# Step 6 — Reliability Signals for FD001

## Reliability-Aware Selective Remaining Useful Life Prediction

### Objective
Construct and validate reliability indicators based on:

1. Sensor-quality behaviour
2. Model predictive uncertainty

These signals will later support selective RUL prediction
(accept prediction vs abstain).

### Experimental discipline
- Frozen FD001 preprocessing
- Frozen GRU predictor from Step 3
- Frozen corruption framework from Step 4
- Step-5 degradation results are not modified
- Official FD001 test set remains untouched

In [ ]:
from pathlib import Path
import json
import random

import numpy as np
import pandas as pd

import torch
import torch.nn as nn


# ---------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ---------------------------------------------------------
# Locate project root
# ---------------------------------------------------------

cwd = Path.cwd()

if (cwd / "02_dataset").exists():
    PROJECT_ROOT = cwd

elif (cwd.parent / "02_dataset").exists():
    PROJECT_ROOT = cwd.parent

else:
    raise FileNotFoundError(
        "Could not locate the project root."
    )


# ---------------------------------------------------------
# Important directories
# ---------------------------------------------------------

PROCESSED_DIR = (
    PROJECT_ROOT
    / "02_dataset"
    / "processed"
    / "FD001"
)

DEGRADATION_DIR = (
    PROJECT_ROOT
    / "05_experiments"
    / "sensor_degradation"
)

MODELS_DIR = (
    PROJECT_ROOT
    / "05_experiments"
    / "models"
)

STEP5_DIR = (
    PROJECT_ROOT
    / "06_results"
    / "step5_degradation_impact"
)

STEP6_DIR = (
    PROJECT_ROOT
    / "06_results"
    / "step6_reliability_signals"
)

STEP6_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ---------------------------------------------------------
# Frozen GRU checkpoint
# ---------------------------------------------------------

GRU_MODEL_PATH = (
    MODELS_DIR
    / "fd001_gru_selected_cap125_window30_seed42.pt"
)


# ---------------------------------------------------------
# Display checks
# ---------------------------------------------------------

print("Project root       :", PROJECT_ROOT)
print("Processed exists   :", PROCESSED_DIR.exists())
print("Degradation exists :", DEGRADATION_DIR.exists())
print("Models exists      :", MODELS_DIR.exists())
print("Step-5 exists      :", STEP5_DIR.exists())
print("Step-6 directory   :", STEP6_DIR)
print("GRU checkpoint     :", GRU_MODEL_PATH.exists())

print("\nPyTorch version :", torch.__version__)
print("CUDA available  :", torch.cuda.is_available())

if torch.cuda.is_available():
    print(
        "GPU             :",
        torch.cuda.get_device_name(0)
    )

In [ ]:
# ---------------------------------------------------------
# Step 6.3 — Inspect frozen preprocessing artifacts
# ---------------------------------------------------------

PREPROCESSING_CONFIG_PATH = (
    PROCESSED_DIR
    / "fd001_preprocessing_config.json"
)


assert PREPROCESSING_CONFIG_PATH.exists(), (
    "Frozen preprocessing configuration not found."
)


with open(
    PREPROCESSING_CONFIG_PATH,
    "r"
) as f:
    preprocessing_config = json.load(f)


print("Preprocessing configuration loaded.")
print("\nConfiguration keys:")

for key in preprocessing_config.keys():
    print(" -", key)


print("\nConfiguration contents:")

for key, value in preprocessing_config.items():

    if isinstance(value, list):

        print(
            f"{key}: "
            f"{len(value)} items"
        )

        print(
            "   ",
            value
        )

    else:

        print(
            f"{key}:",
            value
        )


print("\nFiles currently in processed FD001 directory:")

for path in sorted(
    PROCESSED_DIR.iterdir()
):
    print(
        " -",
        path.name
    )


print("\nFiles currently in sensor-degradation directory:")

for path in sorted(
    DEGRADATION_DIR.iterdir()
):
    print(
        " -",
        path.name
    )

In [ ]:
# ---------------------------------------------------------
# Step 6.4 — Locate critical frozen artifacts
# ---------------------------------------------------------

print("=== Processed FD001 artifacts ===")

for path in sorted(PROCESSED_DIR.iterdir()):

    name = path.name.lower()

    if any(
        keyword in name
        for keyword in [
            "scaler",
            "split",
            "validation",
            "dev_train",
            "config"
        ]
    ):
        print(" -", path.name)


print("\n=== Sensor-degradation artifacts ===")

for path in sorted(DEGRADATION_DIR.iterdir()):

    name = path.name.lower()

    if any(
        keyword in name
        for keyword in [
            "reference",
            "temporal",
            "specification",
            "summary"
        ]
    ):
        print(" -", path.name)


print("\n=== Model artifacts ===")

for path in sorted(MODELS_DIR.iterdir()):

    if "fd001" in path.name.lower():
        print(" -", path.name)

In [ ]:
# ---------------------------------------------------------
# Step 6.5 — Load frozen Step-2 and Step-4 artifacts
# ---------------------------------------------------------

import joblib


# ---------------------------------------------------------
# Paths
# ---------------------------------------------------------

VALIDATION_PATH = (
    PROCESSED_DIR
    / "fd001_validation_scaled.csv"
)

SCALER_PATH = (
    PROCESSED_DIR
    / "fd001_standard_scaler.joblib"
)

SENSOR_REFERENCE_PATH = (
    DEGRADATION_DIR
    / "fd001_sensor_reference_stats.csv"
)

SENSOR_TEMPORAL_PATH = (
    DEGRADATION_DIR
    / "fd001_sensor_temporal_stats.csv"
)

CORRUPTION_SPEC_PATH = (
    DEGRADATION_DIR
    / "fd001_corruption_specification.json"
)


# ---------------------------------------------------------
# Existence checks
# ---------------------------------------------------------

required_paths = [
    VALIDATION_PATH,
    SCALER_PATH,
    SENSOR_REFERENCE_PATH,
    SENSOR_TEMPORAL_PATH,
    CORRUPTION_SPEC_PATH,
    GRU_MODEL_PATH
]

for path in required_paths:
    assert path.exists(), f"Missing required artifact: {path}"


# ---------------------------------------------------------
# Load artifacts
# ---------------------------------------------------------

val_scaled = pd.read_csv(
    VALIDATION_PATH
)

scaler = joblib.load(
    SCALER_PATH
)

sensor_reference_stats = pd.read_csv(
    SENSOR_REFERENCE_PATH,
    index_col=0
)

sensor_temporal_stats = pd.read_csv(
    SENSOR_TEMPORAL_PATH,
    index_col=0
)

with open(
    CORRUPTION_SPEC_PATH,
    "r"
) as f:
    corruption_spec = json.load(f)


# ---------------------------------------------------------
# Frozen configuration
# ---------------------------------------------------------

PRIMARY_INPUT_COLUMNS = (
    preprocessing_config[
        "primary_input_columns"
    ]
)

RETAINED_SENSOR_COLUMNS = [
    column
    for column in PRIMARY_INPUT_COLUMNS
    if column.startswith("sensor_")
]

RETAINED_OPERATING_COLUMNS = [
    column
    for column in PRIMARY_INPUT_COLUMNS
    if column.startswith("op_setting_")
]


# ---------------------------------------------------------
# Basic validation checks
# ---------------------------------------------------------

assert len(PRIMARY_INPUT_COLUMNS) == 17

assert len(RETAINED_SENSOR_COLUMNS) == 15

assert all(
    column in val_scaled.columns
    for column in PRIMARY_INPUT_COLUMNS
)

assert (
    val_scaled[
        "unit_id"
    ].nunique()
    == 20
)


# ---------------------------------------------------------
# Display
# ---------------------------------------------------------

print("Frozen artifacts loaded successfully.")

print(
    "\nValidation shape       :",
    val_scaled.shape
)

print(
    "Validation engines     :",
    val_scaled["unit_id"].nunique()
)

print(
    "Primary inputs         :",
    len(PRIMARY_INPUT_COLUMNS)
)

print(
    "Retained sensors       :",
    len(RETAINED_SENSOR_COLUMNS)
)

print(
    "Operating inputs       :",
    len(RETAINED_OPERATING_COLUMNS)
)

print(
    "\nSensor reference shape :",
    sensor_reference_stats.shape
)

print(
    "Temporal stats shape   :",
    sensor_temporal_stats.shape
)

print(
    "Corruption families    :",
    list(
        corruption_spec[
            "corruption_levels"
        ].keys()
    )
    if "corruption_levels" in corruption_spec
    else list(corruption_spec.keys())
)

print(
    "\nValidation columns:"
)

print(
    val_scaled.columns.tolist()
)

In [ ]:
# ---------------------------------------------------------
# Step 6.5A — Confirm actual corruption families
# ---------------------------------------------------------

EXPECTED_CORRUPTIONS = [
    "noise",
    "bias",
    "drift",
    "dropout",
    "stuck"
]

CORRUPTION_FAMILIES = [
    corruption
    for corruption in EXPECTED_CORRUPTIONS
    if corruption in corruption_spec
]

assert CORRUPTION_FAMILIES == EXPECTED_CORRUPTIONS

assert "levels" in corruption_spec

CORRUPTION_LEVELS = corruption_spec["levels"]


print(
    "Actual corruption families:",
    CORRUPTION_FAMILIES
)

print("\nFrozen severity levels:")

for corruption in CORRUPTION_FAMILIES:
    print(
        corruption,
        ":",
        CORRUPTION_LEVELS[corruption]
    )

In [ ]:
# ---------------------------------------------------------
# Step 6.6 — Reconstruct clean validation sequences
# ---------------------------------------------------------

WINDOW_SIZE = 30
TARGET_COLUMN = "RUL_cap_125"


def create_sequences_with_metadata(
    df,
    feature_columns,
    target_column,
    window_size
):
    """
    Create fixed-length sequences independently for each engine.

    Target and metadata correspond to the final cycle
    of each input window.
    """

    X_sequences = []
    y_targets = []
    metadata_records = []

    df_sorted = (
        df
        .sort_values(
            ["unit_id", "cycle"]
        )
        .reset_index(drop=True)
    )

    for unit_id, engine_df in df_sorted.groupby(
        "unit_id",
        sort=True
    ):

        engine_df = (
            engine_df
            .sort_values("cycle")
            .reset_index(drop=True)
        )

        features = engine_df[
            feature_columns
        ].to_numpy(
            dtype=np.float32
        )

        targets = engine_df[
            target_column
        ].to_numpy(
            dtype=np.float32
        )

        for end_idx in range(
            window_size - 1,
            len(engine_df)
        ):

            start_idx = (
                end_idx
                - window_size
                + 1
            )

            X_sequences.append(
                features[
                    start_idx:end_idx + 1
                ]
            )

            y_targets.append(
                targets[end_idx]
            )

            metadata_records.append(
                {
                    "unit_id":
                        int(
                            engine_df.loc[
                                end_idx,
                                "unit_id"
                            ]
                        ),

                    "cycle":
                        int(
                            engine_df.loc[
                                end_idx,
                                "cycle"
                            ]
                        ),

                    "RUL_raw":
                        float(
                            engine_df.loc[
                                end_idx,
                                "RUL_raw"
                            ]
                        ),

                    "RUL_cap_125":
                        float(
                            engine_df.loc[
                                end_idx,
                                "RUL_cap_125"
                            ]
                        ),

                    "RUL_cap_130":
                        float(
                            engine_df.loc[
                                end_idx,
                                "RUL_cap_130"
                            ]
                        )
                }
            )


    X = np.asarray(
        X_sequences,
        dtype=np.float32
    )

    y = np.asarray(
        y_targets,
        dtype=np.float32
    )

    metadata = pd.DataFrame(
        metadata_records
    )

    return X, y, metadata


X_val_clean, y_val_clean, val_seq_meta = (
    create_sequences_with_metadata(
        df=val_scaled,
        feature_columns=PRIMARY_INPUT_COLUMNS,
        target_column=TARGET_COLUMN,
        window_size=WINDOW_SIZE
    )
)


# ---------------------------------------------------------
# Integrity checks
# ---------------------------------------------------------

assert X_val_clean.shape == (
    3490,
    30,
    17
)

assert y_val_clean.shape == (
    3490,
)

assert len(val_seq_meta) == 3490

assert (
    val_seq_meta["unit_id"].nunique()
    == 20
)

assert np.isfinite(
    X_val_clean
).all()

assert np.isfinite(
    y_val_clean
).all()


# ---------------------------------------------------------
# Display
# ---------------------------------------------------------

print(
    "Validation sequence shape :",
    X_val_clean.shape
)

print(
    "Validation target shape   :",
    y_val_clean.shape
)

print(
    "Metadata rows             :",
    len(val_seq_meta)
)

print(
    "Sequence engines          :",
    val_seq_meta[
        "unit_id"
    ].nunique()
)

print(
    "\nFirst sequence metadata:"
)

display(
    val_seq_meta.head()
)

print(
    "\nLast sequence metadata:"
)

display(
    val_seq_meta.tail()
)

In [ ]:
# ---------------------------------------------------------
# Step 6.7 — Load and verify the frozen selected GRU
# ---------------------------------------------------------

import torch
import torch.nn as nn


DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


class GRURegressor(nn.Module):

    def __init__(
        self,
        input_size=17,
        hidden_size=64,
        dropout=0.2
    ):

        super().__init__()

        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            batch_first=True
        )

        self.regressor = nn.Sequential(
            nn.Linear(
                hidden_size,
                32
            ),
            nn.ReLU(),
            nn.Dropout(
                p=dropout
            ),
            nn.Linear(
                32,
                1
            )
        )


    def forward(self, x):

        gru_output, _ = self.gru(x)

        last_hidden = gru_output[:, -1, :]

        prediction = self.regressor(
            last_hidden
        )

        return prediction.squeeze(-1)


# ---------------------------------------------------------
# Load checkpoint
# ---------------------------------------------------------

checkpoint = torch.load(
    GRU_MODEL_PATH,
    map_location=DEVICE,
    weights_only=True
)


print(
    "Checkpoint keys:",
    list(checkpoint.keys())
)


# ---------------------------------------------------------
# Verify checkpoint metadata
# ---------------------------------------------------------

assert checkpoint["window_size"] == WINDOW_SIZE

assert checkpoint["input_columns"] == PRIMARY_INPUT_COLUMNS

assert checkpoint["target"] == TARGET_COLUMN

assert checkpoint["seed"] == 42


# ---------------------------------------------------------
# Rebuild exact frozen model architecture
# ---------------------------------------------------------

frozen_gru = GRURegressor(
    input_size=len(PRIMARY_INPUT_COLUMNS),
    hidden_size=64,
    dropout=0.2
).to(DEVICE)


frozen_gru.load_state_dict(
    checkpoint["model_state_dict"]
)

frozen_gru.eval()


# ---------------------------------------------------------
# Freeze all model parameters
# ---------------------------------------------------------

for parameter in frozen_gru.parameters():
    parameter.requires_grad = False


# ---------------------------------------------------------
# Verification
# ---------------------------------------------------------

total_parameters = sum(
    p.numel()
    for p in frozen_gru.parameters()
)

trainable_parameters = sum(
    p.numel()
    for p in frozen_gru.parameters()
    if p.requires_grad
)


print(
    "\nDevice              :",
    DEVICE
)

print(
    "Checkpoint best epoch:",
    checkpoint["best_epoch"]
)

print(
    "Checkpoint best RMSE :",
    checkpoint["best_val_rmse"]
)

print(
    "Checkpoint window    :",
    checkpoint["window_size"]
)

print(
    "Checkpoint target    :",
    checkpoint["target"]
)

print(
    "Checkpoint seed      :",
    checkpoint["seed"]
)

print(
    "Total parameters     :",
    total_parameters
)

print(
    "Trainable parameters :",
    trainable_parameters
)

print(
    "\nFrozen selected GRU loaded and verified successfully."
)

In [ ]:
# ---------------------------------------------------------
# Step 6.8 — Reproduce frozen clean-validation baseline
# ---------------------------------------------------------

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)


def predict_gru_batched(
    model,
    X,
    device,
    batch_size=512
):
    """
    Generate deterministic predictions from the frozen GRU.
    """

    model.eval()

    predictions = []

    with torch.no_grad():

        for start in range(
            0,
            len(X),
            batch_size
        ):

            end = min(
                start + batch_size,
                len(X)
            )

            X_batch = torch.tensor(
                X[start:end],
                dtype=torch.float32,
                device=device
            )

            batch_predictions = (
                model(X_batch)
                .detach()
                .cpu()
                .numpy()
            )

            predictions.append(
                batch_predictions
            )

    return np.concatenate(
        predictions
    )


# ---------------------------------------------------------
# Generate clean predictions
# ---------------------------------------------------------

clean_predictions = predict_gru_batched(
    model=frozen_gru,
    X=X_val_clean,
    device=DEVICE
)


# ---------------------------------------------------------
# Metrics
# ---------------------------------------------------------

clean_rmse = np.sqrt(
    mean_squared_error(
        y_val_clean,
        clean_predictions
    )
)

clean_mae = mean_absolute_error(
    y_val_clean,
    clean_predictions
)

clean_r2 = r2_score(
    y_val_clean,
    clean_predictions
)


# ---------------------------------------------------------
# Reproduction checks
# ---------------------------------------------------------

assert clean_predictions.shape == (
    3490,
)

assert np.isfinite(
    clean_predictions
).all()

assert np.isclose(
    clean_rmse,
    checkpoint["best_val_rmse"],
    rtol=1e-5,
    atol=1e-4
)


# ---------------------------------------------------------
# Build clean prediction table
# ---------------------------------------------------------

clean_prediction_df = (
    val_seq_meta.copy()
)

clean_prediction_df[
    "target"
] = y_val_clean

clean_prediction_df[
    "prediction"
] = clean_predictions

clean_prediction_df[
    "signed_error"
] = (
    clean_prediction_df["prediction"]
    - clean_prediction_df["target"]
)

clean_prediction_df[
    "absolute_error"
] = (
    clean_prediction_df[
        "signed_error"
    ].abs()
)


# ---------------------------------------------------------
# Display
# ---------------------------------------------------------

print(
    "Clean predictions:",
    clean_predictions.shape
)

print(
    "\nClean frozen-GRU metrics:"
)

print(
    "RMSE:",
    round(clean_rmse, 6)
)

print(
    "MAE :",
    round(clean_mae, 6)
)

print(
    "R2  :",
    round(clean_r2, 6)
)

print(
    "\nCheckpoint RMSE:",
    checkpoint["best_val_rmse"]
)

print(
    "Absolute RMSE difference:",
    abs(
        clean_rmse
        - checkpoint["best_val_rmse"]
    )
)

print(
    "\nClean baseline reproduced successfully."
)

display(
    clean_prediction_df.head()
)

In [ ]:
# ---------------------------------------------------------
# Step 6.9 — MC-dropout predictive-dispersion signal
# ---------------------------------------------------------

from scipy.stats import (
    pearsonr,
    spearmanr
)


MC_PASSES = 50
MC_SEED = 42


def mc_dropout_predictions(
    model,
    X,
    device,
    n_passes=50,
    batch_size=512,
    seed=42
):
    """
    Generate repeated stochastic predictions with dropout active.

    Model parameters remain frozen.
    The returned standard deviation is used as a
    model-side predictive-dispersion signal.
    """

    # Reproducibility
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # Important:
    # train() activates Dropout.
    # It does NOT retrain the network because:
    #   1. gradients are disabled below
    #   2. all parameters have requires_grad=False
    model.train()

    all_pass_predictions = []

    with torch.no_grad():

        for pass_id in range(n_passes):

            pass_predictions = []

            for start in range(
                0,
                len(X),
                batch_size
            ):

                end = min(
                    start + batch_size,
                    len(X)
                )

                X_batch = torch.tensor(
                    X[start:end],
                    dtype=torch.float32,
                    device=device
                )

                batch_predictions = (
                    model(X_batch)
                    .detach()
                    .cpu()
                    .numpy()
                )

                pass_predictions.append(
                    batch_predictions
                )

            all_pass_predictions.append(
                np.concatenate(
                    pass_predictions
                )
            )

    # Restore deterministic inference mode
    model.eval()

    all_pass_predictions = np.asarray(
        all_pass_predictions,
        dtype=np.float32
    )

    mc_mean = all_pass_predictions.mean(
        axis=0
    )

    mc_std = all_pass_predictions.std(
        axis=0,
        ddof=0
    )

    return (
        all_pass_predictions,
        mc_mean,
        mc_std
    )


(
    mc_predictions,
    mc_mean_prediction,
    mc_std
) = mc_dropout_predictions(
    model=frozen_gru,
    X=X_val_clean,
    device=DEVICE,
    n_passes=MC_PASSES,
    batch_size=512,
    seed=MC_SEED
)


# ---------------------------------------------------------
# Integrity checks
# ---------------------------------------------------------

assert mc_predictions.shape == (
    MC_PASSES,
    3490
)

assert mc_mean_prediction.shape == (
    3490,
)

assert mc_std.shape == (
    3490,
)

assert np.isfinite(
    mc_predictions
).all()

assert np.isfinite(
    mc_std
).all()

assert (
    mc_std >= 0
).all()

assert all(
    not parameter.requires_grad
    for parameter in frozen_gru.parameters()
)


# ---------------------------------------------------------
# Add signal to clean prediction table
# ---------------------------------------------------------

clean_prediction_df[
    "mc_mean_prediction"
] = mc_mean_prediction

clean_prediction_df[
    "mc_dropout_std"
] = mc_std


# ---------------------------------------------------------
# Relationship with deterministic prediction error
# ---------------------------------------------------------

pearson_r, pearson_p = pearsonr(
    clean_prediction_df[
        "mc_dropout_std"
    ],
    clean_prediction_df[
        "absolute_error"
    ]
)

spearman_r, spearman_p = spearmanr(
    clean_prediction_df[
        "mc_dropout_std"
    ],
    clean_prediction_df[
        "absolute_error"
    ]
)


# ---------------------------------------------------------
# Summary
# ---------------------------------------------------------

print(
    "MC passes              :",
    MC_PASSES
)

print(
    "MC prediction matrix   :",
    mc_predictions.shape
)

print(
    "\nMC-dropout std summary:"
)

print(
    "Minimum :",
    round(
        float(mc_std.min()),
        6
    )
)

print(
    "Median  :",
    round(
        float(np.median(mc_std)),
        6
    )
)

print(
    "Mean    :",
    round(
        float(mc_std.mean()),
        6
    )
)

print(
    "Maximum :",
    round(
        float(mc_std.max()),
        6
    )
)

print(
    "\nAssociation with absolute RUL error:"
)

print(
    "Pearson correlation :",
    round(
        float(pearson_r),
        4
    )
)

print(
    "Pearson p-value     :",
    pearson_p
)

print(
    "Spearman correlation:",
    round(
        float(spearman_r),
        4
    )
)

print(
    "Spearman p-value    :",
    spearman_p
)

print(
    "\nTrainable parameters:",
    sum(
        p.numel()
        for p in frozen_gru.parameters()
        if p.requires_grad
    )
)

print(
    "\nMC-dropout reliability signal generated successfully."
)

display(
    clean_prediction_df[
        [
            "unit_id",
            "cycle",
            "RUL_raw",
            "target",
            "prediction",
            "absolute_error",
            "mc_dropout_std"
        ]
    ].head(10)
)

In [ ]:
# ---------------------------------------------------------
# Step 6.10 — MC-dropout uncertainty ranking diagnostic
# ---------------------------------------------------------

mc_ranking_df = clean_prediction_df.copy()


# ---------------------------------------------------------
# Divide samples into 10 uncertainty groups
# 1 = lowest uncertainty
# 10 = highest uncertainty
# ---------------------------------------------------------

mc_ranking_df[
    "mc_uncertainty_decile"
] = (
    pd.qcut(
        mc_ranking_df[
            "mc_dropout_std"
        ],
        q=10,
        labels=False,
        duplicates="drop"
    )
    + 1
)


# ---------------------------------------------------------
# Summarise prediction quality within each uncertainty group
# ---------------------------------------------------------

decile_records = []

for decile, group in mc_ranking_df.groupby(
    "mc_uncertainty_decile"
):

    errors = (
        group["prediction"]
        - group["target"]
    )

    rmse = np.sqrt(
        np.mean(
            errors.to_numpy() ** 2
        )
    )

    mae = np.mean(
        np.abs(
            errors.to_numpy()
        )
    )

    over10_rate = (
        (
            errors.to_numpy()
            > 10
        ).mean()
        * 100
    )

    decile_records.append(
        {
            "uncertainty_decile":
                int(decile),

            "samples":
                len(group),

            "mean_mc_std":
                group[
                    "mc_dropout_std"
                ].mean(),

            "RMSE":
                rmse,

            "MAE":
                mae,

            "over10_rate_pct":
                over10_rate
        }
    )


mc_decile_summary = pd.DataFrame(
    decile_records
)


display(
    mc_decile_summary.round(3)
)


# ---------------------------------------------------------
# Compare lowest 20% with highest 20%
# ---------------------------------------------------------

low_uncertainty = mc_ranking_df[
    mc_ranking_df[
        "mc_uncertainty_decile"
    ] <= 2
]

high_uncertainty = mc_ranking_df[
    mc_ranking_df[
        "mc_uncertainty_decile"
    ] >= 9
]


low_rmse = np.sqrt(
    np.mean(
        (
            low_uncertainty["prediction"]
            - low_uncertainty["target"]
        ).to_numpy() ** 2
    )
)

high_rmse = np.sqrt(
    np.mean(
        (
            high_uncertainty["prediction"]
            - high_uncertainty["target"]
        ).to_numpy() ** 2
    )
)


low_mae = np.mean(
    low_uncertainty["absolute_error"]
)

high_mae = np.mean(
    high_uncertainty["absolute_error"]
)


print(
    "Lowest-20% uncertainty samples:",
    len(low_uncertainty)
)

print(
    "Highest-20% uncertainty samples:",
    len(high_uncertainty)
)

print(
    "\nLowest-20% RMSE :",
    round(low_rmse, 3)
)

print(
    "Highest-20% RMSE:",
    round(high_rmse, 3)
)

print(
    "\nLowest-20% MAE  :",
    round(low_mae, 3)
)

print(
    "Highest-20% MAE :",
    round(high_mae, 3)
)

print(
    "\nRMSE ratio (high / low):",
    round(
        high_rmse / low_rmse,
        3
    )
)

In [ ]:
# ---------------------------------------------------------
# Helper — Create engine-wise sliding-window sequences
# ---------------------------------------------------------

def create_sequences(
    df,
    feature_columns,
    target_column,
    window_size
):
    """
    Create sliding-window sequences separately for each engine.

    The target and metadata correspond to the LAST cycle
    of each window.

    Important:
    Windows never cross engine boundaries.
    """

    X_sequences = []
    y_targets = []
    metadata_records = []

    metadata_columns = [
        column
        for column in [
            "unit_id",
            "cycle",
            "RUL_raw",
            "RUL_cap_125",
            "RUL_cap_130"
        ]
        if column in df.columns
    ]

    for unit_id, engine_df in df.groupby(
        "unit_id",
        sort=True
    ):

        engine_df = (
            engine_df
            .sort_values("cycle")
            .reset_index(drop=True)
        )

        feature_values = (
            engine_df[
                feature_columns
            ]
            .to_numpy(
                dtype=np.float32
            )
        )

        target_values = (
            engine_df[
                target_column
            ]
            .to_numpy(
                dtype=np.float32
            )
        )

        for end_index in range(
            window_size - 1,
            len(engine_df)
        ):

            start_index = (
                end_index
                - window_size
                + 1
            )

            X_sequences.append(
                feature_values[
                    start_index:
                    end_index + 1
                ]
            )

            y_targets.append(
                target_values[
                    end_index
                ]
            )

            metadata_records.append(
                engine_df.loc[
                    end_index,
                    metadata_columns
                ].to_dict()
            )

    X_sequences = np.asarray(
        X_sequences,
        dtype=np.float32
    )

    y_targets = np.asarray(
        y_targets,
        dtype=np.float32
    )

    metadata_df = (
        pd.DataFrame(
            metadata_records
        )
        .reset_index(drop=True)
    )

    return (
        X_sequences,
        y_targets,
        metadata_df
    )


print(
    "create_sequences helper defined successfully."
)

In [ ]:
# ---------------------------------------------------------
# Step 6.11 — Development-only PCA sensor-quality signal
# ---------------------------------------------------------

from sklearn.decomposition import PCA


# ---------------------------------------------------------
# Load CLEAN development data
# ---------------------------------------------------------

dev_scaled = pd.read_csv(
    PROCESSED_DIR
    / "fd001_dev_train_scaled.csv"
)


print(
    "Development rows   :",
    len(dev_scaled)
)

print(
    "Development engines:",
    dev_scaled["unit_id"].nunique()
)


# ---------------------------------------------------------
# Verify retained sensor columns
# ---------------------------------------------------------

assert all(
    sensor in dev_scaled.columns
    for sensor in RETAINED_SENSOR_COLUMNS
)

assert len(
    RETAINED_SENSOR_COLUMNS
) == 15


# ---------------------------------------------------------
# Fit PCA ONLY on clean development sensor measurements
# ---------------------------------------------------------

dev_sensor_matrix = (
    dev_scaled[
        RETAINED_SENSOR_COLUMNS
    ]
    .to_numpy(
        dtype=np.float64
    )
)


sensor_pca = PCA(
    n_components=0.95,
    svd_solver="full"
)


sensor_pca.fit(
    dev_sensor_matrix
)


print(
    "\nOriginal sensor dimensions:",
    dev_sensor_matrix.shape[1]
)

print(
    "PCA retained components    :",
    sensor_pca.n_components_
)

print(
    "Explained variance         :",
    round(
        float(
            sensor_pca.explained_variance_ratio_.sum()
        ),
        6
    )
)


# ---------------------------------------------------------
# Locate sensor channels inside the 17-input GRU sequence
# ---------------------------------------------------------

SENSOR_INPUT_INDICES = [
    PRIMARY_INPUT_COLUMNS.index(
        sensor
    )
    for sensor in RETAINED_SENSOR_COLUMNS
]


print(
    "\nSensor input indices:",
    SENSOR_INPUT_INDICES
)


assert len(
    SENSOR_INPUT_INDICES
) == 15

assert len(
    set(SENSOR_INPUT_INDICES)
) == 15


# ---------------------------------------------------------
# Recreate clean development sequences
# ---------------------------------------------------------

(
    X_dev_clean,
    y_dev_clean,
    dev_seq_meta
) = create_sequences(
    df=dev_scaled,
    feature_columns=PRIMARY_INPUT_COLUMNS,
    target_column=TARGET_COLUMN,
    window_size=WINDOW_SIZE
)


print(
    "\nDevelopment sequence shape:",
    X_dev_clean.shape
)


# ---------------------------------------------------------
# PCA window-quality function
# ---------------------------------------------------------

def calculate_pca_window_quality(
    X,
    sensor_indices,
    fitted_pca
):
    """
    Calculate cross-sensor PCA reconstruction error
    for every 30-cycle sequence.

    Returns:
        row_rmse
        window_mean_rmse
        window_max_rmse
    """

    sensor_windows = X[
        :,
        :,
        sensor_indices
    ]


    n_windows = sensor_windows.shape[0]
    window_length = sensor_windows.shape[1]
    n_sensors = sensor_windows.shape[2]


    flat_sensor_rows = (
        sensor_windows
        .reshape(
            -1,
            n_sensors
        )
        .astype(
            np.float64
        )
    )


    transformed = (
        fitted_pca.transform(
            flat_sensor_rows
        )
    )


    reconstructed = (
        fitted_pca.inverse_transform(
            transformed
        )
    )


    row_rmse = np.sqrt(
        np.mean(
            (
                flat_sensor_rows
                - reconstructed
            ) ** 2,
            axis=1
        )
    )


    row_rmse = row_rmse.reshape(
        n_windows,
        window_length
    )


    window_mean_rmse = (
        row_rmse.mean(
            axis=1
        )
    )


    window_max_rmse = (
        row_rmse.max(
            axis=1
        )
    )


    return (
        row_rmse,
        window_mean_rmse,
        window_max_rmse
    )


# ---------------------------------------------------------
# Development reference scores
# ---------------------------------------------------------

(
    dev_pca_row_rmse,
    dev_pca_mean_rmse,
    dev_pca_max_rmse
) = calculate_pca_window_quality(
    X=X_dev_clean,
    sensor_indices=SENSOR_INPUT_INDICES,
    fitted_pca=sensor_pca
)


# ---------------------------------------------------------
# Clean validation scores
# ---------------------------------------------------------

(
    val_pca_row_rmse,
    val_pca_mean_rmse,
    val_pca_max_rmse
) = calculate_pca_window_quality(
    X=X_val_clean,
    sensor_indices=SENSOR_INPUT_INDICES,
    fitted_pca=sensor_pca
)


# ---------------------------------------------------------
# Convert scores to DEVELOPMENT-reference percentiles
# ---------------------------------------------------------

def reference_percentile(
    reference_values,
    values
):

    sorted_reference = np.sort(
        np.asarray(
            reference_values
        )
    )

    values = np.asarray(
        values
    )

    return (
        np.searchsorted(
            sorted_reference,
            values,
            side="right"
        )
        / len(sorted_reference)
    )


val_pca_mean_percentile = (
    reference_percentile(
        dev_pca_mean_rmse,
        val_pca_mean_rmse
    )
)


val_pca_max_percentile = (
    reference_percentile(
        dev_pca_max_rmse,
        val_pca_max_rmse
    )
)


# ---------------------------------------------------------
# Integrity checks
# ---------------------------------------------------------

assert len(
    val_pca_mean_rmse
) == 3490

assert len(
    val_pca_max_rmse
) == 3490

assert np.isfinite(
    val_pca_mean_rmse
).all()

assert np.isfinite(
    val_pca_max_rmse
).all()

assert (
    val_pca_mean_rmse >= 0
).all()

assert (
    val_pca_max_rmse >= 0
).all()


# ---------------------------------------------------------
# Add sensor-quality signals to prediction table
# ---------------------------------------------------------

clean_prediction_df[
    "pca_sensor_mean_re"
] = val_pca_mean_rmse


clean_prediction_df[
    "pca_sensor_max_re"
] = val_pca_max_rmse


clean_prediction_df[
    "pca_sensor_mean_percentile"
] = val_pca_mean_percentile


clean_prediction_df[
    "pca_sensor_max_percentile"
] = val_pca_max_percentile


# ---------------------------------------------------------
# Summary
# ---------------------------------------------------------

print(
    "\nDevelopment PCA mean-RE:"
)

print(
    "Median:",
    round(
        float(
            np.median(
                dev_pca_mean_rmse
            )
        ),
        6
    )
)

print(
    "95th percentile:",
    round(
        float(
            np.quantile(
                dev_pca_mean_rmse,
                0.95
            )
        ),
        6
    )
)


print(
    "\nClean validation PCA mean-RE:"
)

print(
    "Minimum:",
    round(
        float(
            val_pca_mean_rmse.min()
        ),
        6
    )
)

print(
    "Median :",
    round(
        float(
            np.median(
                val_pca_mean_rmse
            )
        ),
        6
    )
)

print(
    "Maximum:",
    round(
        float(
            val_pca_mean_rmse.max()
        ),
        6
    )
)


print(
    "\nValidation mean-percentile summary:"
)

print(
    "Median:",
    round(
        float(
            np.median(
                val_pca_mean_percentile
            )
        ),
        4
    )
)

print(
    "Samples above dev 95th percentile:",
    int(
        (
            val_pca_mean_percentile
            > 0.95
        ).sum()
    )
)


print(
    "\nPCA sensor-quality signal generated successfully."
)


display(
    clean_prediction_df[
        [
            "unit_id",
            "cycle",
            "target",
            "prediction",
            "absolute_error",
            "mc_dropout_std",
            "pca_sensor_mean_re",
            "pca_sensor_mean_percentile"
        ]
    ].head(10)
)

In [ ]:
# ---------------------------------------------------------
# Step 6.12 — Clean-data behaviour of PCA sensor-quality signal
# ---------------------------------------------------------

from scipy.stats import pearsonr, spearmanr


# ---------------------------------------------------------
# Correlation with absolute RUL prediction error
# ---------------------------------------------------------

pca_pearson_r, pca_pearson_p = pearsonr(
    clean_prediction_df[
        "pca_sensor_mean_re"
    ],
    clean_prediction_df[
        "absolute_error"
    ]
)

pca_spearman_r, pca_spearman_p = spearmanr(
    clean_prediction_df[
        "pca_sensor_mean_re"
    ],
    clean_prediction_df[
        "absolute_error"
    ]
)


print(
    "Association between clean PCA reconstruction error"
    " and absolute RUL error:"
)

print(
    "Pearson correlation :",
    round(
        float(pca_pearson_r),
        4
    )
)

print(
    "Pearson p-value     :",
    pca_pearson_p
)

print(
    "Spearman correlation:",
    round(
        float(pca_spearman_r),
        4
    )
)

print(
    "Spearman p-value    :",
    pca_spearman_p
)


# ---------------------------------------------------------
# PCA-quality deciles
# ---------------------------------------------------------

clean_prediction_df[
    "pca_quality_decile"
] = pd.qcut(
    clean_prediction_df[
        "pca_sensor_mean_re"
    ],
    q=10,
    labels=False,
    duplicates="drop"
) + 1


pca_decile_records = []


for decile, group in (
    clean_prediction_df
    .groupby(
        "pca_quality_decile"
    )
):

    rmse = np.sqrt(
        np.mean(
            (
                group["prediction"]
                - group["target"]
            ) ** 2
        )
    )

    mae = np.mean(
        np.abs(
            group["prediction"]
            - group["target"]
        )
    )

    over10_rate = (
        (
            group["prediction"]
            - group["RUL_raw"]
        )
        > 10
    ).mean() * 100


    pca_decile_records.append(
        {
            "pca_quality_decile":
                int(decile),

            "samples":
                len(group),

            "mean_pca_re":
                group[
                    "pca_sensor_mean_re"
                ].mean(),

            "RMSE":
                rmse,

            "MAE":
                mae,

            "over10_rate_pct":
                over10_rate
        }
    )


pca_decile_df = pd.DataFrame(
    pca_decile_records
)


display(
    pca_decile_df.round(3)
)


# ---------------------------------------------------------
# Compare lowest and highest 20%
# ---------------------------------------------------------

low_threshold = (
    clean_prediction_df[
        "pca_sensor_mean_re"
    ]
    .quantile(0.20)
)


high_threshold = (
    clean_prediction_df[
        "pca_sensor_mean_re"
    ]
    .quantile(0.80)
)


low_pca_group = (
    clean_prediction_df[
        clean_prediction_df[
            "pca_sensor_mean_re"
        ] <= low_threshold
    ]
)


high_pca_group = (
    clean_prediction_df[
        clean_prediction_df[
            "pca_sensor_mean_re"
        ] >= high_threshold
    ]
)


def subset_rmse(df):

    return np.sqrt(
        np.mean(
            (
                df["prediction"]
                - df["target"]
            ) ** 2
        )
    )


def subset_mae(df):

    return np.mean(
        np.abs(
            df["prediction"]
            - df["target"]
        )
    )


low_pca_rmse = subset_rmse(
    low_pca_group
)

high_pca_rmse = subset_rmse(
    high_pca_group
)


low_pca_mae = subset_mae(
    low_pca_group
)

high_pca_mae = subset_mae(
    high_pca_group
)


print(
    "\nLowest-20% PCA-error samples:",
    len(low_pca_group)
)

print(
    "Highest-20% PCA-error samples:",
    len(high_pca_group)
)


print(
    "\nLowest-20% RMSE :",
    round(
        float(low_pca_rmse),
        3
    )
)

print(
    "Highest-20% RMSE:",
    round(
        float(high_pca_rmse),
        3
    )
)


print(
    "\nLowest-20% MAE  :",
    round(
        float(low_pca_mae),
        3
    )
)

print(
    "Highest-20% MAE :",
    round(
        float(high_pca_mae),
        3
    )
)


print(
    "\nRMSE ratio (high / low):",
    round(
        float(
            high_pca_rmse
            / low_pca_rmse
        ),
        3
    )
)

In [ ]:
# ---------------------------------------------------------
# Step 6.13A — Check which Step-4/5 corruption helpers
# are currently available in this notebook
# ---------------------------------------------------------

helper_names = [
    "add_sensor_noise",
    "add_sensor_bias",
    "add_sensor_drift",
    "add_sensor_dropout",
    "add_stuck_sensor",
    "apply_corruption",
    "create_sequences"
]


print("Current helper-function availability:\n")

for name in helper_names:

    available = (
        name in globals()
        and callable(
            globals()[name]
        )
    )

    print(
        f"{name:<22}: {available}"
    )

In [ ]:
# ---------------------------------------------------------
# Step 6.13B — Reconstruct raw validation inputs
# and restore the frozen constant-bias function
# ---------------------------------------------------------

# ---------------------------------------------------------
# 1. Reconstruct original-scale primary inputs
# ---------------------------------------------------------

val_raw_for_corruption = val_scaled.copy()

val_raw_for_corruption[
    PRIMARY_INPUT_COLUMNS
] = scaler.inverse_transform(
    val_scaled[
        PRIMARY_INPUT_COLUMNS
    ]
)


print(
    "Raw validation inputs reconstructed."
)


# ---------------------------------------------------------
# 2. Verify round-trip reconstruction
# ---------------------------------------------------------

roundtrip_scaled = scaler.transform(
    val_raw_for_corruption[
        PRIMARY_INPUT_COLUMNS
    ]
)

max_roundtrip_difference = np.max(
    np.abs(
        roundtrip_scaled
        - val_scaled[
            PRIMARY_INPUT_COLUMNS
        ].to_numpy()
    )
)


print(
    "Maximum scaling round-trip difference:",
    max_roundtrip_difference
)


assert np.allclose(
    roundtrip_scaled,
    val_scaled[
        PRIMARY_INPUT_COLUMNS
    ].to_numpy(),
    rtol=1e-10,
    atol=1e-10
)


# ---------------------------------------------------------
# 3. Restore Step-4 constant-bias corruption
# ---------------------------------------------------------

def add_sensor_bias(
    df,
    sensor,
    alpha,
    direction
):
    """
    Apply a constant sensor bias in the original sensor scale.

    Bias magnitude:
        direction * alpha * development-reference sensor std

    direction must be either -1 or +1.
    """

    if sensor not in RETAINED_SENSOR_COLUMNS:
        raise ValueError(
            f"{sensor} is not a retained sensor."
        )

    if direction not in (-1, 1):
        raise ValueError(
            "direction must be -1 or +1."
        )

    corrupted_df = df.copy()

    reference_std = float(
        sensor_reference_stats.loc[
            sensor,
            "std"
        ]
    )

    bias_amount = (
        direction
        * float(alpha)
        * reference_std
    )

    corrupted_df[
        sensor
    ] = (
        corrupted_df[
            sensor
        ]
        + bias_amount
    )

    return (
        corrupted_df,
        bias_amount
    )


print(
    "Step-4 constant-bias helper restored successfully."
)


# ---------------------------------------------------------
# 4. Small verification case
# ---------------------------------------------------------

TEST_SENSOR = "sensor_9"
TEST_SEVERITY = "high"
TEST_DIRECTION = 1

TEST_ALPHA = (
    CORRUPTION_LEVELS[
        "bias"
    ][
        TEST_SEVERITY
    ]
)


bias_test_df, bias_amount = (
    add_sensor_bias(
        df=val_raw_for_corruption,
        sensor=TEST_SENSOR,
        alpha=TEST_ALPHA,
        direction=TEST_DIRECTION
    )
)


observed_shift = (
    bias_test_df[
        TEST_SENSOR
    ]
    - val_raw_for_corruption[
        TEST_SENSOR
    ]
)


print(
    "\nBias verification:"
)

print(
    "Sensor          :",
    TEST_SENSOR
)

print(
    "Severity        :",
    TEST_SEVERITY
)

print(
    "Alpha           :",
    TEST_ALPHA
)

print(
    "Direction       :",
    TEST_DIRECTION
)

print(
    "Reference std   :",
    sensor_reference_stats.loc[
        TEST_SENSOR,
        "std"
    ]
)

print(
    "Expected shift  :",
    bias_amount
)

print(
    "Observed mean   :",
    observed_shift.mean()
)

print(
    "Observed std    :",
    observed_shift.std(
        ddof=0
    )
)


assert np.isclose(
    observed_shift.mean(),
    bias_amount,
    rtol=1e-10,
    atol=1e-10
)

assert np.isclose(
    observed_shift.std(
        ddof=0
    ),
    0.0,
    atol=1e-10
)


print(
    "\nBias reconstruction and integrity checks passed."
)

In [ ]:
# ---------------------------------------------------------
# Step 6.13C — Inspect currently available PCA objects
# ---------------------------------------------------------

print("PCA-related objects currently in memory:\n")

for name in sorted(globals().keys()):

    if "pca" in name.lower():

        obj = globals()[name]

        print(
            f"{name:<35}"
            f"{type(obj).__name__}"
        )


print("\nImportant Step-6 objects:\n")

important_names = [
    "X_val_clean",
    "y_val_clean",
    "val_seq_meta",
    "SENSOR_INPUT_INDICES",
    "PRIMARY_INPUT_COLUMNS",
    "RETAINED_SENSOR_COLUMNS"
]

for name in important_names:

    if name in globals():

        obj = globals()[name]

        shape = getattr(
            obj,
            "shape",
            None
        )

        print(
            f"{name:<28}"
            f"available=True   "
            f"shape={shape}"
        )

    else:

        print(
            f"{name:<28}"
            f"available=False"
        )

In [ ]:
# ---------------------------------------------------------
# Step 6.13C — Confirm PCA helper signature
# ---------------------------------------------------------

import inspect


print(
    "calculate_pca_window_quality signature:"
)

print(
    inspect.signature(
        calculate_pca_window_quality
    )
)


print("\nSequence objects:")

for name in [
    "X_val_clean",
    "X_dev_clean"
]:

    if name in globals():

        obj = globals()[name]

        print(
            f"{name:<18}: "
            f"available=True, "
            f"shape={obj.shape}"
        )

    else:

        print(
            f"{name:<18}: "
            f"available=False"
        )

In [ ]:
# ---------------------------------------------------------
# Step 6.13D
# Test whether the PCA sensor-quality signal responds
# to high positive bias on sensor_9
# ---------------------------------------------------------


# ---------------------------------------------------------
# 1. Scale the already-corrupted raw validation data
# using the frozen Step-2 scaler
# ---------------------------------------------------------

val_bias_scaled = bias_test_df.copy()

val_bias_scaled[
    PRIMARY_INPUT_COLUMNS
] = scaler.transform(
    bias_test_df[
        PRIMARY_INPUT_COLUMNS
    ]
)


print(
    "Corrupted validation data scaled."
)

print(
    "Missing scaled values:",
    val_bias_scaled[
        PRIMARY_INPUT_COLUMNS
    ].isna().sum().sum()
)

print(
    "Infinite scaled values:",
    np.isinf(
        val_bias_scaled[
            PRIMARY_INPUT_COLUMNS
        ].to_numpy()
    ).sum()
)


# ---------------------------------------------------------
# 2. Recreate sequences from corrupted validation data
# ---------------------------------------------------------

(
    X_val_bias,
    y_val_bias,
    bias_seq_meta
) = create_sequences(
    df=val_bias_scaled,
    feature_columns=PRIMARY_INPUT_COLUMNS,
    target_column=TARGET_COLUMN,
    window_size=WINDOW_SIZE
)


print(
    "\nBias sequence shape:",
    X_val_bias.shape
)

print(
    "Bias target shape:",
    y_val_bias.shape
)


# ---------------------------------------------------------
# 3. Confirm perfect alignment with clean evaluation data
# ---------------------------------------------------------

assert (
    X_val_bias.shape
    == X_val_clean.shape
)

assert np.array_equal(
    y_val_bias,
    y_val_clean
)

assert np.array_equal(
    bias_seq_meta[
        ["unit_id", "cycle"]
    ].to_numpy(),
    val_seq_meta[
        ["unit_id", "cycle"]
    ].to_numpy()
)


print(
    "Clean and biased sequences are perfectly aligned."
)


# ---------------------------------------------------------
# 4. Calculate PCA sensor-quality signals
# ---------------------------------------------------------

(
    bias_pca_row_rmse,
    bias_pca_mean_rmse,
    bias_pca_max_rmse
) = calculate_pca_window_quality(
    X=X_val_bias,
    sensor_indices=SENSOR_INPUT_INDICES,
    fitted_pca=sensor_pca
)


print(
    "\nPCA signals generated for biased data."
)

print(
    "Mean-RMSE shape:",
    bias_pca_mean_rmse.shape
)

print(
    "Max-RMSE shape:",
    bias_pca_max_rmse.shape
)


# ---------------------------------------------------------
# 5. Development-derived clean reference threshold
# ---------------------------------------------------------

DEV_PCA_95 = np.quantile(
    dev_pca_mean_rmse,
    0.95
)


# ---------------------------------------------------------
# 6. Compare clean vs corrupted PCA reconstruction error
# ---------------------------------------------------------

clean_mean = float(
    np.mean(
        val_pca_mean_rmse
    )
)

bias_mean = float(
    np.mean(
        bias_pca_mean_rmse
    )
)


clean_median = float(
    np.median(
        val_pca_mean_rmse
    )
)

bias_median = float(
    np.median(
        bias_pca_mean_rmse
    )
)


mean_change_pct = (
    (
        bias_mean
        - clean_mean
    )
    / clean_mean
    * 100.0
)


median_change_pct = (
    (
        bias_median
        - clean_median
    )
    / clean_median
    * 100.0
)


clean_above_95_pct = (
    np.mean(
        val_pca_mean_rmse
        > DEV_PCA_95
    )
    * 100.0
)


bias_above_95_pct = (
    np.mean(
        bias_pca_mean_rmse
        > DEV_PCA_95
    )
    * 100.0
)


paired_delta = (
    bias_pca_mean_rmse
    - val_pca_mean_rmse
)


# ---------------------------------------------------------
# 7. Print diagnostic comparison
# ---------------------------------------------------------

print(
    "\n=== PCA degradation-response diagnostic ==="
)

print(
    "Corruption       : bias"
)

print(
    "Sensor           : sensor_9"
)

print(
    "Severity         : high"
)

print(
    "Direction        : +1"
)

print(
    "Bias magnitude   :",
    round(
        float(
            bias_amount
        ),
        6
    )
)


print(
    "\nDevelopment PCA mean-RE 95th percentile:",
    round(
        DEV_PCA_95,
        6
    )
)


print(
    "\nClean PCA mean-RE:"
)

print(
    "Mean             :",
    round(
        clean_mean,
        6
    )
)

print(
    "Median           :",
    round(
        clean_median,
        6
    )
)

print(
    "Above dev P95 (%) :",
    round(
        clean_above_95_pct,
        3
    )
)


print(
    "\nBiased PCA mean-RE:"
)

print(
    "Mean             :",
    round(
        bias_mean,
        6
    )
)

print(
    "Median           :",
    round(
        bias_median,
        6
    )
)

print(
    "Above dev P95 (%) :",
    round(
        bias_above_95_pct,
        3
    )
)


print(
    "\nChange caused by corruption:"
)

print(
    "Mean RE change (%)   :",
    round(
        mean_change_pct,
        3
    )
)

print(
    "Median RE change (%) :",
    round(
        median_change_pct,
        3
    )
)

print(
    "Mean paired delta    :",
    round(
        float(
            np.mean(
                paired_delta
            )
        ),
        6
    )
)

print(
    "Median paired delta  :",
    round(
        float(
            np.median(
                paired_delta
            )
        ),
        6
    )
)

print(
    "Fraction increased (%) :",
    round(
        float(
            np.mean(
                paired_delta
                > 0
            )
            * 100.0
        ),
        3
    )
)

In [ ]:
# ---------------------------------------------------------
# Step 6.13E
# Systematic PCA response across all constant-bias scenarios
# ---------------------------------------------------------

bias_pca_records = []

clean_pca_mean = float(
    np.mean(
        val_pca_mean_rmse
    )
)

clean_pca_median = float(
    np.median(
        val_pca_mean_rmse
    )
)

clean_above_dev95_pct = float(
    np.mean(
        val_pca_mean_rmse
        > DEV_PCA_95
    )
    * 100.0
)


total_bias_scenarios = (
    len(RETAINED_SENSOR_COLUMNS)
    * len(CORRUPTION_LEVELS["bias"])
    * 2
)


scenario_counter = 0


for sensor in RETAINED_SENSOR_COLUMNS:

    for severity, alpha in (
        CORRUPTION_LEVELS[
            "bias"
        ].items()
    ):

        for direction in (-1, 1):

            scenario_counter += 1

            # ---------------------------------------------
            # Inject bias in original sensor scale
            # ---------------------------------------------

            corrupted_raw, applied_bias = (
                add_sensor_bias(
                    df=val_raw_for_corruption,
                    sensor=sensor,
                    alpha=alpha,
                    direction=direction
                )
            )


            # ---------------------------------------------
            # Scale using frozen Step-2 scaler
            # ---------------------------------------------

            corrupted_scaled = (
                corrupted_raw.copy()
            )

            corrupted_scaled[
                PRIMARY_INPUT_COLUMNS
            ] = scaler.transform(
                corrupted_raw[
                    PRIMARY_INPUT_COLUMNS
                ]
            )


            # ---------------------------------------------
            # Recreate sequences
            # ---------------------------------------------

            (
                X_corrupted,
                y_corrupted,
                corrupted_meta
            ) = create_sequences(
                df=corrupted_scaled,
                feature_columns=PRIMARY_INPUT_COLUMNS,
                target_column=TARGET_COLUMN,
                window_size=WINDOW_SIZE
            )


            # ---------------------------------------------
            # Alignment checks
            # ---------------------------------------------

            assert (
                X_corrupted.shape
                == X_val_clean.shape
            )

            assert np.array_equal(
                y_corrupted,
                y_val_clean
            )

            assert np.array_equal(
                corrupted_meta[
                    ["unit_id", "cycle"]
                ].to_numpy(),
                val_seq_meta[
                    ["unit_id", "cycle"]
                ].to_numpy()
            )


            # ---------------------------------------------
            # PCA sensor-quality signal
            # ---------------------------------------------

            (
                _,
                corrupted_mean_re,
                _
            ) = calculate_pca_window_quality(
                X=X_corrupted,
                sensor_indices=SENSOR_INPUT_INDICES,
                fitted_pca=sensor_pca
            )


            corrupted_mean = float(
                np.mean(
                    corrupted_mean_re
                )
            )

            corrupted_median = float(
                np.median(
                    corrupted_mean_re
                )
            )

            mean_change_pct = (
                (
                    corrupted_mean
                    - clean_pca_mean
                )
                / clean_pca_mean
                * 100.0
            )

            median_change_pct = (
                (
                    corrupted_median
                    - clean_pca_median
                )
                / clean_pca_median
                * 100.0
            )


            paired_delta = (
                corrupted_mean_re
                - val_pca_mean_rmse
            )


            fraction_increased_pct = float(
                np.mean(
                    paired_delta > 0
                )
                * 100.0
            )


            above_dev95_pct = float(
                np.mean(
                    corrupted_mean_re
                    > DEV_PCA_95
                )
                * 100.0
            )


            bias_pca_records.append(
                {
                    "sensor": sensor,
                    "severity": severity,
                    "alpha": float(alpha),
                    "direction": int(direction),
                    "bias_amount": float(
                        applied_bias
                    ),
                    "clean_mean_re": clean_pca_mean,
                    "corrupted_mean_re": corrupted_mean,
                    "mean_re_change_pct": mean_change_pct,
                    "median_re_change_pct": median_change_pct,
                    "mean_paired_delta": float(
                        np.mean(
                            paired_delta
                        )
                    ),
                    "fraction_increased_pct":
                        fraction_increased_pct,
                    "clean_above_dev95_pct":
                        clean_above_dev95_pct,
                    "corrupted_above_dev95_pct":
                        above_dev95_pct,
                    "above_dev95_delta_pp":
                        above_dev95_pct
                        - clean_above_dev95_pct
                }
            )


            if (
                scenario_counter % 15 == 0
                or scenario_counter
                == total_bias_scenarios
            ):

                print(
                    f"[{scenario_counter:>2}/"
                    f"{total_bias_scenarios}] "
                    f"completed"
                )


# ---------------------------------------------------------
# Results table
# ---------------------------------------------------------

bias_pca_response_df = pd.DataFrame(
    bias_pca_records
)


print(
    "\nBias PCA-response scenarios:",
    len(
        bias_pca_response_df
    )
)


assert (
    len(
        bias_pca_response_df
    )
    == 90
)


assert (
    bias_pca_response_df[
        [
            "corrupted_mean_re",
            "mean_re_change_pct",
            "median_re_change_pct",
            "fraction_increased_pct",
            "corrupted_above_dev95_pct"
        ]
    ]
    .notna()
    .all()
    .all()
)


# ---------------------------------------------------------
# Summarize by severity
# ---------------------------------------------------------

bias_pca_severity_summary = (
    bias_pca_response_df
    .groupby(
        "severity"
    )
    .agg(
        scenarios=(
            "sensor",
            "size"
        ),
        mean_RE_change_pct=(
            "mean_re_change_pct",
            "mean"
        ),
        median_RE_change_pct=(
            "median_re_change_pct",
            "median"
        ),
        mean_fraction_increased_pct=(
            "fraction_increased_pct",
            "mean"
        ),
        mean_above_dev95_pct=(
            "corrupted_above_dev95_pct",
            "mean"
        ),
        mean_above_dev95_delta_pp=(
            "above_dev95_delta_pp",
            "mean"
        )
    )
    .reindex(
        [
            "low",
            "medium",
            "high"
        ]
    )
)


print(
    "\n=== Bias PCA response by severity ==="
)

display(
    bias_pca_severity_summary
    .round(3)
)


# ---------------------------------------------------------
# Strongest PCA responses
# ---------------------------------------------------------

print(
    "\n=== Strongest 15 bias responses ==="
)

display(
    bias_pca_response_df
    .sort_values(
        "mean_re_change_pct",
        ascending=False
    )
    [
        [
            "sensor",
            "severity",
            "direction",
            "bias_amount",
            "mean_re_change_pct",
            "median_re_change_pct",
            "fraction_increased_pct",
            "corrupted_above_dev95_pct",
            "above_dev95_delta_pp"
        ]
    ]
    .head(15)
    .round(3)
)


# ---------------------------------------------------------
# Weakest PCA responses
# ---------------------------------------------------------

print(
    "\n=== Weakest 15 bias responses ==="
)

display(
    bias_pca_response_df
    .sort_values(
        "mean_re_change_pct",
        ascending=True
    )
    [
        [
            "sensor",
            "severity",
            "direction",
            "bias_amount",
            "mean_re_change_pct",
            "median_re_change_pct",
            "fraction_increased_pct",
            "corrupted_above_dev95_pct",
            "above_dev95_delta_pp"
        ]
    ]
    .head(15)
    .round(3)
)


# ---------------------------------------------------------
# Save Step-6 diagnostic artifact
# ---------------------------------------------------------

bias_pca_path = (
    STEP6_DIR
    / "fd001_pca_bias_response_90_scenarios.csv"
)

bias_pca_response_df.to_csv(
    bias_pca_path,
    index=False
)


print(
    "\nSaved:",
    bias_pca_path
)

In [ ]:
# ---------------------------------------------------------
# Step 6.13F
# Sensor-wise summary of PCA sensitivity to constant bias
# ---------------------------------------------------------

bias_sensor_summary_df = (
    bias_pca_response_df
    .groupby("sensor")
    .agg(
        scenarios=(
            "sensor",
            "size"
        ),

        mean_RE_change_pct=(
            "mean_re_change_pct",
            "mean"
        ),

        median_RE_change_pct=(
            "median_re_change_pct",
            "median"
        ),

        min_RE_change_pct=(
            "mean_re_change_pct",
            "min"
        ),

        max_RE_change_pct=(
            "mean_re_change_pct",
            "max"
        ),

        mean_fraction_increased_pct=(
            "fraction_increased_pct",
            "mean"
        ),

        mean_above_dev95_pct=(
            "corrupted_above_dev95_pct",
            "mean"
        ),

        max_above_dev95_pct=(
            "corrupted_above_dev95_pct",
            "max"
        )
    )
    .sort_values(
        "mean_RE_change_pct",
        ascending=False
    )
)


print(
    "=== Sensor-wise PCA sensitivity "
    "across all bias scenarios ==="
)

display(
    bias_sensor_summary_df
    .round(3)
)


# ---------------------------------------------------------
# Integrity checks
# ---------------------------------------------------------

assert len(
    bias_sensor_summary_df
) == len(
    RETAINED_SENSOR_COLUMNS
)

assert (
    bias_sensor_summary_df[
        "scenarios"
    ]
    == 6
).all()


print(
    "\nAll 15 sensors contain "
    "the expected 6 bias scenarios."
)


# ---------------------------------------------------------
# Save
# ---------------------------------------------------------

bias_sensor_summary_path = (
    STEP6_DIR
    / "fd001_pca_bias_sensor_summary.csv"
)

bias_sensor_summary_df.to_csv(
    bias_sensor_summary_path
)


print(
    "\nSaved:",
    bias_sensor_summary_path
)

In [ ]:
# =========================================================
# Step 6.14A
# Restore the frozen Step-4 additive-noise corruption
# =========================================================

def add_sensor_noise(
    df,
    sensor,
    alpha,
    reference_stats,
    seed=42
):
    """
    Add zero-mean Gaussian noise to one sensor.

    Noise standard deviation:
        alpha * development-reference sensor std

    The input dataframe is not modified.
    """

    corrupted = df.copy()

    reference_std = float(
        reference_stats.loc[
            sensor,
            "std"
        ]
    )

    target_noise_std = (
        alpha * reference_std
    )

    rng = np.random.default_rng(
        seed
    )

    injected_noise = rng.normal(
        loc=0.0,
        scale=target_noise_std,
        size=len(corrupted)
    )

    corrupted[sensor] = (
        corrupted[sensor].to_numpy()
        + injected_noise
    )

    return (
        corrupted,
        injected_noise
    )


print(
    "Step-4 additive-noise helper "
    "restored successfully."
)

In [ ]:
# =========================================================
# Step 6.14B
# Verify restored additive-noise helper against Step 4
# =========================================================

NOISE_TEST_SENSOR = "sensor_2"
NOISE_TEST_SEVERITY = "medium"

NOISE_TEST_ALPHA = (
    CORRUPTION_LEVELS["noise"][NOISE_TEST_SEVERITY]
)

# ---------------------------------------------------------
# Reconstruct clean validation data in original/raw scale
# ---------------------------------------------------------

noise_check_raw = val_scaled.copy()

noise_check_raw[
    PRIMARY_INPUT_COLUMNS
] = scaler.inverse_transform(
    val_scaled[
        PRIMARY_INPUT_COLUMNS
    ]
)

# ---------------------------------------------------------
# Apply restored Step-4 noise corruption
# ---------------------------------------------------------

noise_check_corrupted, injected_noise = (
    add_sensor_noise(
        df=noise_check_raw,
        sensor=NOISE_TEST_SENSOR,
        alpha=NOISE_TEST_ALPHA,
        reference_stats=sensor_reference_stats,
        seed=42
    )
)

reference_std = float(
    sensor_reference_stats.loc[
        NOISE_TEST_SENSOR,
        "std"
    ]
)

target_noise_std = (
    NOISE_TEST_ALPHA
    * reference_std
)

actual_noise_std = (
    injected_noise.std(ddof=0)
)

actual_noise_mean = (
    injected_noise.mean()
)

# ---------------------------------------------------------
# Load the frozen Step-4 validation result
# ---------------------------------------------------------

frozen_noise_validation = pd.read_csv(
    DEGRADATION_DIR
    / "fd001_noise_validation.csv"
)

frozen_row = (
    frozen_noise_validation[
        (
            frozen_noise_validation["sensor"]
            == NOISE_TEST_SENSOR
        )
        &
        (
            frozen_noise_validation["severity"]
            == NOISE_TEST_SEVERITY
        )
    ]
    .iloc[0]
)

print("Noise verification:")
print("Sensor              :", NOISE_TEST_SENSOR)
print("Severity            :", NOISE_TEST_SEVERITY)
print("Alpha               :", NOISE_TEST_ALPHA)
print("Reference std       :", reference_std)
print("Target noise std    :", target_noise_std)
print("Actual noise std    :", actual_noise_std)
print("Actual noise mean   :", actual_noise_mean)

print("\nFrozen Step-4 values:")
print(
    "Frozen target std   :",
    frozen_row["target_noise_std"]
)
print(
    "Frozen actual std   :",
    frozen_row["actual_noise_std"]
)

# ---------------------------------------------------------
# Exact consistency checks
# ---------------------------------------------------------

assert np.isclose(
    target_noise_std,
    frozen_row["target_noise_std"],
    rtol=1e-10,
    atol=1e-12
)

assert np.isclose(
    actual_noise_std,
    frozen_row["actual_noise_std"],
    rtol=1e-10,
    atol=1e-12
)

# Only the selected sensor may change
unchanged_columns = [
    col
    for col in noise_check_raw.columns
    if col != NOISE_TEST_SENSOR
]

pd.testing.assert_frame_equal(
    noise_check_corrupted[unchanged_columns],
    noise_check_raw[unchanged_columns]
)

# The change in the selected sensor must equal
# the generated noise exactly
assert np.allclose(
    (
        noise_check_corrupted[NOISE_TEST_SENSOR]
        - noise_check_raw[NOISE_TEST_SENSOR]
    ).to_numpy(),
    injected_noise,
    rtol=1e-12,
    atol=1e-12
)

print(
    "\nStep-4 additive-noise reconstruction "
    "verified successfully."
)

In [ ]:
# ============================================================
# Step 6.15
# PCA sensor-quality response under additive sensor noise
# FD001 — all 15 retained sensors × 3 severity levels
# ============================================================

# This cell:
# 1. reconstructs raw validation inputs from the frozen scaler,
# 2. applies the frozen Step-4 additive-noise definition,
# 3. recreates exactly aligned sequences,
# 4. evaluates the already-fitted development PCA,
# 5. measures the PCA quality-signal response,
# 6. saves scenario-level and sensor-level results.
#
# IMPORTANT:
# - GRU is NOT retrained.
# - PCA is NOT refitted.
# - scaler is NOT refitted.
# - corruption parameters are NOT changed.
# - validation targets are NOT changed.
# ============================================================


# ------------------------------------------------------------
# 1. Frozen additive-noise configuration
# ------------------------------------------------------------

NOISE_SEED = int(
    corruption_spec["corruption_seed"]
)

NOISE_LEVELS = dict(
    CORRUPTION_LEVELS["noise"]
)

print("Noise seed:", NOISE_SEED)
print("Noise severity levels:", NOISE_LEVELS)
print("Retained sensors:", len(RETAINED_SENSOR_COLUMNS))

assert len(RETAINED_SENSOR_COLUMNS) == 15

assert set(NOISE_LEVELS.keys()) == {
    "low",
    "medium",
    "high"
}


# ------------------------------------------------------------
# 2. Reconstruct RAW clean validation data
#
# val_scaled is our frozen processed validation frame.
# Sensor corruption must be applied in original sensor units,
# so we inverse-transform only the 17 model inputs.
# ------------------------------------------------------------

val_raw_for_corruption = (
    val_scaled
    .copy(deep=True)
)

val_raw_for_corruption[
    PRIMARY_INPUT_COLUMNS
] = scaler.inverse_transform(
    val_scaled[
        PRIMARY_INPUT_COLUMNS
    ]
)


# ------------------------------------------------------------
# 3. Verify scaling round-trip
# ------------------------------------------------------------

roundtrip_scaled = scaler.transform(
    val_raw_for_corruption[
        PRIMARY_INPUT_COLUMNS
    ]
)

roundtrip_difference = float(
    np.max(
        np.abs(
            roundtrip_scaled
            -
            val_scaled[
                PRIMARY_INPUT_COLUMNS
            ].to_numpy(
                dtype=float
            )
        )
    )
)

print(
    "\nMaximum scaling round-trip difference:",
    roundtrip_difference
)

assert roundtrip_difference < 1e-10

print(
    "Raw validation inputs reconstructed successfully."
)


# ------------------------------------------------------------
# 4. Keep an immutable clean reference copy
# ------------------------------------------------------------

clean_raw_reference = (
    val_raw_for_corruption
    .copy(deep=True)
)


# ------------------------------------------------------------
# 5. PCA clean-validation reference
# ------------------------------------------------------------

dev_pca_mean_p95 = float(
    np.percentile(
        dev_pca_mean_rmse,
        95
    )
)

clean_mean_re = float(
    np.mean(
        val_pca_mean_rmse
    )
)

clean_median_re = float(
    np.median(
        val_pca_mean_rmse
    )
)

clean_above_dev95_pct = float(
    100.0
    * np.mean(
        val_pca_mean_rmse
        > dev_pca_mean_p95
    )
)

print("\nClean PCA reference:")

print(
    "Development PCA mean-RE 95th percentile:",
    round(
        dev_pca_mean_p95,
        6
    )
)

print(
    "Clean validation mean PCA RE:",
    round(
        clean_mean_re,
        6
    )
)

print(
    "Clean validation median PCA RE:",
    round(
        clean_median_re,
        6
    )
)

print(
    "Clean validation above dev-P95 (%):",
    round(
        clean_above_dev95_pct,
        3
    )
)


# ------------------------------------------------------------
# 6. Metadata normalization
#
# create_sequences may represent unit_id/cycle as float in a
# reconstructed DataFrame although values are still exact
# integers. We verify this explicitly and normalize ONLY those
# identifier columns before strict comparison.
# ------------------------------------------------------------

def normalize_sequence_metadata(metadata_df):

    normalized = (
        metadata_df
        .reset_index(drop=True)
        .copy()
    )

    for column in [
        "unit_id",
        "cycle"
    ]:

        if column in normalized.columns:

            values = pd.to_numeric(
                normalized[column],
                errors="raise"
            ).to_numpy(
                dtype=float
            )

            assert np.isfinite(
                values
            ).all()

            assert np.allclose(
                values,
                np.round(values),
                rtol=0.0,
                atol=1e-12
            ), (
                f"{column} contains "
                "non-integer identifier values."
            )

            normalized[column] = (
                np.round(values)
                .astype(np.int64)
            )

    # Remaining metadata columns are continuous numeric values.
    for column in normalized.columns:

        if column not in [
            "unit_id",
            "cycle"
        ]:

            normalized[column] = (
                pd.to_numeric(
                    normalized[column],
                    errors="raise"
                )
                .astype(np.float64)
            )

    return normalized


clean_meta_check = (
    normalize_sequence_metadata(
        val_seq_meta
    )
)

print(
    "\nClean sequence metadata prepared "
    "for strict alignment checking."
)


# ------------------------------------------------------------
# 7. Expected experimental design
# ------------------------------------------------------------

EXPECTED_NOISE_SCENARIOS = (
    len(RETAINED_SENSOR_COLUMNS)
    * len(NOISE_LEVELS)
)

assert EXPECTED_NOISE_SCENARIOS == 45

print(
    "\nExpected noise scenarios:",
    EXPECTED_NOISE_SCENARIOS
)


# ------------------------------------------------------------
# 8. Run all 45 additive-noise scenarios
# ------------------------------------------------------------

noise_pca_records = []

scenario_counter = 0


for sensor in RETAINED_SENSOR_COLUMNS:

    reference_std = float(
        sensor_reference_stats.loc[
            sensor,
            "std"
        ]
    )

    assert np.isfinite(reference_std)

    assert reference_std > 0.0

    for severity in [
        "low",
        "medium",
        "high"
    ]:

        scenario_counter += 1

        alpha = float(
            NOISE_LEVELS[
                severity
            ]
        )


        # ----------------------------------------------------
        # 8A. Apply frozen Step-4 additive sensor noise
        # ----------------------------------------------------

        (
            val_noise_raw,
            injected_noise
        ) = add_sensor_noise(
            df=val_raw_for_corruption,
            sensor=sensor,
            alpha=alpha,
            seed=NOISE_SEED
        )

        injected_noise = np.asarray(
            injected_noise,
            dtype=float
        )


        # ----------------------------------------------------
        # 8B. Verify corruption magnitude
        # ----------------------------------------------------

        target_noise_std = float(
            alpha
            * reference_std
        )

        actual_noise_std = float(
            injected_noise.std(
                ddof=0
            )
        )

        actual_noise_mean = float(
            injected_noise.mean()
        )


        # ----------------------------------------------------
        # 8C. Verify ONLY selected sensor changed
        # ----------------------------------------------------

        other_input_columns = [
            column
            for column
            in PRIMARY_INPUT_COLUMNS
            if column != sensor
        ]

        np.testing.assert_allclose(
            val_noise_raw[
                other_input_columns
            ].to_numpy(
                dtype=float
            ),
            clean_raw_reference[
                other_input_columns
            ].to_numpy(
                dtype=float
            ),
            rtol=0.0,
            atol=0.0
        )


        # Selected-sensor difference must equal injected noise.
        observed_noise = (
            val_noise_raw[
                sensor
            ].to_numpy(
                dtype=float
            )
            -
            clean_raw_reference[
                sensor
            ].to_numpy(
                dtype=float
            )
        )

        np.testing.assert_allclose(
            observed_noise,
            injected_noise,
            rtol=1e-12,
            atol=1e-12
        )


        # ----------------------------------------------------
        # 8D. Verify clean reference was NOT modified
        # ----------------------------------------------------

        np.testing.assert_allclose(
            val_raw_for_corruption[
                PRIMARY_INPUT_COLUMNS
            ].to_numpy(
                dtype=float
            ),
            clean_raw_reference[
                PRIMARY_INPUT_COLUMNS
            ].to_numpy(
                dtype=float
            ),
            rtol=0.0,
            atol=0.0
        )


        # ----------------------------------------------------
        # 8E. Scale corrupted validation data using the
        #     original frozen scaler
        # ----------------------------------------------------

        val_noise_scaled = (
            val_noise_raw
            .copy(deep=True)
        )

        val_noise_scaled[
            PRIMARY_INPUT_COLUMNS
        ] = scaler.transform(
            val_noise_raw[
                PRIMARY_INPUT_COLUMNS
            ]
        )


        missing_scaled = int(
            val_noise_scaled[
                PRIMARY_INPUT_COLUMNS
            ]
            .isna()
            .sum()
            .sum()
        )

        infinite_scaled = int(
            np.isinf(
                val_noise_scaled[
                    PRIMARY_INPUT_COLUMNS
                ]
                .to_numpy(
                    dtype=float
                )
            ).sum()
        )

        assert missing_scaled == 0
        assert infinite_scaled == 0


        # ----------------------------------------------------
        # 8F. Recreate corrupted sequences
        # ----------------------------------------------------

        (
            X_noise,
            y_noise,
            noise_seq_meta
        ) = create_sequences(
            df=val_noise_scaled,
            feature_columns=PRIMARY_INPUT_COLUMNS,
            target_column=TARGET_COLUMN,
            window_size=WINDOW_SIZE
        )


        # ----------------------------------------------------
        # 8G. Strict clean-corrupted sequence alignment
        # ----------------------------------------------------

        assert (
            X_noise.shape
            ==
            X_val_clean.shape
        )

        assert (
            y_noise.shape
            ==
            y_val_clean.shape
        )

        np.testing.assert_array_equal(
            y_noise,
            y_val_clean
        )

        noise_meta_check = (
            normalize_sequence_metadata(
                noise_seq_meta
            )
        )

        assert (
            noise_meta_check.columns.tolist()
            ==
            clean_meta_check.columns.tolist()
        )

        pd.testing.assert_frame_equal(
            noise_meta_check,
            clean_meta_check,
            check_dtype=True,
            check_exact=True
        )


        # ----------------------------------------------------
        # 8H. Calculate PCA reconstruction-error signal
        #
        # sensor_pca was fitted earlier using DEVELOPMENT data.
        # No refitting occurs here.
        # ----------------------------------------------------

        (
            noise_pca_row_rmse,
            noise_pca_mean_rmse,
            noise_pca_max_rmse
        ) = calculate_pca_window_quality(
            X_noise,
            SENSOR_INPUT_INDICES,
            sensor_pca
        )

        assert (
            noise_pca_mean_rmse.shape
            ==
            val_pca_mean_rmse.shape
        )

        assert np.isfinite(
            noise_pca_mean_rmse
        ).all()


        # ----------------------------------------------------
        # 8I. Paired change relative to clean validation
        # ----------------------------------------------------

        paired_re_delta = (
            noise_pca_mean_rmse
            -
            val_pca_mean_rmse
        )

        safe_clean_re = np.maximum(
            val_pca_mean_rmse,
            1e-12
        )

        paired_re_change_pct = (
            100.0
            * paired_re_delta
            / safe_clean_re
        )

        mean_re_delta = float(
            np.mean(
                paired_re_delta
            )
        )

        median_re_delta = float(
            np.median(
                paired_re_delta
            )
        )

        mean_re_change_pct = float(
            np.mean(
                paired_re_change_pct
            )
        )

        median_re_change_pct = float(
            np.median(
                paired_re_change_pct
            )
        )

        fraction_increased_pct = float(
            100.0
            * np.mean(
                paired_re_delta > 0
            )
        )

        corrupted_above_dev95_pct = float(
            100.0
            * np.mean(
                noise_pca_mean_rmse
                > dev_pca_mean_p95
            )
        )

        above_dev95_delta_pp = float(
            corrupted_above_dev95_pct
            -
            clean_above_dev95_pct
        )


        # ----------------------------------------------------
        # 8J. Record scenario result
        # ----------------------------------------------------

        noise_pca_records.append(
            {
                "sensor":
                    sensor,

                "severity":
                    severity,

                "alpha":
                    alpha,

                "seed":
                    NOISE_SEED,

                "reference_std":
                    reference_std,

                "target_noise_std":
                    target_noise_std,

                "actual_noise_std":
                    actual_noise_std,

                "actual_noise_mean":
                    actual_noise_mean,

                "clean_mean_re":
                    clean_mean_re,

                "corrupted_mean_re":
                    float(
                        np.mean(
                            noise_pca_mean_rmse
                        )
                    ),

                "clean_median_re":
                    clean_median_re,

                "corrupted_median_re":
                    float(
                        np.median(
                            noise_pca_mean_rmse
                        )
                    ),

                "mean_re_delta":
                    mean_re_delta,

                "median_re_delta":
                    median_re_delta,

                "mean_re_change_pct":
                    mean_re_change_pct,

                "median_re_change_pct":
                    median_re_change_pct,

                "fraction_increased_pct":
                    fraction_increased_pct,

                "clean_above_dev95_pct":
                    clean_above_dev95_pct,

                "corrupted_above_dev95_pct":
                    corrupted_above_dev95_pct,

                "above_dev95_delta_pp":
                    above_dev95_delta_pp
            }
        )


        if (
            scenario_counter % 15 == 0
            or
            scenario_counter
            ==
            EXPECTED_NOISE_SCENARIOS
        ):

            print(
                f"[{scenario_counter}/"
                f"{EXPECTED_NOISE_SCENARIOS}] "
                "completed"
            )


# ------------------------------------------------------------
# 9. Scenario-results DataFrame
# ------------------------------------------------------------

noise_pca_df = pd.DataFrame(
    noise_pca_records
)


# ------------------------------------------------------------
# 10. Complete experimental-integrity checks
# ------------------------------------------------------------

assert len(noise_pca_df) == 45

assert (
    noise_pca_df[
        "sensor"
    ].nunique()
    == 15
)

assert (
    noise_pca_df[
        "severity"
    ].nunique()
    == 3
)

assert set(
    noise_pca_df[
        "severity"
    ].unique()
) == {
    "low",
    "medium",
    "high"
}

sensor_scenario_counts = (
    noise_pca_df
    .groupby(
        "sensor"
    )
    .size()
)

assert (
    sensor_scenario_counts == 3
).all()

assert np.isfinite(
    noise_pca_df
    .select_dtypes(
        include=[np.number]
    )
    .to_numpy(
        dtype=float
    )
).all()


print(
    "\nNoise PCA-response scenarios:",
    len(noise_pca_df)
)

print(
    "All 45 additive-noise "
    "PCA-response scenarios verified."
)


# ------------------------------------------------------------
# 11. Verify frozen noise magnitude behaviour
# ------------------------------------------------------------

noise_ratio = (
    noise_pca_df[
        "actual_noise_std"
    ]
    /
    noise_pca_df[
        "target_noise_std"
    ]
)

print(
    "\nActual/target noise-std ratio:"
)

print(
    "Minimum:",
    float(
        noise_ratio.min()
    )
)

print(
    "Maximum:",
    float(
        noise_ratio.max()
    )
)

# Step 4 previously produced ~0.996975 due to the fixed
# finite random realization. This check allows tiny variation
# while still detecting an incorrect corruption implementation.
assert (
    noise_ratio
    .between(
        0.98,
        1.02
    )
    .all()
)

print(
    "Frozen additive-noise magnitude "
    "checks passed."
)


# ------------------------------------------------------------
# 12. Severity-wise PCA response
# ------------------------------------------------------------

noise_severity_summary = (
    noise_pca_df
    .groupby(
        "severity",
        sort=False
    )
    .agg(
        scenarios=(
            "sensor",
            "count"
        ),

        mean_RE_change_pct=(
            "mean_re_change_pct",
            "mean"
        ),

        median_RE_change_pct=(
            "median_re_change_pct",
            "median"
        ),

        mean_fraction_increased_pct=(
            "fraction_increased_pct",
            "mean"
        ),

        mean_above_dev95_pct=(
            "corrupted_above_dev95_pct",
            "mean"
        ),

        mean_above_dev95_delta_pp=(
            "above_dev95_delta_pp",
            "mean"
        )
    )
    .reindex(
        [
            "low",
            "medium",
            "high"
        ]
    )
)

print(
    "\n=== Noise PCA response by severity ==="
)

display(
    noise_severity_summary
    .round(3)
)


# ------------------------------------------------------------
# 13. Strongest individual noise responses
# ------------------------------------------------------------

strongest_noise_pca = (
    noise_pca_df
    .sort_values(
        "mean_re_change_pct",
        ascending=False
    )
    .head(15)
)

print(
    "\n=== Strongest 15 noise PCA responses ==="
)

display(
    strongest_noise_pca[
        [
            "sensor",
            "severity",
            "alpha",
            "target_noise_std",
            "actual_noise_std",
            "mean_re_change_pct",
            "median_re_change_pct",
            "fraction_increased_pct",
            "corrupted_above_dev95_pct",
            "above_dev95_delta_pp"
        ]
    ]
    .round(3)
)


# ------------------------------------------------------------
# 14. Weakest individual noise responses
# ------------------------------------------------------------

weakest_noise_pca = (
    noise_pca_df
    .sort_values(
        "mean_re_change_pct",
        ascending=True
    )
    .head(15)
)

print(
    "\n=== Weakest 15 noise PCA responses ==="
)

display(
    weakest_noise_pca[
        [
            "sensor",
            "severity",
            "alpha",
            "target_noise_std",
            "actual_noise_std",
            "mean_re_change_pct",
            "median_re_change_pct",
            "fraction_increased_pct",
            "corrupted_above_dev95_pct",
            "above_dev95_delta_pp"
        ]
    ]
    .round(3)
)


# ------------------------------------------------------------
# 15. Sensor-wise PCA sensitivity under noise
# ------------------------------------------------------------

noise_sensor_summary = (
    noise_pca_df
    .groupby(
        "sensor"
    )
    .agg(
        scenarios=(
            "severity",
            "count"
        ),

        mean_RE_change_pct=(
            "mean_re_change_pct",
            "mean"
        ),

        median_RE_change_pct=(
            "median_re_change_pct",
            "median"
        ),

        min_RE_change_pct=(
            "mean_re_change_pct",
            "min"
        ),

        max_RE_change_pct=(
            "mean_re_change_pct",
            "max"
        ),

        mean_fraction_increased_pct=(
            "fraction_increased_pct",
            "mean"
        ),

        mean_above_dev95_pct=(
            "corrupted_above_dev95_pct",
            "mean"
        ),

        max_above_dev95_pct=(
            "corrupted_above_dev95_pct",
            "max"
        )
    )
    .sort_values(
        "mean_RE_change_pct",
        ascending=False
    )
)

assert (
    noise_sensor_summary[
        "scenarios"
    ]
    == 3
).all()

print(
    "\n=== Sensor-wise PCA sensitivity "
    "across all noise scenarios ==="
)

display(
    noise_sensor_summary
    .round(3)
)


# ------------------------------------------------------------
# 16. Save results
# ------------------------------------------------------------

NOISE_PCA_RESULTS_PATH = (
    STEP6_DIR
    / "fd001_pca_noise_response_45_scenarios.csv"
)

NOISE_PCA_SENSOR_SUMMARY_PATH = (
    STEP6_DIR
    / "fd001_pca_noise_sensor_summary.csv"
)

NOISE_PCA_SEVERITY_SUMMARY_PATH = (
    STEP6_DIR
    / "fd001_pca_noise_severity_summary.csv"
)


noise_pca_df.to_csv(
    NOISE_PCA_RESULTS_PATH,
    index=False
)

noise_sensor_summary.reset_index().to_csv(
    NOISE_PCA_SENSOR_SUMMARY_PATH,
    index=False
)

noise_severity_summary.reset_index().to_csv(
    NOISE_PCA_SEVERITY_SUMMARY_PATH,
    index=False
)


print(
    "\nSaved scenario-level results:"
)

print(
    NOISE_PCA_RESULTS_PATH
)

print(
    "\nSaved sensor-level summary:"
)

print(
    NOISE_PCA_SENSOR_SUMMARY_PATH
)

print(
    "\nSaved severity-level summary:"
)

print(
    NOISE_PCA_SEVERITY_SUMMARY_PATH
)


# ------------------------------------------------------------
# 17. FINAL Step 6.15 confirmation
# ------------------------------------------------------------

print(
    "\nStep 6.15 completed successfully."
)

In [ ]:
# ============================================================
# STEP 6.15 — PRE-CHECK
# Verify the exact objects/functions currently available
# DO NOT modify any data
# ============================================================

import inspect

objects_to_check = [
    "val_scaled",
    "scaler",
    "sensor_reference_stats",
    "sensor_temporal_stats",
    "sensor_pca",
    "X_val_clean",
    "y_val_clean",
    "val_seq_meta",
    "PRIMARY_INPUT_COLUMNS",
    "RETAINED_SENSOR_COLUMNS",
    "SENSOR_INPUT_INDICES",
    "CORRUPTION_LEVELS",
    "add_sensor_noise",
    "create_sequences",
    "calculate_pca_window_quality",
]

print("=== Step 6.15 pre-check ===\n")

for name in objects_to_check:
    available = name in globals()
    print(f"{name:<30}: {available}")

print("\n=== Function signatures ===")

for name in [
    "add_sensor_noise",
    "create_sequences",
    "calculate_pca_window_quality",
]:
    if name in globals():
        print(f"{name}: {inspect.signature(globals()[name])}")

print("\n=== Important shapes ===")

if "val_scaled" in globals():
    print("val_scaled shape :", val_scaled.shape)

if "X_val_clean" in globals():
    print("X_val_clean shape:", X_val_clean.shape)

if "y_val_clean" in globals():
    print("y_val_clean shape:", y_val_clean.shape)

if "val_seq_meta" in globals():
    print("val_seq_meta shape:", val_seq_meta.shape)

if "sensor_reference_stats" in globals():
    print(
        "sensor_reference_stats shape:",
        sensor_reference_stats.shape
    )

print("\nPre-check completed.")

In [ ]:
# ============================================================
# Step 6.15A — Additive-noise PCA signal: one-scenario smoke test
# Purpose:
#   Verify the complete corruption -> scaling -> sequencing ->
#   PCA-quality pipeline before running all 45 noise scenarios.
#
# IMPORTANT:
#   This cell does NOT modify val_scaled or X_val_clean.
# ============================================================

import numpy as np


print("=== Step 6.15A — Additive-noise smoke test ===")


# ------------------------------------------------------------
# 1. Frozen test configuration
# ------------------------------------------------------------

TEST_SENSOR = "sensor_2"
TEST_SEVERITY = "medium"

TEST_ALPHA = float(
    CORRUPTION_LEVELS["noise"][TEST_SEVERITY]
)

TEST_SEED = 42

TARGET_COLUMN = "RUL_cap_125"
WINDOW_SIZE = 30


print("Sensor        :", TEST_SENSOR)
print("Severity      :", TEST_SEVERITY)
print("Alpha         :", TEST_ALPHA)
print("Seed          :", TEST_SEED)
print("Target        :", TARGET_COLUMN)
print("Window size   :", WINDOW_SIZE)


# ------------------------------------------------------------
# 2. Basic frozen-configuration checks
# ------------------------------------------------------------

assert TEST_SENSOR in RETAINED_SENSOR_COLUMNS
assert TEST_SENSOR in PRIMARY_INPUT_COLUMNS

assert TARGET_COLUMN in val_scaled.columns

assert len(PRIMARY_INPUT_COLUMNS) == 17
assert len(RETAINED_SENSOR_COLUMNS) == 15
assert len(SENSOR_INPUT_INDICES) == 15

assert X_val_clean.shape == (3490, 30, 17)
assert len(y_val_clean) == 3490
assert len(val_seq_meta) == 3490


print("\nFrozen configuration checks passed.")


# ------------------------------------------------------------
# 3. Reconstruct validation data in ORIGINAL sensor units
#
# val_scaled itself is NOT modified.
# ------------------------------------------------------------

val_raw_for_noise = val_scaled.copy(deep=True)

val_raw_for_noise[
    PRIMARY_INPUT_COLUMNS
] = scaler.inverse_transform(
    val_scaled[
        PRIMARY_INPUT_COLUMNS
    ]
)


# Verify inverse -> forward scaling round trip
round_trip_scaled = scaler.transform(
    val_raw_for_noise[
        PRIMARY_INPUT_COLUMNS
    ]
)

original_scaled_values = (
    val_scaled[
        PRIMARY_INPUT_COLUMNS
    ]
    .to_numpy(dtype=float)
)

round_trip_difference = np.max(
    np.abs(
        round_trip_scaled
        - original_scaled_values
    )
)


print(
    "\nMaximum scaling round-trip difference:",
    round_trip_difference
)

assert round_trip_difference < 1e-10


print(
    "Raw validation inputs reconstructed successfully."
)


# ------------------------------------------------------------
# 4. Apply the EXACT frozen Step-4 noise helper
# ------------------------------------------------------------

(
    val_noise_raw,
    injected_noise
) = add_sensor_noise(
    df=val_raw_for_noise.copy(deep=True),
    sensor=TEST_SENSOR,
    alpha=TEST_ALPHA,
    reference_stats=sensor_reference_stats,
    seed=TEST_SEED
)


injected_noise = np.asarray(
    injected_noise,
    dtype=float
)


print(
    "\nInjected-noise vector shape:",
    injected_noise.shape
)

assert injected_noise.shape[0] == len(
    val_raw_for_noise
)


# ------------------------------------------------------------
# 5. Verify that ONLY the selected sensor changed
# ------------------------------------------------------------

other_primary_columns = [
    column
    for column in PRIMARY_INPUT_COLUMNS
    if column != TEST_SENSOR
]


assert np.allclose(
    val_noise_raw[
        other_primary_columns
    ].to_numpy(dtype=float),

    val_raw_for_noise[
        other_primary_columns
    ].to_numpy(dtype=float),

    rtol=0.0,
    atol=1e-12
)


observed_noise = (
    val_noise_raw[
        TEST_SENSOR
    ].to_numpy(dtype=float)
    -
    val_raw_for_noise[
        TEST_SENSOR
    ].to_numpy(dtype=float)
)


assert np.allclose(
    observed_noise,
    injected_noise,
    rtol=0.0,
    atol=1e-12
)


print(
    "Single-sensor corruption integrity check passed."
)

print(
    "Injected-noise mean:",
    float(np.mean(injected_noise))
)

print(
    "Injected-noise std :",
    float(np.std(injected_noise, ddof=0))
)


# ------------------------------------------------------------
# 6. Scale corrupted data using the SAME frozen scaler
# ------------------------------------------------------------

val_noise_scaled = val_noise_raw.copy(deep=True)

val_noise_scaled[
    PRIMARY_INPUT_COLUMNS
] = scaler.transform(
    val_noise_raw[
        PRIMARY_INPUT_COLUMNS
    ]
)


noise_scaled_values = (
    val_noise_scaled[
        PRIMARY_INPUT_COLUMNS
    ]
    .to_numpy(dtype=float)
)


assert np.isfinite(
    noise_scaled_values
).all()


print(
    "\nCorrupted validation data scaled successfully."
)


# ------------------------------------------------------------
# 7. Recreate 30-cycle sequences
# ------------------------------------------------------------

(
    X_noise,
    y_noise,
    noise_seq_meta
) = create_sequences(
    df=val_noise_scaled,
    feature_columns=PRIMARY_INPUT_COLUMNS,
    target_column=TARGET_COLUMN,
    window_size=WINDOW_SIZE
)


print(
    "\nNoise sequence shape:",
    X_noise.shape
)

print(
    "Noise target shape  :",
    y_noise.shape
)

print(
    "Noise metadata shape:",
    noise_seq_meta.shape
)


assert X_noise.shape == X_val_clean.shape

assert np.array_equal(
    y_noise,
    y_val_clean
)


# ------------------------------------------------------------
# 8. Strict metadata alignment check
#
# Compare numeric VALUES rather than pandas dtypes.
# This avoids the previous int64/float64 false failure.
# ------------------------------------------------------------

assert list(
    noise_seq_meta.columns
) == list(
    val_seq_meta.columns
)


noise_meta_values = (
    noise_seq_meta
    .to_numpy(dtype=float)
)

clean_meta_values = (
    val_seq_meta
    .to_numpy(dtype=float)
)


assert np.allclose(
    noise_meta_values,
    clean_meta_values,
    rtol=0.0,
    atol=1e-12
)


print(
    "Clean and corrupted sequences are perfectly aligned."
)


# ------------------------------------------------------------
# 9. Generate PCA sensor-quality outputs
#
# We intentionally inspect the returned object here before
# building the complete 45-scenario loop.
# ------------------------------------------------------------

pca_output = calculate_pca_window_quality(
    X_noise,
    SENSOR_INPUT_INDICES,
    sensor_pca
)


print(
    "\nPCA function return type:",
    type(pca_output).__name__
)


if isinstance(
    pca_output,
    (tuple, list)
):
    print(
        "Number of PCA outputs:",
        len(pca_output)
    )

    for output_index, output_value in enumerate(
        pca_output,
        start=1
    ):
        output_array = np.asarray(
            output_value
        )

        print(
            f"PCA output {output_index} shape:",
            output_array.shape
        )

        print(
            f"PCA output {output_index} min :",
            float(np.min(output_array))
        )

        print(
            f"PCA output {output_index} mean:",
            float(np.mean(output_array))
        )

        print(
            f"PCA output {output_index} max :",
            float(np.max(output_array))
        )

else:
    output_array = np.asarray(
        pca_output
    )

    print(
        "PCA output shape:",
        output_array.shape
    )


print(
    "\nStep 6.15A smoke test completed successfully."
)

In [ ]:
print("=== PCA output structure ===")
print("Type:", type(pca_output).__name__)

if isinstance(pca_output, (tuple, list)):
    print("Number of outputs:", len(pca_output))

    for i, item in enumerate(pca_output, start=1):
        arr = np.asarray(item)
        print(
            f"Output {i}: "
            f"shape={arr.shape}, "
            f"dtype={arr.dtype}"
        )
else:
    arr = np.asarray(pca_output)
    print("Single output shape:", arr.shape)

In [ ]:
# ============================================================
# Step 6.15 — Additive-noise PCA response across 45 scenarios
# ============================================================

import numpy as np
import pandas as pd

print("=== Step 6.15 — Full additive-noise PCA response ===")

# ------------------------------------------------------------
# 1. Frozen configuration
# ------------------------------------------------------------

NOISE_SEED = 42
TARGET_COLUMN = "RUL_cap_125"
WINDOW_SIZE = 30

NOISE_LEVELS = CORRUPTION_LEVELS["noise"]

print("Noise seed:", NOISE_SEED)
print("Noise levels:", NOISE_LEVELS)
print("Retained sensors:", len(RETAINED_SENSOR_COLUMNS))

assert len(RETAINED_SENSOR_COLUMNS) == 15
assert set(NOISE_LEVELS.keys()) == {"low", "medium", "high"}
assert TARGET_COLUMN == "RUL_cap_125"
assert WINDOW_SIZE == 30

print("Frozen configuration verified.")


# ------------------------------------------------------------
# 2. Reconstruct raw validation inputs safely
# ------------------------------------------------------------

val_raw_for_noise = val_scaled.copy()

val_raw_for_noise[
    PRIMARY_INPUT_COLUMNS
] = scaler.inverse_transform(
    val_scaled[
        PRIMARY_INPUT_COLUMNS
    ]
)

# Verify inverse-transform / transform round trip
round_trip = scaler.transform(
    val_raw_for_noise[
        PRIMARY_INPUT_COLUMNS
    ]
)

round_trip_difference = np.max(
    np.abs(
        round_trip
        - val_scaled[
            PRIMARY_INPUT_COLUMNS
        ].to_numpy()
    )
)

print(
    "\nMaximum scaling round-trip difference:",
    round_trip_difference
)

assert round_trip_difference < 1e-10

print("Raw validation inputs reconstructed successfully.")


# ------------------------------------------------------------
# 3. Recompute clean PCA reference
# ------------------------------------------------------------

(
    clean_pca_row_re,
    clean_pca_mean_re,
    clean_pca_max_re
) = calculate_pca_window_quality(
    X_val_clean,
    SENSOR_INPUT_INDICES,
    sensor_pca
)

clean_pca_row_re = np.asarray(
    clean_pca_row_re,
    dtype=float
)

clean_pca_mean_re = np.asarray(
    clean_pca_mean_re,
    dtype=float
)

clean_pca_max_re = np.asarray(
    clean_pca_max_re,
    dtype=float
)

assert clean_pca_row_re.shape == (
    len(X_val_clean),
    WINDOW_SIZE
)

assert clean_pca_mean_re.shape == (
    len(X_val_clean),
)

assert clean_pca_max_re.shape == (
    len(X_val_clean),
)


# ------------------------------------------------------------
# 4. Development-only PCA threshold
# ------------------------------------------------------------

# This quantity was generated earlier when PCA was fitted
# using development data only.
assert "dev_pca_mean_rmse" in globals(), (
    "dev_pca_mean_rmse is not available. "
    "Please do not continue until the previous PCA cell "
    "has been run."
)

dev_pca_mean_rmse_array = np.asarray(
    dev_pca_mean_rmse,
    dtype=float
)

DEV_PCA_P95 = float(
    np.percentile(
        dev_pca_mean_rmse_array,
        95
    )
)

clean_above_dev95_pct = float(
    100.0
    * np.mean(
        clean_pca_mean_re
        > DEV_PCA_P95
    )
)

print("\nClean PCA reference:")
print(
    "Development PCA mean-RE 95th percentile:",
    round(DEV_PCA_P95, 6)
)
print(
    "Clean validation mean PCA RE:",
    round(
        clean_pca_mean_re.mean(),
        6
    )
)
print(
    "Clean validation median PCA RE:",
    round(
        np.median(clean_pca_mean_re),
        6
    )
)
print(
    "Clean validation above dev-P95 (%):",
    round(
        clean_above_dev95_pct,
        3
    )
)


# ------------------------------------------------------------
# 5. Prepare clean metadata for alignment checking
# ------------------------------------------------------------

ALIGNMENT_COLUMNS = [
    "unit_id",
    "cycle",
    "RUL_raw",
    "RUL_cap_125",
    "RUL_cap_130"
]

clean_meta_check = (
    val_seq_meta[
        ALIGNMENT_COLUMNS
    ]
    .astype(float)
    .reset_index(drop=True)
)

assert len(clean_meta_check) == len(X_val_clean)

print(
    "\nClean sequence metadata prepared "
    "for strict alignment checking."
)


# ------------------------------------------------------------
# 6. Run all 45 additive-noise scenarios
# ------------------------------------------------------------

noise_response_records = []

total_scenarios = (
    len(RETAINED_SENSOR_COLUMNS)
    * len(NOISE_LEVELS)
)

scenario_counter = 0

for sensor in RETAINED_SENSOR_COLUMNS:

    for severity, alpha in NOISE_LEVELS.items():

        scenario_counter += 1

        alpha = float(alpha)

        # ----------------------------------------------------
        # 6A. Apply frozen Step-4 additive noise
        # ----------------------------------------------------

        (
            val_noise_raw,
            injected_noise
        ) = add_sensor_noise(
            df=val_raw_for_noise,
            sensor=sensor,
            alpha=alpha,
            reference_stats=sensor_reference_stats,
            seed=NOISE_SEED
        )

        injected_noise = np.asarray(
            injected_noise,
            dtype=float
        )

        assert len(injected_noise) == len(
            val_raw_for_noise
        )


        # ----------------------------------------------------
        # 6B. Verify only intended sensor changed
        # ----------------------------------------------------

        other_input_columns = [
            column
            for column in PRIMARY_INPUT_COLUMNS
            if column != sensor
        ]

        assert np.allclose(
            val_noise_raw[
                other_input_columns
            ].to_numpy(),
            val_raw_for_noise[
                other_input_columns
            ].to_numpy(),
            rtol=0.0,
            atol=1e-12
        )


        # ----------------------------------------------------
        # 6C. Scale corrupted data with frozen scaler
        # ----------------------------------------------------

        val_noise_scaled = val_noise_raw.copy()

        val_noise_scaled[
            PRIMARY_INPUT_COLUMNS
        ] = scaler.transform(
            val_noise_raw[
                PRIMARY_INPUT_COLUMNS
            ]
        )

        scaled_values = (
            val_noise_scaled[
                PRIMARY_INPUT_COLUMNS
            ]
            .to_numpy()
        )

        assert not np.isnan(
            scaled_values
        ).any()

        assert np.isfinite(
            scaled_values
        ).all()


        # ----------------------------------------------------
        # 6D. Recreate corrupted 30-cycle sequences
        # ----------------------------------------------------

        (
            X_noise,
            y_noise,
            noise_seq_meta
        ) = create_sequences(
            df=val_noise_scaled,
            feature_columns=PRIMARY_INPUT_COLUMNS,
            target_column=TARGET_COLUMN,
            window_size=WINDOW_SIZE
        )

        assert X_noise.shape == X_val_clean.shape
        assert y_noise.shape == y_val_clean.shape

        assert np.array_equal(
            y_noise,
            y_val_clean
        )


        # ----------------------------------------------------
        # 6E. Strict clean/corrupted metadata alignment
        #
        # We explicitly convert metadata to float before
        # comparison so harmless pandas dtype differences
        # such as int64 vs float64 cannot create a false error.
        # ----------------------------------------------------

        noise_meta_check = (
            noise_seq_meta[
                ALIGNMENT_COLUMNS
            ]
            .astype(float)
            .reset_index(drop=True)
        )

        np.testing.assert_allclose(
            noise_meta_check.to_numpy(),
            clean_meta_check.to_numpy(),
            rtol=0.0,
            atol=0.0
        )


        # ----------------------------------------------------
        # 6F. Calculate PCA sensor-quality signals
        # ----------------------------------------------------

        (
            noise_pca_row_re,
            noise_pca_mean_re,
            noise_pca_max_re
        ) = calculate_pca_window_quality(
            X_noise,
            SENSOR_INPUT_INDICES,
            sensor_pca
        )

        noise_pca_row_re = np.asarray(
            noise_pca_row_re,
            dtype=float
        )

        noise_pca_mean_re = np.asarray(
            noise_pca_mean_re,
            dtype=float
        )

        noise_pca_max_re = np.asarray(
            noise_pca_max_re,
            dtype=float
        )

        assert noise_pca_row_re.shape == (
            len(X_val_clean),
            WINDOW_SIZE
        )

        assert noise_pca_mean_re.shape == (
            len(X_val_clean),
        )

        assert noise_pca_max_re.shape == (
            len(X_val_clean),
        )


        # ----------------------------------------------------
        # 6G. Paired clean-vs-corrupted PCA response
        # ----------------------------------------------------

        paired_delta = (
            noise_pca_mean_re
            - clean_pca_mean_re
        )

        denominator = np.maximum(
            np.abs(clean_pca_mean_re),
            1e-12
        )

        paired_change_pct = (
            100.0
            * paired_delta
            / denominator
        )

        fraction_increased_pct = float(
            100.0
            * np.mean(
                paired_delta > 0
            )
        )

        corrupted_above_dev95_pct = float(
            100.0
            * np.mean(
                noise_pca_mean_re
                > DEV_PCA_P95
            )
        )

        above_dev95_delta_pp = float(
            corrupted_above_dev95_pct
            - clean_above_dev95_pct
        )


        # ----------------------------------------------------
        # 6H. Store one scenario record
        # ----------------------------------------------------

        noise_response_records.append(
            {
                "sensor": sensor,
                "severity": severity,
                "alpha": alpha,
                "seed": NOISE_SEED,

                "injected_noise_mean":
                    float(
                        np.mean(
                            injected_noise
                        )
                    ),

                "injected_noise_std":
                    float(
                        np.std(
                            injected_noise,
                            ddof=0
                        )
                    ),

                "clean_mean_re":
                    float(
                        np.mean(
                            clean_pca_mean_re
                        )
                    ),

                "corrupted_mean_re":
                    float(
                        np.mean(
                            noise_pca_mean_re
                        )
                    ),

                "mean_re_change_pct":
                    float(
                        np.mean(
                            paired_change_pct
                        )
                    ),

                "median_re_change_pct":
                    float(
                        np.median(
                            paired_change_pct
                        )
                    ),

                "fraction_increased_pct":
                    fraction_increased_pct,

                "clean_above_dev95_pct":
                    clean_above_dev95_pct,

                "corrupted_above_dev95_pct":
                    corrupted_above_dev95_pct,

                "above_dev95_delta_pp":
                    above_dev95_delta_pp,

                "clean_mean_max_re":
                    float(
                        np.mean(
                            clean_pca_max_re
                        )
                    ),

                "corrupted_mean_max_re":
                    float(
                        np.mean(
                            noise_pca_max_re
                        )
                    )
            }
        )


        if (
            scenario_counter % 15 == 0
            or scenario_counter
            == total_scenarios
        ):
            print(
                f"[{scenario_counter}/"
                f"{total_scenarios}] completed"
            )


# ------------------------------------------------------------
# 7. Build result table
# ------------------------------------------------------------

noise_response_df = pd.DataFrame(
    noise_response_records
)

assert len(noise_response_df) == 45

assert (
    noise_response_df[
        ["sensor", "severity"]
    ]
    .duplicated()
    .sum()
    == 0
)

assert (
    noise_response_df["sensor"].nunique()
    == 15
)

assert (
    noise_response_df["severity"].nunique()
    == 3
)

assert (
    noise_response_df
    .isna()
    .sum()
    .sum()
    == 0
)

numeric_values = (
    noise_response_df
    .select_dtypes(
        include=[np.number]
    )
    .to_numpy()
)

assert np.isfinite(
    numeric_values
).all()

print(
    "\nNoise PCA-response scenarios:",
    len(noise_response_df)
)

print(
    "Unique sensors:",
    noise_response_df[
        "sensor"
    ].nunique()
)

print(
    "Unique severities:",
    noise_response_df[
        "severity"
    ].nunique()
)


# ------------------------------------------------------------
# 8. Severity-level summary
# ------------------------------------------------------------

noise_severity_summary = (
    noise_response_df
    .groupby(
        "severity"
    )
    .agg(
        scenarios=(
            "sensor",
            "count"
        ),

        mean_RE_change_pct=(
            "mean_re_change_pct",
            "mean"
        ),

        median_RE_change_pct=(
            "median_re_change_pct",
            "median"
        ),

        mean_fraction_increased_pct=(
            "fraction_increased_pct",
            "mean"
        ),

        mean_above_dev95_pct=(
            "corrupted_above_dev95_pct",
            "mean"
        ),

        mean_above_dev95_delta_pp=(
            "above_dev95_delta_pp",
            "mean"
        )
    )
    .reindex(
        [
            "low",
            "medium",
            "high"
        ]
    )
)

print(
    "\n=== Additive-noise PCA response "
    "by severity ==="
)

display(
    noise_severity_summary.round(3)
)


# ------------------------------------------------------------
# 9. Sensor-wise summary
# ------------------------------------------------------------

noise_sensor_summary = (
    noise_response_df
    .groupby(
        "sensor"
    )
    .agg(
        scenarios=(
            "severity",
            "count"
        ),

        mean_RE_change_pct=(
            "mean_re_change_pct",
            "mean"
        ),

        median_RE_change_pct=(
            "median_re_change_pct",
            "median"
        ),

        min_RE_change_pct=(
            "mean_re_change_pct",
            "min"
        ),

        max_RE_change_pct=(
            "mean_re_change_pct",
            "max"
        ),

        mean_fraction_increased_pct=(
            "fraction_increased_pct",
            "mean"
        ),

        mean_above_dev95_pct=(
            "corrupted_above_dev95_pct",
            "mean"
        ),

        max_above_dev95_pct=(
            "corrupted_above_dev95_pct",
            "max"
        )
    )
    .sort_values(
        "mean_RE_change_pct",
        ascending=False
    )
)

assert (
    noise_sensor_summary[
        "scenarios"
    ]
    == 3
).all()

print(
    "\n=== Sensor-wise PCA sensitivity "
    "across additive-noise scenarios ==="
)

display(
    noise_sensor_summary.round(3)
)


# ------------------------------------------------------------
# 10. Strongest individual noise responses
# ------------------------------------------------------------

strongest_noise_responses = (
    noise_response_df
    .sort_values(
        "mean_re_change_pct",
        ascending=False
    )
    .head(15)
)

print(
    "\n=== Strongest 15 additive-noise "
    "PCA responses ==="
)

display(
    strongest_noise_responses.round(3)
)


# ------------------------------------------------------------
# 11. Save Step-6 noise outputs
# ------------------------------------------------------------

noise_response_path = (
    STEP6_DIR
    / "fd001_pca_noise_response_45_scenarios.csv"
)

noise_sensor_summary_path = (
    STEP6_DIR
    / "fd001_pca_noise_sensor_summary.csv"
)

noise_severity_summary_path = (
    STEP6_DIR
    / "fd001_pca_noise_severity_summary.csv"
)

noise_response_df.to_csv(
    noise_response_path,
    index=False
)

noise_sensor_summary.reset_index().to_csv(
    noise_sensor_summary_path,
    index=False
)

noise_severity_summary.reset_index().to_csv(
    noise_severity_summary_path,
    index=False
)

print("\nSaved:")
print(noise_response_path)
print(noise_sensor_summary_path)
print(noise_severity_summary_path)


# ------------------------------------------------------------
# 12. Final completion check
# ------------------------------------------------------------

print(
    "\nStep 6.15 completed successfully."
)
print(
    "All 45 frozen additive-noise scenarios "
    "were evaluated using the PCA "
    "sensor-quality signal."
)

In [ ]:
# ============================================================
# STEP 6.16 PRE-CHECK
# Purpose:
# Inspect the exact objects needed for PCA evaluation of
# gradual sensor drift before running any corruption experiment.
# This cell does NOT modify data or saved results.
# ============================================================

import inspect

print("=== Step 6.16 pre-check: gradual drift ===\n")

objects_to_check = [
    "add_sensor_drift",
    "CORRUPTION_LEVELS",
    "sensor_temporal_stats",
    "sensor_reference_stats",
    "val_scaled",
    "scaler",
    "create_sequences",
    "calculate_pca_window_quality",
    "sensor_pca",
    "X_val_clean",
    "y_val_clean",
    "val_seq_meta",
    "PRIMARY_INPUT_COLUMNS",
    "RETAINED_SENSOR_COLUMNS",
    "SENSOR_INPUT_INDICES",
]

for name in objects_to_check:
    available = name in globals()
    print(f"{name:<30}: {available}")

print("\n=== Function signatures ===")

for function_name in [
    "add_sensor_drift",
    "create_sequences",
    "calculate_pca_window_quality",
]:
    if function_name in globals():
        obj = globals()[function_name]

        if callable(obj):
            print(
                f"{function_name}: "
                f"{inspect.signature(obj)}"
            )

print("\n=== Frozen drift configuration ===")

if "CORRUPTION_LEVELS" in globals():
    print(
        "Drift severity levels:",
        CORRUPTION_LEVELS.get("drift", "NOT FOUND")
    )

print("\n=== Temporal-statistics information ===")

if "sensor_temporal_stats" in globals():
    print(
        "sensor_temporal_stats shape:",
        sensor_temporal_stats.shape
    )
    print(
        "Columns:",
        sensor_temporal_stats.columns.tolist()
    )

    display(
        sensor_temporal_stats.head()
    )

print("\n=== Important validation shapes ===")

for name in [
    "val_scaled",
    "X_val_clean",
    "y_val_clean",
    "val_seq_meta",
]:
    if name in globals():
        obj = globals()[name]
        print(
            f"{name:<20}:",
            getattr(obj, "shape", None)
        )

print("\nStep 6.16 pre-check completed.")

In [ ]:
# ============================================================
# STEP 6.16A
# Retrieve the EXACT frozen Step-4 add_sensor_drift function
# from the saved Step-4 notebook.
#
# READ-ONLY:
# - does not execute Step 4
# - does not modify files
# - does not change data/results
# ============================================================

from pathlib import Path
import json

print("=== Step 6.16A — Locate frozen add_sensor_drift ===\n")

# ------------------------------------------------------------
# 1. Locate project root robustly
# ------------------------------------------------------------

current_dir = Path.cwd()

if (current_dir / "03_notebooks").exists():
    project_root_616 = current_dir

elif (current_dir.parent / "03_notebooks").exists():
    project_root_616 = current_dir.parent

else:
    raise FileNotFoundError(
        "Could not locate the project root containing "
        "'03_notebooks'."
    )

step4_notebook_path = (
    project_root_616
    / "03_notebooks"
    / "04_sensor_degradation_fd001.ipynb"
)

print("Project root :", project_root_616)
print("Step-4 file  :", step4_notebook_path)
print("File exists  :", step4_notebook_path.exists())

assert step4_notebook_path.exists(), (
    "Step-4 notebook was not found at the expected location."
)

# ------------------------------------------------------------
# 2. Read notebook JSON only
# ------------------------------------------------------------

with open(
    step4_notebook_path,
    "r",
    encoding="utf-8"
) as f:
    step4_notebook_json = json.load(f)

# ------------------------------------------------------------
# 3. Search for the exact function-definition cell
# ------------------------------------------------------------

drift_definition_cells = []

for cell_number, cell in enumerate(
    step4_notebook_json.get("cells", [])
):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(
        cell.get("source", [])
    )

    if "def add_sensor_drift" in source:
        drift_definition_cells.append(
            (cell_number, source)
        )

print(
    "\nNumber of cells containing "
    "'def add_sensor_drift':",
    len(drift_definition_cells)
)

assert len(drift_definition_cells) >= 1, (
    "Could not find the original add_sensor_drift "
    "definition in the Step-4 notebook."
)

# ------------------------------------------------------------
# 4. Display exact saved source
# ------------------------------------------------------------

for cell_number, source in drift_definition_cells:

    print(
        "\n"
        + "=" * 72
    )

    print(
        f"Step-4 notebook cell index: {cell_number}"
    )

    print(
        "=" * 72
    )

    print(source)

    print(
        "=" * 72
    )

print(
    "\nStep 6.16A completed."
)

print(
    "No function has been executed or modified."
)

In [ ]:
# ============================================================
# STEP 6.16B
# Compare the two saved Step-4 add_sensor_drift definitions
#
# READ-ONLY:
# - no function execution
# - no data modification
# - no file modification
# ============================================================

import ast
import hashlib
import difflib

print("=== Step 6.16B — Compare frozen drift definitions ===\n")


def extract_named_function(source, function_name):
    """
    Extract the exact source text of a named function
    from a Python code-cell source string.
    """

    tree = ast.parse(source)

    for node in tree.body:

        if (
            isinstance(
                node,
                (ast.FunctionDef, ast.AsyncFunctionDef)
            )
            and node.name == function_name
        ):

            lines = source.splitlines()

            start = node.lineno - 1
            end = node.end_lineno

            return "\n".join(
                lines[start:end]
            )

    return None


# ------------------------------------------------------------
# 1. Extract exact add_sensor_drift definitions
# ------------------------------------------------------------

drift_functions = []

for cell_number, source in drift_definition_cells:

    function_source = extract_named_function(
        source,
        "add_sensor_drift"
    )

    if function_source is not None:

        drift_functions.append(
            {
                "cell_index": cell_number,
                "source": function_source,
                "lines": len(
                    function_source.splitlines()
                ),
                "sha256": hashlib.sha256(
                    function_source.encode("utf-8")
                ).hexdigest()
            }
        )


print(
    "Number of extracted drift functions:",
    len(drift_functions)
)

assert len(drift_functions) == 2, (
    "Expected exactly two saved add_sensor_drift "
    "definitions."
)


# ------------------------------------------------------------
# 2. Show identification information
# ------------------------------------------------------------

for item in drift_functions:

    print("\n" + "-" * 68)

    print(
        "Notebook cell index :",
        item["cell_index"]
    )

    print(
        "Function lines      :",
        item["lines"]
    )

    print(
        "SHA-256             :",
        item["sha256"]
    )

    print("-" * 68)


# ------------------------------------------------------------
# 3. Check whether both definitions are identical
# ------------------------------------------------------------

same_definition = (
    drift_functions[0]["source"]
    ==
    drift_functions[1]["source"]
)

print(
    "\nDefinitions identical:",
    same_definition
)


# ------------------------------------------------------------
# 4. If different, show exact differences
# ------------------------------------------------------------

if not same_definition:

    source_1 = (
        drift_functions[0]["source"]
        .splitlines()
    )

    source_2 = (
        drift_functions[1]["source"]
        .splitlines()
    )

    diff = difflib.unified_diff(
        source_1,
        source_2,
        fromfile=(
            f"cell_{drift_functions[0]['cell_index']}"
        ),
        tofile=(
            f"cell_{drift_functions[1]['cell_index']}"
        ),
        lineterm=""
    )

    print(
        "\n=== Exact differences between "
        "the two definitions ===\n"
    )

    for line in diff:
        print(line)

else:

    print(
        "\nBoth saved add_sensor_drift definitions "
        "are exactly identical."
    )


print(
    "\nStep 6.16B comparison completed."
)

In [ ]:
# ============================================================
# STEP 6.16C
# Compact line-by-line comparison of the two saved
# add_sensor_drift definitions.
#
# READ-ONLY
# ============================================================

import difflib

print(
    "=== Step 6.16C — Exact changed portions "
    "of add_sensor_drift ===\n"
)

assert len(drift_functions) == 2

source_a = drift_functions[0]["source"].splitlines()
source_b = drift_functions[1]["source"].splitlines()

cell_a = drift_functions[0]["cell_index"]
cell_b = drift_functions[1]["cell_index"]

matcher = difflib.SequenceMatcher(
    None,
    source_a,
    source_b
)

change_count = 0

for tag, i1, i2, j1, j2 in matcher.get_opcodes():

    if tag == "equal":
        continue

    change_count += 1

    print("=" * 72)
    print(
        f"CHANGE BLOCK {change_count}: {tag.upper()}"
    )
    print("=" * 72)

    print(
        f"\nCell {cell_a}: "
        f"function lines {i1 + 1} to {i2}"
    )

    if i1 == i2:
        print("    <no lines>")
    else:
        for idx in range(i1, i2):
            print(
                f"{idx + 1:>3}: "
                f"{source_a[idx]}"
            )

    print(
        f"\nCell {cell_b}: "
        f"function lines {j1 + 1} to {j2}"
    )

    if j1 == j2:
        print("    <no lines>")
    else:
        for idx in range(j1, j2):
            print(
                f"{idx + 1:>3}: "
                f"{source_b[idx]}"
            )

    print()


print("=" * 72)
print(
    "Total changed blocks:",
    change_count
)
print(
    "Step 6.16C completed."
)

In [ ]:
# ============================================================
# STEP 6.16D
# Show ONLY change blocks 3, 4 and 5
# between the two saved add_sensor_drift definitions.
#
# READ-ONLY
# ============================================================

import difflib

print(
    "=== Step 6.16D — Remaining add_sensor_drift differences ===\n"
)

assert "drift_functions" in globals()
assert len(drift_functions) == 2

source_a = drift_functions[0]["source"].splitlines()
source_b = drift_functions[1]["source"].splitlines()

cell_a = drift_functions[0]["cell_index"]
cell_b = drift_functions[1]["cell_index"]

matcher = difflib.SequenceMatcher(
    None,
    source_a,
    source_b
)

all_changes = [
    opcode
    for opcode in matcher.get_opcodes()
    if opcode[0] != "equal"
]

print(
    "Total changed blocks:",
    len(all_changes)
)

assert len(all_changes) == 5

for block_number in [3, 4, 5]:

    tag, i1, i2, j1, j2 = (
        all_changes[block_number - 1]
    )

    print("\n" + "=" * 68)
    print(
        f"CHANGE BLOCK {block_number}: "
        f"{tag.upper()}"
    )
    print("=" * 68)

    print(
        f"\nCell {cell_a}:"
    )

    if i1 == i2:
        print("<no lines>")
    else:
        for idx in range(i1, i2):
            print(
                f"{idx + 1:>3}: "
                f"{source_a[idx]}"
            )

    print(
        f"\nCell {cell_b}:"
    )

    if j1 == j2:
        print("<no lines>")
    else:
        for idx in range(j1, j2):
            print(
                f"{idx + 1:>3}: "
                f"{source_b[idx]}"
            )


print(
    "\nStep 6.16D completed."
)

In [ ]:
# ============================================================
# STEP 6.16E
# Inspect the frozen Step-4 drift-validation artifact
#
# READ-ONLY
# Nothing is modified.
# ============================================================

from pathlib import Path
import pandas as pd


print(
    "=== Step 6.16E — Inspect frozen Step-4 drift artifact ===\n"
)


# ------------------------------------------------------------
# 1. Locate project root
# ------------------------------------------------------------

cwd = Path.cwd()

if (cwd / "02_dataset").exists():
    PROJECT_ROOT_616 = cwd

elif (cwd.parent / "02_dataset").exists():
    PROJECT_ROOT_616 = cwd.parent

else:
    raise FileNotFoundError(
        "Could not locate project root."
    )


# ------------------------------------------------------------
# 2. Locate frozen drift-validation CSV
# ------------------------------------------------------------

DEGRADATION_DIR_616 = (
    PROJECT_ROOT_616
    / "05_experiments"
    / "sensor_degradation"
)

DRIFT_VALIDATION_PATH_616 = (
    DEGRADATION_DIR_616
    / "fd001_drift_validation.csv"
)


print(
    "Project root:",
    PROJECT_ROOT_616
)

print(
    "Drift-validation file:",
    DRIFT_VALIDATION_PATH_616
)

print(
    "File exists:",
    DRIFT_VALIDATION_PATH_616.exists()
)


assert DRIFT_VALIDATION_PATH_616.exists(), (
    "Frozen Step-4 drift-validation file was not found."
)


# ------------------------------------------------------------
# 3. Load frozen validation table
# ------------------------------------------------------------

drift_validation_frozen_616 = pd.read_csv(
    DRIFT_VALIDATION_PATH_616
)


print(
    "\nShape:",
    drift_validation_frozen_616.shape
)

print(
    "Columns:"
)

for column in drift_validation_frozen_616.columns:
    print(
        " -",
        column
    )


# ------------------------------------------------------------
# 4. Basic integrity checks
# ------------------------------------------------------------

assert len(
    drift_validation_frozen_616
) == 90, (
    "Expected exactly 90 frozen drift scenarios."
)


print(
    "\nUnique sensors:",
    drift_validation_frozen_616[
        "sensor"
    ].nunique()
    if "sensor"
    in drift_validation_frozen_616.columns
    else "column unavailable"
)

print(
    "Unique severities:",
    drift_validation_frozen_616[
        "severity"
    ].nunique()
    if "severity"
    in drift_validation_frozen_616.columns
    else "column unavailable"
)


if (
    "direction"
    in drift_validation_frozen_616.columns
):

    print(
        "Directions:",
        sorted(
            drift_validation_frozen_616[
                "direction"
            ].unique().tolist()
        )
    )


# ------------------------------------------------------------
# 5. Show representative rows
# ------------------------------------------------------------

print(
    "\nFirst 10 frozen drift-validation rows:"
)

display(
    drift_validation_frozen_616.head(10)
)


print(
    "\nLast 10 frozen drift-validation rows:"
)

display(
    drift_validation_frozen_616.tail(10)
)


print(
    "\nStep 6.16E completed."
)

In [ ]:
# ============================================================
# STEP 6.16F
# Verify the later/corrected Step-4 add_sensor_drift
# against ALL 90 frozen Step-4 drift-validation scenarios
#
# IMPORTANT:
# - Read-only verification
# - Does NOT overwrite global add_sensor_drift
# - Does NOT modify frozen files
# - Does NOT save anything
# ============================================================

from pathlib import Path
import ast
import json
import numpy as np
import pandas as pd


print(
    "=== Step 6.16F — Verify corrected drift implementation ===\n"
)


# ------------------------------------------------------------
# 1. Locate project and Step-4 notebook
# ------------------------------------------------------------

cwd_616f = Path.cwd()

if (cwd_616f / "02_dataset").exists():
    PROJECT_ROOT_616F = cwd_616f

elif (cwd_616f.parent / "02_dataset").exists():
    PROJECT_ROOT_616F = cwd_616f.parent

else:
    raise FileNotFoundError(
        "Could not locate project root."
    )


STEP4_NOTEBOOK_616F = (
    PROJECT_ROOT_616F
    / "03_notebooks"
    / "04_sensor_degradation_fd001.ipynb"
)


assert STEP4_NOTEBOOK_616F.exists(), (
    "Step-4 notebook was not found."
)


# ------------------------------------------------------------
# 2. Extract ONLY add_sensor_drift definitions
#    from Step-4 notebook
# ------------------------------------------------------------

with open(
    STEP4_NOTEBOOK_616F,
    "r",
    encoding="utf-8"
) as file:

    notebook_616f = json.load(file)


drift_definitions_616f = []


for cell_index_616f, cell_616f in enumerate(
    notebook_616f["cells"]
):

    if cell_616f.get("cell_type") != "code":
        continue

    source_616f = "".join(
        cell_616f.get(
            "source",
            []
        )
    )

    if "def add_sensor_drift" not in source_616f:
        continue

    tree_616f = ast.parse(
        source_616f
    )

    for node_616f in tree_616f.body:

        if (
            isinstance(
                node_616f,
                ast.FunctionDef
            )
            and
            node_616f.name
            == "add_sensor_drift"
        ):

            function_source_616f = (
                ast.get_source_segment(
                    source_616f,
                    node_616f
                )
            )

            drift_definitions_616f.append(
                (
                    cell_index_616f,
                    function_source_616f
                )
            )


definition_indices_616f = [
    item[0]
    for item in drift_definitions_616f
]


print(
    "Drift-definition cell indices:",
    definition_indices_616f
)


assert definition_indices_616f == [29, 34], (
    "Unexpected Step-4 drift-definition structure. "
    "Stop before continuing."
)


# ------------------------------------------------------------
# 3. Select later corrected definition — cell 34
# ------------------------------------------------------------

selected_cell_index_616f = 34


selected_source_616f = [
    source
    for index, source
    in drift_definitions_616f
    if index == selected_cell_index_616f
][0]


print(
    "Candidate definition:",
    f"Step-4 notebook cell {selected_cell_index_616f}"
)


# ------------------------------------------------------------
# 4. Execute candidate ONLY inside a temporary namespace
#
#    This does NOT replace any function in the notebook's
#    normal global namespace.
# ------------------------------------------------------------

temporary_namespace_616f = {
    "__builtins__": __builtins__,
    "np": np,
    "pd": pd,
    "CORRUPTION_LEVELS": CORRUPTION_LEVELS,
}


exec(
    selected_source_616f,
    temporary_namespace_616f
)


candidate_add_sensor_drift_616f = (
    temporary_namespace_616f[
        "add_sensor_drift"
    ]
)


print(
    "Candidate function loaded in temporary namespace only."
)


# ------------------------------------------------------------
# 5. Reconstruct raw validation inputs
#    from frozen scaled validation data
# ------------------------------------------------------------

val_raw_616f = (
    val_scaled.copy(deep=True)
)


raw_primary_values_616f = (
    scaler.inverse_transform(
        val_scaled[
            PRIMARY_INPUT_COLUMNS
        ].to_numpy()
    )
)


val_raw_616f.loc[
    :,
    PRIMARY_INPUT_COLUMNS
] = raw_primary_values_616f


# ------------------------------------------------------------
# 6. Verify scaling round-trip
# ------------------------------------------------------------

roundtrip_scaled_616f = (
    scaler.transform(
        val_raw_616f[
            PRIMARY_INPUT_COLUMNS
        ].to_numpy()
    )
)


original_scaled_616f = (
    val_scaled[
        PRIMARY_INPUT_COLUMNS
    ].to_numpy()
)


max_roundtrip_difference_616f = float(
    np.max(
        np.abs(
            roundtrip_scaled_616f
            -
            original_scaled_616f
        )
    )
)


print(
    "\nMaximum scaling round-trip difference:",
    max_roundtrip_difference_616f
)


assert max_roundtrip_difference_616f < 1e-10, (
    "Scaling round-trip verification failed."
)


# ------------------------------------------------------------
# 7. Load frozen Step-4 validation artifact
# ------------------------------------------------------------

DRIFT_VALIDATION_PATH_616F = (
    PROJECT_ROOT_616F
    / "05_experiments"
    / "sensor_degradation"
    / "fd001_drift_validation.csv"
)


frozen_drift_616f = pd.read_csv(
    DRIFT_VALIDATION_PATH_616F
)


assert len(frozen_drift_616f) == 90


print(
    "Frozen drift scenarios:",
    len(frozen_drift_616f)
)


# ------------------------------------------------------------
# 8. Verify ALL 90 scenarios
# ------------------------------------------------------------

verification_records_616f = []


for scenario_number_616f, (
    _,
    frozen_row_616f
) in enumerate(
    frozen_drift_616f.iterrows(),
    start=1
):

    sensor_616f = str(
        frozen_row_616f["sensor"]
    )

    severity_616f = str(
        frozen_row_616f["severity"]
    )

    direction_616f = int(
        frozen_row_616f["direction"]
    )


    # --------------------------------------------------------
    # Run candidate cell-34 implementation
    # --------------------------------------------------------

    result_616f = (
        candidate_add_sensor_drift_616f(
            df=val_raw_616f,
            sensor=sensor_616f,
            severity=severity_616f,
            direction=direction_616f,
            sensor_stats=sensor_reference_stats
        )
    )


    assert isinstance(
        result_616f,
        tuple
    ), (
        "Drift function did not return a tuple."
    )


    assert len(result_616f) == 2, (
        "Unexpected number of outputs from drift function."
    )


    corrupted_df_616f, drift_values_616f = (
        result_616f
    )


    drift_values_616f = np.asarray(
        drift_values_616f,
        dtype=float
    ).reshape(-1)


    # --------------------------------------------------------
    # Structural integrity
    # --------------------------------------------------------

    assert corrupted_df_616f.shape == val_raw_616f.shape

    assert len(drift_values_616f) == len(
        val_raw_616f
    )

    assert corrupted_df_616f.index.equals(
        val_raw_616f.index
    )


    # --------------------------------------------------------
    # Verify only selected sensor changed
    # --------------------------------------------------------

    unchanged_columns_616f = [
        column
        for column in val_raw_616f.columns
        if column != sensor_616f
    ]


    pd.testing.assert_frame_equal(
        corrupted_df_616f[
            unchanged_columns_616f
        ],
        val_raw_616f[
            unchanged_columns_616f
        ],
        check_dtype=False
    )


    # --------------------------------------------------------
    # Verify returned drift equals actual sensor change
    # --------------------------------------------------------

    observed_sensor_change_616f = (
        corrupted_df_616f[
            sensor_616f
        ].to_numpy(
            dtype=float
        )
        -
        val_raw_616f[
            sensor_616f
        ].to_numpy(
            dtype=float
        )
    )


    max_vector_difference_616f = float(
        np.max(
            np.abs(
                observed_sensor_change_616f
                -
                drift_values_616f
            )
        )
    )


    assert max_vector_difference_616f < 1e-10, (
        "Returned drift vector does not match "
        "the actual sensor modification."
    )


    # --------------------------------------------------------
    # Build engine/cycle drift table
    # --------------------------------------------------------

    drift_check_df_616f = pd.DataFrame(
        {
            "unit_id":
                val_raw_616f[
                    "unit_id"
                ].to_numpy(),

            "cycle":
                val_raw_616f[
                    "cycle"
                ].to_numpy(),

            "drift":
                drift_values_616f
        }
    )


    drift_check_df_616f = (
        drift_check_df_616f
        .sort_values(
            [
                "unit_id",
                "cycle"
            ]
        )
    )


    grouped_drift_616f = (
        drift_check_df_616f
        .groupby(
            "unit_id",
            sort=False
        )["drift"]
    )


    starting_drift_616f = (
        grouped_drift_616f
        .first()
        .to_numpy(
            dtype=float
        )
    )


    final_drift_616f = (
        grouped_drift_616f
        .last()
        .to_numpy(
            dtype=float
        )
    )


    # --------------------------------------------------------
    # Candidate summary
    # --------------------------------------------------------

    observed_max_start_616f = float(
        np.max(
            np.abs(
                starting_drift_616f
            )
        )
    )


    observed_mean_final_616f = float(
        np.mean(
            final_drift_616f
        )
    )


    observed_min_final_616f = float(
        np.min(
            final_drift_616f
        )
    )


    observed_max_final_616f = float(
        np.max(
            final_drift_616f
        )
    )


    # --------------------------------------------------------
    # Differences against frozen Step-4 artifact
    # --------------------------------------------------------

    verification_records_616f.append(
        {
            "scenario":
                scenario_number_616f,

            "sensor":
                sensor_616f,

            "severity":
                severity_616f,

            "direction":
                direction_616f,

            "start_diff":
                abs(
                    observed_max_start_616f
                    -
                    float(
                        frozen_row_616f[
                            "max_abs_starting_drift"
                        ]
                    )
                ),

            "mean_final_diff":
                abs(
                    observed_mean_final_616f
                    -
                    float(
                        frozen_row_616f[
                            "mean_final_drift"
                        ]
                    )
                ),

            "min_final_diff":
                abs(
                    observed_min_final_616f
                    -
                    float(
                        frozen_row_616f[
                            "min_final_drift"
                        ]
                    )
                ),

            "max_final_diff":
                abs(
                    observed_max_final_616f
                    -
                    float(
                        frozen_row_616f[
                            "max_final_drift"
                        ]
                    )
                ),

            "vector_integrity_diff":
                max_vector_difference_616f
        }
    )


    if scenario_number_616f % 15 == 0:

        print(
            f"[{scenario_number_616f}/90] verified"
        )


# ------------------------------------------------------------
# 9. Summarize verification
# ------------------------------------------------------------

verification_df_616f = pd.DataFrame(
    verification_records_616f
)


difference_columns_616f = [
    "start_diff",
    "mean_final_diff",
    "min_final_diff",
    "max_final_diff",
    "vector_integrity_diff"
]


print(
    "\n=== Maximum absolute differences ==="
)


for column_616f in difference_columns_616f:

    print(
        f"{column_616f:<25}:",
        verification_df_616f[
            column_616f
        ].max()
    )


# ------------------------------------------------------------
# 10. Final strict verification
# ------------------------------------------------------------

TOLERANCE_616F = 1e-9


assert (
    verification_df_616f[
        difference_columns_616f
    ]
    .to_numpy()
    <= TOLERANCE_616F
).all(), (
    "Cell-34 drift implementation does NOT exactly reproduce "
    "the frozen Step-4 validation artifact."
)


print(
    "\nAll 90 frozen Step-4 drift scenarios "
    "were reproduced successfully."
)

print(
    "Cell 34 is verified as the canonical "
    "index-safe add_sensor_drift implementation."
)

print(
    "\nStep 6.16F completed successfully."
)

In [ ]:
print("=== Exact candidate drift function around failing line 26 ===\n")

source_lines = selected_source_616f.splitlines()

for line_number in range(20, 31):
    print(
        f"{line_number:02d}: "
        f"{source_lines[line_number - 1]}"
    )

In [ ]:
# ============================================================
# Step 6.16F — Verify candidate frozen drift implementation
#              with correct NUMERIC severity
# ============================================================

print(
    "=== Step 6.16F — Verify candidate "
    "add_sensor_drift implementation ==="
)

# ------------------------------------------------------------
# 1. Select one frozen drift scenario
# ------------------------------------------------------------

frozen_row_616f = (
    drift_plan_df
    .iloc[0]
)

sensor_616f = str(
    frozen_row_616f["sensor"]
)

severity_name_616f = str(
    frozen_row_616f["severity"]
)

direction_616f = int(
    frozen_row_616f["direction"]
)

# IMPORTANT:
# frozen add_sensor_drift expects NUMERIC severity,
# not the textual label.
severity_616f = float(
    CORRUPTION_LEVELS["drift"][
        severity_name_616f
    ]
)

print("\nFrozen test scenario:")
print("Sensor          :", sensor_616f)
print("Severity label  :", severity_name_616f)
print("Severity value  :", severity_616f)
print("Direction       :", direction_616f)

assert severity_name_616f in {
    "low",
    "medium",
    "high",
}

assert np.isclose(
    severity_616f,
    CORRUPTION_LEVELS["drift"][
        severity_name_616f
    ],
)

# ------------------------------------------------------------
# 2. Run candidate cell-34 implementation
# ------------------------------------------------------------

result_616f = (
    candidate_add_sensor_drift_616f(
        df=val_raw_616f,
        sensor=sensor_616f,
        severity=severity_616f,
        direction=direction_616f,
        sensor_stats=sensor_reference_stats,
    )
)

# ------------------------------------------------------------
# 3. Verify return structure
# ------------------------------------------------------------

assert isinstance(
    result_616f,
    tuple,
), (
    "Drift function did not return a tuple."
)

assert len(result_616f) == 2, (
    "Expected add_sensor_drift to return "
    "two objects."
)

(
    val_drift_raw_616f,
    drift_values_616f,
) = result_616f

assert isinstance(
    val_drift_raw_616f,
    pd.DataFrame,
)

drift_values_616f = np.asarray(
    drift_values_616f,
    dtype=float,
)

print("\nCandidate function executed successfully.")
print(
    "Corrupted dataframe shape :",
    val_drift_raw_616f.shape,
)
print(
    "Drift-vector shape        :",
    drift_values_616f.shape,
)

# ------------------------------------------------------------
# 4. Basic integrity checks
# ------------------------------------------------------------

assert (
    val_drift_raw_616f.shape
    == val_raw_616f.shape
)

assert len(drift_values_616f) == len(
    val_raw_616f
)

assert np.isfinite(
    drift_values_616f
).all()

assert np.isfinite(
    val_drift_raw_616f[
        PRIMARY_INPUT_COLUMNS
    ].to_numpy(dtype=float)
).all()

# Only selected sensor may change
other_columns_616f = [
    col
    for col in PRIMARY_INPUT_COLUMNS
    if col != sensor_616f
]

np.testing.assert_allclose(
    val_drift_raw_616f[
        other_columns_616f
    ].to_numpy(dtype=float),
    val_raw_616f[
        other_columns_616f
    ].to_numpy(dtype=float),
    rtol=0.0,
    atol=0.0,
)

print(
    "Non-target input columns remained unchanged."
)

# ------------------------------------------------------------
# 5. Check drift magnitude
# ------------------------------------------------------------

sensor_std_616f = float(
    sensor_reference_stats.loc[
        sensor_616f,
        "std",
    ]
)

expected_final_drift_616f = (
    direction_616f
    * severity_616f
    * sensor_std_616f
)

print("\nDrift magnitude verification:")
print(
    "Reference sensor std :",
    sensor_std_616f,
)
print(
    "Expected final drift :",
    expected_final_drift_616f,
)
print(
    "Observed min drift   :",
    float(np.min(drift_values_616f)),
)
print(
    "Observed max drift   :",
    float(np.max(drift_values_616f)),
)

# Gradual drift should begin at/near zero
assert np.isclose(
    drift_values_616f[0],
    0.0,
    atol=1e-12,
)

# Final magnitude must match frozen formula
assert np.isclose(
    drift_values_616f[-1],
    expected_final_drift_616f,
    rtol=1e-10,
    atol=1e-10,
)

print(
    "\nCandidate frozen drift implementation "
    "passed the single-scenario verification."
)

print(
    "\nStep 6.16F completed successfully."
)

In [ ]:
# ============================================================
# Step 6.16F-1 — Locate existing drift-scenario dataframe
# ============================================================

print(
    "=== Looking for existing drift-scenario "
    "DataFrames in memory ===\n"
)

candidate_dfs_616f = []

for name, obj in list(globals().items()):

    if isinstance(obj, pd.DataFrame):

        cols = set(obj.columns)

        required = {
            "sensor",
            "severity",
            "direction",
        }

        if required.issubset(cols):

            corruption_info = ""

            if "corruption" in cols:
                unique_corruptions = (
                    obj["corruption"]
                    .astype(str)
                    .unique()
                    .tolist()
                )

                corruption_info = (
                    f" | corruptions={unique_corruptions}"
                )

            print(
                f"{name:<35}"
                f" shape={obj.shape}"
                f"{corruption_info}"
            )

            candidate_dfs_616f.append(name)

print(
    "\nNumber of candidate DataFrames:",
    len(candidate_dfs_616f),
)

In [ ]:
# ============================================================
# Step 6.16F-2 — Verify the two frozen drift DataFrames
# ============================================================

print(
    "=== Step 6.16F-2 — Compare frozen drift DataFrames ===\n"
)

print(
    "drift_validation_frozen_616 shape:",
    drift_validation_frozen_616.shape,
)

print(
    "frozen_drift_616f shape:",
    frozen_drift_616f.shape,
)

print(
    "\nColumns of drift_validation_frozen_616:"
)
print(
    drift_validation_frozen_616.columns.tolist()
)

print(
    "\nColumns of frozen_drift_616f:"
)
print(
    frozen_drift_616f.columns.tolist()
)

same_columns_616f = (
    drift_validation_frozen_616.columns.tolist()
    == frozen_drift_616f.columns.tolist()
)

print(
    "\nSame columns:",
    same_columns_616f,
)

same_values_616f = (
    drift_validation_frozen_616
    .reset_index(drop=True)
    .equals(
        frozen_drift_616f
        .reset_index(drop=True)
    )
)

print(
    "Same values:",
    same_values_616f,
)

print(
    "\nFirst 3 rows:"
)

display(
    drift_validation_frozen_616.head(3)
)

print(
    "\nStep 6.16F-2 completed."
)

In [ ]:
# ============================================================
# Step 6.16F-3
# One-scenario verification of frozen gradual-drift function
# ============================================================

print(
    "=== Step 6.16F-3 — Frozen drift smoke test ===\n"
)

# ------------------------------------------------------------
# 1. Confirm the temporary candidate function still exists
# ------------------------------------------------------------

assert (
    "candidate_add_sensor_drift_616f" in globals()
), (
    "candidate_add_sensor_drift_616f is not currently "
    "available in memory. Stop here."
)

assert callable(
    candidate_add_sensor_drift_616f
), "candidate_add_sensor_drift_616f is not callable."

print(
    "Candidate drift function available: True"
)

# ------------------------------------------------------------
# 2. Select first frozen Step-4 drift scenario
# ------------------------------------------------------------

test_row_616f = (
    drift_validation_frozen_616
    .iloc[0]
)

test_sensor_616f = str(
    test_row_616f["sensor"]
)

test_severity_name_616f = str(
    test_row_616f["severity"]
)

# IMPORTANT:
# The frozen function expects numeric severity,
# therefore we pass alpha — NOT the string label.
test_alpha_616f = float(
    test_row_616f["alpha"]
)

test_direction_616f = int(
    test_row_616f["direction"]
)

expected_final_drift_616f = float(
    test_row_616f["expected_final_drift"]
)

print(
    "\nFrozen scenario:"
)

print(
    "Sensor              :",
    test_sensor_616f
)

print(
    "Severity label      :",
    test_severity_name_616f
)

print(
    "Numeric alpha       :",
    test_alpha_616f
)

print(
    "Direction           :",
    test_direction_616f
)

print(
    "Expected final drift:",
    expected_final_drift_616f
)

# ------------------------------------------------------------
# 3. Reconstruct validation inputs in ORIGINAL sensor units
# ------------------------------------------------------------

val_raw_smoke_616f = (
    val_scaled.copy()
)

val_raw_smoke_616f[
    PRIMARY_INPUT_COLUMNS
] = scaler.inverse_transform(
    val_scaled[
        PRIMARY_INPUT_COLUMNS
    ]
)

# Round-trip integrity check
roundtrip_scaled_616f = scaler.transform(
    val_raw_smoke_616f[
        PRIMARY_INPUT_COLUMNS
    ]
)

roundtrip_diff_616f = float(
    np.max(
        np.abs(
            roundtrip_scaled_616f
            - val_scaled[
                PRIMARY_INPUT_COLUMNS
            ].to_numpy()
        )
    )
)

print(
    "\nMaximum scaling round-trip difference:",
    roundtrip_diff_616f
)

assert (
    roundtrip_diff_616f < 1e-10
), (
    "Scaling round-trip check failed."
)

# ------------------------------------------------------------
# 4. Run frozen drift function correctly
# ------------------------------------------------------------

result_616f = (
    candidate_add_sensor_drift_616f(
        df=val_raw_smoke_616f,
        sensor=test_sensor_616f,
        severity=test_alpha_616f,
        direction=test_direction_616f,
        sensor_stats=sensor_reference_stats
    )
)

assert isinstance(
    result_616f,
    tuple
), (
    "Frozen drift function did not return a tuple."
)

assert len(
    result_616f
) == 2, (
    "Frozen drift function did not return exactly "
    "two outputs."
)

(
    corrupted_smoke_616f,
    drift_vector_smoke_616f
) = result_616f

drift_vector_smoke_616f = np.asarray(
    drift_vector_smoke_616f,
    dtype=float
)

print(
    "\nFunction return verified."
)

print(
    "Corrupted shape:",
    corrupted_smoke_616f.shape
)

print(
    "Drift-vector shape:",
    drift_vector_smoke_616f.shape
)

assert (
    corrupted_smoke_616f.shape
    == val_raw_smoke_616f.shape
)

assert (
    drift_vector_smoke_616f.shape
    == (len(val_raw_smoke_616f),)
)

# ------------------------------------------------------------
# 5. Verify only selected sensor was modified
# ------------------------------------------------------------

other_inputs_616f = [
    column
    for column in PRIMARY_INPUT_COLUMNS
    if column != test_sensor_616f
]

max_other_change_616f = float(
    np.max(
        np.abs(
            corrupted_smoke_616f[
                other_inputs_616f
            ].to_numpy()
            -
            val_raw_smoke_616f[
                other_inputs_616f
            ].to_numpy()
        )
    )
)

print(
    "\nMaximum change in non-target inputs:",
    max_other_change_616f
)

assert np.isclose(
    max_other_change_616f,
    0.0,
    atol=1e-12
), (
    "A non-target input was modified."
)

# ------------------------------------------------------------
# 6. Calculate actual injected drift directly from data
# ------------------------------------------------------------

actual_delta_616f = (
    corrupted_smoke_616f[
        test_sensor_616f
    ].to_numpy(dtype=float)
    -
    val_raw_smoke_616f[
        test_sensor_616f
    ].to_numpy(dtype=float)
)

assert np.allclose(
    actual_delta_616f,
    drift_vector_smoke_616f,
    rtol=1e-10,
    atol=1e-10
), (
    "Returned drift vector does not match actual "
    "sensor modification."
)

# ------------------------------------------------------------
# 7. Verify gradual drift independently for every engine
# ------------------------------------------------------------

drift_check_616f = pd.DataFrame(
    {
        "unit_id":
            val_raw_smoke_616f[
                "unit_id"
            ].to_numpy(),

        "cycle":
            val_raw_smoke_616f[
                "cycle"
            ].to_numpy(),

        "drift":
            actual_delta_616f,
    }
)

drift_check_616f = (
    drift_check_616f
    .sort_values(
        ["unit_id", "cycle"]
    )
)

starting_drift_616f = (
    drift_check_616f
    .groupby(
        "unit_id",
        sort=False
    )["drift"]
    .first()
    .to_numpy()
)

final_drift_616f = (
    drift_check_616f
    .groupby(
        "unit_id",
        sort=False
    )["drift"]
    .last()
    .to_numpy()
)

max_abs_start_616f = float(
    np.max(
        np.abs(
            starting_drift_616f
        )
    )
)

mean_final_616f = float(
    np.mean(
        final_drift_616f
    )
)

min_final_616f = float(
    np.min(
        final_drift_616f
    )
)

max_final_616f = float(
    np.max(
        final_drift_616f
    )
)

print(
    "\nObserved gradual-drift behaviour:"
)

print(
    "Maximum absolute starting drift:",
    max_abs_start_616f
)

print(
    "Mean final drift:",
    mean_final_616f
)

print(
    "Minimum final drift:",
    min_final_616f
)

print(
    "Maximum final drift:",
    max_final_616f
)

# Frozen Step-4 artifact comparisons
print(
    "\nFrozen Step-4 reference:"
)

print(
    "Expected final drift:",
    expected_final_drift_616f
)

print(
    "Frozen max abs starting drift:",
    float(
        test_row_616f[
            "max_abs_starting_drift"
        ]
    )
)

print(
    "Frozen mean final drift:",
    float(
        test_row_616f[
            "mean_final_drift"
        ]
    )
)

print(
    "Frozen min final drift:",
    float(
        test_row_616f[
            "min_final_drift"
        ]
    )
)

print(
    "Frozen max final drift:",
    float(
        test_row_616f[
            "max_final_drift"
        ]
    )
)

# ------------------------------------------------------------
# 8. Exact numerical consistency checks
# ------------------------------------------------------------

assert np.isclose(
    max_abs_start_616f,
    float(
        test_row_616f[
            "max_abs_starting_drift"
        ]
    ),
    rtol=1e-8,
    atol=1e-10
)

assert np.isclose(
    mean_final_616f,
    float(
        test_row_616f[
            "mean_final_drift"
        ]
    ),
    rtol=1e-8,
    atol=1e-10
)

assert np.isclose(
    min_final_616f,
    float(
        test_row_616f[
            "min_final_drift"
        ]
    ),
    rtol=1e-8,
    atol=1e-10
)

assert np.isclose(
    max_final_616f,
    float(
        test_row_616f[
            "max_final_drift"
        ]
    ),
    rtol=1e-8,
    atol=1e-10
)

assert np.isclose(
    mean_final_616f,
    expected_final_drift_616f,
    rtol=1e-8,
    atol=1e-10
)

print(
    "\nStep 6.16F-3 completed successfully."
)

print(
    "The candidate gradual-drift implementation "
    "reproduces the frozen Step-4 scenario."
)

In [ ]:
# ============================================================
# Step 6.16F-3a
# Inspect actual frozen drift return structure
# READ-ONLY DIAGNOSTIC
# ============================================================

print(
    "=== Step 6.16F-3a — Inspect drift return structure ===\n"
)

print(
    "Return type:",
    type(result_616f)
)

print(
    "Number of returned objects:",
    len(result_616f)
)

for i, obj in enumerate(
    result_616f,
    start=1
):
    print(
        f"\nOutput {i}:"
    )

    print(
        "  Type :",
        type(obj)
    )

    if hasattr(
        obj,
        "shape"
    ):
        print(
            "  Shape:",
            obj.shape
        )

    elif hasattr(
        obj,
        "__len__"
    ):
        try:
            print(
                "  Length:",
                len(obj)
            )
        except TypeError:
            pass

print(
    "\nDiagnostic inspection completed."
)

In [ ]:
# ============================================================
# Step 6.16G — Verify one frozen drift scenario exactly
# ============================================================

import numpy as np
import pandas as pd

print("=== Step 6.16G — Verify one frozen drift scenario ===")

# ------------------------------------------------------------
# 1. Confirm required objects exist
# ------------------------------------------------------------

required_objects_616g = [
    "candidate_add_sensor_drift_616f",
    "frozen_drift_616f",
    "val_scaled",
    "scaler",
    "PRIMARY_INPUT_COLUMNS",
    "sensor_reference_stats",
]

for name in required_objects_616g:
    assert name in globals(), (
        f"Required object '{name}' is not available in memory. "
        "Do not continue until it is restored."
    )

print("Required objects are available.")

# ------------------------------------------------------------
# 2. Select the first frozen Step-4 drift scenario
# ------------------------------------------------------------

scenario_616g = frozen_drift_616f.iloc[0]

sensor_616g = str(
    scenario_616g["sensor"]
)

severity_label_616g = str(
    scenario_616g["severity"]
)

alpha_616g = float(
    scenario_616g["alpha"]
)

direction_616g = int(
    scenario_616g["direction"]
)

expected_final_drift_616g = float(
    scenario_616g["expected_final_drift"]
)

print("\nFrozen scenario:")
print("Sensor               :", sensor_616g)
print("Severity label       :", severity_label_616g)
print("Numeric alpha        :", alpha_616g)
print("Direction            :", direction_616g)
print("Expected final drift :", expected_final_drift_616g)

# ------------------------------------------------------------
# 3. Reconstruct raw validation inputs safely
# ------------------------------------------------------------

scaled_inputs_616g = (
    val_scaled[
        PRIMARY_INPUT_COLUMNS
    ]
    .to_numpy(dtype=float)
)

raw_inputs_616g = scaler.inverse_transform(
    scaled_inputs_616g
)

val_raw_616g = val_scaled.copy()

val_raw_616g[
    PRIMARY_INPUT_COLUMNS
] = raw_inputs_616g

roundtrip_616g = scaler.transform(
    val_raw_616g[
        PRIMARY_INPUT_COLUMNS
    ]
)

max_roundtrip_difference_616g = float(
    np.max(
        np.abs(
            roundtrip_616g
            -
            scaled_inputs_616g
        )
    )
)

print(
    "\nMaximum scaling round-trip difference:",
    max_roundtrip_difference_616g
)

assert max_roundtrip_difference_616g < 1e-10

# ------------------------------------------------------------
# 4. Run candidate frozen drift implementation
#
# IMPORTANT:
# severity receives NUMERIC alpha, not the text label.
# ------------------------------------------------------------

result_616g = candidate_add_sensor_drift_616f(
    df=val_raw_616g,
    sensor=sensor_616g,
    severity=alpha_616g,
    direction=direction_616g,
    sensor_stats=sensor_reference_stats,
)

assert isinstance(result_616g, tuple), (
    "Drift function did not return a tuple."
)

assert len(result_616g) == 3, (
    f"Expected 3 returned objects, received {len(result_616g)}."
)

(
    corrupted_raw_616g,
    drift_vector_616g,
    returned_scalar_616g,
) = result_616g

drift_vector_616g = np.asarray(
    drift_vector_616g,
    dtype=float
)

returned_scalar_616g = float(
    returned_scalar_616g
)

print("\nReturn structure:")
print(
    "Output 1:",
    type(corrupted_raw_616g).__name__,
    corrupted_raw_616g.shape,
)
print(
    "Output 2:",
    type(drift_vector_616g).__name__,
    drift_vector_616g.shape,
)
print(
    "Output 3:",
    type(returned_scalar_616g).__name__,
    returned_scalar_616g,
)

# ------------------------------------------------------------
# 5. Basic integrity checks
# ------------------------------------------------------------

assert corrupted_raw_616g.shape == val_raw_616g.shape

assert drift_vector_616g.shape == (
    len(val_raw_616g),
)

assert np.isfinite(
    drift_vector_616g
).all()

assert np.isfinite(
    returned_scalar_616g
)

# Only the selected sensor is allowed to change.
unchanged_columns_616g = [
    col
    for col in PRIMARY_INPUT_COLUMNS
    if col != sensor_616g
]

pd.testing.assert_frame_equal(
    corrupted_raw_616g[
        unchanged_columns_616g
    ].reset_index(drop=True),
    val_raw_616g[
        unchanged_columns_616g
    ].reset_index(drop=True),
    check_dtype=False,
)

print(
    "\nNon-target input columns remain unchanged."
)

# ------------------------------------------------------------
# 6. Compare actual corruption with returned drift vector
# ------------------------------------------------------------

observed_change_616g = (
    corrupted_raw_616g[
        sensor_616g
    ].to_numpy(dtype=float)
    -
    val_raw_616g[
        sensor_616g
    ].to_numpy(dtype=float)
)

max_vector_difference_616g = float(
    np.max(
        np.abs(
            observed_change_616g
            -
            drift_vector_616g
        )
    )
)

print(
    "Maximum difference between actual sensor change "
    "and returned drift vector:",
    max_vector_difference_616g
)

assert max_vector_difference_616g < 1e-10

# ------------------------------------------------------------
# 7. Inspect engine-wise drift behaviour
# ------------------------------------------------------------

check_df_616g = pd.DataFrame(
    {
        "unit_id": val_raw_616g[
            "unit_id"
        ].to_numpy(),
        "cycle": val_raw_616g[
            "cycle"
        ].to_numpy(),
        "drift": drift_vector_616g,
    }
)

engine_drift_summary_616g = (
    check_df_616g
    .groupby(
        "unit_id",
        sort=True
    )
    .agg(
        first_drift=("drift", "first"),
        last_drift=("drift", "last"),
        min_drift=("drift", "min"),
        max_drift=("drift", "max"),
    )
)

print("\nFirst 10 engine-wise drift summaries:")
display(
    engine_drift_summary_616g.head(10)
)

# ------------------------------------------------------------
# 8. Compare final drift with frozen Step-4 artifact
# ------------------------------------------------------------

engine_final_drifts_616g = (
    check_df_616g
    .groupby(
        "unit_id",
        sort=True
    )["drift"]
    .last()
    .to_numpy(dtype=float)
)

mean_final_drift_616g = float(
    np.mean(
        engine_final_drifts_616g
    )
)

min_final_drift_616g = float(
    np.min(
        engine_final_drifts_616g
    )
)

max_final_drift_616g = float(
    np.max(
        engine_final_drifts_616g
    )
)

max_abs_starting_drift_616g = float(
    check_df_616g
    .groupby(
        "unit_id",
        sort=True
    )["drift"]
    .first()
    .abs()
    .max()
)

print("\nFrozen-artifact comparison:")
print(
    "Expected final drift      :",
    expected_final_drift_616g
)
print(
    "Returned scalar           :",
    returned_scalar_616g
)
print(
    "Mean engine final drift   :",
    mean_final_drift_616g
)
print(
    "Minimum engine final drift:",
    min_final_drift_616g
)
print(
    "Maximum engine final drift:",
    max_final_drift_616g
)
print(
    "Max |starting drift|      :",
    max_abs_starting_drift_616g
)

# ------------------------------------------------------------
# 9. Exact frozen-behaviour checks
# ------------------------------------------------------------

assert np.isclose(
    returned_scalar_616g,
    expected_final_drift_616g,
    rtol=1e-10,
    atol=1e-10,
), (
    "Returned scalar does not match frozen expected_final_drift."
)

assert np.isclose(
    mean_final_drift_616g,
    expected_final_drift_616g,
    rtol=1e-10,
    atol=1e-10,
), (
    "Mean engine final drift does not match frozen artifact."
)

assert np.isclose(
    min_final_drift_616g,
    expected_final_drift_616g,
    rtol=1e-10,
    atol=1e-10,
), (
    "Minimum engine final drift does not match frozen artifact."
)

assert np.isclose(
    max_final_drift_616g,
    expected_final_drift_616g,
    rtol=1e-10,
    atol=1e-10,
), (
    "Maximum engine final drift does not match frozen artifact."
)

assert max_abs_starting_drift_616g < 1e-10, (
    "Drift does not begin at zero for every validation engine."
)

print(
    "\nStep 6.16G completed successfully."
)
print(
    "Candidate drift implementation reproduces "
    "the selected frozen Step-4 scenario."
)

In [ ]:
# ============================================================
# Step 6.16H — Verify ALL 90 frozen Step-4 drift scenarios
# ============================================================

import numpy as np
import pandas as pd

print(
    "=== Step 6.16H — Verify all frozen drift scenarios ==="
)

# ------------------------------------------------------------
# 1. Required-object check
# ------------------------------------------------------------

required_objects_616h = [
    "candidate_add_sensor_drift_616f",
    "frozen_drift_616f",
    "val_scaled",
    "scaler",
    "PRIMARY_INPUT_COLUMNS",
    "sensor_reference_stats",
]

for name in required_objects_616h:
    assert name in globals(), (
        f"Required object '{name}' is missing."
    )

print("Required objects are available.")

# ------------------------------------------------------------
# 2. Verify frozen drift table
# ------------------------------------------------------------

assert len(frozen_drift_616f) == 90, (
    "Expected exactly 90 frozen drift scenarios."
)

required_drift_columns_616h = [
    "sensor",
    "severity",
    "alpha",
    "direction",
    "expected_final_drift",
    "max_abs_starting_drift",
    "mean_final_drift",
    "min_final_drift",
    "max_final_drift",
]

missing_columns_616h = [
    col
    for col in required_drift_columns_616h
    if col not in frozen_drift_616f.columns
]

assert not missing_columns_616h, (
    f"Frozen drift table is missing columns: "
    f"{missing_columns_616h}"
)

print(
    "Frozen drift scenarios:",
    len(frozen_drift_616f)
)

# ------------------------------------------------------------
# 3. Reconstruct raw validation inputs ONCE
# ------------------------------------------------------------

scaled_inputs_616h = (
    val_scaled[
        PRIMARY_INPUT_COLUMNS
    ]
    .to_numpy(dtype=float)
)

raw_inputs_616h = scaler.inverse_transform(
    scaled_inputs_616h
)

val_raw_616h = val_scaled.copy()

val_raw_616h[
    PRIMARY_INPUT_COLUMNS
] = raw_inputs_616h

roundtrip_616h = scaler.transform(
    val_raw_616h[
        PRIMARY_INPUT_COLUMNS
    ]
)

max_roundtrip_difference_616h = float(
    np.max(
        np.abs(
            roundtrip_616h
            -
            scaled_inputs_616h
        )
    )
)

print(
    "Maximum scaling round-trip difference:",
    max_roundtrip_difference_616h
)

assert max_roundtrip_difference_616h < 1e-10

# ------------------------------------------------------------
# 4. Validation engine IDs
# ------------------------------------------------------------

validation_unit_ids_616h = (
    val_raw_616h[
        "unit_id"
    ]
    .to_numpy()
)

unique_engines_616h = np.unique(
    validation_unit_ids_616h
)

print(
    "Validation engines:",
    len(unique_engines_616h)
)

# ------------------------------------------------------------
# 5. Evaluate every frozen drift scenario
# ------------------------------------------------------------

verification_records_616h = []

for scenario_index_616h, scenario_616h in (
    frozen_drift_616f.reset_index(
        drop=True
    ).iterrows()
):

    sensor_616h = str(
        scenario_616h["sensor"]
    )

    severity_label_616h = str(
        scenario_616h["severity"]
    )

    alpha_616h = float(
        scenario_616h["alpha"]
    )

    direction_616h = int(
        scenario_616h["direction"]
    )

    expected_final_616h = float(
        scenario_616h[
            "expected_final_drift"
        ]
    )

    frozen_starting_616h = float(
        scenario_616h[
            "max_abs_starting_drift"
        ]
    )

    frozen_mean_final_616h = float(
        scenario_616h[
            "mean_final_drift"
        ]
    )

    frozen_min_final_616h = float(
        scenario_616h[
            "min_final_drift"
        ]
    )

    frozen_max_final_616h = float(
        scenario_616h[
            "max_final_drift"
        ]
    )

    # --------------------------------------------------------
    # Apply candidate drift implementation
    #
    # IMPORTANT:
    # severity receives numeric alpha.
    # --------------------------------------------------------

    result_616h = candidate_add_sensor_drift_616f(
        df=val_raw_616h,
        sensor=sensor_616h,
        severity=alpha_616h,
        direction=direction_616h,
        sensor_stats=sensor_reference_stats,
    )

    assert isinstance(
        result_616h,
        tuple
    ), (
        f"Scenario {scenario_index_616h}: "
        "function did not return a tuple."
    )

    assert len(result_616h) == 3, (
        f"Scenario {scenario_index_616h}: "
        f"expected 3 outputs, "
        f"received {len(result_616h)}."
    )

    (
        corrupted_raw_616h,
        drift_vector_616h,
        returned_scalar_616h,
    ) = result_616h

    drift_vector_616h = np.asarray(
        drift_vector_616h,
        dtype=float
    )

    returned_scalar_616h = float(
        returned_scalar_616h
    )

    # --------------------------------------------------------
    # Structural integrity
    # --------------------------------------------------------

    assert (
        corrupted_raw_616h.shape
        ==
        val_raw_616h.shape
    )

    assert drift_vector_616h.shape == (
        len(val_raw_616h),
    )

    assert np.isfinite(
        drift_vector_616h
    ).all()

    assert np.isfinite(
        returned_scalar_616h
    )

    # --------------------------------------------------------
    # Verify non-target inputs unchanged
    # --------------------------------------------------------

    unchanged_columns_616h = [
        col
        for col in PRIMARY_INPUT_COLUMNS
        if col != sensor_616h
    ]

    clean_other_616h = (
        val_raw_616h[
            unchanged_columns_616h
        ]
        .to_numpy(dtype=float)
    )

    corrupted_other_616h = (
        corrupted_raw_616h[
            unchanged_columns_616h
        ]
        .to_numpy(dtype=float)
    )

    max_non_target_change_616h = float(
        np.max(
            np.abs(
                corrupted_other_616h
                -
                clean_other_616h
            )
        )
    )

    assert max_non_target_change_616h < 1e-12, (
        f"Scenario {scenario_index_616h}: "
        "a non-target feature changed."
    )

    # --------------------------------------------------------
    # Verify returned drift vector equals actual sensor change
    # --------------------------------------------------------

    actual_change_616h = (
        corrupted_raw_616h[
            sensor_616h
        ].to_numpy(dtype=float)
        -
        val_raw_616h[
            sensor_616h
        ].to_numpy(dtype=float)
    )

    max_vector_difference_616h = float(
        np.max(
            np.abs(
                actual_change_616h
                -
                drift_vector_616h
            )
        )
    )

    assert max_vector_difference_616h < 1e-10, (
        f"Scenario {scenario_index_616h}: "
        "returned drift vector does not match "
        "actual sensor corruption."
    )

    # --------------------------------------------------------
    # Engine-wise drift behaviour
    # --------------------------------------------------------

    drift_check_616h = pd.DataFrame(
        {
            "unit_id": validation_unit_ids_616h,
            "drift": drift_vector_616h,
        }
    )

    engine_summary_616h = (
        drift_check_616h
        .groupby(
            "unit_id",
            sort=True
        )
        .agg(
            first_drift=("drift", "first"),
            last_drift=("drift", "last"),
        )
    )

    max_abs_starting_616h = float(
        engine_summary_616h[
            "first_drift"
        ]
        .abs()
        .max()
    )

    engine_final_values_616h = (
        engine_summary_616h[
            "last_drift"
        ]
        .to_numpy(dtype=float)
    )

    mean_final_616h = float(
        np.mean(
            engine_final_values_616h
        )
    )

    min_final_616h = float(
        np.min(
            engine_final_values_616h
        )
    )

    max_final_616h = float(
        np.max(
            engine_final_values_616h
        )
    )

    # --------------------------------------------------------
    # Exact frozen-artifact comparisons
    # --------------------------------------------------------

    assert np.isclose(
        returned_scalar_616h,
        expected_final_616h,
        rtol=1e-10,
        atol=1e-10,
    ), (
        f"Scenario {scenario_index_616h}: "
        "returned scalar mismatch."
    )

    assert np.isclose(
        max_abs_starting_616h,
        frozen_starting_616h,
        rtol=1e-10,
        atol=1e-10,
    ), (
        f"Scenario {scenario_index_616h}: "
        "starting drift mismatch."
    )

    assert np.isclose(
        mean_final_616h,
        frozen_mean_final_616h,
        rtol=1e-10,
        atol=1e-10,
    ), (
        f"Scenario {scenario_index_616h}: "
        "mean final drift mismatch."
    )

    assert np.isclose(
        min_final_616h,
        frozen_min_final_616h,
        rtol=1e-10,
        atol=1e-10,
    ), (
        f"Scenario {scenario_index_616h}: "
        "minimum final drift mismatch."
    )

    assert np.isclose(
        max_final_616h,
        frozen_max_final_616h,
        rtol=1e-10,
        atol=1e-10,
    ), (
        f"Scenario {scenario_index_616h}: "
        "maximum final drift mismatch."
    )

    # Every engine should finish at expected final drift.
    assert np.allclose(
        engine_final_values_616h,
        expected_final_616h,
        rtol=1e-10,
        atol=1e-10,
    ), (
        f"Scenario {scenario_index_616h}: "
        "one or more engines did not reach "
        "the expected final drift."
    )

    verification_records_616h.append(
        {
            "scenario_number":
                scenario_index_616h + 1,
            "sensor":
                sensor_616h,
            "severity":
                severity_label_616h,
            "alpha":
                alpha_616h,
            "direction":
                direction_616h,
            "expected_final_drift":
                expected_final_616h,
            "returned_scalar":
                returned_scalar_616h,
            "max_abs_starting_drift":
                max_abs_starting_616h,
            "mean_final_drift":
                mean_final_616h,
            "min_final_drift":
                min_final_616h,
            "max_final_drift":
                max_final_616h,
            "max_vector_difference":
                max_vector_difference_616h,
            "max_non_target_change":
                max_non_target_change_616h,
            "passed":
                True,
        }
    )

    if (
        (scenario_index_616h + 1) % 15 == 0
        or
        (scenario_index_616h + 1) == 90
    ):
        print(
            f"[{scenario_index_616h + 1}/90] "
            "verified"
        )

# ------------------------------------------------------------
# 6. Build verification table
# ------------------------------------------------------------

drift_verification_616h = pd.DataFrame(
    verification_records_616h
)

assert len(
    drift_verification_616h
) == 90

assert drift_verification_616h[
    "passed"
].all()

print(
    "\nVerified scenarios:",
    len(drift_verification_616h)
)

print(
    "Unique sensors:",
    drift_verification_616h[
        "sensor"
    ].nunique()
)

print(
    "Unique severities:",
    drift_verification_616h[
        "severity"
    ].nunique()
)

print(
    "Directions:",
    sorted(
        drift_verification_616h[
            "direction"
        ].unique().tolist()
    )
)

# ------------------------------------------------------------
# 7. Numerical-error summary
# ------------------------------------------------------------

print(
    "\nMaximum drift-vector mismatch:",
    drift_verification_616h[
        "max_vector_difference"
    ].max()
)

print(
    "Maximum non-target change:",
    drift_verification_616h[
        "max_non_target_change"
    ].max()
)

# ------------------------------------------------------------
# 8. Small verification preview
# ------------------------------------------------------------

display(
    drift_verification_616h[
        [
            "sensor",
            "severity",
            "alpha",
            "direction",
            "expected_final_drift",
            "returned_scalar",
            "passed",
        ]
    ].head(12)
)

print(
    "\nStep 6.16H completed successfully."
)

print(
    "All 90 frozen Step-4 drift scenarios were "
    "reproduced by the candidate implementation."
)

In [ ]:
# ============================================================
# Step 6.16I — Full gradual-drift PCA sensor-quality response
# FD001
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd


print(
    "=== Step 6.16I — Full gradual-drift PCA response ==="
)


# ============================================================
# 1. Required-object verification
# ============================================================

required_objects_616i = [
    "candidate_add_sensor_drift_616f",
    "frozen_drift_616f",
    "val_scaled",
    "scaler",
    "sensor_reference_stats",
    "PRIMARY_INPUT_COLUMNS",
    "RETAINED_SENSOR_COLUMNS",
    "SENSOR_INPUT_INDICES",
    "create_sequences",
    "calculate_pca_window_quality",
    "sensor_pca",
    "X_val_clean",
    "y_val_clean",
    "val_seq_meta",
]

missing_objects_616i = [
    name
    for name in required_objects_616i
    if name not in globals()
]

assert not missing_objects_616i, (
    "Missing required objects: "
    f"{missing_objects_616i}"
)

print("Required objects are available.")


# ============================================================
# 2. Frozen configuration
# ============================================================

TARGET_COLUMN_616i = "RUL_cap_125"
WINDOW_SIZE_616i = 30

assert TARGET_COLUMN_616i in val_scaled.columns, (
    f"{TARGET_COLUMN_616i} is missing from validation data."
)

assert X_val_clean.ndim == 3

assert X_val_clean.shape[1] == WINDOW_SIZE_616i, (
    "Validation sequence window does not match "
    "the frozen 30-cycle configuration."
)

assert X_val_clean.shape[2] == len(
    PRIMARY_INPUT_COLUMNS
), (
    "Validation sequence input dimension does not match "
    "PRIMARY_INPUT_COLUMNS."
)

assert len(RETAINED_SENSOR_COLUMNS) == 15

assert len(SENSOR_INPUT_INDICES) == 15

assert len(frozen_drift_616f) == 90

print(
    "Target          :",
    TARGET_COLUMN_616i
)

print(
    "Window size     :",
    WINDOW_SIZE_616i
)

print(
    "Retained sensors:",
    len(RETAINED_SENSOR_COLUMNS)
)

print(
    "Frozen scenarios:",
    len(frozen_drift_616f)
)


# ============================================================
# 3. Prepare Step-6 output directory
# ============================================================

if "STEP6_DIR" in globals():

    STEP6_DIR_616i = Path(
        STEP6_DIR
    )

elif "PROJECT_ROOT" in globals():

    STEP6_DIR_616i = (
        Path(PROJECT_ROOT)
        / "06_results"
        / "step6_reliability_signals"
    )

else:

    raise NameError(
        "Neither STEP6_DIR nor PROJECT_ROOT "
        "is available."
    )

STEP6_DIR_616i.mkdir(
    parents=True,
    exist_ok=True
)

print(
    "Output directory:",
    STEP6_DIR_616i
)


# ============================================================
# 4. Reconstruct raw validation inputs
#    from the frozen scaled validation set
# ============================================================

scaled_inputs_616i = (
    val_scaled[
        PRIMARY_INPUT_COLUMNS
    ]
    .copy()
)

raw_inputs_array_616i = (
    scaler.inverse_transform(
        scaled_inputs_616i
    )
)

val_raw_616i = (
    val_scaled.copy()
)

val_raw_616i[
    PRIMARY_INPUT_COLUMNS
] = raw_inputs_array_616i


# ------------------------------------------------------------
# Scaling round-trip integrity check
# ------------------------------------------------------------

roundtrip_scaled_616i = (
    scaler.transform(
        val_raw_616i[
            PRIMARY_INPUT_COLUMNS
        ]
    )
)

max_roundtrip_difference_616i = float(
    np.max(
        np.abs(
            roundtrip_scaled_616i
            -
            scaled_inputs_616i.to_numpy(
                dtype=float
            )
        )
    )
)

print(
    "\nMaximum scaling round-trip difference:",
    max_roundtrip_difference_616i
)

assert max_roundtrip_difference_616i < 1e-10

print(
    "Raw validation inputs reconstructed successfully."
)


# ============================================================
# 5. Clean PCA sensor-quality reference
# ============================================================

(
    clean_pca_row_616i,
    clean_pca_mean_616i,
    clean_pca_max_616i,
) = calculate_pca_window_quality(
    X_val_clean,
    SENSOR_INPUT_INDICES,
    sensor_pca
)

clean_pca_mean_616i = np.asarray(
    clean_pca_mean_616i,
    dtype=float
)

clean_pca_max_616i = np.asarray(
    clean_pca_max_616i,
    dtype=float
)

assert clean_pca_mean_616i.shape == (
    len(X_val_clean),
)

assert clean_pca_max_616i.shape == (
    len(X_val_clean),
)

assert np.isfinite(
    clean_pca_mean_616i
).all()

assert np.isfinite(
    clean_pca_max_616i
).all()


# ============================================================
# 6. Development-data 95th percentile PCA threshold
# ============================================================

if "dev_pca_mean_rmse" in globals():

    dev_pca_mean_reference_616i = np.asarray(
        dev_pca_mean_rmse,
        dtype=float
    )

elif "X_dev_clean" in globals():

    (
        _,
        dev_pca_mean_reference_616i,
        _,
    ) = calculate_pca_window_quality(
        X_dev_clean,
        SENSOR_INPUT_INDICES,
        sensor_pca
    )

    dev_pca_mean_reference_616i = np.asarray(
        dev_pca_mean_reference_616i,
        dtype=float
    )

else:

    raise NameError(
        "A development PCA mean-RE reference is required. "
        "Neither dev_pca_mean_rmse nor X_dev_clean exists."
    )


DEV_PCA_P95_616i = float(
    np.percentile(
        dev_pca_mean_reference_616i,
        95
    )
)

CLEAN_MEAN_RE_616i = float(
    np.mean(
        clean_pca_mean_616i
    )
)

CLEAN_MEDIAN_RE_616i = float(
    np.median(
        clean_pca_mean_616i
    )
)

CLEAN_ABOVE_DEV95_PCT_616i = float(
    100.0
    * np.mean(
        clean_pca_mean_616i
        >
        DEV_PCA_P95_616i
    )
)

CLEAN_MEAN_MAX_RE_616i = float(
    np.mean(
        clean_pca_max_616i
    )
)


print(
    "\nClean PCA reference:"
)

print(
    "Development PCA mean-RE 95th percentile:",
    round(
        DEV_PCA_P95_616i,
        6
    )
)

print(
    "Clean validation mean PCA RE:",
    round(
        CLEAN_MEAN_RE_616i,
        6
    )
)

print(
    "Clean validation median PCA RE:",
    round(
        CLEAN_MEDIAN_RE_616i,
        6
    )
)

print(
    "Clean validation above dev-P95 (%):",
    round(
        CLEAN_ABOVE_DEV95_PCT_616i,
        3
    )
)


# ============================================================
# 7. Robust metadata-alignment helper
# ============================================================

def assert_metadata_alignment_616i(
    reference_df,
    candidate_df
):

    assert (
        reference_df.shape
        ==
        candidate_df.shape
    ), (
        "Metadata shapes do not match."
    )

    assert (
        reference_df.columns.tolist()
        ==
        candidate_df.columns.tolist()
    ), (
        "Metadata columns do not match."
    )

    for column_616i in reference_df.columns:

        ref_values_616i = (
            reference_df[
                column_616i
            ].to_numpy()
        )

        cand_values_616i = (
            candidate_df[
                column_616i
            ].to_numpy()
        )

        if (
            pd.api.types.is_numeric_dtype(
                reference_df[
                    column_616i
                ]
            )
            and
            pd.api.types.is_numeric_dtype(
                candidate_df[
                    column_616i
                ]
            )
        ):

            assert np.allclose(
                ref_values_616i.astype(
                    float
                ),
                cand_values_616i.astype(
                    float
                ),
                rtol=0.0,
                atol=1e-12
            ), (
                "Metadata mismatch in numeric column "
                f"'{column_616i}'."
            )

        else:

            assert np.array_equal(
                ref_values_616i.astype(
                    str
                ),
                cand_values_616i.astype(
                    str
                )
            ), (
                "Metadata mismatch in column "
                f"'{column_616i}'."
            )


# ============================================================
# 8. Run all 90 gradual-drift scenarios
# ============================================================

drift_pca_records_616i = []

frozen_plan_616i = (
    frozen_drift_616f
    .reset_index(
        drop=True
    )
    .copy()
)

for scenario_index_616i, scenario_616i in (
    frozen_plan_616i.iterrows()
):

    # --------------------------------------------------------
    # Frozen scenario definition
    # --------------------------------------------------------

    sensor_616i = str(
        scenario_616i[
            "sensor"
        ]
    )

    severity_616i = str(
        scenario_616i[
            "severity"
        ]
    )

    alpha_616i = float(
        scenario_616i[
            "alpha"
        ]
    )

    direction_616i = int(
        scenario_616i[
            "direction"
        ]
    )

    expected_final_drift_616i = float(
        scenario_616i[
            "expected_final_drift"
        ]
    )


    # --------------------------------------------------------
    # 8A. Apply VERIFIED gradual-drift implementation
    # --------------------------------------------------------

    drift_result_616i = (
        candidate_add_sensor_drift_616f(
            df=val_raw_616i,
            sensor=sensor_616i,
            severity=alpha_616i,
            direction=direction_616i,
            sensor_stats=sensor_reference_stats,
        )
    )

    assert isinstance(
        drift_result_616i,
        tuple
    )

    assert len(
        drift_result_616i
    ) == 3

    (
        drift_raw_616i,
        drift_vector_616i,
        returned_final_drift_616i,
    ) = drift_result_616i

    drift_vector_616i = np.asarray(
        drift_vector_616i,
        dtype=float
    )

    returned_final_drift_616i = float(
        returned_final_drift_616i
    )


    # --------------------------------------------------------
    # 8B. Frozen-artifact consistency
    # --------------------------------------------------------

    assert np.isclose(
        returned_final_drift_616i,
        expected_final_drift_616i,
        rtol=1e-10,
        atol=1e-10
    ), (
        f"Frozen drift mismatch for "
        f"{sensor_616i}, "
        f"{severity_616i}, "
        f"direction {direction_616i}."
    )


    # --------------------------------------------------------
    # 8C. Scale corrupted validation data
    # --------------------------------------------------------

    drift_scaled_616i = (
        drift_raw_616i.copy()
    )

    drift_scaled_616i[
        PRIMARY_INPUT_COLUMNS
    ] = scaler.transform(
        drift_raw_616i[
            PRIMARY_INPUT_COLUMNS
        ]
    )

    drift_input_array_616i = (
        drift_scaled_616i[
            PRIMARY_INPUT_COLUMNS
        ]
        .to_numpy(
            dtype=float
        )
    )

    assert np.isfinite(
        drift_input_array_616i
    ).all(), (
        "Drift scenario produced invalid scaled values."
    )


    # --------------------------------------------------------
    # 8D. Recreate sequences
    # --------------------------------------------------------

    (
        X_drift_616i,
        y_drift_616i,
        drift_seq_meta_616i,
    ) = create_sequences(
        df=drift_scaled_616i,
        feature_columns=PRIMARY_INPUT_COLUMNS,
        target_column=TARGET_COLUMN_616i,
        window_size=WINDOW_SIZE_616i,
    )


    # --------------------------------------------------------
    # 8E. Strict sequence alignment
    # --------------------------------------------------------

    assert (
        X_drift_616i.shape
        ==
        X_val_clean.shape
    ), (
        "Corrupted sequence shape differs "
        "from clean sequence shape."
    )

    assert (
        y_drift_616i.shape
        ==
        y_val_clean.shape
    )

    assert np.allclose(
        np.asarray(
            y_drift_616i,
            dtype=float
        ),
        np.asarray(
            y_val_clean,
            dtype=float
        ),
        rtol=0.0,
        atol=1e-12
    ), (
        "Targets changed after drift corruption."
    )

    assert_metadata_alignment_616i(
        val_seq_meta.reset_index(
            drop=True
        ),
        drift_seq_meta_616i.reset_index(
            drop=True
        ),
    )


    # --------------------------------------------------------
    # 8F. Generate PCA sensor-quality signal
    # --------------------------------------------------------

    (
        drift_pca_row_616i,
        drift_pca_mean_616i,
        drift_pca_max_616i,
    ) = calculate_pca_window_quality(
        X_drift_616i,
        SENSOR_INPUT_INDICES,
        sensor_pca,
    )

    drift_pca_mean_616i = np.asarray(
        drift_pca_mean_616i,
        dtype=float
    )

    drift_pca_max_616i = np.asarray(
        drift_pca_max_616i,
        dtype=float
    )

    assert drift_pca_mean_616i.shape == (
        len(clean_pca_mean_616i),
    )

    assert np.isfinite(
        drift_pca_mean_616i
    ).all()

    assert np.isfinite(
        drift_pca_max_616i
    ).all()


    # --------------------------------------------------------
    # 8G. Paired clean-vs-drift PCA response
    # --------------------------------------------------------

    paired_delta_616i = (
        drift_pca_mean_616i
        -
        clean_pca_mean_616i
    )

    safe_clean_denominator_616i = np.maximum(
        np.abs(
            clean_pca_mean_616i
        ),
        1e-12
    )

    paired_change_pct_616i = (
        100.0
        *
        paired_delta_616i
        /
        safe_clean_denominator_616i
    )

    corrupted_mean_re_616i = float(
        np.mean(
            drift_pca_mean_616i
        )
    )

    corrupted_median_re_616i = float(
        np.median(
            drift_pca_mean_616i
        )
    )

    mean_re_change_pct_616i = float(
        100.0
        *
        (
            corrupted_mean_re_616i
            -
            CLEAN_MEAN_RE_616i
        )
        /
        CLEAN_MEAN_RE_616i
    )

    median_re_change_pct_616i = float(
        np.median(
            paired_change_pct_616i
        )
    )

    fraction_increased_pct_616i = float(
        100.0
        *
        np.mean(
            drift_pca_mean_616i
            >
            clean_pca_mean_616i
        )
    )

    corrupted_above_dev95_pct_616i = float(
        100.0
        *
        np.mean(
            drift_pca_mean_616i
            >
            DEV_PCA_P95_616i
        )
    )

    above_dev95_delta_pp_616i = float(
        corrupted_above_dev95_pct_616i
        -
        CLEAN_ABOVE_DEV95_PCT_616i
    )

    corrupted_mean_max_re_616i = float(
        np.mean(
            drift_pca_max_616i
        )
    )

    mean_max_re_change_pct_616i = float(
        100.0
        *
        (
            corrupted_mean_max_re_616i
            -
            CLEAN_MEAN_MAX_RE_616i
        )
        /
        CLEAN_MEAN_MAX_RE_616i
    )


    # --------------------------------------------------------
    # 8H. Store scenario result
    # --------------------------------------------------------

    drift_pca_records_616i.append(
        {
            "scenario_id":
                f"FD001_DRIFT_{scenario_index_616i + 1:03d}",

            "sensor":
                sensor_616i,

            "severity":
                severity_616i,

            "alpha":
                alpha_616i,

            "direction":
                direction_616i,

            "expected_final_drift":
                expected_final_drift_616i,

            "returned_final_drift":
                returned_final_drift_616i,

            "clean_mean_re":
                CLEAN_MEAN_RE_616i,

            "corrupted_mean_re":
                corrupted_mean_re_616i,

            "mean_RE_change_pct":
                mean_re_change_pct_616i,

            "clean_median_re":
                CLEAN_MEDIAN_RE_616i,

            "corrupted_median_re":
                corrupted_median_re_616i,

            "median_RE_change_pct":
                median_re_change_pct_616i,

            "fraction_increased_pct":
                fraction_increased_pct_616i,

            "clean_above_dev95_pct":
                CLEAN_ABOVE_DEV95_PCT_616i,

            "corrupted_above_dev95_pct":
                corrupted_above_dev95_pct_616i,

            "above_dev95_delta_pp":
                above_dev95_delta_pp_616i,

            "clean_mean_max_re":
                CLEAN_MEAN_MAX_RE_616i,

            "corrupted_mean_max_re":
                corrupted_mean_max_re_616i,

            "mean_max_RE_change_pct":
                mean_max_re_change_pct_616i,
        }
    )


    if (
        (scenario_index_616i + 1) % 15 == 0
        or
        (scenario_index_616i + 1) == 90
    ):

        print(
            f"[{scenario_index_616i + 1}/90] completed"
        )


# ============================================================
# 9. Build full scenario table
# ============================================================

drift_pca_response_df_616i = pd.DataFrame(
    drift_pca_records_616i
)

assert len(
    drift_pca_response_df_616i
) == 90

assert (
    drift_pca_response_df_616i[
        "sensor"
    ].nunique()
    == 15
)

assert (
    drift_pca_response_df_616i[
        "severity"
    ].nunique()
    == 3
)

assert sorted(
    drift_pca_response_df_616i[
        "direction"
    ].unique().tolist()
) == [-1, 1]


print(
    "\nDrift PCA-response scenarios:",
    len(
        drift_pca_response_df_616i
    )
)

print(
    "Unique sensors:",
    drift_pca_response_df_616i[
        "sensor"
    ].nunique()
)

print(
    "Unique severities:",
    drift_pca_response_df_616i[
        "severity"
    ].nunique()
)

print(
    "Directions:",
    sorted(
        drift_pca_response_df_616i[
            "direction"
        ].unique().tolist()
    )
)


# ============================================================
# 10. Severity summary
# ============================================================

drift_severity_summary_616i = (
    drift_pca_response_df_616i
    .groupby(
        "severity",
        as_index=True
    )
    .agg(
        scenarios=(
            "scenario_id",
            "count"
        ),

        mean_RE_change_pct=(
            "mean_RE_change_pct",
            "mean"
        ),

        median_RE_change_pct=(
            "median_RE_change_pct",
            "median"
        ),

        mean_fraction_increased_pct=(
            "fraction_increased_pct",
            "mean"
        ),

        mean_above_dev95_pct=(
            "corrupted_above_dev95_pct",
            "mean"
        ),

        mean_above_dev95_delta_pp=(
            "above_dev95_delta_pp",
            "mean"
        ),

        max_RE_change_pct=(
            "mean_RE_change_pct",
            "max"
        ),
    )
    .reindex(
        [
            "low",
            "medium",
            "high",
        ]
    )
)


print(
    "\n=== Gradual-drift PCA response by severity ==="
)

display(
    drift_severity_summary_616i.round(
        3
    )
)


# ============================================================
# 11. Sensor-wise summary
# ============================================================

drift_sensor_summary_616i = (
    drift_pca_response_df_616i
    .groupby(
        "sensor"
    )
    .agg(
        scenarios=(
            "scenario_id",
            "count"
        ),

        mean_RE_change_pct=(
            "mean_RE_change_pct",
            "mean"
        ),

        median_RE_change_pct=(
            "median_RE_change_pct",
            "median"
        ),

        min_RE_change_pct=(
            "mean_RE_change_pct",
            "min"
        ),

        max_RE_change_pct=(
            "mean_RE_change_pct",
            "max"
        ),

        mean_fraction_increased_pct=(
            "fraction_increased_pct",
            "mean"
        ),

        mean_above_dev95_pct=(
            "corrupted_above_dev95_pct",
            "mean"
        ),

        max_above_dev95_pct=(
            "corrupted_above_dev95_pct",
            "max"
        ),
    )
    .sort_values(
        "mean_RE_change_pct",
        ascending=False
    )
)


print(
    "\n=== Sensor-wise PCA sensitivity "
    "across gradual-drift scenarios ==="
)

display(
    drift_sensor_summary_616i.round(
        3
    )
)


# ============================================================
# 12. Direction summary
# ============================================================

drift_direction_summary_616i = (
    drift_pca_response_df_616i
    .groupby(
        "direction"
    )
    .agg(
        scenarios=(
            "scenario_id",
            "count"
        ),

        mean_RE_change_pct=(
            "mean_RE_change_pct",
            "mean"
        ),

        median_RE_change_pct=(
            "median_RE_change_pct",
            "median"
        ),

        mean_fraction_increased_pct=(
            "fraction_increased_pct",
            "mean"
        ),

        mean_above_dev95_delta_pp=(
            "above_dev95_delta_pp",
            "mean"
        ),

        max_RE_change_pct=(
            "mean_RE_change_pct",
            "max"
        ),
    )
)


print(
    "\n=== Gradual-drift PCA response by direction ==="
)

display(
    drift_direction_summary_616i.round(
        3
    )
)


# ============================================================
# 13. Strongest PCA responses
# ============================================================

strongest_drift_responses_616i = (
    drift_pca_response_df_616i
    .sort_values(
        "mean_RE_change_pct",
        ascending=False
    )
    .head(15)
)


print(
    "\n=== Strongest 15 gradual-drift PCA responses ==="
)

display(
    strongest_drift_responses_616i.round(
        3
    )
)


# ============================================================
# 14. Save results
# ============================================================

FULL_PATH_616i = (
    STEP6_DIR_616i
    / "fd001_pca_drift_response_90_scenarios.csv"
)

SENSOR_PATH_616i = (
    STEP6_DIR_616i
    / "fd001_pca_drift_sensor_summary.csv"
)

SEVERITY_PATH_616i = (
    STEP6_DIR_616i
    / "fd001_pca_drift_severity_summary.csv"
)

DIRECTION_PATH_616i = (
    STEP6_DIR_616i
    / "fd001_pca_drift_direction_summary.csv"
)


drift_pca_response_df_616i.to_csv(
    FULL_PATH_616i,
    index=False
)

drift_sensor_summary_616i.to_csv(
    SENSOR_PATH_616i
)

drift_severity_summary_616i.to_csv(
    SEVERITY_PATH_616i
)

drift_direction_summary_616i.to_csv(
    DIRECTION_PATH_616i
)


print(
    "\nSaved:"
)

print(
    FULL_PATH_616i
)

print(
    SENSOR_PATH_616i
)

print(
    SEVERITY_PATH_616i
)

print(
    DIRECTION_PATH_616i
)


# ============================================================
# 15. Final Step-6.16I verification
# ============================================================

assert FULL_PATH_616i.exists()
assert SENSOR_PATH_616i.exists()
assert SEVERITY_PATH_616i.exists()
assert DIRECTION_PATH_616i.exists()

assert (
    drift_pca_response_df_616i[
        [
            "mean_RE_change_pct",
            "median_RE_change_pct",
            "fraction_increased_pct",
            "corrupted_above_dev95_pct",
            "above_dev95_delta_pp",
        ]
    ]
    .notna()
    .all()
    .all()
)

assert np.isfinite(
    drift_pca_response_df_616i[
        [
            "mean_RE_change_pct",
            "median_RE_change_pct",
            "fraction_increased_pct",
            "corrupted_above_dev95_pct",
            "above_dev95_delta_pp",
        ]
    ].to_numpy(
        dtype=float
    )
).all()


print(
    "\nStep 6.16I completed successfully."
)

print(
    "All 90 verified gradual-drift scenarios "
    "were evaluated using the frozen PCA "
    "sensor-quality signal."
)

In [ ]:
# ============================================================
# Step 6.17A — Inspect frozen Step-4 dropout implementation
# ============================================================

from pathlib import Path
import json
import inspect
import pandas as pd


print(
    "=== Step 6.17A — Inspect frozen "
    "Step-4 dropout implementation ==="
)


# ------------------------------------------------------------
# 1. Locate project root safely
# ------------------------------------------------------------

cwd = Path.cwd()

if (cwd / "03_notebooks").exists():
    PROJECT_ROOT_617 = cwd
elif (cwd.parent / "03_notebooks").exists():
    PROJECT_ROOT_617 = cwd.parent
else:
    raise FileNotFoundError(
        "Could not locate C:\\RUL_Reliability_Research project root."
    )

STEP4_NOTEBOOK_617 = (
    PROJECT_ROOT_617
    / "03_notebooks"
    / "04_sensor_degradation_fd001.ipynb"
)

print("\nProject root :", PROJECT_ROOT_617)
print("Step-4 file  :", STEP4_NOTEBOOK_617)
print("File exists  :", STEP4_NOTEBOOK_617.exists())

assert STEP4_NOTEBOOK_617.exists(), (
    "Step-4 notebook could not be found."
)


# ------------------------------------------------------------
# 2. Check what dropout-related objects currently exist
#    in memory
# ------------------------------------------------------------

print("\n=== Current in-memory availability ===")

objects_to_check_617 = [
    "add_sensor_dropout",
    "apply_corruption",
    "CORRUPTION_LEVELS",
    "sensor_reference_stats",
    "sensor_temporal_stats",
    "val_scaled",
    "scaler",
    "create_sequences",
    "calculate_pca_window_quality",
    "sensor_pca",
    "X_val_clean",
    "y_val_clean",
    "val_seq_meta",
    "PRIMARY_INPUT_COLUMNS",
    "RETAINED_SENSOR_COLUMNS",
    "SENSOR_INPUT_INDICES",
]

for name in objects_to_check_617:
    print(
        f"{name:<32}:",
        name in globals()
    )


# ------------------------------------------------------------
# 3. If dropout helper is already in memory, inspect signature
#    only. Do NOT execute it.
# ------------------------------------------------------------

print("\n=== In-memory dropout helper signature ===")

if "add_sensor_dropout" in globals():
    try:
        print(
            "add_sensor_dropout:",
            inspect.signature(add_sensor_dropout)
        )
    except Exception as exc:
        print(
            "Could not inspect signature:",
            repr(exc)
        )
else:
    print(
        "add_sensor_dropout is not currently loaded in memory."
    )


# ------------------------------------------------------------
# 4. Read Step-4 notebook as JSON
# ------------------------------------------------------------

with open(
    STEP4_NOTEBOOK_617,
    "r",
    encoding="utf-8"
) as f:
    step4_nb_617 = json.load(f)

cells_617 = step4_nb_617.get("cells", [])

print(
    "\nTotal Step-4 notebook cells:",
    len(cells_617)
)


# ------------------------------------------------------------
# 5. Find every code cell containing dropout helper definition
# ------------------------------------------------------------

dropout_definition_cells_617 = []

for idx, cell in enumerate(cells_617):

    if cell.get("cell_type") != "code":
        continue

    source = "".join(
        cell.get("source", [])
    )

    if "def add_sensor_dropout" in source:
        dropout_definition_cells_617.append(
            (idx, source)
        )


print(
    "\nNumber of cells containing "
    "'def add_sensor_dropout':",
    len(dropout_definition_cells_617)
)


for idx, source in dropout_definition_cells_617:

    print("\n" + "=" * 72)
    print(
        f"Step-4 notebook cell index: {idx}"
    )
    print("=" * 72)

    # Print the full source of the definition-containing cell.
    # We are only inspecting; nothing is executed.
    print(source)


# ------------------------------------------------------------
# 6. Look for frozen dropout-validation CSV artifacts
# ------------------------------------------------------------

EXPERIMENT_DIR_617 = (
    PROJECT_ROOT_617
    / "05_experiments"
    / "sensor_degradation"
)

dropout_files_617 = sorted(
    EXPERIMENT_DIR_617.glob(
        "*dropout*.csv"
    )
)

print("\n=== Dropout-related frozen CSV files ===")

if not dropout_files_617:
    print(
        "No dropout-related CSV files were found."
    )
else:
    for path in dropout_files_617:
        print("-", path.name)


# ------------------------------------------------------------
# 7. Inspect structure of each dropout CSV
# ------------------------------------------------------------

for path in dropout_files_617:

    print("\n" + "=" * 72)
    print("File:", path.name)
    print("=" * 72)

    df_tmp_617 = pd.read_csv(path)

    print("Shape   :", df_tmp_617.shape)
    print("Columns :", df_tmp_617.columns.tolist())

    print("\nFirst 5 rows:")
    display(
        df_tmp_617.head()
    )


# ------------------------------------------------------------
# 8. Frozen dropout severity configuration
# ------------------------------------------------------------

print(
    "\n=== Frozen dropout configuration ==="
)

if "CORRUPTION_LEVELS" in globals():

    if "dropout" in CORRUPTION_LEVELS:
        print(
            "Dropout severity levels:",
            CORRUPTION_LEVELS["dropout"]
        )
    else:
        print(
            "'dropout' key not found in CORRUPTION_LEVELS."
        )

else:
    print(
        "CORRUPTION_LEVELS is not currently loaded."
    )


print(
    "\nStep 6.17A completed."
)
print(
    "No corruption function was executed "
    "and no frozen artifact was modified."
)

In [ ]:
# ============================================================
# Step 6.17B — Extract exact frozen add_sensor_dropout
#               definitions from Step-4 notebook
# ============================================================

from pathlib import Path
import json
import ast
import hashlib


print(
    "=== Step 6.17B — Extract frozen "
    "add_sensor_dropout definitions ==="
)


# ------------------------------------------------------------
# 1. Locate project root
# ------------------------------------------------------------

cwd = Path.cwd()

if (cwd / "03_notebooks").exists():
    PROJECT_ROOT_617B = cwd
elif (cwd.parent / "03_notebooks").exists():
    PROJECT_ROOT_617B = cwd.parent
else:
    raise FileNotFoundError(
        "Could not locate project root."
    )


STEP4_NOTEBOOK_617B = (
    PROJECT_ROOT_617B
    / "03_notebooks"
    / "04_sensor_degradation_fd001.ipynb"
)

assert STEP4_NOTEBOOK_617B.exists(), (
    f"Step-4 notebook not found: "
    f"{STEP4_NOTEBOOK_617B}"
)

print("Project root :", PROJECT_ROOT_617B)
print("Step-4 file  :", STEP4_NOTEBOOK_617B)


# ------------------------------------------------------------
# 2. Load notebook JSON
# ------------------------------------------------------------

with open(
    STEP4_NOTEBOOK_617B,
    "r",
    encoding="utf-8"
) as f:
    nb_617b = json.load(f)


# ------------------------------------------------------------
# 3. Safely extract exact function definitions using AST
# ------------------------------------------------------------

definitions_617b = []

for cell_index, cell in enumerate(
    nb_617b.get("cells", [])
):

    if cell.get("cell_type") != "code":
        continue

    source = "".join(
        cell.get("source", [])
    )

    if "def add_sensor_dropout" not in source:
        continue

    try:
        tree = ast.parse(source)
    except SyntaxError as exc:
        print(
            f"\nCell {cell_index} could not be parsed:",
            exc
        )
        continue

    lines = source.splitlines()

    for node in tree.body:

        if (
            isinstance(
                node,
                (ast.FunctionDef, ast.AsyncFunctionDef)
            )
            and node.name == "add_sensor_dropout"
        ):

            if not hasattr(node, "end_lineno"):
                raise RuntimeError(
                    "Python AST does not expose end_lineno."
                )

            function_source = "\n".join(
                lines[
                    node.lineno - 1:
                    node.end_lineno
                ]
            )

            function_hash = hashlib.sha256(
                function_source.encode("utf-8")
            ).hexdigest()

            definitions_617b.append(
                {
                    "cell_index": cell_index,
                    "source": function_source,
                    "sha256": function_hash,
                    "line_count": (
                        node.end_lineno
                        - node.lineno
                        + 1
                    ),
                }
            )


# ------------------------------------------------------------
# 4. Report exactly what exists
# ------------------------------------------------------------

print(
    "\nNumber of extracted "
    "add_sensor_dropout definitions:",
    len(definitions_617b)
)


if len(definitions_617b) == 0:
    raise RuntimeError(
        "No add_sensor_dropout definition was found."
    )


for i, item in enumerate(
    definitions_617b,
    start=1
):

    print("\n" + "=" * 78)
    print(
        f"DEFINITION {i}"
    )
    print("=" * 78)

    print(
        "Notebook cell index :",
        item["cell_index"]
    )

    print(
        "Function line count :",
        item["line_count"]
    )

    print(
        "SHA-256             :",
        item["sha256"]
    )

    print("\nExact function source:\n")

    print(
        item["source"]
    )


# ------------------------------------------------------------
# 5. Compare definitions if more than one exists
# ------------------------------------------------------------

if len(definitions_617b) > 1:

    unique_hashes_617b = {
        item["sha256"]
        for item in definitions_617b
    }

    print(
        "\nDefinitions identical:",
        len(unique_hashes_617b) == 1
    )

    print(
        "Unique implementation count:",
        len(unique_hashes_617b)
    )

else:

    print(
        "\nOnly one frozen dropout "
        "definition was found."
    )


print(
    "\nStep 6.17B completed."
)

print(
    "Nothing was executed or modified."
)

In [ ]:
# ============================================================
# Step 6.17C — Compare frozen add_sensor_dropout definitions
# ============================================================

from pathlib import Path
import json
import ast
import difflib
import hashlib


print(
    "=== Step 6.17C — Compare frozen "
    "add_sensor_dropout definitions ==="
)


# ------------------------------------------------------------
# 1. Locate project root safely
# ------------------------------------------------------------

cwd = Path.cwd()

if (cwd / "03_notebooks").exists():
    PROJECT_ROOT_617C = cwd
elif (cwd.parent / "03_notebooks").exists():
    PROJECT_ROOT_617C = cwd.parent
else:
    raise FileNotFoundError(
        "Could not locate project root."
    )


STEP4_NOTEBOOK_617C = (
    PROJECT_ROOT_617C
    / "03_notebooks"
    / "04_sensor_degradation_fd001.ipynb"
)

assert STEP4_NOTEBOOK_617C.exists(), (
    f"Step-4 notebook not found: "
    f"{STEP4_NOTEBOOK_617C}"
)


# ------------------------------------------------------------
# 2. Load notebook
# ------------------------------------------------------------

with open(
    STEP4_NOTEBOOK_617C,
    "r",
    encoding="utf-8"
) as f:
    nb_617c = json.load(f)


# ------------------------------------------------------------
# 3. Extract exact add_sensor_dropout definitions
# ------------------------------------------------------------

definitions_617c = []

for cell_index, cell in enumerate(
    nb_617c.get("cells", [])
):

    if cell.get("cell_type") != "code":
        continue

    source = "".join(
        cell.get("source", [])
    )

    if "def add_sensor_dropout" not in source:
        continue

    try:
        tree = ast.parse(source)
    except SyntaxError:
        continue

    source_lines = source.splitlines()

    for node in tree.body:

        if (
            isinstance(node, ast.FunctionDef)
            and node.name == "add_sensor_dropout"
        ):

            function_source = "\n".join(
                source_lines[
                    node.lineno - 1:
                    node.end_lineno
                ]
            )

            definitions_617c.append(
                {
                    "cell_index": cell_index,
                    "source": function_source,
                    "lines": function_source.splitlines(),
                    "node": node,
                    "sha256": hashlib.sha256(
                        function_source.encode("utf-8")
                    ).hexdigest(),
                }
            )


print(
    "\nNumber of extracted definitions:",
    len(definitions_617c)
)

if len(definitions_617c) < 2:
    raise RuntimeError(
        "Expected at least two frozen dropout "
        "definitions for comparison."
    )


# ------------------------------------------------------------
# 4. Basic information about each definition
# ------------------------------------------------------------

for i, item in enumerate(
    definitions_617c,
    start=1
):

    node = item["node"]

    arg_names = [
        arg.arg
        for arg in node.args.args
    ]

    defaults = node.args.defaults

    print("\n" + "-" * 72)
    print(f"DEFINITION {i}")
    print("-" * 72)

    print(
        "Notebook cell index :",
        item["cell_index"]
    )

    print(
        "Function arguments   :",
        arg_names
    )

    print(
        "Default count        :",
        len(defaults)
    )

    print(
        "Function lines       :",
        len(item["lines"])
    )

    print(
        "SHA-256              :",
        item["sha256"]
    )


# ------------------------------------------------------------
# 5. Show all return statements
# ------------------------------------------------------------

print("\n=== Return structures ===")

for i, item in enumerate(
    definitions_617c,
    start=1
):

    return_nodes = [
        node
        for node in ast.walk(item["node"])
        if isinstance(node, ast.Return)
    ]

    print(
        f"\nDefinition {i} "
        f"(cell {item['cell_index']}):"
    )

    if not return_nodes:
        print("  No return statement found.")
        continue

    for j, return_node in enumerate(
        return_nodes,
        start=1
    ):

        try:
            return_text = ast.unparse(
                return_node.value
            )
        except Exception:
            return_text = "<could not unparse>"

        print(
            f"  Return {j}: {return_text}"
        )


# ------------------------------------------------------------
# 6. Compare the first two definitions using changed blocks
# ------------------------------------------------------------

left = definitions_617c[0]
right = definitions_617c[1]

matcher = difflib.SequenceMatcher(
    None,
    left["lines"],
    right["lines"]
)

changed_blocks = []

for tag, i1, i2, j1, j2 in matcher.get_opcodes():

    if tag == "equal":
        continue

    changed_blocks.append(
        {
            "tag": tag,
            "left_start": i1,
            "left_end": i2,
            "right_start": j1,
            "right_end": j2,
        }
    )


print(
    "\nTotal changed blocks:",
    len(changed_blocks)
)


# ------------------------------------------------------------
# 7. Print only the changed portions
# ------------------------------------------------------------

for block_no, block in enumerate(
    changed_blocks,
    start=1
):

    print("\n" + "=" * 78)
    print(
        f"CHANGE BLOCK {block_no}: "
        f"{block['tag'].upper()}"
    )
    print("=" * 78)

    i1 = block["left_start"]
    i2 = block["left_end"]

    j1 = block["right_start"]
    j2 = block["right_end"]

    print(
        f"\nDefinition 1 / "
        f"cell {left['cell_index']}:"
    )

    if i1 == i2:
        print("  <no lines>")
    else:
        for idx in range(i1, i2):
            print(
                f"{idx + 1:>4}: "
                f"{left['lines'][idx]}"
            )

    print(
        f"\nDefinition 2 / "
        f"cell {right['cell_index']}:"
    )

    if j1 == j2:
        print("  <no lines>")
    else:
        for idx in range(j1, j2):
            print(
                f"{idx + 1:>4}: "
                f"{right['lines'][idx]}"
            )


# ------------------------------------------------------------
# 8. Final confirmation
# ------------------------------------------------------------

print(
    "\nStep 6.17C completed."
)

print(
    "No dropout function was executed."
)

print(
    "No frozen artifact was modified."
)

In [ ]:
# ============================================================
# Step 6.17C-2 — Compact comparison of dropout definitions
# ============================================================

from pathlib import Path
import json
import ast
import difflib


print(
    "=== Step 6.17C-2 — Compact dropout comparison ==="
)


# ------------------------------------------------------------
# 1. Locate Step-4 notebook
# ------------------------------------------------------------

cwd = Path.cwd()

if (cwd / "03_notebooks").exists():
    project_root_617c2 = cwd
elif (cwd.parent / "03_notebooks").exists():
    project_root_617c2 = cwd.parent
else:
    raise FileNotFoundError(
        "Could not locate project root."
    )

step4_path_617c2 = (
    project_root_617c2
    / "03_notebooks"
    / "04_sensor_degradation_fd001.ipynb"
)

assert step4_path_617c2.exists()


# ------------------------------------------------------------
# 2. Extract the two exact function definitions
# ------------------------------------------------------------

with open(
    step4_path_617c2,
    "r",
    encoding="utf-8"
) as f:
    nb_617c2 = json.load(f)


defs_617c2 = []

for cell_idx, cell in enumerate(
    nb_617c2["cells"]
):

    if cell.get("cell_type") != "code":
        continue

    source = "".join(
        cell.get("source", [])
    )

    if "def add_sensor_dropout" not in source:
        continue

    tree = ast.parse(source)
    source_lines = source.splitlines()

    for node in tree.body:

        if (
            isinstance(node, ast.FunctionDef)
            and node.name == "add_sensor_dropout"
        ):

            function_lines = source_lines[
                node.lineno - 1:
                node.end_lineno
            ]

            defs_617c2.append(
                {
                    "cell": cell_idx,
                    "lines": function_lines,
                    "node": node,
                }
            )


assert len(defs_617c2) == 2, (
    "Expected exactly two dropout definitions."
)


left_617c2 = defs_617c2[0]
right_617c2 = defs_617c2[1]


print(
    "Definition 1 cell:",
    left_617c2["cell"]
)

print(
    "Definition 2 cell:",
    right_617c2["cell"]
)


# ------------------------------------------------------------
# 3. Print exact return statements
# ------------------------------------------------------------

print("\n=== RETURN STATEMENTS ===")

for number, item in enumerate(
    defs_617c2,
    start=1
):

    returns = [
        node
        for node in ast.walk(item["node"])
        if isinstance(node, ast.Return)
    ]

    print(
        f"\nDefinition {number} "
        f"(cell {item['cell']}):"
    )

    for return_node in returns:

        if return_node.value is None:
            print("return None")
        else:
            print(
                "return",
                ast.unparse(return_node.value)
            )


# ------------------------------------------------------------
# 4. Print only changed code blocks
# ------------------------------------------------------------

print("\n=== EXACT CHANGED BLOCKS ===")

matcher_617c2 = difflib.SequenceMatcher(
    None,
    left_617c2["lines"],
    right_617c2["lines"]
)

change_number_617c2 = 0

for tag, i1, i2, j1, j2 in matcher_617c2.get_opcodes():

    if tag == "equal":
        continue

    change_number_617c2 += 1

    print(
        "\n" + "=" * 65
    )

    print(
        f"CHANGE {change_number_617c2}: "
        f"{tag.upper()}"
    )

    print("=" * 65)

    print(
        f"\nCell {left_617c2['cell']}:"
    )

    if i1 == i2:
        print("<nothing>")
    else:
        for line_no in range(i1, i2):

            print(
                f"{line_no + 1:>3}: "
                f"{left_617c2['lines'][line_no]}"
            )

    print(
        f"\nCell {right_617c2['cell']}:"
    )

    if j1 == j2:
        print("<nothing>")
    else:
        for line_no in range(j1, j2):

            print(
                f"{line_no + 1:>3}: "
                f"{right_617c2['lines'][line_no]}"
            )


print(
    "\nTotal changed blocks:",
    change_number_617c2
)

print(
    "\nStep 6.17C-2 completed."
)

print(
    "No dropout function was executed or modified."
)

In [ ]:
# ============================================================
# Step 6.17C-3 — Compact all dropout differences
# ============================================================

import difflib


print(
    "=== Step 6.17C-3 — All dropout differences, compact ==="
)


# We reuse defs_617c2 produced by Step 6.17C-2.
assert "defs_617c2" in globals(), (
    "defs_617c2 is missing. "
    "Please rerun Step 6.17C-2 first."
)

assert len(defs_617c2) == 2, (
    "Expected exactly two dropout definitions."
)


left = defs_617c2[0]
right = defs_617c2[1]


matcher = difflib.SequenceMatcher(
    None,
    left["lines"],
    right["lines"]
)


change_no = 0

for tag, i1, i2, j1, j2 in matcher.get_opcodes():

    if tag == "equal":
        continue

    change_no += 1

    left_text = "  |  ".join(
        line.strip()
        for line in left["lines"][i1:i2]
        if line.strip()
    )

    right_text = "  |  ".join(
        line.strip()
        for line in right["lines"][j1:j2]
        if line.strip()
    )

    if not left_text:
        left_text = "<nothing>"

    if not right_text:
        right_text = "<nothing>"

    print("\n" + "=" * 70)

    print(
        f"CHANGE {change_no} — {tag.upper()}"
    )

    print(
        f"Cell {left['cell']}: {left_text}"
    )

    print(
        f"Cell {right['cell']}: {right_text}"
    )


print("\n" + "=" * 70)

print(
    "Total changed blocks:",
    change_no
)

assert change_no == 4, (
    f"Expected 4 changed blocks, found {change_no}."
)

print(
    "\nStep 6.17C-3 completed successfully."
)

print(
    "No dropout function was executed."
)

In [ ]:
# ============================================================
# Step 6.17D — Verify one frozen dropout scenario
# ============================================================

from pathlib import Path
import json
import ast

import numpy as np
import pandas as pd


print(
    "=== Step 6.17D — Verify one frozen dropout scenario ==="
)


# ------------------------------------------------------------
# 1. Check required objects already established in Step 6
# ------------------------------------------------------------

required_objects_617d = [
    "val_scaled",
    "scaler",
    "PRIMARY_INPUT_COLUMNS",
    "RETAINED_SENSOR_COLUMNS",
    "sensor_reference_stats",
]

missing_objects_617d = [
    name
    for name in required_objects_617d
    if name not in globals()
]

assert not missing_objects_617d, (
    "Missing required objects: "
    f"{missing_objects_617d}"
)

print("Required in-memory objects are available.")


# ------------------------------------------------------------
# 2. Locate project root and frozen Step-4 files
# ------------------------------------------------------------

cwd_617d = Path.cwd()

if (cwd_617d / "03_notebooks").exists():
    PROJECT_ROOT_617D = cwd_617d

elif (cwd_617d.parent / "03_notebooks").exists():
    PROJECT_ROOT_617D = cwd_617d.parent

else:
    raise FileNotFoundError(
        "Could not locate project root."
    )


STEP4_NOTEBOOK_617D = (
    PROJECT_ROOT_617D
    / "03_notebooks"
    / "04_sensor_degradation_fd001.ipynb"
)

assert STEP4_NOTEBOOK_617D.exists(), (
    f"Step-4 notebook not found: "
    f"{STEP4_NOTEBOOK_617D}"
)


dropout_dir_617d = (
    PROJECT_ROOT_617D
    / "05_experiments"
    / "sensor_degradation"
)

dropout_files_617d = list(
    dropout_dir_617d.glob(
        "*dropout*validation*.csv"
    )
)

assert len(dropout_files_617d) == 1, (
    "Expected exactly one frozen dropout-validation CSV, "
    f"but found {len(dropout_files_617d)}:\n"
    + "\n".join(
        str(path)
        for path in dropout_files_617d
    )
)

DROPOUT_VALIDATION_PATH_617D = (
    dropout_files_617d[0]
)

print(
    "Frozen dropout artifact:",
    DROPOUT_VALIDATION_PATH_617D
)


# ------------------------------------------------------------
# 3. Load frozen validation artifact
# ------------------------------------------------------------

dropout_validation_617d = pd.read_csv(
    DROPOUT_VALIDATION_PATH_617D
)

expected_columns_617d = {
    "sensor",
    "severity",
    "requested_fraction",
    "expected_count",
    "actual_count",
    "actual_fraction",
    "imputation_value",
}

assert expected_columns_617d.issubset(
    dropout_validation_617d.columns
), (
    "Frozen dropout artifact does not contain "
    "the expected columns."
)

assert len(dropout_validation_617d) == 45

print(
    "Frozen dropout scenarios:",
    len(dropout_validation_617d)
)


# ------------------------------------------------------------
# 4. Extract the corrected dropout implementation
#
#    We do NOT simply assume "latest = correct".
#    We require the implementation to contain the verified
#    nested-mask design:
#
#       rng.permutation(...)
#       random_order[:n_drop]
# ------------------------------------------------------------

with open(
    STEP4_NOTEBOOK_617D,
    "r",
    encoding="utf-8"
) as f:
    notebook_617d = json.load(f)


candidate_sources_617d = []

for cell_index_617d, cell_617d in enumerate(
    notebook_617d.get("cells", [])
):

    if cell_617d.get("cell_type") != "code":
        continue

    source_617d = "".join(
        cell_617d.get("source", [])
    )

    if "def add_sensor_dropout" not in source_617d:
        continue

    try:
        tree_617d = ast.parse(source_617d)
    except SyntaxError:
        continue

    source_lines_617d = source_617d.splitlines()

    for node_617d in tree_617d.body:

        if (
            isinstance(node_617d, ast.FunctionDef)
            and node_617d.name
            == "add_sensor_dropout"
        ):

            function_source_617d = "\n".join(
                source_lines_617d[
                    node_617d.lineno - 1:
                    node_617d.end_lineno
                ]
            )

            candidate_sources_617d.append(
                {
                    "cell_index": cell_index_617d,
                    "source": function_source_617d,
                }
            )


corrected_candidates_617d = [
    item
    for item in candidate_sources_617d
    if (
        "rng.permutation" in item["source"]
        and "random_order" in item["source"]
        and "[:n_drop]" in item["source"]
    )
]

assert len(corrected_candidates_617d) == 1, (
    "Could not uniquely identify the corrected "
    "nested-mask dropout implementation."
)

corrected_item_617d = (
    corrected_candidates_617d[0]
)

print(
    "Corrected dropout definition cell:",
    corrected_item_617d["cell_index"]
)


# ------------------------------------------------------------
# 5. Load corrected function ONLY into temporary namespace
# ------------------------------------------------------------

temporary_namespace_617d = {
    "np": np,
    "pd": pd,
}

exec(
    corrected_item_617d["source"],
    temporary_namespace_617d,
)

candidate_add_sensor_dropout_617d = (
    temporary_namespace_617d[
        "add_sensor_dropout"
    ]
)

print(
    "Corrected dropout function loaded "
    "into temporary namespace only."
)


# ------------------------------------------------------------
# 6. Reconstruct original-unit validation inputs
# ------------------------------------------------------------

val_raw_617d = val_scaled.copy()

scaled_input_617d = (
    val_scaled[
        PRIMARY_INPUT_COLUMNS
    ].copy()
)

raw_input_array_617d = (
    scaler.inverse_transform(
        scaled_input_617d
    )
)

val_raw_617d.loc[
    :,
    PRIMARY_INPUT_COLUMNS
] = raw_input_array_617d


# ------------------------------------------------------------
# 7. Verify round trip
# ------------------------------------------------------------

round_trip_scaled_617d = (
    scaler.transform(
        val_raw_617d[
            PRIMARY_INPUT_COLUMNS
        ]
    )
)

round_trip_difference_617d = float(
    np.max(
        np.abs(
            round_trip_scaled_617d
            - val_scaled[
                PRIMARY_INPUT_COLUMNS
            ].to_numpy(dtype=float)
        )
    )
)

print(
    "\nMaximum scaling round-trip difference:",
    round_trip_difference_617d
)

assert round_trip_difference_617d < 1e-10

print(
    "Raw validation inputs reconstructed successfully."
)


# ------------------------------------------------------------
# 8. Select ONE frozen scenario only
# ------------------------------------------------------------

frozen_row_617d = (
    dropout_validation_617d
    .iloc[0]
)

sensor_617d = str(
    frozen_row_617d["sensor"]
)

severity_617d = str(
    frozen_row_617d["severity"]
)

dropout_rate_617d = float(
    frozen_row_617d["requested_fraction"]
)

expected_count_617d = int(
    frozen_row_617d["expected_count"]
)

frozen_actual_count_617d = int(
    frozen_row_617d["actual_count"]
)

frozen_actual_fraction_617d = float(
    frozen_row_617d["actual_fraction"]
)

frozen_impute_value_617d = float(
    frozen_row_617d["imputation_value"]
)


assert sensor_617d in RETAINED_SENSOR_COLUMNS


print("\nFrozen scenario:")
print(
    "Sensor             :",
    sensor_617d
)
print(
    "Severity           :",
    severity_617d
)
print(
    "Requested fraction :",
    dropout_rate_617d
)
print(
    "Expected count     :",
    expected_count_617d
)
print(
    "Frozen actual count:",
    frozen_actual_count_617d
)
print(
    "Frozen imputation  :",
    frozen_impute_value_617d
)


# ------------------------------------------------------------
# 9. Execute corrected function for ONE scenario
# ------------------------------------------------------------

result_617d = (
    candidate_add_sensor_dropout_617d(
        df=val_raw_617d,
        sensor=sensor_617d,
        dropout_rate=dropout_rate_617d,
        sensor_stats=sensor_reference_stats,
        seed=42,
    )
)


# ------------------------------------------------------------
# 10. Verify return contract
# ------------------------------------------------------------

assert isinstance(result_617d, tuple)

assert len(result_617d) == 4, (
    "Expected exactly four returned objects."
)

(
    missing_df_617d,
    imputed_df_617d,
    drop_mask_617d,
    impute_value_617d,
) = result_617d


assert isinstance(
    missing_df_617d,
    pd.DataFrame
)

assert isinstance(
    imputed_df_617d,
    pd.DataFrame
)


drop_mask_617d = np.asarray(
    drop_mask_617d,
    dtype=bool
)

assert drop_mask_617d.shape == (
    len(val_raw_617d),
)


print("\nReturn structure verified:")
print(
    "missing_df  :",
    missing_df_617d.shape
)
print(
    "imputed_df  :",
    imputed_df_617d.shape
)
print(
    "drop_mask   :",
    drop_mask_617d.shape
)
print(
    "impute_value:",
    float(impute_value_617d)
)


# ------------------------------------------------------------
# 11. Verify exact dropout count
# ------------------------------------------------------------

actual_count_617d = int(
    drop_mask_617d.sum()
)

actual_fraction_617d = (
    actual_count_617d
    / len(drop_mask_617d)
)


print("\nObserved dropout:")
print(
    "Mask count      :",
    actual_count_617d
)
print(
    "Actual fraction :",
    actual_fraction_617d
)


assert (
    actual_count_617d
    == expected_count_617d
)

assert (
    actual_count_617d
    == frozen_actual_count_617d
)

assert np.isclose(
    actual_fraction_617d,
    frozen_actual_fraction_617d,
    rtol=0.0,
    atol=1e-12,
)


# ------------------------------------------------------------
# 12. Verify imputation value against frozen artifact
# ------------------------------------------------------------

assert np.isclose(
    float(impute_value_617d),
    frozen_impute_value_617d,
    rtol=0.0,
    atol=1e-10,
), (
    "Returned imputation value does not match "
    "the frozen Step-4 artifact."
)


# ------------------------------------------------------------
# 13. Verify non-target columns are unchanged
# ------------------------------------------------------------

non_target_columns_617d = [
    column
    for column in val_raw_617d.columns
    if column != sensor_617d
]


pd.testing.assert_frame_equal(
    missing_df_617d[
        non_target_columns_617d
    ].reset_index(drop=True),
    val_raw_617d[
        non_target_columns_617d
    ].reset_index(drop=True),
    check_dtype=False,
)

pd.testing.assert_frame_equal(
    imputed_df_617d[
        non_target_columns_617d
    ].reset_index(drop=True),
    val_raw_617d[
        non_target_columns_617d
    ].reset_index(drop=True),
    check_dtype=False,
)

print(
    "Non-target columns remain unchanged."
)


# ------------------------------------------------------------
# 14. Verify missing-data representation
# ------------------------------------------------------------

missing_sensor_617d = (
    missing_df_617d[
        sensor_617d
    ].to_numpy()
)

original_sensor_617d = (
    val_raw_617d[
        sensor_617d
    ].to_numpy()
)


assert np.all(
    pd.isna(
        missing_sensor_617d[
            drop_mask_617d
        ]
    )
)

assert np.allclose(
    missing_sensor_617d[
        ~drop_mask_617d
    ],
    original_sensor_617d[
        ~drop_mask_617d
    ],
    equal_nan=True,
)

print(
    "Missing-data representation verified."
)


# ------------------------------------------------------------
# 15. Verify imputed representation
# ------------------------------------------------------------

imputed_sensor_617d = (
    imputed_df_617d[
        sensor_617d
    ].to_numpy(dtype=float)
)

assert np.allclose(
    imputed_sensor_617d[
        drop_mask_617d
    ],
    float(impute_value_617d),
    rtol=0.0,
    atol=1e-10,
)

assert np.allclose(
    imputed_sensor_617d[
        ~drop_mask_617d
    ],
    original_sensor_617d[
        ~drop_mask_617d
    ],
    equal_nan=True,
)

print(
    "Imputed representation verified."
)


# ------------------------------------------------------------
# 16. Determinism check
# ------------------------------------------------------------

repeat_result_617d = (
    candidate_add_sensor_dropout_617d(
        df=val_raw_617d,
        sensor=sensor_617d,
        dropout_rate=dropout_rate_617d,
        sensor_stats=sensor_reference_stats,
        seed=42,
    )
)

repeat_mask_617d = np.asarray(
    repeat_result_617d[2],
    dtype=bool
)

assert np.array_equal(
    drop_mask_617d,
    repeat_mask_617d
)

print(
    "Same-seed determinism verified."
)


# ------------------------------------------------------------
# 17. Final result
# ------------------------------------------------------------

print(
    "\nFrozen-artifact comparison:"
)

print(
    "Expected count       :",
    expected_count_617d
)

print(
    "Returned count       :",
    actual_count_617d
)

print(
    "Frozen impute value  :",
    frozen_impute_value_617d
)

print(
    "Returned impute value:",
    float(impute_value_617d)
)


print(
    "\nStep 6.17D completed successfully."
)

print(
    "The corrected nested-mask dropout implementation "
    "reproduces the selected frozen Step-4 scenario."
)

In [ ]:
# ============================================================
# Step 6.17D — Verify one frozen dropout scenario
# CORRECTED VERSION
#
# Important:
# - uses AST structure, not fragile text matching
# - executes only ONE frozen dropout scenario
# - modifies no Step-4 artifact
# ============================================================

from pathlib import Path
import json
import ast

import numpy as np
import pandas as pd


print(
    "=== Step 6.17D — Verify one frozen dropout scenario ==="
)


# ------------------------------------------------------------
# 1. Verify required Step-6 objects
# ------------------------------------------------------------

required_objects_617d = [
    "val_scaled",
    "scaler",
    "PRIMARY_INPUT_COLUMNS",
    "RETAINED_SENSOR_COLUMNS",
    "sensor_reference_stats",
]

missing_objects_617d = [
    name
    for name in required_objects_617d
    if name not in globals()
]

assert not missing_objects_617d, (
    "Missing required objects: "
    f"{missing_objects_617d}"
)

print("Required in-memory objects are available.")


# ------------------------------------------------------------
# 2. Locate frozen project artifacts
# ------------------------------------------------------------

cwd_617d = Path.cwd()

if (cwd_617d / "03_notebooks").exists():
    PROJECT_ROOT_617D = cwd_617d

elif (cwd_617d.parent / "03_notebooks").exists():
    PROJECT_ROOT_617D = cwd_617d.parent

else:
    raise FileNotFoundError(
        "Could not locate project root."
    )


STEP4_NOTEBOOK_617D = (
    PROJECT_ROOT_617D
    / "03_notebooks"
    / "04_sensor_degradation_fd001.ipynb"
)

assert STEP4_NOTEBOOK_617D.exists(), (
    f"Step-4 notebook not found: "
    f"{STEP4_NOTEBOOK_617D}"
)


dropout_validation_candidates_617d = list(
    (
        PROJECT_ROOT_617D
        / "05_experiments"
        / "sensor_degradation"
    ).glob("*dropout*validation*.csv")
)

assert len(
    dropout_validation_candidates_617d
) == 1, (
    "Expected exactly one frozen dropout-validation CSV, "
    f"found {len(dropout_validation_candidates_617d)}."
)

DROPOUT_VALIDATION_PATH_617D = (
    dropout_validation_candidates_617d[0]
)

print(
    "Frozen dropout artifact:",
    DROPOUT_VALIDATION_PATH_617D
)


# ------------------------------------------------------------
# 3. Load frozen Step-4 dropout validation table
# ------------------------------------------------------------

dropout_validation_617d = pd.read_csv(
    DROPOUT_VALIDATION_PATH_617D
)

required_columns_617d = {
    "sensor",
    "severity",
    "requested_fraction",
    "expected_count",
    "actual_count",
    "actual_fraction",
    "imputation_value",
}

assert required_columns_617d.issubset(
    dropout_validation_617d.columns
)

assert len(dropout_validation_617d) == 45


print(
    "Frozen dropout scenarios:",
    len(dropout_validation_617d)
)


# ------------------------------------------------------------
# 4. Extract both frozen add_sensor_dropout definitions
# ------------------------------------------------------------

with open(
    STEP4_NOTEBOOK_617D,
    "r",
    encoding="utf-8"
) as f:
    notebook_617d = json.load(f)


candidate_sources_617d = []

for cell_index_617d, cell_617d in enumerate(
    notebook_617d.get("cells", [])
):

    if cell_617d.get("cell_type") != "code":
        continue

    source_617d = "".join(
        cell_617d.get("source", [])
    )

    if "def add_sensor_dropout" not in source_617d:
        continue

    try:
        tree_617d = ast.parse(source_617d)
    except SyntaxError:
        continue

    lines_617d = source_617d.splitlines()

    for node_617d in tree_617d.body:

        if (
            isinstance(node_617d, ast.FunctionDef)
            and node_617d.name == "add_sensor_dropout"
        ):

            function_source_617d = "\n".join(
                lines_617d[
                    node_617d.lineno - 1:
                    node_617d.end_lineno
                ]
            )

            candidate_sources_617d.append(
                {
                    "cell_index": cell_index_617d,
                    "source": function_source_617d,
                }
            )


assert len(candidate_sources_617d) == 2, (
    "Expected exactly two frozen dropout definitions."
)

print(
    "Frozen dropout definition cells:",
    [
        item["cell_index"]
        for item in candidate_sources_617d
    ]
)


# ------------------------------------------------------------
# 5. Structurally detect the corrected nested-mask version
#
# Required logic:
#
# random_order = rng.permutation(n_rows)
# drop_positions = random_order[:n_drop]
#
# This uses AST rather than text formatting.
# ------------------------------------------------------------

def is_nested_dropout_definition_617d(
    function_source
):

    tree = ast.parse(function_source)

    has_permutation = False
    has_prefix_slice = False

    for node in ast.walk(tree):

        # Detect rng.permutation(...)
        if isinstance(node, ast.Call):

            if (
                isinstance(node.func, ast.Attribute)
                and node.func.attr == "permutation"
            ):
                has_permutation = True

        # Detect:
        # drop_positions = random_order[:n_drop]
        if isinstance(node, ast.Assign):

            target_names = [
                target.id
                for target in node.targets
                if isinstance(target, ast.Name)
            ]

            if "drop_positions" not in target_names:
                continue

            value = node.value

            if not isinstance(value, ast.Subscript):
                continue

            if not (
                isinstance(value.value, ast.Name)
                and value.value.id == "random_order"
            ):
                continue

            slice_node = value.slice

            if isinstance(slice_node, ast.Slice):

                lower_is_none = (
                    slice_node.lower is None
                )

                upper_is_n_drop = (
                    isinstance(
                        slice_node.upper,
                        ast.Name
                    )
                    and
                    slice_node.upper.id == "n_drop"
                )

                step_is_none = (
                    slice_node.step is None
                )

                if (
                    lower_is_none
                    and upper_is_n_drop
                    and step_is_none
                ):
                    has_prefix_slice = True

    return (
        has_permutation
        and has_prefix_slice
    )


corrected_candidates_617d = [
    item
    for item in candidate_sources_617d
    if is_nested_dropout_definition_617d(
        item["source"]
    )
]


print(
    "Nested-mask candidates found:",
    len(corrected_candidates_617d)
)

assert len(corrected_candidates_617d) == 1, (
    "Expected exactly one structurally verified "
    "nested-mask dropout definition."
)


corrected_item_617d = (
    corrected_candidates_617d[0]
)


print(
    "Corrected dropout definition cell:",
    corrected_item_617d["cell_index"]
)


# Based on Step 6.17C-3 this must be cell 43.
assert (
    corrected_item_617d["cell_index"] == 43
), (
    "Unexpected corrected dropout cell. "
    "Previously verified Step-4 cell 43 was expected."
)

print(
    "Corrected Step-4 cell 43 identified structurally."
)


# ------------------------------------------------------------
# 6. Load corrected function into TEMPORARY namespace only
# ------------------------------------------------------------

temporary_namespace_617d = {
    "np": np,
    "pd": pd,
}

exec(
    corrected_item_617d["source"],
    temporary_namespace_617d
)

candidate_add_sensor_dropout_617d = (
    temporary_namespace_617d[
        "add_sensor_dropout"
    ]
)

print(
    "Corrected function loaded into temporary "
    "namespace only."
)


# ------------------------------------------------------------
# 7. Reconstruct validation inputs in original sensor units
# ------------------------------------------------------------

val_raw_617d = val_scaled.copy()

scaled_inputs_617d = (
    val_scaled[
        PRIMARY_INPUT_COLUMNS
    ].copy()
)

raw_inputs_617d = scaler.inverse_transform(
    scaled_inputs_617d
)

val_raw_617d.loc[
    :,
    PRIMARY_INPUT_COLUMNS
] = raw_inputs_617d


# ------------------------------------------------------------
# 8. Verify inverse-scaling round trip
# ------------------------------------------------------------

round_trip_617d = scaler.transform(
    val_raw_617d[
        PRIMARY_INPUT_COLUMNS
    ]
)

round_trip_difference_617d = float(
    np.max(
        np.abs(
            round_trip_617d
            -
            val_scaled[
                PRIMARY_INPUT_COLUMNS
            ].to_numpy(dtype=float)
        )
    )
)

print(
    "\nMaximum scaling round-trip difference:",
    round_trip_difference_617d
)

assert round_trip_difference_617d < 1e-10

print(
    "Raw validation inputs reconstructed successfully."
)


# ------------------------------------------------------------
# 9. Select exactly ONE frozen scenario
# ------------------------------------------------------------

frozen_row_617d = (
    dropout_validation_617d
    .iloc[0]
)

sensor_617d = str(
    frozen_row_617d["sensor"]
)

severity_617d = str(
    frozen_row_617d["severity"]
)

dropout_rate_617d = float(
    frozen_row_617d["requested_fraction"]
)

expected_count_617d = int(
    frozen_row_617d["expected_count"]
)

frozen_actual_count_617d = int(
    frozen_row_617d["actual_count"]
)

frozen_actual_fraction_617d = float(
    frozen_row_617d["actual_fraction"]
)

frozen_imputation_value_617d = float(
    frozen_row_617d["imputation_value"]
)


assert sensor_617d in RETAINED_SENSOR_COLUMNS


print("\nFrozen scenario:")
print("Sensor              :", sensor_617d)
print("Severity            :", severity_617d)
print("Requested fraction  :", dropout_rate_617d)
print("Expected count      :", expected_count_617d)
print("Frozen actual count :", frozen_actual_count_617d)
print("Frozen actual frac. :", frozen_actual_fraction_617d)
print("Frozen imputation   :", frozen_imputation_value_617d)


# ------------------------------------------------------------
# 10. Execute ONE frozen scenario
# ------------------------------------------------------------

result_617d = (
    candidate_add_sensor_dropout_617d(
        df=val_raw_617d,
        sensor=sensor_617d,
        dropout_rate=dropout_rate_617d,
        sensor_stats=sensor_reference_stats,
        seed=42,
    )
)


# ------------------------------------------------------------
# 11. Verify exact return structure
# ------------------------------------------------------------

assert isinstance(result_617d, tuple)

assert len(result_617d) == 4, (
    "Frozen dropout function must return four objects."
)

(
    missing_df_617d,
    imputed_df_617d,
    drop_mask_617d,
    impute_value_617d,
) = result_617d


assert isinstance(
    missing_df_617d,
    pd.DataFrame
)

assert isinstance(
    imputed_df_617d,
    pd.DataFrame
)


drop_mask_617d = np.asarray(
    drop_mask_617d,
    dtype=bool
)


assert drop_mask_617d.shape == (
    len(val_raw_617d),
)


print("\nReturn structure:")
print("missing_df   :", missing_df_617d.shape)
print("imputed_df   :", imputed_df_617d.shape)
print("drop_mask    :", drop_mask_617d.shape)
print("impute_value :", float(impute_value_617d))


# ------------------------------------------------------------
# 12. Verify exact number of dropped rows
# ------------------------------------------------------------

actual_count_617d = int(
    drop_mask_617d.sum()
)

actual_fraction_617d = float(
    actual_count_617d
    / len(drop_mask_617d)
)


print("\nObserved dropout:")
print("Actual count    :", actual_count_617d)
print("Actual fraction :", actual_fraction_617d)


assert (
    actual_count_617d
    == expected_count_617d
)

assert (
    actual_count_617d
    == frozen_actual_count_617d
)

assert np.isclose(
    actual_fraction_617d,
    frozen_actual_fraction_617d,
    rtol=0.0,
    atol=1e-12,
)


# ------------------------------------------------------------
# 13. Verify frozen imputation value
# ------------------------------------------------------------

returned_imputation_617d = float(
    impute_value_617d
)

assert np.isclose(
    returned_imputation_617d,
    frozen_imputation_value_617d,
    rtol=0.0,
    atol=1e-10,
), (
    "Returned imputation value does not match "
    "the frozen Step-4 artifact."
)

print(
    "Frozen imputation value reproduced exactly."
)


# ------------------------------------------------------------
# 14. Verify no non-target column changed
# ------------------------------------------------------------

non_target_columns_617d = [
    column
    for column in val_raw_617d.columns
    if column != sensor_617d
]


pd.testing.assert_frame_equal(
    missing_df_617d[
        non_target_columns_617d
    ].reset_index(drop=True),
    val_raw_617d[
        non_target_columns_617d
    ].reset_index(drop=True),
    check_dtype=False,
)


pd.testing.assert_frame_equal(
    imputed_df_617d[
        non_target_columns_617d
    ].reset_index(drop=True),
    val_raw_617d[
        non_target_columns_617d
    ].reset_index(drop=True),
    check_dtype=False,
)


print(
    "Non-target columns remain unchanged."
)


# ------------------------------------------------------------
# 15. Verify missing representation
# ------------------------------------------------------------

original_sensor_617d = (
    val_raw_617d[
        sensor_617d
    ].to_numpy(dtype=float)
)

missing_sensor_617d = (
    missing_df_617d[
        sensor_617d
    ].to_numpy()
)


assert pd.isna(
    missing_sensor_617d[
        drop_mask_617d
    ]
).all()


assert np.allclose(
    missing_sensor_617d[
        ~drop_mask_617d
    ].astype(float),
    original_sensor_617d[
        ~drop_mask_617d
    ],
    rtol=0.0,
    atol=0.0,
    equal_nan=True,
)


print(
    "Missing-data representation verified."
)


# ------------------------------------------------------------
# 16. Verify imputed representation
# ------------------------------------------------------------

imputed_sensor_617d = (
    imputed_df_617d[
        sensor_617d
    ].to_numpy(dtype=float)
)


assert np.allclose(
    imputed_sensor_617d[
        drop_mask_617d
    ],
    returned_imputation_617d,
    rtol=0.0,
    atol=1e-10,
)


assert np.allclose(
    imputed_sensor_617d[
        ~drop_mask_617d
    ],
    original_sensor_617d[
        ~drop_mask_617d
    ],
    rtol=0.0,
    atol=0.0,
    equal_nan=True,
)


print(
    "Imputed-data representation verified."
)


# ------------------------------------------------------------
# 17. Same-seed determinism
# ------------------------------------------------------------

repeat_result_617d = (
    candidate_add_sensor_dropout_617d(
        df=val_raw_617d,
        sensor=sensor_617d,
        dropout_rate=dropout_rate_617d,
        sensor_stats=sensor_reference_stats,
        seed=42,
    )
)


repeat_mask_617d = np.asarray(
    repeat_result_617d[2],
    dtype=bool
)


assert np.array_equal(
    drop_mask_617d,
    repeat_mask_617d
)


print(
    "Same-seed determinism verified."
)


# ------------------------------------------------------------
# 18. Final frozen-artifact comparison
# ------------------------------------------------------------

print("\n=== Frozen-artifact comparison ===")

print(
    "Expected dropped count :",
    expected_count_617d
)

print(
    "Returned dropped count :",
    actual_count_617d
)

print(
    "Frozen impute value    :",
    frozen_imputation_value_617d
)

print(
    "Returned impute value  :",
    returned_imputation_617d
)


print(
    "\nStep 6.17D completed successfully."
)

print(
    "The corrected nested-mask implementation "
    "reproduces the selected frozen Step-4 "
    "dropout scenario."
)

In [ ]:
# ============================================================
# Step 6.17E — Verify ALL frozen dropout scenarios
#              and nested severity masks
# ============================================================

import numpy as np
import pandas as pd


print(
    "=== Step 6.17E — Verify all frozen dropout scenarios "
    "and nested masks ==="
)


# ------------------------------------------------------------
# 1. Required objects from successful Step 6.17D
# ------------------------------------------------------------

required_objects_617e = [
    "candidate_add_sensor_dropout_617d",
    "dropout_validation_617d",
    "val_raw_617d",
    "sensor_reference_stats",
    "RETAINED_SENSOR_COLUMNS",
]

missing_617e = [
    name
    for name in required_objects_617e
    if name not in globals()
]

assert not missing_617e, (
    "Missing required objects: "
    f"{missing_617e}. "
    "Please rerun the successful Step 6.17D first."
)

print("Required Step-6.17D objects are available.")


# ------------------------------------------------------------
# 2. Basic frozen-artifact checks
# ------------------------------------------------------------

assert len(dropout_validation_617d) == 45

assert (
    dropout_validation_617d["sensor"].nunique()
    == 15
)

assert (
    dropout_validation_617d["severity"].nunique()
    == 3
)

expected_severities_617e = {
    "low",
    "medium",
    "high",
}

assert set(
    dropout_validation_617d["severity"].astype(str)
) == expected_severities_617e


print(
    "Frozen scenarios :",
    len(dropout_validation_617d)
)

print(
    "Unique sensors   :",
    dropout_validation_617d["sensor"].nunique()
)

print(
    "Severities       :",
    sorted(
        dropout_validation_617d[
            "severity"
        ].unique()
    )
)


# ------------------------------------------------------------
# 3. Reproduce every frozen scenario
# ------------------------------------------------------------

verification_records_617e = []
masks_by_sensor_617e = {}


for sensor_617e in RETAINED_SENSOR_COLUMNS:

    sensor_rows_617e = (
        dropout_validation_617d[
            dropout_validation_617d["sensor"]
            == sensor_617e
        ]
        .copy()
    )

    assert len(sensor_rows_617e) == 3, (
        f"{sensor_617e} does not have exactly "
        "three frozen dropout severities."
    )

    masks_by_sensor_617e[
        sensor_617e
    ] = {}

    for severity_617e in [
        "low",
        "medium",
        "high",
    ]:

        row_match_617e = (
            sensor_rows_617e[
                sensor_rows_617e["severity"]
                == severity_617e
            ]
        )

        assert len(row_match_617e) == 1

        row_617e = row_match_617e.iloc[0]

        dropout_rate_617e = float(
            row_617e[
                "requested_fraction"
            ]
        )

        frozen_expected_count_617e = int(
            row_617e[
                "expected_count"
            ]
        )

        frozen_actual_count_617e = int(
            row_617e[
                "actual_count"
            ]
        )

        frozen_actual_fraction_617e = float(
            row_617e[
                "actual_fraction"
            ]
        )

        frozen_imputation_617e = float(
            row_617e[
                "imputation_value"
            ]
        )


        # ----------------------------------------------------
        # Execute exact corrected Step-4 helper
        # ----------------------------------------------------

        result_617e = (
            candidate_add_sensor_dropout_617d(
                df=val_raw_617d,
                sensor=sensor_617e,
                dropout_rate=dropout_rate_617e,
                sensor_stats=sensor_reference_stats,
                seed=42,
            )
        )


        assert isinstance(
            result_617e,
            tuple
        )

        assert len(result_617e) == 4


        (
            missing_df_617e,
            imputed_df_617e,
            drop_mask_617e,
            returned_imputation_617e,
        ) = result_617e


        drop_mask_617e = np.asarray(
            drop_mask_617e,
            dtype=bool
        )

        assert drop_mask_617e.shape == (
            len(val_raw_617d),
        )


        # ----------------------------------------------------
        # Verify frozen count/fraction
        # ----------------------------------------------------

        returned_count_617e = int(
            drop_mask_617e.sum()
        )

        returned_fraction_617e = float(
            returned_count_617e
            / len(drop_mask_617e)
        )


        assert (
            returned_count_617e
            == frozen_expected_count_617e
        )

        assert (
            returned_count_617e
            == frozen_actual_count_617e
        )

        assert np.isclose(
            returned_fraction_617e,
            frozen_actual_fraction_617e,
            rtol=0.0,
            atol=1e-12,
        )


        # ----------------------------------------------------
        # Verify frozen imputation value
        # ----------------------------------------------------

        returned_imputation_617e = float(
            returned_imputation_617e
        )

        assert np.isclose(
            returned_imputation_617e,
            frozen_imputation_617e,
            rtol=0.0,
            atol=1e-10,
        )


        # ----------------------------------------------------
        # Verify missing representation
        # ----------------------------------------------------

        missing_values_617e = (
            missing_df_617e[
                sensor_617e
            ].to_numpy()
        )

        assert pd.isna(
            missing_values_617e[
                drop_mask_617e
            ]
        ).all()


        # ----------------------------------------------------
        # Verify imputed representation
        # ----------------------------------------------------

        imputed_values_617e = (
            imputed_df_617e[
                sensor_617e
            ].to_numpy(dtype=float)
        )

        assert np.allclose(
            imputed_values_617e[
                drop_mask_617e
            ],
            returned_imputation_617e,
            rtol=0.0,
            atol=1e-10,
        )


        # Save mask for nesting test
        masks_by_sensor_617e[
            sensor_617e
        ][
            severity_617e
        ] = drop_mask_617e.copy()


        verification_records_617e.append(
            {
                "sensor": sensor_617e,
                "severity": severity_617e,
                "requested_fraction":
                    dropout_rate_617e,
                "frozen_count":
                    frozen_actual_count_617e,
                "returned_count":
                    returned_count_617e,
                "frozen_imputation":
                    frozen_imputation_617e,
                "returned_imputation":
                    returned_imputation_617e,
                "scenario_passed": True,
            }
        )


# ------------------------------------------------------------
# 4. Verify LOW ⊂ MEDIUM ⊂ HIGH for every sensor
# ------------------------------------------------------------

nesting_records_617e = []


for sensor_617e in RETAINED_SENSOR_COLUMNS:

    low_mask_617e = (
        masks_by_sensor_617e[
            sensor_617e
        ]["low"]
    )

    medium_mask_617e = (
        masks_by_sensor_617e[
            sensor_617e
        ]["medium"]
    )

    high_mask_617e = (
        masks_by_sensor_617e[
            sensor_617e
        ]["high"]
    )


    # Every True in low must also be True in medium.
    low_in_medium_617e = bool(
        np.all(
            ~low_mask_617e
            | medium_mask_617e
        )
    )


    # Every True in medium must also be True in high.
    medium_in_high_617e = bool(
        np.all(
            ~medium_mask_617e
            | high_mask_617e
        )
    )


    # Counts should strictly increase here because
    # frozen rates are 0.10, 0.25 and 0.50.
    low_count_617e = int(
        low_mask_617e.sum()
    )

    medium_count_617e = int(
        medium_mask_617e.sum()
    )

    high_count_617e = int(
        high_mask_617e.sum()
    )


    counts_increase_617e = (
        low_count_617e
        < medium_count_617e
        < high_count_617e
    )


    assert low_in_medium_617e, (
        f"{sensor_617e}: low mask is not "
        "contained in medium mask."
    )

    assert medium_in_high_617e, (
        f"{sensor_617e}: medium mask is not "
        "contained in high mask."
    )

    assert counts_increase_617e, (
        f"{sensor_617e}: dropout counts do not "
        "increase low < medium < high."
    )


    nesting_records_617e.append(
        {
            "sensor": sensor_617e,
            "low_count": low_count_617e,
            "medium_count":
                medium_count_617e,
            "high_count":
                high_count_617e,
            "low_subset_medium":
                low_in_medium_617e,
            "medium_subset_high":
                medium_in_high_617e,
            "counts_increase":
                counts_increase_617e,
        }
    )


# ------------------------------------------------------------
# 5. Build verification tables
# ------------------------------------------------------------

dropout_verification_df_617e = pd.DataFrame(
    verification_records_617e
)

dropout_nesting_df_617e = pd.DataFrame(
    nesting_records_617e
)


# ------------------------------------------------------------
# 6. Final integrity checks
# ------------------------------------------------------------

assert len(
    dropout_verification_df_617e
) == 45

assert (
    dropout_verification_df_617e[
        "scenario_passed"
    ].all()
)

assert len(
    dropout_nesting_df_617e
) == 15

assert (
    dropout_nesting_df_617e[
        "low_subset_medium"
    ].all()
)

assert (
    dropout_nesting_df_617e[
        "medium_subset_high"
    ].all()
)

assert (
    dropout_nesting_df_617e[
        "counts_increase"
    ].all()
)


# ------------------------------------------------------------
# 7. Display concise results
# ------------------------------------------------------------

print(
    "\nVerified frozen scenarios:",
    len(dropout_verification_df_617e)
)

print(
    "Sensors with verified nested masks:",
    len(dropout_nesting_df_617e)
)


print(
    "\n=== Nested-mask verification by sensor ==="
)

display(
    dropout_nesting_df_617e
)


print(
    "\nStep 6.17E completed successfully."
)

print(
    "All 45 frozen Step-4 dropout scenarios "
    "were reproduced."
)

print(
    "For every retained sensor:"
)

print(
    "LOW mask ⊂ MEDIUM mask ⊂ HIGH mask."
)

In [ ]:
# ============================================================
# Step 6.17F — Pre-check before full dropout PCA experiment
# ============================================================

import inspect
import numpy as np
import pandas as pd


print(
    "=== Step 6.17F — Dropout PCA experiment pre-check ==="
)


# ------------------------------------------------------------
# 1. Objects we expect from the verified Step-6 pipeline
# ------------------------------------------------------------

objects_617f = [
    "candidate_add_sensor_dropout_617d",
    "dropout_validation_617d",
    "val_raw_617d",
    "val_scaled",
    "scaler",
    "PRIMARY_INPUT_COLUMNS",
    "RETAINED_SENSOR_COLUMNS",
    "SENSOR_INPUT_INDICES",
    "create_sequences",
    "calculate_pca_window_quality",
    "sensor_pca",
    "X_val_clean",
    "y_val_clean",
    "val_seq_meta",
    "dev_pca_mean_rmse",
    "val_pca_mean_rmse",
    "val_pca_max_rmse",
]


for name_617f in objects_617f:

    available_617f = (
        name_617f in globals()
    )

    print(
        f"{name_617f:<35}: "
        f"{available_617f}"
    )


# ------------------------------------------------------------
# 2. Function signatures
# ------------------------------------------------------------

print("\n=== Function signatures ===")

for function_name_617f in [
    "candidate_add_sensor_dropout_617d",
    "create_sequences",
    "calculate_pca_window_quality",
]:

    if function_name_617f in globals():

        function_617f = globals()[
            function_name_617f
        ]

        print(
            f"{function_name_617f}: "
            f"{inspect.signature(function_617f)}"
        )


# ------------------------------------------------------------
# 3. Important shapes
# ------------------------------------------------------------

print("\n=== Important shapes ===")

for object_name_617f in [
    "val_scaled",
    "val_raw_617d",
    "X_val_clean",
    "y_val_clean",
    "val_seq_meta",
    "dev_pca_mean_rmse",
    "val_pca_mean_rmse",
    "val_pca_max_rmse",
]:

    if object_name_617f in globals():

        object_617f = globals()[
            object_name_617f
        ]

        print(
            f"{object_name_617f:<25}: "
            f"{getattr(object_617f, 'shape', None)}"
        )


# ------------------------------------------------------------
# 4. Find current target/window variables WITHOUT assuming names
# ------------------------------------------------------------

print("\n=== Target/window-related variables ===")

candidate_names_617f = []

for name_617f in sorted(globals()):

    lower_name_617f = name_617f.lower()

    if (
        "target" in lower_name_617f
        or "window" in lower_name_617f
    ):

        value_617f = globals()[name_617f]

        if isinstance(
            value_617f,
            (
                str,
                int,
                float,
                np.integer,
                np.floating,
            )
        ):

            candidate_names_617f.append(
                (
                    name_617f,
                    value_617f,
                )
            )


for name_617f, value_617f in candidate_names_617f:

    print(
        f"{name_617f:<35}: "
        f"{value_617f}"
    )


# ------------------------------------------------------------
# 5. PCA clean-reference summary
# ------------------------------------------------------------

print("\n=== Existing PCA reference values ===")

if "dev_pca_mean_rmse" in globals():

    print(
        "Development mean-RE P95:",
        float(
            np.percentile(
                dev_pca_mean_rmse,
                95
            )
        )
    )


if "val_pca_mean_rmse" in globals():

    print(
        "Clean validation mean RE:",
        float(
            np.mean(
                val_pca_mean_rmse
            )
        )
    )

    print(
        "Clean validation median RE:",
        float(
            np.median(
                val_pca_mean_rmse
            )
        )
    )


if (
    "dev_pca_mean_rmse" in globals()
    and
    "val_pca_mean_rmse" in globals()
):

    threshold_617f = float(
        np.percentile(
            dev_pca_mean_rmse,
            95
        )
    )

    clean_above_617f = float(
        100
        * np.mean(
            val_pca_mean_rmse
            > threshold_617f
        )
    )

    print(
        "Clean validation above dev-P95 (%):",
        clean_above_617f
    )


print(
    "\nStep 6.17F pre-check completed."
)

print(
    "No corruption was generated and "
    "no frozen artifact was modified."
)

In [ ]:
# ============================================================
# Step 6.17G — Full dropout PCA sensor-quality response
#
# 15 retained sensors
# × 3 frozen dropout severities
# = 45 verified scenarios
#
# IMPORTANT:
# - uses corrected frozen Step-4 dropout helper
# - uses imputed representation for PCA/model input
# - uses existing frozen PCA
# - does NOT fit PCA again
# - does NOT modify Step-4 artifacts
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


print(
    "=== Step 6.17G — Full dropout PCA response ==="
)


# ------------------------------------------------------------
# 1. Verify all required frozen objects
# ------------------------------------------------------------

required_objects_617g = [
    "candidate_add_sensor_dropout_617d",
    "dropout_validation_617d",
    "val_raw_617d",
    "val_scaled",
    "scaler",
    "sensor_reference_stats",
    "PRIMARY_INPUT_COLUMNS",
    "RETAINED_SENSOR_COLUMNS",
    "SENSOR_INPUT_INDICES",
    "create_sequences",
    "calculate_pca_window_quality",
    "sensor_pca",
    "X_val_clean",
    "y_val_clean",
    "val_seq_meta",
    "dev_pca_mean_rmse",
    "val_pca_mean_rmse",
    "val_pca_max_rmse",
]

missing_objects_617g = [
    name
    for name in required_objects_617g
    if name not in globals()
]

assert not missing_objects_617g, (
    "Missing required objects: "
    f"{missing_objects_617g}"
)

print("Required objects are available.")


# ------------------------------------------------------------
# 2. Freeze target/window from already selected GRU pipeline
# ------------------------------------------------------------

TARGET_COLUMN_617G = "RUL_cap_125"

WINDOW_SIZE_617G = int(
    X_val_clean.shape[1]
)


assert TARGET_COLUMN_617G in val_scaled.columns, (
    f"{TARGET_COLUMN_617G} is missing from validation data."
)

assert WINDOW_SIZE_617G == 30, (
    "Unexpected sequence window size. "
    f"Expected 30, found {WINDOW_SIZE_617G}."
)

assert X_val_clean.shape[2] == len(
    PRIMARY_INPUT_COLUMNS
)

assert len(SENSOR_INPUT_INDICES) == len(
    RETAINED_SENSOR_COLUMNS
)


print(
    "Target          :",
    TARGET_COLUMN_617G
)

print(
    "Window size     :",
    WINDOW_SIZE_617G
)

print(
    "Retained sensors:",
    len(RETAINED_SENSOR_COLUMNS)
)

print(
    "Frozen scenarios:",
    len(dropout_validation_617d)
)


# ------------------------------------------------------------
# 3. Locate output directory
# ------------------------------------------------------------

cwd_617g = Path.cwd()

if (cwd_617g / "06_results").exists():
    PROJECT_ROOT_617G = cwd_617g

elif (cwd_617g.parent / "06_results").exists():
    PROJECT_ROOT_617G = cwd_617g.parent

else:
    raise FileNotFoundError(
        "Could not locate project root."
    )


OUTPUT_DIR_617G = (
    PROJECT_ROOT_617G
    / "06_results"
    / "step6_reliability_signals"
)

OUTPUT_DIR_617G.mkdir(
    parents=True,
    exist_ok=True
)


print(
    "Output directory:",
    OUTPUT_DIR_617G
)


# ------------------------------------------------------------
# 4. Verify clean validation sequences
# ------------------------------------------------------------

assert X_val_clean.ndim == 3

assert X_val_clean.shape[0] == len(
    y_val_clean
)

assert X_val_clean.shape[0] == len(
    val_seq_meta
)

assert len(val_pca_mean_rmse) == len(
    y_val_clean
)

assert len(val_pca_max_rmse) == len(
    y_val_clean
)


print(
    "\nClean sequence shape:",
    X_val_clean.shape
)


# ------------------------------------------------------------
# 5. Freeze clean PCA references
# ------------------------------------------------------------

DEV_P95_617G = float(
    np.percentile(
        dev_pca_mean_rmse,
        95
    )
)

CLEAN_MEAN_RE_617G = float(
    np.mean(
        val_pca_mean_rmse
    )
)

CLEAN_MEDIAN_RE_617G = float(
    np.median(
        val_pca_mean_rmse
    )
)

CLEAN_ABOVE_DEV95_PCT_617G = float(
    100.0
    * np.mean(
        val_pca_mean_rmse
        > DEV_P95_617G
    )
)

CLEAN_MEAN_MAX_RE_617G = float(
    np.mean(
        val_pca_max_rmse
    )
)


print("\nClean PCA reference:")

print(
    "Development mean-RE P95       :",
    round(
        DEV_P95_617G,
        6
    )
)

print(
    "Clean validation mean RE      :",
    round(
        CLEAN_MEAN_RE_617G,
        6
    )
)

print(
    "Clean validation median RE    :",
    round(
        CLEAN_MEDIAN_RE_617G,
        6
    )
)

print(
    "Clean validation above P95 (%) :",
    round(
        CLEAN_ABOVE_DEV95_PCT_617G,
        3
    )
)


# ------------------------------------------------------------
# 6. Strict metadata-alignment helper
#
# We compare values rather than DataFrame dtypes because
# earlier diagnostics showed that integer/float dtype changes
# can occur even when sequence identities are identical.
# ------------------------------------------------------------

def verify_sequence_alignment_617g(
    corrupted_meta,
    clean_meta,
):

    assert list(
        corrupted_meta.columns
    ) == list(
        clean_meta.columns
    ), (
        "Sequence metadata columns differ."
    )

    assert corrupted_meta.shape == clean_meta.shape, (
        "Sequence metadata shape differs."
    )

    for column_617g in clean_meta.columns:

        clean_values_617g = (
            clean_meta[
                column_617g
            ].to_numpy()
        )

        corrupted_values_617g = (
            corrupted_meta[
                column_617g
            ].to_numpy()
        )

        if (
            np.issubdtype(
                clean_values_617g.dtype,
                np.number
            )
            and
            np.issubdtype(
                corrupted_values_617g.dtype,
                np.number
            )
        ):

            assert np.allclose(
                corrupted_values_617g.astype(float),
                clean_values_617g.astype(float),
                rtol=0.0,
                atol=1e-12,
                equal_nan=True,
            ), (
                f"Sequence metadata mismatch "
                f"in column {column_617g}."
            )

        else:

            assert np.array_equal(
                corrupted_values_617g.astype(str),
                clean_values_617g.astype(str),
            ), (
                f"Sequence metadata mismatch "
                f"in column {column_617g}."
            )


# ------------------------------------------------------------
# 7. Run all 45 frozen dropout scenarios
# ------------------------------------------------------------

dropout_pca_records_617g = []


scenario_counter_617g = 0

TOTAL_SCENARIOS_617G = len(
    dropout_validation_617d
)


for sensor_617g in RETAINED_SENSOR_COLUMNS:

    sensor_plan_617g = (
        dropout_validation_617d[
            dropout_validation_617d[
                "sensor"
            ] == sensor_617g
        ]
        .copy()
    )


    assert len(sensor_plan_617g) == 3


    for severity_617g in [
        "low",
        "medium",
        "high",
    ]:

        scenario_counter_617g += 1


        # ----------------------------------------------------
        # 7A. Read frozen scenario
        # ----------------------------------------------------

        scenario_rows_617g = (
            sensor_plan_617g[
                sensor_plan_617g[
                    "severity"
                ] == severity_617g
            ]
        )


        assert len(scenario_rows_617g) == 1


        frozen_row_617g = (
            scenario_rows_617g.iloc[0]
        )


        dropout_rate_617g = float(
            frozen_row_617g[
                "requested_fraction"
            ]
        )

        frozen_count_617g = int(
            frozen_row_617g[
                "actual_count"
            ]
        )

        frozen_fraction_617g = float(
            frozen_row_617g[
                "actual_fraction"
            ]
        )

        frozen_imputation_617g = float(
            frozen_row_617g[
                "imputation_value"
            ]
        )


        # ----------------------------------------------------
        # 7B. Generate exact verified Step-4 dropout
        # ----------------------------------------------------

        (
            missing_df_617g,
            imputed_df_617g,
            drop_mask_617g,
            returned_imputation_617g,
        ) = candidate_add_sensor_dropout_617d(
            df=val_raw_617d,
            sensor=sensor_617g,
            dropout_rate=dropout_rate_617g,
            sensor_stats=sensor_reference_stats,
            seed=42,
        )


        drop_mask_617g = np.asarray(
            drop_mask_617g,
            dtype=bool
        )


        # ----------------------------------------------------
        # 7C. Reconfirm frozen corruption integrity
        # ----------------------------------------------------

        returned_count_617g = int(
            drop_mask_617g.sum()
        )

        returned_fraction_617g = float(
            returned_count_617g
            / len(drop_mask_617g)
        )


        assert (
            returned_count_617g
            == frozen_count_617g
        )

        assert np.isclose(
            returned_fraction_617g,
            frozen_fraction_617g,
            rtol=0.0,
            atol=1e-12,
        )

        assert np.isclose(
            float(
                returned_imputation_617g
            ),
            frozen_imputation_617g,
            rtol=0.0,
            atol=1e-10,
        )


        # ----------------------------------------------------
        # 7D. PCA/model input must use IMPUTED data
        #
        # PCA cannot receive NaN values.
        # ----------------------------------------------------

        corrupted_scaled_617g = (
            imputed_df_617g.copy()
        )


        corrupted_scaled_values_617g = (
            scaler.transform(
                imputed_df_617g[
                    PRIMARY_INPUT_COLUMNS
                ]
            )
        )


        corrupted_scaled_617g.loc[
            :,
            PRIMARY_INPUT_COLUMNS
        ] = corrupted_scaled_values_617g


        # No NaN/infinite input may reach PCA.
        scaled_matrix_617g = (
            corrupted_scaled_617g[
                PRIMARY_INPUT_COLUMNS
            ].to_numpy(dtype=float)
        )


        assert np.isfinite(
            scaled_matrix_617g
        ).all(), (
            f"Non-finite scaled values for "
            f"{sensor_617g}, {severity_617g}."
        )


        # ----------------------------------------------------
        # 7E. Recreate sequences
        # ----------------------------------------------------

        (
            X_dropout_617g,
            y_dropout_617g,
            dropout_meta_617g,
        ) = create_sequences(
            df=corrupted_scaled_617g,
            feature_columns=PRIMARY_INPUT_COLUMNS,
            target_column=TARGET_COLUMN_617G,
            window_size=WINDOW_SIZE_617G,
        )


        # ----------------------------------------------------
        # 7F. Strict alignment with clean validation
        # ----------------------------------------------------

        assert X_dropout_617g.shape == (
            X_val_clean.shape
        ), (
            f"Sequence shape mismatch for "
            f"{sensor_617g}, {severity_617g}."
        )


        assert np.allclose(
            np.asarray(
                y_dropout_617g,
                dtype=float
            ),
            np.asarray(
                y_val_clean,
                dtype=float
            ),
            rtol=0.0,
            atol=1e-12,
        ), (
            f"Target alignment mismatch for "
            f"{sensor_617g}, {severity_617g}."
        )


        verify_sequence_alignment_617g(
            corrupted_meta=dropout_meta_617g,
            clean_meta=val_seq_meta,
        )


        # ----------------------------------------------------
        # 7G. Frozen PCA sensor-quality calculation
        # ----------------------------------------------------

        (
            dropout_pca_row_rmse_617g,
            dropout_pca_mean_rmse_617g,
            dropout_pca_max_rmse_617g,
        ) = calculate_pca_window_quality(
            X_dropout_617g,
            SENSOR_INPUT_INDICES,
            sensor_pca,
        )


        dropout_pca_mean_rmse_617g = (
            np.asarray(
                dropout_pca_mean_rmse_617g,
                dtype=float
            )
        )

        dropout_pca_max_rmse_617g = (
            np.asarray(
                dropout_pca_max_rmse_617g,
                dtype=float
            )
        )


        assert (
            dropout_pca_mean_rmse_617g.shape
            == val_pca_mean_rmse.shape
        )

        assert (
            dropout_pca_max_rmse_617g.shape
            == val_pca_max_rmse.shape
        )


        # ----------------------------------------------------
        # 7H. Scenario-level PCA response metrics
        # ----------------------------------------------------

        corrupted_mean_re_617g = float(
            np.mean(
                dropout_pca_mean_rmse_617g
            )
        )

        corrupted_median_re_617g = float(
            np.median(
                dropout_pca_mean_rmse_617g
            )
        )

        corrupted_above_dev95_pct_617g = float(
            100.0
            * np.mean(
                dropout_pca_mean_rmse_617g
                > DEV_P95_617G
            )
        )

        corrupted_mean_max_re_617g = float(
            np.mean(
                dropout_pca_max_rmse_617g
            )
        )


        mean_re_change_pct_617g = float(
            100.0
            * (
                corrupted_mean_re_617g
                - CLEAN_MEAN_RE_617G
            )
            / CLEAN_MEAN_RE_617G
        )


        median_re_change_pct_617g = float(
            100.0
            * (
                corrupted_median_re_617g
                - CLEAN_MEDIAN_RE_617G
            )
            / CLEAN_MEDIAN_RE_617G
        )


        fraction_increased_pct_617g = float(
            100.0
            * np.mean(
                dropout_pca_mean_rmse_617g
                > val_pca_mean_rmse
            )
        )


        above_dev95_delta_pp_617g = float(
            corrupted_above_dev95_pct_617g
            - CLEAN_ABOVE_DEV95_PCT_617G
        )


        mean_max_re_change_pct_617g = float(
            100.0
            * (
                corrupted_mean_max_re_617g
                - CLEAN_MEAN_MAX_RE_617G
            )
            / CLEAN_MEAN_MAX_RE_617G
        )


        # ----------------------------------------------------
        # 7I. Store scenario result
        # ----------------------------------------------------

        dropout_pca_records_617g.append(
            {
                "scenario_id":
                    f"FD001_DROPOUT_{scenario_counter_617g:03d}",

                "sensor":
                    sensor_617g,

                "severity":
                    severity_617g,

                "requested_fraction":
                    dropout_rate_617g,

                "dropped_count":
                    returned_count_617g,

                "actual_fraction":
                    returned_fraction_617g,

                "imputation_value":
                    float(
                        returned_imputation_617g
                    ),

                "clean_mean_re":
                    CLEAN_MEAN_RE_617G,

                "corrupted_mean_re":
                    corrupted_mean_re_617g,

                "mean_RE_change_pct":
                    mean_re_change_pct_617g,

                "clean_median_re":
                    CLEAN_MEDIAN_RE_617G,

                "corrupted_median_re":
                    corrupted_median_re_617g,

                "median_RE_change_pct":
                    median_re_change_pct_617g,

                "fraction_increased_pct":
                    fraction_increased_pct_617g,

                "clean_above_dev95_pct":
                    CLEAN_ABOVE_DEV95_PCT_617G,

                "corrupted_above_dev95_pct":
                    corrupted_above_dev95_pct_617g,

                "above_dev95_delta_pp":
                    above_dev95_delta_pp_617g,

                "clean_mean_max_re":
                    CLEAN_MEAN_MAX_RE_617G,

                "corrupted_mean_max_re":
                    corrupted_mean_max_re_617g,

                "mean_max_RE_change_pct":
                    mean_max_re_change_pct_617g,
            }
        )


        if (
            scenario_counter_617g % 15 == 0
            or
            scenario_counter_617g
            == TOTAL_SCENARIOS_617G
        ):

            print(
                f"[{scenario_counter_617g}/"
                f"{TOTAL_SCENARIOS_617G}] completed"
            )


# ------------------------------------------------------------
# 8. Build complete scenario DataFrame
# ------------------------------------------------------------

dropout_pca_response_df_617g = pd.DataFrame(
    dropout_pca_records_617g
)


assert len(
    dropout_pca_response_df_617g
) == 45

assert (
    dropout_pca_response_df_617g[
        "sensor"
    ].nunique()
    == 15
)

assert (
    dropout_pca_response_df_617g[
        "severity"
    ].nunique()
    == 3
)


print(
    "\nDropout PCA-response scenarios:",
    len(
        dropout_pca_response_df_617g
    )
)

print(
    "Unique sensors:",
    dropout_pca_response_df_617g[
        "sensor"
    ].nunique()
)

print(
    "Unique severities:",
    dropout_pca_response_df_617g[
        "severity"
    ].nunique()
)


# ------------------------------------------------------------
# 9. Severity summary
# ------------------------------------------------------------

severity_order_617g = [
    "low",
    "medium",
    "high",
]


dropout_severity_summary_617g = (
    dropout_pca_response_df_617g
    .groupby(
        "severity"
    )
    .agg(
        scenarios=(
            "scenario_id",
            "count"
        ),

        mean_RE_change_pct=(
            "mean_RE_change_pct",
            "mean"
        ),

        median_RE_change_pct=(
            "median_RE_change_pct",
            "mean"
        ),

        mean_fraction_increased_pct=(
            "fraction_increased_pct",
            "mean"
        ),

        mean_above_dev95_pct=(
            "corrupted_above_dev95_pct",
            "mean"
        ),

        mean_above_dev95_delta_pp=(
            "above_dev95_delta_pp",
            "mean"
        ),

        mean_max_RE_change_pct=(
            "mean_max_RE_change_pct",
            "mean"
        ),
    )
    .reindex(
        severity_order_617g
    )
)


print(
    "\n=== Dropout PCA response by severity ==="
)

display(
    dropout_severity_summary_617g.round(3)
)


# ------------------------------------------------------------
# 10. Sensor-wise sensitivity summary
# ------------------------------------------------------------

dropout_sensor_summary_617g = (
    dropout_pca_response_df_617g
    .groupby(
        "sensor"
    )
    .agg(
        scenarios=(
            "scenario_id",
            "count"
        ),

        mean_RE_change_pct=(
            "mean_RE_change_pct",
            "mean"
        ),

        median_RE_change_pct=(
            "mean_RE_change_pct",
            "median"
        ),

        min_RE_change_pct=(
            "mean_RE_change_pct",
            "min"
        ),

        max_RE_change_pct=(
            "mean_RE_change_pct",
            "max"
        ),

        mean_fraction_increased_pct=(
            "fraction_increased_pct",
            "mean"
        ),

        mean_above_dev95_pct=(
            "corrupted_above_dev95_pct",
            "mean"
        ),

        max_above_dev95_pct=(
            "corrupted_above_dev95_pct",
            "max"
        ),
    )
    .sort_values(
        "mean_RE_change_pct",
        ascending=False
    )
)


assert (
    dropout_sensor_summary_617g[
        "scenarios"
    ] == 3
).all()


print(
    "\n=== Sensor-wise PCA sensitivity "
    "across dropout scenarios ==="
)

display(
    dropout_sensor_summary_617g.round(3)
)


# ------------------------------------------------------------
# 11. Show strongest dropout PCA responses
# ------------------------------------------------------------

strongest_dropout_617g = (
    dropout_pca_response_df_617g
    .sort_values(
        "mean_RE_change_pct",
        ascending=False
    )
    .head(15)
)


print(
    "\n=== Strongest 15 dropout PCA responses ==="
)

display(
    strongest_dropout_617g.round(3)
)


# ------------------------------------------------------------
# 12. Save results
# ------------------------------------------------------------

response_path_617g = (
    OUTPUT_DIR_617G
    / "fd001_pca_dropout_response_45_scenarios.csv"
)

sensor_summary_path_617g = (
    OUTPUT_DIR_617G
    / "fd001_pca_dropout_sensor_summary.csv"
)

severity_summary_path_617g = (
    OUTPUT_DIR_617G
    / "fd001_pca_dropout_severity_summary.csv"
)


dropout_pca_response_df_617g.to_csv(
    response_path_617g,
    index=False
)

dropout_sensor_summary_617g.to_csv(
    sensor_summary_path_617g
)

dropout_severity_summary_617g.to_csv(
    severity_summary_path_617g
)


# ------------------------------------------------------------
# 13. Final checks
# ------------------------------------------------------------

assert response_path_617g.exists()

assert sensor_summary_path_617g.exists()

assert severity_summary_path_617g.exists()


print("\nSaved:")

print(
    response_path_617g
)

print(
    sensor_summary_path_617g
)

print(
    severity_summary_path_617g
)


print(
    "\nStep 6.17G completed successfully."
)

print(
    "All 45 verified dropout scenarios were evaluated "
    "using the frozen PCA sensor-quality signal."
)

In [ ]:
# ============================================================
# Step 6.18A — Pre-check frozen stuck-sensor implementation
# ============================================================

from pathlib import Path
import inspect
import pandas as pd


print(
    "=== Step 6.18A — Stuck-sensor pre-check ==="
)


# ------------------------------------------------------------
# 1. Current in-memory availability
# ------------------------------------------------------------

objects_618a = [
    "add_stuck_sensor",
    "apply_corruption",
    "CORRUPTION_LEVELS",
    "sensor_reference_stats",
    "sensor_temporal_stats",
    "val_scaled",
    "scaler",
    "create_sequences",
    "calculate_pca_window_quality",
    "sensor_pca",
    "X_val_clean",
    "y_val_clean",
    "val_seq_meta",
    "PRIMARY_INPUT_COLUMNS",
    "RETAINED_SENSOR_COLUMNS",
    "SENSOR_INPUT_INDICES",
]

print(
    "\n=== Current in-memory availability ==="
)

for name_618a in objects_618a:

    print(
        f"{name_618a:<32}: "
        f"{name_618a in globals()}"
    )


# ------------------------------------------------------------
# 2. Show signature only if stuck helper is available
# ------------------------------------------------------------

if "add_stuck_sensor" in globals():

    print(
        "\n=== In-memory stuck helper signature ==="
    )

    print(
        inspect.signature(
            add_stuck_sensor
        )
    )


# ------------------------------------------------------------
# 3. Locate project root
# ------------------------------------------------------------

cwd_618a = Path.cwd()

if (cwd_618a / "03_notebooks").exists():

    PROJECT_ROOT_618A = cwd_618a

elif (cwd_618a.parent / "03_notebooks").exists():

    PROJECT_ROOT_618A = cwd_618a.parent

else:

    raise FileNotFoundError(
        "Could not locate project root."
    )


STEP4_PATH_618A = (
    PROJECT_ROOT_618A
    / "03_notebooks"
    / "04_sensor_degradation_fd001.ipynb"
)


print(
    "\nProject root :",
    PROJECT_ROOT_618A
)

print(
    "Step-4 file  :",
    STEP4_PATH_618A
)

print(
    "File exists  :",
    STEP4_PATH_618A.exists()
)


assert STEP4_PATH_618A.exists()


# ------------------------------------------------------------
# 4. Locate frozen stuck-validation artifact
# ------------------------------------------------------------

stuck_candidates_618a = list(
    (
        PROJECT_ROOT_618A
        / "05_experiments"
        / "sensor_degradation"
    ).glob("*stuck*validation*.csv")
)


print(
    "\nNumber of stuck-validation files:",
    len(stuck_candidates_618a)
)


for path_618a in stuck_candidates_618a:

    print(
        " -",
        path_618a.name
    )


# ------------------------------------------------------------
# 5. Inspect frozen artifact if uniquely found
# ------------------------------------------------------------

if len(stuck_candidates_618a) == 1:

    STUCK_PATH_618A = (
        stuck_candidates_618a[0]
    )

    stuck_validation_618a = pd.read_csv(
        STUCK_PATH_618A
    )

    print(
        "\nFrozen stuck artifact:",
        STUCK_PATH_618A
    )

    print(
        "Shape:",
        stuck_validation_618a.shape
    )

    print(
        "Columns:",
        stuck_validation_618a.columns.tolist()
    )

    print(
        "\nFirst 5 rows:"
    )

    display(
        stuck_validation_618a.head()
    )


# ------------------------------------------------------------
# 6. Frozen stuck configuration
# ------------------------------------------------------------

if "CORRUPTION_LEVELS" in globals():

    if "stuck" in CORRUPTION_LEVELS:

        print(
            "\nFrozen stuck severity levels:",
            CORRUPTION_LEVELS["stuck"]
        )


print(
    "\nStep 6.18A completed."
)

print(
    "No stuck corruption was executed and "
    "no frozen artifact was modified."
)

In [ ]:
# ============================================================
# Step 6.18B — Extract frozen add_stuck_sensor definitions
#               from Step-4 notebook
# ============================================================

from pathlib import Path
import ast
import hashlib
import json


print(
    "=== Step 6.18B — Extract frozen add_stuck_sensor "
    "definitions ==="
)


# ------------------------------------------------------------
# 1. Locate Step-4 notebook safely
# ------------------------------------------------------------

cwd_618b = Path.cwd()

if (cwd_618b / "03_notebooks").exists():
    PROJECT_ROOT_618B = cwd_618b

elif (cwd_618b.parent / "03_notebooks").exists():
    PROJECT_ROOT_618B = cwd_618b.parent

else:
    raise FileNotFoundError(
        "Could not locate project root."
    )


STEP4_PATH_618B = (
    PROJECT_ROOT_618B
    / "03_notebooks"
    / "04_sensor_degradation_fd001.ipynb"
)


assert STEP4_PATH_618B.exists(), (
    f"Step-4 notebook not found: "
    f"{STEP4_PATH_618B}"
)


print(
    "Project root :",
    PROJECT_ROOT_618B
)

print(
    "Step-4 file  :",
    STEP4_PATH_618B
)


# ------------------------------------------------------------
# 2. Read notebook as JSON
# ------------------------------------------------------------

with open(
    STEP4_PATH_618B,
    "r",
    encoding="utf-8",
) as file_618b:

    notebook_618b = json.load(
        file_618b
    )


# ------------------------------------------------------------
# 3. Extract every function named add_stuck_sensor
# ------------------------------------------------------------

stuck_definitions_618b = []


for cell_index_618b, cell_618b in enumerate(
    notebook_618b.get("cells", [])
):

    if cell_618b.get("cell_type") != "code":
        continue


    source_618b = "".join(
        cell_618b.get(
            "source",
            []
        )
    )


    if "def add_stuck_sensor" not in source_618b:
        continue


    try:

        tree_618b = ast.parse(
            source_618b
        )

    except SyntaxError:

        continue


    for node_618b in tree_618b.body:

        if not (
            isinstance(
                node_618b,
                (
                    ast.FunctionDef,
                    ast.AsyncFunctionDef,
                )
            )
            and
            node_618b.name == "add_stuck_sensor"
        ):

            continue


        source_lines_618b = (
            source_618b.splitlines()
        )


        start_line_618b = (
            node_618b.lineno - 1
        )

        end_line_618b = (
            node_618b.end_lineno
        )


        function_source_618b = "\n".join(
            source_lines_618b[
                start_line_618b:
                end_line_618b
            ]
        )


        function_hash_618b = (
            hashlib.sha256(
                function_source_618b.encode(
                    "utf-8"
                )
            )
            .hexdigest()
        )


        # --------------------------------------------
        # Function argument names
        # --------------------------------------------

        positional_args_618b = [
            arg.arg
            for arg in node_618b.args.args
        ]


        keyword_only_args_618b = [
            arg.arg
            for arg in node_618b.args.kwonlyargs
        ]


        # --------------------------------------------
        # Extract all return statements
        # --------------------------------------------

        return_statements_618b = []


        for subnode_618b in ast.walk(
            node_618b
        ):

            if isinstance(
                subnode_618b,
                ast.Return
            ):

                return_text_618b = (
                    ast.get_source_segment(
                        source_618b,
                        subnode_618b
                    )
                )

                if return_text_618b is not None:

                    return_statements_618b.append(
                        return_text_618b
                    )


        stuck_definitions_618b.append(
            {
                "cell_index":
                    cell_index_618b,

                "source":
                    function_source_618b,

                "sha256":
                    function_hash_618b,

                "line_count":
                    len(
                        function_source_618b.splitlines()
                    ),

                "positional_args":
                    positional_args_618b,

                "keyword_only_args":
                    keyword_only_args_618b,

                "default_count":
                    len(
                        node_618b.args.defaults
                    ),

                "return_statements":
                    return_statements_618b,
            }
        )


# ------------------------------------------------------------
# 4. Report exact findings
# ------------------------------------------------------------

print(
    "\nNumber of extracted add_stuck_sensor definitions:",
    len(
        stuck_definitions_618b
    )
)


for definition_number_618b, item_618b in enumerate(
    stuck_definitions_618b,
    start=1,
):

    print(
        "\n"
        + "=" * 72
    )

    print(
        f"DEFINITION {definition_number_618b}"
    )

    print(
        "=" * 72
    )


    print(
        "Notebook cell index :",
        item_618b[
            "cell_index"
        ]
    )

    print(
        "Function line count :",
        item_618b[
            "line_count"
        ]
    )

    print(
        "Positional arguments:",
        item_618b[
            "positional_args"
        ]
    )

    print(
        "Keyword-only args   :",
        item_618b[
            "keyword_only_args"
        ]
    )

    print(
        "Default count       :",
        item_618b[
            "default_count"
        ]
    )

    print(
        "SHA-256             :",
        item_618b[
            "sha256"
        ]
    )


    print(
        "\nReturn statement(s):"
    )


    if item_618b[
        "return_statements"
    ]:

        for return_618b in item_618b[
            "return_statements"
        ]:

            print(
                return_618b
            )

    else:

        print(
            "<no explicit return found>"
        )


    print(
        "\nExact function source:\n"
    )

    print(
        item_618b[
            "source"
        ]
    )


# ------------------------------------------------------------
# 5. Compare definitions without executing them
# ------------------------------------------------------------

if len(
    stuck_definitions_618b
) >= 2:

    unique_hashes_618b = {
        item_618b[
            "sha256"
        ]
        for item_618b
        in stuck_definitions_618b
    }

    print(
        "\nUnique implementation count:",
        len(
            unique_hashes_618b
        )
    )


print(
    "\nStep 6.18B completed."
)

print(
    "No stuck-sensor function was executed "
    "and no frozen artifact was modified."
)

In [ ]:
# ============================================================
# Step 6.18C — Inspect exact frozen stuck-sensor mechanism
#               WITHOUT executing it
# ============================================================

import ast


print(
    "=== Step 6.18C — Inspect exact stuck-sensor mechanism ==="
)


# ------------------------------------------------------------
# 1. Confirm Step-6.18B result is available
# ------------------------------------------------------------

assert "stuck_definitions_618b" in globals(), (
    "Step 6.18B results are not available in memory."
)

assert len(stuck_definitions_618b) == 1, (
    "Expected exactly one frozen add_stuck_sensor definition."
)


stuck_source_618c = (
    stuck_definitions_618b[0]["source"]
)


print(
    "Frozen definition cell:",
    stuck_definitions_618b[0]["cell_index"]
)

print(
    "Function lines:",
    stuck_definitions_618b[0]["line_count"]
)


# ------------------------------------------------------------
# 2. Print complete source with line numbers
# ------------------------------------------------------------

source_lines_618c = (
    stuck_source_618c.splitlines()
)


print(
    "\n=== Complete frozen function with line numbers ===\n"
)


for line_number_618c, line_text_618c in enumerate(
    source_lines_618c,
    start=1,
):

    print(
        f"{line_number_618c:03d}: "
        f"{line_text_618c}"
    )


# ------------------------------------------------------------
# 3. Parse function structurally
# ------------------------------------------------------------

tree_618c = ast.parse(
    stuck_source_618c
)

function_node_618c = tree_618c.body[0]


# ------------------------------------------------------------
# 4. Show only methodologically important statements
# ------------------------------------------------------------

important_keywords_618c = [
    "groupby",
    "unit_id",
    "affected_fraction",
    "n_affected",
    "start",
    "end",
    "stuck",
    "mask",
    "cycle",
    "iloc",
    "loc",
    "index",
    "corrupted",
]


print(
    "\n=== Methodologically important source lines ===\n"
)


for line_number_618c, line_text_618c in enumerate(
    source_lines_618c,
    start=1,
):

    lower_line_618c = (
        line_text_618c.lower()
    )

    if any(
        keyword_618c in lower_line_618c
        for keyword_618c
        in important_keywords_618c
    ):

        print(
            f"{line_number_618c:03d}: "
            f"{line_text_618c}"
        )


# ------------------------------------------------------------
# 5. Show loop structure
# ------------------------------------------------------------

print(
    "\n=== Loop structure ==="
)


for node_618c in ast.walk(
    function_node_618c
):

    if isinstance(
        node_618c,
        ast.For
    ):

        loop_text_618c = (
            ast.get_source_segment(
                stuck_source_618c,
                node_618c.iter
            )
        )

        target_text_618c = (
            ast.get_source_segment(
                stuck_source_618c,
                node_618c.target
            )
        )

        print(
            "FOR",
            target_text_618c,
            "IN",
            loop_text_618c
        )


# ------------------------------------------------------------
# 6. Show assignment statements containing key concepts
# ------------------------------------------------------------

print(
    "\n=== Key assignments ==="
)


for node_618c in ast.walk(
    function_node_618c
):

    if isinstance(
        node_618c,
        (
            ast.Assign,
            ast.AnnAssign,
            ast.AugAssign,
        )
    ):

        assignment_text_618c = (
            ast.get_source_segment(
                stuck_source_618c,
                node_618c
            )
        )

        if assignment_text_618c is None:
            continue


        assignment_lower_618c = (
            assignment_text_618c.lower()
        )


        if any(
            keyword_618c
            in assignment_lower_618c
            for keyword_618c in [
                "affected",
                "stuck",
                "start",
                "mask",
                "corrupted",
            ]
        ):

            print(
                assignment_text_618c
            )


print(
    "\nStep 6.18C completed."
)

print(
    "No stuck corruption was executed and "
    "no frozen artifact was modified."
)

In [ ]:
# ============================================================
# Step 6.18C-2 — Compact inspection of frozen stuck mechanism
# ============================================================

print(
    "=== Step 6.18C-2 — Compact stuck-sensor mechanism ==="
)

assert "stuck_source_618c" in globals(), (
    "Step 6.18C source is not available in memory."
)

source_lines_618c2 = stuck_source_618c.splitlines()

# Terms that identify the actual stuck-sensor mechanism
keywords_618c2 = [
    "groupby",
    "unit_id",
    "affected_fraction",
    "n_affected",
    "start",
    "end",
    "stuck",
    "affected_mask",
    "group.index",
    "iloc",
    ".loc[",
    "sensor]",
    "cycle",
]

# Collect matching lines plus a small amount of surrounding context
selected_line_numbers_618c2 = set()

for idx_618c2, line_618c2 in enumerate(
    source_lines_618c2
):

    lower_line_618c2 = line_618c2.lower()

    if any(
        keyword_618c2.lower() in lower_line_618c2
        for keyword_618c2 in keywords_618c2
    ):

        for neighbour_618c2 in range(
            max(0, idx_618c2 - 2),
            min(
                len(source_lines_618c2),
                idx_618c2 + 3
            ),
        ):
            selected_line_numbers_618c2.add(
                neighbour_618c2
            )


print(
    "\n=== Relevant source blocks ===\n"
)

previous_line_618c2 = None

for idx_618c2 in sorted(
    selected_line_numbers_618c2
):

    if (
        previous_line_618c2 is not None
        and
        idx_618c2 > previous_line_618c2 + 1
    ):
        print("    ...")

    print(
        f"{idx_618c2 + 1:03d}: "
        f"{source_lines_618c2[idx_618c2]}"
    )

    previous_line_618c2 = idx_618c2


print(
    "\nStep 6.18C-2 completed."
)

print(
    "Nothing was executed or modified."
)

In [ ]:
# ============================================================
# Step 6.18C-3 — Directly inspect middle section of the frozen
#                add_stuck_sensor function
#
# Nothing is executed or modified.
# ============================================================

print(
    "=== Step 6.18C-3 — Direct stuck-function inspection ==="
)

assert "stuck_source_618c" in globals(), (
    "Frozen stuck function source is not available."
)

lines_618c3 = stuck_source_618c.splitlines()

assert len(lines_618c3) == 113, (
    f"Expected 113 lines, found {len(lines_618c3)}."
)


# ------------------------------------------------------------
# Print only lines 23–105.
# This removes the already-known header and return section.
# ------------------------------------------------------------

START_LINE_618C3 = 23
END_LINE_618C3 = 105


print(
    f"\nShowing frozen function lines "
    f"{START_LINE_618C3}–{END_LINE_618C3}:\n"
)


for line_number_618c3 in range(
    START_LINE_618C3,
    END_LINE_618C3 + 1
):

    print(
        f"{line_number_618c3:03d}: "
        f"{lines_618c3[line_number_618c3 - 1]}"
    )


print(
    "\nStep 6.18C-3 completed."
)

print(
    "No corruption was executed and "
    "no artifact was modified."
)

In [ ]:
# ============================================================
# Step 6.18C-4 — Inspect stuck mechanism, lines 44–74 only
#
# Inspection only:
#   - no corruption executed
#   - no object modified
#   - no artifact modified
# ============================================================

print(
    "=== Step 6.18C-4 — Stuck mechanism lines 44–74 ==="
)

assert "stuck_source_618c" in globals(), (
    "Frozen stuck function source is not available."
)

lines_618c4 = stuck_source_618c.splitlines()

assert len(lines_618c4) == 113, (
    f"Expected 113 lines, found {len(lines_618c4)}."
)

for line_no_618c4 in range(44, 75):

    print(
        f"{line_no_618c4:03d}: "
        f"{lines_618c4[line_no_618c4 - 1]}"
    )


print(
    "\nStep 6.18C-4 completed."
)

print(
    "Nothing was executed or modified."
)

In [ ]:
# ============================================================
# Step 6.18C-5 — Inspect exact stuck-value assignment
#
# Inspection only:
#   - no corruption executed
#   - no artifact modified
# ============================================================

print(
    "=== Step 6.18C-5 — Exact stuck-value assignment ==="
)

assert "stuck_source_618c" in globals(), (
    "Frozen stuck function source is not available."
)

lines_618c5 = stuck_source_618c.splitlines()

assert len(lines_618c5) == 113, (
    f"Expected 113 lines, found {len(lines_618c5)}."
)

for line_no_618c5 in range(68, 101):

    print(
        f"{line_no_618c5:03d}: "
        f"{lines_618c5[line_no_618c5 - 1]}"
    )

print(
    "\nStep 6.18C-5 completed."
)

print(
    "Nothing was executed or modified."
)

In [ ]:
# ============================================================
# Step 6.18D — Verify one frozen stuck-sensor scenario
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd


print(
    "=== Step 6.18D — Verify one frozen stuck-sensor scenario ==="
)


# ------------------------------------------------------------
# 1. Required objects
# ------------------------------------------------------------

required_618d = [
    "stuck_source_618c",
    "val_scaled",
    "scaler",
    "PRIMARY_INPUT_COLUMNS",
    "RETAINED_SENSOR_COLUMNS",
]

missing_618d = [
    name_618d
    for name_618d in required_618d
    if name_618d not in globals()
]

assert not missing_618d, (
    f"Missing required objects: {missing_618d}"
)

print(
    "Required in-memory objects are available."
)


# ------------------------------------------------------------
# 2. Locate project root
# ------------------------------------------------------------

cwd_618d = Path.cwd()

if (cwd_618d / "05_experiments").exists():

    PROJECT_ROOT_618D = cwd_618d

elif (cwd_618d.parent / "05_experiments").exists():

    PROJECT_ROOT_618D = cwd_618d.parent

else:

    raise FileNotFoundError(
        "Could not locate project root."
    )


# ------------------------------------------------------------
# 3. Load frozen Step-4 stuck validation artifact
# ------------------------------------------------------------

stuck_paths_618d = list(
    (
        PROJECT_ROOT_618D
        / "05_experiments"
        / "sensor_degradation"
    ).glob("*stuck*validation*.csv")
)

assert len(stuck_paths_618d) == 1, (
    "Expected exactly one frozen stuck-validation CSV, "
    f"found {len(stuck_paths_618d)}."
)

STUCK_PATH_618D = stuck_paths_618d[0]

stuck_validation_618d = pd.read_csv(
    STUCK_PATH_618D
)

print(
    "Frozen artifact:",
    STUCK_PATH_618D
)

print(
    "Frozen scenarios:",
    len(stuck_validation_618d)
)


# ------------------------------------------------------------
# 4. Select one deterministic frozen scenario
#
#    Use sensor_2 / low because it is the first known
#    frozen scenario and gives the smallest corruption.
# ------------------------------------------------------------

scenario_rows_618d = (
    stuck_validation_618d[
        (stuck_validation_618d["sensor"] == "sensor_2")
        &
        (stuck_validation_618d["severity"] == "low")
    ]
)

assert len(scenario_rows_618d) == 1, (
    "Could not uniquely locate sensor_2 / low scenario."
)

scenario_618d = (
    scenario_rows_618d.iloc[0]
)

sensor_618d = str(
    scenario_618d["sensor"]
)

severity_618d = str(
    scenario_618d["severity"]
)

fraction_618d = float(
    scenario_618d["requested_fraction"]
)


print(
    "\nFrozen scenario:"
)

print(
    "Sensor             :",
    sensor_618d
)

print(
    "Severity           :",
    severity_618d
)

print(
    "Requested fraction :",
    fraction_618d
)

print(
    "Frozen affected count:",
    int(
        scenario_618d["affected_count"]
    )
)

print(
    "Frozen overall fraction:",
    float(
        scenario_618d["overall_fraction"]
    )
)


# ------------------------------------------------------------
# 5. Reconstruct validation inputs in ORIGINAL sensor units
# ------------------------------------------------------------

val_raw_618d = (
    val_scaled.copy()
)

raw_inputs_618d = (
    scaler.inverse_transform(
        val_scaled[
            PRIMARY_INPUT_COLUMNS
        ]
    )
)

raw_inputs_618d = pd.DataFrame(
    raw_inputs_618d,
    columns=PRIMARY_INPUT_COLUMNS,
    index=val_scaled.index,
)

val_raw_618d[
    PRIMARY_INPUT_COLUMNS
] = raw_inputs_618d


# Verify scaling round trip
roundtrip_618d = scaler.transform(
    val_raw_618d[
        PRIMARY_INPUT_COLUMNS
    ]
)

roundtrip_difference_618d = float(
    np.max(
        np.abs(
            roundtrip_618d
            -
            val_scaled[
                PRIMARY_INPUT_COLUMNS
            ].to_numpy()
        )
    )
)

print(
    "\nMaximum scaling round-trip difference:",
    roundtrip_difference_618d
)

assert roundtrip_difference_618d < 1e-10, (
    "Raw-input reconstruction failed."
)

print(
    "Raw validation inputs reconstructed successfully."
)


# ------------------------------------------------------------
# 6. Load the EXACT frozen function into a TEMPORARY namespace
#
#    This does not overwrite the notebook globals.
# ------------------------------------------------------------

temporary_namespace_618d = dict(
    globals()
)

exec(
    stuck_source_618c,
    temporary_namespace_618d
)

candidate_add_stuck_sensor_618d = (
    temporary_namespace_618d[
        "add_stuck_sensor"
    ]
)

print(
    "\nFrozen add_stuck_sensor loaded "
    "into temporary namespace only."
)


# ------------------------------------------------------------
# 7. Execute ONE frozen scenario
# ------------------------------------------------------------

result_618d = (
    candidate_add_stuck_sensor_618d(
        df=val_raw_618d,
        sensor=sensor_618d,
        affected_fraction=fraction_618d,
    )
)


# ------------------------------------------------------------
# 8. Verify return structure
# ------------------------------------------------------------

assert isinstance(
    result_618d,
    tuple
), (
    "Frozen stuck function did not return a tuple."
)

assert len(result_618d) == 3, (
    "Frozen stuck function did not return "
    "exactly three objects."
)


(
    corrupted_618d,
    affected_mask_618d,
    stuck_info_618d,
) = result_618d


affected_mask_618d = np.asarray(
    affected_mask_618d,
    dtype=bool,
)


print(
    "\nReturn structure:"
)

print(
    "Corrupted DataFrame:",
    corrupted_618d.shape
)

print(
    "Affected mask      :",
    affected_mask_618d.shape
)

print(
    "Stuck info         :",
    stuck_info_618d.shape
)


assert (
    corrupted_618d.shape
    ==
    val_raw_618d.shape
)

assert (
    affected_mask_618d.shape
    ==
    (len(val_raw_618d),)
)


# ------------------------------------------------------------
# 9. Verify ONLY the selected sensor was changed
# ------------------------------------------------------------

non_target_columns_618d = [
    column_618d
    for column_618d in val_raw_618d.columns
    if column_618d != sensor_618d
]


pd.testing.assert_frame_equal(
    corrupted_618d[
        non_target_columns_618d
    ],
    val_raw_618d[
        non_target_columns_618d
    ],
    check_dtype=True,
    check_exact=True,
)


print(
    "\nAll non-target columns remain unchanged."
)


# ------------------------------------------------------------
# 10. Compare total affected count with frozen artifact
# ------------------------------------------------------------

actual_count_618d = int(
    affected_mask_618d.sum()
)

expected_count_618d = int(
    scenario_618d[
        "affected_count"
    ]
)


print(
    "\nAffected-count comparison:"
)

print(
    "Frozen expected:",
    expected_count_618d
)

print(
    "Reproduced      :",
    actual_count_618d
)


assert (
    actual_count_618d
    ==
    expected_count_618d
), (
    "Affected count does not match frozen Step-4 artifact."
)


# ------------------------------------------------------------
# 11. Compare overall affected fraction
# ------------------------------------------------------------

actual_fraction_618d = float(
    affected_mask_618d.mean()
)

frozen_fraction_618d = float(
    scenario_618d[
        "overall_fraction"
    ]
)


print(
    "\nOverall-fraction comparison:"
)

print(
    "Frozen     :",
    frozen_fraction_618d
)

print(
    "Reproduced :",
    actual_fraction_618d
)


assert np.isclose(
    actual_fraction_618d,
    frozen_fraction_618d,
    atol=1e-12,
    rtol=0.0,
), (
    "Overall affected fraction does not "
    "match frozen Step-4 artifact."
)


# ------------------------------------------------------------
# 12. Independent engine-wise verification of the mechanism
# ------------------------------------------------------------

mask_series_618d = pd.Series(
    affected_mask_618d,
    index=val_raw_618d.index,
)


engine_fractions_618d = []

maximum_freeze_error_618d = 0.0

maximum_pre_onset_change_618d = 0.0


for unit_id_618d, group_618d in (
    val_raw_618d.groupby(
        "unit_id",
        sort=False,
    )
):

    group_618d = (
        group_618d
        .sort_values("cycle")
    )

    group_indices_618d = (
        group_618d.index.to_numpy()
    )

    n_rows_618d = len(
        group_618d
    )


    # Same frozen Step-4 rule
    n_affected_618d = int(
        round(
            fraction_618d
            *
            n_rows_618d
        )
    )

    n_affected_618d = max(
        1,
        n_affected_618d,
    )

    n_affected_618d = min(
        n_affected_618d,
        n_rows_618d - 1,
    )

    onset_position_618d = (
        n_rows_618d
        -
        n_affected_618d
    )


    source_index_618d = (
        group_indices_618d[
            onset_position_618d - 1
        ]
    )

    expected_affected_indices_618d = (
        group_indices_618d[
            onset_position_618d:
        ]
    )

    expected_unaffected_indices_618d = (
        group_indices_618d[
            :onset_position_618d
        ]
    )


    # --------------------------------------------
    # Verify mask is exactly the final suffix
    # --------------------------------------------

    actual_engine_mask_618d = (
        mask_series_618d.loc[
            group_indices_618d
        ].to_numpy()
    )

    expected_engine_mask_618d = (
        np.zeros(
            n_rows_618d,
            dtype=bool,
        )
    )

    expected_engine_mask_618d[
        onset_position_618d:
    ] = True


    assert np.array_equal(
        actual_engine_mask_618d,
        expected_engine_mask_618d,
    ), (
        f"Stuck mask mismatch for engine "
        f"{unit_id_618d}."
    )


    # --------------------------------------------
    # Verify frozen value
    # --------------------------------------------

    freeze_value_618d = float(
        val_raw_618d.loc[
            source_index_618d,
            sensor_618d,
        ]
    )

    corrupted_values_618d = (
        corrupted_618d.loc[
            expected_affected_indices_618d,
            sensor_618d,
        ].to_numpy(
            dtype=float
        )
    )


    freeze_error_618d = float(
        np.max(
            np.abs(
                corrupted_values_618d
                -
                freeze_value_618d
            )
        )
    )

    maximum_freeze_error_618d = max(
        maximum_freeze_error_618d,
        freeze_error_618d,
    )


    # --------------------------------------------
    # Verify target sensor is unchanged before onset
    # --------------------------------------------

    original_pre_618d = (
        val_raw_618d.loc[
            expected_unaffected_indices_618d,
            sensor_618d,
        ].to_numpy(
            dtype=float
        )
    )

    corrupted_pre_618d = (
        corrupted_618d.loc[
            expected_unaffected_indices_618d,
            sensor_618d,
        ].to_numpy(
            dtype=float
        )
    )


    pre_change_618d = float(
        np.max(
            np.abs(
                original_pre_618d
                -
                corrupted_pre_618d
            )
        )
    )

    maximum_pre_onset_change_618d = max(
        maximum_pre_onset_change_618d,
        pre_change_618d,
    )


    engine_fractions_618d.append(
        n_affected_618d
        /
        n_rows_618d
    )


# ------------------------------------------------------------
# 13. Verify frozen per-engine fraction range
# ------------------------------------------------------------

minimum_engine_fraction_618d = float(
    np.min(
        engine_fractions_618d
    )
)

maximum_engine_fraction_618d = float(
    np.max(
        engine_fractions_618d
    )
)


print(
    "\nPer-engine affected-fraction comparison:"
)

print(
    "Frozen minimum :",
    float(
        scenario_618d[
            "min_engine_fraction"
        ]
    )
)

print(
    "Actual minimum :",
    minimum_engine_fraction_618d
)

print(
    "Frozen maximum :",
    float(
        scenario_618d[
            "max_engine_fraction"
        ]
    )
)

print(
    "Actual maximum :",
    maximum_engine_fraction_618d
)


assert np.isclose(
    minimum_engine_fraction_618d,
    float(
        scenario_618d[
            "min_engine_fraction"
        ]
    ),
    atol=1e-12,
    rtol=0.0,
)

assert np.isclose(
    maximum_engine_fraction_618d,
    float(
        scenario_618d[
            "max_engine_fraction"
        ]
    ),
    atol=1e-12,
    rtol=0.0,
)


# ------------------------------------------------------------
# 14. Final mechanism checks
# ------------------------------------------------------------

print(
    "\nMaximum stuck-value error:",
    maximum_freeze_error_618d
)

print(
    "Maximum pre-onset sensor change:",
    maximum_pre_onset_change_618d
)


assert maximum_freeze_error_618d < 1e-12, (
    "Affected region is not perfectly frozen."
)

assert maximum_pre_onset_change_618d < 1e-12, (
    "Sensor values changed before stuck onset."
)


# ------------------------------------------------------------
# 15. Show first few engine-level records returned
# ------------------------------------------------------------

print(
    "\nFirst 10 returned engine-level stuck records:"
)

display(
    stuck_info_618d.head(10)
)


# ------------------------------------------------------------
# 16. Success
# ------------------------------------------------------------

print(
    "\nStep 6.18D completed successfully."
)

print(
    "The selected frozen Step-4 stuck-sensor "
    "scenario was reproduced exactly."
)

print(
    "Verified mechanism:"
)

print(
    "For each engine, the final requested fraction "
    "of the trajectory is affected."
)

print(
    "Every affected sensor value is replaced by "
    "the last healthy value immediately before onset."
)

In [ ]:
# ============================================================
# Step 6.18E — Verify ALL frozen stuck-sensor scenarios
#
# Purpose:
#   1. Reproduce all 45 frozen Step-4 stuck scenarios.
#   2. Verify affected counts/fractions.
#   3. Verify only the selected sensor changes.
#   4. Verify every engine has a suffix-style stuck region.
#   5. Verify the stuck value is the final healthy value.
#   6. Verify LOW ⊂ MEDIUM ⊂ HIGH for every sensor.
#
# No frozen artifact is modified.
# ============================================================

import numpy as np
import pandas as pd


print(
    "=== Step 6.18E — Verify all frozen stuck-sensor scenarios ==="
)


# ------------------------------------------------------------
# 1. Required objects from successful Step 6.18D
# ------------------------------------------------------------

required_objects_618e = [
    "candidate_add_stuck_sensor_618d",
    "val_raw_618d",
    "stuck_validation_618d",
    "RETAINED_SENSOR_COLUMNS",
]

missing_objects_618e = [
    name_618e
    for name_618e in required_objects_618e
    if name_618e not in globals()
]

assert not missing_objects_618e, (
    "Missing required Step-6.18D objects: "
    f"{missing_objects_618e}"
)

print(
    "Required Step-6.18D objects are available."
)


# ------------------------------------------------------------
# 2. Prepare frozen scenario table
# ------------------------------------------------------------

stuck_plan_618e = (
    stuck_validation_618d
    .copy()
    .reset_index(drop=True)
)

assert len(stuck_plan_618e) == 45, (
    f"Expected 45 frozen scenarios, "
    f"found {len(stuck_plan_618e)}."
)

assert stuck_plan_618e["sensor"].nunique() == 15, (
    "Expected 15 unique retained sensors."
)

assert stuck_plan_618e["severity"].nunique() == 3, (
    "Expected exactly 3 severity levels."
)


EXPECTED_FRACTIONS_618E = {
    "low": 0.20,
    "medium": 0.40,
    "high": 0.60,
}


for severity_618e, expected_fraction_618e in (
    EXPECTED_FRACTIONS_618E.items()
):

    observed_fractions_618e = (
        stuck_plan_618e.loc[
            stuck_plan_618e["severity"] == severity_618e,
            "requested_fraction",
        ]
        .astype(float)
        .unique()
    )

    assert len(observed_fractions_618e) == 1, (
        f"Severity {severity_618e} does not have "
        "one unique frozen fraction."
    )

    assert np.isclose(
        observed_fractions_618e[0],
        expected_fraction_618e,
        atol=1e-12,
        rtol=0.0,
    ), (
        f"Unexpected frozen fraction for {severity_618e}."
    )


print(
    "Frozen scenarios:",
    len(stuck_plan_618e)
)

print(
    "Unique sensors:",
    stuck_plan_618e["sensor"].nunique()
)

print(
    "Severity levels:",
    EXPECTED_FRACTIONS_618E
)


# ------------------------------------------------------------
# 3. Prepare validation-engine groups once
# ------------------------------------------------------------

engine_groups_618e = {}

for unit_id_618e, group_618e in (
    val_raw_618d.groupby(
        "unit_id",
        sort=False,
    )
):

    engine_groups_618e[unit_id_618e] = (
        group_618e
        .sort_values("cycle")
        .copy()
    )


n_engines_618e = len(
    engine_groups_618e
)

assert n_engines_618e == 20, (
    f"Expected 20 validation engines, "
    f"found {n_engines_618e}."
)

print(
    "Validation engines:",
    n_engines_618e
)


# ------------------------------------------------------------
# 4. Storage
# ------------------------------------------------------------

verification_records_618e = []

scenario_masks_618e = {}

global_max_freeze_error_618e = 0.0

global_max_pre_onset_change_618e = 0.0


# ------------------------------------------------------------
# 5. Reproduce all 45 frozen scenarios
# ------------------------------------------------------------

for scenario_number_618e, scenario_row_618e in (
    stuck_plan_618e.iterrows()
):

    sensor_618e = str(
        scenario_row_618e["sensor"]
    )

    severity_618e = str(
        scenario_row_618e["severity"]
    )

    fraction_618e = float(
        scenario_row_618e[
            "requested_fraction"
        ]
    )


    # --------------------------------------------------------
    # 5A. Generate frozen stuck-sensor corruption
    # --------------------------------------------------------

    result_618e = (
        candidate_add_stuck_sensor_618d(
            df=val_raw_618d,
            sensor=sensor_618e,
            affected_fraction=fraction_618e,
        )
    )


    assert isinstance(
        result_618e,
        tuple,
    ), (
        f"{sensor_618e}/{severity_618e}: "
        "function did not return a tuple."
    )

    assert len(result_618e) == 3, (
        f"{sensor_618e}/{severity_618e}: "
        "expected exactly three returned objects."
    )


    (
        corrupted_618e,
        affected_mask_618e,
        stuck_info_618e,
    ) = result_618e


    affected_mask_618e = np.asarray(
        affected_mask_618e,
        dtype=bool,
    )


    # --------------------------------------------------------
    # 5B. Basic structural checks
    # --------------------------------------------------------

    assert (
        corrupted_618e.shape
        ==
        val_raw_618d.shape
    )

    assert (
        affected_mask_618e.shape
        ==
        (len(val_raw_618d),)
    )

    assert len(stuck_info_618e) == n_engines_618e, (
        f"{sensor_618e}/{severity_618e}: "
        "unexpected number of engine-level records."
    )


    # --------------------------------------------------------
    # 5C. Verify every NON-TARGET column is unchanged
    # --------------------------------------------------------

    non_target_columns_618e = [
        column_618e
        for column_618e in val_raw_618d.columns
        if column_618e != sensor_618e
    ]


    pd.testing.assert_frame_equal(
        corrupted_618e[
            non_target_columns_618e
        ],
        val_raw_618d[
            non_target_columns_618e
        ],
        check_dtype=True,
        check_exact=True,
    )


    # --------------------------------------------------------
    # 5D. Compare overall count and fraction with frozen CSV
    # --------------------------------------------------------

    actual_count_618e = int(
        affected_mask_618e.sum()
    )

    frozen_count_618e = int(
        scenario_row_618e[
            "affected_count"
        ]
    )


    assert actual_count_618e == frozen_count_618e, (
        f"{sensor_618e}/{severity_618e}: "
        f"affected count mismatch "
        f"({actual_count_618e} != {frozen_count_618e})."
    )


    actual_overall_fraction_618e = float(
        affected_mask_618e.mean()
    )

    frozen_overall_fraction_618e = float(
        scenario_row_618e[
            "overall_fraction"
        ]
    )


    assert np.isclose(
        actual_overall_fraction_618e,
        frozen_overall_fraction_618e,
        atol=1e-12,
        rtol=0.0,
    ), (
        f"{sensor_618e}/{severity_618e}: "
        "overall affected fraction mismatch."
    )


    # --------------------------------------------------------
    # 5E. Independent engine-wise verification
    # --------------------------------------------------------

    mask_series_618e = pd.Series(
        affected_mask_618e,
        index=val_raw_618d.index,
    )


    engine_fractions_current_618e = []

    scenario_max_freeze_error_618e = 0.0

    scenario_max_pre_change_618e = 0.0


    for unit_id_618e, group_618e in (
        engine_groups_618e.items()
    ):

        group_indices_618e = (
            group_618e.index.to_numpy()
        )

        n_rows_618e = len(
            group_618e
        )


        # Exact frozen Step-4 formula
        n_affected_618e = int(
            round(
                fraction_618e
                *
                n_rows_618e
            )
        )

        n_affected_618e = max(
            1,
            n_affected_618e,
        )

        n_affected_618e = min(
            n_affected_618e,
            n_rows_618e - 1,
        )


        onset_position_618e = (
            n_rows_618e
            -
            n_affected_618e
        )


        source_index_618e = (
            group_indices_618e[
                onset_position_618e - 1
            ]
        )


        affected_indices_618e = (
            group_indices_618e[
                onset_position_618e:
            ]
        )


        unaffected_indices_618e = (
            group_indices_618e[
                :onset_position_618e
            ]
        )


        # ----------------------------------------------------
        # Expected mask = final suffix only
        # ----------------------------------------------------

        expected_engine_mask_618e = (
            np.zeros(
                n_rows_618e,
                dtype=bool,
            )
        )

        expected_engine_mask_618e[
            onset_position_618e:
        ] = True


        actual_engine_mask_618e = (
            mask_series_618e.loc[
                group_indices_618e
            ]
            .to_numpy(
                dtype=bool
            )
        )


        assert np.array_equal(
            actual_engine_mask_618e,
            expected_engine_mask_618e,
        ), (
            f"{sensor_618e}/{severity_618e}, "
            f"engine {unit_id_618e}: "
            "affected mask is not the expected "
            "final trajectory suffix."
        )


        # ----------------------------------------------------
        # Exact freeze-value verification
        # ----------------------------------------------------

        freeze_value_618e = float(
            val_raw_618d.loc[
                source_index_618e,
                sensor_618e,
            ]
        )


        corrupted_affected_values_618e = (
            corrupted_618e.loc[
                affected_indices_618e,
                sensor_618e,
            ]
            .to_numpy(
                dtype=float
            )
        )


        freeze_error_618e = float(
            np.max(
                np.abs(
                    corrupted_affected_values_618e
                    -
                    freeze_value_618e
                )
            )
        )


        scenario_max_freeze_error_618e = max(
            scenario_max_freeze_error_618e,
            freeze_error_618e,
        )


        # ----------------------------------------------------
        # Target sensor before onset must remain unchanged
        # ----------------------------------------------------

        original_before_618e = (
            val_raw_618d.loc[
                unaffected_indices_618e,
                sensor_618e,
            ]
            .to_numpy(
                dtype=float
            )
        )


        corrupted_before_618e = (
            corrupted_618e.loc[
                unaffected_indices_618e,
                sensor_618e,
            ]
            .to_numpy(
                dtype=float
            )
        )


        pre_change_618e = float(
            np.max(
                np.abs(
                    original_before_618e
                    -
                    corrupted_before_618e
                )
            )
        )


        scenario_max_pre_change_618e = max(
            scenario_max_pre_change_618e,
            pre_change_618e,
        )


        engine_fractions_current_618e.append(
            n_affected_618e
            /
            n_rows_618e
        )


    # --------------------------------------------------------
    # 5F. Frozen per-engine fraction boundaries
    # --------------------------------------------------------

    actual_min_fraction_618e = float(
        np.min(
            engine_fractions_current_618e
        )
    )

    actual_max_fraction_618e = float(
        np.max(
            engine_fractions_current_618e
        )
    )


    frozen_min_fraction_618e = float(
        scenario_row_618e[
            "min_engine_fraction"
        ]
    )

    frozen_max_fraction_618e = float(
        scenario_row_618e[
            "max_engine_fraction"
        ]
    )


    assert np.isclose(
        actual_min_fraction_618e,
        frozen_min_fraction_618e,
        atol=1e-12,
        rtol=0.0,
    ), (
        f"{sensor_618e}/{severity_618e}: "
        "minimum engine fraction mismatch."
    )


    assert np.isclose(
        actual_max_fraction_618e,
        frozen_max_fraction_618e,
        atol=1e-12,
        rtol=0.0,
    ), (
        f"{sensor_618e}/{severity_618e}: "
        "maximum engine fraction mismatch."
    )


    # --------------------------------------------------------
    # 5G. Mechanism must be exact
    # --------------------------------------------------------

    assert scenario_max_freeze_error_618e < 1e-12, (
        f"{sensor_618e}/{severity_618e}: "
        "stuck region is not perfectly frozen."
    )

    assert scenario_max_pre_change_618e < 1e-12, (
        f"{sensor_618e}/{severity_618e}: "
        "target sensor changed before onset."
    )


    global_max_freeze_error_618e = max(
        global_max_freeze_error_618e,
        scenario_max_freeze_error_618e,
    )

    global_max_pre_onset_change_618e = max(
        global_max_pre_onset_change_618e,
        scenario_max_pre_change_618e,
    )


    # Keep mask for nested-severity verification
    scenario_masks_618e[
        (
            sensor_618e,
            severity_618e,
        )
    ] = affected_mask_618e.copy()


    verification_records_618e.append(
        {
            "sensor": sensor_618e,
            "severity": severity_618e,
            "requested_fraction": fraction_618e,
            "frozen_affected_count": frozen_count_618e,
            "actual_affected_count": actual_count_618e,
            "frozen_overall_fraction": frozen_overall_fraction_618e,
            "actual_overall_fraction": actual_overall_fraction_618e,
            "frozen_min_engine_fraction": frozen_min_fraction_618e,
            "actual_min_engine_fraction": actual_min_fraction_618e,
            "frozen_max_engine_fraction": frozen_max_fraction_618e,
            "actual_max_engine_fraction": actual_max_fraction_618e,
            "max_freeze_error": scenario_max_freeze_error_618e,
            "max_pre_onset_change": scenario_max_pre_change_618e,
            "passed": True,
        }
    )


    completed_618e = (
        scenario_number_618e + 1
    )

    if (
        completed_618e % 15 == 0
        or
        completed_618e == len(stuck_plan_618e)
    ):

        print(
            f"[{completed_618e}/"
            f"{len(stuck_plan_618e)}] verified"
        )


# ------------------------------------------------------------
# 6. Verification table
# ------------------------------------------------------------

stuck_verification_618e = pd.DataFrame(
    verification_records_618e
)


assert len(stuck_verification_618e) == 45

assert stuck_verification_618e[
    "passed"
].all()


# ------------------------------------------------------------
# 7. Verify severity masks are nested:
#
#       LOW ⊂ MEDIUM ⊂ HIGH
# ------------------------------------------------------------

nested_records_618e = []


for sensor_618e in RETAINED_SENSOR_COLUMNS:

    low_mask_618e = scenario_masks_618e[
        (
            sensor_618e,
            "low",
        )
    ]

    medium_mask_618e = scenario_masks_618e[
        (
            sensor_618e,
            "medium",
        )
    ]

    high_mask_618e = scenario_masks_618e[
        (
            sensor_618e,
            "high",
        )
    ]


    low_subset_medium_618e = bool(
        np.all(
            (~low_mask_618e)
            |
            medium_mask_618e
        )
    )

    medium_subset_high_618e = bool(
        np.all(
            (~medium_mask_618e)
            |
            high_mask_618e
        )
    )


    low_count_618e = int(
        low_mask_618e.sum()
    )

    medium_count_618e = int(
        medium_mask_618e.sum()
    )

    high_count_618e = int(
        high_mask_618e.sum()
    )


    counts_increase_618e = bool(
        low_count_618e
        <
        medium_count_618e
        <
        high_count_618e
    )


    assert low_subset_medium_618e, (
        f"{sensor_618e}: LOW mask is not "
        "a subset of MEDIUM."
    )

    assert medium_subset_high_618e, (
        f"{sensor_618e}: MEDIUM mask is not "
        "a subset of HIGH."
    )

    assert counts_increase_618e, (
        f"{sensor_618e}: affected counts "
        "do not increase with severity."
    )


    nested_records_618e.append(
        {
            "sensor": sensor_618e,
            "low_count": low_count_618e,
            "medium_count": medium_count_618e,
            "high_count": high_count_618e,
            "low_subset_medium": low_subset_medium_618e,
            "medium_subset_high": medium_subset_high_618e,
            "counts_increase": counts_increase_618e,
        }
    )


stuck_nested_summary_618e = pd.DataFrame(
    nested_records_618e
)


# ------------------------------------------------------------
# 8. Final reporting
# ------------------------------------------------------------

print(
    "\nVerified frozen scenarios:",
    len(stuck_verification_618e)
)

print(
    "Unique sensors:",
    stuck_verification_618e[
        "sensor"
    ].nunique()
)

print(
    "Unique severities:",
    stuck_verification_618e[
        "severity"
    ].nunique()
)

print(
    "\nMaximum stuck-value error across all scenarios:",
    global_max_freeze_error_618e
)

print(
    "Maximum pre-onset sensor change across all scenarios:",
    global_max_pre_onset_change_618e
)


print(
    "\n=== Nested stuck-mask verification by sensor ==="
)

display(
    stuck_nested_summary_618e
)


print(
    "\n=== First 12 frozen-scenario verification records ==="
)

display(
    stuck_verification_618e.head(12)
)


print(
    "\nStep 6.18E completed successfully."
)

print(
    "All 45 frozen Step-4 stuck-sensor scenarios "
    "were reproduced exactly."
)

print(
    "For every retained sensor:"
)

print(
    "LOW mask \u2282 MEDIUM mask \u2282 HIGH mask."
)

print(
    "The affected region is always the final "
    "suffix of each engine trajectory."
)

print(
    "Every affected value is frozen to the "
    "last healthy value immediately before onset."
)

In [ ]:
# ============================================================
# Step 6.18F — Stuck-sensor PCA experiment pre-check
#
# Purpose:
#   Verify every object required for the final stuck-sensor
#   PCA response experiment before running all 45 scenarios.
#
# No corruption generated.
# No file written.
# ============================================================

import inspect
import numpy as np
import pandas as pd


print(
    "=== Step 6.18F — Stuck-sensor PCA experiment pre-check ==="
)


# ------------------------------------------------------------
# 1. Required objects
# ------------------------------------------------------------

required_objects_618f = [
    "candidate_add_stuck_sensor_618d",
    "stuck_validation_618d",
    "val_raw_618d",
    "val_scaled",
    "scaler",
    "PRIMARY_INPUT_COLUMNS",
    "RETAINED_SENSOR_COLUMNS",
    "SENSOR_INPUT_INDICES",
    "create_sequences",
    "calculate_pca_window_quality",
    "sensor_pca",
    "X_val_clean",
    "y_val_clean",
    "val_seq_meta",
    "dev_pca_mean_rmse",
    "val_pca_mean_rmse",
    "val_pca_max_rmse",
]


availability_618f = {}


for name_618f in required_objects_618f:

    availability_618f[name_618f] = (
        name_618f in globals()
    )


for name_618f, available_618f in (
    availability_618f.items()
):

    print(
        f"{name_618f:<35}: {available_618f}"
    )


missing_618f = [
    name_618f
    for name_618f, available_618f
    in availability_618f.items()
    if not available_618f
]


assert not missing_618f, (
    "Missing required objects: "
    f"{missing_618f}"
)


# ------------------------------------------------------------
# 2. Function signatures
# ------------------------------------------------------------

print(
    "\n=== Function signatures ==="
)

print(
    "candidate_add_stuck_sensor_618d:",
    inspect.signature(
        candidate_add_stuck_sensor_618d
    )
)

print(
    "create_sequences:",
    inspect.signature(
        create_sequences
    )
)

print(
    "calculate_pca_window_quality:",
    inspect.signature(
        calculate_pca_window_quality
    )
)


# ------------------------------------------------------------
# 3. Frozen scenario checks
# ------------------------------------------------------------

print(
    "\n=== Frozen stuck scenarios ==="
)

print(
    "Scenario count:",
    len(stuck_validation_618d)
)

print(
    "Unique sensors:",
    stuck_validation_618d[
        "sensor"
    ].nunique()
)

print(
    "Unique severities:",
    stuck_validation_618d[
        "severity"
    ].nunique()
)


assert len(stuck_validation_618d) == 45

assert (
    stuck_validation_618d[
        "sensor"
    ].nunique()
    == 15
)

assert (
    stuck_validation_618d[
        "severity"
    ].nunique()
    == 3
)


# ------------------------------------------------------------
# 4. Input-column consistency
# ------------------------------------------------------------

print(
    "\n=== Input-column consistency ==="
)

print(
    "Primary input columns:",
    len(PRIMARY_INPUT_COLUMNS)
)

print(
    "Retained sensors:",
    len(RETAINED_SENSOR_COLUMNS)
)

print(
    "Sensor input indices:",
    len(SENSOR_INPUT_INDICES)
)


assert len(RETAINED_SENSOR_COLUMNS) == 15

assert len(SENSOR_INPUT_INDICES) == 15

assert set(RETAINED_SENSOR_COLUMNS).issubset(
    set(PRIMARY_INPUT_COLUMNS)
)


# ------------------------------------------------------------
# 5. Clean validation sequence structure
# ------------------------------------------------------------

print(
    "\n=== Clean validation objects ==="
)

print(
    "X_val_clean shape:",
    X_val_clean.shape
)

print(
    "y_val_clean shape:",
    y_val_clean.shape
)

print(
    "val_seq_meta shape:",
    val_seq_meta.shape
)


assert X_val_clean.ndim == 3

assert len(X_val_clean) == len(y_val_clean)

assert len(X_val_clean) == len(val_seq_meta)

assert X_val_clean.shape[2] == len(
    PRIMARY_INPUT_COLUMNS
)


# ------------------------------------------------------------
# 6. PCA reference structure
# ------------------------------------------------------------

print(
    "\n=== PCA reference objects ==="
)

print(
    "sensor_pca components:",
    sensor_pca.n_components_
)

print(
    "sensor_pca input features:",
    sensor_pca.n_features_in_
)

print(
    "Development PCA mean-RE samples:",
    len(dev_pca_mean_rmse)
)

print(
    "Validation PCA mean-RE samples:",
    len(val_pca_mean_rmse)
)

print(
    "Validation PCA max-RE samples:",
    len(val_pca_max_rmse)
)


assert (
    sensor_pca.n_features_in_
    ==
    len(RETAINED_SENSOR_COLUMNS)
)

assert len(val_pca_mean_rmse) == len(
    X_val_clean
)

assert len(val_pca_max_rmse) == len(
    X_val_clean
)


# ------------------------------------------------------------
# 7. Frozen development threshold
# ------------------------------------------------------------

dev_p95_618f = float(
    np.percentile(
        dev_pca_mean_rmse,
        95
    )
)


clean_val_mean_618f = float(
    np.mean(
        val_pca_mean_rmse
    )
)

clean_val_median_618f = float(
    np.median(
        val_pca_mean_rmse
    )
)

clean_val_above_p95_618f = float(
    np.mean(
        val_pca_mean_rmse
        >
        dev_p95_618f
    )
    * 100.0
)


print(
    "\n=== Clean PCA reference ==="
)

print(
    "Development mean-RE P95:",
    round(
        dev_p95_618f,
        6
    )
)

print(
    "Clean validation mean RE:",
    round(
        clean_val_mean_618f,
        6
    )
)

print(
    "Clean validation median RE:",
    round(
        clean_val_median_618f,
        6
    )
)

print(
    "Clean validation above dev-P95 (%):",
    round(
        clean_val_above_p95_618f,
        3
    )
)


# ------------------------------------------------------------
# 8. Verify the known clean reference values
# ------------------------------------------------------------

assert np.isclose(
    dev_p95_618f,
    0.227844,
    atol=5e-7,
), (
    "Development PCA P95 no longer matches "
    "the established Step-6 reference."
)


assert np.isclose(
    clean_val_mean_618f,
    0.206435,
    atol=5e-7,
), (
    "Clean validation PCA mean no longer matches "
    "the established Step-6 reference."
)


assert np.isclose(
    clean_val_median_618f,
    0.206905,
    atol=5e-7,
), (
    "Clean validation PCA median no longer matches "
    "the established Step-6 reference."
)


# ------------------------------------------------------------
# 9. Final pre-check
# ------------------------------------------------------------

print(
    "\nStep 6.18F pre-check completed successfully."
)

print(
    "No corruption was generated "
    "and no frozen artifact was modified."
)

print(
    "The notebook is ready for the full "
    "45-scenario stuck-sensor PCA experiment."
)

In [ ]:
# ============================================================
# Step 6.18G — Full stuck-sensor PCA response experiment
#
# Purpose:
#   Evaluate whether the frozen PCA sensor-quality signal
#   responds to all 45 verified stuck-sensor scenarios:
#
#       15 retained sensors × 3 severity levels
#
# Important:
#   - Uses the EXACT frozen Step-4 stuck corruption.
#   - Uses the EXISTING fitted scaler.
#   - Uses the EXISTING fitted PCA model.
#   - Does NOT refit PCA.
#   - Does NOT modify frozen corruption artifacts.
#   - Does NOT use official test data.
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


print(
    "=== Step 6.18G — Full stuck-sensor PCA response ==="
)


# ------------------------------------------------------------
# 1. Frozen experiment configuration
# ------------------------------------------------------------

TARGET_COLUMN_618G = "RUL_cap_125"
WINDOW_SIZE_618G = 30

EXPECTED_SCENARIOS_618G = 45
EXPECTED_SENSORS_618G = 15

SEVERITY_ORDER_618G = [
    "low",
    "medium",
    "high",
]


# ------------------------------------------------------------
# 2. Required-object check
# ------------------------------------------------------------

required_objects_618g = [
    "candidate_add_stuck_sensor_618d",
    "stuck_validation_618d",
    "val_raw_618d",
    "val_scaled",
    "scaler",
    "PRIMARY_INPUT_COLUMNS",
    "RETAINED_SENSOR_COLUMNS",
    "SENSOR_INPUT_INDICES",
    "create_sequences",
    "calculate_pca_window_quality",
    "sensor_pca",
    "X_val_clean",
    "y_val_clean",
    "val_seq_meta",
    "dev_pca_mean_rmse",
    "val_pca_mean_rmse",
    "val_pca_max_rmse",
]

missing_objects_618g = [
    name_618g
    for name_618g in required_objects_618g
    if name_618g not in globals()
]

assert not missing_objects_618g, (
    "Missing required objects: "
    f"{missing_objects_618g}"
)

print(
    "Required objects are available."
)


# ------------------------------------------------------------
# 3. Verify frozen configuration
# ------------------------------------------------------------

stuck_plan_618g = (
    stuck_validation_618d
    .copy()
    .reset_index(drop=True)
)

assert len(stuck_plan_618g) == EXPECTED_SCENARIOS_618G

assert (
    stuck_plan_618g["sensor"].nunique()
    ==
    EXPECTED_SENSORS_618G
)

assert (
    stuck_plan_618g["severity"].nunique()
    ==
    3
)

assert len(RETAINED_SENSOR_COLUMNS) == 15

assert len(SENSOR_INPUT_INDICES) == 15

assert X_val_clean.shape[1] == WINDOW_SIZE_618G

assert (
    X_val_clean.shape[2]
    ==
    len(PRIMARY_INPUT_COLUMNS)
)

assert TARGET_COLUMN_618G in val_scaled.columns


print(
    "Target          :",
    TARGET_COLUMN_618G
)

print(
    "Window size     :",
    WINDOW_SIZE_618G
)

print(
    "Retained sensors:",
    len(RETAINED_SENSOR_COLUMNS)
)

print(
    "Frozen scenarios:",
    len(stuck_plan_618g)
)


# ------------------------------------------------------------
# 4. Verify scaler compatibility BEFORE experiment
# ------------------------------------------------------------

assert hasattr(
    scaler,
    "n_features_in_",
), (
    "The fitted scaler does not expose n_features_in_."
)

assert (
    scaler.n_features_in_
    ==
    len(PRIMARY_INPUT_COLUMNS)
), (
    "Scaler input-feature count does not match "
    "PRIMARY_INPUT_COLUMNS."
)


if hasattr(
    scaler,
    "feature_names_in_",
):

    fitted_scaler_columns_618g = list(
        scaler.feature_names_in_
    )

    current_scaler_columns_618g = list(
        PRIMARY_INPUT_COLUMNS
    )

    assert (
        fitted_scaler_columns_618g
        ==
        current_scaler_columns_618g
    ), (
        "Scaler feature names/order do not match "
        "PRIMARY_INPUT_COLUMNS."
    )


print(
    "Scaler compatibility verified."
)


# ------------------------------------------------------------
# 5. Clean PCA references
# ------------------------------------------------------------

clean_mean_re_array_618g = np.asarray(
    val_pca_mean_rmse,
    dtype=float,
)

clean_max_re_array_618g = np.asarray(
    val_pca_max_rmse,
    dtype=float,
)


assert clean_mean_re_array_618g.shape == (
    len(X_val_clean),
)

assert clean_max_re_array_618g.shape == (
    len(X_val_clean),
)

assert np.all(
    np.isfinite(
        clean_mean_re_array_618g
    )
)

assert np.all(
    np.isfinite(
        clean_max_re_array_618g
    )
)


dev_p95_618g = float(
    np.percentile(
        np.asarray(
            dev_pca_mean_rmse,
            dtype=float,
        ),
        95,
    )
)


clean_mean_re_618g = float(
    np.mean(
        clean_mean_re_array_618g
    )
)

clean_median_re_618g = float(
    np.median(
        clean_mean_re_array_618g
    )
)

clean_mean_max_re_618g = float(
    np.mean(
        clean_max_re_array_618g
    )
)

clean_above_dev95_pct_618g = float(
    np.mean(
        clean_mean_re_array_618g
        >
        dev_p95_618g
    )
    * 100.0
)


assert clean_mean_re_618g > 0.0
assert clean_median_re_618g > 0.0
assert clean_mean_max_re_618g > 0.0


print(
    "\nClean PCA reference:"
)

print(
    "Development mean-RE P95        :",
    round(
        dev_p95_618g,
        6,
    )
)

print(
    "Clean validation mean RE       :",
    round(
        clean_mean_re_618g,
        6,
    )
)

print(
    "Clean validation median RE     :",
    round(
        clean_median_re_618g,
        6,
    )
)

print(
    "Clean validation above P95 (%) :",
    round(
        clean_above_dev95_pct_618g,
        3,
    )
)


# ------------------------------------------------------------
# 6. Metadata-alignment helper
#
# Avoid dtype-only false alarms such as int64 vs float64.
# Values and ordering must still match exactly/numerically.
# ------------------------------------------------------------

def assert_metadata_aligned_618g(
    candidate_meta,
    reference_meta,
):

    assert list(candidate_meta.columns) == list(
        reference_meta.columns
    ), (
        "Sequence metadata columns do not match."
    )

    assert len(candidate_meta) == len(
        reference_meta
    )


    candidate_meta_618g = (
        candidate_meta
        .reset_index(drop=True)
    )

    reference_meta_618g = (
        reference_meta
        .reset_index(drop=True)
    )


    for column_618g in reference_meta_618g.columns:

        left_618g = (
            candidate_meta_618g[
                column_618g
            ]
        )

        right_618g = (
            reference_meta_618g[
                column_618g
            ]
        )


        left_numeric_618g = pd.to_numeric(
            left_618g,
            errors="coerce",
        )

        right_numeric_618g = pd.to_numeric(
            right_618g,
            errors="coerce",
        )


        if (
            left_numeric_618g.notna().all()
            and
            right_numeric_618g.notna().all()
        ):

            np.testing.assert_allclose(
                left_numeric_618g.to_numpy(
                    dtype=float
                ),
                right_numeric_618g.to_numpy(
                    dtype=float
                ),
                rtol=0.0,
                atol=0.0,
            )

        else:

            assert (
                left_618g.astype(str).tolist()
                ==
                right_618g.astype(str).tolist()
            ), (
                "Metadata mismatch in column "
                f"{column_618g}."
            )


# ------------------------------------------------------------
# 7. Storage for 45 scenario results
# ------------------------------------------------------------

stuck_pca_records_618g = []


# ------------------------------------------------------------
# 8. Run all frozen stuck-sensor scenarios
# ------------------------------------------------------------

for scenario_index_618g, frozen_row_618g in (
    stuck_plan_618g.iterrows()
):

    sensor_618g = str(
        frozen_row_618g["sensor"]
    )

    severity_618g = str(
        frozen_row_618g["severity"]
    )

    requested_fraction_618g = float(
        frozen_row_618g[
            "requested_fraction"
        ]
    )


    scenario_id_618g = (
        f"FD001_STUCK_"
        f"{scenario_index_618g + 1:03d}"
    )


    # --------------------------------------------------------
    # 8A. Reproduce exact frozen stuck corruption
    # --------------------------------------------------------

    (
        stuck_raw_618g,
        affected_mask_618g,
        stuck_info_618g,
    ) = candidate_add_stuck_sensor_618d(
        df=val_raw_618d,
        sensor=sensor_618g,
        affected_fraction=requested_fraction_618g,
    )


    affected_mask_618g = np.asarray(
        affected_mask_618g,
        dtype=bool,
    )


    assert (
        stuck_raw_618g.shape
        ==
        val_raw_618d.shape
    )

    assert (
        affected_mask_618g.shape
        ==
        (len(val_raw_618d),)
    )

    assert len(stuck_info_618g) == 20


    # --------------------------------------------------------
    # 8B. Verify against frozen Step-4 artifact
    # --------------------------------------------------------

    actual_affected_count_618g = int(
        affected_mask_618g.sum()
    )

    frozen_affected_count_618g = int(
        frozen_row_618g[
            "affected_count"
        ]
    )


    assert (
        actual_affected_count_618g
        ==
        frozen_affected_count_618g
    ), (
        f"{scenario_id_618g}: affected-count mismatch."
    )


    actual_fraction_618g = float(
        affected_mask_618g.mean()
    )

    frozen_fraction_618g = float(
        frozen_row_618g[
            "overall_fraction"
        ]
    )


    assert np.isclose(
        actual_fraction_618g,
        frozen_fraction_618g,
        atol=1e-12,
        rtol=0.0,
    ), (
        f"{scenario_id_618g}: "
        "overall-fraction mismatch."
    )


    # --------------------------------------------------------
    # 8C. Scale corrupted sensor data using EXISTING scaler
    #
    # Important:
    #   We copy val_scaled so unit_id, cycle and RUL targets
    #   remain exactly the established validation values.
    # --------------------------------------------------------

    stuck_scaled_618g = (
        val_scaled
        .copy(deep=True)
    )


    scaled_inputs_618g = scaler.transform(
        stuck_raw_618g[
            PRIMARY_INPUT_COLUMNS
        ]
    )


    assert scaled_inputs_618g.shape == (
        len(stuck_raw_618g),
        len(PRIMARY_INPUT_COLUMNS),
    )


    assert np.all(
        np.isfinite(
            scaled_inputs_618g
        )
    )


    stuck_scaled_618g.loc[
        :,
        PRIMARY_INPUT_COLUMNS
    ] = scaled_inputs_618g


    # --------------------------------------------------------
    # 8D. Recreate sequences
    # --------------------------------------------------------

    (
        X_stuck_618g,
        y_stuck_618g,
        meta_stuck_618g,
    ) = create_sequences(
        df=stuck_scaled_618g,
        feature_columns=PRIMARY_INPUT_COLUMNS,
        target_column=TARGET_COLUMN_618G,
        window_size=WINDOW_SIZE_618G,
    )


    assert (
        X_stuck_618g.shape
        ==
        X_val_clean.shape
    ), (
        f"{scenario_id_618g}: "
        "sequence shape changed."
    )


    np.testing.assert_allclose(
        np.asarray(
            y_stuck_618g,
            dtype=float,
        ),
        np.asarray(
            y_val_clean,
            dtype=float,
        ),
        rtol=0.0,
        atol=0.0,
    )


    assert_metadata_aligned_618g(
        meta_stuck_618g,
        val_seq_meta,
    )


    # --------------------------------------------------------
    # 8E. Generate PCA quality signal
    #
    # Function structure already verified:
    #
    #   output 1 = row/time-step reconstruction error
    #   output 2 = window mean reconstruction error
    #   output 3 = window maximum reconstruction error
    # --------------------------------------------------------

    (
        stuck_pca_row_rmse_618g,
        stuck_pca_mean_rmse_618g,
        stuck_pca_max_rmse_618g,
    ) = calculate_pca_window_quality(
        X_stuck_618g,
        SENSOR_INPUT_INDICES,
        sensor_pca,
    )


    stuck_pca_mean_rmse_618g = np.asarray(
        stuck_pca_mean_rmse_618g,
        dtype=float,
    )

    stuck_pca_max_rmse_618g = np.asarray(
        stuck_pca_max_rmse_618g,
        dtype=float,
    )


    assert stuck_pca_mean_rmse_618g.shape == (
        len(X_val_clean),
    )

    assert stuck_pca_max_rmse_618g.shape == (
        len(X_val_clean),
    )

    assert np.all(
        np.isfinite(
            stuck_pca_mean_rmse_618g
        )
    )

    assert np.all(
        np.isfinite(
            stuck_pca_max_rmse_618g
        )
    )


    # --------------------------------------------------------
    # 8F. Scenario-level PCA response metrics
    # --------------------------------------------------------

    corrupted_mean_re_618g = float(
        np.mean(
            stuck_pca_mean_rmse_618g
        )
    )

    corrupted_median_re_618g = float(
        np.median(
            stuck_pca_mean_rmse_618g
        )
    )

    corrupted_mean_max_re_618g = float(
        np.mean(
            stuck_pca_max_rmse_618g
        )
    )


    mean_re_change_pct_618g = float(
        (
            corrupted_mean_re_618g
            /
            clean_mean_re_618g
            -
            1.0
        )
        * 100.0
    )


    median_re_change_pct_618g = float(
        (
            corrupted_median_re_618g
            /
            clean_median_re_618g
            -
            1.0
        )
        * 100.0
    )


    mean_max_re_change_pct_618g = float(
        (
            corrupted_mean_max_re_618g
            /
            clean_mean_max_re_618g
            -
            1.0
        )
        * 100.0
    )


    fraction_increased_pct_618g = float(
        np.mean(
            stuck_pca_mean_rmse_618g
            >
            clean_mean_re_array_618g
        )
        * 100.0
    )


    corrupted_above_dev95_pct_618g = float(
        np.mean(
            stuck_pca_mean_rmse_618g
            >
            dev_p95_618g
        )
        * 100.0
    )


    above_dev95_delta_pp_618g = float(
        corrupted_above_dev95_pct_618g
        -
        clean_above_dev95_pct_618g
    )


    # --------------------------------------------------------
    # 8G. Store scenario record
    # --------------------------------------------------------

    stuck_pca_records_618g.append(
        {
            "scenario_id": scenario_id_618g,

            "sensor": sensor_618g,

            "severity": severity_618g,

            "requested_fraction":
                requested_fraction_618g,

            "affected_count":
                actual_affected_count_618g,

            "actual_fraction":
                actual_fraction_618g,

            "clean_mean_re":
                clean_mean_re_618g,

            "corrupted_mean_re":
                corrupted_mean_re_618g,

            "mean_RE_change_pct":
                mean_re_change_pct_618g,

            "clean_median_re":
                clean_median_re_618g,

            "corrupted_median_re":
                corrupted_median_re_618g,

            "median_RE_change_pct":
                median_re_change_pct_618g,

            "fraction_increased_pct":
                fraction_increased_pct_618g,

            "clean_above_dev95_pct":
                clean_above_dev95_pct_618g,

            "corrupted_above_dev95_pct":
                corrupted_above_dev95_pct_618g,

            "above_dev95_delta_pp":
                above_dev95_delta_pp_618g,

            "clean_mean_max_re":
                clean_mean_max_re_618g,

            "corrupted_mean_max_re":
                corrupted_mean_max_re_618g,

            "mean_max_RE_change_pct":
                mean_max_re_change_pct_618g,
        }
    )


    completed_618g = (
        scenario_index_618g + 1
    )

    if (
        completed_618g % 15 == 0
        or
        completed_618g
        ==
        EXPECTED_SCENARIOS_618G
    ):

        print(
            f"[{completed_618g}/"
            f"{EXPECTED_SCENARIOS_618G}] completed"
        )


# ------------------------------------------------------------
# 9. Create full scenario table
# ------------------------------------------------------------

stuck_pca_response_618g = pd.DataFrame(
    stuck_pca_records_618g
)


assert len(stuck_pca_response_618g) == 45

assert (
    stuck_pca_response_618g[
        "sensor"
    ].nunique()
    ==
    15
)

assert (
    stuck_pca_response_618g[
        "severity"
    ].nunique()
    ==
    3
)


print(
    "\nStuck PCA-response scenarios:",
    len(stuck_pca_response_618g)
)

print(
    "Unique sensors:",
    stuck_pca_response_618g[
        "sensor"
    ].nunique()
)

print(
    "Unique severities:",
    stuck_pca_response_618g[
        "severity"
    ].nunique()
)


# ------------------------------------------------------------
# 10. Severity-level summary
# ------------------------------------------------------------

stuck_severity_summary_618g = (
    stuck_pca_response_618g
    .groupby(
        "severity",
        observed=True,
    )
    .agg(
        scenarios=(
            "scenario_id",
            "count",
        ),

        mean_RE_change_pct=(
            "mean_RE_change_pct",
            "mean",
        ),

        median_RE_change_pct=(
            "median_RE_change_pct",
            "mean",
        ),

        mean_fraction_increased_pct=(
            "fraction_increased_pct",
            "mean",
        ),

        mean_above_dev95_pct=(
            "corrupted_above_dev95_pct",
            "mean",
        ),

        mean_above_dev95_delta_pp=(
            "above_dev95_delta_pp",
            "mean",
        ),

        mean_max_RE_change_pct=(
            "mean_max_RE_change_pct",
            "mean",
        ),
    )
    .reindex(
        SEVERITY_ORDER_618G
    )
)


print(
    "\n=== Stuck-sensor PCA response by severity ==="
)

display(
    stuck_severity_summary_618g.round(3)
)


# ------------------------------------------------------------
# 11. Sensor-level sensitivity summary
# ------------------------------------------------------------

stuck_sensor_summary_618g = (
    stuck_pca_response_618g
    .groupby(
        "sensor",
        observed=True,
    )
    .agg(
        scenarios=(
            "scenario_id",
            "count",
        ),

        mean_RE_change_pct=(
            "mean_RE_change_pct",
            "mean",
        ),

        median_RE_change_pct=(
            "median_RE_change_pct",
            "mean",
        ),

        min_RE_change_pct=(
            "mean_RE_change_pct",
            "min",
        ),

        max_RE_change_pct=(
            "mean_RE_change_pct",
            "max",
        ),

        mean_fraction_increased_pct=(
            "fraction_increased_pct",
            "mean",
        ),

        mean_above_dev95_pct=(
            "corrupted_above_dev95_pct",
            "mean",
        ),

        max_above_dev95_pct=(
            "corrupted_above_dev95_pct",
            "max",
        ),
    )
    .sort_values(
        "mean_RE_change_pct",
        ascending=False,
    )
)


print(
    "\n=== Sensor-wise PCA sensitivity "
    "across stuck-sensor scenarios ==="
)

display(
    stuck_sensor_summary_618g.round(3)
)


# ------------------------------------------------------------
# 12. Strongest 15 individual stuck responses
# ------------------------------------------------------------

strongest_stuck_618g = (
    stuck_pca_response_618g
    .sort_values(
        "mean_RE_change_pct",
        ascending=False,
    )
    .head(15)
)


print(
    "\n=== Strongest 15 stuck-sensor PCA responses ==="
)

display(
    strongest_stuck_618g.round(3)
)


# ------------------------------------------------------------
# 13. Save results
# ------------------------------------------------------------

cwd_618g = Path.cwd()

if (
    cwd_618g
    /
    "06_results"
).exists():

    PROJECT_ROOT_618G = cwd_618g

elif (
    cwd_618g.parent
    /
    "06_results"
).exists():

    PROJECT_ROOT_618G = cwd_618g.parent

else:

    PROJECT_ROOT_618G = Path(
        r"C:\RUL_Reliability_Research"
    )


assert PROJECT_ROOT_618G.exists(), (
    "Could not locate project root."
)


OUTPUT_DIR_618G = (
    PROJECT_ROOT_618G
    /
    "06_results"
    /
    "step6_reliability_signals"
)

OUTPUT_DIR_618G.mkdir(
    parents=True,
    exist_ok=True,
)


response_file_618g = (
    OUTPUT_DIR_618G
    /
    "fd001_pca_stuck_response_45_scenarios.csv"
)

sensor_file_618g = (
    OUTPUT_DIR_618G
    /
    "fd001_pca_stuck_sensor_summary.csv"
)

severity_file_618g = (
    OUTPUT_DIR_618G
    /
    "fd001_pca_stuck_severity_summary.csv"
)


stuck_pca_response_618g.to_csv(
    response_file_618g,
    index=False,
)

stuck_sensor_summary_618g.to_csv(
    sensor_file_618g,
)

stuck_severity_summary_618g.to_csv(
    severity_file_618g,
)


# ------------------------------------------------------------
# 14. Final confirmation
# ------------------------------------------------------------

print(
    "\nSaved:"
)

print(
    response_file_618g
)

print(
    sensor_file_618g
)

print(
    severity_file_618g
)


print(
    "\nStep 6.18G completed successfully."
)

print(
    "All 45 verified stuck-sensor scenarios were "
    "evaluated using the frozen PCA sensor-quality signal."
)

In [ ]:
# ============================================================
# Step 6.19A — Cross-corruption PCA result pre-check
#
# Purpose:
#   Verify the five frozen PCA-response experiment files
#   before any cross-corruption analysis is performed.
#
# Corruption families already completed:
#   1. Constant bias       : 90 scenarios
#   2. Additive noise      : 45 scenarios
#   3. Gradual drift       : 90 scenarios
#   4. Sensor dropout      : 45 scenarios
#   5. Stuck sensor        : 45 scenarios
#
# Expected total          : 315 scenarios
#
# IMPORTANT:
#   - READ ONLY
#   - No corruption generation
#   - No PCA fitting
#   - No CSV modification
#   - No result file is written
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


print(
    "=== Step 6.19A — Cross-corruption PCA result pre-check ==="
)


# ------------------------------------------------------------
# 1. Locate project root
# ------------------------------------------------------------

cwd_619a = Path.cwd()

if (cwd_619a / "06_results").exists():

    PROJECT_ROOT_619A = cwd_619a

elif (cwd_619a.parent / "06_results").exists():

    PROJECT_ROOT_619A = cwd_619a.parent

else:

    PROJECT_ROOT_619A = Path(
        r"C:\RUL_Reliability_Research"
    )


print(
    "\nProject root:",
    PROJECT_ROOT_619A
)

print(
    "Project root exists:",
    PROJECT_ROOT_619A.exists()
)


RESULT_DIR_619A = (
    PROJECT_ROOT_619A
    /
    "06_results"
    /
    "step6_reliability_signals"
)


print(
    "Step-6 result directory:",
    RESULT_DIR_619A
)

print(
    "Result directory exists:",
    RESULT_DIR_619A.exists()
)


# ------------------------------------------------------------
# 2. Define frozen experiment files
# ------------------------------------------------------------

FILE_SPECS_619A = {
    "bias": {
        "filename":
            "fd001_pca_bias_response_90_scenarios.csv",
        "expected_rows": 90,
        "expected_sensors": 15,
        "expected_severities": 3,
    },

    "noise": {
        "filename":
            "fd001_pca_noise_response_45_scenarios.csv",
        "expected_rows": 45,
        "expected_sensors": 15,
        "expected_severities": 3,
    },

    "drift": {
        "filename":
            "fd001_pca_drift_response_90_scenarios.csv",
        "expected_rows": 90,
        "expected_sensors": 15,
        "expected_severities": 3,
    },

    "dropout": {
        "filename":
            "fd001_pca_dropout_response_45_scenarios.csv",
        "expected_rows": 45,
        "expected_sensors": 15,
        "expected_severities": 3,
    },

    "stuck": {
        "filename":
            "fd001_pca_stuck_response_45_scenarios.csv",
        "expected_rows": 45,
        "expected_sensors": 15,
        "expected_severities": 3,
    },
}


# ------------------------------------------------------------
# 3. File-existence check
# ------------------------------------------------------------

file_check_records_619a = []

for corruption_619a, spec_619a in FILE_SPECS_619A.items():

    file_path_619a = (
        RESULT_DIR_619A
        /
        spec_619a["filename"]
    )

    file_check_records_619a.append(
        {
            "corruption": corruption_619a,
            "filename": spec_619a["filename"],
            "exists": file_path_619a.exists(),
            "expected_rows":
                spec_619a["expected_rows"],
        }
    )


file_check_df_619a = pd.DataFrame(
    file_check_records_619a
)


print(
    "\n=== Frozen result-file check ==="
)

display(
    file_check_df_619a
)


all_files_exist_619a = bool(
    file_check_df_619a["exists"].all()
)


print(
    "All five frozen result files exist:",
    all_files_exist_619a
)


# ------------------------------------------------------------
# 4. Load files only if all five exist
# ------------------------------------------------------------

loaded_results_619a = {}

if all_files_exist_619a:

    for corruption_619a, spec_619a in FILE_SPECS_619A.items():

        file_path_619a = (
            RESULT_DIR_619A
            /
            spec_619a["filename"]
        )

        loaded_results_619a[
            corruption_619a
        ] = pd.read_csv(
            file_path_619a
        )

else:

    print(
        "\nSTOP: At least one frozen result file "
        "is missing."
    )


# ------------------------------------------------------------
# 5. Structural summary
# ------------------------------------------------------------

structure_records_619a = []

if all_files_exist_619a:

    for corruption_619a, df_619a in loaded_results_619a.items():

        spec_619a = FILE_SPECS_619A[
            corruption_619a
        ]


        actual_rows_619a = len(
            df_619a
        )

        actual_sensors_619a = (
            df_619a["sensor"].nunique()
            if "sensor" in df_619a.columns
            else np.nan
        )

        actual_severities_619a = (
            df_619a["severity"].nunique()
            if "severity" in df_619a.columns
            else np.nan
        )

        duplicate_ids_619a = (
            int(
                df_619a[
                    "scenario_id"
                ].duplicated().sum()
            )
            if "scenario_id" in df_619a.columns
            else np.nan
        )


        structure_records_619a.append(
            {
                "corruption":
                    corruption_619a,

                "rows":
                    actual_rows_619a,

                "expected_rows":
                    spec_619a[
                        "expected_rows"
                    ],

                "rows_match":
                    actual_rows_619a
                    ==
                    spec_619a[
                        "expected_rows"
                    ],

                "unique_sensors":
                    actual_sensors_619a,

                "sensors_match":
                    actual_sensors_619a
                    ==
                    spec_619a[
                        "expected_sensors"
                    ],

                "unique_severities":
                    actual_severities_619a,

                "severities_match":
                    actual_severities_619a
                    ==
                    spec_619a[
                        "expected_severities"
                    ],

                "duplicate_scenario_ids":
                    duplicate_ids_619a,
            }
        )


structure_df_619a = pd.DataFrame(
    structure_records_619a
)


if all_files_exist_619a:

    print(
        "\n=== Structural verification ==="
    )

    display(
        structure_df_619a
    )


# ------------------------------------------------------------
# 6. Show exact columns in every file
# ------------------------------------------------------------

if all_files_exist_619a:

    print(
        "\n=== Columns in each frozen result file ==="
    )

    for corruption_619a, df_619a in loaded_results_619a.items():

        print(
            f"\n{corruption_619a.upper()}"
        )

        print(
            list(df_619a.columns)
        )


# ------------------------------------------------------------
# 7. Required common PCA-response columns
# ------------------------------------------------------------

REQUIRED_COMMON_COLUMNS_619A = [
    "scenario_id",
    "sensor",
    "severity",

    "clean_mean_re",
    "corrupted_mean_re",
    "mean_RE_change_pct",

    "clean_median_re",
    "corrupted_median_re",
    "median_RE_change_pct",

    "fraction_increased_pct",

    "clean_above_dev95_pct",
    "corrupted_above_dev95_pct",
    "above_dev95_delta_pp",

    "clean_mean_max_re",
    "corrupted_mean_max_re",
    "mean_max_RE_change_pct",
]


schema_records_619a = []

if all_files_exist_619a:

    for corruption_619a, df_619a in loaded_results_619a.items():

        missing_columns_619a = [
            column_619a
            for column_619a
            in REQUIRED_COMMON_COLUMNS_619A
            if column_619a
            not in df_619a.columns
        ]


        schema_records_619a.append(
            {
                "corruption":
                    corruption_619a,

                "required_columns_present":
                    len(
                        missing_columns_619a
                    )
                    == 0,

                "missing_columns":
                    (
                        ", ".join(
                            missing_columns_619a
                        )
                        if missing_columns_619a
                        else ""
                    ),
            }
        )


schema_df_619a = pd.DataFrame(
    schema_records_619a
)


if all_files_exist_619a:

    print(
        "\n=== Common PCA schema verification ==="
    )

    display(
        schema_df_619a
    )


# ------------------------------------------------------------
# 8. Severity-label check
# ------------------------------------------------------------

severity_records_619a = []

EXPECTED_SEVERITIES_619A = {
    "low",
    "medium",
    "high",
}


if all_files_exist_619a:

    for corruption_619a, df_619a in loaded_results_619a.items():

        if "severity" in df_619a.columns:

            observed_619a = set(
                df_619a[
                    "severity"
                ]
                .astype(str)
                .str.lower()
                .unique()
            )

        else:

            observed_619a = set()


        severity_records_619a.append(
            {
                "corruption":
                    corruption_619a,

                "observed_severities":
                    sorted(
                        observed_619a
                    ),

                "severity_labels_match":
                    observed_619a
                    ==
                    EXPECTED_SEVERITIES_619A,
            }
        )


severity_df_619a = pd.DataFrame(
    severity_records_619a
)


if all_files_exist_619a:

    print(
        "\n=== Severity-label verification ==="
    )

    display(
        severity_df_619a
    )


# ------------------------------------------------------------
# 9. Clean-reference consistency
#
# All five corruption experiments used the same clean
# validation windows and the same frozen PCA reference.
#
# Therefore the clean reference quantities should agree.
# ------------------------------------------------------------

CLEAN_REFERENCE_COLUMNS_619A = [
    "clean_mean_re",
    "clean_median_re",
    "clean_above_dev95_pct",
    "clean_mean_max_re",
]


reference_records_619a = []

reference_values_619a = {}


if all_files_exist_619a:

    for corruption_619a, df_619a in loaded_results_619a.items():

        for column_619a in CLEAN_REFERENCE_COLUMNS_619A:

            if column_619a in df_619a.columns:

                values_619a = pd.to_numeric(
                    df_619a[
                        column_619a
                    ],
                    errors="coerce",
                ).to_numpy(
                    dtype=float
                )


                finite_619a = bool(
                    np.all(
                        np.isfinite(
                            values_619a
                        )
                    )
                )


                if finite_619a:

                    min_value_619a = float(
                        np.min(
                            values_619a
                        )
                    )

                    max_value_619a = float(
                        np.max(
                            values_619a
                        )
                    )

                    within_file_constant_619a = bool(
                        np.allclose(
                            values_619a,
                            values_619a[0],
                            rtol=0.0,
                            atol=1e-12,
                        )
                    )

                    reference_values_619a[
                        (
                            corruption_619a,
                            column_619a,
                        )
                    ] = float(
                        values_619a[0]
                    )

                else:

                    min_value_619a = np.nan
                    max_value_619a = np.nan
                    within_file_constant_619a = False


                reference_records_619a.append(
                    {
                        "corruption":
                            corruption_619a,

                        "metric":
                            column_619a,

                        "min":
                            min_value_619a,

                        "max":
                            max_value_619a,

                        "constant_within_file":
                            within_file_constant_619a,
                    }
                )


reference_df_619a = pd.DataFrame(
    reference_records_619a
)


if all_files_exist_619a:

    print(
        "\n=== Clean-reference consistency within files ==="
    )

    display(
        reference_df_619a.round(12)
    )


# ------------------------------------------------------------
# 10. Cross-family clean-reference agreement
# ------------------------------------------------------------

cross_reference_records_619a = []

if all_files_exist_619a:

    for column_619a in CLEAN_REFERENCE_COLUMNS_619A:

        family_values_619a = []

        for corruption_619a in FILE_SPECS_619A:

            key_619a = (
                corruption_619a,
                column_619a,
            )

            if key_619a in reference_values_619a:

                family_values_619a.append(
                    reference_values_619a[
                        key_619a
                    ]
                )


        if len(family_values_619a) == 5:

            family_values_array_619a = np.asarray(
                family_values_619a,
                dtype=float,
            )

            all_match_619a = bool(
                np.allclose(
                    family_values_array_619a,
                    family_values_array_619a[0],
                    rtol=0.0,
                    atol=1e-12,
                )
            )

            min_value_619a = float(
                family_values_array_619a.min()
            )

            max_value_619a = float(
                family_values_array_619a.max()
            )

        else:

            all_match_619a = False
            min_value_619a = np.nan
            max_value_619a = np.nan


        cross_reference_records_619a.append(
            {
                "metric":
                    column_619a,

                "families_found":
                    len(
                        family_values_619a
                    ),

                "minimum":
                    min_value_619a,

                "maximum":
                    max_value_619a,

                "all_five_families_match":
                    all_match_619a,
            }
        )


cross_reference_df_619a = pd.DataFrame(
    cross_reference_records_619a
)


if all_files_exist_619a:

    print(
        "\n=== Clean-reference agreement across corruption families ==="
    )

    display(
        cross_reference_df_619a.round(12)
    )


# ------------------------------------------------------------
# 11. Key numerical-integrity check
# ------------------------------------------------------------

KEY_NUMERIC_COLUMNS_619A = [
    "clean_mean_re",
    "corrupted_mean_re",
    "mean_RE_change_pct",

    "clean_median_re",
    "corrupted_median_re",
    "median_RE_change_pct",

    "fraction_increased_pct",

    "clean_above_dev95_pct",
    "corrupted_above_dev95_pct",
    "above_dev95_delta_pp",

    "clean_mean_max_re",
    "corrupted_mean_max_re",
    "mean_max_RE_change_pct",
]


numeric_records_619a = []

if all_files_exist_619a:

    for corruption_619a, df_619a in loaded_results_619a.items():

        missing_numeric_619a = [
            column_619a
            for column_619a
            in KEY_NUMERIC_COLUMNS_619A
            if column_619a
            not in df_619a.columns
        ]


        if not missing_numeric_619a:

            numeric_matrix_619a = (
                df_619a[
                    KEY_NUMERIC_COLUMNS_619A
                ]
                .apply(
                    pd.to_numeric,
                    errors="coerce",
                )
                .to_numpy(
                    dtype=float
                )
            )

            all_finite_619a = bool(
                np.all(
                    np.isfinite(
                        numeric_matrix_619a
                    )
                )
            )

        else:

            all_finite_619a = False


        numeric_records_619a.append(
            {
                "corruption":
                    corruption_619a,

                "missing_numeric_columns":
                    (
                        ", ".join(
                            missing_numeric_619a
                        )
                        if missing_numeric_619a
                        else ""
                    ),

                "all_key_values_finite":
                    all_finite_619a,
            }
        )


numeric_df_619a = pd.DataFrame(
    numeric_records_619a
)


if all_files_exist_619a:

    print(
        "\n=== Numerical-integrity verification ==="
    )

    display(
        numeric_df_619a
    )


# ------------------------------------------------------------
# 12. Temporary in-memory cross-family preview
#
# This is NOT saved.
# ------------------------------------------------------------

combined_preview_619a = None

if (
    all_files_exist_619a
    and
    schema_df_619a[
        "required_columns_present"
    ].all()
):

    preview_frames_619a = []

    for corruption_619a, df_619a in loaded_results_619a.items():

        temp_619a = (
            df_619a[
                REQUIRED_COMMON_COLUMNS_619A
            ]
            .copy()
        )

        temp_619a.insert(
            0,
            "corruption",
            corruption_619a,
        )

        preview_frames_619a.append(
            temp_619a
        )


    combined_preview_619a = pd.concat(
        preview_frames_619a,
        ignore_index=True,
    )


    print(
        "\n=== Temporary in-memory combined preview ==="
    )

    print(
        "Total rows:",
        len(
            combined_preview_619a
        )
    )

    print(
        "Corruption families:",
        combined_preview_619a[
            "corruption"
        ].nunique()
    )

    print(
        "Unique scenario IDs:",
        combined_preview_619a[
            "scenario_id"
        ].nunique()
    )


    print(
        "\nScenario count by corruption:"
    )

    display(
        combined_preview_619a[
            "corruption"
        ]
        .value_counts()
        .rename(
            "scenarios"
        )
        .to_frame()
    )


# ------------------------------------------------------------
# 13. Overall pre-check decision
# ------------------------------------------------------------

checks_619a = []


checks_619a.append(
    all_files_exist_619a
)


if all_files_exist_619a:

    checks_619a.append(
        bool(
            structure_df_619a[
                "rows_match"
            ].all()
        )
    )

    checks_619a.append(
        bool(
            structure_df_619a[
                "sensors_match"
            ].all()
        )
    )

    checks_619a.append(
        bool(
            structure_df_619a[
                "severities_match"
            ].all()
        )
    )

    checks_619a.append(
        bool(
            (
                structure_df_619a[
                    "duplicate_scenario_ids"
                ]
                ==
                0
            ).all()
        )
    )

    checks_619a.append(
        bool(
            schema_df_619a[
                "required_columns_present"
            ].all()
        )
    )

    checks_619a.append(
        bool(
            severity_df_619a[
                "severity_labels_match"
            ].all()
        )
    )

    checks_619a.append(
        bool(
            cross_reference_df_619a[
                "all_five_families_match"
            ].all()
        )
    )

    checks_619a.append(
        bool(
            numeric_df_619a[
                "all_key_values_finite"
            ].all()
        )
    )


    if combined_preview_619a is not None:

        checks_619a.append(
            len(
                combined_preview_619a
            )
            ==
            315
        )

        checks_619a.append(
            combined_preview_619a[
                "corruption"
            ].nunique()
            ==
            5
        )

        checks_619a.append(
            combined_preview_619a[
                "scenario_id"
            ].nunique()
            ==
            315
        )


PRECHECK_PASSED_619A = bool(
    checks_619a
    and
    all(
        checks_619a
    )
)


print(
    "\n=============================================="
)

if PRECHECK_PASSED_619A:

    print(
        "STEP 6.19A PRE-CHECK PASSED"
    )

    print(
        "Five frozen PCA-response experiments are "
        "structurally compatible."
    )

    print(
        "Expected total verified scenarios: 315"
    )

    print(
        "No result file has been modified or saved."
    )

else:

    print(
        "STEP 6.19A PRE-CHECK DID NOT PASS"
    )

    print(
        "Do NOT continue to cross-corruption analysis yet."
    )

    print(
        "Inspect the tables above and identify the "
        "specific mismatch first."
    )

print(
    "=============================================="
)

In [ ]:
# ============================================================
# Step 6.19A-1 — Exact legacy schema inspection
#
# READ ONLY:
#   - Does not modify any DataFrame
#   - Does not write any file
#   - Does not rerun any experiment
# ============================================================

print(
    "=== Step 6.19A-1 — Exact legacy schema inspection ==="
)

for family_619a1 in ["bias", "noise"]:

    df_619a1 = loaded_results_619a[
        family_619a1
    ]

    print(
        "\n========================================"
    )

    print(
        f"{family_619a1.upper()} FILE"
    )

    print(
        "Shape:",
        df_619a1.shape
    )

    print(
        "\nExact columns:"
    )

    for i_619a1, column_619a1 in enumerate(
        df_619a1.columns,
        start=1
    ):
        print(
            f"{i_619a1:02d}. {column_619a1}"
        )

print(
    "\nStep 6.19A-1 completed."
)

In [ ]:
# ============================================================
# Step 6.19A-2 — Exact NOISE schema inspection
#
# READ ONLY
# Does not modify data or write files.
# ============================================================

print(
    "=== Step 6.19A-2 — Exact NOISE schema ==="
)

noise_619a2 = loaded_results_619a["noise"]

print(
    "Shape:",
    noise_619a2.shape
)

print(
    "\nExact columns:"
)

for i_619a2, col_619a2 in enumerate(
    noise_619a2.columns,
    start=1
):
    print(
        f"{i_619a2:02d}. {col_619a2}"
    )

print(
    "\nStep 6.19A-2 completed."
)

In [ ]:
# ============================================================
# Step 6.19B — Non-destructive cross-corruption harmonization
#
# PURPOSE
#   Harmonize the five frozen PCA-response result tables
#   WITHOUT modifying any original CSV or experiment result.
#
# IMPORTANT
#   - No corruption is generated.
#   - No frozen file is modified.
#   - loaded_results_619a is not modified.
#   - Legacy bias/noise column names are normalized only
#     inside copied DataFrames.
# ============================================================

import numpy as np
import pandas as pd

print(
    "=== Step 6.19B — Non-destructive schema harmonization ==="
)


# ------------------------------------------------------------
# 1. Verify the expected five corruption families
# ------------------------------------------------------------

expected_families_619b = [
    "bias",
    "noise",
    "drift",
    "dropout",
    "stuck",
]

missing_families_619b = [
    family
    for family in expected_families_619b
    if family not in loaded_results_619a
]

if missing_families_619b:
    raise RuntimeError(
        "Missing frozen result families: "
        + ", ".join(missing_families_619b)
    )

print("All five frozen result families are available.")


# ------------------------------------------------------------
# 2. Create completely independent copies
# ------------------------------------------------------------

harmonized_results_619b = {
    family: loaded_results_619a[family].copy(deep=True)
    for family in expected_families_619b
}

print(
    "Independent in-memory copies created."
)


# ------------------------------------------------------------
# 3. Standardize ONLY known legacy column names
# ------------------------------------------------------------

legacy_rename_map_619b = {
    "mean_re_change_pct": "mean_RE_change_pct",
    "median_re_change_pct": "median_RE_change_pct",
}

rename_report_619b = []

for family in expected_families_619b:

    df_619b = harmonized_results_619b[family]

    applied_renames_619b = {}

    for old_name_619b, new_name_619b in (
        legacy_rename_map_619b.items()
    ):

        if (
            old_name_619b in df_619b.columns
            and new_name_619b not in df_619b.columns
        ):
            applied_renames_619b[
                old_name_619b
            ] = new_name_619b

    if applied_renames_619b:

        df_619b = df_619b.rename(
            columns=applied_renames_619b
        )

    harmonized_results_619b[family] = df_619b

    rename_report_619b.append(
        {
            "corruption": family,
            "renamed_columns": (
                ", ".join(
                    [
                        f"{old}->{new}"
                        for old, new
                        in applied_renames_619b.items()
                    ]
                )
                if applied_renames_619b
                else "none"
            ),
        }
    )

rename_report_619b = pd.DataFrame(
    rename_report_619b
)

print(
    "\n=== Legacy-name normalization ==="
)

display(rename_report_619b)


# ------------------------------------------------------------
# 4. Add deterministic scenario IDs only where absent
# ------------------------------------------------------------

scenario_id_report_619b = []

for family in expected_families_619b:

    df_619b = (
        harmonized_results_619b[family]
        .reset_index(drop=True)
        .copy()
    )

    scenario_id_was_present_619b = (
        "scenario_id" in df_619b.columns
    )

    if not scenario_id_was_present_619b:

        family_prefix_619b = family.upper()

        df_619b.insert(
            0,
            "scenario_id",
            [
                (
                    f"FD001_{family_prefix_619b}_"
                    f"{row_number_619b:03d}"
                )
                for row_number_619b
                in range(
                    1,
                    len(df_619b) + 1
                )
            ],
        )

    duplicate_ids_619b = int(
        df_619b["scenario_id"].duplicated().sum()
    )

    harmonized_results_619b[family] = df_619b

    scenario_id_report_619b.append(
        {
            "corruption": family,
            "rows": len(df_619b),
            "original_id_present":
                scenario_id_was_present_619b,
            "generated_id":
                not scenario_id_was_present_619b,
            "duplicate_ids":
                duplicate_ids_619b,
        }
    )

scenario_id_report_619b = pd.DataFrame(
    scenario_id_report_619b
)

print(
    "\n=== Scenario-ID harmonization ==="
)

display(scenario_id_report_619b)


# ------------------------------------------------------------
# 5. Define the scientifically common result schema
#
# These metrics are genuinely comparable across all five
# corruption families.
# ------------------------------------------------------------

common_columns_619b = [
    "scenario_id",
    "sensor",
    "severity",
    "clean_mean_re",
    "corrupted_mean_re",
    "mean_RE_change_pct",
    "median_RE_change_pct",
    "fraction_increased_pct",
    "clean_above_dev95_pct",
    "corrupted_above_dev95_pct",
    "above_dev95_delta_pp",
]


# ------------------------------------------------------------
# 6. Verify common-column availability
# ------------------------------------------------------------

column_check_records_619b = []

for family in expected_families_619b:

    df_619b = harmonized_results_619b[family]

    missing_columns_619b = [
        column
        for column in common_columns_619b
        if column not in df_619b.columns
    ]

    column_check_records_619b.append(
        {
            "corruption": family,
            "rows": len(df_619b),
            "common_columns_present":
                len(missing_columns_619b) == 0,
            "missing_columns": (
                ", ".join(missing_columns_619b)
                if missing_columns_619b
                else ""
            ),
        }
    )

column_check_619b = pd.DataFrame(
    column_check_records_619b
)

print(
    "\n=== Common scientific-schema verification ==="
)

display(column_check_619b)

if not column_check_619b[
    "common_columns_present"
].all():

    raise RuntimeError(
        "Step 6.19B stopped because at least one "
        "family is missing a required common metric."
    )


# ------------------------------------------------------------
# 7. Verify numeric integrity of common PCA metrics
# ------------------------------------------------------------

common_numeric_columns_619b = [
    "clean_mean_re",
    "corrupted_mean_re",
    "mean_RE_change_pct",
    "median_RE_change_pct",
    "fraction_increased_pct",
    "clean_above_dev95_pct",
    "corrupted_above_dev95_pct",
    "above_dev95_delta_pp",
]

numeric_check_records_619b = []

for family in expected_families_619b:

    df_619b = harmonized_results_619b[family]

    nonfinite_total_619b = 0

    for column_619b in common_numeric_columns_619b:

        values_619b = pd.to_numeric(
            df_619b[column_619b],
            errors="coerce",
        ).to_numpy(
            dtype=float
        )

        nonfinite_total_619b += int(
            (~np.isfinite(values_619b)).sum()
        )

    numeric_check_records_619b.append(
        {
            "corruption": family,
            "rows": len(df_619b),
            "nonfinite_common_values":
                nonfinite_total_619b,
            "numeric_integrity_passed":
                nonfinite_total_619b == 0,
        }
    )

numeric_check_619b = pd.DataFrame(
    numeric_check_records_619b
)

print(
    "\n=== Numerical-integrity verification ==="
)

display(numeric_check_619b)

if not numeric_check_619b[
    "numeric_integrity_passed"
].all():

    raise RuntimeError(
        "Step 6.19B stopped because non-finite "
        "values were found in common PCA metrics."
    )


# ------------------------------------------------------------
# 8. Correct clean-reference consistency check
#
# We compare ONLY clean metrics actually available in
# ALL five frozen result families.
# ------------------------------------------------------------

shared_clean_metrics_619b = [
    "clean_mean_re",
    "clean_above_dev95_pct",
]

clean_reference_records_619b = []

for metric_619b in shared_clean_metrics_619b:

    family_values_619b = {}

    for family in expected_families_619b:

        df_619b = harmonized_results_619b[family]

        values_619b = pd.to_numeric(
            df_619b[metric_619b],
            errors="raise",
        ).to_numpy(
            dtype=float
        )

        metric_min_619b = float(
            np.min(values_619b)
        )

        metric_max_619b = float(
            np.max(values_619b)
        )

        constant_within_file_619b = bool(
            np.isclose(
                metric_min_619b,
                metric_max_619b,
                rtol=0.0,
                atol=1e-12,
            )
        )

        if not constant_within_file_619b:
            raise RuntimeError(
                f"{family}: {metric_619b} is not "
                "constant within the frozen result file."
            )

        family_values_619b[family] = (
            metric_min_619b
        )

    all_values_619b = np.array(
        list(
            family_values_619b.values()
        ),
        dtype=float,
    )

    families_agree_619b = bool(
        np.allclose(
            all_values_619b,
            all_values_619b[0],
            rtol=0.0,
            atol=1e-12,
        )
    )

    clean_reference_records_619b.append(
        {
            "metric": metric_619b,
            "bias":
                family_values_619b["bias"],
            "noise":
                family_values_619b["noise"],
            "drift":
                family_values_619b["drift"],
            "dropout":
                family_values_619b["dropout"],
            "stuck":
                family_values_619b["stuck"],
            "all_five_match":
                families_agree_619b,
        }
    )

clean_reference_check_619b = pd.DataFrame(
    clean_reference_records_619b
)

print(
    "\n=== Shared clean-reference agreement ==="
)

display(clean_reference_check_619b)

if not clean_reference_check_619b[
    "all_five_match"
].all():

    raise RuntimeError(
        "Step 6.19B stopped because the shared "
        "clean PCA reference differs across families."
    )


# ------------------------------------------------------------
# 9. Build one common cross-corruption analysis table
# ------------------------------------------------------------

combined_parts_619b = []

for family in expected_families_619b:

    part_619b = (
        harmonized_results_619b[family][
            common_columns_619b
        ]
        .copy()
    )

    part_619b.insert(
        0,
        "corruption",
        family,
    )

    combined_parts_619b.append(
        part_619b
    )

cross_corruption_pca_619b = pd.concat(
    combined_parts_619b,
    axis=0,
    ignore_index=True,
)


# ------------------------------------------------------------
# 10. Final structural verification
# ------------------------------------------------------------

expected_rows_by_family_619b = {
    "bias": 90,
    "noise": 45,
    "drift": 90,
    "dropout": 45,
    "stuck": 45,
}

expected_total_rows_619b = sum(
    expected_rows_by_family_619b.values()
)

family_counts_619b = (
    cross_corruption_pca_619b
    .groupby(
        "corruption",
        sort=False
    )
    .size()
    .rename("rows")
    .reset_index()
)

family_counts_619b[
    "expected_rows"
] = family_counts_619b[
    "corruption"
].map(
    expected_rows_by_family_619b
)

family_counts_619b[
    "rows_match"
] = (
    family_counts_619b["rows"]
    == family_counts_619b["expected_rows"]
)

print(
    "\n=== Harmonized cross-corruption row counts ==="
)

display(family_counts_619b)

if len(cross_corruption_pca_619b) != (
    expected_total_rows_619b
):
    raise RuntimeError(
        "Unexpected total number of harmonized rows."
    )

if not family_counts_619b[
    "rows_match"
].all():
    raise RuntimeError(
        "At least one corruption family has an "
        "unexpected number of rows."
    )


# ------------------------------------------------------------
# 11. Verify uniqueness of family + scenario ID
# ------------------------------------------------------------

duplicate_combined_ids_619b = int(
    cross_corruption_pca_619b[
        [
            "corruption",
            "scenario_id",
        ]
    ]
    .duplicated()
    .sum()
)

if duplicate_combined_ids_619b != 0:
    raise RuntimeError(
        "Duplicate corruption/scenario_id pairs found."
    )


# ------------------------------------------------------------
# 12. Compact final report
# ------------------------------------------------------------

print(
    "\n=== Harmonized dataset summary ==="
)

print(
    "Total rows:",
    len(cross_corruption_pca_619b)
)

print(
    "Corruption families:",
    cross_corruption_pca_619b[
        "corruption"
    ].nunique()
)

print(
    "Unique sensors:",
    cross_corruption_pca_619b[
        "sensor"
    ].nunique()
)

print(
    "Severity labels:",
    sorted(
        cross_corruption_pca_619b[
            "severity"
        ].unique()
    )
)

print(
    "Duplicate family/scenario IDs:",
    duplicate_combined_ids_619b
)

print(
    "\nCommon columns:"
)

for column_619b in (
    cross_corruption_pca_619b.columns
):
    print(
        " -",
        column_619b
    )


print(
    "\nStep 6.19B completed successfully."
)

print(
    "The five frozen PCA-response result families "
    "are now harmonized IN MEMORY only."
)

print(
    "No original CSV file was modified."
)

In [ ]:
# ============================================================
# Step 6.19C — Balanced descriptive cross-corruption comparison
#
# PURPOSE
#   Compare PCA sensor-quality response across the five
#   corruption families after Step 6.19B harmonization.
#
# IMPORTANT
#   Bias and drift contain two directions per sensor/severity.
#   Therefore:
#       1. first average within corruption × sensor × severity
#       2. then summarize corruption families
#
#   This prevents directional corruptions from receiving
#   double weight merely because they contain twice as many
#   scenario rows.
#
#   No original experiment file is modified.
# ============================================================

import numpy as np
import pandas as pd

print(
    "=== Step 6.19C — Balanced cross-corruption PCA comparison ==="
)


# ------------------------------------------------------------
# 1. Verify required Step-6.19B object
# ------------------------------------------------------------

if "cross_corruption_pca_619b" not in globals():
    raise RuntimeError(
        "cross_corruption_pca_619b is not available. "
        "Run Step 6.19B first."
    )

cross_df_619c = cross_corruption_pca_619b.copy(deep=True)

required_columns_619c = [
    "corruption",
    "scenario_id",
    "sensor",
    "severity",
    "clean_mean_re",
    "corrupted_mean_re",
    "mean_RE_change_pct",
    "median_RE_change_pct",
    "fraction_increased_pct",
    "clean_above_dev95_pct",
    "corrupted_above_dev95_pct",
    "above_dev95_delta_pp",
]

missing_columns_619c = [
    column
    for column in required_columns_619c
    if column not in cross_df_619c.columns
]

if missing_columns_619c:
    raise RuntimeError(
        "Missing required columns: "
        + ", ".join(missing_columns_619c)
    )

print("Required harmonized columns are available.")
print("Input scenarios:", len(cross_df_619c))


# ------------------------------------------------------------
# 2. Fix categorical ordering for display only
# ------------------------------------------------------------

family_order_619c = [
    "bias",
    "noise",
    "drift",
    "dropout",
    "stuck",
]

severity_order_619c = [
    "low",
    "medium",
    "high",
]

cross_df_619c["corruption"] = pd.Categorical(
    cross_df_619c["corruption"],
    categories=family_order_619c,
    ordered=True,
)

cross_df_619c["severity"] = pd.Categorical(
    cross_df_619c["severity"],
    categories=severity_order_619c,
    ordered=True,
)


# ------------------------------------------------------------
# 3. Collapse direction / repeated scenario variants
#
# One row now represents:
# corruption × sensor × severity
#
# Expected:
# 5 families × 15 sensors × 3 severities = 225 rows
# ------------------------------------------------------------

balanced_sensor_severity_619c = (
    cross_df_619c
    .groupby(
        [
            "corruption",
            "sensor",
            "severity",
        ],
        observed=True,
        sort=False,
    )
    .agg(
        source_scenarios=(
            "scenario_id",
            "count",
        ),
        clean_mean_re=(
            "clean_mean_re",
            "mean",
        ),
        corrupted_mean_re=(
            "corrupted_mean_re",
            "mean",
        ),
        mean_RE_change_pct=(
            "mean_RE_change_pct",
            "mean",
        ),
        median_RE_change_pct=(
            "median_RE_change_pct",
            "mean",
        ),
        fraction_increased_pct=(
            "fraction_increased_pct",
            "mean",
        ),
        clean_above_dev95_pct=(
            "clean_above_dev95_pct",
            "mean",
        ),
        corrupted_above_dev95_pct=(
            "corrupted_above_dev95_pct",
            "mean",
        ),
        above_dev95_delta_pp=(
            "above_dev95_delta_pp",
            "mean",
        ),
    )
    .reset_index()
)

expected_balanced_rows_619c = (
    5 * 15 * 3
)

if len(balanced_sensor_severity_619c) != (
    expected_balanced_rows_619c
):
    raise RuntimeError(
        "Unexpected number of balanced "
        "corruption/sensor/severity rows. "
        f"Expected {expected_balanced_rows_619c}, "
        f"found {len(balanced_sensor_severity_619c)}."
    )

print(
    "\nBalanced corruption × sensor × severity rows:",
    len(balanced_sensor_severity_619c),
)


# ------------------------------------------------------------
# 4. Confirm source-scenario structure
#
# Bias/drift should normally contribute 2 directional
# scenarios per sensor/severity.
# Other families should contribute 1.
# ------------------------------------------------------------

scenario_structure_619c = (
    balanced_sensor_severity_619c
    .groupby(
        "corruption",
        observed=True,
        sort=False,
    )
    ["source_scenarios"]
    .agg(
        minimum="min",
        maximum="max",
        mean="mean",
    )
    .reset_index()
)

print(
    "\n=== Source scenarios per balanced "
    "sensor/severity cell ==="
)

display(scenario_structure_619c)


# ------------------------------------------------------------
# 5. FAMILY-LEVEL balanced summary
#
# Every family now contributes exactly:
# 15 sensors × 3 severities = 45 balanced cells.
# ------------------------------------------------------------

family_summary_619c = (
    balanced_sensor_severity_619c
    .groupby(
        "corruption",
        observed=True,
        sort=False,
    )
    .agg(
        balanced_cells=(
            "sensor",
            "size",
        ),
        mean_RE_change_pct=(
            "mean_RE_change_pct",
            "mean",
        ),
        median_RE_change_pct=(
            "median_RE_change_pct",
            "mean",
        ),
        mean_fraction_increased_pct=(
            "fraction_increased_pct",
            "mean",
        ),
        mean_corrupted_above_dev95_pct=(
            "corrupted_above_dev95_pct",
            "mean",
        ),
        mean_above_dev95_delta_pp=(
            "above_dev95_delta_pp",
            "mean",
        ),
    )
    .reset_index()
)

if not (
    family_summary_619c["balanced_cells"] == 45
).all():
    raise RuntimeError(
        "At least one corruption family does not "
        "contain exactly 45 balanced cells."
    )

family_summary_619c = (
    family_summary_619c
    .sort_values(
        "mean_RE_change_pct",
        ascending=False,
    )
    .reset_index(drop=True)
)

print(
    "\n=== Balanced PCA response by corruption family ==="
)

display(
    family_summary_619c.round(3)
)


# ------------------------------------------------------------
# 6. FAMILY × SEVERITY summary
#
# Each row contains 15 sensors.
# ------------------------------------------------------------

family_severity_summary_619c = (
    balanced_sensor_severity_619c
    .groupby(
        [
            "corruption",
            "severity",
        ],
        observed=True,
        sort=False,
    )
    .agg(
        sensors=(
            "sensor",
            "nunique",
        ),
        mean_RE_change_pct=(
            "mean_RE_change_pct",
            "mean",
        ),
        median_RE_change_pct=(
            "median_RE_change_pct",
            "mean",
        ),
        mean_fraction_increased_pct=(
            "fraction_increased_pct",
            "mean",
        ),
        mean_corrupted_above_dev95_pct=(
            "corrupted_above_dev95_pct",
            "mean",
        ),
        mean_above_dev95_delta_pp=(
            "above_dev95_delta_pp",
            "mean",
        ),
    )
    .reset_index()
)

if not (
    family_severity_summary_619c["sensors"] == 15
).all():
    raise RuntimeError(
        "At least one family/severity combination "
        "does not contain all 15 sensors."
    )

print(
    "\n=== Balanced PCA response by "
    "corruption family and severity ==="
)

display(
    family_severity_summary_619c.round(3)
)


# ------------------------------------------------------------
# 7. Severity monotonicity check
#
# This is descriptive only.
# We check whether the family-average PCA response increases
# from LOW -> MEDIUM -> HIGH.
# ------------------------------------------------------------

monotonicity_records_619c = []

for family_619c in family_order_619c:

    temp_619c = (
        family_severity_summary_619c[
            family_severity_summary_619c[
                "corruption"
            ] == family_619c
        ]
        .set_index("severity")
        .reindex(severity_order_619c)
    )

    mean_re_values_619c = (
        temp_619c[
            "mean_RE_change_pct"
        ]
        .to_numpy(dtype=float)
    )

    threshold_delta_values_619c = (
        temp_619c[
            "mean_above_dev95_delta_pp"
        ]
        .to_numpy(dtype=float)
    )

    mean_re_monotonic_619c = bool(
        np.all(
            np.diff(
                mean_re_values_619c
            ) >= -1e-12
        )
    )

    threshold_monotonic_619c = bool(
        np.all(
            np.diff(
                threshold_delta_values_619c
            ) >= -1e-12
        )
    )

    monotonicity_records_619c.append(
        {
            "corruption": family_619c,
            "mean_RE_change_monotonic":
                mean_re_monotonic_619c,
            "above_dev95_delta_monotonic":
                threshold_monotonic_619c,
        }
    )

severity_monotonicity_619c = pd.DataFrame(
    monotonicity_records_619c
)

print(
    "\n=== Severity-response monotonicity ==="
)

display(severity_monotonicity_619c)


# ------------------------------------------------------------
# 8. SENSOR robustness across corruption families
#
# For each sensor:
# average each corruption family first,
# then average equally across the five families.
#
# This avoids giving bias/drift extra weight.
# ------------------------------------------------------------

family_sensor_summary_619c = (
    balanced_sensor_severity_619c
    .groupby(
        [
            "corruption",
            "sensor",
        ],
        observed=True,
        sort=False,
    )
    .agg(
        severity_levels=(
            "severity",
            "nunique",
        ),
        mean_RE_change_pct=(
            "mean_RE_change_pct",
            "mean",
        ),
        mean_fraction_increased_pct=(
            "fraction_increased_pct",
            "mean",
        ),
        mean_above_dev95_delta_pp=(
            "above_dev95_delta_pp",
            "mean",
        ),
    )
    .reset_index()
)

sensor_cross_family_619c = (
    family_sensor_summary_619c
    .groupby(
        "sensor",
        sort=False,
    )
    .agg(
        corruption_families=(
            "corruption",
            "nunique",
        ),
        cross_family_mean_RE_change_pct=(
            "mean_RE_change_pct",
            "mean",
        ),
        cross_family_median_RE_change_pct=(
            "mean_RE_change_pct",
            "median",
        ),
        cross_family_min_RE_change_pct=(
            "mean_RE_change_pct",
            "min",
        ),
        cross_family_max_RE_change_pct=(
            "mean_RE_change_pct",
            "max",
        ),
        cross_family_mean_above_dev95_delta_pp=(
            "mean_above_dev95_delta_pp",
            "mean",
        ),
    )
    .reset_index()
)

if not (
    sensor_cross_family_619c[
        "corruption_families"
    ] == 5
).all():
    raise RuntimeError(
        "At least one sensor is missing from one or "
        "more corruption families."
    )

sensor_cross_family_619c = (
    sensor_cross_family_619c
    .sort_values(
        "cross_family_mean_RE_change_pct",
        ascending=False,
    )
    .reset_index(drop=True)
)

print(
    "\n=== Sensor-wise PCA sensitivity "
    "across all five corruption families ==="
)

display(
    sensor_cross_family_619c.round(3)
)


# ------------------------------------------------------------
# 9. Create corruption × sensor response matrix
#
# Values are mean PCA-RE percentage changes,
# averaged across the three severity levels within
# each family.
# ------------------------------------------------------------

sensor_matrix_619c = (
    family_sensor_summary_619c
    .pivot(
        index="sensor",
        columns="corruption",
        values="mean_RE_change_pct",
    )
    .reindex(
        columns=family_order_619c
    )
)

sensor_matrix_619c[
    "equal_family_mean"
] = sensor_matrix_619c.mean(
    axis=1
)

sensor_matrix_619c = (
    sensor_matrix_619c
    .sort_values(
        "equal_family_mean",
        ascending=False,
    )
)

print(
    "\n=== Corruption × sensor PCA-response matrix ==="
)

display(
    sensor_matrix_619c.round(3)
)


# ------------------------------------------------------------
# 10. Final integrity summary
# ------------------------------------------------------------

print(
    "\n=== Step 6.19C integrity summary ==="
)

print(
    "Original harmonized scenarios :",
    len(cross_df_619c),
)

print(
    "Balanced sensor/severity cells:",
    len(balanced_sensor_severity_619c),
)

print(
    "Corruption families           :",
    balanced_sensor_severity_619c[
        "corruption"
    ].nunique(),
)

print(
    "Sensors                       :",
    balanced_sensor_severity_619c[
        "sensor"
    ].nunique(),
)

print(
    "Severity levels               :",
    balanced_sensor_severity_619c[
        "severity"
    ].nunique(),
)

print(
    "\nStep 6.19C completed successfully."
)

print(
    "Cross-corruption PCA responses were compared "
    "using balanced family weighting."
)

print(
    "No frozen result file was modified."
)

In [ ]:
# ============================================================
# Step 6.19C-1 — Compact review of cross-corruption results
# DISPLAY ONLY — no experiment is rerun and nothing is modified
# ============================================================

required_objects_619c1 = [
    "scenario_structure_619c",
    "family_summary_619c",
    "family_severity_summary_619c",
    "severity_monotonicity_619c",
    "sensor_cross_family_619c",
    "sensor_matrix_619c",
]

missing_objects_619c1 = [
    name
    for name in required_objects_619c1
    if name not in globals()
]

if missing_objects_619c1:
    raise RuntimeError(
        "Missing Step 6.19C objects: "
        + ", ".join(missing_objects_619c1)
    )

print("=== 1. Source-scenario structure ===")
display(scenario_structure_619c.round(3))

print("\n=== 2. Balanced PCA response by corruption family ===")
display(family_summary_619c.round(3))

print("\n=== 3. Balanced PCA response by family and severity ===")
display(family_severity_summary_619c.round(3))

print("\n=== 4. Severity-response monotonicity ===")
display(severity_monotonicity_619c)

print("\n=== 5. Cross-family sensor sensitivity ranking ===")
display(sensor_cross_family_619c.round(3))

print("\n=== 6. Corruption × sensor PCA-response matrix ===")
display(sensor_matrix_619c.round(3))

print("\nStep 6.19C-1 completed.")
print("Display only — no data or frozen artifact was modified.")

In [ ]:
# ============================================================
# Step 6.19C-2 — Final severity monotonicity review
# DISPLAY ONLY
# ============================================================

if "severity_monotonicity_619c" not in globals():
    raise RuntimeError(
        "severity_monotonicity_619c is not available. "
        "Run Step 6.19C first."
    )

print(
    "=== Step 6.19C-2 — "
    "Severity monotonicity of PCA response ==="
)

display(
    severity_monotonicity_619c
    .copy()
    .reset_index(drop=True)
)

print(
    "\nStep 6.19C-2 completed. "
    "Display only — nothing was modified."
)

In [ ]:
# ============================================================
# Step 6.20A — Temporal-quality-signal pre-check
#
# PURPOSE
# -------
# 1. Verify all frozen objects required for a temporal signal.
# 2. Inspect the frozen temporal-reference statistics.
# 3. Verify retained-sensor <-> sequence-feature mapping.
# 4. Verify that raw validation inputs can be reconstructed
#    safely from the frozen scaled validation DataFrame.
#
# IMPORTANT
# ---------
# - No corruption is generated.
# - No reliability score is created.
# - No frozen result file is modified.
# ============================================================

import numpy as np
import pandas as pd

print(
    "=== Step 6.20A — "
    "Temporal-quality-signal pre-check ==="
)

# ------------------------------------------------------------
# 1. Required frozen in-memory objects
# ------------------------------------------------------------

required_objects_620a = [
    "sensor_temporal_stats",
    "sensor_reference_stats",
    "val_scaled",
    "scaler",
    "PRIMARY_INPUT_COLUMNS",
    "RETAINED_SENSOR_COLUMNS",
    "SENSOR_INPUT_INDICES",
    "X_val_clean",
    "y_val_clean",
    "val_seq_meta",
]

availability_620a = {
    name: name in globals()
    for name in required_objects_620a
}

print("\n=== Required in-memory objects ===")

for name, available in availability_620a.items():
    print(
        f"{name:<28}: {available}"
    )

missing_objects_620a = [
    name
    for name, available in availability_620a.items()
    if not available
]

if missing_objects_620a:
    raise RuntimeError(
        "Missing required Step-6 objects: "
        + ", ".join(missing_objects_620a)
    )

print(
    "\nAll required frozen objects are available."
)


# ------------------------------------------------------------
# 2. Inspect temporal-reference statistics
# ------------------------------------------------------------

if not isinstance(
    sensor_temporal_stats,
    pd.DataFrame
):
    raise TypeError(
        "sensor_temporal_stats must be a pandas DataFrame."
    )

required_temporal_columns_620a = [
    "diff_std",
    "median_abs_diff",
    "q95_abs_diff",
]

missing_temporal_columns_620a = [
    column
    for column in required_temporal_columns_620a
    if column not in sensor_temporal_stats.columns
]

if missing_temporal_columns_620a:
    raise RuntimeError(
        "sensor_temporal_stats is missing columns: "
        + ", ".join(missing_temporal_columns_620a)
    )

print(
    "\n=== Frozen temporal-reference statistics ==="
)

print(
    "Shape:",
    sensor_temporal_stats.shape
)

print(
    "Columns:",
    list(sensor_temporal_stats.columns)
)

print(
    "Index name:",
    sensor_temporal_stats.index.name
)

print(
    "Duplicate sensor indices:",
    int(
        sensor_temporal_stats.index
        .duplicated()
        .sum()
    )
)

print(
    "Missing values:",
    int(
        sensor_temporal_stats[
            required_temporal_columns_620a
        ]
        .isna()
        .sum()
        .sum()
    )
)

print(
    "Non-finite values:",
    int(
        (
            ~np.isfinite(
                sensor_temporal_stats[
                    required_temporal_columns_620a
                ].to_numpy(dtype=float)
            )
        ).sum()
    )
)


# ------------------------------------------------------------
# 3. Verify retained-sensor coverage
# ------------------------------------------------------------

retained_sensors_620a = list(
    RETAINED_SENSOR_COLUMNS
)

temporal_sensor_index_620a = list(
    sensor_temporal_stats.index
)

missing_from_temporal_stats_620a = sorted(
    set(retained_sensors_620a)
    - set(temporal_sensor_index_620a)
)

extra_in_temporal_stats_620a = sorted(
    set(temporal_sensor_index_620a)
    - set(retained_sensors_620a)
)

print(
    "\n=== Retained-sensor coverage ==="
)

print(
    "Retained sensors:",
    len(retained_sensors_620a)
)

print(
    "Sensors in temporal stats:",
    len(temporal_sensor_index_620a)
)

print(
    "Missing retained sensors:",
    missing_from_temporal_stats_620a
)

print(
    "Unexpected temporal-stat sensors:",
    extra_in_temporal_stats_620a
)

if missing_from_temporal_stats_620a:
    raise RuntimeError(
        "Some retained sensors are missing from "
        "sensor_temporal_stats."
    )


# ------------------------------------------------------------
# 4. Verify sequence-feature mapping
# ------------------------------------------------------------

primary_columns_620a = list(
    PRIMARY_INPUT_COLUMNS
)

sensor_indices_620a = np.asarray(
    SENSOR_INPUT_INDICES,
    dtype=int
)

if sensor_indices_620a.ndim != 1:
    raise RuntimeError(
        "SENSOR_INPUT_INDICES must be one-dimensional."
    )

if len(sensor_indices_620a) != len(
    retained_sensors_620a
):
    raise RuntimeError(
        "Number of sensor input indices does not match "
        "number of retained sensors."
    )

if len(
    np.unique(sensor_indices_620a)
) != len(sensor_indices_620a):
    raise RuntimeError(
        "SENSOR_INPUT_INDICES contains duplicates."
    )

if np.any(sensor_indices_620a < 0):
    raise RuntimeError(
        "SENSOR_INPUT_INDICES contains a negative index."
    )

if np.any(
    sensor_indices_620a
    >= len(primary_columns_620a)
):
    raise RuntimeError(
        "SENSOR_INPUT_INDICES contains an out-of-range index."
    )

mapped_sensor_names_620a = [
    primary_columns_620a[index]
    for index in sensor_indices_620a
]

sensor_mapping_620a = pd.DataFrame({
    "retained_sensor":
        retained_sensors_620a,
    "sequence_feature_index":
        sensor_indices_620a,
    "mapped_input_column":
        mapped_sensor_names_620a,
})

sensor_mapping_620a[
    "name_matches"
] = (
    sensor_mapping_620a[
        "retained_sensor"
    ].to_numpy()
    ==
    sensor_mapping_620a[
        "mapped_input_column"
    ].to_numpy()
)

print(
    "\n=== Retained-sensor to sequence-feature mapping ==="
)

display(
    sensor_mapping_620a
)

print(
    "Exact ordered mapping:",
    bool(
        sensor_mapping_620a[
            "name_matches"
        ].all()
    )
)

if not sensor_mapping_620a[
    "name_matches"
].all():
    raise RuntimeError(
        "Retained-sensor ordering does not match "
        "SENSOR_INPUT_INDICES."
    )


# ------------------------------------------------------------
# 5. Verify clean sequence dimensions
# ------------------------------------------------------------

X_val_clean_620a = np.asarray(
    X_val_clean
)

y_val_clean_620a = np.asarray(
    y_val_clean
)

if X_val_clean_620a.ndim != 3:
    raise RuntimeError(
        "X_val_clean must be a 3-D sequence array."
    )

print(
    "\n=== Clean validation sequence structure ==="
)

print(
    "X_val_clean shape :",
    X_val_clean_620a.shape
)

print(
    "y_val_clean shape :",
    y_val_clean_620a.shape
)

print(
    "val_seq_meta shape:",
    val_seq_meta.shape
)

print(
    "Primary inputs:",
    len(primary_columns_620a)
)

print(
    "Sensor inputs:",
    len(sensor_indices_620a)
)

if (
    X_val_clean_620a.shape[0]
    != len(y_val_clean_620a)
):
    raise RuntimeError(
        "X_val_clean and y_val_clean are not aligned."
    )

if (
    X_val_clean_620a.shape[0]
    != len(val_seq_meta)
):
    raise RuntimeError(
        "X_val_clean and val_seq_meta are not aligned."
    )

if (
    X_val_clean_620a.shape[2]
    != len(primary_columns_620a)
):
    raise RuntimeError(
        "Sequence feature dimension does not match "
        "PRIMARY_INPUT_COLUMNS."
    )


# ------------------------------------------------------------
# 6. Verify scaler feature order
# ------------------------------------------------------------

print(
    "\n=== Frozen scaler compatibility ==="
)

if hasattr(
    scaler,
    "feature_names_in_"
):
    scaler_features_620a = list(
        scaler.feature_names_in_
    )

    print(
        "Scaler feature count:",
        len(scaler_features_620a)
    )

    print(
        "Scaler feature order matches "
        "PRIMARY_INPUT_COLUMNS:",
        scaler_features_620a
        == primary_columns_620a
    )

    if (
        scaler_features_620a
        != primary_columns_620a
    ):
        raise RuntimeError(
            "Scaler feature order does not match "
            "PRIMARY_INPUT_COLUMNS."
        )

else:
    print(
        "Scaler has no feature_names_in_ attribute."
    )

    if getattr(
        scaler,
        "n_features_in_",
        None
    ) != len(primary_columns_620a):
        raise RuntimeError(
            "Scaler feature count does not match "
            "PRIMARY_INPUT_COLUMNS."
        )

    print(
        "Scaler feature count matches "
        "PRIMARY_INPUT_COLUMNS."
    )


# ------------------------------------------------------------
# 7. Reconstruct raw validation PRIMARY inputs safely
# ------------------------------------------------------------

scaled_primary_620a = (
    val_scaled[
        primary_columns_620a
    ]
    .copy()
)

raw_primary_values_620a = (
    scaler.inverse_transform(
        scaled_primary_620a
    )
)

raw_primary_620a = pd.DataFrame(
    raw_primary_values_620a,
    index=scaled_primary_620a.index,
    columns=primary_columns_620a,
)

roundtrip_scaled_620a = pd.DataFrame(
    scaler.transform(
        raw_primary_620a
    ),
    index=scaled_primary_620a.index,
    columns=primary_columns_620a,
)

max_roundtrip_difference_620a = float(
    np.max(
        np.abs(
            roundtrip_scaled_620a.to_numpy()
            -
            scaled_primary_620a.to_numpy()
        )
    )
)

print(
    "\n=== Raw-validation reconstruction check ==="
)

print(
    "Scaled primary shape:",
    scaled_primary_620a.shape
)

print(
    "Reconstructed raw primary shape:",
    raw_primary_620a.shape
)

print(
    "Maximum scaling round-trip difference:",
    max_roundtrip_difference_620a
)

if not np.isfinite(
    max_roundtrip_difference_620a
):
    raise RuntimeError(
        "Scaling round-trip difference is non-finite."
    )

if (
    max_roundtrip_difference_620a
    > 1e-10
):
    raise RuntimeError(
        "Scaling round-trip difference is unexpectedly large."
    )

print(
    "Raw validation PRIMARY inputs can be "
    "reconstructed safely."
)


# ------------------------------------------------------------
# 8. Display frozen temporal statistics
# ------------------------------------------------------------

print(
    "\n=== Frozen sensor temporal statistics ==="
)

display(
    sensor_temporal_stats[
        required_temporal_columns_620a
    ]
    .loc[
        retained_sensors_620a
    ]
    .copy()
)


# ------------------------------------------------------------
# 9. Final status
# ------------------------------------------------------------

print(
    "\nStep 6.20A completed successfully."
)

print(
    "No corruption was generated."
)

print(
    "No temporal reliability score was created."
)

print(
    "No frozen artifact or CSV file was modified."
)

In [ ]:
# ============================================================
# Step 6.20B — Characterize clean temporal behaviour
#
# PURPOSE
# -------
# 1. Reconstruct clean validation sequence windows in the
#    original sensor measurement units.
# 2. Compute within-window absolute temporal differences.
# 3. Compare clean validation temporal behaviour against the
#    frozen development temporal-reference statistics.
# 4. Identify any sensor whose temporal reference is
#    degenerate for ratio-based scoring.
#
# IMPORTANT
# ---------
# - This is diagnostic only.
# - No corruption is generated.
# - No temporal reliability score is frozen.
# - No CSV or frozen artifact is modified.
# ============================================================

import numpy as np
import pandas as pd

print(
    "=== Step 6.20B — "
    "Characterize clean temporal behaviour ==="
)


# ------------------------------------------------------------
# 1. Reconfirm required objects
# ------------------------------------------------------------

required_objects_620b = [
    "sensor_temporal_stats",
    "scaler",
    "PRIMARY_INPUT_COLUMNS",
    "RETAINED_SENSOR_COLUMNS",
    "SENSOR_INPUT_INDICES",
    "X_val_clean",
]

missing_objects_620b = [
    name
    for name in required_objects_620b
    if name not in globals()
]

if missing_objects_620b:
    raise RuntimeError(
        "Missing required objects: "
        + ", ".join(missing_objects_620b)
    )

print(
    "Required frozen objects are available."
)


# ------------------------------------------------------------
# 2. Local copies of frozen configuration
# ------------------------------------------------------------

primary_columns_620b = list(
    PRIMARY_INPUT_COLUMNS
)

retained_sensors_620b = list(
    RETAINED_SENSOR_COLUMNS
)

sensor_indices_620b = np.asarray(
    SENSOR_INPUT_INDICES,
    dtype=int
)

X_val_scaled_620b = np.asarray(
    X_val_clean,
    dtype=float
)

if X_val_scaled_620b.ndim != 3:
    raise RuntimeError(
        "X_val_clean must be a 3-D array."
    )

n_windows_620b, window_size_620b, n_features_620b = (
    X_val_scaled_620b.shape
)

print(
    "\n=== Validation sequence structure ==="
)

print(
    "Windows      :",
    n_windows_620b
)

print(
    "Window size  :",
    window_size_620b
)

print(
    "Features     :",
    n_features_620b
)

print(
    "Sensors      :",
    len(retained_sensors_620b)
)

if n_features_620b != len(
    primary_columns_620b
):
    raise RuntimeError(
        "Sequence feature dimension does not match "
        "PRIMARY_INPUT_COLUMNS."
    )


# ------------------------------------------------------------
# 3. Reconstruct sequence windows in original units
# ------------------------------------------------------------

flat_scaled_620b = (
    X_val_scaled_620b
    .reshape(
        -1,
        n_features_620b
    )
)

flat_scaled_df_620b = pd.DataFrame(
    flat_scaled_620b,
    columns=primary_columns_620b,
)

flat_raw_620b = scaler.inverse_transform(
    flat_scaled_df_620b
)

X_val_raw_620b = (
    np.asarray(
        flat_raw_620b,
        dtype=float
    )
    .reshape(
        n_windows_620b,
        window_size_620b,
        n_features_620b
    )
)

if not np.isfinite(
    X_val_raw_620b
).all():
    raise RuntimeError(
        "Reconstructed raw validation sequences "
        "contain non-finite values."
    )

print(
    "\nRaw validation sequence shape:",
    X_val_raw_620b.shape
)


# ------------------------------------------------------------
# 4. Verify reconstruction by scaling back again
# ------------------------------------------------------------

flat_raw_df_620b = pd.DataFrame(
    X_val_raw_620b.reshape(
        -1,
        n_features_620b
    ),
    columns=primary_columns_620b,
)

flat_rescaled_620b = scaler.transform(
    flat_raw_df_620b
)

X_val_rescaled_620b = (
    np.asarray(
        flat_rescaled_620b,
        dtype=float
    )
    .reshape(
        X_val_scaled_620b.shape
    )
)

max_roundtrip_error_620b = float(
    np.max(
        np.abs(
            X_val_rescaled_620b
            -
            X_val_scaled_620b
        )
    )
)

print(
    "Maximum sequence round-trip error:",
    max_roundtrip_error_620b
)

if max_roundtrip_error_620b > 1e-10:
    raise RuntimeError(
        "Sequence scaling round-trip error "
        "is unexpectedly large."
    )


# ------------------------------------------------------------
# 5. Characterize clean temporal behaviour sensor by sensor
# ------------------------------------------------------------

temporal_summary_rows_620b = []

for sensor_620b, feature_index_620b in zip(
    retained_sensors_620b,
    sensor_indices_620b
):

    # Raw sensor trajectories inside all validation windows
    sensor_windows_620b = (
        X_val_raw_620b[
            :,
            :,
            feature_index_620b
        ]
    )

    # 29 consecutive changes for a 30-cycle window
    abs_diff_620b = np.abs(
        np.diff(
            sensor_windows_620b,
            axis=1
        )
    )

    if not np.isfinite(
        abs_diff_620b
    ).all():
        raise RuntimeError(
            f"Non-finite temporal differences "
            f"found for {sensor_620b}."
        )

    # Clean-window statistics
    window_median_abs_diff_620b = (
        np.median(
            abs_diff_620b,
            axis=1
        )
    )

    window_q95_abs_diff_620b = (
        np.quantile(
            abs_diff_620b,
            0.95,
            axis=1
        )
    )

    # Frozen development references
    ref_diff_std_620b = float(
        sensor_temporal_stats.loc[
            sensor_620b,
            "diff_std"
        ]
    )

    ref_median_620b = float(
        sensor_temporal_stats.loc[
            sensor_620b,
            "median_abs_diff"
        ]
    )

    ref_q95_620b = float(
        sensor_temporal_stats.loc[
            sensor_620b,
            "q95_abs_diff"
        ]
    )

    median_reference_positive_620b = (
        ref_median_620b > 0.0
    )

    q95_reference_positive_620b = (
        ref_q95_620b > 0.0
    )

    row_620b = {
        "sensor":
            sensor_620b,

        "feature_index":
            int(feature_index_620b),

        "ref_diff_std":
            ref_diff_std_620b,

        "ref_median_abs_diff":
            ref_median_620b,

        "ref_q95_abs_diff":
            ref_q95_620b,

        "median_reference_positive":
            median_reference_positive_620b,

        "q95_reference_positive":
            q95_reference_positive_620b,

        "clean_window_median_abs_diff_median":
            float(
                np.median(
                    window_median_abs_diff_620b
                )
            ),

        "clean_window_median_abs_diff_p95":
            float(
                np.quantile(
                    window_median_abs_diff_620b,
                    0.95
                )
            ),

        "clean_window_q95_abs_diff_median":
            float(
                np.median(
                    window_q95_abs_diff_620b
                )
            ),

        "exact_zero_transition_pct":
            float(
                100.0
                * np.mean(
                    abs_diff_620b == 0.0
                )
            ),
    }


    # --------------------------------------------------------
    # Median-difference ratio
    # --------------------------------------------------------

    if median_reference_positive_620b:

        median_ratio_620b = (
            window_median_abs_diff_620b
            /
            ref_median_620b
        )

        row_620b[
            "median_ratio_median"
        ] = float(
            np.median(
                median_ratio_620b
            )
        )

        row_620b[
            "median_ratio_p05"
        ] = float(
            np.quantile(
                median_ratio_620b,
                0.05
            )
        )

        row_620b[
            "median_ratio_p95"
        ] = float(
            np.quantile(
                median_ratio_620b,
                0.95
            )
        )

    else:

        row_620b[
            "median_ratio_median"
        ] = np.nan

        row_620b[
            "median_ratio_p05"
        ] = np.nan

        row_620b[
            "median_ratio_p95"
        ] = np.nan


    # --------------------------------------------------------
    # High-change / q95 reference behaviour
    # --------------------------------------------------------

    if q95_reference_positive_620b:

        q95_ratio_620b = (
            window_q95_abs_diff_620b
            /
            ref_q95_620b
        )

        row_620b[
            "q95_ratio_median"
        ] = float(
            np.median(
                q95_ratio_620b
            )
        )

        row_620b[
            "q95_ratio_p95"
        ] = float(
            np.quantile(
                q95_ratio_620b,
                0.95
            )
        )

        row_620b[
            "transition_above_ref_q95_pct"
        ] = float(
            100.0
            * np.mean(
                abs_diff_620b
                > ref_q95_620b
            )
        )

    else:

        row_620b[
            "q95_ratio_median"
        ] = np.nan

        row_620b[
            "q95_ratio_p95"
        ] = np.nan

        row_620b[
            "transition_above_ref_q95_pct"
        ] = np.nan


    temporal_summary_rows_620b.append(
        row_620b
    )


temporal_clean_summary_620b = pd.DataFrame(
    temporal_summary_rows_620b
)


# ------------------------------------------------------------
# 6. Display reference suitability
# ------------------------------------------------------------

print(
    "\n=== Temporal-reference suitability ==="
)

reference_suitability_620b = (
    temporal_clean_summary_620b[
        [
            "sensor",
            "ref_diff_std",
            "ref_median_abs_diff",
            "ref_q95_abs_diff",
            "median_reference_positive",
            "q95_reference_positive",
            "exact_zero_transition_pct",
        ]
    ]
    .copy()
)

display(
    reference_suitability_620b
)


# ------------------------------------------------------------
# 7. Display clean validation temporal ratios
# ------------------------------------------------------------

print(
    "\n=== Clean validation temporal behaviour ==="
)

clean_ratio_view_620b = (
    temporal_clean_summary_620b[
        [
            "sensor",

            "median_ratio_median",
            "median_ratio_p05",
            "median_ratio_p95",

            "q95_ratio_median",
            "q95_ratio_p95",

            "transition_above_ref_q95_pct",
        ]
    ]
    .copy()
)

display(
    clean_ratio_view_620b
)


# ------------------------------------------------------------
# 8. Explicitly identify degenerate ratio references
# ------------------------------------------------------------

degenerate_temporal_refs_620b = (
    temporal_clean_summary_620b[
        (~temporal_clean_summary_620b[
            "median_reference_positive"
        ])
        |
        (~temporal_clean_summary_620b[
            "q95_reference_positive"
        ])
    ][
        [
            "sensor",
            "ref_diff_std",
            "ref_median_abs_diff",
            "ref_q95_abs_diff",
            "exact_zero_transition_pct",
        ]
    ]
    .copy()
)

print(
    "\n=== Sensors requiring special temporal handling ==="
)

if len(
    degenerate_temporal_refs_620b
) == 0:

    print(
        "None."
    )

else:

    display(
        degenerate_temporal_refs_620b
    )


# ------------------------------------------------------------
# 9. Final integrity summary
# ------------------------------------------------------------

print(
    "\n=== Step 6.20B integrity summary ==="
)

print(
    "Validation windows analysed:",
    n_windows_620b
)

print(
    "Transitions per window:",
    window_size_620b - 1
)

print(
    "Sensors analysed:",
    len(
        temporal_clean_summary_620b
    )
)

print(
    "Sensors with positive median reference:",
    int(
        temporal_clean_summary_620b[
            "median_reference_positive"
        ].sum()
    ),
    "/",
    len(
        temporal_clean_summary_620b
    )
)

print(
    "Sensors with positive q95 reference:",
    int(
        temporal_clean_summary_620b[
            "q95_reference_positive"
        ].sum()
    ),
    "/",
    len(
        temporal_clean_summary_620b
    )
)

print(
    "\nStep 6.20B completed successfully."
)

print(
    "Clean temporal behaviour was characterized only."
)

print(
    "No temporal reliability formula was frozen."
)

print(
    "No corruption was generated."
)

print(
    "No frozen artifact or CSV file was modified."
)

In [ ]:
# ============================================================
# Step 6.20C — Locate clean development sequence source
#
# PURPOSE
# -------
# Identify the exact existing development/training sequence
# objects before temporal calibration.
#
# IMPORTANT
# ---------
# - Diagnostic only.
# - Nothing is modified.
# - No temporal score is created.
# - No corruption is generated.
# ============================================================

import numpy as np
import pandas as pd

print(
    "=== Step 6.20C — "
    "Locate clean development sequence source ==="
)


# ------------------------------------------------------------
# 1. Reference structure from verified validation sequences
# ------------------------------------------------------------

reference_sequence_shape_620c = (
    X_val_clean.shape[1:]
)

print(
    "\nVerified validation sequence tail shape:",
    reference_sequence_shape_620c
)

print(
    "Expected sequence structure: "
    "(n_windows, 30, 17)"
)


# ------------------------------------------------------------
# 2. Snapshot current namespace safely
# ------------------------------------------------------------

namespace_snapshot_620c = list(
    globals().items()
)


# ------------------------------------------------------------
# 3. Locate compatible 3-D NumPy sequence arrays
# ------------------------------------------------------------

array_candidates_620c = []

for name_620c, obj_620c in namespace_snapshot_620c:

    if name_620c.startswith("_"):
        continue

    if not isinstance(
        obj_620c,
        np.ndarray
    ):
        continue

    if obj_620c.ndim != 3:
        continue

    shape_620c = obj_620c.shape

    compatible_620c = (
        shape_620c[1:]
        ==
        reference_sequence_shape_620c
    )

    array_candidates_620c.append(
        {
            "name": name_620c,
            "shape": str(shape_620c),
            "dtype": str(obj_620c.dtype),
            "compatible_30x17":
                bool(compatible_620c),
            "looks_development":
                any(
                    token in name_620c.lower()
                    for token in [
                        "dev",
                        "train",
                        "development",
                    ]
                ),
            "looks_validation":
                "val" in name_620c.lower(),
        }
    )


array_candidates_df_620c = pd.DataFrame(
    array_candidates_620c
)

if len(array_candidates_df_620c) > 0:

    array_candidates_df_620c = (
        array_candidates_df_620c
        .sort_values(
            by=[
                "compatible_30x17",
                "looks_development",
                "name",
            ],
            ascending=[
                False,
                False,
                True,
            ],
        )
        .reset_index(drop=True)
    )


print(
    "\n=== 3-D sequence-array candidates ==="
)

if len(array_candidates_df_620c) == 0:

    print(
        "No 3-D NumPy arrays found."
    )

else:

    display(
        array_candidates_df_620c
    )


# ------------------------------------------------------------
# 4. Locate development/training related DataFrames
# ------------------------------------------------------------

df_candidates_620c = []

for name_620c, obj_620c in namespace_snapshot_620c:

    if name_620c.startswith("_"):
        continue

    if not isinstance(
        obj_620c,
        pd.DataFrame
    ):
        continue

    lower_name_620c = (
        name_620c.lower()
    )

    relevant_name_620c = any(
        token in lower_name_620c
        for token in [
            "dev",
            "train",
            "development",
            "sequence",
            "seq",
        ]
    )

    if not relevant_name_620c:
        continue

    columns_620c = list(
        obj_620c.columns
    )

    df_candidates_620c.append(
        {
            "name": name_620c,
            "shape":
                str(obj_620c.shape),

            "has_unit_id":
                "unit_id"
                in columns_620c,

            "has_cycle":
                "cycle"
                in columns_620c,

            "has_rul_target":
                any(
                    col in columns_620c
                    for col in [
                        "RUL",
                        "RUL_cap_125",
                        "rul",
                    ]
                ),

            "column_count":
                len(columns_620c),
        }
    )


df_candidates_df_620c = pd.DataFrame(
    df_candidates_620c
)

if len(df_candidates_df_620c) > 0:

    df_candidates_df_620c = (
        df_candidates_df_620c
        .sort_values(
            by="name"
        )
        .reset_index(drop=True)
    )


print(
    "\n=== Development/training DataFrame candidates ==="
)

if len(df_candidates_df_620c) == 0:

    print(
        "No matching DataFrame candidates found."
    )

else:

    display(
        df_candidates_df_620c
    )


# ------------------------------------------------------------
# 5. List relevant object names regardless of object type
# ------------------------------------------------------------

relevant_objects_620c = []

for name_620c, obj_620c in namespace_snapshot_620c:

    lower_name_620c = (
        name_620c.lower()
    )

    if not any(
        token in lower_name_620c
        for token in [
            "dev",
            "train",
            "development",
        ]
    ):
        continue

    if name_620c.startswith("_"):
        continue

    try:
        shape_620c = (
            obj_620c.shape
            if hasattr(
                obj_620c,
                "shape"
            )
            else None
        )
    except Exception:
        shape_620c = None

    relevant_objects_620c.append(
        {
            "name":
                name_620c,

            "type":
                type(
                    obj_620c
                ).__name__,

            "shape":
                str(shape_620c)
                if shape_620c is not None
                else "",
        }
    )


relevant_objects_df_620c = (
    pd.DataFrame(
        relevant_objects_620c
    )
)

if len(relevant_objects_df_620c) > 0:

    relevant_objects_df_620c = (
        relevant_objects_df_620c
        .sort_values(
            "name"
        )
        .reset_index(drop=True)
    )


print(
    "\n=== All development/training-named objects ==="
)

if len(relevant_objects_df_620c) == 0:

    print(
        "No development/training-named "
        "objects found."
    )

else:

    display(
        relevant_objects_df_620c
    )


# ------------------------------------------------------------
# 6. Final status
# ------------------------------------------------------------

compatible_arrays_620c = (
    array_candidates_df_620c[
        array_candidates_df_620c[
            "compatible_30x17"
        ]
    ]
    if len(array_candidates_df_620c) > 0
    else pd.DataFrame()
)

print(
    "\n=== Step 6.20C summary ==="
)

print(
    "Compatible 30 x 17 sequence arrays found:",
    len(
        compatible_arrays_620c
    )
)

print(
    "\nStep 6.20C completed."
)

print(
    "No temporal reliability score was created."
)

print(
    "No corruption was generated."
)

print(
    "No frozen object, artifact, or CSV was modified."
)

In [ ]:
# ============================================================
# Step 6.20D — Development-calibrated temporal reference
# ============================================================
#
# Purpose:
#   1. Verify the frozen development sequence structure.
#   2. Verify feature/scaler compatibility.
#   3. Safely reconstruct development inputs using FLOAT64
#      numerical verification.
#   4. Characterize clean development temporal behaviour.
#   5. Build an IN-MEMORY temporal calibration reference.
#
# Important:
#   - No corruption is generated.
#   - No existing frozen result is modified.
#   - No CSV/artifact is overwritten.
#   - No final temporal reliability formula is frozen here.
#
# ============================================================

import numpy as np
import pandas as pd


print(
    "=== Step 6.20D — "
    "Development-calibrated temporal reference ==="
)


# ------------------------------------------------------------
# 1. Verify required frozen objects
# ------------------------------------------------------------

required_objects_620d = [
    "X_dev_clean",
    "y_dev_clean",
    "dev_seq_meta",
    "scaler",
    "PRIMARY_INPUT_COLUMNS",
    "RETAINED_SENSOR_COLUMNS",
    "sensor_temporal_stats",
]

print("\n=== Required objects ===")

for object_name_620d in required_objects_620d:
    available_620d = (
        object_name_620d in globals()
    )

    print(
        f"{object_name_620d:<30}: "
        f"{available_620d}"
    )

    assert available_620d, (
        f"Required object "
        f"'{object_name_620d}' is missing."
    )


# ------------------------------------------------------------
# 2. Verify development sequence structure
# ------------------------------------------------------------

assert isinstance(
    X_dev_clean,
    np.ndarray
), (
    "X_dev_clean must be a NumPy array."
)

assert X_dev_clean.ndim == 3, (
    "X_dev_clean must have shape "
    "(windows, time_steps, features)."
)

n_dev_windows_620d = (
    X_dev_clean.shape[0]
)

window_size_620d = (
    X_dev_clean.shape[1]
)

n_features_620d = (
    X_dev_clean.shape[2]
)


print(
    "\n=== Development sequence structure ==="
)

print(
    "Development windows :",
    n_dev_windows_620d
)

print(
    "Window size         :",
    window_size_620d
)

print(
    "Features            :",
    n_features_620d
)

print(
    "dev_seq_meta rows   :",
    len(dev_seq_meta)
)

print(
    "y_dev_clean length  :",
    len(y_dev_clean)
)


assert (
    n_dev_windows_620d
    ==
    len(dev_seq_meta)
), (
    "Development sequence count and "
    "dev_seq_meta row count do not match."
)


assert (
    n_dev_windows_620d
    ==
    len(y_dev_clean)
), (
    "Development sequence count and "
    "y_dev_clean length do not match."
)


assert (
    n_features_620d
    ==
    len(PRIMARY_INPUT_COLUMNS)
), (
    "Development feature count does not "
    "match PRIMARY_INPUT_COLUMNS."
)


# ------------------------------------------------------------
# 3. Verify scaler compatibility
# ------------------------------------------------------------

if hasattr(
    scaler,
    "feature_names_in_"
):

    scaler_features_620d = list(
        scaler.feature_names_in_
    )

    feature_order_matches_620d = (
        scaler_features_620d
        ==
        list(PRIMARY_INPUT_COLUMNS)
    )

    assert feature_order_matches_620d, (
        "Scaler feature order does not "
        "match PRIMARY_INPUT_COLUMNS."
    )

else:

    assert hasattr(
        scaler,
        "n_features_in_"
    ), (
        "Scaler does not expose "
        "feature-count information."
    )

    assert (
        scaler.n_features_in_
        ==
        len(PRIMARY_INPUT_COLUMNS)
    ), (
        "Scaler feature count does not "
        "match PRIMARY_INPUT_COLUMNS."
    )


print(
    "\nScaler compatibility verified."
)


# ------------------------------------------------------------
# 4. Verify retained-sensor mapping
# ------------------------------------------------------------

sensor_mapping_rows_620d = []

sensor_feature_indices_620d = []


for sensor_name_620d in RETAINED_SENSOR_COLUMNS:

    assert (
        sensor_name_620d
        in PRIMARY_INPUT_COLUMNS
    ), (
        f"{sensor_name_620d} is not present "
        "in PRIMARY_INPUT_COLUMNS."
    )

    feature_index_620d = (
        list(PRIMARY_INPUT_COLUMNS)
        .index(sensor_name_620d)
    )

    sensor_feature_indices_620d.append(
        feature_index_620d
    )

    mapped_name_620d = (
        PRIMARY_INPUT_COLUMNS[
            feature_index_620d
        ]
    )

    sensor_mapping_rows_620d.append(
        {
            "sensor":
                sensor_name_620d,

            "sequence_feature_index":
                feature_index_620d,

            "mapped_column":
                mapped_name_620d,

            "name_matches":
                (
                    mapped_name_620d
                    ==
                    sensor_name_620d
                ),
        }
    )


sensor_mapping_620d = pd.DataFrame(
    sensor_mapping_rows_620d
)


print(
    "\n=== Retained-sensor mapping ==="
)

display(
    sensor_mapping_620d
)


assert (
    sensor_mapping_620d[
        "name_matches"
    ].all()
), (
    "At least one retained sensor is "
    "mapped to the wrong sequence feature."
)


assert (
    len(
        set(
            sensor_feature_indices_620d
        )
    )
    ==
    len(
        sensor_feature_indices_620d
    )
), (
    "Duplicate retained-sensor feature "
    "indices were detected."
)


# ------------------------------------------------------------
# 5. Verify frozen temporal-reference coverage
# ------------------------------------------------------------

missing_temporal_sensors_620d = [
    sensor_name_620d

    for sensor_name_620d
    in RETAINED_SENSOR_COLUMNS

    if sensor_name_620d
    not in sensor_temporal_stats.index
]


unexpected_temporal_sensors_620d = [
    sensor_name_620d

    for sensor_name_620d
    in sensor_temporal_stats.index

    if sensor_name_620d
    not in RETAINED_SENSOR_COLUMNS
]


print(
    "\n=== Temporal-reference coverage ==="
)

print(
    "Missing retained sensors:",
    missing_temporal_sensors_620d
)

print(
    "Unexpected temporal sensors:",
    unexpected_temporal_sensors_620d
)


assert (
    len(
        missing_temporal_sensors_620d
    )
    == 0
), (
    "sensor_temporal_stats is missing "
    "one or more retained sensors."
)


assert (
    len(
        unexpected_temporal_sensors_620d
    )
    == 0
), (
    "sensor_temporal_stats contains "
    "unexpected sensors."
)


# ------------------------------------------------------------
# 6. FLOAT64 scaling round-trip verification
# ------------------------------------------------------------
#
# X_dev_clean is stored as float32.
# The fitted StandardScaler parameters are float64.
#
# Earlier we verified that doing the diagnostic through
# float32 creates an artificial numerical round-trip error.
#
# Therefore this verification is explicitly performed using
# a float64 COPY.  X_dev_clean itself is NOT modified.
# ------------------------------------------------------------

X_dev_flat_620d = (
    X_dev_clean
    .reshape(
        -1,
        n_features_620d
    )
    .astype(
        np.float64,
        copy=True
    )
)


X_dev_flat_df_620d = pd.DataFrame(
    X_dev_flat_620d,
    columns=PRIMARY_INPUT_COLUMNS
)


X_dev_raw_flat_620d = (
    scaler.inverse_transform(
        X_dev_flat_df_620d
    )
)


X_dev_raw_flat_df_620d = pd.DataFrame(
    X_dev_raw_flat_620d,
    columns=PRIMARY_INPUT_COLUMNS
)


X_dev_rescaled_flat_620d = (
    scaler.transform(
        X_dev_raw_flat_df_620d
    )
)


max_roundtrip_error_620d = float(
    np.max(
        np.abs(
            X_dev_rescaled_flat_620d
            -
            X_dev_flat_620d
        )
    )
)


mean_roundtrip_error_620d = float(
    np.mean(
        np.abs(
            X_dev_rescaled_flat_620d
            -
            X_dev_flat_620d
        )
    )
)


print(
    "\n=== FLOAT64 scaling verification ==="
)

print(
    "Original sequence dtype :",
    X_dev_clean.dtype
)

print(
    "Verification dtype      :",
    X_dev_flat_620d.dtype
)

print(
    "Maximum round-trip error:",
    max_roundtrip_error_620d
)

print(
    "Mean round-trip error   :",
    mean_roundtrip_error_620d
)


assert (
    max_roundtrip_error_620d
    <
    1e-10
), (
    "FLOAT64 development scaling "
    "round-trip error is unexpectedly large."
)


print(
    "FLOAT64 scaling round-trip "
    "verification passed."
)


# ------------------------------------------------------------
# 7. Restore raw 3-D development sequence
# ------------------------------------------------------------

X_dev_raw_620d = (
    X_dev_raw_flat_620d
    .reshape(
        X_dev_clean.shape
    )
)


print(
    "\nRaw development sequence shape:",
    X_dev_raw_620d.shape
)


assert (
    X_dev_raw_620d.shape
    ==
    X_dev_clean.shape
), (
    "Raw development sequence shape "
    "does not match X_dev_clean."
)


# ------------------------------------------------------------
# 8. Extract retained sensors
# ------------------------------------------------------------

sensor_feature_indices_array_620d = (
    np.asarray(
        sensor_feature_indices_620d,
        dtype=int
    )
)


X_dev_sensor_raw_620d = (
    X_dev_raw_620d[
        :,
        :,
        sensor_feature_indices_array_620d
    ]
)


print(
    "\nRetained-sensor raw sequence shape:",
    X_dev_sensor_raw_620d.shape
)


assert (
    X_dev_sensor_raw_620d.shape
    ==
    (
        n_dev_windows_620d,
        window_size_620d,
        len(
            RETAINED_SENSOR_COLUMNS
        ),
    )
), (
    "Unexpected retained-sensor "
    "sequence shape."
)


# ------------------------------------------------------------
# 9. Compute clean temporal transitions
# ------------------------------------------------------------
#
# Temporal signal is based on differences between consecutive
# time steps inside each 30-cycle sequence.
#
# Number of transitions in a 30-cycle window = 29.
# ------------------------------------------------------------

dev_sensor_diff_620d = np.diff(
    X_dev_sensor_raw_620d,
    axis=1
)


dev_sensor_abs_diff_620d = np.abs(
    dev_sensor_diff_620d
)


n_transitions_620d = (
    dev_sensor_abs_diff_620d.shape[1]
)


print(
    "\n=== Temporal transition structure ==="
)

print(
    "Development windows :",
    n_dev_windows_620d
)

print(
    "Transitions/window  :",
    n_transitions_620d
)

print(
    "Sensors             :",
    len(
        RETAINED_SENSOR_COLUMNS
    )
)


assert (
    n_transitions_620d
    ==
    window_size_620d - 1
), (
    "Unexpected number of temporal "
    "transitions per window."
)


# ------------------------------------------------------------
# 10. Window-level temporal statistics
# ------------------------------------------------------------
#
# For every clean development window and every retained sensor,
# calculate:
#
#   - mean absolute transition
#   - median absolute transition
#   - 95th-percentile absolute transition
#   - maximum absolute transition
#   - exact-zero transition fraction
#
# These are CHARACTERIZATION statistics only.
# ------------------------------------------------------------

dev_window_mean_abs_diff_620d = (
    np.mean(
        dev_sensor_abs_diff_620d,
        axis=1
    )
)


dev_window_median_abs_diff_620d = (
    np.median(
        dev_sensor_abs_diff_620d,
        axis=1
    )
)


dev_window_q95_abs_diff_620d = (
    np.quantile(
        dev_sensor_abs_diff_620d,
        0.95,
        axis=1
    )
)


dev_window_max_abs_diff_620d = (
    np.max(
        dev_sensor_abs_diff_620d,
        axis=1
    )
)


# Because consecutive identical scaled values remain identical
# after a linear inverse transformation, exact zero is suitable
# here. A tiny numerical safeguard is nevertheless included.

zero_transition_tolerance_620d = (
    1e-12
)


dev_window_zero_fraction_620d = (
    np.mean(
        dev_sensor_abs_diff_620d
        <=
        zero_transition_tolerance_620d,
        axis=1
    )
)


# ------------------------------------------------------------
# 11. Development-calibrated sensor reference
# ------------------------------------------------------------

temporal_reference_rows_620d = []


for sensor_pos_620d, sensor_name_620d in enumerate(
    RETAINED_SENSOR_COLUMNS
):

    mean_values_620d = (
        dev_window_mean_abs_diff_620d[
            :,
            sensor_pos_620d
        ]
    )

    median_values_620d = (
        dev_window_median_abs_diff_620d[
            :,
            sensor_pos_620d
        ]
    )

    q95_values_620d = (
        dev_window_q95_abs_diff_620d[
            :,
            sensor_pos_620d
        ]
    )

    max_values_620d = (
        dev_window_max_abs_diff_620d[
            :,
            sensor_pos_620d
        ]
    )

    zero_values_620d = (
        dev_window_zero_fraction_620d[
            :,
            sensor_pos_620d
        ]
    )


    frozen_diff_std_620d = float(
        sensor_temporal_stats.loc[
            sensor_name_620d,
            "diff_std"
        ]
    )

    frozen_median_abs_diff_620d = float(
        sensor_temporal_stats.loc[
            sensor_name_620d,
            "median_abs_diff"
        ]
    )

    frozen_q95_abs_diff_620d = float(
        sensor_temporal_stats.loc[
            sensor_name_620d,
            "q95_abs_diff"
        ]
    )


    requires_special_handling_620d = bool(
        (
            frozen_median_abs_diff_620d
            <= 0.0
        )
        or
        (
            frozen_q95_abs_diff_620d
            <= 0.0
        )
    )


    temporal_reference_rows_620d.append(
        {
            "sensor":
                sensor_name_620d,

            # Existing frozen temporal statistics
            "frozen_diff_std":
                frozen_diff_std_620d,

            "frozen_median_abs_diff":
                frozen_median_abs_diff_620d,

            "frozen_q95_abs_diff":
                frozen_q95_abs_diff_620d,

            # Development clean distribution:
            # mean absolute transition
            "dev_mean_abs_diff_median":
                float(
                    np.median(
                        mean_values_620d
                    )
                ),

            "dev_mean_abs_diff_p95":
                float(
                    np.quantile(
                        mean_values_620d,
                        0.95
                    )
                ),

            # Development clean distribution:
            # median absolute transition
            "dev_median_abs_diff_median":
                float(
                    np.median(
                        median_values_620d
                    )
                ),

            "dev_median_abs_diff_p95":
                float(
                    np.quantile(
                        median_values_620d,
                        0.95
                    )
                ),

            # Development clean distribution:
            # within-window q95 transition
            "dev_q95_abs_diff_median":
                float(
                    np.median(
                        q95_values_620d
                    )
                ),

            "dev_q95_abs_diff_p95":
                float(
                    np.quantile(
                        q95_values_620d,
                        0.95
                    )
                ),

            # Development clean distribution:
            # maximum transition
            "dev_max_abs_diff_median":
                float(
                    np.median(
                        max_values_620d
                    )
                ),

            "dev_max_abs_diff_p95":
                float(
                    np.quantile(
                        max_values_620d,
                        0.95
                    )
                ),

            # Clean zero-transition behaviour
            "dev_zero_fraction_median":
                float(
                    np.median(
                        zero_values_620d
                    )
                ),

            "dev_zero_fraction_p05":
                float(
                    np.quantile(
                        zero_values_620d,
                        0.05
                    )
                ),

            "dev_zero_fraction_p95":
                float(
                    np.quantile(
                        zero_values_620d,
                        0.95
                    )
                ),

            "requires_special_handling":
                requires_special_handling_620d,
        }
    )


temporal_dev_reference_620d = pd.DataFrame(
    temporal_reference_rows_620d
).set_index(
    "sensor"
)


# ------------------------------------------------------------
# 12. Verify numerical integrity
# ------------------------------------------------------------

numeric_temporal_values_620d = (
    temporal_dev_reference_620d
    .drop(
        columns=[
            "requires_special_handling"
        ]
    )
    .to_numpy(
        dtype=float
    )
)


assert np.isfinite(
    numeric_temporal_values_620d
).all(), (
    "Non-finite value detected in "
    "development temporal reference."
)


# ------------------------------------------------------------
# 13. Development P95 exceedance sanity check
# ------------------------------------------------------------
#
# We expect a clean-development P95 threshold to produce
# approximately <= 5% strict exceedances.
#
# Exact percentage may be lower because of tied/discrete
# temporal values. Therefore this is descriptive only.
# ------------------------------------------------------------

p95_exceedance_rows_620d = []


for sensor_pos_620d, sensor_name_620d in enumerate(
    RETAINED_SENSOR_COLUMNS
):

    median_threshold_620d = float(
        temporal_dev_reference_620d.loc[
            sensor_name_620d,
            "dev_median_abs_diff_p95"
        ]
    )

    q95_threshold_620d = float(
        temporal_dev_reference_620d.loc[
            sensor_name_620d,
            "dev_q95_abs_diff_p95"
        ]
    )


    median_exceed_pct_620d = float(
        100.0
        *
        np.mean(
            dev_window_median_abs_diff_620d[
                :,
                sensor_pos_620d
            ]
            >
            median_threshold_620d
        )
    )


    q95_exceed_pct_620d = float(
        100.0
        *
        np.mean(
            dev_window_q95_abs_diff_620d[
                :,
                sensor_pos_620d
            ]
            >
            q95_threshold_620d
        )
    )


    p95_exceedance_rows_620d.append(
        {
            "sensor":
                sensor_name_620d,

            "median_metric_exceed_p95_pct":
                median_exceed_pct_620d,

            "q95_metric_exceed_p95_pct":
                q95_exceed_pct_620d,
        }
    )


temporal_dev_p95_check_620d = pd.DataFrame(
    p95_exceedance_rows_620d
).set_index(
    "sensor"
)


# ------------------------------------------------------------
# 14. Display development-calibrated reference
# ------------------------------------------------------------

print(
    "\n=== Development-calibrated "
    "temporal reference ==="
)

display(
    temporal_dev_reference_620d
)


print(
    "\n=== Clean-development P95 "
    "exceedance sanity check (%) ==="
)

display(
    temporal_dev_p95_check_620d
)


# ------------------------------------------------------------
# 15. Sensors requiring special treatment
# ------------------------------------------------------------

special_temporal_sensors_620d = (
    temporal_dev_reference_620d[
        temporal_dev_reference_620d[
            "requires_special_handling"
        ]
    ]
)


print(
    "\n=== Sensors requiring special "
    "temporal handling ==="
)

if len(
    special_temporal_sensors_620d
) == 0:

    print(
        "None."
    )

else:

    display(
        special_temporal_sensors_620d[
            [
                "frozen_diff_std",
                "frozen_median_abs_diff",
                "frozen_q95_abs_diff",
                "dev_zero_fraction_median",
                "dev_zero_fraction_p05",
                "dev_zero_fraction_p95",
            ]
        ]
    )


# ------------------------------------------------------------
# 16. Integrity summary
# ------------------------------------------------------------

print(
    "\n=== Step 6.20D integrity summary ==="
)

print(
    "Development windows analysed :",
    n_dev_windows_620d
)

print(
    "Window size                  :",
    window_size_620d
)

print(
    "Transitions per window       :",
    n_transitions_620d
)

print(
    "Retained sensors analysed    :",
    len(
        RETAINED_SENSOR_COLUMNS
    )
)

print(
    "Special-handling sensors     :",
    list(
        special_temporal_sensors_620d.index
    )
)

print(
    "Maximum FLOAT64 round-trip "
    "error:",
    max_roundtrip_error_620d
)


print(
    "\nStep 6.20D completed successfully."
)

print(
    "A clean DEVELOPMENT-calibrated "
    "temporal reference was created "
    "IN MEMORY only."
)

print(
    "No corruption was generated."
)

print(
    "No final temporal reliability "
    "formula was frozen."
)

print(
    "No frozen object, artifact, "
    "or CSV file was modified."
)

In [ ]:
# ============================================================
# Step 6.20D-1 — Diagnose development round-trip precision
#
# Diagnostic only.
# Nothing is modified.
# ============================================================

import numpy as np
import pandas as pd


print(
    "=== Step 6.20D-1 — "
    "Development round-trip precision diagnosis ==="
)


# ------------------------------------------------------------
# 1. Inspect relevant dtypes
# ------------------------------------------------------------

print(
    "\nX_dev_clean dtype:",
    X_dev_clean.dtype
)

print(
    "scaler.mean_ dtype:",
    scaler.mean_.dtype
)

print(
    "scaler.scale_ dtype:",
    scaler.scale_.dtype
)


# ------------------------------------------------------------
# 2. Repeat the round trip explicitly in FLOAT64
# ------------------------------------------------------------

X_dev_flat64_620d1 = (
    X_dev_clean
    .reshape(
        -1,
        X_dev_clean.shape[2]
    )
    .astype(
        np.float64,
        copy=True
    )
)


X_dev_flat64_df_620d1 = pd.DataFrame(
    X_dev_flat64_620d1,
    columns=PRIMARY_INPUT_COLUMNS
)


X_dev_raw64_620d1 = scaler.inverse_transform(
    X_dev_flat64_df_620d1
)


X_dev_raw64_df_620d1 = pd.DataFrame(
    X_dev_raw64_620d1,
    columns=PRIMARY_INPUT_COLUMNS
)


X_dev_rescaled64_620d1 = scaler.transform(
    X_dev_raw64_df_620d1
)


roundtrip_diff64_620d1 = np.abs(
    X_dev_rescaled64_620d1
    -
    X_dev_flat64_620d1
)


max_error64_620d1 = float(
    np.max(
        roundtrip_diff64_620d1
    )
)


mean_error64_620d1 = float(
    np.mean(
        roundtrip_diff64_620d1
    )
)


max_location_620d1 = np.unravel_index(
    np.argmax(
        roundtrip_diff64_620d1
    ),
    roundtrip_diff64_620d1.shape
)


max_row_620d1 = int(
    max_location_620d1[0]
)

max_col_620d1 = int(
    max_location_620d1[1]
)

max_feature_620d1 = (
    PRIMARY_INPUT_COLUMNS[
        max_col_620d1
    ]
)


print(
    "\n=== FLOAT64 round-trip result ==="
)

print(
    "Maximum absolute error:",
    max_error64_620d1
)

print(
    "Mean absolute error   :",
    mean_error64_620d1
)

print(
    "Feature at max error  :",
    max_feature_620d1
)

print(
    "Original scaled value :",
    X_dev_flat64_620d1[
        max_row_620d1,
        max_col_620d1
    ]
)

print(
    "Round-trip value      :",
    X_dev_rescaled64_620d1[
        max_row_620d1,
        max_col_620d1
    ]
)


# ------------------------------------------------------------
# 3. Integrity conclusion
# ------------------------------------------------------------

precision_pass_620d1 = (
    max_error64_620d1
    <
    1e-10
)


print(
    "\nFLOAT64 precision check passed:",
    precision_pass_620d1
)


if precision_pass_620d1:

    print(
        "\nConclusion:"
    )

    print(
        "The earlier Step-6.20D failure was caused "
        "by FLOAT32 round-trip precision."
    )

    print(
        "The frozen scaler and development "
        "sequence structure are compatible."
    )

else:

    print(
        "\nConclusion:"
    )

    print(
        "Do NOT continue yet."
    )

    print(
        "The discrepancy is larger than expected "
        "even in FLOAT64 and requires investigation."
    )


print(
    "\nNothing was modified."
)

In [ ]:
# ================================================================
# Step 6.20E — Candidate temporal components on CLEAN validation
# ================================================================
#
# Purpose
# -------
# 1. Use CLEAN DEVELOPMENT windows only to define temporal
#    reference thresholds.
# 2. Apply those frozen development thresholds to CLEAN VALIDATION.
# 3. Measure clean-validation false-alarm behaviour.
# 4. Treat temporally near-static sensors safely.
#
# IMPORTANT
# ---------
# - No corruption is generated here.
# - No PCA result is modified.
# - No final temporal reliability score is created.
# - No PCA + temporal combination is performed.
# - No threshold is tuned on validation.
# - No CSV or frozen artifact is modified.
# ================================================================

import warnings
import numpy as np
import pandas as pd
from IPython.display import display


print(
    "=== Step 6.20E — Candidate temporal components "
    "on CLEAN validation ==="
)


# ----------------------------------------------------------------
# 1. Required-object verification
# ----------------------------------------------------------------

required_objects_620e = [
    "X_dev_clean",
    "X_val_clean",
    "scaler",
    "PRIMARY_INPUT_COLUMNS",
    "RETAINED_SENSOR_COLUMNS",
    "sensor_temporal_stats",
]

missing_objects_620e = [
    name
    for name in required_objects_620e
    if name not in globals()
]

print("\n=== Required objects ===")

for name in required_objects_620e:
    print(
        f"{name:<28}:",
        name in globals()
    )

assert not missing_objects_620e, (
    "Step 6.20E cannot continue because these required "
    f"objects are missing: {missing_objects_620e}"
)

print("\nAll required objects are available.")


# ----------------------------------------------------------------
# 2. Freeze local FLOAT64 copies
# ----------------------------------------------------------------

X_dev_620e = np.asarray(
    X_dev_clean,
    dtype=np.float64
)

X_val_620e = np.asarray(
    X_val_clean,
    dtype=np.float64
)

primary_columns_620e = list(
    PRIMARY_INPUT_COLUMNS
)

retained_sensors_620e = list(
    RETAINED_SENSOR_COLUMNS
)


assert X_dev_620e.ndim == 3, (
    "X_dev_clean must be a 3-D sequence array."
)

assert X_val_620e.ndim == 3, (
    "X_val_clean must be a 3-D sequence array."
)

assert X_dev_620e.shape[1:] == X_val_620e.shape[1:], (
    "Development and validation sequence structures differ."
)

assert (
    X_dev_620e.shape[2]
    == len(primary_columns_620e)
), (
    "Development feature count does not match "
    "PRIMARY_INPUT_COLUMNS."
)

assert (
    X_val_620e.shape[2]
    == len(primary_columns_620e)
), (
    "Validation feature count does not match "
    "PRIMARY_INPUT_COLUMNS."
)


print("\n=== Sequence structure ===")

print(
    "Development sequence shape :",
    X_dev_620e.shape
)

print(
    "Validation sequence shape  :",
    X_val_620e.shape
)

print(
    "Window size                :",
    X_dev_620e.shape[1]
)

print(
    "Primary features           :",
    X_dev_620e.shape[2]
)

print(
    "Retained sensors           :",
    len(retained_sensors_620e)
)


# ----------------------------------------------------------------
# 3. Exact retained-sensor mapping
# ----------------------------------------------------------------

missing_sensor_columns_620e = [
    sensor
    for sensor in retained_sensors_620e
    if sensor not in primary_columns_620e
]

assert not missing_sensor_columns_620e, (
    "Some retained sensors are missing from "
    f"PRIMARY_INPUT_COLUMNS: {missing_sensor_columns_620e}"
)


sensor_feature_indices_620e = {
    sensor: primary_columns_620e.index(sensor)
    for sensor in retained_sensors_620e
}


mapping_rows_620e = []

for sensor in retained_sensors_620e:

    idx = sensor_feature_indices_620e[sensor]

    mapping_rows_620e.append({
        "sensor": sensor,
        "sequence_feature_index": idx,
        "mapped_column": primary_columns_620e[idx],
        "name_matches": (
            primary_columns_620e[idx]
            == sensor
        ),
    })


temporal_mapping_620e = pd.DataFrame(
    mapping_rows_620e
)


assert temporal_mapping_620e[
    "name_matches"
].all(), (
    "Retained-sensor feature mapping is incorrect."
)


print(
    "\n=== Retained-sensor mapping ==="
)

display(
    temporal_mapping_620e
)


# ----------------------------------------------------------------
# 4. Safe inverse scaling using FLOAT64
# ----------------------------------------------------------------

def inverse_sequence_array_620e(
    X_scaled,
    fitted_scaler
):
    """
    Convert a 3-D scaled sequence array back to raw feature units.

    FLOAT64 is used deliberately so that the verification is not
    affected by FLOAT32 round-trip precision.
    """

    X_scaled = np.asarray(
        X_scaled,
        dtype=np.float64
    )

    original_shape = X_scaled.shape

    flat_scaled = X_scaled.reshape(
        -1,
        original_shape[2]
    )

    with warnings.catch_warnings():

        warnings.filterwarnings(
            "ignore",
            message=(
                "X does not have valid feature names"
            )
        )

        flat_raw = fitted_scaler.inverse_transform(
            flat_scaled
        )

    return np.asarray(
        flat_raw,
        dtype=np.float64
    ).reshape(original_shape)


def scale_sequence_array_620e(
    X_raw,
    fitted_scaler
):
    """
    Re-scale a raw 3-D sequence array for a precision check.
    """

    X_raw = np.asarray(
        X_raw,
        dtype=np.float64
    )

    original_shape = X_raw.shape

    flat_raw = X_raw.reshape(
        -1,
        original_shape[2]
    )

    with warnings.catch_warnings():

        warnings.filterwarnings(
            "ignore",
            message=(
                "X does not have valid feature names"
            )
        )

        flat_scaled = fitted_scaler.transform(
            flat_raw
        )

    return np.asarray(
        flat_scaled,
        dtype=np.float64
    ).reshape(original_shape)


X_dev_raw_620e = inverse_sequence_array_620e(
    X_dev_620e,
    scaler
)

X_val_raw_620e = inverse_sequence_array_620e(
    X_val_620e,
    scaler
)


# ----------------------------------------------------------------
# 5. FLOAT64 round-trip verification
# ----------------------------------------------------------------

X_dev_rescaled_620e = scale_sequence_array_620e(
    X_dev_raw_620e,
    scaler
)

X_val_rescaled_620e = scale_sequence_array_620e(
    X_val_raw_620e,
    scaler
)


dev_roundtrip_error_620e = float(
    np.max(
        np.abs(
            X_dev_620e
            - X_dev_rescaled_620e
        )
    )
)

val_roundtrip_error_620e = float(
    np.max(
        np.abs(
            X_val_620e
            - X_val_rescaled_620e
        )
    )
)


print(
    "\n=== FLOAT64 scaling verification ==="
)

print(
    "Maximum development round-trip error:",
    dev_roundtrip_error_620e
)

print(
    "Maximum validation round-trip error :",
    val_roundtrip_error_620e
)


assert dev_roundtrip_error_620e < 1e-6, (
    "Development FLOAT64 scaling verification failed."
)

assert val_roundtrip_error_620e < 1e-6, (
    "Validation FLOAT64 scaling verification failed."
)

print(
    "FLOAT64 scaling verification passed."
)


# ----------------------------------------------------------------
# 6. Temporal-window metric helper
# ----------------------------------------------------------------

ZERO_TOLERANCE_620E = 1e-12


def temporal_window_metrics_620e(
    X_raw,
    sensor_feature_indices
):
    """
    Compute per-window temporal transition metrics.

    For every retained sensor and every sequence window:
      - mean absolute transition
      - median absolute transition
      - 95th-percentile absolute transition
      - maximum absolute transition
      - fraction of approximately-zero transitions
    """

    metrics = {}

    for sensor, feature_idx in (
        sensor_feature_indices.items()
    ):

        sensor_values = X_raw[
            :,
            :,
            feature_idx
        ]

        abs_diff = np.abs(
            np.diff(
                sensor_values,
                axis=1
            )
        )

        metrics[sensor] = {

            "mean_abs_diff":
                np.mean(
                    abs_diff,
                    axis=1
                ),

            "median_abs_diff":
                np.median(
                    abs_diff,
                    axis=1
                ),

            "q95_abs_diff":
                np.quantile(
                    abs_diff,
                    0.95,
                    axis=1
                ),

            "max_abs_diff":
                np.max(
                    abs_diff,
                    axis=1
                ),

            "zero_fraction":
                np.mean(
                    np.isclose(
                        abs_diff,
                        0.0,
                        atol=ZERO_TOLERANCE_620E,
                        rtol=0.0
                    ),
                    axis=1
                ),
        }

    return metrics


dev_temporal_metrics_620e = (
    temporal_window_metrics_620e(
        X_dev_raw_620e,
        sensor_feature_indices_620e
    )
)

val_temporal_metrics_620e = (
    temporal_window_metrics_620e(
        X_val_raw_620e,
        sensor_feature_indices_620e
    )
)


print(
    "\nTemporal window metrics computed successfully."
)


# ----------------------------------------------------------------
# 7. Identify sensors requiring special handling
# ----------------------------------------------------------------

special_handling_sensors_620e = []


for sensor in retained_sensors_620e:

    frozen_median = float(
        sensor_temporal_stats.loc[
            sensor,
            "median_abs_diff"
        ]
    )

    frozen_q95 = float(
        sensor_temporal_stats.loc[
            sensor,
            "q95_abs_diff"
        ]
    )

    if (
        frozen_median <= ZERO_TOLERANCE_620E
        or
        frozen_q95 <= ZERO_TOLERANCE_620E
    ):
        special_handling_sensors_620e.append(
            sensor
        )


print(
    "\n=== Special temporal handling ==="
)

print(
    "Sensors requiring special handling:",
    special_handling_sensors_620e
)


# ----------------------------------------------------------------
# 8. DEVELOPMENT-ONLY temporal thresholds
# ----------------------------------------------------------------
#
# These thresholds are calibrated exclusively from clean
# development windows.
#
# Validation is NOT used to set them.
# ----------------------------------------------------------------

reference_rows_620e = []


for sensor in retained_sensors_620e:

    dev_m = dev_temporal_metrics_620e[
        sensor
    ]

    reference_rows_620e.append({

        "sensor":
            sensor,

        "mean_abs_diff_p95":
            float(
                np.quantile(
                    dev_m["mean_abs_diff"],
                    0.95
                )
            ),

        "median_abs_diff_p95":
            float(
                np.quantile(
                    dev_m["median_abs_diff"],
                    0.95
                )
            ),

        "q95_abs_diff_p95":
            float(
                np.quantile(
                    dev_m["q95_abs_diff"],
                    0.95
                )
            ),

        "max_abs_diff_p95":
            float(
                np.quantile(
                    dev_m["max_abs_diff"],
                    0.95
                )
            ),

        # For zero-transition fraction, unusually LOW
        # values can indicate excessive movement.
        "zero_fraction_p05":
            float(
                np.quantile(
                    dev_m["zero_fraction"],
                    0.05
                )
            ),

        "zero_fraction_median":
            float(
                np.median(
                    dev_m["zero_fraction"]
                )
            ),

        "special_handling":
            (
                sensor
                in
                special_handling_sensors_620e
            ),
    })


temporal_reference_620e = (
    pd.DataFrame(
        reference_rows_620e
    )
    .set_index("sensor")
)


assert (
    temporal_reference_620e.index.is_unique
), (
    "Temporal reference contains duplicate sensors."
)


print(
    "\n=== DEVELOPMENT-only candidate thresholds ==="
)

display(
    temporal_reference_620e
)


# ----------------------------------------------------------------
# 9. Candidate continuous temporal-excess components
# ----------------------------------------------------------------
#
# We are NOT freezing these as the final score.
#
# Candidate A:
#   excess of window mean absolute transition above development P95
#
# Candidate B:
#   excess of window q95 absolute transition above development P95
#
# Candidate C:
#   zero-transition deficit for special near-static sensors
#
# Values <= threshold produce 0 excess.
# ----------------------------------------------------------------

EPSILON_620E = 1e-12

validation_component_rows_620e = []

mean_flag_columns_620e = []
q95_flag_columns_620e = []
dual_flag_columns_620e = []
zero_flag_columns_620e = []

mean_excess_columns_620e = []
q95_excess_columns_620e = []
zero_deficit_columns_620e = []


for sensor in retained_sensors_620e:

    val_m = val_temporal_metrics_620e[
        sensor
    ]

    ref = temporal_reference_620e.loc[
        sensor
    ]

    mean_threshold = float(
        ref["mean_abs_diff_p95"]
    )

    q95_threshold = float(
        ref["q95_abs_diff_p95"]
    )

    zero_p05 = float(
        ref["zero_fraction_p05"]
    )


    # ------------------------------------------------------------
    # Mean-transition excess
    # ------------------------------------------------------------

    if mean_threshold > EPSILON_620E:

        mean_excess = np.maximum(
            (
                val_m["mean_abs_diff"]
                / mean_threshold
            ) - 1.0,
            0.0
        )

    else:

        mean_excess = np.zeros(
            X_val_620e.shape[0],
            dtype=np.float64
        )


    # ------------------------------------------------------------
    # Q95-transition excess
    # ------------------------------------------------------------

    if q95_threshold > EPSILON_620E:

        q95_excess = np.maximum(
            (
                val_m["q95_abs_diff"]
                / q95_threshold
            ) - 1.0,
            0.0
        )

    else:

        q95_excess = np.zeros(
            X_val_620e.shape[0],
            dtype=np.float64
        )


    # ------------------------------------------------------------
    # Special near-static-sensor component
    # ------------------------------------------------------------

    if (
        sensor
        in
        special_handling_sensors_620e
    ):

        if zero_p05 > EPSILON_620E:

            zero_deficit = np.maximum(
                (
                    zero_p05
                    - val_m["zero_fraction"]
                )
                / zero_p05,
                0.0
            )

        else:

            zero_deficit = np.zeros(
                X_val_620e.shape[0],
                dtype=np.float64
            )

    else:

        zero_deficit = np.zeros(
            X_val_620e.shape[0],
            dtype=np.float64
        )


    mean_flag = (
        mean_excess > 0.0
    )

    q95_flag = (
        q95_excess > 0.0
    )

    dual_flag = (
        mean_flag
        & q95_flag
    )

    zero_flag = (
        zero_deficit > 0.0
    )


    mean_flag_columns_620e.append(
        mean_flag
    )

    q95_flag_columns_620e.append(
        q95_flag
    )

    dual_flag_columns_620e.append(
        dual_flag
    )

    zero_flag_columns_620e.append(
        zero_flag
    )


    mean_excess_columns_620e.append(
        mean_excess
    )

    q95_excess_columns_620e.append(
        q95_excess
    )

    zero_deficit_columns_620e.append(
        zero_deficit
    )


    validation_component_rows_620e.append({

        "sensor":
            sensor,

        "special_handling":
            (
                sensor
                in
                special_handling_sensors_620e
            ),

        "dev_mean_p95":
            mean_threshold,

        "dev_q95_p95":
            q95_threshold,

        "dev_zero_fraction_p05":
            zero_p05,

        "clean_val_mean_exceed_pct":
            float(
                np.mean(
                    mean_flag
                )
                * 100.0
            ),

        "clean_val_q95_exceed_pct":
            float(
                np.mean(
                    q95_flag
                )
                * 100.0
            ),

        "clean_val_dual_exceed_pct":
            float(
                np.mean(
                    dual_flag
                )
                * 100.0
            ),

        "clean_val_zero_deficit_pct":
            float(
                np.mean(
                    zero_flag
                )
                * 100.0
            ),

        "mean_excess_mean":
            float(
                np.mean(
                    mean_excess
                )
            ),

        "q95_excess_mean":
            float(
                np.mean(
                    q95_excess
                )
            ),

        "zero_deficit_mean":
            float(
                np.mean(
                    zero_deficit
                )
            ),
    })


temporal_clean_validation_620e = (
    pd.DataFrame(
        validation_component_rows_620e
    )
)


# ----------------------------------------------------------------
# 10. Build window × sensor matrices
# ----------------------------------------------------------------

mean_flag_matrix_620e = np.column_stack(
    mean_flag_columns_620e
)

q95_flag_matrix_620e = np.column_stack(
    q95_flag_columns_620e
)

dual_flag_matrix_620e = np.column_stack(
    dual_flag_columns_620e
)

zero_flag_matrix_620e = np.column_stack(
    zero_flag_columns_620e
)


mean_excess_matrix_620e = np.column_stack(
    mean_excess_columns_620e
)

q95_excess_matrix_620e = np.column_stack(
    q95_excess_columns_620e
)

zero_deficit_matrix_620e = np.column_stack(
    zero_deficit_columns_620e
)


expected_matrix_shape_620e = (
    X_val_620e.shape[0],
    len(retained_sensors_620e)
)


for matrix_name_620e, matrix_620e in {

    "mean_flag_matrix":
        mean_flag_matrix_620e,

    "q95_flag_matrix":
        q95_flag_matrix_620e,

    "dual_flag_matrix":
        dual_flag_matrix_620e,

    "zero_flag_matrix":
        zero_flag_matrix_620e,

    "mean_excess_matrix":
        mean_excess_matrix_620e,

    "q95_excess_matrix":
        q95_excess_matrix_620e,

    "zero_deficit_matrix":
        zero_deficit_matrix_620e,

}.items():

    assert (
        matrix_620e.shape
        == expected_matrix_shape_620e
    ), (
        f"{matrix_name_620e} has the wrong shape: "
        f"{matrix_620e.shape}"
    )


# ----------------------------------------------------------------
# 11. Clean-validation per-sensor false-alarm behaviour
# ----------------------------------------------------------------

print(
    "\n=== CLEAN validation candidate-component behaviour ==="
)

display(
    temporal_clean_validation_620e[
        [
            "sensor",
            "special_handling",
            "dev_mean_p95",
            "dev_q95_p95",
            "dev_zero_fraction_p05",
            "clean_val_mean_exceed_pct",
            "clean_val_q95_exceed_pct",
            "clean_val_dual_exceed_pct",
            "clean_val_zero_deficit_pct",
        ]
    ]
)


# ----------------------------------------------------------------
# 12. Window-level count behaviour
# ----------------------------------------------------------------
#
# IMPORTANT:
# These are descriptive only.
#
# We are NOT deciding that "any flag" means unreliable because
# with 15 sensors the probability of at least one clean exceedance
# can naturally be much larger than 5%.
# ----------------------------------------------------------------

mean_flag_count_620e = np.sum(
    mean_flag_matrix_620e,
    axis=1
)

q95_flag_count_620e = np.sum(
    q95_flag_matrix_620e,
    axis=1
)

dual_flag_count_620e = np.sum(
    dual_flag_matrix_620e,
    axis=1
)

zero_flag_count_620e = np.sum(
    zero_flag_matrix_620e,
    axis=1
)


def summarize_flag_counts_620e(
    name,
    counts
):

    counts = np.asarray(
        counts,
        dtype=np.float64
    )

    return {
        "candidate": name,
        "mean_flagged_sensors":
            float(
                np.mean(counts)
            ),
        "median_flagged_sensors":
            float(
                np.median(counts)
            ),
        "p95_flagged_sensors":
            float(
                np.quantile(
                    counts,
                    0.95
                )
            ),
        "maximum_flagged_sensors":
            int(
                np.max(counts)
            ),
        "windows_with_any_flag_pct":
            float(
                np.mean(
                    counts > 0
                )
                * 100.0
            ),
    }


temporal_window_summary_620e = pd.DataFrame(
    [
        summarize_flag_counts_620e(
            "mean-transition exceedance",
            mean_flag_count_620e
        ),

        summarize_flag_counts_620e(
            "q95-transition exceedance",
            q95_flag_count_620e
        ),

        summarize_flag_counts_620e(
            "dual mean+q95 exceedance",
            dual_flag_count_620e
        ),

        summarize_flag_counts_620e(
            "special zero-transition deficit",
            zero_flag_count_620e
        ),
    ]
)


print(
    "\n=== CLEAN validation window-level flag-count summary ==="
)

display(
    temporal_window_summary_620e
)


# ----------------------------------------------------------------
# 13. Overall clean-validation exceedance summaries
# ----------------------------------------------------------------

ordinary_sensor_mask_620e = np.array(
    [
        sensor
        not in
        special_handling_sensors_620e
        for sensor in retained_sensors_620e
    ],
    dtype=bool
)


if np.any(
    ordinary_sensor_mask_620e
):

    ordinary_mean_exceed_pct_620e = float(
        np.mean(
            mean_flag_matrix_620e[
                :,
                ordinary_sensor_mask_620e
            ]
        )
        * 100.0
    )

    ordinary_q95_exceed_pct_620e = float(
        np.mean(
            q95_flag_matrix_620e[
                :,
                ordinary_sensor_mask_620e
            ]
        )
        * 100.0
    )

    ordinary_dual_exceed_pct_620e = float(
        np.mean(
            dual_flag_matrix_620e[
                :,
                ordinary_sensor_mask_620e
            ]
        )
        * 100.0
    )

else:

    ordinary_mean_exceed_pct_620e = np.nan
    ordinary_q95_exceed_pct_620e = np.nan
    ordinary_dual_exceed_pct_620e = np.nan


if special_handling_sensors_620e:

    special_sensor_mask_620e = np.array(
        [
            sensor
            in
            special_handling_sensors_620e
            for sensor in retained_sensors_620e
        ],
        dtype=bool
    )

    special_zero_deficit_pct_620e = float(
        np.mean(
            zero_flag_matrix_620e[
                :,
                special_sensor_mask_620e
            ]
        )
        * 100.0
    )

else:

    special_zero_deficit_pct_620e = np.nan


print(
    "\n=== Overall CLEAN validation candidate rates ==="
)

print(
    "Ordinary sensors — mean-transition exceedance (%):",
    ordinary_mean_exceed_pct_620e
)

print(
    "Ordinary sensors — q95-transition exceedance  (%):",
    ordinary_q95_exceed_pct_620e
)

print(
    "Ordinary sensors — dual exceedance            (%):",
    ordinary_dual_exceed_pct_620e
)

print(
    "Special sensors — zero-transition deficit    (%):",
    special_zero_deficit_pct_620e
)


# ----------------------------------------------------------------
# 14. Basic numerical-integrity checks
# ----------------------------------------------------------------

continuous_matrices_620e = [
    mean_excess_matrix_620e,
    q95_excess_matrix_620e,
    zero_deficit_matrix_620e,
]


for matrix_620e in continuous_matrices_620e:

    assert np.isfinite(
        matrix_620e
    ).all(), (
        "A candidate temporal component contains "
        "NaN or infinite values."
    )

    assert np.min(
        matrix_620e
    ) >= 0.0, (
        "Candidate temporal excess components must "
        "not be negative."
    )


assert len(
    temporal_clean_validation_620e
) == len(
    retained_sensors_620e
), (
    "Per-sensor validation summary has the wrong length."
)


# ----------------------------------------------------------------
# 15. Objects intentionally retained for the NEXT step
# ----------------------------------------------------------------
#
# temporal_reference_620e
# temporal_clean_validation_620e
# temporal_window_summary_620e
#
# mean_excess_matrix_620e
# q95_excess_matrix_620e
# zero_deficit_matrix_620e
#
# mean_flag_matrix_620e
# q95_flag_matrix_620e
# dual_flag_matrix_620e
# zero_flag_matrix_620e
#
# These remain IN MEMORY only.
# ----------------------------------------------------------------


print(
    "\n=== Step 6.20E integrity summary ==="
)

print(
    "Development windows analysed :",
    X_dev_620e.shape[0]
)

print(
    "Clean validation windows      :",
    X_val_620e.shape[0]
)

print(
    "Retained sensors analysed     :",
    len(retained_sensors_620e)
)

print(
    "Special-handling sensors      :",
    special_handling_sensors_620e
)

print(
    "Maximum development round-trip error:",
    dev_roundtrip_error_620e
)

print(
    "Maximum validation round-trip error :",
    val_roundtrip_error_620e
)


print(
    "\nStep 6.20E completed successfully."
)

print(
    "Development-calibrated candidate temporal "
    "components were evaluated on CLEAN validation only."
)

print(
    "No corruption was generated."
)

print(
    "No final temporal reliability score was frozen."
)

print(
    "No PCA + temporal combination was performed."
)

print(
    "No frozen artifact or CSV file was modified."
)

In [ ]:
# ================================================================
# Step 6.20F — Two-sided temporal candidate calibration
# ================================================================
#
# Purpose
# -------
# Extend the temporal-quality candidate set so that we can detect:
#
#   1. Excessive temporal activity
#      -> noise / abrupt disturbance / some dropout or drift cases
#
#   2. Insufficient temporal activity
#      -> especially important for stuck-sensor behaviour
#
# All thresholds are calibrated using CLEAN DEVELOPMENT windows.
# CLEAN VALIDATION is used only for independent sanity checking.
#
# IMPORTANT
# ---------
# - No corruption is generated.
# - No PCA result is modified.
# - No final temporal reliability score is frozen.
# - No PCA + temporal combination is performed.
# - No frozen CSV/artifact is modified.
# ================================================================

import numpy as np
import pandas as pd
from IPython.display import display


print(
    "=== Step 6.20F — Two-sided temporal candidate calibration ==="
)


# ----------------------------------------------------------------
# 1. Required objects from Step 6.20E
# ----------------------------------------------------------------

required_objects_620f = [
    "dev_temporal_metrics_620e",
    "val_temporal_metrics_620e",
    "retained_sensors_620e",
    "special_handling_sensors_620e",
    "X_val_620e",
]

missing_objects_620f = [
    name
    for name in required_objects_620f
    if name not in globals()
]


print("\n=== Required Step-6.20E objects ===")

for name in required_objects_620f:
    print(
        f"{name:<35}:",
        name in globals()
    )


assert not missing_objects_620f, (
    "Step 6.20F cannot continue because these objects "
    f"are missing: {missing_objects_620f}"
)

print("\nAll required objects are available.")


# ----------------------------------------------------------------
# 2. Development-only TWO-SIDED reference thresholds
# ----------------------------------------------------------------
#
# Upper-tail thresholds:
#     P95
#
# Lower-tail thresholds:
#     P05
#
# Zero-transition behaviour:
#     high zero-fraction can indicate a sensor becoming stuck.
#
# No validation information is used here.
# ----------------------------------------------------------------

reference_rows_620f = []


for sensor in retained_sensors_620e:

    dev_m = dev_temporal_metrics_620e[sensor]

    reference_rows_620f.append({

        "sensor":
            sensor,

        # --------------------------------------------------------
        # Mean absolute difference
        # --------------------------------------------------------

        "mean_abs_diff_p05":
            float(
                np.quantile(
                    dev_m["mean_abs_diff"],
                    0.05
                )
            ),

        "mean_abs_diff_p50":
            float(
                np.quantile(
                    dev_m["mean_abs_diff"],
                    0.50
                )
            ),

        "mean_abs_diff_p95":
            float(
                np.quantile(
                    dev_m["mean_abs_diff"],
                    0.95
                )
            ),

        # --------------------------------------------------------
        # Q95 absolute difference
        # --------------------------------------------------------

        "q95_abs_diff_p05":
            float(
                np.quantile(
                    dev_m["q95_abs_diff"],
                    0.05
                )
            ),

        "q95_abs_diff_p50":
            float(
                np.quantile(
                    dev_m["q95_abs_diff"],
                    0.50
                )
            ),

        "q95_abs_diff_p95":
            float(
                np.quantile(
                    dev_m["q95_abs_diff"],
                    0.95
                )
            ),

        # --------------------------------------------------------
        # Zero-transition fraction
        # --------------------------------------------------------

        "zero_fraction_p05":
            float(
                np.quantile(
                    dev_m["zero_fraction"],
                    0.05
                )
            ),

        "zero_fraction_p50":
            float(
                np.quantile(
                    dev_m["zero_fraction"],
                    0.50
                )
            ),

        "zero_fraction_p95":
            float(
                np.quantile(
                    dev_m["zero_fraction"],
                    0.95
                )
            ),

        "special_handling":
            sensor in special_handling_sensors_620e,
    })


temporal_reference_620f = (
    pd.DataFrame(reference_rows_620f)
    .set_index("sensor")
)


assert temporal_reference_620f.index.is_unique


print(
    "\n=== DEVELOPMENT-only two-sided reference ==="
)

display(
    temporal_reference_620f
)


# ----------------------------------------------------------------
# 3. Candidate flags on CLEAN VALIDATION
# ----------------------------------------------------------------

EPSILON_620F = 1e-12

validation_rows_620f = []

high_mean_flags_620f = []
low_mean_flags_620f = []

high_q95_flags_620f = []
low_q95_flags_620f = []

high_zero_flags_620f = []

high_mean_excess_620f = []
low_mean_deficit_620f = []

high_q95_excess_620f = []
low_q95_deficit_620f = []

high_zero_excess_620f = []


for sensor in retained_sensors_620e:

    val_m = val_temporal_metrics_620e[sensor]

    ref = temporal_reference_620f.loc[sensor]


    mean_p05 = float(
        ref["mean_abs_diff_p05"]
    )

    mean_p95 = float(
        ref["mean_abs_diff_p95"]
    )

    q95_p05 = float(
        ref["q95_abs_diff_p05"]
    )

    q95_p95 = float(
        ref["q95_abs_diff_p95"]
    )

    zero_p95 = float(
        ref["zero_fraction_p95"]
    )


    # ============================================================
    # HIGH temporal activity
    # ============================================================

    if mean_p95 > EPSILON_620F:

        mean_high_excess = np.maximum(
            (
                val_m["mean_abs_diff"]
                / mean_p95
            ) - 1.0,
            0.0
        )

    else:

        mean_high_excess = np.zeros(
            len(val_m["mean_abs_diff"]),
            dtype=np.float64
        )


    if q95_p95 > EPSILON_620F:

        q95_high_excess = np.maximum(
            (
                val_m["q95_abs_diff"]
                / q95_p95
            ) - 1.0,
            0.0
        )

    else:

        q95_high_excess = np.zeros(
            len(val_m["q95_abs_diff"]),
            dtype=np.float64
        )


    # ============================================================
    # LOW temporal activity
    # ============================================================
    #
    # A stuck sensor may produce unusually small transitions.
    #
    # If the development P05 is exactly zero, a meaningful
    # ratio-based lower-tail test cannot be defined safely.
    # In that case we do NOT invent a threshold.
    # ============================================================

    if mean_p05 > EPSILON_620F:

        mean_low_deficit = np.maximum(
            (
                mean_p05
                - val_m["mean_abs_diff"]
            )
            / mean_p05,
            0.0
        )

    else:

        mean_low_deficit = np.zeros(
            len(val_m["mean_abs_diff"]),
            dtype=np.float64
        )


    if q95_p05 > EPSILON_620F:

        q95_low_deficit = np.maximum(
            (
                q95_p05
                - val_m["q95_abs_diff"]
            )
            / q95_p05,
            0.0
        )

    else:

        q95_low_deficit = np.zeros(
            len(val_m["q95_abs_diff"]),
            dtype=np.float64
        )


    # ============================================================
    # HIGH zero-transition fraction
    # ============================================================
    #
    # This is another stuck-sensor candidate.
    #
    # Example:
    # healthy windows may contain very few exactly unchanged
    # transitions, while a stuck sensor may contain many.
    #
    # If P95 == 1, no upward room exists, so this component is
    # undefined/uninformative and is therefore set to zero.
    # ============================================================

    if zero_p95 < (1.0 - EPSILON_620F):

        denominator = max(
            1.0 - zero_p95,
            EPSILON_620F
        )

        zero_high_excess = np.maximum(
            (
                val_m["zero_fraction"]
                - zero_p95
            )
            / denominator,
            0.0
        )

    else:

        zero_high_excess = np.zeros(
            len(val_m["zero_fraction"]),
            dtype=np.float64
        )


    # ------------------------------------------------------------
    # Boolean flags
    # ------------------------------------------------------------

    mean_high_flag = (
        mean_high_excess > 0.0
    )

    mean_low_flag = (
        mean_low_deficit > 0.0
    )

    q95_high_flag = (
        q95_high_excess > 0.0
    )

    q95_low_flag = (
        q95_low_deficit > 0.0
    )

    zero_high_flag = (
        zero_high_excess > 0.0
    )


    # ------------------------------------------------------------
    # Store matrices
    # ------------------------------------------------------------

    high_mean_flags_620f.append(
        mean_high_flag
    )

    low_mean_flags_620f.append(
        mean_low_flag
    )

    high_q95_flags_620f.append(
        q95_high_flag
    )

    low_q95_flags_620f.append(
        q95_low_flag
    )

    high_zero_flags_620f.append(
        zero_high_flag
    )


    high_mean_excess_620f.append(
        mean_high_excess
    )

    low_mean_deficit_620f.append(
        mean_low_deficit
    )

    high_q95_excess_620f.append(
        q95_high_excess
    )

    low_q95_deficit_620f.append(
        q95_low_deficit
    )

    high_zero_excess_620f.append(
        zero_high_excess
    )


    # ------------------------------------------------------------
    # Per-sensor clean-validation summary
    # ------------------------------------------------------------

    validation_rows_620f.append({

        "sensor":
            sensor,

        "special_handling":
            bool(
                ref["special_handling"]
            ),

        "mean_p05":
            mean_p05,

        "mean_p95":
            mean_p95,

        "q95_p05":
            q95_p05,

        "q95_p95":
            q95_p95,

        "zero_fraction_p95":
            zero_p95,

        "clean_high_mean_pct":
            float(
                np.mean(mean_high_flag)
                * 100.0
            ),

        "clean_low_mean_pct":
            float(
                np.mean(mean_low_flag)
                * 100.0
            ),

        "clean_high_q95_pct":
            float(
                np.mean(q95_high_flag)
                * 100.0
            ),

        "clean_low_q95_pct":
            float(
                np.mean(q95_low_flag)
                * 100.0
            ),

        "clean_high_zero_pct":
            float(
                np.mean(zero_high_flag)
                * 100.0
            ),
    })


temporal_clean_validation_620f = pd.DataFrame(
    validation_rows_620f
)


# ----------------------------------------------------------------
# 4. Convert to window × sensor matrices
# ----------------------------------------------------------------

high_mean_flag_matrix_620f = np.column_stack(
    high_mean_flags_620f
)

low_mean_flag_matrix_620f = np.column_stack(
    low_mean_flags_620f
)

high_q95_flag_matrix_620f = np.column_stack(
    high_q95_flags_620f
)

low_q95_flag_matrix_620f = np.column_stack(
    low_q95_flags_620f
)

high_zero_flag_matrix_620f = np.column_stack(
    high_zero_flags_620f
)


high_mean_excess_matrix_620f = np.column_stack(
    high_mean_excess_620f
)

low_mean_deficit_matrix_620f = np.column_stack(
    low_mean_deficit_620f
)

high_q95_excess_matrix_620f = np.column_stack(
    high_q95_excess_620f
)

low_q95_deficit_matrix_620f = np.column_stack(
    low_q95_deficit_620f
)

high_zero_excess_matrix_620f = np.column_stack(
    high_zero_excess_620f
)


expected_shape_620f = (
    X_val_620e.shape[0],
    len(retained_sensors_620e)
)


all_matrices_620f = {
    "high_mean_flag":
        high_mean_flag_matrix_620f,

    "low_mean_flag":
        low_mean_flag_matrix_620f,

    "high_q95_flag":
        high_q95_flag_matrix_620f,

    "low_q95_flag":
        low_q95_flag_matrix_620f,

    "high_zero_flag":
        high_zero_flag_matrix_620f,

    "high_mean_excess":
        high_mean_excess_matrix_620f,

    "low_mean_deficit":
        low_mean_deficit_matrix_620f,

    "high_q95_excess":
        high_q95_excess_matrix_620f,

    "low_q95_deficit":
        low_q95_deficit_matrix_620f,

    "high_zero_excess":
        high_zero_excess_matrix_620f,
}


for name, matrix in all_matrices_620f.items():

    assert matrix.shape == expected_shape_620f, (
        f"{name} has incorrect shape: "
        f"{matrix.shape}"
    )

    assert np.isfinite(matrix).all(), (
        f"{name} contains NaN/Inf values."
    )


# ----------------------------------------------------------------
# 5. Display per-sensor clean-validation behaviour
# ----------------------------------------------------------------

print(
    "\n=== CLEAN validation TWO-SIDED behaviour ==="
)

display(
    temporal_clean_validation_620f
)


# ----------------------------------------------------------------
# 6. Global per-sensor-event false-alarm rates
# ----------------------------------------------------------------

ordinary_mask_620f = np.array(
    [
        sensor
        not in
        special_handling_sensors_620e
        for sensor in retained_sensors_620e
    ],
    dtype=bool
)


def matrix_rate_pct_620f(
    matrix,
    column_mask=None
):

    if column_mask is not None:
        matrix = matrix[:, column_mask]

    if matrix.size == 0:
        return np.nan

    return float(
        np.mean(matrix)
        * 100.0
    )


global_rates_620f = pd.DataFrame(
    [
        {
            "candidate":
                "HIGH mean transition",

            "ordinary_sensor_event_rate_pct":
                matrix_rate_pct_620f(
                    high_mean_flag_matrix_620f,
                    ordinary_mask_620f
                )
        },

        {
            "candidate":
                "LOW mean transition",

            "ordinary_sensor_event_rate_pct":
                matrix_rate_pct_620f(
                    low_mean_flag_matrix_620f,
                    ordinary_mask_620f
                )
        },

        {
            "candidate":
                "HIGH q95 transition",

            "ordinary_sensor_event_rate_pct":
                matrix_rate_pct_620f(
                    high_q95_flag_matrix_620f,
                    ordinary_mask_620f
                )
        },

        {
            "candidate":
                "LOW q95 transition",

            "ordinary_sensor_event_rate_pct":
                matrix_rate_pct_620f(
                    low_q95_flag_matrix_620f,
                    ordinary_mask_620f
                )
        },

        {
            "candidate":
                "HIGH zero-transition fraction",

            "ordinary_sensor_event_rate_pct":
                matrix_rate_pct_620f(
                    high_zero_flag_matrix_620f,
                    ordinary_mask_620f
                )
        },
    ]
)


print(
    "\n=== Overall CLEAN validation ordinary-sensor rates ==="
)

display(
    global_rates_620f
)


# ----------------------------------------------------------------
# 7. Candidate overlap analysis
# ----------------------------------------------------------------
#
# For excessive activity:
#     HIGH mean AND HIGH q95
#
# For insufficient activity:
#     LOW mean OR HIGH zero fraction
#
# These are diagnostic combinations only.
# They are NOT frozen reliability formulas.
# ----------------------------------------------------------------

high_activity_combined_620f = (
    high_mean_flag_matrix_620f
    &
    high_q95_flag_matrix_620f
)


low_activity_combined_620f = (
    low_mean_flag_matrix_620f
    |
    high_zero_flag_matrix_620f
)


two_sided_any_620f = (
    high_activity_combined_620f
    |
    low_activity_combined_620f
)


combined_rates_620f = pd.DataFrame(
    [
        {
            "diagnostic_combination":
                "HIGH mean AND HIGH q95",

            "ordinary_sensor_event_rate_pct":
                matrix_rate_pct_620f(
                    high_activity_combined_620f,
                    ordinary_mask_620f
                )
        },

        {
            "diagnostic_combination":
                "LOW mean OR HIGH zero-fraction",

            "ordinary_sensor_event_rate_pct":
                matrix_rate_pct_620f(
                    low_activity_combined_620f,
                    ordinary_mask_620f
                )
        },

        {
            "diagnostic_combination":
                "Either high-activity or low-activity",

            "ordinary_sensor_event_rate_pct":
                matrix_rate_pct_620f(
                    two_sided_any_620f,
                    ordinary_mask_620f
                )
        },
    ]
)


print(
    "\n=== Diagnostic combination rates on CLEAN validation ==="
)

display(
    combined_rates_620f
)


# ----------------------------------------------------------------
# 8. Window-level multiplicity summary
# ----------------------------------------------------------------

def summarize_window_flags_620f(
    label,
    flag_matrix
):

    ordinary_flags = flag_matrix[
        :,
        ordinary_mask_620f
    ]

    counts = np.sum(
        ordinary_flags,
        axis=1
    )

    return {
        "candidate":
            label,

        "mean_flagged_sensors":
            float(
                np.mean(counts)
            ),

        "median_flagged_sensors":
            float(
                np.median(counts)
            ),

        "p95_flagged_sensors":
            float(
                np.quantile(
                    counts,
                    0.95
                )
            ),

        "maximum_flagged_sensors":
            int(
                np.max(counts)
            ),

        "windows_with_any_flag_pct":
            float(
                np.mean(
                    counts > 0
                )
                * 100.0
            ),
    }


window_summary_620f = pd.DataFrame(
    [
        summarize_window_flags_620f(
            "HIGH mean",
            high_mean_flag_matrix_620f
        ),

        summarize_window_flags_620f(
            "LOW mean",
            low_mean_flag_matrix_620f
        ),

        summarize_window_flags_620f(
            "HIGH q95",
            high_q95_flag_matrix_620f
        ),

        summarize_window_flags_620f(
            "LOW q95",
            low_q95_flag_matrix_620f
        ),

        summarize_window_flags_620f(
            "HIGH zero fraction",
            high_zero_flag_matrix_620f
        ),

        summarize_window_flags_620f(
            "HIGH activity combined",
            high_activity_combined_620f
        ),

        summarize_window_flags_620f(
            "LOW activity combined",
            low_activity_combined_620f
        ),

        summarize_window_flags_620f(
            "TWO-SIDED diagnostic",
            two_sided_any_620f
        ),
    ]
)


print(
    "\n=== CLEAN validation window-level multiplicity ==="
)

display(
    window_summary_620f
)


# ----------------------------------------------------------------
# 9. Inspect sensors with unavailable lower-tail tests
# ----------------------------------------------------------------

lower_tail_availability_rows_620f = []


for sensor in retained_sensors_620e:

    ref = temporal_reference_620f.loc[
        sensor
    ]

    lower_tail_availability_rows_620f.append(
        {
            "sensor":
                sensor,

            "mean_lower_tail_available":
                bool(
                    float(
                        ref["mean_abs_diff_p05"]
                    )
                    >
                    EPSILON_620F
                ),

            "q95_lower_tail_available":
                bool(
                    float(
                        ref["q95_abs_diff_p05"]
                    )
                    >
                    EPSILON_620F
                ),

            "high_zero_test_available":
                bool(
                    float(
                        ref["zero_fraction_p95"]
                    )
                    <
                    (
                        1.0
                        - EPSILON_620F
                    )
                ),

            "special_handling":
                bool(
                    ref["special_handling"]
                ),
        }
    )


lower_tail_availability_620f = pd.DataFrame(
    lower_tail_availability_rows_620f
)


print(
    "\n=== Lower-activity diagnostic availability ==="
)

display(
    lower_tail_availability_620f
)


# ----------------------------------------------------------------
# 10. Integrity summary
# ----------------------------------------------------------------

print(
    "\n=== Step 6.20F integrity summary ==="
)

print(
    "Development thresholds derived from      : "
    "CLEAN development only"
)

print(
    "Clean validation windows evaluated       :",
    X_val_620e.shape[0]
)

print(
    "Retained sensors                         :",
    len(retained_sensors_620e)
)

print(
    "Special-handling sensors                 :",
    special_handling_sensors_620e
)

print(
    "No corruption generated                  : True"
)

print(
    "No final temporal score frozen           : True"
)

print(
    "No PCA + temporal combination performed  : True"
)

print(
    "No frozen artifact modified              : True"
)


print(
    "\nStep 6.20F completed successfully."
)

print(
    "Two-sided temporal candidates are now calibrated "
    "using development data and sanity-checked on clean validation."
)

print(
    "The next step will test these candidate components "
    "against the frozen corruption families before any final "
    "temporal-quality formulation is selected."
)

In [ ]:
# ================================================================
# Step 6.20G-A — Frozen corruption-generator availability pre-check
# ================================================================
#
# PURPOSE
# -------
# 1. Inspect the CURRENT notebook namespace only.
# 2. Find corruption-related callable objects for:
#       - bias
#       - noise
#       - gradual drift
#       - dropout
#       - stuck sensor
# 3. Show their exact names and function signatures.
# 4. Identify previously verified candidate aliases where available.
#
# IMPORTANT
# ---------
# - NO corruption is generated.
# - NO function is called.
# - NO function is overwritten.
# - NO CSV/artifact is modified.
# - NO temporal score is created.
# - NO PCA + temporal combination is performed.
# ================================================================

import inspect
import re
import pandas as pd
import numpy as np

print(
    "\n=== Step 6.20G-A — "
    "Frozen corruption-generator availability pre-check ===\n"
)


# ------------------------------------------------
# 1. Corruption families and search vocabulary
# ------------------------------------------------

CORRUPTION_SEARCH_TERMS_620ga = {
    "bias": [
        "bias"
    ],

    "noise": [
        "noise"
    ],

    "drift": [
        "drift"
    ],

    "dropout": [
        "dropout",
        "drop_out"
    ],

    "stuck": [
        "stuck"
    ],
}


# ------------------------------------------------
# 2. Helper: safe function signature
# ------------------------------------------------

def _safe_signature_620ga(obj):
    try:
        return str(inspect.signature(obj))
    except Exception:
        return "<signature unavailable>"


# ------------------------------------------------
# 3. Helper: classify callable by corruption family
# ------------------------------------------------

def _family_matches_620ga(name):
    name_lower = str(name).lower()

    matches = []

    for family, terms in CORRUPTION_SEARCH_TERMS_620ga.items():
        if any(term in name_lower for term in terms):
            matches.append(family)

    return matches


# ------------------------------------------------
# 4. Inspect CURRENT in-memory callable objects
# ------------------------------------------------

callable_records_620ga = []

for object_name_620ga, object_value_620ga in list(globals().items()):

    # Ignore helper objects from this cell itself.
    if object_name_620ga.endswith("_620ga"):
        continue

    # We are interested in callable Python objects only.
    if not callable(object_value_620ga):
        continue

    matched_families_620ga = _family_matches_620ga(
        object_name_620ga
    )

    if len(matched_families_620ga) == 0:
        continue

    callable_records_620ga.append(
        {
            "object_name": object_name_620ga,
            "family_matches": ", ".join(
                matched_families_620ga
            ),
            "signature": _safe_signature_620ga(
                object_value_620ga
            ),
            "object_type": type(
                object_value_620ga
            ).__name__,
        }
    )


corruption_callables_620ga = pd.DataFrame(
    callable_records_620ga
)

if len(corruption_callables_620ga) > 0:

    corruption_callables_620ga = (
        corruption_callables_620ga
        .sort_values(
            [
                "family_matches",
                "object_name"
            ]
        )
        .reset_index(drop=True)
    )


print(
    "=== Corruption-related callable objects "
    "currently in memory ==="
)

if len(corruption_callables_620ga) == 0:
    print(
        "No corruption-related callable objects "
        "were found in the current namespace."
    )
else:
    display(
        corruption_callables_620ga
    )


# ------------------------------------------------
# 5. Family-level callable availability
# ------------------------------------------------

family_availability_records_620ga = []

for family_620ga in CORRUPTION_SEARCH_TERMS_620ga:

    if len(corruption_callables_620ga) == 0:

        matched_names_620ga = []

    else:

        matched_names_620ga = (
            corruption_callables_620ga.loc[
                corruption_callables_620ga[
                    "family_matches"
                ]
                .str.contains(
                    rf"(^|,\s*){re.escape(family_620ga)}($|,)",
                    regex=True
                ),
                "object_name"
            ]
            .tolist()
        )

    family_availability_records_620ga.append(
        {
            "corruption_family": family_620ga,
            "callables_found": len(
                matched_names_620ga
            ),
            "callable_names": (
                ", ".join(matched_names_620ga)
                if matched_names_620ga
                else "<none>"
            )
        }
    )


family_availability_620ga = pd.DataFrame(
    family_availability_records_620ga
)

print(
    "\n=== Family-level callable availability ==="
)

display(
    family_availability_620ga
)


# ------------------------------------------------
# 6. Inspect previously verified candidate aliases
#
# We do NOT assume every alias exists.
# These are simply searched for explicitly.
# ------------------------------------------------

known_verified_aliases_620ga = [
    # Dropout — previously verified in Step 6.17
    "candidate_add_sensor_dropout_617d",

    # Stuck sensor — previously verified in Step 6.18
    "candidate_add_stuck_sensor_618d",
]


alias_records_620ga = []

for alias_name_620ga in known_verified_aliases_620ga:

    alias_exists_620ga = (
        alias_name_620ga in globals()
        and callable(
            globals()[alias_name_620ga]
        )
    )

    alias_records_620ga.append(
        {
            "verified_alias": alias_name_620ga,
            "available": alias_exists_620ga,
            "signature": (
                _safe_signature_620ga(
                    globals()[alias_name_620ga]
                )
                if alias_exists_620ga
                else "<not currently available>"
            )
        }
    )


verified_alias_check_620ga = pd.DataFrame(
    alias_records_620ga
)

print(
    "\n=== Previously verified corruption aliases ==="
)

display(
    verified_alias_check_620ga
)


# ------------------------------------------------
# 7. Inspect corruption-related DataFrames
#
# This helps us identify validation tables and
# previously generated scenario metadata without
# modifying them.
# ------------------------------------------------

dataframe_records_620ga = []

for object_name_620ga, object_value_620ga in list(globals().items()):

    if object_name_620ga.endswith("_620ga"):
        continue

    if not isinstance(
        object_value_620ga,
        pd.DataFrame
    ):
        continue

    matched_families_620ga = _family_matches_620ga(
        object_name_620ga
    )

    if len(matched_families_620ga) == 0:
        continue

    dataframe_records_620ga.append(
        {
            "object_name": object_name_620ga,
            "family_matches": ", ".join(
                matched_families_620ga
            ),
            "rows": int(
                object_value_620ga.shape[0]
            ),
            "columns": int(
                object_value_620ga.shape[1]
            )
        }
    )


corruption_dataframes_620ga = pd.DataFrame(
    dataframe_records_620ga
)

if len(corruption_dataframes_620ga) > 0:

    corruption_dataframes_620ga = (
        corruption_dataframes_620ga
        .sort_values(
            [
                "family_matches",
                "object_name"
            ]
        )
        .reset_index(drop=True)
    )


print(
    "\n=== Corruption-related DataFrames "
    "currently in memory ==="
)

if len(corruption_dataframes_620ga) == 0:
    print(
        "No corruption-related DataFrames "
        "were found."
    )
else:
    display(
        corruption_dataframes_620ga
    )


# ------------------------------------------------
# 8. Important frozen experimental objects
# ------------------------------------------------

important_objects_620ga = [
    "X_dev_clean",
    "X_val_clean",
    "y_dev_clean",
    "y_val_clean",
    "dev_seq_meta",
    "val_seq_meta",
    "scaler",
    "PRIMARY_INPUT_COLUMNS",
    "RETAINED_SENSOR_COLUMNS",
    "SENSOR_INPUT_INDICES",
    "sensor_temporal_stats",
]


important_records_620ga = []

for object_name_620ga in important_objects_620ga:

    exists_620ga = (
        object_name_620ga in globals()
    )

    if exists_620ga:

        obj_620ga = globals()[
            object_name_620ga
        ]

        shape_620ga = getattr(
            obj_620ga,
            "shape",
            ""
        )

        important_records_620ga.append(
            {
                "object": object_name_620ga,
                "available": True,
                "type": type(
                    obj_620ga
                ).__name__,
                "shape": str(
                    shape_620ga
                )
            }
        )

    else:

        important_records_620ga.append(
            {
                "object": object_name_620ga,
                "available": False,
                "type": "",
                "shape": ""
            }
        )


important_object_check_620ga = pd.DataFrame(
    important_records_620ga
)

print(
    "\n=== Important frozen-object availability ==="
)

display(
    important_object_check_620ga
)


# ------------------------------------------------
# 9. Readiness summary
# ------------------------------------------------

important_objects_ready_620ga = bool(
    important_object_check_620ga[
        "available"
    ].all()
)

dropout_alias_ready_620ga = bool(
    verified_alias_check_620ga.loc[
        verified_alias_check_620ga[
            "verified_alias"
        ]
        ==
        "candidate_add_sensor_dropout_617d",
        "available"
    ].iloc[0]
)

stuck_alias_ready_620ga = bool(
    verified_alias_check_620ga.loc[
        verified_alias_check_620ga[
            "verified_alias"
        ]
        ==
        "candidate_add_stuck_sensor_618d",
        "available"
    ].iloc[0]
)


print(
    "\n=== Step 6.20G-A readiness summary ==="
)

print(
    "Important frozen experimental objects ready :",
    important_objects_ready_620ga
)

print(
    "Verified dropout generator available         :",
    dropout_alias_ready_620ga
)

print(
    "Verified stuck generator available           :",
    stuck_alias_ready_620ga
)

print(
    "\nBias/noise/drift callable candidates must be "
    "identified from the tables above before "
    "anything is executed."
)


print(
    "\nStep 6.20G-A completed successfully."
)

print(
    "Inspection only."
)

print(
    "No corruption function was executed."
)

print(
    "No corruption scenario was generated."
)

print(
    "No temporal score was created."
)

print(
    "No PCA + temporal combination was performed."
)

print(
    "No frozen artifact or CSV file was modified."
)

In [ ]:
# ================================================================
# Step 6.20G-B — Exact bias/noise/drift generator inspection
# ================================================================
#
# PURPOSE
# -------
# Inspect the exact in-memory implementations of:
#
#   1. add_sensor_bias
#   2. add_sensor_noise
#   3. candidate_add_sensor_drift_616f
#
# We inspect:
#   - availability
#   - callable type
#   - exact signature
#   - source line count
#   - source-code hash
#   - return statements
#   - complete source code
#
# IMPORTANT
# ---------
# NO corruption function is executed.
# NO data are modified.
# NO CSV/artifact is modified.
# NO temporal reliability score is created.
# ================================================================

import inspect
import hashlib
import textwrap
import pandas as pd

print(
    "\n=== Step 6.20G-B — "
    "Exact bias/noise/drift generator inspection ===\n"
)


# ------------------------------------------------
# 1. Exact generators to inspect
# ------------------------------------------------

generator_names_620gb = [
    "add_sensor_bias",
    "add_sensor_noise",
    "candidate_add_sensor_drift_616f",
]


# ------------------------------------------------
# 2. Safe helper functions
# ------------------------------------------------

def _safe_signature_620gb(obj):
    try:
        return str(inspect.signature(obj))
    except Exception as exc:
        return f"<signature unavailable: {exc}>"


def _safe_source_620gb(obj):
    try:
        return inspect.getsource(obj)
    except Exception as exc:
        return f"<source unavailable: {exc}>"


def _source_hash_620gb(source):
    if source.startswith("<source unavailable"):
        return "<not available>"

    return hashlib.sha256(
        source.encode("utf-8")
    ).hexdigest()


def _return_lines_620gb(source):
    if source.startswith("<source unavailable"):
        return []

    lines = source.splitlines()

    return [
        line.strip()
        for line in lines
        if line.strip().startswith("return ")
        or line.strip() == "return"
    ]


# ------------------------------------------------
# 3. Inspect availability and signatures
# ------------------------------------------------

summary_records_620gb = []

generator_sources_620gb = {}

for generator_name_620gb in generator_names_620gb:

    exists_620gb = (
        generator_name_620gb in globals()
        and callable(
            globals()[generator_name_620gb]
        )
    )

    if exists_620gb:

        generator_obj_620gb = globals()[
            generator_name_620gb
        ]

        source_620gb = _safe_source_620gb(
            generator_obj_620gb
        )

        generator_sources_620gb[
            generator_name_620gb
        ] = source_620gb

        line_count_620gb = (
            len(source_620gb.splitlines())
            if not source_620gb.startswith(
                "<source unavailable"
            )
            else None
        )

        summary_records_620gb.append(
            {
                "generator": generator_name_620gb,
                "available": True,
                "object_type": type(
                    generator_obj_620gb
                ).__name__,
                "signature": _safe_signature_620gb(
                    generator_obj_620gb
                ),
                "source_lines": line_count_620gb,
                "sha256": _source_hash_620gb(
                    source_620gb
                ),
            }
        )

    else:

        generator_sources_620gb[
            generator_name_620gb
        ] = "<not available>"

        summary_records_620gb.append(
            {
                "generator": generator_name_620gb,
                "available": False,
                "object_type": "",
                "signature": "<not available>",
                "source_lines": None,
                "sha256": "<not available>",
            }
        )


generator_summary_620gb = pd.DataFrame(
    summary_records_620gb
)


print("=== Generator summary ===")

display(
    generator_summary_620gb
)


# ------------------------------------------------
# 4. Explicit exact signatures
# ------------------------------------------------

print(
    "\n=== Exact function signatures ==="
)

for generator_name_620gb in generator_names_620gb:

    print(
        "\n",
        generator_name_620gb,
        sep=""
    )

    if (
        generator_name_620gb in globals()
        and callable(
            globals()[generator_name_620gb]
        )
    ):

        print(
            _safe_signature_620gb(
                globals()[generator_name_620gb]
            )
        )

    else:

        print("<not available>")


# ------------------------------------------------
# 5. Return-statement inspection
# ------------------------------------------------

print(
    "\n=== Return statements ==="
)

for generator_name_620gb in generator_names_620gb:

    print(
        "\n",
        generator_name_620gb,
        ":",
        sep=""
    )

    source_620gb = generator_sources_620gb[
        generator_name_620gb
    ]

    returns_620gb = _return_lines_620gb(
        source_620gb
    )

    if returns_620gb:

        for return_line_620gb in returns_620gb:
            print(
                "   ",
                return_line_620gb
            )

    else:

        print(
            "   <no directly detectable "
            "return statement>"
        )


# ------------------------------------------------
# 6. Print exact source code
# ------------------------------------------------

for generator_name_620gb in generator_names_620gb:

    print(
        "\n"
        + "=" * 78
    )

    print(
        generator_name_620gb
    )

    print(
        "=" * 78
    )

    source_620gb = generator_sources_620gb[
        generator_name_620gb
    ]

    print(
        source_620gb
    )


# ------------------------------------------------
# 7. Inspect potentially relevant global constants
#
# Do not change anything — only report availability.
# ------------------------------------------------

potential_support_objects_620gb = [
    "CORRUPTION_LEVELS",
    "sensor_reference_stats",
    "sensor_temporal_stats",
    "RETAINED_SENSOR_COLUMNS",
    "PRIMARY_INPUT_COLUMNS",
]


support_records_620gb = []

for name_620gb in potential_support_objects_620gb:

    exists_620gb = (
        name_620gb in globals()
    )

    obj_620gb = (
        globals()[name_620gb]
        if exists_620gb
        else None
    )

    support_records_620gb.append(
        {
            "object": name_620gb,
            "available": exists_620gb,
            "type": (
                type(obj_620gb).__name__
                if exists_620gb
                else ""
            ),
            "shape_or_length": (
                str(getattr(obj_620gb, "shape"))
                if exists_620gb
                and hasattr(obj_620gb, "shape")
                else (
                    str(len(obj_620gb))
                    if exists_620gb
                    and hasattr(obj_620gb, "__len__")
                    else ""
                )
            ),
        }
    )


support_check_620gb = pd.DataFrame(
    support_records_620gb
)

print(
    "\n=== Supporting frozen-object availability ==="
)

display(
    support_check_620gb
)


# ------------------------------------------------
# 8. Integrity conclusion
# ------------------------------------------------

all_three_available_620gb = bool(
    generator_summary_620gb[
        "available"
    ].all()
)

print(
    "\n=== Step 6.20G-B integrity summary ==="
)

print(
    "Bias generator available  :",
    bool(
        generator_summary_620gb.loc[
            generator_summary_620gb[
                "generator"
            ] == "add_sensor_bias",
            "available"
        ].iloc[0]
    )
)

print(
    "Noise generator available :",
    bool(
        generator_summary_620gb.loc[
            generator_summary_620gb[
                "generator"
            ] == "add_sensor_noise",
            "available"
        ].iloc[0]
    )
)

print(
    "Drift generator available :",
    bool(
        generator_summary_620gb.loc[
            generator_summary_620gb[
                "generator"
            ]
            ==
            "candidate_add_sensor_drift_616f",
            "available"
        ].iloc[0]
    )
)

print(
    "All three available       :",
    all_three_available_620gb
)

print(
    "\nStep 6.20G-B completed."
)

print(
    "Inspection only."
)

print(
    "No corruption function was executed."
)

print(
    "No data were modified."
)

print(
    "No temporal reliability score was created."
)

print(
    "No PCA + temporal combination was performed."
)

print(
    "No frozen artifact or CSV file was modified."
)

In [ ]:
# ================================================================
# Step 6.20G-B2 — Extract exact frozen bias/noise/drift definitions
#                  directly from the Step-4 notebook
# ================================================================
#
# PURPOSE
# -------
# Read the frozen Step-4 notebook itself and locate the exact
# definitions of:
#
#     add_sensor_bias
#     add_sensor_noise
#     add_sensor_drift
#
# This avoids relying on inspect.getsource() for dynamically
# loaded aliases such as candidate_add_sensor_drift_616f.
#
# IMPORTANT
# ---------
# Inspection only.
# NO corruption function is executed.
# NO notebook is modified.
# NO CSV/artifact is modified.
# ================================================================

from pathlib import Path
import json
import ast
import hashlib
import pandas as pd


print(
    "\n=== Step 6.20G-B2 — "
    "Extract frozen bias/noise/drift definitions from Step 4 ===\n"
)


# ------------------------------------------------
# 1. Locate project root safely
# ------------------------------------------------

cwd_620gb2 = Path.cwd()

if (cwd_620gb2 / "03_notebooks").exists():
    PROJECT_ROOT_620gb2 = cwd_620gb2

elif (cwd_620gb2.parent / "03_notebooks").exists():
    PROJECT_ROOT_620gb2 = cwd_620gb2.parent

else:
    raise FileNotFoundError(
        "Could not locate project root containing 03_notebooks."
    )


STEP4_NOTEBOOK_620gb2 = (
    PROJECT_ROOT_620gb2
    / "03_notebooks"
    / "04_sensor_degradation_fd001.ipynb"
)

print("Project root :", PROJECT_ROOT_620gb2)
print("Step-4 file  :", STEP4_NOTEBOOK_620gb2)
print("File exists  :", STEP4_NOTEBOOK_620gb2.exists())

assert STEP4_NOTEBOOK_620gb2.exists(), (
    "Frozen Step-4 notebook could not be found."
)


# ------------------------------------------------
# 2. Read notebook JSON
# ------------------------------------------------

with open(
    STEP4_NOTEBOOK_620gb2,
    "r",
    encoding="utf-8"
) as f:

    notebook_620gb2 = json.load(f)


cells_620gb2 = notebook_620gb2.get(
    "cells",
    []
)

print(
    "Notebook cells:",
    len(cells_620gb2)
)


# ------------------------------------------------
# 3. Functions to locate
# ------------------------------------------------

target_function_names_620gb2 = [
    "add_sensor_bias",
    "add_sensor_noise",
    "add_sensor_drift",
]


# ------------------------------------------------
# 4. AST-based exact extraction helper
# ------------------------------------------------

def extract_function_definitions_from_cell_620gb2(
    source_text,
    target_names
):
    """
    Return complete source text for any top-level function
    definitions matching target_names.
    """

    results = []

    try:
        tree = ast.parse(source_text)

    except SyntaxError:
        return results

    lines = source_text.splitlines()

    for node in tree.body:

        if (
            isinstance(
                node,
                (ast.FunctionDef, ast.AsyncFunctionDef)
            )
            and node.name in target_names
        ):

            start = node.lineno - 1

            end = getattr(
                node,
                "end_lineno",
                None
            )

            if end is None:
                continue

            function_source = "\n".join(
                lines[start:end]
            )

            results.append(
                {
                    "function_name": node.name,
                    "function_source": function_source,
                }
            )

    return results


# ------------------------------------------------
# 5. Scan every code cell
# ------------------------------------------------

found_records_620gb2 = []

for cell_index_620gb2, cell_620gb2 in enumerate(
    cells_620gb2
):

    if cell_620gb2.get("cell_type") != "code":
        continue

    source_620gb2 = "".join(
        cell_620gb2.get(
            "source",
            []
        )
    )

    extracted_620gb2 = (
        extract_function_definitions_from_cell_620gb2(
            source_620gb2,
            set(
                target_function_names_620gb2
            )
        )
    )

    for item_620gb2 in extracted_620gb2:

        fn_source_620gb2 = item_620gb2[
            "function_source"
        ]

        try:
            parsed_fn_620gb2 = ast.parse(
                fn_source_620gb2
            ).body[0]

            positional_args_620gb2 = [
                arg.arg
                for arg in parsed_fn_620gb2.args.args
            ]

            defaults_620gb2 = len(
                parsed_fn_620gb2.args.defaults
            )

            return_nodes_620gb2 = [
                node
                for node in ast.walk(
                    parsed_fn_620gb2
                )
                if isinstance(
                    node,
                    ast.Return
                )
            ]

        except Exception:

            positional_args_620gb2 = []
            defaults_620gb2 = None
            return_nodes_620gb2 = []

        found_records_620gb2.append(
            {
                "function_name":
                    item_620gb2[
                        "function_name"
                    ],

                "cell_index":
                    cell_index_620gb2,

                "source_lines":
                    len(
                        fn_source_620gb2.splitlines()
                    ),

                "positional_arguments":
                    positional_args_620gb2,

                "default_count":
                    defaults_620gb2,

                "return_count":
                    len(
                        return_nodes_620gb2
                    ),

                "sha256":
                    hashlib.sha256(
                        fn_source_620gb2.encode(
                            "utf-8"
                        )
                    ).hexdigest(),

                "function_source":
                    fn_source_620gb2,
            }
        )


found_df_620gb2 = pd.DataFrame(
    found_records_620gb2
)


# ------------------------------------------------
# 6. Structural summary
# ------------------------------------------------

print(
    "\n=== Frozen Step-4 definition summary ==="
)

if found_df_620gb2.empty:

    print(
        "No matching function definitions were found."
    )

else:

    display(
        found_df_620gb2[
            [
                "function_name",
                "cell_index",
                "source_lines",
                "positional_arguments",
                "default_count",
                "return_count",
                "sha256",
            ]
        ]
    )


# ------------------------------------------------
# 7. Count definitions per function
# ------------------------------------------------

count_records_620gb2 = []

for target_name_620gb2 in (
    target_function_names_620gb2
):

    if found_df_620gb2.empty:

        count_620gb2 = 0

    else:

        count_620gb2 = int(
            (
                found_df_620gb2[
                    "function_name"
                ]
                == target_name_620gb2
            ).sum()
        )

    count_records_620gb2.append(
        {
            "function_name":
                target_name_620gb2,

            "definitions_found":
                count_620gb2,
        }
    )


definition_count_df_620gb2 = pd.DataFrame(
    count_records_620gb2
)

print(
    "\n=== Definition counts ==="
)

display(
    definition_count_df_620gb2
)


# ------------------------------------------------
# 8. Print every exact extracted source
# ------------------------------------------------

if not found_df_620gb2.empty:

    for _, row_620gb2 in (
        found_df_620gb2
        .sort_values(
            [
                "function_name",
                "cell_index"
            ]
        )
        .iterrows()
    ):

        print(
            "\n"
            + "=" * 82
        )

        print(
            "FUNCTION :",
            row_620gb2[
                "function_name"
            ]
        )

        print(
            "CELL     :",
            row_620gb2[
                "cell_index"
            ]
        )

        print(
            "SHA-256  :",
            row_620gb2[
                "sha256"
            ]
        )

        print(
            "=" * 82
        )

        print(
            row_620gb2[
                "function_source"
            ]
        )


# ------------------------------------------------
# 9. Identify latest definition of each function
#
# If a function was redefined later in Step 4,
# the highest cell index is the final notebook version.
# ------------------------------------------------

latest_records_620gb2 = []

for target_name_620gb2 in (
    target_function_names_620gb2
):

    matches_620gb2 = (
        found_df_620gb2[
            found_df_620gb2[
                "function_name"
            ]
            == target_name_620gb2
        ].copy()
        if not found_df_620gb2.empty
        else pd.DataFrame()
    )

    if matches_620gb2.empty:

        latest_records_620gb2.append(
            {
                "function_name":
                    target_name_620gb2,

                "latest_cell":
                    None,

                "latest_source_lines":
                    None,

                "latest_sha256":
                    None,

                "found":
                    False,
            }
        )

    else:

        latest_row_620gb2 = (
            matches_620gb2
            .sort_values(
                "cell_index"
            )
            .iloc[-1]
        )

        latest_records_620gb2.append(
            {
                "function_name":
                    target_name_620gb2,

                "latest_cell":
                    int(
                        latest_row_620gb2[
                            "cell_index"
                        ]
                    ),

                "latest_source_lines":
                    int(
                        latest_row_620gb2[
                            "source_lines"
                        ]
                    ),

                "latest_sha256":
                    latest_row_620gb2[
                        "sha256"
                    ],

                "found":
                    True,
            }
        )


latest_df_620gb2 = pd.DataFrame(
    latest_records_620gb2
)

print(
    "\n=== Latest frozen Step-4 definitions ==="
)

display(
    latest_df_620gb2
)


# ------------------------------------------------
# 10. Integrity summary
# ------------------------------------------------

all_found_620gb2 = bool(
    latest_df_620gb2[
        "found"
    ].all()
)


print(
    "\n=== Step 6.20G-B2 integrity summary ==="
)

print(
    "Bias definition found  :",
    bool(
        latest_df_620gb2.loc[
            latest_df_620gb2[
                "function_name"
            ]
            == "add_sensor_bias",
            "found"
        ].iloc[0]
    )
)

print(
    "Noise definition found :",
    bool(
        latest_df_620gb2.loc[
            latest_df_620gb2[
                "function_name"
            ]
            == "add_sensor_noise",
            "found"
        ].iloc[0]
    )
)

print(
    "Drift definition found :",
    bool(
        latest_df_620gb2.loc[
            latest_df_620gb2[
                "function_name"
            ]
            == "add_sensor_drift",
            "found"
        ].iloc[0]
    )
)

print(
    "All three found        :",
    all_found_620gb2
)


print(
    "\nStep 6.20G-B2 completed."
)

print(
    "Inspection only."
)

print(
    "No corruption function was executed."
)

print(
    "No notebook or frozen artifact was modified."
)

In [ ]:
# ======================================================================
# Step 6.20G-C — Evaluate two-sided temporal diagnostic across all
#                 five frozen corruption families
# ======================================================================
#
# PURPOSE
# -------
# Evaluate the candidate temporal-quality components from Step 6.20F
# against the exact frozen Step-4 corruption families:
#
#     bias
#     noise
#     drift
#     dropout
#     stuck sensor
#
# IMPORTANT DESIGN RULES
# ----------------------
# 1. Development data determine temporal thresholds.
# 2. Clean validation is only a sanity-check/reference set.
# 3. Existing frozen corruption scenario manifests determine the
#    corruption configurations.
# 4. The original frozen result CSV files are NOT modified.
# 5. No final temporal-quality score is frozen in this step.
# 6. No PCA + temporal combination is performed yet.
#
# TEMPORAL CANDIDATES
# -------------------
# Ordinary retained sensors:
#
#   HIGH-ACTIVITY:
#       mean absolute transition > development P95
#       AND
#       q95 absolute transition > development P95
#
#   LOW-ACTIVITY:
#       mean absolute transition < development P05
#       OR
#       zero-transition fraction > development P95
#
# Special sensor(s), currently sensor_6:
#
#   SPECIAL ZERO-TRANSITION DEFICIT:
#       zero-transition fraction < development P05
#
# TWO-SIDED DIAGNOSTIC:
#
#       HIGH activity
#       OR LOW activity
#       OR special zero-transition deficit
#
# ======================================================================

from pathlib import Path
import numpy as np
import pandas as pd
import ast
import warnings
import time


print(
    "\n=== Step 6.20G-C — "
    "Temporal diagnostic across frozen corruption families ===\n"
)


# ======================================================================
# 1. REQUIRED OBJECT CHECK
# ======================================================================

required_objects_620gc = [
    "X_dev_clean",
    "X_val_clean",
    "scaler",
    "PRIMARY_INPUT_COLUMNS",
    "RETAINED_SENSOR_COLUMNS",
    "SENSOR_INPUT_INDICES",
    "sensor_reference_stats",
    "sensor_temporal_stats",
    "CORRUPTION_LEVELS",
    "found_df_620gb2",
    "candidate_add_sensor_dropout_617d",
    "candidate_add_stuck_sensor_618d",
]

print("=== Required objects ===")

missing_objects_620gc = []

for name_620gc in required_objects_620gc:

    available_620gc = (
        name_620gc in globals()
    )

    print(
        f"{name_620gc:<38}: "
        f"{available_620gc}"
    )

    if not available_620gc:
        missing_objects_620gc.append(
            name_620gc
        )


assert not missing_objects_620gc, (
    "Missing required objects: "
    + ", ".join(
        missing_objects_620gc
    )
)

print(
    "\nAll required objects are available."
)


# ======================================================================
# 2. LOCATE PROJECT ROOT AND FROZEN PCA RESPONSE FILES
# ======================================================================

cwd_620gc = Path.cwd()

if (cwd_620gc / "06_results").exists():

    PROJECT_ROOT_620gc = cwd_620gc

elif (cwd_620gc.parent / "06_results").exists():

    PROJECT_ROOT_620gc = cwd_620gc.parent

else:

    raise FileNotFoundError(
        "Could not locate project root."
    )


RESULT_DIR_620gc = (
    PROJECT_ROOT_620gc
    / "06_results"
    / "step6_reliability_signals"
)


frozen_result_paths_620gc = {

    "bias":
        RESULT_DIR_620gc
        / "fd001_pca_bias_response_90_scenarios.csv",

    "noise":
        RESULT_DIR_620gc
        / "fd001_pca_noise_response_45_scenarios.csv",

    "drift":
        RESULT_DIR_620gc
        / "fd001_pca_drift_response_90_scenarios.csv",

    "dropout":
        RESULT_DIR_620gc
        / "fd001_pca_dropout_response_45_scenarios.csv",

    "stuck":
        RESULT_DIR_620gc
        / "fd001_pca_stuck_response_45_scenarios.csv",
}


print(
    "\n=== Frozen scenario-manifest files ==="
)

manifest_check_records_620gc = []

for family_620gc, path_620gc in (
    frozen_result_paths_620gc.items()
):

    manifest_check_records_620gc.append(
        {
            "corruption":
                family_620gc,

            "filename":
                path_620gc.name,

            "exists":
                path_620gc.exists(),
        }
    )


manifest_check_df_620gc = pd.DataFrame(
    manifest_check_records_620gc
)

display(
    manifest_check_df_620gc
)


assert manifest_check_df_620gc[
    "exists"
].all(), (
    "One or more frozen PCA response files "
    "could not be located."
)


# ======================================================================
# 3. LOAD SCENARIO MANIFESTS
# ======================================================================

scenario_manifests_620gc = {}

for family_620gc, path_620gc in (
    frozen_result_paths_620gc.items()
):

    df_620gc = pd.read_csv(
        path_620gc
    )

    scenario_manifests_620gc[
        family_620gc
    ] = df_620gc.copy()


print(
    "\n=== Frozen scenario counts ==="
)

scenario_count_records_620gc = []

expected_counts_620gc = {
    "bias": 90,
    "noise": 45,
    "drift": 90,
    "dropout": 45,
    "stuck": 45,
}

for family_620gc, df_620gc in (
    scenario_manifests_620gc.items()
):

    scenario_count_records_620gc.append(
        {
            "corruption":
                family_620gc,

            "rows":
                len(df_620gc),

            "expected_rows":
                expected_counts_620gc[
                    family_620gc
                ],

            "rows_match":
                len(df_620gc)
                == expected_counts_620gc[
                    family_620gc
                ],
        }
    )


scenario_count_df_620gc = pd.DataFrame(
    scenario_count_records_620gc
)

display(
    scenario_count_df_620gc
)

assert scenario_count_df_620gc[
    "rows_match"
].all()


# ======================================================================
# 4. LOAD EXACT FINAL FROZEN STEP-4 BIAS / NOISE / DRIFT FUNCTIONS
# ======================================================================

print(
    "\n=== Loading exact latest frozen Step-4 generators ==="
)


def latest_frozen_source_620gc(
    function_name
):

    matches = (
        found_df_620gb2[
            found_df_620gb2[
                "function_name"
            ]
            == function_name
        ]
        .sort_values(
            "cell_index"
        )
    )

    assert len(matches) > 0, (
        f"No frozen definition found for "
        f"{function_name}."
    )

    return (
        matches.iloc[-1][
            "function_source"
        ],
        int(
            matches.iloc[-1][
                "cell_index"
            ]
        )
    )


bias_source_620gc, bias_cell_620gc = (
    latest_frozen_source_620gc(
        "add_sensor_bias"
    )
)

noise_source_620gc, noise_cell_620gc = (
    latest_frozen_source_620gc(
        "add_sensor_noise"
    )
)

drift_source_620gc, drift_cell_620gc = (
    latest_frozen_source_620gc(
        "add_sensor_drift"
    )
)


# Separate namespace so current notebook functions
# are not overwritten.

frozen_generator_namespace_620gc = {

    "np":
        np,

    "pd":
        pd,

    "CORRUPTION_LEVELS":
        CORRUPTION_LEVELS,

    "RETAINED_SENSOR_COLUMNS":
        RETAINED_SENSOR_COLUMNS,

    "PRIMARY_INPUT_COLUMNS":
        PRIMARY_INPUT_COLUMNS,

    "sensor_reference_stats":
        sensor_reference_stats,

    "sensor_temporal_stats":
        sensor_temporal_stats,
}


exec(
    bias_source_620gc,
    frozen_generator_namespace_620gc
)

frozen_add_sensor_bias_620gc = (
    frozen_generator_namespace_620gc[
        "add_sensor_bias"
    ]
)


exec(
    noise_source_620gc,
    frozen_generator_namespace_620gc
)

frozen_add_sensor_noise_620gc = (
    frozen_generator_namespace_620gc[
        "add_sensor_noise"
    ]
)


exec(
    drift_source_620gc,
    frozen_generator_namespace_620gc
)

frozen_add_sensor_drift_620gc = (
    frozen_generator_namespace_620gc[
        "add_sensor_drift"
    ]
)


print(
    "Bias frozen cell :",
    bias_cell_620gc
)

print(
    "Noise frozen cell:",
    noise_cell_620gc
)

print(
    "Drift frozen cell:",
    drift_cell_620gc
)


assert bias_cell_620gc == 23
assert noise_cell_620gc == 14
assert drift_cell_620gc == 34


print(
    "Exact final frozen Step-4 generators loaded "
    "into temporary namespace."
)


# ======================================================================
# 5. RECONSTRUCT RAW VALIDATION DATAFRAME
# ======================================================================

print(
    "\n=== Raw validation reconstruction ==="
)


# Prefer an already verified raw validation object if available.
raw_candidate_names_620gc = [
    "val_raw_618d",
    "val_raw_617d",
    "val_raw_616i",
    "val_raw_for_noise",
]


val_raw_base_620gc = None
selected_raw_name_620gc = None


for name_620gc in raw_candidate_names_620gc:

    if (
        name_620gc in globals()
        and isinstance(
            globals()[name_620gc],
            pd.DataFrame
        )
    ):

        candidate_df_620gc = (
            globals()[
                name_620gc
            ]
        )

        if (
            len(candidate_df_620gc)
            == 4070
            and
            set(
                PRIMARY_INPUT_COLUMNS
            ).issubset(
                candidate_df_620gc.columns
            )
        ):

            val_raw_base_620gc = (
                candidate_df_620gc.copy()
            )

            selected_raw_name_620gc = (
                name_620gc
            )

            break


# Fallback: inverse-transform val_scaled.
if val_raw_base_620gc is None:

    assert "val_scaled" in globals(), (
        "Could not find a verified raw validation "
        "DataFrame and val_scaled is unavailable."
    )

    val_raw_base_620gc = (
        val_scaled.copy()
    )

    val_raw_base_620gc[
        PRIMARY_INPUT_COLUMNS
    ] = scaler.inverse_transform(
        val_scaled[
            PRIMARY_INPUT_COLUMNS
        ].to_numpy(
            dtype=np.float64
        )
    )

    selected_raw_name_620gc = (
        "reconstructed_from_val_scaled"
    )


print(
    "Raw validation source :",
    selected_raw_name_620gc
)

print(
    "Raw validation shape  :",
    val_raw_base_620gc.shape
)


assert (
    len(val_raw_base_620gc)
    == 4070
)

assert {
    "unit_id",
    "cycle",
}.issubset(
    val_raw_base_620gc.columns
)


# ======================================================================
# 6. RETAINED SENSOR MAPPING
# ======================================================================

retained_sensors_620gc = list(
    RETAINED_SENSOR_COLUMNS
)


sensor_feature_indices_620gc = [

    PRIMARY_INPUT_COLUMNS.index(
        sensor_620gc
    )

    for sensor_620gc
    in retained_sensors_620gc
]


assert (
    len(retained_sensors_620gc)
    == 15
)


print(
    "\nRetained sensors:",
    len(retained_sensors_620gc)
)


# ======================================================================
# 7. BUILD RAW SENSOR WINDOW ARRAY FROM DATAFRAME
# ======================================================================

WINDOW_SIZE_620gc = 30


def dataframe_to_raw_sensor_windows_620gc(
    df
):
    """
    Convert a validation DataFrame into raw retained-sensor
    rolling windows.

    Returns
    -------
    ndarray
        Shape:
            (n_windows, 30, 15)
    """

    windows = []

    ordered_df = (
        df.sort_values(
            [
                "unit_id",
                "cycle"
            ]
        )
    )

    for unit_id, group in ordered_df.groupby(
        "unit_id",
        sort=False
    ):

        group = group.sort_values(
            "cycle"
        )

        sensor_matrix = (
            group[
                retained_sensors_620gc
            ]
            .to_numpy(
                dtype=np.float64
            )
        )

        n_rows = len(
            sensor_matrix
        )

        if n_rows < WINDOW_SIZE_620gc:
            continue

        for end_position in range(
            WINDOW_SIZE_620gc,
            n_rows + 1
        ):

            start_position = (
                end_position
                - WINDOW_SIZE_620gc
            )

            windows.append(
                sensor_matrix[
                    start_position:
                    end_position
                ]
            )


    return np.stack(
        windows,
        axis=0
    )


# ======================================================================
# 8. CLEAN VALIDATION WINDOW ALIGNMENT CHECK
# ======================================================================

X_val_raw_sensor_620gc = (
    dataframe_to_raw_sensor_windows_620gc(
        val_raw_base_620gc
    )
)


print(
    "\n=== Clean validation window reconstruction ==="
)

print(
    "Raw sensor window shape:",
    X_val_raw_sensor_620gc.shape
)


assert (
    X_val_raw_sensor_620gc.shape
    == (
        3490,
        30,
        15
    )
)


# Reconstruct the same retained-sensor windows from
# the previously frozen CLEAN validation sequence array.

X_val_clean_float64_620gc = (
    np.asarray(
        X_val_clean,
        dtype=np.float64
    )
)

X_val_clean_flat_620gc = (
    X_val_clean_float64_620gc.reshape(
        -1,
        len(
            PRIMARY_INPUT_COLUMNS
        )
    )
)

X_val_clean_raw_flat_620gc = (
    scaler.inverse_transform(
        X_val_clean_flat_620gc
    )
)

X_val_clean_raw_620gc = (
    X_val_clean_raw_flat_620gc.reshape(
        X_val_clean_float64_620gc.shape
    )
)

X_val_clean_raw_sensor_620gc = (
    X_val_clean_raw_620gc[
        :,
        :,
        sensor_feature_indices_620gc
    ]
)


clean_alignment_error_620gc = float(
    np.max(
        np.abs(
            X_val_raw_sensor_620gc
            -
            X_val_clean_raw_sensor_620gc
        )
    )
)


print(
    "Maximum clean-window alignment error:",
    clean_alignment_error_620gc
)


# Small non-zero differences may occur because X_val_clean
# was stored as float32.
assert (
    clean_alignment_error_620gc
    < 0.01
), (
    "Raw validation windows do not align with "
    "the frozen clean validation sequence array."
)


print(
    "Clean validation window alignment verified."
)


# ======================================================================
# 9. TEMPORAL METRIC FUNCTION
# ======================================================================

def temporal_metrics_620gc(
    X_sensor_raw
):
    """
    Compute temporal metrics for retained sensor windows.

    Parameters
    ----------
    X_sensor_raw:
        shape = (windows, 30, 15)

    Returns
    -------
    dict of arrays:
        mean_abs_diff
        q95_abs_diff
        zero_fraction
    """

    transitions = np.diff(
        X_sensor_raw,
        axis=1
    )

    abs_transitions = np.abs(
        transitions
    )

    mean_abs_diff = np.mean(
        abs_transitions,
        axis=1
    )

    q95_abs_diff = np.quantile(
        abs_transitions,
        0.95,
        axis=1
    )

    zero_fraction = np.mean(
        abs_transitions == 0.0,
        axis=1
    )

    return {
        "mean_abs_diff":
            mean_abs_diff,

        "q95_abs_diff":
            q95_abs_diff,

        "zero_fraction":
            zero_fraction,
    }


# ======================================================================
# 10. DEVELOPMENT RAW WINDOWS
# ======================================================================

X_dev_clean_float64_620gc = (
    np.asarray(
        X_dev_clean,
        dtype=np.float64
    )
)

X_dev_flat_620gc = (
    X_dev_clean_float64_620gc.reshape(
        -1,
        len(
            PRIMARY_INPUT_COLUMNS
        )
    )
)

X_dev_raw_flat_620gc = (
    scaler.inverse_transform(
        X_dev_flat_620gc
    )
)

X_dev_raw_620gc = (
    X_dev_raw_flat_620gc.reshape(
        X_dev_clean_float64_620gc.shape
    )
)

X_dev_raw_sensor_620gc = (
    X_dev_raw_620gc[
        :,
        :,
        sensor_feature_indices_620gc
    ]
)


dev_metrics_620gc = (
    temporal_metrics_620gc(
        X_dev_raw_sensor_620gc
    )
)

clean_val_metrics_620gc = (
    temporal_metrics_620gc(
        X_val_raw_sensor_620gc
    )
)


# ======================================================================
# 11. DEVELOPMENT-ONLY TWO-SIDED REFERENCE
# ======================================================================

reference_records_620gc = []


for sensor_idx_620gc, sensor_620gc in enumerate(
    retained_sensors_620gc
):

    dev_mean_values_620gc = (
        dev_metrics_620gc[
            "mean_abs_diff"
        ][
            :,
            sensor_idx_620gc
        ]
    )

    dev_q95_values_620gc = (
        dev_metrics_620gc[
            "q95_abs_diff"
        ][
            :,
            sensor_idx_620gc
        ]
    )

    dev_zero_values_620gc = (
        dev_metrics_620gc[
            "zero_fraction"
        ][
            :,
            sensor_idx_620gc
        ]
    )


    special_620gc = (
        sensor_620gc
        == "sensor_6"
    )


    reference_records_620gc.append(
        {
            "sensor":
                sensor_620gc,

            "mean_p05":
                float(
                    np.quantile(
                        dev_mean_values_620gc,
                        0.05
                    )
                ),

            "mean_p50":
                float(
                    np.quantile(
                        dev_mean_values_620gc,
                        0.50
                    )
                ),

            "mean_p95":
                float(
                    np.quantile(
                        dev_mean_values_620gc,
                        0.95
                    )
                ),

            "q95_p05":
                float(
                    np.quantile(
                        dev_q95_values_620gc,
                        0.05
                    )
                ),

            "q95_p50":
                float(
                    np.quantile(
                        dev_q95_values_620gc,
                        0.50
                    )
                ),

            "q95_p95":
                float(
                    np.quantile(
                        dev_q95_values_620gc,
                        0.95
                    )
                ),

            "zero_p05":
                float(
                    np.quantile(
                        dev_zero_values_620gc,
                        0.05
                    )
                ),

            "zero_p50":
                float(
                    np.quantile(
                        dev_zero_values_620gc,
                        0.50
                    )
                ),

            "zero_p95":
                float(
                    np.quantile(
                        dev_zero_values_620gc,
                        0.95
                    )
                ),

            "special_handling":
                special_620gc,
        }
    )


temporal_reference_620gc = (
    pd.DataFrame(
        reference_records_620gc
    )
    .set_index(
        "sensor"
    )
)


print(
    "\n=== Development-only temporal reference reconstructed ==="
)

display(
    temporal_reference_620gc
)


# ======================================================================
# 12. DIAGNOSTIC FLAG FUNCTION
# ======================================================================

def temporal_diagnostic_620gc(
    metrics
):
    """
    Compute candidate temporal diagnostic flags.

    Returns sensor-level boolean matrices with shape:
        (n_windows, 15)
    """

    mean_values = (
        metrics[
            "mean_abs_diff"
        ]
    )

    q95_values = (
        metrics[
            "q95_abs_diff"
        ]
    )

    zero_values = (
        metrics[
            "zero_fraction"
        ]
    )


    n_windows = (
        mean_values.shape[0]
    )


    high_activity = np.zeros(
        (
            n_windows,
            len(
                retained_sensors_620gc
            )
        ),
        dtype=bool
    )

    low_activity = np.zeros_like(
        high_activity
    )

    special_deficit = np.zeros_like(
        high_activity
    )


    for sensor_idx, sensor in enumerate(
        retained_sensors_620gc
    ):

        ref = (
            temporal_reference_620gc.loc[
                sensor
            ]
        )


        if bool(
            ref[
                "special_handling"
            ]
        ):

            # Special handling for sensor_6:
            # unusually FEW zero transitions.

            special_deficit[
                :,
                sensor_idx
            ] = (
                zero_values[
                    :,
                    sensor_idx
                ]
                <
                float(
                    ref[
                        "zero_p05"
                    ]
                )
            )

        else:

            # High temporal activity:
            # both mean and q95 transitions unusually large.

            high_activity[
                :,
                sensor_idx
            ] = (
                (
                    mean_values[
                        :,
                        sensor_idx
                    ]
                    >
                    float(
                        ref[
                            "mean_p95"
                        ]
                    )
                )
                &
                (
                    q95_values[
                        :,
                        sensor_idx
                    ]
                    >
                    float(
                        ref[
                            "q95_p95"
                        ]
                    )
                )
            )


            # Low temporal activity:
            # mean transition unusually small
            # OR too many exact-zero transitions.

            low_activity[
                :,
                sensor_idx
            ] = (
                (
                    mean_values[
                        :,
                        sensor_idx
                    ]
                    <
                    float(
                        ref[
                            "mean_p05"
                        ]
                    )
                )
                |
                (
                    zero_values[
                        :,
                        sensor_idx
                    ]
                    >
                    float(
                        ref[
                            "zero_p95"
                        ]
                    )
                )
            )


    two_sided = (
        high_activity
        |
        low_activity
        |
        special_deficit
    )


    return {
        "high_activity":
            high_activity,

        "low_activity":
            low_activity,

        "special_deficit":
            special_deficit,

        "two_sided":
            two_sided,
    }


# ======================================================================
# 13. SUMMARY FUNCTION
# ======================================================================

def summarize_temporal_flags_620gc(
    flags
):
    """
    Produce scenario-level temporal-diagnostic summaries.
    """

    high_activity = (
        flags[
            "high_activity"
        ]
    )

    low_activity = (
        flags[
            "low_activity"
        ]
    )

    special_deficit = (
        flags[
            "special_deficit"
        ]
    )

    two_sided = (
        flags[
            "two_sided"
        ]
    )


    flagged_per_window = np.sum(
        two_sided,
        axis=1
    )


    return {

        "high_activity_sensor_event_pct":
            float(
                100.0
                * np.mean(
                    high_activity
                )
            ),

        "low_activity_sensor_event_pct":
            float(
                100.0
                * np.mean(
                    low_activity
                )
            ),

        "special_deficit_sensor_event_pct":
            float(
                100.0
                * np.mean(
                    special_deficit
                )
            ),

        "two_sided_sensor_event_pct":
            float(
                100.0
                * np.mean(
                    two_sided
                )
            ),

        "mean_flagged_sensors":
            float(
                np.mean(
                    flagged_per_window
                )
            ),

        "median_flagged_sensors":
            float(
                np.median(
                    flagged_per_window
                )
            ),

        "p95_flagged_sensors":
            float(
                np.quantile(
                    flagged_per_window,
                    0.95
                )
            ),

        "maximum_flagged_sensors":
            int(
                np.max(
                    flagged_per_window
                )
            ),

        "windows_with_any_flag_pct":
            float(
                100.0
                * np.mean(
                    flagged_per_window
                    > 0
                )
            ),
    }


# ======================================================================
# 14. CLEAN VALIDATION BASELINE
# ======================================================================

clean_flags_620gc = (
    temporal_diagnostic_620gc(
        clean_val_metrics_620gc
    )
)

clean_summary_620gc = (
    summarize_temporal_flags_620gc(
        clean_flags_620gc
    )
)


print(
    "\n=== CLEAN validation temporal baseline ==="
)

clean_summary_df_620gc = pd.DataFrame(
    [
        clean_summary_620gc
    ]
)

display(
    clean_summary_df_620gc
)


# Expected from Step 6.20F:
#
# two-sided sensor-event rate should be approximately
# 8.64% before special-handling contribution differences.
#
# windows_with_any_flag_pct should be approximately 70%.
#
# We do not require exact equality because the previous
# display combined diagnostic subcomponents separately.


# ======================================================================
# 15. FAMILY-SPECIFIC CORRUPTION EXECUTION
# ======================================================================

def extract_corrupted_dataframe_620gc(
    family,
    result
):
    """
    Extract the corrupted/imputed DataFrame from the
    frozen corruption-generator return structure.
    """

    if family == "dropout":

        # Verified frozen dropout structure:
        #
        # missing_df,
        # imputed_df,
        # drop_mask,
        # impute_value
        #
        # PCA and downstream models operate on the
        # imputed version.

        assert isinstance(
            result,
            (tuple, list)
        )

        assert len(result) >= 2

        corrupted_df = result[1]


    elif family == "stuck":

        # Verified structure:
        #
        # corrupted,
        # affected_mask,
        # stuck_info

        assert isinstance(
            result,
            (tuple, list)
        )

        corrupted_df = result[0]


    else:

        # Bias/noise/drift:
        # frozen functions return corrupted DataFrame
        # as their first returned object.

        if isinstance(
            result,
            pd.DataFrame
        ):

            corrupted_df = result

        else:

            assert isinstance(
                result,
                (tuple, list)
            )

            assert len(result) >= 1

            corrupted_df = result[0]


    assert isinstance(
        corrupted_df,
        pd.DataFrame
    )

    assert (
        len(corrupted_df)
        == len(
            val_raw_base_620gc
        )
    )

    return corrupted_df.copy()


def apply_frozen_corruption_620gc(
    family,
    scenario_row
):
    """
    Apply one exact frozen corruption scenario.
    """

    sensor = str(
        scenario_row[
            "sensor"
        ]
    )

    severity = str(
        scenario_row[
            "severity"
        ]
    )


    if family == "bias":

        direction = int(
            scenario_row[
                "direction"
            ]
        )

        result = (
            frozen_add_sensor_bias_620gc(
                val_raw_base_620gc,
                sensor,
                severity,
                direction,
                sensor_reference_stats
            )
        )


    elif family == "noise":

        seed = (
            int(
                scenario_row[
                    "seed"
                ]
            )
            if "seed" in scenario_row.index
            and pd.notna(
                scenario_row[
                    "seed"
                ]
            )
            else 42
        )

        result = (
            frozen_add_sensor_noise_620gc(
                val_raw_base_620gc,
                sensor,
                severity,
                sensor_reference_stats,
                seed=seed
            )
        )


    elif family == "drift":

        direction = int(
            scenario_row[
                "direction"
            ]
        )

        result = (
            frozen_add_sensor_drift_620gc(
                val_raw_base_620gc,
                sensor,
                severity,
                direction,
                sensor_reference_stats
            )
        )


    elif family == "dropout":

        dropout_rate = float(
            scenario_row[
                "requested_fraction"
            ]
        )

        result = (
            candidate_add_sensor_dropout_617d(
                val_raw_base_620gc,
                sensor,
                dropout_rate,
                sensor_reference_stats,
                seed=42
            )
        )


    elif family == "stuck":

        affected_fraction = float(
            scenario_row[
                "requested_fraction"
            ]
        )

        result = (
            candidate_add_stuck_sensor_618d(
                val_raw_base_620gc,
                sensor,
                affected_fraction
            )
        )


    else:

        raise ValueError(
            f"Unknown corruption family: {family}"
        )


    return extract_corrupted_dataframe_620gc(
        family,
        result
    )


# ======================================================================
# 16. ONE-SCENARIO SMOKE TEST FOR EACH FAMILY
# ======================================================================

print(
    "\n=== One-scenario execution smoke test ==="
)

smoke_records_620gc = []


for family_620gc in [
    "bias",
    "noise",
    "drift",
    "dropout",
    "stuck",
]:

    manifest_620gc = (
        scenario_manifests_620gc[
            family_620gc
        ]
    )

    scenario_row_620gc = (
        manifest_620gc.iloc[0]
    )


    corrupted_df_620gc = (
        apply_frozen_corruption_620gc(
            family_620gc,
            scenario_row_620gc
        )
    )


    non_target_columns_620gc = [
        column_620gc
        for column_620gc
        in val_raw_base_620gc.columns
        if column_620gc
        != scenario_row_620gc[
            "sensor"
        ]
    ]


    max_other_column_change_620gc = float(
        np.max(
            np.abs(
                corrupted_df_620gc[
                    non_target_columns_620gc
                ]
                .select_dtypes(
                    include=[
                        np.number
                    ]
                )
                .to_numpy(
                    dtype=np.float64
                )
                -
                val_raw_base_620gc[
                    non_target_columns_620gc
                ]
                .select_dtypes(
                    include=[
                        np.number
                    ]
                )
                .to_numpy(
                    dtype=np.float64
                )
            )
        )
    )


    target_sensor_620gc = str(
        scenario_row_620gc[
            "sensor"
        ]
    )


    target_changed_count_620gc = int(
        np.sum(
            ~np.isclose(
                corrupted_df_620gc[
                    target_sensor_620gc
                ].to_numpy(
                    dtype=np.float64
                ),
                val_raw_base_620gc[
                    target_sensor_620gc
                ].to_numpy(
                    dtype=np.float64
                ),
                rtol=0.0,
                atol=0.0
            )
        )
    )


    smoke_records_620gc.append(
        {
            "corruption":
                family_620gc,

            "sensor":
                target_sensor_620gc,

            "severity":
                str(
                    scenario_row_620gc[
                        "severity"
                    ]
                ),

            "target_changed_rows":
                target_changed_count_620gc,

            "max_non_target_change":
                max_other_column_change_620gc,

            "passed":
                (
                    target_changed_count_620gc
                    > 0
                    and
                    max_other_column_change_620gc
                    == 0.0
                ),
        }
    )


smoke_df_620gc = pd.DataFrame(
    smoke_records_620gc
)

display(
    smoke_df_620gc
)

assert smoke_df_620gc[
    "passed"
].all(), (
    "At least one frozen corruption generator "
    "failed the execution smoke test."
)


print(
    "All five frozen corruption families "
    "passed the smoke test."
)


# ======================================================================
# 17. FULL 315-SCENARIO TEMPORAL EXPERIMENT
# ======================================================================

print(
    "\n=== Full frozen-corruption temporal experiment ==="
)

print(
    "Total scenarios:",
    sum(
        len(df)
        for df in scenario_manifests_620gc.values()
    )
)

print(
    "This step may take several minutes."
)


scenario_temporal_records_620gc = []

scenario_counter_620gc = 0

total_scenarios_620gc = sum(
    len(df)
    for df in scenario_manifests_620gc.values()
)

start_time_620gc = time.time()


for family_620gc in [
    "bias",
    "noise",
    "drift",
    "dropout",
    "stuck",
]:

    manifest_620gc = (
        scenario_manifests_620gc[
            family_620gc
        ]
    )


    for row_number_620gc, scenario_row_620gc in (
        manifest_620gc.iterrows()
    ):

        scenario_counter_620gc += 1


        corrupted_df_620gc = (
            apply_frozen_corruption_620gc(
                family_620gc,
                scenario_row_620gc
            )
        )


        X_corrupted_raw_sensor_620gc = (
            dataframe_to_raw_sensor_windows_620gc(
                corrupted_df_620gc
            )
        )


        assert (
            X_corrupted_raw_sensor_620gc.shape
            == (
                3490,
                30,
                15
            )
        )


        corrupted_metrics_620gc = (
            temporal_metrics_620gc(
                X_corrupted_raw_sensor_620gc
            )
        )


        corrupted_flags_620gc = (
            temporal_diagnostic_620gc(
                corrupted_metrics_620gc
            )
        )


        temporal_summary_620gc = (
            summarize_temporal_flags_620gc(
                corrupted_flags_620gc
            )
        )


        scenario_id_620gc = (
            str(
                scenario_row_620gc[
                    "scenario_id"
                ]
            )
            if (
                "scenario_id"
                in scenario_row_620gc.index
                and
                pd.notna(
                    scenario_row_620gc[
                        "scenario_id"
                    ]
                )
            )
            else
            (
                f"FD001_"
                f"{family_620gc.upper()}_"
                f"{row_number_620gc + 1:03d}"
            )
        )


        record_620gc = {

            "corruption":
                family_620gc,

            "scenario_id":
                scenario_id_620gc,

            "sensor":
                str(
                    scenario_row_620gc[
                        "sensor"
                    ]
                ),

            "severity":
                str(
                    scenario_row_620gc[
                        "severity"
                    ]
                ),

            **temporal_summary_620gc,
        }


        # Preserve direction where relevant.

        if (
            "direction"
            in scenario_row_620gc.index
        ):

            record_620gc[
                "direction"
            ] = scenario_row_620gc[
                "direction"
            ]


        scenario_temporal_records_620gc.append(
            record_620gc
        )


        if (
            scenario_counter_620gc % 25
            == 0
            or
            scenario_counter_620gc
            == total_scenarios_620gc
        ):

            elapsed_620gc = (
                time.time()
                - start_time_620gc
            )

            print(
                f"[{scenario_counter_620gc}/"
                f"{total_scenarios_620gc}] "
                f"completed "
                f"({elapsed_620gc:.1f} s)"
            )


temporal_response_620gc = pd.DataFrame(
    scenario_temporal_records_620gc
)


print(
    "\nScenario-level temporal response shape:",
    temporal_response_620gc.shape
)

assert len(
    temporal_response_620gc
) == 315


# ======================================================================
# 18. CLEAN-BASELINE DELTAS
# ======================================================================

for metric_620gc in [
    "high_activity_sensor_event_pct",
    "low_activity_sensor_event_pct",
    "special_deficit_sensor_event_pct",
    "two_sided_sensor_event_pct",
    "mean_flagged_sensors",
    "windows_with_any_flag_pct",
]:

    temporal_response_620gc[
        f"{metric_620gc}_delta"
    ] = (
        temporal_response_620gc[
            metric_620gc
        ]
        -
        clean_summary_620gc[
            metric_620gc
        ]
    )


# ======================================================================
# 19. BALANCED FAMILY × SENSOR × SEVERITY AGGREGATION
# ======================================================================

balanced_temporal_620gc = (
    temporal_response_620gc
    .groupby(
        [
            "corruption",
            "sensor",
            "severity"
        ],
        as_index=False
    )
    .agg(
        source_scenarios=(
            "scenario_id",
            "count"
        ),

        mean_two_sided_sensor_event_pct=(
            "two_sided_sensor_event_pct",
            "mean"
        ),

        mean_two_sided_delta_pct=(
            "two_sided_sensor_event_pct_delta",
            "mean"
        ),

        mean_flagged_sensors=(
            "mean_flagged_sensors",
            "mean"
        ),

        mean_flagged_sensors_delta=(
            "mean_flagged_sensors_delta",
            "mean"
        ),

        mean_windows_any_flag_pct=(
            "windows_with_any_flag_pct",
            "mean"
        ),

        mean_windows_any_flag_delta_pct=(
            "windows_with_any_flag_pct_delta",
            "mean"
        ),

        mean_high_activity_pct=(
            "high_activity_sensor_event_pct",
            "mean"
        ),

        mean_low_activity_pct=(
            "low_activity_sensor_event_pct",
            "mean"
        ),

        mean_special_deficit_pct=(
            "special_deficit_sensor_event_pct",
            "mean"
        ),
    )
)


assert len(
    balanced_temporal_620gc
) == 225


print(
    "\nBalanced corruption × sensor × severity cells:",
    len(
        balanced_temporal_620gc
    )
)


# ======================================================================
# 20. FAMILY-LEVEL SUMMARY — EQUAL SENSOR/SEVERITY WEIGHTING
# ======================================================================

family_summary_620gc = (
    balanced_temporal_620gc
    .groupby(
        "corruption",
        as_index=False
    )
    .agg(
        balanced_cells=(
            "sensor",
            "size"
        ),

        mean_two_sided_event_pct=(
            "mean_two_sided_sensor_event_pct",
            "mean"
        ),

        mean_two_sided_delta_pct=(
            "mean_two_sided_delta_pct",
            "mean"
        ),

        mean_flagged_sensors=(
            "mean_flagged_sensors",
            "mean"
        ),

        mean_flagged_sensors_delta=(
            "mean_flagged_sensors_delta",
            "mean"
        ),

        mean_windows_any_flag_pct=(
            "mean_windows_any_flag_pct",
            "mean"
        ),

        mean_windows_any_flag_delta_pct=(
            "mean_windows_any_flag_delta_pct",
            "mean"
        ),

        mean_high_activity_pct=(
            "mean_high_activity_pct",
            "mean"
        ),

        mean_low_activity_pct=(
            "mean_low_activity_pct",
            "mean"
        ),

        mean_special_deficit_pct=(
            "mean_special_deficit_pct",
            "mean"
        ),
    )
    .sort_values(
        "mean_two_sided_delta_pct",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n=== Temporal diagnostic response by corruption family ==="
)

display(
    family_summary_620gc.round(
        3
    )
)


# ======================================================================
# 21. FAMILY × SEVERITY SUMMARY
# ======================================================================

severity_order_620gc = [
    "low",
    "medium",
    "high",
]


family_severity_summary_620gc = (
    balanced_temporal_620gc
    .groupby(
        [
            "corruption",
            "severity"
        ],
        as_index=False
    )
    .agg(
        sensors=(
            "sensor",
            "nunique"
        ),

        mean_two_sided_event_pct=(
            "mean_two_sided_sensor_event_pct",
            "mean"
        ),

        mean_two_sided_delta_pct=(
            "mean_two_sided_delta_pct",
            "mean"
        ),

        mean_flagged_sensors=(
            "mean_flagged_sensors",
            "mean"
        ),

        mean_flagged_sensors_delta=(
            "mean_flagged_sensors_delta",
            "mean"
        ),

        mean_windows_any_flag_pct=(
            "mean_windows_any_flag_pct",
            "mean"
        ),

        mean_windows_any_flag_delta_pct=(
            "mean_windows_any_flag_delta_pct",
            "mean"
        ),
    )
)


family_severity_summary_620gc[
    "severity"
] = pd.Categorical(
    family_severity_summary_620gc[
        "severity"
    ],
    categories=severity_order_620gc,
    ordered=True
)


family_severity_summary_620gc = (
    family_severity_summary_620gc
    .sort_values(
        [
            "corruption",
            "severity"
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n=== Temporal response by family and severity ==="
)

display(
    family_severity_summary_620gc.round(
        3
    )
)


# ======================================================================
# 22. SEVERITY MONOTONICITY CHECK
# ======================================================================

monotonicity_records_620gc = []


for family_620gc in sorted(
    family_severity_summary_620gc[
        "corruption"
    ].unique()
):

    family_rows_620gc = (
        family_severity_summary_620gc[
            family_severity_summary_620gc[
                "corruption"
            ]
            == family_620gc
        ]
        .sort_values(
            "severity"
        )
    )


    event_values_620gc = (
        family_rows_620gc[
            "mean_two_sided_delta_pct"
        ]
        .to_numpy(
            dtype=float
        )
    )


    flagged_values_620gc = (
        family_rows_620gc[
            "mean_flagged_sensors_delta"
        ]
        .to_numpy(
            dtype=float
        )
    )


    monotonicity_records_620gc.append(
        {
            "corruption":
                family_620gc,

            "two_sided_delta_monotonic":
                bool(
                    np.all(
                        np.diff(
                            event_values_620gc
                        )
                        >= -1e-12
                    )
                ),

            "flagged_sensor_delta_monotonic":
                bool(
                    np.all(
                        np.diff(
                            flagged_values_620gc
                        )
                        >= -1e-12
                    )
                ),
        }
    )


temporal_monotonicity_620gc = pd.DataFrame(
    monotonicity_records_620gc
)


print(
    "\n=== Severity monotonicity ==="
)

display(
    temporal_monotonicity_620gc
)


# ======================================================================
# 23. CROSS-FAMILY SENSOR TEMPORAL SENSITIVITY
# ======================================================================

sensor_family_summary_620gc = (
    balanced_temporal_620gc
    .groupby(
        [
            "corruption",
            "sensor"
        ],
        as_index=False
    )
    .agg(
        mean_two_sided_delta_pct=(
            "mean_two_sided_delta_pct",
            "mean"
        ),

        mean_flagged_sensors_delta=(
            "mean_flagged_sensors_delta",
            "mean"
        ),

        mean_windows_any_flag_delta_pct=(
            "mean_windows_any_flag_delta_pct",
            "mean"
        ),
    )
)


cross_family_sensor_620gc = (
    sensor_family_summary_620gc
    .groupby(
        "sensor",
        as_index=False
    )
    .agg(
        corruption_families=(
            "corruption",
            "nunique"
        ),

        cross_family_mean_two_sided_delta_pct=(
            "mean_two_sided_delta_pct",
            "mean"
        ),

        cross_family_min_two_sided_delta_pct=(
            "mean_two_sided_delta_pct",
            "min"
        ),

        cross_family_max_two_sided_delta_pct=(
            "mean_two_sided_delta_pct",
            "max"
        ),

        cross_family_mean_flagged_delta=(
            "mean_flagged_sensors_delta",
            "mean"
        ),
    )
    .sort_values(
        "cross_family_mean_two_sided_delta_pct",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n=== Cross-family temporal sensor sensitivity ==="
)

display(
    cross_family_sensor_620gc.round(
        3
    )
)


# ======================================================================
# 24. CORRUPTION × SENSOR RESPONSE MATRIX
# ======================================================================

temporal_matrix_620gc = (
    sensor_family_summary_620gc
    .pivot(
        index="sensor",
        columns="corruption",
        values="mean_two_sided_delta_pct"
    )
)


family_columns_620gc = [
    family
    for family in [
        "bias",
        "noise",
        "drift",
        "dropout",
        "stuck",
    ]
    if family
    in temporal_matrix_620gc.columns
]


temporal_matrix_620gc[
    "equal_family_mean"
] = (
    temporal_matrix_620gc[
        family_columns_620gc
    ]
    .mean(
        axis=1
    )
)


temporal_matrix_620gc = (
    temporal_matrix_620gc
    .sort_values(
        "equal_family_mean",
        ascending=False
    )
)


print(
    "\n=== Corruption × sensor temporal-response matrix ==="
)

display(
    temporal_matrix_620gc.round(
        3
    )
)


# ======================================================================
# 25. INTEGRITY SUMMARY
# ======================================================================

print(
    "\n=== Step 6.20G-C integrity summary ==="
)

print(
    "Frozen scenarios evaluated        :",
    len(
        temporal_response_620gc
    )
)

print(
    "Balanced sensor/severity cells     :",
    len(
        balanced_temporal_620gc
    )
)

print(
    "Corruption families                :",
    temporal_response_620gc[
        "corruption"
    ].nunique()
)

print(
    "Unique retained sensors            :",
    temporal_response_620gc[
        "sensor"
    ].nunique()
)

print(
    "Severity levels                    :",
    sorted(
        temporal_response_620gc[
            "severity"
        ].unique()
    )
)

print(
    "Clean two-sided event rate (%)     :",
    clean_summary_620gc[
        "two_sided_sensor_event_pct"
    ]
)

print(
    "Clean mean flagged sensors/window  :",
    clean_summary_620gc[
        "mean_flagged_sensors"
    ]
)

print(
    "Clean windows with any flag (%)    :",
    clean_summary_620gc[
        "windows_with_any_flag_pct"
    ]
)


all_numeric_finite_620gc = bool(
    np.isfinite(
        temporal_response_620gc[
            [
                "two_sided_sensor_event_pct",
                "mean_flagged_sensors",
                "windows_with_any_flag_pct",
            ]
        ]
        .to_numpy(
            dtype=float
        )
    ).all()
)


print(
    "All key numerical results finite   :",
    all_numeric_finite_620gc
)


assert len(
    temporal_response_620gc
) == 315

assert len(
    balanced_temporal_620gc
) == 225

assert (
    temporal_response_620gc[
        "corruption"
    ].nunique()
    == 5
)

assert (
    temporal_response_620gc[
        "sensor"
    ].nunique()
    == 15
)

assert all_numeric_finite_620gc


print(
    "\nStep 6.20G-C completed successfully."
)

print(
    "All five frozen corruption families were evaluated "
    "with the development-calibrated two-sided temporal diagnostic."
)

print(
    "No final temporal reliability score was frozen."
)

print(
    "No PCA + temporal combination was performed."
)

print(
    "No original frozen corruption or PCA result CSV was modified."
)

In [ ]:
# ======================================================================
# Step 6.20G-C1
# Correct legacy bias/noise/drift calling convention
# + rerun five-family smoke test
# ======================================================================

import numpy as np
import pandas as pd


print(
    "\n=== Step 6.20G-C1 — "
    "Correct legacy generator calling convention ===\n"
)


# ----------------------------------------------------------------------
# 1. VERIFY THAT THE FROZEN MANIFESTS CONTAIN NUMERICAL ALPHA
# ----------------------------------------------------------------------

print("=== Frozen alpha availability ===")

alpha_check_records_620gc1 = []

for family_620gc1 in [
    "bias",
    "noise",
    "drift",
]:

    manifest_620gc1 = (
        scenario_manifests_620gc[
            family_620gc1
        ]
    )

    alpha_present_620gc1 = (
        "alpha"
        in manifest_620gc1.columns
    )

    if alpha_present_620gc1:

        alpha_numeric_620gc1 = (
            pd.to_numeric(
                manifest_620gc1["alpha"],
                errors="coerce"
            )
        )

        all_alpha_finite_620gc1 = bool(
            np.isfinite(
                alpha_numeric_620gc1
                .to_numpy(
                    dtype=float
                )
            ).all()
        )

        unique_alpha_620gc1 = sorted(
            alpha_numeric_620gc1
            .dropna()
            .unique()
            .tolist()
        )

    else:

        all_alpha_finite_620gc1 = False
        unique_alpha_620gc1 = []


    alpha_check_records_620gc1.append(
        {
            "corruption":
                family_620gc1,

            "alpha_column_present":
                alpha_present_620gc1,

            "all_alpha_values_finite":
                all_alpha_finite_620gc1,

            "unique_alpha_values":
                unique_alpha_620gc1,
        }
    )


alpha_check_df_620gc1 = pd.DataFrame(
    alpha_check_records_620gc1
)

display(
    alpha_check_df_620gc1
)


assert alpha_check_df_620gc1[
    "alpha_column_present"
].all(), (
    "A required frozen alpha column is missing."
)

assert alpha_check_df_620gc1[
    "all_alpha_values_finite"
].all(), (
    "One or more frozen alpha values are invalid."
)


# ----------------------------------------------------------------------
# 2. CORRECT FAMILY-SPECIFIC CORRUPTION CALLER
# ----------------------------------------------------------------------
#
# IMPORTANT:
#
# The legacy Step-4 functions call their third argument
# "severity", but the implementation uses that argument
# numerically.
#
# Therefore:
#
#   bias  -> frozen manifest alpha
#   noise -> frozen manifest alpha
#   drift -> frozen manifest alpha
#
# The categorical LOW/MEDIUM/HIGH label remains metadata.
#
# Dropout and stuck already have explicitly verified numerical
# requested_fraction fields and remain unchanged.
# ----------------------------------------------------------------------

def apply_frozen_corruption_620gc(
    family,
    scenario_row
):
    """
    Apply one exact frozen Step-4 corruption scenario.

    Notes
    -----
    Bias/noise/drift are legacy generators whose argument named
    `severity` is numerically parameterized.  Therefore the frozen
    scenario manifest's `alpha` value must be supplied.

    Dropout and stuck retain their previously verified interfaces.
    """

    sensor = str(
        scenario_row[
            "sensor"
        ]
    )

    severity_label = str(
        scenario_row[
            "severity"
        ]
    )


    # --------------------------------------------------------------
    # BIAS
    # --------------------------------------------------------------

    if family == "bias":

        assert (
            "alpha"
            in scenario_row.index
        )

        alpha = float(
            scenario_row[
                "alpha"
            ]
        )

        direction = int(
            scenario_row[
                "direction"
            ]
        )

        result = (
            frozen_add_sensor_bias_620gc(
                val_raw_base_620gc,
                sensor,
                alpha,
                direction,
                sensor_reference_stats
            )
        )


    # --------------------------------------------------------------
    # NOISE
    # --------------------------------------------------------------

    elif family == "noise":

        assert (
            "alpha"
            in scenario_row.index
        )

        alpha = float(
            scenario_row[
                "alpha"
            ]
        )

        if (
            "seed"
            in scenario_row.index
            and
            pd.notna(
                scenario_row[
                    "seed"
                ]
            )
        ):

            seed = int(
                scenario_row[
                    "seed"
                ]
            )

        else:

            seed = 42


        result = (
            frozen_add_sensor_noise_620gc(
                val_raw_base_620gc,
                sensor,
                alpha,
                sensor_reference_stats,
                seed=seed
            )
        )


    # --------------------------------------------------------------
    # DRIFT
    # --------------------------------------------------------------

    elif family == "drift":

        assert (
            "alpha"
            in scenario_row.index
        )

        alpha = float(
            scenario_row[
                "alpha"
            ]
        )

        direction = int(
            scenario_row[
                "direction"
            ]
        )

        result = (
            frozen_add_sensor_drift_620gc(
                val_raw_base_620gc,
                sensor,
                alpha,
                direction,
                sensor_reference_stats
            )
        )


    # --------------------------------------------------------------
    # DROPOUT
    # --------------------------------------------------------------

    elif family == "dropout":

        requested_fraction = float(
            scenario_row[
                "requested_fraction"
            ]
        )

        result = (
            candidate_add_sensor_dropout_617d(
                val_raw_base_620gc,
                sensor,
                requested_fraction,
                sensor_reference_stats,
                seed=42
            )
        )


    # --------------------------------------------------------------
    # STUCK
    # --------------------------------------------------------------

    elif family == "stuck":

        requested_fraction = float(
            scenario_row[
                "requested_fraction"
            ]
        )

        result = (
            candidate_add_stuck_sensor_618d(
                val_raw_base_620gc,
                sensor,
                requested_fraction
            )
        )


    else:

        raise ValueError(
            f"Unknown corruption family: {family}"
        )


    corrupted_df = (
        extract_corrupted_dataframe_620gc(
            family,
            result
        )
    )


    return corrupted_df


print(
    "\nCorrected corruption caller created."
)


# ----------------------------------------------------------------------
# 3. FIVE-FAMILY ONE-SCENARIO SMOKE TEST
# ----------------------------------------------------------------------

print(
    "\n=== Five-family execution smoke test ==="
)


smoke_records_620gc1 = []


for family_620gc1 in [
    "bias",
    "noise",
    "drift",
    "dropout",
    "stuck",
]:

    manifest_620gc1 = (
        scenario_manifests_620gc[
            family_620gc1
        ]
    )

    scenario_row_620gc1 = (
        manifest_620gc1.iloc[0]
    )

    target_sensor_620gc1 = str(
        scenario_row_620gc1[
            "sensor"
        ]
    )


    # --------------------------------------------------------------
    # Execute ONE frozen scenario only
    # --------------------------------------------------------------

    corrupted_df_620gc1 = (
        apply_frozen_corruption_620gc(
            family_620gc1,
            scenario_row_620gc1
        )
    )


    # --------------------------------------------------------------
    # Structural verification
    # --------------------------------------------------------------

    same_shape_620gc1 = (
        corrupted_df_620gc1.shape
        ==
        val_raw_base_620gc.shape
    )

    same_columns_620gc1 = (
        list(
            corrupted_df_620gc1.columns
        )
        ==
        list(
            val_raw_base_620gc.columns
        )
    )


    # --------------------------------------------------------------
    # Target-sensor change
    # --------------------------------------------------------------

    clean_target_620gc1 = (
        val_raw_base_620gc[
            target_sensor_620gc1
        ]
        .to_numpy(
            dtype=np.float64
        )
    )

    corrupted_target_620gc1 = (
        corrupted_df_620gc1[
            target_sensor_620gc1
        ]
        .to_numpy(
            dtype=np.float64
        )
    )


    target_changed_mask_620gc1 = (
        ~np.isclose(
            corrupted_target_620gc1,
            clean_target_620gc1,
            rtol=0.0,
            atol=0.0,
            equal_nan=True
        )
    )


    target_changed_rows_620gc1 = int(
        np.sum(
            target_changed_mask_620gc1
        )
    )


    # --------------------------------------------------------------
    # Check that all NON-TARGET columns remain unchanged
    # --------------------------------------------------------------

    non_target_columns_620gc1 = [

        column_620gc1

        for column_620gc1
        in val_raw_base_620gc.columns

        if column_620gc1
        != target_sensor_620gc1
    ]


    non_target_match_620gc1 = bool(

        corrupted_df_620gc1[
            non_target_columns_620gc1
        ]
        .equals(
            val_raw_base_620gc[
                non_target_columns_620gc1
            ]
        )
    )


    # --------------------------------------------------------------
    # Record exact parameter used
    # --------------------------------------------------------------

    if family_620gc1 in [
        "bias",
        "noise",
        "drift",
    ]:

        numeric_parameter_620gc1 = float(
            scenario_row_620gc1[
                "alpha"
            ]
        )

        parameter_name_620gc1 = (
            "alpha"
        )

    else:

        numeric_parameter_620gc1 = float(
            scenario_row_620gc1[
                "requested_fraction"
            ]
        )

        parameter_name_620gc1 = (
            "requested_fraction"
        )


    passed_620gc1 = bool(

        same_shape_620gc1
        and

        same_columns_620gc1
        and

        non_target_match_620gc1
        and

        target_changed_rows_620gc1
        > 0
    )


    smoke_records_620gc1.append(
        {
            "corruption":
                family_620gc1,

            "sensor":
                target_sensor_620gc1,

            "severity_label":
                str(
                    scenario_row_620gc1[
                        "severity"
                    ]
                ),

            "numeric_parameter":
                numeric_parameter_620gc1,

            "parameter_source":
                parameter_name_620gc1,

            "target_changed_rows":
                target_changed_rows_620gc1,

            "same_shape":
                same_shape_620gc1,

            "same_columns":
                same_columns_620gc1,

            "non_target_columns_unchanged":
                non_target_match_620gc1,

            "passed":
                passed_620gc1,
        }
    )


smoke_df_620gc1 = pd.DataFrame(
    smoke_records_620gc1
)


display(
    smoke_df_620gc1
)


# ----------------------------------------------------------------------
# 4. FINAL INTEGRITY CHECK
# ----------------------------------------------------------------------

print(
    "\n=== Step 6.20G-C1 integrity summary ==="
)

print(
    "Families tested                  :",
    len(
        smoke_df_620gc1
    )
)

print(
    "Families passed                  :",
    int(
        smoke_df_620gc1[
            "passed"
        ].sum()
    )
)

print(
    "All five families passed        :",
    bool(
        smoke_df_620gc1[
            "passed"
        ].all()
    )
)


assert len(
    smoke_df_620gc1
) == 5

assert smoke_df_620gc1[
    "passed"
].all(), (
    "At least one corruption family still "
    "failed the corrected smoke test."
)


print(
    "\nStep 6.20G-C1 completed successfully."
)

print(
    "The categorical severity labels were preserved "
    "as scenario metadata."
)

print(
    "Bias/noise/drift were executed using their exact "
    "frozen numerical alpha values."
)

print(
    "Dropout/stuck were executed using their exact "
    "frozen requested fractions."
)

print(
    "No full corruption experiment was executed."
)

print(
    "No frozen CSV, model, scaler, or artifact was modified."
)

In [ ]:
# ======================================================================
# Step 6.20G-C2
# Full 315-scenario temporal-response experiment
# ======================================================================

import time
import gc

import numpy as np
import pandas as pd


print(
    "\n=== Step 6.20G-C2 — "
    "Full frozen-corruption temporal response ===\n"
)


# ----------------------------------------------------------------------
# 1. REQUIRED OBJECTS
# ----------------------------------------------------------------------

required_objects_620gc2 = [
    "scenario_manifests_620gc",
    "apply_frozen_corruption_620gc",
    "val_raw_base_620gc",
    "temporal_reference_620gc",
    "RETAINED_SENSOR_COLUMNS",
]

print("=== Required objects ===")

missing_objects_620gc2 = []

for name_620gc2 in required_objects_620gc2:

    available_620gc2 = (
        name_620gc2 in globals()
    )

    print(
        f"{name_620gc2:<32}: "
        f"{available_620gc2}"
    )

    if not available_620gc2:
        missing_objects_620gc2.append(
            name_620gc2
        )


assert not missing_objects_620gc2, (
    "Missing required objects: "
    + ", ".join(
        missing_objects_620gc2
    )
)


print(
    "\nAll required objects are available."
)


# ----------------------------------------------------------------------
# 2. FIX SENSOR ORDER
# ----------------------------------------------------------------------

retained_sensors_620gc2 = list(
    RETAINED_SENSOR_COLUMNS
)

n_sensors_620gc2 = len(
    retained_sensors_620gc2
)

assert n_sensors_620gc2 == 15


missing_sensor_columns_620gc2 = [

    sensor_620gc2

    for sensor_620gc2
    in retained_sensors_620gc2

    if sensor_620gc2
    not in val_raw_base_620gc.columns
]

assert not missing_sensor_columns_620gc2, (
    "Missing retained sensors in raw validation data: "
    + str(
        missing_sensor_columns_620gc2
    )
)


print(
    "\nRetained sensors:",
    n_sensors_620gc2
)


# ----------------------------------------------------------------------
# 3. VERIFY / ALIGN DEVELOPMENT-ONLY TEMPORAL REFERENCE
# ----------------------------------------------------------------------

temporal_ref_620gc2 = (
    temporal_reference_620gc.copy()
)


if (
    "sensor"
    in temporal_ref_620gc2.columns
):

    temporal_ref_620gc2 = (
        temporal_ref_620gc2
        .set_index(
            "sensor"
        )
    )


required_reference_columns_620gc2 = [
    "mean_p05",
    "mean_p50",
    "mean_p95",
    "q95_p05",
    "q95_p50",
    "q95_p95",
    "zero_p05",
    "zero_p50",
    "zero_p95",
    "special_handling",
]


missing_reference_columns_620gc2 = [

    column_620gc2

    for column_620gc2
    in required_reference_columns_620gc2

    if column_620gc2
    not in temporal_ref_620gc2.columns
]


assert not missing_reference_columns_620gc2, (
    "Missing temporal-reference columns: "
    + str(
        missing_reference_columns_620gc2
    )
)


missing_reference_sensors_620gc2 = [

    sensor_620gc2

    for sensor_620gc2
    in retained_sensors_620gc2

    if sensor_620gc2
    not in temporal_ref_620gc2.index
]


assert not missing_reference_sensors_620gc2, (
    "Missing temporal-reference sensors: "
    + str(
        missing_reference_sensors_620gc2
    )
)


temporal_ref_620gc2 = (
    temporal_ref_620gc2
    .loc[
        retained_sensors_620gc2,
        required_reference_columns_620gc2
    ]
    .copy()
)


special_mask_620gc2 = (
    temporal_ref_620gc2[
        "special_handling"
    ]
    .astype(bool)
    .to_numpy()
)


ordinary_mask_620gc2 = (
    ~special_mask_620gc2
)


special_sensors_620gc2 = [

    retained_sensors_620gc2[
        i_620gc2
    ]

    for i_620gc2
    in np.where(
        special_mask_620gc2
    )[0]
]


print(
    "Special-handling sensors:",
    special_sensors_620gc2
)


assert special_sensors_620gc2 == [
    "sensor_6"
], (
    "Unexpected special-handling sensor set."
)


# ----------------------------------------------------------------------
# 4. CREATE VALIDATION WINDOW-ROW INDICES ONCE
# ----------------------------------------------------------------------
#
# The validation DataFrame contains 20 complete validation-engine
# trajectories.  We construct the same 30-cycle rolling-window
# structure once and reuse these row positions for every corruption.
# ----------------------------------------------------------------------

assert (
    val_raw_base_620gc.index.is_unique
), (
    "Validation DataFrame index must be unique."
)


row_position_lookup_620gc2 = pd.Series(
    np.arange(
        len(
            val_raw_base_620gc
        ),
        dtype=np.int64
    ),
    index=val_raw_base_620gc.index
)


window_index_blocks_620gc2 = []


for unit_id_620gc2, group_620gc2 in (
    val_raw_base_620gc
    .groupby(
        "unit_id",
        sort=False
    )
):

    group_620gc2 = (
        group_620gc2
        .sort_values(
            "cycle"
        )
    )

    positions_620gc2 = (
        row_position_lookup_620gc2
        .loc[
            group_620gc2.index
        ]
        .to_numpy(
            dtype=np.int64
        )
    )


    assert len(
        positions_620gc2
    ) >= 30


    engine_windows_620gc2 = (
        np.lib.stride_tricks
        .sliding_window_view(
            positions_620gc2,
            window_shape=30
        )
    )


    window_index_blocks_620gc2.append(
        engine_windows_620gc2
    )


window_row_indices_620gc2 = (
    np.concatenate(
        window_index_blocks_620gc2,
        axis=0
    )
)


print(
    "\n=== Validation rolling-window structure ==="
)

print(
    "Raw validation rows :",
    len(
        val_raw_base_620gc
    )
)

print(
    "Window count        :",
    window_row_indices_620gc2.shape[0]
)

print(
    "Window size         :",
    window_row_indices_620gc2.shape[1]
)


assert (
    window_row_indices_620gc2.shape
    ==
    (3490, 30)
)


# ----------------------------------------------------------------------
# 5. TEMPORAL-METRIC FUNCTION
# ----------------------------------------------------------------------

def compute_temporal_window_metrics_620gc2(
    raw_df
):
    """
    Compute raw-sensor temporal behaviour for each 30-cycle window.

    Returns
    -------
    mean_abs_diff : (n_windows, n_sensors)
    q95_abs_diff  : (n_windows, n_sensors)
    zero_fraction : (n_windows, n_sensors)
    """

    assert (
        raw_df.shape
        ==
        val_raw_base_620gc.shape
    )


    assert list(
        raw_df.columns
    ) == list(
        val_raw_base_620gc.columns
    )


    assert raw_df.index.equals(
        val_raw_base_620gc.index
    )


    sensor_values_620gc2 = (
        raw_df[
            retained_sensors_620gc2
        ]
        .to_numpy(
            dtype=np.float64
        )
    )


    assert np.isfinite(
        sensor_values_620gc2
    ).all(), (
        "Non-finite sensor values detected."
    )


    windows_620gc2 = (
        sensor_values_620gc2[
            window_row_indices_620gc2
        ]
    )


    absolute_diffs_620gc2 = np.abs(
        np.diff(
            windows_620gc2,
            axis=1
        )
    )


    mean_abs_diff_620gc2 = (
        absolute_diffs_620gc2
        .mean(
            axis=1
        )
    )


    q95_abs_diff_620gc2 = (
        np.quantile(
            absolute_diffs_620gc2,
            q=0.95,
            axis=1,
            method="linear"
        )
    )


    zero_fraction_620gc2 = (
        absolute_diffs_620gc2
        .eq(0.0)
        if isinstance(
            absolute_diffs_620gc2,
            pd.DataFrame
        )
        else (
            absolute_diffs_620gc2
            == 0.0
        )
    )


    zero_fraction_620gc2 = (
        np.asarray(
            zero_fraction_620gc2,
            dtype=np.float64
        )
        .mean(
            axis=1
        )
    )


    return (
        mean_abs_diff_620gc2,
        q95_abs_diff_620gc2,
        zero_fraction_620gc2,
    )


# ----------------------------------------------------------------------
# 6. TWO-SIDED TEMPORAL FLAG FUNCTION
# ----------------------------------------------------------------------

mean_p05_620gc2 = (
    temporal_ref_620gc2[
        "mean_p05"
    ]
    .to_numpy(
        dtype=np.float64
    )
)

mean_p95_620gc2 = (
    temporal_ref_620gc2[
        "mean_p95"
    ]
    .to_numpy(
        dtype=np.float64
    )
)

q95_p05_620gc2 = (
    temporal_ref_620gc2[
        "q95_p05"
    ]
    .to_numpy(
        dtype=np.float64
    )
)

q95_p95_620gc2 = (
    temporal_ref_620gc2[
        "q95_p95"
    ]
    .to_numpy(
        dtype=np.float64
    )
)

zero_p05_620gc2 = (
    temporal_ref_620gc2[
        "zero_p05"
    ]
    .to_numpy(
        dtype=np.float64
    )
)

zero_p95_620gc2 = (
    temporal_ref_620gc2[
        "zero_p95"
    ]
    .to_numpy(
        dtype=np.float64
    )
)


def calculate_temporal_flags_620gc2(
    mean_abs_diff,
    q95_abs_diff,
    zero_fraction
):
    """
    Candidate two-sided temporal diagnostic.

    Ordinary sensors:
      HIGH activity =
          mean transition > development P95
          AND
          q95 transition > development P95

      LOW activity =
          mean transition < development P05
          OR
          zero-transition fraction > development P95

    Special sensor(s):
      deficit =
          zero-transition fraction < development P05

    No final reliability score is created here.
    """

    shape_620gc2 = (
        mean_abs_diff.shape
    )


    assert shape_620gc2 == (
        3490,
        n_sensors_620gc2
    )


    high_activity_620gc2 = np.zeros(
        shape_620gc2,
        dtype=bool
    )

    low_activity_620gc2 = np.zeros(
        shape_620gc2,
        dtype=bool
    )

    special_deficit_620gc2 = np.zeros(
        shape_620gc2,
        dtype=bool
    )


    # --------------------------------------------------------------
    # Ordinary-sensor HIGH temporal activity
    # --------------------------------------------------------------

    high_activity_620gc2[
        :,
        ordinary_mask_620gc2
    ] = (

        (
            mean_abs_diff[
                :,
                ordinary_mask_620gc2
            ]
            >
            mean_p95_620gc2[
                ordinary_mask_620gc2
            ]
        )

        &

        (
            q95_abs_diff[
                :,
                ordinary_mask_620gc2
            ]
            >
            q95_p95_620gc2[
                ordinary_mask_620gc2
            ]
        )
    )


    # --------------------------------------------------------------
    # Ordinary-sensor LOW temporal activity
    # --------------------------------------------------------------

    low_activity_620gc2[
        :,
        ordinary_mask_620gc2
    ] = (

        (
            mean_abs_diff[
                :,
                ordinary_mask_620gc2
            ]
            <
            mean_p05_620gc2[
                ordinary_mask_620gc2
            ]
        )

        |

        (
            zero_fraction[
                :,
                ordinary_mask_620gc2
            ]
            >
            zero_p95_620gc2[
                ordinary_mask_620gc2
            ]
        )
    )


    # --------------------------------------------------------------
    # Special nearly-static sensor deficit
    # --------------------------------------------------------------

    special_deficit_620gc2[
        :,
        special_mask_620gc2
    ] = (

        zero_fraction[
            :,
            special_mask_620gc2
        ]

        <

        zero_p05_620gc2[
            special_mask_620gc2
        ]
    )


    combined_620gc2 = (

        high_activity_620gc2

        |

        low_activity_620gc2

        |

        special_deficit_620gc2
    )


    return {
        "high":
            high_activity_620gc2,

        "low":
            low_activity_620gc2,

        "special":
            special_deficit_620gc2,

        "combined":
            combined_620gc2,
    }


# ----------------------------------------------------------------------
# 7. CLEAN VALIDATION BASELINE
# ----------------------------------------------------------------------

print(
    "\n=== Reconstruct CLEAN temporal baseline ==="
)


(
    clean_mean_abs_620gc2,
    clean_q95_abs_620gc2,
    clean_zero_fraction_620gc2,
) = compute_temporal_window_metrics_620gc2(
    val_raw_base_620gc
)


clean_flags_620gc2 = (
    calculate_temporal_flags_620gc2(
        clean_mean_abs_620gc2,
        clean_q95_abs_620gc2,
        clean_zero_fraction_620gc2
    )
)


clean_combined_620gc2 = (
    clean_flags_620gc2[
        "combined"
    ]
)


clean_high_620gc2 = (
    clean_flags_620gc2[
        "high"
    ]
)

clean_low_620gc2 = (
    clean_flags_620gc2[
        "low"
    ]
)

clean_special_620gc2 = (
    clean_flags_620gc2[
        "special"
    ]
)


clean_per_sensor_event_pct_620gc2 = (
    clean_combined_620gc2
    .mean(
        axis=0
    )
    * 100.0
)

clean_per_sensor_high_pct_620gc2 = (
    clean_high_620gc2
    .mean(
        axis=0
    )
    * 100.0
)

clean_per_sensor_low_pct_620gc2 = (
    clean_low_620gc2
    .mean(
        axis=0
    )
    * 100.0
)

clean_per_sensor_special_pct_620gc2 = (
    clean_special_620gc2
    .mean(
        axis=0
    )
    * 100.0
)


clean_flag_count_per_window_620gc2 = (
    clean_combined_620gc2
    .sum(
        axis=1
    )
)


clean_temporal_baseline_620gc2 = pd.DataFrame(
    [
        {
            "high_activity_sensor_event_pct":
                float(
                    clean_high_620gc2.mean()
                    * 100.0
                ),

            "low_activity_sensor_event_pct":
                float(
                    clean_low_620gc2.mean()
                    * 100.0
                ),

            "special_deficit_sensor_event_pct":
                float(
                    clean_special_620gc2.mean()
                    * 100.0
                ),

            "two_sided_sensor_event_pct":
                float(
                    clean_combined_620gc2.mean()
                    * 100.0
                ),

            "mean_flagged_sensors":
                float(
                    clean_flag_count_per_window_620gc2
                    .mean()
                ),

            "median_flagged_sensors":
                float(
                    np.median(
                        clean_flag_count_per_window_620gc2
                    )
                ),

            "p95_flagged_sensors":
                float(
                    np.quantile(
                        clean_flag_count_per_window_620gc2,
                        0.95
                    )
                ),

            "maximum_flagged_sensors":
                int(
                    clean_flag_count_per_window_620gc2
                    .max()
                ),

            "windows_with_any_flag_pct":
                float(
                    (
                        clean_flag_count_per_window_620gc2
                        > 0
                    ).mean()
                    * 100.0
                ),
        }
    ]
)


display(
    clean_temporal_baseline_620gc2
)


# ----------------------------------------------------------------------
# 8. VERIFY FROZEN SCENARIO COUNT
# ----------------------------------------------------------------------

family_order_620gc2 = [
    "bias",
    "noise",
    "drift",
    "dropout",
    "stuck",
]


scenario_count_table_620gc2 = pd.DataFrame(
    [
        {
            "corruption":
                family_620gc2,

            "scenarios":
                len(
                    scenario_manifests_620gc[
                        family_620gc2
                    ]
                ),
        }

        for family_620gc2
        in family_order_620gc2
    ]
)


display(
    scenario_count_table_620gc2
)


total_scenarios_620gc2 = int(
    scenario_count_table_620gc2[
        "scenarios"
    ].sum()
)


print(
    "Total frozen scenarios:",
    total_scenarios_620gc2
)


assert total_scenarios_620gc2 == 315


# ----------------------------------------------------------------------
# 9. EXECUTE ALL 315 FROZEN SCENARIOS
# ----------------------------------------------------------------------

print(
    "\n=== Executing all 315 frozen scenarios ==="
)

print(
    "This may take a few minutes.\n"
)


scenario_records_620gc2 = []

scenario_counter_620gc2 = 0

start_time_620gc2 = (
    time.perf_counter()
)


sensor_to_index_620gc2 = {

    sensor_620gc2:
        index_620gc2

    for index_620gc2,
        sensor_620gc2

    in enumerate(
        retained_sensors_620gc2
    )
}


for family_620gc2 in family_order_620gc2:

    manifest_620gc2 = (
        scenario_manifests_620gc[
            family_620gc2
        ]
    )


    for (
        manifest_row_number_620gc2,
        (_, scenario_row_620gc2)
    ) in enumerate(
        manifest_620gc2.iterrows(),
        start=1
    ):

        scenario_counter_620gc2 += 1


        sensor_620gc2 = str(
            scenario_row_620gc2[
                "sensor"
            ]
        )

        severity_620gc2 = str(
            scenario_row_620gc2[
                "severity"
            ]
        )


        assert (
            sensor_620gc2
            in sensor_to_index_620gc2
        )


        sensor_index_620gc2 = (
            sensor_to_index_620gc2[
                sensor_620gc2
            ]
        )


        # ----------------------------------------------------------
        # Exact frozen corruption
        # ----------------------------------------------------------

        corrupted_df_620gc2 = (
            apply_frozen_corruption_620gc(
                family_620gc2,
                scenario_row_620gc2
            )
        )


        assert (
            corrupted_df_620gc2.shape
            ==
            val_raw_base_620gc.shape
        )

        assert list(
            corrupted_df_620gc2.columns
        ) == list(
            val_raw_base_620gc.columns
        )

        assert corrupted_df_620gc2.index.equals(
            val_raw_base_620gc.index
        )


        # ----------------------------------------------------------
        # Temporal window metrics
        # ----------------------------------------------------------

        (
            corrupt_mean_abs_620gc2,
            corrupt_q95_abs_620gc2,
            corrupt_zero_fraction_620gc2,
        ) = compute_temporal_window_metrics_620gc2(
            corrupted_df_620gc2
        )


        corrupt_flags_620gc2 = (
            calculate_temporal_flags_620gc2(
                corrupt_mean_abs_620gc2,
                corrupt_q95_abs_620gc2,
                corrupt_zero_fraction_620gc2
            )
        )


        corrupt_high_620gc2 = (
            corrupt_flags_620gc2[
                "high"
            ]
        )

        corrupt_low_620gc2 = (
            corrupt_flags_620gc2[
                "low"
            ]
        )

        corrupt_special_620gc2 = (
            corrupt_flags_620gc2[
                "special"
            ]
        )

        corrupt_combined_620gc2 = (
            corrupt_flags_620gc2[
                "combined"
            ]
        )


        corrupt_flag_count_620gc2 = (
            corrupt_combined_620gc2
            .sum(
                axis=1
            )
        )


        # ----------------------------------------------------------
        # Target-sensor event rates
        # ----------------------------------------------------------

        target_event_pct_620gc2 = float(
            corrupt_combined_620gc2[
                :,
                sensor_index_620gc2
            ]
            .mean()
            * 100.0
        )


        target_high_pct_620gc2 = float(
            corrupt_high_620gc2[
                :,
                sensor_index_620gc2
            ]
            .mean()
            * 100.0
        )


        target_low_pct_620gc2 = float(
            corrupt_low_620gc2[
                :,
                sensor_index_620gc2
            ]
            .mean()
            * 100.0
        )


        target_special_pct_620gc2 = float(
            corrupt_special_620gc2[
                :,
                sensor_index_620gc2
            ]
            .mean()
            * 100.0
        )


        clean_target_event_pct_620gc2 = float(
            clean_per_sensor_event_pct_620gc2[
                sensor_index_620gc2
            ]
        )


        clean_target_high_pct_620gc2 = float(
            clean_per_sensor_high_pct_620gc2[
                sensor_index_620gc2
            ]
        )


        clean_target_low_pct_620gc2 = float(
            clean_per_sensor_low_pct_620gc2[
                sensor_index_620gc2
            ]
        )


        clean_target_special_pct_620gc2 = float(
            clean_per_sensor_special_pct_620gc2[
                sensor_index_620gc2
            ]
        )


        # ----------------------------------------------------------
        # Overall event rates
        # ----------------------------------------------------------

        high_event_pct_620gc2 = float(
            corrupt_high_620gc2.mean()
            * 100.0
        )

        low_event_pct_620gc2 = float(
            corrupt_low_620gc2.mean()
            * 100.0
        )

        special_event_pct_620gc2 = float(
            corrupt_special_620gc2.mean()
            * 100.0
        )

        two_sided_event_pct_620gc2 = float(
            corrupt_combined_620gc2.mean()
            * 100.0
        )


        mean_flagged_sensors_620gc2 = float(
            corrupt_flag_count_620gc2.mean()
        )

        median_flagged_sensors_620gc2 = float(
            np.median(
                corrupt_flag_count_620gc2
            )
        )

        p95_flagged_sensors_620gc2 = float(
            np.quantile(
                corrupt_flag_count_620gc2,
                0.95
            )
        )

        maximum_flagged_sensors_620gc2 = int(
            corrupt_flag_count_620gc2.max()
        )

        windows_with_any_flag_pct_620gc2 = float(
            (
                corrupt_flag_count_620gc2
                > 0
            ).mean()
            * 100.0
        )


        # ----------------------------------------------------------
        # Scenario metadata
        # ----------------------------------------------------------

        if (
            "scenario_id"
            in scenario_row_620gc2.index
            and
            pd.notna(
                scenario_row_620gc2[
                    "scenario_id"
                ]
            )
        ):

            scenario_id_620gc2 = str(
                scenario_row_620gc2[
                    "scenario_id"
                ]
            )

        else:

            scenario_id_620gc2 = (
                f"FD001_"
                f"{family_620gc2.upper()}_"
                f"{manifest_row_number_620gc2:03d}"
            )


        if (
            "alpha"
            in scenario_row_620gc2.index
            and
            pd.notna(
                scenario_row_620gc2[
                    "alpha"
                ]
            )
        ):

            alpha_620gc2 = float(
                scenario_row_620gc2[
                    "alpha"
                ]
            )

        else:

            alpha_620gc2 = np.nan


        if (
            "direction"
            in scenario_row_620gc2.index
            and
            pd.notna(
                scenario_row_620gc2[
                    "direction"
                ]
            )
        ):

            direction_620gc2 = int(
                scenario_row_620gc2[
                    "direction"
                ]
            )

        else:

            direction_620gc2 = np.nan


        if (
            "requested_fraction"
            in scenario_row_620gc2.index
            and
            pd.notna(
                scenario_row_620gc2[
                    "requested_fraction"
                ]
            )
        ):

            requested_fraction_620gc2 = float(
                scenario_row_620gc2[
                    "requested_fraction"
                ]
            )

        else:

            requested_fraction_620gc2 = np.nan


        # ----------------------------------------------------------
        # Store result
        # ----------------------------------------------------------

        scenario_records_620gc2.append(
            {
                "scenario_id":
                    scenario_id_620gc2,

                "corruption":
                    family_620gc2,

                "sensor":
                    sensor_620gc2,

                "severity":
                    severity_620gc2,

                "alpha":
                    alpha_620gc2,

                "direction":
                    direction_620gc2,

                "requested_fraction":
                    requested_fraction_620gc2,

                # Target sensor
                "clean_target_event_pct":
                    clean_target_event_pct_620gc2,

                "corrupted_target_event_pct":
                    target_event_pct_620gc2,

                "target_event_delta_pp":
                    (
                        target_event_pct_620gc2
                        -
                        clean_target_event_pct_620gc2
                    ),

                "clean_target_high_pct":
                    clean_target_high_pct_620gc2,

                "corrupted_target_high_pct":
                    target_high_pct_620gc2,

                "target_high_delta_pp":
                    (
                        target_high_pct_620gc2
                        -
                        clean_target_high_pct_620gc2
                    ),

                "clean_target_low_pct":
                    clean_target_low_pct_620gc2,

                "corrupted_target_low_pct":
                    target_low_pct_620gc2,

                "target_low_delta_pp":
                    (
                        target_low_pct_620gc2
                        -
                        clean_target_low_pct_620gc2
                    ),

                "clean_target_special_pct":
                    clean_target_special_pct_620gc2,

                "corrupted_target_special_pct":
                    target_special_pct_620gc2,

                "target_special_delta_pp":
                    (
                        target_special_pct_620gc2
                        -
                        clean_target_special_pct_620gc2
                    ),

                # All sensors
                "high_activity_sensor_event_pct":
                    high_event_pct_620gc2,

                "low_activity_sensor_event_pct":
                    low_event_pct_620gc2,

                "special_deficit_sensor_event_pct":
                    special_event_pct_620gc2,

                "two_sided_sensor_event_pct":
                    two_sided_event_pct_620gc2,

                "two_sided_event_delta_pp":
                    (
                        two_sided_event_pct_620gc2
                        -
                        float(
                            clean_temporal_baseline_620gc2[
                                "two_sided_sensor_event_pct"
                            ].iloc[0]
                        )
                    ),

                "mean_flagged_sensors":
                    mean_flagged_sensors_620gc2,

                "mean_flagged_sensors_delta":
                    (
                        mean_flagged_sensors_620gc2
                        -
                        float(
                            clean_temporal_baseline_620gc2[
                                "mean_flagged_sensors"
                            ].iloc[0]
                        )
                    ),

                "median_flagged_sensors":
                    median_flagged_sensors_620gc2,

                "p95_flagged_sensors":
                    p95_flagged_sensors_620gc2,

                "maximum_flagged_sensors":
                    maximum_flagged_sensors_620gc2,

                "windows_with_any_flag_pct":
                    windows_with_any_flag_pct_620gc2,

                "windows_with_any_flag_delta_pp":
                    (
                        windows_with_any_flag_pct_620gc2
                        -
                        float(
                            clean_temporal_baseline_620gc2[
                                "windows_with_any_flag_pct"
                            ].iloc[0]
                        )
                    ),
            }
        )


        # ----------------------------------------------------------
        # Progress
        # ----------------------------------------------------------

        if (
            scenario_counter_620gc2
            % 25
            == 0
            or
            scenario_counter_620gc2
            == total_scenarios_620gc2
        ):

            elapsed_620gc2 = (
                time.perf_counter()
                -
                start_time_620gc2
            )

            print(
                f"[{scenario_counter_620gc2:03d}/"
                f"{total_scenarios_620gc2}] "
                f"completed — "
                f"{elapsed_620gc2:.1f} s"
            )


        # ----------------------------------------------------------
        # Release large temporary arrays
        # ----------------------------------------------------------

        del corrupted_df_620gc2
        del corrupt_mean_abs_620gc2
        del corrupt_q95_abs_620gc2
        del corrupt_zero_fraction_620gc2
        del corrupt_flags_620gc2
        del corrupt_high_620gc2
        del corrupt_low_620gc2
        del corrupt_special_620gc2
        del corrupt_combined_620gc2
        del corrupt_flag_count_620gc2


        if (
            scenario_counter_620gc2
            % 25
            == 0
        ):
            gc.collect()


elapsed_total_620gc2 = (
    time.perf_counter()
    -
    start_time_620gc2
)


temporal_response_315_620gc2 = pd.DataFrame(
    scenario_records_620gc2
)


print(
    "\nFull execution time:",
    f"{elapsed_total_620gc2:.1f} seconds"
)


# ----------------------------------------------------------------------
# 10. SCENARIO-LEVEL INTEGRITY
# ----------------------------------------------------------------------

print(
    "\n=== Scenario-level integrity ==="
)


print(
    "Rows:",
    len(
        temporal_response_315_620gc2
    )
)

print(
    "Corruption families:",
    temporal_response_315_620gc2[
        "corruption"
    ].nunique()
)

print(
    "Sensors:",
    temporal_response_315_620gc2[
        "sensor"
    ].nunique()
)

print(
    "Severity levels:",
    temporal_response_315_620gc2[
        "severity"
    ].nunique()
)

print(
    "Duplicate scenario IDs:",
    temporal_response_315_620gc2[
        "scenario_id"
    ].duplicated().sum()
)


assert len(
    temporal_response_315_620gc2
) == 315

assert (
    temporal_response_315_620gc2[
        "corruption"
    ].nunique()
    == 5
)

assert (
    temporal_response_315_620gc2[
        "sensor"
    ].nunique()
    == 15
)

assert (
    temporal_response_315_620gc2[
        "severity"
    ].nunique()
    == 3
)

assert (
    temporal_response_315_620gc2[
        "scenario_id"
    ].duplicated().sum()
    == 0
)


key_numeric_columns_620gc2 = [
    "clean_target_event_pct",
    "corrupted_target_event_pct",
    "target_event_delta_pp",
    "two_sided_sensor_event_pct",
    "two_sided_event_delta_pp",
    "mean_flagged_sensors",
    "mean_flagged_sensors_delta",
    "windows_with_any_flag_pct",
    "windows_with_any_flag_delta_pp",
]


for column_620gc2 in key_numeric_columns_620gc2:

    assert np.isfinite(
        temporal_response_315_620gc2[
            column_620gc2
        ].to_numpy(
            dtype=np.float64
        )
    ).all(), (
        f"Non-finite values in "
        f"{column_620gc2}"
    )


print(
    "All key numerical values are finite."
)


# ----------------------------------------------------------------------
# 11. CREATE BALANCED FAMILY × SENSOR × SEVERITY TABLE
# ----------------------------------------------------------------------
#
# Bias and drift contain two directions per sensor/severity cell.
# The other families contain one scenario.
#
# Averaging within each family/sensor/severity cell creates:
#
#     5 families × 15 sensors × 3 severities = 225 cells
#
# This prevents bias/drift from receiving double weight in later
# cross-family comparisons.
# ----------------------------------------------------------------------

balanced_group_columns_620gc2 = [
    "corruption",
    "sensor",
    "severity",
]


balanced_numeric_columns_620gc2 = [
    "clean_target_event_pct",
    "corrupted_target_event_pct",
    "target_event_delta_pp",
    "clean_target_high_pct",
    "corrupted_target_high_pct",
    "target_high_delta_pp",
    "clean_target_low_pct",
    "corrupted_target_low_pct",
    "target_low_delta_pp",
    "clean_target_special_pct",
    "corrupted_target_special_pct",
    "target_special_delta_pp",
    "high_activity_sensor_event_pct",
    "low_activity_sensor_event_pct",
    "special_deficit_sensor_event_pct",
    "two_sided_sensor_event_pct",
    "two_sided_event_delta_pp",
    "mean_flagged_sensors",
    "mean_flagged_sensors_delta",
    "median_flagged_sensors",
    "p95_flagged_sensors",
    "maximum_flagged_sensors",
    "windows_with_any_flag_pct",
    "windows_with_any_flag_delta_pp",
]


temporal_balanced_cells_620gc2 = (
    temporal_response_315_620gc2
    .groupby(
        balanced_group_columns_620gc2,
        as_index=False,
        sort=False
    )[
        balanced_numeric_columns_620gc2
    ]
    .mean()
)


source_scenario_counts_620gc2 = (
    temporal_response_315_620gc2
    .groupby(
        balanced_group_columns_620gc2,
        as_index=False,
        sort=False
    )
    .size()
    .rename(
        columns={
            "size":
                "source_scenarios"
        }
    )
)


temporal_balanced_cells_620gc2 = (
    temporal_balanced_cells_620gc2
    .merge(
        source_scenario_counts_620gc2,
        on=balanced_group_columns_620gc2,
        how="left",
        validate="one_to_one"
    )
)


print(
    "\n=== Balanced-cell integrity ==="
)

print(
    "Balanced cells:",
    len(
        temporal_balanced_cells_620gc2
    )
)


assert len(
    temporal_balanced_cells_620gc2
) == 225


source_structure_620gc2 = (
    temporal_balanced_cells_620gc2
    .groupby(
        "corruption"
    )[
        "source_scenarios"
    ]
    .agg(
        [
            "min",
            "max",
            "mean",
        ]
    )
    .reset_index()
)


display(
    source_structure_620gc2
)


expected_sources_620gc2 = {
    "bias": 2,
    "noise": 1,
    "drift": 2,
    "dropout": 1,
    "stuck": 1,
}


for family_620gc2, expected_620gc2 in (
    expected_sources_620gc2.items()
):

    family_counts_620gc2 = (
        temporal_balanced_cells_620gc2
        .loc[
            temporal_balanced_cells_620gc2[
                "corruption"
            ]
            ==
            family_620gc2,
            "source_scenarios"
        ]
    )

    assert (
        family_counts_620gc2
        ==
        expected_620gc2
    ).all()


print(
    "Balanced weighting verified."
)


# ----------------------------------------------------------------------
# 12. FAMILY-LEVEL TEMPORAL RESPONSE
# ----------------------------------------------------------------------

family_summary_620gc2 = (
    temporal_balanced_cells_620gc2
    .groupby(
        "corruption",
        as_index=False
    )
    .agg(
        balanced_cells=(
            "sensor",
            "size"
        ),

        mean_target_event_delta_pp=(
            "target_event_delta_pp",
            "mean"
        ),

        median_target_event_delta_pp=(
            "target_event_delta_pp",
            "median"
        ),

        mean_target_high_delta_pp=(
            "target_high_delta_pp",
            "mean"
        ),

        mean_target_low_delta_pp=(
            "target_low_delta_pp",
            "mean"
        ),

        mean_target_special_delta_pp=(
            "target_special_delta_pp",
            "mean"
        ),

        mean_two_sided_event_delta_pp=(
            "two_sided_event_delta_pp",
            "mean"
        ),

        mean_flagged_sensors_delta=(
            "mean_flagged_sensors_delta",
            "mean"
        ),

        mean_windows_any_delta_pp=(
            "windows_with_any_flag_delta_pp",
            "mean"
        ),
    )
)


family_order_map_620gc2 = {
    family_620gc2:
        i_620gc2

    for i_620gc2,
        family_620gc2

    in enumerate(
        family_order_620gc2
    )
}


family_summary_620gc2[
    "_order"
] = (
    family_summary_620gc2[
        "corruption"
    ]
    .map(
        family_order_map_620gc2
    )
)


family_summary_620gc2 = (
    family_summary_620gc2
    .sort_values(
        "_order"
    )
    .drop(
        columns="_order"
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n=== Balanced temporal response by corruption family ==="
)

display(
    family_summary_620gc2.round(3)
)


# ----------------------------------------------------------------------
# 13. FAMILY × SEVERITY RESPONSE
# ----------------------------------------------------------------------

severity_order_620gc2 = {
    "low": 0,
    "medium": 1,
    "high": 2,
}


family_severity_summary_620gc2 = (
    temporal_balanced_cells_620gc2
    .groupby(
        [
            "corruption",
            "severity",
        ],
        as_index=False
    )
    .agg(
        sensors=(
            "sensor",
            "nunique"
        ),

        mean_target_event_delta_pp=(
            "target_event_delta_pp",
            "mean"
        ),

        median_target_event_delta_pp=(
            "target_event_delta_pp",
            "median"
        ),

        mean_target_high_delta_pp=(
            "target_high_delta_pp",
            "mean"
        ),

        mean_target_low_delta_pp=(
            "target_low_delta_pp",
            "mean"
        ),

        mean_target_special_delta_pp=(
            "target_special_delta_pp",
            "mean"
        ),

        mean_two_sided_event_delta_pp=(
            "two_sided_event_delta_pp",
            "mean"
        ),

        mean_windows_any_delta_pp=(
            "windows_with_any_flag_delta_pp",
            "mean"
        ),
    )
)


family_severity_summary_620gc2[
    "_family_order"
] = (
    family_severity_summary_620gc2[
        "corruption"
    ]
    .map(
        family_order_map_620gc2
    )
)

family_severity_summary_620gc2[
    "_severity_order"
] = (
    family_severity_summary_620gc2[
        "severity"
    ]
    .map(
        severity_order_620gc2
    )
)


family_severity_summary_620gc2 = (
    family_severity_summary_620gc2
    .sort_values(
        [
            "_family_order",
            "_severity_order",
        ]
    )
    .drop(
        columns=[
            "_family_order",
            "_severity_order",
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n=== Balanced temporal response by family and severity ==="
)

display(
    family_severity_summary_620gc2.round(3)
)


# ----------------------------------------------------------------------
# 14. SEVERITY-MONOTONICITY DIAGNOSTIC
# ----------------------------------------------------------------------

monotonicity_records_620gc2 = []


for family_620gc2 in family_order_620gc2:

    family_table_620gc2 = (
        family_severity_summary_620gc2[
            family_severity_summary_620gc2[
                "corruption"
            ]
            ==
            family_620gc2
        ]
        .set_index(
            "severity"
        )
    )


    target_values_620gc2 = [
        float(
            family_table_620gc2.loc[
                severity_620gc2,
                "mean_target_event_delta_pp"
            ]
        )

        for severity_620gc2
        in [
            "low",
            "medium",
            "high",
        ]
    ]


    global_values_620gc2 = [
        float(
            family_table_620gc2.loc[
                severity_620gc2,
                "mean_two_sided_event_delta_pp"
            ]
        )

        for severity_620gc2
        in [
            "low",
            "medium",
            "high",
        ]
    ]


    target_monotonic_620gc2 = bool(
        target_values_620gc2[0]
        <=
        target_values_620gc2[1]
        <=
        target_values_620gc2[2]
    )


    global_monotonic_620gc2 = bool(
        global_values_620gc2[0]
        <=
        global_values_620gc2[1]
        <=
        global_values_620gc2[2]
    )


    monotonicity_records_620gc2.append(
        {
            "corruption":
                family_620gc2,

            "target_response_monotonic":
                target_monotonic_620gc2,

            "overall_response_monotonic":
                global_monotonic_620gc2,

            "target_low_delta_pp":
                target_values_620gc2[0],

            "target_medium_delta_pp":
                target_values_620gc2[1],

            "target_high_delta_pp":
                target_values_620gc2[2],
        }
    )


severity_monotonicity_620gc2 = pd.DataFrame(
    monotonicity_records_620gc2
)


print(
    "\n=== Severity monotonicity diagnostic ==="
)

display(
    severity_monotonicity_620gc2.round(3)
)


# ----------------------------------------------------------------------
# 15. FINAL INTEGRITY SUMMARY
# ----------------------------------------------------------------------

print(
    "\n=== Step 6.20G-C2 integrity summary ==="
)

print(
    "Frozen scenarios evaluated       :",
    len(
        temporal_response_315_620gc2
    )
)

print(
    "Balanced sensor/severity cells   :",
    len(
        temporal_balanced_cells_620gc2
    )
)

print(
    "Corruption families              :",
    temporal_response_315_620gc2[
        "corruption"
    ].nunique()
)

print(
    "Retained sensors                 :",
    temporal_response_315_620gc2[
        "sensor"
    ].nunique()
)

print(
    "Severity levels                  :",
    temporal_response_315_620gc2[
        "severity"
    ].nunique()
)

print(
    "Special temporal sensors         :",
    special_sensors_620gc2
)

print(
    "Execution time (seconds)         :",
    round(
        elapsed_total_620gc2,
        2
    )
)


print(
    "\nStep 6.20G-C2 completed successfully."
)

print(
    "All 315 frozen corruption scenarios were "
    "evaluated against the development-calibrated "
    "two-sided temporal diagnostic."
)

print(
    "Bias/drift directions were balanced before "
    "cross-family comparison."
)

print(
    "No final temporal reliability score was frozen."
)

print(
    "No PCA + temporal combination was performed."
)

print(
    "No frozen CSV, model, scaler, or previous artifact "
    "was modified."
)

In [ ]:
# ============================================================
# Step 6.20H-A
# Continuous temporal anomaly-score candidates
# DEVELOPMENT calibration + CLEAN validation sanity check
#
# IMPORTANT:
# - Development data calibrate the score.
# - Clean validation is used only for sanity checking.
# - No corruption is generated here.
# - No final temporal score is frozen here.
# - No frozen CSV/artifact is modified.
# ============================================================

import numpy as np
import pandas as pd

print(
    "\n=== Step 6.20H-A — Continuous temporal-score candidates ==="
)


# ------------------------------------------------------------
# 1. Required objects
# ------------------------------------------------------------

required_objects_620ha = [
    "X_dev_clean",
    "X_val_clean",
    "scaler",
    "PRIMARY_INPUT_COLUMNS",
    "RETAINED_SENSOR_COLUMNS",
    "temporal_reference_620gc",
]

print("\n=== Required objects ===")

missing_objects_620ha = []

for name_620ha in required_objects_620ha:
    available_620ha = name_620ha in globals()

    print(
        f"{name_620ha:<30}: "
        f"{available_620ha}"
    )

    if not available_620ha:
        missing_objects_620ha.append(
            name_620ha
        )


assert not missing_objects_620ha, (
    "Missing required objects: "
    f"{missing_objects_620ha}"
)

print(
    "\nAll required objects are available."
)


# ------------------------------------------------------------
# 2. Basic sequence integrity
# ------------------------------------------------------------

X_dev_620ha = np.asarray(
    X_dev_clean
)

X_val_620ha = np.asarray(
    X_val_clean
)

assert X_dev_620ha.ndim == 3
assert X_val_620ha.ndim == 3

assert (
    X_dev_620ha.shape[1]
    ==
    X_val_620ha.shape[1]
)

assert (
    X_dev_620ha.shape[2]
    ==
    len(PRIMARY_INPUT_COLUMNS)
)

assert (
    X_val_620ha.shape[2]
    ==
    len(PRIMARY_INPUT_COLUMNS)
)


WINDOW_SIZE_620HA = int(
    X_dev_620ha.shape[1]
)

TRANSITIONS_PER_WINDOW_620HA = (
    WINDOW_SIZE_620HA - 1
)

print(
    "\n=== Sequence structure ==="
)

print(
    "Development sequence shape :",
    X_dev_620ha.shape
)

print(
    "Validation sequence shape  :",
    X_val_620ha.shape
)

print(
    "Window size                :",
    WINDOW_SIZE_620HA
)

print(
    "Transitions/window         :",
    TRANSITIONS_PER_WINDOW_620HA
)

print(
    "Primary features           :",
    len(PRIMARY_INPUT_COLUMNS)
)

print(
    "Retained sensors           :",
    len(RETAINED_SENSOR_COLUMNS)
)


# ------------------------------------------------------------
# 3. Verify retained-sensor mapping
# ------------------------------------------------------------

sensor_indices_620ha = [
    PRIMARY_INPUT_COLUMNS.index(
        sensor_620ha
    )
    for sensor_620ha
    in RETAINED_SENSOR_COLUMNS
]

mapping_rows_620ha = []

for (
    sensor_620ha,
    index_620ha
) in zip(
    RETAINED_SENSOR_COLUMNS,
    sensor_indices_620ha
):

    mapped_name_620ha = (
        PRIMARY_INPUT_COLUMNS[
            index_620ha
        ]
    )

    mapping_rows_620ha.append({
        "sensor":
            sensor_620ha,

        "sequence_feature_index":
            index_620ha,

        "mapped_column":
            mapped_name_620ha,

        "name_matches":
            mapped_name_620ha
            ==
            sensor_620ha,
    })


mapping_620ha = pd.DataFrame(
    mapping_rows_620ha
)

print(
    "\n=== Retained-sensor mapping ==="
)

display(
    mapping_620ha
)

assert (
    mapping_620ha[
        "name_matches"
    ].all()
), (
    "Retained-sensor mapping "
    "is inconsistent."
)


# ------------------------------------------------------------
# 4. FLOAT64 inverse transformation
#
# We deliberately use FLOAT64 here because Step 6.20D showed
# that FLOAT32 round-trip precision was insufficient for the
# strict reconstruction check.
# ------------------------------------------------------------

def inverse_scale_sequences_620ha(
    X_scaled_620ha
):
    """
    Convert scaled sequence array
    (n_windows, window_size, n_features)
    back to raw primary-input units using FLOAT64.
    """

    X_scaled_620ha = np.asarray(
        X_scaled_620ha,
        dtype=np.float64,
    )

    original_shape_620ha = (
        X_scaled_620ha.shape
    )

    flat_scaled_620ha = (
        X_scaled_620ha.reshape(
            -1,
            original_shape_620ha[-1],
        )
    )

    flat_raw_620ha = (
        scaler.inverse_transform(
            flat_scaled_620ha
        )
    )

    X_raw_620ha = (
        flat_raw_620ha.reshape(
            original_shape_620ha
        )
    )

    return np.asarray(
        X_raw_620ha,
        dtype=np.float64,
    )


X_dev_raw_620ha = (
    inverse_scale_sequences_620ha(
        X_dev_620ha
    )
)

X_val_raw_620ha = (
    inverse_scale_sequences_620ha(
        X_val_620ha
    )
)


# Retained sensors only
dev_sensor_windows_620ha = (
    X_dev_raw_620ha[
        :,
        :,
        sensor_indices_620ha
    ]
)

val_sensor_windows_620ha = (
    X_val_raw_620ha[
        :,
        :,
        sensor_indices_620ha
    ]
)

print(
    "\nRaw development retained-sensor shape:",
    dev_sensor_windows_620ha.shape
)

print(
    "Raw validation retained-sensor shape :",
    val_sensor_windows_620ha.shape
)


# ------------------------------------------------------------
# 5. Temporal metrics per window and sensor
#
# Metrics:
#   mean_abs_diff
#   q95_abs_diff
#   zero_fraction
# ------------------------------------------------------------

def temporal_metrics_620ha(
    sensor_windows_620ha
):
    """
    Compute temporal transition metrics.

    Input:
        shape =
        (n_windows, window_size, n_sensors)

    Returns:
        three arrays, each:
        (n_windows, n_sensors)
    """

    sensor_windows_620ha = np.asarray(
        sensor_windows_620ha,
        dtype=np.float64,
    )

    differences_620ha = np.diff(
        sensor_windows_620ha,
        axis=1,
    )

    abs_differences_620ha = np.abs(
        differences_620ha
    )

    mean_abs_diff_620ha = np.mean(
        abs_differences_620ha,
        axis=1,
    )

    q95_abs_diff_620ha = np.quantile(
        abs_differences_620ha,
        0.95,
        axis=1,
    )

    # Exact zero-transition fraction.
    #
    # C-MAPSS measurements are discrete enough that exact-zero
    # transitions are meaningful for our stuck/low-activity
    # diagnostic.
    zero_fraction_620ha = np.mean(
        abs_differences_620ha == 0.0,
        axis=1,
    )

    return (
        mean_abs_diff_620ha,
        q95_abs_diff_620ha,
        zero_fraction_620ha,
    )


(
    dev_mean_abs_620ha,
    dev_q95_abs_620ha,
    dev_zero_fraction_620ha,
) = temporal_metrics_620ha(
    dev_sensor_windows_620ha
)

(
    val_mean_abs_620ha,
    val_q95_abs_620ha,
    val_zero_fraction_620ha,
) = temporal_metrics_620ha(
    val_sensor_windows_620ha
)


print(
    "\nTemporal metrics computed successfully."
)


# ------------------------------------------------------------
# 6. Prepare development-only temporal reference
# ------------------------------------------------------------

reference_620ha = (
    temporal_reference_620gc.copy()
)

# Ensure sensor-indexed structure.
if "sensor" in reference_620ha.columns:
    reference_620ha = (
        reference_620ha.set_index(
            "sensor"
        )
    )

reference_620ha = (
    reference_620ha.loc[
        RETAINED_SENSOR_COLUMNS
    ].copy()
)


required_reference_columns_620ha = [
    "mean_p05",
    "mean_p50",
    "mean_p95",
    "q95_p05",
    "q95_p50",
    "q95_p95",
    "zero_p05",
    "zero_p50",
    "zero_p95",
    "special_handling",
]

missing_reference_columns_620ha = [
    column_620ha
    for column_620ha
    in required_reference_columns_620ha
    if column_620ha
    not in reference_620ha.columns
]

assert not missing_reference_columns_620ha, (
    "Missing temporal-reference columns: "
    f"{missing_reference_columns_620ha}"
)


print(
    "\n=== Development-only temporal reference ==="
)

display(
    reference_620ha[
        required_reference_columns_620ha
    ]
)


# ------------------------------------------------------------
# 7. Helper for robust tail normalization
# ------------------------------------------------------------

EPS_620HA = 1e-12

ZERO_STEP_620HA = (
    1.0
    /
    TRANSITIONS_PER_WINDOW_620HA
)


def safe_tail_scale_620ha(
    primary_scale_620ha,
    fallback_scale_620ha,
):
    """
    Use the local tail spread when available.
    Otherwise fall back to the full p05-p95 span.
    """

    primary_scale_620ha = np.asarray(
        primary_scale_620ha,
        dtype=np.float64,
    )

    fallback_scale_620ha = np.asarray(
        fallback_scale_620ha,
        dtype=np.float64,
    )

    return np.where(
        primary_scale_620ha
        > EPS_620HA,

        primary_scale_620ha,

        np.where(
            fallback_scale_620ha
            > EPS_620HA,

            fallback_scale_620ha,

            1.0,
        )
    )


# ------------------------------------------------------------
# 8. Build per-sensor continuous temporal anomaly strength
#
# Ordinary sensors:
#   - unusually high mean activity
#   - unusually low mean activity
#   - unusually high q95 activity
#   - unusually low q95 activity
#   - unusually many exact-zero transitions
#
# Special sensor(s), currently sensor_6:
#   - deficit in expected zero-transition behaviour
#
# We use log1p after standardized exceedance to prevent one
# extreme sensor from numerically dominating the whole score.
# ------------------------------------------------------------

def sensor_strength_620ha(
    mean_abs_620ha,
    q95_abs_620ha,
    zero_fraction_620ha,
):
    mean_abs_620ha = np.asarray(
        mean_abs_620ha,
        dtype=np.float64,
    )

    q95_abs_620ha = np.asarray(
        q95_abs_620ha,
        dtype=np.float64,
    )

    zero_fraction_620ha = np.asarray(
        zero_fraction_620ha,
        dtype=np.float64,
    )

    n_windows_620ha = (
        mean_abs_620ha.shape[0]
    )

    n_sensors_620ha = (
        mean_abs_620ha.shape[1]
    )

    strength_620ha = np.zeros(
        (
            n_windows_620ha,
            n_sensors_620ha,
        ),
        dtype=np.float64,
    )


    # Component matrices retained for diagnostics.
    high_mean_component_620ha = (
        np.zeros_like(
            strength_620ha
        )
    )

    low_mean_component_620ha = (
        np.zeros_like(
            strength_620ha
        )
    )

    high_q95_component_620ha = (
        np.zeros_like(
            strength_620ha
        )
    )

    low_q95_component_620ha = (
        np.zeros_like(
            strength_620ha
        )
    )

    high_zero_component_620ha = (
        np.zeros_like(
            strength_620ha
        )
    )

    special_zero_deficit_component_620ha = (
        np.zeros_like(
            strength_620ha
        )
    )


    for j_620ha, sensor_620ha in enumerate(
        RETAINED_SENSOR_COLUMNS
    ):

        row_620ha = (
            reference_620ha.loc[
                sensor_620ha
            ]
        )

        special_620ha = bool(
            row_620ha[
                "special_handling"
            ]
        )


        mean_p05_620ha = float(
            row_620ha["mean_p05"]
        )

        mean_p50_620ha = float(
            row_620ha["mean_p50"]
        )

        mean_p95_620ha = float(
            row_620ha["mean_p95"]
        )


        q95_p05_620ha = float(
            row_620ha["q95_p05"]
        )

        q95_p50_620ha = float(
            row_620ha["q95_p50"]
        )

        q95_p95_620ha = float(
            row_620ha["q95_p95"]
        )


        zero_p05_620ha = float(
            row_620ha["zero_p05"]
        )

        zero_p50_620ha = float(
            row_620ha["zero_p50"]
        )

        zero_p95_620ha = float(
            row_620ha["zero_p95"]
        )


        if not special_620ha:

            # -------------------------
            # Mean-transition activity
            # -------------------------

            mean_upper_scale_620ha = float(
                safe_tail_scale_620ha(
                    mean_p95_620ha
                    -
                    mean_p50_620ha,

                    mean_p95_620ha
                    -
                    mean_p05_620ha,
                )
            )

            mean_lower_scale_620ha = float(
                safe_tail_scale_620ha(
                    mean_p50_620ha
                    -
                    mean_p05_620ha,

                    mean_p95_620ha
                    -
                    mean_p05_620ha,
                )
            )


            high_mean_raw_620ha = np.maximum(
                0.0,

                (
                    mean_abs_620ha[
                        :,
                        j_620ha
                    ]
                    -
                    mean_p95_620ha
                )
                /
                mean_upper_scale_620ha
            )

            low_mean_raw_620ha = np.maximum(
                0.0,

                (
                    mean_p05_620ha
                    -
                    mean_abs_620ha[
                        :,
                        j_620ha
                    ]
                )
                /
                mean_lower_scale_620ha
            )


            # -------------------------
            # q95-transition activity
            # -------------------------

            q95_upper_scale_620ha = float(
                safe_tail_scale_620ha(
                    q95_p95_620ha
                    -
                    q95_p50_620ha,

                    q95_p95_620ha
                    -
                    q95_p05_620ha,
                )
            )

            q95_lower_scale_620ha = float(
                safe_tail_scale_620ha(
                    q95_p50_620ha
                    -
                    q95_p05_620ha,

                    q95_p95_620ha
                    -
                    q95_p05_620ha,
                )
            )


            high_q95_raw_620ha = np.maximum(
                0.0,

                (
                    q95_abs_620ha[
                        :,
                        j_620ha
                    ]
                    -
                    q95_p95_620ha
                )
                /
                q95_upper_scale_620ha
            )

            low_q95_raw_620ha = np.maximum(
                0.0,

                (
                    q95_p05_620ha
                    -
                    q95_abs_620ha[
                        :,
                        j_620ha
                    ]
                )
                /
                q95_lower_scale_620ha
            )


            # -------------------------
            # Excess exact-zero activity
            #
            # zero fraction moves in discrete steps of 1/29
            # for a 30-cycle window, so that step is a natural
            # normalization scale.
            # -------------------------

            high_zero_raw_620ha = np.maximum(
                0.0,

                (
                    zero_fraction_620ha[
                        :,
                        j_620ha
                    ]
                    -
                    zero_p95_620ha
                )
                /
                ZERO_STEP_620HA
            )


            high_mean_component_620ha[
                :,
                j_620ha
            ] = np.log1p(
                high_mean_raw_620ha
            )

            low_mean_component_620ha[
                :,
                j_620ha
            ] = np.log1p(
                low_mean_raw_620ha
            )

            high_q95_component_620ha[
                :,
                j_620ha
            ] = np.log1p(
                high_q95_raw_620ha
            )

            low_q95_component_620ha[
                :,
                j_620ha
            ] = np.log1p(
                low_q95_raw_620ha
            )

            high_zero_component_620ha[
                :,
                j_620ha
            ] = np.log1p(
                high_zero_raw_620ha
            )


            strength_620ha[
                :,
                j_620ha
            ] = np.maximum.reduce([
                high_mean_component_620ha[
                    :,
                    j_620ha
                ],

                low_mean_component_620ha[
                    :,
                    j_620ha
                ],

                high_q95_component_620ha[
                    :,
                    j_620ha
                ],

                low_q95_component_620ha[
                    :,
                    j_620ha
                ],

                high_zero_component_620ha[
                    :,
                    j_620ha
                ],
            ])


        else:

            # ------------------------------------------------
            # Special handling:
            # sensor_6 normally exhibits almost all
            # zero transitions.
            #
            # Abnormally *lower* zero-transition fraction
            # therefore indicates added temporal activity.
            # ------------------------------------------------

            zero_lower_scale_620ha = max(
                zero_p50_620ha
                -
                zero_p05_620ha,

                ZERO_STEP_620HA,
            )

            special_zero_raw_620ha = (
                np.maximum(
                    0.0,

                    (
                        zero_p05_620ha
                        -
                        zero_fraction_620ha[
                            :,
                            j_620ha
                        ]
                    )
                    /
                    zero_lower_scale_620ha
                )
            )

            special_zero_deficit_component_620ha[
                :,
                j_620ha
            ] = np.log1p(
                special_zero_raw_620ha
            )

            strength_620ha[
                :,
                j_620ha
            ] = (
                special_zero_deficit_component_620ha[
                    :,
                    j_620ha
                ]
            )


    components_620ha = {
        "high_mean":
            high_mean_component_620ha,

        "low_mean":
            low_mean_component_620ha,

        "high_q95":
            high_q95_component_620ha,

        "low_q95":
            low_q95_component_620ha,

        "high_zero":
            high_zero_component_620ha,

        "special_zero_deficit":
            special_zero_deficit_component_620ha,
    }

    return (
        strength_620ha,
        components_620ha,
    )


dev_sensor_strength_620ha, dev_components_620ha = (
    sensor_strength_620ha(
        dev_mean_abs_620ha,
        dev_q95_abs_620ha,
        dev_zero_fraction_620ha,
    )
)

val_sensor_strength_620ha, val_components_620ha = (
    sensor_strength_620ha(
        val_mean_abs_620ha,
        val_q95_abs_620ha,
        val_zero_fraction_620ha,
    )
)


assert np.isfinite(
    dev_sensor_strength_620ha
).all()

assert np.isfinite(
    val_sensor_strength_620ha
).all()


print(
    "\nPer-sensor continuous anomaly strengths "
    "computed successfully."
)


# ------------------------------------------------------------
# 9. Window-level candidate aggregation
#
# Candidate 1:
#   strongest single sensor
#
# Candidate 2:
#   mean of two strongest sensors
#
# Candidate 3:
#   mean of three strongest sensors
#
# Candidate 4:
#   hybrid magnitude/robustness score
#   = 0.5 * max + 0.5 * top-3 mean
# ------------------------------------------------------------

def aggregate_candidates_620ha(
    sensor_strength_matrix_620ha
):

    sensor_strength_matrix_620ha = (
        np.asarray(
            sensor_strength_matrix_620ha,
            dtype=np.float64,
        )
    )

    descending_620ha = np.sort(
        sensor_strength_matrix_620ha,
        axis=1,
    )[:, ::-1]


    max_sensor_620ha = (
        descending_620ha[:, 0]
    )

    top2_mean_620ha = np.mean(
        descending_620ha[:, :2],
        axis=1,
    )

    top3_mean_620ha = np.mean(
        descending_620ha[:, :3],
        axis=1,
    )

    hybrid_620ha = (
        0.5
        *
        max_sensor_620ha
        +
        0.5
        *
        top3_mean_620ha
    )


    return pd.DataFrame({
        "max_sensor":
            max_sensor_620ha,

        "top2_mean":
            top2_mean_620ha,

        "top3_mean":
            top3_mean_620ha,

        "hybrid_max_top3":
            hybrid_620ha,
    })


dev_candidate_scores_620ha = (
    aggregate_candidates_620ha(
        dev_sensor_strength_620ha
    )
)

val_candidate_scores_620ha = (
    aggregate_candidates_620ha(
        val_sensor_strength_620ha
    )
)


# ------------------------------------------------------------
# 10. Development-only candidate thresholds
#
# P95 is used here ONLY as a diagnostic candidate threshold.
# It is not the final reliability threshold.
# ------------------------------------------------------------

candidate_thresholds_620ha = (
    dev_candidate_scores_620ha
    .quantile(0.95)
    .rename(
        "development_p95_threshold"
    )
)


print(
    "\n=== DEVELOPMENT-only candidate thresholds ==="
)

display(
    candidate_thresholds_620ha
    .to_frame()
)


# ------------------------------------------------------------
# 11. Development vs clean-validation behaviour
# ------------------------------------------------------------

candidate_summary_rows_620ha = []

for candidate_620ha in (
    dev_candidate_scores_620ha.columns
):

    dev_values_620ha = (
        dev_candidate_scores_620ha[
            candidate_620ha
        ].to_numpy(
            dtype=np.float64
        )
    )

    val_values_620ha = (
        val_candidate_scores_620ha[
            candidate_620ha
        ].to_numpy(
            dtype=np.float64
        )
    )

    threshold_620ha = float(
        candidate_thresholds_620ha[
            candidate_620ha
        ]
    )


    dev_above_pct_620ha = (
        100.0
        *
        np.mean(
            dev_values_620ha
            >
            threshold_620ha
        )
    )

    val_above_pct_620ha = (
        100.0
        *
        np.mean(
            val_values_620ha
            >
            threshold_620ha
        )
    )


    candidate_summary_rows_620ha.append({
        "candidate":
            candidate_620ha,

        "dev_mean":
            float(
                np.mean(
                    dev_values_620ha
                )
            ),

        "dev_median":
            float(
                np.median(
                    dev_values_620ha
                )
            ),

        "dev_p95":
            float(
                np.quantile(
                    dev_values_620ha,
                    0.95,
                )
            ),

        "clean_val_mean":
            float(
                np.mean(
                    val_values_620ha
                )
            ),

        "clean_val_median":
            float(
                np.median(
                    val_values_620ha
                )
            ),

        "clean_val_p95":
            float(
                np.quantile(
                    val_values_620ha,
                    0.95,
                )
            ),

        "dev_above_dev_p95_pct":
            dev_above_pct_620ha,

        "clean_val_above_dev_p95_pct":
            val_above_pct_620ha,

        "clean_val_minus_5pct":
            val_above_pct_620ha
            -
            5.0,
    })


candidate_summary_620ha = (
    pd.DataFrame(
        candidate_summary_rows_620ha
    )
)


print(
    "\n=== Continuous temporal candidate behaviour ==="
)

display(
    candidate_summary_620ha.round(6)
)


# ------------------------------------------------------------
# 12. Candidate correlation diagnostic
#
# If two candidates are effectively identical,
# we do not need both later.
# ------------------------------------------------------------

candidate_corr_620ha = (
    val_candidate_scores_620ha
    .corr()
)


print(
    "\n=== CLEAN validation candidate-score correlations ==="
)

display(
    candidate_corr_620ha.round(4)
)


# ------------------------------------------------------------
# 13. Clean-validation sensor-strength summary
# ------------------------------------------------------------

sensor_strength_summary_rows_620ha = []

for j_620ha, sensor_620ha in enumerate(
    RETAINED_SENSOR_COLUMNS
):

    values_620ha = (
        val_sensor_strength_620ha[
            :,
            j_620ha
        ]
    )

    sensor_strength_summary_rows_620ha.append({
        "sensor":
            sensor_620ha,

        "special_handling":
            bool(
                reference_620ha.loc[
                    sensor_620ha,
                    "special_handling"
                ]
            ),

        "mean_strength":
            float(
                np.mean(
                    values_620ha
                )
            ),

        "median_strength":
            float(
                np.median(
                    values_620ha
                )
            ),

        "p95_strength":
            float(
                np.quantile(
                    values_620ha,
                    0.95,
                )
            ),

        "maximum_strength":
            float(
                np.max(
                    values_620ha
                )
            ),

        "nonzero_windows_pct":
            float(
                100.0
                *
                np.mean(
                    values_620ha > 0
                )
            ),
    })


sensor_strength_summary_620ha = (
    pd.DataFrame(
        sensor_strength_summary_rows_620ha
    )
    .sort_values(
        "mean_strength",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n=== CLEAN validation sensor-strength summary ==="
)

display(
    sensor_strength_summary_620ha.round(6)
)


# ------------------------------------------------------------
# 14. Integrity checks
# ------------------------------------------------------------

assert len(
    dev_candidate_scores_620ha
) == len(
    X_dev_620ha
)

assert len(
    val_candidate_scores_620ha
) == len(
    X_val_620ha
)

assert (
    dev_candidate_scores_620ha
    .notna()
    .all()
    .all()
)

assert (
    val_candidate_scores_620ha
    .notna()
    .all()
    .all()
)

assert np.isfinite(
    dev_candidate_scores_620ha
    .to_numpy()
).all()

assert np.isfinite(
    val_candidate_scores_620ha
    .to_numpy()
).all()


print(
    "\n=== Step 6.20H-A integrity summary ==="
)

print(
    "Development windows analysed :",
    len(
        dev_candidate_scores_620ha
    )
)

print(
    "Clean validation windows      :",
    len(
        val_candidate_scores_620ha
    )
)

print(
    "Retained sensors              :",
    len(
        RETAINED_SENSOR_COLUMNS
    )
)

print(
    "Candidate temporal scores     :",
    list(
        dev_candidate_scores_620ha.columns
    )
)

print(
    "Special temporal sensors      :",
    list(
        reference_620ha.index[
            reference_620ha[
                "special_handling"
            ].astype(bool)
        ]
    )
)


print(
    "\nStep 6.20H-A completed successfully."
)

print(
    "Continuous temporal anomaly candidates were "
    "calibrated from CLEAN DEVELOPMENT only."
)

print(
    "CLEAN validation was used only as a "
    "sanity check."
)

print(
    "No corruption was generated."
)

print(
    "No final temporal score was selected or frozen."
)

print(
    "No PCA + temporal combination was performed."
)

print(
    "No frozen CSV, model, scaler, or artifact "
    "was modified."
)

In [ ]:
# ============================================================
# Step 6.20H-B
# Evaluate continuous temporal-score candidates across
# ALL 315 previously frozen corruption scenarios.
#
# PURPOSE:
#   1. Test the four Step-6.20H-A temporal aggregations.
#   2. Preserve frozen Step-4 corruption definitions exactly.
#   3. Compare corrupted temporal scores against the CLEAN
#      validation baseline using DEVELOPMENT-only thresholds.
#   4. Use balanced corruption-family weighting.
#   5. Diagnose severity monotonicity.
#
# IMPORTANT:
#   - No corruption definition is changed.
#   - No new severity level is invented.
#   - No PCA + temporal combination is performed.
#   - No final temporal score is frozen.
#   - No existing CSV/artifact is modified.
# ============================================================

import time
import numpy as np
import pandas as pd

print(
    "\n=== Step 6.20H-B — Continuous temporal candidates "
    "across frozen corruptions ==="
)


# ------------------------------------------------------------
# 1. Required objects
# ------------------------------------------------------------

required_objects_620hb = [
    "scenario_manifests_620gc",
    "apply_frozen_corruption_620gc",
    "val_raw_base_620gc",
    "RETAINED_SENSOR_COLUMNS",
    "candidate_thresholds_620ha",
    "val_candidate_scores_620ha",
    "temporal_metrics_620ha",
    "sensor_strength_620ha",
    "aggregate_candidates_620ha",
]

print("\n=== Required objects ===")

missing_objects_620hb = []

for name_620hb in required_objects_620hb:

    available_620hb = (
        name_620hb in globals()
    )

    print(
        f"{name_620hb:<34}: "
        f"{available_620hb}"
    )

    if not available_620hb:

        missing_objects_620hb.append(
            name_620hb
        )


assert not missing_objects_620hb, (
    "Missing required objects: "
    f"{missing_objects_620hb}"
)

print(
    "\nAll required objects are available."
)


# ------------------------------------------------------------
# 2. Frozen scenario-family integrity
# ------------------------------------------------------------

expected_families_620hb = [
    "bias",
    "noise",
    "drift",
    "dropout",
    "stuck",
]

expected_counts_620hb = {
    "bias": 90,
    "noise": 45,
    "drift": 90,
    "dropout": 45,
    "stuck": 45,
}


family_integrity_rows_620hb = []

for family_620hb in expected_families_620hb:

    assert (
        family_620hb
        in scenario_manifests_620gc
    ), (
        "Missing frozen scenario manifest: "
        f"{family_620hb}"
    )

    manifest_620hb = (
        scenario_manifests_620gc[
            family_620hb
        ]
    )

    actual_rows_620hb = len(
        manifest_620hb
    )

    expected_rows_620hb = (
        expected_counts_620hb[
            family_620hb
        ]
    )

    family_integrity_rows_620hb.append({
        "corruption":
            family_620hb,

        "rows":
            actual_rows_620hb,

        "expected_rows":
            expected_rows_620hb,

        "rows_match":
            actual_rows_620hb
            ==
            expected_rows_620hb,
    })


family_integrity_620hb = pd.DataFrame(
    family_integrity_rows_620hb
)


print(
    "\n=== Frozen scenario counts ==="
)

display(
    family_integrity_620hb
)


assert (
    family_integrity_620hb[
        "rows_match"
    ].all()
), (
    "Frozen scenario counts do not match "
    "the verified Step-4 design."
)


TOTAL_SCENARIOS_620HB = int(
    family_integrity_620hb[
        "rows"
    ].sum()
)

assert TOTAL_SCENARIOS_620HB == 315

print(
    "\nTotal frozen scenarios:",
    TOTAL_SCENARIOS_620HB
)


# ------------------------------------------------------------
# 3. Basic raw-validation integrity
# ------------------------------------------------------------

val_raw_620hb = (
    val_raw_base_620gc.copy()
)

required_raw_columns_620hb = (
    ["unit_id", "cycle"]
    +
    list(
        RETAINED_SENSOR_COLUMNS
    )
)

missing_raw_columns_620hb = [
    column_620hb
    for column_620hb
    in required_raw_columns_620hb
    if column_620hb
    not in val_raw_620hb.columns
]

assert not missing_raw_columns_620hb, (
    "Raw validation DataFrame is missing: "
    f"{missing_raw_columns_620hb}"
)


print(
    "\n=== Raw validation structure ==="
)

print(
    "Rows              :",
    len(
        val_raw_620hb
    )
)

print(
    "Columns           :",
    val_raw_620hb.shape[1]
)

print(
    "Validation engines:",
    val_raw_620hb[
        "unit_id"
    ].nunique()
)


# ------------------------------------------------------------
# 4. Raw rolling-window constructor
#
# Creates exactly the same 30-cycle rolling structure used
# throughout the frozen validation experiments.
# ------------------------------------------------------------

WINDOW_SIZE_620HB = 30


def build_raw_sensor_windows_620hb(
    raw_df_620hb
):
    """
    Construct rolling retained-sensor windows independently
    within each engine trajectory.

    Output:
        (n_windows, 30, 15)
    """

    windows_620hb = []

    for unit_id_620hb, group_620hb in (
        raw_df_620hb.groupby(
            "unit_id",
            sort=False,
        )
    ):

        group_620hb = (
            group_620hb
            .sort_values(
                "cycle"
            )
        )

        sensor_array_620hb = (
            group_620hb[
                RETAINED_SENSOR_COLUMNS
            ]
            .to_numpy(
                dtype=np.float64
            )
        )

        n_rows_620hb = len(
            sensor_array_620hb
        )

        if (
            n_rows_620hb
            <
            WINDOW_SIZE_620HB
        ):
            continue

        # sliding_window_view over axis 0 gives:
        # (n_windows, n_sensors, window_size)
        engine_windows_620hb = (
            np.lib.stride_tricks
            .sliding_window_view(
                sensor_array_620hb,
                window_shape=WINDOW_SIZE_620HB,
                axis=0,
            )
        )

        # Convert to:
        # (n_windows, window_size, n_sensors)
        engine_windows_620hb = (
            np.transpose(
                engine_windows_620hb,
                (0, 2, 1),
            )
        )

        windows_620hb.append(
            engine_windows_620hb
        )


    assert windows_620hb, (
        "No validation windows were generated."
    )

    return np.concatenate(
        windows_620hb,
        axis=0,
    )


# ------------------------------------------------------------
# 5. CLEAN raw baseline through the exact H-B pathway
# ------------------------------------------------------------

clean_sensor_windows_620hb = (
    build_raw_sensor_windows_620hb(
        val_raw_620hb
    )
)


print(
    "\n=== CLEAN raw-window reconstruction ==="
)

print(
    "Shape:",
    clean_sensor_windows_620hb.shape
)


assert (
    clean_sensor_windows_620hb.shape
    ==
    (
        3490,
        30,
        len(
            RETAINED_SENSOR_COLUMNS
        ),
    )
), (
    "Unexpected clean validation window shape."
)


(
    clean_mean_abs_620hb,
    clean_q95_abs_620hb,
    clean_zero_fraction_620hb,
) = temporal_metrics_620ha(
    clean_sensor_windows_620hb
)


(
    clean_sensor_strength_620hb,
    clean_components_620hb,
) = sensor_strength_620ha(
    clean_mean_abs_620hb,
    clean_q95_abs_620hb,
    clean_zero_fraction_620hb,
)


clean_candidate_scores_620hb = (
    aggregate_candidates_620ha(
        clean_sensor_strength_620hb
    )
)


assert (
    list(
        clean_candidate_scores_620hb.columns
    )
    ==
    list(
        val_candidate_scores_620ha.columns
    )
)


# Compare H-B raw pathway against H-A reconstructed pathway.
clean_pathway_comparison_rows_620hb = []

for candidate_620hb in (
    clean_candidate_scores_620hb.columns
):

    hba_values_620hb = (
        val_candidate_scores_620ha[
            candidate_620hb
        ].to_numpy(
            dtype=np.float64
        )
    )

    hbb_values_620hb = (
        clean_candidate_scores_620hb[
            candidate_620hb
        ].to_numpy(
            dtype=np.float64
        )
    )

    difference_620hb = np.abs(
        hba_values_620hb
        -
        hbb_values_620hb
    )

    clean_pathway_comparison_rows_620hb.append({
        "candidate":
            candidate_620hb,

        "mean_absolute_difference":
            float(
                np.mean(
                    difference_620hb
                )
            ),

        "maximum_absolute_difference":
            float(
                np.max(
                    difference_620hb
                )
            ),
    })


clean_pathway_comparison_620hb = (
    pd.DataFrame(
        clean_pathway_comparison_rows_620hb
    )
)


print(
    "\n=== H-A vs H-B CLEAN-pathway consistency ==="
)

display(
    clean_pathway_comparison_620hb
)


# ------------------------------------------------------------
# 6. CLEAN validation event rates under DEVELOPMENT thresholds
# ------------------------------------------------------------

candidate_names_620hb = list(
    clean_candidate_scores_620hb.columns
)


clean_event_rates_620hb = {}

for candidate_620hb in (
    candidate_names_620hb
):

    threshold_620hb = float(
        candidate_thresholds_620ha[
            candidate_620hb
        ]
    )

    clean_event_rates_620hb[
        candidate_620hb
    ] = float(
        100.0
        *
        np.mean(
            clean_candidate_scores_620hb[
                candidate_620hb
            ].to_numpy(
                dtype=np.float64
            )
            >
            threshold_620hb
        )
    )


clean_event_rate_df_620hb = (
    pd.DataFrame({
        "candidate":
            list(
                clean_event_rates_620hb.keys()
            ),

        "clean_validation_above_dev_p95_pct":
            list(
                clean_event_rates_620hb.values()
            ),
    })
)


print(
    "\n=== CLEAN validation reference event rates ==="
)

display(
    clean_event_rate_df_620hb.round(6)
)


# ------------------------------------------------------------
# 7. Helper:
#    metadata extraction from each frozen manifest row
# ------------------------------------------------------------

def safe_metadata_value_620hb(
    scenario_row_620hb,
    column_620hb,
):
    if column_620hb not in (
        scenario_row_620hb.index
    ):
        return np.nan

    value_620hb = (
        scenario_row_620hb[
            column_620hb
        ]
    )

    if pd.isna(
        value_620hb
    ):
        return np.nan

    return value_620hb


# ------------------------------------------------------------
# 8. Execute all frozen scenarios
# ------------------------------------------------------------

print(
    "\n=== Executing all 315 frozen scenarios ==="
)

print(
    "Continuous temporal candidates will be "
    "evaluated scenario by scenario."
)


scenario_result_rows_620hb = []

start_time_620hb = time.perf_counter()

scenario_counter_620hb = 0


for family_620hb in (
    expected_families_620hb
):

    manifest_620hb = (
        scenario_manifests_620gc[
            family_620hb
        ]
    )

    for manifest_index_620hb, scenario_row_620hb in (
        manifest_620hb.iterrows()
    ):

        scenario_counter_620hb += 1


        # ----------------------------------------------------
        # Execute exact frozen corruption
        # ----------------------------------------------------

        corrupted_df_620hb = (
            apply_frozen_corruption_620gc(
                family_620hb,
                scenario_row_620hb,
            )
        )


        assert (
            corrupted_df_620hb.shape
            ==
            val_raw_620hb.shape
        )

        assert (
            list(
                corrupted_df_620hb.columns
            )
            ==
            list(
                val_raw_620hb.columns
            )
        )


        # ----------------------------------------------------
        # Temporal windows
        # ----------------------------------------------------

        corrupted_sensor_windows_620hb = (
            build_raw_sensor_windows_620hb(
                corrupted_df_620hb
            )
        )

        assert (
            corrupted_sensor_windows_620hb.shape
            ==
            clean_sensor_windows_620hb.shape
        )


        # ----------------------------------------------------
        # Temporal metrics
        # ----------------------------------------------------

        (
            corrupted_mean_abs_620hb,
            corrupted_q95_abs_620hb,
            corrupted_zero_fraction_620hb,
        ) = temporal_metrics_620ha(
            corrupted_sensor_windows_620hb
        )


        (
            corrupted_sensor_strength_620hb,
            corrupted_components_620hb,
        ) = sensor_strength_620ha(
            corrupted_mean_abs_620hb,
            corrupted_q95_abs_620hb,
            corrupted_zero_fraction_620hb,
        )


        corrupted_candidate_scores_620hb = (
            aggregate_candidates_620ha(
                corrupted_sensor_strength_620hb
            )
        )


        assert np.isfinite(
            corrupted_candidate_scores_620hb
            .to_numpy()
        ).all()


        # ----------------------------------------------------
        # Target-sensor diagnostic
        #
        # This is NOT a deployable final score.
        # It is only an oracle diagnostic because here we know
        # which sensor was deliberately corrupted.
        # ----------------------------------------------------

        target_sensor_620hb = str(
            scenario_row_620hb[
                "sensor"
            ]
        )

        target_sensor_index_620hb = (
            RETAINED_SENSOR_COLUMNS.index(
                target_sensor_620hb
            )
        )


        clean_target_strength_620hb = (
            clean_sensor_strength_620hb[
                :,
                target_sensor_index_620hb
            ]
        )

        corrupted_target_strength_620hb = (
            corrupted_sensor_strength_620hb[
                :,
                target_sensor_index_620hb
            ]
        )


        target_strength_delta_620hb = (
            corrupted_target_strength_620hb
            -
            clean_target_strength_620hb
        )


        # ----------------------------------------------------
        # Per-candidate scenario metrics
        # ----------------------------------------------------

        for candidate_620hb in (
            candidate_names_620hb
        ):

            clean_scores_620hb = (
                clean_candidate_scores_620hb[
                    candidate_620hb
                ].to_numpy(
                    dtype=np.float64
                )
            )

            corrupted_scores_620hb = (
                corrupted_candidate_scores_620hb[
                    candidate_620hb
                ].to_numpy(
                    dtype=np.float64
                )
            )


            paired_delta_620hb = (
                corrupted_scores_620hb
                -
                clean_scores_620hb
            )


            threshold_620hb = float(
                candidate_thresholds_620ha[
                    candidate_620hb
                ]
            )


            clean_event_rate_620hb = (
                clean_event_rates_620hb[
                    candidate_620hb
                ]
            )

            corrupted_event_rate_620hb = float(
                100.0
                *
                np.mean(
                    corrupted_scores_620hb
                    >
                    threshold_620hb
                )
            )


            scenario_id_620hb = (
                safe_metadata_value_620hb(
                    scenario_row_620hb,
                    "scenario_id",
                )
            )

            if (
                isinstance(
                    scenario_id_620hb,
                    float
                )
                and
                np.isnan(
                    scenario_id_620hb
                )
            ):
                scenario_id_620hb = (
                    f"{family_620hb.upper()}_"
                    f"{manifest_index_620hb:03d}"
                )


            scenario_result_rows_620hb.append({
                "corruption":
                    family_620hb,

                "scenario_id":
                    scenario_id_620hb,

                "sensor":
                    target_sensor_620hb,

                "severity":
                    str(
                        scenario_row_620hb[
                            "severity"
                        ]
                    ),

                "direction":
                    safe_metadata_value_620hb(
                        scenario_row_620hb,
                        "direction",
                    ),

                "alpha":
                    safe_metadata_value_620hb(
                        scenario_row_620hb,
                        "alpha",
                    ),

                "requested_fraction":
                    safe_metadata_value_620hb(
                        scenario_row_620hb,
                        "requested_fraction",
                    ),

                "candidate":
                    candidate_620hb,

                "clean_mean_score":
                    float(
                        np.mean(
                            clean_scores_620hb
                        )
                    ),

                "corrupted_mean_score":
                    float(
                        np.mean(
                            corrupted_scores_620hb
                        )
                    ),

                "mean_score_delta":
                    float(
                        np.mean(
                            paired_delta_620hb
                        )
                    ),

                "median_score_delta":
                    float(
                        np.median(
                            paired_delta_620hb
                        )
                    ),

                "p95_score_delta":
                    float(
                        np.quantile(
                            paired_delta_620hb,
                            0.95,
                        )
                    ),

                "score_increased_windows_pct":
                    float(
                        100.0
                        *
                        np.mean(
                            paired_delta_620hb
                            >
                            0.0
                        )
                    ),

                "development_p95_threshold":
                    threshold_620hb,

                "clean_above_threshold_pct":
                    clean_event_rate_620hb,

                "corrupted_above_threshold_pct":
                    corrupted_event_rate_620hb,

                "above_threshold_delta_pp":
                    (
                        corrupted_event_rate_620hb
                        -
                        clean_event_rate_620hb
                    ),

                # Oracle target-sensor diagnostic only
                "target_sensor_mean_strength_delta":
                    float(
                        np.mean(
                            target_strength_delta_620hb
                        )
                    ),

                "target_sensor_median_strength_delta":
                    float(
                        np.median(
                            target_strength_delta_620hb
                        )
                    ),

                "target_sensor_strength_increased_pct":
                    float(
                        100.0
                        *
                        np.mean(
                            target_strength_delta_620hb
                            >
                            0.0
                        )
                    ),
            })


        if (
            scenario_counter_620hb
            % 25
            ==
            0
            or
            scenario_counter_620hb
            ==
            TOTAL_SCENARIOS_620HB
        ):

            elapsed_620hb = (
                time.perf_counter()
                -
                start_time_620hb
            )

            print(
                f"[{scenario_counter_620hb:03d}/"
                f"{TOTAL_SCENARIOS_620HB}] "
                f"completed — "
                f"{elapsed_620hb:.1f} s"
            )


elapsed_total_620hb = (
    time.perf_counter()
    -
    start_time_620hb
)


temporal_candidate_results_620hb = (
    pd.DataFrame(
        scenario_result_rows_620hb
    )
)


print(
    "\nFull temporal-candidate execution time:",
    round(
        elapsed_total_620hb,
        2
    ),
    "seconds"
)


# ------------------------------------------------------------
# 9. Result integrity
# ------------------------------------------------------------

expected_result_rows_620hb = (
    TOTAL_SCENARIOS_620HB
    *
    len(
        candidate_names_620hb
    )
)


assert (
    len(
        temporal_candidate_results_620hb
    )
    ==
    expected_result_rows_620hb
)


numeric_result_columns_620hb = [
    "clean_mean_score",
    "corrupted_mean_score",
    "mean_score_delta",
    "median_score_delta",
    "p95_score_delta",
    "score_increased_windows_pct",
    "development_p95_threshold",
    "clean_above_threshold_pct",
    "corrupted_above_threshold_pct",
    "above_threshold_delta_pp",
    "target_sensor_mean_strength_delta",
    "target_sensor_median_strength_delta",
    "target_sensor_strength_increased_pct",
]


assert np.isfinite(
    temporal_candidate_results_620hb[
        numeric_result_columns_620hb
    ].to_numpy(
        dtype=np.float64
    )
).all()


print(
    "\n=== Scenario-result integrity ==="
)

print(
    "Rows:",
    len(
        temporal_candidate_results_620hb
    )
)

print(
    "Expected rows:",
    expected_result_rows_620hb
)

print(
    "Corruption families:",
    temporal_candidate_results_620hb[
        "corruption"
    ].nunique()
)

print(
    "Candidates:",
    temporal_candidate_results_620hb[
        "candidate"
    ].nunique()
)

print(
    "All key numeric values are finite."
)


# ------------------------------------------------------------
# 10. Balance duplicate directions
#
# Bias and drift have two directions per
# sensor/severity combination.
#
# We reduce every family to exactly:
# 15 sensors x 3 severities = 45 balanced cells.
# ------------------------------------------------------------

balanced_cells_620hb = (
    temporal_candidate_results_620hb
    .groupby(
        [
            "corruption",
            "sensor",
            "severity",
            "candidate",
        ],
        as_index=False,
    )
    .agg(
        source_scenarios=(
            "scenario_id",
            "size",
        ),

        mean_score_delta=(
            "mean_score_delta",
            "mean",
        ),

        median_score_delta=(
            "median_score_delta",
            "mean",
        ),

        above_threshold_delta_pp=(
            "above_threshold_delta_pp",
            "mean",
        ),

        score_increased_windows_pct=(
            "score_increased_windows_pct",
            "mean",
        ),

        target_sensor_mean_strength_delta=(
            "target_sensor_mean_strength_delta",
            "mean",
        ),

        target_sensor_strength_increased_pct=(
            "target_sensor_strength_increased_pct",
            "mean",
        ),
    )
)


expected_balanced_rows_620hb = (
    5
    *
    15
    *
    3
    *
    len(
        candidate_names_620hb
    )
)


assert (
    len(
        balanced_cells_620hb
    )
    ==
    expected_balanced_rows_620hb
)


print(
    "\n=== Balanced-cell structure ==="
)

print(
    "Rows:",
    len(
        balanced_cells_620hb
    )
)

print(
    "Expected:",
    expected_balanced_rows_620hb
)


source_structure_620hb = (
    balanced_cells_620hb
    .groupby(
        "corruption"
    )[
        "source_scenarios"
    ]
    .agg(
        ["min", "max", "mean"]
    )
    .reset_index()
)


display(
    source_structure_620hb
)


# ------------------------------------------------------------
# 11. Balanced candidate response by corruption family
# ------------------------------------------------------------

family_candidate_summary_620hb = (
    balanced_cells_620hb
    .groupby(
        [
            "candidate",
            "corruption",
        ],
        as_index=False,
    )
    .agg(
        balanced_cells=(
            "sensor",
            "size",
        ),

        mean_score_delta=(
            "mean_score_delta",
            "mean",
        ),

        median_score_delta=(
            "median_score_delta",
            "median",
        ),

        mean_above_threshold_delta_pp=(
            "above_threshold_delta_pp",
            "mean",
        ),

        median_above_threshold_delta_pp=(
            "above_threshold_delta_pp",
            "median",
        ),

        mean_score_increased_windows_pct=(
            "score_increased_windows_pct",
            "mean",
        ),

        mean_target_sensor_strength_delta=(
            "target_sensor_mean_strength_delta",
            "mean",
        ),
    )
)


print(
    "\n=== Balanced continuous temporal response "
    "by corruption family ==="
)

display(
    family_candidate_summary_620hb
    .round(6)
)


# ------------------------------------------------------------
# 12. Family × severity response
# ------------------------------------------------------------

severity_order_620hb = [
    "low",
    "medium",
    "high",
]


family_severity_summary_620hb = (
    balanced_cells_620hb
    .groupby(
        [
            "candidate",
            "corruption",
            "severity",
        ],
        as_index=False,
    )
    .agg(
        sensors=(
            "sensor",
            "nunique",
        ),

        mean_score_delta=(
            "mean_score_delta",
            "mean",
        ),

        mean_above_threshold_delta_pp=(
            "above_threshold_delta_pp",
            "mean",
        ),

        mean_target_sensor_strength_delta=(
            "target_sensor_mean_strength_delta",
            "mean",
        ),
    )
)


family_severity_summary_620hb[
    "severity"
] = pd.Categorical(
    family_severity_summary_620hb[
        "severity"
    ],
    categories=severity_order_620hb,
    ordered=True,
)


family_severity_summary_620hb = (
    family_severity_summary_620hb
    .sort_values(
        [
            "candidate",
            "corruption",
            "severity",
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n=== Balanced candidate response "
    "by family and severity ==="
)

display(
    family_severity_summary_620hb
    .round(6)
)


# ------------------------------------------------------------
# 13. Severity monotonicity
#
# Bias is shown for completeness, but it is NOT expected
# to generate temporal response because a constant offset
# cancels under temporal differencing.
# ------------------------------------------------------------

monotonicity_rows_620hb = []


for candidate_620hb in (
    candidate_names_620hb
):

    for family_620hb in (
        expected_families_620hb
    ):

        subset_620hb = (
            family_severity_summary_620hb[
                (
                    family_severity_summary_620hb[
                        "candidate"
                    ]
                    ==
                    candidate_620hb
                )
                &
                (
                    family_severity_summary_620hb[
                        "corruption"
                    ]
                    ==
                    family_620hb
                )
            ]
            .copy()
        )

        subset_620hb = (
            subset_620hb
            .set_index(
                "severity"
            )
            .reindex(
                severity_order_620hb
            )
        )


        score_values_620hb = (
            subset_620hb[
                "mean_score_delta"
            ]
            .to_numpy(
                dtype=np.float64
            )
        )

        event_values_620hb = (
            subset_620hb[
                "mean_above_threshold_delta_pp"
            ]
            .to_numpy(
                dtype=np.float64
            )
        )


        score_monotonic_620hb = bool(
            np.all(
                np.diff(
                    score_values_620hb
                )
                >=
                -1e-12
            )
        )

        event_monotonic_620hb = bool(
            np.all(
                np.diff(
                    event_values_620hb
                )
                >=
                -1e-12
            )
        )


        monotonicity_rows_620hb.append({
            "candidate":
                candidate_620hb,

            "corruption":
                family_620hb,

            "low_score_delta":
                score_values_620hb[0],

            "medium_score_delta":
                score_values_620hb[1],

            "high_score_delta":
                score_values_620hb[2],

            "score_delta_monotonic":
                score_monotonic_620hb,

            "low_event_delta_pp":
                event_values_620hb[0],

            "medium_event_delta_pp":
                event_values_620hb[1],

            "high_event_delta_pp":
                event_values_620hb[2],

            "event_delta_monotonic":
                event_monotonic_620hb,
        })


severity_monotonicity_620hb = (
    pd.DataFrame(
        monotonicity_rows_620hb
    )
)


print(
    "\n=== Severity monotonicity diagnostic ==="
)

display(
    severity_monotonicity_620hb
    .round(6)
)


# ------------------------------------------------------------
# 14. Dynamic-family comparison
#
# IMPORTANT:
# Bias is excluded here because temporal differencing should
# intentionally suppress a constant level shift.
#
# Dynamic families:
#   noise, drift, dropout, stuck
# ------------------------------------------------------------

dynamic_families_620hb = [
    "noise",
    "drift",
    "dropout",
    "stuck",
]


dynamic_cells_620hb = (
    balanced_cells_620hb[
        balanced_cells_620hb[
            "corruption"
        ].isin(
            dynamic_families_620hb
        )
    ]
    .copy()
)


candidate_dynamic_summary_620hb = (
    dynamic_cells_620hb
    .groupby(
        "candidate",
        as_index=False,
    )
    .agg(
        balanced_dynamic_cells=(
            "sensor",
            "size",
        ),

        mean_dynamic_score_delta=(
            "mean_score_delta",
            "mean",
        ),

        median_dynamic_score_delta=(
            "mean_score_delta",
            "median",
        ),

        mean_dynamic_event_delta_pp=(
            "above_threshold_delta_pp",
            "mean",
        ),

        median_dynamic_event_delta_pp=(
            "above_threshold_delta_pp",
            "median",
        ),

        mean_dynamic_score_increased_pct=(
            "score_increased_windows_pct",
            "mean",
        ),
    )
)


# Count dynamic families with monotonic mean-score response.
dynamic_monotonic_counts_620hb = (
    severity_monotonicity_620hb[
        severity_monotonicity_620hb[
            "corruption"
        ].isin(
            dynamic_families_620hb
        )
    ]
    .groupby(
        "candidate"
    )
    .agg(
        dynamic_score_monotonic_families=(
            "score_delta_monotonic",
            "sum",
        ),

        dynamic_event_monotonic_families=(
            "event_delta_monotonic",
            "sum",
        ),
    )
    .reset_index()
)


candidate_dynamic_summary_620hb = (
    candidate_dynamic_summary_620hb
    .merge(
        dynamic_monotonic_counts_620hb,
        on="candidate",
        how="left",
    )
)


# Add CLEAN false-positive reference.
candidate_dynamic_summary_620hb[
    "clean_validation_event_pct"
] = (
    candidate_dynamic_summary_620hb[
        "candidate"
    ]
    .map(
        clean_event_rates_620hb
    )
)


print(
    "\n=== Dynamic-family candidate comparison ==="
)

display(
    candidate_dynamic_summary_620hb
    .sort_values(
        "mean_dynamic_event_delta_pp",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
    .round(6)
)


# ------------------------------------------------------------
# 15. Bias invariance diagnostic
#
# A constant additive bias should produce essentially no
# temporal-transition change. This is EXPECTED behaviour,
# not a failure of the temporal component.
# ------------------------------------------------------------

bias_summary_620hb = (
    family_candidate_summary_620hb[
        family_candidate_summary_620hb[
            "corruption"
        ]
        ==
        "bias"
    ]
    [
        [
            "candidate",
            "mean_score_delta",
            "mean_above_threshold_delta_pp",
            "mean_target_sensor_strength_delta",
        ]
    ]
    .reset_index(
        drop=True
    )
)


print(
    "\n=== Constant-bias temporal invariance diagnostic ==="
)

display(
    bias_summary_620hb
    .round(10)
)


# ------------------------------------------------------------
# 16. Integrity summary
# ------------------------------------------------------------

print(
    "\n=== Step 6.20H-B integrity summary ==="
)

print(
    "Frozen scenarios evaluated         :",
    TOTAL_SCENARIOS_620HB
)

print(
    "Candidate scores per scenario      :",
    len(
        candidate_names_620hb
    )
)

print(
    "Scenario/candidate result rows     :",
    len(
        temporal_candidate_results_620hb
    )
)

print(
    "Balanced family/sensor/severity/"
    "candidate cells:",
    len(
        balanced_cells_620hb
    )
)

print(
    "Dynamic corruption families       :",
    dynamic_families_620hb
)

print(
    "CLEAN thresholds source            : "
    "development only"
)

print(
    "Full execution time (seconds)      :",
    round(
        elapsed_total_620hb,
        2
    )
)


print(
    "\nStep 6.20H-B completed successfully."
)

print(
    "All four continuous temporal-score candidates "
    "were evaluated across all 315 frozen scenarios."
)

print(
    "Bias/drift directions were balanced before "
    "cross-family comparison."
)

print(
    "Constant bias was treated as a temporal "
    "invariance diagnostic."
)

print(
    "No final temporal candidate was selected "
    "or frozen."
)

print(
    "No PCA + temporal combination was performed."
)

print(
    "No frozen CSV, model, scaler, or previous "
    "artifact was modified."
)

In [ ]:
# ============================================================
# Step 6.20H-C
# Candidate stability and sensor-dominance diagnostic
#
# PURPOSE:
#   1. Check whether the H-A / H-B pathway differences actually
#      alter clean threshold decisions.
#   2. Determine which sensor dominates the max_sensor score
#      on CLEAN validation.
#   3. Check candidate performance sensor by sensor across the
#      four dynamic corruption families.
#   4. Identify possible blind spots before selecting/fixing
#      the final temporal aggregation.
#
# IMPORTANT:
#   - No corruption is generated.
#   - No 315-scenario experiment is rerun.
#   - No threshold is recalibrated.
#   - No PCA + temporal combination is performed.
#   - No final temporal candidate is frozen.
#   - No artifact or CSV is modified.
# ============================================================

import numpy as np
import pandas as pd

print(
    "\n=== Step 6.20H-C — Candidate stability "
    "and sensor-dominance diagnostic ==="
)


# ------------------------------------------------------------
# 1. Required objects
# ------------------------------------------------------------

required_objects_620hc = [
    "clean_candidate_scores_620hb",
    "val_candidate_scores_620ha",
    "candidate_thresholds_620ha",
    "clean_sensor_strength_620hb",
    "RETAINED_SENSOR_COLUMNS",
    "balanced_cells_620hb",
    "severity_monotonicity_620hb",
    "clean_event_rates_620hb",
]

print("\n=== Required objects ===")

missing_objects_620hc = []

for name_620hc in required_objects_620hc:

    available_620hc = (
        name_620hc in globals()
    )

    print(
        f"{name_620hc:<34}: "
        f"{available_620hc}"
    )

    if not available_620hc:
        missing_objects_620hc.append(
            name_620hc
        )


assert not missing_objects_620hc, (
    "Missing required objects: "
    f"{missing_objects_620hc}"
)

print(
    "\nAll required objects are available."
)


# ------------------------------------------------------------
# 2. Basic structure verification
# ------------------------------------------------------------

candidate_names_620hc = list(
    clean_candidate_scores_620hb.columns
)

assert (
    candidate_names_620hc
    ==
    list(
        val_candidate_scores_620ha.columns
    )
)

assert (
    clean_sensor_strength_620hb.shape[1]
    ==
    len(
        RETAINED_SENSOR_COLUMNS
    )
)

assert (
    clean_sensor_strength_620hb.shape[0]
    ==
    len(
        clean_candidate_scores_620hb
    )
)


print(
    "\nCandidate temporal scores:",
    candidate_names_620hc
)

print(
    "Clean validation windows:",
    len(
        clean_candidate_scores_620hb
    )
)

print(
    "Retained sensors:",
    len(
        RETAINED_SENSOR_COLUMNS
    )
)


# ------------------------------------------------------------
# 3. H-A versus H-B pathway decision agreement
#
# We already saw small numerical score differences.
# The important question now is whether those differences
# change clean anomaly decisions at the frozen development
# P95 threshold.
# ------------------------------------------------------------

pathway_rows_620hc = []

for candidate_620hc in candidate_names_620hc:

    threshold_620hc = float(
        candidate_thresholds_620ha[
            candidate_620hc
        ]
    )

    ha_scores_620hc = (
        val_candidate_scores_620ha[
            candidate_620hc
        ]
        .to_numpy(
            dtype=np.float64
        )
    )

    hb_scores_620hc = (
        clean_candidate_scores_620hb[
            candidate_620hc
        ]
        .to_numpy(
            dtype=np.float64
        )
    )

    assert (
        ha_scores_620hc.shape
        ==
        hb_scores_620hc.shape
    )

    abs_diff_620hc = np.abs(
        ha_scores_620hc
        -
        hb_scores_620hc
    )

    ha_flags_620hc = (
        ha_scores_620hc
        >
        threshold_620hc
    )

    hb_flags_620hc = (
        hb_scores_620hc
        >
        threshold_620hc
    )

    disagreement_620hc = (
        ha_flags_620hc
        !=
        hb_flags_620hc
    )

    pathway_rows_620hc.append({

        "candidate":
            candidate_620hc,

        "threshold":
            threshold_620hc,

        "mean_abs_score_difference":
            float(
                np.mean(
                    abs_diff_620hc
                )
            ),

        "p95_abs_score_difference":
            float(
                np.quantile(
                    abs_diff_620hc,
                    0.95
                )
            ),

        "maximum_abs_score_difference":
            float(
                np.max(
                    abs_diff_620hc
                )
            ),

        "HA_event_pct":
            float(
                100.0
                *
                np.mean(
                    ha_flags_620hc
                )
            ),

        "HB_event_pct":
            float(
                100.0
                *
                np.mean(
                    hb_flags_620hc
                )
            ),

        "decision_disagreement_windows":
            int(
                np.sum(
                    disagreement_620hc
                )
            ),

        "decision_disagreement_pct":
            float(
                100.0
                *
                np.mean(
                    disagreement_620hc
                )
            ),
    })


pathway_stability_620hc = (
    pd.DataFrame(
        pathway_rows_620hc
    )
)


print(
    "\n=== H-A / H-B clean decision stability ==="
)

display(
    pathway_stability_620hc
    .round(8)
)


# ------------------------------------------------------------
# 4. CLEAN max-sensor winner distribution
#
# For each clean validation window, determine which retained
# sensor supplies the maximum temporal anomaly strength.
# ------------------------------------------------------------

winner_indices_620hc = np.argmax(
    clean_sensor_strength_620hb,
    axis=1,
)

winner_sensors_620hc = np.array(
    RETAINED_SENSOR_COLUMNS,
    dtype=object,
)[
    winner_indices_620hc
]


winner_counts_620hc = (
    pd.Series(
        winner_sensors_620hc,
        name="sensor"
    )
    .value_counts()
    .rename_axis(
        "sensor"
    )
    .reset_index(
        name="winning_windows"
    )
)


winner_counts_620hc[
    "winning_windows_pct"
] = (
    100.0
    *
    winner_counts_620hc[
        "winning_windows"
    ]
    /
    len(
        winner_sensors_620hc
    )
)


winner_counts_620hc[
    "cumulative_pct"
] = (
    winner_counts_620hc[
        "winning_windows_pct"
    ]
    .cumsum()
)


print(
    "\n=== CLEAN validation max-strength "
    "winner distribution ==="
)

display(
    winner_counts_620hc
    .round(6)
)


top_winner_sensor_620hc = (
    winner_counts_620hc
    .iloc[0][
        "sensor"
    ]
)

top_winner_share_620hc = float(
    winner_counts_620hc
    .iloc[0][
        "winning_windows_pct"
    ]
)


sensor17_share_620hc = float(
    winner_counts_620hc
    .loc[
        winner_counts_620hc[
            "sensor"
        ]
        ==
        "sensor_17",
        "winning_windows_pct",
    ]
    .sum()
)


print(
    "\nTop CLEAN max-strength sensor:",
    top_winner_sensor_620hc
)

print(
    "Top sensor winner share (%):",
    round(
        top_winner_share_620hc,
        6
    )
)

print(
    "sensor_17 winner share (%):",
    round(
        sensor17_share_620hc,
        6
    )
)


# ------------------------------------------------------------
# 5. Dynamic-corruption cells only
# ------------------------------------------------------------

dynamic_families_620hc = [
    "noise",
    "drift",
    "dropout",
    "stuck",
]


dynamic_cells_620hc = (
    balanced_cells_620hb[
        balanced_cells_620hb[
            "corruption"
        ].isin(
            dynamic_families_620hc
        )
    ]
    .copy()
)


expected_dynamic_cells_620hc = (
    4
    *
    15
    *
    3
    *
    len(
        candidate_names_620hc
    )
)


assert (
    len(
        dynamic_cells_620hc
    )
    ==
    expected_dynamic_cells_620hc
)


print(
    "\nDynamic balanced cells:",
    len(
        dynamic_cells_620hc
    )
)


# ------------------------------------------------------------
# 6. Per-candidate / per-target-sensor dynamic sensitivity
#
# Each candidate/sensor has:
# 4 families × 3 severities = 12 balanced cells.
# ------------------------------------------------------------

sensor_candidate_summary_620hc = (
    dynamic_cells_620hc
    .groupby(
        [
            "candidate",
            "sensor",
        ],
        as_index=False,
    )
    .agg(

        dynamic_cells=(
            "corruption",
            "size",
        ),

        mean_event_delta_pp=(
            "above_threshold_delta_pp",
            "mean",
        ),

        median_event_delta_pp=(
            "above_threshold_delta_pp",
            "median",
        ),

        minimum_event_delta_pp=(
            "above_threshold_delta_pp",
            "min",
        ),

        maximum_event_delta_pp=(
            "above_threshold_delta_pp",
            "max",
        ),

        mean_score_delta=(
            "mean_score_delta",
            "mean",
        ),

        mean_score_increased_windows_pct=(
            "score_increased_windows_pct",
            "mean",
        ),
    )
)


assert (
    sensor_candidate_summary_620hc[
        "dynamic_cells"
    ]
    ==
    12
).all()


print(
    "\n=== Per-target-sensor dynamic sensitivity ==="
)

display(
    sensor_candidate_summary_620hc
    .sort_values(
        [
            "candidate",
            "mean_event_delta_pp",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
    .round(6)
)


# ------------------------------------------------------------
# 7. Candidate robustness across target sensors
#
# This is important:
# a good temporal score should not work only for a small
# subset of sensors.
# ------------------------------------------------------------

candidate_sensor_robustness_620hc = (
    sensor_candidate_summary_620hc
    .groupby(
        "candidate",
        as_index=False,
    )
    .agg(

        sensors=(
            "sensor",
            "nunique",
        ),

        mean_sensor_event_delta_pp=(
            "mean_event_delta_pp",
            "mean",
        ),

        median_sensor_event_delta_pp=(
            "mean_event_delta_pp",
            "median",
        ),

        worst_sensor_event_delta_pp=(
            "mean_event_delta_pp",
            "min",
        ),

        best_sensor_event_delta_pp=(
            "mean_event_delta_pp",
            "max",
        ),

        mean_sensor_score_delta=(
            "mean_score_delta",
            "mean",
        ),
    )
)


positive_sensor_counts_620hc = (
    sensor_candidate_summary_620hc
    .assign(
        positive_sensor=lambda df_620hc:
            (
                df_620hc[
                    "mean_event_delta_pp"
                ]
                >
                0.0
            )
    )
    .groupby(
        "candidate",
        as_index=False,
    )
    .agg(
        sensors_with_positive_mean_response=(
            "positive_sensor",
            "sum",
        )
    )
)


candidate_sensor_robustness_620hc = (
    candidate_sensor_robustness_620hc
    .merge(
        positive_sensor_counts_620hc,
        on="candidate",
        how="left",
    )
)


candidate_sensor_robustness_620hc[
    "clean_validation_event_pct"
] = (
    candidate_sensor_robustness_620hc[
        "candidate"
    ]
    .map(
        clean_event_rates_620hb
    )
)


print(
    "\n=== Candidate robustness across target sensors ==="
)

display(
    candidate_sensor_robustness_620hc
    .sort_values(
        "mean_sensor_event_delta_pp",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
    .round(6)
)


# ------------------------------------------------------------
# 8. Which candidate wins sensor by sensor?
# ------------------------------------------------------------

sensor_candidate_pivot_620hc = (
    sensor_candidate_summary_620hc
    .pivot(
        index="sensor",
        columns="candidate",
        values="mean_event_delta_pp",
    )
)


sensor_candidate_pivot_620hc[
    "best_candidate"
] = (
    sensor_candidate_pivot_620hc[
        candidate_names_620hc
    ]
    .idxmax(
        axis=1
    )
)


sensor_candidate_pivot_620hc[
    "best_event_delta_pp"
] = (
    sensor_candidate_pivot_620hc[
        candidate_names_620hc
    ]
    .max(
        axis=1
    )
)


print(
    "\n=== Best temporal candidate for each "
    "target sensor ==="
)

display(
    sensor_candidate_pivot_620hc
    .sort_values(
        "best_event_delta_pp",
        ascending=False,
    )
    .round(6)
)


candidate_win_counts_620hc = (
    sensor_candidate_pivot_620hc[
        "best_candidate"
    ]
    .value_counts()
    .rename_axis(
        "candidate"
    )
    .reset_index(
        name="target_sensors_won"
    )
)


print(
    "\n=== Sensor-level candidate win counts ==="
)

display(
    candidate_win_counts_620hc
)


# ------------------------------------------------------------
# 9. Worst-performing target sensors for max_sensor
#
# This directly checks whether max_sensor has meaningful
# blind spots.
# ------------------------------------------------------------

max_sensor_target_summary_620hc = (
    sensor_candidate_summary_620hc[
        sensor_candidate_summary_620hc[
            "candidate"
        ]
        ==
        "max_sensor"
    ]
    .sort_values(
        "mean_event_delta_pp",
        ascending=True,
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n=== max_sensor — weakest target-sensor responses ==="
)

display(
    max_sensor_target_summary_620hc
    .head(8)
    .round(6)
)


# ------------------------------------------------------------
# 10. Monotonicity summary for dynamic families
# ------------------------------------------------------------

dynamic_monotonicity_620hc = (
    severity_monotonicity_620hb[
        severity_monotonicity_620hb[
            "corruption"
        ].isin(
            dynamic_families_620hc
        )
    ]
    .groupby(
        "candidate",
        as_index=False,
    )
    .agg(

        score_monotonic_families=(
            "score_delta_monotonic",
            "sum",
        ),

        event_monotonic_families=(
            "event_delta_monotonic",
            "sum",
        ),
    )
)


print(
    "\n=== Dynamic-family monotonicity summary ==="
)

display(
    dynamic_monotonicity_620hc
)


# ------------------------------------------------------------
# 11. Evidence table for final selection
#
# No arbitrary composite score is created.
# We preserve the individual scientific criteria.
# ------------------------------------------------------------

selection_evidence_620hc = (
    candidate_sensor_robustness_620hc
    .merge(
        dynamic_monotonicity_620hc,
        on="candidate",
        how="left",
    )
    .merge(
        pathway_stability_620hc[
            [
                "candidate",
                "decision_disagreement_pct",
                "maximum_abs_score_difference",
            ]
        ],
        on="candidate",
        how="left",
    )
)


selection_evidence_620hc[
    "clean_calibration_error_from_5pct"
] = np.abs(
    selection_evidence_620hc[
        "clean_validation_event_pct"
    ]
    -
    5.0
)


selection_evidence_620hc = (
    selection_evidence_620hc[
        [
            "candidate",
            "clean_validation_event_pct",
            "clean_calibration_error_from_5pct",
            "mean_sensor_event_delta_pp",
            "median_sensor_event_delta_pp",
            "worst_sensor_event_delta_pp",
            "sensors_with_positive_mean_response",
            "score_monotonic_families",
            "event_monotonic_families",
            "decision_disagreement_pct",
            "maximum_abs_score_difference",
        ]
    ]
)


print(
    "\n=== Evidence table for temporal-candidate selection ==="
)

display(
    selection_evidence_620hc
    .sort_values(
        "mean_sensor_event_delta_pp",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
    .round(8)
)


# ------------------------------------------------------------
# 12. Integrity summary
# ------------------------------------------------------------

print(
    "\n=== Step 6.20H-C integrity summary ==="
)

print(
    "Clean validation windows analysed :",
    len(
        clean_candidate_scores_620hb
    )
)

print(
    "Retained sensors analysed         :",
    len(
        RETAINED_SENSOR_COLUMNS
    )
)

print(
    "Temporal candidates compared      :",
    len(
        candidate_names_620hc
    )
)

print(
    "Dynamic corruption families       :",
    dynamic_families_620hc
)

print(
    "Dynamic balanced cells analysed   :",
    len(
        dynamic_cells_620hc
    )
)

print(
    "Corruption experiment rerun       : False"
)

print(
    "Threshold recalibration performed : False"
)

print(
    "PCA + temporal combination done   : False"
)

print(
    "Final temporal candidate frozen   : False"
)


print(
    "\nStep 6.20H-C completed successfully."
)

print(
    "Candidate stability, clean max-sensor dominance, "
    "and per-target-sensor robustness were diagnosed."
)

print(
    "No existing experimental artifact was modified."
)

In [ ]:
# ============================================================
# Step 6.20H-C1
# Export H-C diagnostic results for review
# ============================================================

from pathlib import Path
import pandas as pd

print("\n=== Step 6.20H-C1 — Export diagnostic results ===")


# ------------------------------------------------------------
# 1. Locate project root
# ------------------------------------------------------------

cwd_620hc1 = Path.cwd()

if (cwd_620hc1 / "06_results").exists():
    project_root_620hc1 = cwd_620hc1

elif (cwd_620hc1.parent / "06_results").exists():
    project_root_620hc1 = cwd_620hc1.parent

else:
    raise FileNotFoundError(
        "Could not locate project root containing 06_results."
    )


output_dir_620hc1 = (
    project_root_620hc1
    / "06_results"
    / "step6_reliability_signals"
)

output_dir_620hc1.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# 2. Verify required H-C result tables
# ------------------------------------------------------------

required_tables_620hc1 = {
    "pathway_stability_620hc":
        pathway_stability_620hc,

    "winner_counts_620hc":
        winner_counts_620hc,

    "candidate_sensor_robustness_620hc":
        candidate_sensor_robustness_620hc,

    "selection_evidence_620hc":
        selection_evidence_620hc,
}


for name_620hc1, table_620hc1 in required_tables_620hc1.items():

    assert isinstance(
        table_620hc1,
        pd.DataFrame
    ), f"{name_620hc1} is not a DataFrame."

    assert len(
        table_620hc1
    ) > 0, f"{name_620hc1} is empty."


print("All required Step 6.20H-C tables are available.")


# ------------------------------------------------------------
# 3. Save full tables as CSV
# ------------------------------------------------------------

csv_paths_620hc1 = {}


for name_620hc1, table_620hc1 in required_tables_620hc1.items():

    short_name_620hc1 = (
        name_620hc1
        .replace("_620hc", "")
    )

    path_620hc1 = (
        output_dir_620hc1
        / f"fd001_temporal_{short_name_620hc1}.csv"
    )

    table_620hc1.to_csv(
        path_620hc1,
        index=False
    )

    csv_paths_620hc1[
        short_name_620hc1
    ] = path_620hc1


# ------------------------------------------------------------
# 4. Create compact human-readable review file
# ------------------------------------------------------------

summary_path_620hc1 = (
    output_dir_620hc1
    / "fd001_step620hc_review_summary.txt"
)


with open(
    summary_path_620hc1,
    "w",
    encoding="utf-8"
) as f_620hc1:

    f_620hc1.write(
        "STEP 6.20H-C REVIEW SUMMARY\n"
    )

    f_620hc1.write(
        "=" * 80 + "\n\n"
    )

    # A. Pathway stability
    f_620hc1.write(
        "1. H-A / H-B CLEAN DECISION STABILITY\n"
    )

    f_620hc1.write(
        "-" * 80 + "\n"
    )

    f_620hc1.write(
        pathway_stability_620hc
        .round(8)
        .to_string(index=False)
    )

    f_620hc1.write(
        "\n\n"
    )


    # B. Winner distribution
    f_620hc1.write(
        "2. CLEAN VALIDATION MAX-STRENGTH WINNER DISTRIBUTION\n"
    )

    f_620hc1.write(
        "-" * 80 + "\n"
    )

    f_620hc1.write(
        winner_counts_620hc
        .round(6)
        .to_string(index=False)
    )

    f_620hc1.write(
        "\n\n"
    )


    # C. Candidate robustness
    f_620hc1.write(
        "3. CANDIDATE ROBUSTNESS ACROSS TARGET SENSORS\n"
    )

    f_620hc1.write(
        "-" * 80 + "\n"
    )

    f_620hc1.write(
        candidate_sensor_robustness_620hc
        .sort_values(
            "mean_sensor_event_delta_pp",
            ascending=False
        )
        .round(6)
        .to_string(index=False)
    )

    f_620hc1.write(
        "\n\n"
    )


    # D. Final evidence table
    f_620hc1.write(
        "4. TEMPORAL-CANDIDATE SELECTION EVIDENCE\n"
    )

    f_620hc1.write(
        "-" * 80 + "\n"
    )

    f_620hc1.write(
        selection_evidence_620hc
        .sort_values(
            "mean_sensor_event_delta_pp",
            ascending=False
        )
        .round(8)
        .to_string(index=False)
    )

    f_620hc1.write(
        "\n\n"
    )


    # E. Important compact indicators
    f_620hc1.write(
        "5. KEY INDICATORS\n"
    )

    f_620hc1.write(
        "-" * 80 + "\n"
    )

    f_620hc1.write(
        f"Top clean max-strength sensor: "
        f"{top_winner_sensor_620hc}\n"
    )

    f_620hc1.write(
        f"Top sensor winner share (%): "
        f"{top_winner_share_620hc:.6f}\n"
    )

    f_620hc1.write(
        f"sensor_17 winner share (%): "
        f"{sensor17_share_620hc:.6f}\n"
    )


# ------------------------------------------------------------
# 5. Compact notebook output
# ------------------------------------------------------------

print("\nFiles saved successfully:\n")

for key_620hc1, path_620hc1 in csv_paths_620hc1.items():
    print(
        f"{key_620hc1:<30}: "
        f"{path_620hc1}"
    )


print(
    "\nReview summary:",
    summary_path_620hc1
)


print(
    "\n=== Compact selection evidence ==="
)

display(
    selection_evidence_620hc
    .sort_values(
        "mean_sensor_event_delta_pp",
        ascending=False
    )
    .reset_index(drop=True)
    .round(6)
)


print(
    "\nStep 6.20H-C1 completed successfully."
)

In [ ]:
# ================================================================
# Step 6.20I — Freeze selected temporal-quality signal
# ================================================================
#
# PURPOSE
# -------
# Freeze the temporal-quality candidate selected after Steps
# 6.20H-A and 6.20H-B.
#
# Selected candidate:
#     max_sensor
#
# Important:
# - Threshold comes from CLEAN DEVELOPMENT only.
# - CLEAN validation is used only as a sanity check.
# - No corruption is generated here.
# - No model/scaler/PCA artifact is modified.
# - Existing output files are NEVER silently overwritten.
#
# ================================================================

from pathlib import Path
import json
import hashlib
import numpy as np
import pandas as pd


print(
    "\n=== Step 6.20I — Freeze selected temporal-quality signal ==="
)


# ================================================================
# 1. Required-object verification
# ================================================================

required_objects_620i = [
    "candidate_thresholds_620ha",
    "val_candidate_scores_620ha",
    "temporal_reference_620gc",
    "RETAINED_SENSOR_COLUMNS",
    "PRIMARY_INPUT_COLUMNS",
]

print("\n=== Required objects ===")

required_status_620i = {}

for name_620i in required_objects_620i:
    available_620i = name_620i in globals()
    required_status_620i[name_620i] = available_620i

    print(
        f"{name_620i:<32}: "
        f"{available_620i}"
    )

assert all(required_status_620i.values()), (
    "Step 6.20I cannot continue because one or more required "
    "objects are missing."
)

print("\nAll required objects are available.")


# ================================================================
# 2. Freeze candidate identity
# ================================================================

SELECTED_TEMPORAL_CANDIDATE_620I = "max_sensor"

print("\n=== Candidate selected for freezing ===")
print(
    "Selected temporal candidate :",
    SELECTED_TEMPORAL_CANDIDATE_620I
)


# ================================================================
# 3. Robust threshold extractor
# ================================================================

def extract_candidate_threshold_620i(
    threshold_object,
    candidate_name
):
    """
    Extract the development-calibrated threshold for one candidate.

    Supports the structures used during Step 6.20H-A without
    modifying the original object.
    """

    # ------------------------------------------------------------
    # DataFrame
    # ------------------------------------------------------------
    if isinstance(threshold_object, pd.DataFrame):

        preferred_columns_620i = [
            "development_p95_threshold",
            "dev_p95_threshold",
            "p95_threshold",
            "threshold",
        ]

        # Candidate stored in index
        if candidate_name in threshold_object.index:

            for column_620i in preferred_columns_620i:

                if column_620i in threshold_object.columns:

                    value_620i = (
                        threshold_object
                        .loc[candidate_name, column_620i]
                    )

                    return float(value_620i)

            # If only one numeric column exists
            numeric_cols_620i = (
                threshold_object
                .select_dtypes(include=[np.number])
                .columns
                .tolist()
            )

            if len(numeric_cols_620i) == 1:

                return float(
                    threshold_object.loc[
                        candidate_name,
                        numeric_cols_620i[0]
                    ]
                )

        # Candidate stored in a column
        if "candidate" in threshold_object.columns:

            row_620i = threshold_object[
                threshold_object["candidate"].astype(str)
                == str(candidate_name)
            ]

            if len(row_620i) == 1:

                for column_620i in preferred_columns_620i:

                    if column_620i in row_620i.columns:

                        return float(
                            row_620i.iloc[0][column_620i]
                        )

    # ------------------------------------------------------------
    # Series
    # ------------------------------------------------------------
    if isinstance(threshold_object, pd.Series):

        if candidate_name in threshold_object.index:

            return float(
                threshold_object.loc[candidate_name]
            )

    # ------------------------------------------------------------
    # Dictionary
    # ------------------------------------------------------------
    if isinstance(threshold_object, dict):

        if candidate_name in threshold_object:

            value_620i = threshold_object[candidate_name]

            if np.isscalar(value_620i):

                return float(value_620i)

            if isinstance(value_620i, dict):

                for key_620i in [
                    "development_p95_threshold",
                    "dev_p95_threshold",
                    "p95_threshold",
                    "threshold",
                ]:

                    if key_620i in value_620i:

                        return float(
                            value_620i[key_620i]
                        )

    raise ValueError(
        "Could not extract the development threshold for "
        f"candidate '{candidate_name}'."
    )


temporal_threshold_620i = (
    extract_candidate_threshold_620i(
        candidate_thresholds_620ha,
        SELECTED_TEMPORAL_CANDIDATE_620I,
    )
)

assert np.isfinite(temporal_threshold_620i), (
    "Selected temporal threshold is non-finite."
)


print("\n=== Development-calibrated threshold ===")
print(
    "Candidate        :",
    SELECTED_TEMPORAL_CANDIDATE_620I
)
print(
    "Threshold source : CLEAN DEVELOPMENT only"
)
print(
    "Threshold type   : development p95"
)
print(
    "Threshold value  :",
    temporal_threshold_620i
)


# ------------------------------------------------
# Important consistency check from Step 6.20H-A
# ------------------------------------------------

expected_threshold_620i = 1.098612

assert np.isclose(
    temporal_threshold_620i,
    expected_threshold_620i,
    atol=5e-6,
    rtol=0.0,
), (
    "The extracted max_sensor threshold does not agree with the "
    "previous Step-6.20H-A result. Do not freeze the signal."
)

print(
    "Expected Step-6.20H-A threshold agreement : True"
)


# ================================================================
# 4. Robust candidate-score extractor
# ================================================================

def extract_candidate_scores_620i(
    score_object,
    candidate_name
):
    """
    Extract one candidate's window-level scores without modifying
    the source object.
    """

    # ------------------------------------------------------------
    # DataFrame
    # ------------------------------------------------------------
    if isinstance(score_object, pd.DataFrame):

        # Candidate stored as a column
        if candidate_name in score_object.columns:

            return (
                score_object[candidate_name]
                .to_numpy(dtype=np.float64)
            )

        # Long-format structure
        if "candidate" in score_object.columns:

            subset_620i = score_object[
                score_object["candidate"].astype(str)
                == str(candidate_name)
            ]

            possible_score_columns_620i = [
                "score",
                "temporal_score",
                "candidate_score",
                "value",
            ]

            for column_620i in possible_score_columns_620i:

                if column_620i in subset_620i.columns:

                    return (
                        subset_620i[column_620i]
                        .to_numpy(dtype=np.float64)
                    )

    # ------------------------------------------------------------
    # Dictionary
    # ------------------------------------------------------------
    if isinstance(score_object, dict):

        if candidate_name in score_object:

            return np.asarray(
                score_object[candidate_name],
                dtype=np.float64,
            ).reshape(-1)

    raise ValueError(
        "Could not extract CLEAN-validation scores for "
        f"candidate '{candidate_name}'."
    )


clean_val_temporal_scores_620i = (
    extract_candidate_scores_620i(
        val_candidate_scores_620ha,
        SELECTED_TEMPORAL_CANDIDATE_620I,
    )
)


assert clean_val_temporal_scores_620i.ndim == 1, (
    "Validation temporal score must be one-dimensional."
)

assert len(clean_val_temporal_scores_620i) == 3490, (
    "Unexpected number of clean-validation temporal scores."
)

assert np.isfinite(
    clean_val_temporal_scores_620i
).all(), (
    "Non-finite values found in CLEAN-validation temporal score."
)


# ================================================================
# 5. CLEAN-validation sanity check
# ================================================================

clean_val_above_threshold_pct_620i = float(
    100.0
    * np.mean(
        clean_val_temporal_scores_620i
        > temporal_threshold_620i
    )
)


print("\n=== CLEAN-validation sanity check ===")
print(
    "Validation windows          :",
    len(clean_val_temporal_scores_620i)
)
print(
    "Temporal-score mean         :",
    float(np.mean(clean_val_temporal_scores_620i))
)
print(
    "Temporal-score median       :",
    float(np.median(clean_val_temporal_scores_620i))
)
print(
    "Temporal-score p95          :",
    float(
        np.quantile(
            clean_val_temporal_scores_620i,
            0.95
        )
    )
)
print(
    "Above development threshold : "
    f"{clean_val_above_threshold_pct_620i:.6f}%"
)


# Previous Step 6.20H-A result was approximately 4.871%.
# Use a narrow but non-fragile sanity interval.

assert (
    4.0
    <= clean_val_above_threshold_pct_620i
    <= 6.0
), (
    "Clean-validation event rate is inconsistent with the "
    "previous Step-6.20H-A result. Do not freeze the signal."
)

print(
    "Clean-validation sanity check passed."
)


# ================================================================
# 6. Freeze temporal reference IN MEMORY
# ================================================================

frozen_temporal_reference_620i = (
    temporal_reference_620gc.copy(deep=True)
)


assert isinstance(
    frozen_temporal_reference_620i,
    pd.DataFrame
), (
    "Temporal reference must be a DataFrame."
)


# Make sensor identity explicit
frozen_temporal_reference_620i.index = (
    frozen_temporal_reference_620i
    .index
    .astype(str)
)

frozen_temporal_reference_620i.index.name = "sensor"


# ------------------------------------------------
# Sensor coverage
# ------------------------------------------------

retained_sensors_620i = [
    str(sensor_620i)
    for sensor_620i
    in RETAINED_SENSOR_COLUMNS
]

reference_sensors_620i = (
    frozen_temporal_reference_620i
    .index
    .tolist()
)


assert len(retained_sensors_620i) == 15, (
    "Expected 15 retained sensors."
)

assert set(reference_sensors_620i) == set(
    retained_sensors_620i
), (
    "Temporal reference does not cover exactly the retained "
    "sensor set."
)


# ------------------------------------------------
# Special-handling sensor
# ------------------------------------------------

assert (
    "special_handling"
    in frozen_temporal_reference_620i.columns
), (
    "Temporal reference does not contain special_handling."
)


special_temporal_sensors_620i = (
    frozen_temporal_reference_620i[
        frozen_temporal_reference_620i[
            "special_handling"
        ].astype(bool)
    ]
    .index
    .tolist()
)


print("\n=== Frozen temporal-reference coverage ===")
print(
    "Retained sensors          :",
    len(retained_sensors_620i)
)
print(
    "Reference sensors         :",
    len(reference_sensors_620i)
)
print(
    "Special-handling sensors :",
    special_temporal_sensors_620i
)


assert special_temporal_sensors_620i == [
    "sensor_6"
], (
    "Special temporal handling does not match the previously "
    "verified Step-6.20 results."
)


# ================================================================
# 7. Create compact CLEAN-validation summary
# ================================================================

clean_validation_summary_620i = pd.DataFrame(
    [
        {
            "dataset": "FD001",
            "candidate": (
                SELECTED_TEMPORAL_CANDIDATE_620I
            ),
            "threshold_source": (
                "clean_development_only"
            ),
            "threshold_quantile": "p95",
            "development_threshold": (
                temporal_threshold_620i
            ),
            "clean_validation_windows": (
                int(
                    len(
                        clean_val_temporal_scores_620i
                    )
                )
            ),
            "clean_validation_mean": float(
                np.mean(
                    clean_val_temporal_scores_620i
                )
            ),
            "clean_validation_median": float(
                np.median(
                    clean_val_temporal_scores_620i
                )
            ),
            "clean_validation_p95": float(
                np.quantile(
                    clean_val_temporal_scores_620i,
                    0.95,
                )
            ),
            "clean_validation_min": float(
                np.min(
                    clean_val_temporal_scores_620i
                )
            ),
            "clean_validation_max": float(
                np.max(
                    clean_val_temporal_scores_620i
                )
            ),
            "clean_validation_above_threshold_pct": (
                clean_val_above_threshold_pct_620i
            ),
            "retained_sensor_count": (
                len(retained_sensors_620i)
            ),
            "special_handling_sensors": ",".join(
                special_temporal_sensors_620i
            ),
        }
    ]
)


print(
    "\n=== Frozen CLEAN-validation summary ==="
)

display(
    clean_validation_summary_620i
)


# ================================================================
# 8. Locate project root safely
# ================================================================

cwd_620i = Path.cwd().resolve()

project_root_620i = None

search_locations_620i = [
    cwd_620i,
    *cwd_620i.parents,
]

for candidate_root_620i in search_locations_620i:

    if (
        (
            candidate_root_620i
            / "03_notebooks"
        ).exists()
        and
        (
            candidate_root_620i
            / "06_results"
        ).exists()
    ):
        project_root_620i = candidate_root_620i
        break


if project_root_620i is None:

    raise FileNotFoundError(
        "Could not locate the RUL_Reliability_Research "
        "project root."
    )


output_dir_620i = (
    project_root_620i
    / "06_results"
    / "step6_reliability_signals"
)

output_dir_620i.mkdir(
    parents=True,
    exist_ok=True,
)


print("\n=== Output location ===")
print(
    "Project root :",
    project_root_620i
)
print(
    "Output dir   :",
    output_dir_620i
)


# ================================================================
# 9. Frozen artifact paths
# ================================================================

reference_path_620i = (
    output_dir_620i
    / "fd001_temporal_quality_reference.csv"
)

validation_summary_path_620i = (
    output_dir_620i
    / "fd001_temporal_quality_clean_validation_summary.csv"
)

definition_path_620i = (
    output_dir_620i
    / "fd001_temporal_quality_signal_definition.json"
)


# ================================================================
# 10. Helper: SHA-256
# ================================================================

def sha256_bytes_620i(data_620i):

    return hashlib.sha256(
        data_620i
    ).hexdigest()


# ================================================================
# 11. Helper: frozen safe writer
# ================================================================

def write_frozen_bytes_620i(
    path_620i,
    data_620i
):
    """
    Write a frozen artifact safely.

    If file does not exist:
        write it.

    If file already exists and is identical:
        leave it untouched.

    If file exists but differs:
        STOP rather than overwrite.
    """

    if path_620i.exists():

        existing_620i = (
            path_620i.read_bytes()
        )

        if existing_620i == data_620i:

            print(
                "Already exists and is identical:",
                path_620i
            )

            return "identical"

        raise FileExistsError(
            "\nA frozen artifact already exists but its "
            "content is different:\n"
            f"{path_620i}\n\n"
            "The file was NOT overwritten."
        )

    path_620i.write_bytes(data_620i)

    print(
        "Saved:",
        path_620i
    )

    return "created"


# ================================================================
# 12. Prepare temporal-reference CSV
# ================================================================

reference_csv_text_620i = (
    frozen_temporal_reference_620i
    .to_csv(
        index=True,
        lineterminator="\n",
    )
)

reference_csv_bytes_620i = (
    reference_csv_text_620i
    .encode("utf-8")
)

reference_sha256_620i = (
    sha256_bytes_620i(
        reference_csv_bytes_620i
    )
)


# ================================================================
# 13. Prepare CLEAN-validation summary CSV
# ================================================================

validation_csv_text_620i = (
    clean_validation_summary_620i
    .to_csv(
        index=False,
        lineterminator="\n",
    )
)

validation_csv_bytes_620i = (
    validation_csv_text_620i
    .encode("utf-8")
)

validation_sha256_620i = (
    sha256_bytes_620i(
        validation_csv_bytes_620i
    )
)


# ================================================================
# 14. Create frozen signal definition
# ================================================================

temporal_signal_definition_620i = {

    "schema_version": "1.0",

    "dataset": "NASA C-MAPSS FD001",

    "research_step": "6.20I",

    "artifact_status": "frozen",

    "signal_type": "temporal_sensor_quality",

    "selected_candidate": (
        SELECTED_TEMPORAL_CANDIDATE_620I
    ),

    "aggregation_rule": (
        "maximum per-sensor continuous temporal "
        "anomaly strength within each window"
    ),

    "calibration": {

        "source": (
            "clean development data only"
        ),

        "threshold_quantile": "p95",

        "threshold": (
            temporal_threshold_620i
        ),
    },

    "validation_policy": {

        "clean_validation_role": (
            "sanity check only; not used to set "
            "the threshold"
        ),

        "clean_validation_windows": (
            int(
                len(
                    clean_val_temporal_scores_620i
                )
            )
        ),

        "clean_validation_above_threshold_pct": (
            clean_val_above_threshold_pct_620i
        ),
    },

    "sensor_configuration": {

        "retained_sensor_count": (
            len(retained_sensors_620i)
        ),

        "retained_sensors": (
            retained_sensors_620i
        ),

        "special_handling_sensors": (
            special_temporal_sensors_620i
        ),
    },

    "selection_basis": [

        (
            "Candidate thresholds were calibrated "
            "using clean development data."
        ),

        (
            "Clean validation was used only as a "
            "sanity check."
        ),

        (
            "The selected max_sensor candidate showed "
            "monotonic response across the four dynamic "
            "corruption families evaluated in Step 6.20H-B."
        ),

        (
            "Constant bias was retained as a temporal "
            "invariance diagnostic because a constant "
            "offset does not itself alter temporal "
            "transition behaviour."
        ),
    ],

    "provenance": {

        "candidate_source_step": "6.20H-A",

        "cross_corruption_evaluation_step": (
            "6.20H-B"
        ),

        "temporal_reference_source_step": (
            "6.20G-C"
        ),

        "reference_csv": (
            reference_path_620i.name
        ),

        "reference_csv_sha256": (
            reference_sha256_620i
        ),

        "clean_validation_summary_csv": (
            validation_summary_path_620i.name
        ),

        "clean_validation_summary_sha256": (
            validation_sha256_620i
        ),
    },

    "immutability_statement": (
        "Step 6.20I freezes the selected temporal "
        "signal definition only. No corruption "
        "scenario, model, scaler, PCA artifact, or "
        "previous frozen result is modified."
    ),
}


definition_json_text_620i = (
    json.dumps(
        temporal_signal_definition_620i,
        indent=4,
        sort_keys=True,
        ensure_ascii=False,
    )
    + "\n"
)

definition_json_bytes_620i = (
    definition_json_text_620i
    .encode("utf-8")
)


# ================================================================
# 15. Save frozen artifacts safely
# ================================================================

print("\n=== Freezing temporal artifacts ===")

reference_write_status_620i = (
    write_frozen_bytes_620i(
        reference_path_620i,
        reference_csv_bytes_620i,
    )
)

validation_write_status_620i = (
    write_frozen_bytes_620i(
        validation_summary_path_620i,
        validation_csv_bytes_620i,
    )
)

definition_write_status_620i = (
    write_frozen_bytes_620i(
        definition_path_620i,
        definition_json_bytes_620i,
    )
)


# ================================================================
# 16. Re-read files and verify exact integrity
# ================================================================

reference_hash_from_disk_620i = (
    sha256_bytes_620i(
        reference_path_620i.read_bytes()
    )
)

validation_hash_from_disk_620i = (
    sha256_bytes_620i(
        validation_summary_path_620i.read_bytes()
    )
)

definition_hash_from_disk_620i = (
    sha256_bytes_620i(
        definition_path_620i.read_bytes()
    )
)


assert (
    reference_hash_from_disk_620i
    == reference_sha256_620i
), (
    "Saved temporal-reference hash mismatch."
)

assert (
    validation_hash_from_disk_620i
    == validation_sha256_620i
), (
    "Saved validation-summary hash mismatch."
)


# ================================================================
# 17. Create explicit in-memory frozen aliases for next steps
# ================================================================

FROZEN_TEMPORAL_CANDIDATE_620I = (
    SELECTED_TEMPORAL_CANDIDATE_620I
)

FROZEN_TEMPORAL_THRESHOLD_620I = float(
    temporal_threshold_620i
)

FROZEN_TEMPORAL_REFERENCE_620I = (
    frozen_temporal_reference_620i.copy(
        deep=True
    )
)

FROZEN_TEMPORAL_DEFINITION_620I = (
    temporal_signal_definition_620i.copy()
)


# ================================================================
# 18. Final integrity summary
# ================================================================

print(
    "\n=== Step 6.20I integrity summary ==="
)

print(
    "Selected candidate             :",
    FROZEN_TEMPORAL_CANDIDATE_620I
)

print(
    "Development p95 threshold      :",
    FROZEN_TEMPORAL_THRESHOLD_620I
)

print(
    "Clean-validation event rate (%) :",
    round(
        clean_val_above_threshold_pct_620i,
        6
    )
)

print(
    "Retained sensors               :",
    len(retained_sensors_620i)
)

print(
    "Special-handling sensors       :",
    special_temporal_sensors_620i
)

print(
    "Reference artifact SHA-256     :",
    reference_hash_from_disk_620i
)

print(
    "Definition artifact SHA-256    :",
    definition_hash_from_disk_620i
)


print("\nFrozen files:")

print(
    "1.",
    reference_path_620i
)

print(
    "2.",
    validation_summary_path_620i
)

print(
    "3.",
    definition_path_620i
)


print(
    "\nStep 6.20I completed successfully."
)

print(
    "The max_sensor temporal-quality signal is now "
    "FROZEN for the FD001 Step-6 experiments."
)

print(
    "Its threshold was derived from CLEAN DEVELOPMENT only."
)

print(
    "CLEAN validation was used only as a sanity check."
)

print(
    "No corruption was generated."
)

print(
    "No model, scaler, PCA result, or previous frozen "
    "artifact was modified."
)

print(
    "The notebook is now ready for Step 6.21: "
    "PCA-versus-temporal complementary-signal analysis."
)

In [ ]:
# ================================================================
# Step 6.21A — PCA / Temporal complementarity pre-check
# CORRECTED VERSION
# ================================================================
#
# PURPOSE
# -------
# Before combining PCA and temporal-quality signals:
#
# 1. Confirm the newly frozen temporal signal.
# 2. Locate existing PCA-related objects/results.
# 3. Inspect relevant PCA objects currently in memory.
# 4. Inspect Step-6 result files.
#
# IMPORTANT
# ---------
# Inspection only.
# No PCA-temporal fusion.
# No threshold selection.
# No corruption generation.
# No frozen artifact modification.
#
# ================================================================

from pathlib import Path
import pandas as pd
import numpy as np


print(
    "\n=== Step 6.21A — "
    "PCA / Temporal complementarity pre-check ==="
)


# ================================================================
# 1. Confirm frozen temporal objects
# ================================================================

temporal_required_621a = [
    "FROZEN_TEMPORAL_CANDIDATE_620I",
    "FROZEN_TEMPORAL_THRESHOLD_620I",
    "FROZEN_TEMPORAL_REFERENCE_620I",
    "FROZEN_TEMPORAL_DEFINITION_620I",
]

print("\n=== Frozen temporal-object availability ===")

temporal_status_621a = {}

for name_621a in temporal_required_621a:

    available_621a = name_621a in globals()

    temporal_status_621a[name_621a] = available_621a

    print(
        f"{name_621a:<36}: "
        f"{available_621a}"
    )


assert all(
    temporal_status_621a.values()
), (
    "One or more Step-6.20I frozen temporal objects "
    "are missing."
)


print(
    "\nFrozen temporal candidate :",
    FROZEN_TEMPORAL_CANDIDATE_620I
)

print(
    "Frozen temporal threshold :",
    FROZEN_TEMPORAL_THRESHOLD_620I
)


# ================================================================
# 2. Locate project root
# ================================================================

cwd_621a = Path.cwd().resolve()

project_root_621a = None

for root_candidate_621a in [
    cwd_621a,
    *cwd_621a.parents,
]:

    if (
        (root_candidate_621a / "03_notebooks").exists()
        and
        (root_candidate_621a / "06_results").exists()
    ):

        project_root_621a = root_candidate_621a
        break


if project_root_621a is None:

    raise FileNotFoundError(
        "Could not locate project root."
    )


step6_result_dir_621a = (
    project_root_621a
    / "06_results"
    / "step6_reliability_signals"
)


print("\n=== Project paths ===")

print(
    "Project root :",
    project_root_621a
)

print(
    "Step-6 results:",
    step6_result_dir_621a
)

print(
    "Directory exists:",
    step6_result_dir_621a.exists()
)


assert step6_result_dir_621a.exists(), (
    "Step-6 reliability-signal result directory "
    "does not exist."
)


# ================================================================
# 3. Create a STABLE snapshot of notebook globals
# ================================================================
#
# IMPORTANT:
# Never iterate directly over globals().items() while also
# creating variables in the same notebook cell.
#
# list(globals().items()) creates a fixed snapshot and prevents:
#
# RuntimeError:
# dictionary changed size during iteration
#
# ================================================================

globals_snapshot_621a = list(
    globals().items()
)

print(
    "\nStable notebook-namespace snapshot created."
)

print(
    "Objects in snapshot:",
    len(globals_snapshot_621a)
)


# ================================================================
# 4. Inspect PCA-related objects currently in memory
# ================================================================

pca_memory_rows_621a = []

for (
    object_name_621a,
    object_value_621a
) in globals_snapshot_621a:

    lower_name_621a = str(
        object_name_621a
    ).lower()

    if "pca" not in lower_name_621a:
        continue


    object_type_621a = type(
        object_value_621a
    ).__name__


    shape_621a = None

    if hasattr(
        object_value_621a,
        "shape"
    ):

        try:

            shape_621a = tuple(
                object_value_621a.shape
            )

        except Exception:

            shape_621a = None


    pca_memory_rows_621a.append(
        {
            "object_name":
                str(object_name_621a),

            "type":
                object_type_621a,

            "shape":
                shape_621a,
        }
    )


pca_memory_df_621a = pd.DataFrame(
    pca_memory_rows_621a,
    columns=[
        "object_name",
        "type",
        "shape",
    ]
)


if not pca_memory_df_621a.empty:

    pca_memory_df_621a = (
        pca_memory_df_621a
        .sort_values(
            "object_name"
        )
        .reset_index(
            drop=True
        )
    )


print(
    "\n=== PCA-related objects currently in memory ==="
)

if pca_memory_df_621a.empty:

    print(
        "No PCA-named objects were found "
        "in the current notebook namespace."
    )

else:

    display(
        pca_memory_df_621a
    )


# ================================================================
# 5. Identify especially relevant PCA score/reference objects
# ================================================================

pca_priority_terms_621a = [
    "score",
    "rmse",
    "reference",
    "threshold",
    "quality",
    "reliability",
    "dev",
    "val",
    "clean",
]


priority_rows_621a = []


if not pca_memory_df_621a.empty:

    for _, row_621a in (
        pca_memory_df_621a.iterrows()
    ):

        object_name_lower_621a = str(
            row_621a["object_name"]
        ).lower()


        matched_terms_621a = [
            term_621a
            for term_621a
            in pca_priority_terms_621a
            if term_621a
            in object_name_lower_621a
        ]


        if matched_terms_621a:

            priority_rows_621a.append(
                {
                    "object_name":
                        row_621a[
                            "object_name"
                        ],

                    "type":
                        row_621a[
                            "type"
                        ],

                    "shape":
                        row_621a[
                            "shape"
                        ],

                    "matched_terms":
                        ", ".join(
                            matched_terms_621a
                        ),
                }
            )


pca_priority_df_621a = pd.DataFrame(
    priority_rows_621a,
    columns=[
        "object_name",
        "type",
        "shape",
        "matched_terms",
    ]
)


print(
    "\n=== Priority PCA candidates for "
    "complementarity analysis ==="
)


if pca_priority_df_621a.empty:

    print(
        "No priority PCA objects identified "
        "from names alone."
    )

else:

    display(
        pca_priority_df_621a
    )


# ================================================================
# 6. Inspect Step-6 result files related to PCA / temporal quality
# ================================================================

result_file_rows_621a = []


for path_621a in sorted(
    step6_result_dir_621a.iterdir()
):

    if not path_621a.is_file():
        continue


    lower_filename_621a = (
        path_621a.name.lower()
    )


    if (
        "pca" in lower_filename_621a
        or
        "temporal" in lower_filename_621a
        or
        "quality" in lower_filename_621a
    ):

        result_file_rows_621a.append(
            {
                "filename":
                    path_621a.name,

                "suffix":
                    path_621a.suffix.lower(),

                "size_bytes":
                    path_621a.stat().st_size,
            }
        )


result_files_df_621a = pd.DataFrame(
    result_file_rows_621a,
    columns=[
        "filename",
        "suffix",
        "size_bytes",
    ]
)


print(
    "\n=== Relevant frozen Step-6 result files ==="
)


if result_files_df_621a.empty:

    print(
        "No PCA/temporal/quality-named "
        "Step-6 files were found."
    )

else:

    display(
        result_files_df_621a
    )


# ================================================================
# 7. Search for PCA-related DataFrames
# ================================================================
#
# Use the SAME stable namespace snapshot.
# Do not call globals().items() directly here.
#
# ================================================================

scenario_table_rows_621a = []


for (
    object_name_621a,
    object_value_621a
) in globals_snapshot_621a:

    if not isinstance(
        object_value_621a,
        pd.DataFrame
    ):
        continue


    lower_name_621a = str(
        object_name_621a
    ).lower()


    columns_lower_621a = [
        str(column_621a).lower()
        for column_621a
        in object_value_621a.columns
    ]


    pca_related_621a = (
        "pca" in lower_name_621a
        or
        any(
            "pca" in column_621a
            for column_621a
            in columns_lower_621a
        )
    )


    if not pca_related_621a:
        continue


    scenario_table_rows_621a.append(
        {
            "object_name":
                str(object_name_621a),

            "rows":
                int(
                    len(
                        object_value_621a
                    )
                ),

            "columns":
                int(
                    object_value_621a.shape[1]
                ),

            "has_sensor":
                "sensor"
                in columns_lower_621a,

            "has_severity":
                "severity"
                in columns_lower_621a,

            "has_corruption":
                "corruption"
                in columns_lower_621a,

            "has_scenario_id":
                "scenario_id"
                in columns_lower_621a,
        }
    )


scenario_tables_df_621a = pd.DataFrame(
    scenario_table_rows_621a,
    columns=[
        "object_name",
        "rows",
        "columns",
        "has_sensor",
        "has_severity",
        "has_corruption",
        "has_scenario_id",
    ]
)


print(
    "\n=== PCA-related DataFrame candidates ==="
)


if scenario_tables_df_621a.empty:

    print(
        "No PCA-related DataFrame candidates "
        "were found in memory."
    )

else:

    scenario_tables_df_621a = (
        scenario_tables_df_621a
        .sort_values(
            [
                "rows",
                "object_name",
            ],
            ascending=[
                False,
                True,
            ]
        )
        .reset_index(
            drop=True
        )
    )

    display(
        scenario_tables_df_621a
    )


# ================================================================
# 8. Compact list of PCA-related CSV files
# ================================================================

pca_csv_files_621a = []

for path_621a in sorted(
    step6_result_dir_621a.glob(
        "*.csv"
    )
):

    if "pca" in path_621a.name.lower():

        pca_csv_files_621a.append(
            path_621a.name
        )


print(
    "\n=== PCA CSV files in Step-6 result directory ==="
)

if len(pca_csv_files_621a) == 0:

    print(
        "No PCA-named CSV files found."
    )

else:

    for (
        file_index_621a,
        filename_621a
    ) in enumerate(
        pca_csv_files_621a,
        start=1
    ):

        print(
            f"{file_index_621a:02d}. "
            f"{filename_621a}"
        )


# ================================================================
# 9. Final pre-check summary
# ================================================================

print(
    "\n=========================================="
)

print(
    "STEP 6.21A PRE-CHECK SUMMARY"
)

print(
    "=========================================="
)


print(
    "Frozen temporal objects ready      :",
    all(
        temporal_status_621a.values()
    )
)

print(
    "Frozen temporal candidate          :",
    FROZEN_TEMPORAL_CANDIDATE_620I
)

print(
    "Frozen temporal threshold          :",
    FROZEN_TEMPORAL_THRESHOLD_620I
)

print(
    "PCA-related objects found          :",
    len(
        pca_memory_df_621a
    )
)

print(
    "Priority PCA objects found         :",
    len(
        pca_priority_df_621a
    )
)

print(
    "Relevant Step-6 result files found :",
    len(
        result_files_df_621a
    )
)

print(
    "PCA-related DataFrames found       :",
    len(
        scenario_tables_df_621a
    )
)

print(
    "PCA-named CSV files found          :",
    len(
        pca_csv_files_621a
    )
)


print(
    "\nStep 6.21A completed successfully."
)

print(
    "Inspection only."
)

print(
    "No corruption was generated."
)

print(
    "No PCA-temporal combination was performed."
)

print(
    "No threshold was changed."
)

print(
    "No frozen artifact was modified."
)

In [ ]:
# ================================================================
# Step 6.21B — Exact PCA / Temporal scenario-alignment inspection
# ================================================================
#
# PURPOSE
# -------
# 1. Inspect the exact harmonized PCA scenario table.
# 2. Locate the scenario-level temporal-candidate table from Step 6.20H-B.
# 3. Isolate the already-frozen temporal candidate: max_sensor.
# 4. Verify whether PCA and temporal scenario records can be aligned
#    exactly before any complementarity/fusion analysis.
#
# IMPORTANT
# ---------
# Inspection/alignment only.
# No PCA + temporal score combination.
# No new threshold selection.
# No corruption generation.
# No frozen artifact modification.
#
# ================================================================

import pandas as pd
import numpy as np


print(
    "\n=== Step 6.21B — "
    "Exact PCA / Temporal scenario-alignment inspection ==="
)


# ================================================================
# 1. Required frozen objects
# ================================================================

required_objects_621b = [
    "cross_corruption_pca_619b",
    "FROZEN_TEMPORAL_CANDIDATE_620I",
    "FROZEN_TEMPORAL_THRESHOLD_620I",
]


print("\n=== Required objects ===")

status_621b = {}

for name_621b in required_objects_621b:

    available_621b = (
        name_621b in globals()
    )

    status_621b[name_621b] = (
        available_621b
    )

    print(
        f"{name_621b:<38}: "
        f"{available_621b}"
    )


assert all(
    status_621b.values()
), (
    "One or more required Step-6 objects "
    "are missing."
)


# ================================================================
# 2. Inspect exact PCA scenario table
# ================================================================

pca_scenarios_621b = (
    cross_corruption_pca_619b.copy()
)


print(
    "\n=== Exact harmonized PCA scenario table ==="
)

print(
    "Shape:",
    pca_scenarios_621b.shape
)

print(
    "Rows:",
    len(pca_scenarios_621b)
)

print(
    "Columns:",
    pca_scenarios_621b.shape[1]
)


print("\nExact PCA columns:")

for index_621b, column_621b in enumerate(
    pca_scenarios_621b.columns,
    start=1
):

    print(
        f"{index_621b:02d}. "
        f"{column_621b}"
    )


print("\nPCA dtypes:")

display(
    pca_scenarios_621b.dtypes
    .rename("dtype")
    .to_frame()
)


# ================================================================
# 3. PCA key integrity
# ================================================================

preferred_keys_621b = [
    "corruption",
    "scenario_id",
    "sensor",
    "severity",
]


available_pca_keys_621b = [
    key_621b
    for key_621b
    in preferred_keys_621b
    if key_621b
    in pca_scenarios_621b.columns
]


print(
    "\n=== PCA key availability ==="
)

print(
    "Available preferred keys:",
    available_pca_keys_621b
)


assert (
    "corruption"
    in pca_scenarios_621b.columns
)

assert (
    "sensor"
    in pca_scenarios_621b.columns
)

assert (
    "severity"
    in pca_scenarios_621b.columns
)

assert (
    "scenario_id"
    in pca_scenarios_621b.columns
)


pca_duplicate_full_keys_621b = int(
    pca_scenarios_621b.duplicated(
        subset=preferred_keys_621b
    ).sum()
)


print(
    "Duplicate PCA full keys:",
    pca_duplicate_full_keys_621b
)


assert (
    pca_duplicate_full_keys_621b == 0
), (
    "Duplicate full scenario keys exist "
    "in PCA table."
)


print(
    "PCA full scenario keys are unique."
)


# ================================================================
# 4. PCA family structure
# ================================================================

pca_family_counts_621b = (
    pca_scenarios_621b
    .groupby(
        "corruption",
        dropna=False
    )
    .size()
    .rename("rows")
    .reset_index()
)


print(
    "\n=== PCA scenario counts by corruption ==="
)

display(
    pca_family_counts_621b
)


# ================================================================
# 5. Stable snapshot of notebook namespace
# ================================================================

globals_snapshot_621b = list(
    globals().items()
)


# ================================================================
# 6. Locate temporal/scenario candidate DataFrames
# ================================================================

temporal_df_candidates_621b = []


for (
    object_name_621b,
    object_value_621b
) in globals_snapshot_621b:

    if not isinstance(
        object_value_621b,
        pd.DataFrame
    ):
        continue


    # Do not treat the PCA table itself
    # as a temporal candidate.
    if object_name_621b == (
        "cross_corruption_pca_619b"
    ):
        continue


    columns_621b = [
        str(column_621b)
        for column_621b
        in object_value_621b.columns
    ]

    columns_lower_621b = [
        column_621b.lower()
        for column_621b
        in columns_621b
    ]


    has_candidate_621b = (
        "candidate"
        in columns_lower_621b
    )

    has_corruption_621b = (
        "corruption"
        in columns_lower_621b
    )

    has_sensor_621b = (
        "sensor"
        in columns_lower_621b
    )

    has_severity_621b = (
        "severity"
        in columns_lower_621b
    )

    has_scenario_id_621b = (
        "scenario_id"
        in columns_lower_621b
    )


    name_lower_621b = (
        str(object_name_621b).lower()
    )


    temporal_name_hint_621b = any(
        hint_621b
        in name_lower_621b
        for hint_621b in [
            "620h",
            "temporal",
            "candidate",
            "scenario",
        ]
    )


    likely_temporal_table_621b = (
        temporal_name_hint_621b
        and
        (
            has_candidate_621b
            or
            (
                has_corruption_621b
                and
                has_sensor_621b
                and
                has_severity_621b
            )
        )
    )


    if not likely_temporal_table_621b:
        continue


    temporal_df_candidates_621b.append(
        {
            "object_name":
                str(object_name_621b),

            "rows":
                int(
                    len(
                        object_value_621b
                    )
                ),

            "columns":
                int(
                    object_value_621b.shape[1]
                ),

            "has_candidate":
                has_candidate_621b,

            "has_corruption":
                has_corruption_621b,

            "has_sensor":
                has_sensor_621b,

            "has_severity":
                has_severity_621b,

            "has_scenario_id":
                has_scenario_id_621b,
        }
    )


temporal_df_candidates_621b = pd.DataFrame(
    temporal_df_candidates_621b
)


print(
    "\n=== Candidate temporal scenario DataFrames ==="
)


if temporal_df_candidates_621b.empty:

    print(
        "No temporal scenario DataFrame "
        "was identified automatically."
    )

else:

    temporal_df_candidates_621b = (
        temporal_df_candidates_621b
        .sort_values(
            [
                "has_candidate",
                "has_scenario_id",
                "rows",
                "object_name",
            ],
            ascending=[
                False,
                False,
                False,
                True,
            ]
        )
        .reset_index(
            drop=True
        )
    )

    display(
        temporal_df_candidates_621b
    )


# ================================================================
# 7. Strong search for the Step-6.20H-B 4-candidate table
# ================================================================

four_candidate_tables_621b = []


for (
    object_name_621b,
    object_value_621b
) in globals_snapshot_621b:

    if not isinstance(
        object_value_621b,
        pd.DataFrame
    ):
        continue


    columns_lower_621b = {
        str(column_621b).lower()
        for column_621b
        in object_value_621b.columns
    }


    if "candidate" not in columns_lower_621b:
        continue


    if "corruption" not in columns_lower_621b:
        continue


    candidate_column_621b = next(
        column_621b
        for column_621b
        in object_value_621b.columns
        if str(column_621b).lower()
        == "candidate"
    )


    unique_candidates_621b = sorted(
        object_value_621b[
            candidate_column_621b
        ]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )


    frozen_candidate_present_621b = (
        str(
            FROZEN_TEMPORAL_CANDIDATE_620I
        )
        in unique_candidates_621b
    )


    four_candidate_tables_621b.append(
        {
            "object_name":
                str(object_name_621b),

            "rows":
                int(
                    len(
                        object_value_621b
                    )
                ),

            "columns":
                int(
                    object_value_621b.shape[1]
                ),

            "candidate_count":
                len(
                    unique_candidates_621b
                ),

            "frozen_candidate_present":
                frozen_candidate_present_621b,

            "candidate_values":
                unique_candidates_621b,
        }
    )


four_candidate_tables_621b = pd.DataFrame(
    four_candidate_tables_621b
)


print(
    "\n=== DataFrames containing temporal candidate labels ==="
)


if four_candidate_tables_621b.empty:

    print(
        "No DataFrame with both "
        "'candidate' and 'corruption' "
        "columns was found."
    )

else:

    four_candidate_tables_621b = (
        four_candidate_tables_621b
        .sort_values(
            [
                "frozen_candidate_present",
                "candidate_count",
                "rows",
            ],
            ascending=[
                False,
                False,
                False,
            ]
        )
        .reset_index(
            drop=True
        )
    )

    display(
        four_candidate_tables_621b
    )


# ================================================================
# 8. Automatically identify strongest temporal table candidate
# ================================================================

temporal_source_name_621b = None
temporal_source_df_621b = None


candidate_objects_621b = []


for (
    object_name_621b,
    object_value_621b
) in globals_snapshot_621b:

    if not isinstance(
        object_value_621b,
        pd.DataFrame
    ):
        continue


    columns_lower_621b = {
        str(column_621b).lower()
        for column_621b
        in object_value_621b.columns
    }


    required_candidate_columns_621b = {
        "candidate",
        "corruption",
        "sensor",
        "severity",
    }


    if not (
        required_candidate_columns_621b
        .issubset(
            columns_lower_621b
        )
    ):
        continue


    candidate_column_621b = next(
        column_621b
        for column_621b
        in object_value_621b.columns
        if str(column_621b).lower()
        == "candidate"
    )


    frozen_rows_621b = (
        object_value_621b[
            object_value_621b[
                candidate_column_621b
            ].astype(str)
            ==
            str(
                FROZEN_TEMPORAL_CANDIDATE_620I
            )
        ]
    )


    candidate_objects_621b.append(
        (
            str(object_name_621b),
            object_value_621b,
            len(frozen_rows_621b),
            "scenario_id"
            in columns_lower_621b,
        )
    )


if candidate_objects_621b:

    # Prefer:
    # 1. exactly 315 frozen-candidate rows,
    # 2. scenario_id present,
    # 3. largest table otherwise.

    candidate_objects_621b = sorted(
        candidate_objects_621b,
        key=lambda item_621b: (
            item_621b[2] == 315,
            item_621b[3],
            len(item_621b[1]),
        ),
        reverse=True,
    )


    (
        temporal_source_name_621b,
        temporal_source_df_621b,
        frozen_row_count_621b,
        temporal_has_scenario_id_621b,
    ) = candidate_objects_621b[0]


print(
    "\n=== Automatically selected temporal-table candidate ==="
)

print(
    "Object:",
    temporal_source_name_621b
)


if temporal_source_df_621b is not None:

    print(
        "Shape:",
        temporal_source_df_621b.shape
    )

    print(
        "Frozen candidate:",
        FROZEN_TEMPORAL_CANDIDATE_620I
    )

    print(
        "Rows for frozen candidate:",
        frozen_row_count_621b
    )

    print(
        "scenario_id available:",
        temporal_has_scenario_id_621b
    )


# ================================================================
# 9. Isolate frozen temporal candidate
# ================================================================

temporal_selected_621b = None


if temporal_source_df_621b is not None:

    candidate_column_621b = next(
        column_621b
        for column_621b
        in temporal_source_df_621b.columns
        if str(column_621b).lower()
        == "candidate"
    )


    temporal_selected_621b = (
        temporal_source_df_621b[
            temporal_source_df_621b[
                candidate_column_621b
            ].astype(str)
            ==
            str(
                FROZEN_TEMPORAL_CANDIDATE_620I
            )
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


    print(
        "\n=== Frozen temporal scenario table ==="
    )

    print(
        "Shape:",
        temporal_selected_621b.shape
    )


    print(
        "\nExact temporal columns:"
    )

    for (
        index_621b,
        column_621b
    ) in enumerate(
        temporal_selected_621b.columns,
        start=1
    ):

        print(
            f"{index_621b:02d}. "
            f"{column_621b}"
        )


# ================================================================
# 10. Determine exact common join keys
# ================================================================

common_join_keys_621b = []


if temporal_selected_621b is not None:

    for key_621b in preferred_keys_621b:

        if (
            key_621b
            in pca_scenarios_621b.columns
            and
            key_621b
            in temporal_selected_621b.columns
        ):

            common_join_keys_621b.append(
                key_621b
            )


print(
    "\n=== Common PCA / temporal join keys ==="
)

print(
    common_join_keys_621b
)


# ================================================================
# 11. Verify temporal-key uniqueness
# ================================================================

temporal_duplicate_keys_621b = None


if (
    temporal_selected_621b
    is not None
    and
    len(common_join_keys_621b) > 0
):

    temporal_duplicate_keys_621b = int(
        temporal_selected_621b.duplicated(
            subset=common_join_keys_621b
        ).sum()
    )


    print(
        "Duplicate temporal common keys:",
        temporal_duplicate_keys_621b
    )


# ================================================================
# 12. Exact key-alignment diagnostic
# ================================================================

alignment_summary_621b = None


if (
    temporal_selected_621b
    is not None
    and
    len(common_join_keys_621b) > 0
):

    pca_key_frame_621b = (
        pca_scenarios_621b[
            common_join_keys_621b
        ]
        .drop_duplicates()
        .copy()
    )


    temporal_key_frame_621b = (
        temporal_selected_621b[
            common_join_keys_621b
        ]
        .drop_duplicates()
        .copy()
    )


    key_alignment_621b = (
        pca_key_frame_621b
        .merge(
            temporal_key_frame_621b,
            on=common_join_keys_621b,
            how="outer",
            indicator=True,
        )
    )


    alignment_summary_621b = (
        key_alignment_621b[
            "_merge"
        ]
        .value_counts()
        .rename_axis(
            "alignment_status"
        )
        .rename(
            "rows"
        )
        .reset_index()
    )


    print(
        "\n=== PCA / temporal key-alignment diagnostic ==="
    )

    display(
        alignment_summary_621b
    )


    matched_rows_621b = int(
        (
            key_alignment_621b[
                "_merge"
            ]
            ==
            "both"
        ).sum()
    )


    pca_only_rows_621b = int(
        (
            key_alignment_621b[
                "_merge"
            ]
            ==
            "left_only"
        ).sum()
    )


    temporal_only_rows_621b = int(
        (
            key_alignment_621b[
                "_merge"
            ]
            ==
            "right_only"
        ).sum()
    )


else:

    matched_rows_621b = None
    pca_only_rows_621b = None
    temporal_only_rows_621b = None


# ================================================================
# 13. Compact integrity summary
# ================================================================

print(
    "\n============================================="
)

print(
    "STEP 6.21B ALIGNMENT SUMMARY"
)

print(
    "============================================="
)

print(
    "PCA scenario rows                  :",
    len(
        pca_scenarios_621b
    )
)

print(
    "PCA full-key duplicates            :",
    pca_duplicate_full_keys_621b
)

print(
    "Temporal table selected            :",
    temporal_source_name_621b
)

print(
    "Frozen temporal candidate          :",
    FROZEN_TEMPORAL_CANDIDATE_620I
)

print(
    "Frozen temporal threshold          :",
    FROZEN_TEMPORAL_THRESHOLD_620I
)

print(
    "Frozen temporal scenario rows      :",
    (
        None
        if temporal_selected_621b is None
        else len(
            temporal_selected_621b
        )
    )
)

print(
    "Common join keys                   :",
    common_join_keys_621b
)

print(
    "Temporal duplicate common keys     :",
    temporal_duplicate_keys_621b
)

print(
    "Matched unique keys                :",
    matched_rows_621b
)

print(
    "PCA-only keys                      :",
    pca_only_rows_621b
)

print(
    "Temporal-only keys                 :",
    temporal_only_rows_621b
)


exact_alignment_621b = (
    temporal_selected_621b
    is not None
    and
    len(
        temporal_selected_621b
    ) == 315
    and
    matched_rows_621b == 315
    and
    pca_only_rows_621b == 0
    and
    temporal_only_rows_621b == 0
    and
    temporal_duplicate_keys_621b == 0
)


print(
    "\nExact one-to-one scenario alignment ready :",
    exact_alignment_621b
)


print(
    "\nStep 6.21B completed."
)

print(
    "Inspection/alignment only."
)

print(
    "No PCA-temporal score combination was performed."
)

print(
    "No threshold was selected or modified."
)

print(
    "No corruption was generated."
)

print(
    "No frozen artifact was modified."
)

In [ ]:
# ================================================================
# Step 6.21B-1 — Diagnose bias/noise scenario-ID mismatch
# ================================================================
#
# PURPOSE
# -------
# The exact scenario-ID merge in Step 6.21B matched 180/315 rows.
#
# 180 = drift(90) + dropout(45) + stuck(45)
# 135 = bias(90) + noise(45)
#
# Earlier PCA harmonisation showed that bias/noise did not originally
# contain scenario_id values, whereas the other families did.
#
# This step diagnoses whether the remaining mismatch is ONLY an
# identifier-convention problem.
#
# INSPECTION ONLY:
# - no score fusion
# - no threshold change
# - no corruption generation
# - no frozen artifact modification
# ================================================================

import pandas as pd
import numpy as np


print(
    "\n=== Step 6.21B-1 — "
    "Diagnose bias/noise scenario-ID mismatch ==="
)


# ================================================================
# 1. Required Step-6.21B objects
# ================================================================

required_621b1 = [
    "pca_scenarios_621b",
    "temporal_selected_621b",
]

print("\n=== Required aligned-source objects ===")

for name_621b1 in required_621b1:
    print(
        f"{name_621b1:<32}: "
        f"{name_621b1 in globals()}"
    )

assert all(
    name_621b1 in globals()
    for name_621b1 in required_621b1
), (
    "Run Step 6.21B first."
)


# ================================================================
# 2. Family-level exact-ID overlap
# ================================================================

families_621b1 = [
    "bias",
    "noise",
    "drift",
    "dropout",
    "stuck",
]

family_overlap_rows_621b1 = []


for family_621b1 in families_621b1:

    pca_family_621b1 = (
        pca_scenarios_621b[
            pca_scenarios_621b["corruption"]
            .astype(str)
            .str.lower()
            ==
            family_621b1
        ]
        .copy()
    )

    temporal_family_621b1 = (
        temporal_selected_621b[
            temporal_selected_621b["corruption"]
            .astype(str)
            .str.lower()
            ==
            family_621b1
        ]
        .copy()
    )

    pca_ids_621b1 = set(
        pca_family_621b1[
            "scenario_id"
        ].astype(str)
    )

    temporal_ids_621b1 = set(
        temporal_family_621b1[
            "scenario_id"
        ].astype(str)
    )

    family_overlap_rows_621b1.append(
        {
            "corruption": family_621b1,
            "pca_rows": len(
                pca_family_621b1
            ),
            "temporal_rows": len(
                temporal_family_621b1
            ),
            "exact_id_overlap": len(
                pca_ids_621b1
                & temporal_ids_621b1
            ),
            "pca_only_ids": len(
                pca_ids_621b1
                - temporal_ids_621b1
            ),
            "temporal_only_ids": len(
                temporal_ids_621b1
                - pca_ids_621b1
            ),
        }
    )


family_overlap_621b1 = pd.DataFrame(
    family_overlap_rows_621b1
)


print(
    "\n=== Exact scenario-ID overlap by family ==="
)

display(
    family_overlap_621b1
)


# ================================================================
# 3. Check natural-key agreement without scenario_id
# ================================================================

natural_keys_621b1 = [
    "corruption",
    "sensor",
    "severity",
]


pca_natural_counts_621b1 = (
    pca_scenarios_621b
    .groupby(
        natural_keys_621b1,
        dropna=False
    )
    .size()
    .rename(
        "pca_rows_per_natural_key"
    )
    .reset_index()
)


temporal_natural_counts_621b1 = (
    temporal_selected_621b
    .groupby(
        natural_keys_621b1,
        dropna=False
    )
    .size()
    .rename(
        "temporal_rows_per_natural_key"
    )
    .reset_index()
)


natural_compare_621b1 = (
    pca_natural_counts_621b1
    .merge(
        temporal_natural_counts_621b1,
        on=natural_keys_621b1,
        how="outer",
    )
)


natural_compare_621b1[
    [
        "pca_rows_per_natural_key",
        "temporal_rows_per_natural_key",
    ]
] = (
    natural_compare_621b1[
        [
            "pca_rows_per_natural_key",
            "temporal_rows_per_natural_key",
        ]
    ]
    .fillna(0)
    .astype(int)
)


natural_compare_621b1[
    "counts_match"
] = (
    natural_compare_621b1[
        "pca_rows_per_natural_key"
    ]
    ==
    natural_compare_621b1[
        "temporal_rows_per_natural_key"
    ]
)


natural_family_summary_621b1 = (
    natural_compare_621b1
    .groupby(
        "corruption",
        dropna=False
    )
    .agg(
        natural_keys=(
            "counts_match",
            "size"
        ),
        matching_natural_keys=(
            "counts_match",
            "sum"
        ),
        all_counts_match=(
            "counts_match",
            "all"
        ),
        max_pca_multiplicity=(
            "pca_rows_per_natural_key",
            "max"
        ),
        max_temporal_multiplicity=(
            "temporal_rows_per_natural_key",
            "max"
        ),
    )
    .reset_index()
)


print(
    "\n=== Natural-key structure "
    "(corruption + sensor + severity) ==="
)

display(
    natural_family_summary_621b1
)


# ================================================================
# 4. Compact inspection of bias/noise temporal metadata
# ================================================================

temporal_metadata_columns_621b1 = [
    column_621b1
    for column_621b1 in [
        "corruption",
        "scenario_id",
        "sensor",
        "severity",
        "direction",
        "alpha",
        "requested_fraction",
    ]
    if column_621b1
    in temporal_selected_621b.columns
]


print(
    "\n=== Temporal BIAS examples ==="
)

display(
    temporal_selected_621b[
        temporal_selected_621b[
            "corruption"
        ].astype(str).str.lower()
        ==
        "bias"
    ][
        temporal_metadata_columns_621b1
    ]
    .head(12)
)


print(
    "\n=== Temporal NOISE examples ==="
)

display(
    temporal_selected_621b[
        temporal_selected_621b[
            "corruption"
        ].astype(str).str.lower()
        ==
        "noise"
    ][
        temporal_metadata_columns_621b1
    ]
    .head(8)
)


# ================================================================
# 5. Compact PCA bias/noise scenario-ID examples
# ================================================================

print(
    "\n=== Harmonized PCA BIAS examples ==="
)

display(
    pca_scenarios_621b[
        pca_scenarios_621b[
            "corruption"
        ].astype(str).str.lower()
        ==
        "bias"
    ][
        [
            "scenario_id",
            "sensor",
            "severity",
        ]
    ]
    .head(12)
)


print(
    "\n=== Harmonized PCA NOISE examples ==="
)

display(
    pca_scenarios_621b[
        pca_scenarios_621b[
            "corruption"
        ].astype(str).str.lower()
        ==
        "noise"
    ][
        [
            "scenario_id",
            "sensor",
            "severity",
        ]
    ]
    .head(8)
)


# ================================================================
# 6. Inspect richer in-memory PCA source tables
# ================================================================

source_candidates_621b1 = [
    "bias_pca_response_df",
    "noise_response_df",
    "noise_619a2",
    "drift_pca_response_df_616i",
    "dropout_pca_response_df_617g",
    "stuck_pca_response_618g",
]


source_rows_621b1 = []


for object_name_621b1 in source_candidates_621b1:

    if object_name_621b1 not in globals():
        continue

    object_621b1 = globals()[
        object_name_621b1
    ]

    if not isinstance(
        object_621b1,
        pd.DataFrame
    ):
        continue

    source_rows_621b1.append(
        {
            "object_name":
                object_name_621b1,

            "rows":
                len(
                    object_621b1
                ),

            "columns":
                object_621b1.shape[1],

            "column_names":
                list(
                    object_621b1.columns
                ),
        }
    )


source_inventory_621b1 = pd.DataFrame(
    source_rows_621b1
)


print(
    "\n=== Rich PCA source-table inventory ==="
)

display(
    source_inventory_621b1
)


# ================================================================
# 7. Exact column inspection for likely bias/noise sources
# ================================================================

likely_bias_sources_621b1 = [
    name_621b1
    for name_621b1 in [
        "bias_pca_response_df",
    ]
    if name_621b1 in globals()
]


likely_noise_sources_621b1 = [
    name_621b1
    for name_621b1 in [
        "noise_response_df",
        "noise_619a2",
    ]
    if name_621b1 in globals()
]


print(
    "\n=== Bias PCA source columns ==="
)

for name_621b1 in likely_bias_sources_621b1:

    df_621b1 = globals()[
        name_621b1
    ]

    print(
        f"\n{name_621b1} "
        f"shape={df_621b1.shape}"
    )

    print(
        list(
            df_621b1.columns
        )
    )


print(
    "\n=== Noise PCA source columns ==="
)

for name_621b1 in likely_noise_sources_621b1:

    df_621b1 = globals()[
        name_621b1
    ]

    print(
        f"\n{name_621b1} "
        f"shape={df_621b1.shape}"
    )

    print(
        list(
            df_621b1.columns
        )
    )


# ================================================================
# 8. Show only metadata-like columns from rich source tables
# ================================================================

metadata_tokens_621b1 = [
    "sensor",
    "severity",
    "direction",
    "alpha",
    "scenario",
    "fraction",
    "seed",
]


def metadata_columns_621b1(
    df_621b1
):

    return [
        column_621b1
        for column_621b1
        in df_621b1.columns
        if any(
            token_621b1
            in str(
                column_621b1
            ).lower()
            for token_621b1
            in metadata_tokens_621b1
        )
    ]


for (
    family_name_621b1,
    source_names_621b1
) in [
    (
        "BIAS",
        likely_bias_sources_621b1
    ),
    (
        "NOISE",
        likely_noise_sources_621b1
    ),
]:

    for name_621b1 in source_names_621b1:

        df_621b1 = globals()[
            name_621b1
        ]

        cols_621b1 = (
            metadata_columns_621b1(
                df_621b1
            )
        )

        print(
            f"\n=== {family_name_621b1} "
            f"metadata from "
            f"{name_621b1} ==="
        )

        print(
            "Metadata-like columns:",
            cols_621b1
        )

        if cols_621b1:

            display(
                df_621b1[
                    cols_621b1
                ]
                .head(12)
            )


# ================================================================
# 9. Family-specific mismatch conclusion
# ================================================================

mismatch_family_rows_621b1 = (
    family_overlap_621b1[
        family_overlap_621b1[
            "exact_id_overlap"
        ]
        <
        family_overlap_621b1[
            "pca_rows"
        ]
    ]
    .copy()
)


print(
    "\n=== Families requiring ID reconciliation ==="
)

display(
    mismatch_family_rows_621b1
)


problem_families_621b1 = (
    mismatch_family_rows_621b1[
        "corruption"
    ]
    .tolist()
)


print(
    "\n============================================"
)

print(
    "STEP 6.21B-1 DIAGNOSTIC SUMMARY"
)

print(
    "============================================"
)

print(
    "PCA rows                  :",
    len(
        pca_scenarios_621b
    )
)

print(
    "Temporal rows             :",
    len(
        temporal_selected_621b
    )
)

print(
    "Families needing reconciliation:",
    problem_families_621b1
)

print(
    "No score fusion performed : True"
)

print(
    "No threshold modified     : True"
)

print(
    "No artifact modified      : True"
)


print(
    "\nStep 6.21B-1 completed."
)

In [ ]:
# ================================================================
# Step 6.21B-1C — COMPACT alignment diagnostic
# ================================================================
# Purpose:
# Print only the information needed to reconcile PCA and temporal
# scenario identifiers. No files or frozen artifacts are modified.
# ================================================================

import pandas as pd
import numpy as np


print("\n=== STEP 6.21B-1 COMPACT DIAGNOSTIC ===")


# ------------------------------------------------
# 1. Basic availability
# ------------------------------------------------

required = [
    "pca_scenarios_621b",
    "temporal_selected_621b",
]

for name in required:
    print(f"{name}: {name in globals()}")

assert all(name in globals() for name in required), (
    "Run Step 6.21B first."
)


# ------------------------------------------------
# 2. Exact scenario-ID overlap by corruption family
# ------------------------------------------------

families = [
    "bias",
    "noise",
    "drift",
    "dropout",
    "stuck",
]

print("\n--- Exact ID overlap ---")

for family in families:

    pca_part = (
        pca_scenarios_621b[
            pca_scenarios_621b["corruption"]
            .astype(str)
            .str.lower()
            == family
        ]
    )

    temp_part = (
        temporal_selected_621b[
            temporal_selected_621b["corruption"]
            .astype(str)
            .str.lower()
            == family
        ]
    )

    pca_ids = set(
        pca_part["scenario_id"].astype(str)
    )

    temp_ids = set(
        temp_part["scenario_id"].astype(str)
    )

    overlap = len(
        pca_ids & temp_ids
    )

    print(
        f"{family:<8} "
        f"PCA={len(pca_part):>3} | "
        f"TEMP={len(temp_part):>3} | "
        f"MATCH={overlap:>3}"
    )


# ------------------------------------------------
# 3. Natural-key multiplicity
# ------------------------------------------------

print(
    "\n--- Natural key multiplicity "
    "(corruption + sensor + severity) ---"
)

for family in families:

    pca_part = (
        pca_scenarios_621b[
            pca_scenarios_621b["corruption"]
            .astype(str)
            .str.lower()
            == family
        ]
    )

    temp_part = (
        temporal_selected_621b[
            temporal_selected_621b["corruption"]
            .astype(str)
            .str.lower()
            == family
        ]
    )

    pca_counts = (
        pca_part
        .groupby(
            ["sensor", "severity"],
            dropna=False
        )
        .size()
    )

    temp_counts = (
        temp_part
        .groupby(
            ["sensor", "severity"],
            dropna=False
        )
        .size()
    )

    all_keys = (
        pca_counts.index
        .union(temp_counts.index)
    )

    pca_aligned = (
        pca_counts
        .reindex(
            all_keys,
            fill_value=0
        )
    )

    temp_aligned = (
        temp_counts
        .reindex(
            all_keys,
            fill_value=0
        )
    )

    counts_match = bool(
        np.array_equal(
            pca_aligned.values,
            temp_aligned.values
        )
    )

    print(
        f"{family:<8} "
        f"counts_match={counts_match} | "
        f"PCA_mult={int(pca_aligned.max())} | "
        f"TEMP_mult={int(temp_aligned.max())}"
    )


# ------------------------------------------------
# 4. Temporal metadata available for BIAS and NOISE
# ------------------------------------------------

metadata_cols = [
    "direction",
    "alpha",
    "requested_fraction",
    "seed",
]

print("\n--- Temporal metadata ---")

for family in ["bias", "noise"]:

    part = (
        temporal_selected_621b[
            temporal_selected_621b["corruption"]
            .astype(str)
            .str.lower()
            == family
        ]
    )

    print(f"\n{family.upper()}:")

    for col in metadata_cols:

        if col in part.columns:

            values = (
                part[col]
                .dropna()
                .unique()
                .tolist()
            )

            print(
                f"  {col}: {values}"
            )


# ------------------------------------------------
# 5. PCA source metadata availability
# ------------------------------------------------

source_names = [
    "bias_pca_response_df",
    "noise_response_df",
    "noise_619a2",
]

metadata_tokens = [
    "sensor",
    "severity",
    "direction",
    "alpha",
    "fraction",
    "seed",
    "scenario",
]


print("\n--- PCA source metadata availability ---")

for source_name in source_names:

    if source_name not in globals():
        print(
            f"{source_name}: NOT AVAILABLE"
        )
        continue

    df = globals()[source_name]

    if not isinstance(df, pd.DataFrame):
        print(
            f"{source_name}: "
            f"not a DataFrame"
        )
        continue

    candidate_cols = [
        col
        for col in df.columns
        if any(
            token in str(col).lower()
            for token in metadata_tokens
        )
    ]

    print(
        f"\n{source_name}:"
    )

    print(
        "  shape:",
        df.shape
    )

    print(
        "  metadata columns:",
        candidate_cols
    )

    for col in candidate_cols:

        values = (
            df[col]
            .dropna()
            .unique()
            .tolist()
        )

        if len(values) <= 10:
            print(
                f"  {col}: {values}"
            )
        else:
            print(
                f"  {col}: "
                f"{len(values)} unique values"
            )


# ------------------------------------------------
# 6. Final compact conclusion
# ------------------------------------------------

print("\n=== END COMPACT DIAGNOSTIC ===")
print("Nothing was modified.")

In [ ]:
# ================================================================
# Step 6.21B-2 — Build verified PCA / Temporal scenario alignment
# ================================================================
# Purpose:
# Construct a one-to-one mapping between the frozen PCA scenarios
# and the selected frozen temporal scenarios.
#
# IMPORTANT:
# - Existing scenario IDs are NOT changed.
# - No frozen artifact is modified.
# - Nothing is written to disk.
# - The result is created IN MEMORY only.
# ================================================================

from pathlib import Path
import numpy as np
import pandas as pd


print(
    "\n=== Step 6.21B-2 — Verified PCA / Temporal "
    "scenario alignment ==="
)


# ================================================================
# 1. Required objects
# ================================================================

required_objects_621b2 = [
    "pca_scenarios_621b",
    "temporal_selected_621b",
]

print("\n--- Required objects ---")

for name_621b2 in required_objects_621b2:
    print(
        f"{name_621b2:<30}:",
        name_621b2 in globals()
    )

assert all(
    name_621b2 in globals()
    for name_621b2 in required_objects_621b2
), (
    "Required Step-6.21B objects are missing."
)


# ================================================================
# 2. Independent working copies
# ================================================================

pca_work_621b2 = (
    pca_scenarios_621b
    .copy(deep=True)
)

temporal_work_621b2 = (
    temporal_selected_621b
    .copy(deep=True)
)


# ------------------------------------------------
# Normalize key text only in working copies
# ------------------------------------------------

for df_621b2 in [
    pca_work_621b2,
    temporal_work_621b2,
]:

    df_621b2["corruption_norm"] = (
        df_621b2["corruption"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    df_621b2["sensor_norm"] = (
        df_621b2["sensor"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    df_621b2["severity_norm"] = (
        df_621b2["severity"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    df_621b2["scenario_id_text"] = (
        df_621b2["scenario_id"]
        .astype(str)
    )


assert len(pca_work_621b2) == 315
assert len(temporal_work_621b2) == 315


# ================================================================
# 3. Recover exact legacy BIAS source table
# ================================================================

if (
    "bias_pca_response_df" in globals()
    and isinstance(
        bias_pca_response_df,
        pd.DataFrame,
    )
):
    bias_source_621b2 = (
        bias_pca_response_df
        .copy(deep=True)
    )

    bias_source_origin_621b2 = (
        "in-memory bias_pca_response_df"
    )

else:

    project_root_621b2 = Path(
        r"C:\RUL_Reliability_Research"
    )

    bias_path_621b2 = (
        project_root_621b2
        / "06_results"
        / "step6_reliability_signals"
        / "fd001_pca_bias_response_90_scenarios.csv"
    )

    assert bias_path_621b2.exists(), (
        "Frozen bias PCA scenario file was not found."
    )

    bias_source_621b2 = pd.read_csv(
        bias_path_621b2
    )

    bias_source_origin_621b2 = str(
        bias_path_621b2
    )


print(
    "\nBias metadata source:",
    bias_source_origin_621b2
)

print(
    "Bias source rows:",
    len(bias_source_621b2)
)

assert len(bias_source_621b2) == 90


# ================================================================
# 4. Normalize legacy bias-column names
# ================================================================

bias_source_621b2 = (
    bias_source_621b2
    .rename(
        columns={
            "mean_re_change_pct":
                "mean_RE_change_pct",

            "median_re_change_pct":
                "median_RE_change_pct",
        }
    )
)


bias_source_621b2["sensor_norm"] = (
    bias_source_621b2["sensor"]
    .astype(str)
    .str.strip()
    .str.lower()
)

bias_source_621b2["severity_norm"] = (
    bias_source_621b2["severity"]
    .astype(str)
    .str.strip()
    .str.lower()
)


required_bias_metadata_621b2 = [
    "alpha",
    "direction",
]

assert all(
    col_621b2 in bias_source_621b2.columns
    for col_621b2
    in required_bias_metadata_621b2
), (
    "Bias source does not contain the required "
    "alpha/direction metadata."
)


# ================================================================
# 5. Recover alpha + direction for each harmonized PCA bias row
# ================================================================

bias_metric_candidates_621b2 = [
    "clean_mean_re",
    "corrupted_mean_re",
    "mean_RE_change_pct",
    "median_RE_change_pct",
    "fraction_increased_pct",
    "clean_above_dev95_pct",
    "corrupted_above_dev95_pct",
    "above_dev95_delta_pp",
]


bias_match_metrics_621b2 = [
    col_621b2
    for col_621b2
    in bias_metric_candidates_621b2

    if (
        col_621b2 in pca_work_621b2.columns
        and
        col_621b2 in bias_source_621b2.columns
    )
]


print(
    "\nBias numerical matching metrics:",
    len(bias_match_metrics_621b2)
)

assert len(bias_match_metrics_621b2) >= 4, (
    "Too few shared numerical metrics are available "
    "for safe bias-row reconstruction."
)


pca_bias_621b2 = (
    pca_work_621b2[
        pca_work_621b2[
            "corruption_norm"
        ] == "bias"
    ]
    .copy()
)


recovered_bias_rows_621b2 = []


for pca_index_621b2, pca_row_621b2 \
        in pca_bias_621b2.iterrows():

    candidates_621b2 = (
        bias_source_621b2[
            (
                bias_source_621b2[
                    "sensor_norm"
                ]
                ==
                pca_row_621b2[
                    "sensor_norm"
                ]
            )
            &
            (
                bias_source_621b2[
                    "severity_norm"
                ]
                ==
                pca_row_621b2[
                    "severity_norm"
                ]
            )
        ]
    )


    matched_source_indices_621b2 = []


    for source_index_621b2, source_row_621b2 \
            in candidates_621b2.iterrows():

        all_metrics_match_621b2 = True


        for metric_621b2 \
                in bias_match_metrics_621b2:

            pca_value_621b2 = (
                pca_row_621b2[
                    metric_621b2
                ]
            )

            source_value_621b2 = (
                source_row_621b2[
                    metric_621b2
                ]
            )


            try:

                metric_matches_621b2 = bool(
                    np.isclose(
                        float(pca_value_621b2),
                        float(source_value_621b2),
                        rtol=1e-7,
                        atol=1e-9,
                        equal_nan=True,
                    )
                )

            except Exception:

                metric_matches_621b2 = (
                    pca_value_621b2
                    ==
                    source_value_621b2
                )


            if not metric_matches_621b2:

                all_metrics_match_621b2 = False
                break


        if all_metrics_match_621b2:

            matched_source_indices_621b2.append(
                source_index_621b2
            )


    assert (
        len(
            matched_source_indices_621b2
        )
        == 1
    ), (
        "Bias PCA row could not be mapped uniquely "
        "to the frozen legacy bias source. "
        f"PCA index={pca_index_621b2}, "
        f"matches={len(matched_source_indices_621b2)}"
    )


    source_match_621b2 = (
        bias_source_621b2.loc[
            matched_source_indices_621b2[0]
        ]
    )


    recovered_bias_rows_621b2.append(
        {
            "pca_original_index":
                pca_index_621b2,

            "pca_scenario_id":
                str(
                    pca_row_621b2[
                        "scenario_id"
                    ]
                ),

            "corruption":
                "bias",

            "sensor":
                pca_row_621b2[
                    "sensor_norm"
                ],

            "severity":
                pca_row_621b2[
                    "severity_norm"
                ],

            "alpha":
                float(
                    source_match_621b2[
                        "alpha"
                    ]
                ),

            "direction":
                float(
                    source_match_621b2[
                        "direction"
                    ]
                ),
        }
    )


pca_bias_metadata_621b2 = pd.DataFrame(
    recovered_bias_rows_621b2
)


assert len(pca_bias_metadata_621b2) == 90


# ================================================================
# 6. Align BIAS
# ================================================================

temporal_bias_621b2 = (
    temporal_work_621b2[
        temporal_work_621b2[
            "corruption_norm"
        ] == "bias"
    ]
    .copy()
)


assert (
    "alpha"
    in temporal_bias_621b2.columns
)

assert (
    "direction"
    in temporal_bias_621b2.columns
)


temporal_bias_keys_621b2 = pd.DataFrame(
    {
        "temporal_original_index":
            temporal_bias_621b2.index,

        "temporal_scenario_id":
            temporal_bias_621b2[
                "scenario_id"
            ].astype(str).values,

        "sensor":
            temporal_bias_621b2[
                "sensor_norm"
            ].values,

        "severity":
            temporal_bias_621b2[
                "severity_norm"
            ].values,

        "alpha":
            pd.to_numeric(
                temporal_bias_621b2[
                    "alpha"
                ],
                errors="raise",
            ).values,

        "direction":
            pd.to_numeric(
                temporal_bias_621b2[
                    "direction"
                ],
                errors="raise",
            ).values,
    }
)


# Round only for safe merge-key comparison
pca_bias_metadata_621b2[
    "alpha_key"
] = (
    pca_bias_metadata_621b2[
        "alpha"
    ]
    .round(12)
)

temporal_bias_keys_621b2[
    "alpha_key"
] = (
    temporal_bias_keys_621b2[
        "alpha"
    ]
    .round(12)
)


pca_bias_metadata_621b2[
    "direction_key"
] = (
    pca_bias_metadata_621b2[
        "direction"
    ]
    .round(12)
)

temporal_bias_keys_621b2[
    "direction_key"
] = (
    temporal_bias_keys_621b2[
        "direction"
    ]
    .round(12)
)


bias_alignment_621b2 = (
    pca_bias_metadata_621b2
    .merge(
        temporal_bias_keys_621b2[
            [
                "temporal_original_index",
                "temporal_scenario_id",
                "sensor",
                "severity",
                "alpha_key",
                "direction_key",
            ]
        ],
        on=[
            "sensor",
            "severity",
            "alpha_key",
            "direction_key",
        ],
        how="inner",
        validate="one_to_one",
    )
)


bias_alignment_621b2[
    "alignment_basis"
] = (
    "sensor+severity+alpha+direction"
)


assert len(bias_alignment_621b2) == 90


# ================================================================
# 7. Align NOISE
# ================================================================

pca_noise_621b2 = (
    pca_work_621b2[
        pca_work_621b2[
            "corruption_norm"
        ] == "noise"
    ]
    .copy()
)


temporal_noise_621b2 = (
    temporal_work_621b2[
        temporal_work_621b2[
            "corruption_norm"
        ] == "noise"
    ]
    .copy()
)


assert not pca_noise_621b2.duplicated(
    subset=[
        "sensor_norm",
        "severity_norm",
    ]
).any()


assert not temporal_noise_621b2.duplicated(
    subset=[
        "sensor_norm",
        "severity_norm",
    ]
).any()


noise_alignment_621b2 = (
    pca_noise_621b2[
        [
            "scenario_id",
            "sensor_norm",
            "severity_norm",
        ]
    ]
    .rename(
        columns={
            "scenario_id":
                "pca_scenario_id",

            "sensor_norm":
                "sensor",

            "severity_norm":
                "severity",
        }
    )
    .merge(
        temporal_noise_621b2[
            [
                "scenario_id",
                "sensor_norm",
                "severity_norm",
            ]
        ]
        .rename(
            columns={
                "scenario_id":
                    "temporal_scenario_id",

                "sensor_norm":
                    "sensor",

                "severity_norm":
                    "severity",
            }
        ),
        on=[
            "sensor",
            "severity",
        ],
        how="inner",
        validate="one_to_one",
    )
)


noise_alignment_621b2[
    "pca_scenario_id"
] = (
    noise_alignment_621b2[
        "pca_scenario_id"
    ]
    .astype(str)
)


noise_alignment_621b2[
    "temporal_scenario_id"
] = (
    noise_alignment_621b2[
        "temporal_scenario_id"
    ]
    .astype(str)
)


noise_alignment_621b2[
    "corruption"
] = "noise"


noise_alignment_621b2[
    "alignment_basis"
] = (
    "sensor+severity"
)


assert len(noise_alignment_621b2) == 45


# ================================================================
# 8. Align DRIFT / DROPOUT / STUCK using existing scenario IDs
# ================================================================

dynamic_alignment_parts_621b2 = []


for family_621b2 in [
    "drift",
    "dropout",
    "stuck",
]:

    pca_family_621b2 = (
        pca_work_621b2[
            pca_work_621b2[
                "corruption_norm"
            ] == family_621b2
        ]
        .copy()
    )


    temporal_family_621b2 = (
        temporal_work_621b2[
            temporal_work_621b2[
                "corruption_norm"
            ] == family_621b2
        ]
        .copy()
    )


    family_alignment_621b2 = (
        pca_family_621b2[
            [
                "scenario_id_text",
                "sensor_norm",
                "severity_norm",
            ]
        ]
        .rename(
            columns={
                "scenario_id_text":
                    "scenario_id_key",

                "sensor_norm":
                    "sensor",

                "severity_norm":
                    "severity",
            }
        )
        .merge(
            temporal_family_621b2[
                [
                    "scenario_id_text",
                    "sensor_norm",
                    "severity_norm",
                ]
            ]
            .rename(
                columns={
                    "scenario_id_text":
                        "scenario_id_key",

                    "sensor_norm":
                        "sensor",

                    "severity_norm":
                        "severity",
                }
            ),
            on=[
                "scenario_id_key",
                "sensor",
                "severity",
            ],
            how="inner",
            validate="one_to_one",
        )
    )


    family_alignment_621b2[
        "corruption"
    ] = family_621b2


    family_alignment_621b2[
        "pca_scenario_id"
    ] = (
        family_alignment_621b2[
            "scenario_id_key"
        ]
    )


    family_alignment_621b2[
        "temporal_scenario_id"
    ] = (
        family_alignment_621b2[
            "scenario_id_key"
        ]
    )


    family_alignment_621b2[
        "alignment_basis"
    ] = (
        "existing scenario_id"
    )


    dynamic_alignment_parts_621b2.append(
        family_alignment_621b2[
            [
                "corruption",
                "sensor",
                "severity",
                "pca_scenario_id",
                "temporal_scenario_id",
                "alignment_basis",
            ]
        ]
    )


dynamic_alignment_621b2 = pd.concat(
    dynamic_alignment_parts_621b2,
    ignore_index=True,
)


assert len(dynamic_alignment_621b2) == 180


# ================================================================
# 9. Standardize the three alignment blocks
# ================================================================

bias_final_621b2 = (
    bias_alignment_621b2[
        [
            "corruption",
            "sensor",
            "severity",
            "pca_scenario_id",
            "temporal_scenario_id",
            "alignment_basis",
            "alpha",
            "direction",
        ]
    ]
    .copy()
)


noise_final_621b2 = (
    noise_alignment_621b2[
        [
            "corruption",
            "sensor",
            "severity",
            "pca_scenario_id",
            "temporal_scenario_id",
            "alignment_basis",
        ]
    ]
    .copy()
)


noise_final_621b2[
    "alpha"
] = np.nan

noise_final_621b2[
    "direction"
] = np.nan


dynamic_final_621b2 = (
    dynamic_alignment_621b2
    .copy()
)

dynamic_final_621b2[
    "alpha"
] = np.nan

dynamic_final_621b2[
    "direction"
] = np.nan


# ================================================================
# 10. Final verified alignment table
# ================================================================

scenario_alignment_621b2 = (
    pd.concat(
        [
            bias_final_621b2,
            noise_final_621b2,
            dynamic_final_621b2,
        ],
        ignore_index=True,
    )
)


family_order_621b2 = {
    "bias": 0,
    "noise": 1,
    "drift": 2,
    "dropout": 3,
    "stuck": 4,
}


scenario_alignment_621b2[
    "_family_order"
] = (
    scenario_alignment_621b2[
        "corruption"
    ]
    .map(
        family_order_621b2
    )
)


scenario_alignment_621b2 = (
    scenario_alignment_621b2
    .sort_values(
        [
            "_family_order",
            "sensor",
            "severity",
            "pca_scenario_id",
        ]
    )
    .drop(
        columns="_family_order"
    )
    .reset_index(drop=True)
)


# ================================================================
# 11. Integrity checks
# ================================================================

assert len(
    scenario_alignment_621b2
) == 315


pca_duplicate_count_621b2 = int(
    scenario_alignment_621b2[
        [
            "corruption",
            "pca_scenario_id",
        ]
    ]
    .duplicated()
    .sum()
)


temporal_duplicate_count_621b2 = int(
    scenario_alignment_621b2[
        [
            "corruption",
            "temporal_scenario_id",
        ]
    ]
    .duplicated()
    .sum()
)


assert pca_duplicate_count_621b2 == 0

assert temporal_duplicate_count_621b2 == 0


alignment_counts_621b2 = (
    scenario_alignment_621b2
    .groupby(
        "corruption",
        sort=False,
    )
    .size()
    .reindex(
        [
            "bias",
            "noise",
            "drift",
            "dropout",
            "stuck",
        ]
    )
)


expected_counts_621b2 = pd.Series(
    {
        "bias": 90,
        "noise": 45,
        "drift": 90,
        "dropout": 45,
        "stuck": 45,
    }
)


assert (
    alignment_counts_621b2
    .astype(int)
    .equals(
        expected_counts_621b2
        .astype(int)
    )
)


# ================================================================
# 12. Compact output only
# ================================================================

print(
    "\n--- Verified alignment counts ---"
)

for family_621b2 in [
    "bias",
    "noise",
    "drift",
    "dropout",
    "stuck",
]:

    print(
        f"{family_621b2:<8}: "
        f"{int(alignment_counts_621b2.loc[family_621b2])}"
    )


print(
    "\n--- Alignment method ---"
)

print(
    "bias    : sensor + severity + alpha + direction"
)

print(
    "noise   : sensor + severity"
)

print(
    "drift   : existing scenario_id"
)

print(
    "dropout : existing scenario_id"
)

print(
    "stuck   : existing scenario_id"
)


print(
    "\n--- Integrity ---"
)

print(
    "Aligned scenarios        :",
    len(scenario_alignment_621b2)
)

print(
    "Expected scenarios       : 315"
)

print(
    "Duplicate PCA mappings   :",
    pca_duplicate_count_621b2
)

print(
    "Duplicate TEMP mappings  :",
    temporal_duplicate_count_621b2
)

print(
    "One-to-one alignment     :",
    (
        len(scenario_alignment_621b2) == 315
        and
        pca_duplicate_count_621b2 == 0
        and
        temporal_duplicate_count_621b2 == 0
    )
)


print(
    "\nStep 6.21B-2 completed successfully."
)

print(
    "Verified PCA/temporal scenario mapping was "
    "created IN MEMORY only."
)

print(
    "Original PCA scenario IDs were not modified."
)

print(
    "Original temporal scenario IDs were not modified."
)

print(
    "No frozen file or artifact was modified."
)

In [ ]:
# ================================================================
# Step 6.21C — Scenario-level PCA / Temporal complementarity
# ================================================================
# Purpose:
# Compare the already-frozen PCA response and the already-frozen
# temporal-quality response on the exact same 315 scenarios.
#
# IMPORTANT:
# - Uses the verified Step-6.21B-2 alignment.
# - No threshold is changed.
# - No new corruption is generated.
# - No PCA/temporal combination is created.
# - Nothing is written to disk.
# ================================================================

import numpy as np
import pandas as pd


print(
    "\n=== Step 6.21C — PCA / Temporal "
    "scenario-level complementarity ==="
)


# ================================================================
# 1. Required objects
# ================================================================

required_621c = [
    "scenario_alignment_621b2",
    "pca_scenarios_621b",
    "temporal_selected_621b",
]

print("\n--- Required objects ---")

for object_name_621c in required_621c:
    print(
        f"{object_name_621c:<30}:",
        object_name_621c in globals(),
    )

assert all(
    object_name_621c in globals()
    for object_name_621c in required_621c
), (
    "Required Step-6.21B objects are missing."
)


# ================================================================
# 2. Independent working copies
# ================================================================

alignment_621c = (
    scenario_alignment_621b2
    .copy(deep=True)
)

pca_621c = (
    pca_scenarios_621b
    .copy(deep=True)
)

temporal_621c = (
    temporal_selected_621b
    .copy(deep=True)
)


# ================================================================
# 3. Normalize only temporary matching keys
# ================================================================

alignment_621c["corruption_key"] = (
    alignment_621c["corruption"]
    .astype(str)
    .str.strip()
    .str.lower()
)

alignment_621c["pca_id_key"] = (
    alignment_621c["pca_scenario_id"]
    .astype(str)
)

alignment_621c["temporal_id_key"] = (
    alignment_621c["temporal_scenario_id"]
    .astype(str)
)


pca_621c["corruption_key"] = (
    pca_621c["corruption"]
    .astype(str)
    .str.strip()
    .str.lower()
)

pca_621c["scenario_id_key"] = (
    pca_621c["scenario_id"]
    .astype(str)
)


temporal_621c["corruption_key"] = (
    temporal_621c["corruption"]
    .astype(str)
    .str.strip()
    .str.lower()
)

temporal_621c["scenario_id_key"] = (
    temporal_621c["scenario_id"]
    .astype(str)
)


# ================================================================
# 4. Confirm required scientific response variables
# ================================================================

PCA_PRIMARY_621C = "mean_RE_change_pct"
TEMP_PRIMARY_621C = "mean_score_delta"


assert PCA_PRIMARY_621C in pca_621c.columns, (
    f"Required PCA response column "
    f"'{PCA_PRIMARY_621C}' is missing."
)

assert TEMP_PRIMARY_621C in temporal_621c.columns, (
    f"Required temporal response column "
    f"'{TEMP_PRIMARY_621C}' is missing."
)


# Optional percentage-point event-response variables
PCA_EVENT_621C = (
    "above_dev95_delta_pp"
    if "above_dev95_delta_pp" in pca_621c.columns
    else None
)


temporal_event_candidates_621c = [
    "above_threshold_delta_pp",
    "event_delta_pp",
    "mean_above_threshold_delta_pp",
]


TEMP_EVENT_621C = next(
    (
        col_621c
        for col_621c in temporal_event_candidates_621c
        if col_621c in temporal_621c.columns
    ),
    None,
)


print("\n--- Response variables ---")
print("PCA continuous response     :", PCA_PRIMARY_621C)
print("Temporal continuous response:", TEMP_PRIMARY_621C)
print("PCA event response          :", PCA_EVENT_621C)
print("Temporal event response     :", TEMP_EVENT_621C)


# ================================================================
# 5. Extract only required PCA payload
# ================================================================

pca_payload_columns_621c = [
    "corruption_key",
    "scenario_id_key",
    PCA_PRIMARY_621C,
]

if PCA_EVENT_621C is not None:
    pca_payload_columns_621c.append(
        PCA_EVENT_621C
    )


pca_payload_621c = (
    pca_621c[
        pca_payload_columns_621c
    ]
    .copy()
    .rename(
        columns={
            "scenario_id_key":
                "pca_id_key",

            PCA_PRIMARY_621C:
                "pca_response",
        }
    )
)


if PCA_EVENT_621C is not None:
    pca_payload_621c = (
        pca_payload_621c.rename(
            columns={
                PCA_EVENT_621C:
                    "pca_event_delta_pp"
            }
        )
    )


# ================================================================
# 6. Extract only required temporal payload
# ================================================================

temporal_payload_columns_621c = [
    "corruption_key",
    "scenario_id_key",
    TEMP_PRIMARY_621C,
]

if TEMP_EVENT_621C is not None:
    temporal_payload_columns_621c.append(
        TEMP_EVENT_621C
    )


temporal_payload_621c = (
    temporal_621c[
        temporal_payload_columns_621c
    ]
    .copy()
    .rename(
        columns={
            "scenario_id_key":
                "temporal_id_key",

            TEMP_PRIMARY_621C:
                "temporal_response",
        }
    )
)


if TEMP_EVENT_621C is not None:
    temporal_payload_621c = (
        temporal_payload_621c.rename(
            columns={
                TEMP_EVENT_621C:
                    "temporal_event_delta_pp"
            }
        )
    )


# ================================================================
# 7. Merge PCA response through verified mapping
# ================================================================

comparison_621c = (
    alignment_621c
    .merge(
        pca_payload_621c,
        on=[
            "corruption_key",
            "pca_id_key",
        ],
        how="left",
        validate="one_to_one",
    )
)


comparison_621c = (
    comparison_621c
    .merge(
        temporal_payload_621c,
        on=[
            "corruption_key",
            "temporal_id_key",
        ],
        how="left",
        validate="one_to_one",
    )
)


assert len(comparison_621c) == 315


assert comparison_621c[
    "pca_response"
].notna().all()


assert comparison_621c[
    "temporal_response"
].notna().all()


comparison_621c[
    "pca_response"
] = pd.to_numeric(
    comparison_621c[
        "pca_response"
    ],
    errors="raise",
)


comparison_621c[
    "temporal_response"
] = pd.to_numeric(
    comparison_621c[
        "temporal_response"
    ],
    errors="raise",
)


assert np.isfinite(
    comparison_621c[
        "pca_response"
    ]
).all()


assert np.isfinite(
    comparison_621c[
        "temporal_response"
    ]
).all()


print(
    "\nVerified common scenarios:",
    len(comparison_621c)
)


# ================================================================
# 8. Safe correlation helper
# ================================================================

def safe_corr_621c(
    x_621c,
    y_621c,
    method_621c="pearson",
):

    pair_621c = pd.DataFrame(
        {
            "x": pd.to_numeric(
                x_621c,
                errors="coerce",
            ),
            "y": pd.to_numeric(
                y_621c,
                errors="coerce",
            ),
        }
    ).dropna()


    if len(pair_621c) < 3:
        return np.nan


    if (
        pair_621c["x"].nunique() < 2
        or
        pair_621c["y"].nunique() < 2
    ):
        return np.nan


    return float(
        pair_621c["x"].corr(
            pair_621c["y"],
            method=method_621c,
        )
    )


# ================================================================
# 9. Overall PCA / temporal association
# ================================================================

all_pearson_621c = safe_corr_621c(
    comparison_621c["pca_response"],
    comparison_621c["temporal_response"],
    "pearson",
)

all_spearman_621c = safe_corr_621c(
    comparison_621c["pca_response"],
    comparison_621c["temporal_response"],
    "spearman",
)


dynamic_families_621c = [
    "noise",
    "drift",
    "dropout",
    "stuck",
]


dynamic_621c = (
    comparison_621c[
        comparison_621c[
            "corruption_key"
        ].isin(
            dynamic_families_621c
        )
    ]
    .copy()
)


assert len(dynamic_621c) == 225


dynamic_pearson_621c = safe_corr_621c(
    dynamic_621c["pca_response"],
    dynamic_621c["temporal_response"],
    "pearson",
)

dynamic_spearman_621c = safe_corr_621c(
    dynamic_621c["pca_response"],
    dynamic_621c["temporal_response"],
    "spearman",
)


# ================================================================
# 10. Family-level response relationships
# ================================================================

family_rows_621c = []


for family_621c in [
    "bias",
    "noise",
    "drift",
    "dropout",
    "stuck",
]:

    subset_621c = (
        comparison_621c[
            comparison_621c[
                "corruption_key"
            ] == family_621c
        ]
        .copy()
    )


    family_rows_621c.append(
        {
            "family":
                family_621c,

            "n":
                len(subset_621c),

            "pca_mean":
                float(
                    subset_621c[
                        "pca_response"
                    ].mean()
                ),

            "temporal_mean":
                float(
                    subset_621c[
                        "temporal_response"
                    ].mean()
                ),

            "pearson":
                safe_corr_621c(
                    subset_621c[
                        "pca_response"
                    ],
                    subset_621c[
                        "temporal_response"
                    ],
                    "pearson",
                ),

            "spearman":
                safe_corr_621c(
                    subset_621c[
                        "pca_response"
                    ],
                    subset_621c[
                        "temporal_response"
                    ],
                    "spearman",
                ),
        }
    )


family_summary_621c = pd.DataFrame(
    family_rows_621c
)


# ================================================================
# 11. Rank-overlap diagnostic
# ================================================================
# This is supplementary only.
# It asks whether the scenarios ranked among the strongest 25%
# by PCA are the same scenarios ranked strongest by temporal score.
# ================================================================

def top_quartile_overlap_621c(
    df_621c,
):

    work_621c = (
        df_621c[
            [
                "corruption_key",
                "pca_id_key",
                "temporal_id_key",
                "pca_response",
                "temporal_response",
            ]
        ]
        .copy()
        .reset_index(drop=True)
    )


    work_621c[
        "row_id"
    ] = np.arange(
        len(work_621c)
    )


    number_top_621c = max(
        1,
        int(
            np.ceil(
                0.25
                * len(work_621c)
            )
        ),
    )


    pca_top_621c = set(
        work_621c
        .nlargest(
            number_top_621c,
            "pca_response",
        )[
            "row_id"
        ]
        .tolist()
    )


    temporal_top_621c = set(
        work_621c
        .nlargest(
            number_top_621c,
            "temporal_response",
        )[
            "row_id"
        ]
        .tolist()
    )


    intersection_621c = (
        pca_top_621c
        &
        temporal_top_621c
    )

    union_621c = (
        pca_top_621c
        |
        temporal_top_621c
    )


    return {
        "n":
            len(work_621c),

        "top_each":
            number_top_621c,

        "overlap":
            len(intersection_621c),

        "jaccard":
            (
                len(intersection_621c)
                /
                len(union_621c)
            ),
    }


all_overlap_621c = (
    top_quartile_overlap_621c(
        comparison_621c
    )
)

dynamic_overlap_621c = (
    top_quartile_overlap_621c(
        dynamic_621c
    )
)


# ================================================================
# 12. Constant-bias invariance check
# ================================================================

bias_only_621c = (
    comparison_621c[
        comparison_621c[
            "corruption_key"
        ] == "bias"
    ]
    .copy()
)


bias_temporal_max_abs_621c = float(
    np.max(
        np.abs(
            bias_only_621c[
                "temporal_response"
            ].to_numpy(
                dtype=float
            )
        )
    )
)


bias_temporal_nonzero_621c = int(
    np.sum(
        np.abs(
            bias_only_621c[
                "temporal_response"
            ].to_numpy(
                dtype=float
            )
        )
        >
        1e-12
    )
)


# ================================================================
# 13. Optional event-response correlation
# ================================================================

event_available_621c = bool(
    (
        "pca_event_delta_pp"
        in comparison_621c.columns
    )
    and
    (
        "temporal_event_delta_pp"
        in comparison_621c.columns
    )
)


if event_available_621c:

    comparison_621c[
        "pca_event_delta_pp"
    ] = pd.to_numeric(
        comparison_621c[
            "pca_event_delta_pp"
        ],
        errors="coerce",
    )

    comparison_621c[
        "temporal_event_delta_pp"
    ] = pd.to_numeric(
        comparison_621c[
            "temporal_event_delta_pp"
        ],
        errors="coerce",
    )


    event_spearman_all_621c = (
        safe_corr_621c(
            comparison_621c[
                "pca_event_delta_pp"
            ],
            comparison_621c[
                "temporal_event_delta_pp"
            ],
            "spearman",
        )
    )


    event_spearman_dynamic_621c = (
        safe_corr_621c(
            comparison_621c.loc[
                comparison_621c[
                    "corruption_key"
                ].isin(
                    dynamic_families_621c
                ),
                "pca_event_delta_pp",
            ],
            comparison_621c.loc[
                comparison_621c[
                    "corruption_key"
                ].isin(
                    dynamic_families_621c
                ),
                "temporal_event_delta_pp",
            ],
            "spearman",
        )
    )

else:

    event_spearman_all_621c = np.nan
    event_spearman_dynamic_621c = np.nan


# ================================================================
# 14. Compact output
# ================================================================

print(
    "\n--- Continuous-response association ---"
)

print(
    f"All 315 scenarios   | "
    f"Pearson={all_pearson_621c:.4f} | "
    f"Spearman={all_spearman_621c:.4f}"
)

print(
    f"Dynamic 225 only    | "
    f"Pearson={dynamic_pearson_621c:.4f} | "
    f"Spearman={dynamic_spearman_621c:.4f}"
)


print(
    "\n--- Family-level compact summary ---"
)

print(
    family_summary_621c.round(4).to_string(
        index=False
    )
)


print(
    "\n--- Strongest-quartile overlap ---"
)

print(
    "All scenarios       | "
    f"top_each={all_overlap_621c['top_each']} | "
    f"overlap={all_overlap_621c['overlap']} | "
    f"Jaccard={all_overlap_621c['jaccard']:.4f}"
)

print(
    "Dynamic only        | "
    f"top_each={dynamic_overlap_621c['top_each']} | "
    f"overlap={dynamic_overlap_621c['overlap']} | "
    f"Jaccard={dynamic_overlap_621c['jaccard']:.4f}"
)


print(
    "\n--- Constant-bias temporal invariance ---"
)

print(
    "Bias scenarios                 :",
    len(bias_only_621c),
)

print(
    "Mean PCA response              :",
    round(
        float(
            bias_only_621c[
                "pca_response"
            ].mean()
        ),
        6,
    ),
)

print(
    "Maximum |temporal response|    :",
    f"{bias_temporal_max_abs_621c:.12g}",
)

print(
    "Non-zero temporal responses    :",
    bias_temporal_nonzero_621c,
)


print(
    "\n--- Event-response association ---"
)

if event_available_621c:

    print(
        f"All scenarios Spearman     : "
        f"{event_spearman_all_621c:.4f}"
    )

    print(
        f"Dynamic-only Spearman      : "
        f"{event_spearman_dynamic_621c:.4f}"
    )

else:

    print(
        "Comparable event-delta columns were "
        "not available in both pathways."
    )


# ================================================================
# 15. Integrity summary
# ================================================================

print(
    "\n=== Step 6.21C integrity summary ==="
)

print(
    "Aligned scenarios analysed :",
    len(comparison_621c),
)

print(
    "Dynamic scenarios analysed :",
    len(dynamic_621c),
)

print(
    "Corruption families        :",
    comparison_621c[
        "corruption_key"
    ].nunique(),
)

print(
    "PCA values finite          :",
    bool(
        np.isfinite(
            comparison_621c[
                "pca_response"
            ]
        ).all()
    ),
)

print(
    "Temporal values finite     :",
    bool(
        np.isfinite(
            comparison_621c[
                "temporal_response"
            ]
        ).all()
    ),
)


print(
    "\nStep 6.21C completed successfully."
)

print(
    "PCA and temporal responses were compared "
    "on the verified one-to-one scenario alignment."
)

print(
    "No PCA-temporal combination was created."
)

print(
    "No threshold or frozen artifact was modified."
)

In [ ]:
# ================================================================
# Step 6.21C-1 — Compact complementarity summary
# ================================================================

print("\n=== STEP 6.21C-1 COMPACT SUMMARY ===")


# ------------------------------------------------
# 1. Overall association
# ------------------------------------------------

print("\n--- Overall association ---")

print(
    f"ALL 315     : "
    f"Pearson={all_pearson_621c:.4f}, "
    f"Spearman={all_spearman_621c:.4f}"
)

print(
    f"DYNAMIC 225 : "
    f"Pearson={dynamic_pearson_621c:.4f}, "
    f"Spearman={dynamic_spearman_621c:.4f}"
)


# ------------------------------------------------
# 2. Family-level association
# ------------------------------------------------

print("\n--- Family-level association ---")

for _, row in family_summary_621c.iterrows():

    pearson_text = (
        "NA"
        if pd.isna(row["pearson"])
        else f'{row["pearson"]:.4f}'
    )

    spearman_text = (
        "NA"
        if pd.isna(row["spearman"])
        else f'{row["spearman"]:.4f}'
    )

    print(
        f'{row["family"]:<8} | '
        f'n={int(row["n"]):>3} | '
        f'PCA mean={row["pca_mean"]:.4f} | '
        f'TEMP mean={row["temporal_mean"]:.4f} | '
        f'Pearson={pearson_text} | '
        f'Spearman={spearman_text}'
    )


# ------------------------------------------------
# 3. Strongest-quartile overlap
# ------------------------------------------------

print("\n--- Strongest-quartile overlap ---")

print(
    f'ALL     : top={all_overlap_621c["top_each"]}, '
    f'overlap={all_overlap_621c["overlap"]}, '
    f'Jaccard={all_overlap_621c["jaccard"]:.4f}'
)

print(
    f'DYNAMIC : top={dynamic_overlap_621c["top_each"]}, '
    f'overlap={dynamic_overlap_621c["overlap"]}, '
    f'Jaccard={dynamic_overlap_621c["jaccard"]:.4f}'
)


# ------------------------------------------------
# 4. Bias invariance
# ------------------------------------------------

print("\n--- Bias temporal invariance ---")

print(
    f'Bias scenarios              : {len(bias_only_621c)}'
)

print(
    f'Mean PCA response           : '
    f'{bias_only_621c["pca_response"].mean():.6f}'
)

print(
    f'Max |temporal response|     : '
    f'{bias_temporal_max_abs_621c:.12g}'
)

print(
    f'Non-zero temporal responses : '
    f'{bias_temporal_nonzero_621c}'
)


# ------------------------------------------------
# 5. Event-response association
# ------------------------------------------------

print("\n--- Event-response association ---")

if event_available_621c:

    print(
        f'ALL Spearman     : '
        f'{event_spearman_all_621c:.4f}'
    )

    print(
        f'DYNAMIC Spearman : '
        f'{event_spearman_dynamic_621c:.4f}'
    )

else:

    print("Comparable event responses unavailable.")


# ------------------------------------------------
# 6. Integrity
# ------------------------------------------------

print("\n--- Integrity ---")

print(
    "Aligned scenarios :",
    len(comparison_621c)
)

print(
    "Dynamic scenarios :",
    len(dynamic_621c)
)

print(
    "Families          :",
    comparison_621c["corruption_key"].nunique()
)

print(
    "Finite PCA        :",
    np.isfinite(
        comparison_621c["pca_response"]
    ).all()
)

print(
    "Finite temporal   :",
    np.isfinite(
        comparison_621c["temporal_response"]
    ).all()
)

print("\n=== END COMPACT SUMMARY ===")

In [ ]:
print("=== 6.21C EVENT ASSOCIATION ===")

if event_available_621c:
    print(
        "ALL scenarios Spearman    :",
        round(float(event_spearman_all_621c), 4)
    )
    print(
        "DYNAMIC scenarios Spearman:",
        round(float(event_spearman_dynamic_621c), 4)
    )
else:
    print("Comparable event responses unavailable.")

print("=== END ===")

In [ ]:
# ================================================================
# STEP 6.21D — CLEAN-ONLY PCA + TEMPORAL FUSION CALIBRATION
# ================================================================
#
# Purpose
# -------
# 1. Recover the already-computed PCA and selected temporal scores.
# 2. Normalize both pathways using DEVELOPMENT-calibrated thresholds.
# 3. Construct simple candidate fusion rules.
# 4. Calibrate each fusion rule at DEVELOPMENT p95.
# 5. Sanity-check the resulting false-event rate on CLEAN validation.
#
# IMPORTANT
# ---------
# - CLEAN development is used for calibration.
# - CLEAN validation is used only as a sanity check.
# - No corruption is generated.
# - No final PCA+temporal fusion is selected here.
# - No frozen file is modified.
# ================================================================

import numpy as np
import pandas as pd

print(
    "\n=== Step 6.21D — Clean-only PCA / Temporal fusion calibration ==="
)


# ================================================================
# 1. Required PCA objects
# ================================================================

required_pca_objects_621d = [
    "dev_pca_max_rmse",
    "val_pca_max_rmse",
]

print("\n--- Required PCA objects ---")

for name_621d in required_pca_objects_621d:
    print(
        f"{name_621d:<28}:",
        name_621d in globals()
    )

assert all(
    name_621d in globals()
    for name_621d in required_pca_objects_621d
), (
    "Required PCA score arrays are missing."
)


dev_pca_score_621d = np.asarray(
    dev_pca_max_rmse,
    dtype=np.float64
).reshape(-1)

val_pca_score_621d = np.asarray(
    val_pca_max_rmse,
    dtype=np.float64
).reshape(-1)


# ================================================================
# 2. Helper: extract one candidate array from a candidate container
# ================================================================

def extract_candidate_array_621d(
    object_621d,
    candidate_name_621d,
):
    """
    Extract a 1-D candidate-score array from a common container type.
    """

    # Dictionary
    if isinstance(object_621d, dict):

        if candidate_name_621d in object_621d:
            return np.asarray(
                object_621d[candidate_name_621d],
                dtype=np.float64
            ).reshape(-1)

    # DataFrame
    if isinstance(object_621d, pd.DataFrame):

        if candidate_name_621d in object_621d.columns:
            return np.asarray(
                object_621d[candidate_name_621d],
                dtype=np.float64
            ).reshape(-1)

    # Series
    if isinstance(object_621d, pd.Series):

        return np.asarray(
            object_621d,
            dtype=np.float64
        ).reshape(-1)

    return None


# ================================================================
# 3. Locate DEVELOPMENT temporal candidate scores
# ================================================================

FROZEN_TEMPORAL_NAME_621D = str(
    FROZEN_TEMPORAL_CANDIDATE_620I
)

FROZEN_TEMPORAL_THRESHOLD_621D = float(
    FROZEN_TEMPORAL_THRESHOLD_620I
)

print("\n--- Frozen temporal signal ---")
print(
    "Candidate :",
    FROZEN_TEMPORAL_NAME_621D
)
print(
    "Threshold :",
    FROZEN_TEMPORAL_THRESHOLD_621D
)


dev_temporal_score_621d = None
dev_temporal_source_621d = None


# Preferred exact object
if "dev_candidate_scores_620ha" in globals():

    extracted_621d = extract_candidate_array_621d(
        globals()["dev_candidate_scores_620ha"],
        FROZEN_TEMPORAL_NAME_621D,
    )

    if extracted_621d is not None:
        dev_temporal_score_621d = extracted_621d
        dev_temporal_source_621d = (
            "dev_candidate_scores_620ha"
        )


# Safe search fallback
if dev_temporal_score_621d is None:

    snapshot_621d = list(globals().items())

    dev_candidates_621d = []

    for object_name_621d, object_value_621d in snapshot_621d:

        lower_name_621d = str(
            object_name_621d
        ).lower()

        if "dev" not in lower_name_621d:
            continue

        if (
            "candidate"
            not in lower_name_621d
            and
            "temporal"
            not in lower_name_621d
        ):
            continue

        try:

            extracted_621d = extract_candidate_array_621d(
                object_value_621d,
                FROZEN_TEMPORAL_NAME_621D,
            )

            if (
                extracted_621d is not None
                and
                len(extracted_621d)
                == len(dev_pca_score_621d)
            ):
                dev_candidates_621d.append(
                    (
                        object_name_621d,
                        extracted_621d
                    )
                )

        except Exception:
            pass


    if len(dev_candidates_621d) > 0:

        # Deterministic preference:
        # shortest object name first.
        dev_candidates_621d = sorted(
            dev_candidates_621d,
            key=lambda x: (
                len(str(x[0])),
                str(x[0])
            )
        )

        dev_temporal_source_621d = (
            dev_candidates_621d[0][0]
        )

        dev_temporal_score_621d = (
            dev_candidates_621d[0][1]
        )


assert dev_temporal_score_621d is not None, (
    "Could not safely locate the DEVELOPMENT "
    "temporal candidate-score array."
)


# ================================================================
# 4. Locate CLEAN VALIDATION temporal candidate scores
# ================================================================

val_temporal_score_621d = None
val_temporal_source_621d = None


if "val_candidate_scores_620ha" in globals():

    extracted_621d = extract_candidate_array_621d(
        globals()["val_candidate_scores_620ha"],
        FROZEN_TEMPORAL_NAME_621D,
    )

    if extracted_621d is not None:
        val_temporal_score_621d = extracted_621d
        val_temporal_source_621d = (
            "val_candidate_scores_620ha"
        )


# Safe fallback search
if val_temporal_score_621d is None:

    snapshot_621d = list(globals().items())

    val_candidates_621d = []

    for object_name_621d, object_value_621d in snapshot_621d:

        lower_name_621d = str(
            object_name_621d
        ).lower()

        if "val" not in lower_name_621d:
            continue

        if (
            "candidate"
            not in lower_name_621d
            and
            "temporal"
            not in lower_name_621d
        ):
            continue

        try:

            extracted_621d = extract_candidate_array_621d(
                object_value_621d,
                FROZEN_TEMPORAL_NAME_621D,
            )

            if (
                extracted_621d is not None
                and
                len(extracted_621d)
                == len(val_pca_score_621d)
            ):
                val_candidates_621d.append(
                    (
                        object_name_621d,
                        extracted_621d
                    )
                )

        except Exception:
            pass


    if len(val_candidates_621d) > 0:

        val_candidates_621d = sorted(
            val_candidates_621d,
            key=lambda x: (
                len(str(x[0])),
                str(x[0])
            )
        )

        val_temporal_source_621d = (
            val_candidates_621d[0][0]
        )

        val_temporal_score_621d = (
            val_candidates_621d[0][1]
        )


assert val_temporal_score_621d is not None, (
    "Could not safely locate the CLEAN VALIDATION "
    "temporal candidate-score array."
)


print("\n--- Temporal score sources ---")
print(
    "Development :",
    dev_temporal_source_621d
)
print(
    "Validation  :",
    val_temporal_source_621d
)


# ================================================================
# 5. Integrity checks
# ================================================================

assert (
    len(dev_pca_score_621d)
    ==
    len(dev_temporal_score_621d)
), (
    "Development PCA and temporal lengths differ."
)

assert (
    len(val_pca_score_621d)
    ==
    len(val_temporal_score_621d)
), (
    "Validation PCA and temporal lengths differ."
)


for array_name_621d, array_621d in [
    ("dev_pca", dev_pca_score_621d),
    ("val_pca", val_pca_score_621d),
    ("dev_temporal", dev_temporal_score_621d),
    ("val_temporal", val_temporal_score_621d),
]:

    assert np.isfinite(
        array_621d
    ).all(), (
        f"Non-finite values detected in "
        f"{array_name_621d}."
    )


print("\n--- Score-array structure ---")
print(
    "Development windows :",
    len(dev_pca_score_621d)
)
print(
    "Validation windows  :",
    len(val_pca_score_621d)
)


# ================================================================
# 6. Development-calibrated pathway thresholds
# ================================================================

# PCA p95 is reconstructed strictly from CLEAN DEVELOPMENT.
PCA_P95_621D = float(
    np.quantile(
        dev_pca_score_621d,
        0.95
    )
)

# Temporal threshold is the already-frozen Step-6.20I threshold.
TEMPORAL_P95_621D = float(
    FROZEN_TEMPORAL_THRESHOLD_621D
)


assert PCA_P95_621D > 0
assert TEMPORAL_P95_621D > 0


# Verify frozen temporal threshold against development score p95.
temporal_recomputed_p95_621d = float(
    np.quantile(
        dev_temporal_score_621d,
        0.95
    )
)

temporal_threshold_difference_621d = abs(
    temporal_recomputed_p95_621d
    -
    TEMPORAL_P95_621D
)


print("\n--- Development pathway calibration ---")
print(
    f"PCA development p95      : "
    f"{PCA_P95_621D:.12f}"
)
print(
    f"Temporal frozen p95      : "
    f"{TEMPORAL_P95_621D:.12f}"
)
print(
    f"Temporal recomputed p95  : "
    f"{temporal_recomputed_p95_621d:.12f}"
)
print(
    f"Temporal agreement error : "
    f"{temporal_threshold_difference_621d:.3e}"
)


assert temporal_threshold_difference_621d < 1e-8, (
    "Frozen temporal threshold does not agree "
    "with the development temporal score distribution."
)


# ================================================================
# 7. Normalize both pathways to their DEVELOPMENT p95
# ================================================================

dev_pca_ratio_621d = (
    dev_pca_score_621d
    /
    PCA_P95_621D
)

val_pca_ratio_621d = (
    val_pca_score_621d
    /
    PCA_P95_621D
)

dev_temporal_ratio_621d = (
    dev_temporal_score_621d
    /
    TEMPORAL_P95_621D
)

val_temporal_ratio_621d = (
    val_temporal_score_621d
    /
    TEMPORAL_P95_621D
)


# ================================================================
# 8. Define simple fusion candidates
# ================================================================
#
# MAX
#   OR-like complementary fusion.
#
# MEAN
#   Requires broader agreement between pathways.
#
# RMS
#   Intermediate behaviour; emphasizes the larger pathway more
#   than arithmetic mean, but less strongly than MAX.
#
# GEOMETRIC
#   Strong agreement-oriented reference candidate.
#
# No candidate is selected here.
# ================================================================

def build_fusion_candidates_621d(
    pca_ratio_621d,
    temporal_ratio_621d,
):

    pca_nonnegative_621d = np.maximum(
        pca_ratio_621d,
        0.0
    )

    temporal_nonnegative_621d = np.maximum(
        temporal_ratio_621d,
        0.0
    )

    return {
        "max_fusion":
            np.maximum(
                pca_nonnegative_621d,
                temporal_nonnegative_621d
            ),

        "mean_fusion":
            0.5 * (
                pca_nonnegative_621d
                +
                temporal_nonnegative_621d
            ),

        "rms_fusion":
            np.sqrt(
                0.5 * (
                    pca_nonnegative_621d ** 2
                    +
                    temporal_nonnegative_621d ** 2
                )
            ),

        "geometric_fusion":
            np.sqrt(
                pca_nonnegative_621d
                *
                temporal_nonnegative_621d
            ),
    }


dev_fusion_candidates_621d = (
    build_fusion_candidates_621d(
        dev_pca_ratio_621d,
        dev_temporal_ratio_621d,
    )
)

val_fusion_candidates_621d = (
    build_fusion_candidates_621d(
        val_pca_ratio_621d,
        val_temporal_ratio_621d,
    )
)


# ================================================================
# 9. Calibrate EACH fusion candidate from CLEAN DEVELOPMENT only
# ================================================================

fusion_thresholds_621d = {}

fusion_rows_621d = []


for candidate_621d in dev_fusion_candidates_621d:

    dev_scores_621d = (
        dev_fusion_candidates_621d[
            candidate_621d
        ]
    )

    val_scores_621d = (
        val_fusion_candidates_621d[
            candidate_621d
        ]
    )

    threshold_621d = float(
        np.quantile(
            dev_scores_621d,
            0.95
        )
    )

    fusion_thresholds_621d[
        candidate_621d
    ] = threshold_621d

    dev_event_rate_621d = float(
        100.0
        *
        np.mean(
            dev_scores_621d
            >
            threshold_621d
        )
    )

    val_event_rate_621d = float(
        100.0
        *
        np.mean(
            val_scores_621d
            >
            threshold_621d
        )
    )

    fusion_rows_621d.append(
        {
            "candidate":
                candidate_621d,

            "development_p95_threshold":
                threshold_621d,

            "development_event_pct":
                dev_event_rate_621d,

            "clean_validation_event_pct":
                val_event_rate_621d,

            "clean_validation_mean":
                float(
                    np.mean(
                        val_scores_621d
                    )
                ),

            "clean_validation_median":
                float(
                    np.median(
                        val_scores_621d
                    )
                ),

            "clean_validation_p95":
                float(
                    np.quantile(
                        val_scores_621d,
                        0.95
                    )
                ),
        }
    )


fusion_clean_summary_621d = pd.DataFrame(
    fusion_rows_621d
)


# ================================================================
# 10. Individual pathway clean-validation event rates
# ================================================================

pca_val_event_pct_621d = float(
    100.0
    *
    np.mean(
        val_pca_ratio_621d > 1.0
    )
)

temporal_val_event_pct_621d = float(
    100.0
    *
    np.mean(
        val_temporal_ratio_621d > 1.0
    )
)


both_pathways_event_pct_621d = float(
    100.0
    *
    np.mean(
        (val_pca_ratio_621d > 1.0)
        &
        (val_temporal_ratio_621d > 1.0)
    )
)

either_pathway_event_pct_621d = float(
    100.0
    *
    np.mean(
        (val_pca_ratio_621d > 1.0)
        |
        (val_temporal_ratio_621d > 1.0)
    )
)


clean_overlap_summary_621d = pd.DataFrame(
    [
        {
            "PCA_event_pct":
                pca_val_event_pct_621d,

            "temporal_event_pct":
                temporal_val_event_pct_621d,

            "both_event_pct":
                both_pathways_event_pct_621d,

            "either_event_pct":
                either_pathway_event_pct_621d,
        }
    ]
)


# ================================================================
# 11. Compact output only
# ================================================================

print(
    "\n=== CLEAN validation pathway overlap ==="
)

display(
    clean_overlap_summary_621d.round(4)
)


print(
    "\n=== CLEAN-only fusion calibration summary ==="
)

display(
    fusion_clean_summary_621d[
        [
            "candidate",
            "development_p95_threshold",
            "clean_validation_event_pct",
            "clean_validation_mean",
            "clean_validation_p95",
        ]
    ].round(6)
)


# ================================================================
# 12. Integrity summary
# ================================================================

print(
    "\n=== Step 6.21D integrity summary ==="
)

print(
    "Development windows              :",
    len(dev_pca_score_621d)
)

print(
    "Clean validation windows         :",
    len(val_pca_score_621d)
)

print(
    "Fusion candidates evaluated      :",
    len(fusion_thresholds_621d)
)

print(
    "Threshold calibration source     : "
    "CLEAN DEVELOPMENT only"
)

print(
    "Frozen temporal signal modified : False"
)

print(
    "Frozen PCA artifacts modified    : False"
)

print(
    "Final fusion selected            : False"
)

print(
    "\nStep 6.21D completed successfully."
)

print(
    "Next step: evaluate these calibrated fusion "
    "candidates across the frozen corruption scenarios."
)

In [ ]:
# ================================================================
# STEP 6.21E-A — PCA / Temporal fusion execution pre-check
# ================================================================

import inspect
import numpy as np
import pandas as pd

print("\n=== Step 6.21E-A — Fusion execution pre-check ===")


# ------------------------------------------------
# 1. Core objects needed for frozen-scenario replay
# ------------------------------------------------

required_621ea = [
    "scenario_manifests_620gc",
    "apply_frozen_corruption_620gc",
    "val_raw_base_620gc",
    "fusion_thresholds_621d",
    "PCA_P95_621D",
    "TEMPORAL_P95_621D",
    "FROZEN_TEMPORAL_NAME_621D",
]

print("\n--- Core object availability ---")

for name_621ea in required_621ea:
    print(
        f"{name_621ea:<34}: "
        f"{name_621ea in globals()}"
    )

assert all(
    name_621ea in globals()
    for name_621ea in required_621ea
), "One or more required fusion objects are missing."


# ------------------------------------------------
# 2. Exact PCA scoring function
# ------------------------------------------------

print("\n--- PCA window-scoring function ---")

pca_function_name_621ea = "calculate_pca_window_quality"

pca_function_available_621ea = (
    pca_function_name_621ea in globals()
    and callable(
        globals()[pca_function_name_621ea]
    )
)

print(
    "calculate_pca_window_quality available:",
    pca_function_available_621ea
)

if pca_function_available_621ea:

    pca_function_621ea = globals()[
        pca_function_name_621ea
    ]

    try:
        print(
            "Signature:",
            inspect.signature(
                pca_function_621ea
            )
        )
    except Exception as exc_621ea:
        print(
            "Signature unavailable:",
            type(exc_621ea).__name__
        )


# ------------------------------------------------
# 3. Locate fitted PCA-like objects
# ------------------------------------------------

snapshot_621ea = list(
    globals().items()
)

pca_model_rows_621ea = []

for object_name_621ea, object_value_621ea in snapshot_621ea:

    lower_name_621ea = str(
        object_name_621ea
    ).lower()

    if "pca" not in lower_name_621ea:
        continue

    # We are interested in FITTED PCA-like objects,
    # not arrays, tables, functions, or the PCA class.
    has_transform_621ea = callable(
        getattr(
            object_value_621ea,
            "transform",
            None
        )
    )

    has_components_621ea = hasattr(
        object_value_621ea,
        "components_"
    )

    has_mean_621ea = hasattr(
        object_value_621ea,
        "mean_"
    )

    if (
        has_transform_621ea
        and
        has_components_621ea
        and
        has_mean_621ea
    ):

        components_shape_621ea = tuple(
            np.asarray(
                object_value_621ea.components_
            ).shape
        )

        pca_model_rows_621ea.append(
            {
                "object_name":
                    object_name_621ea,
                "type":
                    type(
                        object_value_621ea
                    ).__name__,
                "components_shape":
                    components_shape_621ea,
            }
        )


pca_model_df_621ea = pd.DataFrame(
    pca_model_rows_621ea
)

print("\n--- Fitted PCA-like objects ---")

if len(pca_model_df_621ea) == 0:
    print(
        "No fitted PCA-like object was "
        "identified automatically."
    )
else:
    display(
        pca_model_df_621ea
    )


# ------------------------------------------------
# 4. Relevant clean PCA arrays
# ------------------------------------------------

important_arrays_621ea = [
    "dev_pca_row_rmse",
    "dev_pca_mean_rmse",
    "dev_pca_max_rmse",
    "val_pca_row_rmse",
    "val_pca_mean_rmse",
    "val_pca_max_rmse",
]

print("\n--- PCA clean-array availability ---")

for name_621ea in important_arrays_621ea:

    if name_621ea in globals():

        value_621ea = globals()[
            name_621ea
        ]

        print(
            f"{name_621ea:<24}: "
            f"True  shape="
            f"{getattr(value_621ea, 'shape', None)}"
        )

    else:

        print(
            f"{name_621ea:<24}: False"
        )


# ------------------------------------------------
# 5. Scenario alignment/frozen-result availability
# ------------------------------------------------

print("\n--- Scenario-level frozen objects ---")

for name_621ea in [
    "pca_scenarios_621b",
    "temporal_selected_621b",
    "scenario_alignment_621b2",
]:

    if name_621ea in globals():

        obj_621ea = globals()[
            name_621ea
        ]

        print(
            f"{name_621ea:<30}: "
            f"True  shape="
            f"{getattr(obj_621ea, 'shape', None)}"
        )

    else:

        print(
            f"{name_621ea:<30}: False"
        )


# ------------------------------------------------
# 6. Compact readiness summary
# ------------------------------------------------

fitted_pca_count_621ea = int(
    len(
        pca_model_df_621ea
    )
)

print(
    "\n========================================"
)
print(
    "STEP 6.21E-A READINESS SUMMARY"
)
print(
    "========================================"
)

print(
    "Frozen corruption replay ready :",
    True
)

print(
    "PCA scoring function available :",
    pca_function_available_621ea
)

print(
    "Fitted PCA-like objects found   :",
    fitted_pca_count_621ea
)

print(
    "Fusion candidates calibrated    :",
    len(
        fusion_thresholds_621d
    )
)

print(
    "Corruption generated            : False"
)

print(
    "Fusion executed                 : False"
)

print(
    "Frozen artifact modified        : False"
)

print(
    "\nStep 6.21E-A completed."
)

In [ ]:
# ================================================================
# STEP 6.21E-B — Exact PCA / Temporal pathway smoke check
# ================================================================

import inspect
import numpy as np
import pandas as pd

print("\n=== Step 6.21E-B — Exact pathway smoke check ===")


# ------------------------------------------------
# 1. Required object availability
# ------------------------------------------------

required_621eb = [
    "X_val_clean",
    "SENSOR_INPUT_INDICES",
    "sensor_pca",
    "calculate_pca_window_quality",
    "val_pca_row_rmse",
    "val_pca_mean_rmse",
    "val_pca_max_rmse",
    "temporal_metrics_620ha",
    "sensor_strength_620ha",
    "aggregate_candidates_620ha",
]

print("\n--- Required object availability ---")

for name_621eb in required_621eb:
    print(
        f"{name_621eb:<32}: "
        f"{name_621eb in globals()}"
    )

assert all(
    name_621eb in globals()
    for name_621eb in required_621eb
), "One or more required Step-6.21E-B objects are missing."


# ------------------------------------------------
# 2. Temporal helper signatures
# ------------------------------------------------

print("\n--- Temporal helper inspection ---")

temporal_helpers_621eb = [
    "temporal_metrics_620ha",
    "sensor_strength_620ha",
    "aggregate_candidates_620ha",
]

for name_621eb in temporal_helpers_621eb:

    obj_621eb = globals()[name_621eb]

    print(
        f"\n{name_621eb}"
    )

    print(
        "Type      :",
        type(obj_621eb).__name__
    )

    print(
        "Callable  :",
        callable(obj_621eb)
    )

    if callable(obj_621eb):
        try:
            print(
                "Signature :",
                inspect.signature(obj_621eb)
            )
        except Exception as exc_621eb:
            print(
                "Signature : unavailable -",
                type(exc_621eb).__name__
            )


# ------------------------------------------------
# 3. PCA scoring smoke test on first 8 CLEAN windows
# ------------------------------------------------

print("\n--- PCA function smoke test ---")

probe_n_621eb = 8

X_probe_621eb = (
    np.asarray(
        X_val_clean[:probe_n_621eb],
        dtype=np.float64
    )
)

pca_probe_621eb = (
    calculate_pca_window_quality(
        X_probe_621eb,
        SENSOR_INPUT_INDICES,
        sensor_pca,
    )
)

print(
    "Input shape :",
    X_probe_621eb.shape
)

print(
    "Return type :",
    type(pca_probe_621eb).__name__
)


# ------------------------------------------------
# 4. Compactly describe PCA return structure
# ------------------------------------------------

def describe_probe_object_621eb(obj_621eb, label_621eb):

    if isinstance(obj_621eb, np.ndarray):

        print(
            f"{label_621eb:<16}"
            f"type=ndarray  "
            f"shape={obj_621eb.shape}  "
            f"dtype={obj_621eb.dtype}"
        )

    elif isinstance(obj_621eb, (list, tuple)):

        print(
            f"{label_621eb:<16}"
            f"type={type(obj_621eb).__name__}  "
            f"length={len(obj_621eb)}"
        )

        for idx_621eb, item_621eb in enumerate(obj_621eb):

            print(
                f"  item {idx_621eb}: "
                f"type={type(item_621eb).__name__}, "
                f"shape={getattr(item_621eb, 'shape', None)}"
            )

    elif isinstance(obj_621eb, dict):

        print(
            f"{label_621eb:<16}"
            f"type=dict  keys={list(obj_621eb.keys())}"
        )

        for key_621eb, item_621eb in obj_621eb.items():

            print(
                f"  {key_621eb}: "
                f"type={type(item_621eb).__name__}, "
                f"shape={getattr(item_621eb, 'shape', None)}"
            )

    else:

        print(
            f"{label_621eb:<16}"
            f"type={type(obj_621eb).__name__}"
        )


describe_probe_object_621eb(
    pca_probe_621eb,
    "PCA output"
)


# ------------------------------------------------
# 5. Identify PCA outputs against frozen arrays
# ------------------------------------------------

print("\n--- PCA output agreement with frozen CLEAN arrays ---")


def compare_array_621eb(
    candidate_621eb,
    reference_621eb,
    reference_name_621eb,
):

    candidate_arr_621eb = np.asarray(
        candidate_621eb,
        dtype=np.float64
    )

    reference_arr_621eb = np.asarray(
        reference_621eb,
        dtype=np.float64
    )

    if (
        candidate_arr_621eb.shape
        !=
        reference_arr_621eb.shape
    ):
        return None

    return float(
        np.max(
            np.abs(
                candidate_arr_621eb
                -
                reference_arr_621eb
            )
        )
    )


reference_arrays_621eb = {
    "row_rmse":
        np.asarray(
            val_pca_row_rmse[
                :probe_n_621eb
            ],
            dtype=np.float64
        ),

    "mean_rmse":
        np.asarray(
            val_pca_mean_rmse[
                :probe_n_621eb
            ],
            dtype=np.float64
        ),

    "max_rmse":
        np.asarray(
            val_pca_max_rmse[
                :probe_n_621eb
            ],
            dtype=np.float64
        ),
}


candidate_outputs_621eb = []

if isinstance(
    pca_probe_621eb,
    dict
):

    candidate_outputs_621eb = [
        (str(key_621eb), value_621eb)
        for key_621eb, value_621eb
        in pca_probe_621eb.items()
    ]

elif isinstance(
    pca_probe_621eb,
    (list, tuple)
):

    candidate_outputs_621eb = [
        (
            f"item_{idx_621eb}",
            value_621eb
        )
        for idx_621eb, value_621eb
        in enumerate(pca_probe_621eb)
    ]

else:

    candidate_outputs_621eb = [
        (
            "output",
            pca_probe_621eb
        )
    ]


comparison_rows_621eb = []

for output_name_621eb, output_value_621eb in candidate_outputs_621eb:

    for ref_name_621eb, ref_value_621eb in reference_arrays_621eb.items():

        try:

            error_621eb = (
                compare_array_621eb(
                    output_value_621eb,
                    ref_value_621eb,
                    ref_name_621eb,
                )
            )

        except Exception:

            error_621eb = None

        if error_621eb is not None:

            comparison_rows_621eb.append(
                {
                    "pca_output":
                        output_name_621eb,

                    "frozen_reference":
                        ref_name_621eb,

                    "maximum_absolute_error":
                        error_621eb,
                }
            )


comparison_df_621eb = pd.DataFrame(
    comparison_rows_621eb
)

if len(comparison_df_621eb) > 0:

    display(
        comparison_df_621eb
        .sort_values(
            "maximum_absolute_error"
        )
        .reset_index(drop=True)
    )

else:

    print(
        "No directly shape-compatible "
        "PCA outputs were found."
    )


# ------------------------------------------------
# 6. Compact readiness summary
# ------------------------------------------------

best_match_error_621eb = (
    float(
        comparison_df_621eb[
            "maximum_absolute_error"
        ].min()
    )
    if len(comparison_df_621eb) > 0
    else np.nan
)

print(
    "\n========================================"
)
print(
    "STEP 6.21E-B READINESS SUMMARY"
)
print(
    "========================================"
)

print(
    "PCA scoring callable           :",
    callable(
        calculate_pca_window_quality
    )
)

print(
    "Temporal helper 1 callable     :",
    callable(
        temporal_metrics_620ha
    )
)

print(
    "Temporal helper 2 callable     :",
    callable(
        sensor_strength_620ha
    )
)

print(
    "Temporal helper 3 callable     :",
    callable(
        aggregate_candidates_620ha
    )
)

print(
    "PCA probe windows              :",
    probe_n_621eb
)

print(
    "Best PCA frozen-array error    :",
    best_match_error_621eb
)

print(
    "Corruption generated           : False"
)

print(
    "Fusion experiment executed     : False"
)

print(
    "Frozen artifact modified       : False"
)

print(
    "\nStep 6.21E-B completed."
)

In [ ]:
import inspect

print("=== EXACT TEMPORAL HELPER SIGNATURES ===")

for name in [
    "temporal_metrics_620ha",
    "sensor_strength_620ha",
    "aggregate_candidates_620ha",
]:
    print(
        f"{name}: {inspect.signature(globals()[name])}"
    )

print("=== END ===")

In [ ]:
# ============================================================
# Step 6.21E-C
# Exact temporal-pathway smoke check — CORRECTED VERSION
#
# Purpose:
#   Reconstruct the frozen temporal-quality pathway exactly on
#   CLEAN validation data before the full PCA/Temporal fusion run.
#
# Important:
#   - No corruption is generated.
#   - No threshold is changed.
#   - No frozen artifact is modified.
#   - No fusion experiment is executed here.
# ============================================================

import numpy as np
import pandas as pd

print(
    "\n=== Step 6.21E-C — "
    "Exact temporal-pathway smoke check ==="
)


# ============================================================
# 1. Required objects
# ============================================================

required_objects_621ec = [
    "X_val_clean",
    "scaler",
    "PRIMARY_INPUT_COLUMNS",
    "RETAINED_SENSOR_COLUMNS",
    "SENSOR_INPUT_INDICES",
    "temporal_metrics_620ha",
    "sensor_strength_620ha",
    "aggregate_candidates_620ha",
    "val_candidate_scores_620ha",
    "FROZEN_TEMPORAL_CANDIDATE_620I",
    "FROZEN_TEMPORAL_THRESHOLD_620I",
]

print("\n--- Required objects ---")

for name_621ec in required_objects_621ec:
    print(
        f"{name_621ec:<34}: "
        f"{name_621ec in globals()}"
    )

missing_621ec = [
    name_621ec
    for name_621ec in required_objects_621ec
    if name_621ec not in globals()
]

if missing_621ec:
    raise RuntimeError(
        "Missing required objects: "
        + ", ".join(missing_621ec)
    )

print("\nAll required objects are available.")


# ============================================================
# 2. Validation sequence structure
# ============================================================

X_val_621ec = np.asarray(X_val_clean)

if X_val_621ec.ndim != 3:
    raise RuntimeError(
        "X_val_clean must be a 3-D sequence array."
    )

n_windows_621ec = int(X_val_621ec.shape[0])
window_size_621ec = int(X_val_621ec.shape[1])
n_primary_features_621ec = int(X_val_621ec.shape[2])

print("\n--- Validation sequence structure ---")
print("X_val_clean shape :", X_val_621ec.shape)
print("Primary features  :", n_primary_features_621ec)
print(
    "Retained sensors  :",
    len(RETAINED_SENSOR_COLUMNS)
)

assert (
    n_primary_features_621ec
    == len(PRIMARY_INPUT_COLUMNS)
), (
    "Feature count does not match "
    "PRIMARY_INPUT_COLUMNS."
)


# ============================================================
# 3. Verify retained-sensor mapping
# ============================================================

sensor_indices_from_names_621ec = [
    PRIMARY_INPUT_COLUMNS.index(sensor_621ec)
    for sensor_621ec in RETAINED_SENSOR_COLUMNS
]

sensor_indices_frozen_621ec = list(
    SENSOR_INPUT_INDICES
)

mapping_match_621ec = (
    sensor_indices_from_names_621ec
    == sensor_indices_frozen_621ec
)

print("\n--- Retained-sensor mapping ---")
print(
    "Name-derived indices :",
    sensor_indices_from_names_621ec
)
print(
    "Frozen indices       :",
    sensor_indices_frozen_621ec
)
print(
    "Exact mapping match  :",
    mapping_match_621ec
)

assert mapping_match_621ec, (
    "Retained-sensor feature mapping does not "
    "match the frozen mapping."
)


# ============================================================
# 4. Reconstruct raw validation sequences
# ============================================================

scaled_flat_621ec = (
    np.asarray(
        X_val_621ec,
        dtype=np.float64
    )
    .reshape(
        -1,
        n_primary_features_621ec
    )
)

raw_flat_621ec = scaler.inverse_transform(
    scaled_flat_621ec
)

raw_primary_621ec = raw_flat_621ec.reshape(
    n_windows_621ec,
    window_size_621ec,
    n_primary_features_621ec,
)

sensor_windows_621ec = raw_primary_621ec[
    :,
    :,
    sensor_indices_frozen_621ec
]

print("\n--- Raw retained-sensor reconstruction ---")
print(
    "Raw primary shape :",
    raw_primary_621ec.shape
)
print(
    "Sensor-window shape:",
    sensor_windows_621ec.shape
)

expected_sensor_shape_621ec = (
    n_windows_621ec,
    window_size_621ec,
    len(RETAINED_SENSOR_COLUMNS),
)

assert (
    sensor_windows_621ec.shape
    == expected_sensor_shape_621ec
), (
    "Unexpected retained-sensor window shape."
)

assert np.isfinite(
    sensor_windows_621ec
).all(), (
    "Non-finite values found in raw "
    "retained-sensor windows."
)


# ============================================================
# 5. Temporal metrics
# ============================================================

temporal_output_621ec = temporal_metrics_620ha(
    sensor_windows_621ec
)

if not isinstance(
    temporal_output_621ec,
    (tuple, list)
):
    raise RuntimeError(
        "temporal_metrics_620ha did not return "
        "the expected tuple/list."
    )

if len(temporal_output_621ec) != 3:
    raise RuntimeError(
        "temporal_metrics_620ha returned "
        f"{len(temporal_output_621ec)} items; "
        "expected 3."
    )

mean_abs_621ec = np.asarray(
    temporal_output_621ec[0],
    dtype=np.float64,
)

q95_abs_621ec = np.asarray(
    temporal_output_621ec[1],
    dtype=np.float64,
)

zero_fraction_621ec = np.asarray(
    temporal_output_621ec[2],
    dtype=np.float64,
)

expected_metric_shape_621ec = (
    n_windows_621ec,
    len(RETAINED_SENSOR_COLUMNS),
)

for metric_name_621ec, metric_621ec in [
    ("mean_abs", mean_abs_621ec),
    ("q95_abs", q95_abs_621ec),
    ("zero_fraction", zero_fraction_621ec),
]:
    assert (
        metric_621ec.shape
        == expected_metric_shape_621ec
    ), (
        f"{metric_name_621ec} has an unexpected shape."
    )

    assert np.isfinite(
        metric_621ec
    ).all(), (
        f"{metric_name_621ec} contains "
        "non-finite values."
    )

print("\n--- temporal_metrics_620ha output ---")
print("mean_abs shape      :", mean_abs_621ec.shape)
print("q95_abs shape       :", q95_abs_621ec.shape)
print(
    "zero_fraction shape :",
    zero_fraction_621ec.shape
)


# ============================================================
# 6. Sensor-strength calculation
#
# IMPORTANT CORRECTION:
# sensor_strength_620ha returns:
#
#     item_0 = (windows x sensors) strength matrix
#     item_1 = diagnostic dictionary
#
# We must NOT convert the entire tuple to np.asarray().
# ============================================================

strength_output_621ec = sensor_strength_620ha(
    mean_abs_621ec,
    q95_abs_621ec,
    zero_fraction_621ec,
)

if not isinstance(
    strength_output_621ec,
    (tuple, list)
):
    raise RuntimeError(
        "sensor_strength_620ha did not return "
        "the expected tuple/list."
    )

if len(strength_output_621ec) != 2:
    raise RuntimeError(
        "sensor_strength_620ha returned "
        f"{len(strength_output_621ec)} items; "
        "expected exactly 2."
    )

# ------------------------------------------------------------
# CORRECT item extraction
# ------------------------------------------------------------

sensor_strength_matrix_621ec = np.asarray(
    strength_output_621ec[0],
    dtype=np.float64,
)

sensor_strength_metadata_621ec = (
    strength_output_621ec[1]
)

print("\n--- sensor_strength_620ha output ---")
print(
    "Strength-matrix shape :",
    sensor_strength_matrix_621ec.shape
)
print(
    "Metadata type         :",
    type(
        sensor_strength_metadata_621ec
    ).__name__
)

if isinstance(
    sensor_strength_metadata_621ec,
    dict
):
    print(
        "Metadata entries      :",
        len(sensor_strength_metadata_621ec)
    )

assert (
    sensor_strength_matrix_621ec.shape
    == expected_metric_shape_621ec
), (
    "Sensor-strength matrix has "
    "an unexpected shape."
)

assert np.isfinite(
    sensor_strength_matrix_621ec
).all(), (
    "Sensor-strength matrix contains "
    "non-finite values."
)


# ============================================================
# 7. Aggregate candidate temporal scores
# ============================================================

aggregate_output_621ec = (
    aggregate_candidates_620ha(
        sensor_strength_matrix_621ec
    )
)

candidate_names_621ec = [
    "max_sensor",
    "top2_mean",
    "top3_mean",
    "hybrid_max_top3",
]


# ============================================================
# 8. Safe candidate-score extractor
# ============================================================

def extract_candidate_scores_621ec(
    obj_621ec,
    label_621ec,
):
    """
    Extract the four temporal candidate vectors without
    silently assuming an unknown structure.
    """

    # --------------------------------------------------------
    # DataFrame
    # --------------------------------------------------------

    if isinstance(obj_621ec, pd.DataFrame):

        if all(
            candidate_621ec in obj_621ec.columns
            for candidate_621ec
            in candidate_names_621ec
        ):
            return {
                candidate_621ec:
                    np.asarray(
                        obj_621ec[candidate_621ec],
                        dtype=np.float64,
                    ).reshape(-1)

                for candidate_621ec
                in candidate_names_621ec
            }

    # --------------------------------------------------------
    # Dictionary
    # --------------------------------------------------------

    if isinstance(obj_621ec, dict):

        if all(
            candidate_621ec in obj_621ec
            for candidate_621ec
            in candidate_names_621ec
        ):
            return {
                candidate_621ec:
                    np.asarray(
                        obj_621ec[candidate_621ec],
                        dtype=np.float64,
                    ).reshape(-1)

                for candidate_621ec
                in candidate_names_621ec
            }

        # Search nested objects safely.
        for nested_value_621ec in obj_621ec.values():

            if isinstance(
                nested_value_621ec,
                (
                    dict,
                    pd.DataFrame,
                    tuple,
                    list,
                ),
            ):
                try:
                    return (
                        extract_candidate_scores_621ec(
                            nested_value_621ec,
                            label_621ec,
                        )
                    )
                except RuntimeError:
                    pass

    # --------------------------------------------------------
    # Tuple/list
    # --------------------------------------------------------

    if isinstance(
        obj_621ec,
        (tuple, list),
    ):

        for nested_value_621ec in obj_621ec:

            if isinstance(
                nested_value_621ec,
                (
                    dict,
                    pd.DataFrame,
                    tuple,
                    list,
                ),
            ):
                try:
                    return (
                        extract_candidate_scores_621ec(
                            nested_value_621ec,
                            label_621ec,
                        )
                    )
                except RuntimeError:
                    pass

    raise RuntimeError(
        f"Could not safely extract the four candidate "
        f"scores from {label_621ec}. "
        "No assumption about candidate ordering was made."
    )


computed_candidates_621ec = (
    extract_candidate_scores_621ec(
        aggregate_output_621ec,
        "aggregate_candidates_620ha output",
    )
)

frozen_candidates_621ec = (
    extract_candidate_scores_621ec(
        val_candidate_scores_620ha,
        "val_candidate_scores_620ha",
    )
)


# ============================================================
# 9. Structural and numerical comparison
# ============================================================

comparison_rows_621ec = []

for candidate_621ec in candidate_names_621ec:

    computed_621ec = np.asarray(
        computed_candidates_621ec[
            candidate_621ec
        ],
        dtype=np.float64,
    ).reshape(-1)

    frozen_621ec = np.asarray(
        frozen_candidates_621ec[
            candidate_621ec
        ],
        dtype=np.float64,
    ).reshape(-1)

    if (
        len(computed_621ec)
        != n_windows_621ec
    ):
        raise RuntimeError(
            f"{candidate_621ec}: computed score "
            "length is incorrect."
        )

    if (
        len(frozen_621ec)
        != n_windows_621ec
    ):
        raise RuntimeError(
            f"{candidate_621ec}: frozen score "
            "length is incorrect."
        )

    if not np.isfinite(
        computed_621ec
    ).all():
        raise RuntimeError(
            f"{candidate_621ec}: computed scores "
            "contain non-finite values."
        )

    if not np.isfinite(
        frozen_621ec
    ).all():
        raise RuntimeError(
            f"{candidate_621ec}: frozen scores "
            "contain non-finite values."
        )

    abs_diff_621ec = np.abs(
        computed_621ec - frozen_621ec
    )

    comparison_rows_621ec.append(
        {
            "candidate": candidate_621ec,
            "computed_mean":
                float(
                    np.mean(computed_621ec)
                ),
            "frozen_mean":
                float(
                    np.mean(frozen_621ec)
                ),
            "mean_absolute_error":
                float(
                    np.mean(abs_diff_621ec)
                ),
            "maximum_absolute_error":
                float(
                    np.max(abs_diff_621ec)
                ),
        }
    )

comparison_621ec = pd.DataFrame(
    comparison_rows_621ec
)

print(
    "\n--- Recomputed vs frozen "
    "CLEAN-validation candidate scores ---"
)

display(
    comparison_621ec.round(12)
)


# ============================================================
# 10. Frozen selected temporal signal agreement
# ============================================================

selected_candidate_621ec = str(
    FROZEN_TEMPORAL_CANDIDATE_620I
)

if (
    selected_candidate_621ec
    not in computed_candidates_621ec
):
    raise RuntimeError(
        "Frozen temporal candidate is absent "
        "from recomputed candidates."
    )

selected_score_621ec = np.asarray(
    computed_candidates_621ec[
        selected_candidate_621ec
    ],
    dtype=np.float64,
).reshape(-1)

frozen_selected_score_621ec = np.asarray(
    frozen_candidates_621ec[
        selected_candidate_621ec
    ],
    dtype=np.float64,
).reshape(-1)

selected_error_621ec = np.abs(
    selected_score_621ec
    - frozen_selected_score_621ec
)

threshold_621ec = float(
    FROZEN_TEMPORAL_THRESHOLD_620I
)

computed_event_pct_621ec = float(
    100.0
    * np.mean(
        selected_score_621ec
        > threshold_621ec
    )
)

frozen_event_pct_621ec = float(
    100.0
    * np.mean(
        frozen_selected_score_621ec
        > threshold_621ec
    )
)

print(
    "\n--- Frozen temporal-signal agreement ---"
)

print(
    "Frozen candidate          :",
    selected_candidate_621ec
)

print(
    "Frozen threshold          :",
    threshold_621ec
)

print(
    "Maximum score difference  :",
    float(
        np.max(
            selected_error_621ec
        )
    )
)

print(
    "Mean score difference     :",
    float(
        np.mean(
            selected_error_621ec
        )
    )
)

print(
    "Recomputed event rate (%) :",
    computed_event_pct_621ec
)

print(
    "Frozen event rate (%)     :",
    frozen_event_pct_621ec
)

print(
    "Event-rate difference (pp):",
    float(
        computed_event_pct_621ec
        - frozen_event_pct_621ec
    )
)


# ============================================================
# 11. Smoke-check decision
# ============================================================

max_candidate_error_621ec = float(
    comparison_621ec[
        "maximum_absolute_error"
    ].max()
)

event_rate_difference_621ec = abs(
    computed_event_pct_621ec
    - frozen_event_pct_621ec
)

# The important requirement is practical numerical
# agreement with the already-frozen validation pathway.
score_agreement_621ec = (
    max_candidate_error_621ec
    <= 1e-8
)

event_agreement_621ec = (
    event_rate_difference_621ec
    <= 1e-9
)

print(
    "\n=============================================="
)
print(
    "STEP 6.21E-C READINESS SUMMARY"
)
print(
    "=============================================="
)

print(
    "Validation windows             :",
    n_windows_621ec
)

print(
    "Window size                    :",
    window_size_621ec
)

print(
    "Retained sensors               :",
    len(RETAINED_SENSOR_COLUMNS)
)

print(
    "Sensor-strength shape          :",
    sensor_strength_matrix_621ec.shape
)

print(
    "Frozen temporal candidate      :",
    selected_candidate_621ec
)

print(
    "Frozen temporal threshold      :",
    threshold_621ec
)

print(
    "Maximum candidate-score error  :",
    max_candidate_error_621ec
)

print(
    "Score pathway agreement        :",
    score_agreement_621ec
)

print(
    "Event-rate agreement           :",
    event_agreement_621ec
)

print(
    "Corruption generated           : False"
)

print(
    "Fusion experiment executed     : False"
)

print(
    "Frozen artifact modified       : False"
)


if not score_agreement_621ec:
    raise RuntimeError(
        "Temporal pathway did not reproduce "
        "the frozen validation candidate scores "
        "within numerical tolerance. "
        "Do not proceed to fusion."
    )

if not event_agreement_621ec:
    raise RuntimeError(
        "Temporal event pathway does not match "
        "the frozen validation event rate. "
        "Do not proceed to fusion."
    )


print(
    "\nStep 6.21E-C completed successfully."
)

print(
    "Exact temporal CLEAN-validation pathway "
    "has been reproduced."
)

print(
    "The PCA and temporal pathways are now ready "
    "for frozen-corruption fusion evaluation."
)

In [ ]:
# ============================================================
# Step 6.21E-C1
# Inspect exact sensor_strength_620ha return structure
# ============================================================

import numpy as np
import pandas as pd
import inspect

print("\n=== Step 6.21E-C1 — sensor_strength return diagnostic ===")

# Re-use the temporal metrics already created before the failure.
required_621ec1 = [
    "mean_abs_621ec",
    "q95_abs_621ec",
    "zero_fraction_621ec",
    "sensor_strength_620ha",
    "aggregate_candidates_620ha",
]

for name_621ec1 in required_621ec1:
    print(
        f"{name_621ec1:<30}: "
        f"{name_621ec1 in globals()}"
    )

missing_621ec1 = [
    name_621ec1
    for name_621ec1 in required_621ec1
    if name_621ec1 not in globals()
]

if missing_621ec1:
    raise RuntimeError(
        "Missing objects: "
        + ", ".join(missing_621ec1)
    )


# ------------------------------------------------------------
# Exact sensor-strength output
# ------------------------------------------------------------

strength_output_621ec1 = sensor_strength_620ha(
    mean_abs_621ec,
    q95_abs_621ec,
    zero_fraction_621ec,
)

print("\n--- sensor_strength_620ha return ---")
print("Python type :", type(strength_output_621ec1).__name__)


def describe_item_621ec1(label, value):
    print(f"\n{label}")
    print("  type :", type(value).__name__)

    if isinstance(value, pd.DataFrame):
        print("  shape:", value.shape)
        print("  columns:", list(value.columns))

    elif isinstance(value, pd.Series):
        print("  shape:", value.shape)
        print("  name :", value.name)

    else:
        try:
            arr = np.asarray(value)
            print("  shape:", arr.shape)
            print("  dtype:", arr.dtype)

            if np.issubdtype(arr.dtype, np.number):
                print("  finite:", bool(np.isfinite(arr).all()))

        except Exception as exc:
            print("  array conversion unavailable:", repr(exc))


if isinstance(strength_output_621ec1, dict):

    print("Dictionary keys:", list(strength_output_621ec1.keys()))

    for key, value in strength_output_621ec1.items():
        describe_item_621ec1(
            f"key={key!r}",
            value
        )

elif isinstance(strength_output_621ec1, (tuple, list)):

    print("Number of returned items:", len(strength_output_621ec1))

    for idx, value in enumerate(strength_output_621ec1):
        describe_item_621ec1(
            f"item_{idx}",
            value
        )

else:

    describe_item_621ec1(
        "single return object",
        strength_output_621ec1
    )


# ------------------------------------------------------------
# Exact aggregate helper signature
# ------------------------------------------------------------

print("\n--- aggregate_candidates_620ha ---")
print(
    "Signature:",
    inspect.signature(
        aggregate_candidates_620ha
    )
)


# ------------------------------------------------------------
# Try to show only return-related source lines
# ------------------------------------------------------------

print("\n--- sensor_strength_620ha return statement(s) ---")

try:
    source_621ec1 = inspect.getsource(
        sensor_strength_620ha
    )

    source_lines_621ec1 = source_621ec1.splitlines()

    return_lines_621ec1 = [
        line.strip()
        for line in source_lines_621ec1
        if "return" in line
    ]

    if return_lines_621ec1:
        for line in return_lines_621ec1:
            print(line)
    else:
        print("<no return line detected>")

except Exception as exc:
    print(
        "<source unavailable:",
        type(exc).__name__,
        ">"
    )


print("\n=== END Step 6.21E-C1 ===")
print("Nothing was modified.")

In [ ]:
# ============================================================
# Step 6.21E-D1
# Five-family end-to-end PCA / Temporal fusion smoke test
#
# Purpose:
#   Verify the complete corrupted-data fusion pathway on one
#   frozen scenario from each corruption family BEFORE running
#   the full 315-scenario experiment.
#
# Nothing is frozen or written to disk in this step.
# ============================================================

import numpy as np
import pandas as pd

print(
    "\n=== Step 6.21E-D1 — "
    "Five-family end-to-end fusion smoke test ==="
)


# ============================================================
# 1. Required objects
# ============================================================

required_objects_621ed1 = [
    "scenario_manifests_620gc",
    "apply_frozen_corruption_620gc",
    "val_raw_base_620gc",
    "X_val_clean",
    "scaler",
    "PRIMARY_INPUT_COLUMNS",
    "RETAINED_SENSOR_COLUMNS",
    "SENSOR_INPUT_INDICES",
    "sensor_pca",
    "calculate_pca_window_quality",
    "temporal_metrics_620ha",
    "sensor_strength_620ha",
    "aggregate_candidates_620ha",
    "val_candidate_scores_620ha",
    "PCA_P95_621D",
    "TEMPORAL_P95_621D",
    "fusion_thresholds_621d",
    "FROZEN_TEMPORAL_CANDIDATE_620I",
]

print("\n--- Required objects ---")

for name_621ed1 in required_objects_621ed1:
    print(
        f"{name_621ed1:<36}: "
        f"{name_621ed1 in globals()}"
    )

missing_621ed1 = [
    name_621ed1
    for name_621ed1 in required_objects_621ed1
    if name_621ed1 not in globals()
]

if missing_621ed1:
    raise RuntimeError(
        "Missing required objects: "
        + ", ".join(missing_621ed1)
    )

print("\nAll required objects are available.")


# ============================================================
# 2. Fixed experiment constants
# ============================================================

WINDOW_SIZE_621ED1 = int(X_val_clean.shape[1])

FUSION_CANDIDATES_621ED1 = [
    "max_fusion",
    "mean_fusion",
    "rms_fusion",
    "geometric_fusion",
]

CORRUPTION_FAMILIES_621ED1 = [
    "bias",
    "noise",
    "drift",
    "dropout",
    "stuck",
]

pca_p95_621ed1 = float(PCA_P95_621D)
temporal_p95_621ed1 = float(TEMPORAL_P95_621D)

if pca_p95_621ed1 <= 0:
    raise RuntimeError(
        "Frozen PCA development p95 must be positive."
    )

if temporal_p95_621ed1 <= 0:
    raise RuntimeError(
        "Frozen temporal development p95 must be positive."
    )

print("\n--- Frozen pathway calibration ---")
print("PCA development p95      :", pca_p95_621ed1)
print("Temporal development p95 :", temporal_p95_621ed1)
print(
    "Frozen temporal signal   :",
    FROZEN_TEMPORAL_CANDIDATE_620I,
)


# ============================================================
# 3. Extract frozen fusion thresholds safely
# ============================================================

def extract_fusion_thresholds_621ed1(obj_621ed1):

    thresholds_621ed1 = {}

    if isinstance(obj_621ed1, pd.DataFrame):

        if (
            "candidate" in obj_621ed1.columns
            and
            "development_p95_threshold"
            in obj_621ed1.columns
        ):
            for _, row_621ed1 in obj_621ed1.iterrows():
                thresholds_621ed1[
                    str(row_621ed1["candidate"])
                ] = float(
                    row_621ed1[
                        "development_p95_threshold"
                    ]
                )

        elif (
            obj_621ed1.index.is_unique
            and
            "development_p95_threshold"
            in obj_621ed1.columns
        ):
            for candidate_621ed1 in obj_621ed1.index:
                thresholds_621ed1[
                    str(candidate_621ed1)
                ] = float(
                    obj_621ed1.loc[
                        candidate_621ed1,
                        "development_p95_threshold",
                    ]
                )

    elif isinstance(obj_621ed1, dict):

        for candidate_621ed1 in FUSION_CANDIDATES_621ED1:

            if candidate_621ed1 in obj_621ed1:

                value_621ed1 = obj_621ed1[
                    candidate_621ed1
                ]

                if np.isscalar(value_621ed1):
                    thresholds_621ed1[
                        candidate_621ed1
                    ] = float(value_621ed1)

                elif isinstance(
                    value_621ed1,
                    dict,
                ):
                    for key_621ed1 in [
                        "development_p95_threshold",
                        "threshold",
                        "p95",
                    ]:
                        if (
                            key_621ed1
                            in value_621ed1
                        ):
                            thresholds_621ed1[
                                candidate_621ed1
                            ] = float(
                                value_621ed1[
                                    key_621ed1
                                ]
                            )
                            break

    return thresholds_621ed1


fusion_threshold_map_621ed1 = (
    extract_fusion_thresholds_621ed1(
        fusion_thresholds_621d
    )
)

print("\n--- Frozen fusion thresholds ---")

for candidate_621ed1 in FUSION_CANDIDATES_621ED1:

    if candidate_621ed1 in fusion_threshold_map_621ed1:
        print(
            f"{candidate_621ed1:<20}: "
            f"{fusion_threshold_map_621ed1[candidate_621ed1]}"
        )
    else:
        print(
            f"{candidate_621ed1:<20}: "
            "<not extracted>"
        )

if set(FUSION_CANDIDATES_621ED1) != set(
    fusion_threshold_map_621ed1.keys()
):
    raise RuntimeError(
        "Could not safely recover all four frozen "
        "fusion thresholds."
    )


# ============================================================
# 4. Window builder
#
# Raw validation data -> scaler -> engine-safe 30-cycle windows
# ============================================================

def build_scaled_windows_621ed1(
    raw_df_621ed1,
):

    if not isinstance(
        raw_df_621ed1,
        pd.DataFrame,
    ):
        raise TypeError(
            "Expected a pandas DataFrame."
        )

    required_columns_621ed1 = (
        ["unit_id", "cycle"]
        + list(PRIMARY_INPUT_COLUMNS)
    )

    missing_columns_621ed1 = [
        column_621ed1
        for column_621ed1
        in required_columns_621ed1
        if column_621ed1
        not in raw_df_621ed1.columns
    ]

    if missing_columns_621ed1:
        raise RuntimeError(
            "Raw validation DataFrame is missing: "
            + ", ".join(missing_columns_621ed1)
        )

    ordered_df_621ed1 = (
        raw_df_621ed1
        .sort_values(
            ["unit_id", "cycle"]
        )
        .reset_index(drop=True)
    )

    windows_621ed1 = []

    for _, engine_df_621ed1 in (
        ordered_df_621ed1.groupby(
            "unit_id",
            sort=True,
        )
    ):

        engine_values_621ed1 = (
            engine_df_621ed1[
                PRIMARY_INPUT_COLUMNS
            ]
            .to_numpy(
                dtype=np.float64
            )
        )

        if not np.isfinite(
            engine_values_621ed1
        ).all():
            raise RuntimeError(
                "Non-finite value found before scaling."
            )

        engine_scaled_621ed1 = (
            scaler.transform(
                engine_values_621ed1
            )
        )

        engine_rows_621ed1 = int(
            engine_scaled_621ed1.shape[0]
        )

        if (
            engine_rows_621ed1
            < WINDOW_SIZE_621ED1
        ):
            continue

        for end_621ed1 in range(
            WINDOW_SIZE_621ED1,
            engine_rows_621ed1 + 1,
        ):

            start_621ed1 = (
                end_621ed1
                - WINDOW_SIZE_621ED1
            )

            windows_621ed1.append(
                engine_scaled_621ed1[
                    start_621ed1:end_621ed1
                ]
            )

    if len(windows_621ed1) == 0:
        raise RuntimeError(
            "No sequence windows were constructed."
        )

    result_621ed1 = np.stack(
        windows_621ed1,
        axis=0,
    ).astype(
        np.float64,
        copy=False,
    )

    return result_621ed1


# ============================================================
# 5. Verify CLEAN sequence reconstruction first
# ============================================================

clean_rebuilt_621ed1 = (
    build_scaled_windows_621ed1(
        val_raw_base_620gc
    )
)

clean_reference_621ed1 = np.asarray(
    X_val_clean,
    dtype=np.float64,
)

print("\n--- CLEAN sequence reconstruction ---")
print(
    "Rebuilt shape   :",
    clean_rebuilt_621ed1.shape
)
print(
    "Reference shape :",
    clean_reference_621ed1.shape
)

if (
    clean_rebuilt_621ed1.shape
    != clean_reference_621ed1.shape
):
    raise RuntimeError(
        "Rebuilt clean validation sequence shape "
        "does not match X_val_clean."
    )

clean_sequence_error_621ed1 = float(
    np.max(
        np.abs(
            clean_rebuilt_621ed1
            - clean_reference_621ed1
        )
    )
)

print(
    "Maximum clean sequence difference:",
    clean_sequence_error_621ed1
)

# Float32 frozen sequence versus float64 reconstruction
# may differ very slightly.
if clean_sequence_error_621ed1 > 1e-5:
    raise RuntimeError(
        "Clean sequence reconstruction differs "
        "more than expected. Do not proceed."
    )

print(
    "Clean validation sequence reconstruction verified."
)


# ============================================================
# 6. Temporal candidate extractor
# ============================================================

TEMPORAL_CANDIDATES_621ED1 = [
    "max_sensor",
    "top2_mean",
    "top3_mean",
    "hybrid_max_top3",
]


def extract_temporal_candidates_621ed1(
    obj_621ed1,
):

    if isinstance(obj_621ed1, pd.DataFrame):

        if all(
            name_621ed1
            in obj_621ed1.columns
            for name_621ed1
            in TEMPORAL_CANDIDATES_621ED1
        ):
            return {
                name_621ed1:
                np.asarray(
                    obj_621ed1[name_621ed1],
                    dtype=np.float64,
                ).reshape(-1)

                for name_621ed1
                in TEMPORAL_CANDIDATES_621ED1
            }

    if isinstance(obj_621ed1, dict):

        if all(
            name_621ed1 in obj_621ed1
            for name_621ed1
            in TEMPORAL_CANDIDATES_621ED1
        ):
            return {
                name_621ed1:
                np.asarray(
                    obj_621ed1[name_621ed1],
                    dtype=np.float64,
                ).reshape(-1)

                for name_621ed1
                in TEMPORAL_CANDIDATES_621ED1
            }

        for value_621ed1 in obj_621ed1.values():

            if isinstance(
                value_621ed1,
                (
                    dict,
                    pd.DataFrame,
                    tuple,
                    list,
                ),
            ):
                try:
                    return (
                        extract_temporal_candidates_621ed1(
                            value_621ed1
                        )
                    )
                except RuntimeError:
                    pass

    if isinstance(
        obj_621ed1,
        (tuple, list),
    ):

        for value_621ed1 in obj_621ed1:

            if isinstance(
                value_621ed1,
                (
                    dict,
                    pd.DataFrame,
                    tuple,
                    list,
                ),
            ):
                try:
                    return (
                        extract_temporal_candidates_621ed1(
                            value_621ed1
                        )
                    )
                except RuntimeError:
                    pass

    raise RuntimeError(
        "Could not safely extract temporal "
        "candidate score arrays."
    )


# ============================================================
# 7. Complete PCA pathway
# ============================================================

def compute_pca_score_621ed1(
    X_scaled_621ed1,
):

    pca_output_621ed1 = (
        calculate_pca_window_quality(
            X_scaled_621ed1,
            SENSOR_INPUT_INDICES,
            sensor_pca,
        )
    )

    if not isinstance(
        pca_output_621ed1,
        (tuple, list),
    ):
        raise RuntimeError(
            "Unexpected PCA scoring output."
        )

    if len(pca_output_621ed1) < 3:
        raise RuntimeError(
            "PCA scoring output has fewer than "
            "three returned arrays."
        )

    # Verified previously:
    # item_0 = row RMSE
    # item_1 = mean RMSE
    # item_2 = max RMSE

    pca_max_621ed1 = np.asarray(
        pca_output_621ed1[2],
        dtype=np.float64,
    ).reshape(-1)

    if (
        len(pca_max_621ed1)
        != X_scaled_621ed1.shape[0]
    ):
        raise RuntimeError(
            "Unexpected PCA max-RMSE length."
        )

    if not np.isfinite(
        pca_max_621ed1
    ).all():
        raise RuntimeError(
            "PCA score contains non-finite values."
        )

    return pca_max_621ed1


# ============================================================
# 8. Complete temporal pathway
# ============================================================

def compute_temporal_score_621ed1(
    X_scaled_621ed1,
):

    n_windows_local_621ed1 = int(
        X_scaled_621ed1.shape[0]
    )

    n_features_local_621ed1 = int(
        X_scaled_621ed1.shape[2]
    )

    raw_flat_local_621ed1 = (
        scaler.inverse_transform(
            np.asarray(
                X_scaled_621ed1,
                dtype=np.float64,
            ).reshape(
                -1,
                n_features_local_621ed1,
            )
        )
    )

    raw_primary_local_621ed1 = (
        raw_flat_local_621ed1.reshape(
            n_windows_local_621ed1,
            WINDOW_SIZE_621ED1,
            n_features_local_621ed1,
        )
    )

    sensor_windows_local_621ed1 = (
        raw_primary_local_621ed1[
            :,
            :,
            SENSOR_INPUT_INDICES,
        ]
    )

    temporal_metrics_local_621ed1 = (
        temporal_metrics_620ha(
            sensor_windows_local_621ed1
        )
    )

    if (
        not isinstance(
            temporal_metrics_local_621ed1,
            (tuple, list),
        )
        or
        len(
            temporal_metrics_local_621ed1
        ) != 3
    ):
        raise RuntimeError(
            "Unexpected temporal_metrics output."
        )

    mean_abs_local_621ed1 = np.asarray(
        temporal_metrics_local_621ed1[0],
        dtype=np.float64,
    )

    q95_abs_local_621ed1 = np.asarray(
        temporal_metrics_local_621ed1[1],
        dtype=np.float64,
    )

    zero_fraction_local_621ed1 = np.asarray(
        temporal_metrics_local_621ed1[2],
        dtype=np.float64,
    )

    strength_output_local_621ed1 = (
        sensor_strength_620ha(
            mean_abs_local_621ed1,
            q95_abs_local_621ed1,
            zero_fraction_local_621ed1,
        )
    )

    if (
        not isinstance(
            strength_output_local_621ed1,
            (tuple, list),
        )
        or
        len(
            strength_output_local_621ed1
        ) != 2
    ):
        raise RuntimeError(
            "Unexpected sensor_strength output."
        )

    strength_matrix_local_621ed1 = (
        np.asarray(
            strength_output_local_621ed1[0],
            dtype=np.float64,
        )
    )

    aggregate_local_621ed1 = (
        aggregate_candidates_620ha(
            strength_matrix_local_621ed1
        )
    )

    candidate_scores_local_621ed1 = (
        extract_temporal_candidates_621ed1(
            aggregate_local_621ed1
        )
    )

    frozen_name_local_621ed1 = str(
        FROZEN_TEMPORAL_CANDIDATE_620I
    )

    temporal_score_local_621ed1 = (
        np.asarray(
            candidate_scores_local_621ed1[
                frozen_name_local_621ed1
            ],
            dtype=np.float64,
        )
        .reshape(-1)
    )

    if (
        len(temporal_score_local_621ed1)
        != n_windows_local_621ed1
    ):
        raise RuntimeError(
            "Unexpected temporal-score length."
        )

    if not np.isfinite(
        temporal_score_local_621ed1
    ).all():
        raise RuntimeError(
            "Temporal score contains non-finite values."
        )

    return temporal_score_local_621ed1


# ============================================================
# 9. Fusion calculation
# ============================================================

def compute_fusion_scores_621ed1(
    pca_score_621ed1,
    temporal_score_621ed1,
):

    pca_norm_621ed1 = (
        np.asarray(
            pca_score_621ed1,
            dtype=np.float64,
        )
        / pca_p95_621ed1
    )

    temporal_norm_621ed1 = (
        np.asarray(
            temporal_score_621ed1,
            dtype=np.float64,
        )
        / temporal_p95_621ed1
    )

    # Both pathways are non-negative anomaly quantities.
    pca_norm_621ed1 = np.maximum(
        pca_norm_621ed1,
        0.0,
    )

    temporal_norm_621ed1 = np.maximum(
        temporal_norm_621ed1,
        0.0,
    )

    fusion_621ed1 = {
        "max_fusion":
            np.maximum(
                pca_norm_621ed1,
                temporal_norm_621ed1,
            ),

        "mean_fusion":
            (
                pca_norm_621ed1
                + temporal_norm_621ed1
            ) / 2.0,

        "rms_fusion":
            np.sqrt(
                (
                    pca_norm_621ed1**2
                    + temporal_norm_621ed1**2
                ) / 2.0
            ),

        "geometric_fusion":
            np.sqrt(
                pca_norm_621ed1
                * temporal_norm_621ed1
            ),
    }

    for (
        fusion_name_621ed1,
        fusion_values_621ed1,
    ) in fusion_621ed1.items():

        if not np.isfinite(
            fusion_values_621ed1
        ).all():
            raise RuntimeError(
                f"{fusion_name_621ed1} contains "
                "non-finite values."
            )

    return fusion_621ed1


# ============================================================
# 10. Helper to safely obtain corrupted DataFrame
# ============================================================

def extract_corrupted_dataframe_621ed1(
    corruption_output_621ed1,
):

    if isinstance(
        corruption_output_621ed1,
        pd.DataFrame,
    ):
        return corruption_output_621ed1

    if isinstance(
        corruption_output_621ed1,
        (tuple, list),
    ):
        dataframe_items_621ed1 = [
            item_621ed1
            for item_621ed1
            in corruption_output_621ed1
            if isinstance(
                item_621ed1,
                pd.DataFrame,
            )
        ]

        if len(
            dataframe_items_621ed1
        ) == 1:
            return dataframe_items_621ed1[0]

    raise RuntimeError(
        "Could not safely identify the corrupted "
        "validation DataFrame."
    )


# ============================================================
# 11. Execute ONE frozen scenario per corruption family
# ============================================================

smoke_rows_621ed1 = []

print(
    "\n--- Executing five-family smoke test ---"
)

for family_621ed1 in CORRUPTION_FAMILIES_621ED1:

    manifest_621ed1 = (
        scenario_manifests_620gc[
            family_621ed1
        ]
    )

    if not isinstance(
        manifest_621ed1,
        pd.DataFrame,
    ):
        raise RuntimeError(
            f"{family_621ed1} manifest is not "
            "a DataFrame."
        )

    if len(manifest_621ed1) == 0:
        raise RuntimeError(
            f"{family_621ed1} manifest is empty."
        )

    # Use first frozen scenario only.
    scenario_row_621ed1 = (
        manifest_621ed1.iloc[0]
    )

    corruption_output_621ed1 = (
        apply_frozen_corruption_620gc(
            family_621ed1,
            scenario_row_621ed1,
        )
    )

    corrupted_df_621ed1 = (
        extract_corrupted_dataframe_621ed1(
            corruption_output_621ed1
        )
    )

    X_corrupted_621ed1 = (
        build_scaled_windows_621ed1(
            corrupted_df_621ed1
        )
    )

    if (
        X_corrupted_621ed1.shape
        != clean_reference_621ed1.shape
    ):
        raise RuntimeError(
            f"{family_621ed1}: corrupted sequence "
            "shape differs from clean validation."
        )

    pca_score_family_621ed1 = (
        compute_pca_score_621ed1(
            X_corrupted_621ed1
        )
    )

    temporal_score_family_621ed1 = (
        compute_temporal_score_621ed1(
            X_corrupted_621ed1
        )
    )

    fusion_family_621ed1 = (
        compute_fusion_scores_621ed1(
            pca_score_family_621ed1,
            temporal_score_family_621ed1,
        )
    )

    row_result_621ed1 = {
        "corruption":
            family_621ed1,

        "sensor":
            scenario_row_621ed1.get(
                "sensor",
                "<unknown>",
            ),

        "severity":
            scenario_row_621ed1.get(
                "severity",
                "<unknown>",
            ),

        "windows":
            int(
                X_corrupted_621ed1.shape[0]
            ),

        "pca_mean":
            float(
                np.mean(
                    pca_score_family_621ed1
                )
            ),

        "temporal_mean":
            float(
                np.mean(
                    temporal_score_family_621ed1
                )
            ),
    }

    for fusion_name_621ed1 in (
        FUSION_CANDIDATES_621ED1
    ):

        fusion_values_621ed1 = (
            fusion_family_621ed1[
                fusion_name_621ed1
            ]
        )

        fusion_threshold_621ed1 = (
            fusion_threshold_map_621ed1[
                fusion_name_621ed1
            ]
        )

        row_result_621ed1[
            fusion_name_621ed1 + "_mean"
        ] = float(
            np.mean(
                fusion_values_621ed1
            )
        )

        row_result_621ed1[
            fusion_name_621ed1 + "_event_pct"
        ] = float(
            100.0
            * np.mean(
                fusion_values_621ed1
                > fusion_threshold_621ed1
            )
        )

    smoke_rows_621ed1.append(
        row_result_621ed1
    )

    print(
        f"{family_621ed1:<8} : "
        f"PASS | windows="
        f"{X_corrupted_621ed1.shape[0]}"
    )


smoke_summary_621ed1 = pd.DataFrame(
    smoke_rows_621ed1
)


# ============================================================
# 12. Compact smoke-test summary
# ============================================================

compact_columns_621ed1 = [
    "corruption",
    "sensor",
    "severity",
    "windows",
    "pca_mean",
    "temporal_mean",
    "max_fusion_event_pct",
    "mean_fusion_event_pct",
    "rms_fusion_event_pct",
    "geometric_fusion_event_pct",
]

print(
    "\n--- Five-family fusion smoke-test summary ---"
)

display(
    smoke_summary_621ed1[
        compact_columns_621ed1
    ].round(6)
)


# ============================================================
# 13. Final integrity checks
# ============================================================

expected_family_count_621ed1 = 5

family_count_ok_621ed1 = (
    len(smoke_summary_621ed1)
    == expected_family_count_621ed1
)

window_count_ok_621ed1 = bool(
    (
        smoke_summary_621ed1[
            "windows"
        ]
        == X_val_clean.shape[0]
    ).all()
)

numeric_columns_621ed1 = (
    smoke_summary_621ed1.select_dtypes(
        include=[np.number]
    )
)

all_finite_621ed1 = bool(
    np.isfinite(
        numeric_columns_621ed1.to_numpy(
            dtype=np.float64
        )
    ).all()
)

print(
    "\n=============================================="
)
print(
    "STEP 6.21E-D1 READINESS SUMMARY"
)
print(
    "=============================================="
)

print(
    "Corruption families tested   :",
    len(smoke_summary_621ed1)
)

print(
    "All five families present    :",
    family_count_ok_621ed1
)

print(
    "Window counts preserved      :",
    window_count_ok_621ed1
)

print(
    "All numerical outputs finite :",
    all_finite_621ed1
)

print(
    "Clean reconstruction error   :",
    clean_sequence_error_621ed1
)

print(
    "Fusion candidates tested     :",
    FUSION_CANDIDATES_621ED1
)

print(
    "Frozen artifacts modified    : False"
)

print(
    "Files written                : False"
)


if not family_count_ok_621ed1:
    raise RuntimeError(
        "Not all five corruption families "
        "completed the smoke test."
    )

if not window_count_ok_621ed1:
    raise RuntimeError(
        "Window counts were not preserved."
    )

if not all_finite_621ed1:
    raise RuntimeError(
        "Non-finite fusion results detected."
    )


print(
    "\nStep 6.21E-D1 completed successfully."
)

print(
    "The complete PCA + temporal fusion pathway "
    "worked on one frozen scenario from each family."
)

print(
    "The notebook is ready for the full "
    "315-scenario fusion evaluation."
)

In [ ]:
# ============================================================
# Step 6.21E-D2
# Full 315-scenario PCA / Temporal fusion evaluation
#
# Purpose:
#   Evaluate PCA-only, Temporal-only, and all four calibrated
#   fusion candidates across every frozen corruption scenario.
#
# Important:
#   - Development-calibrated thresholds remain unchanged.
#   - CLEAN validation is not used for threshold tuning.
#   - Bias/drift directions are balanced before comparisons.
#   - Nothing is written to disk.
#   - No final fusion candidate is selected in this step.
# ============================================================

import time
import numpy as np
import pandas as pd

print(
    "\n=== Step 6.21E-D2 — "
    "Full 315-scenario PCA / Temporal fusion evaluation ==="
)


# ============================================================
# 1. Required objects
# ============================================================

required_objects_621ed2 = [
    "scenario_manifests_620gc",
    "apply_frozen_corruption_620gc",
    "val_raw_base_620gc",
    "X_val_clean",
    "scaler",
    "PRIMARY_INPUT_COLUMNS",
    "RETAINED_SENSOR_COLUMNS",
    "SENSOR_INPUT_INDICES",
    "sensor_pca",
    "calculate_pca_window_quality",
    "temporal_metrics_620ha",
    "sensor_strength_620ha",
    "aggregate_candidates_620ha",
    "PCA_P95_621D",
    "TEMPORAL_P95_621D",
    "fusion_thresholds_621d",
    "FROZEN_TEMPORAL_CANDIDATE_620I",
]

print("\n--- Required objects ---")

for name_621ed2 in required_objects_621ed2:
    print(
        f"{name_621ed2:<36}: "
        f"{name_621ed2 in globals()}"
    )

missing_621ed2 = [
    name_621ed2
    for name_621ed2 in required_objects_621ed2
    if name_621ed2 not in globals()
]

if missing_621ed2:
    raise RuntimeError(
        "Missing required objects: "
        + ", ".join(missing_621ed2)
    )

print("\nAll required objects are available.")


# ============================================================
# 2. Fixed definitions
# ============================================================

WINDOW_SIZE_621ED2 = int(
    X_val_clean.shape[1]
)

CORRUPTION_FAMILIES_621ED2 = [
    "bias",
    "noise",
    "drift",
    "dropout",
    "stuck",
]

FUSION_CANDIDATES_621ED2 = [
    "max_fusion",
    "mean_fusion",
    "rms_fusion",
    "geometric_fusion",
]

ALL_SIGNALS_621ED2 = [
    "pca_only",
    "temporal_only",
    "max_fusion",
    "mean_fusion",
    "rms_fusion",
    "geometric_fusion",
]

TEMPORAL_CANDIDATES_621ED2 = [
    "max_sensor",
    "top2_mean",
    "top3_mean",
    "hybrid_max_top3",
]

SEVERITY_ORDER_621ED2 = {
    "low": 0,
    "medium": 1,
    "high": 2,
}

pca_p95_621ed2 = float(
    PCA_P95_621D
)

temporal_p95_621ed2 = float(
    TEMPORAL_P95_621D
)

if pca_p95_621ed2 <= 0:
    raise RuntimeError(
        "PCA development p95 must be positive."
    )

if temporal_p95_621ed2 <= 0:
    raise RuntimeError(
        "Temporal development p95 must be positive."
    )

print("\n--- Development calibration ---")
print("PCA p95      :", pca_p95_621ed2)
print("Temporal p95 :", temporal_p95_621ed2)
print(
    "Temporal candidate:",
    FROZEN_TEMPORAL_CANDIDATE_620I
)


# ============================================================
# 3. Recover calibrated fusion thresholds
# ============================================================

def extract_fusion_thresholds_621ed2(
    obj_621ed2,
):

    thresholds_621ed2 = {}

    if isinstance(
        obj_621ed2,
        pd.DataFrame,
    ):

        if (
            "candidate" in obj_621ed2.columns
            and
            "development_p95_threshold"
            in obj_621ed2.columns
        ):

            for _, row_621ed2 in (
                obj_621ed2.iterrows()
            ):

                thresholds_621ed2[
                    str(
                        row_621ed2[
                            "candidate"
                        ]
                    )
                ] = float(
                    row_621ed2[
                        "development_p95_threshold"
                    ]
                )

        elif (
            "development_p95_threshold"
            in obj_621ed2.columns
        ):

            for idx_621ed2 in (
                obj_621ed2.index
            ):

                thresholds_621ed2[
                    str(idx_621ed2)
                ] = float(
                    obj_621ed2.loc[
                        idx_621ed2,
                        "development_p95_threshold",
                    ]
                )

    elif isinstance(
        obj_621ed2,
        pd.Series,
    ):

        for key_621ed2, value_621ed2 in (
            obj_621ed2.items()
        ):

            if np.isscalar(
                value_621ed2
            ):

                thresholds_621ed2[
                    str(key_621ed2)
                ] = float(
                    value_621ed2
                )

    elif isinstance(
        obj_621ed2,
        dict,
    ):

        for candidate_621ed2 in (
            FUSION_CANDIDATES_621ED2
        ):

            if (
                candidate_621ed2
                not in obj_621ed2
            ):
                continue

            value_621ed2 = obj_621ed2[
                candidate_621ed2
            ]

            if np.isscalar(
                value_621ed2
            ):

                thresholds_621ed2[
                    candidate_621ed2
                ] = float(
                    value_621ed2
                )

            elif isinstance(
                value_621ed2,
                dict,
            ):

                for key_621ed2 in [
                    "development_p95_threshold",
                    "threshold",
                    "p95",
                ]:

                    if (
                        key_621ed2
                        in value_621ed2
                    ):

                        thresholds_621ed2[
                            candidate_621ed2
                        ] = float(
                            value_621ed2[
                                key_621ed2
                            ]
                        )

                        break

    return thresholds_621ed2


fusion_threshold_map_621ed2 = (
    extract_fusion_thresholds_621ed2(
        fusion_thresholds_621d
    )
)

if set(
    FUSION_CANDIDATES_621ED2
) != set(
    fusion_threshold_map_621ed2.keys()
):

    raise RuntimeError(
        "Could not recover all four "
        "development-calibrated fusion thresholds."
    )


signal_threshold_map_621ed2 = {
    "pca_only": 1.0,
    "temporal_only": 1.0,
    **fusion_threshold_map_621ed2,
}

print("\n--- Signal thresholds ---")

for signal_621ed2 in (
    ALL_SIGNALS_621ED2
):

    print(
        f"{signal_621ed2:<20}: "
        f"{signal_threshold_map_621ed2[signal_621ed2]:.12f}"
    )


# ============================================================
# 4. Warning-free raw -> scaled window reconstruction
#
# DataFrames with the original feature names are deliberately
# passed to StandardScaler to avoid sklearn feature-name
# warnings seen in Step 6.21E-D1.
# ============================================================

def build_scaled_windows_621ed2(
    raw_df_621ed2,
):

    if not isinstance(
        raw_df_621ed2,
        pd.DataFrame,
    ):
        raise TypeError(
            "Expected a pandas DataFrame."
        )

    required_columns_621ed2 = (
        ["unit_id", "cycle"]
        + list(
            PRIMARY_INPUT_COLUMNS
        )
    )

    missing_columns_621ed2 = [
        column_621ed2
        for column_621ed2
        in required_columns_621ed2
        if column_621ed2
        not in raw_df_621ed2.columns
    ]

    if missing_columns_621ed2:

        raise RuntimeError(
            "Missing required columns: "
            + ", ".join(
                missing_columns_621ed2
            )
        )

    ordered_df_621ed2 = (
        raw_df_621ed2
        .sort_values(
            ["unit_id", "cycle"]
        )
        .reset_index(drop=True)
    )

    windows_621ed2 = []

    for _, engine_df_621ed2 in (
        ordered_df_621ed2.groupby(
            "unit_id",
            sort=True,
        )
    ):

        engine_features_621ed2 = (
            engine_df_621ed2[
                PRIMARY_INPUT_COLUMNS
            ]
            .astype(np.float64)
        )

        if not np.isfinite(
            engine_features_621ed2
            .to_numpy()
        ).all():

            raise RuntimeError(
                "Non-finite values before scaling."
            )

        # DataFrame is intentionally passed here.
        engine_scaled_621ed2 = (
            scaler.transform(
                engine_features_621ed2
            )
        )

        engine_scaled_621ed2 = (
            np.asarray(
                engine_scaled_621ed2,
                dtype=np.float64,
            )
        )

        n_rows_621ed2 = int(
            engine_scaled_621ed2.shape[0]
        )

        if (
            n_rows_621ed2
            < WINDOW_SIZE_621ED2
        ):
            continue

        for end_621ed2 in range(
            WINDOW_SIZE_621ED2,
            n_rows_621ed2 + 1,
        ):

            start_621ed2 = (
                end_621ed2
                - WINDOW_SIZE_621ED2
            )

            windows_621ed2.append(
                engine_scaled_621ed2[
                    start_621ed2:
                    end_621ed2
                ]
            )

    if len(windows_621ed2) == 0:

        raise RuntimeError(
            "No sequence windows constructed."
        )

    return np.stack(
        windows_621ed2,
        axis=0,
    ).astype(
        np.float64,
        copy=False,
    )


# ============================================================
# 5. Verify clean reconstruction
# ============================================================

clean_rebuilt_621ed2 = (
    build_scaled_windows_621ed2(
        val_raw_base_620gc
    )
)

clean_reference_621ed2 = np.asarray(
    X_val_clean,
    dtype=np.float64,
)

if (
    clean_rebuilt_621ed2.shape
    != clean_reference_621ed2.shape
):

    raise RuntimeError(
        "Clean reconstructed shape does "
        "not match X_val_clean."
    )

clean_reconstruction_error_621ed2 = (
    float(
        np.max(
            np.abs(
                clean_rebuilt_621ed2
                - clean_reference_621ed2
            )
        )
    )
)

print(
    "\nClean reconstruction error:",
    clean_reconstruction_error_621ed2,
)

if (
    clean_reconstruction_error_621ed2
    > 1e-5
):

    raise RuntimeError(
        "Clean reconstruction error is "
        "larger than expected."
    )

print(
    "Clean validation reconstruction verified."
)


# ============================================================
# 6. Corruption-output extractor
# ============================================================

def extract_corrupted_df_621ed2(
    corruption_output_621ed2,
):

    if isinstance(
        corruption_output_621ed2,
        pd.DataFrame,
    ):

        return corruption_output_621ed2

    if isinstance(
        corruption_output_621ed2,
        (tuple, list),
    ):

        df_items_621ed2 = [
            item_621ed2
            for item_621ed2
            in corruption_output_621ed2
            if isinstance(
                item_621ed2,
                pd.DataFrame,
            )
        ]

        if len(
            df_items_621ed2
        ) == 1:

            return df_items_621ed2[0]

    raise RuntimeError(
        "Could not safely identify "
        "the corrupted DataFrame."
    )


# ============================================================
# 7. PCA score pathway
# ============================================================

def compute_pca_score_621ed2(
    X_scaled_621ed2,
):

    pca_output_621ed2 = (
        calculate_pca_window_quality(
            X_scaled_621ed2,
            SENSOR_INPUT_INDICES,
            sensor_pca,
        )
    )

    if (
        not isinstance(
            pca_output_621ed2,
            (tuple, list),
        )
        or
        len(pca_output_621ed2) < 3
    ):

        raise RuntimeError(
            "Unexpected PCA scoring output."
        )

    # Previously verified:
    # item 0 = row RMSE
    # item 1 = mean RMSE
    # item 2 = max RMSE

    pca_score_621ed2 = np.asarray(
        pca_output_621ed2[2],
        dtype=np.float64,
    ).reshape(-1)

    if (
        len(pca_score_621ed2)
        != X_scaled_621ed2.shape[0]
    ):

        raise RuntimeError(
            "PCA score length mismatch."
        )

    if not np.isfinite(
        pca_score_621ed2
    ).all():

        raise RuntimeError(
            "Non-finite PCA score detected."
        )

    return pca_score_621ed2


# ============================================================
# 8. Temporal aggregate extractor
# ============================================================

def extract_temporal_candidates_621ed2(
    obj_621ed2,
):

    if isinstance(
        obj_621ed2,
        pd.DataFrame,
    ):

        if all(
            candidate_621ed2
            in obj_621ed2.columns

            for candidate_621ed2
            in TEMPORAL_CANDIDATES_621ED2
        ):

            return {
                candidate_621ed2:
                np.asarray(
                    obj_621ed2[
                        candidate_621ed2
                    ],
                    dtype=np.float64,
                ).reshape(-1)

                for candidate_621ed2
                in TEMPORAL_CANDIDATES_621ED2
            }

    if isinstance(
        obj_621ed2,
        dict,
    ):

        if all(
            candidate_621ed2
            in obj_621ed2

            for candidate_621ed2
            in TEMPORAL_CANDIDATES_621ED2
        ):

            return {
                candidate_621ed2:
                np.asarray(
                    obj_621ed2[
                        candidate_621ed2
                    ],
                    dtype=np.float64,
                ).reshape(-1)

                for candidate_621ed2
                in TEMPORAL_CANDIDATES_621ED2
            }

        for value_621ed2 in (
            obj_621ed2.values()
        ):

            if isinstance(
                value_621ed2,
                (
                    dict,
                    pd.DataFrame,
                    tuple,
                    list,
                ),
            ):

                try:
                    return (
                        extract_temporal_candidates_621ed2(
                            value_621ed2
                        )
                    )

                except RuntimeError:
                    pass

    if isinstance(
        obj_621ed2,
        (tuple, list),
    ):

        for value_621ed2 in obj_621ed2:

            if isinstance(
                value_621ed2,
                (
                    dict,
                    pd.DataFrame,
                    tuple,
                    list,
                ),
            ):

                try:
                    return (
                        extract_temporal_candidates_621ed2(
                            value_621ed2
                        )
                    )

                except RuntimeError:
                    pass

    raise RuntimeError(
        "Could not safely extract "
        "temporal candidate scores."
    )


# ============================================================
# 9. Temporal score pathway
#
# Feature names are preserved during inverse_transform to
# eliminate the repeated sklearn warnings seen in D1.
# ============================================================

def compute_temporal_score_621ed2(
    X_scaled_621ed2,
):

    n_windows_621ed2 = int(
        X_scaled_621ed2.shape[0]
    )

    n_features_621ed2 = int(
        X_scaled_621ed2.shape[2]
    )

    flat_scaled_621ed2 = (
        np.asarray(
            X_scaled_621ed2,
            dtype=np.float64,
        )
        .reshape(
            -1,
            n_features_621ed2,
        )
    )

    flat_scaled_df_621ed2 = (
        pd.DataFrame(
            flat_scaled_621ed2,
            columns=
            PRIMARY_INPUT_COLUMNS,
        )
    )

    raw_flat_621ed2 = (
        scaler.inverse_transform(
            flat_scaled_df_621ed2
        )
    )

    raw_primary_621ed2 = (
        np.asarray(
            raw_flat_621ed2,
            dtype=np.float64,
        )
        .reshape(
            n_windows_621ed2,
            WINDOW_SIZE_621ED2,
            n_features_621ed2,
        )
    )

    sensor_windows_621ed2 = (
        raw_primary_621ed2[
            :,
            :,
            SENSOR_INPUT_INDICES,
        ]
    )

    temporal_output_621ed2 = (
        temporal_metrics_620ha(
            sensor_windows_621ed2
        )
    )

    if (
        not isinstance(
            temporal_output_621ed2,
            (tuple, list),
        )
        or
        len(temporal_output_621ed2)
        != 3
    ):

        raise RuntimeError(
            "Unexpected temporal_metrics output."
        )

    mean_abs_621ed2 = np.asarray(
        temporal_output_621ed2[0],
        dtype=np.float64,
    )

    q95_abs_621ed2 = np.asarray(
        temporal_output_621ed2[1],
        dtype=np.float64,
    )

    zero_fraction_621ed2 = (
        np.asarray(
            temporal_output_621ed2[2],
            dtype=np.float64,
        )
    )

    strength_output_621ed2 = (
        sensor_strength_620ha(
            mean_abs_621ed2,
            q95_abs_621ed2,
            zero_fraction_621ed2,
        )
    )

    if (
        not isinstance(
            strength_output_621ed2,
            (tuple, list),
        )
        or
        len(strength_output_621ed2)
        != 2
    ):

        raise RuntimeError(
            "Unexpected sensor_strength output."
        )

    strength_matrix_621ed2 = (
        np.asarray(
            strength_output_621ed2[0],
            dtype=np.float64,
        )
    )

    aggregate_output_621ed2 = (
        aggregate_candidates_620ha(
            strength_matrix_621ed2
        )
    )

    temporal_candidates_621ed2 = (
        extract_temporal_candidates_621ed2(
            aggregate_output_621ed2
        )
    )

    frozen_temporal_name_621ed2 = (
        str(
            FROZEN_TEMPORAL_CANDIDATE_620I
        )
    )

    if (
        frozen_temporal_name_621ed2
        not in temporal_candidates_621ed2
    ):

        raise RuntimeError(
            "Frozen temporal candidate "
            "not found in computed scores."
        )

    temporal_score_621ed2 = (
        np.asarray(
            temporal_candidates_621ed2[
                frozen_temporal_name_621ed2
            ],
            dtype=np.float64,
        )
        .reshape(-1)
    )

    if (
        len(temporal_score_621ed2)
        != n_windows_621ed2
    ):

        raise RuntimeError(
            "Temporal score length mismatch."
        )

    if not np.isfinite(
        temporal_score_621ed2
    ).all():

        raise RuntimeError(
            "Non-finite temporal score detected."
        )

    return temporal_score_621ed2


# ============================================================
# 10. Build all six signal arrays
# ============================================================

def compute_all_signals_621ed2(
    X_scaled_621ed2,
):

    pca_raw_621ed2 = (
        compute_pca_score_621ed2(
            X_scaled_621ed2
        )
    )

    temporal_raw_621ed2 = (
        compute_temporal_score_621ed2(
            X_scaled_621ed2
        )
    )

    pca_norm_621ed2 = np.maximum(
        pca_raw_621ed2
        / pca_p95_621ed2,
        0.0,
    )

    temporal_norm_621ed2 = (
        np.maximum(
            temporal_raw_621ed2
            / temporal_p95_621ed2,
            0.0,
        )
    )

    signal_dict_621ed2 = {

        "pca_only":
            pca_norm_621ed2,

        "temporal_only":
            temporal_norm_621ed2,

        "max_fusion":
            np.maximum(
                pca_norm_621ed2,
                temporal_norm_621ed2,
            ),

        "mean_fusion":
            (
                pca_norm_621ed2
                + temporal_norm_621ed2
            ) / 2.0,

        "rms_fusion":
            np.sqrt(
                (
                    pca_norm_621ed2**2
                    + temporal_norm_621ed2**2
                ) / 2.0
            ),

        "geometric_fusion":
            np.sqrt(
                pca_norm_621ed2
                * temporal_norm_621ed2
            ),
    }

    for (
        signal_name_621ed2,
        values_621ed2,
    ) in signal_dict_621ed2.items():

        if not np.isfinite(
            values_621ed2
        ).all():

            raise RuntimeError(
                f"{signal_name_621ed2} "
                "contains non-finite values."
            )

    return signal_dict_621ed2


# ============================================================
# 11. CLEAN validation baselines
# ============================================================

print(
    "\n--- Computing clean signal baselines ---"
)

clean_signals_621ed2 = (
    compute_all_signals_621ed2(
        clean_reference_621ed2
    )
)

clean_baseline_rows_621ed2 = []

for signal_621ed2 in (
    ALL_SIGNALS_621ED2
):

    values_621ed2 = (
        clean_signals_621ed2[
            signal_621ed2
        ]
    )

    threshold_621ed2 = (
        signal_threshold_map_621ed2[
            signal_621ed2
        ]
    )

    clean_baseline_rows_621ed2.append(
        {
            "signal":
                signal_621ed2,

            "threshold":
                threshold_621ed2,

            "clean_mean":
                float(
                    np.mean(
                        values_621ed2
                    )
                ),

            "clean_median":
                float(
                    np.median(
                        values_621ed2
                    )
                ),

            "clean_p95":
                float(
                    np.percentile(
                        values_621ed2,
                        95,
                    )
                ),

            "clean_event_pct":
                float(
                    100.0
                    * np.mean(
                        values_621ed2
                        > threshold_621ed2
                    )
                ),
        }
    )


fusion_clean_baseline_621ed2 = (
    pd.DataFrame(
        clean_baseline_rows_621ed2
    )
)

print(
    "\n--- CLEAN validation signal baselines ---"
)

display(
    fusion_clean_baseline_621ed2[
        [
            "signal",
            "threshold",
            "clean_event_pct",
            "clean_mean",
        ]
    ].round(6)
)


# ============================================================
# 12. Total frozen-scenario count
# ============================================================

total_scenarios_621ed2 = int(
    sum(
        len(
            scenario_manifests_620gc[
                family_621ed2
            ]
        )

        for family_621ed2
        in CORRUPTION_FAMILIES_621ED2
    )
)

print(
    "\nFrozen scenarios to evaluate:",
    total_scenarios_621ed2,
)

if total_scenarios_621ed2 != 315:

    raise RuntimeError(
        "Expected exactly 315 frozen scenarios."
    )


# ============================================================
# 13. Execute all 315 frozen corruption scenarios
# ============================================================

scenario_rows_621ed2 = []

scenario_counter_621ed2 = 0

start_time_621ed2 = time.time()

print(
    "\n=== Executing all 315 frozen scenarios ==="
)

for family_621ed2 in (
    CORRUPTION_FAMILIES_621ED2
):

    manifest_621ed2 = (
        scenario_manifests_620gc[
            family_621ed2
        ]
    )

    if not isinstance(
        manifest_621ed2,
        pd.DataFrame,
    ):

        raise RuntimeError(
            f"{family_621ed2} manifest "
            "is not a DataFrame."
        )

    for (
        manifest_index_621ed2,
        scenario_row_621ed2,
    ) in manifest_621ed2.iterrows():

        scenario_counter_621ed2 += 1

        corruption_output_621ed2 = (
            apply_frozen_corruption_620gc(
                family_621ed2,
                scenario_row_621ed2,
            )
        )

        corrupted_df_621ed2 = (
            extract_corrupted_df_621ed2(
                corruption_output_621ed2
            )
        )

        X_corrupted_621ed2 = (
            build_scaled_windows_621ed2(
                corrupted_df_621ed2
            )
        )

        if (
            X_corrupted_621ed2.shape
            != clean_reference_621ed2.shape
        ):

            raise RuntimeError(
                f"{family_621ed2}: "
                "window structure changed."
            )

        corrupted_signals_621ed2 = (
            compute_all_signals_621ed2(
                X_corrupted_621ed2
            )
        )

        sensor_621ed2 = str(
            scenario_row_621ed2.get(
                "sensor",
                "<missing>",
            )
        )

        severity_621ed2 = str(
            scenario_row_621ed2.get(
                "severity",
                "<missing>",
            )
        ).lower()

        scenario_id_621ed2 = str(
            scenario_row_621ed2.get(
                "scenario_id",
                (
                    f"{family_621ed2}_"
                    f"{manifest_index_621ed2}"
                ),
            )
        )

        direction_621ed2 = (
            scenario_row_621ed2.get(
                "direction",
                np.nan,
            )
        )

        alpha_621ed2 = (
            scenario_row_621ed2.get(
                "alpha",
                np.nan,
            )
        )

        requested_fraction_621ed2 = (
            scenario_row_621ed2.get(
                "requested_fraction",
                np.nan,
            )
        )

        if sensor_621ed2 == "<missing>":

            raise RuntimeError(
                "Scenario sensor metadata missing."
            )

        if (
            severity_621ed2
            not in SEVERITY_ORDER_621ED2
        ):

            raise RuntimeError(
                "Unexpected severity label: "
                + severity_621ed2
            )

        for signal_621ed2 in (
            ALL_SIGNALS_621ED2
        ):

            corrupted_values_621ed2 = (
                corrupted_signals_621ed2[
                    signal_621ed2
                ]
            )

            clean_values_621ed2 = (
                clean_signals_621ed2[
                    signal_621ed2
                ]
            )

            threshold_621ed2 = (
                signal_threshold_map_621ed2[
                    signal_621ed2
                ]
            )

            clean_mean_621ed2 = float(
                np.mean(
                    clean_values_621ed2
                )
            )

            corrupted_mean_621ed2 = float(
                np.mean(
                    corrupted_values_621ed2
                )
            )

            clean_p95_local_621ed2 = float(
                np.percentile(
                    clean_values_621ed2,
                    95,
                )
            )

            corrupted_p95_621ed2 = float(
                np.percentile(
                    corrupted_values_621ed2,
                    95,
                )
            )

            clean_event_pct_621ed2 = float(
                100.0
                * np.mean(
                    clean_values_621ed2
                    > threshold_621ed2
                )
            )

            corrupted_event_pct_621ed2 = (
                float(
                    100.0
                    * np.mean(
                        corrupted_values_621ed2
                        > threshold_621ed2
                    )
                )
            )

            increased_windows_pct_621ed2 = (
                float(
                    100.0
                    * np.mean(
                        corrupted_values_621ed2
                        >
                        clean_values_621ed2
                        + 1e-12
                    )
                )
            )

            scenario_rows_621ed2.append(
                {
                    "signal":
                        signal_621ed2,

                    "corruption":
                        family_621ed2,

                    "scenario_id":
                        scenario_id_621ed2,

                    "sensor":
                        sensor_621ed2,

                    "severity":
                        severity_621ed2,

                    "direction":
                        direction_621ed2,

                    "alpha":
                        alpha_621ed2,

                    "requested_fraction":
                        requested_fraction_621ed2,

                    "windows":
                        int(
                            len(
                                corrupted_values_621ed2
                            )
                        ),

                    "clean_mean":
                        clean_mean_621ed2,

                    "corrupted_mean":
                        corrupted_mean_621ed2,

                    "mean_score_delta":
                        (
                            corrupted_mean_621ed2
                            - clean_mean_621ed2
                        ),

                    "clean_p95":
                        clean_p95_local_621ed2,

                    "corrupted_p95":
                        corrupted_p95_621ed2,

                    "p95_score_delta":
                        (
                            corrupted_p95_621ed2
                            - clean_p95_local_621ed2
                        ),

                    "clean_event_pct":
                        clean_event_pct_621ed2,

                    "corrupted_event_pct":
                        corrupted_event_pct_621ed2,

                    "event_delta_pp":
                        (
                            corrupted_event_pct_621ed2
                            - clean_event_pct_621ed2
                        ),

                    "score_increased_windows_pct":
                        increased_windows_pct_621ed2,
                }
            )

        if (
            scenario_counter_621ed2 % 25
            == 0
            or
            scenario_counter_621ed2
            == total_scenarios_621ed2
        ):

            elapsed_621ed2 = (
                time.time()
                - start_time_621ed2
            )

            print(
                f"[{scenario_counter_621ed2:03d}/"
                f"{total_scenarios_621ed2}] "
                f"completed — "
                f"{elapsed_621ed2:.1f} s"
            )


execution_time_621ed2 = float(
    time.time()
    - start_time_621ed2
)

fusion_scenario_results_621ed2 = (
    pd.DataFrame(
        scenario_rows_621ed2
    )
)

print(
    "\nFull execution time:",
    round(
        execution_time_621ed2,
        2,
    ),
    "seconds",
)


# ============================================================
# 14. Scenario-level integrity
# ============================================================

expected_scenario_result_rows_621ed2 = (
    total_scenarios_621ed2
    * len(
        ALL_SIGNALS_621ED2
    )
)

if (
    len(
        fusion_scenario_results_621ed2
    )
    != expected_scenario_result_rows_621ed2
):

    raise RuntimeError(
        "Unexpected scenario-result row count."
    )

numeric_scenario_621ed2 = (
    fusion_scenario_results_621ed2
    .select_dtypes(
        include=[np.number]
    )
)

if not np.isfinite(
    numeric_scenario_621ed2
    .to_numpy(
        dtype=np.float64
    )
).all():

    raise RuntimeError(
        "Non-finite scenario result detected."
    )


# ============================================================
# 15. Balanced sensor/severity cells
#
# Bias and drift have two directions.
# Their directions are averaged here so every
# corruption/sensor/severity cell contributes once.
# ============================================================

fusion_balanced_cells_621ed2 = (
    fusion_scenario_results_621ed2
    .groupby(
        [
            "signal",
            "corruption",
            "sensor",
            "severity",
        ],
        as_index=False,
    )
    .agg(
        source_scenarios=(
            "scenario_id",
            "size",
        ),

        mean_score_delta=(
            "mean_score_delta",
            "mean",
        ),

        median_score_delta=(
            "mean_score_delta",
            "median",
        ),

        mean_event_delta_pp=(
            "event_delta_pp",
            "mean",
        ),

        median_event_delta_pp=(
            "event_delta_pp",
            "median",
        ),

        mean_increased_windows_pct=(
            "score_increased_windows_pct",
            "mean",
        ),
    )
)

expected_balanced_rows_621ed2 = (
    len(
        ALL_SIGNALS_621ED2
    )
    * 5
    * 15
    * 3
)

if (
    len(
        fusion_balanced_cells_621ed2
    )
    != expected_balanced_rows_621ed2
):

    raise RuntimeError(
        "Unexpected balanced-cell row count."
    )


# Verify direction multiplicity.
for family_621ed2 in (
    CORRUPTION_FAMILIES_621ED2
):

    expected_multiplicity_621ed2 = (
        2
        if family_621ed2
        in ["bias", "drift"]
        else 1
    )

    family_cells_621ed2 = (
        fusion_balanced_cells_621ed2[
            fusion_balanced_cells_621ed2[
                "corruption"
            ]
            == family_621ed2
        ]
    )

    if not (
        family_cells_621ed2[
            "source_scenarios"
        ]
        == expected_multiplicity_621ed2
    ).all():

        raise RuntimeError(
            f"Unexpected source multiplicity "
            f"for {family_621ed2}."
        )


# ============================================================
# 16. Family summaries
# ============================================================

fusion_family_summary_621ed2 = (
    fusion_balanced_cells_621ed2
    .groupby(
        [
            "signal",
            "corruption",
        ],
        as_index=False,
    )
    .agg(
        balanced_cells=(
            "sensor",
            "size",
        ),

        mean_score_delta=(
            "mean_score_delta",
            "mean",
        ),

        median_score_delta=(
            "mean_score_delta",
            "median",
        ),

        mean_event_delta_pp=(
            "mean_event_delta_pp",
            "mean",
        ),

        median_event_delta_pp=(
            "mean_event_delta_pp",
            "median",
        ),

        mean_increased_windows_pct=(
            "mean_increased_windows_pct",
            "mean",
        ),
    )
)


# ============================================================
# 17. Family/severity summaries
# ============================================================

fusion_family_severity_621ed2 = (
    fusion_balanced_cells_621ed2
    .groupby(
        [
            "signal",
            "corruption",
            "severity",
        ],
        as_index=False,
    )
    .agg(
        sensors=(
            "sensor",
            "size",
        ),

        mean_score_delta=(
            "mean_score_delta",
            "mean",
        ),

        mean_event_delta_pp=(
            "mean_event_delta_pp",
            "mean",
        ),
    )
)


# ============================================================
# 18. Severity monotonicity by sensor
# ============================================================

monotonic_rows_621ed2 = []

for (
    signal_621ed2,
    family_621ed2,
    sensor_621ed2,
), group_621ed2 in (
    fusion_balanced_cells_621ed2
    .groupby(
        [
            "signal",
            "corruption",
            "sensor",
        ]
    )
):

    local_621ed2 = (
        group_621ed2.copy()
    )

    local_621ed2[
        "severity_order"
    ] = (
        local_621ed2[
            "severity"
        ]
        .map(
            SEVERITY_ORDER_621ED2
        )
    )

    local_621ed2 = (
        local_621ed2
        .sort_values(
            "severity_order"
        )
    )

    if len(local_621ed2) != 3:
        raise RuntimeError(
            "Expected three severity levels."
        )

    score_values_621ed2 = (
        local_621ed2[
            "mean_score_delta"
        ]
        .to_numpy(
            dtype=np.float64
        )
    )

    event_values_621ed2 = (
        local_621ed2[
            "mean_event_delta_pp"
        ]
        .to_numpy(
            dtype=np.float64
        )
    )

    score_monotonic_621ed2 = bool(
        np.all(
            np.diff(
                score_values_621ed2
            )
            >= -1e-9
        )
    )

    event_monotonic_621ed2 = bool(
        np.all(
            np.diff(
                event_values_621ed2
            )
            >= -1e-9
        )
    )

    monotonic_rows_621ed2.append(
        {
            "signal":
                signal_621ed2,

            "corruption":
                family_621ed2,

            "sensor":
                sensor_621ed2,

            "low_score_delta":
                score_values_621ed2[0],

            "medium_score_delta":
                score_values_621ed2[1],

            "high_score_delta":
                score_values_621ed2[2],

            "low_event_delta_pp":
                event_values_621ed2[0],

            "medium_event_delta_pp":
                event_values_621ed2[1],

            "high_event_delta_pp":
                event_values_621ed2[2],

            "score_monotonic":
                score_monotonic_621ed2,

            "event_monotonic":
                event_monotonic_621ed2,

            "score_high_minus_low":
                (
                    score_values_621ed2[2]
                    - score_values_621ed2[0]
                ),

            "event_high_minus_low_pp":
                (
                    event_values_621ed2[2]
                    - event_values_621ed2[0]
                ),
        }
    )


fusion_monotonicity_621ed2 = (
    pd.DataFrame(
        monotonic_rows_621ed2
    )
)


monotonic_summary_621ed2 = (
    fusion_monotonicity_621ed2
    .groupby(
        "signal",
        as_index=False,
    )
    .agg(
        score_monotonic_pct=(
            "score_monotonic",
            lambda x:
                100.0
                * np.mean(x),
        ),

        event_monotonic_pct=(
            "event_monotonic",
            lambda x:
                100.0
                * np.mean(x),
        ),

        score_high_gt_low_pct=(
            "score_high_minus_low",
            lambda x:
                100.0
                * np.mean(
                    np.asarray(x)
                    > 1e-9
                ),
        ),

        event_high_gt_low_pct=(
            "event_high_minus_low_pp",
            lambda x:
                100.0
                * np.mean(
                    np.asarray(x)
                    > 1e-9
                ),
        ),
    )
)


# ============================================================
# 19. Overall balanced candidate summaries
# ============================================================

overall_summary_621ed2 = (
    fusion_balanced_cells_621ed2
    .groupby(
        "signal",
        as_index=False,
    )
    .agg(
        mean_score_delta=(
            "mean_score_delta",
            "mean",
        ),

        median_score_delta=(
            "mean_score_delta",
            "median",
        ),

        mean_event_delta_pp=(
            "mean_event_delta_pp",
            "mean",
        ),

        median_event_delta_pp=(
            "mean_event_delta_pp",
            "median",
        ),

        positive_score_cells_pct=(
            "mean_score_delta",
            lambda x:
                100.0
                * np.mean(
                    np.asarray(x)
                    > 1e-9
                ),
        ),

        positive_event_cells_pct=(
            "mean_event_delta_pp",
            lambda x:
                100.0
                * np.mean(
                    np.asarray(x)
                    > 1e-9
                ),
        ),
    )
)


# Dynamic corruption families only:
# noise, drift, dropout, stuck
dynamic_summary_621ed2 = (
    fusion_balanced_cells_621ed2[
        fusion_balanced_cells_621ed2[
            "corruption"
        ]
        != "bias"
    ]
    .groupby(
        "signal",
        as_index=False,
    )
    .agg(
        dynamic_mean_event_delta_pp=(
            "mean_event_delta_pp",
            "mean",
        ),

        dynamic_mean_score_delta=(
            "mean_score_delta",
            "mean",
        ),
    )
)


bias_summary_621ed2 = (
    fusion_balanced_cells_621ed2[
        fusion_balanced_cells_621ed2[
            "corruption"
        ]
        == "bias"
    ]
    .groupby(
        "signal",
        as_index=False,
    )
    .agg(
        bias_mean_event_delta_pp=(
            "mean_event_delta_pp",
            "mean",
        ),

        bias_mean_score_delta=(
            "mean_score_delta",
            "mean",
        ),
    )
)


high_summary_621ed2 = (
    fusion_balanced_cells_621ed2[
        fusion_balanced_cells_621ed2[
            "severity"
        ]
        == "high"
    ]
    .groupby(
        "signal",
        as_index=False,
    )
    .agg(
        high_severity_mean_event_delta_pp=(
            "mean_event_delta_pp",
            "mean",
        ),

        high_severity_mean_score_delta=(
            "mean_score_delta",
            "mean",
        ),
    )
)


family_robustness_621ed2 = (
    fusion_family_summary_621ed2
    .groupby(
        "signal",
        as_index=False,
    )
    .agg(
        worst_family_event_delta_pp=(
            "mean_event_delta_pp",
            "min",
        ),

        best_family_event_delta_pp=(
            "mean_event_delta_pp",
            "max",
        ),

        family_positive_count=(
            "mean_event_delta_pp",
            lambda x:
                int(
                    np.sum(
                        np.asarray(x)
                        > 1e-9
                    )
                ),
        ),
    )
)


# ============================================================
# 20. Merge overall evidence
# ============================================================

fusion_candidate_summary_621ed2 = (
    fusion_clean_baseline_621ed2[
        [
            "signal",
            "clean_event_pct",
        ]
    ]
    .merge(
        overall_summary_621ed2,
        on="signal",
        how="left",
        validate="one_to_one",
    )
    .merge(
        dynamic_summary_621ed2,
        on="signal",
        how="left",
        validate="one_to_one",
    )
    .merge(
        bias_summary_621ed2,
        on="signal",
        how="left",
        validate="one_to_one",
    )
    .merge(
        high_summary_621ed2,
        on="signal",
        how="left",
        validate="one_to_one",
    )
    .merge(
        monotonic_summary_621ed2,
        on="signal",
        how="left",
        validate="one_to_one",
    )
    .merge(
        family_robustness_621ed2,
        on="signal",
        how="left",
        validate="one_to_one",
    )
)


signal_order_map_621ed2 = {
    signal_621ed2:
    index_621ed2

    for index_621ed2,
    signal_621ed2
    in enumerate(
        ALL_SIGNALS_621ED2
    )
}

fusion_candidate_summary_621ed2[
    "_order"
] = (
    fusion_candidate_summary_621ed2[
        "signal"
    ]
    .map(
        signal_order_map_621ed2
    )
)

fusion_candidate_summary_621ed2 = (
    fusion_candidate_summary_621ed2
    .sort_values(
        "_order"
    )
    .drop(
        columns="_order"
    )
    .reset_index(drop=True)
)


# ============================================================
# 21. Compact family-event matrix
# ============================================================

family_event_matrix_621ed2 = (
    fusion_family_summary_621ed2
    .pivot(
        index="signal",
        columns="corruption",
        values="mean_event_delta_pp",
    )
    .reindex(
        ALL_SIGNALS_621ED2
    )
    .reindex(
        columns=
        CORRUPTION_FAMILIES_621ED2
    )
)


# ============================================================
# 22. Compact output only
# ============================================================

print(
    "\n================================================"
)
print(
    "BALANCED MEAN EVENT DELTA BY CORRUPTION FAMILY"
)
print(
    "================================================"
)

display(
    family_event_matrix_621ed2
    .round(4)
)


print(
    "\n========================================="
)
print(
    "CANDIDATE SUMMARY — RESPONSE"
)
print(
    "========================================="
)

display(
    fusion_candidate_summary_621ed2[
        [
            "signal",
            "clean_event_pct",
            "mean_event_delta_pp",
            "dynamic_mean_event_delta_pp",
            "bias_mean_event_delta_pp",
            "high_severity_mean_event_delta_pp",
            "positive_event_cells_pct",
        ]
    ]
    .round(4)
)


print(
    "\n========================================="
)
print(
    "CANDIDATE SUMMARY — ROBUSTNESS"
)
print(
    "========================================="
)

display(
    fusion_candidate_summary_621ed2[
        [
            "signal",
            "mean_score_delta",
            "event_monotonic_pct",
            "score_monotonic_pct",
            "event_high_gt_low_pct",
            "family_positive_count",
            "worst_family_event_delta_pp",
        ]
    ]
    .round(4)
)


# ============================================================
# 23. Final integrity summary
# ============================================================

scenario_rows_ok_621ed2 = bool(
    len(
        fusion_scenario_results_621ed2
    )
    == expected_scenario_result_rows_621ed2
)

balanced_rows_ok_621ed2 = bool(
    len(
        fusion_balanced_cells_621ed2
    )
    == expected_balanced_rows_621ed2
)

all_finite_final_621ed2 = bool(
    np.isfinite(
        fusion_candidate_summary_621ed2
        .select_dtypes(
            include=[np.number]
        )
        .to_numpy(
            dtype=np.float64
        )
    ).all()
)


print(
    "\n=============================================="
)
print(
    "STEP 6.21E-D2 INTEGRITY SUMMARY"
)
print(
    "=============================================="
)

print(
    "Frozen corruption scenarios evaluated :",
    total_scenarios_621ed2,
)

print(
    "Signals evaluated per scenario         :",
    len(
        ALL_SIGNALS_621ED2
    ),
)

print(
    "Scenario/signal result rows            :",
    len(
        fusion_scenario_results_621ed2
    ),
)

print(
    "Expected scenario/signal rows          :",
    expected_scenario_result_rows_621ed2,
)

print(
    "Balanced comparison cells              :",
    len(
        fusion_balanced_cells_621ed2
    ),
)

print(
    "Expected balanced cells                :",
    expected_balanced_rows_621ed2,
)

print(
    "Scenario row count correct             :",
    scenario_rows_ok_621ed2,
)

print(
    "Balanced row count correct             :",
    balanced_rows_ok_621ed2,
)

print(
    "All final numerical values finite      :",
    all_finite_final_621ed2,
)

print(
    "Clean reconstruction error             :",
    clean_reconstruction_error_621ed2,
)

print(
    "Development thresholds modified        : False"
)

print(
    "Frozen PCA artifact modified           : False"
)

print(
    "Frozen temporal artifact modified      : False"
)

print(
    "Final fusion candidate selected        : False"
)

print(
    "Files written                          : False"
)

print(
    "Execution time (seconds)               :",
    round(
        execution_time_621ed2,
        2,
    ),
)


if not scenario_rows_ok_621ed2:
    raise RuntimeError(
        "Scenario-result integrity failed."
    )

if not balanced_rows_ok_621ed2:
    raise RuntimeError(
        "Balanced-cell integrity failed."
    )

if not all_finite_final_621ed2:
    raise RuntimeError(
        "Non-finite final summary values detected."
    )


print(
    "\nStep 6.21E-D2 completed successfully."
)

print(
    "All 315 frozen scenarios were evaluated "
    "using PCA-only, Temporal-only, and four "
    "PCA/Temporal fusion candidates."
)

print(
    "Bias/drift direction multiplicity was "
    "balanced before cross-family comparison."
)

print(
    "No final fusion candidate has been selected yet."
)

print(
    "The results are now ready for evidence-based "
    "fusion-candidate selection."
)

In [ ]:
# ============================================================
# Step 6.21E-D2R
# Recovery after optional-metadata NaN integrity check
#
# IMPORTANT:
#   - DOES NOT rerun the 315 corruption scenarios.
#   - Uses fusion_scenario_results_621ed2 already in memory.
#   - Optional metadata NaNs are allowed where not applicable.
#   - Actual experimental response values must all be finite.
# ============================================================

import numpy as np
import pandas as pd

print(
    "\n=== Step 6.21E-D2R — "
    "Recover completed 315-scenario fusion evaluation ==="
)


# ============================================================
# 1. Verify that the completed result table exists
# ============================================================

if "fusion_scenario_results_621ed2" not in globals():
    raise RuntimeError(
        "fusion_scenario_results_621ed2 is missing. "
        "Do NOT continue."
    )

print(
    "Scenario/signal result rows:",
    len(fusion_scenario_results_621ed2)
)

expected_scenario_result_rows_621ed2 = (
    315 * len(ALL_SIGNALS_621ED2)
)

print(
    "Expected rows:",
    expected_scenario_result_rows_621ed2
)

if (
    len(fusion_scenario_results_621ed2)
    != expected_scenario_result_rows_621ed2
):
    raise RuntimeError(
        "Completed scenario-result row count "
        "is not the expected 1890."
    )


# ============================================================
# 2. Correct numerical integrity check
#
# Optional metadata:
#   direction
#   alpha
#   requested_fraction
#
# These may legitimately contain NaN when that parameter
# does not apply to a corruption family.
# ============================================================

required_numeric_metrics_621ed2 = [
    "windows",
    "clean_mean",
    "corrupted_mean",
    "mean_score_delta",
    "clean_p95",
    "corrupted_p95",
    "p95_score_delta",
    "clean_event_pct",
    "corrupted_event_pct",
    "event_delta_pp",
    "score_increased_windows_pct",
]

missing_metric_columns_621ed2 = [
    col_621ed2
    for col_621ed2 in required_numeric_metrics_621ed2
    if col_621ed2
    not in fusion_scenario_results_621ed2.columns
]

if missing_metric_columns_621ed2:
    raise RuntimeError(
        "Missing experimental metric columns: "
        + ", ".join(missing_metric_columns_621ed2)
    )


required_numeric_array_621ed2 = (
    fusion_scenario_results_621ed2[
        required_numeric_metrics_621ed2
    ]
    .to_numpy(dtype=np.float64)
)

all_experimental_metrics_finite_621ed2 = bool(
    np.isfinite(
        required_numeric_array_621ed2
    ).all()
)

print("\n--- Corrected numerical integrity ---")
print(
    "All actual experimental metrics finite :",
    all_experimental_metrics_finite_621ed2,
)

if not all_experimental_metrics_finite_621ed2:
    raise RuntimeError(
        "A true experimental response metric "
        "contains a non-finite value."
    )


# ============================================================
# 3. Optional metadata audit
# ============================================================

optional_metadata_621ed2 = [
    col_621ed2
    for col_621ed2 in [
        "direction",
        "alpha",
        "requested_fraction",
    ]
    if col_621ed2
    in fusion_scenario_results_621ed2.columns
]

print("\n--- Optional metadata NaN counts ---")

for col_621ed2 in optional_metadata_621ed2:
    print(
        f"{col_621ed2:<20}:",
        int(
            fusion_scenario_results_621ed2[
                col_621ed2
            ].isna().sum()
        ),
    )

print(
    "\nThese metadata NaNs are permitted because "
    "the parameters are family-specific."
)


# ============================================================
# 4. Balanced sensor/severity cells
#
# Bias and drift contain two directional scenarios.
# Average those directions so each
# corruption/sensor/severity combination contributes once.
# ============================================================

fusion_balanced_cells_621ed2 = (
    fusion_scenario_results_621ed2
    .groupby(
        [
            "signal",
            "corruption",
            "sensor",
            "severity",
        ],
        as_index=False,
    )
    .agg(
        source_scenarios=(
            "scenario_id",
            "size",
        ),

        mean_score_delta=(
            "mean_score_delta",
            "mean",
        ),

        median_score_delta=(
            "mean_score_delta",
            "median",
        ),

        mean_event_delta_pp=(
            "event_delta_pp",
            "mean",
        ),

        median_event_delta_pp=(
            "event_delta_pp",
            "median",
        ),

        mean_increased_windows_pct=(
            "score_increased_windows_pct",
            "mean",
        ),
    )
)


expected_balanced_rows_621ed2 = (
    len(ALL_SIGNALS_621ED2)
    * 5
    * 15
    * 3
)

print("\n--- Balanced-cell integrity ---")
print(
    "Balanced rows :",
    len(fusion_balanced_cells_621ed2),
)
print(
    "Expected      :",
    expected_balanced_rows_621ed2,
)

if (
    len(fusion_balanced_cells_621ed2)
    != expected_balanced_rows_621ed2
):
    raise RuntimeError(
        "Unexpected balanced-cell row count."
    )


# Verify correct direction multiplicity.
for family_621ed2 in CORRUPTION_FAMILIES_621ED2:

    expected_mult_621ed2 = (
        2
        if family_621ed2
        in ["bias", "drift"]
        else 1
    )

    local_621ed2 = (
        fusion_balanced_cells_621ed2[
            fusion_balanced_cells_621ed2[
                "corruption"
            ]
            == family_621ed2
        ]
    )

    if not (
        local_621ed2[
            "source_scenarios"
        ]
        == expected_mult_621ed2
    ).all():
        raise RuntimeError(
            f"Direction multiplicity failed "
            f"for {family_621ed2}."
        )

print(
    "Bias/drift direction balancing verified."
)


# ============================================================
# 5. Family-level response
# ============================================================

fusion_family_summary_621ed2 = (
    fusion_balanced_cells_621ed2
    .groupby(
        [
            "signal",
            "corruption",
        ],
        as_index=False,
    )
    .agg(
        balanced_cells=(
            "sensor",
            "size",
        ),

        mean_score_delta=(
            "mean_score_delta",
            "mean",
        ),

        median_score_delta=(
            "mean_score_delta",
            "median",
        ),

        mean_event_delta_pp=(
            "mean_event_delta_pp",
            "mean",
        ),

        median_event_delta_pp=(
            "mean_event_delta_pp",
            "median",
        ),

        mean_increased_windows_pct=(
            "mean_increased_windows_pct",
            "mean",
        ),
    )
)


# ============================================================
# 6. Severity monotonicity
# ============================================================

SEVERITY_ORDER_RECOVERY_621ed2 = {
    "low": 0,
    "medium": 1,
    "high": 2,
}

monotonic_rows_621ed2 = []

for (
    signal_621ed2,
    corruption_621ed2,
    sensor_621ed2,
), group_621ed2 in (
    fusion_balanced_cells_621ed2.groupby(
        [
            "signal",
            "corruption",
            "sensor",
        ]
    )
):

    local_621ed2 = group_621ed2.copy()

    local_621ed2[
        "severity_order"
    ] = (
        local_621ed2[
            "severity"
        ].map(
            SEVERITY_ORDER_RECOVERY_621ed2
        )
    )

    local_621ed2 = (
        local_621ed2
        .sort_values(
            "severity_order"
        )
    )

    if len(local_621ed2) != 3:
        raise RuntimeError(
            "A signal/family/sensor combination "
            "does not contain all 3 severity levels."
        )

    score_values_621ed2 = (
        local_621ed2[
            "mean_score_delta"
        ].to_numpy(
            dtype=np.float64
        )
    )

    event_values_621ed2 = (
        local_621ed2[
            "mean_event_delta_pp"
        ].to_numpy(
            dtype=np.float64
        )
    )

    monotonic_rows_621ed2.append(
        {
            "signal":
                signal_621ed2,

            "corruption":
                corruption_621ed2,

            "sensor":
                sensor_621ed2,

            "score_monotonic":
                bool(
                    np.all(
                        np.diff(
                            score_values_621ed2
                        )
                        >= -1e-9
                    )
                ),

            "event_monotonic":
                bool(
                    np.all(
                        np.diff(
                            event_values_621ed2
                        )
                        >= -1e-9
                    )
                ),

            "score_high_minus_low":
                float(
                    score_values_621ed2[2]
                    - score_values_621ed2[0]
                ),

            "event_high_minus_low_pp":
                float(
                    event_values_621ed2[2]
                    - event_values_621ed2[0]
                ),
        }
    )


fusion_monotonicity_621ed2 = pd.DataFrame(
    monotonic_rows_621ed2
)


monotonic_summary_621ed2 = (
    fusion_monotonicity_621ed2
    .groupby(
        "signal",
        as_index=False,
    )
    .agg(
        score_monotonic_pct=(
            "score_monotonic",
            lambda x:
                100.0
                * np.mean(x),
        ),

        event_monotonic_pct=(
            "event_monotonic",
            lambda x:
                100.0
                * np.mean(x),
        ),

        score_high_gt_low_pct=(
            "score_high_minus_low",
            lambda x:
                100.0
                * np.mean(
                    np.asarray(x)
                    > 1e-9
                ),
        ),

        event_high_gt_low_pct=(
            "event_high_minus_low_pp",
            lambda x:
                100.0
                * np.mean(
                    np.asarray(x)
                    > 1e-9
                ),
        ),
    )
)


# ============================================================
# 7. Overall candidate summaries
# ============================================================

overall_summary_621ed2 = (
    fusion_balanced_cells_621ed2
    .groupby(
        "signal",
        as_index=False,
    )
    .agg(
        mean_score_delta=(
            "mean_score_delta",
            "mean",
        ),

        median_score_delta=(
            "mean_score_delta",
            "median",
        ),

        mean_event_delta_pp=(
            "mean_event_delta_pp",
            "mean",
        ),

        median_event_delta_pp=(
            "mean_event_delta_pp",
            "median",
        ),

        positive_score_cells_pct=(
            "mean_score_delta",
            lambda x:
                100.0
                * np.mean(
                    np.asarray(x)
                    > 1e-9
                ),
        ),

        positive_event_cells_pct=(
            "mean_event_delta_pp",
            lambda x:
                100.0
                * np.mean(
                    np.asarray(x)
                    > 1e-9
                ),
        ),
    )
)


# Exclude constant bias for dynamic-corruption summary.
dynamic_summary_621ed2 = (
    fusion_balanced_cells_621ed2[
        fusion_balanced_cells_621ed2[
            "corruption"
        ]
        != "bias"
    ]
    .groupby(
        "signal",
        as_index=False,
    )
    .agg(
        dynamic_mean_event_delta_pp=(
            "mean_event_delta_pp",
            "mean",
        ),

        dynamic_mean_score_delta=(
            "mean_score_delta",
            "mean",
        ),
    )
)


bias_summary_621ed2 = (
    fusion_balanced_cells_621ed2[
        fusion_balanced_cells_621ed2[
            "corruption"
        ]
        == "bias"
    ]
    .groupby(
        "signal",
        as_index=False,
    )
    .agg(
        bias_mean_event_delta_pp=(
            "mean_event_delta_pp",
            "mean",
        ),

        bias_mean_score_delta=(
            "mean_score_delta",
            "mean",
        ),
    )
)


high_summary_621ed2 = (
    fusion_balanced_cells_621ed2[
        fusion_balanced_cells_621ed2[
            "severity"
        ]
        == "high"
    ]
    .groupby(
        "signal",
        as_index=False,
    )
    .agg(
        high_severity_mean_event_delta_pp=(
            "mean_event_delta_pp",
            "mean",
        ),

        high_severity_mean_score_delta=(
            "mean_score_delta",
            "mean",
        ),
    )
)


family_robustness_621ed2 = (
    fusion_family_summary_621ed2
    .groupby(
        "signal",
        as_index=False,
    )
    .agg(
        worst_family_event_delta_pp=(
            "mean_event_delta_pp",
            "min",
        ),

        best_family_event_delta_pp=(
            "mean_event_delta_pp",
            "max",
        ),

        family_positive_count=(
            "mean_event_delta_pp",
            lambda x:
                int(
                    np.sum(
                        np.asarray(x)
                        > 1e-9
                    )
                ),
        ),
    )
)


# ============================================================
# 8. Merge evidence
# ============================================================

fusion_candidate_summary_621ed2 = (
    fusion_clean_baseline_621ed2[
        [
            "signal",
            "clean_event_pct",
        ]
    ]
    .merge(
        overall_summary_621ed2,
        on="signal",
        how="left",
        validate="one_to_one",
    )
    .merge(
        dynamic_summary_621ed2,
        on="signal",
        how="left",
        validate="one_to_one",
    )
    .merge(
        bias_summary_621ed2,
        on="signal",
        how="left",
        validate="one_to_one",
    )
    .merge(
        high_summary_621ed2,
        on="signal",
        how="left",
        validate="one_to_one",
    )
    .merge(
        monotonic_summary_621ed2,
        on="signal",
        how="left",
        validate="one_to_one",
    )
    .merge(
        family_robustness_621ed2,
        on="signal",
        how="left",
        validate="one_to_one",
    )
)


signal_order_621ed2 = {
    signal_621ed2:
        idx_621ed2
    for idx_621ed2,
        signal_621ed2
    in enumerate(
        ALL_SIGNALS_621ED2
    )
}

fusion_candidate_summary_621ed2[
    "_order"
] = (
    fusion_candidate_summary_621ed2[
        "signal"
    ].map(
        signal_order_621ed2
    )
)

fusion_candidate_summary_621ed2 = (
    fusion_candidate_summary_621ed2
    .sort_values(
        "_order"
    )
    .drop(
        columns="_order"
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 9. Compact family matrix
# ============================================================

family_event_matrix_621ed2 = (
    fusion_family_summary_621ed2
    .pivot(
        index="signal",
        columns="corruption",
        values="mean_event_delta_pp",
    )
    .reindex(
        ALL_SIGNALS_621ED2
    )
    .reindex(
        columns=
        CORRUPTION_FAMILIES_621ED2
    )
)


# ============================================================
# 10. Final numerical-integrity check
# ============================================================

summary_numeric_621ed2 = (
    fusion_candidate_summary_621ed2
    .select_dtypes(
        include=[np.number]
    )
)

all_final_summary_values_finite_621ed2 = bool(
    np.isfinite(
        summary_numeric_621ed2
        .to_numpy(
            dtype=np.float64
        )
    ).all()
)

if not all_final_summary_values_finite_621ed2:
    raise RuntimeError(
        "A final candidate-summary metric "
        "contains a non-finite value."
    )


# ============================================================
# 11. COMPACT OUTPUTS
# ============================================================

print(
    "\n==============================================="
)
print(
    "BALANCED MEAN EVENT DELTA BY CORRUPTION FAMILY"
)
print(
    "==============================================="
)

display(
    family_event_matrix_621ed2
    .round(4)
)


print(
    "\n==================================="
)
print(
    "CANDIDATE SUMMARY — RESPONSE"
)
print(
    "==================================="
)

display(
    fusion_candidate_summary_621ed2[
        [
            "signal",
            "clean_event_pct",
            "mean_event_delta_pp",
            "dynamic_mean_event_delta_pp",
            "bias_mean_event_delta_pp",
            "high_severity_mean_event_delta_pp",
            "positive_event_cells_pct",
        ]
    ]
    .round(4)
)


print(
    "\n==================================="
)
print(
    "CANDIDATE SUMMARY — ROBUSTNESS"
)
print(
    "==================================="
)

display(
    fusion_candidate_summary_621ed2[
        [
            "signal",
            "mean_score_delta",
            "event_monotonic_pct",
            "score_monotonic_pct",
            "event_high_gt_low_pct",
            "family_positive_count",
            "worst_family_event_delta_pp",
        ]
    ]
    .round(4)
)


# ============================================================
# 12. Integrity summary
# ============================================================

print(
    "\n=============================================="
)
print(
    "STEP 6.21E-D2R INTEGRITY SUMMARY"
)
print(
    "=============================================="
)

print(
    "Frozen scenarios already evaluated      : 315"
)

print(
    "Signals per scenario                    :",
    len(ALL_SIGNALS_621ED2),
)

print(
    "Scenario/signal rows                    :",
    len(fusion_scenario_results_621ed2),
)

print(
    "Expected scenario/signal rows           :",
    expected_scenario_result_rows_621ed2,
)

print(
    "Balanced comparison cells               :",
    len(fusion_balanced_cells_621ed2),
)

print(
    "Expected balanced cells                 :",
    expected_balanced_rows_621ed2,
)

print(
    "Actual experimental metrics finite      :",
    all_experimental_metrics_finite_621ed2,
)

print(
    "Final summary values finite             :",
    all_final_summary_values_finite_621ed2,
)

print(
    "Bias/drift direction balancing          : True"
)

print(
    "Development thresholds modified         : False"
)

print(
    "Frozen PCA artifact modified            : False"
)

print(
    "Frozen temporal artifact modified       : False"
)

print(
    "Final fusion candidate selected         : False"
)

print(
    "Files written                           : False"
)

print(
    "\nStep 6.21E-D2R completed successfully."
)

print(
    "The original 315-scenario computation was retained; "
    "only the integrity logic was corrected."
)

print(
    "Results are ready for fusion-candidate selection."
)

In [ ]:
# ============================================================
# Step 6.21F
# Freeze complementary PCA + temporal sensor-quality channels
#
# Final Step-6 quality representation:
#   1. PCA reconstruction-quality channel
#   2. Temporal max_sensor quality channel
#
# rms_fusion is retained only as a scalar ablation benchmark.
#
# No corruption experiment is rerun.
# No model/scaler/PCA object is modified.
# ============================================================

from pathlib import Path
import json
import hashlib
import pandas as pd
import numpy as np


print(
    "\n=== Step 6.21F — Freeze complementary "
    "PCA + temporal quality representation ==="
)


# ------------------------------------------------------------
# 1. Required-object verification
# ------------------------------------------------------------

required_objects_621f = [
    "fusion_candidate_summary_621ed2",
    "fusion_family_summary_621ed2",
    "PCA_P95_621D",
    "TEMPORAL_P95_621D",
    "FROZEN_TEMPORAL_CANDIDATE_620I",
]

print("\n--- Required objects ---")

for name_621f in required_objects_621f:
    available_621f = name_621f in globals()

    print(
        f"{name_621f:<38}:",
        available_621f
    )

    if not available_621f:
        raise RuntimeError(
            f"Required object missing: {name_621f}"
        )


# ------------------------------------------------------------
# 2. Freeze design decision
# ------------------------------------------------------------

FINAL_QUALITY_REPRESENTATION_621F = (
    "dual_channel_pca_plus_temporal"
)

FINAL_PCA_CHANNEL_621F = (
    "pca_reconstruction_quality"
)

FINAL_TEMPORAL_CHANNEL_621F = (
    str(FROZEN_TEMPORAL_CANDIDATE_620I)
)

SCALAR_FUSION_ABLATION_621F = (
    "rms_fusion"
)


print("\n--- Frozen design decision ---")

print(
    "Final representation :",
    FINAL_QUALITY_REPRESENTATION_621F
)

print(
    "PCA channel          :",
    FINAL_PCA_CHANNEL_621F
)

print(
    "Temporal channel     :",
    FINAL_TEMPORAL_CHANNEL_621F
)

print(
    "Scalar ablation      :",
    SCALAR_FUSION_ABLATION_621F
)


# ------------------------------------------------------------
# 3. Development-only calibration values
# ------------------------------------------------------------

PCA_THRESHOLD_621F = float(
    PCA_P95_621D
)

TEMPORAL_THRESHOLD_621F = float(
    TEMPORAL_P95_621D
)


print("\n--- Development-only calibration ---")

print(
    "PCA development p95 threshold      :",
    PCA_THRESHOLD_621F
)

print(
    "Temporal development p95 threshold :",
    TEMPORAL_THRESHOLD_621F
)


if not (
    np.isfinite(PCA_THRESHOLD_621F)
    and np.isfinite(TEMPORAL_THRESHOLD_621F)
):
    raise RuntimeError(
        "Frozen development thresholds "
        "must be finite."
    )


# ------------------------------------------------------------
# 4. Extract evidence for the frozen decision
# ------------------------------------------------------------

selection_evidence_621f = (
    fusion_candidate_summary_621ed2
    .copy()
)

selection_evidence_621f[
    "final_role"
] = "comparison_only"

selection_evidence_621f.loc[
    selection_evidence_621f[
        "signal"
    ] == "pca_only",
    "final_role"
] = "final_quality_channel_pca"

selection_evidence_621f.loc[
    selection_evidence_621f[
        "signal"
    ] == "temporal_only",
    "final_role"
] = "final_quality_channel_temporal"

selection_evidence_621f.loc[
    selection_evidence_621f[
        "signal"
    ] == SCALAR_FUSION_ABLATION_621F,
    "final_role"
] = "scalar_fusion_ablation"


# ------------------------------------------------------------
# 5. Family-response evidence
# ------------------------------------------------------------

family_evidence_621f = (
    fusion_family_summary_621ed2
    .copy()
)


# ------------------------------------------------------------
# 6. Extract clean-validation event rates
# ------------------------------------------------------------

def get_clean_event_pct_621f(signal_name_621f):

    values_621f = (
        fusion_candidate_summary_621ed2.loc[
            fusion_candidate_summary_621ed2[
                "signal"
            ] == signal_name_621f,
            "clean_event_pct"
        ]
    )

    if len(values_621f) != 1:
        raise RuntimeError(
            f"Expected exactly one clean-event value "
            f"for {signal_name_621f}."
        )

    return float(
        values_621f.iloc[0]
    )


PCA_CLEAN_EVENT_PCT_621F = (
    get_clean_event_pct_621f(
        "pca_only"
    )
)

TEMPORAL_CLEAN_EVENT_PCT_621F = (
    get_clean_event_pct_621f(
        "temporal_only"
    )
)

RMS_CLEAN_EVENT_PCT_621F = (
    get_clean_event_pct_621f(
        SCALAR_FUSION_ABLATION_621F
    )
)


# ------------------------------------------------------------
# 7. Output directory
# ------------------------------------------------------------

PROJECT_ROOT_621F = Path(
    r"C:\RUL_Reliability_Research"
)

OUTPUT_DIR_621F = (
    PROJECT_ROOT_621F
    / "06_results"
    / "step6_reliability_signals"
)

OUTPUT_DIR_621F.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# 8. Frozen definition
# ------------------------------------------------------------

quality_definition_621f = {

    "dataset": "FD001",

    "stage": "Step 6.21F",

    "representation":
        FINAL_QUALITY_REPRESENTATION_621F,

    "selection_principle":
        (
            "Preserve complementary degradation information "
            "instead of collapsing PCA and temporal pathways "
            "into a single scalar."
        ),

    "pca_channel": {
        "name":
            FINAL_PCA_CHANNEL_621F,

        "development_p95_threshold":
            PCA_THRESHOLD_621F,

        "clean_validation_event_pct":
            PCA_CLEAN_EVENT_PCT_621F,

        "role":
            (
                "Complementary sensor-quality indicator; "
                "particularly informative for constant bias."
            ),
    },

    "temporal_channel": {
        "name":
            FINAL_TEMPORAL_CHANNEL_621F,

        "development_p95_threshold":
            TEMPORAL_THRESHOLD_621F,

        "clean_validation_event_pct":
            TEMPORAL_CLEAN_EVENT_PCT_621F,

        "role":
            (
                "Complementary sensor-quality indicator; "
                "particularly informative for dynamic "
                "sensor degradation."
            ),
    },

    "scalar_fusion_ablation": {
        "name":
            SCALAR_FUSION_ABLATION_621F,

        "clean_validation_event_pct":
            RMS_CLEAN_EVENT_PCT_621F,

        "role":
            (
                "Single-score fusion benchmark only; "
                "not the final quality representation."
            ),
    },

    "constant_bias_handling":
        (
            "PCA channel retained because temporal transition "
            "signal is invariant to constant additive bias."
        ),

    "threshold_source":
        "clean_development_only",

    "clean_validation_role":
        "sanity_check_only",

    "official_test_used":
        False,

    "final_hard_accept_reject_threshold_selected":
        False,

    "prediction_uncertainty_combined":
        False,
}


# ------------------------------------------------------------
# 9. Save evidence artifacts
# ------------------------------------------------------------

selection_path_621f = (
    OUTPUT_DIR_621F
    / "fd001_quality_channel_selection_evidence.csv"
)

family_path_621f = (
    OUTPUT_DIR_621F
    / "fd001_quality_channel_family_response.csv"
)

definition_path_621f = (
    OUTPUT_DIR_621F
    / "fd001_quality_channel_definition.json"
)


selection_evidence_621f.to_csv(
    selection_path_621f,
    index=False
)

family_evidence_621f.to_csv(
    family_path_621f,
    index=False
)

with open(
    definition_path_621f,
    "w",
    encoding="utf-8"
) as file_621f:

    json.dump(
        quality_definition_621f,
        file_621f,
        indent=4
    )


# ------------------------------------------------------------
# 10. SHA-256 hashes
# ------------------------------------------------------------

def sha256_file_621f(path_621f):

    digest_621f = hashlib.sha256()

    with open(
        path_621f,
        "rb"
    ) as file_621f:

        while True:

            block_621f = file_621f.read(
                1024 * 1024
            )

            if not block_621f:
                break

            digest_621f.update(
                block_621f
            )

    return digest_621f.hexdigest()


saved_files_621f = [
    selection_path_621f,
    family_path_621f,
    definition_path_621f,
]


print("\n--- Frozen files ---")

for path_621f in saved_files_621f:

    print(
        path_621f.name
    )

    print(
        "SHA-256:",
        sha256_file_621f(
            path_621f
        )
    )


# ------------------------------------------------------------
# 11. Compact evidence
# ------------------------------------------------------------

compact_selection_621f = (
    selection_evidence_621f[
        [
            "signal",
            "clean_event_pct",
            "mean_event_delta_pp",
            "dynamic_mean_event_delta_pp",
            "bias_mean_event_delta_pp",
            "high_severity_mean_event_delta_pp",
            "positive_event_cells_pct",
            "event_monotonic_pct",
            "family_positive_count",
            "final_role",
        ]
    ]
    .round(4)
)

print(
    "\n=============================================="
)

print(
    "FINAL STEP-6 QUALITY-SIGNAL EVIDENCE"
)

print(
    "=============================================="
)

display(
    compact_selection_621f
)


# ------------------------------------------------------------
# 12. Integrity summary
# ------------------------------------------------------------

print(
    "\n=============================================="
)

print(
    "STEP 6.21F INTEGRITY SUMMARY"
)

print(
    "=============================================="
)

print(
    "Final quality representation :",
    FINAL_QUALITY_REPRESENTATION_621F
)

print(
    "PCA quality channel          :",
    FINAL_PCA_CHANNEL_621F
)

print(
    "Temporal quality channel     :",
    FINAL_TEMPORAL_CHANNEL_621F
)

print(
    "Scalar fusion ablation       :",
    SCALAR_FUSION_ABLATION_621F
)

print(
    "Threshold source             : CLEAN DEVELOPMENT only"
)

print(
    "Clean validation used        : sanity check only"
)

print(
    "Official test used           : False"
)

print(
    "Previous frozen PCA modified : False"
)

print(
    "Previous temporal modified   : False"
)

print(
    "Hard abstention rule frozen  : False"
)

print(
    "Uncertainty combined yet     : False"
)

print(
    "Files written                : 3"
)

print(
    "\nStep 6.21F completed successfully."
)

print(
    "Complementary PCA + temporal sensor-quality "
    "representation is now frozen for FD001."
)

In [ ]:
# ============================================================
# Step 6.22A
# RUL-side readiness inspection
#
# Purpose:
#   Identify existing RUL prediction, target, model,
#   error and uncertainty-related objects/files before
#   constructing the reliability framework.
#
# INSPECTION ONLY.
# Nothing is trained, modified, selected or frozen here.
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np


print(
    "\n=== Step 6.22A — RUL-side readiness inspection ==="
)


# ------------------------------------------------------------
# 1. Stable snapshot of current notebook namespace
# ------------------------------------------------------------

namespace_snapshot_622a = list(
    globals().items()
)

print(
    "\nNotebook namespace snapshot created."
)

print(
    "Objects in snapshot:",
    len(namespace_snapshot_622a)
)


# ------------------------------------------------------------
# 2. Search terms
# ------------------------------------------------------------

search_terms_622a = [
    "rul",
    "pred",
    "prediction",
    "target",
    "label",
    "model",
    "ensemble",
    "uncert",
    "variance",
    "std",
    "residual",
    "error",
    "rmse",
    "mae",
]


# ------------------------------------------------------------
# 3. Find potentially relevant in-memory objects
# ------------------------------------------------------------

rows_622a = []

for object_name_622a, object_value_622a in namespace_snapshot_622a:

    lower_name_622a = str(
        object_name_622a
    ).lower()

    matched_terms_622a = [
        term_622a
        for term_622a in search_terms_622a
        if term_622a in lower_name_622a
    ]

    if not matched_terms_622a:
        continue

    object_type_622a = type(
        object_value_622a
    ).__name__

    shape_622a = None
    length_622a = None

    try:
        shape_622a = tuple(
            object_value_622a.shape
        )
    except Exception:
        pass

    if shape_622a is None:

        try:
            length_622a = len(
                object_value_622a
            )
        except Exception:
            pass

    rows_622a.append(
        {
            "object_name":
                object_name_622a,

            "type":
                object_type_622a,

            "shape":
                shape_622a,

            "length":
                length_622a,

            "matched_terms":
                ", ".join(
                    matched_terms_622a
                ),
        }
    )


rul_memory_inventory_622a = pd.DataFrame(
    rows_622a
)


if len(rul_memory_inventory_622a) > 0:

    rul_memory_inventory_622a = (
        rul_memory_inventory_622a
        .sort_values(
            [
                "matched_terms",
                "object_name",
            ]
        )
        .reset_index(
            drop=True
        )
    )


print(
    "\n--- Candidate RUL-related objects in memory ---"
)

print(
    "Objects found:",
    len(
        rul_memory_inventory_622a
    )
)


# ------------------------------------------------------------
# 4. Compact priority inventory
#
# Focus on objects whose dimensions could correspond to
# development or validation sequence counts.
# ------------------------------------------------------------

DEV_WINDOWS_622A = 14241
VAL_WINDOWS_622A = 3490


priority_rows_622a = []

for _, row_622a in (
    rul_memory_inventory_622a.iterrows()
    if len(rul_memory_inventory_622a) > 0
    else []
):

    shape_622a = row_622a[
        "shape"
    ]

    length_622a = row_622a[
        "length"
    ]

    relevant_size_622a = False

    if isinstance(
        shape_622a,
        tuple
    ):

        relevant_size_622a = any(
            dimension_622a
            in (
                DEV_WINDOWS_622A,
                VAL_WINDOWS_622A,
            )
            for dimension_622a
            in shape_622a
        )

    if length_622a in (
        DEV_WINDOWS_622A,
        VAL_WINDOWS_622A,
    ):

        relevant_size_622a = True

    if relevant_size_622a:

        priority_rows_622a.append(
            row_622a.to_dict()
        )


rul_priority_inventory_622a = pd.DataFrame(
    priority_rows_622a
)


print(
    "\n--- Priority sequence-aligned objects ---"
)

if len(
    rul_priority_inventory_622a
) == 0:

    print(
        "No obvious 14241/3490-aligned "
        "RUL objects were identified."
    )

else:

    display(
        rul_priority_inventory_622a[
            [
                "object_name",
                "type",
                "shape",
                "length",
                "matched_terms",
            ]
        ].head(40)
    )


# ------------------------------------------------------------
# 5. Inspect model-like objects separately
# ------------------------------------------------------------

model_terms_622a = [
    "model",
    "regressor",
    "ensemble",
    "forest",
    "boost",
    "xgb",
    "lgb",
    "gru",
    "lstm",
    "cnn",
    "torch",
]


model_rows_622a = []

for object_name_622a, object_value_622a in namespace_snapshot_622a:

    lower_name_622a = str(
        object_name_622a
    ).lower()

    matched_model_terms_622a = [
        term_622a
        for term_622a in model_terms_622a
        if term_622a in lower_name_622a
    ]

    if not matched_model_terms_622a:
        continue

    model_rows_622a.append(
        {
            "object_name":
                object_name_622a,

            "type":
                type(
                    object_value_622a
                ).__name__,

            "matched_terms":
                ", ".join(
                    matched_model_terms_622a
                ),
        }
    )


model_inventory_622a = pd.DataFrame(
    model_rows_622a
)


print(
    "\n--- Model-like objects ---"
)

if len(
    model_inventory_622a
) == 0:

    print(
        "No model-like object names detected."
    )

else:

    display(
        model_inventory_622a.head(
            30
        )
    )


# ------------------------------------------------------------
# 6. Search project result files
# ------------------------------------------------------------

PROJECT_ROOT_622A = Path(
    r"C:\RUL_Reliability_Research"
)

candidate_directories_622a = [
    PROJECT_ROOT_622A / "05_experiments",
    PROJECT_ROOT_622A / "06_results",
]


file_terms_622a = [
    "rul",
    "prediction",
    "pred",
    "model",
    "baseline",
    "uncert",
    "ensemble",
    "residual",
    "error",
]


file_rows_622a = []

for directory_622a in candidate_directories_622a:

    if not directory_622a.exists():
        continue

    for path_622a in directory_622a.rglob(
        "*"
    ):

        if not path_622a.is_file():
            continue

        lower_filename_622a = (
            path_622a.name.lower()
        )

        matched_file_terms_622a = [
            term_622a
            for term_622a in file_terms_622a
            if term_622a
            in lower_filename_622a
        ]

        if not matched_file_terms_622a:
            continue

        file_rows_622a.append(
            {
                "filename":
                    path_622a.name,

                "parent":
                    str(
                        path_622a.parent.relative_to(
                            PROJECT_ROOT_622A
                        )
                    ),

                "suffix":
                    path_622a.suffix,

                "size_bytes":
                    path_622a.stat().st_size,

                "matched_terms":
                    ", ".join(
                        matched_file_terms_622a
                    ),
            }
        )


rul_file_inventory_622a = pd.DataFrame(
    file_rows_622a
)


print(
    "\n--- Potential RUL/model/result files ---"
)

print(
    "Files found:",
    len(
        rul_file_inventory_622a
    )
)


if len(
    rul_file_inventory_622a
) > 0:

    display(
        rul_file_inventory_622a
        .sort_values(
            [
                "parent",
                "filename",
            ]
        )
        .head(
            50
        )
    )


# ------------------------------------------------------------
# 7. Look specifically for uncertainty-related evidence
# ------------------------------------------------------------

uncertainty_terms_622a = [
    "uncert",
    "ensemble",
    "variance",
    "std",
    "interval",
    "quantile",
]


uncertainty_objects_622a = []

for object_name_622a, object_value_622a in namespace_snapshot_622a:

    lower_name_622a = str(
        object_name_622a
    ).lower()

    matched_uncertainty_622a = [
        term_622a
        for term_622a in uncertainty_terms_622a
        if term_622a in lower_name_622a
    ]

    if matched_uncertainty_622a:

        uncertainty_objects_622a.append(
            {
                "object_name":
                    object_name_622a,

                "type":
                    type(
                        object_value_622a
                    ).__name__,

                "matched_terms":
                    ", ".join(
                        matched_uncertainty_622a
                    ),
            }
        )


uncertainty_inventory_622a = pd.DataFrame(
    uncertainty_objects_622a
)


print(
    "\n--- Uncertainty-related objects ---"
)

if len(
    uncertainty_inventory_622a
) == 0:

    print(
        "No obvious uncertainty objects detected."
    )

else:

    display(
        uncertainty_inventory_622a.head(
            30
        )
    )


# ------------------------------------------------------------
# 8. Compact readiness summary
# ------------------------------------------------------------

priority_count_622a = len(
    rul_priority_inventory_622a
)

model_count_622a = len(
    model_inventory_622a
)

uncertainty_count_622a = len(
    uncertainty_inventory_622a
)

file_count_622a = len(
    rul_file_inventory_622a
)


print(
    "\n=============================================="
)

print(
    "STEP 6.22A READINESS SUMMARY"
)

print(
    "=============================================="
)

print(
    "Sequence-aligned RUL candidates :",
    priority_count_622a
)

print(
    "Model-like objects              :",
    model_count_622a
)

print(
    "Uncertainty-related objects      :",
    uncertainty_count_622a
)

print(
    "Potential RUL/result files       :",
    file_count_622a
)

print(
    "Training/retraining performed    : False"
)

print(
    "Official test used               : False"
)

print(
    "Quality artifacts modified       : False"
)

print(
    "Abstention rule created          : False"
)

print(
    "\nStep 6.22A completed."
)

print(
    "Inspection only — the next step will identify "
    "the exact clean RUL prediction/error pathway "
    "before uncertainty construction."
)

In [ ]:
# ============================================================
# Step 6.22B
# Verify existing RUL prediction / MC-uncertainty pathway
#
# PURPOSE:
#   1. Verify the exact relationship between:
#        clean_predictions
#        mc_predictions
#        mc_mean_prediction
#        mc_std
#        selected_error_621ec
#
#   2. Identify candidate TRUE-RUL validation targets.
#
# INSPECTION ONLY.
# No model training.
# No uncertainty selection.
# No abstention threshold.
# No official test use.
# ============================================================

import numpy as np
import pandas as pd


print(
    "\n=== Step 6.22B — Verify existing RUL / MC pathway ==="
)


# ------------------------------------------------------------
# 1. Required candidate objects
# ------------------------------------------------------------

required_names_622b = [
    "clean_predictions",
    "mc_mean_prediction",
    "mc_predictions",
    "mc_std",
    "selected_error_621ec",
    "clean_prediction_df",
]


print(
    "\n--- Core object availability ---"
)

for name_622b in required_names_622b:

    print(
        f"{name_622b:24s}:",
        name_622b in globals()
    )


missing_622b = [
    name_622b
    for name_622b in required_names_622b
    if name_622b not in globals()
]

if missing_622b:

    raise RuntimeError(
        "Missing required objects: "
        + ", ".join(missing_622b)
    )


# ------------------------------------------------------------
# 2. Compact shape/type summary
# ------------------------------------------------------------

print(
    "\n--- Core object structure ---"
)

for name_622b in required_names_622b:

    obj_622b = globals()[
        name_622b
    ]

    shape_622b = getattr(
        obj_622b,
        "shape",
        None
    )

    print(
        f"{name_622b:24s} "
        f"type={type(obj_622b).__name__:12s} "
        f"shape={shape_622b}"
    )


# ------------------------------------------------------------
# 3. Convert primary numerical objects safely
# ------------------------------------------------------------

clean_pred_622b = np.asarray(
    clean_predictions,
    dtype=np.float64
).reshape(-1)

mc_mean_622b = np.asarray(
    mc_mean_prediction,
    dtype=np.float64
).reshape(-1)

mc_draws_622b = np.asarray(
    mc_predictions,
    dtype=np.float64
)

mc_std_saved_622b = np.asarray(
    mc_std,
    dtype=np.float64
).reshape(-1)

selected_error_arr_622b = np.asarray(
    selected_error_621ec,
    dtype=np.float64
).reshape(-1)


# ------------------------------------------------------------
# 4. Basic numerical integrity
# ------------------------------------------------------------

print(
    "\n--- Numerical integrity ---"
)

arrays_622b = {
    "clean_predictions":
        clean_pred_622b,

    "mc_mean_prediction":
        mc_mean_622b,

    "mc_predictions":
        mc_draws_622b,

    "mc_std":
        mc_std_saved_622b,

    "selected_error_621ec":
        selected_error_arr_622b,
}


for name_622b, arr_622b in arrays_622b.items():

    print(
        f"{name_622b:24s} "
        f"finite={bool(np.isfinite(arr_622b).all())}"
    )


# ------------------------------------------------------------
# 5. Determine MC orientation
# ------------------------------------------------------------

VAL_WINDOWS_622B = 3490


if (
    mc_draws_622b.ndim == 2
    and mc_draws_622b.shape[1] == VAL_WINDOWS_622B
):

    mc_axis_622b = 0

elif (
    mc_draws_622b.ndim == 2
    and mc_draws_622b.shape[0] == VAL_WINDOWS_622B
):

    mc_axis_622b = 1

else:

    raise RuntimeError(
        "Could not identify MC prediction orientation."
    )


print(
    "\n--- MC structure ---"
)

print(
    "MC prediction shape :",
    mc_draws_622b.shape
)

print(
    "MC sample axis      :",
    mc_axis_622b
)

print(
    "MC passes           :",
    mc_draws_622b.shape[
        mc_axis_622b
    ]
)


# ------------------------------------------------------------
# 6. Recompute MC mean and standard deviation
# ------------------------------------------------------------

mc_mean_recomputed_622b = np.mean(
    mc_draws_622b,
    axis=mc_axis_622b
)

mc_std_ddof0_622b = np.std(
    mc_draws_622b,
    axis=mc_axis_622b,
    ddof=0
)

mc_std_ddof1_622b = np.std(
    mc_draws_622b,
    axis=mc_axis_622b,
    ddof=1
)


mean_error_622b = np.abs(
    mc_mean_recomputed_622b
    - mc_mean_622b
)

std0_error_622b = np.abs(
    mc_std_ddof0_622b
    - mc_std_saved_622b
)

std1_error_622b = np.abs(
    mc_std_ddof1_622b
    - mc_std_saved_622b
)


print(
    "\n--- MC consistency ---"
)

print(
    "MC mean maximum error :",
    float(
        np.max(
            mean_error_622b
        )
    )
)

print(
    "MC mean mean error    :",
    float(
        np.mean(
            mean_error_622b
        )
    )
)

print(
    "mc_std vs ddof=0 max  :",
    float(
        np.max(
            std0_error_622b
        )
    )
)

print(
    "mc_std vs ddof=1 max  :",
    float(
        np.max(
            std1_error_622b
        )
    )
)


if np.max(std0_error_622b) <= np.max(
    std1_error_622b
):

    detected_std_convention_622b = (
        "population std (ddof=0)"
    )

else:

    detected_std_convention_622b = (
        "sample std (ddof=1)"
    )


print(
    "Closest std convention:",
    detected_std_convention_622b
)


# ------------------------------------------------------------
# 7. CLEAN prediction vs MC mean
# ------------------------------------------------------------

clean_mc_difference_622b = np.abs(
    clean_pred_622b
    - mc_mean_622b
)


if (
    np.std(clean_pred_622b) > 0
    and np.std(mc_mean_622b) > 0
):

    clean_mc_corr_622b = float(
        np.corrcoef(
            clean_pred_622b,
            mc_mean_622b
        )[0, 1]
    )

else:

    clean_mc_corr_622b = np.nan


print(
    "\n--- Deterministic clean prediction vs MC mean ---"
)

print(
    "Mean absolute difference :",
    float(
        np.mean(
            clean_mc_difference_622b
        )
    )
)

print(
    "Median absolute difference:",
    float(
        np.median(
            clean_mc_difference_622b
        )
    )
)

print(
    "Maximum absolute difference:",
    float(
        np.max(
            clean_mc_difference_622b
        )
    )
)

print(
    "Pearson correlation       :",
    clean_mc_corr_622b
)


# ------------------------------------------------------------
# 8. MC uncertainty descriptive statistics
# ------------------------------------------------------------

print(
    "\n--- Existing mc_std distribution ---"
)

print(
    "minimum :",
    float(
        np.min(
            mc_std_saved_622b
        )
    )
)

print(
    "median  :",
    float(
        np.median(
            mc_std_saved_622b
        )
    )
)

print(
    "mean    :",
    float(
        np.mean(
            mc_std_saved_622b
        )
    )
)

print(
    "p95     :",
    float(
        np.quantile(
            mc_std_saved_622b,
            0.95
        )
    )
)

print(
    "maximum :",
    float(
        np.max(
            mc_std_saved_622b
        )
    )
)


# ------------------------------------------------------------
# 9. Inspect clean_prediction_df column names only
# ------------------------------------------------------------

print(
    "\n--- clean_prediction_df columns ---"
)

for index_622b, column_622b in enumerate(
    clean_prediction_df.columns,
    start=1
):

    print(
        f"{index_622b:02d}. "
        f"{column_622b}"
    )


# ------------------------------------------------------------
# 10. Search for possible TRUE-RUL target arrays
#
# We inspect ALL current objects, not only names previously
# captured by Step 6.22A.
# ------------------------------------------------------------

namespace_snapshot_622b = list(
    globals().items()
)


target_name_terms_622b = [
    "rul",
    "true",
    "actual",
    "label",
    "target",
    "y_val",
    "y_validation",
    "y_clean",
]


exclude_name_terms_622b = [
    "strength",
    "temporal",
    "pca",
    "threshold",
    "corruption",
    "quality",
]


target_candidates_622b = []


for object_name_622b, object_value_622b in namespace_snapshot_622b:

    lower_name_622b = str(
        object_name_622b
    ).lower()

    if not any(
        term_622b in lower_name_622b
        for term_622b
        in target_name_terms_622b
    ):
        continue

    if any(
        term_622b in lower_name_622b
        for term_622b
        in exclude_name_terms_622b
    ):
        continue

    try:

        arr_622b = np.asarray(
            object_value_622b
        )

    except Exception:

        continue


    if arr_622b.size != VAL_WINDOWS_622B:

        continue


    if not np.issubdtype(
        arr_622b.dtype,
        np.number
    ):

        continue


    arr_flat_622b = np.asarray(
        arr_622b,
        dtype=np.float64
    ).reshape(-1)


    target_candidates_622b.append(
        {
            "object_name":
                object_name_622b,

            "shape":
                arr_622b.shape,

            "finite":
                bool(
                    np.isfinite(
                        arr_flat_622b
                    ).all()
                ),

            "minimum":
                float(
                    np.nanmin(
                        arr_flat_622b
                    )
                ),

            "median":
                float(
                    np.nanmedian(
                        arr_flat_622b
                    )
                ),

            "maximum":
                float(
                    np.nanmax(
                        arr_flat_622b
                    )
                ),
        }
    )


target_candidates_df_622b = pd.DataFrame(
    target_candidates_622b
)


print(
    "\n--- Candidate TRUE-RUL arrays ---"
)

if len(
    target_candidates_df_622b
) == 0:

    print(
        "No obvious 3490-element RUL target "
        "array detected by name."
    )

else:

    display(
        target_candidates_df_622b
    )


# ------------------------------------------------------------
# 11. Search 3490-row DataFrames for RUL/error/pred columns
# ------------------------------------------------------------

df_candidates_622b = []


for object_name_622b, object_value_622b in namespace_snapshot_622b:

    if not isinstance(
        object_value_622b,
        pd.DataFrame
    ):

        continue

    if len(
        object_value_622b
    ) != VAL_WINDOWS_622B:

        continue


    interesting_columns_622b = [
        str(column_622b)

        for column_622b
        in object_value_622b.columns

        if any(
            term_622b
            in str(
                column_622b
            ).lower()

            for term_622b in [
                "rul",
                "true",
                "actual",
                "target",
                "label",
                "pred",
                "error",
                "uncert",
                "std",
            ]
        )
    ]


    if interesting_columns_622b:

        df_candidates_622b.append(
            {
                "object_name":
                    object_name_622b,

                "rows":
                    len(
                        object_value_622b
                    ),

                "interesting_columns":
                    ", ".join(
                        interesting_columns_622b
                    ),
            }
        )


rul_dataframe_candidates_622b = pd.DataFrame(
    df_candidates_622b
)


print(
    "\n--- 3490-row DataFrames with RUL/prediction/error columns ---"
)

if len(
    rul_dataframe_candidates_622b
) == 0:

    print(
        "No candidate DataFrames detected."
    )

else:

    display(
        rul_dataframe_candidates_622b
    )


# ------------------------------------------------------------
# 12. Check whether selected_error can be tied to one of
#     the candidate true-target arrays
# ------------------------------------------------------------

error_match_rows_622b = []


if len(
    target_candidates_df_622b
) > 0:

    for object_name_622b in (
        target_candidates_df_622b[
            "object_name"
        ].tolist()
    ):

        target_arr_622b = np.asarray(
            globals()[
                object_name_622b
            ],
            dtype=np.float64
        ).reshape(-1)


        abs_error_clean_622b = np.abs(
            clean_pred_622b
            - target_arr_622b
        )

        abs_error_mc_622b = np.abs(
            mc_mean_622b
            - target_arr_622b
        )


        clean_match_error_622b = float(
            np.max(
                np.abs(
                    abs_error_clean_622b
                    - selected_error_arr_622b
                )
            )
        )

        mc_match_error_622b = float(
            np.max(
                np.abs(
                    abs_error_mc_622b
                    - selected_error_arr_622b
                )
            )
        )


        error_match_rows_622b.append(
            {
                "target_object":
                    object_name_622b,

                "selected_error_vs_clean_abs_maxdiff":
                    clean_match_error_622b,

                "selected_error_vs_mcmean_abs_maxdiff":
                    mc_match_error_622b,
            }
        )


error_match_df_622b = pd.DataFrame(
    error_match_rows_622b
)


print(
    "\n--- selected_error relationship check ---"
)

if len(
    error_match_df_622b
) == 0:

    print(
        "Cannot test yet because no candidate "
        "TRUE-RUL array was identified."
    )

else:

    display(
        error_match_df_622b
    )


# ------------------------------------------------------------
# 13. Compact final summary
# ------------------------------------------------------------

print(
    "\n=============================================="
)

print(
    "STEP 6.22B COMPACT SUMMARY"
)

print(
    "=============================================="
)

print(
    "Validation predictions       :",
    len(
        clean_pred_622b
    )
)

print(
    "MC passes                    :",
    mc_draws_622b.shape[
        mc_axis_622b
    ]
)

print(
    "MC mean reconstruction error :",
    float(
        np.max(
            mean_error_622b
        )
    )
)

print(
    "Detected std convention      :",
    detected_std_convention_622b
)

print(
    "Clean-vs-MC mean MAE         :",
    float(
        np.mean(
            clean_mc_difference_622b
        )
    )
)

print(
    "Clean-vs-MC correlation      :",
    clean_mc_corr_622b
)

print(
    "Candidate true-RUL arrays    :",
    len(
        target_candidates_df_622b
    )
)

print(
    "Training/retraining          : False"
)

print(
    "Official test used           : False"
)

print(
    "Uncertainty signal frozen    : False"
)

print(
    "Abstention rule created      : False"
)

print(
    "\nStep 6.22B completed."
)

In [ ]:
# ============================================================
# Step 6.22C
# Exact RUL target / prediction / error pathway verification
#
# PURPOSE:
#   - Verify which RUL target clean_predictions uses.
#   - Check equivalence of candidate validation targets.
#   - Inspect clean_prediction_df compactly.
#   - Determine what selected_error_621ec represents.
#
# INSPECTION ONLY.
# No retraining.
# No official test.
# No uncertainty threshold.
# No abstention rule.
# ============================================================

import numpy as np
import pandas as pd


print(
    "\n=== Step 6.22C — Exact RUL target / error pathway ==="
)


# ------------------------------------------------------------
# 1. Verify candidate target equality
# ------------------------------------------------------------

required_targets_622c = [
    "y_val_clean",
    "y_val_bias",
    "y_val_clean_620a",
]

print("\n--- Candidate target equality ---")

for name_622c in required_targets_622c:
    print(
        f"{name_622c:20s}:",
        name_622c in globals()
    )


y_clean_622c = np.asarray(
    y_val_clean,
    dtype=np.float64
).reshape(-1)

y_bias_622c = np.asarray(
    y_val_bias,
    dtype=np.float64
).reshape(-1)

y_clean620a_622c = np.asarray(
    y_val_clean_620a,
    dtype=np.float64
).reshape(-1)


print(
    "\nmax |y_val_clean - y_val_bias|      :",
    float(
        np.max(
            np.abs(
                y_clean_622c
                - y_bias_622c
            )
        )
    )
)

print(
    "max |y_val_clean - y_val_clean_620a|:",
    float(
        np.max(
            np.abs(
                y_clean_622c
                - y_clean620a_622c
            )
        )
    )
)


# ------------------------------------------------------------
# 2. Verify against metadata RUL targets
# ------------------------------------------------------------

print(
    "\n--- Validation metadata target comparison ---"
)

metadata_checks_622c = []


if "val_seq_meta" in globals():

    for col_622c in [
        "RUL_raw",
        "RUL_cap_125",
        "RUL_cap_130",
    ]:

        if col_622c not in val_seq_meta.columns:
            continue

        arr_622c = np.asarray(
            val_seq_meta[col_622c],
            dtype=np.float64
        ).reshape(-1)

        if len(arr_622c) != len(
            y_clean_622c
        ):
            continue

        metadata_checks_622c.append(
            {
                "metadata_column":
                    col_622c,

                "max_abs_difference_vs_y_val_clean":
                    float(
                        np.max(
                            np.abs(
                                arr_622c
                                - y_clean_622c
                            )
                        )
                    ),

                "mean_abs_difference_vs_y_val_clean":
                    float(
                        np.mean(
                            np.abs(
                                arr_622c
                                - y_clean_622c
                            )
                        )
                    ),

                "minimum":
                    float(
                        np.min(
                            arr_622c
                        )
                    ),

                "maximum":
                    float(
                        np.max(
                            arr_622c
                        )
                    ),
            }
        )


metadata_checks_df_622c = pd.DataFrame(
    metadata_checks_622c
)

if len(metadata_checks_df_622c):

    display(
        metadata_checks_df_622c
    )

else:

    print(
        "No compatible val_seq_meta target columns found."
    )


# ------------------------------------------------------------
# 3. Exact clean_prediction_df columns
# ------------------------------------------------------------

print(
    "\n--- Exact clean_prediction_df columns ---"
)

print(
    list(
        clean_prediction_df.columns
    )
)


# ------------------------------------------------------------
# 4. Show only columns relevant to prediction / target / error
# ------------------------------------------------------------

keywords_622c = [
    "rul",
    "target",
    "pred",
    "error",
    "resid",
    "actual",
    "true",
]


interesting_cols_622c = [
    col_622c

    for col_622c
    in clean_prediction_df.columns

    if any(
        key_622c
        in str(
            col_622c
        ).lower()

        for key_622c
        in keywords_622c
    )
]


print(
    "\nRelevant columns:",
    interesting_cols_622c
)


if interesting_cols_622c:

    print(
        "\nFirst 5 rows of relevant columns:"
    )

    display(
        clean_prediction_df[
            interesting_cols_622c
        ].head()
    )


# ------------------------------------------------------------
# 5. Numerical comparison of DataFrame columns with known arrays
# ------------------------------------------------------------

reference_arrays_622c = {
    "y_val_clean":
        y_clean_622c,

    "clean_predictions":
        np.asarray(
            clean_predictions,
            dtype=np.float64
        ).reshape(-1),

    "mc_mean_prediction":
        np.asarray(
            mc_mean_prediction,
            dtype=np.float64
        ).reshape(-1),

    "mc_std":
        np.asarray(
            mc_std,
            dtype=np.float64
        ).reshape(-1),

    "selected_error_621ec":
        np.asarray(
            selected_error_621ec,
            dtype=np.float64
        ).reshape(-1),
}


column_match_rows_622c = []


for col_622c in interesting_cols_622c:

    if not pd.api.types.is_numeric_dtype(
        clean_prediction_df[
            col_622c
        ]
    ):
        continue

    arr_622c = np.asarray(
        clean_prediction_df[
            col_622c
        ],
        dtype=np.float64
    ).reshape(-1)

    if len(arr_622c) != 3490:
        continue


    for ref_name_622c, ref_arr_622c in (
        reference_arrays_622c.items()
    ):

        max_diff_622c = float(
            np.max(
                np.abs(
                    arr_622c
                    - ref_arr_622c
                )
            )
        )

        if max_diff_622c < 1e-6:

            column_match_rows_622c.append(
                {
                    "dataframe_column":
                        col_622c,

                    "exactly_matches":
                        ref_name_622c,

                    "maximum_absolute_difference":
                        max_diff_622c,
                }
            )


column_matches_df_622c = pd.DataFrame(
    column_match_rows_622c
)


print(
    "\n--- Exact / near-exact column matches ---"
)

if len(
    column_matches_df_622c
):

    display(
        column_matches_df_622c
    )

else:

    print(
        "No exact numerical matches detected."
    )


# ------------------------------------------------------------
# 6. Construct canonical RUL errors ourselves
# ------------------------------------------------------------

clean_pred_622c = np.asarray(
    clean_predictions,
    dtype=np.float64
).reshape(-1)

mc_mean_622c = np.asarray(
    mc_mean_prediction,
    dtype=np.float64
).reshape(-1)

selected_error_622c = np.asarray(
    selected_error_621ec,
    dtype=np.float64
).reshape(-1)


signed_clean_error_622c = (
    clean_pred_622c
    - y_clean_622c
)

abs_clean_error_622c = np.abs(
    signed_clean_error_622c
)

sq_clean_error_622c = (
    signed_clean_error_622c ** 2
)


signed_mc_error_622c = (
    mc_mean_622c
    - y_clean_622c
)

abs_mc_error_622c = np.abs(
    signed_mc_error_622c
)

sq_mc_error_622c = (
    signed_mc_error_622c ** 2
)


# ------------------------------------------------------------
# 7. Which standard error definition resembles selected_error?
# ------------------------------------------------------------

candidate_error_forms_622c = {
    "clean_signed":
        signed_clean_error_622c,

    "clean_absolute":
        abs_clean_error_622c,

    "clean_squared":
        sq_clean_error_622c,

    "mcmean_signed":
        signed_mc_error_622c,

    "mcmean_absolute":
        abs_mc_error_622c,

    "mcmean_squared":
        sq_mc_error_622c,
}


error_relation_rows_622c = []


for form_name_622c, form_arr_622c in (
    candidate_error_forms_622c.items()
):

    error_relation_rows_622c.append(
        {
            "candidate_error_form":
                form_name_622c,

            "mean_absolute_difference_vs_selected":
                float(
                    np.mean(
                        np.abs(
                            form_arr_622c
                            - selected_error_622c
                        )
                    )
                ),

            "maximum_absolute_difference_vs_selected":
                float(
                    np.max(
                        np.abs(
                            form_arr_622c
                            - selected_error_622c
                        )
                    )
                ),
        }
    )


error_relation_df_622c = pd.DataFrame(
    error_relation_rows_622c
).sort_values(
    "mean_absolute_difference_vs_selected"
)


print(
    "\n--- What resembles selected_error_621ec? ---"
)

display(
    error_relation_df_622c
)


# ------------------------------------------------------------
# 8. Canonical clean-RUL performance summary
# ------------------------------------------------------------

rmse_clean_622c = float(
    np.sqrt(
        np.mean(
            sq_clean_error_622c
        )
    )
)

mae_clean_622c = float(
    np.mean(
        abs_clean_error_622c
    )
)


rmse_mc_622c = float(
    np.sqrt(
        np.mean(
            sq_mc_error_622c
        )
    )
)

mae_mc_622c = float(
    np.mean(
        abs_mc_error_622c
    )
)


print(
    "\n--- Canonical validation RUL-error summary ---"
)

print(
    "Deterministic prediction MAE :",
    mae_clean_622c
)

print(
    "Deterministic prediction RMSE:",
    rmse_clean_622c
)

print(
    "MC-mean prediction MAE       :",
    mae_mc_622c
)

print(
    "MC-mean prediction RMSE      :",
    rmse_mc_622c
)


# ------------------------------------------------------------
# 9. Compact summary
# ------------------------------------------------------------

print(
    "\n=============================================="
)

print(
    "STEP 6.22C COMPACT SUMMARY"
)

print(
    "=============================================="
)

print(
    "Validation windows                :",
    len(
        y_clean_622c
    )
)

print(
    "Target minimum                    :",
    float(
        np.min(
            y_clean_622c
        )
    )
)

print(
    "Target maximum                    :",
    float(
        np.max(
            y_clean_622c
        )
    )
)

print(
    "y_val_clean = y_val_bias          :",
    bool(
        np.allclose(
            y_clean_622c,
            y_bias_622c
        )
    )
)

print(
    "y_val_clean = y_val_clean_620a    :",
    bool(
        np.allclose(
            y_clean_622c,
            y_clean620a_622c
        )
    )
)

print(
    "Deterministic MAE                 :",
    mae_clean_622c
)

print(
    "Deterministic RMSE                :",
    rmse_clean_622c
)

print(
    "MC-mean MAE                       :",
    mae_mc_622c
)

print(
    "MC-mean RMSE                      :",
    rmse_mc_622c
)

print(
    "Training/retraining               : False"
)

print(
    "Official test used                : False"
)

print(
    "Uncertainty signal frozen         : False"
)

print(
    "Abstention rule created           : False"
)

print(
    "\nStep 6.22C completed."
)

In [ ]:
# ============================================================
# Step 6.22D
# Clean-validation MC-uncertainty / RUL-error diagnostic
#
# PURPOSE:
#   Determine whether the existing MC-dropout standard
#   deviation provides useful information about actual RUL
#   prediction error.
#
# CLEAN VALIDATION ONLY.
# NO official test.
# NO uncertainty threshold selection.
# NO abstention rule.
# NO retraining.
# ============================================================

import numpy as np
import pandas as pd


print(
    "\n=== Step 6.22D — MC uncertainty / RUL-error diagnostic ==="
)


# ------------------------------------------------------------
# 1. Canonical arrays
# ------------------------------------------------------------

y_622d = np.asarray(
    y_val_clean,
    dtype=np.float64
).reshape(-1)

pred_622d = np.asarray(
    clean_predictions,
    dtype=np.float64
).reshape(-1)

mc_mean_622d = np.asarray(
    mc_mean_prediction,
    dtype=np.float64
).reshape(-1)

uncertainty_622d = np.asarray(
    mc_std,
    dtype=np.float64
).reshape(-1)


n_622d = len(
    y_622d
)


assert len(pred_622d) == n_622d
assert len(mc_mean_622d) == n_622d
assert len(uncertainty_622d) == n_622d


if not (
    np.isfinite(y_622d).all()
    and np.isfinite(pred_622d).all()
    and np.isfinite(mc_mean_622d).all()
    and np.isfinite(uncertainty_622d).all()
):
    raise RuntimeError(
        "Non-finite values detected."
    )


print(
    "\nValidation windows:",
    n_622d
)

print(
    "All canonical arrays finite: True"
)


# ------------------------------------------------------------
# 2. Verify saved mc_dropout_std if present
# ------------------------------------------------------------

print(
    "\n--- MC uncertainty pathway agreement ---"
)


mc_std_dataframe_match_622d = None
mc_std_dataframe_maxdiff_622d = None


if (
    "clean_prediction_df" in globals()
    and "mc_dropout_std"
    in clean_prediction_df.columns
):

    df_mc_std_622d = np.asarray(
        clean_prediction_df[
            "mc_dropout_std"
        ],
        dtype=np.float64
    ).reshape(-1)

    mc_std_dataframe_maxdiff_622d = float(
        np.max(
            np.abs(
                df_mc_std_622d
                - uncertainty_622d
            )
        )
    )

    mc_std_dataframe_match_622d = bool(
        np.allclose(
            df_mc_std_622d,
            uncertainty_622d,
            rtol=1e-10,
            atol=1e-10
        )
    )

    print(
        "mc_std matches clean_prediction_df['mc_dropout_std']:",
        mc_std_dataframe_match_622d
    )

    print(
        "Maximum absolute difference:",
        mc_std_dataframe_maxdiff_622d
    )

else:

    print(
        "clean_prediction_df['mc_dropout_std'] "
        "not available — using verified mc_std array."
    )


# ------------------------------------------------------------
# 3. Canonical error arrays
# ------------------------------------------------------------

signed_error_det_622d = (
    pred_622d
    - y_622d
)

abs_error_det_622d = np.abs(
    signed_error_det_622d
)


signed_error_mc_622d = (
    mc_mean_622d
    - y_622d
)

abs_error_mc_622d = np.abs(
    signed_error_mc_622d
)


# ------------------------------------------------------------
# 4. Correlation helpers
# ------------------------------------------------------------

def pearson_622d(
    x_622d,
    y_622d_local
):

    x_622d = np.asarray(
        x_622d,
        dtype=np.float64
    )

    y_622d_local = np.asarray(
        y_622d_local,
        dtype=np.float64
    )

    return float(
        np.corrcoef(
            x_622d,
            y_622d_local
        )[0, 1]
    )


def spearman_622d(
    x_622d,
    y_622d_local
):

    x_rank_622d = (
        pd.Series(
            np.asarray(
                x_622d,
                dtype=np.float64
            )
        )
        .rank(
            method="average"
        )
        .to_numpy()
    )

    y_rank_622d = (
        pd.Series(
            np.asarray(
                y_622d_local,
                dtype=np.float64
            )
        )
        .rank(
            method="average"
        )
        .to_numpy()
    )

    return float(
        np.corrcoef(
            x_rank_622d,
            y_rank_622d
        )[0, 1]
    )


# ------------------------------------------------------------
# 5. Overall uncertainty/error association
# ------------------------------------------------------------

association_rows_622d = []


for error_name_622d, error_arr_622d in {
    "deterministic_absolute_error":
        abs_error_det_622d,

    "mc_mean_absolute_error":
        abs_error_mc_622d,

}.items():

    association_rows_622d.append(
        {
            "error_definition":
                error_name_622d,

            "pearson_uncertainty_vs_error":
                pearson_622d(
                    uncertainty_622d,
                    error_arr_622d
                ),

            "spearman_uncertainty_vs_error":
                spearman_622d(
                    uncertainty_622d,
                    error_arr_622d
                ),
        }
    )


association_df_622d = pd.DataFrame(
    association_rows_622d
)


print(
    "\n--- Overall uncertainty / error association ---"
)

display(
    association_df_622d
)


# ------------------------------------------------------------
# 6. Uncertainty distribution
# ------------------------------------------------------------

uncertainty_summary_622d = pd.DataFrame(
    {
        "minimum": [
            float(
                np.min(
                    uncertainty_622d
                )
            )
        ],

        "median": [
            float(
                np.median(
                    uncertainty_622d
                )
            )
        ],

        "mean": [
            float(
                np.mean(
                    uncertainty_622d
                )
            )
        ],

        "p75": [
            float(
                np.percentile(
                    uncertainty_622d,
                    75
                )
            )
        ],

        "p90": [
            float(
                np.percentile(
                    uncertainty_622d,
                    90
                )
            )
        ],

        "p95": [
            float(
                np.percentile(
                    uncertainty_622d,
                    95
                )
            )
        ],

        "maximum": [
            float(
                np.max(
                    uncertainty_622d
                )
            )
        ],
    }
)


print(
    "\n--- MC uncertainty distribution ---"
)

display(
    uncertainty_summary_622d
)


# ------------------------------------------------------------
# 7. Uncertainty deciles
#
# Decile 1 = lowest uncertainty
# Decile 10 = highest uncertainty
# ------------------------------------------------------------

ranking_622d = (
    pd.Series(
        uncertainty_622d
    )
    .rank(
        method="first"
    )
)


uncertainty_decile_622d = (
    pd.qcut(
        ranking_622d,
        q=10,
        labels=False
    )
    + 1
)


decile_frame_622d = pd.DataFrame(
    {
        "uncertainty_decile":
            np.asarray(
                uncertainty_decile_622d,
                dtype=int
            ),

        "uncertainty":
            uncertainty_622d,

        "det_abs_error":
            abs_error_det_622d,

        "mc_abs_error":
            abs_error_mc_622d,
    }
)


decile_summary_622d = (
    decile_frame_622d
    .groupby(
        "uncertainty_decile",
        as_index=False
    )
    .agg(
        windows=(
            "uncertainty",
            "size"
        ),

        mean_uncertainty=(
            "uncertainty",
            "mean"
        ),

        deterministic_MAE=(
            "det_abs_error",
            "mean"
        ),

        deterministic_median_AE=(
            "det_abs_error",
            "median"
        ),

        mc_mean_MAE=(
            "mc_abs_error",
            "mean"
        ),
    )
)


print(
    "\n--- Error by MC-uncertainty decile ---"
)

display(
    decile_summary_622d.round(4)
)


# ------------------------------------------------------------
# 8. Low- vs high-uncertainty quartiles
# ------------------------------------------------------------

q25_622d = float(
    np.percentile(
        uncertainty_622d,
        25
    )
)

q75_622d = float(
    np.percentile(
        uncertainty_622d,
        75
    )
)


low_mask_622d = (
    uncertainty_622d
    <= q25_622d
)

high_mask_622d = (
    uncertainty_622d
    >= q75_622d
)


quartile_summary_622d = pd.DataFrame(
    [
        {
            "uncertainty_group":
                "lowest_25pct",

            "windows":
                int(
                    low_mask_622d.sum()
                ),

            "mean_uncertainty":
                float(
                    uncertainty_622d[
                        low_mask_622d
                    ].mean()
                ),

            "deterministic_MAE":
                float(
                    abs_error_det_622d[
                        low_mask_622d
                    ].mean()
                ),

            "deterministic_RMSE":
                float(
                    np.sqrt(
                        np.mean(
                            signed_error_det_622d[
                                low_mask_622d
                            ] ** 2
                        )
                    )
                ),
        },

        {
            "uncertainty_group":
                "highest_25pct",

            "windows":
                int(
                    high_mask_622d.sum()
                ),

            "mean_uncertainty":
                float(
                    uncertainty_622d[
                        high_mask_622d
                    ].mean()
                ),

            "deterministic_MAE":
                float(
                    abs_error_det_622d[
                        high_mask_622d
                    ].mean()
                ),

            "deterministic_RMSE":
                float(
                    np.sqrt(
                        np.mean(
                            signed_error_det_622d[
                                high_mask_622d
                            ] ** 2
                        )
                    )
                ),
        },
    ]
)


print(
    "\n--- Lowest vs highest uncertainty quartile ---"
)

display(
    quartile_summary_622d.round(4)
)


low_mae_622d = float(
    abs_error_det_622d[
        low_mask_622d
    ].mean()
)

high_mae_622d = float(
    abs_error_det_622d[
        high_mask_622d
    ].mean()
)


if low_mae_622d > 0:

    high_low_mae_ratio_622d = (
        high_mae_622d
        / low_mae_622d
    )

else:

    high_low_mae_ratio_622d = np.nan


# ------------------------------------------------------------
# 9. Highest-error / highest-uncertainty overlap
# ------------------------------------------------------------

n_top_622d = int(
    np.ceil(
        0.25
        * n_622d
    )
)


top_uncertainty_idx_622d = set(
    np.argsort(
        uncertainty_622d
    )[
        -n_top_622d:
    ].tolist()
)


top_error_idx_622d = set(
    np.argsort(
        abs_error_det_622d
    )[
        -n_top_622d:
    ].tolist()
)


overlap_count_622d = len(
    top_uncertainty_idx_622d
    & top_error_idx_622d
)


union_count_622d = len(
    top_uncertainty_idx_622d
    | top_error_idx_622d
)


top_error_capture_pct_622d = (
    100.0
    * overlap_count_622d
    / n_top_622d
)


jaccard_622d = (
    overlap_count_622d
    / union_count_622d
)


print(
    "\n--- High-uncertainty / high-error overlap ---"
)

print(
    "Top-quartile size:",
    n_top_622d
)

print(
    "Overlap count:",
    overlap_count_622d
)

print(
    "High-error windows captured by "
    "highest-uncertainty quartile (%):",
    top_error_capture_pct_622d
)

print(
    "Jaccard overlap:",
    jaccard_622d
)


# ------------------------------------------------------------
# 10. Preliminary uncertainty-only risk–coverage diagnostic
#
# Keep LOWEST uncertainty windows first.
#
# This is descriptive only.
# No operating threshold is selected.
# ------------------------------------------------------------

coverage_levels_622d = [
    1.00,
    0.90,
    0.80,
    0.70,
    0.60,
    0.50,
]


ascending_uncertainty_idx_622d = np.argsort(
    uncertainty_622d
)


baseline_mae_622d = float(
    np.mean(
        abs_error_det_622d
    )
)


risk_coverage_rows_622d = []


for coverage_622d in coverage_levels_622d:

    keep_n_622d = max(
        1,
        int(
            np.floor(
                coverage_622d
                * n_622d
            )
        )
    )

    keep_idx_622d = (
        ascending_uncertainty_idx_622d[
            :keep_n_622d
        ]
    )

    accepted_signed_error_622d = (
        signed_error_det_622d[
            keep_idx_622d
        ]
    )

    accepted_abs_error_622d = np.abs(
        accepted_signed_error_622d
    )


    accepted_mae_622d = float(
        accepted_abs_error_622d.mean()
    )

    accepted_rmse_622d = float(
        np.sqrt(
            np.mean(
                accepted_signed_error_622d ** 2
            )
        )
    )


    risk_coverage_rows_622d.append(
        {
            "coverage":
                coverage_622d,

            "accepted_windows":
                keep_n_622d,

            "rejected_windows":
                n_622d
                - keep_n_622d,

            "accepted_MAE":
                accepted_mae_622d,

            "accepted_RMSE":
                accepted_rmse_622d,

            "MAE_change_vs_100pct":
                accepted_mae_622d
                - baseline_mae_622d,
        }
    )


risk_coverage_df_622d = pd.DataFrame(
    risk_coverage_rows_622d
)


print(
    "\n--- Preliminary uncertainty-only risk–coverage diagnostic ---"
)

display(
    risk_coverage_df_622d.round(4)
)


# ------------------------------------------------------------
# 11. Compact summary
# ------------------------------------------------------------

primary_pearson_622d = float(
    association_df_622d.loc[
        association_df_622d[
            "error_definition"
        ]
        == "deterministic_absolute_error",

        "pearson_uncertainty_vs_error"
    ].iloc[0]
)


primary_spearman_622d = float(
    association_df_622d.loc[
        association_df_622d[
            "error_definition"
        ]
        == "deterministic_absolute_error",

        "spearman_uncertainty_vs_error"
    ].iloc[0]
)


print(
    "\n============================================"
)

print(
    "STEP 6.22D COMPACT SUMMARY"
)

print(
    "============================================"
)

print(
    "Validation windows               :",
    n_622d
)

print(
    "Primary issued RUL prediction    : deterministic"
)

print(
    "Uncertainty candidate            : MC-dropout std"
)

print(
    "Pearson(uncertainty, abs error)  :",
    primary_pearson_622d
)

print(
    "Spearman(uncertainty, abs error) :",
    primary_spearman_622d
)

print(
    "Lowest-quartile MAE              :",
    low_mae_622d
)

print(
    "Highest-quartile MAE             :",
    high_mae_622d
)

print(
    "High/low uncertainty MAE ratio   :",
    high_low_mae_ratio_622d
)

print(
    "Top-error capture by top-UQ (%)  :",
    top_error_capture_pct_622d
)

print(
    "Official test used               : False"
)

print(
    "Uncertainty threshold selected   : False"
)

print(
    "Uncertainty signal frozen        : False"
)

print(
    "Abstention rule created          : False"
)

print(
    "\nStep 6.22D completed."
)

In [ ]:
# ============================================================
# Step 6.22E
# MC uncertainty / RUL-phase confounding diagnostic
#
# PURPOSE:
#   Determine whether MC-dropout uncertainty is primarily
#   associated with RUL/life-cycle position rather than
#   prediction error.
#
# CLEAN VALIDATION ONLY.
# NO threshold selection.
# NO abstention.
# NO official test.
# ============================================================

import numpy as np
import pandas as pd


print(
    "\n=== Step 6.22E — MC uncertainty / RUL-phase diagnostic ==="
)


# ------------------------------------------------------------
# 1. Reuse canonical verified arrays
# ------------------------------------------------------------

required_622e = [
    "y_622d",
    "pred_622d",
    "uncertainty_622d",
    "abs_error_det_622d",
]

for name_622e in required_622e:
    if name_622e not in globals():
        raise RuntimeError(
            f"Required Step-6.22D object missing: {name_622e}"
        )


target_622e = np.asarray(
    y_622d,
    dtype=np.float64
).reshape(-1)

prediction_622e = np.asarray(
    pred_622d,
    dtype=np.float64
).reshape(-1)

uncertainty_622e = np.asarray(
    uncertainty_622d,
    dtype=np.float64
).reshape(-1)

abs_error_622e = np.asarray(
    abs_error_det_622d,
    dtype=np.float64
).reshape(-1)


n_622e = len(target_622e)

assert len(prediction_622e) == n_622e
assert len(uncertainty_622e) == n_622e
assert len(abs_error_622e) == n_622e


print(
    "Validation windows:",
    n_622e
)


# ------------------------------------------------------------
# 2. Correlation helpers
# ------------------------------------------------------------

def pearson_622e(x, y):

    x = np.asarray(
        x,
        dtype=np.float64
    )

    y = np.asarray(
        y,
        dtype=np.float64
    )

    return float(
        np.corrcoef(
            x,
            y
        )[0, 1]
    )


def spearman_622e(x, y):

    x_rank = (
        pd.Series(x)
        .rank(method="average")
        .to_numpy()
    )

    y_rank = (
        pd.Series(y)
        .rank(method="average")
        .to_numpy()
    )

    return float(
        np.corrcoef(
            x_rank,
            y_rank
        )[0, 1]
    )


# ------------------------------------------------------------
# 3. Overall target / uncertainty relationship
# ------------------------------------------------------------

target_uncertainty_pearson_622e = (
    pearson_622e(
        target_622e,
        uncertainty_622e
    )
)

target_uncertainty_spearman_622e = (
    spearman_622e(
        target_622e,
        uncertainty_622e
    )
)


target_error_pearson_622e = (
    pearson_622e(
        target_622e,
        abs_error_622e
    )
)

target_error_spearman_622e = (
    spearman_622e(
        target_622e,
        abs_error_622e
    )
)


print(
    "\n--- Overall association ---"
)

print(
    "Pearson(target, uncertainty) :",
    target_uncertainty_pearson_622e
)

print(
    "Spearman(target, uncertainty):",
    target_uncertainty_spearman_622e
)

print(
    "Pearson(target, abs error)   :",
    target_error_pearson_622e
)

print(
    "Spearman(target, abs error)  :",
    target_error_spearman_622e
)


# ------------------------------------------------------------
# 4. Define RUL regions
#
# Target is confirmed RUL_cap_125.
#
# These are descriptive bins only.
# They are NOT operating thresholds.
# ------------------------------------------------------------

def rul_phase_622e(value_622e):

    if value_622e == 125:
        return "capped_125"

    elif value_622e > 100:
        return "101_124"

    elif value_622e > 75:
        return "76_100"

    elif value_622e > 50:
        return "51_75"

    elif value_622e > 25:
        return "26_50"

    else:
        return "0_25"


phase_order_622e = [
    "capped_125",
    "101_124",
    "76_100",
    "51_75",
    "26_50",
    "0_25",
]


phase_622e = np.array(
    [
        rul_phase_622e(v)
        for v in target_622e
    ],
    dtype=object
)


phase_df_622e = pd.DataFrame(
    {
        "rul_phase":
            phase_622e,

        "target":
            target_622e,

        "uncertainty":
            uncertainty_622e,

        "absolute_error":
            abs_error_622e,
    }
)


# ------------------------------------------------------------
# 5. RUL-phase summary
# ------------------------------------------------------------

phase_summary_622e = (
    phase_df_622e
    .groupby(
        "rul_phase",
        as_index=False
    )
    .agg(
        windows=(
            "target",
            "size"
        ),

        mean_target=(
            "target",
            "mean"
        ),

        mean_uncertainty=(
            "uncertainty",
            "mean"
        ),

        median_uncertainty=(
            "uncertainty",
            "median"
        ),

        mean_absolute_error=(
            "absolute_error",
            "mean"
        ),

        median_absolute_error=(
            "absolute_error",
            "median"
        ),
    )
)


phase_summary_622e[
    "rul_phase"
] = pd.Categorical(
    phase_summary_622e[
        "rul_phase"
    ],
    categories=phase_order_622e,
    ordered=True
)


phase_summary_622e = (
    phase_summary_622e
    .sort_values(
        "rul_phase"
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n--- RUL-phase summary ---"
)

display(
    phase_summary_622e.round(4)
)


# ------------------------------------------------------------
# 6. Within-phase uncertainty/error association
# ------------------------------------------------------------

within_phase_rows_622e = []


for phase_name_622e in phase_order_622e:

    phase_mask_622e = (
        phase_622e
        == phase_name_622e
    )

    phase_n_622e = int(
        phase_mask_622e.sum()
    )


    if phase_n_622e < 3:

        within_phase_rows_622e.append(
            {
                "rul_phase":
                    phase_name_622e,

                "windows":
                    phase_n_622e,

                "pearson_uncertainty_error":
                    np.nan,

                "spearman_uncertainty_error":
                    np.nan,
            }
        )

        continue


    phase_u_622e = (
        uncertainty_622e[
            phase_mask_622e
        ]
    )

    phase_e_622e = (
        abs_error_622e[
            phase_mask_622e
        ]
    )


    # Avoid meaningless correlation
    # if either variable is constant.
    if (
        np.std(phase_u_622e) == 0
        or np.std(phase_e_622e) == 0
    ):

        pearson_value_622e = np.nan
        spearman_value_622e = np.nan

    else:

        pearson_value_622e = (
            pearson_622e(
                phase_u_622e,
                phase_e_622e
            )
        )

        spearman_value_622e = (
            spearman_622e(
                phase_u_622e,
                phase_e_622e
            )
        )


    within_phase_rows_622e.append(
        {
            "rul_phase":
                phase_name_622e,

            "windows":
                phase_n_622e,

            "pearson_uncertainty_error":
                pearson_value_622e,

            "spearman_uncertainty_error":
                spearman_value_622e,
        }
    )


within_phase_df_622e = pd.DataFrame(
    within_phase_rows_622e
)


print(
    "\n--- Within-RUL-phase uncertainty/error association ---"
)

display(
    within_phase_df_622e.round(4)
)


# ------------------------------------------------------------
# 7. Compare capped vs non-capped target regions
# ------------------------------------------------------------

capped_mask_622e = (
    target_622e
    == 125.0
)

noncapped_mask_622e = (
    target_622e
    < 125.0
)


capped_summary_622e = pd.DataFrame(
    [
        {
            "target_region":
                "capped_125",

            "windows":
                int(
                    capped_mask_622e.sum()
                ),

            "mean_uncertainty":
                float(
                    uncertainty_622e[
                        capped_mask_622e
                    ].mean()
                ),

            "MAE":
                float(
                    abs_error_622e[
                        capped_mask_622e
                    ].mean()
                ),
        },

        {
            "target_region":
                "uncapped_below_125",

            "windows":
                int(
                    noncapped_mask_622e.sum()
                ),

            "mean_uncertainty":
                float(
                    uncertainty_622e[
                        noncapped_mask_622e
                    ].mean()
                ),

            "MAE":
                float(
                    abs_error_622e[
                        noncapped_mask_622e
                    ].mean()
                ),
        },
    ]
)


print(
    "\n--- Capped vs non-capped target region ---"
)

display(
    capped_summary_622e.round(4)
)


# ------------------------------------------------------------
# 8. Check where the highest uncertainty lies
# ------------------------------------------------------------

q75_u_622e = float(
    np.percentile(
        uncertainty_622e,
        75
    )
)


high_u_mask_622e = (
    uncertainty_622e
    >= q75_u_622e
)


high_u_phase_distribution_622e = (
    pd.Series(
        phase_622e[
            high_u_mask_622e
        ]
    )
    .value_counts(
        normalize=True
    )
    .reindex(
        phase_order_622e,
        fill_value=0
    )
    .mul(
        100
    )
    .rename(
        "highest_uncertainty_quartile_pct"
    )
    .reset_index()
    .rename(
        columns={
            "index":
                "rul_phase"
        }
    )
)


print(
    "\n--- Composition of highest-uncertainty quartile ---"
)

display(
    high_u_phase_distribution_622e.round(3)
)


# ------------------------------------------------------------
# 9. Compact summary
# ------------------------------------------------------------

print(
    "\n============================================"
)

print(
    "STEP 6.22E COMPACT SUMMARY"
)

print(
    "============================================"
)

print(
    "Validation windows                 :",
    n_622e
)

print(
    "Canonical target                   : RUL_cap_125"
)

print(
    "Pearson(target, uncertainty)       :",
    target_uncertainty_pearson_622e
)

print(
    "Spearman(target, uncertainty)      :",
    target_uncertainty_spearman_622e
)

print(
    "Pearson(target, abs error)         :",
    target_error_pearson_622e
)

print(
    "Spearman(target, abs error)        :",
    target_error_spearman_622e
)

print(
    "Official test used                 : False"
)

print(
    "Uncertainty threshold selected     : False"
)

print(
    "Uncertainty signal frozen          : False"
)

print(
    "Abstention rule created            : False"
)

print(
    "\nStep 6.22E completed."
)

In [ ]:
# ============================================================
# Step 6.22F-A
# Inspect DEVELOPMENT MC uncertainty / prediction objects
#
# PURPOSE:
#   Identify an existing development-set MC-dropout pathway
#   before constructing prediction-conditioned uncertainty.
#
# INSPECTION ONLY.
# NO recalibration.
# NO threshold selection.
# NO retraining.
# NO official test.
# ============================================================

import numpy as np
import pandas as pd


print(
    "\n=== Step 6.22F-A — Development MC-pathway inspection ==="
)


# ------------------------------------------------------------
# 1. Known reference lengths
# ------------------------------------------------------------

DEV_N_622fa = 14241
VAL_N_622fa = 3490


print(
    "Expected development windows :",
    DEV_N_622fa
)

print(
    "Expected validation windows  :",
    VAL_N_622fa
)


# ------------------------------------------------------------
# 2. Search namespace safely
# ------------------------------------------------------------

namespace_622fa = list(
    globals().items()
)

rows_622fa = []


search_terms_622fa = [
    "mc",
    "dropout",
    "uncert",
    "std",
    "prediction",
    "pred",
]


for name_622fa, obj_622fa in namespace_622fa:

    lower_622fa = str(
        name_622fa
    ).lower()

    if not any(
        term_622fa in lower_622fa
        for term_622fa in search_terms_622fa
    ):
        continue


    obj_type_622fa = type(
        obj_622fa
    ).__name__


    shape_622fa = None

    try:
        shape_622fa = tuple(
            obj_622fa.shape
        )
    except Exception:
        pass


    length_622fa = None

    try:
        length_622fa = len(
            obj_622fa
        )
    except Exception:
        pass


    development_aligned_622fa = False
    validation_aligned_622fa = False


    if shape_622fa is not None:

        development_aligned_622fa = (
            DEV_N_622fa
            in shape_622fa
        )

        validation_aligned_622fa = (
            VAL_N_622fa
            in shape_622fa
        )

    elif length_622fa is not None:

        development_aligned_622fa = (
            length_622fa
            == DEV_N_622fa
        )

        validation_aligned_622fa = (
            length_622fa
            == VAL_N_622fa
        )


    rows_622fa.append(
        {
            "object_name":
                name_622fa,

            "type":
                obj_type_622fa,

            "shape":
                shape_622fa,

            "length":
                length_622fa,

            "development_aligned":
                development_aligned_622fa,

            "validation_aligned":
                validation_aligned_622fa,
        }
    )


objects_622fa = pd.DataFrame(
    rows_622fa
)


# ------------------------------------------------------------
# 3. Priority DEVELOPMENT-aligned objects
# ------------------------------------------------------------

if len(
    objects_622fa
) > 0:

    dev_objects_622fa = (
        objects_622fa[
            objects_622fa[
                "development_aligned"
            ]
        ]
        .copy()
        .sort_values(
            "object_name"
        )
        .reset_index(
            drop=True
        )
    )

else:

    dev_objects_622fa = pd.DataFrame()


print(
    "\n--- DEVELOPMENT-aligned uncertainty/prediction candidates ---"
)


if len(
    dev_objects_622fa
) == 0:

    print(
        "No obvious development-aligned MC object found."
    )

else:

    display(
        dev_objects_622fa[
            [
                "object_name",
                "type",
                "shape",
            ]
        ]
    )


# ------------------------------------------------------------
# 4. Explicitly inspect likely names
# ------------------------------------------------------------

likely_names_622fa = [
    "mc_predictions",
    "mc_mean_prediction",
    "mc_std",
    "clean_predictions",
    "dev_predictions",
    "development_predictions",
    "mc_predictions_dev",
    "dev_mc_predictions",
    "mc_mean_prediction_dev",
    "dev_mc_mean_prediction",
    "mc_std_dev",
    "dev_mc_std",
]


explicit_rows_622fa = []


for name_622fa in likely_names_622fa:

    available_622fa = (
        name_622fa
        in globals()
    )


    if available_622fa:

        obj_622fa = globals()[
            name_622fa
        ]

        try:
            shape_622fa = tuple(
                obj_622fa.shape
            )
        except Exception:
            shape_622fa = None

    else:

        shape_622fa = None


    explicit_rows_622fa.append(
        {
            "object_name":
                name_622fa,

            "available":
                available_622fa,

            "shape":
                shape_622fa,
        }
    )


explicit_df_622fa = pd.DataFrame(
    explicit_rows_622fa
)


print(
    "\n--- Explicit likely-object check ---"
)

display(
    explicit_df_622fa
)


# ------------------------------------------------------------
# 5. Search development-sized DataFrames for predictions
# ------------------------------------------------------------

dev_df_rows_622fa = []


for name_622fa, obj_622fa in namespace_622fa:

    if not isinstance(
        obj_622fa,
        pd.DataFrame
    ):
        continue


    if len(
        obj_622fa
    ) != DEV_N_622fa:
        continue


    relevant_cols_622fa = [
        str(col_622fa)
        for col_622fa
        in obj_622fa.columns
        if any(
            term_622fa
            in str(
                col_622fa
            ).lower()
            for term_622fa
            in [
                "pred",
                "mc",
                "std",
                "uncert",
                "target",
                "rul",
                "error",
            ]
        )
    ]


    if len(
        relevant_cols_622fa
    ) == 0:
        continue


    dev_df_rows_622fa.append(
        {
            "object_name":
                name_622fa,

            "rows":
                len(
                    obj_622fa
                ),

            "relevant_columns":
                ", ".join(
                    relevant_cols_622fa
                ),
        }
    )


dev_df_candidates_622fa = pd.DataFrame(
    dev_df_rows_622fa
)


print(
    "\n--- Development DataFrames containing useful columns ---"
)


if len(
    dev_df_candidates_622fa
) == 0:

    print(
        "No development-sized DataFrame with obvious "
        "prediction/uncertainty columns found."
    )

else:

    display(
        dev_df_candidates_622fa
    )


# ------------------------------------------------------------
# 6. Compact readiness result
# ------------------------------------------------------------

dev_candidate_count_622fa = int(
    len(
        dev_objects_622fa
    )
)


print(
    "\n============================================"
)

print(
    "STEP 6.22F-A COMPACT SUMMARY"
)

print(
    "============================================"
)

print(
    "Development windows expected        :",
    DEV_N_622fa
)

print(
    "Development-aligned candidates      :",
    dev_candidate_count_622fa
)

print(
    "Validation MC std already verified  :",
    "mc_std" in globals()
)

print(
    "Calibration performed               : False"
)

print(
    "Uncertainty threshold selected      : False"
)

print(
    "Uncertainty signal frozen           : False"
)

print(
    "Official test used                  : False"
)

print(
    "Abstention rule created             : False"
)

print(
    "\nStep 6.22F-A completed."
)

In [ ]:
# ============================================================
# Step 6.22F-B
# Identify the exact existing MC-dropout model / pathway
#
# INSPECTION ONLY.
# NO prediction execution.
# NO retraining.
# NO calibration.
# NO official test.
# ============================================================

import inspect
import numpy as np
import pandas as pd


print(
    "\n=== Step 6.22F-B — Exact MC-dropout pathway inspection ==="
)


# ------------------------------------------------------------
# 1. Core object availability
# ------------------------------------------------------------

core_names_622fb = [
    "X_dev_clean",
    "X_val_clean",
    "mc_predictions",
    "mc_mean_prediction",
    "mc_std",
    "clean_predictions",
]


print(
    "\n--- Core objects ---"
)

for name_622fb in core_names_622fb:

    available_622fb = (
        name_622fb in globals()
    )

    if available_622fb:

        obj_622fb = globals()[
            name_622fb
        ]

        try:
            shape_622fb = tuple(
                obj_622fb.shape
            )
        except Exception:
            shape_622fb = None

    else:

        shape_622fb = None


    print(
        f"{name_622fb:<22}: "
        f"{available_622fb}  "
        f"shape={shape_622fb}"
    )


# ------------------------------------------------------------
# 2. Search for model-like objects
# ------------------------------------------------------------

namespace_622fb = list(
    globals().items()
)

model_rows_622fb = []


for name_622fb, obj_622fb in namespace_622fb:

    # Avoid classes themselves.
    if inspect.isclass(
        obj_622fb
    ):
        continue

    has_predict_622fb = callable(
        getattr(
            obj_622fb,
            "predict",
            None
        )
    )

    has_fit_622fb = callable(
        getattr(
            obj_622fb,
            "fit",
            None
        )
    )

    has_call_622fb = callable(
        obj_622fb
    )

    class_name_622fb = (
        type(
            obj_622fb
        ).__name__
    )

    module_name_622fb = (
        type(
            obj_622fb
        ).__module__
    )

    lower_name_622fb = str(
        name_622fb
    ).lower()

    likely_model_name_622fb = any(
        term_622fb in lower_name_622fb
        for term_622fb in [
            "model",
            "network",
            "gru",
            "lstm",
            "cnn",
            "rul",
        ]
    )


    if not (
        has_predict_622fb
        or (
            likely_model_name_622fb
            and has_call_622fb
        )
    ):
        continue


    input_shape_622fb = None
    output_shape_622fb = None


    try:
        input_shape_622fb = (
            obj_622fb.input_shape
        )
    except Exception:
        pass


    try:
        output_shape_622fb = (
            obj_622fb.output_shape
        )
    except Exception:
        pass


    model_rows_622fb.append(
        {
            "object_name":
                name_622fb,

            "class":
                class_name_622fb,

            "module":
                module_name_622fb,

            "has_predict":
                has_predict_622fb,

            "has_fit":
                has_fit_622fb,

            "callable":
                has_call_622fb,

            "input_shape":
                str(
                    input_shape_622fb
                ),

            "output_shape":
                str(
                    output_shape_622fb
                ),
        }
    )


model_candidates_622fb = pd.DataFrame(
    model_rows_622fb
)


print(
    "\n--- Model-like candidates ---"
)


if len(
    model_candidates_622fb
) == 0:

    print(
        "No model-like object found."
    )

else:

    display(
        model_candidates_622fb
    )


# ------------------------------------------------------------
# 3. Search for MC / dropout / prediction helper functions
# ------------------------------------------------------------

function_rows_622fb = []


for name_622fb, obj_622fb in namespace_622fb:

    if not inspect.isfunction(
        obj_622fb
    ):
        continue


    lower_name_622fb = str(
        name_622fb
    ).lower()


    if not any(
        term_622fb in lower_name_622fb
        for term_622fb in [
            "mc",
            "dropout",
            "predict",
            "uncert",
        ]
    ):
        continue


    try:

        signature_622fb = str(
            inspect.signature(
                obj_622fb
            )
        )

    except Exception:

        signature_622fb = (
            "<signature unavailable>"
        )


    function_rows_622fb.append(
        {
            "function_name":
                name_622fb,

            "signature":
                signature_622fb,
        }
    )


function_candidates_622fb = pd.DataFrame(
    function_rows_622fb
)


print(
    "\n--- MC / dropout / prediction helper functions ---"
)


if len(
    function_candidates_622fb
) == 0:

    print(
        "No obvious helper function found."
    )

else:

    display(
        function_candidates_622fb
    )


# ------------------------------------------------------------
# 4. Search likely MC-related scalar configuration
# ------------------------------------------------------------

config_rows_622fb = []


for name_622fb, obj_622fb in namespace_622fb:

    lower_name_622fb = str(
        name_622fb
    ).lower()


    if not any(
        term_622fb in lower_name_622fb
        for term_622fb in [
            "mc_pass",
            "mc_sample",
            "dropout_rate",
            "seed",
        ]
    ):
        continue


    if isinstance(
        obj_622fb,
        (
            int,
            float,
            np.integer,
            np.floating,
            str,
            bool,
        )
    ):

        config_rows_622fb.append(
            {
                "object_name":
                    name_622fb,

                "type":
                    type(
                        obj_622fb
                    ).__name__,

                "value":
                    obj_622fb,
            }
        )


config_candidates_622fb = pd.DataFrame(
    config_rows_622fb
)


print(
    "\n--- Likely MC configuration values ---"
)


if len(
    config_candidates_622fb
) == 0:

    print(
        "No obvious scalar MC configuration found."
    )

else:

    display(
        config_candidates_622fb
    )


# ------------------------------------------------------------
# 5. Validation MC structural fingerprint
# ------------------------------------------------------------

print(
    "\n--- Existing validation MC fingerprint ---"
)


if (
    "mc_predictions" in globals()
    and
    "mc_mean_prediction" in globals()
    and
    "mc_std" in globals()
):

    mc_pred_622fb = np.asarray(
        mc_predictions,
        dtype=float
    )

    mc_mean_622fb = np.asarray(
        mc_mean_prediction,
        dtype=float
    ).reshape(-1)

    mc_std_622fb = np.asarray(
        mc_std,
        dtype=float
    ).reshape(-1)


    print(
        "MC prediction matrix shape :",
        mc_pred_622fb.shape
    )

    print(
        "MC passes                  :",
        mc_pred_622fb.shape[0]
    )

    print(
        "Prediction windows          :",
        mc_pred_622fb.shape[1]
    )


    reconstructed_mean_622fb = (
        np.mean(
            mc_pred_622fb,
            axis=0
        )
    )

    reconstructed_std0_622fb = (
        np.std(
            mc_pred_622fb,
            axis=0,
            ddof=0
        )
    )

    reconstructed_std1_622fb = (
        np.std(
            mc_pred_622fb,
            axis=0,
            ddof=1
        )
    )


    print(
        "Mean reconstruction max error:",
        float(
            np.max(
                np.abs(
                    reconstructed_mean_622fb
                    -
                    mc_mean_622fb
                )
            )
        )
    )


    print(
        "STD ddof=0 max error         :",
        float(
            np.max(
                np.abs(
                    reconstructed_std0_622fb
                    -
                    mc_std_622fb
                )
            )
        )
    )


    print(
        "STD ddof=1 max error         :",
        float(
            np.max(
                np.abs(
                    reconstructed_std1_622fb
                    -
                    mc_std_622fb
                )
            )
        )
    )


# ------------------------------------------------------------
# 6. Compact summary
# ------------------------------------------------------------

print(
    "\n============================================"
)

print(
    "STEP 6.22F-B COMPACT SUMMARY"
)

print(
    "============================================"
)

print(
    "Development sequence available :",
    "X_dev_clean" in globals()
)

if "X_dev_clean" in globals():

    print(
        "Development sequence shape     :",
        tuple(
            X_dev_clean.shape
        )
    )


print(
    "Validation MC passes           :",
    (
        np.asarray(
            mc_predictions
        ).shape[0]
        if "mc_predictions" in globals()
        else None
    )
)

print(
    "Model-like candidates found    :",
    len(
        model_candidates_622fb
    )
)

print(
    "MC helper functions found      :",
    len(
        function_candidates_622fb
    )
)

print(
    "Prediction execution performed : False"
)

print(
    "Retraining performed           : False"
)

print(
    "Calibration performed          : False"
)

print(
    "Official test used             : False"
)

print(
    "Uncertainty signal frozen      : False"
)

print(
    "Abstention rule created        : False"
)

print(
    "\nStep 6.22F-B completed."
)

In [ ]:
# ============================================================
# Step 6.22F-C
# Exact MC-dropout helper / model inspection
#
# INSPECTION ONLY.
# NO prediction execution.
# NO retraining.
# NO calibration.
# NO official test.
# ============================================================

import inspect
import numpy as np


print(
    "\n=== Step 6.22F-C — Exact MC helper/model verification ==="
)


# ------------------------------------------------------------
# 1. Required objects
# ------------------------------------------------------------

required_622fc = [
    "frozen_gru",
    "mc_dropout_predictions",
    "X_dev_clean",
]


print(
    "\n--- Required objects ---"
)

for name_622fc in required_622fc:
    print(
        f"{name_622fc:<24}:",
        name_622fc in globals()
    )


if not all(
    name_622fc in globals()
    for name_622fc in required_622fc
):
    raise RuntimeError(
        "Required MC-pathway objects are missing."
    )


# ------------------------------------------------------------
# 2. Exact helper signature
# ------------------------------------------------------------

print(
    "\n--- Exact MC helper signature ---"
)

mc_signature_622fc = inspect.signature(
    mc_dropout_predictions
)

print(
    "mc_dropout_predictions",
    mc_signature_622fc
)


# ------------------------------------------------------------
# 3. Compact helper-source fingerprint
# ------------------------------------------------------------

print(
    "\n--- Relevant MC helper source lines ---"
)


try:

    mc_source_622fc = inspect.getsource(
        mc_dropout_predictions
    )

    relevant_terms_622fc = [
        "train(",
        "eval(",
        "dropout",
        "manual_seed",
        "seed",
        "no_grad",
        "inference_mode",
        "n_passes",
        "batch_size",
        "predict",
        "stack",
        "mean",
        "std",
        "return",
    ]

    source_lines_622fc = (
        mc_source_622fc.splitlines()
    )


    printed_622fc = 0

    for line_number_622fc, line_622fc in enumerate(
        source_lines_622fc,
        start=1
    ):

        lower_line_622fc = (
            line_622fc.lower()
        )

        if any(
            term_622fc.lower()
            in lower_line_622fc
            for term_622fc
            in relevant_terms_622fc
        ):

            print(
                f"{line_number_622fc:03d}: "
                f"{line_622fc}"
            )

            printed_622fc += 1


    print(
        "\nRelevant source lines printed:",
        printed_622fc
    )


except Exception as exc_622fc:

    print(
        "Source unavailable:",
        repr(
            exc_622fc
        )
    )


# ------------------------------------------------------------
# 4. Frozen model structure
# ------------------------------------------------------------

print(
    "\n--- Frozen GRU model ---"
)

print(
    "Class:",
    type(
        frozen_gru
    ).__name__
)

print(
    "Current training flag:",
    getattr(
        frozen_gru,
        "training",
        None
    )
)


# Count parameters when supported.
try:

    total_params_622fc = sum(
        p.numel()
        for p in frozen_gru.parameters()
    )

    trainable_params_622fc = sum(
        p.numel()
        for p in frozen_gru.parameters()
        if p.requires_grad
    )

    print(
        "Total parameters:",
        total_params_622fc
    )

    print(
        "Trainable-flag parameters:",
        trainable_params_622fc
    )

except Exception as exc_622fc:

    print(
        "Parameter inspection unavailable:",
        repr(
            exc_622fc
        )
    )


# ------------------------------------------------------------
# 5. Dropout-module inspection
# ------------------------------------------------------------

print(
    "\n--- Dropout modules in frozen_gru ---"
)

dropout_rows_622fc = []


try:

    for module_name_622fc, module_622fc in (
        frozen_gru.named_modules()
    ):

        class_name_622fc = (
            type(
                module_622fc
            ).__name__
        )

        if (
            "dropout"
            in class_name_622fc.lower()
        ):

            dropout_rows_622fc.append(
                (
                    module_name_622fc,
                    class_name_622fc,
                    getattr(
                        module_622fc,
                        "p",
                        None
                    ),
                    getattr(
                        module_622fc,
                        "training",
                        None
                    ),
                )
            )


    if len(
        dropout_rows_622fc
    ) == 0:

        print(
            "No explicit Dropout module found."
        )

    else:

        for row_622fc in dropout_rows_622fc:

            print(
                "name=",
                row_622fc[0],
                "| class=",
                row_622fc[1],
                "| p=",
                row_622fc[2],
                "| training=",
                row_622fc[3]
            )


except Exception as exc_622fc:

    print(
        "Dropout inspection unavailable:",
        repr(
            exc_622fc
        )
    )


# ------------------------------------------------------------
# 6. Device inspection
# ------------------------------------------------------------

print(
    "\n--- Possible device objects ---"
)


device_candidates_622fc = []


for name_622fc, obj_622fc in list(
    globals().items()
):

    lower_name_622fc = (
        str(
            name_622fc
        ).lower()
    )

    if (
        "device"
        not in lower_name_622fc
    ):
        continue

    try:

        value_622fc = str(
            obj_622fc
        )

    except Exception:

        value_622fc = (
            "<unprintable>"
        )


    device_candidates_622fc.append(
        (
            name_622fc,
            type(
                obj_622fc
            ).__name__,
            value_622fc
        )
    )


if len(
    device_candidates_622fc
) == 0:

    print(
        "No obvious device-named object found."
    )

else:

    for row_622fc in device_candidates_622fc:

        print(
            f"{row_622fc[0]:<25} "
            f"type={row_622fc[1]:<15} "
            f"value={row_622fc[2]}"
        )


# ------------------------------------------------------------
# 7. Existing MC convention confirmation
# ------------------------------------------------------------

print(
    "\n--- Existing MC convention ---"
)


mc_existing_622fc = np.asarray(
    mc_predictions,
    dtype=float
)

mc_std_existing_622fc = np.asarray(
    mc_std,
    dtype=float
).reshape(-1)


std0_622fc = np.std(
    mc_existing_622fc,
    axis=0,
    ddof=0
)

std1_622fc = np.std(
    mc_existing_622fc,
    axis=0,
    ddof=1
)


error_std0_622fc = float(
    np.max(
        np.abs(
            std0_622fc
            -
            mc_std_existing_622fc
        )
    )
)

error_std1_622fc = float(
    np.max(
        np.abs(
            std1_622fc
            -
            mc_std_existing_622fc
        )
    )
)


print(
    "MC passes:",
    mc_existing_622fc.shape[0]
)

print(
    "ddof=0 maximum error:",
    error_std0_622fc
)

print(
    "ddof=1 maximum error:",
    error_std1_622fc
)

print(
    "Detected stored convention:",
    (
        "population_std_ddof_0"
        if error_std0_622fc
        <
        error_std1_622fc
        else
        "sample_std_ddof_1"
    )
)


# ------------------------------------------------------------
# 8. Compact summary
# ------------------------------------------------------------

print(
    "\n============================================"
)

print(
    "STEP 6.22F-C COMPACT SUMMARY"
)

print(
    "============================================"
)

print(
    "Frozen GRU available          :",
    True
)

print(
    "MC helper available           :",
    True
)

print(
    "Development windows           :",
    X_dev_clean.shape[0]
)

print(
    "Existing validation MC passes :",
    mc_existing_622fc.shape[0]
)

print(
    "Explicit dropout modules      :",
    len(
        dropout_rows_622fc
    )
)

print(
    "Detected std convention       :",
    (
        "ddof=0"
        if error_std0_622fc
        <
        error_std1_622fc
        else
        "ddof=1"
    )
)

print(
    "Prediction execution performed: False"
)

print(
    "Retraining performed          : False"
)

print(
    "Calibration performed         : False"
)

print(
    "Official test used            : False"
)

print(
    "Uncertainty signal frozen     : False"
)

print(
    "Abstention rule created       : False"
)

print(
    "\nStep 6.22F-C completed."
)

In [ ]:
# ============================================================
# Step 6.22F-C2
# Final compact MC-pathway verification
# INSPECTION ONLY
# ============================================================

import inspect
import torch

print("\n=== STEP 6.22F-C2 FINAL MC CHECK ===")

# 1. Model device
try:
    model_device_622fc2 = next(frozen_gru.parameters()).device
except StopIteration:
    model_device_622fc2 = None

print("Model device                 :", model_device_622fc2)


# 2. Explicit dropout modules / probabilities
dropout_info_622fc2 = []

for name_622fc2, module_622fc2 in frozen_gru.named_modules():
    if isinstance(module_622fc2, torch.nn.Dropout):
        dropout_info_622fc2.append(
            (
                name_622fc2,
                float(module_622fc2.p)
            )
        )

print("Explicit dropout modules     :", dropout_info_622fc2)


# 3. Exact return statements from MC helper
source_622fc2 = inspect.getsource(
    mc_dropout_predictions
)

return_lines_622fc2 = [
    line.strip()
    for line in source_622fc2.splitlines()
    if line.strip().startswith("return")
]

print("MC helper return line(s)     :", return_lines_622fc2)


# 4. Existing validation MC structure
print("Existing MC matrix shape     :", mc_predictions.shape)
print("Existing MC std shape        :", mc_std.shape)


# 5. Stored standard-deviation convention
std0_error_622fc2 = float(
    abs(
        np.std(mc_predictions, axis=0, ddof=0)
        -
        mc_std
    ).max()
)

std1_error_622fc2 = float(
    abs(
        np.std(mc_predictions, axis=0, ddof=1)
        -
        mc_std
    ).max()
)

print(
    "Stored std convention        :",
    "ddof=0"
    if std0_error_622fc2 < std1_error_622fc2
    else "ddof=1"
)

print("Prediction execution         : False")
print("Official test used           : False")
print("=== END ===")

In [ ]:
# ============================================================
# Step 6.22F-D
# Generate CLEAN DEVELOPMENT MC-dropout uncertainty
#
# Uses:
#   - frozen GRU
#   - existing verified MC helper
#   - 50 stochastic passes
#   - seed = 42
#   - ddof = 0 convention
#
# IMPORTANT:
#   No retraining
#   No official test
#   No uncertainty threshold selected
#   No abstention rule created
#   No files written
# ============================================================

import time
import numpy as np
import torch

print("\n=== Step 6.22F-D — Development MC-dropout execution ===")

# ------------------------------------------------------------
# 1. Required-object verification
# ------------------------------------------------------------

required_622fd = [
    "frozen_gru",
    "mc_dropout_predictions",
    "X_dev_clean",
]

print("\n--- Required objects ---")

missing_622fd = []

for name_622fd in required_622fd:
    available_622fd = name_622fd in globals()

    print(
        f"{name_622fd:<28}:",
        available_622fd
    )

    if not available_622fd:
        missing_622fd.append(
            name_622fd
        )

if missing_622fd:
    raise RuntimeError(
        "Missing required objects: "
        + ", ".join(missing_622fd)
    )


# ------------------------------------------------------------
# 2. Development sequence verification
# ------------------------------------------------------------

X_dev_mc_622fd = X_dev_clean

n_dev_windows_622fd = int(
    X_dev_mc_622fd.shape[0]
)

print("\n--- Development sequence ---")
print(
    "Shape               :",
    X_dev_mc_622fd.shape
)
print(
    "Development windows :",
    n_dev_windows_622fd
)

if n_dev_windows_622fd != 14241:
    raise RuntimeError(
        "Unexpected development-window count."
    )


# ------------------------------------------------------------
# 3. Exact MC configuration
# ------------------------------------------------------------

MC_PASSES_622FD = 50
MC_BATCH_SIZE_622FD = 512
MC_SEED_622FD = 42

model_device_622fd = next(
    frozen_gru.parameters()
).device

print("\n--- Frozen MC configuration ---")
print(
    "Device       :",
    model_device_622fd
)
print(
    "MC passes    :",
    MC_PASSES_622FD
)
print(
    "Batch size   :",
    MC_BATCH_SIZE_622FD
)
print(
    "Seed         :",
    MC_SEED_622FD
)
print(
    "Std rule     : ddof=0"
)


# ------------------------------------------------------------
# 4. Execute existing verified MC-dropout pathway
# ------------------------------------------------------------

print(
    "\nExecuting 50-pass MC dropout on "
    f"{n_dev_windows_622fd} development windows..."
)

model_state_before_622fd = bool(
    frozen_gru.training
)

start_time_622fd = time.perf_counter()

try:

    mc_result_622fd = (
        mc_dropout_predictions(
            frozen_gru,
            X_dev_mc_622fd,
            model_device_622fd,
            n_passes=MC_PASSES_622FD,
            batch_size=MC_BATCH_SIZE_622FD,
            seed=MC_SEED_622FD,
        )
    )

finally:

    # Restore the exact model mode that existed
    # before the MC helper was called.
    frozen_gru.train(
        model_state_before_622fd
    )

elapsed_622fd = (
    time.perf_counter()
    - start_time_622fd
)

print(
    "Execution completed in",
    f"{elapsed_622fd:.2f} seconds."
)


# ------------------------------------------------------------
# 5. Verify helper return structure
# ------------------------------------------------------------

if not isinstance(
    mc_result_622fd,
    tuple
):
    raise RuntimeError(
        "MC helper did not return a tuple."
    )

if len(mc_result_622fd) != 3:
    raise RuntimeError(
        "Expected exactly three MC helper outputs."
    )

mc_items_622fd = [
    np.asarray(item_622fd)
    for item_622fd
    in mc_result_622fd
]

print("\n--- Returned structures ---")

for i_622fd, arr_622fd in enumerate(
    mc_items_622fd
):
    print(
        f"item_{i_622fd}:",
        "shape=",
        arr_622fd.shape,
        "| dtype=",
        arr_622fd.dtype,
    )


# ------------------------------------------------------------
# 6. Identify prediction matrix safely
# ------------------------------------------------------------

matrix_indices_622fd = [
    i_622fd
    for i_622fd, arr_622fd
    in enumerate(mc_items_622fd)
    if arr_622fd.ndim == 2
]

if len(matrix_indices_622fd) != 1:
    raise RuntimeError(
        "Could not uniquely identify "
        "the MC prediction matrix."
    )

matrix_index_622fd = (
    matrix_indices_622fd[0]
)

dev_mc_predictions_622fd = (
    mc_items_622fd[
        matrix_index_622fd
    ]
)

# Expected orientation:
# passes × windows
if dev_mc_predictions_622fd.shape == (
    n_dev_windows_622fd,
    MC_PASSES_622FD,
):

    dev_mc_predictions_622fd = (
        dev_mc_predictions_622fd.T
    )

expected_mc_shape_622fd = (
    MC_PASSES_622FD,
    n_dev_windows_622fd,
)

if dev_mc_predictions_622fd.shape != (
    expected_mc_shape_622fd
):
    raise RuntimeError(
        "Unexpected MC prediction-matrix shape: "
        f"{dev_mc_predictions_622fd.shape}"
    )


# ------------------------------------------------------------
# 7. Reconstruct mean/std directly from MC matrix
# ------------------------------------------------------------

recomputed_mean_622fd = np.mean(
    dev_mc_predictions_622fd,
    axis=0
)

recomputed_std_622fd = np.std(
    dev_mc_predictions_622fd,
    axis=0,
    ddof=0
)


# ------------------------------------------------------------
# 8. Identify returned mean and std vectors empirically
# ------------------------------------------------------------

vector_items_622fd = [
    arr_622fd
    for arr_622fd
    in mc_items_622fd
    if (
        arr_622fd.ndim == 1
        and
        arr_622fd.shape[0]
        == n_dev_windows_622fd
    )
]

if len(vector_items_622fd) != 2:
    raise RuntimeError(
        "Expected exactly two "
        "development-length vectors."
    )

v0_622fd = vector_items_622fd[0]
v1_622fd = vector_items_622fd[1]

assignment_a_error_622fd = (
    np.max(
        np.abs(
            v0_622fd
            - recomputed_mean_622fd
        )
    )
    +
    np.max(
        np.abs(
            v1_622fd
            - recomputed_std_622fd
        )
    )
)

assignment_b_error_622fd = (
    np.max(
        np.abs(
            v1_622fd
            - recomputed_mean_622fd
        )
    )
    +
    np.max(
        np.abs(
            v0_622fd
            - recomputed_std_622fd
        )
    )
)

if (
    assignment_a_error_622fd
    <=
    assignment_b_error_622fd
):

    dev_mc_mean_prediction_622fd = (
        v0_622fd.copy()
    )

    dev_mc_std_622fd = (
        v1_622fd.copy()
    )

else:

    dev_mc_mean_prediction_622fd = (
        v1_622fd.copy()
    )

    dev_mc_std_622fd = (
        v0_622fd.copy()
    )


# ------------------------------------------------------------
# 9. Numerical agreement
# ------------------------------------------------------------

mean_max_error_622fd = float(
    np.max(
        np.abs(
            dev_mc_mean_prediction_622fd
            - recomputed_mean_622fd
        )
    )
)

std_max_error_622fd = float(
    np.max(
        np.abs(
            dev_mc_std_622fd
            - recomputed_std_622fd
        )
    )
)

all_finite_622fd = bool(
    np.isfinite(
        dev_mc_predictions_622fd
    ).all()
    and
    np.isfinite(
        dev_mc_mean_prediction_622fd
    ).all()
    and
    np.isfinite(
        dev_mc_std_622fd
    ).all()
)

print("\n--- MC reconstruction agreement ---")
print(
    "Prediction matrix shape :",
    dev_mc_predictions_622fd.shape
)
print(
    "Mean vector shape       :",
    dev_mc_mean_prediction_622fd.shape
)
print(
    "STD vector shape        :",
    dev_mc_std_622fd.shape
)
print(
    "Mean max error          :",
    mean_max_error_622fd
)
print(
    "STD max error           :",
    std_max_error_622fd
)
print(
    "All values finite       :",
    all_finite_622fd
)


# ------------------------------------------------------------
# 10. Development uncertainty distribution
# ------------------------------------------------------------

uncertainty_quantiles_622fd = {
    "minimum": float(
        np.min(dev_mc_std_622fd)
    ),
    "p25": float(
        np.quantile(
            dev_mc_std_622fd,
            0.25
        )
    ),
    "median": float(
        np.quantile(
            dev_mc_std_622fd,
            0.50
        )
    ),
    "p75": float(
        np.quantile(
            dev_mc_std_622fd,
            0.75
        )
    ),
    "p90": float(
        np.quantile(
            dev_mc_std_622fd,
            0.90
        )
    ),
    "p95": float(
        np.quantile(
            dev_mc_std_622fd,
            0.95
        )
    ),
    "p99": float(
        np.quantile(
            dev_mc_std_622fd,
            0.99
        )
    ),
    "maximum": float(
        np.max(dev_mc_std_622fd)
    ),
}

print(
    "\n--- CLEAN DEVELOPMENT "
    "MC-uncertainty distribution ---"
)

for key_622fd, value_622fd in (
    uncertainty_quantiles_622fd.items()
):
    print(
        f"{key_622fd:<8}: "
        f"{value_622fd:.6f}"
    )


# ------------------------------------------------------------
# 11. Integrity checks
# ------------------------------------------------------------

if not all_finite_622fd:
    raise RuntimeError(
        "Non-finite development MC result detected."
    )

if mean_max_error_622fd > 1e-3:
    raise RuntimeError(
        "Returned MC mean does not agree "
        "with prediction matrix."
    )

if std_max_error_622fd > 1e-3:
    raise RuntimeError(
        "Returned MC std does not agree "
        "with ddof=0 reconstruction."
    )


# ------------------------------------------------------------
# 12. Compact final summary
# ------------------------------------------------------------

print(
    "\n"
    "============================================"
)
print(
    "STEP 6.22F-D COMPACT SUMMARY"
)
print(
    "============================================"
)

print(
    "Development windows          :",
    n_dev_windows_622fd
)
print(
    "MC passes                    :",
    MC_PASSES_622FD
)
print(
    "Prediction matrix            :",
    dev_mc_predictions_622fd.shape
)
print(
    "MC uncertainty vector        :",
    dev_mc_std_622fd.shape
)
print(
    "Development uncertainty p95  :",
    uncertainty_quantiles_622fd[
        "p95"
    ]
)
print(
    "Numerical integrity          :",
    all_finite_622fd
)
print(
    "Model state restored         :",
    frozen_gru.training
    ==
    model_state_before_622fd
)
print(
    "Training/retraining          : False"
)
print(
    "Official test used           : False"
)
print(
    "Threshold selected           : False"
)
print(
    "Uncertainty signal frozen    : False"
)
print(
    "Abstention rule created      : False"
)
print(
    "Files written                : False"
)

print(
    "\nStep 6.22F-D completed successfully."
)

In [ ]:
print("\n=== STEP 6.22F-D FINAL COMPACT CHECK ===")

print("MC matrix shape       :", dev_mc_predictions_622fd.shape)
print("MC mean shape         :", dev_mc_mean_prediction_622fd.shape)
print("MC std shape          :", dev_mc_std_622fd.shape)

print("Mean reconstruction err:", mean_max_error_622fd)
print("STD reconstruction err :", std_max_error_622fd)
print("All finite             :", all_finite_622fd)

print("\nDevelopment uncertainty:")
print("minimum :", uncertainty_quantiles_622fd["minimum"])
print("median  :", uncertainty_quantiles_622fd["median"])
print("p75     :", uncertainty_quantiles_622fd["p75"])
print("p90     :", uncertainty_quantiles_622fd["p90"])
print("p95     :", uncertainty_quantiles_622fd["p95"])
print("p99     :", uncertainty_quantiles_622fd["p99"])
print("maximum :", uncertainty_quantiles_622fd["maximum"])

print("\n=== END ===")

In [ ]:
# ============================================================
# Step 6.22F-E — Development uncertainty / RUL-phase diagnostic
# ============================================================

import numpy as np
import pandas as pd

print("\n=== Step 6.22F-E — Development MC uncertainty / RUL-phase diagnostic ===")

# ------------------------------------------------------------
# 1. Required objects
# ------------------------------------------------------------
required_622fe = {
    "dev_mc_std_622fd": "dev_mc_std_622fd" in globals(),
    "dev_mc_mean_prediction_622fd": "dev_mc_mean_prediction_622fd" in globals(),
    "dev_seq_meta": "dev_seq_meta" in globals(),
}

print("\n--- Required objects ---")
for name_622fe, available_622fe in required_622fe.items():
    print(f"{name_622fe:32s}: {available_622fe}")

if not all(required_622fe.values()):
    raise RuntimeError(
        "Step 6.22F-E cannot continue because one or more required "
        "development objects are unavailable."
    )

# ------------------------------------------------------------
# 2. Canonical arrays
# ------------------------------------------------------------
if "RUL_cap_125" not in dev_seq_meta.columns:
    raise RuntimeError(
        "dev_seq_meta does not contain the canonical RUL_cap_125 target."
    )

target_dev_622fe = (
    dev_seq_meta["RUL_cap_125"]
    .to_numpy(dtype=np.float64)
)

uncertainty_dev_622fe = np.asarray(
    dev_mc_std_622fd,
    dtype=np.float64
)

mcmean_dev_622fe = np.asarray(
    dev_mc_mean_prediction_622fd,
    dtype=np.float64
)

n_dev_622fe = len(target_dev_622fe)

if not (
    len(uncertainty_dev_622fe) == n_dev_622fe
    and len(mcmean_dev_622fe) == n_dev_622fe
):
    raise RuntimeError(
        "Development target, uncertainty, and MC-mean prediction "
        "are not sequence aligned."
    )

if not (
    np.isfinite(target_dev_622fe).all()
    and np.isfinite(uncertainty_dev_622fe).all()
    and np.isfinite(mcmean_dev_622fe).all()
):
    raise RuntimeError("Non-finite value detected in canonical development arrays.")

# ------------------------------------------------------------
# 3. Correlation helpers
# ------------------------------------------------------------
def pearson_622fe(a, b):
    return float(np.corrcoef(a, b)[0, 1])


def spearman_622fe(a, b):
    return float(
        pd.Series(a).rank(method="average").corr(
            pd.Series(b).rank(method="average")
        )
    )


target_unc_p_622fe = pearson_622fe(
    target_dev_622fe,
    uncertainty_dev_622fe
)
target_unc_s_622fe = spearman_622fe(
    target_dev_622fe,
    uncertainty_dev_622fe
)

pred_unc_p_622fe = pearson_622fe(
    mcmean_dev_622fe,
    uncertainty_dev_622fe
)
pred_unc_s_622fe = spearman_622fe(
    mcmean_dev_622fe,
    uncertainty_dev_622fe
)

target_pred_p_622fe = pearson_622fe(
    target_dev_622fe,
    mcmean_dev_622fe
)
target_pred_s_622fe = spearman_622fe(
    target_dev_622fe,
    mcmean_dev_622fe
)

# ------------------------------------------------------------
# 4. Same RUL-phase convention used in validation diagnostic
# ------------------------------------------------------------
def rul_phase_622fe(values):
    values = np.asarray(values)

    return np.select(
        [
            values >= 125,
            (values >= 101) & (values < 125),
            (values >= 76) & (values <= 100),
            (values >= 51) & (values <= 75),
            (values >= 26) & (values <= 50),
            values <= 25,
        ],
        [
            "capped_125",
            "101_124",
            "76_100",
            "51_75",
            "26_50",
            "0_25",
        ],
        default="other",
    )


true_phase_622fe = rul_phase_622fe(target_dev_622fe)

diagnostic_df_622fe = pd.DataFrame({
    "target": target_dev_622fe,
    "mc_mean_prediction": mcmean_dev_622fe,
    "mc_uncertainty": uncertainty_dev_622fe,
    "true_rul_phase": true_phase_622fe,
})

phase_order_622fe = [
    "capped_125",
    "101_124",
    "76_100",
    "51_75",
    "26_50",
    "0_25",
]

phase_summary_622fe = (
    diagnostic_df_622fe
    .groupby("true_rul_phase", observed=True)
    .agg(
        windows=("target", "size"),
        mean_target=("target", "mean"),
        mean_mc_prediction=("mc_mean_prediction", "mean"),
        mean_uncertainty=("mc_uncertainty", "mean"),
        median_uncertainty=("mc_uncertainty", "median"),
    )
    .reindex(phase_order_622fe)
    .reset_index()
)

# ------------------------------------------------------------
# 5. Compact output only
# ------------------------------------------------------------
print("\n--- Overall development association ---")
print(
    "Pearson(target, uncertainty)      :",
    target_unc_p_622fe
)
print(
    "Spearman(target, uncertainty)     :",
    target_unc_s_622fe
)
print(
    "Pearson(MC mean, uncertainty)     :",
    pred_unc_p_622fe
)
print(
    "Spearman(MC mean, uncertainty)    :",
    pred_unc_s_622fe
)
print(
    "Pearson(target, MC mean)          :",
    target_pred_p_622fe
)
print(
    "Spearman(target, MC mean)         :",
    target_pred_s_622fe
)

print("\n--- Development RUL-phase summary ---")
print(
    phase_summary_622fe.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

print("\n==============================================")
print("STEP 6.22F-E COMPACT SUMMARY")
print("==============================================")
print("Development windows            :", n_dev_622fe)
print("Canonical target               : RUL_cap_125")
print("Uncertainty source             : MC-dropout std, ddof=0")
print("True RUL used for calibration  : False")
print("Threshold selected             : False")
print("Uncertainty signal frozen      : False")
print("Official test used             : False")
print("Abstention rule created        : False")
print("\nStep 6.22F-E completed.")

In [ ]:
# ============================================================
# Step 6.22F-F — Development-only phase-adjusted MC uncertainty
# ============================================================

import numpy as np
import pandas as pd

from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import GroupKFold

print("\n=== Step 6.22F-F — Phase-adjusted MC uncertainty candidates ===")

# ------------------------------------------------------------
# 1. Required objects
# ------------------------------------------------------------
required_622ff = {
    "dev_mc_std_622fd": "dev_mc_std_622fd" in globals(),
    "dev_mc_mean_prediction_622fd": "dev_mc_mean_prediction_622fd" in globals(),
    "dev_seq_meta": "dev_seq_meta" in globals(),
    "mc_std": "mc_std" in globals(),
    "mc_mean_prediction": "mc_mean_prediction" in globals(),
}

print("\n--- Required objects ---")
for name_622ff, available_622ff in required_622ff.items():
    print(f"{name_622ff:34s}: {available_622ff}")

if not all(required_622ff.values()):
    raise RuntimeError(
        "Step 6.22F-F cannot continue because one or more "
        "required objects are unavailable."
    )

if "unit_id" not in dev_seq_meta.columns:
    raise RuntimeError(
        "dev_seq_meta must contain unit_id for engine-aware cross-fitting."
    )

# ------------------------------------------------------------
# 2. Canonical arrays
# ------------------------------------------------------------
dev_pred_622ff = np.asarray(
    dev_mc_mean_prediction_622fd,
    dtype=np.float64
)

dev_unc_622ff = np.asarray(
    dev_mc_std_622fd,
    dtype=np.float64
)

dev_groups_622ff = (
    dev_seq_meta["unit_id"]
    .to_numpy()
)

val_pred_622ff = np.asarray(
    mc_mean_prediction,
    dtype=np.float64
)

val_unc_622ff = np.asarray(
    mc_std,
    dtype=np.float64
)

if not (
    len(dev_pred_622ff)
    == len(dev_unc_622ff)
    == len(dev_groups_622ff)
):
    raise RuntimeError("Development arrays are not aligned.")

if len(val_pred_622ff) != len(val_unc_622ff):
    raise RuntimeError("Validation arrays are not aligned.")

for arr_name_622ff, arr_622ff in {
    "dev_pred": dev_pred_622ff,
    "dev_unc": dev_unc_622ff,
    "val_pred": val_pred_622ff,
    "val_unc": val_unc_622ff,
}.items():

    if not np.isfinite(arr_622ff).all():
        raise RuntimeError(
            f"Non-finite values detected in {arr_name_622ff}."
        )

    if "unc" in arr_name_622ff and np.any(arr_622ff <= 0):
        raise RuntimeError(
            f"Non-positive uncertainty detected in {arr_name_622ff}."
        )

print("\nDevelopment windows :", len(dev_pred_622ff))
print("Development engines :", len(np.unique(dev_groups_622ff)))
print("Validation windows  :", len(val_pred_622ff))

# ------------------------------------------------------------
# 3. Engine-aware cross-fitted expected uncertainty
# ------------------------------------------------------------
unique_engines_622ff = np.unique(dev_groups_622ff)

n_splits_622ff = min(
    5,
    len(unique_engines_622ff)
)

if n_splits_622ff < 2:
    raise RuntimeError(
        "At least two development engines are required."
    )

gkf_622ff = GroupKFold(
    n_splits=n_splits_622ff
)

dev_expected_cf_622ff = np.full(
    len(dev_unc_622ff),
    np.nan,
    dtype=np.float64
)

for train_idx_622ff, hold_idx_622ff in gkf_622ff.split(
    dev_pred_622ff,
    groups=dev_groups_622ff,
):

    iso_fold_622ff = IsotonicRegression(
        increasing=True,
        out_of_bounds="clip",
    )

    iso_fold_622ff.fit(
        dev_pred_622ff[train_idx_622ff],
        dev_unc_622ff[train_idx_622ff],
    )

    dev_expected_cf_622ff[hold_idx_622ff] = (
        iso_fold_622ff.predict(
            dev_pred_622ff[hold_idx_622ff]
        )
    )

if not np.isfinite(dev_expected_cf_622ff).all():
    raise RuntimeError(
        "Cross-fitted expected uncertainty contains non-finite values."
    )

if np.any(dev_expected_cf_622ff <= 0):
    raise RuntimeError(
        "Cross-fitted expected uncertainty must remain positive."
    )

# ------------------------------------------------------------
# 4. Final development-only mapping
#    This may later be used at inference time.
# ------------------------------------------------------------
uncertainty_phase_model_622ff = IsotonicRegression(
    increasing=True,
    out_of_bounds="clip",
)

uncertainty_phase_model_622ff.fit(
    dev_pred_622ff,
    dev_unc_622ff,
)

dev_expected_full_622ff = (
    uncertainty_phase_model_622ff.predict(
        dev_pred_622ff
    )
)

val_expected_622ff = (
    uncertainty_phase_model_622ff.predict(
        val_pred_622ff
    )
)

# ------------------------------------------------------------
# 5. Candidate phase-adjusted uncertainty scores
# ------------------------------------------------------------
eps_622ff = 1e-12

dev_adjusted_622ff = {
    "raw_mc_std":
        dev_unc_622ff,

    "residual":
        dev_unc_622ff
        - dev_expected_cf_622ff,

    "relative_excess":
        (
            dev_unc_622ff
            - dev_expected_cf_622ff
        )
        / np.maximum(
            dev_expected_cf_622ff,
            eps_622ff
        ),

    "log_ratio":
        np.log(
            dev_unc_622ff
            / np.maximum(
                dev_expected_cf_622ff,
                eps_622ff
            )
        ),
}

val_adjusted_622ff = {
    "raw_mc_std":
        val_unc_622ff,

    "residual":
        val_unc_622ff
        - val_expected_622ff,

    "relative_excess":
        (
            val_unc_622ff
            - val_expected_622ff
        )
        / np.maximum(
            val_expected_622ff,
            eps_622ff
        ),

    "log_ratio":
        np.log(
            val_unc_622ff
            / np.maximum(
                val_expected_622ff,
                eps_622ff
            )
        ),
}

# ------------------------------------------------------------
# 6. Correlation helpers
# ------------------------------------------------------------
def pearson_622ff(a, b):
    return float(
        np.corrcoef(a, b)[0, 1]
    )


def spearman_622ff(a, b):
    return float(
        pd.Series(a).rank(
            method="average"
        ).corr(
            pd.Series(b).rank(
                method="average"
            )
        )
    )


# ------------------------------------------------------------
# 7. DEVELOPMENT-ONLY candidate comparison
# ------------------------------------------------------------
candidate_rows_622ff = []

for candidate_622ff, score_622ff in dev_adjusted_622ff.items():

    score_622ff = np.asarray(
        score_622ff,
        dtype=np.float64
    )

    candidate_rows_622ff.append({
        "candidate": candidate_622ff,
        "pearson_vs_mcmean":
            pearson_622ff(
                dev_pred_622ff,
                score_622ff
            ),
        "spearman_vs_mcmean":
            spearman_622ff(
                dev_pred_622ff,
                score_622ff
            ),
        "mean":
            float(np.mean(score_622ff)),
        "std":
            float(np.std(score_622ff)),
        "p95":
            float(np.percentile(
                score_622ff,
                95
            )),
    })

candidate_summary_622ff = pd.DataFrame(
    candidate_rows_622ff
)

# ------------------------------------------------------------
# 8. Selection rule fixed from DEVELOPMENT only
#
# Exclude raw_mc_std.
# Primary goal:
#   minimize residual phase dependence.
# Ranking:
#   max(|Pearson|, |Spearman|)
# ------------------------------------------------------------
adjusted_only_622ff = (
    candidate_summary_622ff[
        candidate_summary_622ff[
            "candidate"
        ] != "raw_mc_std"
    ]
    .copy()
)

adjusted_only_622ff[
    "phase_dependence_score"
] = np.maximum(
    adjusted_only_622ff[
        "pearson_vs_mcmean"
    ].abs(),
    adjusted_only_622ff[
        "spearman_vs_mcmean"
    ].abs(),
)

adjusted_only_622ff = (
    adjusted_only_622ff
    .sort_values(
        [
            "phase_dependence_score",
            "candidate",
        ],
        ascending=[
            True,
            True,
        ],
    )
    .reset_index(drop=True)
)

selected_uncertainty_candidate_622ff = (
    adjusted_only_622ff
    .iloc[0]["candidate"]
)

# ------------------------------------------------------------
# 9. CLEAN validation sanity check
#    Validation DOES NOT participate in candidate selection.
# ------------------------------------------------------------
val_rows_622ff = []

for candidate_622ff in [
    "raw_mc_std",
    selected_uncertainty_candidate_622ff,
]:

    score_622ff = np.asarray(
        val_adjusted_622ff[
            candidate_622ff
        ],
        dtype=np.float64
    )

    val_rows_622ff.append({
        "candidate": candidate_622ff,
        "pearson_vs_mcmean":
            pearson_622ff(
                val_pred_622ff,
                score_622ff
            ),
        "spearman_vs_mcmean":
            spearman_622ff(
                val_pred_622ff,
                score_622ff
            ),
        "mean":
            float(np.mean(score_622ff)),
        "std":
            float(np.std(score_622ff)),
        "p95":
            float(np.percentile(
                score_622ff,
                95
            )),
    })

val_summary_622ff = pd.DataFrame(
    val_rows_622ff
)

# ------------------------------------------------------------
# 10. Compact output
# ------------------------------------------------------------
print("\n--- DEVELOPMENT-only candidate comparison ---")
print(
    candidate_summary_622ff.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print("\n--- DEVELOPMENT-only selection ranking ---")
print(
    adjusted_only_622ff[
        [
            "candidate",
            "phase_dependence_score",
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print(
    "\nSelected candidate from DEVELOPMENT only:",
    selected_uncertainty_candidate_622ff
)

print("\n--- CLEAN validation sanity check ---")
print(
    val_summary_622ff.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print("\n==============================================")
print("STEP 6.22F-F COMPACT SUMMARY")
print("==============================================")
print(
    "Development windows              :",
    len(dev_pred_622ff)
)
print(
    "Engine-aware cross-fitting       :",
    True
)
print(
    "Cross-validation folds           :",
    n_splits_622ff
)
print(
    "Conditioning variable            : MC mean predicted RUL"
)
print(
    "True RUL used in uncertainty map :",
    False
)
print(
    "Selection source                 : CLEAN DEVELOPMENT only"
)
print(
    "Selected adjusted candidate      :",
    selected_uncertainty_candidate_622ff
)
print(
    "Clean validation used            : sanity check only"
)
print(
    "Official test used               :",
    False
)
print(
    "Uncertainty threshold selected   :",
    False
)
print(
    "Uncertainty signal frozen        :",
    False
)
print(
    "Abstention rule created          :",
    False
)

print("\nStep 6.22F-F completed.")

In [ ]:
# ============================================================
# Step 6.22F-F1 — Compact validation check
# ============================================================

print("\n=== STEP 6.22F-F1 COMPACT VALIDATION CHECK ===")

print(
    "Selected from DEVELOPMENT only :",
    selected_uncertainty_candidate_622ff
)

dev_selected_622ff = (
    candidate_summary_622ff[
        candidate_summary_622ff["candidate"]
        == selected_uncertainty_candidate_622ff
    ]
    .iloc[0]
)

print("\n--- Development selected candidate ---")
print(
    f"Pearson vs MC mean  : "
    f"{dev_selected_622ff['pearson_vs_mcmean']:.6f}"
)
print(
    f"Spearman vs MC mean : "
    f"{dev_selected_622ff['spearman_vs_mcmean']:.6f}"
)

print("\n--- CLEAN validation sanity check ---")

for _, row_622ff1 in val_summary_622ff.iterrows():
    print(
        f"{row_622ff1['candidate']:16s} | "
        f"Pearson={row_622ff1['pearson_vs_mcmean']:.6f} | "
        f"Spearman={row_622ff1['spearman_vs_mcmean']:.6f} | "
        f"mean={row_622ff1['mean']:.6f} | "
        f"std={row_622ff1['std']:.6f} | "
        f"p95={row_622ff1['p95']:.6f}"
    )

print("\nOfficial test used             : False")
print("Uncertainty signal frozen      : False")
print("Uncertainty threshold selected : False")
print("Abstention rule created        : False")
print("=== END ===")

In [ ]:
# ============================================================
# Step 6.22G — Phase-adjusted uncertainty error-utility test
# ============================================================

import numpy as np
import pandas as pd

print("\n=== Step 6.22G — Phase-adjusted uncertainty / RUL-error utility ===")


# ------------------------------------------------------------
# 1. Small conversion helper
# ------------------------------------------------------------

def to_numpy_1d_622g(x):
    if hasattr(x, "detach"):
        x = x.detach().cpu().numpy()

    arr = np.asarray(x, dtype=np.float64).reshape(-1)
    return arr


# ------------------------------------------------------------
# 2. Locate the already-computed residual candidate safely
#    We do NOT recompute or refit the phase adjustment here.
# ------------------------------------------------------------

def find_candidate_array_622g(
    split_name,
    candidate_name,
    expected_length
):
    split_tokens = (
        ["dev", "development"]
        if split_name == "development"
        else ["val", "validation"]
    )

    hits = []

    namespace_snapshot = list(globals().items())

    for object_name, object_value in namespace_snapshot:

        lower_name = str(object_name).lower()

        # ----------------------------------------------------
        # Case A: dictionary containing candidate
        # ----------------------------------------------------
        if isinstance(object_value, dict):

            if candidate_name in object_value:
                try:
                    arr = to_numpy_1d_622g(
                        object_value[candidate_name]
                    )

                    if (
                        len(arr) == expected_length
                        and np.isfinite(arr).all()
                    ):
                        score = 0

                        if "622ff" in lower_name:
                            score += 10

                        if any(
                            token in lower_name
                            for token in split_tokens
                        ):
                            score += 5

                        hits.append(
                            (
                                score,
                                f"{object_name}[{candidate_name!r}]",
                                arr
                            )
                        )

                except Exception:
                    pass

        # ----------------------------------------------------
        # Case B: DataFrame containing candidate column
        # ----------------------------------------------------
        if isinstance(object_value, pd.DataFrame):

            if candidate_name in object_value.columns:
                try:
                    arr = to_numpy_1d_622g(
                        object_value[candidate_name]
                    )

                    if (
                        len(arr) == expected_length
                        and np.isfinite(arr).all()
                    ):
                        score = 0

                        if "622ff" in lower_name:
                            score += 10

                        if any(
                            token in lower_name
                            for token in split_tokens
                        ):
                            score += 5

                        hits.append(
                            (
                                score,
                                f"{object_name}[{candidate_name!r}]",
                                arr
                            )
                        )

                except Exception:
                    pass

        # ----------------------------------------------------
        # Case C: direct residual array
        # ----------------------------------------------------
        if (
            candidate_name in lower_name
            and any(
                token in lower_name
                for token in split_tokens
            )
        ):
            try:
                arr = to_numpy_1d_622g(object_value)

                if (
                    len(arr) == expected_length
                    and np.isfinite(arr).all()
                ):
                    score = 5

                    if "622ff" in lower_name:
                        score += 10

                    hits.append(
                        (
                            score,
                            object_name,
                            arr
                        )
                    )

            except Exception:
                pass

    if len(hits) == 0:
        raise RuntimeError(
            f"Could not locate {split_name} "
            f"{candidate_name!r} array."
        )

    hits.sort(
        key=lambda x: (-x[0], x[1])
    )

    selected_source = hits[0][1]
    selected_array = hits[0][2]

    return selected_source, selected_array


DEV_N_622G = 14241
VAL_N_622G = 3490

dev_residual_source_622g, dev_residual_622g = (
    find_candidate_array_622g(
        "development",
        "residual",
        DEV_N_622G
    )
)

val_residual_source_622g, val_residual_622g = (
    find_candidate_array_622g(
        "validation",
        "residual",
        VAL_N_622G
    )
)

print("\n--- Residual-array sources ---")
print("Development :", dev_residual_source_622g)
print("Validation  :", val_residual_source_622g)


# ------------------------------------------------------------
# 3. Canonical development target
# ------------------------------------------------------------

if (
    "dev_seq_meta" not in globals()
    or not isinstance(dev_seq_meta, pd.DataFrame)
    or "RUL_cap_125" not in dev_seq_meta.columns
):
    raise RuntimeError(
        "Canonical development target "
        "dev_seq_meta['RUL_cap_125'] is unavailable."
    )

dev_target_622g = to_numpy_1d_622g(
    dev_seq_meta["RUL_cap_125"]
)

if len(dev_target_622g) != DEV_N_622G:
    raise RuntimeError(
        "Unexpected development target length."
    )


# ------------------------------------------------------------
# 4. Deterministic development prediction
#
#    This is prediction only:
#    - no fitting
#    - no retraining
#    - no official test
# ------------------------------------------------------------

if "predict_gru_batched" not in globals():
    raise RuntimeError(
        "predict_gru_batched is unavailable."
    )

if "frozen_gru" not in globals():
    raise RuntimeError(
        "frozen_gru is unavailable."
    )

if "X_dev_clean" not in globals():
    raise RuntimeError(
        "X_dev_clean is unavailable."
    )

model_device_622g = next(
    frozen_gru.parameters()
).device

dev_prediction_622g = to_numpy_1d_622g(
    predict_gru_batched(
        frozen_gru,
        X_dev_clean,
        model_device_622g,
        batch_size=512
    )
)

if len(dev_prediction_622g) != DEV_N_622G:
    raise RuntimeError(
        "Unexpected development prediction length."
    )


# ------------------------------------------------------------
# 5. Canonical validation pathway
# ------------------------------------------------------------

val_target_622g = to_numpy_1d_622g(
    y_val_clean
)

val_prediction_622g = to_numpy_1d_622g(
    clean_predictions
)

if (
    len(val_target_622g) != VAL_N_622G
    or len(val_prediction_622g) != VAL_N_622G
):
    raise RuntimeError(
        "Unexpected validation array length."
    )


# ------------------------------------------------------------
# 6. Raw MC std — comparison baseline only
# ------------------------------------------------------------

dev_raw_uncertainty_622g = to_numpy_1d_622g(
    dev_mc_std_622fd
)

val_raw_uncertainty_622g = to_numpy_1d_622g(
    mc_std
)


# ------------------------------------------------------------
# 7. Metric helper
# ------------------------------------------------------------

def uncertainty_utility_622g(
    uncertainty,
    prediction,
    target
):
    uncertainty = to_numpy_1d_622g(
        uncertainty
    )

    prediction = to_numpy_1d_622g(
        prediction
    )

    target = to_numpy_1d_622g(
        target
    )

    abs_error = np.abs(
        prediction - target
    )

    pearson = float(
        np.corrcoef(
            uncertainty,
            abs_error
        )[0, 1]
    )

    spearman = float(
        pd.Series(
            uncertainty
        ).corr(
            pd.Series(abs_error),
            method="spearman"
        )
    )

    n = len(uncertainty)
    quartile_n = int(
        np.ceil(0.25 * n)
    )

    order_unc = np.argsort(
        uncertainty
    )

    low_indices = order_unc[
        :quartile_n
    ]

    high_indices = order_unc[
        -quartile_n:
    ]

    low_mae = float(
        abs_error[
            low_indices
        ].mean()
    )

    high_mae = float(
        abs_error[
            high_indices
        ].mean()
    )

    high_low_ratio = float(
        high_mae / low_mae
    )

    high_error_indices = set(
        np.argsort(
            abs_error
        )[-quartile_n:]
    )

    high_unc_indices = set(
        high_indices
    )

    capture_pct = (
        100.0
        * len(
            high_error_indices
            & high_unc_indices
        )
        / quartile_n
    )

    return {
        "pearson": pearson,
        "spearman": spearman,
        "low_q_mae": low_mae,
        "high_q_mae": high_mae,
        "ratio": high_low_ratio,
        "capture_pct": capture_pct,
        "abs_error": abs_error,
    }


# ------------------------------------------------------------
# 8. Evaluate RAW and RESIDUAL
# ------------------------------------------------------------

dev_raw_result_622g = (
    uncertainty_utility_622g(
        dev_raw_uncertainty_622g,
        dev_prediction_622g,
        dev_target_622g
    )
)

dev_res_result_622g = (
    uncertainty_utility_622g(
        dev_residual_622g,
        dev_prediction_622g,
        dev_target_622g
    )
)

val_raw_result_622g = (
    uncertainty_utility_622g(
        val_raw_uncertainty_622g,
        val_prediction_622g,
        val_target_622g
    )
)

val_res_result_622g = (
    uncertainty_utility_622g(
        val_residual_622g,
        val_prediction_622g,
        val_target_622g
    )
)


# ------------------------------------------------------------
# 9. Compact comparison
# ------------------------------------------------------------

print("\n=== DEVELOPMENT — decision evidence ===")

print(
    "RAW MC std  | "
    f"Pearson={dev_raw_result_622g['pearson']:.4f} | "
    f"Spearman={dev_raw_result_622g['spearman']:.4f} | "
    f"LowQ MAE={dev_raw_result_622g['low_q_mae']:.4f} | "
    f"HighQ MAE={dev_raw_result_622g['high_q_mae']:.4f} | "
    f"Ratio={dev_raw_result_622g['ratio']:.3f} | "
    f"Top-error capture={dev_raw_result_622g['capture_pct']:.2f}%"
)

print(
    "RESIDUAL    | "
    f"Pearson={dev_res_result_622g['pearson']:.4f} | "
    f"Spearman={dev_res_result_622g['spearman']:.4f} | "
    f"LowQ MAE={dev_res_result_622g['low_q_mae']:.4f} | "
    f"HighQ MAE={dev_res_result_622g['high_q_mae']:.4f} | "
    f"Ratio={dev_res_result_622g['ratio']:.3f} | "
    f"Top-error capture={dev_res_result_622g['capture_pct']:.2f}%"
)


print("\n=== CLEAN VALIDATION — sanity only ===")

print(
    "RAW MC std  | "
    f"Pearson={val_raw_result_622g['pearson']:.4f} | "
    f"Spearman={val_raw_result_622g['spearman']:.4f} | "
    f"LowQ MAE={val_raw_result_622g['low_q_mae']:.4f} | "
    f"HighQ MAE={val_raw_result_622g['high_q_mae']:.4f} | "
    f"Ratio={val_raw_result_622g['ratio']:.3f} | "
    f"Top-error capture={val_raw_result_622g['capture_pct']:.2f}%"
)

print(
    "RESIDUAL    | "
    f"Pearson={val_res_result_622g['pearson']:.4f} | "
    f"Spearman={val_res_result_622g['spearman']:.4f} | "
    f"LowQ MAE={val_res_result_622g['low_q_mae']:.4f} | "
    f"HighQ MAE={val_res_result_622g['high_q_mae']:.4f} | "
    f"Ratio={val_res_result_622g['ratio']:.3f} | "
    f"Top-error capture={val_res_result_622g['capture_pct']:.2f}%"
)


# ------------------------------------------------------------
# 10. Residual-only preliminary risk–coverage
# ------------------------------------------------------------

def risk_coverage_622g(
    uncertainty,
    abs_error
):
    uncertainty = to_numpy_1d_622g(
        uncertainty
    )

    abs_error = to_numpy_1d_622g(
        abs_error
    )

    order = np.argsort(
        uncertainty
    )

    output = []

    for coverage in [
        1.00,
        0.90,
        0.80,
        0.70,
        0.60,
        0.50
    ]:

        accepted_n = int(
            round(
                coverage
                * len(order)
            )
        )

        accepted_idx = order[
            :accepted_n
        ]

        mae = float(
            abs_error[
                accepted_idx
            ].mean()
        )

        output.append(
            (
                coverage,
                accepted_n,
                mae
            )
        )

    return output


print(
    "\n=== RESIDUAL preliminary risk–coverage ==="
)

print(
    "coverage | DEV_MAE | VAL_MAE"
)

dev_rc_622g = risk_coverage_622g(
    dev_residual_622g,
    dev_res_result_622g[
        "abs_error"
    ]
)

val_rc_622g = risk_coverage_622g(
    val_residual_622g,
    val_res_result_622g[
        "abs_error"
    ]
)

for dev_row, val_row in zip(
    dev_rc_622g,
    val_rc_622g
):
    print(
        f"{dev_row[0]:.2f}     | "
        f"{dev_row[2]:.4f} | "
        f"{val_row[2]:.4f}"
    )


print("\n--- Integrity ---")
print(
    "Development residual finite :",
    np.isfinite(
        dev_residual_622g
    ).all()
)
print(
    "Validation residual finite  :",
    np.isfinite(
        val_residual_622g
    ).all()
)
print(
    "Official test used           : False"
)
print(
    "Uncertainty signal frozen    : False"
)
print(
    "Threshold selected           : False"
)
print(
    "Abstention rule created      : False"
)

print(
    "\n=== Step 6.22G completed ==="
)

In [ ]:
# ============================================================
# Step 6.22G-2 — Compare all phase-adjusted MC candidates
# ============================================================

import numpy as np
import pandas as pd

print(
    "\n=== Step 6.22G-2 — "
    "Phase-adjusted uncertainty candidate comparison ==="
)


# ------------------------------------------------------------
# 1. Required objects
# ------------------------------------------------------------

required_622g2 = [
    "dev_adjusted_622ff",
    "val_adjusted_622ff",
    "dev_mc_std_622fd",
    "mc_std",
    "dev_mc_mean_prediction_622fd",
    "mc_mean_prediction",
    "dev_prediction_622g",
    "dev_target_622g",
    "val_prediction_622g",
    "val_target_622g",
]

missing_622g2 = [
    name
    for name in required_622g2
    if name not in globals()
]

if missing_622g2:
    raise RuntimeError(
        "Missing required objects: "
        + ", ".join(missing_622g2)
    )

print("All required objects are available.")


# ------------------------------------------------------------
# 2. Array helper
# ------------------------------------------------------------

def arr1d_622g2(x):
    if hasattr(x, "detach"):
        x = x.detach().cpu().numpy()

    return np.asarray(
        x,
        dtype=np.float64
    ).reshape(-1)


# ------------------------------------------------------------
# 3. Candidate extractor
# ------------------------------------------------------------

def get_adjusted_candidate_622g2(
    source,
    candidate
):

    if isinstance(source, dict):
        if candidate not in source:
            raise KeyError(
                f"{candidate!r} not found."
            )

        return arr1d_622g2(
            source[candidate]
        )

    if isinstance(source, pd.DataFrame):
        if candidate not in source.columns:
            raise KeyError(
                f"{candidate!r} not found."
            )

        return arr1d_622g2(
            source[candidate]
        )

    raise TypeError(
        "Adjusted candidate source must "
        "be dict or DataFrame."
    )


candidate_names_622g2 = [
    "residual",
    "relative_excess",
    "log_ratio",
]


# ------------------------------------------------------------
# 4. Correlation helper
# ------------------------------------------------------------

def safe_corr_622g2(
    x,
    y,
    method="pearson"
):
    x = arr1d_622g2(x)
    y = arr1d_622g2(y)

    if method == "pearson":
        return float(
            np.corrcoef(x, y)[0, 1]
        )

    return float(
        pd.Series(x).corr(
            pd.Series(y),
            method="spearman"
        )
    )


# ------------------------------------------------------------
# 5. Candidate utility metrics
# ------------------------------------------------------------

def candidate_metrics_622g2(
    score,
    prediction,
    target,
    phase_proxy
):

    score = arr1d_622g2(score)
    prediction = arr1d_622g2(prediction)
    target = arr1d_622g2(target)
    phase_proxy = arr1d_622g2(
        phase_proxy
    )

    if not np.isfinite(score).all():
        raise RuntimeError(
            "Candidate contains "
            "non-finite values."
        )

    absolute_error = np.abs(
        prediction - target
    )

    # Phase dependence
    phase_pearson = safe_corr_622g2(
        score,
        phase_proxy,
        "pearson"
    )

    phase_spearman = safe_corr_622g2(
        score,
        phase_proxy,
        "spearman"
    )

    # Error utility
    error_pearson = safe_corr_622g2(
        score,
        absolute_error,
        "pearson"
    )

    error_spearman = safe_corr_622g2(
        score,
        absolute_error,
        "spearman"
    )

    # Quartile separation
    n = len(score)
    qn = int(
        np.ceil(0.25 * n)
    )

    order = np.argsort(score)

    low_idx = order[:qn]
    high_idx = order[-qn:]

    low_mae = float(
        absolute_error[
            low_idx
        ].mean()
    )

    high_mae = float(
        absolute_error[
            high_idx
        ].mean()
    )

    ratio = float(
        high_mae / low_mae
    )

    # Risk–coverage utility
    full_mae = float(
        absolute_error.mean()
    )

    gains = {}

    for coverage in [
        0.90,
        0.80,
        0.70,
        0.60,
        0.50
    ]:

        keep_n = int(
            round(
                coverage * n
            )
        )

        accepted = order[
            :keep_n
        ]

        accepted_mae = float(
            absolute_error[
                accepted
            ].mean()
        )

        gains[coverage] = (
            full_mae
            - accepted_mae
        )

    mean_gain = float(
        np.mean(
            list(
                gains.values()
            )
        )
    )

    return {
        "phase_pearson": phase_pearson,
        "phase_spearman": phase_spearman,
        "error_pearson": error_pearson,
        "error_spearman": error_spearman,
        "lowQ_MAE": low_mae,
        "highQ_MAE": high_mae,
        "high_low_ratio": ratio,
        "gain_90": gains[0.90],
        "gain_70": gains[0.70],
        "gain_50": gains[0.50],
        "mean_RC_gain": mean_gain,
    }


# ------------------------------------------------------------
# 6. Assemble RAW + adjusted candidates
# ------------------------------------------------------------

dev_candidates_622g2 = {
    "raw_mc_std":
        arr1d_622g2(
            dev_mc_std_622fd
        )
}

val_candidates_622g2 = {
    "raw_mc_std":
        arr1d_622g2(
            mc_std
        )
}

for candidate in candidate_names_622g2:

    dev_candidates_622g2[
        candidate
    ] = get_adjusted_candidate_622g2(
        dev_adjusted_622ff,
        candidate
    )

    val_candidates_622g2[
        candidate
    ] = get_adjusted_candidate_622g2(
        val_adjusted_622ff,
        candidate
    )


# ------------------------------------------------------------
# 7. DEVELOPMENT — actual decision evidence
# ------------------------------------------------------------

dev_rows_622g2 = []

for name, score in dev_candidates_622g2.items():

    result = candidate_metrics_622g2(
        score,
        dev_prediction_622g,
        dev_target_622g,
        dev_mc_mean_prediction_622fd
    )

    dev_rows_622g2.append(
        {
            "candidate": name,
            **result
        }
    )

dev_table_622g2 = pd.DataFrame(
    dev_rows_622g2
)


# ------------------------------------------------------------
# 8. VALIDATION — sanity check only
# ------------------------------------------------------------

val_rows_622g2 = []

for name, score in val_candidates_622g2.items():

    result = candidate_metrics_622g2(
        score,
        val_prediction_622g,
        val_target_622g,
        mc_mean_prediction
    )

    val_rows_622g2.append(
        {
            "candidate": name,
            **result
        }
    )

val_table_622g2 = pd.DataFrame(
    val_rows_622g2
)


# ------------------------------------------------------------
# 9. Compact output
# ------------------------------------------------------------

cols_622g2 = [
    "candidate",
    "phase_spearman",
    "error_spearman",
    "lowQ_MAE",
    "highQ_MAE",
    "high_low_ratio",
    "gain_90",
    "gain_70",
    "gain_50",
    "mean_RC_gain",
]


print(
    "\n=== DEVELOPMENT — selection evidence ==="
)

print(
    dev_table_622g2[
        cols_622g2
    ].round(4).to_string(
        index=False
    )
)


print(
    "\n=== CLEAN VALIDATION — sanity only ==="
)

print(
    val_table_622g2[
        cols_622g2
    ].round(4).to_string(
        index=False
    )
)


# ------------------------------------------------------------
# 10. Development-only ranking among adjusted candidates
#
# Higher error association and positive RC gain are useful;
# lower absolute phase dependence is desirable.
#
# We do NOT freeze anything here.
# ------------------------------------------------------------

adjusted_dev_622g2 = (
    dev_table_622g2[
        dev_table_622g2[
            "candidate"
        ].isin(
            candidate_names_622g2
        )
    ]
    .copy()
)

adjusted_dev_622g2[
    "abs_phase_spearman"
] = (
    adjusted_dev_622g2[
        "phase_spearman"
    ].abs()
)

adjusted_dev_622g2 = (
    adjusted_dev_622g2
    .sort_values(
        by=[
            "mean_RC_gain",
            "error_spearman",
            "abs_phase_spearman"
        ],
        ascending=[
            False,
            False,
            True
        ]
    )
)


print(
    "\n=== DEVELOPMENT-only adjusted-candidate ranking ==="
)

print(
    adjusted_dev_622g2[
        [
            "candidate",
            "abs_phase_spearman",
            "error_spearman",
            "high_low_ratio",
            "mean_RC_gain",
        ]
    ]
    .round(4)
    .to_string(
        index=False
    )
)


print("\n--- Integrity ---")
print(
    "Validation used for selection : False"
)
print(
    "Official test used            : False"
)
print(
    "Uncertainty signal frozen     : False"
)
print(
    "Threshold selected            : False"
)
print(
    "Abstention rule created       : False"
)

print(
    "\n=== Step 6.22G-2 completed ==="
)

In [ ]:
# ============================================================
# Step 6.22H — Development-calibrated within-phase
#               MC-uncertainty percentile candidates
# ============================================================

import numpy as np
import pandas as pd


print(
    "\n=== Step 6.22H — "
    "Within-phase uncertainty ranking ==="
)


# ------------------------------------------------------------
# 1. Required objects
# ------------------------------------------------------------

required_622h = [
    "dev_mc_std_622fd",
    "mc_std",
    "dev_mc_mean_prediction_622fd",
    "mc_mean_prediction",
    "dev_prediction_622g",
    "dev_target_622g",
    "val_prediction_622g",
    "val_target_622g",
]

missing_622h = [
    name
    for name in required_622h
    if name not in globals()
]

if missing_622h:
    raise RuntimeError(
        "Missing required objects: "
        + ", ".join(missing_622h)
    )

print("All required objects are available.")


# ------------------------------------------------------------
# 2. Array helper
# ------------------------------------------------------------

def arr1d_622h(x):

    if hasattr(x, "detach"):
        x = x.detach().cpu().numpy()

    return np.asarray(
        x,
        dtype=np.float64
    ).reshape(-1)


dev_unc_622h = arr1d_622h(
    dev_mc_std_622fd
)

val_unc_622h = arr1d_622h(
    mc_std
)

dev_pred_622h = arr1d_622h(
    dev_prediction_622g
)

val_pred_622h = arr1d_622h(
    val_prediction_622g
)

dev_mcmean_622h = arr1d_622h(
    dev_mc_mean_prediction_622fd
)

val_mcmean_622h = arr1d_622h(
    mc_mean_prediction
)

dev_target_622h = arr1d_622h(
    dev_target_622g
)

val_target_622h = arr1d_622h(
    val_target_622g
)


# ------------------------------------------------------------
# 3. Fit development phase-bin reference
# ------------------------------------------------------------

def fit_phase_reference_622h(
    phase,
    uncertainty,
    n_bins
):

    phase = arr1d_622h(phase)
    uncertainty = arr1d_622h(
        uncertainty
    )

    quantiles = np.linspace(
        0.0,
        1.0,
        n_bins + 1
    )

    raw_edges = np.quantile(
        phase,
        quantiles
    )

    # Interior edges only.
    # Unique protects against tied predictions.
    edges = np.unique(
        raw_edges[1:-1]
    )

    bin_ids = np.digitize(
        phase,
        edges,
        right=True
    )

    references = {}

    for b in np.unique(bin_ids):

        values = uncertainty[
            bin_ids == b
        ]

        references[int(b)] = np.sort(
            values
        )

    return edges, references


# ------------------------------------------------------------
# 4. Apply development-fitted reference
#
# Score = empirical percentile of MC uncertainty
# among DEVELOPMENT windows of similar predicted RUL.
# ------------------------------------------------------------

def apply_phase_percentile_622h(
    phase,
    uncertainty,
    edges,
    references
):

    phase = arr1d_622h(phase)
    uncertainty = arr1d_622h(
        uncertainty
    )

    bin_ids = np.digitize(
        phase,
        edges,
        right=True
    )

    result = np.zeros(
        len(phase),
        dtype=np.float64
    )

    for b in np.unique(bin_ids):

        mask = (
            bin_ids == b
        )

        if int(b) not in references:
            raise RuntimeError(
                f"No development reference "
                f"for phase bin {b}."
            )

        ref = references[int(b)]

        x = uncertainty[mask]

        left = np.searchsorted(
            ref,
            x,
            side="left"
        )

        right = np.searchsorted(
            ref,
            x,
            side="right"
        )

        # Mid-rank empirical percentile.
        result[mask] = (
            left + right
        ) / (
            2.0 * len(ref)
        )

    return result


# ------------------------------------------------------------
# 5. Evaluation helper
# ------------------------------------------------------------

def spearman_622h(x, y):

    return float(
        pd.Series(
            arr1d_622h(x)
        ).corr(
            pd.Series(
                arr1d_622h(y)
            ),
            method="spearman"
        )
    )


def evaluate_uncertainty_622h(
    score,
    prediction,
    target
):

    score = arr1d_622h(score)

    prediction = arr1d_622h(
        prediction
    )

    target = arr1d_622h(
        target
    )

    absolute_error = np.abs(
        prediction - target
    )

    phase_rho = spearman_622h(
        score,
        target
    )

    error_rho = spearman_622h(
        score,
        absolute_error
    )

    order = np.argsort(score)

    n = len(score)

    q = int(
        np.ceil(
            0.25 * n
        )
    )

    low_mae = float(
        absolute_error[
            order[:q]
        ].mean()
    )

    high_mae = float(
        absolute_error[
            order[-q:]
        ].mean()
    )

    high_low_ratio = (
        high_mae / low_mae
    )

    baseline_mae = float(
        absolute_error.mean()
    )

    coverage_gains = {}

    for coverage in [
        0.90,
        0.70,
        0.50
    ]:

        keep = int(
            round(
                coverage * n
            )
        )

        accepted_idx = order[:keep]

        accepted_mae = float(
            absolute_error[
                accepted_idx
            ].mean()
        )

        coverage_gains[
            coverage
        ] = (
            baseline_mae
            - accepted_mae
        )

    mean_gain = float(
        np.mean(
            list(
                coverage_gains.values()
            )
        )
    )

    return {
        "phase_rho": phase_rho,
        "error_rho": error_rho,
        "high_low_ratio":
            high_low_ratio,
        "gain90":
            coverage_gains[0.90],
        "gain70":
            coverage_gains[0.70],
        "gain50":
            coverage_gains[0.50],
        "mean_gain":
            mean_gain,
    }


# ------------------------------------------------------------
# 6. Baseline raw MC uncertainty
# ------------------------------------------------------------

dev_rows_622h = []
val_rows_622h = []


dev_raw_metrics = (
    evaluate_uncertainty_622h(
        dev_unc_622h,
        dev_pred_622h,
        dev_target_622h
    )
)

val_raw_metrics = (
    evaluate_uncertainty_622h(
        val_unc_622h,
        val_pred_622h,
        val_target_622h
    )
)


dev_rows_622h.append(
    {
        "candidate": "raw_mc_std",
        **dev_raw_metrics
    }
)

val_rows_622h.append(
    {
        "candidate": "raw_mc_std",
        **val_raw_metrics
    }
)


# ------------------------------------------------------------
# 7. Development-calibrated conditional percentiles
# ------------------------------------------------------------

phase_sources_622h = {
    "det_pred": (
        dev_pred_622h,
        val_pred_622h
    ),
    "mcmean": (
        dev_mcmean_622h,
        val_mcmean_622h
    ),
}

bin_counts_622h = [
    5,
    10,
    20
]


candidate_arrays_dev_622h = {}
candidate_arrays_val_622h = {}

phase_references_622h = {}


for phase_name, (
    dev_phase,
    val_phase
) in phase_sources_622h.items():

    for n_bins in bin_counts_622h:

        candidate_name = (
            f"{phase_name}_pct_b{n_bins}"
        )

        edges, refs = (
            fit_phase_reference_622h(
                dev_phase,
                dev_unc_622h,
                n_bins
            )
        )

        dev_score = (
            apply_phase_percentile_622h(
                dev_phase,
                dev_unc_622h,
                edges,
                refs
            )
        )

        val_score = (
            apply_phase_percentile_622h(
                val_phase,
                val_unc_622h,
                edges,
                refs
            )
        )

        candidate_arrays_dev_622h[
            candidate_name
        ] = dev_score

        candidate_arrays_val_622h[
            candidate_name
        ] = val_score

        phase_references_622h[
            candidate_name
        ] = {
            "edges": edges,
            "references": refs,
            "phase_source":
                phase_name,
            "requested_bins":
                n_bins,
            "actual_bins":
                len(refs),
        }


        dev_metrics = (
            evaluate_uncertainty_622h(
                dev_score,
                dev_pred_622h,
                dev_target_622h
            )
        )

        val_metrics = (
            evaluate_uncertainty_622h(
                val_score,
                val_pred_622h,
                val_target_622h
            )
        )


        dev_rows_622h.append(
            {
                "candidate":
                    candidate_name,
                **dev_metrics
            }
        )

        val_rows_622h.append(
            {
                "candidate":
                    candidate_name,
                **val_metrics
            }
        )


# ------------------------------------------------------------
# 8. Compact tables
# ------------------------------------------------------------

dev_table_622h = pd.DataFrame(
    dev_rows_622h
)

val_table_622h = pd.DataFrame(
    val_rows_622h
)


display_cols_622h = [
    "candidate",
    "phase_rho",
    "error_rho",
    "high_low_ratio",
    "gain90",
    "gain70",
    "gain50",
    "mean_gain",
]


print(
    "\n=== DEVELOPMENT — "
    "decision evidence ==="
)

print(
    dev_table_622h[
        display_cols_622h
    ]
    .round(4)
    .to_string(
        index=False
    )
)


# ------------------------------------------------------------
# 9. Development-only adjusted ranking
# ------------------------------------------------------------

adjusted_dev_622h = (
    dev_table_622h[
        dev_table_622h[
            "candidate"
        ] != "raw_mc_std"
    ]
    .copy()
)

adjusted_dev_622h[
    "abs_phase_rho"
] = (
    adjusted_dev_622h[
        "phase_rho"
    ].abs()
)

ranking_622h = (
    adjusted_dev_622h
    .sort_values(
        by=[
            "mean_gain",
            "error_rho",
            "abs_phase_rho",
        ],
        ascending=[
            False,
            False,
            True,
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n=== DEVELOPMENT-only "
    "within-phase ranking ==="
)

print(
    ranking_622h[
        [
            "candidate",
            "abs_phase_rho",
            "error_rho",
            "high_low_ratio",
            "mean_gain",
        ]
    ]
    .round(4)
    .to_string(
        index=False
    )
)


# ------------------------------------------------------------
# 10. Validation sanity for DEVELOPMENT winner
# ------------------------------------------------------------

best_candidate_622h = (
    ranking_622h.loc[
        0,
        "candidate"
    ]
)

best_val_row_622h = (
    val_table_622h[
        val_table_622h[
            "candidate"
        ] == best_candidate_622h
    ]
)


print(
    "\n=== VALIDATION sanity for "
    "DEVELOPMENT winner ==="
)

print(
    best_val_row_622h[
        display_cols_622h
    ]
    .round(4)
    .to_string(
        index=False
    )
)


print(
    "\nDevelopment winner:",
    best_candidate_622h
)


print("\n--- Integrity ---")
print(
    "Transformation fitted from DEVELOPMENT only : True"
)
print(
    "True RUL required at inference              : False"
)
print(
    "Validation used for candidate selection     : False"
)
print(
    "Official test used                         : False"
)
print(
    "Uncertainty signal frozen                  : False"
)
print(
    "Threshold selected                         : False"
)
print(
    "Abstention rule created                    : False"
)

print(
    "\n=== Step 6.22H completed ==="
)

In [ ]:
# ============================================================
# Step 6.22H-1 — Compact validation result for dev winner
# ============================================================

winner_622h = best_candidate_622h

dev_row = (
    dev_table_622h[
        dev_table_622h["candidate"] == winner_622h
    ]
    .iloc[0]
)

val_row = (
    val_table_622h[
        val_table_622h["candidate"] == winner_622h
    ]
    .iloc[0]
)

print("\n=== STEP 6.22H-1 COMPACT CHECK ===")
print("Development-selected winner:", winner_622h)

print("\nDEVELOPMENT")
print(
    f"phase_rho={dev_row['phase_rho']:.4f} | "
    f"error_rho={dev_row['error_rho']:.4f} | "
    f"high_low_ratio={dev_row['high_low_ratio']:.4f} | "
    f"mean_gain={dev_row['mean_gain']:.4f}"
)

print("\nCLEAN VALIDATION — sanity only")
print(
    f"phase_rho={val_row['phase_rho']:.4f} | "
    f"error_rho={val_row['error_rho']:.4f} | "
    f"high_low_ratio={val_row['high_low_ratio']:.4f} | "
    f"gain90={val_row['gain90']:.4f} | "
    f"gain70={val_row['gain70']:.4f} | "
    f"gain50={val_row['gain50']:.4f} | "
    f"mean_gain={val_row['mean_gain']:.4f}"
)

print("\nOfficial test used: False")
print("Signal frozen: False")
print("Threshold selected: False")
print("Abstention rule created: False")
print("=== END ===")

In [ ]:
# ============================================================
# Step 6.22I-A — Locate exact selected-uncertainty recipe
# Inspection only — nothing is modified or written
# ============================================================

import numpy as np
import pandas as pd

SELECTED_UNCERTAINTY_622I = "det_pred_pct_b5"

print("\n=== Step 6.22I-A — Selected uncertainty recipe inspection ===")
print("Selected candidate:", SELECTED_UNCERTAINTY_622I)

matches_622i = []

for obj_name, obj_value in list(globals().items()):
    name_lower = str(obj_name).lower()

    # Direct name match
    if SELECTED_UNCERTAINTY_622I.lower() in name_lower:
        matches_622i.append(
            {
                "object_name": obj_name,
                "type": type(obj_value).__name__,
                "shape": getattr(obj_value, "shape", None),
                "match_reason": "object_name",
            }
        )

    # Dictionary containing selected candidate
    elif isinstance(obj_value, dict):
        try:
            if SELECTED_UNCERTAINTY_622I in obj_value:
                selected_obj = obj_value[SELECTED_UNCERTAINTY_622I]
                matches_622i.append(
                    {
                        "object_name": obj_name,
                        "type": type(obj_value).__name__,
                        "shape": getattr(selected_obj, "shape", None),
                        "match_reason": "dictionary_key",
                    }
                )
        except Exception:
            pass

    # DataFrame containing candidate label
    elif isinstance(obj_value, pd.DataFrame):
        try:
            if "candidate" in obj_value.columns:
                candidate_values = (
                    obj_value["candidate"]
                    .astype(str)
                    .unique()
                    .tolist()
                )

                if SELECTED_UNCERTAINTY_622I in candidate_values:
                    matches_622i.append(
                        {
                            "object_name": obj_name,
                            "type": "DataFrame",
                            "shape": obj_value.shape,
                            "match_reason": "candidate_column",
                        }
                    )
        except Exception:
            pass

matches_df_622i = pd.DataFrame(matches_622i)

print("\n--- Objects connected with selected candidate ---")
if len(matches_df_622i):
    display(matches_df_622i)
else:
    print("No direct candidate-containing object found.")

print("\n--- Key current evidence ---")
print("Development-selected winner :", winner_622h)

dev_selected_622i = dev_table_622h[
    dev_table_622h["candidate"] == winner_622h
].iloc[0]

val_selected_622i = val_table_622h[
    val_table_622h["candidate"] == winner_622h
].iloc[0]

print(
    "Development phase_rho / error_rho / mean_gain :",
    round(float(dev_selected_622i["phase_rho"]), 6),
    "/",
    round(float(dev_selected_622i["error_rho"]), 6),
    "/",
    round(float(dev_selected_622i["mean_gain"]), 6),
)

print(
    "Validation phase_rho / error_rho / mean_gain  :",
    round(float(val_selected_622i["phase_rho"]), 6),
    "/",
    round(float(val_selected_622i["error_rho"]), 6),
    "/",
    round(float(val_selected_622i["mean_gain"]), 6),
)

print("\nOfficial test used        : False")
print("Signal frozen            : False")
print("Threshold selected       : False")
print("Abstention rule created  : False")
print("Nothing was modified.")
print("=== END ===")

In [ ]:
# ============================================================
# Step 6.22I-B — Recover exact det_pred_pct_b5 construction
# Inspection only
# ============================================================

SELECTED_622I = "det_pred_pct_b5"

print("\n=== STEP 6.22I-B — EXACT RECIPE RECOVERY ===")

# ------------------------------------------------------------
# 1. Inspect stored phase-reference entry
# ------------------------------------------------------------
print("\n--- Stored phase reference ---")

if (
    "phase_references_622h" in globals()
    and isinstance(phase_references_622h, dict)
    and SELECTED_622I in phase_references_622h
):
    ref_622i = phase_references_622h[SELECTED_622I]

    print("Type :", type(ref_622i).__name__)

    if isinstance(ref_622i, dict):
        print("Keys :", list(ref_622i.keys()))
        for k, v in ref_622i.items():
            if np.isscalar(v) or isinstance(v, (str, list, tuple)):
                print(f"{k}: {v}")
            else:
                print(
                    f"{k}: type={type(v).__name__}, "
                    f"shape={getattr(v, 'shape', None)}"
                )
    else:
        print("Value:", ref_622i)

else:
    print("No direct stored phase-reference entry found.")


# ------------------------------------------------------------
# 2. Find the executed notebook cell that created the candidate
# ------------------------------------------------------------
print("\n--- Notebook-history recipe search ---")

matching_cells_622i = []

for cell_number, cell_source in enumerate(In):
    if (
        isinstance(cell_source, str)
        and SELECTED_622I in cell_source
        and "candidate_arrays_dev_622h" in cell_source
    ):
        matching_cells_622i.append((cell_number, cell_source))

print("Matching construction cells:", len(matching_cells_622i))

if matching_cells_622i:
    # Use the most recent construction cell
    cell_number_622i, source_622i = matching_cells_622i[-1]

    print("Selected history cell:", cell_number_622i)

    lines_622i = source_622i.splitlines()

    hit_indices_622i = [
        i for i, line in enumerate(lines_622i)
        if (
            SELECTED_622I in line
            or "det_pred_pct" in line
            or "candidate_arrays_dev_622h" in line
            or "phase_references_622h" in line
        )
    ]

    if hit_indices_622i:
        start_622i = max(0, min(hit_indices_622i) - 15)
        end_622i = min(len(lines_622i), max(hit_indices_622i) + 20)

        print("\nRelevant source excerpt:\n")

        for i in range(start_622i, end_622i):
            print(f"{i+1:03d}: {lines_622i[i]}")
    else:
        print("Candidate name found, but nearby construction lines were not isolated.")

else:
    print("No construction cell found in current Jupyter history.")


# ------------------------------------------------------------
# 3. Confirm stored selected arrays
# ------------------------------------------------------------
print("\n--- Selected candidate arrays ---")

dev_arr_622i = np.asarray(
    candidate_arrays_dev_622h[SELECTED_622I],
    dtype=float
)

val_arr_622i = np.asarray(
    candidate_arrays_val_622h[SELECTED_622I],
    dtype=float
)

print(
    "Development:",
    "shape =", dev_arr_622i.shape,
    "| finite =", bool(np.isfinite(dev_arr_622i).all()),
    "| min =", round(float(dev_arr_622i.min()), 6),
    "| median =", round(float(np.median(dev_arr_622i)), 6),
    "| max =", round(float(dev_arr_622i.max()), 6),
)

print(
    "Validation :",
    "shape =", val_arr_622i.shape,
    "| finite =", bool(np.isfinite(val_arr_622i).all()),
    "| min =", round(float(val_arr_622i.min()), 6),
    "| median =", round(float(np.median(val_arr_622i)), 6),
    "| max =", round(float(val_arr_622i.max()), 6),
)

print("\nNothing was modified.")
print("=== END ===")

In [ ]:
# ============================================================
# Step 6.22I-C — Compact exact recipe inspection
# Inspection only
# ============================================================

import numpy as np

NAME_622IC = "det_pred_pct_b5"

print("\n=== STEP 6.22I-C — COMPACT RECIPE INSPECTION ===")

# ------------------------------------------------------------
# 1. Inspect the frozen/stored phase reference itself
# ------------------------------------------------------------
ref_622ic = phase_references_622h[NAME_622IC]

print("\n--- Phase-reference definition ---")
print("phase_source   :", ref_622ic.get("phase_source"))
print("requested_bins :", ref_622ic.get("requested_bins"))
print("actual_bins    :", ref_622ic.get("actual_bins"))

edges_622ic = np.asarray(ref_622ic.get("edges"), dtype=float)

print(
    "edges          :",
    np.round(edges_622ic, 6).tolist()
)

references_622ic = ref_622ic.get("references")

print("\n--- Reference dictionary ---")
print("type :", type(references_622ic).__name__)

if isinstance(references_622ic, dict):

    print("keys :", list(references_622ic.keys()))

    for key_622ic, value_622ic in references_622ic.items():

        arr_622ic = np.asarray(value_622ic)

        print(
            f"{key_622ic}: "
            f"shape={arr_622ic.shape}, "
            f"dtype={arr_622ic.dtype}",
            end=""
        )

        if np.issubdtype(arr_622ic.dtype, np.number) and arr_622ic.size > 0:
            arr_num_622ic = arr_622ic.astype(float).ravel()

            print(
                f", min={np.min(arr_num_622ic):.6f}, "
                f"median={np.median(arr_num_622ic):.6f}, "
                f"max={np.max(arr_num_622ic):.6f}"
            )
        else:
            print()

else:
    print("Reference object is not a dictionary.")


# ------------------------------------------------------------
# 2. Search ONLY older notebook cells
#    Exclude current recovery/inspection cells
# ------------------------------------------------------------
print("\n--- Older construction-code search ---")

search_terms_622ic = [
    "phase_references_622h",
    "candidate_arrays_dev_622h",
    "det_pred_pct",
    "percentile",
    "searchsorted",
    "rank",
]

hits_622ic = []

# Exclude last few inspection cells automatically
for cell_number_622ic, source_622ic in enumerate(In[:-2]):

    if not isinstance(source_622ic, str):
        continue

    lower_622ic = source_622ic.lower()

    if (
        "6.22i-b" in lower_622ic
        or "6.22i-c" in lower_622ic
        or "exact recipe recovery" in lower_622ic
    ):
        continue

    matched_622ic = [
        term
        for term in search_terms_622ic
        if term.lower() in lower_622ic
    ]

    if matched_622ic:
        hits_622ic.append(
            (
                cell_number_622ic,
                matched_622ic,
                source_622ic
            )
        )

print("Older candidate cells found:", len(hits_622ic))

# Show only the most relevant recent matches
for cell_number_622ic, matched_622ic, source_622ic in hits_622ic[-4:]:

    print(
        f"\nCell {cell_number_622ic} | "
        f"matched={matched_622ic}"
    )

    lines_622ic = source_622ic.splitlines()

    interesting_622ic = []

    for i_622ic, line_622ic in enumerate(lines_622ic):
        if any(
            term.lower() in line_622ic.lower()
            for term in search_terms_622ic
        ):
            interesting_622ic.append(i_622ic)

    if interesting_622ic:

        start_622ic = max(0, min(interesting_622ic) - 3)
        end_622ic = min(
            len(lines_622ic),
            max(interesting_622ic) + 5
        )

        for i_622ic in range(start_622ic, end_622ic):
            print(
                f"{i_622ic + 1:03d}: "
                f"{lines_622ic[i_622ic]}"
            )


print("\nNothing was modified.")
print("=== END ===")

In [ ]:
# ============================================================
# Step 6.22I-D — Recover exact det_pred_pct_b5 construction
# Inspection only
# ============================================================

SELECTED_622ID = "det_pred_pct_b5"

print("\n=== STEP 6.22I-D — EXACT CONSTRUCTION SEARCH ===")

important_terms_622id = [
    "det_pred_pct_b5",
    "phase_references_622h",
    "candidate_arrays_dev_622h",
    "candidate_arrays_val_622h",
    "searchsorted",
    "digitize",
    "quantile",
    "percentile",
]

hits_622id = []

for cell_no_622id, src_622id in enumerate(In):

    if not isinstance(src_622id, str):
        continue

    low_622id = src_622id.lower()

    # Exclude the recent inspection/recovery cells
    if (
        "step 6.22i-a" in low_622id
        or "step 6.22i-b" in low_622id
        or "step 6.22i-c" in low_622id
        or "step 6.22i-d" in low_622id
    ):
        continue

    if SELECTED_622ID.lower() not in low_622id:
        continue

    score_622id = sum(
        term.lower() in low_622id
        for term in important_terms_622id
    )

    hits_622id.append(
        (score_622id, cell_no_622id, src_622id)
    )


hits_622id.sort(reverse=True)

print("Matching older cells:", len(hits_622id))

if len(hits_622id) == 0:
    print("No older construction cell found.")

else:
    # Only inspect the strongest match
    score_622id, cell_no_622id, src_622id = hits_622id[0]

    print("Best matching cell :", cell_no_622id)
    print("Match score        :", score_622id)

    lines_622id = src_622id.splitlines()

    relevant_idx_622id = []

    for i_622id, line_622id in enumerate(lines_622id):
        if any(
            term.lower() in line_622id.lower()
            for term in important_terms_622id
        ):
            relevant_idx_622id.append(i_622id)

    if relevant_idx_622id:

        # Merge small relevant regions rather than printing the whole cell
        regions_622id = []

        for idx_622id in relevant_idx_622id:
            start_622id = max(0, idx_622id - 3)
            end_622id = min(len(lines_622id), idx_622id + 5)

            if (
                regions_622id
                and start_622id <= regions_622id[-1][1]
            ):
                regions_622id[-1] = (
                    regions_622id[-1][0],
                    max(regions_622id[-1][1], end_622id)
                )
            else:
                regions_622id.append(
                    (start_622id, end_622id)
                )

        # Limit output to the first few compact regions
        for region_no_622id, (
            start_622id,
            end_622id
        ) in enumerate(regions_622id[:4], start=1):

            print(
                f"\n--- Relevant region {region_no_622id} ---"
            )

            for i_622id in range(
                start_622id,
                end_622id
            ):
                print(
                    f"{i_622id + 1:03d}: "
                    f"{lines_622id[i_622id]}"
                )

print("\nNothing was modified.")
print("=== END ===")

In [ ]:
# ============================================================
# Step 6.22I-E — Numerically identify exact det_pred_pct_b5 recipe
# Inspection only
# ============================================================

import numpy as np
import pandas as pd

print("\n=== STEP 6.22I-E — NUMERICAL RECIPE IDENTIFICATION ===")

required_622ie = [
    "phase_references_622h",
    "candidate_arrays_val_622h",
    "clean_predictions",
    "mc_std",
]

for name_622ie in required_622ie:
    print(
        f"{name_622ie:30s}:",
        name_622ie in globals()
    )

missing_622ie = [
    x for x in required_622ie
    if x not in globals()
]

if missing_622ie:
    raise RuntimeError(
        f"Missing required objects: {missing_622ie}"
    )

# ------------------------------------------------------------
# 1. Recover stored objects
# ------------------------------------------------------------

selected_name_622ie = "det_pred_pct_b5"

stored_622ie = np.asarray(
    candidate_arrays_val_622h[selected_name_622ie],
    dtype=np.float64
)

det_pred_622ie = np.asarray(
    clean_predictions,
    dtype=np.float64
)

uncertainty_622ie = np.asarray(
    mc_std,
    dtype=np.float64
)

edges_622ie = np.asarray(
    phase_references_622h["edges"],
    dtype=np.float64
)

refs_622ie = phase_references_622h["references"]

print("\n--- Basic structure ---")
print("Stored candidate shape :", stored_622ie.shape)
print("Prediction shape       :", det_pred_622ie.shape)
print("Uncertainty shape      :", uncertainty_622ie.shape)
print("Phase edges            :", edges_622ie.tolist())
print("Reference bins         :", sorted(refs_622ie.keys()))

if not (
    stored_622ie.shape
    == det_pred_622ie.shape
    == uncertainty_622ie.shape
):
    raise RuntimeError(
        "Validation arrays are not aligned."
    )


# ------------------------------------------------------------
# 2. Candidate percentile formulas
# ------------------------------------------------------------

def percentile_formula_622ie(
    x,
    ref,
    search_side,
    formula_name
):
    ref = np.sort(
        np.asarray(ref, dtype=np.float64)
    )

    n = len(ref)

    pos = np.searchsorted(
        ref,
        x,
        side=search_side
    ).astype(np.float64)

    if formula_name == "pos_div_n":
        out = pos / n

    elif formula_name == "pos_div_nplus1":
        out = pos / (n + 1.0)

    elif formula_name == "posplus1_div_nplus1":
        out = (pos + 1.0) / (n + 1.0)

    elif formula_name == "clip_pos_1_n_div_nplus1":
        out = (
            np.clip(pos, 1.0, float(n))
            / (n + 1.0)
        )

    elif formula_name == "clip_posplus1_1_n_div_nplus1":
        out = (
            np.clip(pos + 1.0, 1.0, float(n))
            / (n + 1.0)
        )

    elif formula_name == "posplus_half_div_n":
        out = (pos + 0.5) / n

    elif formula_name == "posminus_half_div_n":
        out = (pos - 0.5) / n

    else:
        raise ValueError(formula_name)

    return out


formula_names_622ie = [
    "pos_div_n",
    "pos_div_nplus1",
    "posplus1_div_nplus1",
    "clip_pos_1_n_div_nplus1",
    "clip_posplus1_1_n_div_nplus1",
    "posplus_half_div_n",
    "posminus_half_div_n",
]


# ------------------------------------------------------------
# 3. Test BOTH phase-bin conventions and BOTH percentile
#    search-side conventions
# ------------------------------------------------------------

results_622ie = []

for phase_side_622ie in ["left", "right"]:

    phase_bin_622ie = np.searchsorted(
        edges_622ie,
        det_pred_622ie,
        side=phase_side_622ie
    )

    if (
        phase_bin_622ie.min() < 0
        or phase_bin_622ie.max() > 4
    ):
        raise RuntimeError(
            "Unexpected phase-bin index."
        )

    for percentile_side_622ie in ["left", "right"]:

        for formula_622ie in formula_names_622ie:

            reconstructed_622ie = np.empty_like(
                stored_622ie
            )

            for b_622ie in range(5):

                mask_622ie = (
                    phase_bin_622ie == b_622ie
                )

                if not np.any(mask_622ie):
                    continue

                reconstructed_622ie[mask_622ie] = (
                    percentile_formula_622ie(
                        uncertainty_622ie[mask_622ie],
                        refs_622ie[b_622ie],
                        percentile_side_622ie,
                        formula_622ie
                    )
                )

            diff_622ie = np.abs(
                reconstructed_622ie
                - stored_622ie
            )

            results_622ie.append({
                "phase_side":
                    phase_side_622ie,
                "percentile_side":
                    percentile_side_622ie,
                "formula":
                    formula_622ie,
                "mean_abs_error":
                    float(diff_622ie.mean()),
                "max_abs_error":
                    float(diff_622ie.max()),
                "exact_allclose":
                    bool(
                        np.allclose(
                            reconstructed_622ie,
                            stored_622ie,
                            rtol=0.0,
                            atol=1e-12
                        )
                    )
            })


results_df_622ie = (
    pd.DataFrame(results_622ie)
    .sort_values(
        [
            "max_abs_error",
            "mean_abs_error"
        ]
    )
    .reset_index(drop=True)
)

print("\n--- Best numerical matches ---")
display(results_df_622ie.head(10))


# ------------------------------------------------------------
# 4. Best recipe compact result
# ------------------------------------------------------------

best_622ie = results_df_622ie.iloc[0]

print("\n============================================")
print("STEP 6.22I-E BEST RECIPE")
print("============================================")
print(
    "Phase-bin search side       :",
    best_622ie["phase_side"]
)
print(
    "Percentile search side      :",
    best_622ie["percentile_side"]
)
print(
    "Percentile formula          :",
    best_622ie["formula"]
)
print(
    "Mean absolute difference    :",
    best_622ie["mean_abs_error"]
)
print(
    "Maximum absolute difference :",
    best_622ie["max_abs_error"]
)
print(
    "Exact within 1e-12          :",
    best_622ie["exact_allclose"]
)

print("\nStored candidate range:")
print(
    "min =",
    float(stored_622ie.min()),
    "| median =",
    float(np.median(stored_622ie)),
    "| max =",
    float(stored_622ie.max())
)

print("\nNothing was modified.")
print("=== END ===")

In [ ]:
# ============================================================
# Step 6.22I-E0 — Locate exact stored phase-reference object
# Inspection only
# ============================================================

print("\n=== STEP 6.22I-E0 — PHASE REFERENCE LOCATION DIAGNOSTIC ===")

# ------------------------------------------------------------
# 1. Inspect current phase_references_622h
# ------------------------------------------------------------

print("\n--- Current phase_references_622h ---")

if "phase_references_622h" not in globals():
    print("Object does not exist.")
else:
    obj_622ie0 = phase_references_622h

    print("Type :", type(obj_622ie0).__name__)

    if isinstance(obj_622ie0, dict):

        top_keys_622ie0 = list(obj_622ie0.keys())

        print("Number of top-level keys :", len(top_keys_622ie0))
        print("Top-level keys :", top_keys_622ie0)

        print("\nNested dictionary structure:")

        for key_622ie0, value_622ie0 in obj_622ie0.items():

            if isinstance(value_622ie0, dict):
                print(
                    f"{key_622ie0!r} -> dict keys:",
                    list(value_622ie0.keys())
                )

            else:
                shape_622ie0 = getattr(
                    value_622ie0,
                    "shape",
                    None
                )

                print(
                    f"{key_622ie0!r} -> "
                    f"type={type(value_622ie0).__name__}, "
                    f"shape={shape_622ie0}"
                )


# ------------------------------------------------------------
# 2. Search notebook namespace for dictionaries containing
#    BOTH 'edges' and 'references'
# ------------------------------------------------------------

print("\n--- Namespace search for exact phase-reference structure ---")

matches_622ie0 = []

namespace_snapshot_622ie0 = list(
    globals().items()
)

for name_622ie0, value_622ie0 in namespace_snapshot_622ie0:

    if not isinstance(value_622ie0, dict):
        continue

    # Top-level match
    keys_622ie0 = set(value_622ie0.keys())

    if {"edges", "references"}.issubset(keys_622ie0):

        matches_622ie0.append(
            (
                name_622ie0,
                "top_level",
                list(value_622ie0.keys())
            )
        )

    # One-level nested match
    for nested_key_622ie0, nested_value_622ie0 in value_622ie0.items():

        if not isinstance(nested_value_622ie0, dict):
            continue

        nested_keys_622ie0 = set(
            nested_value_622ie0.keys()
        )

        if {"edges", "references"}.issubset(
            nested_keys_622ie0
        ):

            matches_622ie0.append(
                (
                    name_622ie0,
                    f"nested[{nested_key_622ie0!r}]",
                    list(nested_value_622ie0.keys())
                )
            )


if len(matches_622ie0) == 0:

    print("No dictionary containing BOTH 'edges' and 'references' was found.")

else:

    print(
        "Matching phase-reference structures found:",
        len(matches_622ie0)
    )

    for i_622ie0, (
        name_622ie0,
        location_622ie0,
        keys_found_622ie0
    ) in enumerate(matches_622ie0, start=1):

        print(
            f"{i_622ie0:02d}. "
            f"object={name_622ie0} | "
            f"location={location_622ie0}"
        )

        print(
            "    keys=",
            keys_found_622ie0
        )


# ------------------------------------------------------------
# 3. Search likely phase/reference-named dictionaries
# ------------------------------------------------------------

print("\n--- Other phase/reference dictionaries ---")

likely_622ie0 = []

for name_622ie0, value_622ie0 in namespace_snapshot_622ie0:

    lower_622ie0 = str(name_622ie0).lower()

    if (
        isinstance(value_622ie0, dict)
        and (
            "phase" in lower_622ie0
            or "reference" in lower_622ie0
        )
    ):

        likely_622ie0.append(
            (
                name_622ie0,
                list(value_622ie0.keys())[:20]
            )
        )


for name_622ie0, keys_622ie0 in likely_622ie0:
    print(
        f"{name_622ie0}: {keys_622ie0}"
    )


print("\nNothing was modified.")
print("=== END ===")

In [ ]:
# ============================================================
# Step 6.22I-E1
# Numerically verify exact det_pred_pct_b5 recipe
# Inspection only — nothing is modified
# ============================================================

import numpy as np
import pandas as pd

print("\n=== STEP 6.22I-E1 — EXACT NUMERICAL RECIPE VERIFICATION ===")

SELECTED_622IE1 = "det_pred_pct_b5"

# ------------------------------------------------------------
# 1. Required objects
# ------------------------------------------------------------

required_622ie1 = [
    "phase_references_622h",
    "candidate_arrays_val_622h",
    "clean_predictions",
    "mc_std",
]

for name_622ie1 in required_622ie1:
    print(
        f"{name_622ie1:28s}:",
        name_622ie1 in globals()
    )

missing_622ie1 = [
    x for x in required_622ie1
    if x not in globals()
]

if missing_622ie1:
    raise RuntimeError(
        f"Missing required objects: {missing_622ie1}"
    )


# ------------------------------------------------------------
# 2. Recover SELECTED nested phase reference
# ------------------------------------------------------------

if SELECTED_622IE1 not in phase_references_622h:
    raise KeyError(
        f"{SELECTED_622IE1} not found in phase_references_622h"
    )

selected_ref_622ie1 = (
    phase_references_622h[SELECTED_622IE1]
)

edges_622ie1 = np.asarray(
    selected_ref_622ie1["edges"],
    dtype=np.float64
)

references_622ie1 = (
    selected_ref_622ie1["references"]
)

stored_val_622ie1 = np.asarray(
    candidate_arrays_val_622h[SELECTED_622IE1],
    dtype=np.float64
)

det_pred_val_622ie1 = np.asarray(
    clean_predictions,
    dtype=np.float64
)

uncertainty_val_622ie1 = np.asarray(
    mc_std,
    dtype=np.float64
)

print("\n--- Selected reference ---")
print("Candidate       :", SELECTED_622IE1)
print(
    "Phase source    :",
    selected_ref_622ie1.get("phase_source")
)
print(
    "Requested bins  :",
    selected_ref_622ie1.get("requested_bins")
)
print(
    "Actual bins     :",
    selected_ref_622ie1.get("actual_bins")
)
print("Edges           :", edges_622ie1.tolist())

print("\nValidation arrays:")
print("det_pred shape  :", det_pred_val_622ie1.shape)
print("uncertainty     :", uncertainty_val_622ie1.shape)
print("stored candidate:", stored_val_622ie1.shape)


# ------------------------------------------------------------
# 3. Reference getter
# ------------------------------------------------------------

def get_reference_622ie1(ref_dict, bin_index):

    if bin_index in ref_dict:
        value = ref_dict[bin_index]

    elif str(bin_index) in ref_dict:
        value = ref_dict[str(bin_index)]

    else:
        raise KeyError(
            f"No reference array for phase {bin_index}"
        )

    return np.sort(
        np.asarray(
            value,
            dtype=np.float64
        )
    )


# ------------------------------------------------------------
# 4. Candidate reconstruction
#
# Test the plausible empirical-CDF conventions.
# ------------------------------------------------------------

def reconstruct_622ie1(
    phase_side,
    score_rule
):

    phase_index = np.searchsorted(
        edges_622ie1,
        det_pred_val_622ie1,
        side=phase_side
    )

    reconstructed = np.empty(
        len(det_pred_val_622ie1),
        dtype=np.float64
    )

    for phase_622ie1 in np.unique(phase_index):

        mask_622ie1 = (
            phase_index == phase_622ie1
        )

        ref_622ie1 = get_reference_622ie1(
            references_622ie1,
            int(phase_622ie1)
        )

        values_622ie1 = (
            uncertainty_val_622ie1[
                mask_622ie1
            ]
        )

        n_622ie1 = len(ref_622ie1)

        left_622ie1 = np.searchsorted(
            ref_622ie1,
            values_622ie1,
            side="left"
        )

        right_622ie1 = np.searchsorted(
            ref_622ie1,
            values_622ie1,
            side="right"
        )

        if score_rule == "right_over_n":

            score_622ie1 = (
                right_622ie1 / n_622ie1
            )

        elif score_rule == "left_over_n":

            score_622ie1 = (
                left_622ie1 / n_622ie1
            )

        elif score_rule == "midrank_over_n":

            score_622ie1 = (
                0.5
                * (
                    left_622ie1
                    + right_622ie1
                )
                / n_622ie1
            )

        elif score_rule == "right_over_nplus1":

            score_622ie1 = (
                right_622ie1
                / (n_622ie1 + 1.0)
            )

        elif score_rule == "leftplus1_over_nplus1":

            score_622ie1 = (
                (left_622ie1 + 1.0)
                / (n_622ie1 + 1.0)
            )

        else:
            raise ValueError(score_rule)

        reconstructed[
            mask_622ie1
        ] = score_622ie1

    return reconstructed


score_rules_622ie1 = [
    "right_over_n",
    "left_over_n",
    "midrank_over_n",
    "right_over_nplus1",
    "leftplus1_over_nplus1",
]

rows_622ie1 = []

for phase_side_622ie1 in [
    "right",
    "left",
]:

    for score_rule_622ie1 in (
        score_rules_622ie1
    ):

        reconstructed_622ie1 = (
            reconstruct_622ie1(
                phase_side_622ie1,
                score_rule_622ie1
            )
        )

        difference_622ie1 = np.abs(
            reconstructed_622ie1
            - stored_val_622ie1
        )

        rows_622ie1.append(
            {
                "phase_side":
                    phase_side_622ie1,

                "score_rule":
                    score_rule_622ie1,

                "mean_abs_error":
                    float(
                        np.mean(
                            difference_622ie1
                        )
                    ),

                "max_abs_error":
                    float(
                        np.max(
                            difference_622ie1
                        )
                    ),

                "exact_allclose":
                    bool(
                        np.allclose(
                            reconstructed_622ie1,
                            stored_val_622ie1,
                            rtol=0.0,
                            atol=1e-12
                        )
                    ),
            }
        )


comparison_622ie1 = (
    pd.DataFrame(rows_622ie1)
    .sort_values(
        [
            "max_abs_error",
            "mean_abs_error"
        ]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 5. Compact result
# ------------------------------------------------------------

print("\n--- Reconstruction comparison ---")

print(
    comparison_622ie1
    .head(10)
    .to_string(index=False)
)

best_622ie1 = (
    comparison_622ie1.iloc[0]
)

print("\n--- BEST MATCH ---")
print(
    "Phase assignment :",
    best_622ie1["phase_side"]
)
print(
    "Percentile rule  :",
    best_622ie1["score_rule"]
)
print(
    "Mean abs error   :",
    best_622ie1["mean_abs_error"]
)
print(
    "Maximum error    :",
    best_622ie1["max_abs_error"]
)
print(
    "Exact allclose   :",
    best_622ie1["exact_allclose"]
)


# ------------------------------------------------------------
# 6. Integrity
# ------------------------------------------------------------

print("\nOfficial test used       : False")
print("Signal frozen            : False")
print("Threshold selected       : False")
print("Abstention rule created  : False")
print("Nothing was modified.")

print("=== END ===")

In [ ]:
# ============================================================
# Step 6.22I-E2
# Recover exact phase-bin boundary convention from notebook
# Inspection only — nothing is modified
# ============================================================

import re

print("\n=== STEP 6.22I-E2 — PHASE-BOUNDARY CONVENTION RECOVERY ===")

ip_622ie2 = get_ipython()

history_622ie2 = list(
    ip_622ie2.history_manager.input_hist_raw
)

rows_622ie2 = []

for cell_no_622ie2, source_622ie2 in enumerate(
    history_622ie2
):

    if not source_622ie2:
        continue

    contains_candidate_622ie2 = (
        "candidate_arrays_dev_622h"
        in source_622ie2
    )

    contains_reference_622ie2 = (
        "phase_references_622h"
        in source_622ie2
    )

    contains_searchsorted_622ie2 = (
        "searchsorted"
        in source_622ie2
    )

    contains_digitize_622ie2 = (
        "digitize"
        in source_622ie2
    )

    assignment_622ie2 = (
        "candidate_arrays_dev_622h ="
        in source_622ie2
        or
        "phase_references_622h ="
        in source_622ie2
    )

    if (
        contains_candidate_622ie2
        or contains_reference_622ie2
    ):

        score_622ie2 = 0

        score_622ie2 += (
            10 if assignment_622ie2 else 0
        )

        score_622ie2 += (
            5 if contains_searchsorted_622ie2
            else 0
        )

        score_622ie2 += (
            5 if contains_digitize_622ie2
            else 0
        )

        score_622ie2 += (
            3
            if "edges" in source_622ie2
            else 0
        )

        score_622ie2 += (
            3
            if "references" in source_622ie2
            else 0
        )

        rows_622ie2.append(
            (
                score_622ie2,
                cell_no_622ie2,
                source_622ie2
            )
        )


rows_622ie2.sort(
    key=lambda x: x[0],
    reverse=True
)

print(
    "Candidate history cells found:",
    len(rows_622ie2)
)


# ------------------------------------------------------------
# Find strongest likely construction cell
# ------------------------------------------------------------

selected_source_622ie2 = None
selected_cell_622ie2 = None

for (
    score_622ie2,
    cell_no_622ie2,
    source_622ie2
) in rows_622ie2:

    if (
        "searchsorted" in source_622ie2
        or
        "digitize" in source_622ie2
    ):

        selected_source_622ie2 = (
            source_622ie2
        )

        selected_cell_622ie2 = (
            cell_no_622ie2
        )

        break


print(
    "Best likely construction cell:",
    selected_cell_622ie2
)


# ------------------------------------------------------------
# Compact relevant-line extraction
# ------------------------------------------------------------

if selected_source_622ie2 is None:

    print(
        "\nNo construction cell containing "
        "searchsorted/digitize was recovered."
    )

else:

    lines_622ie2 = (
        selected_source_622ie2.splitlines()
    )

    keywords_622ie2 = [
        "searchsorted",
        "digitize",
        "side=",
        "side =",
        "right=",
        "right =",
        "edges",
        "phase_index",
        "phase_idx",
        "bin_index",
        "bin_idx",
    ]

    relevant_numbers_622ie2 = set()

    for i_622ie2, line_622ie2 in enumerate(
        lines_622ie2
    ):

        if any(
            key_622ie2 in line_622ie2
            for key_622ie2
            in keywords_622ie2
        ):

            # one line before and after
            for j_622ie2 in range(
                max(0, i_622ie2 - 1),
                min(
                    len(lines_622ie2),
                    i_622ie2 + 2
                )
            ):
                relevant_numbers_622ie2.add(
                    j_622ie2
                )

    print(
        "\n--- Relevant original source lines ---"
    )

    for j_622ie2 in sorted(
        relevant_numbers_622ie2
    ):

        print(
            f"{j_622ie2 + 1:03d}: "
            f"{lines_622ie2[j_622ie2]}"
        )


    # --------------------------------------------------------
    # Try automatic convention identification
    # --------------------------------------------------------

    explicit_side_622ie2 = re.findall(
        r"""side\s*=\s*['"]"""
        r"""(left|right)['"]""",
        selected_source_622ie2
    )

    explicit_right_622ie2 = re.findall(
        r"""right\s*=\s*(True|False)""",
        selected_source_622ie2
    )

    print("\n--- Automatic interpretation ---")

    if explicit_side_622ie2:

        print(
            "Explicit searchsorted side:",
            explicit_side_622ie2
        )

    elif (
        "np.searchsorted"
        in selected_source_622ie2
    ):

        print(
            "searchsorted found, but no "
            "explicit side detected."
        )

        print(
            "NumPy default would therefore "
            "be side='left'."
        )

    if explicit_right_622ie2:

        print(
            "Explicit digitize right:",
            explicit_right_622ie2
        )

    elif (
        "np.digitize"
        in selected_source_622ie2
    ):

        print(
            "digitize found, but no explicit "
            "right argument detected."
        )

        print(
            "NumPy default would therefore "
            "be right=False."
        )


print("\nNothing was modified.")
print("=== END ===")

In [ ]:
# ============================================================
# Step 6.22I-F
# Freeze selected uncertainty signal
#
# Selected signal:
# det_pred_pct_b5
#
# IMPORTANT:
# - Development-derived definition only
# - Clean validation used only for reproducibility/sanity check
# - No uncertainty threshold is selected here
# - No abstention rule is created here
# - Official test is NOT used
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import json
import hashlib
from datetime import datetime

print(
    "\n=== Step 6.22I-F — "
    "Freeze selected uncertainty signal ==="
)

# ------------------------------------------------------------
# 1. Frozen candidate identity
# ------------------------------------------------------------

FROZEN_UNCERTAINTY_CANDIDATE_622IF = (
    "det_pred_pct_b5"
)

FROZEN_UNCERTAINTY_PHASE_SOURCE_622IF = (
    "deterministic_prediction"
)

FROZEN_UNCERTAINTY_PHASE_BINS_622IF = 5

FROZEN_UNCERTAINTY_BIN_RULE_622IF = (
    "numpy_digitize_right_true"
)

FROZEN_UNCERTAINTY_RANK_RULE_622IF = (
    "empirical_midrank_percentile_within_phase"
)


# ------------------------------------------------------------
# 2. Required in-memory objects
# ------------------------------------------------------------

required_622if = [
    "phase_references_622h",
    "candidate_arrays_dev_622h",
    "candidate_arrays_val_622h",
]

print("\n--- Required objects ---")

for name_622if in required_622if:
    print(
        f"{name_622if:<30}:",
        name_622if in globals()
    )

missing_622if = [
    x
    for x in required_622if
    if x not in globals()
]

if missing_622if:
    raise RuntimeError(
        "Missing required object(s): "
        + ", ".join(missing_622if)
    )


# ------------------------------------------------------------
# 3. Recover exact frozen reference
# ------------------------------------------------------------

reference_622if = (
    phase_references_622h[
        FROZEN_UNCERTAINTY_CANDIDATE_622IF
    ]
)

edges_622if = np.asarray(
    reference_622if["edges"],
    dtype=np.float64
)

references_622if = (
    reference_622if["references"]
)

phase_source_622if = (
    reference_622if["phase_source"]
)

requested_bins_622if = int(
    reference_622if["requested_bins"]
)

actual_bins_622if = int(
    reference_622if["actual_bins"]
)


print("\n--- Exact recovered recipe ---")
print(
    "Candidate        :",
    FROZEN_UNCERTAINTY_CANDIDATE_622IF
)
print(
    "Phase source     :",
    phase_source_622if
)
print(
    "Requested bins   :",
    requested_bins_622if
)
print(
    "Actual bins      :",
    actual_bins_622if
)
print(
    "Boundary rule    : np.digitize(..., right=True)"
)
print(
    "Ranking rule     : empirical mid-rank / N"
)
print(
    "Interior edges   :",
    edges_622if.tolist()
)


# ------------------------------------------------------------
# 4. Basic structural integrity
# ------------------------------------------------------------

if requested_bins_622if != 5:
    raise RuntimeError(
        "Unexpected requested-bin count."
    )

if actual_bins_622if != 5:
    raise RuntimeError(
        "Unexpected actual-bin count."
    )

if len(edges_622if) != 4:
    raise RuntimeError(
        "Five bins require four interior edges."
    )

if sorted(references_622if.keys()) != [
    0, 1, 2, 3, 4
]:
    raise RuntimeError(
        "Unexpected reference-bin keys."
    )

for k_622if in range(5):

    arr_622if = np.asarray(
        references_622if[k_622if],
        dtype=np.float64
    )

    if arr_622if.ndim != 1:
        raise RuntimeError(
            f"Reference bin {k_622if} "
            "is not one-dimensional."
        )

    if len(arr_622if) == 0:
        raise RuntimeError(
            f"Reference bin {k_622if} "
            "is empty."
        )

    if not np.isfinite(arr_622if).all():
        raise RuntimeError(
            f"Reference bin {k_622if} "
            "contains non-finite values."
        )


# ------------------------------------------------------------
# 5. Freeze existing development/validation candidate arrays
# ------------------------------------------------------------

dev_uncertainty_622if = np.asarray(
    candidate_arrays_dev_622h[
        FROZEN_UNCERTAINTY_CANDIDATE_622IF
    ],
    dtype=np.float64
)

val_uncertainty_622if = np.asarray(
    candidate_arrays_val_622h[
        FROZEN_UNCERTAINTY_CANDIDATE_622IF
    ],
    dtype=np.float64
)

if dev_uncertainty_622if.shape != (14241,):
    raise RuntimeError(
        "Unexpected development uncertainty shape: "
        f"{dev_uncertainty_622if.shape}"
    )

if val_uncertainty_622if.shape != (3490,):
    raise RuntimeError(
        "Unexpected validation uncertainty shape: "
        f"{val_uncertainty_622if.shape}"
    )

if not np.isfinite(
    dev_uncertainty_622if
).all():
    raise RuntimeError(
        "Development uncertainty contains "
        "non-finite values."
    )

if not np.isfinite(
    val_uncertainty_622if
).all():
    raise RuntimeError(
        "Validation uncertainty contains "
        "non-finite values."
    )


# ------------------------------------------------------------
# 6. Exact reusable transformation
# ------------------------------------------------------------

def apply_frozen_uncertainty_622if(
    deterministic_prediction,
    mc_dropout_std,
    edges=edges_622if,
    references=references_622if,
):
    """
    Apply the frozen det_pred_pct_b5 uncertainty recipe.

    Phase:
        deterministic predicted RUL

    Phase assignment:
        np.digitize(..., right=True)

    Within each phase:
        empirical mid-rank percentile of MC-dropout std

        score =
        (count(reference < x) +
         count(reference <= x))
        / (2 * N)
    """

    pred_622if = np.asarray(
        deterministic_prediction,
        dtype=np.float64
    )

    std_622if = np.asarray(
        mc_dropout_std,
        dtype=np.float64
    )

    if pred_622if.shape != std_622if.shape:
        raise ValueError(
            "Prediction and uncertainty arrays "
            "must have identical shapes."
        )

    if not np.isfinite(pred_622if).all():
        raise ValueError(
            "Prediction array contains "
            "non-finite values."
        )

    if not np.isfinite(std_622if).all():
        raise ValueError(
            "MC-dropout std array contains "
            "non-finite values."
        )

    phase_ids_622if = np.digitize(
        pred_622if,
        edges,
        right=True
    )

    score_622if = np.empty(
        len(std_622if),
        dtype=np.float64
    )

    for phase_622if in range(5):

        mask_622if = (
            phase_ids_622if
            == phase_622if
        )

        if not np.any(mask_622if):
            continue

        ref_622if = np.sort(
            np.asarray(
                references[phase_622if],
                dtype=np.float64
            )
        )

        x_622if = std_622if[
            mask_622if
        ]

        n_ref_622if = len(
            ref_622if
        )

        count_lt_622if = np.searchsorted(
            ref_622if,
            x_622if,
            side="left"
        )

        count_le_622if = np.searchsorted(
            ref_622if,
            x_622if,
            side="right"
        )

        score_622if[
            mask_622if
        ] = (
            count_lt_622if
            + count_le_622if
        ) / (
            2.0
            * n_ref_622if
        )

    return score_622if


# ------------------------------------------------------------
# 7. Validation reconstruction check
# ------------------------------------------------------------

if (
    "clean_predictions"
    not in globals()
    or
    "mc_std"
    not in globals()
):
    raise RuntimeError(
        "clean_predictions and mc_std "
        "are required for the final "
        "validation reconstruction check."
    )

val_recomputed_622if = (
    apply_frozen_uncertainty_622if(
        clean_predictions,
        mc_std
    )
)

val_max_error_622if = float(
    np.max(
        np.abs(
            val_recomputed_622if
            - val_uncertainty_622if
        )
    )
)

val_mean_error_622if = float(
    np.mean(
        np.abs(
            val_recomputed_622if
            - val_uncertainty_622if
        )
    )
)

val_exact_622if = bool(
    np.allclose(
        val_recomputed_622if,
        val_uncertainty_622if,
        rtol=0.0,
        atol=1e-12
    )
)

print(
    "\n--- CLEAN validation reconstruction ---"
)
print(
    "Maximum absolute error :",
    val_max_error_622if
)
print(
    "Mean absolute error    :",
    val_mean_error_622if
)
print(
    "Exact allclose         :",
    val_exact_622if
)

if not val_exact_622if:
    raise RuntimeError(
        "Frozen recipe failed to reproduce "
        "the selected validation candidate."
    )


# ------------------------------------------------------------
# 8. Summary statistics
# ------------------------------------------------------------

summary_622if = pd.DataFrame(
    [
        {
            "split": "development",
            "windows": len(
                dev_uncertainty_622if
            ),
            "minimum": np.min(
                dev_uncertainty_622if
            ),
            "median": np.median(
                dev_uncertainty_622if
            ),
            "mean": np.mean(
                dev_uncertainty_622if
            ),
            "p90": np.quantile(
                dev_uncertainty_622if,
                0.90
            ),
            "p95": np.quantile(
                dev_uncertainty_622if,
                0.95
            ),
            "maximum": np.max(
                dev_uncertainty_622if
            ),
        },
        {
            "split": "clean_validation",
            "windows": len(
                val_uncertainty_622if
            ),
            "minimum": np.min(
                val_uncertainty_622if
            ),
            "median": np.median(
                val_uncertainty_622if
            ),
            "mean": np.mean(
                val_uncertainty_622if
            ),
            "p90": np.quantile(
                val_uncertainty_622if,
                0.90
            ),
            "p95": np.quantile(
                val_uncertainty_622if,
                0.95
            ),
            "maximum": np.max(
                val_uncertainty_622if
            ),
        },
    ]
)

print("\n--- Frozen signal summary ---")
display(summary_622if)


# ------------------------------------------------------------
# 9. Output directory
# ------------------------------------------------------------

PROJECT_ROOT_622IF = Path(
    r"C:\RUL_Reliability_Research"
)

OUTPUT_DIR_622IF = (
    PROJECT_ROOT_622IF
    / "06_results"
    / "step6_reliability_signals"
)

OUTPUT_DIR_622IF.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# 10. Save reference distribution as NPZ
# ------------------------------------------------------------

reference_path_622if = (
    OUTPUT_DIR_622IF
    / (
        "fd001_uncertainty_"
        "det_pred_pct_b5_reference.npz"
    )
)

np.savez_compressed(
    reference_path_622if,
    edges=edges_622if,
    reference_bin_0=np.asarray(
        references_622if[0],
        dtype=np.float64
    ),
    reference_bin_1=np.asarray(
        references_622if[1],
        dtype=np.float64
    ),
    reference_bin_2=np.asarray(
        references_622if[2],
        dtype=np.float64
    ),
    reference_bin_3=np.asarray(
        references_622if[3],
        dtype=np.float64
    ),
    reference_bin_4=np.asarray(
        references_622if[4],
        dtype=np.float64
    ),
)


# ------------------------------------------------------------
# 11. Save summary CSV
# ------------------------------------------------------------

summary_path_622if = (
    OUTPUT_DIR_622IF
    / (
        "fd001_uncertainty_"
        "det_pred_pct_b5_summary.csv"
    )
)

summary_622if.to_csv(
    summary_path_622if,
    index=False
)


# ------------------------------------------------------------
# 12. SHA helper
# ------------------------------------------------------------

def sha256_file_622if(path):
    h_622if = hashlib.sha256()

    with open(path, "rb") as f_622if:
        for block_622if in iter(
            lambda: f_622if.read(
                1024 * 1024
            ),
            b""
        ):
            h_622if.update(
                block_622if
            )

    return h_622if.hexdigest()


reference_sha_622if = (
    sha256_file_622if(
        reference_path_622if
    )
)

summary_sha_622if = (
    sha256_file_622if(
        summary_path_622if
    )
)


# ------------------------------------------------------------
# 13. Definition JSON
# ------------------------------------------------------------

definition_622if = {
    "dataset": "FD001",

    "signal_role":
        "model_side_predictive_uncertainty",

    "selected_candidate":
        FROZEN_UNCERTAINTY_CANDIDATE_622IF,

    "phase_source":
        "deterministic_RUL_prediction",

    "requested_phase_bins":
        requested_bins_622if,

    "actual_phase_bins":
        actual_bins_622if,

    "phase_edges":
        edges_622if.tolist(),

    "phase_assignment":
        "numpy.digitize(prediction, edges, right=True)",

    "base_uncertainty":
        "MC_dropout_prediction_standard_deviation",

    "mc_standard_deviation_rule":
        "population_standard_deviation_ddof_0",

    "within_phase_transformation":
        (
            "empirical_midrank_percentile: "
            "(count(ref < x) + "
            "count(ref <= x)) / (2*N)"
        ),

    "selection_source":
        "development_only",

    "clean_validation_role":
        "sanity_and_reproducibility_check_only",

    "validation_reconstruction_max_abs_error":
        val_max_error_622if,

    "validation_reconstruction_mean_abs_error":
        val_mean_error_622if,

    "validation_reconstruction_exact_allclose":
        val_exact_622if,

    "official_test_used":
        False,

    "uncertainty_threshold_selected":
        False,

    "abstention_rule_created":
        False,

    "quality_channels_frozen_previously": {
        "pca":
            "pca_reconstruction_quality",
        "temporal":
            "max_sensor",
    },

    "reference_file":
        reference_path_622if.name,

    "reference_file_sha256":
        reference_sha_622if,

    "summary_file":
        summary_path_622if.name,

    "summary_file_sha256":
        summary_sha_622if,
}


definition_path_622if = (
    OUTPUT_DIR_622IF
    / (
        "fd001_uncertainty_"
        "signal_definition.json"
    )
)

with open(
    definition_path_622if,
    "w",
    encoding="utf-8"
) as f_622if:

    json.dump(
        definition_622if,
        f_622if,
        indent=2
    )


definition_sha_622if = (
    sha256_file_622if(
        definition_path_622if
    )
)


# ------------------------------------------------------------
# 14. Freeze convenient in-memory objects
# ------------------------------------------------------------

FROZEN_UNCERTAINTY_NAME_622IF = (
    FROZEN_UNCERTAINTY_CANDIDATE_622IF
)

FROZEN_UNCERTAINTY_REFERENCE_622IF = {
    "edges": edges_622if.copy(),

    "references": {
        k_622if: np.asarray(
            references_622if[k_622if],
            dtype=np.float64
        ).copy()
        for k_622if in range(5)
    },

    "phase_source":
        phase_source_622if,

    "requested_bins":
        requested_bins_622if,

    "actual_bins":
        actual_bins_622if,

    "right":
        True,
}

FROZEN_DEV_UNCERTAINTY_622IF = (
    dev_uncertainty_622if.copy()
)

FROZEN_VAL_UNCERTAINTY_622IF = (
    val_uncertainty_622if.copy()
)


# ------------------------------------------------------------
# 15. Final compact integrity summary
# ------------------------------------------------------------

print(
    "\n======================================"
)
print(
    "STEP 6.22I-F INTEGRITY SUMMARY"
)
print(
    "======================================"
)

print(
    "Frozen uncertainty signal     :",
    FROZEN_UNCERTAINTY_NAME_622IF
)

print(
    "Phase source                  :",
    "deterministic RUL prediction"
)

print(
    "Phase bins                    :",
    actual_bins_622if
)

print(
    "Boundary convention           :",
    "np.digitize(..., right=True)"
)

print(
    "Base uncertainty              :",
    "MC-dropout std (ddof=0)"
)

print(
    "Within-phase transformation   :",
    "empirical mid-rank percentile"
)

print(
    "Development windows           :",
    len(FROZEN_DEV_UNCERTAINTY_622IF)
)

print(
    "Validation windows            :",
    len(FROZEN_VAL_UNCERTAINTY_622IF)
)

print(
    "Validation reconstruction err :",
    val_max_error_622if
)

print(
    "Validation exact reproduction :",
    val_exact_622if
)

print(
    "Official test used            :",
    False
)

print(
    "Uncertainty threshold selected:",
    False
)

print(
    "Abstention rule created       :",
    False
)

print("\nFrozen files:")
print("1.", reference_path_622if)
print("2.", summary_path_622if)
print("3.", definition_path_622if)

print(
    "\nReference SHA-256 :",
    reference_sha_622if
)

print(
    "Definition SHA-256:",
    definition_sha_622if
)

print(
    "\nStep 6.22I-F completed successfully."
)

In [ ]:
# ============================================================
# Step 6.23A
# Reliability-channel alignment pre-check
#
# Purpose:
#   Put PCA quality, temporal quality and predictive
#   uncertainty on comparable [0,1] "higher = worse" scales.
#
# IMPORTANT:
# - Development reference only
# - Clean validation is sanity-check only
# - No combined reliability score selected
# - No threshold selected
# - No abstention rule created
# - Official test NOT used
# ============================================================

import numpy as np
import pandas as pd

print(
    "\n=== Step 6.23A — "
    "Reliability-channel alignment pre-check ==="
)


# ------------------------------------------------------------
# 1. Required objects
# ------------------------------------------------------------

required_623a = [
    "dev_pca_max_rmse",
    "val_pca_max_rmse",
    "dev_candidate_scores_620ha",
    "val_candidate_scores_620ha",
    "FROZEN_DEV_UNCERTAINTY_622IF",
    "FROZEN_VAL_UNCERTAINTY_622IF",
]

print("\n--- Required objects ---")

for name_623a in required_623a:
    print(
        f"{name_623a:<36}:",
        name_623a in globals()
    )

missing_623a = [
    x
    for x in required_623a
    if x not in globals()
]

if missing_623a:
    raise RuntimeError(
        "Missing required object(s): "
        + ", ".join(missing_623a)
    )


# ------------------------------------------------------------
# 2. Recover exact three channels
# ------------------------------------------------------------

dev_pca_raw_623a = np.asarray(
    dev_pca_max_rmse,
    dtype=np.float64
)

val_pca_raw_623a = np.asarray(
    val_pca_max_rmse,
    dtype=np.float64
)


dev_temporal_raw_623a = np.asarray(
    dev_candidate_scores_620ha[
        "max_sensor"
    ],
    dtype=np.float64
)

val_temporal_raw_623a = np.asarray(
    val_candidate_scores_620ha[
        "max_sensor"
    ],
    dtype=np.float64
)


dev_uncertainty_623a = np.asarray(
    FROZEN_DEV_UNCERTAINTY_622IF,
    dtype=np.float64
)

val_uncertainty_623a = np.asarray(
    FROZEN_VAL_UNCERTAINTY_622IF,
    dtype=np.float64
)


# ------------------------------------------------------------
# 3. Shape and numerical checks
# ------------------------------------------------------------

expected_dev_623a = 14241
expected_val_623a = 3490

channel_arrays_623a = {
    "dev_pca": dev_pca_raw_623a,
    "val_pca": val_pca_raw_623a,

    "dev_temporal": dev_temporal_raw_623a,
    "val_temporal": val_temporal_raw_623a,

    "dev_uncertainty": dev_uncertainty_623a,
    "val_uncertainty": val_uncertainty_623a,
}

for name_623a, arr_623a in channel_arrays_623a.items():

    if arr_623a.ndim != 1:
        raise RuntimeError(
            f"{name_623a} is not one-dimensional: "
            f"{arr_623a.shape}"
        )

    if not np.isfinite(arr_623a).all():
        raise RuntimeError(
            f"{name_623a} contains non-finite values."
        )


if len(dev_pca_raw_623a) != expected_dev_623a:
    raise RuntimeError(
        "Unexpected development PCA length."
    )

if len(dev_temporal_raw_623a) != expected_dev_623a:
    raise RuntimeError(
        "Unexpected development temporal length."
    )

if len(dev_uncertainty_623a) != expected_dev_623a:
    raise RuntimeError(
        "Unexpected development uncertainty length."
    )


if len(val_pca_raw_623a) != expected_val_623a:
    raise RuntimeError(
        "Unexpected validation PCA length."
    )

if len(val_temporal_raw_623a) != expected_val_623a:
    raise RuntimeError(
        "Unexpected validation temporal length."
    )

if len(val_uncertainty_623a) != expected_val_623a:
    raise RuntimeError(
        "Unexpected validation uncertainty length."
    )


print("\nAll channel shapes and numerical values are valid.")


# ------------------------------------------------------------
# 4. Development-reference empirical percentile helper
#
# Mid-rank empirical percentile:
#
#   (count(ref < x) + count(ref <= x)) / (2*N)
#
# This handles ties correctly.
# ------------------------------------------------------------

def empirical_midrank_percentile_623a(
    values,
    reference
):

    values_623a = np.asarray(
        values,
        dtype=np.float64
    )

    reference_623a = np.sort(
        np.asarray(
            reference,
            dtype=np.float64
        )
    )

    n_623a = len(reference_623a)

    if n_623a == 0:
        raise ValueError(
            "Reference array is empty."
        )

    left_623a = np.searchsorted(
        reference_623a,
        values_623a,
        side="left"
    )

    right_623a = np.searchsorted(
        reference_623a,
        values_623a,
        side="right"
    )

    score_623a = (
        left_623a + right_623a
    ) / (
        2.0 * n_623a
    )

    return score_623a


# ------------------------------------------------------------
# 5. Convert quality channels to comparable [0,1] scores
#
# Higher raw PCA reconstruction error = worse
# Higher temporal score = worse
#
# Therefore no sign reversal is required.
# ------------------------------------------------------------

dev_pca_pct_623a = (
    empirical_midrank_percentile_623a(
        dev_pca_raw_623a,
        dev_pca_raw_623a
    )
)

val_pca_pct_623a = (
    empirical_midrank_percentile_623a(
        val_pca_raw_623a,
        dev_pca_raw_623a
    )
)


dev_temporal_pct_623a = (
    empirical_midrank_percentile_623a(
        dev_temporal_raw_623a,
        dev_temporal_raw_623a
    )
)

val_temporal_pct_623a = (
    empirical_midrank_percentile_623a(
        val_temporal_raw_623a,
        dev_temporal_raw_623a
    )
)


# Uncertainty is already development-reference percentile based.
dev_uncertainty_pct_623a = (
    dev_uncertainty_623a.copy()
)

val_uncertainty_pct_623a = (
    val_uncertainty_623a.copy()
)


# ------------------------------------------------------------
# 6. Range check
# ------------------------------------------------------------

aligned_arrays_623a = {
    "dev_pca_quality":
        dev_pca_pct_623a,

    "dev_temporal_quality":
        dev_temporal_pct_623a,

    "dev_predictive_uncertainty":
        dev_uncertainty_pct_623a,

    "val_pca_quality":
        val_pca_pct_623a,

    "val_temporal_quality":
        val_temporal_pct_623a,

    "val_predictive_uncertainty":
        val_uncertainty_pct_623a,
}


for name_623a, arr_623a in aligned_arrays_623a.items():

    if (
        np.min(arr_623a) < 0.0
        or
        np.max(arr_623a) > 1.0
    ):
        raise RuntimeError(
            f"{name_623a} lies outside [0,1]."
        )


# ------------------------------------------------------------
# 7. Compact channel summaries
# ------------------------------------------------------------

rows_623a = []

for split_623a, arrays_623a in [
    (
        "development",
        {
            "pca_quality":
                dev_pca_pct_623a,

            "temporal_quality":
                dev_temporal_pct_623a,

            "predictive_uncertainty":
                dev_uncertainty_pct_623a,
        }
    ),
    (
        "clean_validation",
        {
            "pca_quality":
                val_pca_pct_623a,

            "temporal_quality":
                val_temporal_pct_623a,

            "predictive_uncertainty":
                val_uncertainty_pct_623a,
        }
    ),
]:

    for channel_623a, arr_623a in arrays_623a.items():

        rows_623a.append(
            {
                "split": split_623a,
                "channel": channel_623a,
                "windows": len(arr_623a),
                "minimum": float(
                    np.min(arr_623a)
                ),
                "median": float(
                    np.median(arr_623a)
                ),
                "mean": float(
                    np.mean(arr_623a)
                ),
                "p90": float(
                    np.quantile(
                        arr_623a,
                        0.90
                    )
                ),
                "p95": float(
                    np.quantile(
                        arr_623a,
                        0.95
                    )
                ),
                "maximum": float(
                    np.max(arr_623a)
                ),
            }
        )


channel_summary_623a = pd.DataFrame(
    rows_623a
)

print("\n--- Aligned channel summary ---")

display(
    channel_summary_623a.round(6)
)


# ------------------------------------------------------------
# 8. Complementarity / redundancy diagnostic
# ------------------------------------------------------------

dev_channel_df_623a = pd.DataFrame(
    {
        "pca_quality":
            dev_pca_pct_623a,

        "temporal_quality":
            dev_temporal_pct_623a,

        "predictive_uncertainty":
            dev_uncertainty_pct_623a,
    }
)

val_channel_df_623a = pd.DataFrame(
    {
        "pca_quality":
            val_pca_pct_623a,

        "temporal_quality":
            val_temporal_pct_623a,

        "predictive_uncertainty":
            val_uncertainty_pct_623a,
    }
)


print(
    "\n--- DEVELOPMENT Spearman correlations ---"
)

display(
    dev_channel_df_623a
    .corr(method="spearman")
    .round(4)
)


print(
    "\n--- CLEAN VALIDATION Spearman correlations "
    "(sanity only) ---"
)

display(
    val_channel_df_623a
    .corr(method="spearman")
    .round(4)
)


# ------------------------------------------------------------
# 9. Convenient in-memory objects for next step
# ------------------------------------------------------------

RELIABILITY_DEV_CHANNELS_623A = (
    dev_channel_df_623a.copy()
)

RELIABILITY_VAL_CHANNELS_623A = (
    val_channel_df_623a.copy()
)


# ------------------------------------------------------------
# 10. Final compact status
# ------------------------------------------------------------

print(
    "\n========================================"
)
print(
    "STEP 6.23A COMPACT SUMMARY"
)
print(
    "========================================"
)

print(
    "Development windows       :",
    len(RELIABILITY_DEV_CHANNELS_623A)
)

print(
    "Validation windows        :",
    len(RELIABILITY_VAL_CHANNELS_623A)
)

print(
    "Reliability channels      :",
    list(
        RELIABILITY_DEV_CHANNELS_623A.columns
    )
)

print(
    "Common orientation        : higher = worse"
)

print(
    "Common range              : [0,1]"
)

print(
    "Scaling reference         : development only"
)

print(
    "Combined score selected   :",
    False
)

print(
    "Reliability threshold set :",
    False
)

print(
    "Abstention rule created   :",
    False
)

print(
    "Official test used        :",
    False
)

print(
    "\nStep 6.23A completed successfully."
)

In [ ]:
# ============================================================
# Step 6.23B
# Construct and clean-calibrate reliability-score candidates
#
# IMPORTANT:
# - Development data defines all thresholds
# - Clean validation = sanity check only
# - No final reliability score selected
# - No abstention rule created
# - Official test NOT used
# ============================================================

import numpy as np
import pandas as pd

print(
    "\n=== Step 6.23B — "
    "Reliability-score candidate construction ==="
)


# ------------------------------------------------------------
# 1. Required objects
# ------------------------------------------------------------

required_623b = [
    "RELIABILITY_DEV_CHANNELS_623A",
    "RELIABILITY_VAL_CHANNELS_623A",
]

print("\n--- Required objects ---")

for name_623b in required_623b:
    print(
        f"{name_623b:<38}:",
        name_623b in globals()
    )

missing_623b = [
    name_623b
    for name_623b in required_623b
    if name_623b not in globals()
]

if missing_623b:
    raise RuntimeError(
        "Missing required object(s): "
        + ", ".join(missing_623b)
    )


# ------------------------------------------------------------
# 2. Recover aligned channels
# ------------------------------------------------------------

dev_channels_623b = (
    RELIABILITY_DEV_CHANNELS_623A.copy()
)

val_channels_623b = (
    RELIABILITY_VAL_CHANNELS_623A.copy()
)


expected_columns_623b = [
    "pca_quality",
    "temporal_quality",
    "predictive_uncertainty",
]

if list(dev_channels_623b.columns) != expected_columns_623b:
    raise RuntimeError(
        "Unexpected development channel columns."
    )

if list(val_channels_623b.columns) != expected_columns_623b:
    raise RuntimeError(
        "Unexpected validation channel columns."
    )


print(
    "\nDevelopment shape:",
    dev_channels_623b.shape
)

print(
    "Validation shape :",
    val_channels_623b.shape
)


# ------------------------------------------------------------
# 3. Candidate-construction helper
# ------------------------------------------------------------

def build_reliability_candidates_623b(df_623b):

    pca_623b = np.asarray(
        df_623b["pca_quality"],
        dtype=np.float64
    )

    temporal_623b = np.asarray(
        df_623b["temporal_quality"],
        dtype=np.float64
    )

    uncertainty_623b = np.asarray(
        df_623b["predictive_uncertainty"],
        dtype=np.float64
    )

    stacked_623b = np.column_stack(
        [
            pca_623b,
            temporal_623b,
            uncertainty_623b,
        ]
    )

    sensor_stacked_623b = np.column_stack(
        [
            pca_623b,
            temporal_623b,
        ]
    )

    sorted_623b = np.sort(
        stacked_623b,
        axis=1
    )

    candidates_623b = pd.DataFrame(
        {
            # Individual-channel ablations
            "pca_only":
                pca_623b,

            "temporal_only":
                temporal_623b,

            "uncertainty_only":
                uncertainty_623b,

            # Sensor-quality-only combination
            "sensor_quality_max":
                np.max(
                    sensor_stacked_623b,
                    axis=1
                ),

            # Three-channel combinations
            "max3":
                np.max(
                    stacked_623b,
                    axis=1
                ),

            "mean3":
                np.mean(
                    stacked_623b,
                    axis=1
                ),

            "rms3":
                np.sqrt(
                    np.mean(
                        stacked_623b ** 2,
                        axis=1
                    )
                ),

            # Mean of the two strongest channels
            "top2_mean":
                np.mean(
                    sorted_623b[:, -2:],
                    axis=1
                ),
        }
    )

    return candidates_623b


# ------------------------------------------------------------
# 4. Construct development and validation candidates
# ------------------------------------------------------------

DEV_RELIABILITY_CANDIDATES_623B = (
    build_reliability_candidates_623b(
        dev_channels_623b
    )
)

VAL_RELIABILITY_CANDIDATES_623B = (
    build_reliability_candidates_623b(
        val_channels_623b
    )
)


print(
    "\nCandidate names:",
    list(
        DEV_RELIABILITY_CANDIDATES_623B.columns
    )
)


# ------------------------------------------------------------
# 5. Numerical/range integrity
# ------------------------------------------------------------

for split_name_623b, df_623b in [
    (
        "development",
        DEV_RELIABILITY_CANDIDATES_623B
    ),
    (
        "validation",
        VAL_RELIABILITY_CANDIDATES_623B
    ),
]:

    arr_623b = df_623b.to_numpy(
        dtype=np.float64
    )

    if not np.isfinite(arr_623b).all():
        raise RuntimeError(
            f"{split_name_623b} candidates "
            "contain non-finite values."
        )

    if (
        np.min(arr_623b) < 0.0
        or
        np.max(arr_623b) > 1.0
    ):
        raise RuntimeError(
            f"{split_name_623b} candidate values "
            "fall outside [0,1]."
        )


print(
    "\nAll candidate values are finite "
    "and lie in [0,1]."
)


# ------------------------------------------------------------
# 6. DEVELOPMENT-only p95 thresholds
# ------------------------------------------------------------

RELIABILITY_THRESHOLDS_623B = {}

for candidate_623b in (
    DEV_RELIABILITY_CANDIDATES_623B.columns
):

    threshold_623b = float(
        np.quantile(
            DEV_RELIABILITY_CANDIDATES_623B[
                candidate_623b
            ],
            0.95
        )
    )

    RELIABILITY_THRESHOLDS_623B[
        candidate_623b
    ] = threshold_623b


# ------------------------------------------------------------
# 7. Clean development/validation calibration summary
# ------------------------------------------------------------

summary_rows_623b = []

for candidate_623b in (
    DEV_RELIABILITY_CANDIDATES_623B.columns
):

    dev_score_623b = np.asarray(
        DEV_RELIABILITY_CANDIDATES_623B[
            candidate_623b
        ],
        dtype=np.float64
    )

    val_score_623b = np.asarray(
        VAL_RELIABILITY_CANDIDATES_623B[
            candidate_623b
        ],
        dtype=np.float64
    )

    threshold_623b = (
        RELIABILITY_THRESHOLDS_623B[
            candidate_623b
        ]
    )

    dev_event_pct_623b = (
        100.0
        * np.mean(
            dev_score_623b
            > threshold_623b
        )
    )

    val_event_pct_623b = (
        100.0
        * np.mean(
            val_score_623b
            > threshold_623b
        )
    )

    summary_rows_623b.append(
        {
            "candidate":
                candidate_623b,

            "development_p95_threshold":
                threshold_623b,

            "development_event_pct":
                dev_event_pct_623b,

            "clean_validation_event_pct":
                val_event_pct_623b,

            "clean_validation_mean":
                float(
                    np.mean(
                        val_score_623b
                    )
                ),

            "clean_validation_median":
                float(
                    np.median(
                        val_score_623b
                    )
                ),

            "clean_validation_p95":
                float(
                    np.quantile(
                        val_score_623b,
                        0.95
                    )
                ),
        }
    )


reliability_clean_summary_623b = (
    pd.DataFrame(
        summary_rows_623b
    )
)


print(
    "\n--- CLEAN calibration summary ---"
)

display(
    reliability_clean_summary_623b
    .round(6)
)


# ------------------------------------------------------------
# 8. Development candidate correlations
#
# Useful only for understanding how the aggregation rules
# relate to one another.
# ------------------------------------------------------------

print(
    "\n--- DEVELOPMENT candidate Spearman "
    "correlation ---"
)

display(
    DEV_RELIABILITY_CANDIDATES_623B
    .corr(method="spearman")
    .round(3)
)


# ------------------------------------------------------------
# 9. Compact formulas / interpretation
# ------------------------------------------------------------

candidate_definition_623b = pd.DataFrame(
    [
        {
            "candidate": "pca_only",
            "definition":
                "PCA quality only",
        },
        {
            "candidate": "temporal_only",
            "definition":
                "Temporal quality only",
        },
        {
            "candidate": "uncertainty_only",
            "definition":
                "Predictive uncertainty only",
        },
        {
            "candidate": "sensor_quality_max",
            "definition":
                "max(PCA, temporal)",
        },
        {
            "candidate": "max3",
            "definition":
                "max(PCA, temporal, uncertainty)",
        },
        {
            "candidate": "mean3",
            "definition":
                "equal-weight arithmetic mean",
        },
        {
            "candidate": "rms3",
            "definition":
                "equal-weight RMS",
        },
        {
            "candidate": "top2_mean",
            "definition":
                "mean of two largest channels",
        },
    ]
)


print(
    "\n--- Candidate definitions ---"
)

display(
    candidate_definition_623b
)


# ------------------------------------------------------------
# 10. Integrity summary
# ------------------------------------------------------------

print(
    "\n=========================================="
)

print(
    "STEP 6.23B COMPACT SUMMARY"
)

print(
    "=========================================="
)

print(
    "Development windows         :",
    len(
        DEV_RELIABILITY_CANDIDATES_623B
    )
)

print(
    "Validation windows          :",
    len(
        VAL_RELIABILITY_CANDIDATES_623B
    )
)

print(
    "Candidate scores evaluated  :",
    DEV_RELIABILITY_CANDIDATES_623B.shape[1]
)

print(
    "Threshold source            : "
    "CLEAN DEVELOPMENT only"
)

print(
    "Threshold quantile          : p95"
)

print(
    "Final score selected        :",
    False
)

print(
    "Reliability rule frozen     :",
    False
)

print(
    "Abstention rule created     :",
    False
)

print(
    "Official test used          :",
    False
)

print(
    "\nStep 6.23B completed successfully."
)

In [ ]:
# ============================================================
# Step 6.23C-A
# Frozen reliability-channel replay readiness check
#
# Inspection only.
# No corruption generated.
# No reliability candidate selected.
# No threshold changed.
# No abstention rule created.
# Official test NOT used.
# ============================================================

import numpy as np
import pandas as pd
import inspect

print(
    "\n=== Step 6.23C-A — "
    "Frozen reliability-channel replay readiness ==="
)


# ------------------------------------------------------------
# 1. Core objects that should support corruption replay
# ------------------------------------------------------------

core_objects_623ca = [
    # Frozen corruption infrastructure
    "scenario_manifests_620gc",
    "apply_frozen_corruption_620gc",
    "val_raw_base_620gc",

    # Clean validation sequence pathway
    "X_val_clean",
    "scaler",
    "PRIMARY_INPUT_COLUMNS",
    "RETAINED_SENSOR_COLUMNS",
    "SENSOR_INPUT_INDICES",

    # PCA quality pathway
    "sensor_pca",
    "calculate_pca_window_quality",

    # Temporal quality pathway
    "temporal_metrics_620ha",
    "sensor_strength_620ha",
    "aggregate_candidates_620ha",

    # Predictive model / MC pathway
    "frozen_gru",
    "mc_dropout_predictions",

    # Frozen uncertainty recipe
    "phase_references_622h",

    # Reliability channels already constructed
    "RELIABILITY_DEV_CHANNELS_623A",
    "RELIABILITY_VAL_CHANNELS_623A",

    # Current candidate objects
    "DEV_RELIABILITY_CANDIDATES_623B",
    "VAL_RELIABILITY_CANDIDATES_623B",
    "RELIABILITY_THRESHOLDS_623B",
]


print("\n--- Core object availability ---")

missing_623ca = []

for name_623ca in core_objects_623ca:

    available_623ca = (
        name_623ca in globals()
    )

    print(
        f"{name_623ca:<40}: "
        f"{available_623ca}"
    )

    if not available_623ca:
        missing_623ca.append(
            name_623ca
        )


# ------------------------------------------------------------
# 2. Important callable signatures
# ------------------------------------------------------------

print(
    "\n--- Replay helper signatures ---"
)

for name_623ca in [
    "apply_frozen_corruption_620gc",
    "calculate_pca_window_quality",
    "temporal_metrics_620ha",
    "sensor_strength_620ha",
    "aggregate_candidates_620ha",
    "mc_dropout_predictions",
]:

    if name_623ca not in globals():
        continue

    obj_623ca = globals()[name_623ca]

    if callable(obj_623ca):

        try:
            sig_623ca = inspect.signature(
                obj_623ca
            )

        except Exception:
            sig_623ca = "<signature unavailable>"

        print(
            f"{name_623ca}: "
            f"{sig_623ca}"
        )


# ------------------------------------------------------------
# 3. Exact selected uncertainty phase reference
# ------------------------------------------------------------

print(
    "\n--- Frozen uncertainty reference ---"
)

selected_uncertainty_623ca = (
    "det_pred_pct_b5"
)

if "phase_references_622h" in globals():

    print(
        "Selected uncertainty candidate :",
        selected_uncertainty_623ca
    )

    print(
        "Candidate present              :",
        selected_uncertainty_623ca
        in phase_references_622h
    )

    if (
        selected_uncertainty_623ca
        in phase_references_622h
    ):

        ref_623ca = (
            phase_references_622h[
                selected_uncertainty_623ca
            ]
        )

        print(
            "Reference keys                 :",
            list(ref_623ca.keys())
        )

        print(
            "Phase source                   :",
            ref_623ca.get(
                "phase_source",
                None
            )
        )

        print(
            "Requested bins                 :",
            ref_623ca.get(
                "requested_bins",
                None
            )
        )

        print(
            "Actual bins                    :",
            ref_623ca.get(
                "actual_bins",
                None
            )
        )


# ------------------------------------------------------------
# 4. Search namespace for Step-6.23A transformation references
#
# We need to know whether the exact percentile/rank mapping
# helper/reference was retained in memory.
# ------------------------------------------------------------

snapshot_623ca = list(
    globals().items()
)

search_tokens_623ca = (
    "623a",
    "percentile",
    "midrank",
    "ecdf",
    "reliability",
    "reference",
)

reference_candidates_623ca = []

for name_623ca, obj_623ca in snapshot_623ca:

    lower_623ca = (
        str(name_623ca).lower()
    )

    if not any(
        token_623ca in lower_623ca
        for token_623ca
        in search_tokens_623ca
    ):
        continue

    # Avoid reporting the large final channel DataFrames again
    if name_623ca in [
        "RELIABILITY_DEV_CHANNELS_623A",
        "RELIABILITY_VAL_CHANNELS_623A",
        "DEV_RELIABILITY_CANDIDATES_623B",
        "VAL_RELIABILITY_CANDIDATES_623B",
    ]:
        continue

    shape_623ca = getattr(
        obj_623ca,
        "shape",
        None
    )

    reference_candidates_623ca.append(
        (
            name_623ca,
            type(obj_623ca).__name__,
            shape_623ca,
        )
    )


# Keep output compact
reference_candidates_623ca = (
    reference_candidates_623ca[:30]
)

print(
    "\n--- Potential normalization/reference "
    "objects ---"
)

if len(reference_candidates_623ca) == 0:

    print(
        "No matching helper/reference "
        "objects found."
    )

else:

    for (
        name_623ca,
        type_623ca,
        shape_623ca,
    ) in reference_candidates_623ca:

        print(
            f"{name_623ca:<42} "
            f"type={type_623ca:<14} "
            f"shape={shape_623ca}"
        )


# ------------------------------------------------------------
# 5. Compact readiness summary
# ------------------------------------------------------------

critical_623ca = [
    "scenario_manifests_620gc",
    "apply_frozen_corruption_620gc",
    "val_raw_base_620gc",
    "X_val_clean",
    "scaler",
    "sensor_pca",
    "calculate_pca_window_quality",
    "temporal_metrics_620ha",
    "sensor_strength_620ha",
    "aggregate_candidates_620ha",
    "frozen_gru",
    "mc_dropout_predictions",
    "phase_references_622h",
]

critical_ready_623ca = all(
    name_623ca in globals()
    for name_623ca
    in critical_623ca
)


print(
    "\n=========================================="
)

print(
    "STEP 6.23C-A COMPACT SUMMARY"
)

print(
    "=========================================="
)

print(
    "Critical replay objects ready :",
    critical_ready_623ca
)

print(
    "Missing core objects          :",
    len(missing_623ca)
)

print(
    "Frozen uncertainty candidate  :",
    selected_uncertainty_623ca
)

print(
    "Corruption generated          :",
    False
)

print(
    "Final reliability selected    :",
    False
)

print(
    "Threshold changed             :",
    False
)

print(
    "Abstention rule created       :",
    False
)

print(
    "Official test used            :",
    False
)

print(
    "\nStep 6.23C-A completed."
)

In [ ]:
# ============================================================
# Step 6.23C-B0 — Exact frozen-corruption replay interface check
# Inspection only
# ============================================================

import inspect

print("=== STEP 6.23C-B0 — REPLAY INTERFACE CHECK ===\n")

print("--- apply_frozen_corruption_620gc ---")
print("Callable :", callable(apply_frozen_corruption_620gc))
print("Signature:", inspect.signature(apply_frozen_corruption_620gc))

print("\n--- scenario_manifests_620gc ---")
print("Type:", type(scenario_manifests_620gc).__name__)

if isinstance(scenario_manifests_620gc, dict):
    print("Top-level keys:", list(scenario_manifests_620gc.keys()))

    for key, value in scenario_manifests_620gc.items():
        try:
            n = len(value)
        except Exception:
            n = None

        print(
            f"{key:10s} | "
            f"type={type(value).__name__:<12s} | "
            f"length={n}"
        )

elif hasattr(scenario_manifests_620gc, "shape"):
    print("Shape:", scenario_manifests_620gc.shape)
    print("Columns:", list(scenario_manifests_620gc.columns))

print("\nNo corruption executed.")
print("No files written.")
print("Official test used: False")
print("Nothing was modified.")
print("=== END ===")

In [ ]:
# ============================================================
# Step 6.23C-B1
# Five-family frozen-corruption replay return check
# ============================================================

import numpy as np
import pandas as pd

print("=== Step 6.23C-B1 — FIVE-FAMILY REPLAY RETURN CHECK ===\n")

families_623cb1 = ["bias", "noise", "drift", "dropout", "stuck"]

replay_outputs_623cb1 = {}

for family in families_623cb1:

    manifest = scenario_manifests_620gc[family]

    # Inspection/smoke-test scenario only.
    scenario_row = manifest.iloc[0].copy()

    print(f"--- {family.upper()} ---")
    print("Manifest rows :", len(manifest))

    # Execute exactly one already-frozen corruption scenario.
    result = apply_frozen_corruption_620gc(
        family,
        scenario_row
    )

    replay_outputs_623cb1[family] = result

    print("Return type   :", type(result).__name__)

    # --------------------------------------------------------
    # Compact structure inspection
    # --------------------------------------------------------
    if isinstance(result, pd.DataFrame):

        print("Shape         :", result.shape)
        print("Columns       :", len(result.columns))

        numeric = result.select_dtypes(include=[np.number])

        if numeric.shape[1] > 0:
            finite = np.isfinite(
                numeric.to_numpy(dtype=np.float64)
            ).all()

            print("Numeric finite:", bool(finite))
        else:
            print("Numeric finite: no numeric columns")

    elif isinstance(result, np.ndarray):

        print("Shape         :", result.shape)

        if np.issubdtype(result.dtype, np.number):
            print(
                "Numeric finite:",
                bool(np.isfinite(result).all())
            )

    elif isinstance(result, (tuple, list)):

        print("Returned items:", len(result))

        for i, item in enumerate(result):
            shape = getattr(item, "shape", None)

            print(
                f"  item_{i}: "
                f"type={type(item).__name__}, "
                f"shape={shape}"
            )

    elif isinstance(result, dict):

        print("Dictionary keys:", list(result.keys()))

        for key, item in result.items():
            shape = getattr(item, "shape", None)

            print(
                f"  {key}: "
                f"type={type(item).__name__}, "
                f"shape={shape}"
            )

    else:
        print(
            "Shape         :",
            getattr(result, "shape", None)
        )

    print()


print("=== INTEGRITY ===")
print("Families executed       :", len(replay_outputs_623cb1))
print("Final score selected     : False")
print("Reliability threshold set: False")
print("Abstention rule created  : False")
print("Official test used       : False")
print("Files written            : False")
print("Frozen artifacts modified: False")

print("\n=== END Step 6.23C-B1 ===")

In [ ]:
# ============================================================
# Step 6.23C-B1 compact confirmation
# Does NOT regenerate any corruption
# ============================================================

print("=== B1 COMPACT CONFIRMATION ===")

for family in ["bias", "noise", "drift", "dropout", "stuck"]:

    x = replay_outputs_623cb1[family]

    if isinstance(x, pd.DataFrame):
        numeric = x.select_dtypes(include=[np.number])

        finite = (
            np.isfinite(
                numeric.to_numpy(dtype=np.float64)
            ).all()
            if numeric.shape[1] > 0
            else True
        )

        print(
            f"{family:8s} | "
            f"type=DataFrame | "
            f"shape={x.shape} | "
            f"finite={finite}"
        )

    else:
        print(
            f"{family:8s} | "
            f"type={type(x).__name__} | "
            f"shape={getattr(x, 'shape', None)}"
        )

print("=== END ===")

In [ ]:
# ============================================================
# Step 6.23C-B2
# Five-family end-to-end reliability-channel smoke test
#
# Purpose:
#   1. Reconstruct the exact validation sequence pathway
#   2. Verify CLEAN frozen reliability-channel reproduction
#   3. Evaluate PCA + temporal + predictive-uncertainty
#      channels for one already-replayed scenario per family
#
# IMPORTANT:
#   - Uses DEVELOPMENT references only
#   - Uses CLEAN validation only for reproduction sanity check
#   - Does NOT use official test
#   - Does NOT select a reliability threshold
#   - Does NOT create an abstention rule
#   - Does NOT modify frozen artifacts
#   - Does NOT write files
# ============================================================

import numpy as np
import pandas as pd

print(
    "=== Step 6.23C-B2 — Five-family "
    "reliability-channel smoke test ==="
)

# ------------------------------------------------------------
# 1. Required-object check
# ------------------------------------------------------------

required_623cb2 = [
    "replay_outputs_623cb1",
    "val_raw_base_620gc",
    "X_val_clean",
    "scaler",
    "PRIMARY_INPUT_COLUMNS",
    "RETAINED_SENSOR_COLUMNS",
    "SENSOR_INPUT_INDICES",
    "sensor_pca",
    "calculate_pca_window_quality",
    "temporal_metrics_620ha",
    "sensor_strength_620ha",
    "aggregate_candidates_620ha",
    "dev_pca_max_rmse",
    "val_pca_max_rmse",
    "dev_candidate_scores_620ha",
    "val_candidate_scores_620ha",
    "FROZEN_TEMPORAL_CANDIDATE_620I",
    "phase_references_622h",
    "FROZEN_VAL_UNCERTAINTY_622IF",
    "RELIABILITY_VAL_CHANNELS_623A",
    "frozen_gru",
    "predict_gru_batched",
    "mc_dropout_predictions",
]

missing_623cb2 = [
    name for name in required_623cb2
    if name not in globals()
]

print("\n--- Required objects ---")

for name in required_623cb2:
    print(f"{name:36s}: {name in globals()}")

if missing_623cb2:
    raise RuntimeError(
        "Missing required objects: "
        + ", ".join(missing_623cb2)
    )

# ------------------------------------------------------------
# 2. Small utility helpers
# ------------------------------------------------------------

WINDOW_SIZE_623CB2 = int(X_val_clean.shape[1])
N_FEATURES_623CB2 = len(PRIMARY_INPUT_COLUMNS)

EXPECTED_WINDOWS_623CB2 = int(X_val_clean.shape[0])


def midrank_percentile_623cb2(values, reference):
    """
    Development-reference empirical mid-rank percentile.

    score(x) =
        [#(reference < x) + #(reference <= x)] / (2N)

    Therefore:
        below development minimum -> 0
        above development maximum -> 1

    Development itself has mean approximately/exactly 0.5,
    subject to ties.
    """

    values = np.asarray(
        values,
        dtype=np.float64
    ).reshape(-1)

    reference = np.asarray(
        reference,
        dtype=np.float64
    ).reshape(-1)

    if not np.isfinite(values).all():
        raise RuntimeError(
            "Non-finite query values in percentile transform."
        )

    if not np.isfinite(reference).all():
        raise RuntimeError(
            "Non-finite development reference."
        )

    ref_sorted = np.sort(reference)

    left = np.searchsorted(
        ref_sorted,
        values,
        side="left"
    )

    right = np.searchsorted(
        ref_sorted,
        values,
        side="right"
    )

    score = (
        left.astype(np.float64)
        + right.astype(np.float64)
    ) / (2.0 * len(ref_sorted))

    return np.clip(score, 0.0, 1.0)


def find_candidate_array_623cb2(
    obj,
    candidate_name,
    expected_length=None
):
    """
    Recover a named candidate array from the previously
    verified temporal helper outputs.
    """

    # Dictionary
    if isinstance(obj, dict):

        if candidate_name in obj:
            arr = np.asarray(
                obj[candidate_name]
            ).reshape(-1)

            if (
                expected_length is None
                or len(arr) == expected_length
            ):
                return arr.astype(np.float64)

        for value in obj.values():
            try:
                return find_candidate_array_623cb2(
                    value,
                    candidate_name,
                    expected_length
                )
            except (KeyError, ValueError, TypeError):
                pass

    # DataFrame
    if isinstance(obj, pd.DataFrame):

        if candidate_name in obj.columns:
            arr = obj[
                candidate_name
            ].to_numpy(
                dtype=np.float64
            ).reshape(-1)

            if (
                expected_length is None
                or len(arr) == expected_length
            ):
                return arr

    # Series
    if isinstance(obj, pd.Series):

        if obj.name == candidate_name:
            arr = obj.to_numpy(
                dtype=np.float64
            ).reshape(-1)

            if (
                expected_length is None
                or len(arr) == expected_length
            ):
                return arr

    # Tuple / list
    if isinstance(obj, (tuple, list)):

        for value in obj:
            try:
                return find_candidate_array_623cb2(
                    value,
                    candidate_name,
                    expected_length
                )
            except (KeyError, ValueError, TypeError):
                pass

    raise KeyError(
        f"Could not recover candidate "
        f"'{candidate_name}'."
    )


def frozen_channel_623cb2(
    obj,
    column_name,
    fallback_index
):
    """
    Read one frozen Step-6.23A reliability channel.
    """

    if isinstance(obj, pd.DataFrame):

        if column_name not in obj.columns:
            raise KeyError(
                f"{column_name} not found in "
                "RELIABILITY_VAL_CHANNELS_623A."
            )

        return obj[
            column_name
        ].to_numpy(
            dtype=np.float64
        )

    arr = np.asarray(
        obj,
        dtype=np.float64
    )

    if arr.ndim != 2 or arr.shape[1] < 3:
        raise RuntimeError(
            "Unexpected frozen reliability-channel structure."
        )

    return arr[:, fallback_index]


# ------------------------------------------------------------
# 3. Exact raw -> scaled sequence reconstruction
# ------------------------------------------------------------

def build_sequences_623cb2(raw_df):

    required_columns = (
        ["unit_id", "cycle"]
        + list(PRIMARY_INPUT_COLUMNS)
    )

    missing_columns = [
        c for c in required_columns
        if c not in raw_df.columns
    ]

    if missing_columns:
        raise RuntimeError(
            "Raw replay table is missing columns: "
            + ", ".join(missing_columns)
        )

    ordered = (
        raw_df
        .sort_values(
            ["unit_id", "cycle"],
            kind="mergesort"
        )
        .reset_index(drop=True)
    )

    windows = []

    for _, engine_df in ordered.groupby(
        "unit_id",
        sort=True
    ):

        engine_df = (
            engine_df
            .sort_values(
                "cycle",
                kind="mergesort"
            )
            .reset_index(drop=True)
        )

        feature_df = engine_df.loc[
            :,
            list(PRIMARY_INPUT_COLUMNS)
        ]

        scaled_engine = scaler.transform(
            feature_df
        )

        scaled_engine = np.asarray(
            scaled_engine,
            dtype=np.float32
        )

        if len(scaled_engine) < WINDOW_SIZE_623CB2:
            continue

        for end_idx in range(
            WINDOW_SIZE_623CB2 - 1,
            len(scaled_engine)
        ):

            start_idx = (
                end_idx
                - WINDOW_SIZE_623CB2
                + 1
            )

            windows.append(
                scaled_engine[
                    start_idx:end_idx + 1
                ]
            )

    if not windows:
        raise RuntimeError(
            "No sequence windows were reconstructed."
        )

    X = np.stack(
        windows,
        axis=0
    ).astype(np.float32)

    return X


# ------------------------------------------------------------
# 4. PCA raw-quality pathway
# ------------------------------------------------------------

def pca_raw_quality_623cb2(X):

    result = calculate_pca_window_quality(
        X,
        SENSOR_INPUT_INDICES,
        sensor_pca
    )

    if not isinstance(
        result,
        (tuple, list)
    ):
        raise RuntimeError(
            "Unexpected PCA helper return type."
        )

    if len(result) < 3:
        raise RuntimeError(
            "PCA helper returned fewer than 3 items."
        )

    # Previously verified ordering:
    # item_0 = row RMSE
    # item_1 = mean RMSE
    # item_2 = max RMSE
    max_rmse = np.asarray(
        result[2],
        dtype=np.float64
    ).reshape(-1)

    return max_rmse


# ------------------------------------------------------------
# 5. Temporal raw-quality pathway
# ------------------------------------------------------------

def temporal_raw_quality_623cb2(X):

    n_windows = X.shape[0]

    # Return scaled model inputs to original feature space.
    flat_scaled = X.reshape(
        -1,
        N_FEATURES_623CB2
    )

    flat_scaled_df = pd.DataFrame(
        flat_scaled,
        columns=list(PRIMARY_INPUT_COLUMNS)
    )

    flat_raw = scaler.inverse_transform(
        flat_scaled_df
    )

    raw_primary = np.asarray(
        flat_raw,
        dtype=np.float64
    ).reshape(
        n_windows,
        WINDOW_SIZE_623CB2,
        N_FEATURES_623CB2
    )

    raw_sensor_windows = raw_primary[
        :,
        :,
        SENSOR_INPUT_INDICES
    ]

    temporal_items = temporal_metrics_620ha(
        raw_sensor_windows
    )

    if not isinstance(
        temporal_items,
        (tuple, list)
    ):
        raise RuntimeError(
            "Unexpected temporal_metrics_620ha return type."
        )

    if len(temporal_items) < 3:
        raise RuntimeError(
            "temporal_metrics_620ha returned "
            "fewer than 3 items."
        )

    strength_result = sensor_strength_620ha(
        temporal_items[0],
        temporal_items[1],
        temporal_items[2]
    )

    # Earlier verified helper returns
    # (sensor_strength_matrix, metadata).
    if isinstance(
        strength_result,
        (tuple, list)
    ):
        strength_matrix = np.asarray(
            strength_result[0],
            dtype=np.float64
        )
    else:
        strength_matrix = np.asarray(
            strength_result,
            dtype=np.float64
        )

    aggregated = aggregate_candidates_620ha(
        strength_matrix
    )

    score = find_candidate_array_623cb2(
        aggregated,
        FROZEN_TEMPORAL_CANDIDATE_620I,
        expected_length=n_windows
    )

    return score


# ------------------------------------------------------------
# 6. Exact frozen uncertainty-recipe reconstruction
# ------------------------------------------------------------

SELECTED_UNCERTAINTY_623CB2 = "det_pred_pct_b5"

if (
    SELECTED_UNCERTAINTY_623CB2
    not in phase_references_622h
):
    raise RuntimeError(
        "Frozen selected uncertainty recipe "
        "not found in phase_references_622h."
    )

unc_recipe_623cb2 = (
    phase_references_622h[
        SELECTED_UNCERTAINTY_623CB2
    ]
)

edges_623cb2 = np.asarray(
    unc_recipe_623cb2["edges"],
    dtype=np.float64
)

refs_623cb2 = (
    unc_recipe_623cb2[
        "references"
    ]
)


def reference_for_bin_623cb2(bin_id):

    if bin_id in refs_623cb2:
        return refs_623cb2[bin_id]

    if str(bin_id) in refs_623cb2:
        return refs_623cb2[str(bin_id)]

    raise KeyError(
        f"No uncertainty reference for phase bin {bin_id}."
    )


def phase_adjusted_uncertainty_623cb2(
    deterministic_prediction,
    mc_std_values
):

    deterministic_prediction = np.asarray(
        deterministic_prediction,
        dtype=np.float64
    ).reshape(-1)

    mc_std_values = np.asarray(
        mc_std_values,
        dtype=np.float64
    ).reshape(-1)

    if len(deterministic_prediction) != len(
        mc_std_values
    ):
        raise RuntimeError(
            "Prediction / uncertainty length mismatch."
        )

    bin_ids = np.digitize(
        deterministic_prediction,
        edges_623cb2,
        right=True
    )

    adjusted = np.empty(
        len(mc_std_values),
        dtype=np.float64
    )

    for bin_id in np.unique(bin_ids):

        mask = (
            bin_ids == bin_id
        )

        reference = np.asarray(
            reference_for_bin_623cb2(
                int(bin_id)
            ),
            dtype=np.float64
        )

        adjusted[mask] = (
            midrank_percentile_623cb2(
                mc_std_values[mask],
                reference
            )
        )

    return adjusted


# ------------------------------------------------------------
# 7. MC-dropout extraction helper
# ------------------------------------------------------------

def mc_uncertainty_623cb2(X):

    model_device = next(
        frozen_gru.parameters()
    ).device

    deterministic_prediction = np.asarray(
        predict_gru_batched(
            frozen_gru,
            X,
            model_device,
            batch_size=512
        ),
        dtype=np.float64
    ).reshape(-1)

    mc_result = mc_dropout_predictions(
        frozen_gru,
        X,
        model_device,
        n_passes=50,
        batch_size=512,
        seed=42
    )

    items = (
        list(mc_result)
        if isinstance(
            mc_result,
            (tuple, list)
        )
        else [mc_result]
    )

    mc_matrix = None

    for item in items:

        try:
            arr = np.asarray(item)
        except Exception:
            continue

        if arr.ndim != 2:
            continue

        if arr.shape[1] == X.shape[0]:
            mc_matrix = arr.astype(
                np.float64
            )
            break

        if arr.shape[0] == X.shape[0]:
            mc_matrix = arr.T.astype(
                np.float64
            )
            break

    if mc_matrix is None:
        raise RuntimeError(
            "Could not identify MC prediction matrix."
        )

    if mc_matrix.shape[0] != 50:
        raise RuntimeError(
            "Unexpected number of MC passes: "
            f"{mc_matrix.shape[0]}"
        )

    mc_std_values = np.std(
        mc_matrix,
        axis=0,
        ddof=0
    )

    adjusted_uncertainty = (
        phase_adjusted_uncertainty_623cb2(
            deterministic_prediction,
            mc_std_values
        )
    )

    return (
        deterministic_prediction,
        mc_std_values,
        adjusted_uncertainty
    )


# ------------------------------------------------------------
# 8. Development references for channel scaling
# ------------------------------------------------------------

pca_dev_reference_623cb2 = np.asarray(
    dev_pca_max_rmse,
    dtype=np.float64
).reshape(-1)

temporal_dev_reference_623cb2 = (
    find_candidate_array_623cb2(
        dev_candidate_scores_620ha,
        FROZEN_TEMPORAL_CANDIDATE_620I,
        expected_length=14241
    )
)

temporal_val_raw_623cb2 = (
    find_candidate_array_623cb2(
        val_candidate_scores_620ha,
        FROZEN_TEMPORAL_CANDIDATE_620I,
        expected_length=EXPECTED_WINDOWS_623CB2
    )
)


# ------------------------------------------------------------
# 9. CLEAN reconstruction sanity checks
# ------------------------------------------------------------

print(
    "\n--- CLEAN pathway reproduction check ---"
)

X_clean_rebuilt_623cb2 = (
    build_sequences_623cb2(
        val_raw_base_620gc
    )
)

print(
    "Reconstructed CLEAN windows:",
    X_clean_rebuilt_623cb2.shape
)

if (
    X_clean_rebuilt_623cb2.shape
    != X_val_clean.shape
):
    raise RuntimeError(
        "Clean sequence shape does not match X_val_clean."
    )

clean_sequence_error_623cb2 = float(
    np.max(
        np.abs(
            X_clean_rebuilt_623cb2.astype(
                np.float64
            )
            -
            np.asarray(
                X_val_clean,
                dtype=np.float64
            )
        )
    )
)

print(
    "CLEAN sequence maximum error:",
    clean_sequence_error_623cb2
)

if clean_sequence_error_623cb2 > 1e-5:
    raise RuntimeError(
        "Clean sequence reconstruction mismatch."
    )


# Reconstruct the two development-scaled quality channels
pca_val_score_check_623cb2 = (
    midrank_percentile_623cb2(
        val_pca_max_rmse,
        pca_dev_reference_623cb2
    )
)

temporal_val_score_check_623cb2 = (
    midrank_percentile_623cb2(
        temporal_val_raw_623cb2,
        temporal_dev_reference_623cb2
    )
)

# Reconstruct frozen selected uncertainty from the
# already-verified CLEAN validation MC arrays.
unc_val_score_check_623cb2 = (
    phase_adjusted_uncertainty_623cb2(
        clean_predictions,
        mc_std
    )
)


frozen_val_pca_623cb2 = (
    frozen_channel_623cb2(
        RELIABILITY_VAL_CHANNELS_623A,
        "pca_quality",
        0
    )
)

frozen_val_temporal_623cb2 = (
    frozen_channel_623cb2(
        RELIABILITY_VAL_CHANNELS_623A,
        "temporal_quality",
        1
    )
)

frozen_val_uncertainty_623cb2 = (
    frozen_channel_623cb2(
        RELIABILITY_VAL_CHANNELS_623A,
        "predictive_uncertainty",
        2
    )
)


reproduction_errors_623cb2 = {
    "pca_quality": float(
        np.max(
            np.abs(
                pca_val_score_check_623cb2
                - frozen_val_pca_623cb2
            )
        )
    ),

    "temporal_quality": float(
        np.max(
            np.abs(
                temporal_val_score_check_623cb2
                - frozen_val_temporal_623cb2
            )
        )
    ),

    "predictive_uncertainty": float(
        np.max(
            np.abs(
                unc_val_score_check_623cb2
                - frozen_val_uncertainty_623cb2
            )
        )
    ),
}

for name, error in reproduction_errors_623cb2.items():
    print(
        f"{name:24s} max error = {error:.12g}"
    )

if max(
    reproduction_errors_623cb2.values()
) > 1e-8:
    raise RuntimeError(
        "Frozen CLEAN reliability-channel "
        "reproduction failed."
    )

print(
    "Frozen CLEAN reliability-channel "
    "reproduction: PASS"
)


# ------------------------------------------------------------
# 10. Five-family corrupted-data smoke test
# ------------------------------------------------------------

print(
    "\n--- Executing one replayed scenario "
    "from each corruption family ---"
)

families_623cb2 = [
    "bias",
    "noise",
    "drift",
    "dropout",
    "stuck",
]

SMOKE_RELIABILITY_623CB2 = {}
summary_rows_623cb2 = []

for family in families_623cb2:

    raw_corrupted = (
        replay_outputs_623cb1[
            family
        ]
    )

    X_corrupted = (
        build_sequences_623cb2(
            raw_corrupted
        )
    )

    if (
        X_corrupted.shape[0]
        != EXPECTED_WINDOWS_623CB2
    ):
        raise RuntimeError(
            f"{family}: expected "
            f"{EXPECTED_WINDOWS_623CB2} windows, "
            f"got {X_corrupted.shape[0]}."
        )

    # ---------------- PCA ----------------
    pca_raw = (
        pca_raw_quality_623cb2(
            X_corrupted
        )
    )

    pca_channel = (
        midrank_percentile_623cb2(
            pca_raw,
            pca_dev_reference_623cb2
        )
    )

    # ---------------- Temporal ----------------
    temporal_raw = (
        temporal_raw_quality_623cb2(
            X_corrupted
        )
    )

    temporal_channel = (
        midrank_percentile_623cb2(
            temporal_raw,
            temporal_dev_reference_623cb2
        )
    )

    # ---------------- Predictive uncertainty ----------------
    (
        deterministic_prediction,
        mc_std_values,
        uncertainty_channel
    ) = mc_uncertainty_623cb2(
        X_corrupted
    )

    channel_matrix = np.column_stack(
        [
            pca_channel,
            temporal_channel,
            uncertainty_channel,
        ]
    )

    all_finite = bool(
        np.isfinite(
            channel_matrix
        ).all()
    )

    all_in_unit_interval = bool(
        (
            (channel_matrix >= 0.0)
            &
            (channel_matrix <= 1.0)
        ).all()
    )

    if not all_finite:
        raise RuntimeError(
            f"{family}: non-finite "
            "reliability-channel value."
        )

    if not all_in_unit_interval:
        raise RuntimeError(
            f"{family}: channel value "
            "outside [0,1]."
        )

    SMOKE_RELIABILITY_623CB2[
        family
    ] = {
        "X": X_corrupted,
        "pca_raw": pca_raw,
        "temporal_raw": temporal_raw,
        "deterministic_prediction":
            deterministic_prediction,
        "mc_std": mc_std_values,
        "pca_quality": pca_channel,
        "temporal_quality": temporal_channel,
        "predictive_uncertainty":
            uncertainty_channel,
    }

    summary_rows_623cb2.append(
        {
            "family": family,
            "windows": X_corrupted.shape[0],
            "pca_mean":
                float(np.mean(pca_channel)),
            "temporal_mean":
                float(np.mean(temporal_channel)),
            "uncertainty_mean":
                float(np.mean(uncertainty_channel)),
            "pca_p95":
                float(
                    np.quantile(
                        pca_channel,
                        0.95
                    )
                ),
            "temporal_p95":
                float(
                    np.quantile(
                        temporal_channel,
                        0.95
                    )
                ),
            "uncertainty_p95":
                float(
                    np.quantile(
                        uncertainty_channel,
                        0.95
                    )
                ),
            "finite":
                all_finite,
        }
    )

    print(
        f"{family:8s}: PASS | "
        f"windows={X_corrupted.shape[0]}"
    )


SMOKE_SUMMARY_623CB2 = pd.DataFrame(
    summary_rows_623cb2
)

print(
    "\n--- Compact five-family channel summary ---"
)

display(
    SMOKE_SUMMARY_623CB2.round(6)
)


# ------------------------------------------------------------
# 11. Final integrity summary
# ------------------------------------------------------------

print(
    "\n=============================================="
)
print(
    "STEP 6.23C-B2 READINESS SUMMARY"
)
print(
    "=============================================="
)

print(
    "Clean sequence reproduction      :",
    clean_sequence_error_623cb2 <= 1e-5
)

print(
    "Frozen channel reproduction      :",
    max(
        reproduction_errors_623cb2.values()
    ) <= 1e-8
)

print(
    "Corruption families evaluated    :",
    len(SMOKE_SUMMARY_623CB2)
)

print(
    "All families have 3490 windows   :",
    bool(
        (
            SMOKE_SUMMARY_623CB2[
                "windows"
            ]
            == EXPECTED_WINDOWS_623CB2
        ).all()
    )
)

print(
    "All channel outputs finite       :",
    bool(
        SMOKE_SUMMARY_623CB2[
            "finite"
        ].all()
    )
)

print(
    "Official test used               : False"
)

print(
    "Reliability score selected       : False"
)

print(
    "Reliability threshold selected   : False"
)

print(
    "Abstention rule created          : False"
)

print(
    "Frozen artifacts modified        : False"
)

print(
    "Files written                    : False"
)

print(
    "\nStep 6.23C-B2 completed successfully."
)

In [ ]:
# ============================================================
# Step 6.23C-C
# Full 315-scenario reliability-candidate evaluation
#
# Purpose:
#   Evaluate the frozen PCA, temporal and predictive-uncertainty
#   channels — and all Step-6.23B candidate combinations —
#   across every frozen validation corruption scenario.
#
# IMPORTANT:
#   - No official test
#   - No retraining
#   - No candidate is selected here
#   - No threshold is selected here
#   - No abstention rule is created here
#   - No frozen artifact is modified
#   - No files are written
# ============================================================

import time
import numpy as np
import pandas as pd

print(
    "=== Step 6.23C-C — Full 315-scenario "
    "reliability-candidate evaluation ==="
)

# ------------------------------------------------------------
# 1. Required objects
# ------------------------------------------------------------

required_623cc = [
    "scenario_manifests_620gc",
    "apply_frozen_corruption_620gc",
    "build_sequences_623cb2",
    "pca_raw_quality_623cb2",
    "temporal_raw_quality_623cb2",
    "mc_uncertainty_623cb2",
    "midrank_percentile_623cb2",
    "pca_dev_reference_623cb2",
    "temporal_dev_reference_623cb2",
    "RELIABILITY_VAL_CHANNELS_623A",
    "frozen_channel_623cb2",
    "y_val_clean",
    "clean_predictions",
]

missing_623cc = [
    name for name in required_623cc
    if name not in globals()
]

print("\n--- Required objects ---")

for name in required_623cc:
    print(f"{name:36s}: {name in globals()}")

if missing_623cc:
    raise RuntimeError(
        "Missing required objects: "
        + ", ".join(missing_623cc)
    )


# ------------------------------------------------------------
# 2. Canonical validation target
# ------------------------------------------------------------

y_true_623cc = np.asarray(
    y_val_clean,
    dtype=np.float64
).reshape(-1)

if y_true_623cc.shape != (3490,):
    raise RuntimeError(
        f"Unexpected validation target shape: "
        f"{y_true_623cc.shape}"
    )

if not np.isfinite(y_true_623cc).all():
    raise RuntimeError(
        "Validation target contains non-finite values."
    )

clean_prediction_623cc = np.asarray(
    clean_predictions,
    dtype=np.float64
).reshape(-1)

if clean_prediction_623cc.shape != y_true_623cc.shape:
    raise RuntimeError(
        "Clean prediction / target shape mismatch."
    )

clean_absolute_error_623cc = np.abs(
    clean_prediction_623cc
    - y_true_623cc
)

CLEAN_MAE_623CC = float(
    np.mean(clean_absolute_error_623cc)
)

print(
    "\nCanonical CLEAN validation MAE:",
    CLEAN_MAE_623CC
)


# ------------------------------------------------------------
# 3. Reliability-score candidate constructor
#
# All channels use:
#     0 = better / less concerning
#     1 = worse  / more concerning
# ------------------------------------------------------------

CANDIDATE_NAMES_623CC = [
    "pca_only",
    "temporal_only",
    "uncertainty_only",
    "sensor_quality_max",
    "max3",
    "mean3",
    "rms3",
    "top2_mean",
]


def build_reliability_candidates_623cc(
    pca_channel,
    temporal_channel,
    uncertainty_channel
):

    p = np.asarray(
        pca_channel,
        dtype=np.float64
    ).reshape(-1)

    t = np.asarray(
        temporal_channel,
        dtype=np.float64
    ).reshape(-1)

    u = np.asarray(
        uncertainty_channel,
        dtype=np.float64
    ).reshape(-1)

    if not (
        len(p) == len(t) == len(u)
    ):
        raise RuntimeError(
            "Reliability-channel length mismatch."
        )

    channels = np.column_stack(
        [p, t, u]
    )

    sorted_channels = np.sort(
        channels,
        axis=1
    )

    candidates = {
        "pca_only":
            p,

        "temporal_only":
            t,

        "uncertainty_only":
            u,

        "sensor_quality_max":
            np.maximum(p, t),

        "max3":
            np.max(
                channels,
                axis=1
            ),

        "mean3":
            np.mean(
                channels,
                axis=1
            ),

        "rms3":
            np.sqrt(
                np.mean(
                    channels ** 2,
                    axis=1
                )
            ),

        "top2_mean":
            np.mean(
                sorted_channels[:, -2:],
                axis=1
            ),
    }

    for name, values in candidates.items():

        if not np.isfinite(values).all():
            raise RuntimeError(
                f"{name} contains non-finite values."
            )

        if not (
            (
                (values >= 0.0)
                &
                (values <= 1.0)
            ).all()
        ):
            raise RuntimeError(
                f"{name} contains values outside [0,1]."
            )

    return candidates


# ------------------------------------------------------------
# 4. CLEAN candidate references
# ------------------------------------------------------------

clean_pca_623cc = frozen_channel_623cb2(
    RELIABILITY_VAL_CHANNELS_623A,
    "pca_quality",
    0
)

clean_temporal_623cc = frozen_channel_623cb2(
    RELIABILITY_VAL_CHANNELS_623A,
    "temporal_quality",
    1
)

clean_uncertainty_623cc = frozen_channel_623cb2(
    RELIABILITY_VAL_CHANNELS_623A,
    "predictive_uncertainty",
    2
)

CLEAN_CANDIDATES_623CC = (
    build_reliability_candidates_623cc(
        clean_pca_623cc,
        clean_temporal_623cc,
        clean_uncertainty_623cc
    )
)

CLEAN_CANDIDATE_MEANS_623CC = {
    name: float(
        np.mean(values)
    )
    for name, values
    in CLEAN_CANDIDATES_623CC.items()
}


# ------------------------------------------------------------
# 5. Safe correlation helper
# ------------------------------------------------------------

def safe_corr_623cc(x, y, method):

    x = np.asarray(
        x,
        dtype=np.float64
    ).reshape(-1)

    y = np.asarray(
        y,
        dtype=np.float64
    ).reshape(-1)

    if len(x) != len(y):
        raise RuntimeError(
            "Correlation-array length mismatch."
        )

    if (
        np.std(x) == 0.0
        or np.std(y) == 0.0
    ):
        return np.nan

    value = pd.Series(x).corr(
        pd.Series(y),
        method=method
    )

    if pd.isna(value):
        return np.nan

    return float(value)


# ------------------------------------------------------------
# 6. Selective-risk diagnostics
# ------------------------------------------------------------

COVERAGES_623CC = [
    0.90,
    0.70,
    0.50,
]


def reliability_utility_623cc(
    score,
    absolute_error
):

    score = np.asarray(
        score,
        dtype=np.float64
    ).reshape(-1)

    absolute_error = np.asarray(
        absolute_error,
        dtype=np.float64
    ).reshape(-1)

    n = len(score)

    if len(absolute_error) != n:
        raise RuntimeError(
            "Score / error length mismatch."
        )

    full_mae = float(
        np.mean(
            absolute_error
        )
    )

    # Higher reliability score = worse.
    # Therefore accept the LOWEST-score windows.
    order = np.argsort(
        score,
        kind="mergesort"
    )

    coverage_results = {}

    for coverage in COVERAGES_623CC:

        accepted_n = int(
            np.floor(
                coverage * n
            )
        )

        accepted_n = max(
            1,
            min(
                accepted_n,
                n
            )
        )

        accepted_idx = order[
            :accepted_n
        ]

        accepted_mae = float(
            np.mean(
                absolute_error[
                    accepted_idx
                ]
            )
        )

        coverage_results[
            coverage
        ] = {
            "accepted_mae":
                accepted_mae,

            # Positive = selective rejection helped.
            "mae_gain":
                full_mae
                - accepted_mae,
        }

    # Bottom / top quartile comparison
    quartile_n = int(
        np.ceil(
            0.25 * n
        )
    )

    lowest_idx = order[
        :quartile_n
    ]

    highest_idx = order[
        -quartile_n:
    ]

    low_mae = float(
        np.mean(
            absolute_error[
                lowest_idx
            ]
        )
    )

    high_mae = float(
        np.mean(
            absolute_error[
                highest_idx
            ]
        )
    )

    high_low_ratio = (
        high_mae / low_mae
        if low_mae > 0
        else np.nan
    )

    # How many of the largest-error windows are captured
    # by the highest reliability-score quartile?
    error_order = np.argsort(
        absolute_error,
        kind="mergesort"
    )

    top_error_idx = set(
        error_order[
            -quartile_n:
        ].tolist()
    )

    top_score_idx = set(
        highest_idx.tolist()
    )

    overlap = len(
        top_error_idx.intersection(
            top_score_idx
        )
    )

    top_error_capture_pct = (
        100.0
        * overlap
        / quartile_n
    )

    return {
        "full_mae":
            full_mae,

        "error_spearman":
            safe_corr_623cc(
                score,
                absolute_error,
                "spearman"
            ),

        "error_pearson":
            safe_corr_623cc(
                score,
                absolute_error,
                "pearson"
            ),

        "low_quartile_mae":
            low_mae,

        "high_quartile_mae":
            high_mae,

        "high_low_ratio":
            high_low_ratio,

        "top_error_capture_pct":
            top_error_capture_pct,

        "gain90":
            coverage_results[
                0.90
            ]["mae_gain"],

        "gain70":
            coverage_results[
                0.70
            ]["mae_gain"],

        "gain50":
            coverage_results[
                0.50
            ]["mae_gain"],

        "mean_rc_gain":
            float(
                np.mean(
                    [
                        coverage_results[
                            c
                        ]["mae_gain"]
                        for c in COVERAGES_623CC
                    ]
                )
            ),
    }


# ------------------------------------------------------------
# 7. Execute all frozen corruption scenarios
# ------------------------------------------------------------

total_scenarios_623cc = int(
    sum(
        len(df)
        for df in scenario_manifests_620gc.values()
    )
)

print(
    "\nFrozen scenarios to evaluate:",
    total_scenarios_623cc
)

if total_scenarios_623cc != 315:
    raise RuntimeError(
        "Expected exactly 315 frozen scenarios."
    )

scenario_metric_rows_623cc = []

scenario_counter_623cc = 0

start_time_623cc = time.perf_counter()

print(
    "\n=== Executing 315 frozen scenarios ==="
)

for family in [
    "bias",
    "noise",
    "drift",
    "dropout",
    "stuck",
]:

    manifest = (
        scenario_manifests_620gc[
            family
        ]
    )

    for manifest_index, scenario_row in (
        manifest.iterrows()
    ):

        scenario_counter_623cc += 1

        # ------------------------------------
        # Replay frozen corruption
        # ------------------------------------

        corrupted_raw = (
            apply_frozen_corruption_620gc(
                family,
                scenario_row
            )
        )

        # ------------------------------------
        # Reconstruct exact validation windows
        # ------------------------------------

        X_corrupted = (
            build_sequences_623cb2(
                corrupted_raw
            )
        )

        if X_corrupted.shape[0] != 3490:
            raise RuntimeError(
                f"{family} scenario "
                f"{manifest_index}: "
                f"expected 3490 windows, got "
                f"{X_corrupted.shape[0]}."
            )

        # ------------------------------------
        # PCA quality
        # ------------------------------------

        pca_raw = (
            pca_raw_quality_623cb2(
                X_corrupted
            )
        )

        pca_channel = (
            midrank_percentile_623cb2(
                pca_raw,
                pca_dev_reference_623cb2
            )
        )

        # ------------------------------------
        # Temporal quality
        # ------------------------------------

        temporal_raw = (
            temporal_raw_quality_623cb2(
                X_corrupted
            )
        )

        temporal_channel = (
            midrank_percentile_623cb2(
                temporal_raw,
                temporal_dev_reference_623cb2
            )
        )

        # ------------------------------------
        # Predictive uncertainty + prediction
        # ------------------------------------

        (
            prediction,
            mc_std_values,
            uncertainty_channel
        ) = mc_uncertainty_623cb2(
            X_corrupted
        )

        prediction = np.asarray(
            prediction,
            dtype=np.float64
        ).reshape(-1)

        if prediction.shape != y_true_623cc.shape:
            raise RuntimeError(
                f"{family} scenario "
                f"{manifest_index}: prediction "
                "shape mismatch."
            )

        absolute_error = np.abs(
            prediction
            - y_true_623cc
        )

        scenario_mae = float(
            np.mean(
                absolute_error
            )
        )

        mae_delta_vs_clean = (
            scenario_mae
            - CLEAN_MAE_623CC
        )

        # ------------------------------------
        # Candidate reliability scores
        # ------------------------------------

        candidates = (
            build_reliability_candidates_623cc(
                pca_channel,
                temporal_channel,
                uncertainty_channel
            )
        )

        # ------------------------------------
        # Scenario metadata
        # ------------------------------------

        def meta_623cc(column):
            if column in scenario_row.index:
                return scenario_row[column]
            return np.nan

        # ------------------------------------
        # Candidate utility metrics
        # ------------------------------------

        for candidate_name, score in (
            candidates.items()
        ):

            utility = (
                reliability_utility_623cc(
                    score,
                    absolute_error
                )
            )

            row = {
                "family":
                    family,

                "manifest_index":
                    manifest_index,

                "scenario_id":
                    meta_623cc(
                        "scenario_id"
                    ),

                "sensor":
                    meta_623cc(
                        "sensor"
                    ),

                "severity":
                    meta_623cc(
                        "severity"
                    ),

                "direction":
                    meta_623cc(
                        "direction"
                    ),

                "alpha":
                    meta_623cc(
                        "alpha"
                    ),

                "requested_fraction":
                    meta_623cc(
                        "requested_fraction"
                    ),

                "candidate":
                    candidate_name,

                "scenario_mae":
                    scenario_mae,

                "mae_delta_vs_clean":
                    mae_delta_vs_clean,

                "score_mean":
                    float(
                        np.mean(
                            score
                        )
                    ),

                "score_mean_delta":
                    float(
                        np.mean(
                            score
                        )
                    )
                    - CLEAN_CANDIDATE_MEANS_623CC[
                        candidate_name
                    ],
            }

            row.update(
                utility
            )

            scenario_metric_rows_623cc.append(
                row
            )

        # ------------------------------------
        # Compact progress only
        # ------------------------------------

        if (
            scenario_counter_623cc % 25 == 0
            or
            scenario_counter_623cc
            == total_scenarios_623cc
        ):

            elapsed = (
                time.perf_counter()
                - start_time_623cc
            )

            print(
                f"[{scenario_counter_623cc:03d}/"
                f"{total_scenarios_623cc}] "
                f"completed — "
                f"{elapsed:.1f} s"
            )


elapsed_623cc = (
    time.perf_counter()
    - start_time_623cc
)

print(
    "\nFull execution time:",
    round(
        elapsed_623cc,
        2
    ),
    "seconds"
)


# ------------------------------------------------------------
# 8. Scenario-level result table
# ------------------------------------------------------------

RELIABILITY_SCENARIO_RESULTS_623CC = (
    pd.DataFrame(
        scenario_metric_rows_623cc
    )
)

expected_rows_623cc = (
    315
    * len(
        CANDIDATE_NAMES_623CC
    )
)

if (
    len(
        RELIABILITY_SCENARIO_RESULTS_623CC
    )
    != expected_rows_623cc
):
    raise RuntimeError(
        "Unexpected scenario/candidate row count."
    )


# ------------------------------------------------------------
# 9. FAMILY-balanced candidate summaries
#
# First average within each corruption family.
# Then average the five families equally.
# This prevents bias/drift from receiving extra weight merely
# because they contain more directional scenarios.
# ------------------------------------------------------------

metrics_for_balance_623cc = [
    "error_spearman",
    "error_pearson",
    "high_low_ratio",
    "top_error_capture_pct",
    "gain90",
    "gain70",
    "gain50",
    "mean_rc_gain",
    "score_mean_delta",
    "mae_delta_vs_clean",
]

RELIABILITY_FAMILY_RESULTS_623CC = (
    RELIABILITY_SCENARIO_RESULTS_623CC
    .groupby(
        [
            "family",
            "candidate",
        ],
        as_index=False
    )[
        metrics_for_balance_623cc
    ]
    .mean()
)


# ------------------------------------------------------------
# 10. Scenario-response association:
#     Does mean reliability-score increase when scenario MAE
#     increases?
#
# Computed within each family, then family-balanced.
# ------------------------------------------------------------

delta_rows_623cc = []

for candidate_name in CANDIDATE_NAMES_623CC:

    candidate_df = (
        RELIABILITY_SCENARIO_RESULTS_623CC[
            RELIABILITY_SCENARIO_RESULTS_623CC[
                "candidate"
            ]
            == candidate_name
        ]
    )

    for family in [
        "bias",
        "noise",
        "drift",
        "dropout",
        "stuck",
    ]:

        family_df = (
            candidate_df[
                candidate_df[
                    "family"
                ]
                == family
            ]
        )

        rho = safe_corr_623cc(
            family_df[
                "score_mean_delta"
            ],
            family_df[
                "mae_delta_vs_clean"
            ],
            "spearman"
        )

        delta_rows_623cc.append(
            {
                "candidate":
                    candidate_name,

                "family":
                    family,

                "delta_error_spearman":
                    rho,
            }
        )


DELTA_ASSOCIATION_623CC = pd.DataFrame(
    delta_rows_623cc
)

delta_balanced_623cc = (
    DELTA_ASSOCIATION_623CC
    .groupby(
        "candidate",
        as_index=False
    )[
        "delta_error_spearman"
    ]
    .mean()
)


# ------------------------------------------------------------
# 11. Family-positive selective-utility counts
# ------------------------------------------------------------

family_gain_counts_623cc = (
    RELIABILITY_FAMILY_RESULTS_623CC
    .assign(
        positive_gain70=lambda x:
            x["gain70"] > 0.0
    )
    .groupby(
        "candidate",
        as_index=False
    )[
        "positive_gain70"
    ]
    .sum()
    .rename(
        columns={
            "positive_gain70":
                "families_positive_gain70"
        }
    )
)


# ------------------------------------------------------------
# 12. Final family-balanced candidate table
# ------------------------------------------------------------

RELIABILITY_CANDIDATE_SUMMARY_623CC = (
    RELIABILITY_FAMILY_RESULTS_623CC
    .groupby(
        "candidate",
        as_index=False
    )
    .agg(
        fb_error_spearman=(
            "error_spearman",
            "mean"
        ),

        fb_high_low_ratio=(
            "high_low_ratio",
            "mean"
        ),

        fb_top_error_capture_pct=(
            "top_error_capture_pct",
            "mean"
        ),

        fb_gain90=(
            "gain90",
            "mean"
        ),

        fb_gain70=(
            "gain70",
            "mean"
        ),

        fb_gain50=(
            "gain50",
            "mean"
        ),

        fb_mean_rc_gain=(
            "mean_rc_gain",
            "mean"
        ),

        fb_score_mean_delta=(
            "score_mean_delta",
            "mean"
        ),
    )
    .merge(
        delta_balanced_623cc,
        on="candidate",
        how="left"
    )
    .merge(
        family_gain_counts_623cc,
        on="candidate",
        how="left"
    )
)


# Diagnostic ordering only.
# NO final candidate is selected here.
RELIABILITY_CANDIDATE_SUMMARY_623CC = (
    RELIABILITY_CANDIDATE_SUMMARY_623CC
    .sort_values(
        [
            "fb_mean_rc_gain",
            "fb_error_spearman",
        ],
        ascending=[
            False,
            False,
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n--- FAMILY-BALANCED candidate summary ---"
)

display(
    RELIABILITY_CANDIDATE_SUMMARY_623CC[
        [
            "candidate",
            "fb_error_spearman",
            "fb_high_low_ratio",
            "fb_top_error_capture_pct",
            "fb_gain90",
            "fb_gain70",
            "fb_gain50",
            "fb_mean_rc_gain",
            "delta_error_spearman",
            "families_positive_gain70",
        ]
    ].round(4)
)


# ------------------------------------------------------------
# 13. Compact integrity checks
# ------------------------------------------------------------

numeric_cols_623cc = [
    "scenario_mae",
    "mae_delta_vs_clean",
    "score_mean",
    "score_mean_delta",
    "full_mae",
    "low_quartile_mae",
    "high_quartile_mae",
    "high_low_ratio",
    "top_error_capture_pct",
    "gain90",
    "gain70",
    "gain50",
    "mean_rc_gain",
]

all_actual_numeric_finite_623cc = bool(
    np.isfinite(
        RELIABILITY_SCENARIO_RESULTS_623CC[
            numeric_cols_623cc
        ].to_numpy(
            dtype=np.float64
        )
    ).all()
)

families_present_623cc = sorted(
    RELIABILITY_SCENARIO_RESULTS_623CC[
        "family"
    ].unique().tolist()
)

candidates_present_623cc = sorted(
    RELIABILITY_SCENARIO_RESULTS_623CC[
        "candidate"
    ].unique().tolist()
)


print(
    "\n=============================================="
)
print(
    "STEP 6.23C-C INTEGRITY SUMMARY"
)
print(
    "=============================================="
)

print(
    "Frozen corruption scenarios evaluated :",
    scenario_counter_623cc
)

print(
    "Reliability candidates per scenario    :",
    len(
        CANDIDATE_NAMES_623CC
    )
)

print(
    "Scenario/candidate result rows         :",
    len(
        RELIABILITY_SCENARIO_RESULTS_623CC
    )
)

print(
    "Expected result rows                   :",
    expected_rows_623cc
)

print(
    "Corruption families present            :",
    families_present_623cc
)

print(
    "All candidate names present            :",
    len(
        candidates_present_623cc
    )
    == len(
        CANDIDATE_NAMES_623CC
    )
)

print(
    "Actual experimental metrics finite     :",
    all_actual_numeric_finite_623cc
)

print(
    "Family-balanced aggregation used       : True"
)

print(
    "Official test used                     : False"
)

print(
    "Final reliability score selected       : False"
)

print(
    "Reliability threshold selected         : False"
)

print(
    "Abstention rule created                : False"
)

print(
    "Frozen artifacts modified              : False"
)

print(
    "Files written                          : False"
)

print(
    "\nStep 6.23C-C completed successfully."
)

In [ ]:
# ============================================================
# Step 6.23D-A
# Development corruption-replay readiness inspection
# INSPECTION ONLY
# ============================================================

import inspect

print(
    "=== Step 6.23D-A — Development corruption replay "
    "readiness inspection ==="
)

# ------------------------------------------------------------
# 1. Look for development raw/base objects
# ------------------------------------------------------------

priority_names_623da = [
    "dev_raw_base_620gc",
    "dev_raw_base",
    "development_raw_base",
    "dev_base",
    "dev_df",
    "development_df",
    "dev_seq_meta",
    "X_dev_clean",
]

print("\n--- Priority development objects ---")

for name in priority_names_623da:
    if name in globals():

        obj = globals()[name]

        shape = getattr(
            obj,
            "shape",
            None
        )

        print(
            f"{name:30s} : True"
            f" | type={type(obj).__name__}"
            f" | shape={shape}"
        )

    else:
        print(
            f"{name:30s} : False"
        )


# ------------------------------------------------------------
# 2. Search namespace for likely raw development DataFrames
# ------------------------------------------------------------

rows_623da = []

for name, obj in list(globals().items()):

    lname = str(name).lower()

    if not (
        "dev" in lname
        or
        "development" in lname
    ):
        continue

    if isinstance(obj, pd.DataFrame):

        rows_623da.append(
            {
                "object_name": name,
                "rows": len(obj),
                "columns": obj.shape[1],
                "first_columns":
                    ", ".join(
                        map(
                            str,
                            obj.columns[:8]
                        )
                    ),
            }
        )


print(
    "\n--- Development-related DataFrames ---"
)

if rows_623da:

    display(
        pd.DataFrame(
            rows_623da
        ).sort_values(
            [
                "rows",
                "object_name"
            ],
            ascending=[
                False,
                True
            ]
        ).head(25)
    )

else:

    print(
        "No development-related DataFrame found."
    )


# ------------------------------------------------------------
# 3. Inspect exact corruption-replay function source
# ------------------------------------------------------------

print(
    "\n--- apply_frozen_corruption_620gc source ---"
)

print(
    inspect.getsource(
        apply_frozen_corruption_620gc
    )
)


# ------------------------------------------------------------
# 4. Check manifest independence
# ------------------------------------------------------------

print(
    "\n--- Frozen scenario manifests ---"
)

for family in [
    "bias",
    "noise",
    "drift",
    "dropout",
    "stuck",
]:

    manifest = scenario_manifests_620gc[
        family
    ]

    print(
        f"{family:8s}"
        f" | rows={len(manifest)}"
        f" | columns={list(manifest.columns)}"
    )


print(
    "\nOfficial test used       : False"
)
print(
    "Corruption generated     : False"
)
print(
    "Candidate selected       : False"
)
print(
    "Threshold selected       : False"
)
print(
    "Abstention rule created  : False"
)
print(
    "Files written            : False"
)
print(
    "Nothing was modified."
)
print(
    "=== END ==="
)

In [ ]:
# ============================================================
# Step 6.23D-B
# Verify canonical raw development sequence source
# INSPECTION ONLY — nothing is modified
# ============================================================

import numpy as np
import pandas as pd

print(
    "=== Step 6.23D-B — Canonical development raw-source "
    "verification ==="
)

# ------------------------------------------------------------
# 1. Required objects
# ------------------------------------------------------------

required_623db = [
    "X_dev_clean",
    "scaler",
    "PRIMARY_INPUT_COLUMNS",
]

print("\n--- Required objects ---")

for name in required_623db:
    print(
        f"{name:25s}: {name in globals()}"
    )

assert all(
    name in globals()
    for name in required_623db
)


X_dev_ref_623db = np.asarray(
    X_dev_clean,
    dtype=np.float64
)

n_windows_623db, window_size_623db, n_features_623db = (
    X_dev_ref_623db.shape
)

expected_flat_rows_623db = (
    n_windows_623db
    * window_size_623db
)

print(
    "\nDevelopment sequence shape :",
    X_dev_ref_623db.shape
)
print(
    "Expected flat rows         :",
    expected_flat_rows_623db
)
print(
    "Primary feature count      :",
    len(PRIMARY_INPUT_COLUMNS)
)


# ------------------------------------------------------------
# 2. Candidate raw development DataFrames
# ------------------------------------------------------------

candidate_names_623db = [
    "X_dev_flat64_df_620d1",
    "X_dev_flat_df_620d",
    "X_dev_raw64_df_620d1",
    "X_dev_raw_flat_df_620d",
    "dev_flat_raw_df_620d",
    "dev_flat_scaled_df_620d",
]

rows_623db = []


for name in candidate_names_623db:

    if name not in globals():
        continue

    df = globals()[name]

    if not isinstance(
        df,
        pd.DataFrame
    ):
        continue

    feature_ok = all(
        c in df.columns
        for c in PRIMARY_INPUT_COLUMNS
    )

    record = {
        "object_name": name,
        "rows": len(df),
        "columns": df.shape[1],
        "feature_columns_ok": feature_ok,
        "row_count_ok":
            len(df) == expected_flat_rows_623db,
    }

    if not (
        feature_ok
        and
        len(df) == expected_flat_rows_623db
    ):

        record[
            "scaled_vs_Xdev_max_error"
        ] = np.nan

        record[
            "raw_direct_vs_Xdev_max_error"
        ] = np.nan

        rows_623db.append(
            record
        )

        continue


    # --------------------------------------------------------
    # Raw candidate values
    # --------------------------------------------------------

    raw_flat = (
        df[
            PRIMARY_INPUT_COLUMNS
        ]
        .to_numpy(
            dtype=np.float64
        )
    )


    # --------------------------------------------------------
    # A. Direct comparison
    # This identifies candidates that may already be scaled.
    # --------------------------------------------------------

    direct_seq = raw_flat.reshape(
        n_windows_623db,
        window_size_623db,
        n_features_623db
    )

    direct_error = float(
        np.max(
            np.abs(
                direct_seq
                -
                X_dev_ref_623db
            )
        )
    )


    # --------------------------------------------------------
    # B. Apply frozen scaler
    # This identifies true raw candidates.
    # --------------------------------------------------------

    raw_feature_df = pd.DataFrame(
        raw_flat,
        columns=PRIMARY_INPUT_COLUMNS
    )

    scaled_flat = scaler.transform(
        raw_feature_df
    )

    scaled_seq = np.asarray(
        scaled_flat,
        dtype=np.float64
    ).reshape(
        n_windows_623db,
        window_size_623db,
        n_features_623db
    )

    scaled_error = float(
        np.max(
            np.abs(
                scaled_seq
                -
                X_dev_ref_623db
            )
        )
    )


    record[
        "scaled_vs_Xdev_max_error"
    ] = scaled_error

    record[
        "raw_direct_vs_Xdev_max_error"
    ] = direct_error

    rows_623db.append(
        record
    )


comparison_623db = pd.DataFrame(
    rows_623db
)

print(
    "\n--- Development source comparison ---"
)

display(
    comparison_623db
    .sort_values(
        [
            "scaled_vs_Xdev_max_error",
            "raw_direct_vs_Xdev_max_error",
        ],
        na_position="last"
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# 3. Candidate equality check
# ------------------------------------------------------------

valid_names_623db = [
    name
    for name in candidate_names_623db
    if (
        name in globals()
        and
        isinstance(
            globals()[name],
            pd.DataFrame
        )
        and
        len(
            globals()[name]
        ) == expected_flat_rows_623db
        and
        all(
            c in globals()[name].columns
            for c in PRIMARY_INPUT_COLUMNS
        )
    )
]

print(
    "\n--- Pairwise equality of candidate feature matrices ---"
)

for i in range(
    len(valid_names_623db)
):

    for j in range(
        i + 1,
        len(valid_names_623db)
    ):

        name_a = valid_names_623db[i]
        name_b = valid_names_623db[j]

        arr_a = (
            globals()[name_a][
                PRIMARY_INPUT_COLUMNS
            ]
            .to_numpy(
                dtype=np.float64
            )
        )

        arr_b = (
            globals()[name_b][
                PRIMARY_INPUT_COLUMNS
            ]
            .to_numpy(
                dtype=np.float64
            )
        )

        maxdiff = float(
            np.max(
                np.abs(
                    arr_a - arr_b
                )
            )
        )

        print(
            f"{name_a:28s}"
            f" vs "
            f"{name_b:28s}"
            f" : {maxdiff:.12g}"
        )


# ------------------------------------------------------------
# 4. Compact conclusion
# ------------------------------------------------------------

finite_candidates_623db = (
    comparison_623db[
        comparison_623db[
            "scaled_vs_Xdev_max_error"
        ].notna()
    ]
    .copy()
)

if len(
    finite_candidates_623db
) > 0:

    best_row_623db = (
        finite_candidates_623db
        .sort_values(
            "scaled_vs_Xdev_max_error"
        )
        .iloc[0]
    )

    print(
        "\n=== COMPACT RESULT ==="
    )

    print(
        "Best raw-source candidate :",
        best_row_623db[
            "object_name"
        ]
    )

    print(
        "Scaled reconstruction err :",
        best_row_623db[
            "scaled_vs_Xdev_max_error"
        ]
    )

    print(
        "Direct reconstruction err :",
        best_row_623db[
            "raw_direct_vs_Xdev_max_error"
        ]
    )

else:

    print(
        "\nNo valid development candidate "
        "could be evaluated."
    )


print(
    "\nOfficial test used      : False"
)
print(
    "Corruption executed     : False"
)
print(
    "Reliability selected    : False"
)
print(
    "Threshold selected      : False"
)
print(
    "Abstention rule created : False"
)
print(
    "Files written           : False"
)
print(
    "Nothing was modified."
)
print(
    "=== END ==="
)

In [ ]:
# ============================================================
# Step 6.23D-C
# Recover canonical UNIQUE-ROW development raw base
#
# Purpose:
#   1. Verify dev_scaled reproduces X_dev_clean exactly.
#   2. Invert the frozen scaler to recover unique-cycle raw data.
#   3. Verify those raw rows reproduce the already-confirmed
#      flattened raw window representation.
#
# No corruption.
# No threshold selection.
# No official test.
# No files written.
# ============================================================

import numpy as np
import pandas as pd

print(
    "=== Step 6.23D-C — Unique-row development "
    "raw-base recovery ==="
)

# ------------------------------------------------------------
# 1. Required objects
# ------------------------------------------------------------

required_623dc = [
    "dev_scaled",
    "dev_seq_meta",
    "X_dev_clean",
    "X_dev_raw_flat_df_620d",
    "scaler",
    "PRIMARY_INPUT_COLUMNS",
]

print("\n--- Required objects ---")

for name in required_623dc:
    print(
        f"{name:28s}: {name in globals()}"
    )

assert all(
    name in globals()
    for name in required_623dc
)


# ------------------------------------------------------------
# 2. Basic structure
# ------------------------------------------------------------

WINDOW_SIZE_623DC = 30

dev_scaled_623dc = (
    dev_scaled
    .copy()
)

dev_meta_623dc = (
    dev_seq_meta
    .copy()
)

X_dev_reference_623dc = np.asarray(
    X_dev_clean,
    dtype=np.float64
)

raw_flat_reference_623dc = (
    X_dev_raw_flat_df_620d[
        PRIMARY_INPUT_COLUMNS
    ]
    .to_numpy(
        dtype=np.float64
    )
)

assert "unit_id" in dev_scaled_623dc.columns
assert "cycle" in dev_scaled_623dc.columns

assert "unit_id" in dev_meta_623dc.columns
assert "cycle" in dev_meta_623dc.columns

assert all(
    c in dev_scaled_623dc.columns
    for c in PRIMARY_INPUT_COLUMNS
)


n_unique_rows_623dc = len(
    dev_scaled_623dc
)

n_engines_623dc = (
    dev_scaled_623dc[
        "unit_id"
    ]
    .nunique()
)

expected_windows_623dc = int(
    sum(
        max(
            0,
            len(group) - WINDOW_SIZE_623DC + 1
        )
        for _, group
        in dev_scaled_623dc.groupby(
            "unit_id"
        )
    )
)

print(
    "\n--- Unique-row development structure ---"
)

print(
    "Unique development rows :",
    n_unique_rows_623dc
)

print(
    "Development engines     :",
    n_engines_623dc
)

print(
    "Expected windows        :",
    expected_windows_623dc
)

print(
    "Stored windows          :",
    X_dev_reference_623dc.shape[0]
)

print(
    "Metadata windows        :",
    len(dev_meta_623dc)
)


# ------------------------------------------------------------
# 3. Recover raw unique-cycle feature values
# ------------------------------------------------------------

scaled_unique_features_623dc = (
    dev_scaled_623dc[
        PRIMARY_INPUT_COLUMNS
    ]
    .copy()
)

raw_unique_features_623dc = (
    scaler.inverse_transform(
        scaled_unique_features_623dc
    )
)

raw_unique_features_623dc = np.asarray(
    raw_unique_features_623dc,
    dtype=np.float64
)

assert np.isfinite(
    raw_unique_features_623dc
).all()


DEV_RAW_BASE_623DC = (
    dev_scaled_623dc
    .copy()
)

DEV_RAW_BASE_623DC.loc[
    :,
    PRIMARY_INPUT_COLUMNS
] = raw_unique_features_623dc


# ------------------------------------------------------------
# 4. Inverse/forward scaler consistency
# ------------------------------------------------------------

raw_feature_df_623dc = pd.DataFrame(
    raw_unique_features_623dc,
    columns=PRIMARY_INPUT_COLUMNS,
    index=DEV_RAW_BASE_623DC.index,
)

re_scaled_unique_623dc = (
    scaler.transform(
        raw_feature_df_623dc
    )
)

re_scaled_unique_623dc = np.asarray(
    re_scaled_unique_623dc,
    dtype=np.float64
)

original_scaled_unique_623dc = (
    scaled_unique_features_623dc
    .to_numpy(
        dtype=np.float64
    )
)

unique_rescale_error_623dc = float(
    np.max(
        np.abs(
            re_scaled_unique_623dc
            -
            original_scaled_unique_623dc
        )
    )
)


# ------------------------------------------------------------
# 5. Create engine caches
# ------------------------------------------------------------

scaled_cache_623dc = {}
raw_cache_623dc = {}

sorted_dev_623dc = (
    dev_scaled_623dc
    .sort_values(
        ["unit_id", "cycle"],
        kind="mergesort"
    )
)

sorted_raw_623dc = (
    DEV_RAW_BASE_623DC
    .sort_values(
        ["unit_id", "cycle"],
        kind="mergesort"
    )
)


for unit_id, group in sorted_dev_623dc.groupby(
    "unit_id",
    sort=True
):

    group = group.sort_values(
        "cycle",
        kind="mergesort"
    )

    cycles = (
        group["cycle"]
        .to_numpy()
    )

    scaled_values = (
        group[
            PRIMARY_INPUT_COLUMNS
        ]
        .to_numpy(
            dtype=np.float64
        )
    )

    cycle_to_position = {
        cycle: idx
        for idx, cycle
        in enumerate(cycles)
    }

    scaled_cache_623dc[
        unit_id
    ] = (
        scaled_values,
        cycle_to_position
    )


for unit_id, group in sorted_raw_623dc.groupby(
    "unit_id",
    sort=True
):

    group = group.sort_values(
        "cycle",
        kind="mergesort"
    )

    cycles = (
        group["cycle"]
        .to_numpy()
    )

    raw_values = (
        group[
            PRIMARY_INPUT_COLUMNS
        ]
        .to_numpy(
            dtype=np.float64
        )
    )

    cycle_to_position = {
        cycle: idx
        for idx, cycle
        in enumerate(cycles)
    }

    raw_cache_623dc[
        unit_id
    ] = (
        raw_values,
        cycle_to_position
    )


# ------------------------------------------------------------
# 6. Reconstruct windows in EXACT dev_seq_meta order
# ------------------------------------------------------------

scaled_windows_623dc = []
raw_windows_623dc = []

for row in dev_meta_623dc.itertuples(
    index=False
):

    unit_id = getattr(
        row,
        "unit_id"
    )

    end_cycle = getattr(
        row,
        "cycle"
    )

    scaled_values, scaled_positions = (
        scaled_cache_623dc[
            unit_id
        ]
    )

    raw_values, raw_positions = (
        raw_cache_623dc[
            unit_id
        ]
    )

    end_pos_scaled = (
        scaled_positions[
            end_cycle
        ]
    )

    end_pos_raw = (
        raw_positions[
            end_cycle
        ]
    )

    start_pos_scaled = (
        end_pos_scaled
        - WINDOW_SIZE_623DC
        + 1
    )

    start_pos_raw = (
        end_pos_raw
        - WINDOW_SIZE_623DC
        + 1
    )

    assert start_pos_scaled >= 0
    assert start_pos_raw >= 0

    scaled_window = (
        scaled_values[
            start_pos_scaled:
            end_pos_scaled + 1
        ]
    )

    raw_window = (
        raw_values[
            start_pos_raw:
            end_pos_raw + 1
        ]
    )

    assert scaled_window.shape == (
        WINDOW_SIZE_623DC,
        len(PRIMARY_INPUT_COLUMNS)
    )

    assert raw_window.shape == (
        WINDOW_SIZE_623DC,
        len(PRIMARY_INPUT_COLUMNS)
    )

    scaled_windows_623dc.append(
        scaled_window
    )

    raw_windows_623dc.append(
        raw_window
    )


scaled_windows_623dc = np.stack(
    scaled_windows_623dc,
    axis=0
)

raw_windows_623dc = np.stack(
    raw_windows_623dc,
    axis=0
)


# ------------------------------------------------------------
# 7. Compare with canonical scaled development windows
# ------------------------------------------------------------

scaled_window_error_623dc = float(
    np.max(
        np.abs(
            scaled_windows_623dc
            -
            X_dev_reference_623dc
        )
    )
)


# ------------------------------------------------------------
# 8. Compare recovered raw windows with confirmed raw-flat source
# ------------------------------------------------------------

raw_window_flat_623dc = (
    raw_windows_623dc
    .reshape(
        -1,
        len(PRIMARY_INPUT_COLUMNS)
    )
)

assert (
    raw_window_flat_623dc.shape
    ==
    raw_flat_reference_623dc.shape
)

raw_window_error_623dc = float(
    np.max(
        np.abs(
            raw_window_flat_623dc
            -
            raw_flat_reference_623dc
        )
    )
)


# ------------------------------------------------------------
# 9. Integrity
# ------------------------------------------------------------

all_finite_623dc = bool(
    np.isfinite(
        raw_unique_features_623dc
    ).all()
    and
    np.isfinite(
        scaled_windows_623dc
    ).all()
    and
    np.isfinite(
        raw_windows_623dc
    ).all()
)

scaled_reproduction_ok_623dc = (
    scaled_window_error_623dc
    < 1e-8
)

raw_reproduction_ok_623dc = (
    raw_window_error_623dc
    < 1e-8
)


# ------------------------------------------------------------
# 10. Compact result
# ------------------------------------------------------------

print(
    "\n=== STEP 6.23D-C COMPACT RESULT ==="
)

print(
    "Unique raw rows                 :",
    len(DEV_RAW_BASE_623DC)
)

print(
    "Development engines             :",
    n_engines_623dc
)

print(
    "Reconstructed windows           :",
    scaled_windows_623dc.shape[0]
)

print(
    "Scaler inverse/forward error    :",
    unique_rescale_error_623dc
)

print(
    "Scaled-window reconstruction err:",
    scaled_window_error_623dc
)

print(
    "Raw-window reconstruction error :",
    raw_window_error_623dc
)

print(
    "Scaled reproduction exact       :",
    scaled_reproduction_ok_623dc
)

print(
    "Raw reproduction exact          :",
    raw_reproduction_ok_623dc
)

print(
    "All numerical values finite     :",
    all_finite_623dc
)

print(
    "\nCanonical in-memory raw base     : "
    "DEV_RAW_BASE_623DC"
)

print(
    "Corruption executed             : False"
)

print(
    "Official test used              : False"
)

print(
    "Reliability score selected      : False"
)

print(
    "Reliability threshold selected  : False"
)

print(
    "Abstention rule created         : False"
)

print(
    "Files written                   : False"
)

print(
    "Frozen artifacts modified       : False"
)

print(
    "=== END ==="
)

In [ ]:
# ============================================================
# Step 6.23D-D
# Recover canonical UNIQUE development rows directly from the
# already-verified raw overlapping-window representation.
#
# No corruption.
# No threshold selection.
# No official test.
# No files written.
# ============================================================

import numpy as np
import pandas as pd

print(
    "=== Step 6.23D-D — Exact unique-row recovery "
    "from canonical raw windows ==="
)

# ------------------------------------------------------------
# 1. Required objects
# ------------------------------------------------------------

required_623dd = [
    "X_dev_raw_flat_df_620d",
    "X_dev_clean",
    "dev_seq_meta",
    "dev_scaled",
    "scaler",
    "PRIMARY_INPUT_COLUMNS",
]

for name in required_623dd:
    assert name in globals(), f"Missing required object: {name}"

FEATURES_623DD = list(PRIMARY_INPUT_COLUMNS)
WINDOW_623DD = 30

meta_623dd = (
    dev_seq_meta[
        ["unit_id", "cycle"]
    ]
    .reset_index(drop=True)
    .copy()
)

raw_flat_623dd = (
    X_dev_raw_flat_df_620d[
        FEATURES_623DD
    ]
    .to_numpy(dtype=np.float64)
)

Xdev_623dd = np.asarray(
    X_dev_clean,
    dtype=np.float64
)

n_windows_623dd = len(meta_623dd)

assert raw_flat_623dd.shape == (
    n_windows_623dd * WINDOW_623DD,
    len(FEATURES_623DD),
)

# ------------------------------------------------------------
# 2. Recover engine/cycle identity for every flattened row
# ------------------------------------------------------------

unit_ids_623dd = np.repeat(
    meta_623dd["unit_id"].to_numpy(),
    WINDOW_623DD
)

end_cycles_623dd = np.repeat(
    meta_623dd["cycle"].to_numpy(),
    WINDOW_623DD
)

cycle_offsets_623dd = np.tile(
    np.arange(
        -(WINDOW_623DD - 1),
        1
    ),
    n_windows_623dd
)

cycles_623dd = (
    end_cycles_623dd
    +
    cycle_offsets_623dd
)

raw_overlap_623dd = pd.DataFrame(
    raw_flat_623dd,
    columns=FEATURES_623DD
)

raw_overlap_623dd.insert(
    0,
    "cycle",
    cycles_623dd
)

raw_overlap_623dd.insert(
    0,
    "unit_id",
    unit_ids_623dd
)

# ------------------------------------------------------------
# 3. Verify repeated overlapping observations are identical
# ------------------------------------------------------------

grouped_623dd = raw_overlap_623dd.groupby(
    ["unit_id", "cycle"],
    sort=True
)

group_min_623dd = grouped_623dd[
    FEATURES_623DD
].min()

group_max_623dd = grouped_623dd[
    FEATURES_623DD
].max()

spread_623dd = (
    group_max_623dd
    -
    group_min_623dd
)

max_overlap_spread_623dd = float(
    spread_623dd
    .to_numpy(dtype=np.float64)
    .max()
)

# ------------------------------------------------------------
# 4. Collapse overlapping rows into one canonical row/cycle
# ------------------------------------------------------------

unique_raw_623dd = (
    grouped_623dd[
        FEATURES_623DD
    ]
    .first()
    .reset_index()
)

unique_raw_623dd = (
    unique_raw_623dd
    .sort_values(
        ["unit_id", "cycle"],
        kind="mergesort"
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 5. Check exact key coverage against dev_scaled
# ------------------------------------------------------------

dev_keys_623dd = (
    dev_scaled[
        ["unit_id", "cycle"]
    ]
    .sort_values(
        ["unit_id", "cycle"],
        kind="mergesort"
    )
    .reset_index(drop=True)
)

raw_keys_623dd = (
    unique_raw_623dd[
        ["unit_id", "cycle"]
    ]
    .reset_index(drop=True)
)

key_coverage_exact_623dd = bool(
    len(dev_keys_623dd)
    ==
    len(raw_keys_623dd)
    and
    dev_keys_623dd.equals(
        raw_keys_623dd
    )
)

# ------------------------------------------------------------
# 6. Construct canonical unique raw development DataFrame
# ------------------------------------------------------------

dev_nonfeature_623dd = (
    dev_scaled
    .drop(
        columns=FEATURES_623DD
    )
    .copy()
)

CANONICAL_DEV_RAW_623DD = (
    dev_nonfeature_623dd
    .merge(
        unique_raw_623dd,
        on=["unit_id", "cycle"],
        how="inner",
        validate="one_to_one",
    )
    .sort_values(
        ["unit_id", "cycle"],
        kind="mergesort"
    )
    .reset_index(drop=True)
)

assert len(CANONICAL_DEV_RAW_623DD) == len(dev_scaled)

# ------------------------------------------------------------
# 7. Build raw engine caches
# ------------------------------------------------------------

raw_cache_623dd = {}

for unit_id, group in CANONICAL_DEV_RAW_623DD.groupby(
    "unit_id",
    sort=True
):

    group = group.sort_values(
        "cycle",
        kind="mergesort"
    )

    cycles = group[
        "cycle"
    ].to_numpy()

    values = group[
        FEATURES_623DD
    ].to_numpy(dtype=np.float64)

    positions = {
        cycle: idx
        for idx, cycle in enumerate(cycles)
    }

    raw_cache_623dd[unit_id] = (
        values,
        positions
    )

# ------------------------------------------------------------
# 8. Rebuild raw windows in exact metadata order
# ------------------------------------------------------------

rebuilt_raw_windows_623dd = []

for row in meta_623dd.itertuples(index=False):

    unit_id = row.unit_id
    end_cycle = row.cycle

    values, positions = raw_cache_623dd[
        unit_id
    ]

    end_pos = positions[
        end_cycle
    ]

    start_pos = (
        end_pos
        -
        WINDOW_623DD
        +
        1
    )

    assert start_pos >= 0

    window = values[
        start_pos:
        end_pos + 1
    ]

    assert window.shape == (
        WINDOW_623DD,
        len(FEATURES_623DD)
    )

    rebuilt_raw_windows_623dd.append(
        window
    )

rebuilt_raw_windows_623dd = np.stack(
    rebuilt_raw_windows_623dd,
    axis=0
)

rebuilt_raw_flat_623dd = (
    rebuilt_raw_windows_623dd
    .reshape(
        -1,
        len(FEATURES_623DD)
    )
)

raw_replay_error_623dd = float(
    np.max(
        np.abs(
            rebuilt_raw_flat_623dd
            -
            raw_flat_623dd
        )
    )
)

# ------------------------------------------------------------
# 9. Scale rebuilt raw windows and compare to X_dev_clean
# ------------------------------------------------------------

rebuilt_raw_df_623dd = pd.DataFrame(
    rebuilt_raw_flat_623dd,
    columns=FEATURES_623DD
)

rebuilt_scaled_flat_623dd = np.asarray(
    scaler.transform(
        rebuilt_raw_df_623dd
    ),
    dtype=np.float64
)

rebuilt_scaled_windows_623dd = (
    rebuilt_scaled_flat_623dd
    .reshape(
        n_windows_623dd,
        WINDOW_623DD,
        len(FEATURES_623DD)
    )
)

scaled_replay_error_623dd = float(
    np.max(
        np.abs(
            rebuilt_scaled_windows_623dd
            -
            Xdev_623dd
        )
    )
)

# ------------------------------------------------------------
# 10. Final integrity
# ------------------------------------------------------------

all_finite_623dd = bool(
    np.isfinite(
        rebuilt_raw_flat_623dd
    ).all()
    and
    np.isfinite(
        rebuilt_scaled_windows_623dd
    ).all()
)

# Raw overlapping observations should reproduce exactly or
# essentially exactly. Scaled comparison allows only tiny
# floating-point numerical error.
overlap_consistent_623dd = (
    max_overlap_spread_623dd
    <= 1e-12
)

raw_replay_ok_623dd = (
    raw_replay_error_623dd
    <= 1e-12
)

scaled_replay_ok_623dd = (
    scaled_replay_error_623dd
    <= 1e-8
)

canonical_ready_623dd = bool(
    key_coverage_exact_623dd
    and
    overlap_consistent_623dd
    and
    raw_replay_ok_623dd
    and
    scaled_replay_ok_623dd
    and
    all_finite_623dd
)

# ------------------------------------------------------------
# 11. Compact output only
# ------------------------------------------------------------

print("\n=== STEP 6.23D-D COMPACT RESULT ===")

print(
    "Flattened raw rows             :",
    len(raw_flat_623dd)
)

print(
    "Recovered unique rows          :",
    len(unique_raw_623dd)
)

print(
    "Expected unique rows           :",
    len(dev_scaled)
)

print(
    "Unique development engines     :",
    unique_raw_623dd["unit_id"].nunique()
)

print(
    "Exact engine/cycle key coverage:",
    key_coverage_exact_623dd
)

print(
    "Maximum overlap-value spread   :",
    max_overlap_spread_623dd
)

print(
    "Raw-window replay error        :",
    raw_replay_error_623dd
)

print(
    "Scaled-window replay error     :",
    scaled_replay_error_623dd
)

print(
    "All numerical values finite    :",
    all_finite_623dd
)

print(
    "Canonical unique raw base ready:",
    canonical_ready_623dd
)

print(
    "\nCanonical object:",
    "CANONICAL_DEV_RAW_623DD"
)

print(
    "\nCorruption executed            : False"
)

print(
    "Official test used             : False"
)

print(
    "Reliability score selected     : False"
)

print(
    "Reliability threshold selected : False"
)

print(
    "Abstention rule created        : False"
)

print(
    "Files written                  : False"
)

print(
    "Frozen artifacts modified      : False"
)

print("=== END ===")

In [ ]:
# ============================================================
# Step 6.23D-E
# Diagnose the only remaining failure:
# engine/cycle key equality.
#
# Inspection only.
# Nothing modified.
# ============================================================

import numpy as np
import pandas as pd

print("=== Step 6.23D-E — Engine/cycle key diagnostic ===")

# ------------------------------------------------------------
# 1. Recreate sorted key tables
# ------------------------------------------------------------

dev_keys_623de = (
    dev_scaled[["unit_id", "cycle"]]
    .sort_values(["unit_id", "cycle"], kind="mergesort")
    .reset_index(drop=True)
)

raw_keys_623de = (
    unique_raw_623dd[["unit_id", "cycle"]]
    .sort_values(["unit_id", "cycle"], kind="mergesort")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 2. Basic structure
# ------------------------------------------------------------

print("\n--- Structure ---")
print("DEV rows :", len(dev_keys_623de))
print("RAW rows :", len(raw_keys_623de))

print(
    "DEV dtypes:",
    {
        c: str(dev_keys_623de[c].dtype)
        for c in ["unit_id", "cycle"]
    }
)

print(
    "RAW dtypes:",
    {
        c: str(raw_keys_623de[c].dtype)
        for c in ["unit_id", "cycle"]
    }
)

# ------------------------------------------------------------
# 3. Numerical comparison
# ------------------------------------------------------------

dev_num_623de = dev_keys_623de.to_numpy(
    dtype=np.float64
)

raw_num_623de = raw_keys_623de.to_numpy(
    dtype=np.float64
)

max_numeric_difference_623de = float(
    np.max(
        np.abs(
            dev_num_623de
            -
            raw_num_623de
        )
    )
)

numeric_keys_equal_623de = bool(
    np.allclose(
        dev_num_623de,
        raw_num_623de,
        rtol=0.0,
        atol=0.0
    )
)

# ------------------------------------------------------------
# 4. Verify keys are truly integer-valued
# ------------------------------------------------------------

dev_integer_like_623de = bool(
    np.allclose(
        dev_num_623de,
        np.round(dev_num_623de),
        rtol=0.0,
        atol=1e-12
    )
)

raw_integer_like_623de = bool(
    np.allclose(
        raw_num_623de,
        np.round(raw_num_623de),
        rtol=0.0,
        atol=1e-12
    )
)

# ------------------------------------------------------------
# 5. Normalize only for comparison
# ------------------------------------------------------------

dev_int_623de = np.round(
    dev_num_623de
).astype(np.int64)

raw_int_623de = np.round(
    raw_num_623de
).astype(np.int64)

normalized_equal_623de = bool(
    np.array_equal(
        dev_int_623de,
        raw_int_623de
    )
)

row_mismatch_mask_623de = np.any(
    dev_int_623de
    !=
    raw_int_623de,
    axis=1
)

mismatch_count_623de = int(
    row_mismatch_mask_623de.sum()
)

# ------------------------------------------------------------
# 6. Duplicate-key check
# ------------------------------------------------------------

dev_duplicate_keys_623de = int(
    dev_keys_623de.duplicated(
        ["unit_id", "cycle"]
    ).sum()
)

raw_duplicate_keys_623de = int(
    raw_keys_623de.duplicated(
        ["unit_id", "cycle"]
    ).sum()
)

# ------------------------------------------------------------
# 7. Compact conclusion
# ------------------------------------------------------------

print("\n--- Exact diagnostic ---")

print(
    "Pandas DataFrame.equals       :",
    dev_keys_623de.equals(
        raw_keys_623de
    )
)

print(
    "Maximum numerical difference  :",
    max_numeric_difference_623de
)

print(
    "Numerically exactly equal      :",
    numeric_keys_equal_623de
)

print(
    "DEV keys integer-like          :",
    dev_integer_like_623de
)

print(
    "RAW keys integer-like          :",
    raw_integer_like_623de
)

print(
    "Equal after int64 normalization:",
    normalized_equal_623de
)

print(
    "Mismatched normalized rows     :",
    mismatch_count_623de
)

print(
    "DEV duplicate keys             :",
    dev_duplicate_keys_623de
)

print(
    "RAW duplicate keys             :",
    raw_duplicate_keys_623de
)

if mismatch_count_623de > 0:
    first_bad_623de = int(
        np.flatnonzero(
            row_mismatch_mask_623de
        )[0]
    )

    print(
        "First mismatch DEV            :",
        dev_int_623de[first_bad_623de].tolist()
    )

    print(
        "First mismatch RAW            :",
        raw_int_623de[first_bad_623de].tolist()
    )

print("\nNothing was modified.")
print("=== END ===")

In [ ]:
# ============================================================
# Step 6.23D-F
# Certify canonical DEVELOPMENT raw base after verified
# engine/cycle dtype normalization.
#
# No original object is modified.
# No file is written.
# Official test is NOT used.
# ============================================================

import numpy as np
import pandas as pd

print("=== Step 6.23D-F — Canonical DEV raw-base certification ===")

# ------------------------------------------------------------
# 1. Recover the verified source from Step 6.23D-D
# ------------------------------------------------------------

if "CANONICAL_DEV_RAW_623DD" in globals():
    source_623df = CANONICAL_DEV_RAW_623DD
    source_name_623df = "CANONICAL_DEV_RAW_623DD"
elif "unique_raw_623dd" in globals():
    source_623df = unique_raw_623dd
    source_name_623df = "unique_raw_623dd"
else:
    raise RuntimeError(
        "Verified Step 6.23D-D development raw source not found."
    )

required_key_cols_623df = ["unit_id", "cycle"]

missing_623df = [
    c for c in required_key_cols_623df
    if c not in source_623df.columns
]

if missing_623df:
    raise RuntimeError(
        f"Missing required key columns: {missing_623df}"
    )

# ------------------------------------------------------------
# 2. Work on a COPY only
# ------------------------------------------------------------

CANONICAL_DEV_RAW_623DF = source_623df.copy(deep=True)

# Verify that keys are integer-valued before conversion.
for col_623df in required_key_cols_623df:

    arr_623df = pd.to_numeric(
        CANONICAL_DEV_RAW_623DF[col_623df],
        errors="raise"
    ).to_numpy(dtype=np.float64)

    if not np.isfinite(arr_623df).all():
        raise RuntimeError(
            f"Non-finite values found in {col_623df}."
        )

    if not np.allclose(
        arr_623df,
        np.rint(arr_623df),
        rtol=0.0,
        atol=1e-12
    ):
        raise RuntimeError(
            f"{col_623df} contains non-integer-valued identifiers."
        )

    CANONICAL_DEV_RAW_623DF[col_623df] = (
        np.rint(arr_623df).astype(np.int64)
    )

# Canonical ordering.
CANONICAL_DEV_RAW_623DF = (
    CANONICAL_DEV_RAW_623DF
    .sort_values(
        ["unit_id", "cycle"],
        kind="mergesort"
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 3. Build normalized DEVELOPMENT reference keys
# ------------------------------------------------------------

dev_keys_623df = (
    dev_scaled[["unit_id", "cycle"]]
    .copy()
)

for col_623df in required_key_cols_623df:
    dev_keys_623df[col_623df] = (
        pd.to_numeric(
            dev_keys_623df[col_623df],
            errors="raise"
        )
        .round()
        .astype(np.int64)
    )

dev_keys_623df = (
    dev_keys_623df
    .sort_values(
        ["unit_id", "cycle"],
        kind="mergesort"
    )
    .reset_index(drop=True)
)

raw_keys_623df = (
    CANONICAL_DEV_RAW_623DF[
        ["unit_id", "cycle"]
    ]
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 4. Integrity checks
# ------------------------------------------------------------

key_equality_623df = bool(
    raw_keys_623df.equals(
        dev_keys_623df
    )
)

raw_duplicate_keys_623df = int(
    raw_keys_623df.duplicated(
        ["unit_id", "cycle"]
    ).sum()
)

dev_duplicate_keys_623df = int(
    dev_keys_623df.duplicated(
        ["unit_id", "cycle"]
    ).sum()
)

row_count_ok_623df = bool(
    len(raw_keys_623df)
    ==
    len(dev_keys_623df)
)

engine_count_ok_623df = bool(
    raw_keys_623df["unit_id"].nunique()
    ==
    dev_keys_623df["unit_id"].nunique()
)

feature_cols_available_623df = bool(
    all(
        c in CANONICAL_DEV_RAW_623DF.columns
        for c in PRIMARY_INPUT_COLUMNS
    )
)

if feature_cols_available_623df:
    feature_values_623df = (
        CANONICAL_DEV_RAW_623DF[
            PRIMARY_INPUT_COLUMNS
        ]
        .to_numpy(dtype=np.float64)
    )

    feature_values_finite_623df = bool(
        np.isfinite(
            feature_values_623df
        ).all()
    )
else:
    feature_values_finite_623df = False

CANONICAL_DEV_RAW_READY_623DF = bool(
    key_equality_623df
    and row_count_ok_623df
    and engine_count_ok_623df
    and raw_duplicate_keys_623df == 0
    and dev_duplicate_keys_623df == 0
    and feature_cols_available_623df
    and feature_values_finite_623df
)

# ------------------------------------------------------------
# 5. Compact evidence only
# ------------------------------------------------------------

print("\nSource object              :", source_name_623df)
print("Canonical rows             :", len(CANONICAL_DEV_RAW_623DF))
print(
    "Canonical engines          :",
    CANONICAL_DEV_RAW_623DF["unit_id"].nunique()
)
print(
    "unit_id dtype              :",
    CANONICAL_DEV_RAW_623DF["unit_id"].dtype
)
print(
    "cycle dtype                :",
    CANONICAL_DEV_RAW_623DF["cycle"].dtype
)
print("Exact key coverage         :", key_equality_623df)
print("RAW duplicate keys         :", raw_duplicate_keys_623df)
print("DEV duplicate keys         :", dev_duplicate_keys_623df)
print("Primary features available :", feature_cols_available_623df)
print("Primary features finite    :", feature_values_finite_623df)

print("\n--------------------------------------------")
print(
    "CANONICAL DEV RAW BASE READY :",
    CANONICAL_DEV_RAW_READY_623DF
)
print("--------------------------------------------")

print("Original source modified     : False")
print("Files written                : False")
print("Official test used           : False")
print("Reliability score selected   : False")
print("Abstention rule created      : False")
print("=== END ===")

In [ ]:
# ============================================================
# Step 6.23D-G
# Inspect frozen corruption replay base dependency.
#
# Inspection only.
# No corruption executed.
# Nothing modified.
# No files written.
# ============================================================

import inspect
import re

print("=== Step 6.23D-G — Corruption replay base-dependency check ===")

required_623dg = [
    "apply_frozen_corruption_620gc",
    "CANONICAL_DEV_RAW_623DF",
]

for name_623dg in required_623dg:
    print(
        f"{name_623dg:35s}:",
        name_623dg in globals()
    )

if not all(
    name_623dg in globals()
    for name_623dg in required_623dg
):
    raise RuntimeError(
        "Required objects are missing."
    )

# ------------------------------------------------------------
# 1. Exact function signature
# ------------------------------------------------------------

sig_623dg = inspect.signature(
    apply_frozen_corruption_620gc
)

print("\n--- Replay function ---")
print("Signature:", sig_623dg)

# ------------------------------------------------------------
# 2. Source inspection
# ------------------------------------------------------------

src_623dg = inspect.getsource(
    apply_frozen_corruption_620gc
)

possible_base_names_623dg = sorted(
    set(
        re.findall(
            r"\b[A-Za-z_][A-Za-z0-9_]*\b",
            src_623dg
        )
    )
)

base_like_623dg = [
    x for x in possible_base_names_623dg
    if any(
        token in x.lower()
        for token in [
            "raw_base",
            "val_raw",
            "dev_raw",
            "canonical_dev"
        ]
    )
]

print("\n--- Base-like names referenced in source ---")

if base_like_623dg:
    for x_623dg in base_like_623dg:
        print(x_623dg)
else:
    print("None detected by name scan.")

# ------------------------------------------------------------
# 3. Check direct references to known bases
# ------------------------------------------------------------

known_base_checks_623dg = {
    "val_raw_base_620gc":
        "val_raw_base_620gc" in src_623dg,

    "CANONICAL_DEV_RAW_623DF":
        "CANONICAL_DEV_RAW_623DF" in src_623dg,

    "CANONICAL_DEV_RAW_623DD":
        "CANONICAL_DEV_RAW_623DD" in src_623dg,
}

print("\n--- Known base references ---")

for k_623dg, v_623dg in known_base_checks_623dg.items():
    print(f"{k_623dg:28s}: {v_623dg}")

# ------------------------------------------------------------
# 4. Relevant source lines only
# ------------------------------------------------------------

print("\n--- Relevant source lines ---")

relevant_lines_623dg = []

for i_623dg, line_623dg in enumerate(
    src_623dg.splitlines(),
    start=1
):
    lower_623dg = line_623dg.lower()

    if any(
        token in lower_623dg
        for token in [
            "raw_base",
            "val_raw",
            "dev_raw",
            "copy(",
            "candidate_",
            "return "
        ]
    ):
        relevant_lines_623dg.append(
            (i_623dg, line_623dg)
        )

for i_623dg, line_623dg in relevant_lines_623dg[:40]:
    print(
        f"{i_623dg:03d}: {line_623dg}"
    )

# ------------------------------------------------------------
# 5. Compact conclusion
# ------------------------------------------------------------

validation_bound_623dg = bool(
    known_base_checks_623dg[
        "val_raw_base_620gc"
    ]
)

print("\n--------------------------------------")
print(
    "Replay function validation-bound :",
    validation_bound_623dg
)
print("--------------------------------------")

print("Corruption executed              : False")
print("Official test used               : False")
print("Frozen artifact modified         : False")
print("Reliability score selected       : False")
print("Abstention rule created          : False")
print("Files written                    : False")
print("=== END ===")

In [ ]:
# ============================================================
# Step 6.23D-H
# Safe DEVELOPMENT corruption replay wrapper +
# one-scenario-per-family smoke test
#
# Reuses the EXACT frozen Step-4 corruption function.
# Original validation replay function is NOT modified.
# Original validation raw base is restored after every call.
#
# No files written.
# Official test NOT used.
# No reliability score selected.
# No abstention rule created.
# ============================================================

import numpy as np
import pandas as pd

print(
    "=== Step 6.23D-H — Safe DEVELOPMENT replay smoke test ==="
)

# ------------------------------------------------------------
# 1. Required objects
# ------------------------------------------------------------

required_623dh = [
    "apply_frozen_corruption_620gc",
    "scenario_manifests_620gc",
    "CANONICAL_DEV_RAW_623DF",
    "PRIMARY_INPUT_COLUMNS",
]

for name_623dh in required_623dh:
    print(
        f"{name_623dh:35s}:",
        name_623dh in globals()
    )

if not all(
    name_623dh in globals()
    for name_623dh in required_623dh
):
    raise RuntimeError(
        "Required Step 6.23D-H objects are missing."
    )


# ------------------------------------------------------------
# 2. Exact DEVELOPMENT replay wrapper
# ------------------------------------------------------------

def apply_frozen_corruption_dev_623dh(
    family,
    scenario_row
):
    """
    Execute the existing frozen corruption implementation
    against the certified DEVELOPMENT raw base.

    The original validation base is restored in a finally
    block even if corruption execution fails.
    """

    frozen_function_623dh = (
        apply_frozen_corruption_620gc
    )

    fn_globals_623dh = (
        frozen_function_623dh.__globals__
    )

    if "val_raw_base_620gc" not in fn_globals_623dh:
        raise RuntimeError(
            "Expected validation raw-base global was not "
            "found inside the frozen corruption function."
        )

    original_val_base_623dh = (
        fn_globals_623dh["val_raw_base_620gc"]
    )

    # Give the frozen function its own DEVELOPMENT copy.
    development_input_623dh = (
        CANONICAL_DEV_RAW_623DF.copy(deep=True)
    )

    try:
        fn_globals_623dh[
            "val_raw_base_620gc"
        ] = development_input_623dh

        corrupted_623dh = (
            frozen_function_623dh(
                family,
                scenario_row
            )
        )

    finally:
        # Critical: always restore original validation base.
        fn_globals_623dh[
            "val_raw_base_620gc"
        ] = original_val_base_623dh

    if not isinstance(
        corrupted_623dh,
        pd.DataFrame
    ):
        raise RuntimeError(
            f"{family}: replay did not return a DataFrame."
        )

    return corrupted_623dh


# ------------------------------------------------------------
# 3. Reference DEVELOPMENT structure
# ------------------------------------------------------------

clean_dev_623dh = (
    CANONICAL_DEV_RAW_623DF
    .copy(deep=True)
    .reset_index(drop=True)
)

reference_keys_623dh = (
    clean_dev_623dh[
        ["unit_id", "cycle"]
    ]
    .astype(np.int64)
    .reset_index(drop=True)
)

clean_features_623dh = (
    clean_dev_623dh[
        PRIMARY_INPUT_COLUMNS
    ]
    .to_numpy(dtype=np.float64)
)


# ------------------------------------------------------------
# 4. Smoke test: first frozen scenario from each family
# ------------------------------------------------------------

families_623dh = [
    "bias",
    "noise",
    "drift",
    "dropout",
    "stuck",
]

smoke_rows_623dh = []

print("\n--- Executing five-family DEVELOPMENT smoke test ---")

for family_623dh in families_623dh:

    manifest_623dh = (
        scenario_manifests_620gc[
            family_623dh
        ]
    )

    if len(manifest_623dh) == 0:
        raise RuntimeError(
            f"{family_623dh}: empty scenario manifest."
        )

    scenario_row_623dh = (
        manifest_623dh.iloc[0].copy()
    )

    corrupted_623dh = (
        apply_frozen_corruption_dev_623dh(
            family_623dh,
            scenario_row_623dh
        )
    )

    corrupted_623dh = (
        corrupted_623dh.reset_index(drop=True)
    )

    # ------------------------
    # Row count
    # ------------------------

    row_count_ok_623dh = bool(
        len(corrupted_623dh)
        ==
        len(clean_dev_623dh)
    )

    # ------------------------
    # Engine/cycle integrity
    # ------------------------

    if not all(
        col_623dh in corrupted_623dh.columns
        for col_623dh in ["unit_id", "cycle"]
    ):
        raise RuntimeError(
            f"{family_623dh}: missing engine/cycle keys."
        )

    corrupt_keys_623dh = (
        corrupted_623dh[
            ["unit_id", "cycle"]
        ]
        .copy()
    )

    for col_623dh in ["unit_id", "cycle"]:
        corrupt_keys_623dh[
            col_623dh
        ] = (
            pd.to_numeric(
                corrupt_keys_623dh[
                    col_623dh
                ],
                errors="raise"
            )
            .round()
            .astype(np.int64)
        )

    corrupt_keys_623dh = (
        corrupt_keys_623dh.reset_index(
            drop=True
        )
    )

    key_alignment_ok_623dh = bool(
        corrupt_keys_623dh.equals(
            reference_keys_623dh
        )
    )

    duplicate_keys_623dh = int(
        corrupt_keys_623dh.duplicated(
            ["unit_id", "cycle"]
        ).sum()
    )

    # ------------------------
    # Primary features
    # ------------------------

    missing_features_623dh = [
        col_623dh
        for col_623dh in PRIMARY_INPUT_COLUMNS
        if col_623dh not in corrupted_623dh.columns
    ]

    if missing_features_623dh:
        raise RuntimeError(
            f"{family_623dh}: missing features: "
            f"{missing_features_623dh}"
        )

    corrupt_features_623dh = (
        corrupted_623dh[
            PRIMARY_INPUT_COLUMNS
        ]
        .to_numpy(dtype=np.float64)
    )

    finite_623dh = bool(
        np.isfinite(
            corrupt_features_623dh
        ).all()
    )

    max_feature_change_623dh = float(
        np.max(
            np.abs(
                corrupt_features_623dh
                -
                clean_features_623dh
            )
        )
    )

    changed_623dh = bool(
        max_feature_change_623dh > 0.0
    )

    family_pass_623dh = bool(
        row_count_ok_623dh
        and key_alignment_ok_623dh
        and duplicate_keys_623dh == 0
        and finite_623dh
        and changed_623dh
    )

    smoke_rows_623dh.append(
        {
            "family": family_623dh,
            "rows": len(corrupted_623dh),
            "row_count_ok": row_count_ok_623dh,
            "key_alignment_ok": key_alignment_ok_623dh,
            "duplicate_keys": duplicate_keys_623dh,
            "finite": finite_623dh,
            "max_feature_change":
                max_feature_change_623dh,
            "changed": changed_623dh,
            "PASS": family_pass_623dh,
        }
    )

    print(
        f"{family_623dh:8s} : "
        f"{'PASS' if family_pass_623dh else 'FAIL'}"
    )


# ------------------------------------------------------------
# 5. Confirm validation base restoration
# ------------------------------------------------------------

fn_globals_after_623dh = (
    apply_frozen_corruption_620gc.__globals__
)

validation_base_restored_623dh = bool(
    fn_globals_after_623dh[
        "val_raw_base_620gc"
    ]
    is val_raw_base_620gc
)


# ------------------------------------------------------------
# 6. Compact summary
# ------------------------------------------------------------

smoke_summary_623dh = pd.DataFrame(
    smoke_rows_623dh
)

print("\n--- Compact DEVELOPMENT replay summary ---")

display(
    smoke_summary_623dh[
        [
            "family",
            "rows",
            "key_alignment_ok",
            "duplicate_keys",
            "finite",
            "changed",
            "PASS",
        ]
    ]
)

all_families_pass_623dh = bool(
    smoke_summary_623dh[
        "PASS"
    ].all()
)

DEV_REPLAY_READY_623DH = bool(
    all_families_pass_623dh
    and validation_base_restored_623dh
)

print("\n==========================================")
print("STEP 6.23D-H READINESS SUMMARY")
print("==========================================")
print(
    "Development raw rows       :",
    len(clean_dev_623dh)
)
print(
    "Development engines        :",
    clean_dev_623dh[
        "unit_id"
    ].nunique()
)
print(
    "Families smoke-tested      :",
    len(families_623dh)
)
print(
    "All families passed        :",
    all_families_pass_623dh
)
print(
    "Validation base restored   :",
    validation_base_restored_623dh
)
print(
    "DEV replay ready           :",
    DEV_REPLAY_READY_623DH
)

print("\nOfficial test used          : False")
print("Reliability score selected : False")
print("Threshold selected         : False")
print("Abstention rule created    : False")
print("Files written              : False")
print("Original function modified : False")
print("=== END ===")

In [ ]:
# ============================================================
# Step 6.23D-I0
# Compact end-to-end DEVELOPMENT interface check
#
# Inspection only.
# No corruption executed.
# No prediction executed.
# No files written.
# ============================================================

import inspect
import numpy as np
import pandas as pd

print("=== STEP 6.23D-I0 — COMPACT INTERFACE CHECK ===")

# ------------------------------------------------------------
# 1. Core helper signatures
# ------------------------------------------------------------

helper_names_623di0 = [
    "build_sequences_623cb2",
    "pca_raw_quality_623cb2",
    "temporal_raw_quality_623cb2",
    "mc_uncertainty_623cb2",
    "midrank_percentile_623cb2",
    "frozen_channel_623cb2",
    "predict_gru_batched",
]

print("\n--- Helper signatures ---")

for name_623di0 in helper_names_623di0:

    obj_623di0 = globals().get(
        name_623di0,
        None
    )

    if obj_623di0 is None:
        print(
            f"{name_623di0:30s}: MISSING"
        )

    elif callable(obj_623di0):
        try:
            sig_623di0 = inspect.signature(
                obj_623di0
            )
        except Exception:
            sig_623di0 = "<signature unavailable>"

        print(
            f"{name_623di0:30s}: "
            f"{sig_623di0}"
        )

    else:
        print(
            f"{name_623di0:30s}: "
            f"NOT CALLABLE"
        )


# ------------------------------------------------------------
# 2. Development target structure
# ------------------------------------------------------------

print("\n--- Development target structure ---")

if "dev_seq_meta" not in globals():
    print("dev_seq_meta: MISSING")

else:

    print(
        "dev_seq_meta shape:",
        dev_seq_meta.shape
    )

    target_cols_623di0 = [
        col_623di0
        for col_623di0 in [
            "RUL_raw",
            "RUL_cap_125",
            "RUL_cap_130",
            "target",
        ]
        if col_623di0
        in dev_seq_meta.columns
    ]

    print(
        "Available target columns:",
        target_cols_623di0
    )


# ------------------------------------------------------------
# 3. Development reliability candidate structure
# ------------------------------------------------------------

print("\n--- Reliability candidates ---")

for name_623di0 in [
    "RELIABILITY_DEV_CHANNELS_623A",
    "DEV_RELIABILITY_CANDIDATES_623B",
    "RELIABILITY_THRESHOLDS_623B",
]:

    obj_623di0 = globals().get(
        name_623di0,
        None
    )

    if obj_623di0 is None:
        print(
            f"{name_623di0}: MISSING"
        )

    elif isinstance(
        obj_623di0,
        pd.DataFrame
    ):
        print(
            f"{name_623di0}: "
            f"shape={obj_623di0.shape}, "
            f"columns={list(obj_623di0.columns)}"
        )

    else:
        print(
            f"{name_623di0}: "
            f"type={type(obj_623di0).__name__}"
        )


# ------------------------------------------------------------
# 4. Development calibration references
# ------------------------------------------------------------

print("\n--- Calibration references ---")

for name_623di0 in [
    "pca_dev_reference_623cb2",
    "temporal_dev_reference_623cb2",
    "phase_references_622h",
]:

    obj_623di0 = globals().get(
        name_623di0,
        None
    )

    if obj_623di0 is None:
        print(
            f"{name_623di0}: MISSING"
        )

    else:
        try:
            length_623di0 = len(
                obj_623di0
            )
        except Exception:
            length_623di0 = "NA"

        print(
            f"{name_623di0}: "
            f"type={type(obj_623di0).__name__}, "
            f"len={length_623di0}"
        )


# ------------------------------------------------------------
# 5. Core development MC objects
# ------------------------------------------------------------

print("\n--- Development uncertainty objects ---")

for name_623di0 in [
    "dev_mc_mean_prediction_622fd",
    "dev_mc_std_622fd",
]:

    obj_623di0 = globals().get(
        name_623di0,
        None
    )

    if obj_623di0 is None:
        print(
            f"{name_623di0}: MISSING"
        )

    else:
        print(
            f"{name_623di0}: "
            f"shape={np.asarray(obj_623di0).shape}"
        )


print("\nCorruption executed       : False")
print("Official test used        : False")
print("Score selected            : False")
print("Threshold selected        : False")
print("Abstention rule created   : False")
print("Files written             : False")
print("=== END ===")

In [ ]:
# ============================================================
# Step 6.23D-I1
# Exact CLEAN DEVELOPMENT reliability-channel reproduction
#
# Purpose:
#   Re-run the clean development pathway through the current
#   helper functions and compare against the already-established
#   Step 6.23A development reliability channels.
#
# No corruption.
# No threshold selection.
# No abstention.
# No files written.
# ============================================================

import numpy as np
import pandas as pd

print(
    "=== Step 6.23D-I1 — CLEAN DEV pathway reproduction ==="
)

# ------------------------------------------------------------
# 1. Required objects
# ------------------------------------------------------------

required_623di1 = [
    "CANONICAL_DEV_RAW_623DF",
    "build_sequences_623cb2",
    "pca_raw_quality_623cb2",
    "temporal_raw_quality_623cb2",
    "mc_uncertainty_623cb2",
    "midrank_percentile_623cb2",
    "pca_dev_reference_623cb2",
    "temporal_dev_reference_623cb2",
    "phase_references_622h",
    "RELIABILITY_DEV_CHANNELS_623A",
    "frozen_gru",
    "predict_gru_batched",
]

for name_623di1 in required_623di1:
    print(
        f"{name_623di1:35s}:",
        name_623di1 in globals()
    )

if not all(
    name_623di1 in globals()
    for name_623di1 in required_623di1
):
    raise RuntimeError(
        "One or more required objects are missing."
    )


# ------------------------------------------------------------
# 2. Rebuild CLEAN development sequences
# ------------------------------------------------------------

built_623di1 = build_sequences_623cb2(
    CANONICAL_DEV_RAW_623DF
)

print("\n--- build_sequences_623cb2 return ---")
print("Type:", type(built_623di1).__name__)

if isinstance(built_623di1, tuple):

    print("Returned items:", len(built_623di1))

    for i_623di1, obj_623di1 in enumerate(
        built_623di1
    ):
        try:
            shape_623di1 = np.asarray(
                obj_623di1
            ).shape
        except Exception:
            shape_623di1 = None

        print(
            f"item_{i_623di1}: "
            f"type={type(obj_623di1).__name__}, "
            f"shape={shape_623di1}"
        )

else:
    print(
        "Shape:",
        np.asarray(built_623di1).shape
    )


# ------------------------------------------------------------
# 3. Identify sequence tensor
# ------------------------------------------------------------

sequence_candidates_623di1 = []

if isinstance(built_623di1, tuple):
    iterable_623di1 = built_623di1
else:
    iterable_623di1 = [built_623di1]

for i_623di1, obj_623di1 in enumerate(
    iterable_623di1
):

    try:
        arr_623di1 = np.asarray(
            obj_623di1
        )
    except Exception:
        continue

    if (
        arr_623di1.ndim == 3
        and arr_623di1.shape[0] == 14241
        and arr_623di1.shape[1] == 30
    ):
        sequence_candidates_623di1.append(
            (i_623di1, arr_623di1)
        )

if len(sequence_candidates_623di1) != 1:
    raise RuntimeError(
        "Could not uniquely identify the clean "
        "development sequence tensor."
    )

sequence_item_index_623di1 = (
    sequence_candidates_623di1[0][0]
)

X_dev_rebuilt_623di1 = (
    sequence_candidates_623di1[0][1]
)

print(
    "\nIdentified sequence item :",
    sequence_item_index_623di1
)

print(
    "Rebuilt sequence shape   :",
    X_dev_rebuilt_623di1.shape
)


# ------------------------------------------------------------
# 4. Exact sequence reproduction check
# ------------------------------------------------------------

if "X_dev_clean" in globals():

    xdev_error_623di1 = float(
        np.max(
            np.abs(
                np.asarray(
                    X_dev_rebuilt_623di1,
                    dtype=np.float64
                )
                -
                np.asarray(
                    X_dev_clean,
                    dtype=np.float64
                )
            )
        )
    )

    print(
        "Max sequence difference :",
        xdev_error_623di1
    )

else:

    xdev_error_623di1 = np.nan

    print(
        "X_dev_clean not found; "
        "direct sequence comparison skipped."
    )


# ------------------------------------------------------------
# 5. Recompute raw PCA / temporal / uncertainty
# ------------------------------------------------------------

pca_raw_623di1 = np.asarray(
    pca_raw_quality_623cb2(
        X_dev_rebuilt_623di1
    ),
    dtype=np.float64
)

temporal_raw_623di1 = np.asarray(
    temporal_raw_quality_623cb2(
        X_dev_rebuilt_623di1
    ),
    dtype=np.float64
)

uncertainty_raw_623di1 = np.asarray(
    mc_uncertainty_623cb2(
        X_dev_rebuilt_623di1
    ),
    dtype=np.float64
)

print("\n--- Raw channel shapes ---")
print(
    "PCA         :", pca_raw_623di1.shape
)
print(
    "Temporal    :", temporal_raw_623di1.shape
)
print(
    "Uncertainty :", uncertainty_raw_623di1.shape
)

if not (
    pca_raw_623di1.shape == (14241,)
    and temporal_raw_623di1.shape == (14241,)
    and uncertainty_raw_623di1.shape == (14241,)
):
    raise RuntimeError(
        "Unexpected clean development raw-channel shape."
    )

if not (
    np.isfinite(pca_raw_623di1).all()
    and np.isfinite(temporal_raw_623di1).all()
    and np.isfinite(uncertainty_raw_623di1).all()
):
    raise RuntimeError(
        "Non-finite clean development channel detected."
    )


# ------------------------------------------------------------
# 6. Convert PCA and temporal signals using development
#    empirical references.
# ------------------------------------------------------------

pca_pct_623di1 = np.asarray(
    midrank_percentile_623cb2(
        pca_raw_623di1,
        pca_dev_reference_623cb2
    ),
    dtype=np.float64
)

temporal_pct_623di1 = np.asarray(
    midrank_percentile_623cb2(
        temporal_raw_623di1,
        temporal_dev_reference_623cb2
    ),
    dtype=np.float64
)


# ------------------------------------------------------------
# 7. Recover exact frozen predictive-uncertainty channel
#
# For the clean development set we already froze this channel
# in Step 6.22I-F. Use it only as the canonical comparison.
# ------------------------------------------------------------

if "FROZEN_DEV_UNCERTAINTY_622IF" in globals():

    uncertainty_pct_623di1 = np.asarray(
        FROZEN_DEV_UNCERTAINTY_622IF,
        dtype=np.float64
    )

elif "RELIABILITY_DEV_CHANNELS_623A" in globals():

    uncertainty_pct_623di1 = (
        RELIABILITY_DEV_CHANNELS_623A[
            "predictive_uncertainty"
        ]
        .to_numpy(dtype=np.float64)
    )

else:
    raise RuntimeError(
        "Frozen development uncertainty channel unavailable."
    )


# ------------------------------------------------------------
# 8. Reconstructed channel table
# ------------------------------------------------------------

reconstructed_channels_623di1 = pd.DataFrame(
    {
        "pca_quality":
            pca_pct_623di1,
        "temporal_quality":
            temporal_pct_623di1,
        "predictive_uncertainty":
            uncertainty_pct_623di1,
    }
)

reference_channels_623di1 = (
    RELIABILITY_DEV_CHANNELS_623A[
        [
            "pca_quality",
            "temporal_quality",
            "predictive_uncertainty",
        ]
    ]
    .reset_index(drop=True)
    .astype(np.float64)
)


# ------------------------------------------------------------
# 9. Exact / near-exact reproduction errors
# ------------------------------------------------------------

comparison_rows_623di1 = []

for col_623di1 in [
    "pca_quality",
    "temporal_quality",
    "predictive_uncertainty",
]:

    reconstructed_623di1 = (
        reconstructed_channels_623di1[
            col_623di1
        ].to_numpy(dtype=np.float64)
    )

    reference_623di1 = (
        reference_channels_623di1[
            col_623di1
        ].to_numpy(dtype=np.float64)
    )

    diff_623di1 = np.abs(
        reconstructed_623di1
        -
        reference_623di1
    )

    comparison_rows_623di1.append(
        {
            "channel": col_623di1,
            "mean_absolute_error":
                float(diff_623di1.mean()),
            "maximum_absolute_error":
                float(diff_623di1.max()),
            "exact_allclose":
                bool(
                    np.allclose(
                        reconstructed_623di1,
                        reference_623di1,
                        rtol=0.0,
                        atol=1e-12
                    )
                ),
        }
    )

comparison_623di1 = pd.DataFrame(
    comparison_rows_623di1
)

print(
    "\n--- CLEAN development channel reproduction ---"
)

display(comparison_623di1)


# ------------------------------------------------------------
# 10. Overall readiness
# ------------------------------------------------------------

channels_exact_623di1 = bool(
    comparison_623di1[
        "exact_allclose"
    ].all()
)

if np.isfinite(xdev_error_623di1):
    sequence_ok_623di1 = bool(
        xdev_error_623di1 <= 1e-10
    )
else:
    sequence_ok_623di1 = True

CLEAN_DEV_PATHWAY_READY_623DI1 = bool(
    channels_exact_623di1
    and sequence_ok_623di1
)

print("\n========================================")
print("STEP 6.23D-I1 READINESS SUMMARY")
print("========================================")

print(
    "Development windows          :",
    len(reconstructed_channels_623di1)
)

print(
    "Sequence reproduction OK     :",
    sequence_ok_623di1
)

print(
    "All channel reproductions OK :",
    channels_exact_623di1
)

print(
    "CLEAN DEV pathway ready      :",
    CLEAN_DEV_PATHWAY_READY_623DI1
)

print("\nCorruption executed           : False")
print("Official test used            : False")
print("Reliability score selected    : False")
print("Threshold selected            : False")
print("Abstention rule created       : False")
print("Files written                 : False")
print("Frozen artifacts modified     : False")
print("=== END ===")

In [ ]:
# ============================================================
# Step 6.23D-I1A
# Identify exact mc_uncertainty_623cb2 return components
#
# Diagnostic only.
# Nothing selected or modified.
# ============================================================

import numpy as np
import inspect

print("=== STEP 6.23D-I1A — MC HELPER RETURN IDENTIFICATION ===")

# ------------------------------------------------------------
# 1. Existing helper output
# ------------------------------------------------------------

u_623di1a = np.asarray(
    uncertainty_raw_623di1,
    dtype=np.float64
)

print("Stored helper-output shape:", u_623di1a.shape)

if u_623di1a.shape != (3, 14241):
    raise RuntimeError(
        "Unexpected helper-output structure."
    )

# ------------------------------------------------------------
# 2. Compact statistics for the three rows
# ------------------------------------------------------------

print("\n--- Component statistics ---")

for i_623di1a in range(3):
    x_623di1a = u_623di1a[i_623di1a]

    print(
        f"row_{i_623di1a}: "
        f"min={x_623di1a.min():.6f} | "
        f"mean={x_623di1a.mean():.6f} | "
        f"p95={np.percentile(x_623di1a,95):.6f} | "
        f"max={x_623di1a.max():.6f}"
    )

# ------------------------------------------------------------
# 3. Compare against known canonical DEV uncertainty objects
# ------------------------------------------------------------

references_623di1a = {}

for name_623di1a in [
    "dev_mc_mean_prediction_622fd",
    "dev_mc_std_622fd",
    "FROZEN_DEV_UNCERTAINTY_622IF",
]:

    if name_623di1a in globals():

        arr_623di1a = np.asarray(
            globals()[name_623di1a],
            dtype=np.float64
        ).reshape(-1)

        if arr_623di1a.shape == (14241,):
            references_623di1a[
                name_623di1a
            ] = arr_623di1a


if "RELIABILITY_DEV_CHANNELS_623A" in globals():

    references_623di1a[
        "RELIABILITY_DEV_CHANNELS_623A.predictive_uncertainty"
    ] = (
        RELIABILITY_DEV_CHANNELS_623A[
            "predictive_uncertainty"
        ]
        .to_numpy(dtype=np.float64)
    )


print("\n--- Maximum absolute differences ---")

best_matches_623di1a = {}

for i_623di1a in range(3):

    row_623di1a = u_623di1a[
        i_623di1a
    ]

    errors_623di1a = {}

    for ref_name_623di1a, ref_623di1a in (
        references_623di1a.items()
    ):

        errors_623di1a[
            ref_name_623di1a
        ] = float(
            np.max(
                np.abs(
                    row_623di1a
                    -
                    ref_623di1a
                )
            )
        )

    best_name_623di1a = min(
        errors_623di1a,
        key=errors_623di1a.get
    )

    best_error_623di1a = (
        errors_623di1a[
            best_name_623di1a
        ]
    )

    best_matches_623di1a[
        i_623di1a
    ] = (
        best_name_623di1a,
        best_error_623di1a
    )

    print(
        f"row_{i_623di1a} best match -> "
        f"{best_name_623di1a} | "
        f"max_err={best_error_623di1a:.12g}"
    )


# ------------------------------------------------------------
# 4. Show only source lines near RETURN
# ------------------------------------------------------------

print("\n--- Helper return source ---")

src_lines_623di1a = inspect.getsource(
    mc_uncertainty_623cb2
).splitlines()

return_indices_623di1a = [
    i_623di1a
    for i_623di1a, line_623di1a
    in enumerate(src_lines_623di1a)
    if "return" in line_623di1a
]

shown_623di1a = set()

for idx_623di1a in return_indices_623di1a:

    start_623di1a = max(
        0,
        idx_623di1a - 2
    )

    end_623di1a = min(
        len(src_lines_623di1a),
        idx_623di1a + 4
    )

    for j_623di1a in range(
        start_623di1a,
        end_623di1a
    ):
        if j_623di1a not in shown_623di1a:

            print(
                f"{j_623di1a + 1:03d}: "
                f"{src_lines_623di1a[j_623di1a]}"
            )

            shown_623di1a.add(
                j_623di1a
            )


print("\nOfficial test used      : False")
print("Signal selected         : False")
print("Threshold selected      : False")
print("Abstention created      : False")
print("Nothing was modified.")
print("=== END ===")

In [ ]:
# ============================================================
# Step 6.23D-I2
# Correct CLEAN DEVELOPMENT reliability-channel reproduction
#
# Uses the exact frozen predictive-uncertainty recipe:
#
# deterministic prediction
#       -> development-frozen 5 phase bins
# MC-dropout std
#       -> empirical within-phase mid-rank percentile
#
# No corruption.
# No score selection.
# No threshold selection.
# No abstention.
# No files written.
# ============================================================

import numpy as np
import pandas as pd

print(
    "=== Step 6.23D-I2 — Correct CLEAN DEV pathway reproduction ==="
)

# ------------------------------------------------------------
# 1. Required objects
# ------------------------------------------------------------

required_623di2 = [
    "CANONICAL_DEV_RAW_623DF",
    "build_sequences_623cb2",
    "pca_raw_quality_623cb2",
    "temporal_raw_quality_623cb2",
    "mc_uncertainty_623cb2",
    "pca_dev_reference_623cb2",
    "temporal_dev_reference_623cb2",
    "midrank_percentile_623cb2",
    "phase_references_622h",
    "FROZEN_DEV_UNCERTAINTY_622IF",
    "RELIABILITY_DEV_CHANNELS_623A",
]

for name_623di2 in required_623di2:
    print(
        f"{name_623di2:35s}:",
        name_623di2 in globals()
    )

if not all(
    name_623di2 in globals()
    for name_623di2 in required_623di2
):
    raise RuntimeError(
        "Missing required Step 6.23D-I2 objects."
    )


# ------------------------------------------------------------
# 2. Rebuild exact clean DEV sequences
# ------------------------------------------------------------

X_dev_623di2 = np.asarray(
    build_sequences_623cb2(
        CANONICAL_DEV_RAW_623DF
    ),
    dtype=np.float64
)

print("\n--- Sequence reproduction ---")
print(
    "Rebuilt shape:",
    X_dev_623di2.shape
)

if X_dev_623di2.shape != (14241, 30, 17):
    raise RuntimeError(
        "Unexpected development sequence shape."
    )

if "X_dev_clean" in globals():

    sequence_error_623di2 = float(
        np.max(
            np.abs(
                X_dev_623di2
                -
                np.asarray(
                    X_dev_clean,
                    dtype=np.float64
                )
            )
        )
    )

else:

    sequence_error_623di2 = np.nan

print(
    "Maximum difference vs X_dev_clean:",
    sequence_error_623di2
)


# ------------------------------------------------------------
# 3. PCA and temporal raw channels
# ------------------------------------------------------------

pca_raw_623di2 = np.asarray(
    pca_raw_quality_623cb2(
        X_dev_623di2
    ),
    dtype=np.float64
).reshape(-1)

temporal_raw_623di2 = np.asarray(
    temporal_raw_quality_623cb2(
        X_dev_623di2
    ),
    dtype=np.float64
).reshape(-1)


# ------------------------------------------------------------
# 4. MC helper — unpack correctly
# ------------------------------------------------------------

mc_return_623di2 = (
    mc_uncertainty_623cb2(
        X_dev_623di2
    )
)

if (
    not isinstance(
        mc_return_623di2,
        (tuple, list)
    )
    or len(mc_return_623di2) != 3
):
    raise RuntimeError(
        "Unexpected mc_uncertainty_623cb2 return structure."
    )

det_prediction_623di2 = np.asarray(
    mc_return_623di2[0],
    dtype=np.float64
).reshape(-1)

mc_std_raw_623di2 = np.asarray(
    mc_return_623di2[1],
    dtype=np.float64
).reshape(-1)

helper_adjusted_623di2 = np.asarray(
    mc_return_623di2[2],
    dtype=np.float64
).reshape(-1)

print("\n--- Correct MC component interpretation ---")
print(
    "deterministic prediction:",
    det_prediction_623di2.shape
)
print(
    "raw MC std              :",
    mc_std_raw_623di2.shape
)
print(
    "helper adjusted output  :",
    helper_adjusted_623di2.shape
)


# ------------------------------------------------------------
# 5. Convert PCA + temporal using frozen DEV references
# ------------------------------------------------------------

pca_pct_623di2 = np.asarray(
    midrank_percentile_623cb2(
        pca_raw_623di2,
        pca_dev_reference_623cb2
    ),
    dtype=np.float64
).reshape(-1)

temporal_pct_623di2 = np.asarray(
    midrank_percentile_623cb2(
        temporal_raw_623di2,
        temporal_dev_reference_623cb2
    ),
    dtype=np.float64
).reshape(-1)


# ------------------------------------------------------------
# 6. Recover exact frozen predictive-uncertainty recipe
# ------------------------------------------------------------

SELECTED_UNCERTAINTY_623DI2 = (
    "det_pred_pct_b5"
)

if (
    SELECTED_UNCERTAINTY_623DI2
    not in phase_references_622h
):
    raise RuntimeError(
        "Frozen uncertainty phase reference not found."
    )

phase_ref_623di2 = (
    phase_references_622h[
        SELECTED_UNCERTAINTY_623DI2
    ]
)

edges_623di2 = np.asarray(
    phase_ref_623di2["edges"],
    dtype=np.float64
)

references_623di2 = (
    phase_ref_623di2[
        "references"
    ]
)

actual_bins_623di2 = int(
    phase_ref_623di2[
        "actual_bins"
    ]
)

print("\n--- Frozen uncertainty recipe ---")
print(
    "Candidate   :",
    SELECTED_UNCERTAINTY_623DI2
)
print(
    "Phase source:",
    phase_ref_623di2[
        "phase_source"
    ]
)
print(
    "Phase bins  :",
    actual_bins_623di2
)
print(
    "Edges       :",
    edges_623di2.tolist()
)
print(
    "Boundary    : np.digitize(..., right=True)"
)


# ------------------------------------------------------------
# 7. Exact empirical mid-rank / N helper
#
# For each new value:
#
#     (# reference values below x
#      + 0.5 * # reference values equal x) / N
#
# This is the convention previously recovered and frozen.
# ------------------------------------------------------------

def empirical_midrank_over_n_623di2(
    values,
    reference
):

    values = np.asarray(
        values,
        dtype=np.float64
    )

    reference = np.sort(
        np.asarray(
            reference,
            dtype=np.float64
        )
    )

    if len(reference) == 0:
        raise RuntimeError(
            "Empty phase reference encountered."
        )

    left = np.searchsorted(
        reference,
        values,
        side="left"
    )

    right = np.searchsorted(
        reference,
        values,
        side="right"
    )

    return (
        0.5
        * (
            left.astype(np.float64)
            +
            right.astype(np.float64)
        )
        / float(len(reference))
    )


# ------------------------------------------------------------
# 8. Assign deterministic predictions to frozen phase bins
# ------------------------------------------------------------

phase_ids_623di2 = np.digitize(
    det_prediction_623di2,
    edges_623di2,
    right=True
)

if not np.all(
    np.isin(
        phase_ids_623di2,
        np.arange(actual_bins_623di2)
    )
):
    raise RuntimeError(
        "Unexpected phase ID generated."
    )


# ------------------------------------------------------------
# 9. Reconstruct predictive-uncertainty channel
# ------------------------------------------------------------

uncertainty_pct_623di2 = np.empty(
    len(mc_std_raw_623di2),
    dtype=np.float64
)

phase_counts_623di2 = {}

for phase_id_623di2 in range(
    actual_bins_623di2
):

    mask_623di2 = (
        phase_ids_623di2
        ==
        phase_id_623di2
    )

    phase_counts_623di2[
        phase_id_623di2
    ] = int(
        mask_623di2.sum()
    )

    if not np.any(mask_623di2):
        continue

    if phase_id_623di2 not in references_623di2:
        raise RuntimeError(
            f"Missing frozen reference for phase "
            f"{phase_id_623di2}."
        )

    uncertainty_pct_623di2[
        mask_623di2
    ] = (
        empirical_midrank_over_n_623di2(
            mc_std_raw_623di2[
                mask_623di2
            ],
            references_623di2[
                phase_id_623di2
            ]
        )
    )

print(
    "\nPhase counts:",
    phase_counts_623di2
)


# ------------------------------------------------------------
# 10. Reconstructed reliability channels
# ------------------------------------------------------------

reconstructed_623di2 = pd.DataFrame(
    {
        "pca_quality":
            pca_pct_623di2,
        "temporal_quality":
            temporal_pct_623di2,
        "predictive_uncertainty":
            uncertainty_pct_623di2,
    }
)

reference_623di2 = (
    RELIABILITY_DEV_CHANNELS_623A[
        [
            "pca_quality",
            "temporal_quality",
            "predictive_uncertainty",
        ]
    ]
    .reset_index(drop=True)
    .astype(np.float64)
)


# ------------------------------------------------------------
# 11. Numerical comparison
# ------------------------------------------------------------

comparison_rows_623di2 = []

for channel_623di2 in [
    "pca_quality",
    "temporal_quality",
    "predictive_uncertainty",
]:

    a_623di2 = (
        reconstructed_623di2[
            channel_623di2
        ]
        .to_numpy(dtype=np.float64)
    )

    b_623di2 = (
        reference_623di2[
            channel_623di2
        ]
        .to_numpy(dtype=np.float64)
    )

    diff_623di2 = np.abs(
        a_623di2
        -
        b_623di2
    )

    comparison_rows_623di2.append(
        {
            "channel":
                channel_623di2,
            "mean_abs_error":
                float(
                    diff_623di2.mean()
                ),
            "max_abs_error":
                float(
                    diff_623di2.max()
                ),
            "correlation":
                float(
                    np.corrcoef(
                        a_623di2,
                        b_623di2
                    )[0, 1]
                ),
        }
    )

comparison_623di2 = pd.DataFrame(
    comparison_rows_623di2
)

print(
    "\n--- Frozen CLEAN DEV channel comparison ---"
)

display(
    comparison_623di2
)


# ------------------------------------------------------------
# 12. Additional uncertainty comparisons
# ------------------------------------------------------------

frozen_unc_623di2 = np.asarray(
    FROZEN_DEV_UNCERTAINTY_622IF,
    dtype=np.float64
).reshape(-1)

exact_uncertainty_error_623di2 = float(
    np.max(
        np.abs(
            uncertainty_pct_623di2
            -
            frozen_unc_623di2
        )
    )
)

helper_adjusted_error_623di2 = float(
    np.max(
        np.abs(
            helper_adjusted_623di2
            -
            frozen_unc_623di2
        )
    )
)

print(
    "\n--- Uncertainty recipe comparison ---"
)

print(
    "Exact frozen-recipe max error :",
    exact_uncertainty_error_623di2
)

print(
    "Helper adjusted max error     :",
    helper_adjusted_error_623di2
)


# ------------------------------------------------------------
# 13. Readiness
#
# PCA/temporal should reproduce exactly or essentially exactly.
# MC recomputation can show tiny floating-point differences,
# therefore assess reconstruction numerically rather than
# requiring GPU bitwise identity.
# ------------------------------------------------------------

pca_error_623di2 = float(
    comparison_623di2.loc[
        comparison_623di2[
            "channel"
        ] == "pca_quality",
        "max_abs_error"
    ].iloc[0]
)

temporal_error_623di2 = float(
    comparison_623di2.loc[
        comparison_623di2[
            "channel"
        ] == "temporal_quality",
        "max_abs_error"
    ].iloc[0]
)

unc_corr_623di2 = float(
    comparison_623di2.loc[
        comparison_623di2[
            "channel"
        ] == "predictive_uncertainty",
        "correlation"
    ].iloc[0]
)

CLEAN_DEV_PATHWAY_READY_623DI2 = bool(
    pca_error_623di2 <= 1e-12
    and temporal_error_623di2 <= 1e-12
    and unc_corr_623di2 >= 0.999
)

print("\n==========================================")
print("STEP 6.23D-I2 READINESS SUMMARY")
print("==========================================")

print(
    "Development windows          :",
    len(reconstructed_623di2)
)

print(
    "Sequence reproduction error  :",
    sequence_error_623di2
)

print(
    "PCA reproduction max error   :",
    pca_error_623di2
)

print(
    "Temporal reproduction error  :",
    temporal_error_623di2
)

print(
    "Uncertainty correlation      :",
    unc_corr_623di2
)

print(
    "CLEAN DEV pathway ready      :",
    CLEAN_DEV_PATHWAY_READY_623DI2
)

print("\nOfficial test used            : False")
print("Reliability score selected    : False")
print("Threshold selected            : False")
print("Abstention rule created       : False")
print("Files written                 : False")
print("Frozen artifacts modified     : False")
print("=== END ===")

In [ ]:
# ============================================================
# Step 6.23D-J
# Five-family DEVELOPMENT end-to-end reliability smoke test
#
# One frozen scenario from each corruption family.
#
# Purpose:
#   raw DEV
#     -> frozen corruption
#     -> sequence reconstruction
#     -> PCA quality
#     -> temporal quality
#     -> MC predictive uncertainty
#     -> exact phase adjustment
#     -> 8 reliability candidates
#     -> deterministic RUL prediction / absolute error
#
# NO score selection.
# NO threshold selection.
# NO abstention.
# NO official test.
# NO files written.
# ============================================================

import numpy as np
import pandas as pd
import time

print(
    "=== Step 6.23D-J — "
    "Five-family DEVELOPMENT end-to-end smoke test ==="
)


# ------------------------------------------------------------
# 1. Required objects
# ------------------------------------------------------------

required_623dj = [
    "CANONICAL_DEV_RAW_623DF",
    "scenario_manifests_620gc",
    "apply_frozen_corruption_620gc",
    "val_raw_base_620gc",
    "PRIMARY_INPUT_COLUMNS",
    "build_sequences_623cb2",
    "pca_raw_quality_623cb2",
    "temporal_raw_quality_623cb2",
    "mc_uncertainty_623cb2",
    "midrank_percentile_623cb2",
    "pca_dev_reference_623cb2",
    "temporal_dev_reference_623cb2",
    "phase_references_622h",
    "dev_seq_meta",
]

missing_623dj = [
    name_623dj
    for name_623dj in required_623dj
    if name_623dj not in globals()
]

if missing_623dj:
    raise RuntimeError(
        "Missing required objects: "
        + str(missing_623dj)
    )

print("\nAll required objects are available.")


# ------------------------------------------------------------
# 2. Canonical target
# ------------------------------------------------------------

if "RUL_cap_125" not in dev_seq_meta.columns:
    raise RuntimeError(
        "RUL_cap_125 missing from dev_seq_meta."
    )

y_dev_623dj = (
    dev_seq_meta["RUL_cap_125"]
    .to_numpy(dtype=np.float64)
)

if y_dev_623dj.shape != (14241,):
    raise RuntimeError(
        "Unexpected DEV target shape."
    )

if not np.isfinite(y_dev_623dj).all():
    raise RuntimeError(
        "Non-finite development target."
    )


# ------------------------------------------------------------
# 3. Frozen phase-adjustment information
# ------------------------------------------------------------

SELECTED_UNCERTAINTY_623DJ = (
    "det_pred_pct_b5"
)

if (
    SELECTED_UNCERTAINTY_623DJ
    not in phase_references_622h
):
    raise RuntimeError(
        "Frozen phase reference unavailable."
    )

phase_ref_623dj = (
    phase_references_622h[
        SELECTED_UNCERTAINTY_623DJ
    ]
)

phase_edges_623dj = np.asarray(
    phase_ref_623dj["edges"],
    dtype=np.float64
)

phase_reference_values_623dj = (
    phase_ref_623dj["references"]
)

n_phase_bins_623dj = int(
    phase_ref_623dj["actual_bins"]
)


# ------------------------------------------------------------
# 4. Exact empirical mid-rank / N transformation
# ------------------------------------------------------------

def empirical_midrank_over_n_623dj(
    values,
    reference
):

    values = np.asarray(
        values,
        dtype=np.float64
    )

    reference = np.sort(
        np.asarray(
            reference,
            dtype=np.float64
        )
    )

    if len(reference) == 0:
        raise RuntimeError(
            "Empty phase reference."
        )

    left_623dj = np.searchsorted(
        reference,
        values,
        side="left"
    )

    right_623dj = np.searchsorted(
        reference,
        values,
        side="right"
    )

    return (
        0.5
        * (
            left_623dj.astype(np.float64)
            +
            right_623dj.astype(np.float64)
        )
        / float(len(reference))
    )


# ------------------------------------------------------------
# 5. Exact frozen uncertainty transformation
# ------------------------------------------------------------

def frozen_uncertainty_623dj(
    deterministic_prediction,
    raw_mc_std
):

    deterministic_prediction = np.asarray(
        deterministic_prediction,
        dtype=np.float64
    ).reshape(-1)

    raw_mc_std = np.asarray(
        raw_mc_std,
        dtype=np.float64
    ).reshape(-1)

    if (
        deterministic_prediction.shape
        != raw_mc_std.shape
    ):
        raise RuntimeError(
            "Prediction/std shape mismatch."
        )

    phase_ids_623dj = np.digitize(
        deterministic_prediction,
        phase_edges_623dj,
        right=True
    )

    adjusted_623dj = np.empty(
        len(raw_mc_std),
        dtype=np.float64
    )

    for phase_id_623dj in range(
        n_phase_bins_623dj
    ):

        mask_623dj = (
            phase_ids_623dj
            ==
            phase_id_623dj
        )

        if not np.any(mask_623dj):
            continue

        if (
            phase_id_623dj
            not in phase_reference_values_623dj
        ):
            raise RuntimeError(
                "Missing phase reference "
                f"{phase_id_623dj}."
            )

        adjusted_623dj[
            mask_623dj
        ] = empirical_midrank_over_n_623dj(
            raw_mc_std[
                mask_623dj
            ],
            phase_reference_values_623dj[
                phase_id_623dj
            ]
        )

    return adjusted_623dj


# ------------------------------------------------------------
# 6. Reliability-candidate construction
# ------------------------------------------------------------

def reliability_candidates_623dj(
    pca_quality,
    temporal_quality,
    predictive_uncertainty
):

    pca_quality = np.asarray(
        pca_quality,
        dtype=np.float64
    ).reshape(-1)

    temporal_quality = np.asarray(
        temporal_quality,
        dtype=np.float64
    ).reshape(-1)

    predictive_uncertainty = np.asarray(
        predictive_uncertainty,
        dtype=np.float64
    ).reshape(-1)

    channels_623dj = np.column_stack(
        [
            pca_quality,
            temporal_quality,
            predictive_uncertainty,
        ]
    )

    ordered_623dj = np.sort(
        channels_623dj,
        axis=1
    )

    candidates_623dj = pd.DataFrame(
        {
            "pca_only":
                pca_quality,

            "temporal_only":
                temporal_quality,

            "uncertainty_only":
                predictive_uncertainty,

            "sensor_quality_max":
                np.maximum(
                    pca_quality,
                    temporal_quality
                ),

            "max3":
                np.max(
                    channels_623dj,
                    axis=1
                ),

            "mean3":
                np.mean(
                    channels_623dj,
                    axis=1
                ),

            "rms3":
                np.sqrt(
                    np.mean(
                        channels_623dj ** 2,
                        axis=1
                    )
                ),

            "top2_mean":
                (
                    ordered_623dj[:, -1]
                    +
                    ordered_623dj[:, -2]
                )
                / 2.0,
        }
    )

    return candidates_623dj


# ------------------------------------------------------------
# 7. Safe DEVELOPMENT corruption replay
#
# The original frozen Step-4 replay function internally
# references val_raw_base_620gc.
#
# Therefore:
#   - temporarily provide a COPY of canonical DEV raw data,
#   - execute exactly the frozen corruption,
#   - restore the original validation object in finally.
#
# This preserves the frozen corruption implementation.
# ------------------------------------------------------------

def apply_dev_corruption_623dj(
    family,
    scenario_row
):

    global val_raw_base_620gc

    original_validation_base_623dj = (
        val_raw_base_620gc
    )

    try:

        val_raw_base_620gc = (
            CANONICAL_DEV_RAW_623DF
            .copy(deep=True)
        )

        corrupted_623dj = (
            apply_frozen_corruption_620gc(
                family,
                scenario_row
            )
        )

    finally:

        val_raw_base_620gc = (
            original_validation_base_623dj
        )

    return corrupted_623dj


# ------------------------------------------------------------
# 8. Canonical DEV feature fingerprint
# ------------------------------------------------------------

canonical_features_623dj = (
    CANONICAL_DEV_RAW_623DF[
        PRIMARY_INPUT_COLUMNS
    ]
    .to_numpy(dtype=np.float64)
    .copy()
)

if not np.isfinite(
    canonical_features_623dj
).all():
    raise RuntimeError(
        "Canonical DEV features are non-finite."
    )


# ------------------------------------------------------------
# 9. One frozen scenario from every corruption family
# ------------------------------------------------------------

families_623dj = [
    "bias",
    "noise",
    "drift",
    "dropout",
    "stuck",
]

results_623dj = []

smoke_details_623dj = {}

start_all_623dj = time.time()

print(
    "\n--- Executing five-family DEVELOPMENT smoke test ---"
)


for family_623dj in families_623dj:

    family_start_623dj = time.time()

    manifest_623dj = (
        scenario_manifests_620gc[
            family_623dj
        ]
    )

    if len(manifest_623dj) == 0:
        raise RuntimeError(
            f"Empty manifest for {family_623dj}."
        )

    # First frozen scenario only.
    scenario_row_623dj = (
        manifest_623dj.iloc[0]
    )

    # --------------------------------------------------------
    # A. Apply exact frozen corruption to DEV raw base
    # --------------------------------------------------------

    corrupted_raw_623dj = (
        apply_dev_corruption_623dj(
            family_623dj,
            scenario_row_623dj
        )
    )

    if not isinstance(
        corrupted_raw_623dj,
        pd.DataFrame
    ):
        raise RuntimeError(
            f"{family_623dj}: "
            "corruption did not return DataFrame."
        )

    if len(corrupted_raw_623dj) != 16561:
        raise RuntimeError(
            f"{family_623dj}: "
            "unexpected raw row count."
        )

    corrupted_features_623dj = (
        corrupted_raw_623dj[
            PRIMARY_INPUT_COLUMNS
        ]
        .to_numpy(dtype=np.float64)
    )

    if not np.isfinite(
        corrupted_features_623dj
    ).all():
        raise RuntimeError(
            f"{family_623dj}: "
            "non-finite corrupted features."
        )

    raw_max_change_623dj = float(
        np.max(
            np.abs(
                corrupted_features_623dj
                -
                canonical_features_623dj
            )
        )
    )


    # --------------------------------------------------------
    # B. Build exact 30-cycle sequences
    # --------------------------------------------------------

    X_corrupt_623dj = np.asarray(
        build_sequences_623cb2(
            corrupted_raw_623dj
        ),
        dtype=np.float64
    )

    if X_corrupt_623dj.shape != (
        14241,
        30,
        17
    ):
        raise RuntimeError(
            f"{family_623dj}: "
            f"unexpected sequence shape "
            f"{X_corrupt_623dj.shape}."
        )

    if not np.isfinite(
        X_corrupt_623dj
    ).all():
        raise RuntimeError(
            f"{family_623dj}: "
            "non-finite sequences."
        )


    # --------------------------------------------------------
    # C. PCA raw quality
    # --------------------------------------------------------

    pca_raw_623dj = np.asarray(
        pca_raw_quality_623cb2(
            X_corrupt_623dj
        ),
        dtype=np.float64
    ).reshape(-1)

    pca_quality_623dj = np.asarray(
        midrank_percentile_623cb2(
            pca_raw_623dj,
            pca_dev_reference_623cb2
        ),
        dtype=np.float64
    ).reshape(-1)


    # --------------------------------------------------------
    # D. Temporal raw quality
    # --------------------------------------------------------

    temporal_raw_623dj = np.asarray(
        temporal_raw_quality_623cb2(
            X_corrupt_623dj
        ),
        dtype=np.float64
    ).reshape(-1)

    temporal_quality_623dj = np.asarray(
        midrank_percentile_623cb2(
            temporal_raw_623dj,
            temporal_dev_reference_623cb2
        ),
        dtype=np.float64
    ).reshape(-1)


    # --------------------------------------------------------
    # E. RUL prediction + raw MC uncertainty
    # --------------------------------------------------------

    mc_return_623dj = (
        mc_uncertainty_623cb2(
            X_corrupt_623dj
        )
    )

    if (
        not isinstance(
            mc_return_623dj,
            (tuple, list)
        )
        or len(mc_return_623dj) != 3
    ):
        raise RuntimeError(
            f"{family_623dj}: "
            "unexpected MC return."
        )

    deterministic_prediction_623dj = (
        np.asarray(
            mc_return_623dj[0],
            dtype=np.float64
        )
        .reshape(-1)
    )

    raw_mc_std_623dj = (
        np.asarray(
            mc_return_623dj[1],
            dtype=np.float64
        )
        .reshape(-1)
    )


    # --------------------------------------------------------
    # F. Exact frozen predictive-uncertainty recipe
    # --------------------------------------------------------

    uncertainty_quality_623dj = (
        frozen_uncertainty_623dj(
            deterministic_prediction_623dj,
            raw_mc_std_623dj
        )
    )


    # --------------------------------------------------------
    # G. Integrity of all three channels
    # --------------------------------------------------------

    channel_matrix_623dj = np.column_stack(
        [
            pca_quality_623dj,
            temporal_quality_623dj,
            uncertainty_quality_623dj,
        ]
    )

    if channel_matrix_623dj.shape != (
        14241,
        3
    ):
        raise RuntimeError(
            f"{family_623dj}: "
            "unexpected channel shape."
        )

    if not np.isfinite(
        channel_matrix_623dj
    ).all():
        raise RuntimeError(
            f"{family_623dj}: "
            "non-finite reliability channel."
        )

    if (
        channel_matrix_623dj.min()
        < -1e-12
        or
        channel_matrix_623dj.max()
        > 1.0 + 1e-12
    ):
        raise RuntimeError(
            f"{family_623dj}: "
            "channel outside [0,1]."
        )


    # --------------------------------------------------------
    # H. Build eight candidate reliability scores
    # --------------------------------------------------------

    candidates_623dj = (
        reliability_candidates_623dj(
            pca_quality_623dj,
            temporal_quality_623dj,
            uncertainty_quality_623dj
        )
    )

    if candidates_623dj.shape != (
        14241,
        8
    ):
        raise RuntimeError(
            f"{family_623dj}: "
            "unexpected candidate shape."
        )

    if not np.isfinite(
        candidates_623dj.to_numpy()
    ).all():
        raise RuntimeError(
            f"{family_623dj}: "
            "non-finite candidate score."
        )


    # --------------------------------------------------------
    # I. RUL-error pathway
    # --------------------------------------------------------

    if deterministic_prediction_623dj.shape != (
        14241,
    ):
        raise RuntimeError(
            f"{family_623dj}: "
            "unexpected prediction shape."
        )

    absolute_error_623dj = np.abs(
        deterministic_prediction_623dj
        -
        y_dev_623dj
    )

    corrupted_mae_623dj = float(
        np.mean(
            absolute_error_623dj
        )
    )


    # --------------------------------------------------------
    # J. Compact result
    # --------------------------------------------------------

    sensor_623dj = (
        scenario_row_623dj.get(
            "sensor",
            None
        )
    )

    severity_623dj = (
        scenario_row_623dj.get(
            "severity",
            None
        )
    )

    elapsed_623dj = (
        time.time()
        -
        family_start_623dj
    )

    results_623dj.append(
        {
            "family":
                family_623dj,

            "sensor":
                sensor_623dj,

            "severity":
                severity_623dj,

            "raw_max_change":
                raw_max_change_623dj,

            "windows":
                len(
                    deterministic_prediction_623dj
                ),

            "pca_mean":
                float(
                    np.mean(
                        pca_quality_623dj
                    )
                ),

            "temporal_mean":
                float(
                    np.mean(
                        temporal_quality_623dj
                    )
                ),

            "uncertainty_mean":
                float(
                    np.mean(
                        uncertainty_quality_623dj
                    )
                ),

            "mean3_mean":
                float(
                    candidates_623dj[
                        "mean3"
                    ].mean()
                ),

            "rul_mae":
                corrupted_mae_623dj,

            "finite":
                True,

            "seconds":
                elapsed_623dj,
        }
    )

    smoke_details_623dj[
        family_623dj
    ] = {
        "scenario":
            scenario_row_623dj.copy(),

        "channels":
            channel_matrix_623dj,

        "candidates":
            candidates_623dj,

        "prediction":
            deterministic_prediction_623dj,

        "absolute_error":
            absolute_error_623dj,
    }

    print(
        f"{family_623dj:8s}: PASS "
        f"| windows=14241 "
        f"| MAE={corrupted_mae_623dj:.4f} "
        f"| {elapsed_623dj:.2f}s"
    )


# ------------------------------------------------------------
# 10. Verify original validation base restoration
# ------------------------------------------------------------

validation_restored_623dj = bool(
    val_raw_base_620gc is not None
)

canonical_unchanged_623dj = bool(
    np.array_equal(
        CANONICAL_DEV_RAW_623DF[
            PRIMARY_INPUT_COLUMNS
        ].to_numpy(dtype=np.float64),
        canonical_features_623dj
    )
)


# ------------------------------------------------------------
# 11. Summary
# ------------------------------------------------------------

smoke_summary_623dj = pd.DataFrame(
    results_623dj
)

total_seconds_623dj = (
    time.time()
    -
    start_all_623dj
)

print(
    "\n--- Five-family DEVELOPMENT "
    "reliability smoke-test summary ---"
)

display(
    smoke_summary_623dj
)

all_families_present_623dj = (
    set(
        smoke_summary_623dj[
            "family"
        ]
    )
    ==
    set(
        families_623dj
    )
)

all_windows_ok_623dj = bool(
    (
        smoke_summary_623dj[
            "windows"
        ]
        ==
        14241
    ).all()
)

all_finite_623dj = bool(
    smoke_summary_623dj[
        "finite"
    ].all()
)

all_corruptions_changed_623dj = bool(
    (
        smoke_summary_623dj[
            "raw_max_change"
        ]
        >
        0.0
    ).all()
)

DEV_FULL_PATH_READY_623DJ = bool(
    all_families_present_623dj
    and all_windows_ok_623dj
    and all_finite_623dj
    and all_corruptions_changed_623dj
    and canonical_unchanged_623dj
)


print("\n============================================")
print("STEP 6.23D-J READINESS SUMMARY")
print("============================================")

print(
    "Corruption families tested    :",
    len(smoke_summary_623dj)
)

print(
    "All five families present     :",
    all_families_present_623dj
)

print(
    "All have 14241 windows        :",
    all_windows_ok_623dj
)

print(
    "All outputs finite            :",
    all_finite_623dj
)

print(
    "All corruptions changed data  :",
    all_corruptions_changed_623dj
)

print(
    "Canonical DEV unchanged       :",
    canonical_unchanged_623dj
)

print(
    "Validation base restored      :",
    validation_restored_623dj
)

print(
    "Total runtime (seconds)       :",
    total_seconds_623dj
)

print(
    "DEV full pathway ready        :",
    DEV_FULL_PATH_READY_623DJ
)

print("\nOfficial test used            : False")
print("Reliability score selected    : False")
print("Reliability threshold selected: False")
print("Abstention rule created       : False")
print("Files written                 : False")
print("Frozen artifacts modified     : False")

print("=== END ===")

In [ ]:
# ============================================================
# Step 6.23D-K
# Full 315-scenario DEVELOPMENT reliability-candidate evaluation
#
# PURPOSE
# -------
# Evaluate all frozen sensor-degradation scenarios using
# DEVELOPMENT data only.
#
# For every scenario:
#   frozen corruption
#       -> sequence reconstruction
#       -> PCA quality
#       -> temporal quality
#       -> predictive uncertainty
#       -> 8 reliability candidates
#       -> corrupted RUL error
#       -> selective-risk diagnostics
#
# IMPORTANT
# ---------
# - NO official test.
# - NO reliability-score selection in this cell.
# - NO threshold selection.
# - NO abstention rule.
# - NO files written.
#
# Family-balanced aggregation is used so that corruption
# families with more scenarios do not dominate the comparison.
# ============================================================

import numpy as np
import pandas as pd
import time

print(
    "=== Step 6.23D-K — Full 315-scenario "
    "DEVELOPMENT reliability evaluation ==="
)


# ------------------------------------------------------------
# 1. Required objects
# ------------------------------------------------------------

required_623dk = [
    "CANONICAL_DEV_RAW_623DF",
    "scenario_manifests_620gc",
    "PRIMARY_INPUT_COLUMNS",
    "dev_seq_meta",
    "apply_dev_corruption_623dj",
    "build_sequences_623cb2",
    "pca_raw_quality_623cb2",
    "temporal_raw_quality_623cb2",
    "mc_uncertainty_623cb2",
    "midrank_percentile_623cb2",
    "pca_dev_reference_623cb2",
    "temporal_dev_reference_623cb2",
    "frozen_uncertainty_623dj",
    "reliability_candidates_623dj",
]

missing_623dk = [
    name
    for name in required_623dk
    if name not in globals()
]

if missing_623dk:
    raise RuntimeError(
        "Missing required Step 6.23D-K objects: "
        + str(missing_623dk)
    )

print("All required objects are available.")


# ------------------------------------------------------------
# 2. Canonical DEVELOPMENT target
# ------------------------------------------------------------

if "RUL_cap_125" not in dev_seq_meta.columns:
    raise RuntimeError(
        "RUL_cap_125 missing from dev_seq_meta."
    )

y_dev_623dk = (
    dev_seq_meta["RUL_cap_125"]
    .to_numpy(dtype=np.float64)
)

if y_dev_623dk.shape != (14241,):
    raise RuntimeError(
        f"Unexpected target shape: {y_dev_623dk.shape}"
    )

if not np.isfinite(y_dev_623dk).all():
    raise RuntimeError(
        "Non-finite development targets."
    )


# ------------------------------------------------------------
# 3. Clean-development deterministic prediction
#
# Needed only as a reference for corruption-induced error
# change. Reuse the verified result if available.
# ------------------------------------------------------------

if (
    "det_prediction_623di2" in globals()
    and
    np.asarray(
        det_prediction_623di2
    ).shape == (14241,)
):

    CLEAN_DEV_PRED_623DK = np.asarray(
        det_prediction_623di2,
        dtype=np.float64
    ).copy()

    clean_prediction_source_623dk = (
        "verified Step 6.23D-I2"
    )

else:

    X_clean_623dk = np.asarray(
        build_sequences_623cb2(
            CANONICAL_DEV_RAW_623DF
        ),
        dtype=np.float64
    )

    mc_clean_return_623dk = (
        mc_uncertainty_623cb2(
            X_clean_623dk
        )
    )

    CLEAN_DEV_PRED_623DK = np.asarray(
        mc_clean_return_623dk[0],
        dtype=np.float64
    ).reshape(-1)

    clean_prediction_source_623dk = (
        "recomputed"
    )

if CLEAN_DEV_PRED_623DK.shape != (14241,):
    raise RuntimeError(
        "Unexpected clean prediction shape."
    )

CLEAN_DEV_ABS_ERROR_623DK = np.abs(
    CLEAN_DEV_PRED_623DK
    -
    y_dev_623dk
)

CLEAN_DEV_MAE_623DK = float(
    np.mean(
        CLEAN_DEV_ABS_ERROR_623DK
    )
)

print(
    "\nClean prediction source :",
    clean_prediction_source_623dk
)

print(
    "Clean DEV MAE          :",
    CLEAN_DEV_MAE_623DK
)


# ------------------------------------------------------------
# 4. Candidate names
# ------------------------------------------------------------

CANDIDATE_NAMES_623DK = [
    "pca_only",
    "temporal_only",
    "uncertainty_only",
    "sensor_quality_max",
    "max3",
    "mean3",
    "rms3",
    "top2_mean",
]

N_CANDIDATES_623DK = len(
    CANDIDATE_NAMES_623DK
)

print(
    "Reliability candidates :",
    CANDIDATE_NAMES_623DK
)


# ------------------------------------------------------------
# 5. Create canonical ordered scenario registry
# ------------------------------------------------------------

FAMILIES_623DK = [
    "bias",
    "noise",
    "drift",
    "dropout",
    "stuck",
]

scenario_registry_623dk = []

global_scenario_index_623dk = 0

for family_623dk in FAMILIES_623DK:

    manifest_623dk = (
        scenario_manifests_620gc[
            family_623dk
        ]
    )

    for local_position_623dk, (
        original_index_623dk,
        scenario_row_623dk
    ) in enumerate(
        manifest_623dk.iterrows()
    ):

        scenario_registry_623dk.append(
            {
                "global_index":
                    global_scenario_index_623dk,

                "family":
                    family_623dk,

                "local_position":
                    local_position_623dk,

                "manifest_index":
                    original_index_623dk,

                "scenario_row":
                    scenario_row_623dk.copy(),
            }
        )

        global_scenario_index_623dk += 1


N_SCENARIOS_623DK = len(
    scenario_registry_623dk
)

print(
    "\nFrozen scenarios found :",
    N_SCENARIOS_623DK
)

if N_SCENARIOS_623DK != 315:
    raise RuntimeError(
        "Expected exactly 315 frozen scenarios."
    )


# ------------------------------------------------------------
# 6. Scenario counts by family
# ------------------------------------------------------------

family_counts_623dk = {
    family_623dk:
    len(
        scenario_manifests_620gc[
            family_623dk
        ]
    )
    for family_623dk in FAMILIES_623DK
}

print(
    "Family counts          :",
    family_counts_623dk
)

expected_family_counts_623dk = {
    "bias": 90,
    "noise": 45,
    "drift": 90,
    "dropout": 45,
    "stuck": 45,
}

if family_counts_623dk != expected_family_counts_623dk:
    raise RuntimeError(
        "Unexpected frozen family counts."
    )


# ------------------------------------------------------------
# 7. Preserve fingerprints before execution
# ------------------------------------------------------------

canonical_features_before_623dk = (
    CANONICAL_DEV_RAW_623DF[
        PRIMARY_INPUT_COLUMNS
    ]
    .to_numpy(dtype=np.float64)
    .copy()
)

validation_object_id_before_623dk = (
    id(val_raw_base_620gc)
)

validation_features_before_623dk = (
    val_raw_base_620gc[
        PRIMARY_INPUT_COLUMNS
    ]
    .to_numpy(dtype=np.float64)
    .copy()
)


# ------------------------------------------------------------
# 8. Metric helpers
#
# Higher reliability score = worse / less trustworthy.
#
# Therefore selective acceptance keeps the LOWEST scores.
# ------------------------------------------------------------

def exact_coverage_mae_623dk(
    score,
    absolute_error,
    coverage
):

    score = np.asarray(
        score,
        dtype=np.float64
    ).reshape(-1)

    absolute_error = np.asarray(
        absolute_error,
        dtype=np.float64
    ).reshape(-1)

    if score.shape != absolute_error.shape:
        raise RuntimeError(
            "Score/error shape mismatch."
        )

    n_623dk = len(score)

    k_623dk = int(
        np.floor(
            float(coverage)
            *
            n_623dk
        )
    )

    k_623dk = max(
        1,
        min(
            n_623dk,
            k_623dk
        )
    )

    # Stable ordering makes tie-handling deterministic.
    order_623dk = np.argsort(
        score,
        kind="mergesort"
    )

    accepted_623dk = (
        order_623dk[
            :k_623dk
        ]
    )

    return float(
        np.mean(
            absolute_error[
                accepted_623dk
            ]
        )
    )


def top_quartile_capture_623dk(
    score,
    absolute_error
):

    score = np.asarray(
        score,
        dtype=np.float64
    ).reshape(-1)

    absolute_error = np.asarray(
        absolute_error,
        dtype=np.float64
    ).reshape(-1)

    n_623dk = len(score)

    q_623dk = max(
        1,
        int(
            np.floor(
                0.25
                *
                n_623dk
            )
        )
    )

    score_order_623dk = np.argsort(
        score,
        kind="mergesort"
    )

    error_order_623dk = np.argsort(
        absolute_error,
        kind="mergesort"
    )

    high_score_indices_623dk = set(
        score_order_623dk[
            -q_623dk:
        ]
        .tolist()
    )

    high_error_indices_623dk = set(
        error_order_623dk[
            -q_623dk:
        ]
        .tolist()
    )

    overlap_623dk = len(
        high_score_indices_623dk
        &
        high_error_indices_623dk
    )

    return (
        100.0
        *
        overlap_623dk
        /
        q_623dk
    )


def high_low_error_ratio_623dk(
    score,
    absolute_error
):

    score = np.asarray(
        score,
        dtype=np.float64
    ).reshape(-1)

    absolute_error = np.asarray(
        absolute_error,
        dtype=np.float64
    ).reshape(-1)

    n_623dk = len(score)

    q_623dk = max(
        1,
        int(
            np.floor(
                0.25
                *
                n_623dk
            )
        )
    )

    order_623dk = np.argsort(
        score,
        kind="mergesort"
    )

    low_indices_623dk = (
        order_623dk[
            :q_623dk
        ]
    )

    high_indices_623dk = (
        order_623dk[
            -q_623dk:
        ]
    )

    low_mae_623dk = float(
        np.mean(
            absolute_error[
                low_indices_623dk
            ]
        )
    )

    high_mae_623dk = float(
        np.mean(
            absolute_error[
                high_indices_623dk
            ]
        )
    )

    if low_mae_623dk <= 0.0:
        return np.nan

    return (
        high_mae_623dk
        /
        low_mae_623dk
    )


def spearman_623dk(
    a,
    b
):

    a = pd.Series(
        np.asarray(
            a,
            dtype=np.float64
        )
    )

    b = pd.Series(
        np.asarray(
            b,
            dtype=np.float64
        )
    )

    return float(
        a.corr(
            b,
            method="spearman"
        )
    )


# ------------------------------------------------------------
# 9. Compact in-memory storage
#
# float32 storage is used ONLY after float64 computation.
#
# Approximately:
#   score cube ~144 MB
#   error matrix ~18 MB
#
# This avoids having to rerun all 315 MC experiments later
# when we analyse thresholds and abstention.
# ------------------------------------------------------------

FULL_DEV_SCORE_CUBE_623DK = np.empty(
    (
        N_SCENARIOS_623DK,
        14241,
        N_CANDIDATES_623DK,
    ),
    dtype=np.float32
)

FULL_DEV_ERROR_MATRIX_623DK = np.empty(
    (
        N_SCENARIOS_623DK,
        14241,
    ),
    dtype=np.float32
)

scenario_metric_rows_623dk = []
scenario_index_rows_623dk = []


# ------------------------------------------------------------
# 10. Execute all 315 scenarios
# ------------------------------------------------------------

print(
    "\n=== Executing all 315 DEVELOPMENT scenarios ==="
)

full_start_623dk = time.time()


for scenario_record_623dk in scenario_registry_623dk:

    global_idx_623dk = (
        scenario_record_623dk[
            "global_index"
        ]
    )

    family_623dk = (
        scenario_record_623dk[
            "family"
        ]
    )

    local_position_623dk = (
        scenario_record_623dk[
            "local_position"
        ]
    )

    manifest_index_623dk = (
        scenario_record_623dk[
            "manifest_index"
        ]
    )

    scenario_row_623dk = (
        scenario_record_623dk[
            "scenario_row"
        ]
    )

    scenario_id_623dk = (
        f"{family_623dk}_"
        f"{local_position_623dk:03d}"
    )


    # --------------------------------------------------------
    # A. Exact frozen corruption replay on canonical DEV
    # --------------------------------------------------------

    corrupted_raw_623dk = (
        apply_dev_corruption_623dj(
            family_623dk,
            scenario_row_623dk
        )
    )

    if not isinstance(
        corrupted_raw_623dk,
        pd.DataFrame
    ):
        raise RuntimeError(
            f"{scenario_id_623dk}: "
            "corruption output is not DataFrame."
        )

    if len(corrupted_raw_623dk) != 16561:
        raise RuntimeError(
            f"{scenario_id_623dk}: "
            "unexpected raw row count."
        )


    # --------------------------------------------------------
    # B. Build windows
    # --------------------------------------------------------

    X_corrupt_623dk = np.asarray(
        build_sequences_623cb2(
            corrupted_raw_623dk
        ),
        dtype=np.float64
    )

    if X_corrupt_623dk.shape != (
        14241,
        30,
        17
    ):
        raise RuntimeError(
            f"{scenario_id_623dk}: "
            f"unexpected sequence shape "
            f"{X_corrupt_623dk.shape}."
        )

    if not np.isfinite(
        X_corrupt_623dk
    ).all():
        raise RuntimeError(
            f"{scenario_id_623dk}: "
            "non-finite sequences."
        )


    # --------------------------------------------------------
    # C. PCA quality
    # --------------------------------------------------------

    pca_raw_623dk = np.asarray(
        pca_raw_quality_623cb2(
            X_corrupt_623dk
        ),
        dtype=np.float64
    ).reshape(-1)

    pca_quality_623dk = np.asarray(
        midrank_percentile_623cb2(
            pca_raw_623dk,
            pca_dev_reference_623cb2
        ),
        dtype=np.float64
    ).reshape(-1)


    # --------------------------------------------------------
    # D. Temporal quality
    # --------------------------------------------------------

    temporal_raw_623dk = np.asarray(
        temporal_raw_quality_623cb2(
            X_corrupt_623dk
        ),
        dtype=np.float64
    ).reshape(-1)

    temporal_quality_623dk = np.asarray(
        midrank_percentile_623cb2(
            temporal_raw_623dk,
            temporal_dev_reference_623cb2
        ),
        dtype=np.float64
    ).reshape(-1)


    # --------------------------------------------------------
    # E. Corrupted RUL prediction + MC standard deviation
    # --------------------------------------------------------

    mc_result_623dk = (
        mc_uncertainty_623cb2(
            X_corrupt_623dk
        )
    )

    if (
        not isinstance(
            mc_result_623dk,
            (tuple, list)
        )
        or
        len(mc_result_623dk) != 3
    ):
        raise RuntimeError(
            f"{scenario_id_623dk}: "
            "unexpected MC helper return."
        )

    corrupted_prediction_623dk = np.asarray(
        mc_result_623dk[0],
        dtype=np.float64
    ).reshape(-1)

    corrupted_mc_std_623dk = np.asarray(
        mc_result_623dk[1],
        dtype=np.float64
    ).reshape(-1)


    # --------------------------------------------------------
    # F. Exact frozen predictive-uncertainty representation
    # --------------------------------------------------------

    uncertainty_quality_623dk = (
        frozen_uncertainty_623dj(
            corrupted_prediction_623dk,
            corrupted_mc_std_623dk
        )
    )


    # --------------------------------------------------------
    # G. Build eight reliability candidates
    # --------------------------------------------------------

    candidate_df_623dk = (
        reliability_candidates_623dj(
            pca_quality_623dk,
            temporal_quality_623dk,
            uncertainty_quality_623dk
        )
    )

    candidate_df_623dk = (
        candidate_df_623dk[
            CANDIDATE_NAMES_623DK
        ]
    )

    if candidate_df_623dk.shape != (
        14241,
        8
    ):
        raise RuntimeError(
            f"{scenario_id_623dk}: "
            "unexpected candidate matrix shape."
        )

    candidate_matrix_623dk = (
        candidate_df_623dk
        .to_numpy(dtype=np.float64)
    )

    if not np.isfinite(
        candidate_matrix_623dk
    ).all():
        raise RuntimeError(
            f"{scenario_id_623dk}: "
            "non-finite reliability candidate."
        )

    if (
        candidate_matrix_623dk.min()
        < -1e-12
        or
        candidate_matrix_623dk.max()
        > 1.0 + 1e-12
    ):
        raise RuntimeError(
            f"{scenario_id_623dk}: "
            "candidate outside [0,1]."
        )


    # --------------------------------------------------------
    # H. RUL errors
    # --------------------------------------------------------

    absolute_error_623dk = np.abs(
        corrupted_prediction_623dk
        -
        y_dev_623dk
    )

    delta_absolute_error_623dk = (
        absolute_error_623dk
        -
        CLEAN_DEV_ABS_ERROR_623DK
    )

    corrupted_mae_623dk = float(
        np.mean(
            absolute_error_623dk
        )
    )

    mae_delta_623dk = (
        corrupted_mae_623dk
        -
        CLEAN_DEV_MAE_623DK
    )


    # --------------------------------------------------------
    # I. Store compact matrices for later threshold analysis
    # --------------------------------------------------------

    FULL_DEV_SCORE_CUBE_623DK[
        global_idx_623dk,
        :,
        :
    ] = candidate_matrix_623dk.astype(
        np.float32
    )

    FULL_DEV_ERROR_MATRIX_623DK[
        global_idx_623dk,
        :
    ] = absolute_error_623dk.astype(
        np.float32
    )


    # --------------------------------------------------------
    # J. Scenario metadata
    # --------------------------------------------------------

    sensor_623dk = (
        scenario_row_623dk.get(
            "sensor",
            None
        )
    )

    severity_623dk = (
        scenario_row_623dk.get(
            "severity",
            None
        )
    )

    direction_623dk = (
        scenario_row_623dk.get(
            "direction",
            np.nan
        )
    )

    alpha_623dk = (
        scenario_row_623dk.get(
            "alpha",
            np.nan
        )
    )

    requested_fraction_623dk = (
        scenario_row_623dk.get(
            "requested_fraction",
            np.nan
        )
    )

    scenario_index_rows_623dk.append(
        {
            "global_index":
                global_idx_623dk,

            "scenario_id":
                scenario_id_623dk,

            "family":
                family_623dk,

            "local_position":
                local_position_623dk,

            "manifest_index":
                manifest_index_623dk,

            "sensor":
                sensor_623dk,

            "severity":
                severity_623dk,

            "direction":
                direction_623dk,

            "alpha":
                alpha_623dk,

            "requested_fraction":
                requested_fraction_623dk,

            "corrupted_mae":
                corrupted_mae_623dk,

            "mae_delta_vs_clean":
                mae_delta_623dk,
        }
    )


    # --------------------------------------------------------
    # K. Candidate diagnostics
    # --------------------------------------------------------

    for candidate_idx_623dk, candidate_623dk \
        in enumerate(
            CANDIDATE_NAMES_623DK
        ):

        score_623dk = (
            candidate_matrix_623dk[
                :,
                candidate_idx_623dk
            ]
        )

        error_spearman_623dk = (
            spearman_623dk(
                score_623dk,
                absolute_error_623dk
            )
        )

        delta_error_spearman_623dk = (
            spearman_623dk(
                score_623dk,
                delta_absolute_error_623dk
            )
        )

        high_low_ratio_623dk = (
            high_low_error_ratio_623dk(
                score_623dk,
                absolute_error_623dk
            )
        )

        top_error_capture_623dk = (
            top_quartile_capture_623dk(
                score_623dk,
                absolute_error_623dk
            )
        )

        mae90_623dk = (
            exact_coverage_mae_623dk(
                score_623dk,
                absolute_error_623dk,
                0.90
            )
        )

        mae70_623dk = (
            exact_coverage_mae_623dk(
                score_623dk,
                absolute_error_623dk,
                0.70
            )
        )

        mae50_623dk = (
            exact_coverage_mae_623dk(
                score_623dk,
                absolute_error_623dk,
                0.50
            )
        )

        gain90_623dk = (
            corrupted_mae_623dk
            -
            mae90_623dk
        )

        gain70_623dk = (
            corrupted_mae_623dk
            -
            mae70_623dk
        )

        gain50_623dk = (
            corrupted_mae_623dk
            -
            mae50_623dk
        )

        mean_rc_gain_623dk = float(
            np.mean(
                [
                    gain90_623dk,
                    gain70_623dk,
                    gain50_623dk,
                ]
            )
        )

        scenario_metric_rows_623dk.append(
            {
                "global_index":
                    global_idx_623dk,

                "scenario_id":
                    scenario_id_623dk,

                "family":
                    family_623dk,

                "sensor":
                    sensor_623dk,

                "severity":
                    severity_623dk,

                "candidate":
                    candidate_623dk,

                "corrupted_mae":
                    corrupted_mae_623dk,

                "mae_delta_vs_clean":
                    mae_delta_623dk,

                "score_mean":
                    float(
                        np.mean(
                            score_623dk
                        )
                    ),

                "score_p95":
                    float(
                        np.quantile(
                            score_623dk,
                            0.95
                        )
                    ),

                "error_spearman":
                    error_spearman_623dk,

                "delta_error_spearman":
                    delta_error_spearman_623dk,

                "high_low_ratio":
                    high_low_ratio_623dk,

                "top_error_capture_pct":
                    top_error_capture_623dk,

                "gain90":
                    gain90_623dk,

                "gain70":
                    gain70_623dk,

                "gain50":
                    gain50_623dk,

                "mean_rc_gain":
                    mean_rc_gain_623dk,
            }
        )


    # --------------------------------------------------------
    # L. Progress
    # --------------------------------------------------------

    completed_623dk = (
        global_idx_623dk
        +
        1
    )

    if (
        completed_623dk % 25 == 0
        or
        completed_623dk
        ==
        N_SCENARIOS_623DK
    ):

        elapsed_623dk = (
            time.time()
            -
            full_start_623dk
        )

        rate_623dk = (
            elapsed_623dk
            /
            completed_623dk
        )

        remaining_seconds_623dk = (
            rate_623dk
            *
            (
                N_SCENARIOS_623DK
                -
                completed_623dk
            )
        )

        print(
            f"[{completed_623dk:03d}/"
            f"{N_SCENARIOS_623DK}] "
            f"completed | "
            f"elapsed={elapsed_623dk:.1f}s | "
            f"remaining≈"
            f"{remaining_seconds_623dk/60:.1f} min"
        )


# ------------------------------------------------------------
# 11. Build result tables
# ------------------------------------------------------------

FULL_DEV_SCENARIO_INDEX_623DK = pd.DataFrame(
    scenario_index_rows_623dk
)

FULL_DEV_SCENARIO_METRICS_623DK = pd.DataFrame(
    scenario_metric_rows_623dk
)

full_elapsed_623dk = (
    time.time()
    -
    full_start_623dk
)

print(
    "\nFull execution time:",
    f"{full_elapsed_623dk:.2f} seconds",
    f"({full_elapsed_623dk/60:.2f} minutes)"
)


# ------------------------------------------------------------
# 12. Numerical-integrity verification
# ------------------------------------------------------------

expected_metric_rows_623dk = (
    315
    *
    8
)

if (
    len(
        FULL_DEV_SCENARIO_METRICS_623DK
    )
    !=
    expected_metric_rows_623dk
):
    raise RuntimeError(
        "Unexpected scenario/candidate result count."
    )

numeric_metric_columns_623dk = [
    "corrupted_mae",
    "mae_delta_vs_clean",
    "score_mean",
    "score_p95",
    "error_spearman",
    "delta_error_spearman",
    "high_low_ratio",
    "top_error_capture_pct",
    "gain90",
    "gain70",
    "gain50",
    "mean_rc_gain",
]

metric_numeric_values_623dk = (
    FULL_DEV_SCENARIO_METRICS_623DK[
        numeric_metric_columns_623dk
    ]
    .to_numpy(dtype=np.float64)
)

all_metric_values_finite_623dk = bool(
    np.isfinite(
        metric_numeric_values_623dk
    ).all()
)

all_score_cube_finite_623dk = bool(
    np.isfinite(
        FULL_DEV_SCORE_CUBE_623DK
    ).all()
)

all_error_matrix_finite_623dk = bool(
    np.isfinite(
        FULL_DEV_ERROR_MATRIX_623DK
    ).all()
)

if not all_metric_values_finite_623dk:
    raise RuntimeError(
        "Non-finite scenario metric detected."
    )

if not all_score_cube_finite_623dk:
    raise RuntimeError(
        "Non-finite score-cube value detected."
    )

if not all_error_matrix_finite_623dk:
    raise RuntimeError(
        "Non-finite error-matrix value detected."
    )


# ------------------------------------------------------------
# 13. Family-level summaries
#
# First average within each corruption family.
# ------------------------------------------------------------

metric_columns_for_summary_623dk = [
    "error_spearman",
    "delta_error_spearman",
    "high_low_ratio",
    "top_error_capture_pct",
    "gain90",
    "gain70",
    "gain50",
    "mean_rc_gain",
]

DEV_FAMILY_CANDIDATE_SUMMARY_623DK = (
    FULL_DEV_SCENARIO_METRICS_623DK
    .groupby(
        [
            "family",
            "candidate",
        ],
        as_index=False
    )[
        metric_columns_for_summary_623dk
    ]
    .mean()
)


# ------------------------------------------------------------
# 14. Family-balanced candidate summary
#
# Every corruption family receives equal weight.
# ------------------------------------------------------------

DEV_FAMILY_BALANCED_SUMMARY_623DK = (
    DEV_FAMILY_CANDIDATE_SUMMARY_623DK
    .groupby(
        "candidate",
        as_index=False
    )[
        metric_columns_for_summary_623dk
    ]
    .mean()
)

DEV_FAMILY_BALANCED_SUMMARY_623DK = (
    DEV_FAMILY_BALANCED_SUMMARY_623DK
    .rename(
        columns={
            "error_spearman":
                "fb_error_spearman",

            "delta_error_spearman":
                "fb_delta_error_spearman",

            "high_low_ratio":
                "fb_high_low_ratio",

            "top_error_capture_pct":
                "fb_top_error_capture_pct",

            "gain90":
                "fb_gain90",

            "gain70":
                "fb_gain70",

            "gain50":
                "fb_gain50",

            "mean_rc_gain":
                "fb_mean_rc_gain",
        }
    )
)


# ------------------------------------------------------------
# 15. Family-consistency counts
# ------------------------------------------------------------

positive_gain70_counts_623dk = (
    DEV_FAMILY_CANDIDATE_SUMMARY_623DK
    .assign(
        positive_gain70=lambda df:
            df["gain70"]
            >
            0.0
    )
    .groupby(
        "candidate"
    )[
        "positive_gain70"
    ]
    .sum()
)

positive_mean_gain_counts_623dk = (
    DEV_FAMILY_CANDIDATE_SUMMARY_623DK
    .assign(
        positive_mean_gain=lambda df:
            df["mean_rc_gain"]
            >
            0.0
    )
    .groupby(
        "candidate"
    )[
        "positive_mean_gain"
    ]
    .sum()
)

positive_delta_rho_counts_623dk = (
    DEV_FAMILY_CANDIDATE_SUMMARY_623DK
    .assign(
        positive_delta_rho=lambda df:
            df[
                "delta_error_spearman"
            ]
            >
            0.0
    )
    .groupby(
        "candidate"
    )[
        "positive_delta_rho"
    ]
    .sum()
)

DEV_FAMILY_BALANCED_SUMMARY_623DK[
    "families_positive_gain70"
] = (
    DEV_FAMILY_BALANCED_SUMMARY_623DK[
        "candidate"
    ]
    .map(
        positive_gain70_counts_623dk
    )
    .astype(int)
)

DEV_FAMILY_BALANCED_SUMMARY_623DK[
    "families_positive_mean_rc_gain"
] = (
    DEV_FAMILY_BALANCED_SUMMARY_623DK[
        "candidate"
    ]
    .map(
        positive_mean_gain_counts_623dk
    )
    .astype(int)
)

DEV_FAMILY_BALANCED_SUMMARY_623DK[
    "families_positive_delta_rho"
] = (
    DEV_FAMILY_BALANCED_SUMMARY_623DK[
        "candidate"
    ]
    .map(
        positive_delta_rho_counts_623dk
    )
    .astype(int)
)


# ------------------------------------------------------------
# 16. Display family-balanced evidence
#
# Sorting here is ONLY for readability.
# It is NOT final score selection.
# ------------------------------------------------------------

DEV_FAMILY_BALANCED_SUMMARY_623DK = (
    DEV_FAMILY_BALANCED_SUMMARY_623DK
    .sort_values(
        by=[
            "fb_mean_rc_gain",
            "fb_delta_error_spearman",
            "fb_error_spearman",
        ],
        ascending=[
            False,
            False,
            False,
        ]
    )
    .reset_index(drop=True)
)

print(
    "\n=== FAMILY-BALANCED DEVELOPMENT "
    "candidate evidence ==="
)

display(
    DEV_FAMILY_BALANCED_SUMMARY_623DK
)


# ------------------------------------------------------------
# 17. Per-family evidence table
# ------------------------------------------------------------

print(
    "\n=== Per-family mean risk-coverage gain ==="
)

family_gain_table_623dk = (
    DEV_FAMILY_CANDIDATE_SUMMARY_623DK
    .pivot(
        index="candidate",
        columns="family",
        values="mean_rc_gain"
    )
    .reindex(
        CANDIDATE_NAMES_623DK
    )
)

display(
    family_gain_table_623dk
)


# ------------------------------------------------------------
# 18. Corruption-family MAE response
# ------------------------------------------------------------

family_mae_response_623dk = (
    FULL_DEV_SCENARIO_INDEX_623DK
    .groupby(
        "family",
        as_index=False
    )
    .agg(
        scenarios=(
            "scenario_id",
            "count"
        ),
        mean_corrupted_mae=(
            "corrupted_mae",
            "mean"
        ),
        mean_mae_delta_vs_clean=(
            "mae_delta_vs_clean",
            "mean"
        ),
        median_mae_delta_vs_clean=(
            "mae_delta_vs_clean",
            "median"
        ),
        maximum_mae_delta_vs_clean=(
            "mae_delta_vs_clean",
            "max"
        ),
    )
)

print(
    "\n=== Corruption-family RUL response ==="
)

display(
    family_mae_response_623dk
)


# ------------------------------------------------------------
# 19. Verify canonical DEV and validation state unchanged
# ------------------------------------------------------------

canonical_unchanged_623dk = bool(
    np.array_equal(
        CANONICAL_DEV_RAW_623DF[
            PRIMARY_INPUT_COLUMNS
        ].to_numpy(dtype=np.float64),
        canonical_features_before_623dk
    )
)

validation_object_restored_623dk = bool(
    id(
        val_raw_base_620gc
    )
    ==
    validation_object_id_before_623dk
)

validation_values_restored_623dk = bool(
    np.array_equal(
        val_raw_base_620gc[
            PRIMARY_INPUT_COLUMNS
        ].to_numpy(dtype=np.float64),
        validation_features_before_623dk
    )
)


# ------------------------------------------------------------
# 20. Final readiness
# ------------------------------------------------------------

families_present_623dk = sorted(
    FULL_DEV_SCENARIO_INDEX_623DK[
        "family"
    ]
    .unique()
    .tolist()
)

candidates_present_623dk = sorted(
    FULL_DEV_SCENARIO_METRICS_623DK[
        "candidate"
    ]
    .unique()
    .tolist()
)

FULL_DEV_RELIABILITY_EVIDENCE_READY_623DK = bool(
    len(
        FULL_DEV_SCENARIO_INDEX_623DK
    )
    ==
    315

    and

    len(
        FULL_DEV_SCENARIO_METRICS_623DK
    )
    ==
    2520

    and

    set(
        families_present_623dk
    )
    ==
    set(
        FAMILIES_623DK
    )

    and

    set(
        candidates_present_623dk
    )
    ==
    set(
        CANDIDATE_NAMES_623DK
    )

    and

    all_metric_values_finite_623dk

    and

    all_score_cube_finite_623dk

    and

    all_error_matrix_finite_623dk

    and

    canonical_unchanged_623dk

    and

    validation_object_restored_623dk

    and

    validation_values_restored_623dk
)


print("\n==============================================")
print("STEP 6.23D-K INTEGRITY SUMMARY")
print("==============================================")

print(
    "Frozen DEV scenarios evaluated :",
    len(
        FULL_DEV_SCENARIO_INDEX_623DK
    )
)

print(
    "Candidates per scenario         :",
    N_CANDIDATES_623DK
)

print(
    "Scenario/candidate result rows  :",
    len(
        FULL_DEV_SCENARIO_METRICS_623DK
    )
)

print(
    "Expected result rows            :",
    315 * 8
)

print(
    "Families present                :",
    families_present_623dk
)

print(
    "All metrics finite              :",
    all_metric_values_finite_623dk
)

print(
    "Stored score cube finite        :",
    all_score_cube_finite_623dk
)

print(
    "Stored error matrix finite      :",
    all_error_matrix_finite_623dk
)

print(
    "Canonical DEV unchanged         :",
    canonical_unchanged_623dk
)

print(
    "Validation object restored      :",
    validation_object_restored_623dk
)

print(
    "Validation values restored      :",
    validation_values_restored_623dk
)

print(
    "Family-balanced aggregation     : True"
)

print(
    "Full DEV evidence ready         :",
    FULL_DEV_RELIABILITY_EVIDENCE_READY_623DK
)

print(
    "Total runtime (minutes)         :",
    full_elapsed_623dk / 60.0
)

print("\nOfficial test used              : False")
print("Final reliability score selected: False")
print("Reliability threshold selected  : False")
print("Abstention rule created         : False")
print("Files written                   : False")
print("Frozen artifacts modified       : False")

print("=== END ===")

In [ ]:
# ============================================================
# Step 6.23D-K1
# Diagnose non-finite metrics from completed 315-scenario run
#
# IMPORTANT
# ---------
# - Does NOT rerun any corruption scenario.
# - Does NOT rerun MC dropout.
# - Does NOT select a reliability score.
# - Does NOT modify stored results.
# ============================================================

import numpy as np
import pandas as pd

print(
    "=== Step 6.23D-K1 — "
    "Non-finite metric diagnostic ==="
)

# ------------------------------------------------------------
# 1. Confirm expensive experiment objects survived
# ------------------------------------------------------------

required_623dk1 = [
    "FULL_DEV_SCENARIO_METRICS_623DK",
    "FULL_DEV_SCENARIO_INDEX_623DK",
    "FULL_DEV_SCORE_CUBE_623DK",
    "FULL_DEV_ERROR_MATRIX_623DK",
    "CANDIDATE_NAMES_623DK",
]

for name_623dk1 in required_623dk1:
    print(
        f"{name_623dk1:<38}:",
        name_623dk1 in globals()
    )

missing_623dk1 = [
    name
    for name in required_623dk1
    if name not in globals()
]

if missing_623dk1:
    raise RuntimeError(
        "Missing completed-run objects: "
        + str(missing_623dk1)
    )


# ------------------------------------------------------------
# 2. Basic result counts
# ------------------------------------------------------------

print("\n--- Completed-run structure ---")

print(
    "Scenario rows          :",
    len(FULL_DEV_SCENARIO_INDEX_623DK)
)

print(
    "Scenario/candidate rows:",
    len(FULL_DEV_SCENARIO_METRICS_623DK)
)

print(
    "Score cube shape       :",
    FULL_DEV_SCORE_CUBE_623DK.shape
)

print(
    "Error matrix shape     :",
    FULL_DEV_ERROR_MATRIX_623DK.shape
)


# ------------------------------------------------------------
# 3. Check stored numerical arrays separately
# ------------------------------------------------------------

score_cube_finite_623dk1 = bool(
    np.isfinite(
        FULL_DEV_SCORE_CUBE_623DK
    ).all()
)

error_matrix_finite_623dk1 = bool(
    np.isfinite(
        FULL_DEV_ERROR_MATRIX_623DK
    ).all()
)

print("\n--- Stored-array integrity ---")

print(
    "Score cube all finite  :",
    score_cube_finite_623dk1
)

print(
    "Error matrix all finite:",
    error_matrix_finite_623dk1
)


# ------------------------------------------------------------
# 4. Find exactly which metric columns contain NaN/Inf
# ------------------------------------------------------------

metric_columns_623dk1 = [
    "corrupted_mae",
    "mae_delta_vs_clean",
    "score_mean",
    "score_p95",
    "error_spearman",
    "delta_error_spearman",
    "high_low_ratio",
    "top_error_capture_pct",
    "gain90",
    "gain70",
    "gain50",
    "mean_rc_gain",
]

nonfinite_counts_623dk1 = {}

for column_623dk1 in metric_columns_623dk1:

    values_623dk1 = (
        FULL_DEV_SCENARIO_METRICS_623DK[
            column_623dk1
        ]
        .to_numpy(dtype=np.float64)
    )

    nonfinite_counts_623dk1[
        column_623dk1
    ] = int(
        (~np.isfinite(values_623dk1)).sum()
    )

NONFINITE_COUNTS_623DK1 = pd.DataFrame(
    {
        "metric":
            list(
                nonfinite_counts_623dk1.keys()
            ),

        "nonfinite_count":
            list(
                nonfinite_counts_623dk1.values()
            ),
    }
)

print("\n--- Non-finite count by metric ---")

display(
    NONFINITE_COUNTS_623DK1
)


# ------------------------------------------------------------
# 5. Extract all problematic scenario/candidate rows
# ------------------------------------------------------------

numeric_matrix_623dk1 = (
    FULL_DEV_SCENARIO_METRICS_623DK[
        metric_columns_623dk1
    ]
    .to_numpy(dtype=np.float64)
)

problem_mask_623dk1 = (
    ~np.isfinite(
        numeric_matrix_623dk1
    )
).any(axis=1)

PROBLEM_ROWS_623DK1 = (
    FULL_DEV_SCENARIO_METRICS_623DK
    .loc[
        problem_mask_623dk1
    ]
    .copy()
)

print(
    "\nProblematic scenario/candidate rows:",
    len(PROBLEM_ROWS_623DK1)
)

if len(PROBLEM_ROWS_623DK1) > 0:

    columns_to_show_623dk1 = [
        "global_index",
        "scenario_id",
        "family",
        "sensor",
        "severity",
        "candidate",
        "corrupted_mae",
        "error_spearman",
        "delta_error_spearman",
        "high_low_ratio",
        "gain90",
        "gain70",
        "gain50",
        "mean_rc_gain",
    ]

    display(
        PROBLEM_ROWS_623DK1[
            columns_to_show_623dk1
        ]
    )


# ------------------------------------------------------------
# 6. Diagnose whether the relevant score/error arrays
#    are constant in each problematic row
# ------------------------------------------------------------

diagnostic_rows_623dk1 = []

for _, row_623dk1 in PROBLEM_ROWS_623DK1.iterrows():

    scenario_idx_623dk1 = int(
        row_623dk1["global_index"]
    )

    candidate_name_623dk1 = str(
        row_623dk1["candidate"]
    )

    candidate_idx_623dk1 = (
        CANDIDATE_NAMES_623DK.index(
            candidate_name_623dk1
        )
    )

    score_623dk1 = (
        FULL_DEV_SCORE_CUBE_623DK[
            scenario_idx_623dk1,
            :,
            candidate_idx_623dk1
        ]
        .astype(np.float64)
    )

    error_623dk1 = (
        FULL_DEV_ERROR_MATRIX_623DK[
            scenario_idx_623dk1,
            :
        ]
        .astype(np.float64)
    )

    score_range_623dk1 = float(
        np.max(score_623dk1)
        -
        np.min(score_623dk1)
    )

    error_range_623dk1 = float(
        np.max(error_623dk1)
        -
        np.min(error_623dk1)
    )

    score_unique_623dk1 = int(
        np.unique(
            score_623dk1
        ).size
    )

    error_unique_623dk1 = int(
        np.unique(
            error_623dk1
        ).size
    )

    diagnostic_rows_623dk1.append(
        {
            "global_index":
                scenario_idx_623dk1,

            "scenario_id":
                row_623dk1["scenario_id"],

            "family":
                row_623dk1["family"],

            "candidate":
                candidate_name_623dk1,

            "score_unique_values":
                score_unique_623dk1,

            "score_range":
                score_range_623dk1,

            "score_constant":
                score_unique_623dk1 == 1,

            "error_unique_values":
                error_unique_623dk1,

            "error_range":
                error_range_623dk1,

            "error_constant":
                error_unique_623dk1 == 1,
        }
    )

CONSTANT_INPUT_DIAGNOSTIC_623DK1 = pd.DataFrame(
    diagnostic_rows_623dk1
)

print(
    "\n--- Constant-input diagnostic ---"
)

display(
    CONSTANT_INPUT_DIAGNOSTIC_623DK1
)


# ------------------------------------------------------------
# 7. Compact conclusion
# ------------------------------------------------------------

total_nonfinite_metrics_623dk1 = int(
    NONFINITE_COUNTS_623DK1[
        "nonfinite_count"
    ].sum()
)

print("\n========================================")
print("STEP 6.23D-K1 COMPACT SUMMARY")
print("========================================")

print(
    "315 scenarios retained        :",
    len(FULL_DEV_SCENARIO_INDEX_623DK)
    == 315
)

print(
    "2520 candidate rows retained  :",
    len(FULL_DEV_SCENARIO_METRICS_623DK)
    == 2520
)

print(
    "Score cube finite             :",
    score_cube_finite_623dk1
)

print(
    "Error matrix finite           :",
    error_matrix_finite_623dk1
)

print(
    "Problem result rows           :",
    len(PROBLEM_ROWS_623DK1)
)

print(
    "Total non-finite metric cells :",
    total_nonfinite_metrics_623dk1
)

print("\nNo scenarios were rerun.")
print("No score selected.")
print("No threshold selected.")
print("No abstention rule created.")
print("Official test used            : False")
print("Nothing was modified.")
print("=== END ===")

In [ ]:
# ============================================================
# Step 6.23D-K2
# Verify the delta-error operand for the isolated NaN scenario
#
# Diagnostic only:
# - no corruption replay
# - no MC-dropout rerun
# - no score selection
# - no threshold selection
# - no file writing
# ============================================================

import numpy as np
import pandas as pd

print(
    "=== Step 6.23D-K2 — "
    "stuck_009 delta-error verification ==="
)

# ------------------------------------------------------------
# 1. Required objects
# ------------------------------------------------------------

required_623dk2 = [
    "FULL_DEV_ERROR_MATRIX_623DK",
    "dev_seq_meta",
    "X_dev_clean",
    "frozen_gru",
    "predict_gru_batched",
]

for name_623dk2 in required_623dk2:
    print(
        f"{name_623dk2:<34}:",
        name_623dk2 in globals()
    )

missing_623dk2 = [
    name
    for name in required_623dk2
    if name not in globals()
]

if missing_623dk2:
    raise RuntimeError(
        "Missing required object(s): "
        + str(missing_623dk2)
    )


# ------------------------------------------------------------
# 2. Canonical development target
# ------------------------------------------------------------

if "RUL_cap_125" not in dev_seq_meta.columns:
    raise RuntimeError(
        "RUL_cap_125 not found in dev_seq_meta."
    )

target_623dk2 = (
    dev_seq_meta["RUL_cap_125"]
    .to_numpy(dtype=np.float64)
)

if target_623dk2.shape != (14241,):
    raise RuntimeError(
        "Unexpected development target shape: "
        + str(target_623dk2.shape)
    )


# ------------------------------------------------------------
# 3. Reproduce deterministic CLEAN development prediction
#    One deterministic pass only — not MC dropout.
# ------------------------------------------------------------

try:
    model_device_623dk2 = next(
        frozen_gru.parameters()
    ).device
except Exception as exc:
    raise RuntimeError(
        "Could not identify frozen_gru device."
    ) from exc


clean_pred_623dk2 = np.asarray(
    predict_gru_batched(
        frozen_gru,
        X_dev_clean,
        model_device_623dk2,
        batch_size=512,
    ),
    dtype=np.float64,
).reshape(-1)

if clean_pred_623dk2.shape != (14241,):
    raise RuntimeError(
        "Unexpected clean prediction shape: "
        + str(clean_pred_623dk2.shape)
    )


clean_abs_error_623dk2 = np.abs(
    clean_pred_623dk2
    -
    target_623dk2
)


# ------------------------------------------------------------
# 4. Retrieve already-computed stuck_009 error vector
# ------------------------------------------------------------

scenario_index_623dk2 = 279

scenario_error_623dk2 = np.asarray(
    FULL_DEV_ERROR_MATRIX_623DK[
        scenario_index_623dk2
    ],
    dtype=np.float64,
)

if scenario_error_623dk2.shape != (14241,):
    raise RuntimeError(
        "Unexpected scenario error shape: "
        + str(scenario_error_623dk2.shape)
    )


# ------------------------------------------------------------
# 5. Exact delta-error calculation
# ------------------------------------------------------------

delta_error_623dk2 = (
    scenario_error_623dk2
    -
    clean_abs_error_623dk2
)

clean_mae_623dk2 = float(
    np.mean(clean_abs_error_623dk2)
)

scenario_mae_623dk2 = float(
    np.mean(scenario_error_623dk2)
)

delta_min_623dk2 = float(
    np.min(delta_error_623dk2)
)

delta_max_623dk2 = float(
    np.max(delta_error_623dk2)
)

delta_range_623dk2 = float(
    delta_max_623dk2
    -
    delta_min_623dk2
)

delta_std_623dk2 = float(
    np.std(
        delta_error_623dk2,
        ddof=0,
    )
)

max_abs_delta_623dk2 = float(
    np.max(
        np.abs(
            delta_error_623dk2
        )
    )
)

exact_zero_count_623dk2 = int(
    np.sum(
        delta_error_623dk2 == 0.0
    )
)

unique_delta_count_623dk2 = int(
    np.unique(
        delta_error_623dk2
    ).size
)

allclose_zero_623dk2 = bool(
    np.allclose(
        delta_error_623dk2,
        0.0,
        rtol=0.0,
        atol=1e-10,
    )
)


# ------------------------------------------------------------
# 6. Comparison summary
# ------------------------------------------------------------

summary_623dk2 = pd.DataFrame(
    [
        {
            "clean_MAE":
                clean_mae_623dk2,

            "stuck_009_MAE":
                scenario_mae_623dk2,

            "MAE_difference":
                scenario_mae_623dk2
                -
                clean_mae_623dk2,

            "delta_min":
                delta_min_623dk2,

            "delta_max":
                delta_max_623dk2,

            "delta_range":
                delta_range_623dk2,

            "delta_std":
                delta_std_623dk2,

            "max_abs_delta":
                max_abs_delta_623dk2,

            "unique_delta_values":
                unique_delta_count_623dk2,

            "exact_zero_count":
                exact_zero_count_623dk2,

            "allclose_zero_1e-10":
                allclose_zero_623dk2,
        }
    ]
)

print(
    "\n--- CLEAN vs stuck_009 error comparison ---"
)

display(summary_623dk2)


# ------------------------------------------------------------
# 7. Additional exact/allclose checks
# ------------------------------------------------------------

print(
    "\n--- Vector agreement ---"
)

print(
    "Exact array equality      :",
    np.array_equal(
        scenario_error_623dk2,
        clean_abs_error_623dk2,
    )
)

print(
    "Allclose (atol=1e-10)     :",
    np.allclose(
        scenario_error_623dk2,
        clean_abs_error_623dk2,
        rtol=0.0,
        atol=1e-10,
    )
)

print(
    "Maximum absolute difference:",
    max_abs_delta_623dk2
)


# ------------------------------------------------------------
# 8. Interpretation flag
# ------------------------------------------------------------

delta_effectively_constant_623dk2 = bool(
    delta_range_623dk2 <= 1e-10
)

print(
    "\n--- Delta-error diagnostic ---"
)

print(
    "Delta-error effectively constant:",
    delta_effectively_constant_623dk2
)

if delta_effectively_constant_623dk2:

    print(
        "Interpretation: delta-error has no "
        "rank variation in stuck_009."
    )

    print(
        "Therefore delta_error_spearman is "
        "mathematically undefined for this scenario."
    )

else:

    print(
        "Interpretation: delta-error is NOT constant."
    )

    print(
        "The NaN requires further investigation "
        "before any repair."
    )


# ------------------------------------------------------------
# 9. Integrity statement
# ------------------------------------------------------------

print("\n========================================")
print("STEP 6.23D-K2 COMPACT SUMMARY")
print("========================================")

print(
    "Scenario examined           : stuck_009"
)

print(
    "Scenario global index       :",
    scenario_index_623dk2
)

print(
    "Clean error finite          :",
    bool(
        np.isfinite(
            clean_abs_error_623dk2
        ).all()
    )
)

print(
    "Scenario error finite       :",
    bool(
        np.isfinite(
            scenario_error_623dk2
        ).all()
    )
)

print(
    "Delta error finite          :",
    bool(
        np.isfinite(
            delta_error_623dk2
        ).all()
    )
)

print(
    "Delta effectively constant :",
    delta_effectively_constant_623dk2
)

print("\n315-scenario experiment was NOT rerun.")
print("Reliability score selected   : False")
print("Threshold selected           : False")
print("Abstention rule created      : False")
print("Official test used           : False")
print("Files written                : False")
print("Nothing was modified.")
print("=== END ===")

In [ ]:
# ============================================================
# Step 6.23D-K3
# Recover the exact delta_error_spearman construction
# and locate the stored clean-error baseline.
#
# Inspection only.
# ============================================================

import numpy as np
import pandas as pd

print(
    "=== Step 6.23D-K3 — "
    "Exact delta-error construction recovery ==="
)


# ------------------------------------------------------------
# 1. Search notebook history for delta_error_spearman
# ------------------------------------------------------------

print(
    "\n--- Notebook-history search for "
    "delta_error_spearman ---"
)

history_623dk3 = globals().get("In", [])

matching_cells_623dk3 = []

for idx_623dk3, source_623dk3 in enumerate(
    history_623dk3
):
    if not isinstance(source_623dk3, str):
        continue

    if (
        "delta_error_spearman"
        in source_623dk3
        and "Step 6.23D-K3"
        not in source_623dk3
    ):
        matching_cells_623dk3.append(
            idx_623dk3
        )


print(
    "Matching cells:",
    matching_cells_623dk3
)


# Show the most recent relevant cells
for cell_idx_623dk3 in matching_cells_623dk3[-4:]:

    source_623dk3 = history_623dk3[
        cell_idx_623dk3
    ]

    lines_623dk3 = (
        source_623dk3
        .splitlines()
    )

    hit_lines_623dk3 = [
        i
        for i, line
        in enumerate(lines_623dk3)
        if "delta_error_spearman" in line
    ]

    print(
        "\n========================================"
    )
    print(
        "CELL:",
        cell_idx_623dk3
    )
    print(
        "========================================"
    )

    for hit_623dk3 in hit_lines_623dk3:

        start_623dk3 = max(
            0,
            hit_623dk3 - 12
        )

        end_623dk3 = min(
            len(lines_623dk3),
            hit_623dk3 + 13
        )

        for line_no_623dk3 in range(
            start_623dk3,
            end_623dk3
        ):
            print(
                f"{line_no_623dk3 + 1:04d}: "
                f"{lines_623dk3[line_no_623dk3]}"
            )


# ------------------------------------------------------------
# 2. Retrieve stuck_009 stored error vector
# ------------------------------------------------------------

if (
    "FULL_DEV_ERROR_MATRIX_623DK"
    not in globals()
):
    raise RuntimeError(
        "FULL_DEV_ERROR_MATRIX_623DK "
        "is unavailable."
    )


scenario_error_623dk3 = np.asarray(
    FULL_DEV_ERROR_MATRIX_623DK[279],
    dtype=np.float64,
)


if scenario_error_623dk3.shape != (14241,):
    raise RuntimeError(
        "Unexpected stuck_009 error-vector shape."
    )


# ------------------------------------------------------------
# 3. Search existing 14241-element arrays for possible
#    CLEAN error baselines.
# ------------------------------------------------------------

print(
    "\n--- Existing sequence-aligned candidate "
    "baseline objects ---"
)

candidate_rows_623dk3 = []


def inspect_candidate_623dk3(
    object_name,
    array_value,
    source_kind,
):

    try:
        arr_623dk3 = np.asarray(
            array_value,
            dtype=np.float64,
        ).reshape(-1)

    except Exception:
        return

    if arr_623dk3.shape != (14241,):
        return

    if not np.isfinite(
        arr_623dk3
    ).all():
        return

    delta_623dk3 = (
        scenario_error_623dk3
        -
        arr_623dk3
    )

    candidate_rows_623dk3.append(
        {
            "object":
                object_name,

            "source":
                source_kind,

            "max_abs_diff_vs_stuck009":
                float(
                    np.max(
                        np.abs(
                            delta_623dk3
                        )
                    )
                ),

            "mean_abs_diff_vs_stuck009":
                float(
                    np.mean(
                        np.abs(
                            delta_623dk3
                        )
                    )
                ),

            "delta_unique_values":
                int(
                    np.unique(
                        delta_623dk3
                    ).size
                ),

            "delta_range":
                float(
                    np.max(
                        delta_623dk3
                    )
                    -
                    np.min(
                        delta_623dk3
                    )
                ),

            "exact_match":
                bool(
                    np.array_equal(
                        scenario_error_623dk3,
                        arr_623dk3,
                    )
                ),
        }
    )


# Search standalone arrays / Series.
for name_623dk3, obj_623dk3 in list(
    globals().items()
):

    if name_623dk3.startswith("__"):
        continue

    lower_name_623dk3 = (
        name_623dk3.lower()
    )

    # Restrict to relevant-looking names.
    if not any(
        token_623dk3 in lower_name_623dk3
        for token_623dk3 in [
            "error",
            "clean",
            "dev",
            "abs",
        ]
    ):
        continue

    if isinstance(
        obj_623dk3,
        (np.ndarray, pd.Series),
    ):
        shape_623dk3 = np.shape(
            obj_623dk3
        )

        if shape_623dk3 == (14241,):

            inspect_candidate_623dk3(
                name_623dk3,
                obj_623dk3,
                "array_or_series",
            )


# ------------------------------------------------------------
# 4. Search relevant DataFrame columns
# ------------------------------------------------------------

for name_623dk3, obj_623dk3 in list(
    globals().items()
):

    if not isinstance(
        obj_623dk3,
        pd.DataFrame,
    ):
        continue

    if len(obj_623dk3) != 14241:
        continue

    for column_623dk3 in obj_623dk3.columns:

        column_lower_623dk3 = str(
            column_623dk3
        ).lower()

        if not any(
            token_623dk3
            in column_lower_623dk3
            for token_623dk3 in [
                "error",
                "absolute",
                "abs_error",
            ]
        ):
            continue

        inspect_candidate_623dk3(
            (
                f"{name_623dk3}"
                f"['{column_623dk3}']"
            ),
            obj_623dk3[
                column_623dk3
            ],
            "dataframe_column",
        )


# ------------------------------------------------------------
# 5. Rank possible clean-error baselines
# ------------------------------------------------------------

candidate_df_623dk3 = pd.DataFrame(
    candidate_rows_623dk3
)

if len(candidate_df_623dk3) == 0:

    print(
        "No sequence-aligned error baseline "
        "was found in the current namespace."
    )

else:

    candidate_df_623dk3 = (
        candidate_df_623dk3
        .sort_values(
            [
                "max_abs_diff_vs_stuck009",
                "mean_abs_diff_vs_stuck009",
            ]
        )
        .reset_index(drop=True)
    )

    display(
        candidate_df_623dk3.head(20)
    )


# ------------------------------------------------------------
# 6. Exact-match report
# ------------------------------------------------------------

if len(candidate_df_623dk3) > 0:

    exact_matches_623dk3 = (
        candidate_df_623dk3[
            candidate_df_623dk3[
                "exact_match"
            ]
        ]
    )

    print(
        "\n--- Exact stored baseline matches ---"
    )

    if len(exact_matches_623dk3) == 0:

        print(
            "No exact 14241-element stored error "
            "baseline matched stuck_009."
        )

    else:

        display(
            exact_matches_623dk3
        )


# ------------------------------------------------------------
# 7. Integrity
# ------------------------------------------------------------

print(
    "\n========================================"
)
print(
    "STEP 6.23D-K3 COMPACT SUMMARY"
)
print(
    "========================================"
)

print(
    "Scenario inspected          : stuck_009"
)

print(
    "Scenario global index       : 279"
)

print(
    "Stored scenario error finite:",
    bool(
        np.isfinite(
            scenario_error_623dk3
        ).all()
    )
)

print(
    "History matches found       :",
    len(
        matching_cells_623dk3
    )
)

print(
    "Candidate baselines found   :",
    len(
        candidate_df_623dk3
    )
)

print(
    "315 scenarios rerun         : False"
)

print(
    "Reliability score selected  : False"
)

print(
    "Threshold selected          : False"
)

print(
    "Abstention rule created     : False"
)

print(
    "Official test used          : False"
)

print(
    "Files written               : False"
)

print(
    "Nothing was modified."
)

print(
    "=== END ==="
)

In [ ]:
# ============================================================
# Step 6.23D-K4
# Exact delta_error_spearman source recovery
#
# Inspection only:
# - no scenario rerun
# - no repair
# - no score selection
# - no files written
# ============================================================

print(
    "=== Step 6.23D-K4 — "
    "Exact delta_error_spearman source recovery ==="
)

history_623dk4 = globals().get("In", [])


# ------------------------------------------------------------
# Terms whose original construction we need to recover
# ------------------------------------------------------------

search_terms_623dk4 = [
    "delta_error_spearman",
    "CLEAN_DEV_ABS_ERROR_623DK",
    "delta_error",
    "spearman",
]


# ------------------------------------------------------------
# Search only relevant earlier notebook cells
# ------------------------------------------------------------

candidate_cells_623dk4 = []

for cell_idx_623dk4, source_623dk4 in enumerate(
    history_623dk4
):

    if not isinstance(
        source_623dk4,
        str,
    ):
        continue

    # Do not inspect this diagnostic itself.
    if "Step 6.23D-K4" in source_623dk4:
        continue

    if (
        "delta_error_spearman"
        in source_623dk4
        or
        "CLEAN_DEV_ABS_ERROR_623DK"
        in source_623dk4
    ):
        candidate_cells_623dk4.append(
            cell_idx_623dk4
        )


print(
    "\nRelevant cells:",
    candidate_cells_623dk4
)


# ------------------------------------------------------------
# Print compact excerpts around every important occurrence
# ------------------------------------------------------------

for cell_idx_623dk4 in candidate_cells_623dk4:

    source_623dk4 = history_623dk4[
        cell_idx_623dk4
    ]

    lines_623dk4 = (
        source_623dk4.splitlines()
    )

    hit_indices_623dk4 = []

    for line_idx_623dk4, line_623dk4 in enumerate(
        lines_623dk4
    ):

        if any(
            term_623dk4 in line_623dk4
            for term_623dk4
            in search_terms_623dk4
        ):
            hit_indices_623dk4.append(
                line_idx_623dk4
            )


    if not hit_indices_623dk4:
        continue


    print(
        "\n"
        "============================================"
    )
    print(
        f"CELL {cell_idx_623dk4}"
    )
    print(
        "============================================"
    )


    # Merge overlapping source windows so output stays compact.
    source_ranges_623dk4 = []

    for hit_623dk4 in hit_indices_623dk4:

        start_623dk4 = max(
            0,
            hit_623dk4 - 15,
        )

        end_623dk4 = min(
            len(lines_623dk4),
            hit_623dk4 + 16,
        )

        if (
            source_ranges_623dk4
            and
            start_623dk4
            <= source_ranges_623dk4[-1][1]
        ):

            source_ranges_623dk4[-1] = (
                source_ranges_623dk4[-1][0],
                max(
                    source_ranges_623dk4[-1][1],
                    end_623dk4,
                ),
            )

        else:

            source_ranges_623dk4.append(
                (
                    start_623dk4,
                    end_623dk4,
                )
            )


    for (
        start_623dk4,
        end_623dk4,
    ) in source_ranges_623dk4:

        print(
            "\n--- source excerpt ---"
        )

        for line_no_623dk4 in range(
            start_623dk4,
            end_623dk4,
        ):

            print(
                f"{line_no_623dk4 + 1:04d}: "
                f"{lines_623dk4[line_no_623dk4]}"
            )


# ------------------------------------------------------------
# Also inspect the stored problematic row directly
# ------------------------------------------------------------

print(
    "\n--- Stored stuck_009 metric rows ---"
)

if (
    "FULL_DEV_SCENARIO_METRICS_623DK"
    in globals()
):

    metrics_623dk4 = (
        FULL_DEV_SCENARIO_METRICS_623DK
    )

    stuck_rows_623dk4 = (
        metrics_623dk4[
            metrics_623dk4[
                "scenario_id"
            ].astype(str)
            == "stuck_009"
        ]
        .copy()
    )

    columns_623dk4 = [
        col_623dk4
        for col_623dk4 in [
            "scenario_id",
            "family",
            "sensor",
            "severity",
            "candidate",
            "error_spearman",
            "delta_error_spearman",
            "high_low_ratio",
            "gain90",
            "gain70",
            "gain50",
            "mean_rc_gain",
        ]
        if col_623dk4
        in stuck_rows_623dk4.columns
    ]

    display(
        stuck_rows_623dk4[
            columns_623dk4
        ]
    )

else:

    print(
        "FULL_DEV_SCENARIO_METRICS_623DK "
        "not available."
    )


# ------------------------------------------------------------
# Integrity statement
# ------------------------------------------------------------

print(
    "\n============================================"
)
print(
    "STEP 6.23D-K4 COMPACT SUMMARY"
)
print(
    "============================================"
)

print(
    "Relevant history cells found :",
    len(candidate_cells_623dk4)
)

print(
    "315 scenarios rerun          : False"
)

print(
    "Metrics modified             : False"
)

print(
    "Reliability score selected   : False"
)

print(
    "Threshold selected           : False"
)

print(
    "Abstention rule created      : False"
)

print(
    "Official test used           : False"
)

print(
    "Files written                : False"
)

print(
    "Nothing was modified."
)

print(
    "=== END ==="
)

In [ ]:
# ============================================================
# Step 6.23D-K5
# Recover exact delta-error Spearman computation
#
# Inspection only:
# - no scenario rerun
# - no metric repair
# - no score selection
# - no threshold selection
# - no files written
# ============================================================

print(
    "=== Step 6.23D-K5 — "
    "Exact delta-error Spearman computation ==="
)

history_623dk5 = globals().get("In", [])


# ------------------------------------------------------------
# We already know Cell 151 contains the original computation.
# Recover it safely from notebook history.
# ------------------------------------------------------------

target_cell_idx_623dk5 = 151

if (
    target_cell_idx_623dk5
    >= len(history_623dk5)
):
    raise RuntimeError(
        "Cell 151 is not available in notebook history."
    )

source_623dk5 = history_623dk5[
    target_cell_idx_623dk5
]

lines_623dk5 = source_623dk5.splitlines()


# ------------------------------------------------------------
# Find exact delta_error_spearman storage locations
# ------------------------------------------------------------

delta_store_hits_623dk5 = [
    idx_623dk5
    for idx_623dk5, line_623dk5
    in enumerate(lines_623dk5)
    if "delta_error_spearman" in line_623dk5
]

print(
    "\nCell:",
    target_cell_idx_623dk5
)

print(
    "delta_error_spearman occurrences:",
    [
        x_623dk5 + 1
        for x_623dk5
        in delta_store_hits_623dk5
    ]
)


# ------------------------------------------------------------
# For every stored occurrence, inspect preceding source and
# retain only lines useful for reconstructing rho.
# ------------------------------------------------------------

important_terms_623dk5 = [
    "rho",
    "spearman",
    "delta",
    "error",
    "abs_error",
    "clean",
    "scenario",
    "constant",
    "unique",
    "std",
]


for hit_623dk5 in delta_store_hits_623dk5:

    # Look sufficiently far backward to capture construction.
    start_623dk5 = max(
        0,
        hit_623dk5 - 120,
    )

    end_623dk5 = min(
        len(lines_623dk5),
        hit_623dk5 + 8,
    )

    print(
        "\n"
        "=============================================="
    )

    print(
        f"Context for source line {hit_623dk5 + 1}"
    )

    print(
        "=============================================="
    )

    for idx_623dk5 in range(
        start_623dk5,
        end_623dk5,
    ):

        text_623dk5 = (
            lines_623dk5[idx_623dk5]
        )

        if any(
            term_623dk5.lower()
            in text_623dk5.lower()
            for term_623dk5
            in important_terms_623dk5
        ):

            print(
                f"{idx_623dk5 + 1:04d}: "
                f"{text_623dk5}"
            )


# ------------------------------------------------------------
# Independently locate every spearmanr call in Cell 151.
# ------------------------------------------------------------

print(
    "\n--- All scipy/pandas Spearman-related lines "
    "in Cell 151 ---"
)

for idx_623dk5, text_623dk5 in enumerate(
    lines_623dk5
):

    lower_623dk5 = text_623dk5.lower()

    if (
        "spearmanr" in lower_623dk5
        or
        'method="spearman"' in lower_623dk5
        or
        "method='spearman'" in lower_623dk5
    ):

        lo_623dk5 = max(
            0,
            idx_623dk5 - 8,
        )

        hi_623dk5 = min(
            len(lines_623dk5),
            idx_623dk5 + 9,
        )

        print(
            f"\n--- around line "
            f"{idx_623dk5 + 1} ---"
        )

        for j_623dk5 in range(
            lo_623dk5,
            hi_623dk5,
        ):
            print(
                f"{j_623dk5 + 1:04d}: "
                f"{lines_623dk5[j_623dk5]}"
            )


# ------------------------------------------------------------
# Search for explicit construction of the delta-error vector.
# ------------------------------------------------------------

print(
    "\n--- Delta-error construction candidates ---"
)

for idx_623dk5, text_623dk5 in enumerate(
    lines_623dk5
):

    lower_623dk5 = text_623dk5.lower()

    if (
        "delta_error" in lower_623dk5
        or
        (
            "scenario_error" in lower_623dk5
            and "-" in text_623dk5
        )
        or
        (
            "clean" in lower_623dk5
            and "error" in lower_623dk5
            and "-" in text_623dk5
        )
    ):

        lo_623dk5 = max(
            0,
            idx_623dk5 - 5,
        )

        hi_623dk5 = min(
            len(lines_623dk5),
            idx_623dk5 + 6,
        )

        print(
            f"\n--- around line "
            f"{idx_623dk5 + 1} ---"
        )

        for j_623dk5 in range(
            lo_623dk5,
            hi_623dk5,
        ):
            print(
                f"{j_623dk5 + 1:04d}: "
                f"{lines_623dk5[j_623dk5]}"
            )


# ------------------------------------------------------------
# Integrity summary
# ------------------------------------------------------------

print(
    "\n=============================================="
)
print(
    "STEP 6.23D-K5 COMPACT SUMMARY"
)
print(
    "=============================================="
)

print(
    "Original computation cell    :",
    target_cell_idx_623dk5
)

print(
    "315 scenarios rerun          : False"
)

print(
    "Metric values modified       : False"
)

print(
    "Reliability score selected   : False"
)

print(
    "Threshold selected           : False"
)

print(
    "Abstention rule created      : False"
)

print(
    "Official test used           : False"
)

print(
    "Files written                : False"
)

print(
    "Nothing was modified."
)

print(
    "=== END ==="
)

In [ ]:
# ============================================================
# Step 6.23D-K6
# Exact source windows around delta_error_spearman
#
# Inspection only.
# Nothing is recomputed or modified.
# ============================================================

print(
    "=== Step 6.23D-K6 — "
    "Exact delta-error source windows ==="
)

history_623dk6 = globals().get("In", [])

cell_idx_623dk6 = 151

if cell_idx_623dk6 >= len(history_623dk6):
    raise RuntimeError(
        "Cell 151 is unavailable in notebook history."
    )

source_623dk6 = history_623dk6[cell_idx_623dk6]
lines_623dk6 = source_623dk6.splitlines()


def print_window_623dk6(start_line, end_line, title):
    """
    start_line/end_line are 1-based inclusive source lines.
    """
    print("\n" + "=" * 64)
    print(title)
    print("=" * 64)

    start_idx = max(0, start_line - 1)
    end_idx = min(len(lines_623dk6), end_line)

    for idx in range(start_idx, end_idx):
        print(
            f"{idx + 1:04d}: "
            f"{lines_623dk6[idx]}"
        )


# ------------------------------------------------------------
# The first two occurrences are close together and are likely
# the important scenario-response computation.
# ------------------------------------------------------------

print_window_623dk6(
    895,
    965,
    "SOURCE LINES 895–965"
)


# ------------------------------------------------------------
# Third occurrence: likely later aggregation/reporting.
# ------------------------------------------------------------

print_window_623dk6(
    1068,
    1098,
    "SOURCE LINES 1068–1098"
)


# ------------------------------------------------------------
# Also list every line assigning to rho-like variables.
# ------------------------------------------------------------

print(
    "\n" + "=" * 64
)
print(
    "RHO / SPEARMAN ASSIGNMENT LINES"
)
print(
    "=" * 64
)

for idx_623dk6, line_623dk6 in enumerate(
    lines_623dk6
):
    stripped_623dk6 = line_623dk6.strip()
    lower_623dk6 = stripped_623dk6.lower()

    if (
        "spearmanr(" in lower_623dk6
        or
        "rho =" in lower_623dk6
        or
        "rho=" in lower_623dk6
    ):
        print(
            f"{idx_623dk6 + 1:04d}: "
            f"{line_623dk6}"
        )


print(
    "\n=============================================="
)
print(
    "STEP 6.23D-K6 COMPACT SUMMARY"
)
print(
    "=============================================="
)

print("Source cell inspected       : 151")
print("315 scenarios rerun         : False")
print("Metrics modified            : False")
print("Reliability score selected  : False")
print("Threshold selected          : False")
print("Abstention rule created     : False")
print("Official test used          : False")
print("Files written               : False")
print("Nothing was modified.")
print("=== END ===")

In [ ]:
# ============================================================
# Step 6.23D-K6A
# Isolate exact scenario-level delta-error Spearman calculation
#
# Inspection only.
# No recomputation, no modification, no file writing.
# ============================================================

print(
    "=== Step 6.23D-K6A — "
    "Exact scenario-level delta-error calculation ==="
)

source_623dk6a = In[151]
lines_623dk6a = source_623dk6a.splitlines()

# We only need the small source region surrounding
# the scenario-response calculation.
START_LINE_623DK6A = 915
END_LINE_623DK6A = 962

for line_no_623dk6a in range(
    START_LINE_623DK6A,
    END_LINE_623DK6A + 1
):
    if line_no_623dk6a <= len(lines_623dk6a):
        print(
            f"{line_no_623dk6a:04d}: "
            f"{lines_623dk6a[line_no_623dk6a - 1]}"
        )

print("\n--- Exact important-line search ---")

keywords_623dk6a = [
    "spearmanr",
    "rho",
    "delta_error",
    "mae_delta",
    "corrupted_mae",
    "clean",
]

for idx_623dk6a, line_623dk6a in enumerate(
    lines_623dk6a,
    start=1
):
    if 915 <= idx_623dk6a <= 962:
        lower_623dk6a = line_623dk6a.lower()

        if any(
            keyword_623dk6a in lower_623dk6a
            for keyword_623dk6a in keywords_623dk6a
        ):
            print(
                f"{idx_623dk6a:04d}: "
                f"{line_623dk6a}"
            )

print("\n--- Integrity ---")
print("Source cell             : 151")
print("315 scenarios rerun     : False")
print("Metrics modified        : False")
print("Score selected          : False")
print("Threshold selected      : False")
print("Abstention rule created : False")
print("Official test used      : False")
print("Files written           : False")
print("Nothing was modified.")
print("=== END ===")

In [ ]:
# ============================================================
# Step 6.23D-K7
# Locate exact delta_error_spearman construction in FULL DEV run
#
# Inspection only.
# No scenario rerun.
# No metric modification.
# No files written.
# ============================================================

print(
    "=== Step 6.23D-K7 — "
    "Locate FULL-DEV delta_error_spearman construction ==="
)

source_623dk7 = In[165]
lines_623dk7 = source_623dk7.splitlines()

target_623dk7 = "delta_error_spearman"

hits_623dk7 = [
    i
    for i, line in enumerate(lines_623dk7, start=1)
    if target_623dk7 in line
]

print("Source cell                  : 165")
print("delta_error_spearman hits    :", hits_623dk7)

for hit_623dk7 in hits_623dk7:
    print("\n" + "=" * 70)
    print(f"CONTEXT AROUND SOURCE LINE {hit_623dk7}")
    print("=" * 70)

    start_623dk7 = max(1, hit_623dk7 - 25)
    end_623dk7 = min(
        len(lines_623dk7),
        hit_623dk7 + 20
    )

    for line_no_623dk7 in range(
        start_623dk7,
        end_623dk7 + 1
    ):
        marker_623dk7 = (
            ">>>"
            if line_no_623dk7 == hit_623dk7
            else "   "
        )

        print(
            f"{marker_623dk7} "
            f"{line_no_623dk7:04d}: "
            f"{lines_623dk7[line_no_623dk7 - 1]}"
        )

# ------------------------------------------------------------
# Also locate the correlation helper used by this cell.
# ------------------------------------------------------------

print("\n--- Correlation-related lines in In[165] ---")

search_terms_623dk7 = [
    "spearman",
    "safe_corr",
    "corr(",
    "score_delta",
    "error_delta",
    "mae_delta_vs_clean",
]

for i_623dk7, line_623dk7 in enumerate(
    lines_623dk7,
    start=1
):
    low_623dk7 = line_623dk7.lower()

    if any(
        term_623dk7.lower() in low_623dk7
        for term_623dk7 in search_terms_623dk7
    ):
        print(
            f"{i_623dk7:04d}: "
            f"{line_623dk7}"
        )

print("\n--- Integrity ---")
print("315 scenarios rerun       : False")
print("Stored metrics modified   : False")
print("Reliability score selected: False")
print("Threshold selected        : False")
print("Abstention rule created   : False")
print("Official test used        : False")
print("Files written             : False")
print("Nothing was modified.")
print("=== END ===")

In [ ]:
# ============================================================
# Step 6.23D-K7A
# Show ONLY the exact first delta_error_spearman construction
#
# Inspection only.
# Nothing modified.
# ============================================================

print(
    "=== Step 6.23D-K7A — "
    "Exact first delta_error_spearman construction ==="
)

source_623dk7a = In[165]
lines_623dk7a = source_623dk7a.splitlines()

# First occurrence is at line 995.
# Show only the immediately relevant region.
for line_no_623dk7a in range(988, 1007):

    if line_no_623dk7a <= len(lines_623dk7a):

        marker_623dk7a = (
            ">>>"
            if line_no_623dk7a == 995
            else "   "
        )

        print(
            f"{marker_623dk7a} "
            f"{line_no_623dk7a:04d}: "
            f"{lines_623dk7a[line_no_623dk7a - 1]}"
        )

print("\nNothing was modified.")
print("Official test used : False")
print("Files written      : False")
print("=== END ===")

In [ ]:
# ============================================================
# Step 6.23D-K8
# Root-cause inspection for delta_error_spearman NaN
#
# Inspection only.
# No scenario rerun.
# No metric modification.
# No files written.
# ============================================================

print(
    "=== Step 6.23D-K8 — "
    "delta-error Spearman root-cause inspection ==="
)

source_623dk8 = In[165]
lines_623dk8 = source_623dk8.splitlines()

# ------------------------------------------------------------
# 1. Find every occurrence of delta_absolute_error_623dk
# ------------------------------------------------------------

term_623dk8 = "delta_absolute_error_623dk"

hits_delta_623dk8 = [
    i
    for i, line in enumerate(lines_623dk8, start=1)
    if term_623dk8 in line
]

print(
    "\ndelta_absolute_error_623dk occurrences:",
    hits_delta_623dk8
)

for hit_623dk8 in hits_delta_623dk8:

    print("\n" + "=" * 70)
    print(
        f"CONTEXT AROUND delta_absolute_error "
        f"LINE {hit_623dk8}"
    )
    print("=" * 70)

    start_623dk8 = max(
        1,
        hit_623dk8 - 12
    )

    end_623dk8 = min(
        len(lines_623dk8),
        hit_623dk8 + 8
    )

    for line_no_623dk8 in range(
        start_623dk8,
        end_623dk8 + 1
    ):

        marker_623dk8 = (
            ">>>"
            if line_no_623dk8 == hit_623dk8
            else "   "
        )

        print(
            f"{marker_623dk8} "
            f"{line_no_623dk8:04d}: "
            f"{lines_623dk8[line_no_623dk8 - 1]}"
        )


# ------------------------------------------------------------
# 2. Locate exact spearman_623dk definition
# ------------------------------------------------------------

spearman_def_hits_623dk8 = [
    i
    for i, line in enumerate(lines_623dk8, start=1)
    if "def spearman_623dk" in line
]

print(
    "\nspearman_623dk definition line(s):",
    spearman_def_hits_623dk8
)

for hit_623dk8 in spearman_def_hits_623dk8:

    print("\n" + "=" * 70)
    print(
        f"SPEARMAN HELPER SOURCE FROM LINE "
        f"{hit_623dk8}"
    )
    print("=" * 70)

    start_623dk8 = hit_623dk8

    end_623dk8 = min(
        len(lines_623dk8),
        hit_623dk8 + 30
    )

    for line_no_623dk8 in range(
        start_623dk8,
        end_623dk8 + 1
    ):
        print(
            f"{line_no_623dk8:04d}: "
            f"{lines_623dk8[line_no_623dk8 - 1]}"
        )


# ------------------------------------------------------------
# 3. Stored stuck_009 numerical diagnostic
#    No recalculation of the 315 scenarios.
# ------------------------------------------------------------

scenario_idx_623dk8 = 279

scenario_error_623dk8 = np.asarray(
    FULL_DEV_ERROR_MATRIX_623DK[
        scenario_idx_623dk8
    ],
    dtype=np.float64
)

print("\n--- Stored stuck_009 error vector ---")
print("shape        :", scenario_error_623dk8.shape)
print("finite       :", np.isfinite(
    scenario_error_623dk8
).all())
print("minimum      :", scenario_error_623dk8.min())
print("maximum      :", scenario_error_623dk8.max())
print("std          :", scenario_error_623dk8.std())
print(
    "unique values:",
    np.unique(scenario_error_623dk8).size
)


print("\n--- Integrity ---")
print("315 scenarios rerun        : False")
print("Stored metrics modified    : False")
print("Reliability score selected : False")
print("Threshold selected         : False")
print("Abstention rule created    : False")
print("Official test used         : False")
print("Files written              : False")
print("Nothing was modified.")
print("=== END ===")

In [ ]:
# ============================================================
# Step 6.23D-K9
# Final diagnosis of stuck_009 delta-error Spearman NaN
#
# Inspection only.
# No scenarios rerun.
# No stored results modified.
# ============================================================

import numpy as np

print(
    "=== Step 6.23D-K9 — "
    "Final stuck_009 Spearman diagnosis ==="
)

# ------------------------------------------------------------
# 1. Show exact spearman_623dk helper
# ------------------------------------------------------------

source_623dk9 = In[165]
lines_623dk9 = source_623dk9.splitlines()

helper_hits_623dk9 = [
    i
    for i, line in enumerate(lines_623dk9, start=1)
    if "def spearman_623dk" in line
]

print("\nSpearman helper definition line(s):",
      helper_hits_623dk9)

for hit_623dk9 in helper_hits_623dk9:

    print("\n--- spearman_623dk source ---")

    for line_no_623dk9 in range(
        hit_623dk9,
        min(hit_623dk9 + 18, len(lines_623dk9)) + 1
    ):
        print(
            f"{line_no_623dk9:04d}: "
            f"{lines_623dk9[line_no_623dk9 - 1]}"
        )


# ------------------------------------------------------------
# 2. Recover stuck_009 delta-error vector
# ------------------------------------------------------------

SCENARIO_INDEX_623DK9 = 279

scenario_error_623dk9 = np.asarray(
    FULL_DEV_ERROR_MATRIX_623DK[
        SCENARIO_INDEX_623DK9
    ],
    dtype=np.float64
)

clean_error_623dk9 = np.asarray(
    CLEAN_DEV_ABS_ERROR_623DK,
    dtype=np.float64
)

delta_error_623dk9 = (
    scenario_error_623dk9
    - clean_error_623dk9
)

print("\n--- stuck_009 delta-error statistics ---")
print("windows              :", delta_error_623dk9.size)
print("finite               :", np.isfinite(delta_error_623dk9).all())
print("minimum              :", delta_error_623dk9.min())
print("maximum              :", delta_error_623dk9.max())
print("range                :", np.ptp(delta_error_623dk9))
print("standard deviation   :", delta_error_623dk9.std())
print("unique values        :", np.unique(delta_error_623dk9).size)
print(
    "exact zeros          :",
    np.sum(delta_error_623dk9 == 0.0)
)
print(
    "|delta| <= 1e-6      :",
    np.sum(np.abs(delta_error_623dk9) <= 1e-6)
)
print(
    "|delta| <= 1e-5      :",
    np.sum(np.abs(delta_error_623dk9) <= 1e-5)
)


# ------------------------------------------------------------
# 3. Evaluate the EXISTING helper for all 8 candidates
# ------------------------------------------------------------

print("\n--- Existing helper output for stuck_009 ---")

for candidate_idx_623dk9, candidate_name_623dk9 in enumerate(
    CANDIDATE_NAMES_623DK
):

    score_623dk9 = np.asarray(
        FULL_DEV_SCORE_CUBE_623DK[
            SCENARIO_INDEX_623DK9,
            :,
            candidate_idx_623dk9
        ],
        dtype=np.float64
    )

    rho_623dk9 = spearman_623dk(
        score_623dk9,
        delta_error_623dk9
    )

    print(
        f"{candidate_name_623dk9:20s} "
        f"| score_unique={np.unique(score_623dk9).size:5d} "
        f"| rho={rho_623dk9}"
    )


print("\n--- Integrity ---")
print("315 scenarios rerun        : False")
print("Stored metrics modified    : False")
print("Reliability score selected : False")
print("Threshold selected         : False")
print("Abstention rule created    : False")
print("Official test used         : False")
print("Files written              : False")
print("Nothing was modified.")
print("=== END ===")

In [ ]:
# ============================================================
# Step 6.23D-K10
# Final Spearman-helper + stuck_009 numerical verification
#
# Inspection only.
# No scenario rerun.
# No stored object modified.
# No file written.
# ============================================================

import numpy as np
import pandas as pd

print(
    "=== Step 6.23D-K10 — "
    "Spearman helper and stuck_009 verification ==="
)

# ------------------------------------------------------------
# A. Show ONLY remaining spearman_623dk helper source
# ------------------------------------------------------------

src_623dk10 = In[165].splitlines()

print("\n--- Remaining spearman_623dk source ---")

for line_no in range(554, 580):
    if line_no <= len(src_623dk10):
        print(
            f"{line_no:04d}: "
            f"{src_623dk10[line_no - 1]}"
        )


# ------------------------------------------------------------
# B. Reconstruct stored stuck_009 delta error
# ------------------------------------------------------------

scenario_index = 279

scenario_error = np.asarray(
    FULL_DEV_ERROR_MATRIX_623DK[scenario_index],
    dtype=np.float64
)

clean_error = np.asarray(
    CLEAN_DEV_ABS_ERROR_623DK,
    dtype=np.float64
)

delta_error = scenario_error - clean_error

print("\n--- stuck_009 delta-error structure ---")
print("windows              :", delta_error.size)
print("finite               :", np.isfinite(delta_error).all())
print("unique values        :", np.unique(delta_error).size)
print("exact zero count     :", int(np.sum(delta_error == 0.0)))
print("nonzero count        :", int(np.sum(delta_error != 0.0)))
print("minimum              :", float(delta_error.min()))
print("maximum              :", float(delta_error.max()))
print("range                :", float(np.ptp(delta_error)))
print("std                  :", float(np.std(delta_error)))


# ------------------------------------------------------------
# C. Compare correlation implementations for all 8 candidates
# ------------------------------------------------------------

rows = []

for candidate_idx, candidate_name in enumerate(
    CANDIDATE_NAMES_623DK
):

    score = np.asarray(
        FULL_DEV_SCORE_CUBE_623DK[
            scenario_index,
            :,
            candidate_idx
        ],
        dtype=np.float64
    )

    helper_rho = spearman_623dk(
        score,
        delta_error
    )

    pandas_rho = pd.Series(score).corr(
        pd.Series(delta_error),
        method="spearman"
    )

    score_rank = pd.Series(score).rank(
        method="average"
    ).to_numpy()

    delta_rank = pd.Series(delta_error).rank(
        method="average"
    ).to_numpy()

    rank_pearson = np.corrcoef(
        score_rank,
        delta_rank
    )[0, 1]

    rows.append(
        {
            "candidate": candidate_name,
            "score_unique": np.unique(score).size,
            "delta_unique": np.unique(delta_error).size,
            "helper_rho": helper_rho,
            "pandas_rho": pandas_rho,
            "rank_pearson": rank_pearson,
        }
    )

comparison_623dk10 = pd.DataFrame(rows)

print("\n--- Spearman comparison ---")
display(comparison_623dk10)


# ------------------------------------------------------------
# D. Basic diagnosis
# ------------------------------------------------------------

print("\n--- Diagnostic flags ---")

print(
    "delta exactly constant :",
    bool(np.ptp(delta_error) == 0.0)
)

print(
    "delta nearly zero scale:",
    bool(np.max(np.abs(delta_error)) < 1e-5)
)

print(
    "all helper results finite:",
    bool(
        np.isfinite(
            comparison_623dk10["helper_rho"]
        ).all()
    )
)

print(
    "all pandas results finite:",
    bool(
        np.isfinite(
            comparison_623dk10["pandas_rho"]
        ).all()
    )
)

print(
    "all rank-Pearson finite:",
    bool(
        np.isfinite(
            comparison_623dk10["rank_pearson"]
        ).all()
    )
)


print("\n--- Integrity ---")
print("315 scenarios rerun        : False")
print("Stored metrics modified    : False")
print("Reliability score selected : False")
print("Threshold selected         : False")
print("Abstention rule created    : False")
print("Official test used         : False")
print("Files written              : False")
print("Nothing was modified.")
print("=== END ===")

In [ ]:
# ============================================================
# Step 6.23D-K11
# Exact pre-storage verification for stuck_009
#
# Replays ONE development scenario only.
# No 315-scenario rerun.
# No files written.
# No score/threshold/abstention selection.
# ============================================================

import numpy as np
import pandas as pd
import torch

print(
    "=== Step 6.23D-K11 — "
    "Exact pre-storage stuck_009 verification ==="
)

# ------------------------------------------------------------
# 1. Locate exact frozen scenario
# ------------------------------------------------------------

stuck_manifest_623dk11 = scenario_manifests_620gc["stuck"]

scenario_rows_623dk11 = stuck_manifest_623dk11[
    stuck_manifest_623dk11["scenario_id"].astype(str)
    == "stuck_009"
]

if len(scenario_rows_623dk11) != 1:
    raise RuntimeError(
        "Could not identify exactly one stuck_009 scenario."
    )

scenario_row_623dk11 = scenario_rows_623dk11.iloc[0]

print("\n--- Frozen scenario ---")
print("scenario_id :", scenario_row_623dk11["scenario_id"])
print("sensor      :", scenario_row_623dk11["sensor"])
print("severity    :", scenario_row_623dk11["severity"])


# ------------------------------------------------------------
# 2. Safely replay this ONE corruption on DEVELOPMENT raw data
#
# apply_frozen_corruption_620gc internally uses
# val_raw_base_620gc, so temporarily point that name to the
# certified development raw base and ALWAYS restore it.
# ------------------------------------------------------------

_original_val_base_623dk11 = val_raw_base_620gc

try:
    val_raw_base_620gc = CANONICAL_DEV_RAW_623DF

    corrupted_raw_623dk11 = (
        apply_frozen_corruption_620gc(
            "stuck",
            scenario_row_623dk11
        )
    )

finally:
    val_raw_base_620gc = _original_val_base_623dk11


print("\n--- Raw corruption check ---")

feature_diff_623dk11 = np.max(
    np.abs(
        corrupted_raw_623dk11[
            PRIMARY_INPUT_COLUMNS
        ].to_numpy(dtype=np.float64)
        -
        CANONICAL_DEV_RAW_623DF[
            PRIMARY_INPUT_COLUMNS
        ].to_numpy(dtype=np.float64)
    )
)

print("maximum raw-data change :", feature_diff_623dk11)


# ------------------------------------------------------------
# 3. Rebuild development sequences
# ------------------------------------------------------------

X_stuck009_623dk11 = build_sequences_623cb2(
    corrupted_raw_623dk11
)

X_stuck009_623dk11 = np.asarray(
    X_stuck009_623dk11,
    dtype=np.float32
)

print("\n--- Sequence structure ---")
print("clean shape     :", X_dev_clean.shape)
print("corrupted shape :", X_stuck009_623dk11.shape)

if X_stuck009_623dk11.shape != X_dev_clean.shape:
    raise RuntimeError(
        "stuck_009 sequence shape does not match clean DEV."
    )


# ------------------------------------------------------------
# 4. Deterministic clean/corrupted RUL prediction
# ------------------------------------------------------------

model_device_623dk11 = next(
    frozen_gru.parameters()
).device

clean_pred_623dk11 = np.asarray(
    predict_gru_batched(
        frozen_gru,
        X_dev_clean,
        model_device_623dk11,
        batch_size=512
    ),
    dtype=np.float64
)

stuck_pred_623dk11 = np.asarray(
    predict_gru_batched(
        frozen_gru,
        X_stuck009_623dk11,
        model_device_623dk11,
        batch_size=512
    ),
    dtype=np.float64
)


# ------------------------------------------------------------
# 5. Canonical DEV target
# ------------------------------------------------------------

y_dev_623dk11 = (
    dev_seq_meta["RUL_cap_125"]
    .to_numpy(dtype=np.float64)
)

if not (
    len(y_dev_623dk11)
    == len(clean_pred_623dk11)
    == len(stuck_pred_623dk11)
):
    raise RuntimeError(
        "Prediction/target alignment failure."
    )


# ------------------------------------------------------------
# 6. PRE-STORAGE error comparison
# ------------------------------------------------------------

clean_abs_error_623dk11 = np.abs(
    clean_pred_623dk11 - y_dev_623dk11
)

stuck_abs_error_623dk11 = np.abs(
    stuck_pred_623dk11 - y_dev_623dk11
)

delta_abs_error_623dk11 = (
    stuck_abs_error_623dk11
    - clean_abs_error_623dk11
)


print("\n--- Prediction difference ---")
print(
    "max |corrupted-clean prediction| :",
    float(
        np.max(
            np.abs(
                stuck_pred_623dk11
                - clean_pred_623dk11
            )
        )
    )
)

print(
    "mean |corrupted-clean prediction|:",
    float(
        np.mean(
            np.abs(
                stuck_pred_623dk11
                - clean_pred_623dk11
            )
        )
    )
)


print("\n--- PRE-STORAGE delta-error diagnostic ---")
print("windows            :", delta_abs_error_623dk11.size)
print(
    "finite             :",
    np.isfinite(delta_abs_error_623dk11).all()
)
print(
    "minimum            :",
    float(delta_abs_error_623dk11.min())
)
print(
    "maximum            :",
    float(delta_abs_error_623dk11.max())
)
print(
    "max absolute delta :",
    float(np.max(np.abs(delta_abs_error_623dk11)))
)
print(
    "range              :",
    float(np.ptp(delta_abs_error_623dk11))
)
print(
    "std                :",
    float(np.std(delta_abs_error_623dk11))
)
print(
    "unique values      :",
    int(np.unique(delta_abs_error_623dk11).size)
)
print(
    "exact zero count   :",
    int(np.sum(delta_abs_error_623dk11 == 0.0))
)
print(
    "ALL exactly zero   :",
    bool(np.all(delta_abs_error_623dk11 == 0.0))
)


# ------------------------------------------------------------
# 7. Compare against stored float32 scenario error
# ------------------------------------------------------------

stored_error_623dk11 = np.asarray(
    FULL_DEV_ERROR_MATRIX_623DK[279],
    dtype=np.float64
)

stored_delta_623dk11 = (
    stored_error_623dk11
    - clean_abs_error_623dk11
)

print("\n--- Stored-matrix comparison ---")
print(
    "stored delta max abs :",
    float(np.max(np.abs(stored_delta_623dk11)))
)
print(
    "stored delta range   :",
    float(np.ptp(stored_delta_623dk11))
)
print(
    "stored delta unique  :",
    int(np.unique(stored_delta_623dk11).size)
)


# ------------------------------------------------------------
# 8. Final diagnostic interpretation flag
# ------------------------------------------------------------

PRE_STORAGE_CONSTANT_623DK11 = bool(
    np.ptp(delta_abs_error_623dk11) == 0.0
)

print("\n--- Diagnostic conclusion flag ---")
print(
    "Pre-storage delta-error constant :",
    PRE_STORAGE_CONSTANT_623DK11
)

if PRE_STORAGE_CONSTANT_623DK11:
    print(
        "Interpretation: original delta-error Spearman "
        "is mathematically undefined for stuck_009."
    )
else:
    print(
        "Interpretation: original delta-error contains "
        "real variation; further diagnosis is required."
    )


print("\n--- Integrity ---")
print("Scenarios replayed         : 1")
print("315 scenarios rerun        : False")
print("Stored metrics modified    : False")
print("Reliability score selected : False")
print("Threshold selected         : False")
print("Abstention rule created    : False")
print("Official test used         : False")
print("Files written              : False")
print("Frozen artifacts modified  : False")
print("=== END ===")

In [ ]:
# ============================================================
# Step 6.23D-K11A
# Locate the exact stuck_009 frozen-manifest row
#
# Inspection only.
# Nothing modified.
# ============================================================

import pandas as pd
import numpy as np

print(
    "=== Step 6.23D-K11A — "
    "Locate stuck_009 manifest row ==="
)

stuck_manifest_623dk11a = (
    scenario_manifests_620gc["stuck"].copy()
)

print("\n--- STUCK manifest structure ---")
print("type   :", type(stuck_manifest_623dk11a).__name__)
print("shape  :", stuck_manifest_623dk11a.shape)
print("index  :", stuck_manifest_623dk11a.index.tolist())
print("columns:", stuck_manifest_623dk11a.columns.tolist())


# ------------------------------------------------------------
# 1. Show scenario-ID values exactly as stored
# ------------------------------------------------------------

if "scenario_id" in stuck_manifest_623dk11a.columns:

    print("\n--- Stored scenario_id values ---")

    preview_623dk11a = pd.DataFrame({
        "manifest_position": np.arange(
            len(stuck_manifest_623dk11a)
        ),
        "manifest_index": stuck_manifest_623dk11a.index,
        "scenario_id": stuck_manifest_623dk11a[
            "scenario_id"
        ].astype(str).to_numpy()
    })

    display(preview_623dk11a.head(15))

    exact_mask_623dk11a = (
        stuck_manifest_623dk11a["scenario_id"]
        .astype(str)
        .str.strip()
        .eq("stuck_009")
    )

    print(
        "\nExact normalized 'stuck_009' matches:",
        int(exact_mask_623dk11a.sum())
    )

    if exact_mask_623dk11a.any():
        display(
            stuck_manifest_623dk11a.loc[
                exact_mask_623dk11a
            ]
        )


# ------------------------------------------------------------
# 2. Inspect the row corresponding to STUCK offset 9
#
# Combined scenario counts before STUCK:
# bias=90, noise=45, drift=90, dropout=45
# total = 270
# global index 279 -> STUCK positional offset 9
#
# Inspection only — not yet accepting it as canonical.
# ------------------------------------------------------------

stuck_offset_623dk11a = 279 - (
    90 + 45 + 90 + 45
)

print(
    "\nGlobal index 279 corresponds to "
    "STUCK positional offset:",
    stuck_offset_623dk11a
)

if (
    0 <= stuck_offset_623dk11a
    < len(stuck_manifest_623dk11a)
):

    print("\n--- Manifest row at positional offset 9 ---")

    display(
        stuck_manifest_623dk11a.iloc[
            [stuck_offset_623dk11a]
        ]
    )


# ------------------------------------------------------------
# 3. Search using known stored-result attributes
# ------------------------------------------------------------

known_filters_623dk11a = pd.Series(
    True,
    index=stuck_manifest_623dk11a.index
)

if "sensor" in stuck_manifest_623dk11a.columns:
    known_filters_623dk11a &= (
        stuck_manifest_623dk11a["sensor"]
        .astype(str)
        .str.strip()
        .eq("sensor_6")
    )

if "severity" in stuck_manifest_623dk11a.columns:
    known_filters_623dk11a &= (
        stuck_manifest_623dk11a["severity"]
        .astype(str)
        .str.strip()
        .str.lower()
        .eq("low")
    )

print(
    "\nRows matching known attributes "
    "(sensor_6 + low, where available):",
    int(known_filters_623dk11a.sum())
)

if known_filters_623dk11a.any():
    display(
        stuck_manifest_623dk11a.loc[
            known_filters_623dk11a
        ]
    )


print("\n--- Integrity ---")
print("Corruption executed         : False")
print("Prediction executed         : False")
print("315 scenarios rerun         : False")
print("Reliability score selected  : False")
print("Threshold selected          : False")
print("Abstention rule created     : False")
print("Official test used          : False")
print("Files written               : False")
print("Nothing was modified.")
print("=== END ===")

In [ ]:
# ============================================================
# Step 6.23D-K11B
# Exact PRE-STORAGE verification for the verified
# global-index-279 STUCK scenario
#
# Replays ONE development scenario only.
# No 315-scenario rerun.
# No stored metric modification.
# No files written.
# ============================================================

import numpy as np
import pandas as pd

print(
    "=== Step 6.23D-K11B — "
    "Verified pre-storage STUCK scenario check ==="
)

# ------------------------------------------------------------
# 1. Recover exact manifest position from global scenario index
# ------------------------------------------------------------

GLOBAL_INDEX_623DK11B = 279

family_order_623dk11b = [
    "bias",
    "noise",
    "drift",
    "dropout",
    "stuck",
]

stuck_start_623dk11b = sum(
    len(scenario_manifests_620gc[f])
    for f in family_order_623dk11b[:-1]
)

stuck_position_623dk11b = (
    GLOBAL_INDEX_623DK11B
    - stuck_start_623dk11b
)

stuck_manifest_623dk11b = (
    scenario_manifests_620gc["stuck"]
)

if not (
    0 <= stuck_position_623dk11b
    < len(stuck_manifest_623dk11b)
):
    raise RuntimeError(
        "Recovered STUCK position is outside the manifest."
    )

scenario_row_623dk11b = (
    stuck_manifest_623dk11b.iloc[
        stuck_position_623dk11b
    ]
)

print("\n--- Verified scenario mapping ---")
print("Global scenario index :", GLOBAL_INDEX_623DK11B)
print("STUCK family starts   :", stuck_start_623dk11b)
print("Manifest position     :", stuck_position_623dk11b)
print(
    "Manifest scenario_id  :",
    scenario_row_623dk11b["scenario_id"]
)
print(
    "Sensor                :",
    scenario_row_623dk11b["sensor"]
)
print(
    "Severity              :",
    scenario_row_623dk11b["severity"]
)

# Known mapping already observed in K11A.
if str(scenario_row_623dk11b["sensor"]) != "sensor_6":
    raise RuntimeError(
        "Verified global index no longer maps to sensor_6."
    )

if str(
    scenario_row_623dk11b["severity"]
).lower() != "low":
    raise RuntimeError(
        "Verified global index no longer maps to LOW severity."
    )


# ------------------------------------------------------------
# 2. Replay exactly ONE frozen corruption on canonical DEV raw
#
# The legacy replay helper uses val_raw_base_620gc internally.
# Temporarily redirect it to canonical DEV and ALWAYS restore.
# ------------------------------------------------------------

_original_val_raw_base_623dk11b = (
    val_raw_base_620gc
)

try:

    val_raw_base_620gc = (
        CANONICAL_DEV_RAW_623DF
    )

    corrupted_raw_623dk11b = (
        apply_frozen_corruption_620gc(
            "stuck",
            scenario_row_623dk11b
        )
    )

finally:

    val_raw_base_620gc = (
        _original_val_raw_base_623dk11b
    )


# ------------------------------------------------------------
# 3. Verify corruption actually changed the raw data
# ------------------------------------------------------------

clean_raw_matrix_623dk11b = (
    CANONICAL_DEV_RAW_623DF[
        PRIMARY_INPUT_COLUMNS
    ]
    .to_numpy(dtype=np.float64)
)

corrupt_raw_matrix_623dk11b = (
    corrupted_raw_623dk11b[
        PRIMARY_INPUT_COLUMNS
    ]
    .to_numpy(dtype=np.float64)
)

raw_difference_623dk11b = (
    corrupt_raw_matrix_623dk11b
    - clean_raw_matrix_623dk11b
)

print("\n--- Raw corruption check ---")
print(
    "Maximum absolute raw change :",
    float(
        np.max(
            np.abs(raw_difference_623dk11b)
        )
    )
)
print(
    "Changed raw cells            :",
    int(
        np.sum(
            raw_difference_623dk11b != 0.0
        )
    )
)


# ------------------------------------------------------------
# 4. Rebuild corrupted DEV sequences
# ------------------------------------------------------------

X_stuck_623dk11b = (
    build_sequences_623cb2(
        corrupted_raw_623dk11b
    )
)

X_stuck_623dk11b = np.asarray(
    X_stuck_623dk11b,
    dtype=np.float32
)

print("\n--- Sequence check ---")
print("Clean DEV shape     :", X_dev_clean.shape)
print("Corrupted DEV shape :", X_stuck_623dk11b.shape)

if X_stuck_623dk11b.shape != X_dev_clean.shape:
    raise RuntimeError(
        "Corrupted DEV sequence shape mismatch."
    )


# ------------------------------------------------------------
# 5. Deterministic clean and corrupted RUL predictions
# ------------------------------------------------------------

device_623dk11b = next(
    frozen_gru.parameters()
).device

clean_pred_623dk11b = np.asarray(
    predict_gru_batched(
        frozen_gru,
        X_dev_clean,
        device_623dk11b,
        batch_size=512
    ),
    dtype=np.float64
)

corrupt_pred_623dk11b = np.asarray(
    predict_gru_batched(
        frozen_gru,
        X_stuck_623dk11b,
        device_623dk11b,
        batch_size=512
    ),
    dtype=np.float64
)


# ------------------------------------------------------------
# 6. Canonical development target
# ------------------------------------------------------------

y_dev_623dk11b = (
    dev_seq_meta[
        "RUL_cap_125"
    ]
    .to_numpy(dtype=np.float64)
)

if not (
    len(y_dev_623dk11b)
    == len(clean_pred_623dk11b)
    == len(corrupt_pred_623dk11b)
    == 14241
):
    raise RuntimeError(
        "DEV target/prediction alignment failure."
    )


# ------------------------------------------------------------
# 7. PRE-STORAGE RUL-error calculation
# ------------------------------------------------------------

clean_error_623dk11b = np.abs(
    clean_pred_623dk11b
    - y_dev_623dk11b
)

corrupt_error_623dk11b = np.abs(
    corrupt_pred_623dk11b
    - y_dev_623dk11b
)

delta_error_623dk11b = (
    corrupt_error_623dk11b
    - clean_error_623dk11b
)


# ------------------------------------------------------------
# 8. Confirm clean-error pathway reproduces stored baseline
# ------------------------------------------------------------

canonical_clean_error_623dk11b = np.asarray(
    CLEAN_DEV_ABS_ERROR_623DK,
    dtype=np.float64
)

clean_reproduction_error_623dk11b = (
    np.max(
        np.abs(
            clean_error_623dk11b
            - canonical_clean_error_623dk11b
        )
    )
)

print("\n--- Clean-pathway reproduction ---")
print(
    "Clean DEV MAE              :",
    float(clean_error_623dk11b.mean())
)
print(
    "Stored clean DEV MAE       :",
    float(canonical_clean_error_623dk11b.mean())
)
print(
    "Maximum clean-error diff   :",
    float(clean_reproduction_error_623dk11b)
)


# ------------------------------------------------------------
# 9. Prediction-level effect
# ------------------------------------------------------------

prediction_delta_623dk11b = (
    corrupt_pred_623dk11b
    - clean_pred_623dk11b
)

print("\n--- Prediction effect ---")
print(
    "Maximum |prediction delta| :",
    float(
        np.max(
            np.abs(
                prediction_delta_623dk11b
            )
        )
    )
)
print(
    "Mean |prediction delta|    :",
    float(
        np.mean(
            np.abs(
                prediction_delta_623dk11b
            )
        )
    )
)
print(
    "Exactly unchanged predictions:",
    int(
        np.sum(
            prediction_delta_623dk11b == 0.0
        )
    )
)


# ------------------------------------------------------------
# 10. PRE-STORAGE delta-error diagnosis
# ------------------------------------------------------------

print("\n--- PRE-STORAGE delta-error diagnostic ---")

print("Windows              :", delta_error_623dk11b.size)
print(
    "All finite           :",
    bool(np.isfinite(delta_error_623dk11b).all())
)
print(
    "Minimum              :",
    float(delta_error_623dk11b.min())
)
print(
    "Maximum              :",
    float(delta_error_623dk11b.max())
)
print(
    "Maximum absolute     :",
    float(
        np.max(
            np.abs(delta_error_623dk11b)
        )
    )
)
print(
    "Range                :",
    float(np.ptp(delta_error_623dk11b))
)
print(
    "Standard deviation   :",
    float(np.std(delta_error_623dk11b))
)
print(
    "Unique values        :",
    int(np.unique(delta_error_623dk11b).size)
)
print(
    "Exact zero count     :",
    int(
        np.sum(
            delta_error_623dk11b == 0.0
        )
    )
)

PRE_STORAGE_CONSTANT_623DK11B = bool(
    np.ptp(delta_error_623dk11b) == 0.0
)

PRE_STORAGE_ALL_ZERO_623DK11B = bool(
    np.all(delta_error_623dk11b == 0.0)
)

print(
    "Exactly constant     :",
    PRE_STORAGE_CONSTANT_623DK11B
)
print(
    "All exactly zero     :",
    PRE_STORAGE_ALL_ZERO_623DK11B
)


# ------------------------------------------------------------
# 11. Compare against stored float32 error matrix
# ------------------------------------------------------------

stored_error_623dk11b = np.asarray(
    FULL_DEV_ERROR_MATRIX_623DK[
        GLOBAL_INDEX_623DK11B
    ],
    dtype=np.float64
)

stored_delta_vs_clean_623dk11b = (
    stored_error_623dk11b
    - canonical_clean_error_623dk11b
)

print("\n--- Stored float32 matrix comparison ---")

print(
    "Stored delta range       :",
    float(
        np.ptp(
            stored_delta_vs_clean_623dk11b
        )
    )
)

print(
    "Stored delta max abs     :",
    float(
        np.max(
            np.abs(
                stored_delta_vs_clean_623dk11b
            )
        )
    )
)

print(
    "Stored delta unique      :",
    int(
        np.unique(
            stored_delta_vs_clean_623dk11b
        ).size
    )
)


# ------------------------------------------------------------
# 12. Interpretation
# ------------------------------------------------------------

print("\n--- Interpretation flag ---")

if PRE_STORAGE_CONSTANT_623DK11B:

    print(
        "PRE-STORAGE delta-error is constant."
    )
    print(
        "Therefore its Spearman correlation "
        "is mathematically undefined."
    )

else:

    print(
        "PRE-STORAGE delta-error contains variation."
    )
    print(
        "The original NaN therefore needs a "
        "different explanation."
    )


# ------------------------------------------------------------
# 13. Integrity
# ------------------------------------------------------------

print("\n--- Integrity ---")
print("Scenarios replayed          : 1")
print("315 scenarios rerun         : False")
print("Stored metrics modified     : False")
print("Reliability score selected  : False")
print("Threshold selected          : False")
print("Abstention rule created     : False")
print("Official test used          : False")
print("Files written               : False")
print("Frozen artifacts modified   : False")
print("=== END ===")

In [ ]:
# ============================================================
# Step 6.23D-K11C
# Target-sensor / model-input root-cause check
#
# Inspection only.
# Uses objects already produced by K11B.
# Nothing modified.
# ============================================================

import numpy as np
import pandas as pd

print(
    "=== Step 6.23D-K11C — "
    "Target-sensor/model-input root-cause check ==="
)

TARGET_SENSOR_623DK11C = "sensor_6"

# ------------------------------------------------------------
# 1. Is the target sensor actually used by the RUL model?
# ------------------------------------------------------------

print("\n--- Feature-membership check ---")

print(
    "sensor_6 in PRIMARY_INPUT_COLUMNS :",
    TARGET_SENSOR_623DK11C
    in list(PRIMARY_INPUT_COLUMNS)
)

if "RETAINED_SENSOR_COLUMNS" in globals():
    print(
        "sensor_6 in RETAINED_SENSOR_COLUMNS:",
        TARGET_SENSOR_623DK11C
        in list(RETAINED_SENSOR_COLUMNS)
    )

print(
    "\nPrimary input columns:"
)
print(list(PRIMARY_INPUT_COLUMNS))


# ------------------------------------------------------------
# 2. Compare sensor_6 itself in clean vs corrupted RAW data
# ------------------------------------------------------------

if (
    TARGET_SENSOR_623DK11C
    not in CANONICAL_DEV_RAW_623DF.columns
):
    raise RuntimeError(
        "sensor_6 is absent from canonical DEV raw data."
    )

if (
    TARGET_SENSOR_623DK11C
    not in corrupted_raw_623dk11b.columns
):
    raise RuntimeError(
        "sensor_6 is absent from corrupted DEV raw data."
    )

clean_sensor_623dk11c = (
    CANONICAL_DEV_RAW_623DF[
        TARGET_SENSOR_623DK11C
    ]
    .to_numpy(dtype=np.float64)
)

corrupted_sensor_623dk11c = (
    corrupted_raw_623dk11b[
        TARGET_SENSOR_623DK11C
    ]
    .to_numpy(dtype=np.float64)
)

sensor_delta_623dk11c = (
    corrupted_sensor_623dk11c
    - clean_sensor_623dk11c
)

print("\n--- Exact sensor_6 raw-data check ---")

print(
    "Rows:",
    len(clean_sensor_623dk11c)
)

print(
    "Clean unique values:",
    int(np.unique(clean_sensor_623dk11c).size)
)

print(
    "Clean minimum:",
    float(np.min(clean_sensor_623dk11c))
)

print(
    "Clean maximum:",
    float(np.max(clean_sensor_623dk11c))
)

print(
    "Clean std:",
    float(np.std(clean_sensor_623dk11c))
)

print(
    "Changed sensor_6 cells:",
    int(np.sum(sensor_delta_623dk11c != 0.0))
)

print(
    "Maximum |sensor_6 change|:",
    float(np.max(np.abs(sensor_delta_623dk11c)))
)


# ------------------------------------------------------------
# 3. Compare ALL shared numeric raw columns
# ------------------------------------------------------------

shared_numeric_623dk11c = [
    c
    for c in CANONICAL_DEV_RAW_623DF.columns
    if (
        c in corrupted_raw_623dk11b.columns
        and pd.api.types.is_numeric_dtype(
            CANONICAL_DEV_RAW_623DF[c]
        )
        and pd.api.types.is_numeric_dtype(
            corrupted_raw_623dk11b[c]
        )
    )
]

clean_all_623dk11c = (
    CANONICAL_DEV_RAW_623DF[
        shared_numeric_623dk11c
    ]
    .to_numpy(dtype=np.float64)
)

corrupt_all_623dk11c = (
    corrupted_raw_623dk11b[
        shared_numeric_623dk11c
    ]
    .to_numpy(dtype=np.float64)
)

all_delta_623dk11c = (
    corrupt_all_623dk11c
    - clean_all_623dk11c
)

print("\n--- Entire raw DataFrame check ---")

print(
    "Shared numeric columns:",
    len(shared_numeric_623dk11c)
)

print(
    "Changed numeric cells:",
    int(np.sum(all_delta_623dk11c != 0.0))
)

print(
    "Maximum raw numeric change:",
    float(np.max(np.abs(all_delta_623dk11c)))
)


# ------------------------------------------------------------
# 4. Sequence-level confirmation
# ------------------------------------------------------------

sequence_delta_623dk11c = (
    np.asarray(
        X_stuck_623dk11b,
        dtype=np.float64
    )
    -
    np.asarray(
        X_dev_clean,
        dtype=np.float64
    )
)

print("\n--- Model-sequence effect ---")

print(
    "Changed sequence values:",
    int(np.sum(sequence_delta_623dk11c != 0.0))
)

print(
    "Maximum sequence change:",
    float(np.max(np.abs(sequence_delta_623dk11c)))
)


# ------------------------------------------------------------
# 5. Classification
# ------------------------------------------------------------

sensor_is_model_input_623dk11c = bool(
    TARGET_SENSOR_623DK11C
    in list(PRIMARY_INPUT_COLUMNS)
)

sensor_changed_623dk11c = bool(
    np.any(sensor_delta_623dk11c != 0.0)
)

any_raw_changed_623dk11c = bool(
    np.any(all_delta_623dk11c != 0.0)
)

sequence_changed_623dk11c = bool(
    np.any(sequence_delta_623dk11c != 0.0)
)


print("\n--- Root-cause classification ---")

if (
    sensor_is_model_input_623dk11c
    and not sensor_changed_623dk11c
):
    print(
        "CASE A: sensor_6 is a model input, "
        "but this frozen STUCK scenario is a raw-data no-op."
    )

elif (
    not sensor_is_model_input_623dk11c
    and sensor_changed_623dk11c
    and not sequence_changed_623dk11c
):
    print(
        "CASE B: sensor_6 is corrupted in raw data "
        "but is not used by the RUL model."
    )

elif (
    not any_raw_changed_623dk11c
):
    print(
        "CASE C: the frozen scenario changes no numeric "
        "raw value anywhere."
    )

else:
    print(
        "CASE D: another pathway issue exists; "
        "further inspection will be needed."
    )


print("\n--- Integrity ---")
print("New corruption executed       : False")
print("New prediction executed       : False")
print("315 scenarios rerun           : False")
print("Stored metrics modified       : False")
print("Reliability score selected    : False")
print("Threshold selected            : False")
print("Abstention rule created       : False")
print("Official test used            : False")
print("Files written                 : False")
print("Nothing was modified.")
print("=== END ===")

In [ ]:
# ============================================================
# Step 6.23D-K12
# Full frozen-scenario RAW-EFFECT validity audit
#
# Purpose:
#   Identify any frozen scenario that produces NO numerical
#   change in the canonical DEVELOPMENT raw input.
#
# No model inference.
# No MC dropout.
# No score selection.
# No threshold selection.
# No official test.
# No files written.
# ============================================================

import numpy as np
import pandas as pd
import time

print(
    "=== Step 6.23D-K12 — "
    "315-scenario raw-effect validity audit ==="
)

# ------------------------------------------------------------
# 1. Required objects
# ------------------------------------------------------------

required_623dk12 = [
    "scenario_manifests_620gc",
    "apply_frozen_corruption_620gc",
    "CANONICAL_DEV_RAW_623DF",
    "PRIMARY_INPUT_COLUMNS",
]

for name_623dk12 in required_623dk12:
    print(
        f"{name_623dk12:32s}:",
        name_623dk12 in globals()
    )

missing_623dk12 = [
    x for x in required_623dk12
    if x not in globals()
]

if missing_623dk12:
    raise RuntimeError(
        f"Missing objects: {missing_623dk12}"
    )


# ------------------------------------------------------------
# 2. Fixed scenario-family order
# ------------------------------------------------------------

family_order_623dk12 = [
    "bias",
    "noise",
    "drift",
    "dropout",
    "stuck",
]

expected_counts_623dk12 = {
    "bias": 90,
    "noise": 45,
    "drift": 90,
    "dropout": 45,
    "stuck": 45,
}

print("\n--- Frozen manifest counts ---")

for family_623dk12 in family_order_623dk12:
    actual_n_623dk12 = len(
        scenario_manifests_620gc[
            family_623dk12
        ]
    )

    print(
        f"{family_623dk12:8s}:",
        actual_n_623dk12
    )

    if actual_n_623dk12 != expected_counts_623dk12[
        family_623dk12
    ]:
        raise RuntimeError(
            f"Unexpected {family_623dk12} "
            f"scenario count."
        )


# ------------------------------------------------------------
# 3. Certified clean-development reference
# ------------------------------------------------------------

canonical_623dk12 = (
    CANONICAL_DEV_RAW_623DF.copy(deep=True)
)

primary_clean_623dk12 = (
    canonical_623dk12[
        PRIMARY_INPUT_COLUMNS
    ]
    .to_numpy(dtype=np.float64)
)

shared_numeric_columns_623dk12 = [
    c
    for c in canonical_623dk12.columns
    if pd.api.types.is_numeric_dtype(
        canonical_623dk12[c]
    )
]

all_clean_numeric_623dk12 = (
    canonical_623dk12[
        shared_numeric_columns_623dk12
    ]
    .to_numpy(dtype=np.float64)
)


# ------------------------------------------------------------
# 4. Audit every frozen scenario
# ------------------------------------------------------------

audit_rows_623dk12 = []

global_index_623dk12 = 0
start_time_623dk12 = time.time()

_original_val_base_623dk12 = val_raw_base_620gc

try:

    for family_623dk12 in family_order_623dk12:

        manifest_623dk12 = (
            scenario_manifests_620gc[
                family_623dk12
            ]
        )

        print(
            f"\nAuditing {family_623dk12.upper()} "
            f"({len(manifest_623dk12)} scenarios)..."
        )

        for local_index_623dk12 in range(
            len(manifest_623dk12)
        ):

            row_623dk12 = (
                manifest_623dk12.iloc[
                    local_index_623dk12
                ]
            )

            # Give the legacy function an isolated clean copy.
            # This prevents accidental mutation of our
            # certified canonical source.
            val_raw_base_620gc = (
                canonical_623dk12.copy(deep=True)
            )

            corrupted_623dk12 = (
                apply_frozen_corruption_620gc(
                    family_623dk12,
                    row_623dk12
                )
            )

            # ----------------------------------------------
            # Primary-model-input difference
            # ----------------------------------------------

            corrupted_primary_623dk12 = (
                corrupted_623dk12[
                    PRIMARY_INPUT_COLUMNS
                ]
                .to_numpy(dtype=np.float64)
            )

            primary_delta_623dk12 = (
                corrupted_primary_623dk12
                - primary_clean_623dk12
            )

            primary_changed_623dk12 = int(
                np.count_nonzero(
                    primary_delta_623dk12
                )
            )

            primary_max_change_623dk12 = float(
                np.max(
                    np.abs(
                        primary_delta_623dk12
                    )
                )
            )

            # ----------------------------------------------
            # Entire numeric raw-data difference
            # ----------------------------------------------

            corrupted_all_numeric_623dk12 = (
                corrupted_623dk12[
                    shared_numeric_columns_623dk12
                ]
                .to_numpy(dtype=np.float64)
            )

            all_delta_623dk12 = (
                corrupted_all_numeric_623dk12
                - all_clean_numeric_623dk12
            )

            all_changed_623dk12 = int(
                np.count_nonzero(
                    all_delta_623dk12
                )
            )

            all_max_change_623dk12 = float(
                np.max(
                    np.abs(
                        all_delta_623dk12
                    )
                )
            )

            # ----------------------------------------------
            # Target-sensor difference
            # ----------------------------------------------

            sensor_623dk12 = (
                str(row_623dk12["sensor"])
                if "sensor" in row_623dk12.index
                else None
            )

            target_changed_623dk12 = np.nan
            target_max_change_623dk12 = np.nan

            if (
                sensor_623dk12 is not None
                and sensor_623dk12
                in canonical_623dk12.columns
            ):

                clean_target_623dk12 = (
                    canonical_623dk12[
                        sensor_623dk12
                    ]
                    .to_numpy(dtype=np.float64)
                )

                corrupt_target_623dk12 = (
                    corrupted_623dk12[
                        sensor_623dk12
                    ]
                    .to_numpy(dtype=np.float64)
                )

                target_delta_623dk12 = (
                    corrupt_target_623dk12
                    - clean_target_623dk12
                )

                target_changed_623dk12 = int(
                    np.count_nonzero(
                        target_delta_623dk12
                    )
                )

                target_max_change_623dk12 = float(
                    np.max(
                        np.abs(
                            target_delta_623dk12
                        )
                    )
                )

            # ----------------------------------------------
            # Validity
            # ----------------------------------------------

            effective_primary_623dk12 = bool(
                primary_changed_623dk12 > 0
            )

            effective_any_raw_623dk12 = bool(
                all_changed_623dk12 > 0
            )

            manifest_id_623dk12 = (
                str(row_623dk12["scenario_id"])
                if "scenario_id" in row_623dk12.index
                else None
            )

            severity_623dk12 = (
                str(row_623dk12["severity"])
                if "severity" in row_623dk12.index
                else None
            )

            # This reproduces the zero-based label convention
            # used by the completed FULL-DEV experiment.
            evaluation_label_623dk12 = (
                f"{family_623dk12}_"
                f"{local_index_623dk12:03d}"
            )

            audit_rows_623dk12.append(
                {
                    "global_index":
                        global_index_623dk12,

                    "family":
                        family_623dk12,

                    "family_position":
                        local_index_623dk12,

                    "evaluation_label":
                        evaluation_label_623dk12,

                    "manifest_scenario_id":
                        manifest_id_623dk12,

                    "sensor":
                        sensor_623dk12,

                    "severity":
                        severity_623dk12,

                    "target_changed_cells":
                        target_changed_623dk12,

                    "target_max_abs_change":
                        target_max_change_623dk12,

                    "primary_changed_cells":
                        primary_changed_623dk12,

                    "primary_max_abs_change":
                        primary_max_change_623dk12,

                    "all_raw_changed_cells":
                        all_changed_623dk12,

                    "all_raw_max_abs_change":
                        all_max_change_623dk12,

                    "effective_primary":
                        effective_primary_623dk12,

                    "effective_any_raw":
                        effective_any_raw_623dk12,
                }
            )

            global_index_623dk12 += 1

finally:

    val_raw_base_620gc = (
        _original_val_base_623dk12
    )


# ------------------------------------------------------------
# 5. Build audit table
# ------------------------------------------------------------

SCENARIO_RAW_EFFECT_AUDIT_623DK12 = (
    pd.DataFrame(
        audit_rows_623dk12
    )
)

elapsed_623dk12 = (
    time.time()
    - start_time_623dk12
)

if len(SCENARIO_RAW_EFFECT_AUDIT_623DK12) != 315:
    raise RuntimeError(
        "Audit did not produce exactly 315 rows."
    )


# ------------------------------------------------------------
# 6. Identify ineffective scenarios
# ------------------------------------------------------------

NOOP_SCENARIOS_623DK12 = (
    SCENARIO_RAW_EFFECT_AUDIT_623DK12[
        ~SCENARIO_RAW_EFFECT_AUDIT_623DK12[
            "effective_primary"
        ]
    ]
    .copy()
)

print("\n--- Audit summary ---")

print(
    "Total scenarios             :",
    len(SCENARIO_RAW_EFFECT_AUDIT_623DK12)
)

print(
    "Effective model-input cases :",
    int(
        SCENARIO_RAW_EFFECT_AUDIT_623DK12[
            "effective_primary"
        ].sum()
    )
)

print(
    "No-op model-input cases     :",
    len(NOOP_SCENARIOS_623DK12)
)

print(
    "Effective any-raw cases     :",
    int(
        SCENARIO_RAW_EFFECT_AUDIT_623DK12[
            "effective_any_raw"
        ].sum()
    )
)

print(
    "Runtime seconds             :",
    elapsed_623dk12
)


# ------------------------------------------------------------
# 7. Family-level validity table
# ------------------------------------------------------------

family_summary_623dk12 = (
    SCENARIO_RAW_EFFECT_AUDIT_623DK12
    .groupby("family", sort=False)
    .agg(
        scenarios=(
            "global_index",
            "size"
        ),
        effective_primary=(
            "effective_primary",
            "sum"
        ),
        ineffective_primary=(
            "effective_primary",
            lambda x: int(
                (~x).sum()
            )
        ),
    )
    .reset_index()
)

print("\n--- Family-level validity ---")
display(family_summary_623dk12)


# ------------------------------------------------------------
# 8. Display all no-op scenarios
# ------------------------------------------------------------

print("\n--- Model-input NO-OP scenarios ---")

if len(NOOP_SCENARIOS_623DK12) == 0:

    print(
        "No no-op scenarios found."
    )

else:

    display(
        NOOP_SCENARIOS_623DK12[
            [
                "global_index",
                "family",
                "family_position",
                "evaluation_label",
                "manifest_scenario_id",
                "sensor",
                "severity",
                "target_changed_cells",
                "primary_changed_cells",
                "all_raw_changed_cells",
                "primary_max_abs_change",
            ]
        ]
    )


# ------------------------------------------------------------
# 9. Verify the known problematic scenario
# ------------------------------------------------------------

known_279_623dk12 = (
    SCENARIO_RAW_EFFECT_AUDIT_623DK12[
        SCENARIO_RAW_EFFECT_AUDIT_623DK12[
            "global_index"
        ] == 279
    ]
)

print("\n--- Known index 279 verification ---")
display(known_279_623dk12)


# ------------------------------------------------------------
# 10. Integrity
# ------------------------------------------------------------

canonical_after_623dk12 = (
    CANONICAL_DEV_RAW_623DF[
        PRIMARY_INPUT_COLUMNS
    ]
    .to_numpy(dtype=np.float64)
)

canonical_unchanged_623dk12 = bool(
    np.array_equal(
        canonical_after_623dk12,
        primary_clean_623dk12
    )
)

print("\n==========================================")
print("STEP 6.23D-K12 COMPACT SUMMARY")
print("==========================================")

print(
    "Scenarios audited             :",
    len(SCENARIO_RAW_EFFECT_AUDIT_623DK12)
)

print(
    "No-op model-input scenarios   :",
    len(NOOP_SCENARIOS_623DK12)
)

print(
    "Canonical DEV unchanged       :",
    canonical_unchanged_623dk12
)

print("Model inference executed       : False")
print("MC dropout executed            : False")
print("315 RUL experiments rerun      : False")
print("Stored results modified        : False")
print("Reliability score selected     : False")
print("Threshold selected             : False")
print("Abstention rule created        : False")
print("Official test used             : False")
print("Files written                  : False")
print("=== END ===")

In [ ]:
# ============================================================
# Step 6.23D-K13
# Effective-scenario filtering and family-balanced
# reliability-candidate re-summary
#
# Uses stored FULL-DEV results.
# NO scenario rerun.
# NO model inference.
# NO MC dropout.
# NO reliability score selection.
# NO threshold selection.
# NO official test.
# NO files written.
# ============================================================

import numpy as np
import pandas as pd

print(
    "=== Step 6.23D-K13 — "
    "Effective-scenario reliability re-summary ==="
)

# ------------------------------------------------------------
# 1. Required objects
# ------------------------------------------------------------

required_623dk13 = [
    "FULL_DEV_SCENARIO_METRICS_623DK",
    "SCENARIO_RAW_EFFECT_AUDIT_623DK12",
    "CANDIDATE_NAMES_623DK",
]

for name_623dk13 in required_623dk13:
    print(
        f"{name_623dk13:36s}:",
        name_623dk13 in globals()
    )

missing_623dk13 = [
    name
    for name in required_623dk13
    if name not in globals()
]

if missing_623dk13:
    raise RuntimeError(
        f"Missing required objects: {missing_623dk13}"
    )


# ------------------------------------------------------------
# 2. Certified effective scenario indices
# ------------------------------------------------------------

audit_623dk13 = (
    SCENARIO_RAW_EFFECT_AUDIT_623DK12
    .copy()
)

effective_indices_623dk13 = (
    audit_623dk13.loc[
        audit_623dk13[
            "effective_primary"
        ].astype(bool),
        "global_index"
    ]
    .astype(int)
    .to_numpy()
)

ineffective_indices_623dk13 = (
    audit_623dk13.loc[
        ~audit_623dk13[
            "effective_primary"
        ].astype(bool),
        "global_index"
    ]
    .astype(int)
    .to_numpy()
)

print("\n--- Scenario validity ---")
print(
    "Total frozen scenarios     :",
    len(audit_623dk13)
)
print(
    "Effective scenarios        :",
    len(effective_indices_623dk13)
)
print(
    "Ineffective scenarios      :",
    len(ineffective_indices_623dk13)
)
print(
    "Ineffective global indices :",
    ineffective_indices_623dk13.tolist()
)

if len(effective_indices_623dk13) != 314:
    raise RuntimeError(
        "Expected exactly 314 effective scenarios."
    )

if not np.array_equal(
    ineffective_indices_623dk13,
    np.array([279])
):
    raise RuntimeError(
        "Unexpected ineffective-scenario set."
    )


# ------------------------------------------------------------
# 3. Original stored scenario/candidate metrics
# ------------------------------------------------------------

full_metrics_623dk13 = (
    FULL_DEV_SCENARIO_METRICS_623DK
    .copy()
)

print("\n--- Stored FULL-DEV metric structure ---")
print("Rows       :", len(full_metrics_623dk13))
print("Columns    :", len(full_metrics_623dk13.columns))
print(
    "Candidates :",
    full_metrics_623dk13[
        "candidate"
    ].nunique()
)
print(
    "Scenarios  :",
    full_metrics_623dk13[
        "global_index"
    ].nunique()
)

if len(full_metrics_623dk13) != 315 * 8:
    raise RuntimeError(
        "Stored FULL-DEV table does not have "
        "315 × 8 rows."
    )


# ------------------------------------------------------------
# 4. Filter ONLY the certified no-op scenario
# ------------------------------------------------------------

EFFECTIVE_DEV_SCENARIO_METRICS_623DK13 = (
    full_metrics_623dk13[
        full_metrics_623dk13[
            "global_index"
        ].isin(
            effective_indices_623dk13
        )
    ]
    .copy()
    .reset_index(drop=True)
)

print("\n--- Effective analysis set ---")
print(
    "Rows       :",
    len(EFFECTIVE_DEV_SCENARIO_METRICS_623DK13)
)
print(
    "Scenarios  :",
    EFFECTIVE_DEV_SCENARIO_METRICS_623DK13[
        "global_index"
    ].nunique()
)
print(
    "Candidates :",
    EFFECTIVE_DEV_SCENARIO_METRICS_623DK13[
        "candidate"
    ].nunique()
)

if len(
    EFFECTIVE_DEV_SCENARIO_METRICS_623DK13
) != 314 * 8:
    raise RuntimeError(
        "Effective scenario table must contain "
        "314 × 8 = 2512 rows."
    )


# ------------------------------------------------------------
# 5. Verify candidate coverage
# ------------------------------------------------------------

expected_candidates_623dk13 = list(
    CANDIDATE_NAMES_623DK
)

candidate_counts_623dk13 = (
    EFFECTIVE_DEV_SCENARIO_METRICS_623DK13[
        "candidate"
    ]
    .value_counts()
    .sort_index()
)

print("\n--- Candidate scenario counts ---")
display(
    candidate_counts_623dk13
    .rename("effective_scenarios")
    .to_frame()
)

if not (
    candidate_counts_623dk13
    == 314
).all():
    raise RuntimeError(
        "One or more candidates do not have "
        "all 314 effective scenarios."
    )


# ------------------------------------------------------------
# 6. Verify family counts
# ------------------------------------------------------------

scenario_family_counts_623dk13 = (
    EFFECTIVE_DEV_SCENARIO_METRICS_623DK13[
        [
            "global_index",
            "family"
        ]
    ]
    .drop_duplicates()
    ["family"]
    .value_counts()
    .reindex(
        [
            "bias",
            "noise",
            "drift",
            "dropout",
            "stuck",
        ]
    )
)

print("\n--- Effective family counts ---")
display(
    scenario_family_counts_623dk13
    .rename("scenarios")
    .to_frame()
)

expected_family_counts_623dk13 = {
    "bias": 90,
    "noise": 45,
    "drift": 90,
    "dropout": 45,
    "stuck": 44,
}

for (
    family_623dk13,
    expected_n_623dk13
) in expected_family_counts_623dk13.items():

    if int(
        scenario_family_counts_623dk13.loc[
            family_623dk13
        ]
    ) != expected_n_623dk13:

        raise RuntimeError(
            f"Unexpected effective count for "
            f"{family_623dk13}."
        )


# ------------------------------------------------------------
# 7. Metrics used in candidate evidence
# ------------------------------------------------------------

candidate_metric_columns_623dk13 = [
    "corrupted_mae",
    "mae_delta_vs_clean",
    "error_spearman",
    "delta_error_spearman",
    "high_low_ratio",
    "top_error_capture_pct",
    "gain90",
    "gain70",
    "gain50",
    "mean_rc_gain",
]

missing_metric_columns_623dk13 = [
    c
    for c in candidate_metric_columns_623dk13
    if c not in EFFECTIVE_DEV_SCENARIO_METRICS_623DK13.columns
]

if missing_metric_columns_623dk13:
    raise RuntimeError(
        "Missing expected metric columns: "
        f"{missing_metric_columns_623dk13}"
    )


# ------------------------------------------------------------
# 8. Finite-value verification AFTER no-op exclusion
# ------------------------------------------------------------

finite_matrix_623dk13 = (
    EFFECTIVE_DEV_SCENARIO_METRICS_623DK13[
        candidate_metric_columns_623dk13
    ]
    .to_numpy(dtype=np.float64)
)

all_effective_metrics_finite_623dk13 = bool(
    np.isfinite(
        finite_matrix_623dk13
    ).all()
)

print("\n--- Finite-value verification ---")
print(
    "All effective metric values finite:",
    all_effective_metrics_finite_623dk13
)

if not all_effective_metrics_finite_623dk13:

    bad_mask_623dk13 = (
        ~np.isfinite(
            finite_matrix_623dk13
        )
    )

    bad_row_positions_623dk13 = np.unique(
        np.where(
            bad_mask_623dk13
        )[0]
    )

    display(
        EFFECTIVE_DEV_SCENARIO_METRICS_623DK13
        .iloc[
            bad_row_positions_623dk13
        ]
    )

    raise RuntimeError(
        "Non-finite metric remains after "
        "excluding the certified no-op scenario."
    )


# ------------------------------------------------------------
# 9. Candidate × family means
#
# We first average scenarios WITHIN each family.
# This prevents families with 90 scenarios from receiving
# twice the weight of families with ~45 scenarios.
# ------------------------------------------------------------

FAMILY_CANDIDATE_SUMMARY_623DK13 = (
    EFFECTIVE_DEV_SCENARIO_METRICS_623DK13
    .groupby(
        [
            "candidate",
            "family"
        ],
        as_index=False
    )[
        candidate_metric_columns_623dk13
    ]
    .mean()
)

print("\n--- Candidate × family summary shape ---")
print(
    FAMILY_CANDIDATE_SUMMARY_623DK13.shape
)

if len(
    FAMILY_CANDIDATE_SUMMARY_623DK13
) != 8 * 5:
    raise RuntimeError(
        "Expected exactly 40 candidate-family rows."
    )


# ------------------------------------------------------------
# 10. Family-balanced candidate aggregation
#
# Each of the five corruption families contributes equally.
# ------------------------------------------------------------

family_balanced_means_623dk13 = (
    FAMILY_CANDIDATE_SUMMARY_623DK13
    .groupby(
        "candidate",
        as_index=False
    )[
        candidate_metric_columns_623dk13
    ]
    .mean()
)

rename_map_623dk13 = {
    c: f"fb_{c}"
    for c in candidate_metric_columns_623dk13
}

family_balanced_means_623dk13 = (
    family_balanced_means_623dk13
    .rename(
        columns=rename_map_623dk13
    )
)


# ------------------------------------------------------------
# 11. Family-consistency diagnostics
# ------------------------------------------------------------

positive_gain70_623dk13 = (
    FAMILY_CANDIDATE_SUMMARY_623DK13
    .assign(
        positive_gain70=lambda d:
            d["gain70"] > 0.0
    )
    .groupby(
        "candidate"
    )[
        "positive_gain70"
    ]
    .sum()
    .astype(int)
    .rename(
        "families_positive_gain70"
    )
)

positive_mean_rc_623dk13 = (
    FAMILY_CANDIDATE_SUMMARY_623DK13
    .assign(
        positive_mean_rc=lambda d:
            d["mean_rc_gain"] > 0.0
    )
    .groupby(
        "candidate"
    )[
        "positive_mean_rc"
    ]
    .sum()
    .astype(int)
    .rename(
        "families_positive_mean_rc_gain"
    )
)

min_family_gain70_623dk13 = (
    FAMILY_CANDIDATE_SUMMARY_623DK13
    .groupby(
        "candidate"
    )[
        "gain70"
    ]
    .min()
    .rename(
        "worst_family_gain70"
    )
)

min_family_mean_rc_623dk13 = (
    FAMILY_CANDIDATE_SUMMARY_623DK13
    .groupby(
        "candidate"
    )[
        "mean_rc_gain"
    ]
    .min()
    .rename(
        "worst_family_mean_rc_gain"
    )
)


# ------------------------------------------------------------
# 12. Final corrected evidence table
#
# IMPORTANT:
# Sorted for readability only.
# This is NOT the final score selection.
# ------------------------------------------------------------

EFFECTIVE_CANDIDATE_SUMMARY_623DK13 = (
    family_balanced_means_623dk13
    .set_index("candidate")
    .join(
        positive_gain70_623dk13
    )
    .join(
        positive_mean_rc_623dk13
    )
    .join(
        min_family_gain70_623dk13
    )
    .join(
        min_family_mean_rc_623dk13
    )
    .reset_index()
)

EFFECTIVE_CANDIDATE_SUMMARY_623DK13 = (
    EFFECTIVE_CANDIDATE_SUMMARY_623DK13
    .sort_values(
        [
            "fb_mean_rc_gain",
            "fb_gain70",
            "fb_error_spearman",
        ],
        ascending=[
            False,
            False,
            False,
        ]
    )
    .reset_index(drop=True)
)

display_columns_623dk13 = [
    "candidate",
    "fb_error_spearman",
    "fb_delta_error_spearman",
    "fb_high_low_ratio",
    "fb_top_error_capture_pct",
    "fb_gain90",
    "fb_gain70",
    "fb_gain50",
    "fb_mean_rc_gain",
    "families_positive_gain70",
    "families_positive_mean_rc_gain",
    "worst_family_gain70",
    "worst_family_mean_rc_gain",
]

print(
    "\n--- Corrected FAMILY-BALANCED "
    "candidate evidence ---"
)

display(
    EFFECTIVE_CANDIDATE_SUMMARY_623DK13[
        display_columns_623dk13
    ]
)


# ------------------------------------------------------------
# 13. Explicit no-op exclusion record
# ------------------------------------------------------------

EXCLUDED_NOOP_SCENARIOS_623DK13 = (
    audit_623dk13[
        ~audit_623dk13[
            "effective_primary"
        ].astype(bool)
    ]
    [
        [
            "global_index",
            "family",
            "family_position",
            "evaluation_label",
            "manifest_scenario_id",
            "sensor",
            "severity",
            "primary_changed_cells",
            "all_raw_changed_cells",
        ]
    ]
    .copy()
)

print("\n--- Excluded no-op audit record ---")
display(
    EXCLUDED_NOOP_SCENARIOS_623DK13
)


# ------------------------------------------------------------
# 14. Integrity summary
# ------------------------------------------------------------

print("\n==========================================")
print("STEP 6.23D-K13 INTEGRITY SUMMARY")
print("==========================================")

print("Frozen scenarios originally evaluated : 315")
print("Effective scenarios retained          : 314")
print("No-op scenarios excluded              : 1")
print(
    "Scenario/candidate rows retained      :",
    len(EFFECTIVE_DEV_SCENARIO_METRICS_623DK13)
)
print(
    "All retained metrics finite           :",
    all_effective_metrics_finite_623dk13
)
print("Family-balanced aggregation used      : True")
print("315 scenarios rerun                   : False")
print("Model inference executed              : False")
print("MC dropout executed                   : False")
print("Stored FULL-DEV results modified      : False")
print("Reliability score selected            : False")
print("Threshold selected                    : False")
print("Abstention rule created               : False")
print("Official test used                    : False")
print("Files written                         : False")
print("=== END ===")

In [ ]:
# ============================================================
# Step 6.23D-K14
# Freeze DEVELOPMENT-selected reliability signal
#
# Selection basis:
#   Corrected 314-effective-scenario,
#   family-balanced DEVELOPMENT evidence.
#
# No threshold selection.
# No abstention rule.
# No official test.
# No files written.
# ============================================================

import numpy as np
import pandas as pd

print(
    "=== Step 6.23D-K14 — "
    "Freeze development-selected reliability signal ==="
)

# ------------------------------------------------------------
# 1. Required objects
# ------------------------------------------------------------

required_623dk14 = [
    "EFFECTIVE_CANDIDATE_SUMMARY_623DK13",
    "DEV_RELIABILITY_CANDIDATES_623B",
    "VAL_RELIABILITY_CANDIDATES_623B",
    "FROZEN_DEV_UNCERTAINTY_622IF",
    "FROZEN_VAL_UNCERTAINTY_622IF",
]

for name_623dk14 in required_623dk14:
    print(
        f"{name_623dk14:40s}:",
        name_623dk14 in globals()
    )

missing_623dk14 = [
    name
    for name in required_623dk14
    if name not in globals()
]

if missing_623dk14:
    raise RuntimeError(
        f"Missing required objects: {missing_623dk14}"
    )


# ------------------------------------------------------------
# 2. Reproducible DEVELOPMENT-only eligibility rule
#
# A candidate must:
#   - improve gain70 in ALL five corruption families
#   - improve mean RC gain in ALL five families
#   - remain positive even in its worst family
#
# This avoids choosing from a single aggregate number.
# ------------------------------------------------------------

summary_623dk14 = (
    EFFECTIVE_CANDIDATE_SUMMARY_623DK13
    .copy()
)

eligibility_mask_623dk14 = (
    (
        summary_623dk14[
            "families_positive_gain70"
        ] == 5
    )
    &
    (
        summary_623dk14[
            "families_positive_mean_rc_gain"
        ] == 5
    )
    &
    (
        summary_623dk14[
            "worst_family_gain70"
        ] > 0.0
    )
    &
    (
        summary_623dk14[
            "worst_family_mean_rc_gain"
        ] > 0.0
    )
)

eligible_623dk14 = (
    summary_623dk14[
        eligibility_mask_623dk14
    ]
    .copy()
)

print("\n--- DEVELOPMENT-only eligible candidates ---")

display(
    eligible_623dk14[
        [
            "candidate",
            "fb_error_spearman",
            "fb_high_low_ratio",
            "fb_gain90",
            "fb_gain70",
            "fb_gain50",
            "fb_mean_rc_gain",
            "families_positive_gain70",
            "families_positive_mean_rc_gain",
            "worst_family_gain70",
            "worst_family_mean_rc_gain",
        ]
    ]
)

if len(eligible_623dk14) == 0:
    raise RuntimeError(
        "No reliability candidate satisfies "
        "the DEVELOPMENT-only robustness criteria."
    )


# ------------------------------------------------------------
# 3. Selection rule among eligible candidates
#
# Primary ranking:
#   highest family-balanced mean risk–coverage gain
#
# Tie breakers:
#   gain70, then error Spearman
# ------------------------------------------------------------

eligible_ranked_623dk14 = (
    eligible_623dk14
    .sort_values(
        [
            "fb_mean_rc_gain",
            "fb_gain70",
            "fb_error_spearman",
        ],
        ascending=[
            False,
            False,
            False,
        ]
    )
    .reset_index(drop=True)
)

SELECTED_RELIABILITY_SCORE_623DK14 = str(
    eligible_ranked_623dk14
    .iloc[0]["candidate"]
)

print("\n--- Selected reliability score ---")
print(
    "Selected candidate:",
    SELECTED_RELIABILITY_SCORE_623DK14
)

if (
    SELECTED_RELIABILITY_SCORE_623DK14
    != "uncertainty_only"
):
    raise RuntimeError(
        "Unexpected DEVELOPMENT-selected candidate. "
        "Inspect before proceeding."
    )


# ------------------------------------------------------------
# 4. Recover exact DEV / clean-validation score arrays
# ------------------------------------------------------------

SELECTED_DEV_RELIABILITY_623DK14 = np.asarray(
    DEV_RELIABILITY_CANDIDATES_623B[
        SELECTED_RELIABILITY_SCORE_623DK14
    ],
    dtype=np.float64
)

SELECTED_VAL_RELIABILITY_623DK14 = np.asarray(
    VAL_RELIABILITY_CANDIDATES_623B[
        SELECTED_RELIABILITY_SCORE_623DK14
    ],
    dtype=np.float64
)

print("\n--- Selected signal arrays ---")
print(
    "Development shape:",
    SELECTED_DEV_RELIABILITY_623DK14.shape
)
print(
    "Validation shape :",
    SELECTED_VAL_RELIABILITY_623DK14.shape
)

if (
    SELECTED_DEV_RELIABILITY_623DK14.shape
    != (14241,)
):
    raise RuntimeError(
        "Unexpected development score shape."
    )

if (
    SELECTED_VAL_RELIABILITY_623DK14.shape
    != (3490,)
):
    raise RuntimeError(
        "Unexpected validation score shape."
    )

if not np.isfinite(
    SELECTED_DEV_RELIABILITY_623DK14
).all():
    raise RuntimeError(
        "Non-finite development reliability score."
    )

if not np.isfinite(
    SELECTED_VAL_RELIABILITY_623DK14
).all():
    raise RuntimeError(
        "Non-finite validation reliability score."
    )


# ------------------------------------------------------------
# 5. Verify exact connection to previously frozen
#    phase-adjusted predictive-uncertainty signal
# ------------------------------------------------------------

frozen_dev_unc_623dk14 = np.asarray(
    FROZEN_DEV_UNCERTAINTY_622IF,
    dtype=np.float64
)

frozen_val_unc_623dk14 = np.asarray(
    FROZEN_VAL_UNCERTAINTY_622IF,
    dtype=np.float64
)

dev_max_error_623dk14 = float(
    np.max(
        np.abs(
            SELECTED_DEV_RELIABILITY_623DK14
            - frozen_dev_unc_623dk14
        )
    )
)

val_max_error_623dk14 = float(
    np.max(
        np.abs(
            SELECTED_VAL_RELIABILITY_623DK14
            - frozen_val_unc_623dk14
        )
    )
)

print("\n--- Frozen-signal linkage ---")
print(
    "DEV max difference vs frozen uncertainty:",
    dev_max_error_623dk14
)
print(
    "VAL max difference vs frozen uncertainty:",
    val_max_error_623dk14
)

if dev_max_error_623dk14 > 1e-12:
    raise RuntimeError(
        "Development uncertainty linkage mismatch."
    )

if val_max_error_623dk14 > 1e-12:
    raise RuntimeError(
        "Validation uncertainty linkage mismatch."
    )


# ------------------------------------------------------------
# 6. Freeze decision metadata IN MEMORY
# ------------------------------------------------------------

selected_row_623dk14 = (
    eligible_ranked_623dk14
    .iloc[0]
)

RELIABILITY_SCORE_DEFINITION_623DK14 = {
    "selected_candidate":
        SELECTED_RELIABILITY_SCORE_623DK14,

    "base_signal":
        "MC-dropout predictive standard deviation",

    "base_std_convention":
        "ddof=0",

    "phase_source":
        "deterministic RUL prediction",

    "phase_bins":
        5,

    "within_phase_transform":
        "empirical mid-rank percentile",

    "selection_dataset":
        "development only",

    "effective_corruption_scenarios":
        314,

    "excluded_noop_scenarios":
        1,

    "excluded_global_index":
        279,

    "family_balanced_selection":
        True,

    "fb_error_spearman":
        float(
            selected_row_623dk14[
                "fb_error_spearman"
            ]
        ),

    "fb_gain70":
        float(
            selected_row_623dk14[
                "fb_gain70"
            ]
        ),

    "fb_mean_rc_gain":
        float(
            selected_row_623dk14[
                "fb_mean_rc_gain"
            ]
        ),

    "families_positive_gain70":
        int(
            selected_row_623dk14[
                "families_positive_gain70"
            ]
        ),

    "families_positive_mean_rc_gain":
        int(
            selected_row_623dk14[
                "families_positive_mean_rc_gain"
            ]
        ),

    "worst_family_gain70":
        float(
            selected_row_623dk14[
                "worst_family_gain70"
            ]
        ),

    "worst_family_mean_rc_gain":
        float(
            selected_row_623dk14[
                "worst_family_mean_rc_gain"
            ]
        ),

    "threshold_selected":
        False,

    "abstention_rule_created":
        False,

    "official_test_used":
        False,
}


# ------------------------------------------------------------
# 7. Compact frozen-decision summary
# ------------------------------------------------------------

print("\n==========================================")
print("STEP 6.23D-K14 FROZEN DECISION SUMMARY")
print("==========================================")

print(
    "Selected reliability signal     :",
    SELECTED_RELIABILITY_SCORE_623DK14
)
print(
    "Eligible robust candidates      :",
    len(eligible_623dk14)
)
print(
    "Effective scenarios considered  : 314"
)
print(
    "No-op scenarios excluded        : 1"
)
print(
    "Family-balanced evidence used   : True"
)

print(
    "Selected fb_error_spearman      :",
    RELIABILITY_SCORE_DEFINITION_623DK14[
        "fb_error_spearman"
    ]
)
print(
    "Selected fb_gain70              :",
    RELIABILITY_SCORE_DEFINITION_623DK14[
        "fb_gain70"
    ]
)
print(
    "Selected fb_mean_rc_gain        :",
    RELIABILITY_SCORE_DEFINITION_623DK14[
        "fb_mean_rc_gain"
    ]
)
print(
    "Families positive gain70        :",
    RELIABILITY_SCORE_DEFINITION_623DK14[
        "families_positive_gain70"
    ],
    "/ 5"
)
print(
    "Families positive mean RC gain  :",
    RELIABILITY_SCORE_DEFINITION_623DK14[
        "families_positive_mean_rc_gain"
    ],
    "/ 5"
)

print(
    "Worst-family gain70             :",
    RELIABILITY_SCORE_DEFINITION_623DK14[
        "worst_family_gain70"
    ]
)
print(
    "Worst-family mean RC gain       :",
    RELIABILITY_SCORE_DEFINITION_623DK14[
        "worst_family_mean_rc_gain"
    ]
)

print(
    "DEV frozen-signal match         :",
    dev_max_error_623dk14 <= 1e-12
)
print(
    "VAL frozen-signal match         :",
    val_max_error_623dk14 <= 1e-12
)

print("Reliability score selected      : True")
print("Reliability threshold selected  : False")
print("Abstention rule created         : False")
print("Official test used              : False")
print("Model retrained                 : False")
print("315 scenarios rerun             : False")
print("Files written                   : False")
print("=== END ===")

In [ ]:
# ============================================================
# Step 6.23D-L1
# DEVELOPMENT-only threshold / coverage candidate evaluation
#
# Selected reliability signal:
#   uncertainty_only
#
# Purpose:
#   Evaluate possible abstention operating points WITHOUT
#   selecting the final threshold yet.
#
# No model inference.
# No MC dropout.
# No scenario rerun.
# No official test.
# No files written.
# ============================================================

import numpy as np
import pandas as pd

print(
    "=== Step 6.23D-L1 — "
    "DEVELOPMENT threshold/coverage candidate evaluation ==="
)

# ------------------------------------------------------------
# 1. Required frozen objects
# ------------------------------------------------------------

required_623dl1 = [
    "SELECTED_RELIABILITY_SCORE_623DK14",
    "SELECTED_DEV_RELIABILITY_623DK14",
    "SELECTED_VAL_RELIABILITY_623DK14",
    "FULL_DEV_SCORE_CUBE_623DK",
    "FULL_DEV_ERROR_MATRIX_623DK",
    "CANDIDATE_NAMES_623DK",
    "SCENARIO_RAW_EFFECT_AUDIT_623DK12",
]

for name_623dl1 in required_623dl1:
    print(
        f"{name_623dl1:40s}:",
        name_623dl1 in globals()
    )

missing_623dl1 = [
    name
    for name in required_623dl1
    if name not in globals()
]

if missing_623dl1:
    raise RuntimeError(
        f"Missing required objects: {missing_623dl1}"
    )


# ------------------------------------------------------------
# 2. Confirm frozen selected signal
# ------------------------------------------------------------

if SELECTED_RELIABILITY_SCORE_623DK14 != "uncertainty_only":
    raise RuntimeError(
        "Frozen reliability signal is not uncertainty_only."
    )

candidate_names_623dl1 = list(
    CANDIDATE_NAMES_623DK
)

if "uncertainty_only" not in candidate_names_623dl1:
    raise RuntimeError(
        "uncertainty_only missing from score cube candidates."
    )

uncertainty_idx_623dl1 = (
    candidate_names_623dl1.index(
        "uncertainty_only"
    )
)

print("\n--- Frozen reliability signal ---")
print(
    "Selected signal      :",
    SELECTED_RELIABILITY_SCORE_623DK14
)
print(
    "Score-cube index     :",
    uncertainty_idx_623dl1
)


# ------------------------------------------------------------
# 3. Certified effective scenario indices
# ------------------------------------------------------------

audit_623dl1 = (
    SCENARIO_RAW_EFFECT_AUDIT_623DK12
    .copy()
)

effective_audit_623dl1 = (
    audit_623dl1[
        audit_623dl1[
            "effective_primary"
        ].astype(bool)
    ]
    .copy()
    .sort_values("global_index")
    .reset_index(drop=True)
)

effective_indices_623dl1 = (
    effective_audit_623dl1[
        "global_index"
    ]
    .astype(int)
    .to_numpy()
)

print("\n--- Scenario set ---")
print(
    "Effective scenarios :",
    len(effective_indices_623dl1)
)

if len(effective_indices_623dl1) != 314:
    raise RuntimeError(
        "Expected exactly 314 effective scenarios."
    )


# ------------------------------------------------------------
# 4. Extract selected score cube and error matrix
# ------------------------------------------------------------

full_score_cube_623dl1 = np.asarray(
    FULL_DEV_SCORE_CUBE_623DK,
    dtype=np.float64
)

full_error_matrix_623dl1 = np.asarray(
    FULL_DEV_ERROR_MATRIX_623DK,
    dtype=np.float64
)

print("\n--- Stored array structure ---")
print(
    "Full score cube :",
    full_score_cube_623dl1.shape
)
print(
    "Full error matrix:",
    full_error_matrix_623dl1.shape
)

if full_score_cube_623dl1.shape != (
    315,
    14241,
    len(candidate_names_623dl1),
):
    raise RuntimeError(
        "Unexpected FULL_DEV score-cube shape."
    )

if full_error_matrix_623dl1.shape != (
    315,
    14241,
):
    raise RuntimeError(
        "Unexpected FULL_DEV error-matrix shape."
    )

scenario_scores_623dl1 = (
    full_score_cube_623dl1[
        effective_indices_623dl1,
        :,
        uncertainty_idx_623dl1,
    ]
)

scenario_errors_623dl1 = (
    full_error_matrix_623dl1[
        effective_indices_623dl1,
        :
    ]
)

if not np.isfinite(
    scenario_scores_623dl1
).all():
    raise RuntimeError(
        "Non-finite selected reliability score detected."
    )

if not np.isfinite(
    scenario_errors_623dl1
).all():
    raise RuntimeError(
        "Non-finite RUL error detected."
    )


# ------------------------------------------------------------
# 5. Candidate nominal CLEAN-development coverages
#
# Diagnostics extend to 50%, but later threshold selection
# will not automatically use the lowest coverage.
# ------------------------------------------------------------

TARGET_COVERAGES_623DL1 = np.array(
    [
        0.95,
        0.90,
        0.85,
        0.80,
        0.75,
        0.70,
        0.60,
        0.50,
    ],
    dtype=np.float64
)

dev_score_623dl1 = np.asarray(
    SELECTED_DEV_RELIABILITY_623DK14,
    dtype=np.float64
)

val_score_623dl1 = np.asarray(
    SELECTED_VAL_RELIABILITY_623DK14,
    dtype=np.float64
)

threshold_rows_623dl1 = []

for target_cov_623dl1 in TARGET_COVERAGES_623DL1:

    # 'higher' gives a deterministic observed score threshold.
    threshold_623dl1 = float(
        np.quantile(
            dev_score_623dl1,
            target_cov_623dl1,
            method="higher",
        )
    )

    clean_dev_coverage_623dl1 = float(
        np.mean(
            dev_score_623dl1
            <= threshold_623dl1
        )
    )

    clean_val_coverage_623dl1 = float(
        np.mean(
            val_score_623dl1
            <= threshold_623dl1
        )
    )

    threshold_rows_623dl1.append(
        {
            "target_clean_coverage":
                float(target_cov_623dl1),

            "threshold":
                threshold_623dl1,

            "clean_dev_coverage":
                clean_dev_coverage_623dl1,

            "clean_val_coverage":
                clean_val_coverage_623dl1,
        }
    )

THRESHOLD_CANDIDATES_623DL1 = (
    pd.DataFrame(
        threshold_rows_623dl1
    )
)

print("\n--- Candidate thresholds ---")
display(
    THRESHOLD_CANDIDATES_623DL1
)


# ------------------------------------------------------------
# 6. Scenario-level selective-risk evaluation
#
# For every effective corruption scenario and every threshold:
#
#   coverage = accepted fraction
#   selective MAE = MAE only among accepted predictions
#
# Gain is relative reduction against the SAME scenario's
# full-coverage MAE:
#
#   gain = (full_MAE - selective_MAE) / full_MAE
#
# Positive gain = useful rejection.
# ------------------------------------------------------------

scenario_result_rows_623dl1 = []

for threshold_row_623dl1 in (
    THRESHOLD_CANDIDATES_623DL1
    .itertuples(index=False)
):

    target_cov_623dl1 = float(
        threshold_row_623dl1.target_clean_coverage
    )

    threshold_623dl1 = float(
        threshold_row_623dl1.threshold
    )

    for scenario_pos_623dl1 in range(
        len(effective_indices_623dl1)
    ):

        score_623dl1 = (
            scenario_scores_623dl1[
                scenario_pos_623dl1
            ]
        )

        error_623dl1 = (
            scenario_errors_623dl1[
                scenario_pos_623dl1
            ]
        )

        accepted_623dl1 = (
            score_623dl1
            <= threshold_623dl1
        )

        accepted_n_623dl1 = int(
            accepted_623dl1.sum()
        )

        total_n_623dl1 = int(
            accepted_623dl1.size
        )

        coverage_623dl1 = float(
            accepted_n_623dl1
            / total_n_623dl1
        )

        full_mae_623dl1 = float(
            np.mean(
                error_623dl1
            )
        )

        if accepted_n_623dl1 == 0:

            selective_mae_623dl1 = np.nan
            relative_gain_623dl1 = np.nan

        else:

            selective_mae_623dl1 = float(
                np.mean(
                    error_623dl1[
                        accepted_623dl1
                    ]
                )
            )

            if full_mae_623dl1 > 0.0:

                relative_gain_623dl1 = float(
                    (
                        full_mae_623dl1
                        - selective_mae_623dl1
                    )
                    / full_mae_623dl1
                )

            else:
                relative_gain_623dl1 = np.nan

        audit_row_623dl1 = (
            effective_audit_623dl1
            .iloc[
                scenario_pos_623dl1
            ]
        )

        scenario_result_rows_623dl1.append(
            {
                "target_clean_coverage":
                    target_cov_623dl1,

                "threshold":
                    threshold_623dl1,

                "global_index":
                    int(
                        audit_row_623dl1[
                            "global_index"
                        ]
                    ),

                "family":
                    str(
                        audit_row_623dl1[
                            "family"
                        ]
                    ),

                "sensor":
                    str(
                        audit_row_623dl1[
                            "sensor"
                        ]
                    ),

                "severity":
                    str(
                        audit_row_623dl1[
                            "severity"
                        ]
                    ),

                "coverage":
                    coverage_623dl1,

                "full_mae":
                    full_mae_623dl1,

                "selective_mae":
                    selective_mae_623dl1,

                "relative_mae_gain":
                    relative_gain_623dl1,
            }
        )


THRESHOLD_SCENARIO_RESULTS_623DL1 = (
    pd.DataFrame(
        scenario_result_rows_623dl1
    )
)

expected_rows_623dl1 = (
    len(TARGET_COVERAGES_623DL1)
    * 314
)

if len(
    THRESHOLD_SCENARIO_RESULTS_623DL1
) != expected_rows_623dl1:
    raise RuntimeError(
        "Unexpected threshold/scenario row count."
    )


# ------------------------------------------------------------
# 7. Check finite selective metrics
# ------------------------------------------------------------

zero_accept_rows_623dl1 = (
    THRESHOLD_SCENARIO_RESULTS_623DL1[
        THRESHOLD_SCENARIO_RESULTS_623DL1[
            "coverage"
        ] <= 0.0
    ]
)

print("\n--- Zero-acceptance scenarios ---")
print(
    "Count:",
    len(zero_accept_rows_623dl1)
)

if len(zero_accept_rows_623dl1) > 0:
    display(
        zero_accept_rows_623dl1[
            [
                "target_clean_coverage",
                "global_index",
                "family",
                "sensor",
                "severity",
                "coverage",
            ]
        ]
    )


# ------------------------------------------------------------
# 8. Family-level aggregation
#
# Each scenario is first evaluated individually.
# Then scenarios are averaged inside each family.
# ------------------------------------------------------------

FAMILY_THRESHOLD_SUMMARY_623DL1 = (
    THRESHOLD_SCENARIO_RESULTS_623DL1
    .groupby(
        [
            "target_clean_coverage",
            "threshold",
            "family",
        ],
        as_index=False
    )
    .agg(
        mean_coverage=(
            "coverage",
            "mean"
        ),

        mean_full_mae=(
            "full_mae",
            "mean"
        ),

        mean_selective_mae=(
            "selective_mae",
            "mean"
        ),

        mean_relative_gain=(
            "relative_mae_gain",
            "mean"
        ),

        median_relative_gain=(
            "relative_mae_gain",
            "median"
        ),

        scenarios_positive_gain=(
            "relative_mae_gain",
            lambda x: int(
                np.sum(
                    np.asarray(x) > 0.0
                )
            )
        ),

        scenarios=(
            "global_index",
            "size"
        ),
    )
)


# ------------------------------------------------------------
# 9. Family-balanced threshold summary
# ------------------------------------------------------------

family_balanced_rows_623dl1 = []

for (
    target_cov_623dl1,
    threshold_623dl1
), group_623dl1 in (
    FAMILY_THRESHOLD_SUMMARY_623DL1
    .groupby(
        [
            "target_clean_coverage",
            "threshold",
        ]
    )
):

    if group_623dl1["family"].nunique() != 5:
        raise RuntimeError(
            "Threshold result missing a corruption family."
        )

    family_balanced_rows_623dl1.append(
        {
            "target_clean_coverage":
                float(target_cov_623dl1),

            "threshold":
                float(threshold_623dl1),

            "family_balanced_coverage":
                float(
                    group_623dl1[
                        "mean_coverage"
                    ].mean()
                ),

            "family_balanced_gain":
                float(
                    group_623dl1[
                        "mean_relative_gain"
                    ].mean()
                ),

            "families_positive_gain":
                int(
                    np.sum(
                        group_623dl1[
                            "mean_relative_gain"
                        ] > 0.0
                    )
                ),

            "worst_family_gain":
                float(
                    group_623dl1[
                        "mean_relative_gain"
                    ].min()
                ),

            "best_family_gain":
                float(
                    group_623dl1[
                        "mean_relative_gain"
                    ].max()
                ),

            "worst_family_coverage":
                float(
                    group_623dl1[
                        "mean_coverage"
                    ].min()
                ),

            "best_family_coverage":
                float(
                    group_623dl1[
                        "mean_coverage"
                    ].max()
                ),
        }
    )


THRESHOLD_EVIDENCE_623DL1 = (
    pd.DataFrame(
        family_balanced_rows_623dl1
    )
    .merge(
        THRESHOLD_CANDIDATES_623DL1,
        on=[
            "target_clean_coverage",
            "threshold",
        ],
        how="left",
        validate="one_to_one",
    )
)


# ------------------------------------------------------------
# 10. Conservative threshold eligibility diagnostic
#
# IMPORTANT:
# This does NOT select a threshold.
#
# A candidate is provisionally "robust" only if:
#   - nominal clean coverage >= 70%
#   - all five corruption families show positive mean gain
#   - worst-family gain is positive
#
# Later we choose among eligible operating points.
# ------------------------------------------------------------

THRESHOLD_EVIDENCE_623DL1[
    "robust_candidate"
] = (
    (
        THRESHOLD_EVIDENCE_623DL1[
            "target_clean_coverage"
        ] >= 0.70
    )
    &
    (
        THRESHOLD_EVIDENCE_623DL1[
            "families_positive_gain"
        ] == 5
    )
    &
    (
        THRESHOLD_EVIDENCE_623DL1[
            "worst_family_gain"
        ] > 0.0
    )
)


# ------------------------------------------------------------
# 11. Display decision evidence
# ------------------------------------------------------------

display_columns_623dl1 = [
    "target_clean_coverage",
    "threshold",
    "clean_dev_coverage",
    "clean_val_coverage",
    "family_balanced_coverage",
    "family_balanced_gain",
    "families_positive_gain",
    "worst_family_gain",
    "best_family_gain",
    "worst_family_coverage",
    "robust_candidate",
]

print(
    "\n--- DEVELOPMENT threshold evidence ---"
)

display(
    THRESHOLD_EVIDENCE_623DL1[
        display_columns_623dl1
    ]
    .sort_values(
        "target_clean_coverage",
        ascending=False,
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 12. Family-by-threshold evidence
# ------------------------------------------------------------

print(
    "\n--- FAMILY-level gain evidence ---"
)

family_gain_pivot_623dl1 = (
    FAMILY_THRESHOLD_SUMMARY_623DL1
    .pivot(
        index="target_clean_coverage",
        columns="family",
        values="mean_relative_gain",
    )
    .reindex(
        TARGET_COVERAGES_623DL1
    )
)

display(
    family_gain_pivot_623dl1
)


# ------------------------------------------------------------
# 13. Integrity
# ------------------------------------------------------------

print("\n==========================================")
print("STEP 6.23D-L1 INTEGRITY SUMMARY")
print("==========================================")

print(
    "Frozen reliability signal       :",
    SELECTED_RELIABILITY_SCORE_623DK14
)
print(
    "Effective scenarios evaluated   :",
    314
)
print(
    "Threshold candidates evaluated  :",
    len(TARGET_COVERAGES_623DL1)
)
print(
    "Threshold/scenario rows          :",
    len(THRESHOLD_SCENARIO_RESULTS_623DL1)
)
print(
    "Zero-acceptance scenario cases   :",
    len(zero_accept_rows_623dl1)
)

print("Reliability score selected       : True")
print("Final threshold selected         : False")
print("Abstention rule created          : False")
print("Model inference executed         : False")
print("MC dropout executed              : False")
print("315 scenarios rerun              : False")
print("Official test used               : False")
print("Files written                    : False")
print("=== END ===")

In [ ]:
# ============================================================
# Step 6.23D-L2
# Threshold scenario-level robustness and CLEAN-validation
# selective-risk diagnostic
#
# No threshold is selected here.
# No model inference.
# No MC dropout.
# No corruption rerun.
# No official test.
# No files written.
# ============================================================

import numpy as np
import pandas as pd

print(
    "=== Step 6.23D-L2 — "
    "Scenario-level threshold robustness diagnostic ==="
)

# ------------------------------------------------------------
# 1. Required objects
# ------------------------------------------------------------

required_623dl2 = [
    "THRESHOLD_SCENARIO_RESULTS_623DL1",
    "THRESHOLD_EVIDENCE_623DL1",
    "THRESHOLD_CANDIDATES_623DL1",
    "SELECTED_VAL_RELIABILITY_623DK14",
    "y_val_clean",
    "clean_predictions",
]

for name_623dl2 in required_623dl2:
    print(
        f"{name_623dl2:40s}:",
        name_623dl2 in globals()
    )

missing_623dl2 = [
    name for name in required_623dl2
    if name not in globals()
]

if missing_623dl2:
    raise RuntimeError(
        f"Missing required objects: {missing_623dl2}"
    )


# ------------------------------------------------------------
# 2. Stored scenario-level results
# ------------------------------------------------------------

scenario_results_623dl2 = (
    THRESHOLD_SCENARIO_RESULTS_623DL1
    .copy()
)

print("\n--- Stored scenario results ---")
print("Rows:", len(scenario_results_623dl2))
print(
    "Scenarios:",
    scenario_results_623dl2[
        "global_index"
    ].nunique()
)
print(
    "Thresholds:",
    scenario_results_623dl2[
        "target_clean_coverage"
    ].nunique()
)

if len(scenario_results_623dl2) != 314 * 8:
    raise RuntimeError(
        "Expected 314 × 8 scenario-threshold rows."
    )


# ------------------------------------------------------------
# 3. Scenario-level robustness by threshold
# ------------------------------------------------------------

scenario_robustness_rows_623dl2 = []

for target_cov_623dl2, group_623dl2 in (
    scenario_results_623dl2
    .groupby("target_clean_coverage")
):

    gains_623dl2 = (
        group_623dl2[
            "relative_mae_gain"
        ]
        .to_numpy(dtype=np.float64)
    )

    coverages_623dl2 = (
        group_623dl2[
            "coverage"
        ]
        .to_numpy(dtype=np.float64)
    )

    if not np.isfinite(gains_623dl2).all():
        raise RuntimeError(
            "Non-finite scenario gain detected."
        )

    scenario_robustness_rows_623dl2.append(
        {
            "target_clean_coverage":
                float(target_cov_623dl2),

            "scenarios":
                int(len(group_623dl2)),

            "positive_gain_scenarios":
                int(np.sum(gains_623dl2 > 0.0)),

            "positive_gain_pct":
                float(
                    100.0 *
                    np.mean(gains_623dl2 > 0.0)
                ),

            "nonnegative_gain_pct":
                float(
                    100.0 *
                    np.mean(gains_623dl2 >= 0.0)
                ),

            "median_gain":
                float(np.median(gains_623dl2)),

            "gain_p10":
                float(
                    np.quantile(
                        gains_623dl2,
                        0.10
                    )
                ),

            "gain_p25":
                float(
                    np.quantile(
                        gains_623dl2,
                        0.25
                    )
                ),

            "gain_p75":
                float(
                    np.quantile(
                        gains_623dl2,
                        0.75
                    )
                ),

            "gain_p90":
                float(
                    np.quantile(
                        gains_623dl2,
                        0.90
                    )
                ),

            "minimum_gain":
                float(np.min(gains_623dl2)),

            "mean_scenario_coverage":
                float(np.mean(coverages_623dl2)),

            "coverage_p10":
                float(
                    np.quantile(
                        coverages_623dl2,
                        0.10
                    )
                ),

            "coverage_p90":
                float(
                    np.quantile(
                        coverages_623dl2,
                        0.90
                    )
                ),
        }
    )


SCENARIO_ROBUSTNESS_623DL2 = (
    pd.DataFrame(
        scenario_robustness_rows_623dl2
    )
    .sort_values(
        "target_clean_coverage",
        ascending=False
    )
    .reset_index(drop=True)
)

print(
    "\n--- Scenario-level robustness ---"
)

display(
    SCENARIO_ROBUSTNESS_623DL2
)


# ------------------------------------------------------------
# 4. CLEAN validation absolute error
# ------------------------------------------------------------

y_val_623dl2 = np.asarray(
    y_val_clean,
    dtype=np.float64
).reshape(-1)

pred_val_623dl2 = np.asarray(
    clean_predictions,
    dtype=np.float64
).reshape(-1)

val_score_623dl2 = np.asarray(
    SELECTED_VAL_RELIABILITY_623DK14,
    dtype=np.float64
).reshape(-1)

print("\n--- CLEAN validation arrays ---")
print("Target shape     :", y_val_623dl2.shape)
print("Prediction shape :", pred_val_623dl2.shape)
print("Reliability shape:", val_score_623dl2.shape)

if not (
    y_val_623dl2.shape
    == pred_val_623dl2.shape
    == val_score_623dl2.shape
    == (3490,)
):
    raise RuntimeError(
        "Unexpected clean-validation array shapes."
    )

clean_val_abs_error_623dl2 = np.abs(
    pred_val_623dl2
    - y_val_623dl2
)

clean_val_full_mae_623dl2 = float(
    np.mean(
        clean_val_abs_error_623dl2
    )
)

print(
    "Full-coverage CLEAN validation MAE:",
    clean_val_full_mae_623dl2
)


# ------------------------------------------------------------
# 5. CLEAN validation selective MAE at each threshold
# ------------------------------------------------------------

clean_val_rows_623dl2 = []

for threshold_row_623dl2 in (
    THRESHOLD_CANDIDATES_623DL1
    .itertuples(index=False)
):

    target_cov_623dl2 = float(
        threshold_row_623dl2.target_clean_coverage
    )

    threshold_623dl2 = float(
        threshold_row_623dl2.threshold
    )

    accepted_623dl2 = (
        val_score_623dl2
        <= threshold_623dl2
    )

    accepted_n_623dl2 = int(
        accepted_623dl2.sum()
    )

    coverage_623dl2 = float(
        np.mean(accepted_623dl2)
    )

    if accepted_n_623dl2 == 0:
        raise RuntimeError(
            "Zero accepted CLEAN-validation predictions."
        )

    selective_mae_623dl2 = float(
        np.mean(
            clean_val_abs_error_623dl2[
                accepted_623dl2
            ]
        )
    )

    relative_gain_623dl2 = float(
        (
            clean_val_full_mae_623dl2
            - selective_mae_623dl2
        )
        / clean_val_full_mae_623dl2
    )

    clean_val_rows_623dl2.append(
        {
            "target_clean_coverage":
                target_cov_623dl2,

            "threshold":
                threshold_623dl2,

            "clean_val_coverage":
                coverage_623dl2,

            "clean_val_selective_mae":
                selective_mae_623dl2,

            "clean_val_relative_gain":
                relative_gain_623dl2,

            "clean_val_abstention_pct":
                100.0 * (
                    1.0 - coverage_623dl2
                ),
        }
    )


CLEAN_VAL_THRESHOLD_UTILITY_623DL2 = (
    pd.DataFrame(
        clean_val_rows_623dl2
    )
)


# ------------------------------------------------------------
# 6. Join corruption robustness + clean-validation evidence
# ------------------------------------------------------------

THRESHOLD_DECISION_EVIDENCE_623DL2 = (
    THRESHOLD_EVIDENCE_623DL1
    .merge(
        SCENARIO_ROBUSTNESS_623DL2,
        on="target_clean_coverage",
        how="left",
        validate="one_to_one",
    )
    .merge(
        CLEAN_VAL_THRESHOLD_UTILITY_623DL2,
        on=[
            "target_clean_coverage",
            "threshold",
        ],
        how="left",
        validate="one_to_one",
    )
)


# ------------------------------------------------------------
# 7. Display the decision-facing columns
# ------------------------------------------------------------

decision_columns_623dl2 = [
    "target_clean_coverage",
    "threshold",
    "clean_val_coverage",
    "clean_val_selective_mae",
    "clean_val_relative_gain",
    "family_balanced_gain",
    "worst_family_gain",
    "positive_gain_pct",
    "median_gain",
    "gain_p10",
    "gain_p25",
    "minimum_gain",
    "mean_scenario_coverage",
    "robust_candidate",
]

print(
    "\n--- Combined threshold decision evidence ---"
)

display(
    THRESHOLD_DECISION_EVIDENCE_623DL2[
        decision_columns_623dl2
    ]
    .sort_values(
        "target_clean_coverage",
        ascending=False
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 8. Family × threshold positive-scenario fraction
# ------------------------------------------------------------

family_positive_623dl2 = (
    scenario_results_623dl2
    .assign(
        positive_gain=lambda d:
            d["relative_mae_gain"] > 0.0
    )
    .groupby(
        [
            "target_clean_coverage",
            "family"
        ],
        as_index=False
    )
    .agg(
        positive_scenarios=(
            "positive_gain",
            "sum"
        ),
        scenarios=(
            "positive_gain",
            "size"
        )
    )
)

family_positive_623dl2[
    "positive_pct"
] = (
    100.0
    * family_positive_623dl2[
        "positive_scenarios"
    ]
    / family_positive_623dl2[
        "scenarios"
    ]
)

FAMILY_POSITIVE_GAIN_PIVOT_623DL2 = (
    family_positive_623dl2
    .pivot(
        index="target_clean_coverage",
        columns="family",
        values="positive_pct"
    )
    .sort_index(
        ascending=False
    )
)

print(
    "\n--- Percent of scenarios with positive gain, by family ---"
)

display(
    FAMILY_POSITIVE_GAIN_PIVOT_623DL2
)


# ------------------------------------------------------------
# 9. Integrity
# ------------------------------------------------------------

print("\n==========================================")
print("STEP 6.23D-L2 INTEGRITY SUMMARY")
print("==========================================")

print("Effective scenarios analysed     : 314")
print("Threshold candidates analysed    : 8")
print(
    "Full CLEAN validation MAE        :",
    clean_val_full_mae_623dl2
)
print(
    "All scenario gains finite        :",
    bool(
        np.isfinite(
            scenario_results_623dl2[
                "relative_mae_gain"
            ].to_numpy(
                dtype=np.float64
            )
        ).all()
    )
)

print("Reliability signal frozen        : True")
print("Final threshold selected         : False")
print("Abstention rule created          : False")
print("Model inference executed         : False")
print("MC dropout executed              : False")
print("Corruption scenarios rerun       : False")
print("Official test used               : False")
print("Files written                    : False")
print("=== END ===")

In [ ]:
# ============================================================
# Step 6.23D-L2A
# Repair reporting merge after duplicate clean_val_coverage name
#
# IMPORTANT:
# - Reuses completed L2 calculations.
# - No model inference.
# - No MC dropout.
# - No corruption replay.
# - No scenario rerun.
# - No official test.
# - No files written.
# ============================================================

import numpy as np
import pandas as pd

print(
    "=== Step 6.23D-L2A — "
    "Repair threshold decision evidence merge ==="
)

# ------------------------------------------------------------
# 1. Required already-computed objects
# ------------------------------------------------------------

required_623dl2a = [
    "THRESHOLD_EVIDENCE_623DL1",
    "SCENARIO_ROBUSTNESS_623DL2",
    "CLEAN_VAL_THRESHOLD_UTILITY_623DL2",
    "THRESHOLD_SCENARIO_RESULTS_623DL1",
]

for name_623dl2a in required_623dl2a:
    print(
        f"{name_623dl2a:42s}:",
        name_623dl2a in globals()
    )

missing_623dl2a = [
    name
    for name in required_623dl2a
    if name not in globals()
]

if missing_623dl2a:
    raise RuntimeError(
        f"Missing required objects: {missing_623dl2a}"
    )


# ------------------------------------------------------------
# 2. Preserve both clean-validation coverage calculations
#    explicitly so there is no ambiguous _x / _y suffix.
# ------------------------------------------------------------

l1_evidence_623dl2a = (
    THRESHOLD_EVIDENCE_623DL1
    .copy()
)

clean_utility_623dl2a = (
    CLEAN_VAL_THRESHOLD_UTILITY_623DL2
    .copy()
)

# Rename the L1 version.
if "clean_val_coverage" in l1_evidence_623dl2a.columns:
    l1_evidence_623dl2a = (
        l1_evidence_623dl2a.rename(
            columns={
                "clean_val_coverage":
                    "clean_val_coverage_l1"
            }
        )
    )

# Rename the directly recomputed L2 version.
if "clean_val_coverage" in clean_utility_623dl2a.columns:
    clean_utility_623dl2a = (
        clean_utility_623dl2a.rename(
            columns={
                "clean_val_coverage":
                    "clean_val_coverage_l2"
            }
        )
    )


# ------------------------------------------------------------
# 3. Rebuild combined decision table cleanly
# ------------------------------------------------------------

THRESHOLD_DECISION_EVIDENCE_623DL2A = (
    l1_evidence_623dl2a
    .merge(
        SCENARIO_ROBUSTNESS_623DL2,
        on="target_clean_coverage",
        how="left",
        validate="one_to_one",
    )
    .merge(
        clean_utility_623dl2a,
        on=[
            "target_clean_coverage",
            "threshold",
        ],
        how="left",
        validate="one_to_one",
    )
)


# ------------------------------------------------------------
# 4. Verify the two clean-validation coverage calculations
# ------------------------------------------------------------

coverage_diff_623dl2a = np.abs(
    THRESHOLD_DECISION_EVIDENCE_623DL2A[
        "clean_val_coverage_l1"
    ].to_numpy(dtype=np.float64)
    -
    THRESHOLD_DECISION_EVIDENCE_623DL2A[
        "clean_val_coverage_l2"
    ].to_numpy(dtype=np.float64)
)

max_coverage_diff_623dl2a = float(
    np.max(coverage_diff_623dl2a)
)

print("\n--- Coverage reconstruction check ---")
print(
    "Maximum |L1 coverage - L2 coverage| :",
    max_coverage_diff_623dl2a
)

if max_coverage_diff_623dl2a > 1e-12:
    raise RuntimeError(
        "L1 and L2 clean-validation coverages disagree."
    )


# Canonical display name after successful verification.
THRESHOLD_DECISION_EVIDENCE_623DL2A[
    "clean_val_coverage"
] = (
    THRESHOLD_DECISION_EVIDENCE_623DL2A[
        "clean_val_coverage_l2"
    ]
)


# ------------------------------------------------------------
# 5. Decision-facing table
# ------------------------------------------------------------

decision_columns_623dl2a = [
    "target_clean_coverage",
    "threshold",
    "clean_dev_coverage",
    "clean_val_coverage",
    "clean_val_selective_mae",
    "clean_val_relative_gain",
    "family_balanced_coverage",
    "family_balanced_gain",
    "families_positive_gain",
    "worst_family_gain",
    "positive_gain_scenarios",
    "positive_gain_pct",
    "median_gain",
    "gain_p10",
    "gain_p25",
    "minimum_gain",
    "mean_scenario_coverage",
    "robust_candidate",
]

print(
    "\n--- Combined threshold decision evidence ---"
)

display(
    THRESHOLD_DECISION_EVIDENCE_623DL2A[
        decision_columns_623dl2a
    ]
    .sort_values(
        "target_clean_coverage",
        ascending=False
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 6. Family × threshold individual-scenario success rates
# ------------------------------------------------------------

scenario_results_623dl2a = (
    THRESHOLD_SCENARIO_RESULTS_623DL1
    .copy()
)

family_positive_623dl2a = (
    scenario_results_623dl2a
    .assign(
        positive_gain=lambda d:
            d["relative_mae_gain"] > 0.0
    )
    .groupby(
        [
            "target_clean_coverage",
            "family",
        ],
        as_index=False,
    )
    .agg(
        positive_scenarios=(
            "positive_gain",
            "sum"
        ),
        scenarios=(
            "positive_gain",
            "size"
        ),
    )
)

family_positive_623dl2a[
    "positive_pct"
] = (
    100.0
    * family_positive_623dl2a[
        "positive_scenarios"
    ]
    / family_positive_623dl2a[
        "scenarios"
    ]
)

FAMILY_POSITIVE_GAIN_PIVOT_623DL2A = (
    family_positive_623dl2a
    .pivot(
        index="target_clean_coverage",
        columns="family",
        values="positive_pct",
    )
    .sort_index(
        ascending=False
    )
)

print(
    "\n--- Percent of scenarios with positive gain, by family ---"
)

display(
    FAMILY_POSITIVE_GAIN_PIVOT_623DL2A
)


# ------------------------------------------------------------
# 7. Focused comparison of the realistic operating region
#    (95% through 70%; 60%/50% remain diagnostics only)
# ------------------------------------------------------------

FOCUSED_THRESHOLD_EVIDENCE_623DL2A = (
    THRESHOLD_DECISION_EVIDENCE_623DL2A[
        THRESHOLD_DECISION_EVIDENCE_623DL2A[
            "target_clean_coverage"
        ] >= 0.70
    ]
    .copy()
    .sort_values(
        "target_clean_coverage",
        ascending=False
    )
)

print(
    "\n--- Focused robust operating region (>=70% nominal coverage) ---"
)

display(
    FOCUSED_THRESHOLD_EVIDENCE_623DL2A[
        [
            "target_clean_coverage",
            "threshold",
            "clean_val_coverage",
            "clean_val_selective_mae",
            "clean_val_relative_gain",
            "family_balanced_gain",
            "worst_family_gain",
            "positive_gain_pct",
            "gain_p10",
            "minimum_gain",
        ]
    ]
)


# ------------------------------------------------------------
# 8. Integrity summary
# ------------------------------------------------------------

print("\n============================================")
print("STEP 6.23D-L2A INTEGRITY SUMMARY")
print("============================================")

print(
    "Decision-table rows              :",
    len(THRESHOLD_DECISION_EVIDENCE_623DL2A)
)

print(
    "Coverage reconstruction exact    :",
    bool(max_coverage_diff_623dl2a <= 1e-12)
)

print(
    "Scenario result rows             :",
    len(scenario_results_623dl2a)
)

print(
    "Unique effective scenarios       :",
    scenario_results_623dl2a[
        "global_index"
    ].nunique()
)

print(
    "Threshold candidates             :",
    scenario_results_623dl2a[
        "target_clean_coverage"
    ].nunique()
)

print(
    "All combined numeric evidence finite:",
    bool(
        np.isfinite(
            THRESHOLD_DECISION_EVIDENCE_623DL2A[
                [
                    "threshold",
                    "clean_val_coverage",
                    "clean_val_selective_mae",
                    "clean_val_relative_gain",
                    "family_balanced_gain",
                    "worst_family_gain",
                    "positive_gain_pct",
                    "median_gain",
                    "gain_p10",
                    "gain_p25",
                    "minimum_gain",
                    "mean_scenario_coverage",
                ]
            ].to_numpy(dtype=np.float64)
        ).all()
    )
)

print("Reliability signal frozen        : True")
print("Final threshold selected         : False")
print("Abstention rule created          : False")
print("Model inference executed         : False")
print("MC dropout executed              : False")
print("Corruption scenarios rerun       : False")
print("Official test used               : False")
print("Files written                    : False")
print("=== END ===")

In [ ]:
# ============================================================
# Step 6.23D-L3
# Freeze final DEVELOPMENT-selected reliability threshold
# and abstention rule
#
# Decision:
#   reliability signal = uncertainty_only
#   target clean coverage = 0.75
#
# No model inference.
# No MC dropout.
# No corruption replay.
# No official test.
# No files written yet.
# ============================================================

import numpy as np
import pandas as pd

print(
    "=== Step 6.23D-L3 — "
    "Freeze final reliability threshold / abstention rule ==="
)

# ------------------------------------------------------------
# 1. Required objects
# ------------------------------------------------------------

required_623dl3 = [
    "THRESHOLD_DECISION_EVIDENCE_623DL2A",
    "SELECTED_RELIABILITY_SCORE_623DK14",
    "SELECTED_DEV_RELIABILITY_623DK14",
    "SELECTED_VAL_RELIABILITY_623DK14",
    "y_val_clean",
    "clean_predictions",
]

for name_623dl3 in required_623dl3:
    print(
        f"{name_623dl3:42s}:",
        name_623dl3 in globals()
    )

missing_623dl3 = [
    name
    for name in required_623dl3
    if name not in globals()
]

if missing_623dl3:
    raise RuntimeError(
        f"Missing required objects: {missing_623dl3}"
    )


# ------------------------------------------------------------
# 2. Freeze selected operating point
# ------------------------------------------------------------

FROZEN_TARGET_CLEAN_COVERAGE_623DL3 = 0.75

selected_row_623dl3 = (
    THRESHOLD_DECISION_EVIDENCE_623DL2A.loc[
        np.isclose(
            THRESHOLD_DECISION_EVIDENCE_623DL2A[
                "target_clean_coverage"
            ].to_numpy(dtype=np.float64),
            FROZEN_TARGET_CLEAN_COVERAGE_623DL3,
            atol=1e-12,
            rtol=0.0,
        )
    ]
    .copy()
)

if len(selected_row_623dl3) != 1:
    raise RuntimeError(
        "Could not identify exactly one 75% threshold row."
    )

selected_row_623dl3 = selected_row_623dl3.iloc[0]


# Full-precision threshold from development evidence.
FROZEN_RELIABILITY_THRESHOLD_623DL3 = float(
    selected_row_623dl3["threshold"]
)

FROZEN_RELIABILITY_SIGNAL_NAME_623DL3 = str(
    SELECTED_RELIABILITY_SCORE_623DK14
)


# ------------------------------------------------------------
# 3. Frozen reliability arrays
# ------------------------------------------------------------

FROZEN_DEV_RELIABILITY_623DL3 = np.asarray(
    SELECTED_DEV_RELIABILITY_623DK14,
    dtype=np.float64,
).copy()

FROZEN_VAL_RELIABILITY_623DL3 = np.asarray(
    SELECTED_VAL_RELIABILITY_623DK14,
    dtype=np.float64,
).copy()

if FROZEN_DEV_RELIABILITY_623DL3.shape != (14241,):
    raise RuntimeError(
        "Unexpected DEV reliability shape."
    )

if FROZEN_VAL_RELIABILITY_623DL3.shape != (3490,):
    raise RuntimeError(
        "Unexpected validation reliability shape."
    )

if not (
    np.isfinite(
        FROZEN_DEV_RELIABILITY_623DL3
    ).all()
    and
    np.isfinite(
        FROZEN_VAL_RELIABILITY_623DL3
    ).all()
):
    raise RuntimeError(
        "Non-finite frozen reliability value detected."
    )


# ------------------------------------------------------------
# 4. Create the final abstention rule
#
# Reliability orientation:
# HIGHER = WORSE
#
# Therefore:
# ACCEPT  -> score <= threshold
# ABSTAIN -> score >  threshold
# ------------------------------------------------------------

FROZEN_DEV_ACCEPT_MASK_623DL3 = (
    FROZEN_DEV_RELIABILITY_623DL3
    <= FROZEN_RELIABILITY_THRESHOLD_623DL3
)

FROZEN_VAL_ACCEPT_MASK_623DL3 = (
    FROZEN_VAL_RELIABILITY_623DL3
    <= FROZEN_RELIABILITY_THRESHOLD_623DL3
)

FROZEN_DEV_ABSTAIN_MASK_623DL3 = (
    ~FROZEN_DEV_ACCEPT_MASK_623DL3
)

FROZEN_VAL_ABSTAIN_MASK_623DL3 = (
    ~FROZEN_VAL_ACCEPT_MASK_623DL3
)


# ------------------------------------------------------------
# 5. Coverage verification
# ------------------------------------------------------------

dev_coverage_623dl3 = float(
    FROZEN_DEV_ACCEPT_MASK_623DL3.mean()
)

val_coverage_623dl3 = float(
    FROZEN_VAL_ACCEPT_MASK_623DL3.mean()
)

stored_dev_coverage_623dl3 = float(
    selected_row_623dl3[
        "clean_dev_coverage"
    ]
)

stored_val_coverage_623dl3 = float(
    selected_row_623dl3[
        "clean_val_coverage"
    ]
)

dev_cov_error_623dl3 = abs(
    dev_coverage_623dl3
    - stored_dev_coverage_623dl3
)

val_cov_error_623dl3 = abs(
    val_coverage_623dl3
    - stored_val_coverage_623dl3
)

print("\n--- Coverage verification ---")

print(
    "DEV reconstructed coverage :",
    dev_coverage_623dl3
)

print(
    "DEV stored coverage        :",
    stored_dev_coverage_623dl3
)

print(
    "DEV coverage difference    :",
    dev_cov_error_623dl3
)

print()

print(
    "VAL reconstructed coverage :",
    val_coverage_623dl3
)

print(
    "VAL stored coverage        :",
    stored_val_coverage_623dl3
)

print(
    "VAL coverage difference    :",
    val_cov_error_623dl3
)

if dev_cov_error_623dl3 > 1e-12:
    raise RuntimeError(
        "DEV threshold coverage reconstruction failed."
    )

if val_cov_error_623dl3 > 1e-12:
    raise RuntimeError(
        "Validation threshold coverage reconstruction failed."
    )


# ------------------------------------------------------------
# 6. Clean-validation selective-risk verification
# ------------------------------------------------------------

y_val_623dl3 = np.asarray(
    y_val_clean,
    dtype=np.float64,
).reshape(-1)

pred_val_623dl3 = np.asarray(
    clean_predictions,
    dtype=np.float64,
).reshape(-1)

if not (
    y_val_623dl3.shape
    == pred_val_623dl3.shape
    == FROZEN_VAL_ACCEPT_MASK_623DL3.shape
    == (3490,)
):
    raise RuntimeError(
        "Validation arrays are not aligned."
    )

val_abs_error_623dl3 = np.abs(
    pred_val_623dl3
    - y_val_623dl3
)

full_val_mae_623dl3 = float(
    np.mean(
        val_abs_error_623dl3
    )
)

accepted_val_mae_623dl3 = float(
    np.mean(
        val_abs_error_623dl3[
            FROZEN_VAL_ACCEPT_MASK_623DL3
        ]
    )
)

val_relative_gain_623dl3 = float(
    (
        full_val_mae_623dl3
        - accepted_val_mae_623dl3
    )
    / full_val_mae_623dl3
)

stored_selective_mae_623dl3 = float(
    selected_row_623dl3[
        "clean_val_selective_mae"
    ]
)

stored_relative_gain_623dl3 = float(
    selected_row_623dl3[
        "clean_val_relative_gain"
    ]
)

print(
    "\n--- CLEAN validation selective-risk verification ---"
)

print(
    "Full-coverage validation MAE :",
    full_val_mae_623dl3
)

print(
    "Accepted-prediction MAE      :",
    accepted_val_mae_623dl3
)

print(
    "Relative MAE gain            :",
    val_relative_gain_623dl3
)

print(
    "Accepted predictions         :",
    int(
        FROZEN_VAL_ACCEPT_MASK_623DL3.sum()
    ),
    "/",
    len(
        FROZEN_VAL_ACCEPT_MASK_623DL3
    )
)

print(
    "Abstained predictions        :",
    int(
        FROZEN_VAL_ABSTAIN_MASK_623DL3.sum()
    ),
    "/",
    len(
        FROZEN_VAL_ABSTAIN_MASK_623DL3
    )
)

if not np.isclose(
    accepted_val_mae_623dl3,
    stored_selective_mae_623dl3,
    atol=1e-12,
    rtol=0.0,
):
    raise RuntimeError(
        "Stored selective MAE was not reproduced."
    )

if not np.isclose(
    val_relative_gain_623dl3,
    stored_relative_gain_623dl3,
    atol=1e-12,
    rtol=0.0,
):
    raise RuntimeError(
        "Stored clean-validation gain was not reproduced."
    )


# ------------------------------------------------------------
# 7. Freeze rule definition
# ------------------------------------------------------------

FROZEN_ABSTENTION_RULE_623DL3 = {
    "reliability_signal":
        FROZEN_RELIABILITY_SIGNAL_NAME_623DL3,

    "orientation":
        "higher_is_worse",

    "target_clean_coverage":
        FROZEN_TARGET_CLEAN_COVERAGE_623DL3,

    "threshold":
        FROZEN_RELIABILITY_THRESHOLD_623DL3,

    "accept_rule":
        "reliability_score <= threshold",

    "abstain_rule":
        "reliability_score > threshold",

    "threshold_source":
        "clean development only",

    "selection_basis":
        (
            "balanced clean-validation selective risk, "
            "family-balanced corruption robustness, "
            "scenario-level robustness, and coverage"
        ),

    "effective_corruption_scenarios":
        314,

    "no_op_scenarios_excluded":
        1,

    "official_test_used":
        False,
}


# ------------------------------------------------------------
# 8. Frozen decision evidence
# ------------------------------------------------------------

FROZEN_THRESHOLD_EVIDENCE_623DL3 = (
    selected_row_623dl3.to_dict()
)


print("\n============================================")
print("STEP 6.23D-L3 FROZEN DECISION SUMMARY")
print("============================================")

print(
    "Reliability signal          :",
    FROZEN_RELIABILITY_SIGNAL_NAME_623DL3
)

print(
    "Signal orientation          : higher = worse"
)

print(
    "Target clean coverage       :",
    FROZEN_TARGET_CLEAN_COVERAGE_623DL3
)

print(
    "Frozen threshold            :",
    repr(
        FROZEN_RELIABILITY_THRESHOLD_623DL3
    )
)

print(
    "DEV achieved coverage       :",
    dev_coverage_623dl3
)

print(
    "Clean VAL achieved coverage :",
    val_coverage_623dl3
)

print(
    "Clean VAL full MAE          :",
    full_val_mae_623dl3
)

print(
    "Clean VAL selective MAE     :",
    accepted_val_mae_623dl3
)

print(
    "Clean VAL relative gain     :",
    val_relative_gain_623dl3
)

print(
    "Corruption scenario gain %  :",
    float(
        selected_row_623dl3[
            "positive_gain_pct"
        ]
    )
)

print(
    "Family-balanced gain        :",
    float(
        selected_row_623dl3[
            "family_balanced_gain"
        ]
    )
)

print(
    "Worst-family gain           :",
    float(
        selected_row_623dl3[
            "worst_family_gain"
        ]
    )
)

print(
    "Scenario gain p10           :",
    float(
        selected_row_623dl3[
            "gain_p10"
        ]
    )
)

print(
    "Minimum scenario gain       :",
    float(
        selected_row_623dl3[
            "minimum_gain"
        ]
    )
)

print()
print("Accept rule                 : score <= threshold")
print("Abstain rule                : score > threshold")

print()
print("Reliability score selected  : True")
print("Final threshold selected    : True")
print("Abstention rule created     : True")
print("Model inference executed    : False")
print("MC dropout executed         : False")
print("Corruption scenarios rerun  : False")
print("Official test used          : False")
print("Files written               : False")
print("=== END ===")

In [ ]:
# ============================================================
# Step 6.23D-L4
# Persist frozen reliability signal, threshold, abstention rule,
# evidence table, and reproducibility arrays.
#
# No model inference.
# No MC dropout.
# No corruption replay.
# No official test.
# ============================================================

from pathlib import Path
import json
import hashlib
import numpy as np
import pandas as pd

print(
    "=== Step 6.23D-L4 — "
    "Persist frozen reliability decision ==="
)

# ------------------------------------------------------------
# 1. Required frozen objects
# ------------------------------------------------------------

required_623dl4 = [
    "FROZEN_RELIABILITY_SIGNAL_NAME_623DL3",
    "FROZEN_TARGET_CLEAN_COVERAGE_623DL3",
    "FROZEN_RELIABILITY_THRESHOLD_623DL3",
    "FROZEN_DEV_RELIABILITY_623DL3",
    "FROZEN_VAL_RELIABILITY_623DL3",
    "FROZEN_DEV_ACCEPT_MASK_623DL3",
    "FROZEN_VAL_ACCEPT_MASK_623DL3",
    "FROZEN_ABSTENTION_RULE_623DL3",
    "FROZEN_THRESHOLD_EVIDENCE_623DL3",
    "THRESHOLD_DECISION_EVIDENCE_623DL2A",
]

for name_623dl4 in required_623dl4:
    print(
        f"{name_623dl4:45s}:",
        name_623dl4 in globals()
    )

missing_623dl4 = [
    name
    for name in required_623dl4
    if name not in globals()
]

if missing_623dl4:
    raise RuntimeError(
        f"Missing required frozen objects: {missing_623dl4}"
    )


# ------------------------------------------------------------
# 2. Final freeze integrity checks
# ------------------------------------------------------------

if FROZEN_RELIABILITY_SIGNAL_NAME_623DL3 != "uncertainty_only":
    raise RuntimeError(
        "Unexpected frozen reliability signal."
    )

if not np.isclose(
    FROZEN_TARGET_CLEAN_COVERAGE_623DL3,
    0.75,
    atol=1e-12,
    rtol=0.0,
):
    raise RuntimeError(
        "Unexpected frozen target coverage."
    )

if not np.isfinite(
    FROZEN_RELIABILITY_THRESHOLD_623DL3
):
    raise RuntimeError(
        "Frozen reliability threshold is non-finite."
    )

if FROZEN_DEV_RELIABILITY_623DL3.shape != (14241,):
    raise RuntimeError(
        "Unexpected DEV reliability shape."
    )

if FROZEN_VAL_RELIABILITY_623DL3.shape != (3490,):
    raise RuntimeError(
        "Unexpected VAL reliability shape."
    )

if not (
    np.isfinite(
        FROZEN_DEV_RELIABILITY_623DL3
    ).all()
    and
    np.isfinite(
        FROZEN_VAL_RELIABILITY_623DL3
    ).all()
):
    raise RuntimeError(
        "Non-finite reliability value detected."
    )


# ------------------------------------------------------------
# 3. Locate project root
# ------------------------------------------------------------

cwd_623dl4 = Path.cwd()

if (cwd_623dl4 / "06_results").exists():
    PROJECT_ROOT_623DL4 = cwd_623dl4

elif (cwd_623dl4.parent / "06_results").exists():
    PROJECT_ROOT_623DL4 = cwd_623dl4.parent

else:
    raise FileNotFoundError(
        "Could not locate project root containing 06_results."
    )


OUTPUT_DIR_623DL4 = (
    PROJECT_ROOT_623DL4
    / "06_results"
    / "step6_reliability_signals"
)

OUTPUT_DIR_623DL4.mkdir(
    parents=True,
    exist_ok=True,
)

print(
    "\nOutput directory:",
    OUTPUT_DIR_623DL4
)


# ------------------------------------------------------------
# 4. Output paths
# ------------------------------------------------------------

rule_path_623dl4 = (
    OUTPUT_DIR_623DL4
    / "fd001_frozen_reliability_decision.json"
)

candidate_path_623dl4 = (
    OUTPUT_DIR_623DL4
    / "fd001_threshold_candidate_evidence.csv"
)

arrays_path_623dl4 = (
    OUTPUT_DIR_623DL4
    / "fd001_frozen_reliability_decision_arrays.npz"
)

manifest_path_623dl4 = (
    OUTPUT_DIR_623DL4
    / "fd001_frozen_reliability_decision_manifest.json"
)


# ------------------------------------------------------------
# 5. JSON-safe conversion helper
# ------------------------------------------------------------

def json_safe_623dl4(value):
    if isinstance(value, np.generic):
        return value.item()

    if isinstance(value, np.ndarray):
        return value.tolist()

    if isinstance(value, dict):
        return {
            str(k): json_safe_623dl4(v)
            for k, v in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [
            json_safe_623dl4(v)
            for v in value
        ]

    if pd.isna(value):
        return None

    return value


# ------------------------------------------------------------
# 6. Construct frozen rule record
# ------------------------------------------------------------

selected_evidence_623dl4 = {
    str(k): json_safe_623dl4(v)
    for k, v in
    FROZEN_THRESHOLD_EVIDENCE_623DL3.items()
}

rule_record_623dl4 = {
    "dataset": "NASA C-MAPSS FD001",

    "decision_status": "FROZEN",

    "reliability_signal":
        FROZEN_RELIABILITY_SIGNAL_NAME_623DL3,

    "signal_orientation":
        "higher_is_worse",

    "target_clean_development_coverage":
        float(
            FROZEN_TARGET_CLEAN_COVERAGE_623DL3
        ),

    "reliability_threshold":
        float(
            FROZEN_RELIABILITY_THRESHOLD_623DL3
        ),

    "accept_rule":
        "reliability_score <= reliability_threshold",

    "abstain_rule":
        "reliability_score > reliability_threshold",

    "threshold_calibration_source":
        "clean development data",

    "effective_corruption_scenarios":
        314,

    "excluded_no_op_scenarios":
        1,

    "official_test_used_for_selection":
        False,

    "selection_locked":
        True,

    "selected_threshold_evidence":
        selected_evidence_623dl4,
}


# ------------------------------------------------------------
# 7. Write frozen rule JSON
# ------------------------------------------------------------

with open(
    rule_path_623dl4,
    "w",
    encoding="utf-8",
) as f_623dl4:

    json.dump(
        rule_record_623dl4,
        f_623dl4,
        indent=2,
        sort_keys=True,
    )


# ------------------------------------------------------------
# 8. Save complete candidate evidence
# ------------------------------------------------------------

THRESHOLD_DECISION_EVIDENCE_623DL2A.to_csv(
    candidate_path_623dl4,
    index=False,
)


# ------------------------------------------------------------
# 9. Save exact frozen arrays
# ------------------------------------------------------------

np.savez_compressed(
    arrays_path_623dl4,

    dev_reliability=np.asarray(
        FROZEN_DEV_RELIABILITY_623DL3,
        dtype=np.float64,
    ),

    val_reliability=np.asarray(
        FROZEN_VAL_RELIABILITY_623DL3,
        dtype=np.float64,
    ),

    dev_accept_mask=np.asarray(
        FROZEN_DEV_ACCEPT_MASK_623DL3,
        dtype=bool,
    ),

    val_accept_mask=np.asarray(
        FROZEN_VAL_ACCEPT_MASK_623DL3,
        dtype=bool,
    ),

    threshold=np.asarray(
        [
            FROZEN_RELIABILITY_THRESHOLD_623DL3
        ],
        dtype=np.float64,
    ),

    target_clean_coverage=np.asarray(
        [
            FROZEN_TARGET_CLEAN_COVERAGE_623DL3
        ],
        dtype=np.float64,
    ),
)


# ------------------------------------------------------------
# 10. SHA-256 helper
# ------------------------------------------------------------

def sha256_file_623dl4(path):
    digest = hashlib.sha256()

    with open(path, "rb") as file_obj:
        while True:
            chunk = file_obj.read(1024 * 1024)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


# ------------------------------------------------------------
# 11. Reproducibility manifest
# ------------------------------------------------------------

manifest_623dl4 = {
    "dataset": "NASA C-MAPSS FD001",

    "frozen_reliability_signal":
        FROZEN_RELIABILITY_SIGNAL_NAME_623DL3,

    "frozen_threshold":
        float(
            FROZEN_RELIABILITY_THRESHOLD_623DL3
        ),

    "target_clean_coverage":
        float(
            FROZEN_TARGET_CLEAN_COVERAGE_623DL3
        ),

    "development_windows":
        int(
            len(
                FROZEN_DEV_RELIABILITY_623DL3
            )
        ),

    "validation_windows":
        int(
            len(
                FROZEN_VAL_RELIABILITY_623DL3
            )
        ),

    "development_accepted":
        int(
            np.sum(
                FROZEN_DEV_ACCEPT_MASK_623DL3
            )
        ),

    "validation_accepted":
        int(
            np.sum(
                FROZEN_VAL_ACCEPT_MASK_623DL3
            )
        ),

    "official_test_used":
        False,

    "files": {},
}


# Hash files already written.
for file_path_623dl4 in [
    rule_path_623dl4,
    candidate_path_623dl4,
    arrays_path_623dl4,
]:

    manifest_623dl4["files"][
        file_path_623dl4.name
    ] = {
        "sha256":
            sha256_file_623dl4(
                file_path_623dl4
            )
    }


with open(
    manifest_path_623dl4,
    "w",
    encoding="utf-8",
) as f_623dl4:

    json.dump(
        manifest_623dl4,
        f_623dl4,
        indent=2,
        sort_keys=True,
    )


# ------------------------------------------------------------
# 12. Verify written artifacts exist
# ------------------------------------------------------------

written_files_623dl4 = [
    rule_path_623dl4,
    candidate_path_623dl4,
    arrays_path_623dl4,
    manifest_path_623dl4,
]

for path_623dl4 in written_files_623dl4:
    if not path_623dl4.exists():
        raise RuntimeError(
            f"Expected frozen artifact missing: "
            f"{path_623dl4}"
        )


# ------------------------------------------------------------
# 13. Compact frozen summary
# ------------------------------------------------------------

print("\n============================================")
print("STEP 6.23D-L4 FROZEN ARTIFACT SUMMARY")
print("============================================")

print(
    "Reliability signal     :",
    FROZEN_RELIABILITY_SIGNAL_NAME_623DL3
)

print(
    "Target DEV coverage    :",
    FROZEN_TARGET_CLEAN_COVERAGE_623DL3
)

print(
    "Exact frozen threshold :",
    repr(
        FROZEN_RELIABILITY_THRESHOLD_623DL3
    )
)

print(
    "DEV accepted           :",
    int(
        FROZEN_DEV_ACCEPT_MASK_623DL3.sum()
    ),
    "/",
    len(
        FROZEN_DEV_ACCEPT_MASK_623DL3
    )
)

print(
    "VAL accepted           :",
    int(
        FROZEN_VAL_ACCEPT_MASK_623DL3.sum()
    ),
    "/",
    len(
        FROZEN_VAL_ACCEPT_MASK_623DL3
    )
)

print()
print("Frozen files:")

for i_623dl4, path_623dl4 in enumerate(
    written_files_623dl4,
    start=1,
):
    print(
        f"{i_623dl4}. {path_623dl4}"
    )

print()
print("Reliability score frozen : True")
print("Threshold frozen         : True")
print("Abstention rule frozen   : True")
print("Official test used       : False")
print("Model inference executed : False")
print("Corruptions rerun        : False")
print("Files written            : True")
print("=== END ===")

In [ ]:
# ============================================================
# Step 6.23D-L5
# Reload and independently verify frozen reliability artifacts
#
# No model inference.
# No MC dropout.
# No corruption replay.
# No official test.
# ============================================================

from pathlib import Path
import json
import hashlib
import numpy as np
import pandas as pd

print(
    "=== Step 6.23D-L5 — "
    "Reload / verify frozen reliability artifacts ==="
)

# ------------------------------------------------------------
# 1. Locate project root
# ------------------------------------------------------------

cwd_623dl5 = Path.cwd()

if (cwd_623dl5 / "06_results").exists():
    PROJECT_ROOT_623DL5 = cwd_623dl5

elif (cwd_623dl5.parent / "06_results").exists():
    PROJECT_ROOT_623DL5 = cwd_623dl5.parent

else:
    raise FileNotFoundError(
        "Could not locate project root containing 06_results."
    )


artifact_dir_623dl5 = (
    PROJECT_ROOT_623DL5
    / "06_results"
    / "step6_reliability_signals"
)


# ------------------------------------------------------------
# 2. Expected files
# ------------------------------------------------------------

rule_path_623dl5 = (
    artifact_dir_623dl5
    / "fd001_frozen_reliability_decision.json"
)

candidate_path_623dl5 = (
    artifact_dir_623dl5
    / "fd001_threshold_candidate_evidence.csv"
)

arrays_path_623dl5 = (
    artifact_dir_623dl5
    / "fd001_frozen_reliability_decision_arrays.npz"
)

manifest_path_623dl5 = (
    artifact_dir_623dl5
    / "fd001_frozen_reliability_decision_manifest.json"
)

paths_623dl5 = [
    rule_path_623dl5,
    candidate_path_623dl5,
    arrays_path_623dl5,
    manifest_path_623dl5,
]

print("\n--- Frozen file availability ---")

for path_623dl5 in paths_623dl5:
    print(
        f"{path_623dl5.name:52s}:",
        path_623dl5.exists()
    )

missing_623dl5 = [
    str(path)
    for path in paths_623dl5
    if not path.exists()
]

if missing_623dl5:
    raise RuntimeError(
        f"Missing frozen artifacts: {missing_623dl5}"
    )


# ------------------------------------------------------------
# 3. Reload artifacts
# ------------------------------------------------------------

with open(
    rule_path_623dl5,
    "r",
    encoding="utf-8",
) as f_623dl5:
    RELOADED_RULE_623DL5 = json.load(f_623dl5)


RELOADED_THRESHOLD_TABLE_623DL5 = pd.read_csv(
    candidate_path_623dl5
)


with np.load(
    arrays_path_623dl5,
    allow_pickle=False,
) as npz_623dl5:

    RELOADED_DEV_RELIABILITY_623DL5 = (
        npz_623dl5[
            "dev_reliability"
        ].astype(
            np.float64,
            copy=True,
        )
    )

    RELOADED_VAL_RELIABILITY_623DL5 = (
        npz_623dl5[
            "val_reliability"
        ].astype(
            np.float64,
            copy=True,
        )
    )

    RELOADED_DEV_ACCEPT_MASK_623DL5 = (
        npz_623dl5[
            "dev_accept_mask"
        ].astype(
            bool,
            copy=True,
        )
    )

    RELOADED_VAL_ACCEPT_MASK_623DL5 = (
        npz_623dl5[
            "val_accept_mask"
        ].astype(
            bool,
            copy=True,
        )
    )

    RELOADED_THRESHOLD_623DL5 = float(
        npz_623dl5[
            "threshold"
        ].reshape(-1)[0]
    )

    RELOADED_TARGET_COVERAGE_623DL5 = float(
        npz_623dl5[
            "target_clean_coverage"
        ].reshape(-1)[0]
    )


with open(
    manifest_path_623dl5,
    "r",
    encoding="utf-8",
) as f_623dl5:
    RELOADED_MANIFEST_623DL5 = json.load(f_623dl5)


# ------------------------------------------------------------
# 4. SHA-256 verification
# ------------------------------------------------------------

def sha256_file_623dl5(path):
    digest_623dl5 = hashlib.sha256()

    with open(path, "rb") as file_obj_623dl5:
        while True:
            chunk_623dl5 = file_obj_623dl5.read(
                1024 * 1024
            )

            if not chunk_623dl5:
                break

            digest_623dl5.update(
                chunk_623dl5
            )

    return digest_623dl5.hexdigest()


hash_rows_623dl5 = []

for path_623dl5 in [
    rule_path_623dl5,
    candidate_path_623dl5,
    arrays_path_623dl5,
]:

    actual_hash_623dl5 = sha256_file_623dl5(
        path_623dl5
    )

    expected_hash_623dl5 = (
        RELOADED_MANIFEST_623DL5[
            "files"
        ][
            path_623dl5.name
        ][
            "sha256"
        ]
    )

    hash_rows_623dl5.append(
        {
            "file":
                path_623dl5.name,

            "hash_match":
                (
                    actual_hash_623dl5
                    == expected_hash_623dl5
                ),
        }
    )


HASH_CHECK_623DL5 = pd.DataFrame(
    hash_rows_623dl5
)

print("\n--- SHA-256 verification ---")

display(
    HASH_CHECK_623DL5
)

if not HASH_CHECK_623DL5[
    "hash_match"
].all():

    raise RuntimeError(
        "Frozen artifact hash verification failed."
    )


# ------------------------------------------------------------
# 5. Decision consistency
# ------------------------------------------------------------

rule_signal_623dl5 = (
    RELOADED_RULE_623DL5[
        "reliability_signal"
    ]
)

rule_threshold_623dl5 = float(
    RELOADED_RULE_623DL5[
        "reliability_threshold"
    ]
)

rule_target_623dl5 = float(
    RELOADED_RULE_623DL5[
        "target_clean_development_coverage"
    ]
)


if rule_signal_623dl5 != "uncertainty_only":
    raise RuntimeError(
        "Reloaded reliability signal is incorrect."
    )


if not np.isclose(
    rule_target_623dl5,
    0.75,
    atol=1e-15,
    rtol=0.0,
):
    raise RuntimeError(
        "Reloaded target coverage is incorrect."
    )


if not np.isclose(
    rule_threshold_623dl5,
    RELOADED_THRESHOLD_623DL5,
    atol=0.0,
    rtol=0.0,
):
    raise RuntimeError(
        "Threshold mismatch between JSON and NPZ."
    )


# ------------------------------------------------------------
# 6. Candidate-table threshold consistency
# ------------------------------------------------------------

row75_623dl5 = (
    RELOADED_THRESHOLD_TABLE_623DL5.loc[
        np.isclose(
            RELOADED_THRESHOLD_TABLE_623DL5[
                "target_clean_coverage"
            ].to_numpy(
                dtype=np.float64
            ),
            0.75,
            atol=1e-12,
            rtol=0.0,
        )
    ]
)

if len(row75_623dl5) != 1:
    raise RuntimeError(
        "Could not identify exactly one frozen 75% row."
    )

table_threshold_623dl5 = float(
    row75_623dl5.iloc[0][
        "threshold"
    ]
)

if not np.isclose(
    table_threshold_623dl5,
    rule_threshold_623dl5,
    atol=1e-15,
    rtol=0.0,
):
    raise RuntimeError(
        "CSV threshold does not match frozen threshold."
    )


# ------------------------------------------------------------
# 7. Reloaded array integrity
# ------------------------------------------------------------

if (
    RELOADED_DEV_RELIABILITY_623DL5.shape
    != (14241,)
):
    raise RuntimeError(
        "Unexpected reloaded DEV reliability shape."
    )

if (
    RELOADED_VAL_RELIABILITY_623DL5.shape
    != (3490,)
):
    raise RuntimeError(
        "Unexpected reloaded VAL reliability shape."
    )


if not (
    np.isfinite(
        RELOADED_DEV_RELIABILITY_623DL5
    ).all()
    and
    np.isfinite(
        RELOADED_VAL_RELIABILITY_623DL5
    ).all()
):
    raise RuntimeError(
        "Non-finite value in frozen reliability arrays."
    )


# ------------------------------------------------------------
# 8. Independently reconstruct masks
# ------------------------------------------------------------

dev_mask_reconstructed_623dl5 = (
    RELOADED_DEV_RELIABILITY_623DL5
    <= rule_threshold_623dl5
)

val_mask_reconstructed_623dl5 = (
    RELOADED_VAL_RELIABILITY_623DL5
    <= rule_threshold_623dl5
)


dev_mask_match_623dl5 = bool(
    np.array_equal(
        dev_mask_reconstructed_623dl5,
        RELOADED_DEV_ACCEPT_MASK_623DL5,
    )
)

val_mask_match_623dl5 = bool(
    np.array_equal(
        val_mask_reconstructed_623dl5,
        RELOADED_VAL_ACCEPT_MASK_623DL5,
    )
)

if not dev_mask_match_623dl5:
    raise RuntimeError(
        "Reloaded DEV accept mask failed reproduction."
    )

if not val_mask_match_623dl5:
    raise RuntimeError(
        "Reloaded VAL accept mask failed reproduction."
    )


# ------------------------------------------------------------
# 9. Compare against current in-memory freeze
# ------------------------------------------------------------

memory_comparison_available_623dl5 = all(
    name in globals()
    for name in [
        "FROZEN_DEV_RELIABILITY_623DL3",
        "FROZEN_VAL_RELIABILITY_623DL3",
        "FROZEN_RELIABILITY_THRESHOLD_623DL3",
    ]
)

if memory_comparison_available_623dl5:

    dev_memory_error_623dl5 = float(
        np.max(
            np.abs(
                RELOADED_DEV_RELIABILITY_623DL5
                - FROZEN_DEV_RELIABILITY_623DL3
            )
        )
    )

    val_memory_error_623dl5 = float(
        np.max(
            np.abs(
                RELOADED_VAL_RELIABILITY_623DL5
                - FROZEN_VAL_RELIABILITY_623DL3
            )
        )
    )

    threshold_memory_error_623dl5 = abs(
        rule_threshold_623dl5
        - float(
            FROZEN_RELIABILITY_THRESHOLD_623DL3
        )
    )

else:
    dev_memory_error_623dl5 = np.nan
    val_memory_error_623dl5 = np.nan
    threshold_memory_error_623dl5 = np.nan


# ------------------------------------------------------------
# 10. Final compact summary
# ------------------------------------------------------------

print("\n============================================")
print("STEP 6.23D-L5 FREEZE VERIFICATION SUMMARY")
print("============================================")

print(
    "Reloaded reliability signal :",
    rule_signal_623dl5
)

print(
    "Reloaded target coverage    :",
    rule_target_623dl5
)

print(
    "Reloaded exact threshold    :",
    repr(
        rule_threshold_623dl5
    )
)

print(
    "DEV reliability shape       :",
    RELOADED_DEV_RELIABILITY_623DL5.shape
)

print(
    "VAL reliability shape       :",
    RELOADED_VAL_RELIABILITY_623DL5.shape
)

print(
    "DEV accepted                :",
    int(
        RELOADED_DEV_ACCEPT_MASK_623DL5.sum()
    ),
    "/",
    len(
        RELOADED_DEV_ACCEPT_MASK_623DL5
    )
)

print(
    "VAL accepted                :",
    int(
        RELOADED_VAL_ACCEPT_MASK_623DL5.sum()
    ),
    "/",
    len(
        RELOADED_VAL_ACCEPT_MASK_623DL5
    )
)

print(
    "DEV mask reproduced         :",
    dev_mask_match_623dl5
)

print(
    "VAL mask reproduced         :",
    val_mask_match_623dl5
)

print(
    "All stored hashes valid     :",
    bool(
        HASH_CHECK_623DL5[
            "hash_match"
        ].all()
    )
)

if memory_comparison_available_623dl5:

    print(
        "DEV max error vs memory     :",
        dev_memory_error_623dl5
    )

    print(
        "VAL max error vs memory     :",
        val_memory_error_623dl5
    )

    print(
        "Threshold error vs memory   :",
        threshold_memory_error_623dl5
    )

print()
print("Reliability signal frozen   : True")
print("Threshold frozen            : True")
print("Abstention rule frozen      : True")
print("Official test used          : False")
print("Model inference executed    : False")
print("Corruptions rerun           : False")
print("Files modified              : False")
print("=== END ===")

In [ ]:
# ============================================================
# Step 6.24A
# Official FD001 test readiness / namespace inspection
#
# Inspection only:
# - no model inference
# - no MC dropout
# - no corruption
# - no threshold tuning
# - no files written
# ============================================================

import numpy as np
import pandas as pd

print(
    "=== Step 6.24A — "
    "Official FD001 test readiness inspection ==="
)


# ------------------------------------------------------------
# 1. Confirm frozen decision remains available
# ------------------------------------------------------------

frozen_required_624a = [
    "RELOADED_RULE_623DL5",
    "RELOADED_THRESHOLD_623DL5",
    "RELOADED_TARGET_COVERAGE_623DL5",
]

print("\n--- Frozen decision objects ---")

for name_624a in frozen_required_624a:
    print(
        f"{name_624a:40s}:",
        name_624a in globals()
    )

missing_frozen_624a = [
    name
    for name in frozen_required_624a
    if name not in globals()
]

if missing_frozen_624a:
    raise RuntimeError(
        "Frozen reliability decision is not available: "
        f"{missing_frozen_624a}"
    )


print(
    "\nFrozen signal    :",
    RELOADED_RULE_623DL5[
        "reliability_signal"
    ]
)

print(
    "Frozen threshold :",
    repr(
        RELOADED_THRESHOLD_623DL5
    )
)

print(
    "Target coverage  :",
    RELOADED_TARGET_COVERAGE_623DL5
)


# ------------------------------------------------------------
# 2. Search namespace for likely official-test objects
# ------------------------------------------------------------

keywords_624a = [
    "test",
    "fd001",
    "rul",
]

exclude_tokens_624a = [
    "dev",
    "val",
    "train",
    "threshold",
    "candidate",
    "scenario",
    "corrupt",
    "audit",
]


rows_624a = []

for name_624a, obj_624a in list(globals().items()):

    name_lower_624a = name_624a.lower()

    if not any(
        token in name_lower_624a
        for token in keywords_624a
    ):
        continue

    # Keep potentially useful objects even when their
    # names also contain other terms, but flag them.
    excluded_hint_624a = any(
        token in name_lower_624a
        for token in exclude_tokens_624a
    )

    try:
        obj_type_624a = type(
            obj_624a
        ).__name__

        if isinstance(
            obj_624a,
            pd.DataFrame
        ):
            shape_624a = obj_624a.shape
            columns_624a = list(
                obj_624a.columns[:10]
            )

        elif isinstance(
            obj_624a,
            pd.Series
        ):
            shape_624a = obj_624a.shape
            columns_624a = [
                str(
                    obj_624a.name
                )
            ]

        elif isinstance(
            obj_624a,
            np.ndarray
        ):
            shape_624a = obj_624a.shape
            columns_624a = None

        elif isinstance(
            obj_624a,
            (list, tuple)
        ):
            shape_624a = (
                len(
                    obj_624a
                ),
            )
            columns_624a = None

        else:
            try:
                shape_624a = obj_624a.shape
            except Exception:
                shape_624a = None

            columns_624a = None

        rows_624a.append(
            {
                "object_name":
                    name_624a,

                "type":
                    obj_type_624a,

                "shape":
                    str(
                        shape_624a
                    ),

                "first_columns":
                    (
                        str(
                            columns_624a
                        )
                        if columns_624a
                        is not None
                        else ""
                    ),

                "mixed_name_hint":
                    excluded_hint_624a,
            }
        )

    except Exception:
        pass


TEST_NAMESPACE_OBJECTS_624A = (
    pd.DataFrame(
        rows_624a
    )
    .sort_values(
        [
            "mixed_name_hint",
            "object_name",
        ],
        ascending=[
            True,
            True,
        ],
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n--- Likely official-test related objects ---"
)

if len(
    TEST_NAMESPACE_OBJECTS_624A
) == 0:

    print(
        "No obvious test-related objects found."
    )

else:
    display(
        TEST_NAMESPACE_OBJECTS_624A
    )


# ------------------------------------------------------------
# 3. Specifically inspect likely model-input arrays
# ------------------------------------------------------------

model_input_rows_624a = []

for name_624a, obj_624a in list(
    globals().items()
):

    if not isinstance(
        obj_624a,
        np.ndarray
    ):
        continue

    if obj_624a.ndim not in [
        2,
        3,
    ]:
        continue

    name_lower_624a = name_624a.lower()

    if "test" not in name_lower_624a:
        continue

    finite_624a = bool(
        np.isfinite(
            obj_624a
        ).all()
    )

    model_input_rows_624a.append(
        {
            "object_name":
                name_624a,

            "shape":
                str(
                    obj_624a.shape
                ),

            "dtype":
                str(
                    obj_624a.dtype
                ),

            "finite":
                finite_624a,

            "minimum":
                float(
                    np.nanmin(
                        obj_624a
                    )
                ),

            "maximum":
                float(
                    np.nanmax(
                        obj_624a
                    )
                ),
        }
    )


TEST_ARRAY_CANDIDATES_624A = (
    pd.DataFrame(
        model_input_rows_624a
    )
)


print(
    "\n--- Candidate TEST model-input arrays ---"
)

if len(
    TEST_ARRAY_CANDIDATES_624A
) == 0:

    print(
        "No ndarray with 'test' in its name found."
    )

else:
    display(
        TEST_ARRAY_CANDIDATES_624A
    )


# ------------------------------------------------------------
# 4. Locate likely test metadata tables
# ------------------------------------------------------------

metadata_rows_624a = []

for name_624a, obj_624a in list(
    globals().items()
):

    if not isinstance(
        obj_624a,
        pd.DataFrame
    ):
        continue

    name_lower_624a = name_624a.lower()

    if "test" not in name_lower_624a:
        continue

    cols_lower_624a = {
        str(c).lower()
        for c in obj_624a.columns
    }

    has_unit_624a = (
        "unit_id"
        in cols_lower_624a
    )

    has_cycle_624a = (
        "cycle"
        in cols_lower_624a
    )

    has_rul_624a = any(
        "rul"
        in str(c).lower()
        for c in obj_624a.columns
    )

    metadata_rows_624a.append(
        {
            "object_name":
                name_624a,

            "rows":
                len(
                    obj_624a
                ),

            "columns":
                len(
                    obj_624a.columns
                ),

            "has_unit_id":
                has_unit_624a,

            "has_cycle":
                has_cycle_624a,

            "has_rul_column":
                has_rul_624a,

            "first_columns":
                str(
                    list(
                        obj_624a.columns[:12]
                    )
                ),
        }
    )


TEST_DATAFRAME_CANDIDATES_624A = (
    pd.DataFrame(
        metadata_rows_624a
    )
)


print(
    "\n--- Candidate TEST DataFrames ---"
)

if len(
    TEST_DATAFRAME_CANDIDATES_624A
) == 0:

    print(
        "No DataFrame with 'test' in its name found."
    )

else:
    display(
        TEST_DATAFRAME_CANDIDATES_624A
    )


# ------------------------------------------------------------
# 5. Search specifically for official RUL label-like objects
# ------------------------------------------------------------

rul_rows_624a = []

for name_624a, obj_624a in list(
    globals().items()
):

    name_lower_624a = name_624a.lower()

    if "rul" not in name_lower_624a:
        continue

    if not (
        "test" in name_lower_624a
        or "fd001" in name_lower_624a
    ):
        continue

    try:

        if isinstance(
            obj_624a,
            pd.DataFrame
        ):
            length_624a = len(
                obj_624a
            )

            shape_624a = (
                obj_624a.shape
            )

        elif isinstance(
            obj_624a,
            pd.Series
        ):
            length_624a = len(
                obj_624a
            )

            shape_624a = (
                obj_624a.shape
            )

        elif isinstance(
            obj_624a,
            np.ndarray
        ):
            length_624a = (
                obj_624a.size
            )

            shape_624a = (
                obj_624a.shape
            )

        else:
            continue

        rul_rows_624a.append(
            {
                "object_name":
                    name_624a,

                "type":
                    type(
                        obj_624a
                    ).__name__,

                "shape":
                    str(
                        shape_624a
                    ),

                "length":
                    length_624a,
            }
        )

    except Exception:
        pass


TEST_RUL_CANDIDATES_624A = (
    pd.DataFrame(
        rul_rows_624a
    )
)


print(
    "\n--- Candidate official TEST RUL objects ---"
)

if len(
    TEST_RUL_CANDIDATES_624A
) == 0:

    print(
        "No obvious test-RUL object found."
    )

else:
    display(
        TEST_RUL_CANDIDATES_624A
    )


# ------------------------------------------------------------
# 6. Frozen-model/helper availability
# ------------------------------------------------------------

helper_names_624a = [
    "frozen_gru",
    "predict_gru_batched",
    "mc_dropout_predictions",
    "mc_uncertainty_623cb2",
    "build_sequences_623cb2",
    "scaler",
    "PRIMARY_INPUT_COLUMNS",
]

print(
    "\n--- Frozen model / helper availability ---"
)

for name_624a in helper_names_624a:
    print(
        f"{name_624a:32s}:",
        name_624a in globals()
    )


# ------------------------------------------------------------
# 7. Integrity statement
# ------------------------------------------------------------

print(
    "\n============================================"
)

print(
    "STEP 6.24A READINESS SUMMARY"
)

print(
    "============================================"
)

print(
    "Frozen reliability signal :",
    RELOADED_RULE_623DL5[
        "reliability_signal"
    ]
)

print(
    "Frozen threshold          :",
    repr(
        RELOADED_THRESHOLD_623DL5
    )
)

print(
    "Candidate namespace rows  :",
    len(
        TEST_NAMESPACE_OBJECTS_624A
    )
)

print(
    "Candidate TEST arrays     :",
    len(
        TEST_ARRAY_CANDIDATES_624A
    )
)

print(
    "Candidate TEST DataFrames :",
    len(
        TEST_DATAFRAME_CANDIDATES_624A
    )
)

print(
    "Candidate TEST RUL objects:",
    len(
        TEST_RUL_CANDIDATES_624A
    )
)

print()
print(
    "Model inference executed  : False"
)

print(
    "MC dropout executed       : False"
)

print(
    "Corruption executed       : False"
)

print(
    "Threshold modified        : False"
)

print(
    "Frozen artifacts modified : False"
)

print(
    "Official test evaluated   : False"
)

print(
    "Files written             : False"
)

print(
    "=== END ==="
)

In [ ]:
# ============================================================
# Step 6.24B
# Official FD001 source-file verification +
# preprocessing provenance inspection
#
# READ / INSPECT ONLY
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

print(
    "=== Step 6.24B — Official FD001 source and "
    "preprocessing provenance inspection ==="
)


# ------------------------------------------------------------
# 1. Locate project root safely
# ------------------------------------------------------------

cwd_624b = Path.cwd()

candidate_roots_624b = [
    cwd_624b,
    cwd_624b.parent,
    cwd_624b.parent.parent,
]

PROJECT_ROOT_624B = None

for root_624b in candidate_roots_624b:
    if (
        root_624b
        / "02_dataset"
        / "raw"
        / "CMAPSSData"
    ).exists():
        PROJECT_ROOT_624B = root_624b
        break

if PROJECT_ROOT_624B is None:
    raise FileNotFoundError(
        "Could not locate project root containing "
        "02_dataset/raw/CMAPSSData."
    )


DATA_DIR_624B = (
    PROJECT_ROOT_624B
    / "02_dataset"
    / "raw"
    / "CMAPSSData"
)

TEST_PATH_624B = (
    DATA_DIR_624B
    / "test_FD001.txt"
)

RUL_PATH_624B = (
    DATA_DIR_624B
    / "RUL_FD001.txt"
)


print("\n--- Official source paths ---")
print("Project root :", PROJECT_ROOT_624B)
print("Dataset dir  :", DATA_DIR_624B)
print("TEST exists  :", TEST_PATH_624B.exists())
print("RUL exists   :", RUL_PATH_624B.exists())

if not TEST_PATH_624B.exists():
    raise FileNotFoundError(TEST_PATH_624B)

if not RUL_PATH_624B.exists():
    raise FileNotFoundError(RUL_PATH_624B)


# ------------------------------------------------------------
# 2. Load untouched NASA FD001 test files
# ------------------------------------------------------------

column_names_624b = (
    ["unit_id", "cycle"]
    + [
        f"op_setting_{i}"
        for i in range(1, 4)
    ]
    + [
        f"sensor_{i}"
        for i in range(1, 22)
    ]
)

if len(column_names_624b) != 26:
    raise RuntimeError(
        "Unexpected C-MAPSS column count."
    )


OFFICIAL_TEST_RAW_624B = pd.read_csv(
    TEST_PATH_624B,
    sep=r"\s+",
    header=None,
    names=column_names_624b,
)

OFFICIAL_TEST_FINAL_RUL_624B = pd.read_csv(
    RUL_PATH_624B,
    sep=r"\s+",
    header=None,
    names=["final_RUL"],
)


print("\n--- Raw official FD001 structure ---")
print(
    "Test shape      :",
    OFFICIAL_TEST_RAW_624B.shape
)
print(
    "RUL shape       :",
    OFFICIAL_TEST_FINAL_RUL_624B.shape
)
print(
    "Test engines    :",
    OFFICIAL_TEST_RAW_624B[
        "unit_id"
    ].nunique()
)
print(
    "Missing values  :",
    int(
        OFFICIAL_TEST_RAW_624B
        .isna()
        .sum()
        .sum()
    )
)
print(
    "Duplicate keys  :",
    int(
        OFFICIAL_TEST_RAW_624B
        .duplicated(
            ["unit_id", "cycle"]
        )
        .sum()
    )
)


# ------------------------------------------------------------
# 3. Verify official engine/RUL alignment
# ------------------------------------------------------------

test_units_624b = np.sort(
    OFFICIAL_TEST_RAW_624B[
        "unit_id"
    ].unique()
)

expected_units_624b = np.arange(
    1,
    len(test_units_624b) + 1
)

unit_sequence_ok_624b = (
    np.array_equal(
        test_units_624b,
        expected_units_624b
    )
)

rul_count_ok_624b = (
    len(
        OFFICIAL_TEST_FINAL_RUL_624B
    )
    ==
    len(
        test_units_624b
    )
)

print(
    "\nUnit IDs consecutive from 1 :",
    unit_sequence_ok_624b
)

print(
    "One final RUL per engine     :",
    rul_count_ok_624b
)


if not unit_sequence_ok_624b:
    raise RuntimeError(
        "Unexpected official test engine IDs."
    )

if not rul_count_ok_624b:
    raise RuntimeError(
        "Official test/RUL engine-count mismatch."
    )


# ------------------------------------------------------------
# 4. Check per-engine cycle structure
# ------------------------------------------------------------

cycle_check_rows_624b = []

for unit_624b, grp_624b in (
    OFFICIAL_TEST_RAW_624B
    .groupby(
        "unit_id",
        sort=True
    )
):

    cycles_624b = (
        grp_624b["cycle"]
        .to_numpy(
            dtype=np.int64
        )
    )

    expected_cycles_624b = np.arange(
        1,
        len(cycles_624b) + 1
    )

    cycle_check_rows_624b.append(
        {
            "unit_id":
                int(unit_624b),

            "rows":
                len(cycles_624b),

            "first_cycle":
                int(
                    cycles_624b[0]
                ),

            "last_cycle":
                int(
                    cycles_624b[-1]
                ),

            "contiguous":
                bool(
                    np.array_equal(
                        cycles_624b,
                        expected_cycles_624b
                    )
                ),
        }
    )


OFFICIAL_TEST_CYCLE_CHECK_624B = (
    pd.DataFrame(
        cycle_check_rows_624b
    )
)

print(
    "\nAll test trajectories contiguous :",
    bool(
        OFFICIAL_TEST_CYCLE_CHECK_624B[
            "contiguous"
        ].all()
    )
)

display(
    OFFICIAL_TEST_CYCLE_CHECK_624B.head(
        10
    )
)


# ------------------------------------------------------------
# 5. Build engine-level endpoint audit ONLY
#
# final_RUL is the official RUL at each engine's
# final observed test cycle.
# ------------------------------------------------------------

last_cycles_624b = (
    OFFICIAL_TEST_RAW_624B
    .groupby(
        "unit_id"
    )["cycle"]
    .max()
    .sort_index()
    .to_numpy(
        dtype=np.int64
    )
)

final_rul_624b = (
    OFFICIAL_TEST_FINAL_RUL_624B[
        "final_RUL"
    ]
    .to_numpy(
        dtype=np.float64
    )
)

OFFICIAL_TEST_ENDPOINT_AUDIT_624B = (
    pd.DataFrame(
        {
            "unit_id":
                test_units_624b,

            "last_observed_cycle":
                last_cycles_624b,

            "official_final_RUL":
                final_rul_624b,

            "implied_failure_cycle":
                (
                    last_cycles_624b
                    +
                    final_rul_624b
                ),
        }
    )
)

print(
    "\n--- First 10 official endpoint records ---"
)

display(
    OFFICIAL_TEST_ENDPOINT_AUDIT_624B.head(
        10
    )
)


# ------------------------------------------------------------
# 6. Inspect current frozen preprocessing objects
# ------------------------------------------------------------

print(
    "\n--- Frozen preprocessing structure ---"
)

print(
    "PRIMARY_INPUT_COLUMNS available :",
    "PRIMARY_INPUT_COLUMNS"
    in globals()
)

print(
    "scaler available                :",
    "scaler"
    in globals()
)

print(
    "val_raw_base_620gc available    :",
    "val_raw_base_620gc"
    in globals()
)

print(
    "X_val_clean available           :",
    "X_val_clean"
    in globals()
)


if "PRIMARY_INPUT_COLUMNS" in globals():

    print(
        "\nPrimary input count :",
        len(PRIMARY_INPUT_COLUMNS)
    )

    print(
        "Primary input columns:"
    )

    print(
        list(
            PRIMARY_INPUT_COLUMNS
        )
    )


if "scaler" in globals():

    print(
        "\nScaler type :",
        type(
            scaler
        ).__name__
    )

    if hasattr(
        scaler,
        "n_features_in_"
    ):
        print(
            "Scaler n_features_in_ :",
            scaler.n_features_in_
        )

    if hasattr(
        scaler,
        "feature_names_in_"
    ):
        print(
            "Scaler feature_names_in_:"
        )

        print(
            list(
                scaler.feature_names_in_
            )
        )


if "val_raw_base_620gc" in globals():

    print(
        "\nValidation raw-base shape :",
        val_raw_base_620gc.shape
    )

    print(
        "Validation raw-base columns:"
    )

    print(
        list(
            val_raw_base_620gc.columns
        )
    )


if "X_val_clean" in globals():

    print(
        "\nValidation sequence shape :",
        np.asarray(
            X_val_clean
        ).shape
    )


# ------------------------------------------------------------
# 7. Compare NASA raw columns with model input columns
# ------------------------------------------------------------

if "PRIMARY_INPUT_COLUMNS" in globals():

    raw_columns_set_624b = set(
        OFFICIAL_TEST_RAW_624B.columns
    )

    missing_primary_624b = [
        col
        for col in PRIMARY_INPUT_COLUMNS
        if col not in raw_columns_set_624b
    ]

    extra_nasa_columns_624b = [
        col
        for col in OFFICIAL_TEST_RAW_624B.columns
        if col not in PRIMARY_INPUT_COLUMNS
    ]

    print(
        "\n--- Raw/model feature compatibility ---"
    )

    print(
        "Primary model features missing "
        "from official raw TEST:",
        missing_primary_624b
    )

    print(
        "NASA raw columns not directly "
        "used as primary inputs:"
    )

    print(
        extra_nasa_columns_624b
    )


# ------------------------------------------------------------
# 8. Search notebook history for exact preprocessing provenance
# ------------------------------------------------------------

search_terms_624b = [
    "val_raw_base_620gc",
    "X_val_clean",
    "PRIMARY_INPUT_COLUMNS",
    "test_FD001",
    "RUL_FD001",
    "val_scaled",
    "build_sequences",
]


history_hits_624b = []

try:

    ip_624b = get_ipython()

    input_history_624b = (
        ip_624b.user_ns.get(
            "In",
            []
        )
    )

    for cell_no_624b, source_624b in enumerate(
        input_history_624b
    ):

        if not isinstance(
            source_624b,
            str
        ):
            continue

        matched_624b = [
            term
            for term in search_terms_624b
            if term.lower()
            in source_624b.lower()
        ]

        if matched_624b:

            source_lines_624b = (
                source_624b.splitlines()
            )

            preview_624b = "\n".join(
                source_lines_624b[:35]
            )

            history_hits_624b.append(
                {
                    "cell":
                        cell_no_624b,

                    "matched":
                        matched_624b,

                    "preview":
                        preview_624b,
                }
            )

except Exception as exc_624b:

    print(
        "\nNotebook-history search unavailable:",
        repr(
            exc_624b
        )
    )


print(
    "\n--- Relevant preprocessing-history cells ---"
)

print(
    "Matching cells:",
    [
        item["cell"]
        for item in history_hits_624b
    ]
)


for item_624b in history_hits_624b[-12:]:

    print(
        "\n"
        +
        "=" * 70
    )

    print(
        "CELL:",
        item_624b[
            "cell"
        ],
        "| matched:",
        item_624b[
            "matched"
        ]
    )

    print(
        "=" * 70
    )

    print(
        item_624b[
            "preview"
        ]
    )


# ------------------------------------------------------------
# 9. Integrity summary
# ------------------------------------------------------------

print(
    "\n=============================================="
)

print(
    "STEP 6.24B INTEGRITY SUMMARY"
)

print(
    "=============================================="
)

print(
    "Official TEST raw loaded      : True"
)

print(
    "Official TEST rows            :",
    len(
        OFFICIAL_TEST_RAW_624B
    )
)

print(
    "Official TEST engines         :",
    OFFICIAL_TEST_RAW_624B[
        "unit_id"
    ].nunique()
)

print(
    "Official final-RUL values     :",
    len(
        OFFICIAL_TEST_FINAL_RUL_624B
    )
)

print(
    "Engine/RUL alignment verified:",
    bool(
        unit_sequence_ok_624b
        and rul_count_ok_624b
    )
)

print(
    "Duplicate engine/cycle keys   :",
    int(
        OFFICIAL_TEST_RAW_624B
        .duplicated(
            ["unit_id", "cycle"]
        )
        .sum()
    )
)

print(
    "Missing raw values            :",
    int(
        OFFICIAL_TEST_RAW_624B
        .isna()
        .sum()
        .sum()
    )
)

print(
    "Model inference executed      : False"
)

print(
    "MC dropout executed           : False"
)

print(
    "Reliability threshold changed : False"
)

print(
    "Official TEST scored          : False"
)

print(
    "Frozen artifacts modified     : False"
)

print(
    "Files written                 : False"
)

print(
    "=== END ==="
)

In [ ]:
# ============================================================
# Step 6.24C
# Exact validation preprocessing / target recovery
# + official FD001 TEST sequence construction
#
# NO MODEL INFERENCE
# NO MC DROPOUT
# NO OFFICIAL-TEST SCORING
# NO FILE WRITING
# ============================================================

import numpy as np
import pandas as pd

print(
    "=== Step 6.24C — Exact preprocessing/target recovery "
    "and official TEST sequence construction ==="
)


# ------------------------------------------------------------
# 1. Required-object check
# ------------------------------------------------------------

required_624c = [
    "OFFICIAL_TEST_RAW_624B",
    "OFFICIAL_TEST_FINAL_RUL_624B",
    "PRIMARY_INPUT_COLUMNS",
    "scaler",
    "build_sequences_623cb2",
    "val_raw_base_620gc",
    "X_val_clean",
    "y_val_clean",
]

missing_624c = [
    name
    for name in required_624c
    if name not in globals()
]

print("\n--- Required objects ---")

for name in required_624c:
    print(
        f"{name:<35}:",
        name in globals()
    )

if missing_624c:
    raise RuntimeError(
        "Missing required objects: "
        + ", ".join(missing_624c)
    )


# ------------------------------------------------------------
# 2. Reproduce CLEAN validation sequences exactly
# ------------------------------------------------------------

X_val_rebuilt_624c = np.asarray(
    build_sequences_623cb2(
        val_raw_base_620gc.copy()
    ),
    dtype=np.float64,
)

X_val_reference_624c = np.asarray(
    X_val_clean,
    dtype=np.float64,
)

print(
    "\n--- Validation sequence reproduction ---"
)

print(
    "Stored shape  :",
    X_val_reference_624c.shape
)

print(
    "Rebuilt shape :",
    X_val_rebuilt_624c.shape
)

if (
    X_val_rebuilt_624c.shape
    != X_val_reference_624c.shape
):
    raise RuntimeError(
        "Validation sequence shape mismatch."
    )

val_sequence_max_error_624c = float(
    np.max(
        np.abs(
            X_val_rebuilt_624c
            - X_val_reference_624c
        )
    )
)

print(
    "Maximum absolute sequence error :",
    val_sequence_max_error_624c
)

val_sequence_exact_624c = bool(
    np.allclose(
        X_val_rebuilt_624c,
        X_val_reference_624c,
        rtol=0.0,
        atol=1e-10,
    )
)

print(
    "Validation preprocessing reproduced :",
    val_sequence_exact_624c
)

if not val_sequence_exact_624c:
    raise RuntimeError(
        "Frozen validation preprocessing was not "
        "reproduced exactly enough."
    )


# ------------------------------------------------------------
# 3. Recover exact training/validation target convention
#
# Build one target per 30-cycle window:
# target = target value at the FINAL cycle of each window.
# ------------------------------------------------------------

WINDOW_LENGTH_624C = int(
    X_val_reference_624c.shape[1]
)

target_candidates_624c = [
    col
    for col in [
        "RUL_raw",
        "RUL_cap_125",
        "RUL_cap_130",
    ]
    if col in val_raw_base_620gc.columns
]

if not target_candidates_624c:
    raise RuntimeError(
        "No known RUL target columns found "
        "in validation raw base."
    )


def endpoint_targets_624c(
    raw_df,
    target_column,
    window_length,
):
    values = []

    for _, grp in (
        raw_df
        .sort_values(
            ["unit_id", "cycle"]
        )
        .groupby(
            "unit_id",
            sort=True
        )
    ):
        target_arr = (
            grp[target_column]
            .to_numpy(
                dtype=np.float64
            )
        )

        if len(target_arr) < window_length:
            continue

        values.extend(
            target_arr[
                window_length - 1:
            ]
        )

    return np.asarray(
        values,
        dtype=np.float64,
    )


y_val_reference_624c = np.asarray(
    y_val_clean,
    dtype=np.float64,
).reshape(-1)


target_match_rows_624c = []

TARGET_ARRAYS_624C = {}

for target_col_624c in target_candidates_624c:

    candidate_y_624c = (
        endpoint_targets_624c(
            val_raw_base_620gc,
            target_col_624c,
            WINDOW_LENGTH_624C,
        )
    )

    TARGET_ARRAYS_624C[
        target_col_624c
    ] = candidate_y_624c

    if (
        candidate_y_624c.shape
        ==
        y_val_reference_624c.shape
    ):

        difference_624c = np.abs(
            candidate_y_624c
            - y_val_reference_624c
        )

        max_error_624c = float(
            np.max(
                difference_624c
            )
        )

        mean_error_624c = float(
            np.mean(
                difference_624c
            )
        )

        exact_624c = bool(
            np.allclose(
                candidate_y_624c,
                y_val_reference_624c,
                rtol=0.0,
                atol=1e-10,
            )
        )

    else:

        max_error_624c = np.inf
        mean_error_624c = np.inf
        exact_624c = False

    target_match_rows_624c.append(
        {
            "target":
                target_col_624c,

            "candidate_shape":
                candidate_y_624c.shape,

            "reference_shape":
                y_val_reference_624c.shape,

            "max_abs_error":
                max_error_624c,

            "mean_abs_error":
                mean_error_624c,

            "exact_match":
                exact_624c,
        }
    )


TARGET_MATCH_TABLE_624C = pd.DataFrame(
    target_match_rows_624c
)

print(
    "\n--- Validation target recovery ---"
)

display(
    TARGET_MATCH_TABLE_624C
)


exact_targets_624c = (
    TARGET_MATCH_TABLE_624C.loc[
        TARGET_MATCH_TABLE_624C[
            "exact_match"
        ],
        "target",
    ]
    .tolist()
)

if len(exact_targets_624c) != 1:
    raise RuntimeError(
        "Expected exactly one validation target "
        "convention to reproduce y_val_clean; "
        f"found {exact_targets_624c}."
    )

FROZEN_RUL_TARGET_624C = (
    exact_targets_624c[0]
)

print(
    "\nRecovered frozen RUL target :",
    FROZEN_RUL_TARGET_624C
)


# ------------------------------------------------------------
# 4. Construct true per-cycle RUL for official TEST
#
# NASA RUL_FD001 gives RUL at the FINAL observed cycle.
#
# Therefore for any earlier observed cycle:
#
# RUL_raw(cycle)
# =
# final_RUL
# +
# (last_observed_cycle - cycle)
# ------------------------------------------------------------

OFFICIAL_TEST_BASE_624C = (
    OFFICIAL_TEST_RAW_624B
    .copy()
)

last_cycle_map_624c = (
    OFFICIAL_TEST_BASE_624C
    .groupby(
        "unit_id"
    )["cycle"]
    .max()
    .to_dict()
)

official_rul_map_624c = dict(
    zip(
        np.arange(
            1,
            len(
                OFFICIAL_TEST_FINAL_RUL_624B
            ) + 1
        ),
        OFFICIAL_TEST_FINAL_RUL_624B[
            "final_RUL"
        ].to_numpy(
            dtype=np.float64
        ),
    )
)

OFFICIAL_TEST_BASE_624C[
    "RUL_raw"
] = (
    OFFICIAL_TEST_BASE_624C[
        "unit_id"
    ]
    .map(
        official_rul_map_624c
    )
    .astype(
        np.float64
    )
    +
    OFFICIAL_TEST_BASE_624C[
        "unit_id"
    ]
    .map(
        last_cycle_map_624c
    )
    .astype(
        np.float64
    )
    -
    OFFICIAL_TEST_BASE_624C[
        "cycle"
    ]
    .astype(
        np.float64
    )
)

OFFICIAL_TEST_BASE_624C[
    "RUL_cap_125"
] = np.minimum(
    OFFICIAL_TEST_BASE_624C[
        "RUL_raw"
    ],
    125.0,
)

OFFICIAL_TEST_BASE_624C[
    "RUL_cap_130"
] = np.minimum(
    OFFICIAL_TEST_BASE_624C[
        "RUL_raw"
    ],
    130.0,
)


# ------------------------------------------------------------
# 5. Match canonical validation raw-base column layout
# ------------------------------------------------------------

canonical_columns_624c = (
    [
        "unit_id",
        "cycle",
        "RUL_raw",
        "RUL_cap_125",
        "RUL_cap_130",
    ]
    +
    list(
        PRIMARY_INPUT_COLUMNS
    )
)

missing_columns_624c = [
    col
    for col in canonical_columns_624c
    if col not in OFFICIAL_TEST_BASE_624C.columns
]

if missing_columns_624c:
    raise RuntimeError(
        "Official TEST is missing canonical columns: "
        + str(
            missing_columns_624c
        )
    )

OFFICIAL_TEST_BASE_624C = (
    OFFICIAL_TEST_BASE_624C[
        canonical_columns_624c
    ]
    .copy()
)


print(
    "\n--- Canonical official TEST raw base ---"
)

print(
    "Shape :",
    OFFICIAL_TEST_BASE_624C.shape
)

print(
    "Columns:",
    list(
        OFFICIAL_TEST_BASE_624C.columns
    )
)

print(
    "All values finite :",
    bool(
        np.isfinite(
            OFFICIAL_TEST_BASE_624C[
                canonical_columns_624c
            ].to_numpy(
                dtype=np.float64
            )
        ).all()
    )
)


# ------------------------------------------------------------
# 6. Build official TEST sequences using EXACT frozen helper
# ------------------------------------------------------------

X_TEST_CLEAN_624C = np.asarray(
    build_sequences_623cb2(
        OFFICIAL_TEST_BASE_624C.copy()
    ),
    dtype=np.float64,
)

expected_windows_624c = int(
    sum(
        max(
            0,
            len(grp)
            - WINDOW_LENGTH_624C
            + 1
        )
        for _, grp in (
            OFFICIAL_TEST_BASE_624C
            .groupby(
                "unit_id",
                sort=True
            )
        )
    )
)

print(
    "\n--- Official TEST sequences ---"
)

print(
    "Window length       :",
    WINDOW_LENGTH_624C
)

print(
    "Feature count       :",
    len(
        PRIMARY_INPUT_COLUMNS
    )
)

print(
    "Expected windows    :",
    expected_windows_624c
)

print(
    "Constructed shape   :",
    X_TEST_CLEAN_624C.shape
)

print(
    "All sequence values finite :",
    bool(
        np.isfinite(
            X_TEST_CLEAN_624C
        ).all()
    )
)

if (
    X_TEST_CLEAN_624C.shape
    != (
        expected_windows_624c,
        WINDOW_LENGTH_624C,
        len(
            PRIMARY_INPUT_COLUMNS
        ),
    )
):
    raise RuntimeError(
        "Unexpected official TEST sequence shape."
    )


# ------------------------------------------------------------
# 7. Build exact official TEST window metadata + targets
# ------------------------------------------------------------

test_meta_rows_624c = []

for unit_624c, grp_624c in (
    OFFICIAL_TEST_BASE_624C
    .sort_values(
        ["unit_id", "cycle"]
    )
    .groupby(
        "unit_id",
        sort=True
    )
):

    grp_624c = (
        grp_624c
        .reset_index(
            drop=True
        )
    )

    for end_idx_624c in range(
        WINDOW_LENGTH_624C - 1,
        len(
            grp_624c
        ),
    ):

        row_624c = grp_624c.iloc[
            end_idx_624c
        ]

        test_meta_rows_624c.append(
            {
                "unit_id":
                    int(
                        unit_624c
                    ),

                "cycle":
                    int(
                        row_624c[
                            "cycle"
                        ]
                    ),

                "RUL_raw":
                    float(
                        row_624c[
                            "RUL_raw"
                        ]
                    ),

                "RUL_cap_125":
                    float(
                        row_624c[
                            "RUL_cap_125"
                        ]
                    ),

                "RUL_cap_130":
                    float(
                        row_624c[
                            "RUL_cap_130"
                        ]
                    ),
            }
        )


OFFICIAL_TEST_SEQ_META_624C = (
    pd.DataFrame(
        test_meta_rows_624c
    )
)

Y_TEST_CLEAN_624C = (
    OFFICIAL_TEST_SEQ_META_624C[
        FROZEN_RUL_TARGET_624C
    ]
    .to_numpy(
        dtype=np.float64
    )
)

print(
    "\n--- Official TEST target structure ---"
)

print(
    "Frozen target column :",
    FROZEN_RUL_TARGET_624C
)

print(
    "Metadata shape       :",
    OFFICIAL_TEST_SEQ_META_624C.shape
)

print(
    "Target shape         :",
    Y_TEST_CLEAN_624C.shape
)

print(
    "Target finite        :",
    bool(
        np.isfinite(
            Y_TEST_CLEAN_624C
        ).all()
    )
)


# ------------------------------------------------------------
# 8. Endpoint-label verification against NASA RUL file
# ------------------------------------------------------------

endpoint_rows_624c = (
    OFFICIAL_TEST_SEQ_META_624C
    .sort_values(
        ["unit_id", "cycle"]
    )
    .groupby(
        "unit_id",
        as_index=False
    )
    .tail(
        1
    )
    .sort_values(
        "unit_id"
    )
    .reset_index(
        drop=True
    )
)

official_final_raw_624c = (
    OFFICIAL_TEST_FINAL_RUL_624B[
        "final_RUL"
    ]
    .to_numpy(
        dtype=np.float64
    )
)

endpoint_raw_error_624c = float(
    np.max(
        np.abs(
            endpoint_rows_624c[
                "RUL_raw"
            ].to_numpy(
                dtype=np.float64
            )
            -
            official_final_raw_624c
        )
    )
)

print(
    "\n--- NASA endpoint-label verification ---"
)

print(
    "Endpoint engines :",
    len(
        endpoint_rows_624c
    )
)

print(
    "Maximum raw-RUL endpoint error :",
    endpoint_raw_error_624c
)

print(
    "Exact endpoint RUL reproduction :",
    bool(
        np.allclose(
            endpoint_rows_624c[
                "RUL_raw"
            ].to_numpy(
                dtype=np.float64
            ),
            official_final_raw_624c,
            rtol=0.0,
            atol=0.0,
        )
    )
)


# ------------------------------------------------------------
# 9. Compact preview
# ------------------------------------------------------------

print(
    "\n--- First five TEST-window metadata rows ---"
)

display(
    OFFICIAL_TEST_SEQ_META_624C.head()
)

print(
    "\n--- Last TEST window of first five engines ---"
)

display(
    endpoint_rows_624c.head()
)


# ------------------------------------------------------------
# 10. Integrity summary
# ------------------------------------------------------------

print(
    "\n=============================================="
)

print(
    "STEP 6.24C INTEGRITY SUMMARY"
)

print(
    "=============================================="
)

print(
    "Validation preprocessing exact :",
    val_sequence_exact_624c
)

print(
    "Recovered RUL target           :",
    FROZEN_RUL_TARGET_624C
)

print(
    "Official TEST raw rows         :",
    len(
        OFFICIAL_TEST_BASE_624C
    )
)

print(
    "Official TEST engines          :",
    OFFICIAL_TEST_BASE_624C[
        "unit_id"
    ].nunique()
)

print(
    "Official TEST windows          :",
    len(
        X_TEST_CLEAN_624C
    )
)

print(
    "Official TEST sequence shape   :",
    X_TEST_CLEAN_624C.shape
)

print(
    "Official TEST target shape     :",
    Y_TEST_CLEAN_624C.shape
)

print(
    "NASA endpoint RUL error        :",
    endpoint_raw_error_624c
)

print(
    "Model inference executed       : False"
)

print(
    "MC dropout executed            : False"
)

print(
    "Reliability decision changed   : False"
)

print(
    "Official TEST scored           : False"
)

print(
    "Files written                  : False"
)

print(
    "=== END ==="
)

In [ ]:
# ============================================================
# Step 6.24D0
# Final pre-inference evaluation / uncertainty-recipe lock
#
# NO MODEL INFERENCE
# NO MC DROPOUT
# NO TEST SCORING
# NO FILE WRITING
# ============================================================

import inspect
import numpy as np
import pandas as pd

print(
    "=== Step 6.24D0 — Final pre-inference recipe lock ==="
)


# ------------------------------------------------------------
# 1. Required objects
# ------------------------------------------------------------

required_624d0 = [
    "X_TEST_CLEAN_624C",
    "Y_TEST_CLEAN_624C",
    "OFFICIAL_TEST_SEQ_META_624C",
    "OFFICIAL_TEST_FINAL_RUL_624B",
    "FROZEN_RUL_TARGET_624C",
    "frozen_gru",
    "predict_gru_batched",
    "mc_dropout_predictions",
    "mc_uncertainty_623cb2",
    "phase_references_622h",
    "RELOADED_RULE_623DL5",
    "RELOADED_THRESHOLD_623DL5",
    "RELOADED_TARGET_COVERAGE_623DL5",
]

missing_624d0 = [
    name
    for name in required_624d0
    if name not in globals()
]

print("\n--- Required objects ---")

for name in required_624d0:
    print(
        f"{name:<38}:",
        name in globals()
    )

if missing_624d0:
    raise RuntimeError(
        "Missing required objects: "
        + ", ".join(missing_624d0)
    )


# ------------------------------------------------------------
# 2. Freeze conventional NASA endpoint positions
#
# These 100 positions correspond to the final available
# 30-cycle window of each official TEST engine.
# ------------------------------------------------------------

endpoint_table_624d0 = (
    OFFICIAL_TEST_SEQ_META_624C
    .reset_index()
    .rename(
        columns={
            "index": "window_position"
        }
    )
    .sort_values(
        ["unit_id", "cycle"]
    )
    .groupby(
        "unit_id",
        as_index=False
    )
    .tail(1)
    .sort_values(
        "unit_id"
    )
    .reset_index(
        drop=True
    )
)

OFFICIAL_TEST_ENDPOINT_POSITIONS_624D0 = (
    endpoint_table_624d0[
        "window_position"
    ]
    .to_numpy(
        dtype=np.int64
    )
)

print(
    "\n--- Conventional FD001 endpoint evaluation set ---"
)

print(
    "Endpoint windows :",
    len(
        OFFICIAL_TEST_ENDPOINT_POSITIONS_624D0
    )
)

print(
    "Endpoint engines :",
    endpoint_table_624d0[
        "unit_id"
    ].nunique()
)

print(
    "Unique positions :",
    len(
        np.unique(
            OFFICIAL_TEST_ENDPOINT_POSITIONS_624D0
        )
    )
)


# ------------------------------------------------------------
# 3. Verify endpoint target against official NASA labels
#    under the recovered frozen cap-125 convention
# ------------------------------------------------------------

official_final_raw_624d0 = (
    OFFICIAL_TEST_FINAL_RUL_624B[
        "final_RUL"
    ]
    .to_numpy(
        dtype=np.float64
    )
)

if FROZEN_RUL_TARGET_624C == "RUL_cap_125":

    official_endpoint_target_624d0 = (
        np.minimum(
            official_final_raw_624d0,
            125.0,
        )
    )

elif FROZEN_RUL_TARGET_624C == "RUL_cap_130":

    official_endpoint_target_624d0 = (
        np.minimum(
            official_final_raw_624d0,
            130.0,
        )
    )

elif FROZEN_RUL_TARGET_624C == "RUL_raw":

    official_endpoint_target_624d0 = (
        official_final_raw_624d0.copy()
    )

else:
    raise RuntimeError(
        "Unexpected frozen RUL target: "
        + str(
            FROZEN_RUL_TARGET_624C
        )
    )


recovered_endpoint_target_624d0 = (
    Y_TEST_CLEAN_624C[
        OFFICIAL_TEST_ENDPOINT_POSITIONS_624D0
    ]
)

endpoint_target_error_624d0 = float(
    np.max(
        np.abs(
            recovered_endpoint_target_624d0
            -
            official_endpoint_target_624d0
        )
    )
)

print(
    "\n--- Endpoint target verification ---"
)

print(
    "Frozen target convention :",
    FROZEN_RUL_TARGET_624C
)

print(
    "Endpoint target shape     :",
    recovered_endpoint_target_624d0.shape
)

print(
    "Maximum target error      :",
    endpoint_target_error_624d0
)

print(
    "Endpoint targets exact    :",
    bool(
        np.allclose(
            recovered_endpoint_target_624d0,
            official_endpoint_target_624d0,
            rtol=0.0,
            atol=0.0,
        )
    )
)

if endpoint_target_error_624d0 != 0.0:
    raise RuntimeError(
        "Official endpoint target reconstruction mismatch."
    )


# ------------------------------------------------------------
# 4. Reloaded frozen decision
# ------------------------------------------------------------

print(
    "\n--- Reloaded frozen reliability decision ---"
)

print(
    "Rule object type :",
    type(
        RELOADED_RULE_623DL5
    ).__name__
)

if isinstance(
    RELOADED_RULE_623DL5,
    dict,
):
    print(
        "Rule keys        :",
        list(
            RELOADED_RULE_623DL5.keys()
        )
    )

    for key_624d0, value_624d0 in (
        RELOADED_RULE_623DL5.items()
    ):
        if np.isscalar(
            value_624d0
        ) or isinstance(
            value_624d0,
            str,
        ):
            print(
                f"  {key_624d0}:",
                value_624d0
            )

print(
    "Frozen threshold  :",
    RELOADED_THRESHOLD_623DL5
)

print(
    "Target coverage    :",
    RELOADED_TARGET_COVERAGE_623DL5
)


# ------------------------------------------------------------
# 5. Frozen uncertainty-rule metadata
# ------------------------------------------------------------

uncertainty_metadata_names_624d0 = [
    "FROZEN_UNCERTAINTY_BIN_RULE_622IF",
    "FROZEN_UNCERTAINTY_RANK_RULE_622IF",
    "FROZEN_DEV_UNCERTAINTY_622IF",
    "FROZEN_VAL_UNCERTAINTY_622IF",
]

print(
    "\n--- Frozen uncertainty metadata ---"
)

for name_624d0 in (
    uncertainty_metadata_names_624d0
):

    exists_624d0 = (
        name_624d0 in globals()
    )

    print(
        f"{name_624d0:<38}:",
        exists_624d0,
        end=""
    )

    if exists_624d0:

        obj_624d0 = globals()[
            name_624d0
        ]

        if isinstance(
            obj_624d0,
            np.ndarray,
        ):
            print(
                " | shape=",
                obj_624d0.shape,
                sep=""
            )

        else:
            print(
                " |",
                obj_624d0
            )

    else:
        print()


# ------------------------------------------------------------
# 6. Inspect phase-reference structure without dumping
#    thousands of stored values
# ------------------------------------------------------------

def compact_object_description_624d0(
    obj
):
    if isinstance(
        obj,
        np.ndarray,
    ):

        arr = np.asarray(
            obj
        )

        text = (
            f"ndarray shape={arr.shape}, "
            f"dtype={arr.dtype}"
        )

        if (
            arr.size > 0
            and np.issubdtype(
                arr.dtype,
                np.number
            )
        ):
            finite_arr = arr[
                np.isfinite(
                    arr
                )
            ]

            if finite_arr.size:
                text += (
                    f", min={finite_arr.min():.9f}, "
                    f"max={finite_arr.max():.9f}"
                )

        return text

    if isinstance(
        obj,
        pd.Series,
    ):
        return (
            f"Series len={len(obj)}, "
            f"dtype={obj.dtype}"
        )

    if isinstance(
        obj,
        pd.DataFrame,
    ):
        return (
            f"DataFrame shape={obj.shape}, "
            f"columns={list(obj.columns)}"
        )

    if isinstance(
        obj,
        dict,
    ):
        return (
            "dict keys="
            + str(
                list(
                    obj.keys()
                )
            )
        )

    if isinstance(
        obj,
        (list, tuple),
    ):
        return (
            f"{type(obj).__name__} "
            f"len={len(obj)}"
        )

    return (
        f"{type(obj).__name__}: "
        f"{obj}"
    )


print(
    "\n--- phase_references_622h structure ---"
)

print(
    "Type :",
    type(
        phase_references_622h
    ).__name__
)

if isinstance(
    phase_references_622h,
    dict,
):

    print(
        "Keys :",
        list(
            phase_references_622h.keys()
        )
    )

    for key_624d0, value_624d0 in (
        phase_references_622h.items()
    ):

        print(
            f"{repr(key_624d0):<25} ->",
            compact_object_description_624d0(
                value_624d0
            )
        )

else:

    print(
        compact_object_description_624d0(
            phase_references_622h
        )
    )


# ------------------------------------------------------------
# 7. Inspect model / MC helper interfaces
# ------------------------------------------------------------

print(
    "\n--- Helper signatures ---"
)

for fn_name_624d0 in [
    "predict_gru_batched",
    "mc_dropout_predictions",
    "mc_uncertainty_623cb2",
]:

    fn_624d0 = globals()[
        fn_name_624d0
    ]

    print(
        f"{fn_name_624d0:<28}:",
        inspect.signature(
            fn_624d0
        )
    )


# ------------------------------------------------------------
# 8. Show only uncertainty-relevant source lines
# ------------------------------------------------------------

keywords_624d0 = [
    "seed",
    "manual_seed",
    "random",
    "dropout",
    "std",
    "ddof",
    "return",
    "predict",
    "sample",
    "pass",
]


for fn_name_624d0 in [
    "mc_dropout_predictions",
    "mc_uncertainty_623cb2",
]:

    print(
        f"\n--- Relevant source lines: "
        f"{fn_name_624d0} ---"
    )

    try:

        source_lines_624d0 = (
            inspect
            .getsource(
                globals()[
                    fn_name_624d0
                ]
            )
            .splitlines()
        )

        found_line_624d0 = False

        for line_no_624d0, line_624d0 in enumerate(
            source_lines_624d0,
            start=1,
        ):

            lower_624d0 = (
                line_624d0.lower()
            )

            if any(
                keyword_624d0
                in lower_624d0
                for keyword_624d0
                in keywords_624d0
            ):

                print(
                    f"{line_no_624d0:03d}: "
                    f"{line_624d0}"
                )

                found_line_624d0 = True

        if not found_line_624d0:
            print(
                "No keyword-matched lines."
            )

    except Exception as exc_624d0:

        print(
            "Source inspection unavailable:",
            repr(
                exc_624d0
            )
        )


# ------------------------------------------------------------
# 9. Final lock summary
# ------------------------------------------------------------

print(
    "\n=============================================="
)

print(
    "STEP 6.24D0 PRE-INFERENCE LOCK SUMMARY"
)

print(
    "=============================================="
)

print(
    "Official TEST windows          :",
    len(
        X_TEST_CLEAN_624C
    )
)

print(
    "Conventional endpoint windows  :",
    len(
        OFFICIAL_TEST_ENDPOINT_POSITIONS_624D0
    )
)

print(
    "Frozen target                  :",
    FROZEN_RUL_TARGET_624C
)

print(
    "Endpoint target error          :",
    endpoint_target_error_624d0
)

print(
    "Frozen threshold               :",
    RELOADED_THRESHOLD_623DL5
)

print(
    "Frozen target coverage         :",
    RELOADED_TARGET_COVERAGE_623DL5
)

print(
    "Model inference executed       : False"
)

print(
    "MC dropout executed            : False"
)

print(
    "Official TEST scored           : False"
)

print(
    "Frozen decision modified       : False"
)

print(
    "Files written                  : False"
)

print(
    "=== END ==="
)

In [ ]:
# ============================================================
# Step 6.24D0A
# Compact extraction of the remaining pre-inference recipe
#
# NO MODEL INFERENCE
# NO MC DROPOUT
# NO TEST SCORING
# NO FILE WRITING
# ============================================================

import inspect
import numpy as np
import pandas as pd

print(
    "=== Step 6.24D0A — Compact uncertainty-recipe extraction ==="
)


# ------------------------------------------------------------
# 1. Endpoint verification result
# ------------------------------------------------------------

print("\n--- Endpoint verification ---")

print(
    "Endpoint target error :",
    endpoint_target_error_624d0
)

print(
    "Endpoint targets exact:",
    bool(
        endpoint_target_error_624d0 == 0.0
    )
)


# ------------------------------------------------------------
# 2. Frozen reliability rule
# ------------------------------------------------------------

print("\n--- Frozen reliability rule ---")

print(
    "Rule:",
    RELOADED_RULE_623DL5
)

print(
    "Threshold:",
    RELOADED_THRESHOLD_623DL5
)

print(
    "Target coverage:",
    RELOADED_TARGET_COVERAGE_623DL5
)


# ------------------------------------------------------------
# 3. Frozen uncertainty metadata
# ------------------------------------------------------------

print("\n--- Frozen uncertainty metadata ---")

for name in [
    "FROZEN_UNCERTAINTY_BIN_RULE_622IF",
    "FROZEN_UNCERTAINTY_RANK_RULE_622IF",
]:

    print(
        name,
        "=",
        globals().get(
            name,
            "<missing>"
        )
    )


# ------------------------------------------------------------
# 4. phase_references_622h exact structure
# ------------------------------------------------------------

print("\n--- phase_references_622h ---")

print(
    "Type:",
    type(
        phase_references_622h
    ).__name__
)

if isinstance(
    phase_references_622h,
    dict,
):

    print(
        "Keys:",
        list(
            phase_references_622h.keys()
        )
    )

    for key, value in (
        phase_references_622h.items()
    ):

        print(
            "\nKEY:",
            repr(key)
        )

        print(
            "TYPE:",
            type(value).__name__
        )

        if isinstance(
            value,
            np.ndarray,
        ):

            print(
                "SHAPE:",
                value.shape
            )

            print(
                "DTYPE:",
                value.dtype
            )

            if value.size <= 20:
                print(
                    "VALUE:",
                    value
                )

            else:
                print(
                    "FIRST 5:",
                    value.reshape(-1)[:5]
                )

                print(
                    "LAST 5 :",
                    value.reshape(-1)[-5:]
                )

                if np.issubdtype(
                    value.dtype,
                    np.number
                ):
                    print(
                        "MIN:",
                        np.nanmin(value)
                    )

                    print(
                        "MAX:",
                        np.nanmax(value)
                    )

        elif isinstance(
            value,
            (list, tuple),
        ):

            print(
                "LENGTH:",
                len(value)
            )

            print(
                "VALUE:",
                value
                if len(value) <= 20
                else value[:5]
            )

        else:

            print(
                "VALUE:",
                value
            )

else:

    print(
        phase_references_622h
    )


# ------------------------------------------------------------
# 5. Helper signatures
# ------------------------------------------------------------

print("\n--- Helper signatures ---")

print(
    "mc_dropout_predictions:",
    inspect.signature(
        mc_dropout_predictions
    )
)

print(
    "mc_uncertainty_623cb2:",
    inspect.signature(
        mc_uncertainty_623cb2
    )
)


# ------------------------------------------------------------
# 6. Only uncertainty-relevant source fragments
# ------------------------------------------------------------

def print_relevant_source(
    fn,
    fn_name
):

    print(
        f"\n--- Relevant source: {fn_name} ---"
    )

    source = (
        inspect
        .getsource(fn)
        .splitlines()
    )

    keywords = [
        "seed",
        "manual_seed",
        "dropout",
        "std",
        "ddof",
        "mean",
        "predict",
        "return",
        "sample",
        "passes",
        "iterations",
    ]

    selected = set()

    for i, line in enumerate(
        source
    ):

        if any(
            word in line.lower()
            for word in keywords
        ):

            for j in range(
                max(0, i - 1),
                min(
                    len(source),
                    i + 2
                )
            ):
                selected.add(j)

    for i in sorted(
        selected
    ):
        print(
            f"{i + 1:03d}: "
            f"{source[i]}"
        )


print_relevant_source(
    mc_dropout_predictions,
    "mc_dropout_predictions"
)

print_relevant_source(
    mc_uncertainty_623cb2,
    "mc_uncertainty_623cb2"
)


# ------------------------------------------------------------
# 7. Integrity
# ------------------------------------------------------------

print(
    "\n========================================"
)

print(
    "STEP 6.24D0A COMPACT SUMMARY"
)

print(
    "========================================"
)

print(
    "Official TEST inference executed : False"
)

print(
    "MC dropout executed              : False"
)

print(
    "Official TEST scored             : False"
)

print(
    "Frozen decision modified         : False"
)

print(
    "Files written                    : False"
)

print(
    "=== END ==="
)

In [ ]:
print("=== Step 6.24D0B — Compact frozen uncertainty-reference check ===")

selected_key_624d0b = "det_pred_pct_b5"

ref_624d0b = phase_references_622h[selected_key_624d0b]

print("\nSelected reference key :", selected_key_624d0b)
print("Reference type         :", type(ref_624d0b).__name__)
print("Reference fields       :", list(ref_624d0b.keys()))

print("\nPhase source            :", ref_624d0b.get("phase_source"))
print("Requested bins          :", ref_624d0b.get("requested_bins"))
print("Actual bins             :", ref_624d0b.get("actual_bins"))

edges_624d0b = np.asarray(ref_624d0b["edges"])
references_624d0b = ref_624d0b["references"]

print("\nEdges shape             :", edges_624d0b.shape)
print("Edges                   :", edges_624d0b.tolist())

print("\nReference phases        :", sorted(references_624d0b.keys()))

for phase_id_624d0b in sorted(references_624d0b.keys()):
    arr_624d0b = np.asarray(references_624d0b[phase_id_624d0b])
    print(
        f"Phase {phase_id_624d0b}: "
        f"shape={arr_624d0b.shape}, "
        f"finite={np.isfinite(arr_624d0b).all()}, "
        f"min={arr_624d0b.min():.6f}, "
        f"max={arr_624d0b.max():.6f}"
    )

print("\nFrozen threshold        :", RELOADED_THRESHOLD_623DL5)
print("Frozen target coverage  :", RELOADED_TARGET_COVERAGE_623DL5)

print("\nOfficial TEST scored    : False")
print("Nothing was modified.")
print("=== END ===")

In [ ]:
import inspect
import re

print("=== Step 6.24D0C — MC-dropout recipe / reproducibility inspection ===")

# ------------------------------------------------------------
# 1. Main uncertainty helper
# ------------------------------------------------------------
print("\n--- mc_uncertainty_623cb2 ---")
print("Signature:", inspect.signature(mc_uncertainty_623cb2))

source_624d0c = inspect.getsource(mc_uncertainty_623cb2)
source_lines_624d0c = source_624d0c.splitlines()

keywords_624d0c = (
    "seed",
    "manual_seed",
    "random",
    "dropout",
    "mc_",
    "sample",
    "pass",
    "predict",
    "batch"
)

print("\nRelevant source lines:")

printed_624d0c = set()

for i_624d0c, line_624d0c in enumerate(
    source_lines_624d0c,
    start=1
):
    lower_624d0c = line_624d0c.lower()

    if any(
        keyword_624d0c in lower_624d0c
        for keyword_624d0c in keywords_624d0c
    ):
        # Include one surrounding line for context.
        for j_624d0c in range(
            max(1, i_624d0c - 1),
            min(len(source_lines_624d0c), i_624d0c + 1) + 1
        ):
            if j_624d0c not in printed_624d0c:
                print(
                    f"{j_624d0c:03d}: "
                    f"{source_lines_624d0c[j_624d0c - 1]}"
                )
                printed_624d0c.add(j_624d0c)

# ------------------------------------------------------------
# 2. Referenced global names
# ------------------------------------------------------------
print("\n--- Potential MC configuration globals ---")

global_names_624d0c = sorted(
    set(mc_uncertainty_623cb2.__code__.co_names)
)

interesting_names_624d0c = [
    name_624d0c
    for name_624d0c in global_names_624d0c
    if any(
        token_624d0c in name_624d0c.lower()
        for token_624d0c in [
            "seed",
            "mc",
            "drop",
            "sample",
            "pass",
            "batch"
        ]
    )
]

if not interesting_names_624d0c:
    print("No obvious MC configuration globals referenced.")
else:
    for name_624d0c in interesting_names_624d0c:
        if name_624d0c in globals():
            value_624d0c = globals()[name_624d0c]

            if isinstance(
                value_624d0c,
                (str, int, float, bool, type(None))
            ):
                print(
                    f"{name_624d0c} = "
                    f"{value_624d0c!r}"
                )
            else:
                print(
                    f"{name_624d0c}: "
                    f"type={type(value_624d0c).__name__}"
                )

# ------------------------------------------------------------
# 3. Lower-level MC helper
# ------------------------------------------------------------
print("\n--- mc_dropout_predictions ---")

if "mc_dropout_predictions" in globals():
    print(
        "Signature:",
        inspect.signature(mc_dropout_predictions)
    )

    mc_source_624d0c = inspect.getsource(
        mc_dropout_predictions
    )

    mc_lines_624d0c = mc_source_624d0c.splitlines()

    for i_624d0c, line_624d0c in enumerate(
        mc_lines_624d0c,
        start=1
    ):
        if any(
            keyword_624d0c in line_624d0c.lower()
            for keyword_624d0c in keywords_624d0c
        ):
            print(
                f"{i_624d0c:03d}: "
                f"{line_624d0c}"
            )
else:
    print(
        "mc_dropout_predictions not found "
        "as a standalone global object."
    )

print("\nOfficial TEST inference executed : False")
print("Official TEST scored             : False")
print("Frozen decision modified         : False")
print("Files written                    : False")
print("Nothing was modified.")
print("=== END ===")

In [ ]:
import inspect

print(
    "=== Step 6.24D0D — "
    "Frozen uncertainty transformation verification ==="
)

source_624d0d = inspect.getsource(
    mc_uncertainty_623cb2
)

lines_624d0d = source_624d0d.splitlines()

keywords_624d0d = [
    "adjusted",
    "uncertainty",
    "phase",
    "reference",
    "edge",
    "digitize",
    "percentile",
    "midrank",
    "det_pred"
]

print(
    "\n--- Relevant mc_uncertainty_623cb2 "
    "source lines ---"
)

printed_624d0d = set()

for i_624d0d, line_624d0d in enumerate(
    lines_624d0d,
    start=1
):
    lower_624d0d = line_624d0d.lower()

    if any(
        key_624d0d in lower_624d0d
        for key_624d0d in keywords_624d0d
    ):
        # Show two surrounding lines for context.
        for j_624d0d in range(
            max(1, i_624d0d - 2),
            min(
                len(lines_624d0d),
                i_624d0d + 2
            ) + 1
        ):
            if j_624d0d not in printed_624d0d:
                print(
                    f"{j_624d0d:03d}: "
                    f"{lines_624d0d[j_624d0d - 1]}"
                )
                printed_624d0d.add(j_624d0d)

print("\n--- Frozen reference availability ---")

print(
    "Selected candidate:",
    "det_pred_pct_b5"
)

print(
    "phase_references_622h available:",
    "phase_references_622h" in globals()
)

if "phase_references_622h" in globals():
    print(
        "Selected reference present:",
        "det_pred_pct_b5"
        in phase_references_622h
    )

    if (
        "det_pred_pct_b5"
        in phase_references_622h
    ):
        ref_624d0d = (
            phase_references_622h[
                "det_pred_pct_b5"
            ]
        )

        print(
            "Reference fields:",
            list(ref_624d0d.keys())
        )

        print(
            "Phase source:",
            ref_624d0d.get(
                "phase_source"
            )
        )

        print(
            "Actual bins:",
            ref_624d0d.get(
                "actual_bins"
            )
        )

print(
    "\nFrozen threshold:",
    RELOADED_THRESHOLD_623DL5
)

print(
    "Target DEV coverage:",
    RELOADED_TARGET_COVERAGE_623DL5
)

print("\nOfficial TEST inference executed : False")
print("Official TEST scored             : False")
print("Frozen decision modified         : False")
print("Files written                    : False")
print("Nothing was modified.")
print("=== END ===")

In [ ]:
import inspect

print(
    "=== Step 6.24D0E — "
    "Phase-adjusted uncertainty helper verification ==="
)

required_name_624d0e = "phase_adjusted_uncertainty_623cb2"

print(
    "\nHelper available:",
    required_name_624d0e in globals()
)

if required_name_624d0e not in globals():
    raise RuntimeError(
        "phase_adjusted_uncertainty_623cb2 is unavailable."
    )

helper_624d0e = phase_adjusted_uncertainty_623cb2

print(
    "Signature:",
    inspect.signature(helper_624d0e)
)

source_624d0e = inspect.getsource(
    helper_624d0e
)

lines_624d0e = source_624d0e.splitlines()

print(
    "\n--- Complete helper source ---"
)

for i_624d0e, line_624d0e in enumerate(
    lines_624d0e,
    start=1
):
    print(
        f"{i_624d0e:03d}: "
        f"{line_624d0e}"
    )

print(
    "\n--- Referenced global/function names ---"
)

names_624d0e = list(
    helper_624d0e.__code__.co_names
)

for name_624d0e in names_624d0e:
    print(name_624d0e)

print(
    "\n--- Frozen-reference indicators ---"
)

reference_terms_624d0e = [
    "phase_references_622h",
    "det_pred_pct_b5",
    "midrank",
    "percentile",
    "reference",
    "edges",
    "digitize",
]

source_lower_624d0e = (
    source_624d0e.lower()
)

for term_624d0e in reference_terms_624d0e:
    print(
        f"{term_624d0e:25s}:",
        term_624d0e.lower()
        in source_lower_624d0e
        or term_624d0e
        in names_624d0e
    )

print(
    "\nphase_references_622h available:",
    "phase_references_622h" in globals()
)

if "phase_references_622h" in globals():
    print(
        "det_pred_pct_b5 present:",
        "det_pred_pct_b5"
        in phase_references_622h
    )

print(
    "\nOfficial TEST inference executed : False"
)

print(
    "Official TEST scored             : False"
)

print(
    "Frozen reliability decision changed:",
    False
)

print(
    "Files written:",
    False
)

print("Nothing was modified.")
print("=== END ===")

In [ ]:
import inspect
import re

print(
    "=== Step 6.24D0F — "
    "Compact leakage-safety verification ==="
)

helper_624d0f = phase_adjusted_uncertainty_623cb2

# ---------------------------------------------------------
# 1. Names actually referenced by the helper
# ---------------------------------------------------------

code_names_624d0f = list(
    helper_624d0f.__code__.co_names
)

print("\n--- Names referenced by helper ---")
for name in code_names_624d0f:
    print(name)


# ---------------------------------------------------------
# 2. Relevant source lines only
# ---------------------------------------------------------

source_624d0f = inspect.getsource(
    helper_624d0f
)

keywords_624d0f = [
    "phase",
    "reference",
    "midrank",
    "percentile",
    "rank",
    "edge",
    "digitize",
    "quantile",
    "searchsorted",
    "phase_references_622h",
    "det_pred_pct_b5",
]

print(
    "\n--- Relevant helper source lines ---"
)

for i, line in enumerate(
    source_624d0f.splitlines(),
    start=1
):
    if any(
        k.lower() in line.lower()
        for k in keywords_624d0f
    ):
        print(
            f"{i:03d}: {line}"
        )


# ---------------------------------------------------------
# 3. Referenced globals
# ---------------------------------------------------------

closure_624d0f = inspect.getclosurevars(
    helper_624d0f
)

print(
    "\n--- Referenced global objects ---"
)

for name, obj in closure_624d0f.globals.items():

    print(
        f"{name}: "
        f"{type(obj).__name__}"
    )

    if isinstance(obj, dict):
        print(
            "    keys:",
            list(obj.keys())
        )


# ---------------------------------------------------------
# 4. Specifically inspect frozen DEV reference
# ---------------------------------------------------------

print(
    "\n--- Frozen development reference ---"
)

print(
    "phase_references_622h available:",
    "phase_references_622h" in globals()
)

if "phase_references_622h" in globals():

    print(
        "det_pred_pct_b5 available:",
        "det_pred_pct_b5"
        in phase_references_622h
    )

    if (
        "det_pred_pct_b5"
        in phase_references_622h
    ):
        ref = phase_references_622h[
            "det_pred_pct_b5"
        ]

        print(
            "reference fields:",
            list(ref.keys())
        )

        print(
            "phase_source:",
            ref.get(
                "phase_source",
                None
            )
        )

        print(
            "requested_bins:",
            ref.get(
                "requested_bins",
                None
            )
        )

        print(
            "actual_bins:",
            ref.get(
                "actual_bins",
                None
            )
        )


# ---------------------------------------------------------
# 5. Identify user-defined helper functions called by it
# ---------------------------------------------------------

print(
    "\n--- Called user-defined helpers ---"
)

called_helpers_624d0f = []

for name in code_names_624d0f:

    obj = helper_624d0f.__globals__.get(
        name,
        None
    )

    if inspect.isfunction(obj):
        called_helpers_624d0f.append(
            name
        )

        print(
            name,
            inspect.signature(obj)
        )


print(
    "\nOfficial TEST scored:",
    False
)

print(
    "Frozen decision modified:",
    False
)

print(
    "Files written:",
    False
)

print(
    "=== END ==="
)

In [ ]:
import numpy as np

print(
    "=== Step 6.24D0G — "
    "Exact frozen-reference linkage verification ==="
)

# ---------------------------------------------------------
# 1. Frozen reference selected earlier
# ---------------------------------------------------------

ref_624d0g = phase_references_622h[
    "det_pred_pct_b5"
]

frozen_edges_624d0g = np.asarray(
    ref_624d0g["edges"],
    dtype=np.float64
).reshape(-1)

frozen_refs_624d0g = ref_624d0g[
    "references"
]


# ---------------------------------------------------------
# 2. Globals actually used by uncertainty helper
# ---------------------------------------------------------

helper_edges_624d0g = np.asarray(
    edges_623cb2,
    dtype=np.float64
).reshape(-1)

helper_refs_624d0g = (
    reference_for_bin_623cb2
)


# ---------------------------------------------------------
# 3. Edge comparison
# ---------------------------------------------------------

print("\n--- Edge linkage ---")

print(
    "Frozen DEV edge shape :",
    frozen_edges_624d0g.shape
)

print(
    "Helper edge shape     :",
    helper_edges_624d0g.shape
)

edge_shape_match_624d0g = (
    frozen_edges_624d0g.shape
    ==
    helper_edges_624d0g.shape
)

if edge_shape_match_624d0g:

    edge_max_diff_624d0g = float(
        np.max(
            np.abs(
                frozen_edges_624d0g
                -
                helper_edges_624d0g
            )
        )
    )

else:
    edge_max_diff_624d0g = np.inf


print(
    "Edge shape match      :",
    edge_shape_match_624d0g
)

print(
    "Maximum edge diff     :",
    edge_max_diff_624d0g
)


# ---------------------------------------------------------
# 4. Per-phase reference comparison
# ---------------------------------------------------------

print(
    "\n--- Phase-reference linkage ---"
)

frozen_keys_624d0g = sorted(
    frozen_refs_624d0g.keys()
)

helper_keys_624d0g = sorted(
    helper_refs_624d0g.keys()
)

print(
    "Frozen DEV phases :",
    frozen_keys_624d0g
)

print(
    "Helper phases     :",
    helper_keys_624d0g
)

key_match_624d0g = (
    frozen_keys_624d0g
    ==
    helper_keys_624d0g
)

print(
    "Phase keys match  :",
    key_match_624d0g
)


phase_results_624d0g = []

if key_match_624d0g:

    for phase_624d0g in frozen_keys_624d0g:

        a_624d0g = np.asarray(
            frozen_refs_624d0g[
                phase_624d0g
            ],
            dtype=np.float64
        ).reshape(-1)

        b_624d0g = np.asarray(
            helper_refs_624d0g[
                phase_624d0g
            ],
            dtype=np.float64
        ).reshape(-1)

        shape_match_624d0g = (
            a_624d0g.shape
            ==
            b_624d0g.shape
        )

        if shape_match_624d0g:
            max_diff_624d0g = float(
                np.max(
                    np.abs(
                        a_624d0g
                        -
                        b_624d0g
                    )
                )
            )
        else:
            max_diff_624d0g = np.inf

        exact_624d0g = bool(
            shape_match_624d0g
            and
            max_diff_624d0g == 0.0
        )

        phase_results_624d0g.append(
            exact_624d0g
        )

        print(
            f"Phase {phase_624d0g}: "
            f"shape={a_624d0g.shape} | "
            f"max_diff={max_diff_624d0g:.12g} | "
            f"exact={exact_624d0g}"
        )


all_refs_exact_624d0g = bool(
    key_match_624d0g
    and
    all(phase_results_624d0g)
)


# ---------------------------------------------------------
# 5. Final leakage-safety verdict
# ---------------------------------------------------------

edge_exact_624d0g = bool(
    edge_shape_match_624d0g
    and
    edge_max_diff_624d0g == 0.0
)

leakage_safe_624d0g = bool(
    edge_exact_624d0g
    and
    all_refs_exact_624d0g
)


print(
    "\n========================================"
)

print(
    "Edge linkage exact       :",
    edge_exact_624d0g
)

print(
    "All phase refs exact     :",
    all_refs_exact_624d0g
)

print(
    "DEV-reference linkage OK :",
    leakage_safe_624d0g
)

print(
    "Official TEST scored     :",
    False
)

print(
    "Frozen decision modified :",
    False
)

print(
    "Files written            :",
    False
)

print(
    "========================================"
)

if not leakage_safe_624d0g:
    raise RuntimeError(
        "Frozen DEV reference linkage "
        "could not be certified."
    )

print(
    "PASS — uncertainty transformation "
    "uses the exact frozen DEV references."
)

print("=== END ===")

In [ ]:
import inspect

print(
    "=== Step 6.24D0H — "
    "reference_for_bin helper inspection ==="
)

# ---------------------------------------------------------
# 1. Basic function identity
# ---------------------------------------------------------

print(
    "\n--- Helper identity ---"
)

print(
    "Available :",
    "reference_for_bin_623cb2" in globals()
)

print(
    "Type      :",
    type(reference_for_bin_623cb2).__name__
)

print(
    "Callable  :",
    callable(reference_for_bin_623cb2)
)

print(
    "Signature :",
    inspect.signature(
        reference_for_bin_623cb2
    )
)


# ---------------------------------------------------------
# 2. Complete source
# ---------------------------------------------------------

print(
    "\n--- Complete helper source ---"
)

source_624d0h = inspect.getsource(
    reference_for_bin_623cb2
)

print(source_624d0h)


# ---------------------------------------------------------
# 3. Names referenced by helper
# ---------------------------------------------------------

code_624d0h = (
    reference_for_bin_623cb2.__code__
)

names_624d0h = list(
    code_624d0h.co_names
)

print(
    "\n--- Names referenced by helper ---"
)

for name_624d0h in names_624d0h:
    print(name_624d0h)


# ---------------------------------------------------------
# 4. Relevant global objects
# ---------------------------------------------------------

print(
    "\n--- Relevant referenced globals ---"
)

keywords_624d0h = (
    "reference",
    "phase",
    "bin",
    "edge",
    "622",
    "623",
)

for name_624d0h in names_624d0h:

    if name_624d0h not in globals():
        continue

    if not any(
        key_624d0h.lower()
        in name_624d0h.lower()
        for key_624d0h
        in keywords_624d0h
    ):
        continue

    obj_624d0h = globals()[
        name_624d0h
    ]

    print(
        f"{name_624d0h:<35} "
        f"type={type(obj_624d0h).__name__}"
    )


# ---------------------------------------------------------
# 5. Integrity
# ---------------------------------------------------------

print(
    "\n--- Integrity ---"
)

print(
    "Official TEST scored     :",
    False
)

print(
    "Frozen decision modified :",
    False
)

print(
    "Files written            :",
    False
)

print(
    "Nothing was modified."
)

print("=== END ===")

In [ ]:
import numpy as np
import pandas as pd

print(
    "=== Step 6.24D0I — "
    "Exact phase-reference linkage verification ==="
)


# ---------------------------------------------------------
# 1. Required objects
# ---------------------------------------------------------

required_624d0i = [
    "phase_references_622h",
    "refs_623cb2",
    "reference_for_bin_623cb2",
]

print("\n--- Required objects ---")

for name_624d0i in required_624d0i:
    print(
        f"{name_624d0i:<32}:",
        name_624d0i in globals()
    )

missing_624d0i = [
    name_624d0i
    for name_624d0i in required_624d0i
    if name_624d0i not in globals()
]

if missing_624d0i:
    raise RuntimeError(
        "Missing required objects: "
        + str(missing_624d0i)
    )


# ---------------------------------------------------------
# 2. Recover the frozen DEVELOPMENT reference
# ---------------------------------------------------------

selected_key_624d0i = "det_pred_pct_b5"

if selected_key_624d0i not in phase_references_622h:
    raise KeyError(
        f"Missing frozen reference key: "
        f"{selected_key_624d0i}"
    )

frozen_record_624d0i = (
    phase_references_622h[
        selected_key_624d0i
    ]
)

frozen_refs_624d0i = (
    frozen_record_624d0i[
        "references"
    ]
)


print(
    "\n--- Reference containers ---"
)

print(
    "Frozen reference type :",
    type(frozen_refs_624d0i).__name__
)

print(
    "Helper backing type   :",
    type(refs_623cb2).__name__
)

print(
    "Same Python object    :",
    refs_623cb2 is frozen_refs_624d0i
)


# ---------------------------------------------------------
# 3. Safe key retrieval
# ---------------------------------------------------------

def get_phase_reference_624d0i(
    mapping,
    phase_id
):
    if phase_id in mapping:
        return mapping[phase_id]

    if str(phase_id) in mapping:
        return mapping[str(phase_id)]

    raise KeyError(
        f"Phase {phase_id} not found."
    )


# ---------------------------------------------------------
# 4. Compare every frozen phase
# ---------------------------------------------------------

rows_624d0i = []

frozen_phase_ids_624d0i = sorted(
    [
        int(k)
        for k
        in frozen_refs_624d0i.keys()
    ]
)


for phase_id_624d0i in frozen_phase_ids_624d0i:

    frozen_arr_624d0i = np.asarray(
        get_phase_reference_624d0i(
            frozen_refs_624d0i,
            phase_id_624d0i
        ),
        dtype=np.float64
    ).reshape(-1)

    backing_arr_624d0i = np.asarray(
        get_phase_reference_624d0i(
            refs_623cb2,
            phase_id_624d0i
        ),
        dtype=np.float64
    ).reshape(-1)

    helper_arr_624d0i = np.asarray(
        reference_for_bin_623cb2(
            phase_id_624d0i
        ),
        dtype=np.float64
    ).reshape(-1)

    if (
        frozen_arr_624d0i.shape
        != helper_arr_624d0i.shape
    ):
        max_diff_624d0i = np.nan

    elif frozen_arr_624d0i.size == 0:
        max_diff_624d0i = 0.0

    else:
        max_diff_624d0i = float(
            np.max(
                np.abs(
                    frozen_arr_624d0i
                    - helper_arr_624d0i
                )
            )
        )

    rows_624d0i.append(
        {
            "phase":
                phase_id_624d0i,

            "frozen_shape":
                frozen_arr_624d0i.shape,

            "backing_shape":
                backing_arr_624d0i.shape,

            "helper_shape":
                helper_arr_624d0i.shape,

            "frozen_finite":
                bool(
                    np.isfinite(
                        frozen_arr_624d0i
                    ).all()
                ),

            "helper_finite":
                bool(
                    np.isfinite(
                        helper_arr_624d0i
                    ).all()
                ),

            "backing_vs_helper_exact":
                bool(
                    np.array_equal(
                        backing_arr_624d0i,
                        helper_arr_624d0i
                    )
                ),

            "frozen_vs_helper_exact":
                bool(
                    np.array_equal(
                        frozen_arr_624d0i,
                        helper_arr_624d0i
                    )
                ),

            "frozen_vs_helper_max_diff":
                max_diff_624d0i,
        }
    )


REFERENCE_LINKAGE_624D0I = pd.DataFrame(
    rows_624d0i
)

print(
    "\n--- Phase-by-phase comparison ---"
)

display(
    REFERENCE_LINKAGE_624D0I
)


# ---------------------------------------------------------
# 5. Overall verification
# ---------------------------------------------------------

all_shapes_match_624d0i = bool(
    all(
        row["frozen_shape"]
        == row["helper_shape"]
        for row in rows_624d0i
    )
)

all_finite_624d0i = bool(
    REFERENCE_LINKAGE_624D0I[
        [
            "frozen_finite",
            "helper_finite",
        ]
    ]
    .to_numpy()
    .all()
)

all_backing_exact_624d0i = bool(
    REFERENCE_LINKAGE_624D0I[
        "backing_vs_helper_exact"
    ].all()
)

all_frozen_exact_624d0i = bool(
    REFERENCE_LINKAGE_624D0I[
        "frozen_vs_helper_exact"
    ].all()
)

max_reference_error_624d0i = float(
    REFERENCE_LINKAGE_624D0I[
        "frozen_vs_helper_max_diff"
    ].max()
)


print(
    "\n=========================================="
)

print(
    "STEP 6.24D0I LINKAGE SUMMARY"
)

print(
    "=========================================="
)

print(
    "Frozen phases              :",
    frozen_phase_ids_624d0i
)

print(
    "All phase shapes match     :",
    all_shapes_match_624d0i
)

print(
    "All references finite      :",
    all_finite_624d0i
)

print(
    "Backing/helper exact       :",
    all_backing_exact_624d0i
)

print(
    "Frozen/helper exact        :",
    all_frozen_exact_624d0i
)

print(
    "Maximum reference error    :",
    max_reference_error_624d0i
)

print(
    "Official TEST scored       :",
    False
)

print(
    "Frozen decision modified   :",
    False
)

print(
    "Files written              :",
    False
)

print("=== END ===")

In [ ]:
import numpy as np
import pandas as pd

print(
    "=== Step 6.24E — Official FD001 endpoint inference "
    "with frozen reliability rule ==="
)


# =========================================================
# 1. Required frozen objects
# =========================================================

required_624e = [
    "X_TEST_CLEAN_624C",
    "OFFICIAL_TEST_SEQ_META_624C",
    "mc_uncertainty_623cb2",
    "RELOADED_THRESHOLD_623DL5",
    "RELOADED_RULE_623DL5",
]

print("\n--- Required objects ---")

for name_624e in required_624e:
    print(
        f"{name_624e:<32}:",
        name_624e in globals()
    )

missing_624e = [
    name_624e
    for name_624e in required_624e
    if name_624e not in globals()
]

if missing_624e:
    raise RuntimeError(
        "Missing required objects: "
        + str(missing_624e)
    )


# =========================================================
# 2. Verify complete TEST sequence structure
# =========================================================

X_test_all_624e = np.asarray(
    X_TEST_CLEAN_624C
)

meta_test_all_624e = (
    OFFICIAL_TEST_SEQ_META_624C
    .copy()
    .reset_index(drop=True)
)

print(
    "\n--- Complete official TEST sequence set ---"
)

print(
    "Sequence shape :",
    X_test_all_624e.shape
)

print(
    "Metadata shape :",
    meta_test_all_624e.shape
)

if len(X_test_all_624e) != len(
    meta_test_all_624e
):
    raise RuntimeError(
        "TEST sequence/metadata length mismatch."
    )

if X_test_all_624e.shape != (
    10196,
    30,
    17,
):
    raise RuntimeError(
        "Unexpected official TEST sequence shape."
    )

if not np.isfinite(
    X_test_all_624e
).all():
    raise RuntimeError(
        "Non-finite official TEST sequence value."
    )


# =========================================================
# 3. Identify exactly one final window per engine
#    ONLY unit_id/cycle are used for this selection.
# =========================================================

required_meta_cols_624e = [
    "unit_id",
    "cycle",
]

missing_meta_cols_624e = [
    col
    for col in required_meta_cols_624e
    if col not in meta_test_all_624e.columns
]

if missing_meta_cols_624e:
    raise RuntimeError(
        "Missing TEST metadata columns: "
        + str(missing_meta_cols_624e)
    )


last_cycle_by_engine_624e = (
    meta_test_all_624e
    .groupby(
        "unit_id"
    )["cycle"]
    .transform("max")
)

endpoint_mask_624e = (
    meta_test_all_624e["cycle"]
    .to_numpy()
    ==
    last_cycle_by_engine_624e
    .to_numpy()
)

endpoint_positions_624e = np.flatnonzero(
    endpoint_mask_624e
)

TEST_ENDPOINT_META_624E = (
    meta_test_all_624e
    .iloc[
        endpoint_positions_624e
    ]
    .copy()
    .reset_index(drop=True)
)

X_TEST_ENDPOINT_624E = (
    X_test_all_624e[
        endpoint_positions_624e
    ]
    .copy()
)


print(
    "\n--- Conventional endpoint set ---"
)

print(
    "Endpoint windows :",
    len(
        X_TEST_ENDPOINT_624E
    )
)

print(
    "Endpoint shape   :",
    X_TEST_ENDPOINT_624E.shape
)

print(
    "Endpoint engines :",
    TEST_ENDPOINT_META_624E[
        "unit_id"
    ].nunique()
)

print(
    "Duplicate engines:",
    int(
        TEST_ENDPOINT_META_624E[
            "unit_id"
        ].duplicated().sum()
    )
)


if len(
    X_TEST_ENDPOINT_624E
) != 100:
    raise RuntimeError(
        "Expected exactly 100 FD001 endpoint windows."
    )

if TEST_ENDPOINT_META_624E[
    "unit_id"
].nunique() != 100:
    raise RuntimeError(
        "Expected exactly 100 endpoint engines."
    )

if TEST_ENDPOINT_META_624E[
    "unit_id"
].duplicated().any():
    raise RuntimeError(
        "Duplicate engine in endpoint set."
    )


# =========================================================
# 4. Run the already-frozen uncertainty recipe
#
# IMPORTANT:
# mc_uncertainty_623cb2 returns:
#
#   deterministic prediction
#   raw MC standard deviation
#   phase-adjusted uncertainty
#
# No fitting or calibration happens here.
# =========================================================

print(
    "\n--- Running frozen endpoint inference ---"
)

mc_result_624e = (
    mc_uncertainty_623cb2(
        X_TEST_ENDPOINT_624E
    )
)

if not isinstance(
    mc_result_624e,
    (
        tuple,
        list,
    )
):
    raise RuntimeError(
        "Unexpected MC helper return type."
    )

if len(
    mc_result_624e
) != 3:
    raise RuntimeError(
        "Expected exactly three MC-helper outputs."
    )


TEST_ENDPOINT_PRED_624E = np.asarray(
    mc_result_624e[0],
    dtype=np.float64
).reshape(-1)

TEST_ENDPOINT_MC_STD_624E = np.asarray(
    mc_result_624e[1],
    dtype=np.float64
).reshape(-1)

TEST_ENDPOINT_RELIABILITY_624E = np.asarray(
    mc_result_624e[2],
    dtype=np.float64
).reshape(-1)


# =========================================================
# 5. Structural checks
# =========================================================

expected_n_624e = 100

for array_name_624e, array_624e in [
    (
        "prediction",
        TEST_ENDPOINT_PRED_624E,
    ),
    (
        "MC standard deviation",
        TEST_ENDPOINT_MC_STD_624E,
    ),
    (
        "reliability score",
        TEST_ENDPOINT_RELIABILITY_624E,
    ),
]:
    if array_624e.shape != (
        expected_n_624e,
    ):
        raise RuntimeError(
            f"Unexpected {array_name_624e} shape: "
            f"{array_624e.shape}"
        )

    if not np.isfinite(
        array_624e
    ).all():
        raise RuntimeError(
            f"Non-finite values detected in "
            f"{array_name_624e}."
        )


# =========================================================
# 6. Apply the FROZEN reliability threshold
#
# Frozen rule:
# ACCEPT  when reliability_score <= threshold
# ABSTAIN when reliability_score >  threshold
# =========================================================

FROZEN_TEST_THRESHOLD_624E = float(
    RELOADED_THRESHOLD_623DL5
)

TEST_ENDPOINT_ACCEPT_MASK_624E = (
    TEST_ENDPOINT_RELIABILITY_624E
    <=
    FROZEN_TEST_THRESHOLD_624E
)

TEST_ENDPOINT_ABSTAIN_MASK_624E = (
    ~TEST_ENDPOINT_ACCEPT_MASK_624E
)


accepted_n_624e = int(
    TEST_ENDPOINT_ACCEPT_MASK_624E.sum()
)

abstained_n_624e = int(
    TEST_ENDPOINT_ABSTAIN_MASK_624E.sum()
)

endpoint_coverage_624e = float(
    TEST_ENDPOINT_ACCEPT_MASK_624E.mean()
)


# =========================================================
# 7. Construct inference record
#
# Deliberately DO NOT attach RUL target values here.
# =========================================================

OFFICIAL_TEST_ENDPOINT_INFERENCE_624E = (
    TEST_ENDPOINT_META_624E[
        [
            "unit_id",
            "cycle",
        ]
    ]
    .copy()
)

OFFICIAL_TEST_ENDPOINT_INFERENCE_624E[
    "prediction"
] = TEST_ENDPOINT_PRED_624E

OFFICIAL_TEST_ENDPOINT_INFERENCE_624E[
    "mc_std"
] = TEST_ENDPOINT_MC_STD_624E

OFFICIAL_TEST_ENDPOINT_INFERENCE_624E[
    "reliability_score"
] = TEST_ENDPOINT_RELIABILITY_624E

OFFICIAL_TEST_ENDPOINT_INFERENCE_624E[
    "accept"
] = TEST_ENDPOINT_ACCEPT_MASK_624E

OFFICIAL_TEST_ENDPOINT_INFERENCE_624E[
    "decision"
] = np.where(
    TEST_ENDPOINT_ACCEPT_MASK_624E,
    "ACCEPT",
    "ABSTAIN",
)


# =========================================================
# 8. Compact non-label inspection
# =========================================================

print(
    "\n--- First ten endpoint inference records ---"
)

display(
    OFFICIAL_TEST_ENDPOINT_INFERENCE_624E
    .head(10)
)


print(
    "\n--- Reliability distribution ---"
)

reliability_summary_624e = pd.Series(
    TEST_ENDPOINT_RELIABILITY_624E
).describe(
    percentiles=[
        0.10,
        0.25,
        0.50,
        0.75,
        0.90,
    ]
)

display(
    reliability_summary_624e
    .to_frame(
        "reliability_score"
    )
)


# =========================================================
# 9. Final integrity summary
# =========================================================

print(
    "\n=============================================="
)

print(
    "STEP 6.24E PRE-SCORING INFERENCE SUMMARY"
)

print(
    "=============================================="
)

print(
    "Official endpoint engines     :",
    100
)

print(
    "Endpoint sequence shape       :",
    X_TEST_ENDPOINT_624E.shape
)

print(
    "Prediction values finite      :",
    bool(
        np.isfinite(
            TEST_ENDPOINT_PRED_624E
        ).all()
    )
)

print(
    "MC-STD values finite          :",
    bool(
        np.isfinite(
            TEST_ENDPOINT_MC_STD_624E
        ).all()
    )
)

print(
    "Reliability values finite     :",
    bool(
        np.isfinite(
            TEST_ENDPOINT_RELIABILITY_624E
        ).all()
    )
)

print(
    "Frozen reliability threshold  :",
    FROZEN_TEST_THRESHOLD_624E
)

print(
    "Accepted endpoints            :",
    f"{accepted_n_624e} / 100"
)

print(
    "Abstained endpoints           :",
    f"{abstained_n_624e} / 100"
)

print(
    "Official endpoint coverage    :",
    endpoint_coverage_624e
)

print(
    "Frozen rule modified          :",
    False
)

print(
    "Threshold modified            :",
    False
)

print(
    "Official labels used for score:",
    False
)

print(
    "Files written                 :",
    False
)

print("=== END ===")

In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd


print(
    "=== Step 6.24E1 — Freeze official FD001 "
    "pre-score endpoint inference ==="
)


# =========================================================
# 1. Required objects
# =========================================================

required_624e1 = [
    "OFFICIAL_TEST_ENDPOINT_INFERENCE_624E",
    "TEST_ENDPOINT_PRED_624E",
    "TEST_ENDPOINT_MC_STD_624E",
    "TEST_ENDPOINT_RELIABILITY_624E",
    "TEST_ENDPOINT_ACCEPT_MASK_624E",
    "FROZEN_TEST_THRESHOLD_624E",
    "RELOADED_TARGET_COVERAGE_623DL5",
    "FROZEN_RELIABILITY_SIGNAL_NAME_623DL3",
]

print("\n--- Required objects ---")

for name_624e1 in required_624e1:
    print(
        f"{name_624e1:<40}:",
        name_624e1 in globals(),
    )

missing_624e1 = [
    name
    for name in required_624e1
    if name not in globals()
]

if missing_624e1:
    raise RuntimeError(
        "Missing required objects: "
        + str(missing_624e1)
    )


# =========================================================
# 2. Copy and certify the pre-score inference table
# =========================================================

prescore_df_624e1 = (
    OFFICIAL_TEST_ENDPOINT_INFERENCE_624E
    .copy()
    .reset_index(drop=True)
)


expected_columns_624e1 = [
    "unit_id",
    "cycle",
    "prediction",
    "mc_std",
    "reliability_score",
    "accept",
    "decision",
]


if list(
    prescore_df_624e1.columns
) != expected_columns_624e1:
    raise RuntimeError(
        "Unexpected pre-score inference columns."
    )


# Explicitly guarantee that NO RUL labels are present.

for forbidden_624e1 in [
    "RUL",
    "rul",
    "target",
    "true_rul",
    "RUL_raw",
    "RUL_cap_125",
    "RUL_cap_130",
]:
    if forbidden_624e1 in prescore_df_624e1.columns:
        raise RuntimeError(
            "Ground-truth/target column detected "
            "in pre-score artifact: "
            + forbidden_624e1
        )


if len(
    prescore_df_624e1
) != 100:
    raise RuntimeError(
        "Expected exactly 100 endpoint rows."
    )


if prescore_df_624e1[
    "unit_id"
].nunique() != 100:
    raise RuntimeError(
        "Expected exactly 100 unique engines."
    )


numeric_cols_624e1 = [
    "unit_id",
    "cycle",
    "prediction",
    "mc_std",
    "reliability_score",
]

if not np.isfinite(
    prescore_df_624e1[
        numeric_cols_624e1
    ].to_numpy(
        dtype=np.float64
    )
).all():
    raise RuntimeError(
        "Non-finite pre-score value detected."
    )


# =========================================================
# 3. Cross-check frozen arrays against the table
# =========================================================

prediction_error_624e1 = float(
    np.max(
        np.abs(
            prescore_df_624e1[
                "prediction"
            ].to_numpy(
                dtype=np.float64
            )
            -
            np.asarray(
                TEST_ENDPOINT_PRED_624E,
                dtype=np.float64,
            )
        )
    )
)

mc_error_624e1 = float(
    np.max(
        np.abs(
            prescore_df_624e1[
                "mc_std"
            ].to_numpy(
                dtype=np.float64
            )
            -
            np.asarray(
                TEST_ENDPOINT_MC_STD_624E,
                dtype=np.float64,
            )
        )
    )
)

reliability_error_624e1 = float(
    np.max(
        np.abs(
            prescore_df_624e1[
                "reliability_score"
            ].to_numpy(
                dtype=np.float64
            )
            -
            np.asarray(
                TEST_ENDPOINT_RELIABILITY_624E,
                dtype=np.float64,
            )
        )
    )
)

mask_exact_624e1 = bool(
    np.array_equal(
        prescore_df_624e1[
            "accept"
        ].to_numpy(
            dtype=bool
        ),
        np.asarray(
            TEST_ENDPOINT_ACCEPT_MASK_624E,
            dtype=bool,
        ),
    )
)


print(
    "\n--- In-memory reproduction check ---"
)

print(
    "Prediction max error  :",
    prediction_error_624e1,
)

print(
    "MC-STD max error      :",
    mc_error_624e1,
)

print(
    "Reliability max error :",
    reliability_error_624e1,
)

print(
    "Accept mask exact     :",
    mask_exact_624e1,
)


if prediction_error_624e1 != 0.0:
    raise RuntimeError(
        "Prediction reproduction mismatch."
    )

if mc_error_624e1 != 0.0:
    raise RuntimeError(
        "MC-STD reproduction mismatch."
    )

if reliability_error_624e1 != 0.0:
    raise RuntimeError(
        "Reliability reproduction mismatch."
    )

if not mask_exact_624e1:
    raise RuntimeError(
        "Accept-mask reproduction mismatch."
    )


# =========================================================
# 4. Output directory
# =========================================================

if "PROJECT_ROOT" not in globals():
    raise RuntimeError(
        "PROJECT_ROOT is not available."
    )


output_dir_624e1 = (
    Path(PROJECT_ROOT)
    / "06_results"
    / "step6_official_test"
)

output_dir_624e1.mkdir(
    parents=True,
    exist_ok=True,
)


csv_path_624e1 = (
    output_dir_624e1
    / "fd001_official_endpoint_prescore_inference.csv"
)

npz_path_624e1 = (
    output_dir_624e1
    / "fd001_official_endpoint_prescore_arrays.npz"
)

manifest_path_624e1 = (
    output_dir_624e1
    / "fd001_official_endpoint_prescore_manifest.json"
)


# =========================================================
# 5. Persist WITHOUT official labels
# =========================================================

prescore_df_624e1.to_csv(
    csv_path_624e1,
    index=False,
    float_format="%.17g",
)


np.savez_compressed(
    npz_path_624e1,
    unit_id=prescore_df_624e1[
        "unit_id"
    ].to_numpy(
        dtype=np.int64
    ),
    cycle=prescore_df_624e1[
        "cycle"
    ].to_numpy(
        dtype=np.int64
    ),
    prediction=np.asarray(
        TEST_ENDPOINT_PRED_624E,
        dtype=np.float64,
    ),
    mc_std=np.asarray(
        TEST_ENDPOINT_MC_STD_624E,
        dtype=np.float64,
    ),
    reliability_score=np.asarray(
        TEST_ENDPOINT_RELIABILITY_624E,
        dtype=np.float64,
    ),
    accept=np.asarray(
        TEST_ENDPOINT_ACCEPT_MASK_624E,
        dtype=bool,
    ),
)


# =========================================================
# 6. SHA-256 helper
# =========================================================

def sha256_file_624e1(path):
    digest = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as file_obj:
        for block in iter(
            lambda: file_obj.read(
                1024 * 1024
            ),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


csv_hash_624e1 = sha256_file_624e1(
    csv_path_624e1
)

npz_hash_624e1 = sha256_file_624e1(
    npz_path_624e1
)


# =========================================================
# 7. Pre-score manifest
# =========================================================

accepted_624e1 = int(
    prescore_df_624e1[
        "accept"
    ].sum()
)

abstained_624e1 = int(
    len(
        prescore_df_624e1
    )
    - accepted_624e1
)

coverage_624e1 = float(
    accepted_624e1
    /
    len(
        prescore_df_624e1
    )
)


manifest_624e1 = {
    "dataset": "NASA C-MAPSS FD001",
    "evaluation_stage":
        "official_endpoint_prescore_freeze",

    "endpoint_definition":
        "last available 30-cycle window per test engine",

    "endpoint_engines": 100,

    "reliability_signal": str(
        FROZEN_RELIABILITY_SIGNAL_NAME_623DL3
    ),

    "frozen_threshold": float(
        FROZEN_TEST_THRESHOLD_624E
    ),

    "development_target_coverage": float(
        RELOADED_TARGET_COVERAGE_623DL5
    ),

    "observed_official_test_coverage": float(
        coverage_624e1
    ),

    "accepted_endpoints": int(
        accepted_624e1
    ),

    "abstained_endpoints": int(
        abstained_624e1
    ),

    "accept_rule":
        "reliability_score <= frozen_threshold",

    "abstain_rule":
        "reliability_score > frozen_threshold",

    "official_ground_truth_used_for_scoring":
        False,

    "official_ground_truth_stored_in_artifact":
        False,

    "reliability_rule_modified":
        False,

    "threshold_modified":
        False,

    "files": {
        csv_path_624e1.name: {
            "sha256": csv_hash_624e1,
        },
        npz_path_624e1.name: {
            "sha256": npz_hash_624e1,
        },
    },
}


with open(
    manifest_path_624e1,
    "w",
    encoding="utf-8",
) as file_obj:
    json.dump(
        manifest_624e1,
        file_obj,
        indent=2,
    )


manifest_hash_624e1 = sha256_file_624e1(
    manifest_path_624e1
)


# =========================================================
# 8. Reload and verify persisted table
# =========================================================

reloaded_df_624e1 = pd.read_csv(
    csv_path_624e1
)


if len(
    reloaded_df_624e1
) != 100:
    raise RuntimeError(
        "Reloaded pre-score table has wrong size."
    )


reloaded_prediction_error_624e1 = float(
    np.max(
        np.abs(
            reloaded_df_624e1[
                "prediction"
            ].to_numpy(
                dtype=np.float64
            )
            -
            TEST_ENDPOINT_PRED_624E
        )
    )
)

reloaded_reliability_error_624e1 = float(
    np.max(
        np.abs(
            reloaded_df_624e1[
                "reliability_score"
            ].to_numpy(
                dtype=np.float64
            )
            -
            TEST_ENDPOINT_RELIABILITY_624E
        )
    )
)


# =========================================================
# 9. Final summary
# =========================================================

print(
    "\n=============================================="
)

print(
    "STEP 6.24E1 PRE-SCORE FREEZE SUMMARY"
)

print(
    "=============================================="
)

print(
    "Endpoint records       :",
    len(
        prescore_df_624e1
    )
)

print(
    "Reliability signal     :",
    FROZEN_RELIABILITY_SIGNAL_NAME_623DL3,
)

print(
    "Frozen threshold       :",
    FROZEN_TEST_THRESHOLD_624E,
)

print(
    "Accepted               :",
    f"{accepted_624e1} / 100",
)

print(
    "Abstained              :",
    f"{abstained_624e1} / 100",
)

print(
    "Observed coverage      :",
    coverage_624e1,
)

print(
    "Ground-truth columns   :",
    False,
)

print(
    "Reload prediction err  :",
    reloaded_prediction_error_624e1,
)

print(
    "Reload reliability err :",
    reloaded_reliability_error_624e1,
)

print(
    "CSV SHA-256            :",
    csv_hash_624e1,
)

print(
    "NPZ SHA-256            :",
    npz_hash_624e1,
)

print(
    "Manifest SHA-256       :",
    manifest_hash_624e1,
)

print(
    "\nFrozen files:"
)

print(
    "1.",
    csv_path_624e1,
)

print(
    "2.",
    npz_path_624e1,
)

print(
    "3.",
    manifest_path_624e1,
)

print(
    "\nOfficial labels scored :",
    False,
)

print(
    "Frozen rule modified   :",
    False,
)

print(
    "Files written          :",
    True,
)

print("=== END ===")

In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd


print(
    "=== Step 6.24F — Official FD001 endpoint scoring "
    "of frozen pre-score predictions ==="
)


# =========================================================
# 1. Required objects
# =========================================================

required_624f = [
    "PROJECT_ROOT",
    "OFFICIAL_TEST_FINAL_RUL_624B",
]

print("\n--- Required objects ---")

for name_624f in required_624f:
    print(
        f"{name_624f:<36}:",
        name_624f in globals(),
    )

missing_624f = [
    name
    for name in required_624f
    if name not in globals()
]

if missing_624f:
    raise RuntimeError(
        "Missing required objects: "
        + str(missing_624f)
    )


# =========================================================
# 2. Frozen pre-score artifact paths
# =========================================================

score_dir_624f = (
    Path(PROJECT_ROOT)
    / "06_results"
    / "step6_official_test"
)

prescore_csv_624f = (
    score_dir_624f
    / "fd001_official_endpoint_prescore_inference.csv"
)

prescore_npz_624f = (
    score_dir_624f
    / "fd001_official_endpoint_prescore_arrays.npz"
)

prescore_manifest_path_624f = (
    score_dir_624f
    / "fd001_official_endpoint_prescore_manifest.json"
)


for path_624f in [
    prescore_csv_624f,
    prescore_npz_624f,
    prescore_manifest_path_624f,
]:
    if not path_624f.exists():
        raise FileNotFoundError(
            f"Missing frozen pre-score artifact: "
            f"{path_624f}"
        )


# =========================================================
# 3. SHA-256 helper
# =========================================================

def sha256_file_624f(path):
    digest = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as file_obj:
        for block in iter(
            lambda: file_obj.read(
                1024 * 1024
            ),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


# =========================================================
# 4. Verify frozen pre-score artifact integrity
# =========================================================

with open(
    prescore_manifest_path_624f,
    "r",
    encoding="utf-8",
) as file_obj:
    prescore_manifest_624f = json.load(
        file_obj
    )


if (
    prescore_manifest_624f.get(
        "official_ground_truth_used_for_scoring"
    )
    is not False
):
    raise RuntimeError(
        "Pre-score manifest does not certify "
        "ground-truth-free inference."
    )


stored_file_info_624f = (
    prescore_manifest_624f[
        "files"
    ]
)


csv_hash_before_624f = sha256_file_624f(
    prescore_csv_624f
)

npz_hash_before_624f = sha256_file_624f(
    prescore_npz_624f
)


expected_csv_hash_624f = (
    stored_file_info_624f[
        prescore_csv_624f.name
    ][
        "sha256"
    ]
)

expected_npz_hash_624f = (
    stored_file_info_624f[
        prescore_npz_624f.name
    ][
        "sha256"
    ]
)


csv_hash_valid_624f = bool(
    csv_hash_before_624f
    == expected_csv_hash_624f
)

npz_hash_valid_624f = bool(
    npz_hash_before_624f
    == expected_npz_hash_624f
)


print(
    "\n--- Frozen pre-score integrity ---"
)

print(
    "CSV hash valid :",
    csv_hash_valid_624f,
)

print(
    "NPZ hash valid :",
    npz_hash_valid_624f,
)


if not (
    csv_hash_valid_624f
    and npz_hash_valid_624f
):
    raise RuntimeError(
        "Frozen pre-score artifact hash mismatch."
    )


# =========================================================
# 5. Reload frozen predictions / decisions
# =========================================================

prescore_df_624f = pd.read_csv(
    prescore_csv_624f
)


expected_prescore_columns_624f = [
    "unit_id",
    "cycle",
    "prediction",
    "mc_std",
    "reliability_score",
    "accept",
    "decision",
]


if list(
    prescore_df_624f.columns
) != expected_prescore_columns_624f:
    raise RuntimeError(
        "Unexpected frozen pre-score columns."
    )


if len(
    prescore_df_624f
) != 100:
    raise RuntimeError(
        "Expected 100 frozen endpoint records."
    )


if prescore_df_624f[
    "unit_id"
].nunique() != 100:
    raise RuntimeError(
        "Frozen endpoint table does not contain "
        "100 unique engines."
    )


# Normalize persisted boolean only if necessary.

if not pd.api.types.is_bool_dtype(
    prescore_df_624f[
        "accept"
    ]
):
    normalized_accept_624f = (
        prescore_df_624f[
            "accept"
        ]
        .astype(str)
        .str.strip()
        .str.lower()
        .map(
            {
                "true": True,
                "false": False,
            }
        )
    )

    if normalized_accept_624f.isna().any():
        raise RuntimeError(
            "Could not safely normalize persisted "
            "accept decisions."
        )

    prescore_df_624f[
        "accept"
    ] = normalized_accept_624f.astype(
        bool
    )


# =========================================================
# 6. Reconstruct frozen decision rule WITHOUT labels
# =========================================================

frozen_threshold_624f = float(
    prescore_manifest_624f[
        "frozen_threshold"
    ]
)

frozen_signal_624f = str(
    prescore_manifest_624f[
        "reliability_signal"
    ]
)

target_coverage_624f = float(
    prescore_manifest_624f[
        "development_target_coverage"
    ]
)


derived_accept_624f = (
    prescore_df_624f[
        "reliability_score"
    ].to_numpy(
        dtype=np.float64
    )
    <= frozen_threshold_624f
)


stored_accept_624f = (
    prescore_df_624f[
        "accept"
    ].to_numpy(
        dtype=bool
    )
)


decision_exact_624f = bool(
    np.array_equal(
        derived_accept_624f,
        stored_accept_624f,
    )
)


if not decision_exact_624f:
    raise RuntimeError(
        "Frozen accept decisions do not reproduce "
        "from the frozen threshold."
    )


print(
    "\n--- Frozen decision lock ---"
)

print(
    "Reliability signal :",
    frozen_signal_624f,
)

print(
    "Frozen threshold   :",
    frozen_threshold_624f,
)

print(
    "Target coverage    :",
    target_coverage_624f,
)

print(
    "Decision exact     :",
    decision_exact_624f,
)


# =========================================================
# 7. Normalize official NASA endpoint labels
# =========================================================

if not isinstance(
    OFFICIAL_TEST_FINAL_RUL_624B,
    pd.DataFrame,
):
    raise RuntimeError(
        "OFFICIAL_TEST_FINAL_RUL_624B is not "
        "the expected DataFrame."
    )


required_label_columns_624f = [
    "unit_id",
    "last_observed_cycle",
    "official_final_RUL",
]


missing_label_columns_624f = [
    col
    for col in required_label_columns_624f
    if col
    not in OFFICIAL_TEST_FINAL_RUL_624B.columns
]


if missing_label_columns_624f:
    raise RuntimeError(
        "Official endpoint-label structure differs "
        "from the verified Step 6.24B structure. "
        "Missing: "
        + str(
            missing_label_columns_624f
        )
    )


official_labels_624f = (
    OFFICIAL_TEST_FINAL_RUL_624B[
        required_label_columns_624f
    ]
    .copy()
    .sort_values(
        "unit_id"
    )
    .reset_index(
        drop=True
    )
)


if len(
    official_labels_624f
) != 100:
    raise RuntimeError(
        "Expected exactly 100 official endpoint labels."
    )


if official_labels_624f[
    "unit_id"
].nunique() != 100:
    raise RuntimeError(
        "Official endpoint labels contain "
        "duplicate engine IDs."
    )


expected_unit_ids_624f = np.arange(
    1,
    101,
    dtype=np.int64,
)


actual_unit_ids_624f = (
    official_labels_624f[
        "unit_id"
    ].to_numpy(
        dtype=np.int64
    )
)


if not np.array_equal(
    actual_unit_ids_624f,
    expected_unit_ids_624f,
):
    raise RuntimeError(
        "Official endpoint unit IDs are not "
        "exactly 1..100."
    )


if not np.isfinite(
    official_labels_624f[
        "official_final_RUL"
    ].to_numpy(
        dtype=np.float64
    )
).all():
    raise RuntimeError(
        "Official endpoint RUL contains "
        "non-finite values."
    )


print(
    "\n--- Official NASA endpoint labels ---"
)

print(
    "Endpoint labels      :",
    len(
        official_labels_624f
    )
)

print(
    "Unique engines       :",
    official_labels_624f[
        "unit_id"
    ].nunique(),
)

print(
    "Raw RUL minimum      :",
    float(
        official_labels_624f[
            "official_final_RUL"
        ].min()
    ),
)

print(
    "Raw RUL maximum      :",
    float(
        official_labels_624f[
            "official_final_RUL"
        ].max()
    ),
)


# =========================================================
# 8. Join labels AFTER frozen decisions
# =========================================================

scored_df_624f = (
    prescore_df_624f
    .merge(
        official_labels_624f,
        on="unit_id",
        how="inner",
        validate="one_to_one",
    )
    .sort_values(
        "unit_id"
    )
    .reset_index(
        drop=True
    )
)


if len(
    scored_df_624f
) != 100:
    raise RuntimeError(
        "Endpoint scoring join did not produce "
        "exactly 100 rows."
    )


cycle_alignment_exact_624f = bool(
    np.array_equal(
        scored_df_624f[
            "cycle"
        ].to_numpy(
            dtype=np.int64
        ),
        scored_df_624f[
            "last_observed_cycle"
        ].to_numpy(
            dtype=np.int64
        ),
    )
)


if not cycle_alignment_exact_624f:
    raise RuntimeError(
        "Frozen endpoint cycle does not match "
        "official last-observed cycle."
    )


# =========================================================
# 9. Official raw-RUL errors
# =========================================================

true_rul_624f = (
    scored_df_624f[
        "official_final_RUL"
    ].to_numpy(
        dtype=np.float64
    )
)

pred_624f = (
    scored_df_624f[
        "prediction"
    ].to_numpy(
        dtype=np.float64
    )
)


# Signed error convention:
#
# prediction - true RUL
#
# Positive = RUL overestimation
# Negative = RUL underestimation

error_624f = (
    pred_624f
    - true_rul_624f
)

abs_error_624f = np.abs(
    error_624f
)

squared_error_624f = np.square(
    error_624f
)


# NASA / PHM asymmetric penalty.
#
# Overestimation (error >= 0) receives
# the stronger denominator 10 penalty.
#
# Underestimation (error < 0) uses 13.

nasa_penalty_624f = np.where(
    error_624f < 0.0,
    np.exp(
        -error_624f / 13.0
    ) - 1.0,
    np.exp(
        error_624f / 10.0
    ) - 1.0,
)


if not (
    np.isfinite(
        error_624f
    ).all()
    and np.isfinite(
        nasa_penalty_624f
    ).all()
):
    raise RuntimeError(
        "Non-finite official endpoint error metric."
    )


scored_df_624f[
    "signed_error_pred_minus_true"
] = error_624f

scored_df_624f[
    "absolute_error"
] = abs_error_624f

scored_df_624f[
    "squared_error"
] = squared_error_624f

scored_df_624f[
    "nasa_penalty"
] = nasa_penalty_624f


# =========================================================
# 10. Metric helper
# =========================================================

def endpoint_metrics_624f(df):
    if len(
        df
    ) == 0:
        raise RuntimeError(
            "Cannot score an empty endpoint subset."
        )

    signed_error = (
        df[
            "signed_error_pred_minus_true"
        ].to_numpy(
            dtype=np.float64
        )
    )

    absolute_error = (
        df[
            "absolute_error"
        ].to_numpy(
            dtype=np.float64
        )
    )

    squared_error = (
        df[
            "squared_error"
        ].to_numpy(
            dtype=np.float64
        )
    )

    nasa_penalty = (
        df[
            "nasa_penalty"
        ].to_numpy(
            dtype=np.float64
        )
    )

    return {
        "n": int(
            len(
                df
            )
        ),
        "mae": float(
            np.mean(
                absolute_error
            )
        ),
        "rmse": float(
            np.sqrt(
                np.mean(
                    squared_error
                )
            )
        ),
        "mean_signed_error": float(
            np.mean(
                signed_error
            )
        ),
        "median_absolute_error": float(
            np.median(
                absolute_error
            )
        ),
        "nasa_score_sum": float(
            np.sum(
                nasa_penalty
            )
        ),
        "nasa_penalty_mean": float(
            np.mean(
                nasa_penalty
            )
        ),
        "overestimation_count": int(
            np.sum(
                signed_error > 0.0
            )
        ),
        "underestimation_count": int(
            np.sum(
                signed_error < 0.0
            )
        ),
        "exact_prediction_count": int(
            np.sum(
                signed_error == 0.0
            )
        ),
    }


# =========================================================
# 11. Full / accepted / abstained metrics
# =========================================================

accepted_df_624f = (
    scored_df_624f[
        scored_df_624f[
            "accept"
        ]
    ]
    .copy()
)

abstained_df_624f = (
    scored_df_624f[
        ~scored_df_624f[
            "accept"
        ]
    ]
    .copy()
)


full_metrics_624f = endpoint_metrics_624f(
    scored_df_624f
)

accepted_metrics_624f = endpoint_metrics_624f(
    accepted_df_624f
)

abstained_metrics_624f = endpoint_metrics_624f(
    abstained_df_624f
)


official_coverage_624f = float(
    len(
        accepted_df_624f
    )
    /
    len(
        scored_df_624f
    )
)


mae_relative_gain_624f = float(
    (
        full_metrics_624f[
            "mae"
        ]
        -
        accepted_metrics_624f[
            "mae"
        ]
    )
    /
    full_metrics_624f[
        "mae"
    ]
)


nasa_mean_relative_gain_624f = float(
    (
        full_metrics_624f[
            "nasa_penalty_mean"
        ]
        -
        accepted_metrics_624f[
            "nasa_penalty_mean"
        ]
    )
    /
    full_metrics_624f[
        "nasa_penalty_mean"
    ]
)


# =========================================================
# 12. Compact per-engine preview
# =========================================================

preview_columns_624f = [
    "unit_id",
    "cycle",
    "official_final_RUL",
    "prediction",
    "signed_error_pred_minus_true",
    "absolute_error",
    "reliability_score",
    "accept",
    "decision",
]


print(
    "\n--- First 10 officially scored endpoints ---"
)

display(
    scored_df_624f[
        preview_columns_624f
    ].head(
        10
    )
)


# =========================================================
# 13. Compact metric table
# =========================================================

metrics_table_624f = pd.DataFrame(
    [
        {
            "evaluation_set":
                "FULL_100_ENDPOINTS",
            "coverage":
                1.0,
            **full_metrics_624f,
        },
        {
            "evaluation_set":
                "ACCEPTED_ONLY",
            "coverage":
                official_coverage_624f,
            **accepted_metrics_624f,
        },
        {
            "evaluation_set":
                "ABSTAINED_DIAGNOSTIC",
            "coverage":
                1.0
                - official_coverage_624f,
            **abstained_metrics_624f,
        },
    ]
)


print(
    "\n--- Official FD001 endpoint metrics ---"
)

display(
    metrics_table_624f
)


# =========================================================
# 14. Save scored artifacts
# =========================================================

scored_csv_624f = (
    score_dir_624f
    / "fd001_official_endpoint_scored_results.csv"
)

summary_json_624f = (
    score_dir_624f
    / "fd001_official_endpoint_score_summary.json"
)


scored_df_624f.to_csv(
    scored_csv_624f,
    index=False,
    float_format="%.17g",
)


summary_624f = {
    "dataset":
        "NASA C-MAPSS FD001",

    "evaluation_protocol":
        "conventional final-window endpoint evaluation",

    "endpoint_engines":
        100,

    "official_test_label":
        "raw NASA final RUL",

    "training_target_convention":
        "RUL_cap_125",

    "test_labels_capped_for_primary_scoring":
        False,

    "prediction_clipping_applied":
        False,

    "decision_locked_before_scoring":
        True,

    "reliability_signal":
        frozen_signal_624f,

    "frozen_threshold":
        frozen_threshold_624f,

    "development_target_coverage":
        target_coverage_624f,

    "official_endpoint_coverage":
        official_coverage_624f,

    "accepted_endpoints":
        int(
            len(
                accepted_df_624f
            )
        ),

    "abstained_endpoints":
        int(
            len(
                abstained_df_624f
            )
        ),

    "full_coverage_metrics":
        full_metrics_624f,

    "accepted_selective_metrics":
        accepted_metrics_624f,

    "abstained_diagnostic_metrics":
        abstained_metrics_624f,

    "accepted_mae_relative_gain":
        mae_relative_gain_624f,

    "accepted_mean_nasa_penalty_relative_gain":
        nasa_mean_relative_gain_624f,

    "signed_error_definition":
        "prediction_minus_true_RUL",

    "nasa_score_definition": {
        "if_error_negative":
            "exp(-error/13)-1",
        "if_error_nonnegative":
            "exp(error/10)-1",
    },

    "pre_score_csv_sha256":
        csv_hash_before_624f,

    "pre_score_npz_sha256":
        npz_hash_before_624f,

    "frozen_rule_modified_after_test_labels":
        False,
}


with open(
    summary_json_624f,
    "w",
    encoding="utf-8",
) as file_obj:
    json.dump(
        summary_624f,
        file_obj,
        indent=2,
    )


scored_csv_hash_624f = sha256_file_624f(
    scored_csv_624f
)

summary_json_hash_624f = sha256_file_624f(
    summary_json_624f
)


# =========================================================
# 15. Verify frozen pre-score files were untouched
# =========================================================

csv_hash_after_624f = sha256_file_624f(
    prescore_csv_624f
)

npz_hash_after_624f = sha256_file_624f(
    prescore_npz_624f
)


prescore_unchanged_624f = bool(
    (
        csv_hash_before_624f
        == csv_hash_after_624f
    )
    and
    (
        npz_hash_before_624f
        == npz_hash_after_624f
    )
)


if not prescore_unchanged_624f:
    raise RuntimeError(
        "Frozen pre-score artifacts changed "
        "during scoring."
    )


# =========================================================
# 16. Freeze useful in-memory scored objects
# =========================================================

OFFICIAL_TEST_SCORED_RESULTS_624F = (
    scored_df_624f.copy()
)

OFFICIAL_TEST_METRICS_624F = (
    metrics_table_624f.copy()
)

OFFICIAL_TEST_SCORE_SUMMARY_624F = (
    dict(
        summary_624f
    )
)


# =========================================================
# 17. Final compact summary
# =========================================================

print(
    "\n=============================================="
)

print(
    "STEP 6.24F OFFICIAL FD001 SCORE SUMMARY"
)

print(
    "=============================================="
)

print(
    "Official endpoint engines :",
    len(
        scored_df_624f
    )
)

print(
    "Official label convention : raw NASA final RUL"
)

print(
    "Training target convention:",
    "RUL_cap_125",
)

print(
    "Frozen reliability signal :",
    frozen_signal_624f,
)

print(
    "Frozen threshold          :",
    frozen_threshold_624f,
)

print(
    "Accepted endpoints        :",
    f"{len(accepted_df_624f)} / 100",
)

print(
    "Abstained endpoints       :",
    f"{len(abstained_df_624f)} / 100",
)

print(
    "Official coverage         :",
    official_coverage_624f,
)


print(
    "\nFULL-COVERAGE endpoint performance"
)

print(
    "MAE                       :",
    full_metrics_624f[
        "mae"
    ],
)

print(
    "RMSE                      :",
    full_metrics_624f[
        "rmse"
    ],
)

print(
    "NASA score sum            :",
    full_metrics_624f[
        "nasa_score_sum"
    ],
)

print(
    "Mean NASA penalty         :",
    full_metrics_624f[
        "nasa_penalty_mean"
    ],
)


print(
    "\nSELECTIVE accepted-endpoint performance"
)

print(
    "Selective risk (MAE)      :",
    accepted_metrics_624f[
        "mae"
    ],
)

print(
    "Selective RMSE            :",
    accepted_metrics_624f[
        "rmse"
    ],
)

print(
    "Mean NASA penalty         :",
    accepted_metrics_624f[
        "nasa_penalty_mean"
    ],
)

print(
    "MAE relative gain         :",
    mae_relative_gain_624f,
)

print(
    "NASA-mean relative gain   :",
    nasa_mean_relative_gain_624f,
)


print(
    "\nABSTAINED-set diagnostic"
)

print(
    "Abstained MAE             :",
    abstained_metrics_624f[
        "mae"
    ],
)

print(
    "Abstained RMSE            :",
    abstained_metrics_624f[
        "rmse"
    ],
)

print(
    "Abstained mean NASA pen.  :",
    abstained_metrics_624f[
        "nasa_penalty_mean"
    ],
)


print(
    "\nCycle alignment exact     :",
    cycle_alignment_exact_624f,
)

print(
    "Pre-score artifacts intact:",
    prescore_unchanged_624f,
)

print(
    "Frozen rule modified      :",
    False,
)

print(
    "Official TEST scored      :",
    True,
)

print(
    "\nScored CSV SHA-256        :",
    scored_csv_hash_624f,
)

print(
    "Summary JSON SHA-256      :",
    summary_json_hash_624f,
)

print(
    "\nScored files:"
)

print(
    "1.",
    scored_csv_624f,
)

print(
    "2.",
    summary_json_624f,
)

print("=== END ===")

In [ ]:
print(
    "=== Step 6.24F-A — Locate verified official endpoint-label object ==="
)

import numpy as np
import pandas as pd


# ---------------------------------------------------------
# 1. Inspect the currently named object
# ---------------------------------------------------------

print("\n--- Current OFFICIAL_TEST_FINAL_RUL_624B ---")

obj_624fa = globals().get(
    "OFFICIAL_TEST_FINAL_RUL_624B",
    None
)

print(
    "Exists :",
    obj_624fa is not None
)

if obj_624fa is not None:
    print(
        "Type   :",
        type(obj_624fa).__name__
    )

    if isinstance(
        obj_624fa,
        pd.DataFrame
    ):
        print(
            "Shape  :",
            obj_624fa.shape
        )

        print(
            "Columns:",
            list(
                obj_624fa.columns
            )
        )

        print(
            "\nFirst five rows:"
        )

        display(
            obj_624fa.head()
        )


# ---------------------------------------------------------
# 2. Locate plausible 100-row endpoint DataFrames
# ---------------------------------------------------------

candidate_rows_624fa = []

for name_624fa, value_624fa in list(
    globals().items()
):
    if not isinstance(
        value_624fa,
        pd.DataFrame
    ):
        continue

    if len(
        value_624fa
    ) != 100:
        continue

    cols_624fa = [
        str(c)
        for c in value_624fa.columns
    ]

    cols_lower_624fa = [
        c.lower()
        for c in cols_624fa
    ]

    relevance_624fa = any(
        (
            "rul" in c
            or "cycle" in c
            or "unit" in c
            or "engine" in c
        )
        for c in cols_lower_624fa
    )

    if relevance_624fa:
        candidate_rows_624fa.append(
            {
                "object_name":
                    name_624fa,
                "shape":
                    str(
                        value_624fa.shape
                    ),
                "columns":
                    cols_624fa,
            }
        )


candidates_624fa = pd.DataFrame(
    candidate_rows_624fa
)


print(
    "\n--- Candidate 100-row endpoint DataFrames ---"
)

if len(
    candidates_624fa
) == 0:
    print(
        "No plausible 100-row endpoint DataFrame found."
    )
else:
    display(
        candidates_624fa
    )


# ---------------------------------------------------------
# 3. Search specifically for expected endpoint fields
# ---------------------------------------------------------

expected_624fa = {
    "unit_id",
    "last_observed_cycle",
    "official_final_RUL",
}

exact_candidates_624fa = []

for name_624fa, value_624fa in list(
    globals().items()
):
    if not isinstance(
        value_624fa,
        pd.DataFrame
    ):
        continue

    cols_624fa = set(
        map(
            str,
            value_624fa.columns
        )
    )

    overlap_624fa = sorted(
        expected_624fa.intersection(
            cols_624fa
        )
    )

    if overlap_624fa:
        exact_candidates_624fa.append(
            {
                "object_name":
                    name_624fa,
                "rows":
                    len(
                        value_624fa
                    ),
                "matching_expected_columns":
                    overlap_624fa,
                "all_columns":
                    list(
                        value_624fa.columns
                    ),
            }
        )


exact_candidates_df_624fa = pd.DataFrame(
    exact_candidates_624fa
)


print(
    "\n--- Objects containing expected endpoint fields ---"
)

if len(
    exact_candidates_df_624fa
) == 0:
    print(
        "No DataFrame currently contains "
        "the expected endpoint field names."
    )
else:
    display(
        exact_candidates_df_624fa
    )


print(
    "\nOfficial TEST scored : False"
)

print(
    "Files written       : False"
)

print(
    "Nothing was modified."
)

print("=== END ===")

In [ ]:
print(
    "=== Step 6.24F-B — Exact official endpoint-label alignment verification ==="
)

import numpy as np
import pandas as pd


# ---------------------------------------------------------
# 1. Required objects
# ---------------------------------------------------------

required_624fb = [
    "OFFICIAL_TEST_FINAL_RUL_624B",
    "OFFICIAL_TEST_ENDPOINT_AUDIT_624B",
    "OFFICIAL_TEST_ENDPOINT_INFERENCE_624E",
]

print("\n--- Required objects ---")

for name_624fb in required_624fb:
    print(
        f"{name_624fb:<42}:",
        name_624fb in globals()
    )

missing_624fb = [
    name
    for name in required_624fb
    if name not in globals()
]

if missing_624fb:
    raise RuntimeError(
        "Required object(s) missing: "
        + str(missing_624fb)
    )


# ---------------------------------------------------------
# 2. Inspect endpoint audit structure
# ---------------------------------------------------------

audit_624fb = (
    OFFICIAL_TEST_ENDPOINT_AUDIT_624B
    .copy()
)

raw_rul_624fb = (
    OFFICIAL_TEST_FINAL_RUL_624B
    .copy()
)

prescore_624fb = (
    OFFICIAL_TEST_ENDPOINT_INFERENCE_624E
    .copy()
)


print(
    "\n--- OFFICIAL_TEST_ENDPOINT_AUDIT_624B ---"
)

print(
    "Shape   :",
    audit_624fb.shape
)

print(
    "Columns :",
    list(
        audit_624fb.columns
    )
)

display(
    audit_624fb.head(10)
)


# ---------------------------------------------------------
# 3. Verify required audit fields
# ---------------------------------------------------------

required_audit_cols_624fb = [
    "unit_id",
    "last_observed_cycle",
    "official_final_RUL",
]

missing_audit_cols_624fb = [
    col
    for col in required_audit_cols_624fb
    if col not in audit_624fb.columns
]

print(
    "\nMissing required audit columns :",
    missing_audit_cols_624fb
)

if missing_audit_cols_624fb:
    raise RuntimeError(
        "Endpoint audit is missing required fields: "
        + str(missing_audit_cols_624fb)
    )


# ---------------------------------------------------------
# 4. Row-count / ID integrity
# ---------------------------------------------------------

audit_rows_ok_624fb = (
    len(audit_624fb) == 100
)

raw_rows_ok_624fb = (
    len(raw_rul_624fb) == 100
)

prescore_rows_ok_624fb = (
    len(prescore_624fb) == 100
)

audit_ids_624fb = (
    audit_624fb["unit_id"]
    .to_numpy(
        dtype=np.int64
    )
)

prescore_ids_624fb = (
    prescore_624fb["unit_id"]
    .to_numpy(
        dtype=np.int64
    )
)

expected_ids_624fb = np.arange(
    1,
    101,
    dtype=np.int64
)

audit_ids_exact_624fb = bool(
    np.array_equal(
        audit_ids_624fb,
        expected_ids_624fb
    )
)

prescore_ids_exact_624fb = bool(
    np.array_equal(
        prescore_ids_624fb,
        expected_ids_624fb
    )
)

cross_ids_exact_624fb = bool(
    np.array_equal(
        audit_ids_624fb,
        prescore_ids_624fb
    )
)


# ---------------------------------------------------------
# 5. Raw NASA RUL vs endpoint audit
# ---------------------------------------------------------

raw_final_rul_624fb = (
    raw_rul_624fb["final_RUL"]
    .to_numpy(
        dtype=np.float64
    )
)

audit_final_rul_624fb = (
    audit_624fb["official_final_RUL"]
    .to_numpy(
        dtype=np.float64
    )
)

rul_max_error_624fb = float(
    np.max(
        np.abs(
            raw_final_rul_624fb
            - audit_final_rul_624fb
        )
    )
)

rul_exact_624fb = bool(
    np.array_equal(
        raw_final_rul_624fb,
        audit_final_rul_624fb
    )
)


# ---------------------------------------------------------
# 6. Endpoint cycle vs frozen pre-score inference
# ---------------------------------------------------------

audit_cycle_624fb = (
    audit_624fb["last_observed_cycle"]
    .to_numpy(
        dtype=np.int64
    )
)

prescore_cycle_624fb = (
    prescore_624fb["cycle"]
    .to_numpy(
        dtype=np.int64
    )
)

cycle_exact_624fb = bool(
    np.array_equal(
        audit_cycle_624fb,
        prescore_cycle_624fb
    )
)

cycle_max_error_624fb = float(
    np.max(
        np.abs(
            audit_cycle_624fb
            - prescore_cycle_624fb
        )
    )
)


# ---------------------------------------------------------
# 7. Finite / uniqueness checks
# ---------------------------------------------------------

rul_finite_624fb = bool(
    np.isfinite(
        audit_final_rul_624fb
    ).all()
)

unique_engines_624fb = int(
    audit_624fb["unit_id"].nunique()
)

duplicate_units_624fb = int(
    audit_624fb["unit_id"].duplicated().sum()
)


# ---------------------------------------------------------
# 8. Compact comparison table
# ---------------------------------------------------------

comparison_624fb = pd.DataFrame(
    {
        "unit_id":
            audit_ids_624fb,
        "last_observed_cycle":
            audit_cycle_624fb,
        "official_final_RUL":
            audit_final_rul_624fb,
        "raw_final_RUL":
            raw_final_rul_624fb,
        "prescore_cycle":
            prescore_cycle_624fb,
    }
)

comparison_624fb[
    "RUL_difference"
] = (
    comparison_624fb["official_final_RUL"]
    - comparison_624fb["raw_final_RUL"]
)

comparison_624fb[
    "cycle_difference"
] = (
    comparison_624fb["last_observed_cycle"]
    - comparison_624fb["prescore_cycle"]
)


print(
    "\n--- First ten aligned endpoint records ---"
)

display(
    comparison_624fb.head(10)
)


# ---------------------------------------------------------
# 9. Integrity summary
# ---------------------------------------------------------

print(
    "\n=============================================="
)

print(
    "STEP 6.24F-B ALIGNMENT SUMMARY"
)

print(
    "=============================================="
)

print(
    "Audit rows                     :",
    len(audit_624fb)
)

print(
    "Raw NASA RUL rows              :",
    len(raw_rul_624fb)
)

print(
    "Frozen pre-score rows          :",
    len(prescore_624fb)
)

print(
    "Unique endpoint engines        :",
    unique_engines_624fb
)

print(
    "Duplicate endpoint engines     :",
    duplicate_units_624fb
)

print(
    "Audit IDs exactly 1..100       :",
    audit_ids_exact_624fb
)

print(
    "Pre-score IDs exactly 1..100   :",
    prescore_ids_exact_624fb
)

print(
    "Audit/pre-score IDs exact      :",
    cross_ids_exact_624fb
)

print(
    "Raw NASA / audit RUL exact     :",
    rul_exact_624fb
)

print(
    "Maximum RUL difference         :",
    rul_max_error_624fb
)

print(
    "Audit/pre-score cycles exact   :",
    cycle_exact_624fb
)

print(
    "Maximum cycle difference       :",
    cycle_max_error_624fb
)

print(
    "Official RUL values finite     :",
    rul_finite_624fb
)


alignment_ready_624fb = bool(
    audit_rows_ok_624fb
    and raw_rows_ok_624fb
    and prescore_rows_ok_624fb
    and audit_ids_exact_624fb
    and prescore_ids_exact_624fb
    and cross_ids_exact_624fb
    and rul_exact_624fb
    and cycle_exact_624fb
    and rul_finite_624fb
    and unique_engines_624fb == 100
    and duplicate_units_624fb == 0
)

print(
    "\nOFFICIAL ENDPOINT LABEL ALIGNMENT READY :",
    alignment_ready_624fb
)

print(
    "\nOfficial TEST scored : False"
)

print(
    "Frozen rule modified : False"
)

print(
    "Files written        : False"
)

print(
    "Nothing was modified."
)

print("=== END ===")

In [ ]:
print(
    "=== Step 6.24F-C — Correct official FD001 endpoint scoring ==="
)

import numpy as np
import pandas as pd


# =========================================================
# 1. Required frozen / verified objects
# =========================================================

required_624fc = [
    "OFFICIAL_TEST_ENDPOINT_INFERENCE_624E",
    "OFFICIAL_TEST_ENDPOINT_AUDIT_624B",
    "OFFICIAL_TEST_FINAL_RUL_624B",
    "FROZEN_RUL_TARGET_624C",
    "FROZEN_TEST_THRESHOLD_624E",
]

print("\n--- Required objects ---")

for name_624fc in required_624fc:
    print(
        f"{name_624fc:<42}:",
        name_624fc in globals()
    )

missing_624fc = [
    name
    for name in required_624fc
    if name not in globals()
]

if missing_624fc:
    raise RuntimeError(
        "Required object(s) missing: "
        + str(missing_624fc)
    )


# =========================================================
# 2. Copy already-frozen pre-score inference
# =========================================================

pred_df_624fc = (
    OFFICIAL_TEST_ENDPOINT_INFERENCE_624E
    .copy()
)

audit_df_624fc = (
    OFFICIAL_TEST_ENDPOINT_AUDIT_624B
    .copy()
)

raw_rul_df_624fc = (
    OFFICIAL_TEST_FINAL_RUL_624B
    .copy()
)


print(
    "\n--- Frozen pre-score structure ---"
)

print(
    "Prediction rows :",
    len(pred_df_624fc)
)

print(
    "Audit rows      :",
    len(audit_df_624fc)
)

print(
    "Raw RUL rows    :",
    len(raw_rul_df_624fc)
)


# =========================================================
# 3. Structural safeguards
# =========================================================

required_pred_cols_624fc = [
    "unit_id",
    "cycle",
    "prediction",
    "mc_std",
    "reliability_score",
    "accept",
]

missing_pred_cols_624fc = [
    col
    for col in required_pred_cols_624fc
    if col not in pred_df_624fc.columns
]

if missing_pred_cols_624fc:
    raise RuntimeError(
        "Frozen endpoint inference is missing: "
        + str(missing_pred_cols_624fc)
    )


required_audit_cols_624fc = [
    "unit_id",
    "last_observed_cycle",
    "official_final_RUL",
]

missing_audit_cols_624fc = [
    col
    for col in required_audit_cols_624fc
    if col not in audit_df_624fc.columns
]

if missing_audit_cols_624fc:
    raise RuntimeError(
        "Endpoint audit is missing: "
        + str(missing_audit_cols_624fc)
    )


if len(pred_df_624fc) != 100:
    raise RuntimeError(
        "Frozen endpoint inference does not contain 100 engines."
    )

if len(audit_df_624fc) != 100:
    raise RuntimeError(
        "Endpoint audit does not contain 100 engines."
    )

if len(raw_rul_df_624fc) != 100:
    raise RuntimeError(
        "Raw NASA endpoint RUL does not contain 100 engines."
    )


# =========================================================
# 4. Merge using engine identity
# =========================================================

scored_df_624fc = (
    pred_df_624fc[
        [
            "unit_id",
            "cycle",
            "prediction",
            "mc_std",
            "reliability_score",
            "accept",
        ]
    ]
    .merge(
        audit_df_624fc[
            [
                "unit_id",
                "last_observed_cycle",
                "official_final_RUL",
            ]
        ],
        on="unit_id",
        how="inner",
        validate="one_to_one",
    )
    .sort_values("unit_id")
    .reset_index(drop=True)
)


if len(scored_df_624fc) != 100:
    raise RuntimeError(
        "Endpoint merge did not preserve all 100 engines."
    )


# =========================================================
# 5. Verify endpoint cycle alignment again
# =========================================================

cycle_exact_624fc = bool(
    np.array_equal(
        scored_df_624fc["cycle"]
        .to_numpy(dtype=np.int64),

        scored_df_624fc["last_observed_cycle"]
        .to_numpy(dtype=np.int64),
    )
)

if not cycle_exact_624fc:
    raise RuntimeError(
        "Endpoint cycle mismatch detected before scoring."
    )


# =========================================================
# 6. Verify raw NASA label identity
# =========================================================

raw_nasa_rul_624fc = (
    raw_rul_df_624fc["final_RUL"]
    .to_numpy(dtype=np.float64)
)

audit_rul_624fc = (
    scored_df_624fc["official_final_RUL"]
    .to_numpy(dtype=np.float64)
)

raw_label_exact_624fc = bool(
    np.array_equal(
        raw_nasa_rul_624fc,
        audit_rul_624fc
    )
)

if not raw_label_exact_624fc:
    raise RuntimeError(
        "Raw NASA RUL and verified endpoint audit no longer agree."
    )


# =========================================================
# 7. Recover the frozen target convention
# =========================================================

frozen_target_name_624fc = str(
    FROZEN_RUL_TARGET_624C
)

print(
    "\nFrozen target convention :",
    frozen_target_name_624fc
)


if frozen_target_name_624fc == "RUL_raw":

    frozen_target_624fc = (
        audit_rul_624fc.copy()
    )

elif frozen_target_name_624fc == "RUL_cap_125":

    frozen_target_624fc = np.minimum(
        audit_rul_624fc,
        125.0
    )

elif frozen_target_name_624fc == "RUL_cap_130":

    frozen_target_624fc = np.minimum(
        audit_rul_624fc,
        130.0
    )

else:

    raise RuntimeError(
        "Unsupported frozen RUL target convention: "
        + frozen_target_name_624fc
    )


scored_df_624fc[
    "frozen_target"
] = frozen_target_624fc


# =========================================================
# 8. Verify frozen abstention rule
# =========================================================

threshold_624fc = float(
    FROZEN_TEST_THRESHOLD_624E
)

stored_accept_624fc = (
    scored_df_624fc["accept"]
    .astype(bool)
    .to_numpy()
)

reconstructed_accept_624fc = (
    scored_df_624fc[
        "reliability_score"
    ]
    .to_numpy(dtype=np.float64)
    <= threshold_624fc
)

accept_mask_exact_624fc = bool(
    np.array_equal(
        stored_accept_624fc,
        reconstructed_accept_624fc
    )
)

if not accept_mask_exact_624fc:
    raise RuntimeError(
        "Frozen accept mask differs from frozen reliability threshold."
    )


accepted_count_624fc = int(
    reconstructed_accept_624fc.sum()
)

abstained_count_624fc = int(
    (~reconstructed_accept_624fc).sum()
)

coverage_624fc = float(
    reconstructed_accept_624fc.mean()
)


print(
    "\n--- Frozen decision verification ---"
)

print(
    "Threshold             :",
    threshold_624fc
)

print(
    "Accept mask exact     :",
    accept_mask_exact_624fc
)

print(
    "Accepted endpoints    :",
    accepted_count_624fc
)

print(
    "Abstained endpoints   :",
    abstained_count_624fc
)

print(
    "Official coverage     :",
    coverage_624fc
)


# =========================================================
# 9. Metric helper
# =========================================================

def endpoint_metrics_624fc(
    y_true,
    y_pred,
):

    y_true = np.asarray(
        y_true,
        dtype=np.float64
    )

    y_pred = np.asarray(
        y_pred,
        dtype=np.float64
    )

    if len(y_true) == 0:
        return {
            "n": 0,
            "mae": np.nan,
            "rmse": np.nan,
            "median_ae": np.nan,
            "mean_error": np.nan,
        }

    error = (
        y_pred - y_true
    )

    abs_error = np.abs(
        error
    )

    return {
        "n":
            int(len(y_true)),

        "mae":
            float(
                np.mean(abs_error)
            ),

        "rmse":
            float(
                np.sqrt(
                    np.mean(
                        error ** 2
                    )
                )
            ),

        "median_ae":
            float(
                np.median(abs_error)
            ),

        "mean_error":
            float(
                np.mean(error)
            ),
    }


# =========================================================
# 10. Arrays for scoring
# =========================================================

prediction_624fc = (
    scored_df_624fc["prediction"]
    .to_numpy(dtype=np.float64)
)

raw_target_624fc = (
    scored_df_624fc[
        "official_final_RUL"
    ]
    .to_numpy(dtype=np.float64)
)

frozen_target_624fc = (
    scored_df_624fc[
        "frozen_target"
    ]
    .to_numpy(dtype=np.float64)
)

accept_624fc = (
    reconstructed_accept_624fc
)


# =========================================================
# 11. Frozen-target metrics
# =========================================================

frozen_full_624fc = (
    endpoint_metrics_624fc(
        frozen_target_624fc,
        prediction_624fc,
    )
)

frozen_accept_624fc = (
    endpoint_metrics_624fc(
        frozen_target_624fc[
            accept_624fc
        ],
        prediction_624fc[
            accept_624fc
        ],
    )
)

frozen_abstain_624fc = (
    endpoint_metrics_624fc(
        frozen_target_624fc[
            ~accept_624fc
        ],
        prediction_624fc[
            ~accept_624fc
        ],
    )
)


frozen_selective_gain_624fc = (
    1.0
    - (
        frozen_accept_624fc["mae"]
        /
        frozen_full_624fc["mae"]
    )
)


# =========================================================
# 12. Raw NASA endpoint metrics
# =========================================================

raw_full_624fc = (
    endpoint_metrics_624fc(
        raw_target_624fc,
        prediction_624fc,
    )
)

raw_accept_624fc = (
    endpoint_metrics_624fc(
        raw_target_624fc[
            accept_624fc
        ],
        prediction_624fc[
            accept_624fc
        ],
    )
)

raw_abstain_624fc = (
    endpoint_metrics_624fc(
        raw_target_624fc[
            ~accept_624fc
        ],
        prediction_624fc[
            ~accept_624fc
        ],
    )
)


raw_selective_gain_624fc = (
    1.0
    - (
        raw_accept_624fc["mae"]
        /
        raw_full_624fc["mae"]
    )
)


# =========================================================
# 13. Add error columns for later analysis
# =========================================================

scored_df_624fc[
    "frozen_target_abs_error"
] = np.abs(
    prediction_624fc
    - frozen_target_624fc
)

scored_df_624fc[
    "raw_nasa_abs_error"
] = np.abs(
    prediction_624fc
    - raw_target_624fc
)

scored_df_624fc[
    "accept_verified"
] = accept_624fc


# =========================================================
# 14. Compact result table
# =========================================================

OFFICIAL_TEST_SCORING_624FC = pd.DataFrame(
    [
        {
            "label_basis":
                frozen_target_name_624fc,

            "subset":
                "all",

            "n":
                frozen_full_624fc["n"],

            "coverage":
                1.0,

            "MAE":
                frozen_full_624fc["mae"],

            "RMSE":
                frozen_full_624fc["rmse"],

            "median_AE":
                frozen_full_624fc["median_ae"],

            "mean_error":
                frozen_full_624fc["mean_error"],

            "selective_MAE_gain":
                0.0,
        },

        {
            "label_basis":
                frozen_target_name_624fc,

            "subset":
                "accepted",

            "n":
                frozen_accept_624fc["n"],

            "coverage":
                coverage_624fc,

            "MAE":
                frozen_accept_624fc["mae"],

            "RMSE":
                frozen_accept_624fc["rmse"],

            "median_AE":
                frozen_accept_624fc["median_ae"],

            "mean_error":
                frozen_accept_624fc["mean_error"],

            "selective_MAE_gain":
                frozen_selective_gain_624fc,
        },

        {
            "label_basis":
                frozen_target_name_624fc,

            "subset":
                "abstained",

            "n":
                frozen_abstain_624fc["n"],

            "coverage":
                abstained_count_624fc / 100.0,

            "MAE":
                frozen_abstain_624fc["mae"],

            "RMSE":
                frozen_abstain_624fc["rmse"],

            "median_AE":
                frozen_abstain_624fc["median_ae"],

            "mean_error":
                frozen_abstain_624fc["mean_error"],

            "selective_MAE_gain":
                np.nan,
        },

        {
            "label_basis":
                "NASA_raw_RUL",

            "subset":
                "all",

            "n":
                raw_full_624fc["n"],

            "coverage":
                1.0,

            "MAE":
                raw_full_624fc["mae"],

            "RMSE":
                raw_full_624fc["rmse"],

            "median_AE":
                raw_full_624fc["median_ae"],

            "mean_error":
                raw_full_624fc["mean_error"],

            "selective_MAE_gain":
                0.0,
        },

        {
            "label_basis":
                "NASA_raw_RUL",

            "subset":
                "accepted",

            "n":
                raw_accept_624fc["n"],

            "coverage":
                coverage_624fc,

            "MAE":
                raw_accept_624fc["mae"],

            "RMSE":
                raw_accept_624fc["rmse"],

            "median_AE":
                raw_accept_624fc["median_ae"],

            "mean_error":
                raw_accept_624fc["mean_error"],

            "selective_MAE_gain":
                raw_selective_gain_624fc,
        },

        {
            "label_basis":
                "NASA_raw_RUL",

            "subset":
                "abstained",

            "n":
                raw_abstain_624fc["n"],

            "coverage":
                abstained_count_624fc / 100.0,

            "MAE":
                raw_abstain_624fc["mae"],

            "RMSE":
                raw_abstain_624fc["rmse"],

            "median_AE":
                raw_abstain_624fc["median_ae"],

            "mean_error":
                raw_abstain_624fc["mean_error"],

            "selective_MAE_gain":
                np.nan,
        },
    ]
)


OFFICIAL_TEST_ENDPOINT_SCORED_624FC = (
    scored_df_624fc.copy()
)


print(
    "\n--- Official FD001 endpoint scoring ---"
)

display(
    OFFICIAL_TEST_SCORING_624FC
)


# =========================================================
# 15. First ten scored engines
# =========================================================

print(
    "\n--- First ten scored endpoint records ---"
)

display(
    OFFICIAL_TEST_ENDPOINT_SCORED_624FC[
        [
            "unit_id",
            "cycle",
            "official_final_RUL",
            "frozen_target",
            "prediction",
            "reliability_score",
            "accept_verified",
            "frozen_target_abs_error",
            "raw_nasa_abs_error",
        ]
    ].head(10)
)


# =========================================================
# 16. Final integrity summary
# =========================================================

all_numeric_finite_624fc = bool(
    np.isfinite(
        OFFICIAL_TEST_SCORING_624FC[
            [
                "MAE",
                "RMSE",
                "median_AE",
                "mean_error",
            ]
        ].to_numpy(
            dtype=np.float64
        )
    ).all()
)


print(
    "\n==============================================="
)

print(
    "STEP 6.24F-C OFFICIAL SCORING SUMMARY"
)

print(
    "==============================================="
)

print(
    "Endpoint engines               :",
    len(scored_df_624fc)
)

print(
    "Frozen target                  :",
    frozen_target_name_624fc
)

print(
    "Frozen reliability threshold   :",
    threshold_624fc
)

print(
    "Accepted                       :",
    f"{accepted_count_624fc} / 100"
)

print(
    "Abstained                      :",
    f"{abstained_count_624fc} / 100"
)

print(
    "Coverage                       :",
    coverage_624fc
)

print(
    "\nFrozen-target full MAE         :",
    frozen_full_624fc["mae"]
)

print(
    "Frozen-target selective MAE    :",
    frozen_accept_624fc["mae"]
)

print(
    "Frozen-target MAE gain         :",
    frozen_selective_gain_624fc
)

print(
    "Abstained frozen-target MAE    :",
    frozen_abstain_624fc["mae"]
)

print(
    "\nNASA-raw full MAE              :",
    raw_full_624fc["mae"]
)

print(
    "NASA-raw selective MAE         :",
    raw_accept_624fc["mae"]
)

print(
    "NASA-raw MAE gain              :",
    raw_selective_gain_624fc
)

print(
    "Abstained NASA-raw MAE         :",
    raw_abstain_624fc["mae"]
)

print(
    "\nEndpoint alignment verified    :",
    cycle_exact_624fc
    and raw_label_exact_624fc
)

print(
    "Frozen accept mask verified    :",
    accept_mask_exact_624fc
)

print(
    "All reported metrics finite    :",
    all_numeric_finite_624fc
)

print(
    "\nOfficial TEST labels used for score : True"
)

print(
    "New model inference executed       : False"
)

print(
    "MC dropout rerun                   : False"
)

print(
    "Frozen reliability rule modified   : False"
)

print(
    "Threshold modified                 : False"
)

print(
    "Files written                      : False"
)

print(
    "In-memory scoring objects created only."
)

print("=== END ===")

In [ ]:
# ============================================================
# Step 6.24F-D
# Persist and verify CLEAN official FD001 scoring artifacts
# ============================================================

from pathlib import Path
import json
import hashlib
import numpy as np
import pandas as pd


print(
    "=== Step 6.24F-D — Persist and verify "
    "clean official FD001 scoring artifacts ==="
)


# ------------------------------------------------------------
# 1. Required frozen / verified objects
# ------------------------------------------------------------

required_objects_624fd = [
    "OFFICIAL_TEST_ENDPOINT_INFERENCE_624E",
    "OFFICIAL_TEST_ENDPOINT_AUDIT_624B",
    "TEST_ENDPOINT_META_624E",
    "FROZEN_RUL_TARGET_624C",
    "FROZEN_TEST_THRESHOLD_624E",
    "FROZEN_RELIABILITY_SIGNAL_NAME_623DL3",
]

print("\n--- Required objects ---")

missing_objects_624fd = []

for name_624fd in required_objects_624fd:
    exists_624fd = name_624fd in globals()
    print(f"{name_624fd:42s}: {exists_624fd}")

    if not exists_624fd:
        missing_objects_624fd.append(name_624fd)

if missing_objects_624fd:
    raise RuntimeError(
        "Missing required object(s): "
        + str(missing_objects_624fd)
    )


# ------------------------------------------------------------
# 2. Defensive copies only
# ------------------------------------------------------------

inference_624fd = (
    OFFICIAL_TEST_ENDPOINT_INFERENCE_624E
    .copy(deep=True)
)

audit_624fd = (
    OFFICIAL_TEST_ENDPOINT_AUDIT_624B
    .copy(deep=True)
)

endpoint_meta_624fd = (
    TEST_ENDPOINT_META_624E
    .copy(deep=True)
)

frozen_target_name_624fd = str(
    FROZEN_RUL_TARGET_624C
)

frozen_threshold_624fd = float(
    FROZEN_TEST_THRESHOLD_624E
)

frozen_signal_name_624fd = str(
    FROZEN_RELIABILITY_SIGNAL_NAME_623DL3
)


# ------------------------------------------------------------
# 3. Verify expected structures
# ------------------------------------------------------------

required_inference_cols_624fd = [
    "unit_id",
    "cycle",
    "prediction",
    "mc_std",
    "reliability_score",
    "accept",
]

required_audit_cols_624fd = [
    "unit_id",
    "last_observed_cycle",
    "official_final_RUL",
]

required_meta_cols_624fd = [
    "unit_id",
    "cycle",
    frozen_target_name_624fd,
]

for label_624fd, df_624fd, cols_624fd in [
    (
        "inference",
        inference_624fd,
        required_inference_cols_624fd,
    ),
    (
        "audit",
        audit_624fd,
        required_audit_cols_624fd,
    ),
    (
        "endpoint metadata",
        endpoint_meta_624fd,
        required_meta_cols_624fd,
    ),
]:
    missing_cols_624fd = [
        c
        for c in cols_624fd
        if c not in df_624fd.columns
    ]

    if missing_cols_624fd:
        raise RuntimeError(
            f"Missing {label_624fd} column(s): "
            f"{missing_cols_624fd}"
        )


# ------------------------------------------------------------
# 4. Restrict to exactly one official endpoint per engine
# ------------------------------------------------------------

if len(inference_624fd) != 100:
    raise RuntimeError(
        "Expected exactly 100 frozen endpoint inference rows."
    )

if inference_624fd["unit_id"].nunique() != 100:
    raise RuntimeError(
        "Frozen inference does not contain 100 unique engines."
    )

if len(audit_624fd) != 100:
    raise RuntimeError(
        "Expected exactly 100 official endpoint audit rows."
    )

if audit_624fd["unit_id"].nunique() != 100:
    raise RuntimeError(
        "Official endpoint audit does not contain "
        "100 unique engines."
    )


# Endpoint metadata should already be the 100-row endpoint set.
# Protect against accidental duplicate engine rows anyway.
if endpoint_meta_624fd["unit_id"].duplicated().any():
    raise RuntimeError(
        "Endpoint metadata contains duplicate unit_id values."
    )

if len(endpoint_meta_624fd) != 100:
    raise RuntimeError(
        "Expected exactly 100 endpoint metadata rows."
    )


# ------------------------------------------------------------
# 5. Construct canonical clean official scoring table
# ------------------------------------------------------------

scores_624fd = (
    inference_624fd[
        required_inference_cols_624fd
        + (
            ["decision"]
            if "decision" in inference_624fd.columns
            else []
        )
    ]
    .merge(
        endpoint_meta_624fd[
            [
                "unit_id",
                "cycle",
                frozen_target_name_624fd,
            ]
        ],
        on=[
            "unit_id",
            "cycle",
        ],
        how="left",
        validate="one_to_one",
    )
    .merge(
        audit_624fd[
            [
                "unit_id",
                "last_observed_cycle",
                "official_final_RUL",
            ]
        ],
        on="unit_id",
        how="left",
        validate="one_to_one",
    )
    .sort_values("unit_id")
    .reset_index(drop=True)
)


if scores_624fd[
    [
        frozen_target_name_624fd,
        "official_final_RUL",
    ]
].isna().any().any():
    raise RuntimeError(
        "Missing official endpoint label after alignment."
    )


# Exact cycle alignment check
cycle_diff_624fd = (
    scores_624fd["cycle"].to_numpy(dtype=np.float64)
    -
    scores_624fd[
        "last_observed_cycle"
    ].to_numpy(dtype=np.float64)
)

max_cycle_diff_624fd = float(
    np.max(np.abs(cycle_diff_624fd))
)

if max_cycle_diff_624fd != 0.0:
    raise RuntimeError(
        "Frozen inference cycle does not exactly match "
        "official endpoint cycle."
    )


# ------------------------------------------------------------
# 6. Add frozen-target and raw-NASA error columns
# ------------------------------------------------------------

scores_624fd["frozen_target"] = (
    scores_624fd[
        frozen_target_name_624fd
    ].astype(np.float64)
)

scores_624fd["frozen_target_error"] = (
    scores_624fd["prediction"].astype(np.float64)
    -
    scores_624fd["frozen_target"]
)

scores_624fd["frozen_target_abs_error"] = np.abs(
    scores_624fd["frozen_target_error"]
)

scores_624fd["raw_nasa_error"] = (
    scores_624fd["prediction"].astype(np.float64)
    -
    scores_624fd["official_final_RUL"].astype(np.float64)
)

scores_624fd["raw_nasa_abs_error"] = np.abs(
    scores_624fd["raw_nasa_error"]
)

scores_624fd["accept"] = (
    scores_624fd["accept"].astype(bool)
)

scores_624fd["decision"] = np.where(
    scores_624fd["accept"],
    "ACCEPT",
    "ABSTAIN",
)


# ------------------------------------------------------------
# 7. Verify frozen decision exactly
# ------------------------------------------------------------

reconstructed_accept_624fd = (
    scores_624fd[
        "reliability_score"
    ].to_numpy(dtype=np.float64)
    <= frozen_threshold_624fd
)

stored_accept_624fd = (
    scores_624fd[
        "accept"
    ].to_numpy(dtype=bool)
)

accept_mask_exact_624fd = bool(
    np.array_equal(
        reconstructed_accept_624fd,
        stored_accept_624fd,
    )
)

if not accept_mask_exact_624fd:
    raise RuntimeError(
        "Frozen accept mask does not reproduce exactly."
    )


accepted_count_624fd = int(
    stored_accept_624fd.sum()
)

abstained_count_624fd = int(
    len(stored_accept_624fd)
    -
    accepted_count_624fd
)

coverage_624fd = float(
    stored_accept_624fd.mean()
)


# ------------------------------------------------------------
# 8. Metric helper
# ------------------------------------------------------------

def endpoint_metrics_624fd(
    target,
    prediction,
):
    target = np.asarray(
        target,
        dtype=np.float64,
    ).reshape(-1)

    prediction = np.asarray(
        prediction,
        dtype=np.float64,
    ).reshape(-1)

    if len(target) != len(prediction):
        raise RuntimeError(
            "Target/prediction length mismatch."
        )

    if len(target) == 0:
        raise RuntimeError(
            "Cannot score empty endpoint subset."
        )

    error = prediction - target
    abs_error = np.abs(error)

    return {
        "n": int(len(target)),
        "MAE": float(
            np.mean(abs_error)
        ),
        "RMSE": float(
            np.sqrt(
                np.mean(
                    np.square(error)
                )
            )
        ),
        "median_AE": float(
            np.median(abs_error)
        ),
        "mean_error": float(
            np.mean(error)
        ),
    }


# ------------------------------------------------------------
# 9. Recompute frozen official metrics
# ------------------------------------------------------------

all_mask_624fd = np.ones(
    len(scores_624fd),
    dtype=bool,
)

accept_mask_624fd = (
    scores_624fd["accept"]
    .to_numpy(dtype=bool)
)

abstain_mask_624fd = (
    ~accept_mask_624fd
)


def subset_metrics_624fd(
    target_column,
    mask,
):
    return endpoint_metrics_624fd(
        scores_624fd.loc[
            mask,
            target_column,
        ],
        scores_624fd.loc[
            mask,
            "prediction",
        ],
    )


frozen_all_624fd = subset_metrics_624fd(
    "frozen_target",
    all_mask_624fd,
)

frozen_accept_624fd = subset_metrics_624fd(
    "frozen_target",
    accept_mask_624fd,
)

frozen_abstain_624fd = subset_metrics_624fd(
    "frozen_target",
    abstain_mask_624fd,
)

raw_all_624fd = subset_metrics_624fd(
    "official_final_RUL",
    all_mask_624fd,
)

raw_accept_624fd = subset_metrics_624fd(
    "official_final_RUL",
    accept_mask_624fd,
)

raw_abstain_624fd = subset_metrics_624fd(
    "official_final_RUL",
    abstain_mask_624fd,
)


frozen_mae_gain_624fd = float(
    (
        frozen_all_624fd["MAE"]
        -
        frozen_accept_624fd["MAE"]
    )
    /
    frozen_all_624fd["MAE"]
)

raw_mae_gain_624fd = float(
    (
        raw_all_624fd["MAE"]
        -
        raw_accept_624fd["MAE"]
    )
    /
    raw_all_624fd["MAE"]
)


# ------------------------------------------------------------
# 10. Finite-value integrity check
# ------------------------------------------------------------

numeric_score_cols_624fd = [
    "prediction",
    "mc_std",
    "reliability_score",
    "frozen_target",
    "official_final_RUL",
    "frozen_target_error",
    "frozen_target_abs_error",
    "raw_nasa_error",
    "raw_nasa_abs_error",
]

all_scores_finite_624fd = bool(
    np.isfinite(
        scores_624fd[
            numeric_score_cols_624fd
        ].to_numpy(dtype=np.float64)
    ).all()
)

if not all_scores_finite_624fd:
    raise RuntimeError(
        "Non-finite value found in clean official scoring table."
    )


# ------------------------------------------------------------
# 11. Compact summary object
# ------------------------------------------------------------

summary_624fd = {
    "dataset": "NASA C-MAPSS FD001",
    "evaluation": "official_clean_test_endpoints",
    "endpoint_engines": 100,

    "frozen_target": frozen_target_name_624fd,
    "reliability_signal": frozen_signal_name_624fd,
    "frozen_threshold": frozen_threshold_624fd,

    "accepted": accepted_count_624fd,
    "abstained": abstained_count_624fd,
    "coverage": coverage_624fd,

    "frozen_target_metrics": {
        "all": frozen_all_624fd,
        "accepted": frozen_accept_624fd,
        "abstained": frozen_abstain_624fd,
        "relative_MAE_gain": frozen_mae_gain_624fd,
    },

    "NASA_raw_RUL_metrics": {
        "all": raw_all_624fd,
        "accepted": raw_accept_624fd,
        "abstained": raw_abstain_624fd,
        "relative_MAE_gain": raw_mae_gain_624fd,
    },

    "integrity": {
        "endpoint_cycle_alignment_exact": True,
        "max_cycle_difference": max_cycle_diff_624fd,
        "frozen_accept_mask_exact": accept_mask_exact_624fd,
        "all_reported_values_finite": all_scores_finite_624fd,

        # Methodological lock:
        "reliability_rule_modified_after_test_scoring": False,
        "threshold_modified_after_test_scoring": False,
        "model_retrained_for_official_test": False,
        "official_labels_used_only_for_scoring": True,
    },
}


# ------------------------------------------------------------
# 12. Locate project root
# ------------------------------------------------------------

cwd_624fd = Path.cwd().resolve()

project_root_624fd = None

for candidate_624fd in [
    cwd_624fd,
    *cwd_624fd.parents,
]:
    if (
        candidate_624fd
        / "06_results"
    ).exists():
        project_root_624fd = (
            candidate_624fd
        )
        break

if project_root_624fd is None:
    raise FileNotFoundError(
        "Could not locate project root containing 06_results."
    )


output_dir_624fd = (
    project_root_624fd
    / "06_results"
    / "step6_official_fd001"
)

output_dir_624fd.mkdir(
    parents=True,
    exist_ok=True,
)


csv_path_624fd = (
    output_dir_624fd
    / "fd001_official_clean_endpoint_scores.csv"
)

summary_path_624fd = (
    output_dir_624fd
    / "fd001_official_clean_scoring_summary.json"
)

manifest_path_624fd = (
    output_dir_624fd
    / "fd001_official_clean_scoring_manifest.json"
)


print(
    "\nOutput directory:",
    output_dir_624fd,
)


# ------------------------------------------------------------
# 13. Deterministic serialization
# ------------------------------------------------------------

csv_bytes_624fd = (
    scores_624fd
    .to_csv(
        index=False,
        float_format="%.12g",
    )
    .encode("utf-8")
)

summary_bytes_624fd = (
    json.dumps(
        summary_624fd,
        indent=2,
        sort_keys=True,
        allow_nan=False,
    )
    .encode("utf-8")
)


def sha256_bytes_624fd(data):
    return hashlib.sha256(
        data
    ).hexdigest()


csv_hash_624fd = sha256_bytes_624fd(
    csv_bytes_624fd
)

summary_hash_624fd = sha256_bytes_624fd(
    summary_bytes_624fd
)


manifest_624fd = {
    "dataset": "NASA C-MAPSS FD001",
    "artifact_stage": "clean_official_endpoint_scoring",
    "files": {
        csv_path_624fd.name: {
            "sha256": csv_hash_624fd,
            "rows": int(len(scores_624fd)),
        },
        summary_path_624fd.name: {
            "sha256": summary_hash_624fd,
        },
    },
}

manifest_bytes_624fd = (
    json.dumps(
        manifest_624fd,
        indent=2,
        sort_keys=True,
        allow_nan=False,
    )
    .encode("utf-8")
)


# ------------------------------------------------------------
# 14. Write-once / verify-existing helper
# ------------------------------------------------------------

def write_once_or_verify_624fd(
    path,
    expected_bytes,
):
    if path.exists():
        existing_bytes = (
            path.read_bytes()
        )

        if existing_bytes != expected_bytes:
            raise RuntimeError(
                "Existing frozen artifact differs "
                f"from current result:\n{path}"
            )

        return "VERIFIED_EXISTING"

    path.write_bytes(
        expected_bytes
    )

    return "WRITTEN"


csv_status_624fd = (
    write_once_or_verify_624fd(
        csv_path_624fd,
        csv_bytes_624fd,
    )
)

summary_status_624fd = (
    write_once_or_verify_624fd(
        summary_path_624fd,
        summary_bytes_624fd,
    )
)

manifest_status_624fd = (
    write_once_or_verify_624fd(
        manifest_path_624fd,
        manifest_bytes_624fd,
    )
)


# ------------------------------------------------------------
# 15. Reload from disk and independently verify
# ------------------------------------------------------------

reloaded_scores_624fd = pd.read_csv(
    csv_path_624fd
)

with open(
    summary_path_624fd,
    "r",
    encoding="utf-8",
) as f:
    reloaded_summary_624fd = (
        json.load(f)
    )

with open(
    manifest_path_624fd,
    "r",
    encoding="utf-8",
) as f:
    reloaded_manifest_624fd = (
        json.load(f)
    )


disk_csv_hash_624fd = hashlib.sha256(
    csv_path_624fd.read_bytes()
).hexdigest()

disk_summary_hash_624fd = hashlib.sha256(
    summary_path_624fd.read_bytes()
).hexdigest()


csv_hash_valid_624fd = bool(
    disk_csv_hash_624fd
    ==
    reloaded_manifest_624fd[
        "files"
    ][
        csv_path_624fd.name
    ][
        "sha256"
    ]
)

summary_hash_valid_624fd = bool(
    disk_summary_hash_624fd
    ==
    reloaded_manifest_624fd[
        "files"
    ][
        summary_path_624fd.name
    ][
        "sha256"
    ]
)


if not (
    csv_hash_valid_624fd
    and summary_hash_valid_624fd
):
    raise RuntimeError(
        "Frozen clean-test artifact hash verification failed."
    )


if len(reloaded_scores_624fd) != 100:
    raise RuntimeError(
        "Reloaded clean scoring CSV does not contain 100 rows."
    )


# ------------------------------------------------------------
# 16. Display frozen clean official result
# ------------------------------------------------------------

print(
    "\n--- Frozen clean FD001 official result ---"
)

print(
    "Reliability signal       :",
    frozen_signal_name_624fd,
)

print(
    "Threshold                :",
    frozen_threshold_624fd,
)

print(
    "Frozen target            :",
    frozen_target_name_624fd,
)

print(
    "Accepted                 :",
    f"{accepted_count_624fd} / 100",
)

print(
    "Abstained                :",
    f"{abstained_count_624fd} / 100",
)

print(
    "Coverage                 :",
    coverage_624fd,
)

print(
    "Frozen-target full MAE   :",
    frozen_all_624fd["MAE"],
)

print(
    "Frozen-target selective MAE:",
    frozen_accept_624fd["MAE"],
)

print(
    "Frozen-target MAE gain   :",
    frozen_mae_gain_624fd,
)

print(
    "NASA-raw full MAE        :",
    raw_all_624fd["MAE"],
)

print(
    "NASA-raw selective MAE   :",
    raw_accept_624fd["MAE"],
)

print(
    "NASA-raw MAE gain        :",
    raw_mae_gain_624fd,
)


# ------------------------------------------------------------
# 17. Final integrity summary
# ------------------------------------------------------------

print(
    "\n=============================================="
)

print(
    "STEP 6.24F-D FREEZE / VERIFICATION SUMMARY"
)

print(
    "=============================================="
)

print(
    "Canonical score rows        :",
    len(scores_624fd),
)

print(
    "Unique endpoint engines     :",
    scores_624fd["unit_id"].nunique(),
)

print(
    "Endpoint alignment exact    :",
    max_cycle_diff_624fd == 0.0,
)

print(
    "Frozen accept mask exact    :",
    accept_mask_exact_624fd,
)

print(
    "All score values finite     :",
    all_scores_finite_624fd,
)

print(
    "CSV artifact                :",
    csv_status_624fd,
)

print(
    "Summary artifact            :",
    summary_status_624fd,
)

print(
    "Manifest artifact           :",
    manifest_status_624fd,
)

print(
    "CSV SHA-256 valid           :",
    csv_hash_valid_624fd,
)

print(
    "Summary SHA-256 valid       :",
    summary_hash_valid_624fd,
)

print(
    "Model inference rerun       : False"
)

print(
    "MC dropout rerun            : False"
)

print(
    "Threshold modified          : False"
)

print(
    "Reliability rule modified   : False"
)

print(
    "Official TEST retuned       : False"
)

print(
    "\nFrozen files:"
)

print(
    "1.",
    csv_path_624fd,
)

print(
    "2.",
    summary_path_624fd,
)

print(
    "3.",
    manifest_path_624fd,
)

print(
    "\n=== END ==="
)

In [ ]:
print("=== Step 6.24F-D COMPACT FINAL CHECK ===")

print(
    "Endpoint alignment exact :",
    max_cycle_diff_624fd == 0.0
)

print(
    "Frozen accept mask exact :",
    accept_mask_exact_624fd
)

print(
    "All score values finite  :",
    all_scores_finite_624fd
)

print(
    "CSV artifact status       :",
    csv_status_624fd
)

print(
    "Summary artifact status   :",
    summary_status_624fd
)

print(
    "Manifest artifact status  :",
    manifest_status_624fd
)

print(
    "CSV SHA-256 valid         :",
    csv_hash_valid_624fd
)

print(
    "Summary SHA-256 valid     :",
    summary_hash_valid_624fd
)

print(
    "Stored score rows         :",
    len(reloaded_scores_624fd)
)

print(
    "Unique engines            :",
    reloaded_scores_624fd["unit_id"].nunique()
)

print("\nNo inference rerun.")
print("No MC-dropout rerun.")
print("No threshold change.")
print("No reliability-rule change.")
print("=== END ===")

In [ ]:
# ============================================================
# Step 6.25A
# Official FD001 corruption-transfer readiness audit
# LABEL-FREE / NO MODEL INFERENCE / NO SCORING
# ============================================================

import numpy as np
import pandas as pd

print(
    "=== Step 6.25A — "
    "Official FD001 corruption-transfer readiness audit ==="
)

# ------------------------------------------------------------
# 1. Required frozen objects
# ------------------------------------------------------------

required_objects_625a = [
    "OFFICIAL_TEST_RAW_624B",
    "scenario_manifests_620gc",
    "apply_frozen_corruption_620gc",
    "PRIMARY_INPUT_COLUMNS",
    "scaler",
    "build_sequences_623cb2",
    "frozen_gru",
    "mc_uncertainty_623cb2",
    "RELOADED_RULE_623DL5",
    "RELOADED_THRESHOLD_623DL5",
    "RELOADED_TARGET_COVERAGE_623DL5",
]

print("\n--- Required frozen objects ---")

missing_objects_625a = []

for name_625a in required_objects_625a:

    present_625a = name_625a in globals()

    print(
        f"{name_625a:<35}: "
        f"{present_625a}"
    )

    if not present_625a:
        missing_objects_625a.append(
            name_625a
        )

if missing_objects_625a:

    raise RuntimeError(
        "Missing required frozen objects: "
        + str(
            missing_objects_625a
        )
    )


# ------------------------------------------------------------
# 2. Official TEST raw-base integrity
# ------------------------------------------------------------

test_raw_625a = (
    OFFICIAL_TEST_RAW_624B.copy()
)

print(
    "\n--- Official TEST raw structure ---"
)

print(
    "Rows              :",
    len(test_raw_625a)
)

print(
    "Columns           :",
    test_raw_625a.shape[1]
)

print(
    "Engines           :",
    test_raw_625a["unit_id"].nunique()
)

print(
    "Duplicate keys    :",
    int(
        test_raw_625a.duplicated(
            ["unit_id", "cycle"]
        ).sum()
    )
)

numeric_test_625a = (
    test_raw_625a.select_dtypes(
        include=[np.number]
    )
)

print(
    "All numeric finite:",
    bool(
        np.isfinite(
            numeric_test_625a.to_numpy(
                dtype=np.float64
            )
        ).all()
    )
)


# ------------------------------------------------------------
# 3. Primary-model feature compatibility
# ------------------------------------------------------------

missing_primary_625a = [
    col_625a
    for col_625a
    in PRIMARY_INPUT_COLUMNS
    if col_625a
    not in test_raw_625a.columns
]

print(
    "\n--- Primary feature compatibility ---"
)

print(
    "Primary feature count :",
    len(
        PRIMARY_INPUT_COLUMNS
    )
)

print(
    "Missing primary inputs:",
    missing_primary_625a
)

if missing_primary_625a:

    raise RuntimeError(
        "Official TEST is missing "
        "primary model features."
    )


# ------------------------------------------------------------
# 4. Frozen corruption-manifest inventory
# ------------------------------------------------------------

print(
    "\n--- Frozen corruption manifests ---"
)

manifest_inventory_625a = []

for family_625a, manifest_625a \
    in scenario_manifests_620gc.items():

    if not isinstance(
        manifest_625a,
        pd.DataFrame
    ):

        raise TypeError(
            f"{family_625a} manifest is "
            f"{type(manifest_625a).__name__}, "
            "expected DataFrame."
        )

    manifest_inventory_625a.append(
        {
            "family":
                family_625a,

            "scenarios":
                len(
                    manifest_625a
                ),
        }
    )

manifest_inventory_625a = (
    pd.DataFrame(
        manifest_inventory_625a
    )
)

display(
    manifest_inventory_625a
)

total_scenarios_625a = int(
    manifest_inventory_625a[
        "scenarios"
    ].sum()
)

print(
    "Total frozen scenarios:",
    total_scenarios_625a
)


# ------------------------------------------------------------
# 5. Sensor-name compatibility
# ------------------------------------------------------------

sensor_records_625a = []

for family_625a, manifest_625a \
    in scenario_manifests_620gc.items():

    if "sensor" not in \
        manifest_625a.columns:

        raise RuntimeError(
            f"'sensor' column missing "
            f"from {family_625a} manifest."
        )

    for sensor_625a in sorted(
        manifest_625a[
            "sensor"
        ]
        .astype(str)
        .unique()
    ):

        sensor_records_625a.append(
            {
                "family":
                    family_625a,

                "sensor":
                    sensor_625a,

                "exists_in_raw_test":
                    sensor_625a
                    in test_raw_625a.columns,

                "primary_model_input":
                    sensor_625a
                    in PRIMARY_INPUT_COLUMNS,
            }
        )

sensor_check_625a = (
    pd.DataFrame(
        sensor_records_625a
    )
    .drop_duplicates()
    .reset_index(
        drop=True
    )
)

print(
    "\n--- Frozen corruption sensor compatibility ---"
)

display(
    sensor_check_625a
)

missing_corruption_sensors_625a = (
    sensor_check_625a.loc[
        ~sensor_check_625a[
            "exists_in_raw_test"
        ],
        "sensor",
    ]
    .unique()
    .tolist()
)

print(
    "Corruption sensors missing "
    "from TEST:",
    missing_corruption_sensors_625a
)

if missing_corruption_sensors_625a:

    raise RuntimeError(
        "Frozen corruption sensors "
        "are missing from official TEST."
    )


# ------------------------------------------------------------
# 6. Frozen decision lock
# ------------------------------------------------------------

print(
    "\n--- Frozen reliability decision ---"
)

print(
    "Frozen rule:",
    RELOADED_RULE_623DL5
)

print(
    "Frozen threshold:",
    RELOADED_THRESHOLD_623DL5
)

print(
    "Frozen target coverage:",
    RELOADED_TARGET_COVERAGE_623DL5
)


# ------------------------------------------------------------
# 7. Safety / leakage assertions
# ------------------------------------------------------------

print(
    "\n--- Safety status ---"
)

print(
    "Official TEST labels "
    "used for tuning : False"
)

print(
    "Model inference "
    "executed         : False"
)

print(
    "MC-dropout "
    "executed         : False"
)

print(
    "Corruptions "
    "executed         : False"
)

print(
    "Threshold "
    "modified         : False"
)

print(
    "Reliability rule "
    "modified         : False"
)

print(
    "Files written    : False"
)


# ------------------------------------------------------------
# 8. Final readiness result
# ------------------------------------------------------------

ready_625a = bool(
    total_scenarios_625a == 315
    and len(
        missing_primary_625a
    ) == 0
    and len(
        missing_corruption_sensors_625a
    ) == 0
    and test_raw_625a[
        "unit_id"
    ].nunique() == 100
    and int(
        test_raw_625a.duplicated(
            ["unit_id", "cycle"]
        ).sum()
    ) == 0
)

print(
    "\n========================================"
)

print(
    "STEP 6.25A READINESS SUMMARY"
)

print(
    "========================================"
)

print(
    "Official TEST raw rows       :",
    len(
        test_raw_625a
    )
)

print(
    "Official TEST engines        :",
    test_raw_625a[
        "unit_id"
    ].nunique()
)

print(
    "Frozen corruption scenarios  :",
    total_scenarios_625a
)

print(
    "Primary features compatible  :",
    len(
        missing_primary_625a
    ) == 0
)

print(
    "Corruption sensors available :",
    len(
        missing_corruption_sensors_625a
    ) == 0
)

print(
    "Frozen decision unchanged    : True"
)

print(
    "OFFICIAL TEST TRANSFER READY :",
    ready_625a
)

print(
    "\nNo inference was run."
)

print(
    "No corruption was applied."
)

print(
    "No official labels were "
    "used for any decision."
)

print(
    "=== END ==="
)

In [ ]:
# ============================================================
# Step 6.25B0
# Frozen corruption-helper / manifest contract inspection
# READ-ONLY / LABEL-FREE / NO CORRUPTION EXECUTION
# ============================================================

import inspect
import pandas as pd

print(
    "=== Step 6.25B0 — "
    "Frozen corruption-helper / manifest contract inspection ==="
)

# ------------------------------------------------------------
# 1. Helper identity
# ------------------------------------------------------------

helper_name_625b0 = "apply_frozen_corruption_620gc"

print("\n--- Corruption helper ---")

print(
    "Available :",
    helper_name_625b0 in globals()
)

if helper_name_625b0 not in globals():
    raise RuntimeError(
        f"{helper_name_625b0} is not available."
    )

helper_625b0 = globals()[
    helper_name_625b0
]

print(
    "Type      :",
    type(
        helper_625b0
    ).__name__
)

print(
    "Callable  :",
    callable(
        helper_625b0
    )
)

print(
    "Signature :",
    inspect.signature(
        helper_625b0
    )
)


# ------------------------------------------------------------
# 2. Complete helper source
# ------------------------------------------------------------

print(
    "\n--- Complete helper source ---"
)

try:
    helper_source_625b0 = (
        inspect.getsource(
            helper_625b0
        )
    )

    print(
        helper_source_625b0
    )

except Exception as exc_625b0:

    print(
        "Could not recover source:",
        repr(
            exc_625b0
        )
    )


# ------------------------------------------------------------
# 3. Manifest-family structure
# ------------------------------------------------------------

print(
    "\n--- Manifest family structure ---"
)

manifest_summary_rows_625b0 = []

for family_625b0, manifest_625b0 \
    in scenario_manifests_620gc.items():

    manifest_summary_rows_625b0.append(
        {
            "family":
                family_625b0,

            "type":
                type(
                    manifest_625b0
                ).__name__,

            "rows":
                len(
                    manifest_625b0
                )
                if hasattr(
                    manifest_625b0,
                    "__len__"
                )
                else None,

            "columns":
                list(
                    manifest_625b0.columns
                )
                if isinstance(
                    manifest_625b0,
                    pd.DataFrame
                )
                else None,
        }
    )

MANIFEST_STRUCTURE_625B0 = (
    pd.DataFrame(
        manifest_summary_rows_625b0
    )
)

display(
    MANIFEST_STRUCTURE_625B0
)


# ------------------------------------------------------------
# 4. One representative row from each family
# ------------------------------------------------------------

print(
    "\n--- Representative frozen scenario rows ---"
)

for family_625b0, manifest_625b0 \
    in scenario_manifests_620gc.items():

    print(
        f"\nFAMILY: {family_625b0}"
    )

    if not isinstance(
        manifest_625b0,
        pd.DataFrame
    ):
        print(
            "Not a DataFrame."
        )
        continue

    if len(
        manifest_625b0
    ) == 0:
        print(
            "Manifest is empty."
        )
        continue

    display(
        manifest_625b0
        .iloc[[0]]
        .copy()
    )


# ------------------------------------------------------------
# 5. Column intersection across all manifests
# ------------------------------------------------------------

manifest_frames_625b0 = [
    df_625b0
    for df_625b0
    in scenario_manifests_620gc.values()
    if isinstance(
        df_625b0,
        pd.DataFrame
    )
]

if not manifest_frames_625b0:
    raise RuntimeError(
        "No DataFrame corruption manifests found."
    )

common_columns_625b0 = set(
    manifest_frames_625b0[0].columns
)

for df_625b0 in manifest_frames_625b0[1:]:
    common_columns_625b0 &= set(
        df_625b0.columns
    )

common_columns_625b0 = sorted(
    common_columns_625b0
)

print(
    "\n--- Common manifest columns ---"
)

print(
    common_columns_625b0
)


# ------------------------------------------------------------
# 6. Compact scenario-ID inspection
# ------------------------------------------------------------

print(
    "\n--- Scenario ID samples ---"
)

for family_625b0, manifest_625b0 \
    in scenario_manifests_620gc.items():

    if (
        isinstance(
            manifest_625b0,
            pd.DataFrame
        )
        and
        "scenario_id"
        in manifest_625b0.columns
    ):

        values_625b0 = (
            manifest_625b0[
                "scenario_id"
            ]
            .astype(str)
            .head(3)
            .tolist()
        )

        print(
            f"{family_625b0:<8}:",
            values_625b0
        )


# ------------------------------------------------------------
# 7. Safety summary
# ------------------------------------------------------------

print(
    "\n========================================"
)

print(
    "STEP 6.25B0 INSPECTION SUMMARY"
)

print(
    "========================================"
)

print(
    "Corruption executed       : False"
)

print(
    "Model inference executed  : False"
)

print(
    "MC-dropout executed       : False"
)

print(
    "Official TEST scored      : False"
)

print(
    "Official labels inspected : False"
)

print(
    "Frozen rule modified      : False"
)

print(
    "Threshold modified        : False"
)

print(
    "Files written             : False"
)

print(
    "\nNothing was modified."
)

print(
    "=== END ==="
)

In [ ]:
# ============================================================
# Step 6.25B1
# Exact frozen-corruption helper dependency inspection
# READ-ONLY / NO CORRUPTION / NO INFERENCE
# ============================================================

import inspect
import types

print(
    "=== Step 6.25B1 — "
    "Exact frozen-corruption helper dependency inspection ==="
)

helper_625b1 = apply_frozen_corruption_620gc


# ------------------------------------------------------------
# 1. Closure / namespace dependencies
# ------------------------------------------------------------

print("\n--- Closure variables ---")

closure_625b1 = inspect.getclosurevars(
    helper_625b1
)

print(
    "Nonlocals:",
    sorted(
        closure_625b1.nonlocals.keys()
    )
)

print(
    "Globals:",
    sorted(
        closure_625b1.globals.keys()
    )
)

print(
    "Unbound:",
    sorted(
        closure_625b1.unbound
    )
)


# ------------------------------------------------------------
# 2. Compact type/shape inspection of referenced globals
# ------------------------------------------------------------

print(
    "\n--- Referenced global objects ---"
)

dependency_rows_625b1 = []

for name_625b1, obj_625b1 \
    in closure_625b1.globals.items():

    shape_625b1 = getattr(
        obj_625b1,
        "shape",
        None
    )

    dependency_rows_625b1.append(
        {
            "name":
                name_625b1,

            "type":
                type(
                    obj_625b1
                ).__name__,

            "shape":
                shape_625b1,

            "callable":
                callable(
                    obj_625b1
                ),
        }
    )

DEPENDENCIES_625B1 = (
    pd.DataFrame(
        dependency_rows_625b1
    )
    .sort_values(
        "name"
    )
    .reset_index(
        drop=True
    )
)

display(
    DEPENDENCIES_625B1
)


# ------------------------------------------------------------
# 3. Recover source and print only important lines
# ------------------------------------------------------------

source_625b1 = inspect.getsource(
    helper_625b1
)

source_lines_625b1 = (
    source_625b1.splitlines()
)

important_tokens_625b1 = [
    "dataframe",
    "data_frame",
    "_df",
    "raw",
    "dev",
    "val",
    "train",
    "test",
    "canonical",
    "copy",
    "bias",
    "noise",
    "drift",
    "dropout",
    "stuck",
    "corrupt",
    "generator",
]

print(
    "\n--- Relevant helper-source lines ---"
)

for i_625b1, line_625b1 \
    in enumerate(
        source_lines_625b1,
        start=1
    ):

    low_625b1 = (
        line_625b1.lower()
    )

    if any(
        token_625b1 in low_625b1
        for token_625b1
        in important_tokens_625b1
    ):

        lo_625b1 = max(
            1,
            i_625b1 - 2
        )

        hi_625b1 = min(
            len(
                source_lines_625b1
            ),
            i_625b1 + 2
        )

        print(
            "\n"
            + "-" * 60
        )

        for j_625b1 in range(
            lo_625b1,
            hi_625b1 + 1
        ):

            marker_625b1 = (
                ">>>"
                if j_625b1 == i_625b1
                else "   "
            )

            print(
                f"{marker_625b1} "
                f"{j_625b1:03d}: "
                f"{source_lines_625b1[j_625b1 - 1]}"
            )


# ------------------------------------------------------------
# 4. Identify DataFrame-like dependencies specifically
# ------------------------------------------------------------

print(
    "\n--- DataFrame-like dependencies ---"
)

df_dependencies_625b1 = []

for name_625b1, obj_625b1 \
    in closure_625b1.globals.items():

    if isinstance(
        obj_625b1,
        pd.DataFrame
    ):

        df_dependencies_625b1.append(
            {
                "name":
                    name_625b1,

                "shape":
                    obj_625b1.shape,

                "first_columns":
                    list(
                        obj_625b1.columns[:8]
                    ),
            }
        )

DF_DEPENDENCIES_625B1 = (
    pd.DataFrame(
        df_dependencies_625b1
    )
)

if len(
    DF_DEPENDENCIES_625B1
) > 0:

    display(
        DF_DEPENDENCIES_625B1
    )

else:

    print(
        "No directly referenced pandas DataFrames."
    )


# ------------------------------------------------------------
# 5. Safety summary
# ------------------------------------------------------------

print(
    "\n========================================"
)

print(
    "STEP 6.25B1 INSPECTION SUMMARY"
)

print(
    "========================================"
)

print(
    "Corruption executed       : False"
)

print(
    "Model inference executed  : False"
)

print(
    "MC-dropout executed       : False"
)

print(
    "Official TEST scored      : False"
)

print(
    "Official labels used      : False"
)

print(
    "Frozen rule modified      : False"
)

print(
    "Threshold modified        : False"
)

print(
    "Files written             : False"
)

print(
    "\nNothing was modified."
)

print(
    "=== END ==="
)

In [ ]:
# ============================================================
# Step 6.25B2
# Underlying frozen-corruption generator contract inspection
# READ-ONLY / NO CORRUPTION / NO INFERENCE / NO TEST SCORING
# ============================================================

import inspect
import pandas as pd
import numpy as np

print(
    "=== Step 6.25B2 — "
    "Underlying frozen-corruption generator contract inspection ==="
)


# ------------------------------------------------------------
# 1. Frozen corruption-related functions
# ------------------------------------------------------------

generators_625b2 = {
    "bias":
        frozen_add_sensor_bias_620gc,

    "noise":
        frozen_add_sensor_noise_620gc,

    "drift":
        frozen_add_sensor_drift_620gc,

    "dropout":
        candidate_add_sensor_dropout_617d,

    "stuck":
        candidate_add_stuck_sensor_618d,

    "extractor":
        extract_corrupted_dataframe_620gc,
}


# ------------------------------------------------------------
# 2. Signature + closure dependency audit
# ------------------------------------------------------------

contract_rows_625b2 = []

for name_625b2, func_625b2 \
    in generators_625b2.items():

    closure_625b2 = inspect.getclosurevars(
        func_625b2
    )

    dataframe_globals_625b2 = []

    for global_name_625b2, global_obj_625b2 \
        in closure_625b2.globals.items():

        if isinstance(
            global_obj_625b2,
            pd.DataFrame
        ):
            dataframe_globals_625b2.append(
                (
                    global_name_625b2,
                    global_obj_625b2.shape
                )
            )

    contract_rows_625b2.append(
        {
            "family/helper":
                name_625b2,

            "function_name":
                func_625b2.__name__,

            "signature":
                str(
                    inspect.signature(
                        func_625b2
                    )
                ),

            "nonlocal_names":
                sorted(
                    closure_625b2.nonlocals.keys()
                ),

            "global_dataframe_dependencies":
                dataframe_globals_625b2,

            "unbound_names":
                sorted(
                    closure_625b2.unbound
                ),
        }
    )


GENERATOR_CONTRACTS_625B2 = pd.DataFrame(
    contract_rows_625b2
)

print(
    "\n--- Generator contracts ---"
)

display(
    GENERATOR_CONTRACTS_625B2
)


# ------------------------------------------------------------
# 3. Exact locations where apply_frozen_corruption_620gc
#    references DEV data or calls the generators
# ------------------------------------------------------------

print(
    "\n--- Exact wrapper dependency/call lines ---"
)

wrapper_source_625b2 = inspect.getsource(
    apply_frozen_corruption_620gc
)

wrapper_lines_625b2 = (
    wrapper_source_625b2.splitlines()
)

wrapper_tokens_625b2 = [
    "val_raw_base_620gc",
    "sensor_reference_stats",
    "frozen_add_sensor_bias_620gc",
    "frozen_add_sensor_noise_620gc",
    "frozen_add_sensor_drift_620gc",
    "candidate_add_sensor_dropout_617d",
    "candidate_add_stuck_sensor_618d",
    "extract_corrupted_dataframe_620gc",
]

printed_wrapper_lines_625b2 = set()

for i_625b2, line_625b2 \
    in enumerate(
        wrapper_lines_625b2,
        start=1
    ):

    if any(
        token_625b2 in line_625b2
        for token_625b2
        in wrapper_tokens_625b2
    ):

        lo_625b2 = max(
            1,
            i_625b2 - 3
        )

        hi_625b2 = min(
            len(
                wrapper_lines_625b2
            ),
            i_625b2 + 5
        )

        for j_625b2 in range(
            lo_625b2,
            hi_625b2 + 1
        ):

            printed_wrapper_lines_625b2.add(
                j_625b2
            )


for j_625b2 in sorted(
    printed_wrapper_lines_625b2
):

    marker_625b2 = (
        ">>>"
        if any(
            token_625b2
            in wrapper_lines_625b2[
                j_625b2 - 1
            ]
            for token_625b2
            in wrapper_tokens_625b2
        )
        else "   "
    )

    print(
        f"{marker_625b2} "
        f"{j_625b2:03d}: "
        f"{wrapper_lines_625b2[j_625b2 - 1]}"
    )


# ------------------------------------------------------------
# 4. sensor_reference_stats structure
# ------------------------------------------------------------

print(
    "\n--- Frozen sensor_reference_stats ---"
)

print(
    "Type:",
    type(
        sensor_reference_stats
    ).__name__
)

print(
    "Shape:",
    sensor_reference_stats.shape
)

print(
    "Index:",
    list(
        sensor_reference_stats.index
    )
)

print(
    "Columns:",
    list(
        sensor_reference_stats.columns
    )
)

display(
    sensor_reference_stats.head()
)


# ------------------------------------------------------------
# 5. Search notebook-accessible objects for likely source
#    of sensor_reference_stats WITHOUT recalculating anything
# ------------------------------------------------------------

print(
    "\n--- Likely sensor-reference provenance objects ---"
)

provenance_rows_625b2 = []

for obj_name_625b2, obj_625b2 \
    in list(
        globals().items()
    ):

    name_lower_625b2 = (
        obj_name_625b2.lower()
    )

    if (
        "sensor_reference"
        in name_lower_625b2
        or
        "reference_stats"
        in name_lower_625b2
    ):

        provenance_rows_625b2.append(
            {
                "object_name":
                    obj_name_625b2,

                "type":
                    type(
                        obj_625b2
                    ).__name__,

                "shape":
                    getattr(
                        obj_625b2,
                        "shape",
                        None
                    ),
            }
        )


PROVENANCE_OBJECTS_625B2 = pd.DataFrame(
    provenance_rows_625b2
)

if len(
    PROVENANCE_OBJECTS_625B2
) > 0:

    display(
        PROVENANCE_OBJECTS_625B2
        .sort_values(
            "object_name"
        )
        .reset_index(
            drop=True
        )
    )

else:

    print(
        "No additional sensor-reference objects found."
    )


# ------------------------------------------------------------
# 6. Integrity summary
# ------------------------------------------------------------

print(
    "\n=========================================="
)

print(
    "STEP 6.25B2 INSPECTION SUMMARY"
)

print(
    "=========================================="
)

print(
    "Corruption executed              : False"
)

print(
    "Official TEST data modified      : False"
)

print(
    "Model inference executed         : False"
)

print(
    "MC-dropout executed              : False"
)

print(
    "Official TEST scored             : False"
)

print(
    "Official labels used             : False"
)

print(
    "Frozen manifests modified        : False"
)

print(
    "Frozen reliability rule modified : False"
)

print(
    "Frozen threshold modified        : False"
)

print(
    "Files written                    : False"
)

print(
    "\nNothing was modified."
)

print(
    "=== END ==="
)

In [ ]:
# ============================================================
# Step 6.25B3
# Frozen generator return-contract source +
# sensor-reference provenance inspection
#
# READ-ONLY
# NO CORRUPTION
# NO MODEL INFERENCE
# NO OFFICIAL TEST SCORING
# ============================================================

import inspect
from pathlib import Path
import pandas as pd
import numpy as np

print(
    "=== Step 6.25B3 — "
    "Generator return-contract and sensor-reference provenance audit ==="
)


# ------------------------------------------------------------
# 1. Inspect return-related source lines of each generator
# ------------------------------------------------------------

generator_functions_625b3 = {
    "bias":
        frozen_add_sensor_bias_620gc,

    "noise":
        frozen_add_sensor_noise_620gc,

    "drift":
        frozen_add_sensor_drift_620gc,

    "dropout":
        candidate_add_sensor_dropout_617d,

    "stuck":
        candidate_add_stuck_sensor_618d,

    "extractor":
        extract_corrupted_dataframe_620gc,
}


print(
    "\n--- Generator return-related source ---"
)

for family_625b3, func_625b3 \
    in generator_functions_625b3.items():

    print(
        "\n"
        + "=" * 70
    )

    print(
        "HELPER:",
        family_625b3,
        "|",
        func_625b3.__name__
    )

    print(
        "Signature:",
        inspect.signature(
            func_625b3
        )
    )

    print(
        "=" * 70
    )

    source_625b3 = inspect.getsource(
        func_625b3
    )

    lines_625b3 = (
        source_625b3.splitlines()
    )

    relevant_indices_625b3 = []

    search_terms_625b3 = [
        "return ",
        "result",
        "corrupted",
        "DataFrame",
        "tuple",
        "dict",
    ]

    for i_625b3, line_625b3 \
        in enumerate(
            lines_625b3,
            start=1
        ):

        stripped_625b3 = (
            line_625b3.strip()
        )

        if (
            stripped_625b3.startswith(
                "return"
            )
            or
            any(
                term_625b3
                in line_625b3
                for term_625b3
                in search_terms_625b3
            )
        ):

            relevant_indices_625b3.append(
                i_625b3
            )

    selected_lines_625b3 = set()

    for i_625b3 in relevant_indices_625b3:

        lo_625b3 = max(
            1,
            i_625b3 - 3
        )

        hi_625b3 = min(
            len(
                lines_625b3
            ),
            i_625b3 + 5
        )

        selected_lines_625b3.update(
            range(
                lo_625b3,
                hi_625b3 + 1
            )
        )

    if not selected_lines_625b3:

        print(
            "No return-related lines detected."
        )

    else:

        for j_625b3 in sorted(
            selected_lines_625b3
        ):

            marker_625b3 = (
                ">>>"
                if lines_625b3[
                    j_625b3 - 1
                ]
                .strip()
                .startswith(
                    "return"
                )
                else "   "
            )

            print(
                f"{marker_625b3} "
                f"{j_625b3:03d}: "
                f"{lines_625b3[j_625b3 - 1]}"
            )


# ------------------------------------------------------------
# 2. Full extractor source
#
# This is especially important because B2 showed that the
# extractor itself depends on val_raw_base_620gc.
# ------------------------------------------------------------

print(
    "\n\n--- Complete existing extractor source ---"
)

extractor_source_625b3 = inspect.getsource(
    extract_corrupted_dataframe_620gc
)

print(
    extractor_source_625b3
)


# ------------------------------------------------------------
# 3. SENSOR_REFERENCE_PATH inspection
# ------------------------------------------------------------

print(
    "\n--- SENSOR_REFERENCE_PATH ---"
)

if "SENSOR_REFERENCE_PATH" \
    in globals():

    path_625b3 = Path(
        SENSOR_REFERENCE_PATH
    )

    print(
        "Object type :",
        type(
            SENSOR_REFERENCE_PATH
        ).__name__
    )

    print(
        "Path        :",
        path_625b3
    )

    print(
        "Exists      :",
        path_625b3.exists()
    )

    if path_625b3.exists():

        print(
            "Suffix      :",
            path_625b3.suffix
        )

        print(
            "Size bytes  :",
            path_625b3.stat().st_size
        )

else:

    print(
        "SENSOR_REFERENCE_PATH is not available."
    )


# ------------------------------------------------------------
# 4. Search notebook cell history/source for creation of
#    sensor_reference_stats and SENSOR_REFERENCE_PATH
# ------------------------------------------------------------

print(
    "\n--- Notebook-history provenance search ---"
)

history_hits_625b3 = []

try:

    ip_625b3 = get_ipython()

    history_manager_625b3 = (
        ip_625b3.history_manager
    )

    for (
        session_625b3,
        line_no_625b3,
        source_625b3
    ) in history_manager_625b3.get_range(
        output=False,
        raw=True
    ):

        if (
            "sensor_reference_stats"
            in source_625b3
            or
            "SENSOR_REFERENCE_PATH"
            in source_625b3
        ):

            history_hits_625b3.append(
                {
                    "session":
                        session_625b3,

                    "cell":
                        line_no_625b3,

                    "source":
                        source_625b3,
                }
            )

except Exception as exc_625b3:

    print(
        "History search unavailable:",
        repr(
            exc_625b3
        )
    )


print(
    "Matching history cells:",
    len(
        history_hits_625b3
    )
)

for hit_625b3 in history_hits_625b3:

    print(
        "\n"
        + "-" * 70
    )

    print(
        "SESSION:",
        hit_625b3[
            "session"
        ],
        "CELL:",
        hit_625b3[
            "cell"
        ]
    )

    src_lines_625b3 = (
        hit_625b3[
            "source"
        ]
        .splitlines()
    )

    matching_line_numbers_625b3 = []

    for i_625b3, line_625b3 \
        in enumerate(
            src_lines_625b3,
            start=1
        ):

        if (
            "sensor_reference_stats"
            in line_625b3
            or
            "SENSOR_REFERENCE_PATH"
            in line_625b3
        ):

            matching_line_numbers_625b3.append(
                i_625b3
            )

    context_lines_625b3 = set()

    for i_625b3 \
        in matching_line_numbers_625b3:

        lo_625b3 = max(
            1,
            i_625b3 - 8
        )

        hi_625b3 = min(
            len(
                src_lines_625b3
            ),
            i_625b3 + 12
        )

        context_lines_625b3.update(
            range(
                lo_625b3,
                hi_625b3 + 1
            )
        )

    for j_625b3 in sorted(
        context_lines_625b3
    ):

        marker_625b3 = (
            ">>>"
            if (
                "sensor_reference_stats"
                in src_lines_625b3[
                    j_625b3 - 1
                ]
                or
                "SENSOR_REFERENCE_PATH"
                in src_lines_625b3[
                    j_625b3 - 1
                ]
            )
            else "   "
        )

        print(
            f"{marker_625b3} "
            f"{j_625b3:03d}: "
            f"{src_lines_625b3[j_625b3 - 1]}"
        )


# ------------------------------------------------------------
# 5. If the reference artifact is a CSV, reload it read-only
#    and compare against the in-memory frozen table.
#
# No table is recomputed here.
# ------------------------------------------------------------

print(
    "\n--- Stored sensor-reference artifact comparison ---"
)

reference_file_match_625b3 = None

if (
    "SENSOR_REFERENCE_PATH"
    in globals()
):

    path_625b3 = Path(
        SENSOR_REFERENCE_PATH
    )

    if (
        path_625b3.exists()
        and
        path_625b3.suffix.lower()
        == ".csv"
    ):

        stored_reference_625b3 = (
            pd.read_csv(
                path_625b3,
                index_col=0
            )
        )

        print(
            "Stored shape:",
            stored_reference_625b3.shape
        )

        print(
            "Memory shape:",
            sensor_reference_stats.shape
        )

        same_index_625b3 = (
            list(
                stored_reference_625b3.index
            )
            ==
            list(
                sensor_reference_stats.index
            )
        )

        same_columns_625b3 = (
            list(
                stored_reference_625b3.columns
            )
            ==
            list(
                sensor_reference_stats.columns
            )
        )

        print(
            "Index exact:",
            same_index_625b3
        )

        print(
            "Columns exact:",
            same_columns_625b3
        )

        if (
            same_index_625b3
            and
            same_columns_625b3
            and
            stored_reference_625b3.shape
            ==
            sensor_reference_stats.shape
        ):

            max_reference_diff_625b3 = (
                np.max(
                    np.abs(
                        stored_reference_625b3
                        .to_numpy(
                            dtype=np.float64
                        )
                        -
                        sensor_reference_stats
                        .to_numpy(
                            dtype=np.float64
                        )
                    )
                )
            )

            reference_file_match_625b3 = (
                max_reference_diff_625b3
                == 0.0
            )

            print(
                "Maximum numeric difference:",
                max_reference_diff_625b3
            )

            print(
                "Stored/memory exact:",
                reference_file_match_625b3
            )

    else:

        print(
            "Reference path is absent or is not a CSV;"
            " no reload comparison performed."
        )


# ------------------------------------------------------------
# 6. Integrity summary
# ------------------------------------------------------------

print(
    "\n=========================================="
)

print(
    "STEP 6.25B3 INSPECTION SUMMARY"
)

print(
    "=========================================="
)

print(
    "Corruption executed               : False"
)

print(
    "Official TEST modified            : False"
)

print(
    "Model inference executed          : False"
)

print(
    "MC-dropout executed               : False"
)

print(
    "Official TEST scored              : False"
)

print(
    "Official labels used              : False"
)

print(
    "Sensor statistics recomputed      : False"
)

print(
    "Frozen corruption manifests changed: False"
)

print(
    "Frozen reliability rule changed  : False"
)

print(
    "Threshold changed                 : False"
)

print(
    "Files written                     : False"
)

print(
    "\nNothing was modified."
)

print(
    "=== END ==="
)

In [ ]:
# ============================================================
# Step 6.25B3A
# Robust generator return-contract + provenance recovery
#
# READ-ONLY
# NO CORRUPTION
# NO MODEL INFERENCE
# NO MC DROPOUT
# NO OFFICIAL TEST SCORING
# NO FILE WRITES
# ============================================================

import inspect
import ast
import re
from pathlib import Path
import pandas as pd
import numpy as np


print(
    "=== Step 6.25B3A — Robust generator source "
    "and sensor-reference provenance recovery ==="
)


# ------------------------------------------------------------
# 1. Obtain notebook history once
# ------------------------------------------------------------

history_records_625b3a = []

try:

    ip_625b3a = get_ipython()

    for (
        session_625b3a,
        cell_625b3a,
        source_625b3a
    ) in ip_625b3a.history_manager.get_range(
        output=False,
        raw=True
    ):

        history_records_625b3a.append(
            (
                session_625b3a,
                cell_625b3a,
                source_625b3a,
            )
        )

    print(
        "\nNotebook history records available:",
        len(history_records_625b3a)
    )

except Exception as exc_625b3a:

    print(
        "\nNotebook history recovery failed:",
        repr(exc_625b3a)
    )


# ------------------------------------------------------------
# 2. Robust source recovery helper
# ------------------------------------------------------------

def recover_function_source_625b3a(func):

    func_name = func.__name__

    # First try standard inspect
    try:

        src = inspect.getsource(func)

        return {
            "source": src,
            "origin": "inspect.getsource",
            "session": None,
            "cell": None,
        }

    except Exception:

        pass

    # Fallback: search notebook execution history,
    # newest matching definition first
    for (
        session_id,
        cell_id,
        cell_source
    ) in reversed(history_records_625b3a):

        if (
            f"def {func_name}("
            not in cell_source
            and
            f"def {func_name} ("
            not in cell_source
        ):

            continue

        # Try AST extraction
        try:

            tree = ast.parse(
                cell_source
            )

            matches = [
                node
                for node in ast.walk(tree)
                if (
                    isinstance(
                        node,
                        (
                            ast.FunctionDef,
                            ast.AsyncFunctionDef,
                        )
                    )
                    and
                    node.name == func_name
                )
            ]

            if matches:

                node = matches[-1]

                lines = (
                    cell_source
                    .splitlines()
                )

                start = (
                    node.lineno - 1
                )

                end = getattr(
                    node,
                    "end_lineno",
                    None
                )

                if end is not None:

                    recovered = "\n".join(
                        lines[
                            start:end
                        ]
                    )

                    return {
                        "source": recovered,
                        "origin":
                            "notebook_history_ast",
                        "session":
                            session_id,
                        "cell":
                            cell_id,
                    }

        except Exception:

            pass

        # Plain-text fallback
        lines = (
            cell_source
            .splitlines()
        )

        start_idx = None

        for i, line in enumerate(lines):

            if re.match(
                rf"^\s*def\s+{re.escape(func_name)}\s*\(",
                line
            ):

                start_idx = i
                break

        if start_idx is not None:

            def_indent = (
                len(lines[start_idx])
                -
                len(
                    lines[start_idx]
                    .lstrip()
                )
            )

            collected = []

            for j in range(
                start_idx,
                len(lines)
            ):

                line = lines[j]

                if (
                    j > start_idx
                    and
                    line.strip()
                    and
                    (
                        len(line)
                        -
                        len(
                            line.lstrip()
                        )
                    )
                    <= def_indent
                    and
                    re.match(
                        r"^\s*(def|class)\s+",
                        line
                    )
                ):

                    break

                collected.append(
                    line
                )

            return {
                "source":
                    "\n".join(
                        collected
                    ),

                "origin":
                    "notebook_history_text",

                "session":
                    session_id,

                "cell":
                    cell_id,
            }

    return {
        "source": None,
        "origin": "NOT_FOUND",
        "session": None,
        "cell": None,
    }


# ------------------------------------------------------------
# 3. Generator functions
# ------------------------------------------------------------

generator_functions_625b3a = {
    "bias":
        frozen_add_sensor_bias_620gc,

    "noise":
        frozen_add_sensor_noise_620gc,

    "drift":
        frozen_add_sensor_drift_620gc,

    "dropout":
        candidate_add_sensor_dropout_617d,

    "stuck":
        candidate_add_stuck_sensor_618d,

    "extractor":
        extract_corrupted_dataframe_620gc,
}


generator_source_records_625b3a = []


print(
    "\n--- Generator return-contract source ---"
)


for family_name, func in \
    generator_functions_625b3a.items():

    recovered = (
        recover_function_source_625b3a(
            func
        )
    )

    src = recovered[
        "source"
    ]

    print(
        "\n"
        + "=" * 76
    )

    print(
        "HELPER:",
        family_name,
        "|",
        func.__name__
    )

    print(
        "Signature:",
        inspect.signature(
            func
        )
    )

    print(
        "Source recovery:",
        recovered[
            "origin"
        ]
    )

    if recovered[
        "cell"
    ] is not None:

        print(
            "Source session/cell:",
            recovered[
                "session"
            ],
            "/",
            recovered[
                "cell"
            ]
        )

    print(
        "=" * 76
    )

    generator_source_records_625b3a.append(
        {
            "family":
                family_name,

            "function_name":
                func.__name__,

            "signature":
                str(
                    inspect.signature(
                        func
                    )
                ),

            "source_origin":
                recovered[
                    "origin"
                ],

            "source_cell":
                recovered[
                    "cell"
                ],

            "source_found":
                src is not None,
        }
    )

    if src is None:

        print(
            "SOURCE COULD NOT BE RECOVERED."
        )

        continue

    src_lines = (
        src.splitlines()
    )

    # Extract return-related context
    return_line_indices = []

    for i, line in enumerate(
        src_lines
    ):

        if (
            re.match(
                r"^\s*return\b",
                line
            )
            or
            "corrupted_df"
            in line
            or
            "result"
            in line
        ):

            return_line_indices.append(
                i
            )

    context_indices = set()

    for idx in return_line_indices:

        lo = max(
            0,
            idx - 4
        )

        hi = min(
            len(src_lines),
            idx + 7
        )

        context_indices.update(
            range(
                lo,
                hi
            )
        )

    if not context_indices:

        print(
            "No return-related lines detected."
        )

    else:

        for idx in sorted(
            context_indices
        ):

            line = (
                src_lines[idx]
            )

            marker = (
                ">>>"
                if re.match(
                    r"^\s*return\b",
                    line
                )
                else "   "
            )

            print(
                f"{marker} "
                f"{idx + 1:03d}: "
                f"{line}"
            )


# ------------------------------------------------------------
# 4. Compact generator source-recovery summary
# ------------------------------------------------------------

GENERATOR_SOURCE_AUDIT_625B3A = (
    pd.DataFrame(
        generator_source_records_625b3a
    )
)

print(
    "\n--- Generator source recovery summary ---"
)

display(
    GENERATOR_SOURCE_AUDIT_625B3A
)


# ------------------------------------------------------------
# 5. Complete extractor source
# ------------------------------------------------------------

print(
    "\n--- Complete extractor source ---"
)

extractor_recovered_625b3a = (
    recover_function_source_625b3a(
        extract_corrupted_dataframe_620gc
    )
)

if (
    extractor_recovered_625b3a[
        "source"
    ]
    is not None
):

    print(
        extractor_recovered_625b3a[
            "source"
        ]
    )

else:

    print(
        "Extractor source could not be recovered."
    )


# ------------------------------------------------------------
# 6. Sensor-reference path
# ------------------------------------------------------------

print(
    "\n--- SENSOR_REFERENCE_PATH ---"
)

if (
    "SENSOR_REFERENCE_PATH"
    in globals()
):

    reference_path_625b3a = Path(
        SENSOR_REFERENCE_PATH
    )

    print(
        "Type   :",
        type(
            SENSOR_REFERENCE_PATH
        ).__name__
    )

    print(
        "Path   :",
        reference_path_625b3a
    )

    print(
        "Exists :",
        reference_path_625b3a.exists()
    )

    if reference_path_625b3a.exists():

        print(
            "Suffix :",
            reference_path_625b3a.suffix
        )

        print(
            "Bytes  :",
            reference_path_625b3a.stat().st_size
        )

else:

    reference_path_625b3a = None

    print(
        "SENSOR_REFERENCE_PATH unavailable."
    )


# ------------------------------------------------------------
# 7. Search only likely CREATION/LOAD assignments
# ------------------------------------------------------------

print(
    "\n--- Sensor-reference provenance assignment search ---"
)

provenance_hits_625b3a = []


assignment_patterns_625b3a = [
    r"sensor_reference_stats\s*=",
    r"SENSOR_REFERENCE_PATH\s*=",
    r"to_csv\s*\(",
    r"read_csv\s*\(",
]


for (
    session_id,
    cell_id,
    source
) in history_records_625b3a:

    contains_reference_name = (
        "sensor_reference_stats"
        in source
        or
        "SENSOR_REFERENCE_PATH"
        in source
    )

    if not contains_reference_name:

        continue

    important_lines = []

    lines = (
        source.splitlines()
    )

    for i, line in enumerate(
        lines,
        start=1
    ):

        if (
            "sensor_reference_stats"
            in line
            or
            "SENSOR_REFERENCE_PATH"
            in line
        ):

            important_lines.append(
                i
            )

    if not important_lines:

        continue

    context_set = set()

    for line_no in important_lines:

        context_set.update(
            range(
                max(
                    1,
                    line_no - 10
                ),
                min(
                    len(lines),
                    line_no + 15
                )
                + 1
            )
        )

    provenance_hits_625b3a.append(
        (
            session_id,
            cell_id,
            lines,
            sorted(
                context_set
            ),
        )
    )


print(
    "History cells mentioning reference objects:",
    len(
        provenance_hits_625b3a
    )
)


# Show newest matches first, maximum 12 cells
for (
    session_id,
    cell_id,
    lines,
    context_indices
) in reversed(
    provenance_hits_625b3a[
        -12:
    ]
):

    print(
        "\n"
        + "-" * 76
    )

    print(
        "SESSION:",
        session_id,
        "CELL:",
        cell_id
    )

    for line_no in context_indices:

        line = (
            lines[
                line_no - 1
            ]
        )

        marker = (
            ">>>"
            if (
                "sensor_reference_stats"
                in line
                or
                "SENSOR_REFERENCE_PATH"
                in line
            )
            else "   "
        )

        print(
            f"{marker} "
            f"{line_no:03d}: "
            f"{line}"
        )


# ------------------------------------------------------------
# 8. Stored reference artifact comparison
# ------------------------------------------------------------

print(
    "\n--- Stored sensor-reference artifact comparison ---"
)

reference_exact_625b3a = None


if (
    reference_path_625b3a is not None
    and
    reference_path_625b3a.exists()
    and
    reference_path_625b3a.suffix.lower()
    == ".csv"
):

    stored_reference_625b3a = (
        pd.read_csv(
            reference_path_625b3a,
            index_col=0
        )
    )

    print(
        "Stored shape :",
        stored_reference_625b3a.shape
    )

    print(
        "Memory shape :",
        sensor_reference_stats.shape
    )

    index_exact = (
        list(
            stored_reference_625b3a.index
        )
        ==
        list(
            sensor_reference_stats.index
        )
    )

    columns_exact = (
        list(
            stored_reference_625b3a.columns
        )
        ==
        list(
            sensor_reference_stats.columns
        )
    )

    print(
        "Index exact   :",
        index_exact
    )

    print(
        "Columns exact :",
        columns_exact
    )

    if (
        index_exact
        and
        columns_exact
        and
        stored_reference_625b3a.shape
        ==
        sensor_reference_stats.shape
    ):

        max_diff_625b3a = float(
            np.max(
                np.abs(
                    stored_reference_625b3a
                    .to_numpy(
                        dtype=np.float64
                    )
                    -
                    sensor_reference_stats
                    .to_numpy(
                        dtype=np.float64
                    )
                )
            )
        )

        reference_exact_625b3a = (
            max_diff_625b3a
            == 0.0
        )

        print(
            "Maximum numeric difference:",
            max_diff_625b3a
        )

        print(
            "Stored/memory exact:",
            reference_exact_625b3a
        )

else:

    print(
        "No CSV reload comparison performed."
    )


# ------------------------------------------------------------
# 9. Integrity summary
# ------------------------------------------------------------

print(
    "\n=============================================="
)

print(
    "STEP 6.25B3A INTEGRITY SUMMARY"
)

print(
    "=============================================="
)

print(
    "Generator functions executed       : False"
)

print(
    "Corruption executed                : False"
)

print(
    "Official TEST modified             : False"
)

print(
    "Model inference executed           : False"
)

print(
    "MC-dropout executed                : False"
)

print(
    "Official TEST scored               : False"
)

print(
    "Official labels used               : False"
)

print(
    "Sensor statistics recomputed       : False"
)

print(
    "Frozen manifests modified          : False"
)

print(
    "Frozen reliability rule modified  : False"
)

print(
    "Threshold modified                 : False"
)

print(
    "Files written                     : False"
)

print(
    "\nNothing was modified."
)

print(
    "=== END ==="
)

In [ ]:
# ============================================================
# Step 6.25B3B
# Behavioural corruption-generator contract probe
#
# FIVE representative DEVELOPMENT corruptions only
# NO MODEL INFERENCE
# NO MC DROPOUT
# NO OFFICIAL TEST
# NO OFFICIAL LABELS
# NO FILE WRITES
# NO FROZEN-DECISION CHANGES
# ============================================================

import numpy as np
import pandas as pd


print(
    "=== Step 6.25B3B — Behavioural generator-contract probe ==="
)


# ------------------------------------------------------------
# 1. Required objects
# ------------------------------------------------------------

required_objects_625b3b = [
    "scenario_manifests_620gc",
    "val_raw_base_620gc",
    "sensor_reference_stats",
    "frozen_add_sensor_bias_620gc",
    "frozen_add_sensor_noise_620gc",
    "frozen_add_sensor_drift_620gc",
    "candidate_add_sensor_dropout_617d",
    "candidate_add_stuck_sensor_618d",
    "extract_corrupted_dataframe_620gc",
    "PRIMARY_INPUT_COLUMNS",
]


print("\n--- Required objects ---")

for name in required_objects_625b3b:

    print(
        f"{name:<42}:",
        name in globals()
    )

    if name not in globals():

        raise RuntimeError(
            f"Required object unavailable: {name}"
        )


# ------------------------------------------------------------
# 2. Freeze original validation raw base for mutation check
# ------------------------------------------------------------

VAL_RAW_SNAPSHOT_625B3B = (
    val_raw_base_620gc.copy(
        deep=True
    )
)


# ------------------------------------------------------------
# 3. Helper: compact description of return object
# ------------------------------------------------------------

def describe_return_625b3b(obj):

    info = {
        "result_type":
            type(obj).__name__,

        "container_length":
            np.nan,

        "component_types":
            "",
    }

    if isinstance(
        obj,
        (tuple, list)
    ):

        info[
            "container_length"
        ] = len(obj)

        parts = []

        for item in obj:

            item_type = (
                type(item).__name__
            )

            item_shape = getattr(
                item,
                "shape",
                None
            )

            if item_shape is not None:

                parts.append(
                    f"{item_type}{tuple(item_shape)}"
                )

            else:

                parts.append(
                    item_type
                )

        info[
            "component_types"
        ] = " | ".join(
            parts
        )

    elif isinstance(
        obj,
        dict
    ):

        info[
            "container_length"
        ] = len(obj)

        info[
            "component_types"
        ] = (
            "keys="
            + str(
                list(obj.keys())
            )
        )

    else:

        item_shape = getattr(
            obj,
            "shape",
            None
        )

        if item_shape is not None:

            info[
                "component_types"
            ] = (
                f"{type(obj).__name__}"
                f"{tuple(item_shape)}"
            )

    return info


# ------------------------------------------------------------
# 4. Exact underlying-generator invocation
#
# IMPORTANT:
# Bias/noise/drift legacy argument called `severity`
# actually receives frozen numeric alpha, as already verified
# in apply_frozen_corruption_620gc.
# ------------------------------------------------------------

def invoke_underlying_generator_625b3b(
    family,
    scenario_row,
    base_df
):

    sensor = str(
        scenario_row[
            "sensor"
        ]
    )

    if family == "bias":

        alpha = float(
            scenario_row[
                "alpha"
            ]
        )

        direction = int(
            scenario_row[
                "direction"
            ]
        )

        return frozen_add_sensor_bias_620gc(
            base_df.copy(
                deep=True
            ),
            sensor,
            alpha,
            direction,
            sensor_reference_stats,
        )

    if family == "noise":

        alpha = float(
            scenario_row[
                "alpha"
            ]
        )

        if (
            "seed"
            in scenario_row.index
            and
            pd.notna(
                scenario_row[
                    "seed"
                ]
            )
        ):

            seed = int(
                scenario_row[
                    "seed"
                ]
            )

        else:

            seed = 42

        return frozen_add_sensor_noise_620gc(
            base_df.copy(
                deep=True
            ),
            sensor,
            alpha,
            sensor_reference_stats,
            seed=seed,
        )

    if family == "drift":

        alpha = float(
            scenario_row[
                "alpha"
            ]
        )

        direction = int(
            scenario_row[
                "direction"
            ]
        )

        return frozen_add_sensor_drift_620gc(
            base_df.copy(
                deep=True
            ),
            sensor,
            alpha,
            direction,
            sensor_reference_stats,
        )

    if family == "dropout":

        requested_fraction = float(
            scenario_row[
                "requested_fraction"
            ]
        )

        if (
            "seed"
            in scenario_row.index
            and
            pd.notna(
                scenario_row[
                    "seed"
                ]
            )
        ):

            seed = int(
                scenario_row[
                    "seed"
                ]
            )

        else:

            seed = 42

        return candidate_add_sensor_dropout_617d(
            base_df.copy(
                deep=True
            ),
            sensor,
            requested_fraction,
            sensor_reference_stats,
            seed=seed,
        )

    if family == "stuck":

        requested_fraction = float(
            scenario_row[
                "requested_fraction"
            ]
        )

        return candidate_add_stuck_sensor_618d(
            base_df.copy(
                deep=True
            ),
            sensor,
            requested_fraction,
        )

    raise ValueError(
        f"Unsupported family: {family}"
    )


# ------------------------------------------------------------
# 5. Probe ONE representative frozen scenario per family
# ------------------------------------------------------------

families_625b3b = [
    "bias",
    "noise",
    "drift",
    "dropout",
    "stuck",
]


probe_records_625b3b = []


print(
    "\n--- Running five temporary DEVELOPMENT probes ---"
)


for family in families_625b3b:

    manifest = (
        scenario_manifests_620gc[
            family
        ]
    )

    if not isinstance(
        manifest,
        pd.DataFrame
    ):

        raise RuntimeError(
            f"{family} manifest is not a DataFrame."
        )

    if len(manifest) == 0:

        raise RuntimeError(
            f"{family} manifest is empty."
        )

    # First frozen scenario only.
    scenario_row = (
        manifest.iloc[
            0
        ].copy()
    )

    sensor = str(
        scenario_row[
            "sensor"
        ]
    )

    scenario_id = (
        str(
            scenario_row[
                "scenario_id"
            ]
        )
        if (
            "scenario_id"
            in scenario_row.index
            and
            pd.notna(
                scenario_row[
                    "scenario_id"
                ]
            )
        )
        else
        f"{family}_representative"
    )


    print(
        f"\n{family.upper()} "
        f"| {scenario_id} "
        f"| {sensor}"
    )


    # --------------------------------------------------------
    # Execute generator on COPY only
    # --------------------------------------------------------

    result = (
        invoke_underlying_generator_625b3b(
            family,
            scenario_row,
            VAL_RAW_SNAPSHOT_625B3B,
        )
    )


    return_info = (
        describe_return_625b3b(
            result
        )
    )


    # --------------------------------------------------------
    # Extract corrupted DataFrame using frozen extractor
    # --------------------------------------------------------

    extraction_error = None
    extracted_df = None

    try:

        extracted_df = (
            extract_corrupted_dataframe_620gc(
                family,
                result
            )
        )

    except Exception as exc:

        extraction_error = (
            type(exc).__name__
            + ": "
            + str(exc)
        )


    # --------------------------------------------------------
    # Validate extracted DataFrame
    # --------------------------------------------------------

    extracted_shape = None
    same_shape = False
    same_columns = False
    same_index = False
    target_changed_cells = np.nan
    primary_changed_cells = np.nan
    all_numeric_changed_cells = np.nan
    max_abs_target_change = np.nan


    if isinstance(
        extracted_df,
        pd.DataFrame
    ):

        extracted_shape = (
            extracted_df.shape
        )

        same_shape = (
            extracted_df.shape
            ==
            VAL_RAW_SNAPSHOT_625B3B.shape
        )

        same_columns = (
            list(
                extracted_df.columns
            )
            ==
            list(
                VAL_RAW_SNAPSHOT_625B3B.columns
            )
        )

        same_index = (
            extracted_df.index.equals(
                VAL_RAW_SNAPSHOT_625B3B.index
            )
        )


        # Target sensor effect
        clean_target = (
            VAL_RAW_SNAPSHOT_625B3B[
                sensor
            ]
            .to_numpy(
                dtype=np.float64
            )
        )

        corrupted_target = (
            extracted_df[
                sensor
            ]
            .to_numpy(
                dtype=np.float64
            )
        )

        target_delta = (
            corrupted_target
            -
            clean_target
        )

        target_changed_cells = int(
            np.sum(
                np.abs(
                    target_delta
                )
                > 1e-12
            )
        )

        max_abs_target_change = float(
            np.max(
                np.abs(
                    target_delta
                )
            )
        )


        # Primary model inputs
        clean_primary = (
            VAL_RAW_SNAPSHOT_625B3B[
                PRIMARY_INPUT_COLUMNS
            ]
            .to_numpy(
                dtype=np.float64
            )
        )

        corrupted_primary = (
            extracted_df[
                PRIMARY_INPUT_COLUMNS
            ]
            .to_numpy(
                dtype=np.float64
            )
        )

        primary_changed_cells = int(
            np.sum(
                np.abs(
                    corrupted_primary
                    -
                    clean_primary
                )
                > 1e-12
            )
        )


        # All shared numeric columns
        shared_numeric = [
            col
            for col
            in VAL_RAW_SNAPSHOT_625B3B.columns
            if (
                col in extracted_df.columns
                and
                pd.api.types.is_numeric_dtype(
                    VAL_RAW_SNAPSHOT_625B3B[
                        col
                    ]
                )
                and
                pd.api.types.is_numeric_dtype(
                    extracted_df[
                        col
                    ]
                )
            )
        ]

        clean_numeric = (
            VAL_RAW_SNAPSHOT_625B3B[
                shared_numeric
            ]
            .to_numpy(
                dtype=np.float64
            )
        )

        corrupted_numeric = (
            extracted_df[
                shared_numeric
            ]
            .to_numpy(
                dtype=np.float64
            )
        )

        all_numeric_changed_cells = int(
            np.sum(
                np.abs(
                    corrupted_numeric
                    -
                    clean_numeric
                )
                > 1e-12
            )
        )


    # --------------------------------------------------------
    # Confirm original in-memory validation base untouched
    # --------------------------------------------------------

    original_unchanged = (
        val_raw_base_620gc.equals(
            VAL_RAW_SNAPSHOT_625B3B
        )
    )


    probe_records_625b3b.append(
        {
            "family":
                family,

            "scenario_id":
                scenario_id,

            "sensor":
                sensor,

            "result_type":
                return_info[
                    "result_type"
                ],

            "container_length":
                return_info[
                    "container_length"
                ],

            "component_types":
                return_info[
                    "component_types"
                ],

            "extracted_type":
                (
                    type(
                        extracted_df
                    ).__name__
                    if extracted_df
                    is not None
                    else None
                ),

            "extracted_shape":
                str(
                    extracted_shape
                ),

            "same_shape":
                same_shape,

            "same_columns":
                same_columns,

            "same_index":
                same_index,

            "target_changed_cells":
                target_changed_cells,

            "primary_changed_cells":
                primary_changed_cells,

            "all_numeric_changed_cells":
                all_numeric_changed_cells,

            "max_abs_target_change":
                max_abs_target_change,

            "extraction_error":
                extraction_error,

            "original_validation_unchanged":
                original_unchanged,
        }
    )


# ------------------------------------------------------------
# 6. Compact result table
# ------------------------------------------------------------

GENERATOR_CONTRACT_AUDIT_625B3B = (
    pd.DataFrame(
        probe_records_625b3b
    )
)


print(
    "\n--- Behavioural return/extraction contract ---"
)

display(
    GENERATOR_CONTRACT_AUDIT_625B3B
)


# ------------------------------------------------------------
# 7. Critical checks
# ------------------------------------------------------------

all_extracted_625b3b = bool(
    GENERATOR_CONTRACT_AUDIT_625B3B[
        "extraction_error"
    ]
    .isna()
    .all()
)

all_dataframe_625b3b = bool(
    (
        GENERATOR_CONTRACT_AUDIT_625B3B[
            "extracted_type"
        ]
        ==
        "DataFrame"
    )
    .all()
)

all_shapes_625b3b = bool(
    GENERATOR_CONTRACT_AUDIT_625B3B[
        "same_shape"
    ].all()
)

all_columns_625b3b = bool(
    GENERATOR_CONTRACT_AUDIT_625B3B[
        "same_columns"
    ].all()
)

all_indices_625b3b = bool(
    GENERATOR_CONTRACT_AUDIT_625B3B[
        "same_index"
    ].all()
)

all_effective_625b3b = bool(
    (
        GENERATOR_CONTRACT_AUDIT_625B3B[
            "target_changed_cells"
        ]
        > 0
    ).all()
)

original_safe_625b3b = bool(
    GENERATOR_CONTRACT_AUDIT_625B3B[
        "original_validation_unchanged"
    ].all()
)


# ------------------------------------------------------------
# 8. Integrity summary
# ------------------------------------------------------------

print(
    "\n=============================================="
)

print(
    "STEP 6.25B3B CONTRACT SUMMARY"
)

print(
    "=============================================="
)

print(
    "Representative scenarios executed : 5"
)

print(
    "Families probed                  :",
    families_625b3b
)

print(
    "Extractor succeeded all families:",
    all_extracted_625b3b
)

print(
    "All extracted objects DataFrames :",
    all_dataframe_625b3b
)

print(
    "All extracted shapes preserved   :",
    all_shapes_625b3b
)

print(
    "All column orders preserved      :",
    all_columns_625b3b
)

print(
    "All indices preserved            :",
    all_indices_625b3b
)

print(
    "All representative corruptions effective:",
    all_effective_625b3b
)

print(
    "Original validation base unchanged:",
    original_safe_625b3b
)

print(
    "\nOfficial TEST touched            : False"
)

print(
    "Official TEST labels used        : False"
)

print(
    "Model inference executed         : False"
)

print(
    "MC-dropout executed              : False"
)

print(
    "Reliability score evaluated      : False"
)

print(
    "Threshold modified               : False"
)

print(
    "Frozen manifests modified        : False"
)

print(
    "Frozen decision modified         : False"
)

print(
    "Files written                    : False"
)

print(
    "\nOnly five temporary DEVELOPMENT corruption"
    " probes were executed."
)

print(
    "=== END ==="
)

In [ ]:
# ============================================================
# Step 6.25B4
# Compact extractor portability inspection
#
# INSPECTION ONLY
# NO CORRUPTION
# NO MODEL INFERENCE
# NO MC DROPOUT
# NO OFFICIAL TEST SCORING
# NO FILE WRITES
# ============================================================

import inspect


print(
    "=== Step 6.25B4 — Extractor TEST-portability inspection ==="
)


# ------------------------------------------------------------
# 1. Basic identity
# ------------------------------------------------------------

extractor_625b4 = (
    extract_corrupted_dataframe_620gc
)

print(
    "\n--- Extractor identity ---"
)

print(
    "Available :",
    "extract_corrupted_dataframe_620gc"
    in globals()
)

print(
    "Type      :",
    type(
        extractor_625b4
    ).__name__
)

print(
    "Signature :",
    inspect.signature(
        extractor_625b4
    )
)


# ------------------------------------------------------------
# 2. Recover exact source
# ------------------------------------------------------------

source_625b4 = inspect.getsource(
    extractor_625b4
)

lines_625b4 = (
    source_625b4.splitlines()
)

print(
    "\nSource line count:",
    len(
        lines_625b4
    )
)


# ------------------------------------------------------------
# 3. Print ONLY portability-relevant source contexts
# ------------------------------------------------------------

keywords_625b4 = [
    "val_raw_base_620gc",
    "return ",
    "result[",
    "family ==",
    "family in",
    "isinstance",
    ".shape",
    ".columns",
]


matched_indices_625b4 = set()


for idx, line in enumerate(
    lines_625b4
):

    if any(
        keyword in line
        for keyword
        in keywords_625b4
    ):

        # ±2 lines of context
        for j in range(
            max(
                0,
                idx - 2
            ),
            min(
                len(
                    lines_625b4
                ),
                idx + 3
            )
        ):

            matched_indices_625b4.add(
                j
            )


print(
    "\n--- Portability-relevant source context ---"
)

previous_625b4 = None

for idx in sorted(
    matched_indices_625b4
):

    if (
        previous_625b4
        is not None
        and
        idx
        >
        previous_625b4 + 1
    ):

        print(
            "      ..."
        )

    print(
        f"{idx + 1:03d}: "
        f"{lines_625b4[idx]}"
    )

    previous_625b4 = idx


# ------------------------------------------------------------
# 4. Exact global-name dependency check
# ------------------------------------------------------------

code_names_625b4 = set(
    extractor_625b4
    .__code__
    .co_names
)

print(
    "\n--- Dependency check ---"
)

print(
    "References val_raw_base_620gc :",
    "val_raw_base_620gc"
    in code_names_625b4
)

print(
    "References pandas/DataFrame    :",
    (
        "pd"
        in code_names_625b4
        or
        "DataFrame"
        in code_names_625b4
    )
)


# ------------------------------------------------------------
# 5. Safety summary
# ------------------------------------------------------------

print(
    "\n--- Integrity ---"
)

print(
    "Corruption executed        : False"
)

print(
    "Official TEST modified     : False"
)

print(
    "Official TEST scored       : False"
)

print(
    "Official labels used       : False"
)

print(
    "Model inference executed   : False"
)

print(
    "MC-dropout executed        : False"
)

print(
    "Frozen manifests modified  : False"
)

print(
    "Frozen threshold modified  : False"
)

print(
    "Frozen rule modified       : False"
)

print(
    "Files written              : False"
)

print(
    "Nothing was modified."
)

print(
    "=== END ==="
)

In [ ]:
# ============================================================
# Step 6.25B4A
# Exact val_raw_base_620gc usage inside extractor
# INSPECTION ONLY
# ============================================================

import inspect

source_625b4a = inspect.getsource(
    extract_corrupted_dataframe_620gc
)

lines_625b4a = source_625b4a.splitlines()

print(
    "=== Step 6.25B4A — Exact DEV-base dependency context ==="
)

hits_625b4a = [
    i
    for i, line in enumerate(lines_625b4a)
    if "val_raw_base_620gc" in line
]

print(
    "\nval_raw_base_620gc occurrence lines:",
    [
        i + 1
        for i in hits_625b4a
    ]
)

for hit in hits_625b4a:

    print(
        "\n" + "=" * 70
    )

    start = max(
        0,
        hit - 6
    )

    end = min(
        len(lines_625b4a),
        hit + 7
    )

    for j in range(
        start,
        end
    ):

        marker = (
            ">>>"
            if j == hit
            else "   "
        )

        print(
            f"{marker} {j + 1:03d}: "
            f"{lines_625b4a[j]}"
        )


print(
    "\n--- Return-related lines ---"
)

for i, line in enumerate(
    lines_625b4a,
    start=1
):

    stripped = line.strip()

    if (
        stripped.startswith("return ")
        or
        "result[" in stripped
    ):

        print(
            f"{i:03d}: {line}"
        )


print(
    "\n--- Integrity ---"
)

print(
    "Corruption executed      : False"
)

print(
    "Official TEST touched    : False"
)

print(
    "Model inference executed : False"
)

print(
    "Official TEST scored     : False"
)

print(
    "Files written            : False"
)

print(
    "Nothing was modified."
)

print(
    "=== END ==="
)

In [ ]:
# ============================================================
# Step 6.25B4B
# Exact family-specific extraction branches
# INSPECTION ONLY
# ============================================================

import inspect

source_625b4b = inspect.getsource(
    extract_corrupted_dataframe_620gc
)

lines_625b4b = source_625b4b.splitlines()

print(
    "=== Step 6.25B4B — Exact extractor family branches ==="
)

keywords_625b4b = [
    'family ==',
    'family in',
    'result[',
    'corrupted_df =',
    'raise',
    'return corrupted_df'
]

for i, line in enumerate(
    lines_625b4b,
    start=1
):
    stripped = line.strip()

    if any(
        key in stripped
        for key in keywords_625b4b
    ):
        print(
            f"{i:03d}: {line}"
        )


print(
    "\n--- Focused source region 020-074 ---"
)

for i in range(
    19,
    min(
        74,
        len(lines_625b4b)
    )
):
    print(
        f"{i + 1:03d}: "
        f"{lines_625b4b[i]}"
    )


print(
    "\n--- Integrity ---"
)

print(
    "Corruption executed      : False"
)

print(
    "Official TEST touched    : False"
)

print(
    "Model inference executed : False"
)

print(
    "MC-dropout executed      : False"
)

print(
    "Official TEST scored     : False"
)

print(
    "Frozen extractor modified: False"
)

print(
    "Files written            : False"
)

print(
    "Nothing was modified."
)

print(
    "=== END ==="
)

In [ ]:
# ============================================================
# Step 6.25B5
# Official FD001 TEST corruption smoke test
#
# Purpose:
#   1. Create TEST-portable corruption application logic.
#   2. Execute exactly ONE frozen scenario per corruption family.
#   3. Verify structural integrity and actual raw-data effects.
#
# IMPORTANT:
#   - No model inference.
#   - No MC-dropout.
#   - No official RUL labels.
#   - No scoring.
#   - No threshold/rule changes.
#   - No files written.
# ============================================================

import numpy as np
import pandas as pd


print(
    "=== Step 6.25B5 — Official FD001 TEST "
    "five-family corruption smoke test ==="
)


# ------------------------------------------------------------
# 1. Required-object verification
# ------------------------------------------------------------

required_objects_625b5 = [
    "OFFICIAL_TEST_RAW_624B",
    "scenario_manifests_620gc",
    "PRIMARY_INPUT_COLUMNS",
    "sensor_reference_stats",
    "frozen_add_sensor_bias_620gc",
    "frozen_add_sensor_noise_620gc",
    "frozen_add_sensor_drift_620gc",
    "candidate_add_sensor_dropout_617d",
    "candidate_add_stuck_sensor_618d",
]

print(
    "\n--- Required objects ---"
)

for name_625b5 in required_objects_625b5:
    available_625b5 = (
        name_625b5 in globals()
    )

    print(
        f"{name_625b5:<38}: "
        f"{available_625b5}"
    )

    if not available_625b5:
        raise RuntimeError(
            f"Missing required object: "
            f"{name_625b5}"
        )


# ------------------------------------------------------------
# 2. Freeze pristine official TEST copy
# ------------------------------------------------------------

official_test_before_625b5 = (
    OFFICIAL_TEST_RAW_624B.copy(
        deep=True
    )
)

if len(
    official_test_before_625b5
) != 13096:
    raise RuntimeError(
        "Unexpected official TEST row count."
    )


# ------------------------------------------------------------
# 3. TEST-portable extractor
# ------------------------------------------------------------

def extract_corrupted_dataframe_test_625b5(
    family,
    result,
    reference_df
):
    """
    TEST-portable equivalent of the frozen DEV extractor.

    The frozen family-specific return contracts are preserved.

    No dependency on val_raw_base_620gc.
    """

    family = str(
        family
    ).lower()

    if family == "dropout":

        if not isinstance(
            result,
            (tuple, list)
        ):
            raise TypeError(
                "DROP_OUT result must be "
                "tuple/list."
            )

        if len(result) < 2:
            raise RuntimeError(
                "DROP_OUT result has fewer "
                "than two components."
            )

        # Frozen downstream convention:
        # use the IMPUTED DataFrame.
        corrupted_df = result[1]

    elif family in {
        "bias",
        "noise",
        "drift",
        "stuck",
    }:

        if isinstance(
            result,
            pd.DataFrame
        ):
            corrupted_df = result

        elif isinstance(
            result,
            (tuple, list)
        ):

            if len(result) < 1:
                raise RuntimeError(
                    f"{family} result is empty."
                )

            corrupted_df = result[0]

        else:
            raise TypeError(
                f"Unsupported {family} "
                f"result type: "
                f"{type(result).__name__}"
            )

    else:
        raise ValueError(
            f"Unsupported corruption family: "
            f"{family}"
        )


    if not isinstance(
        corrupted_df,
        pd.DataFrame
    ):
        raise TypeError(
            f"Extracted {family} object is not "
            f"a DataFrame."
        )


    # TEST-safe structural contract
    if corrupted_df.shape != reference_df.shape:
        raise RuntimeError(
            f"{family}: corrupted shape "
            f"{corrupted_df.shape} differs "
            f"from reference shape "
            f"{reference_df.shape}."
        )


    if list(
        corrupted_df.columns
    ) != list(
        reference_df.columns
    ):
        raise RuntimeError(
            f"{family}: column order changed."
        )


    if not corrupted_df.index.equals(
        reference_df.index
    ):
        raise RuntimeError(
            f"{family}: index changed."
        )


    return corrupted_df.copy(
        deep=True
    )


# ------------------------------------------------------------
# 4. TEST-portable frozen corruption applicator
# ------------------------------------------------------------

def apply_frozen_corruption_test_625b5(
    base_df,
    family,
    scenario_row
):

    family = str(
        family
    ).lower()

    row = scenario_row

    sensor = str(
        row["sensor"]
    )


    # Always operate on a temporary copy.
    working_df = base_df.copy(
        deep=True
    )


    if family == "bias":

        alpha = float(
            row["alpha"]
        )

        direction = int(
            row["direction"]
        )

        result = (
            frozen_add_sensor_bias_620gc(
                working_df,
                sensor,
                alpha,
                direction,
                sensor_reference_stats,
            )
        )


    elif family == "noise":

        alpha = float(
            row["alpha"]
        )

        seed = (
            int(row["seed"])
            if "seed" in row.index
            and pd.notna(
                row["seed"]
            )
            else 42
        )

        result = (
            frozen_add_sensor_noise_620gc(
                working_df,
                sensor,
                alpha,
                sensor_reference_stats,
                seed=seed,
            )
        )


    elif family == "drift":

        alpha = float(
            row["alpha"]
        )

        direction = int(
            row["direction"]
        )

        result = (
            frozen_add_sensor_drift_620gc(
                working_df,
                sensor,
                alpha,
                direction,
                sensor_reference_stats,
            )
        )


    elif family == "dropout":

        requested_fraction = float(
            row["requested_fraction"]
        )

        seed = (
            int(row["seed"])
            if "seed" in row.index
            and pd.notna(
                row["seed"]
            )
            else 42
        )

        result = (
            candidate_add_sensor_dropout_617d(
                working_df,
                sensor,
                requested_fraction,
                sensor_reference_stats,
                seed=seed,
            )
        )


    elif family == "stuck":

        requested_fraction = float(
            row["requested_fraction"]
        )

        result = (
            candidate_add_stuck_sensor_618d(
                working_df,
                sensor,
                requested_fraction,
            )
        )


    else:
        raise ValueError(
            f"Unsupported family: {family}"
        )


    corrupted_df = (
        extract_corrupted_dataframe_test_625b5(
            family=family,
            result=result,
            reference_df=base_df,
        )
    )


    return result, corrupted_df


# ------------------------------------------------------------
# 5. Run exactly ONE scenario per family
# ------------------------------------------------------------

families_625b5 = [
    "bias",
    "noise",
    "drift",
    "dropout",
    "stuck",
]

smoke_rows_625b5 = []


print(
    "\n--- Running five temporary "
    "official-TEST corruption probes ---"
)


for family_625b5 in families_625b5:

    manifest_625b5 = (
        scenario_manifests_620gc[
            family_625b5
        ]
    )

    if not isinstance(
        manifest_625b5,
        pd.DataFrame
    ):
        raise TypeError(
            f"{family_625b5} manifest "
            f"is not a DataFrame."
        )

    if len(
        manifest_625b5
    ) < 1:
        raise RuntimeError(
            f"{family_625b5} manifest empty."
        )


    scenario_row_625b5 = (
        manifest_625b5.iloc[0]
    )

    sensor_625b5 = str(
        scenario_row_625b5["sensor"]
    )


    scenario_id_625b5 = (
        str(
            scenario_row_625b5[
                "scenario_id"
            ]
        )
        if (
            "scenario_id"
            in scenario_row_625b5.index
            and pd.notna(
                scenario_row_625b5[
                    "scenario_id"
                ]
            )
        )
        else
        f"{family_625b5}_representative"
    )


    print(
        f"\n{family_625b5.upper()} | "
        f"{scenario_id_625b5} | "
        f"{sensor_625b5}"
    )


    result_625b5, corrupted_625b5 = (
        apply_frozen_corruption_test_625b5(
            base_df=OFFICIAL_TEST_RAW_624B,
            family=family_625b5,
            scenario_row=scenario_row_625b5,
        )
    )


    # --------------------------------------------------------
    # Target-sensor effect
    # --------------------------------------------------------

    clean_sensor_625b5 = (
        OFFICIAL_TEST_RAW_624B[
            sensor_625b5
        ]
        .to_numpy(
            dtype=np.float64
        )
    )

    corrupt_sensor_625b5 = (
        corrupted_625b5[
            sensor_625b5
        ]
        .to_numpy(
            dtype=np.float64
        )
    )


    target_change_mask_625b5 = (
        ~np.isclose(
            clean_sensor_625b5,
            corrupt_sensor_625b5,
            rtol=0.0,
            atol=0.0,
            equal_nan=True,
        )
    )

    target_changed_cells_625b5 = int(
        target_change_mask_625b5.sum()
    )


    target_max_change_625b5 = (
        float(
            np.nanmax(
                np.abs(
                    corrupt_sensor_625b5
                    -
                    clean_sensor_625b5
                )
            )
        )
        if len(
            clean_sensor_625b5
        ) > 0
        else np.nan
    )


    # --------------------------------------------------------
    # Primary-input effect
    # --------------------------------------------------------

    clean_primary_625b5 = (
        OFFICIAL_TEST_RAW_624B[
            PRIMARY_INPUT_COLUMNS
        ]
        .to_numpy(
            dtype=np.float64
        )
    )

    corrupt_primary_625b5 = (
        corrupted_625b5[
            PRIMARY_INPUT_COLUMNS
        ]
        .to_numpy(
            dtype=np.float64
        )
    )


    primary_change_mask_625b5 = (
        ~np.isclose(
            clean_primary_625b5,
            corrupt_primary_625b5,
            rtol=0.0,
            atol=0.0,
            equal_nan=True,
        )
    )

    primary_changed_cells_625b5 = int(
        primary_change_mask_625b5.sum()
    )


    primary_max_change_625b5 = (
        float(
            np.nanmax(
                np.abs(
                    corrupt_primary_625b5
                    -
                    clean_primary_625b5
                )
            )
        )
        if clean_primary_625b5.size
        else np.nan
    )


    smoke_rows_625b5.append(
        {
            "family":
                family_625b5,

            "scenario_id":
                scenario_id_625b5,

            "sensor":
                sensor_625b5,

            "result_type":
                type(
                    result_625b5
                ).__name__,

            "corrupted_shape":
                str(
                    corrupted_625b5.shape
                ),

            "shape_exact":
                (
                    corrupted_625b5.shape
                    ==
                    OFFICIAL_TEST_RAW_624B.shape
                ),

            "columns_exact":
                (
                    list(
                        corrupted_625b5.columns
                    )
                    ==
                    list(
                        OFFICIAL_TEST_RAW_624B.columns
                    )
                ),

            "index_exact":
                corrupted_625b5.index.equals(
                    OFFICIAL_TEST_RAW_624B.index
                ),

            "target_changed_cells":
                target_changed_cells_625b5,

            "primary_changed_cells":
                primary_changed_cells_625b5,

            "target_max_abs_change":
                target_max_change_625b5,

            "primary_max_abs_change":
                primary_max_change_625b5,
        }
    )


SMOKE_TEST_RESULTS_625B5 = (
    pd.DataFrame(
        smoke_rows_625b5
    )
)


print(
    "\n--- Smoke-test corruption effects ---"
)

display(
    SMOKE_TEST_RESULTS_625B5
)


# ------------------------------------------------------------
# 6. Verify pristine official TEST object
# ------------------------------------------------------------

pd.testing.assert_frame_equal(
    OFFICIAL_TEST_RAW_624B,
    official_test_before_625b5,
    check_exact=True,
)


all_structure_exact_625b5 = bool(
    (
        SMOKE_TEST_RESULTS_625B5[
            [
                "shape_exact",
                "columns_exact",
                "index_exact",
            ]
        ]
        .to_numpy()
        .all()
    )
)


all_effective_625b5 = bool(
    (
        SMOKE_TEST_RESULTS_625B5[
            "primary_changed_cells"
        ]
        > 0
    ).all()
)


print(
    "\n============================================"
)

print(
    "STEP 6.25B5 SMOKE-TEST SUMMARY"
)

print(
    "============================================"
)

print(
    "Families tested                 :",
    len(
        SMOKE_TEST_RESULTS_625B5
    )
)

print(
    "Families                       :",
    families_625b5
)

print(
    "All TEST structures preserved  :",
    all_structure_exact_625b5
)

print(
    "All five corruptions effective :",
    all_effective_625b5
)

print(
    "Original official TEST unchanged:",
    True
)

print(
    ""
)

print(
    "Model inference executed       : False"
)

print(
    "MC-dropout executed            : False"
)

print(
    "Official RUL labels used       : False"
)

print(
    "Official TEST scored           : False"
)

print(
    "Frozen manifests modified      : False"
)

print(
    "Reliability rule modified      : False"
)

print(
    "Threshold modified             : False"
)

print(
    "Files written                  : False"
)

print(
    "=== END ==="
)

In [ ]:
# ============================================================
# Step 6.25C
# Full 315-scenario official FD001 TEST raw-effect audit
#
# Purpose:
#   Transfer every frozen corruption scenario to official TEST
#   and verify its RAW effect before any model inference.
#
# IMPORTANT:
#   - No GRU inference.
#   - No MC-dropout.
#   - No official RUL labels.
#   - No scoring.
#   - No reliability-threshold changes.
#   - No files written.
#   - Frozen manifests remain untouched.
# ============================================================

import copy
import numpy as np
import pandas as pd


print(
    "=== Step 6.25C — Full 315-scenario "
    "official FD001 TEST raw-effect audit ==="
)


# ------------------------------------------------------------
# 1. Required objects
# ------------------------------------------------------------

required_objects_625c = [
    "OFFICIAL_TEST_RAW_624B",
    "scenario_manifests_620gc",
    "PRIMARY_INPUT_COLUMNS",
    "apply_frozen_corruption_test_625b5",
    "RELOADED_RULE_623DL5",
    "RELOADED_THRESHOLD_623DL5",
    "RELOADED_TARGET_COVERAGE_623DL5",
]

print("\n--- Required objects ---")

for name_625c in required_objects_625c:

    available_625c = (
        name_625c in globals()
    )

    print(
        f"{name_625c:<40}: "
        f"{available_625c}"
    )

    if not available_625c:
        raise RuntimeError(
            f"Missing required object: "
            f"{name_625c}"
        )


# ------------------------------------------------------------
# 2. Frozen family/count contract
# ------------------------------------------------------------

family_order_625c = [
    "bias",
    "noise",
    "drift",
    "dropout",
    "stuck",
]

expected_counts_625c = {
    "bias": 90,
    "noise": 45,
    "drift": 90,
    "dropout": 45,
    "stuck": 45,
}


print(
    "\n--- Frozen manifest counts ---"
)

total_manifest_scenarios_625c = 0

for family_625c in family_order_625c:

    manifest_625c = (
        scenario_manifests_620gc[
            family_625c
        ]
    )

    count_625c = len(
        manifest_625c
    )

    total_manifest_scenarios_625c += (
        count_625c
    )

    print(
        f"{family_625c:<10}: "
        f"{count_625c}"
    )

    if (
        count_625c
        != expected_counts_625c[
            family_625c
        ]
    ):
        raise RuntimeError(
            f"Unexpected frozen count for "
            f"{family_625c}: "
            f"{count_625c}"
        )


if total_manifest_scenarios_625c != 315:
    raise RuntimeError(
        "Frozen scenario total is not 315."
    )


# ------------------------------------------------------------
# 3. Integrity snapshots
# ------------------------------------------------------------

official_test_snapshot_625c = (
    OFFICIAL_TEST_RAW_624B.copy(
        deep=True
    )
)

manifest_snapshots_625c = {
    family: (
        scenario_manifests_620gc[
            family
        ].copy(
            deep=True
        )
    )
    for family
    in family_order_625c
}


rule_snapshot_625c = repr(
    RELOADED_RULE_623DL5
)

threshold_snapshot_625c = float(
    RELOADED_THRESHOLD_623DL5
)

coverage_snapshot_625c = float(
    RELOADED_TARGET_COVERAGE_623DL5
)


# ------------------------------------------------------------
# 4. Clean reference arrays
# ------------------------------------------------------------

numeric_columns_625c = (
    OFFICIAL_TEST_RAW_624B
    .select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


clean_primary_array_625c = (
    OFFICIAL_TEST_RAW_624B[
        PRIMARY_INPUT_COLUMNS
    ]
    .to_numpy(
        dtype=np.float64
    )
)


clean_numeric_array_625c = (
    OFFICIAL_TEST_RAW_624B[
        numeric_columns_625c
    ]
    .to_numpy(
        dtype=np.float64
    )
)


print(
    "\n--- Official TEST reference structure ---"
)

print(
    "Rows                 :",
    len(
        OFFICIAL_TEST_RAW_624B
    )
)

print(
    "Columns              :",
    OFFICIAL_TEST_RAW_624B.shape[1]
)

print(
    "Primary inputs       :",
    len(
        PRIMARY_INPUT_COLUMNS
    )
)

print(
    "Numeric columns      :",
    len(
        numeric_columns_625c
    )
)


# ------------------------------------------------------------
# 5. Audit all 315 scenarios
# ------------------------------------------------------------

audit_rows_625c = []

global_index_625c = 0


print(
    "\n--- Auditing 315 frozen scenarios on "
    "official TEST raw data ---"
)


for family_625c in family_order_625c:

    manifest_625c = (
        scenario_manifests_620gc[
            family_625c
        ]
    )

    print(
        f"\nAuditing "
        f"{family_625c.upper()} "
        f"({len(manifest_625c)} scenarios)..."
    )


    for family_position_625c in range(
        len(
            manifest_625c
        )
    ):

        scenario_row_625c = (
            manifest_625c.iloc[
                family_position_625c
            ]
        )

        sensor_625c = str(
            scenario_row_625c[
                "sensor"
            ]
        )


        # --------------------------------------------
        # Preserve existing manifest scenario ID
        # where one exists.
        # --------------------------------------------

        if (
            "scenario_id"
            in scenario_row_625c.index
            and pd.notna(
                scenario_row_625c[
                    "scenario_id"
                ]
            )
        ):

            manifest_scenario_id_625c = (
                str(
                    scenario_row_625c[
                        "scenario_id"
                    ]
                )
            )

        else:
            manifest_scenario_id_625c = None


        # Internal audit label only.
        # We do NOT modify the frozen manifest.
        evaluation_label_625c = (
            manifest_scenario_id_625c
            if manifest_scenario_id_625c
            is not None
            else
            (
                f"{family_625c}_"
                f"{family_position_625c + 1:03d}"
            )
        )


        # --------------------------------------------
        # Apply exact frozen corruption
        # --------------------------------------------

        result_625c, corrupted_625c = (
            apply_frozen_corruption_test_625b5(
                base_df=OFFICIAL_TEST_RAW_624B,
                family=family_625c,
                scenario_row=scenario_row_625c,
            )
        )


        # --------------------------------------------
        # Structural checks
        # --------------------------------------------

        shape_exact_625c = (
            corrupted_625c.shape
            ==
            OFFICIAL_TEST_RAW_624B.shape
        )

        columns_exact_625c = (
            list(
                corrupted_625c.columns
            )
            ==
            list(
                OFFICIAL_TEST_RAW_624B.columns
            )
        )

        index_exact_625c = (
            corrupted_625c.index.equals(
                OFFICIAL_TEST_RAW_624B.index
            )
        )


        # --------------------------------------------
        # Target-sensor raw effect
        # --------------------------------------------

        clean_target_625c = (
            OFFICIAL_TEST_RAW_624B[
                sensor_625c
            ]
            .to_numpy(
                dtype=np.float64
            )
        )

        corrupt_target_625c = (
            corrupted_625c[
                sensor_625c
            ]
            .to_numpy(
                dtype=np.float64
            )
        )


        target_change_mask_625c = (
            ~np.isclose(
                clean_target_625c,
                corrupt_target_625c,
                rtol=0.0,
                atol=0.0,
                equal_nan=True,
            )
        )


        target_changed_cells_625c = int(
            target_change_mask_625c.sum()
        )


        target_changed_fraction_625c = (
            target_changed_cells_625c
            /
            len(
                OFFICIAL_TEST_RAW_624B
            )
        )


        target_max_abs_change_625c = (
            float(
                np.nanmax(
                    np.abs(
                        corrupt_target_625c
                        -
                        clean_target_625c
                    )
                )
            )
        )


        # --------------------------------------------
        # Primary-model-input effect
        # --------------------------------------------

        corrupt_primary_625c = (
            corrupted_625c[
                PRIMARY_INPUT_COLUMNS
            ]
            .to_numpy(
                dtype=np.float64
            )
        )


        primary_change_mask_625c = (
            ~np.isclose(
                clean_primary_array_625c,
                corrupt_primary_625c,
                rtol=0.0,
                atol=0.0,
                equal_nan=True,
            )
        )


        primary_changed_cells_625c = int(
            primary_change_mask_625c.sum()
        )


        primary_max_abs_change_625c = (
            float(
                np.nanmax(
                    np.abs(
                        corrupt_primary_625c
                        -
                        clean_primary_array_625c
                    )
                )
            )
        )


        primary_finite_625c = bool(
            np.isfinite(
                corrupt_primary_625c
            ).all()
        )


        # --------------------------------------------
        # Entire numeric raw-data effect
        # --------------------------------------------

        corrupt_numeric_625c = (
            corrupted_625c[
                numeric_columns_625c
            ]
            .to_numpy(
                dtype=np.float64
            )
        )


        all_numeric_change_mask_625c = (
            ~np.isclose(
                clean_numeric_array_625c,
                corrupt_numeric_625c,
                rtol=0.0,
                atol=0.0,
                equal_nan=True,
            )
        )


        all_numeric_changed_cells_625c = int(
            all_numeric_change_mask_625c.sum()
        )


        # Every corruption is supposed to operate
        # only on its declared target sensor.
        off_target_changed_cells_625c = int(
            all_numeric_changed_cells_625c
            -
            target_changed_cells_625c
        )


        sensor_is_primary_625c = (
            sensor_625c
            in PRIMARY_INPUT_COLUMNS
        )


        effective_primary_625c = (
            primary_changed_cells_625c
            > 0
        )


        # --------------------------------------------
        # Parameter capture for audit only
        # --------------------------------------------

        severity_625c = (
            scenario_row_625c.get(
                "severity",
                None
            )
        )

        alpha_625c = (
            scenario_row_625c.get(
                "alpha",
                np.nan
            )
        )

        direction_625c = (
            scenario_row_625c.get(
                "direction",
                np.nan
            )
        )

        requested_fraction_625c = (
            scenario_row_625c.get(
                "requested_fraction",
                np.nan
            )
        )


        audit_rows_625c.append(
            {
                "global_index":
                    global_index_625c,

                "family":
                    family_625c,

                "family_position":
                    family_position_625c,

                "evaluation_label":
                    evaluation_label_625c,

                "manifest_scenario_id":
                    manifest_scenario_id_625c,

                "sensor":
                    sensor_625c,

                "severity":
                    severity_625c,

                "alpha":
                    alpha_625c,

                "direction":
                    direction_625c,

                "requested_fraction":
                    requested_fraction_625c,

                "shape_exact":
                    shape_exact_625c,

                "columns_exact":
                    columns_exact_625c,

                "index_exact":
                    index_exact_625c,

                "sensor_is_primary":
                    sensor_is_primary_625c,

                "target_changed_cells":
                    target_changed_cells_625c,

                "target_changed_fraction":
                    target_changed_fraction_625c,

                "primary_changed_cells":
                    primary_changed_cells_625c,

                "all_numeric_changed_cells":
                    all_numeric_changed_cells_625c,

                "off_target_changed_cells":
                    off_target_changed_cells_625c,

                "target_max_abs_change":
                    target_max_abs_change_625c,

                "primary_max_abs_change":
                    primary_max_abs_change_625c,

                "primary_finite":
                    primary_finite_625c,

                "effective_primary":
                    effective_primary_625c,
            }
        )


        global_index_625c += 1


        # Explicitly release temporary scenario objects.
        del result_625c
        del corrupted_625c
        del corrupt_primary_625c
        del corrupt_numeric_625c


# ------------------------------------------------------------
# 6. Construct audit DataFrame
# ------------------------------------------------------------

OFFICIAL_TEST_RAW_EFFECT_AUDIT_625C = (
    pd.DataFrame(
        audit_rows_625c
    )
)


if len(
    OFFICIAL_TEST_RAW_EFFECT_AUDIT_625C
) != 315:
    raise RuntimeError(
        "Audit did not produce exactly "
        "315 scenario records."
    )


# ------------------------------------------------------------
# 7. Hard structural/integrity checks
# ------------------------------------------------------------

structural_ok_625c = bool(
    (
        OFFICIAL_TEST_RAW_EFFECT_AUDIT_625C[
            [
                "shape_exact",
                "columns_exact",
                "index_exact",
                "sensor_is_primary",
                "primary_finite",
            ]
        ]
        .to_numpy()
        .all()
    )
)


off_target_violation_count_625c = int(
    (
        OFFICIAL_TEST_RAW_EFFECT_AUDIT_625C[
            "off_target_changed_cells"
        ]
        != 0
    )
    .sum()
)


if not structural_ok_625c:
    raise RuntimeError(
        "At least one scenario failed a "
        "TEST structural/integrity check."
    )


if off_target_violation_count_625c != 0:
    raise RuntimeError(
        "At least one corruption modified "
        "non-target numeric cells."
    )


# ------------------------------------------------------------
# 8. Verify pristine TEST and frozen objects unchanged
# ------------------------------------------------------------

pd.testing.assert_frame_equal(
    OFFICIAL_TEST_RAW_624B,
    official_test_snapshot_625c,
    check_exact=True,
)


manifests_unchanged_625c = True

for family_625c in family_order_625c:

    try:

        pd.testing.assert_frame_equal(
            scenario_manifests_620gc[
                family_625c
            ],
            manifest_snapshots_625c[
                family_625c
            ],
            check_exact=True,
        )

    except AssertionError:

        manifests_unchanged_625c = False


rule_unchanged_625c = (
    repr(
        RELOADED_RULE_623DL5
    )
    ==
    rule_snapshot_625c
)


threshold_unchanged_625c = (
    float(
        RELOADED_THRESHOLD_623DL5
    )
    ==
    threshold_snapshot_625c
)


coverage_unchanged_625c = (
    float(
        RELOADED_TARGET_COVERAGE_623DL5
    )
    ==
    coverage_snapshot_625c
)


# ------------------------------------------------------------
# 9. Family-level summary
# ------------------------------------------------------------

family_summary_rows_625c = []

for family_625c in family_order_625c:

    group_625c = (
        OFFICIAL_TEST_RAW_EFFECT_AUDIT_625C[
            OFFICIAL_TEST_RAW_EFFECT_AUDIT_625C[
                "family"
            ]
            ==
            family_625c
        ]
    )

    family_summary_rows_625c.append(
        {
            "family":
                family_625c,

            "scenarios":
                len(
                    group_625c
                ),

            "effective_primary":
                int(
                    group_625c[
                        "effective_primary"
                    ]
                    .sum()
                ),

            "no_op_primary":
                int(
                    (
                        ~group_625c[
                            "effective_primary"
                        ]
                    )
                    .sum()
                ),

            "off_target_violations":
                int(
                    (
                        group_625c[
                            "off_target_changed_cells"
                        ]
                        != 0
                    )
                    .sum()
                ),

            "nonfinite_primary":
                int(
                    (
                        ~group_625c[
                            "primary_finite"
                        ]
                    )
                    .sum()
                ),
        }
    )


OFFICIAL_TEST_FAMILY_AUDIT_625C = (
    pd.DataFrame(
        family_summary_rows_625c
    )
)


print(
    "\n--- Family-level TEST raw-effect validity ---"
)

display(
    OFFICIAL_TEST_FAMILY_AUDIT_625C
)


# ------------------------------------------------------------
# 10. Explicit no-op report
# ------------------------------------------------------------

TEST_NO_OP_SCENARIOS_625C = (
    OFFICIAL_TEST_RAW_EFFECT_AUDIT_625C[
        ~OFFICIAL_TEST_RAW_EFFECT_AUDIT_625C[
            "effective_primary"
        ]
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


print(
    "\n--- Official TEST model-input NO-OP scenarios ---"
)

print(
    "Count:",
    len(
        TEST_NO_OP_SCENARIOS_625C
    )
)


if len(
    TEST_NO_OP_SCENARIOS_625C
) > 0:

    display(
        TEST_NO_OP_SCENARIOS_625C[
            [
                "global_index",
                "family",
                "family_position",
                "evaluation_label",
                "manifest_scenario_id",
                "sensor",
                "severity",
                "target_changed_cells",
                "primary_changed_cells",
                "target_max_abs_change",
            ]
        ]
    )

else:

    print(
        "No TEST no-op scenarios found."
    )


# ------------------------------------------------------------
# 11. Known DEVELOPMENT no-op lookup
# ------------------------------------------------------------

known_dev_case_625c = (
    OFFICIAL_TEST_RAW_EFFECT_AUDIT_625C[
        (
            OFFICIAL_TEST_RAW_EFFECT_AUDIT_625C[
                "global_index"
            ]
            == 279
        )
    ]
)


print(
    "\n--- Known DEVELOPMENT index 279 "
    "TEST-transfer check ---"
)

display(
    known_dev_case_625c[
        [
            "global_index",
            "family",
            "family_position",
            "evaluation_label",
            "manifest_scenario_id",
            "sensor",
            "severity",
            "target_changed_cells",
            "primary_changed_cells",
            "target_max_abs_change",
            "effective_primary",
        ]
    ]
)


# ------------------------------------------------------------
# 12. Final compact summary
# ------------------------------------------------------------

effective_count_625c = int(
    OFFICIAL_TEST_RAW_EFFECT_AUDIT_625C[
        "effective_primary"
    ]
    .sum()
)


no_op_count_625c = int(
    len(
        OFFICIAL_TEST_RAW_EFFECT_AUDIT_625C
    )
    -
    effective_count_625c
)


print(
    "\n=============================================="
)

print(
    "STEP 6.25C OFFICIAL TEST RAW-EFFECT SUMMARY"
)

print(
    "=============================================="
)

print(
    "Frozen scenarios transferred    :",
    len(
        OFFICIAL_TEST_RAW_EFFECT_AUDIT_625C
    )
)

print(
    "Effective model-input scenarios :",
    effective_count_625c
)

print(
    "Model-input no-op scenarios     :",
    no_op_count_625c
)

print(
    "Structural checks all valid     :",
    structural_ok_625c
)

print(
    "Off-target corruption cases     :",
    off_target_violation_count_625c
)

print(
    "Official TEST base unchanged    :",
    True
)

print(
    "Frozen manifests unchanged      :",
    manifests_unchanged_625c
)

print(
    "Frozen reliability rule unchanged:",
    rule_unchanged_625c
)

print(
    "Frozen threshold unchanged      :",
    threshold_unchanged_625c
)

print(
    "Frozen target coverage unchanged:",
    coverage_unchanged_625c
)

print("")

print(
    "Model inference executed        : False"
)

print(
    "MC-dropout executed             : False"
)

print(
    "Official RUL labels used        : False"
)

print(
    "Official TEST scored            : False"
)

print(
    "Files written                   : False"
)

print(
    "=== END ==="
)

In [ ]:
# ============================================================
# Step 6.25C1
# Freeze official FD001 TEST corruption-evaluation scenario set
#
# Purpose:
#   Persist the verified 315-scenario raw-effect audit BEFORE
#   any corrupted-model inference or official-label scoring.
#
# IMPORTANT:
#   - No corruption rerun.
#   - No model inference.
#   - No MC-dropout.
#   - No official RUL labels.
#   - No scoring.
#   - No threshold/rule changes.
# ============================================================

from pathlib import Path
import hashlib
import json
import pandas as pd
import numpy as np


print(
    "=== Step 6.25C1 — Freeze official FD001 "
    "TEST corruption evaluation set ==="
)


# ------------------------------------------------------------
# 1. Required objects
# ------------------------------------------------------------

required_objects_625c1 = [
    "OFFICIAL_TEST_RAW_EFFECT_AUDIT_625C",
    "OFFICIAL_TEST_FAMILY_AUDIT_625C",
    "RELOADED_RULE_623DL5",
    "RELOADED_THRESHOLD_623DL5",
    "RELOADED_TARGET_COVERAGE_623DL5",
]

print("\n--- Required objects ---")

for name_625c1 in required_objects_625c1:

    available_625c1 = (
        name_625c1 in globals()
    )

    print(
        f"{name_625c1:<42}: "
        f"{available_625c1}"
    )

    if not available_625c1:
        raise RuntimeError(
            f"Missing required object: "
            f"{name_625c1}"
        )


# ------------------------------------------------------------
# 2. Verify the audit contract again
# ------------------------------------------------------------

audit_625c1 = (
    OFFICIAL_TEST_RAW_EFFECT_AUDIT_625C
    .copy(deep=True)
)

family_audit_625c1 = (
    OFFICIAL_TEST_FAMILY_AUDIT_625C
    .copy(deep=True)
)


if len(audit_625c1) != 315:
    raise RuntimeError(
        "Expected exactly 315 frozen scenarios."
    )


if audit_625c1["global_index"].nunique() != 315:
    raise RuntimeError(
        "global_index is not unique across "
        "the 315 scenarios."
    )


expected_global_indices_625c1 = set(
    range(315)
)

observed_global_indices_625c1 = set(
    audit_625c1[
        "global_index"
    ].astype(int)
)


if (
    observed_global_indices_625c1
    != expected_global_indices_625c1
):
    raise RuntimeError(
        "Frozen global scenario indices are not "
        "exactly 0..314."
    )


all_structural_valid_625c1 = bool(
    audit_625c1[
        [
            "shape_exact",
            "columns_exact",
            "index_exact",
            "sensor_is_primary",
            "primary_finite",
        ]
    ]
    .to_numpy()
    .all()
)


all_effective_625c1 = bool(
    audit_625c1[
        "effective_primary"
    ]
    .all()
)


no_off_target_625c1 = bool(
    (
        audit_625c1[
            "off_target_changed_cells"
        ]
        == 0
    )
    .all()
)


if not all_structural_valid_625c1:
    raise RuntimeError(
        "Structural TEST corruption audit "
        "is not fully valid."
    )


if not all_effective_625c1:
    raise RuntimeError(
        "At least one official TEST corruption "
        "scenario is not effective."
    )


if not no_off_target_625c1:
    raise RuntimeError(
        "At least one scenario modified "
        "off-target numeric cells."
    )


# ------------------------------------------------------------
# 3. Freeze inclusion policy
# ------------------------------------------------------------
#
# IMPORTANT:
# Every one of the original 315 frozen scenarios
# is retained for official TEST evaluation.
#
# We do NOT re-select scenarios using TEST performance.
# ------------------------------------------------------------

audit_625c1[
    "include_in_official_test_evaluation"
] = True


audit_625c1[
    "known_development_noop_case"
] = (
    audit_625c1[
        "global_index"
    ]
    .astype(int)
    == 279
)


included_count_625c1 = int(
    audit_625c1[
        "include_in_official_test_evaluation"
    ]
    .sum()
)


if included_count_625c1 != 315:
    raise RuntimeError(
        "Official TEST evaluation set "
        "must contain all 315 scenarios."
    )


# ------------------------------------------------------------
# 4. Verify known DEVELOPMENT no-op transfer case
# ------------------------------------------------------------

index279_625c1 = (
    audit_625c1[
        audit_625c1[
            "global_index"
        ]
        .astype(int)
        == 279
    ]
    .copy()
)


if len(index279_625c1) != 1:
    raise RuntimeError(
        "Expected exactly one global index 279."
    )


idx279_row_625c1 = (
    index279_625c1.iloc[0]
)


if (
    str(
        idx279_row_625c1[
            "family"
        ]
    )
    != "stuck"
):
    raise RuntimeError(
        "Global index 279 family mismatch."
    )


if (
    str(
        idx279_row_625c1[
            "sensor"
        ]
    )
    != "sensor_6"
):
    raise RuntimeError(
        "Global index 279 sensor mismatch."
    )


if not bool(
    idx279_row_625c1[
        "effective_primary"
    ]
):
    raise RuntimeError(
        "Global index 279 should be effective "
        "on official TEST."
    )


# ------------------------------------------------------------
# 5. Locate project/output directory
# ------------------------------------------------------------

if "PROJECT_ROOT" in globals():

    project_root_625c1 = Path(
        PROJECT_ROOT
    )

else:

    cwd_625c1 = Path.cwd()

    candidates_625c1 = [
        cwd_625c1,
        cwd_625c1.parent,
        cwd_625c1.parent.parent,
    ]

    project_root_625c1 = None

    for candidate_625c1 in candidates_625c1:

        if (
            candidate_625c1
            / "06_results"
        ).exists():

            project_root_625c1 = (
                candidate_625c1
            )

            break


    if project_root_625c1 is None:

        raise RuntimeError(
            "Could not locate project root."
        )


output_dir_625c1 = (
    project_root_625c1
    / "06_results"
    / "step6_official_fd001"
    / "corruption_transfer"
)


output_dir_625c1.mkdir(
    parents=True,
    exist_ok=True,
)


print(
    "\nOutput directory:",
    output_dir_625c1
)


# ------------------------------------------------------------
# 6. Output paths
# ------------------------------------------------------------

audit_path_625c1 = (
    output_dir_625c1
    / "fd001_official_test_corruption_raw_effect_audit.csv"
)


family_path_625c1 = (
    output_dir_625c1
    / "fd001_official_test_corruption_family_summary.csv"
)


manifest_path_625c1 = (
    output_dir_625c1
    / "fd001_official_test_corruption_evaluation_manifest.json"
)


# ------------------------------------------------------------
# 7. Persist audit tables
# ------------------------------------------------------------

audit_625c1.to_csv(
    audit_path_625c1,
    index=False,
)


family_audit_625c1.to_csv(
    family_path_625c1,
    index=False,
)


# ------------------------------------------------------------
# 8. SHA-256 helper
# ------------------------------------------------------------

def sha256_file_625c1(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for block in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(block)

    return h.hexdigest()


audit_hash_625c1 = (
    sha256_file_625c1(
        audit_path_625c1
    )
)


family_hash_625c1 = (
    sha256_file_625c1(
        family_path_625c1
    )
)


# ------------------------------------------------------------
# 9. Build frozen TEST-evaluation manifest
# ------------------------------------------------------------

manifest_625c1 = {

    "dataset":
        "NASA C-MAPSS FD001",

    "evaluation_stage":
        "official_test_corruption_transfer",

    "scenario_source":
        "frozen development corruption manifests",

    "frozen_scenarios_total":
        315,

    "official_test_scenarios_included":
        315,

    "scenario_global_indices":
        "0..314",

    "inclusion_policy":
        (
            "Retain the complete frozen 315-scenario "
            "manifest set for official TEST evaluation. "
            "No scenario is selected or removed using "
            "official TEST prediction performance or "
            "official TEST RUL labels."
        ),

    "families": {
        "bias": 90,
        "noise": 45,
        "drift": 90,
        "dropout": 45,
        "stuck": 45,
    },

    "raw_effect_audit": {
        "all_315_effective_on_primary_inputs":
            all_effective_625c1,

        "all_structural_checks_valid":
            all_structural_valid_625c1,

        "off_target_corruption_cases":
            int(
                (
                    audit_625c1[
                        "off_target_changed_cells"
                    ]
                    != 0
                )
                .sum()
            ),

        "nonfinite_primary_cases":
            int(
                (
                    ~audit_625c1[
                        "primary_finite"
                    ]
                )
                .sum()
            ),
    },

    "development_noop_transfer_case": {
        "global_index": 279,

        "manifest_scenario_id":
            str(
                idx279_row_625c1[
                    "manifest_scenario_id"
                ]
            ),

        "family":
            str(
                idx279_row_625c1[
                    "family"
                ]
            ),

        "sensor":
            str(
                idx279_row_625c1[
                    "sensor"
                ]
            ),

        "severity":
            str(
                idx279_row_625c1[
                    "severity"
                ]
            ),

        "official_test_target_changed_cells":
            int(
                idx279_row_625c1[
                    "target_changed_cells"
                ]
            ),

        "official_test_primary_changed_cells":
            int(
                idx279_row_625c1[
                    "primary_changed_cells"
                ]
            ),

        "official_test_target_max_abs_change":
            float(
                idx279_row_625c1[
                    "target_max_abs_change"
                ]
            ),

        "official_test_effective":
            bool(
                idx279_row_625c1[
                    "effective_primary"
                ]
            ),
    },

    "frozen_reliability_decision": {
        "signal":
            "uncertainty_only",

        "threshold":
            float(
                RELOADED_THRESHOLD_623DL5
            ),

        "target_development_coverage":
            float(
                RELOADED_TARGET_COVERAGE_623DL5
            ),

        "rule_repr":
            repr(
                RELOADED_RULE_623DL5
            ),
    },

    "leakage_safety": {
        "official_test_rul_labels_used":
            False,

        "official_test_prediction_performance_used":
            False,

        "model_inference_executed_in_this_step":
            False,

        "mc_dropout_executed_in_this_step":
            False,

        "threshold_modified":
            False,

        "reliability_rule_modified":
            False,

        "corruption_manifests_modified":
            False,
    },

    "artifact_hashes": {
        audit_path_625c1.name:
            audit_hash_625c1,

        family_path_625c1.name:
            family_hash_625c1,
    },
}


with open(
    manifest_path_625c1,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        manifest_625c1,
        f,
        indent=2,
    )


# ------------------------------------------------------------
# 10. Reload / verification
# ------------------------------------------------------------

reloaded_audit_625c1 = (
    pd.read_csv(
        audit_path_625c1
    )
)


reloaded_family_625c1 = (
    pd.read_csv(
        family_path_625c1
    )
)


with open(
    manifest_path_625c1,
    "r",
    encoding="utf-8",
) as f:

    reloaded_manifest_625c1 = (
        json.load(f)
    )


audit_hash_valid_625c1 = (
    sha256_file_625c1(
        audit_path_625c1
    )
    ==
    audit_hash_625c1
)


family_hash_valid_625c1 = (
    sha256_file_625c1(
        family_path_625c1
    )
    ==
    family_hash_625c1
)


rows_valid_625c1 = (
    len(
        reloaded_audit_625c1
    )
    == 315
)


included_valid_625c1 = (
    int(
        reloaded_audit_625c1[
            "include_in_official_test_evaluation"
        ]
        .sum()
    )
    == 315
)


manifest_count_valid_625c1 = (
    int(
        reloaded_manifest_625c1[
            "official_test_scenarios_included"
        ]
    )
    == 315
)


# ------------------------------------------------------------
# 11. Final compact verification
# ------------------------------------------------------------

print(
    "\n=============================================="
)

print(
    "STEP 6.25C1 FROZEN TEST-SCENARIO SUMMARY"
)

print(
    "=============================================="
)

print(
    "Stored audit rows                 :",
    len(
        reloaded_audit_625c1
    )
)

print(
    "Included TEST scenarios           :",
    int(
        reloaded_audit_625c1[
            "include_in_official_test_evaluation"
        ]
        .sum()
    )
)

print(
    "All TEST corruptions effective    :",
    all_effective_625c1
)

print(
    "All structural checks valid       :",
    all_structural_valid_625c1
)

print(
    "Off-target corruption cases       :",
    int(
        (
            audit_625c1[
                "off_target_changed_cells"
            ]
            != 0
        )
        .sum()
    )
)

print(
    "Index 279 effective on TEST       :",
    bool(
        idx279_row_625c1[
            "effective_primary"
        ]
    )
)

print(
    "Index 279 changed cells           :",
    int(
        idx279_row_625c1[
            "primary_changed_cells"
        ]
    )
)

print(
    "Index 279 max absolute change     :",
    float(
        idx279_row_625c1[
            "target_max_abs_change"
        ]
    )
)

print(
    "Audit SHA-256 valid               :",
    audit_hash_valid_625c1
)

print(
    "Family-summary SHA-256 valid      :",
    family_hash_valid_625c1
)

print(
    "Reloaded row count valid          :",
    rows_valid_625c1
)

print(
    "Reloaded inclusion count valid    :",
    included_valid_625c1
)

print(
    "Manifest inclusion count valid    :",
    manifest_count_valid_625c1
)

print("")

print(
    "Official TEST labels used         : False"
)

print(
    "Model inference executed          : False"
)

print(
    "MC-dropout executed               : False"
)

print(
    "Reliability rule modified         : False"
)

print(
    "Threshold modified                : False"
)

print("")

print(
    "Frozen files:"
)

print(
    "1.",
    audit_path_625c1
)

print(
    "2.",
    family_path_625c1
)

print(
    "3.",
    manifest_path_625c1
)

print(
    "=== END ==="
)

In [ ]:
# ============================================================
# Step 6.25D
# Five-family end-to-end corrupted official-TEST smoke test
#
# PURPOSE
# -------
# Verify the COMPLETE frozen TEST pathway on one scenario from
# each corruption family:
#
# raw TEST
#   -> frozen corruption
#   -> frozen scaler / feature order
#   -> endpoint sequence
#   -> frozen GRU
#   -> MC-dropout
#   -> frozen reliability score
#   -> frozen accept/abstain rule
#
# IMPORTANT
# ---------
# - Official RUL labels are NOT accessed.
# - No threshold selection.
# - No reliability-score selection.
# - No corruption parameter selection.
# - No model training.
# - No files written.
# ============================================================

import numpy as np
import pandas as pd


print(
    "=== Step 6.25D — Five-family end-to-end "
    "official TEST inference smoke test ==="
)


# ------------------------------------------------------------
# 1. Required objects
# ------------------------------------------------------------

required_objects_625d = [
    "OFFICIAL_TEST_RAW_624B",
    "OFFICIAL_TEST_RAW_EFFECT_AUDIT_625C",
    "scenario_manifests_620gc",
    "apply_frozen_corruption_test_625b5",
    "PRIMARY_INPUT_COLUMNS",
    "scaler",
    "X_TEST_CLEAN_624C",
    "OFFICIAL_TEST_SEQ_META_624C",
    "OFFICIAL_TEST_ENDPOINT_INFERENCE_624E",
    "mc_uncertainty_623cb2",
    "RELOADED_THRESHOLD_623DL5",
    "RELOADED_RULE_623DL5",
]

print("\n--- Required objects ---")

for name_625d in required_objects_625d:

    available_625d = (
        name_625d in globals()
    )

    print(
        f"{name_625d:<42}: "
        f"{available_625d}"
    )

    if not available_625d:
        raise RuntimeError(
            f"Missing required object: {name_625d}"
        )


# ------------------------------------------------------------
# 2. Freeze clean raw snapshot
# ------------------------------------------------------------

official_test_snapshot_625d = (
    OFFICIAL_TEST_RAW_624B
    .copy(deep=True)
)


# ------------------------------------------------------------
# 3. Recover frozen CLEAN endpoint sequences
# ------------------------------------------------------------

seq_meta_625d = (
    OFFICIAL_TEST_SEQ_META_624C
    .copy()
)


endpoint_meta_625d = (
    seq_meta_625d
    .sort_values(
        ["unit_id", "cycle"]
    )
    .groupby(
        "unit_id",
        sort=True,
        as_index=False,
    )
    .tail(1)
    .sort_values("unit_id")
)


endpoint_positions_625d = (
    endpoint_meta_625d
    .index
    .to_numpy(dtype=int)
)


X_clean_endpoint_frozen_625d = (
    np.asarray(
        X_TEST_CLEAN_624C[
            endpoint_positions_625d
        ],
        dtype=np.float32,
    )
)


if X_clean_endpoint_frozen_625d.shape != (
    100,
    30,
    17,
):
    raise RuntimeError(
        "Unexpected frozen clean endpoint shape: "
        f"{X_clean_endpoint_frozen_625d.shape}"
    )


# ------------------------------------------------------------
# 4. Label-free TEST endpoint preprocessing helper
# ------------------------------------------------------------

def build_endpoint_sequences_625d(
    raw_df,
):

    """
    Build one final 30-cycle model sequence per TEST engine.

    No RUL labels are used.
    """

    required_cols = (
        ["unit_id", "cycle"]
        + list(PRIMARY_INPUT_COLUMNS)
    )

    missing_cols = [
        col
        for col in required_cols
        if col not in raw_df.columns
    ]

    if missing_cols:
        raise RuntimeError(
            "Missing TEST preprocessing columns: "
            + str(missing_cols)
        )

    work_df = (
        raw_df[
            required_cols
        ]
        .copy()
    )

    # Important:
    # same frozen StandardScaler,
    # same 17-feature order.
    scaled_values = scaler.transform(
        work_df[
            PRIMARY_INPUT_COLUMNS
        ]
    )

    scaled_df = pd.DataFrame(
        scaled_values,
        columns=PRIMARY_INPUT_COLUMNS,
        index=work_df.index,
    )

    scaled_df.insert(
        0,
        "cycle",
        work_df["cycle"].to_numpy(),
    )

    scaled_df.insert(
        0,
        "unit_id",
        work_df["unit_id"].to_numpy(),
    )


    sequences = []
    meta_records = []


    for unit_id, group in scaled_df.groupby(
        "unit_id",
        sort=True,
    ):

        group = (
            group
            .sort_values("cycle")
        )

        if len(group) < 30:
            raise RuntimeError(
                f"Engine {unit_id} has fewer "
                "than 30 cycles."
            )

        endpoint_block = (
            group[
                PRIMARY_INPUT_COLUMNS
            ]
            .tail(30)
            .to_numpy(
                dtype=np.float32
            )
        )

        if endpoint_block.shape != (
            30,
            len(PRIMARY_INPUT_COLUMNS),
        ):
            raise RuntimeError(
                "Endpoint block has unexpected "
                f"shape for engine {unit_id}: "
                f"{endpoint_block.shape}"
            )

        sequences.append(
            endpoint_block
        )

        meta_records.append(
            {
                "unit_id": int(unit_id),
                "cycle": int(
                    group[
                        "cycle"
                    ].iloc[-1]
                ),
            }
        )


    X_endpoint = np.stack(
        sequences,
        axis=0,
    )

    endpoint_meta = pd.DataFrame(
        meta_records
    )


    if not np.isfinite(
        X_endpoint
    ).all():
        raise RuntimeError(
            "Non-finite values found in "
            "endpoint sequences."
        )


    return (
        X_endpoint,
        endpoint_meta,
    )


# ------------------------------------------------------------
# 5. Clean preprocessing reproduction check
# ------------------------------------------------------------

(
    X_clean_rebuilt_625d,
    clean_meta_rebuilt_625d,
) = build_endpoint_sequences_625d(
    OFFICIAL_TEST_RAW_624B
)


clean_sequence_max_error_625d = float(
    np.max(
        np.abs(
            X_clean_rebuilt_625d
            - X_clean_endpoint_frozen_625d
        )
    )
)


clean_id_match_625d = bool(
    np.array_equal(
        clean_meta_rebuilt_625d[
            "unit_id"
        ].to_numpy(),
        endpoint_meta_625d[
            "unit_id"
        ].to_numpy(),
    )
)


clean_cycle_match_625d = bool(
    np.array_equal(
        clean_meta_rebuilt_625d[
            "cycle"
        ].to_numpy(),
        endpoint_meta_625d[
            "cycle"
        ].to_numpy(),
    )
)


print(
    "\n--- Clean TEST preprocessing reproduction ---"
)

print(
    "Frozen endpoint shape      :",
    X_clean_endpoint_frozen_625d.shape
)

print(
    "Rebuilt endpoint shape     :",
    X_clean_rebuilt_625d.shape
)

print(
    "Maximum sequence difference:",
    clean_sequence_max_error_625d
)

print(
    "Endpoint engine IDs exact  :",
    clean_id_match_625d
)

print(
    "Endpoint cycles exact      :",
    clean_cycle_match_625d
)


if clean_sequence_max_error_625d != 0.0:
    raise RuntimeError(
        "Clean TEST preprocessing does not "
        "exactly reproduce the frozen pathway."
    )


if not clean_id_match_625d:
    raise RuntimeError(
        "Clean endpoint engine-ID order differs "
        "from frozen endpoint order."
    )


if not clean_cycle_match_625d:
    raise RuntimeError(
        "Clean endpoint cycle order differs "
        "from frozen endpoint order."
    )


# ------------------------------------------------------------
# 6. Clean frozen endpoint inference reference
# ------------------------------------------------------------

clean_inference_625d = (
    OFFICIAL_TEST_ENDPOINT_INFERENCE_624E
    .copy()
    .sort_values("unit_id")
    .reset_index(drop=True)
)


if len(clean_inference_625d) != 100:
    raise RuntimeError(
        "Expected 100 clean endpoint "
        "inference records."
    )


required_clean_inference_cols_625d = [
    "unit_id",
    "cycle",
    "prediction",
    "mc_std",
    "reliability_score",
    "accept",
]


missing_clean_cols_625d = [
    col
    for col in required_clean_inference_cols_625d
    if col not in clean_inference_625d.columns
]


if missing_clean_cols_625d:
    raise RuntimeError(
        "Missing clean inference fields: "
        + str(missing_clean_cols_625d)
    )


clean_prediction_625d = (
    clean_inference_625d[
        "prediction"
    ]
    .to_numpy(dtype=float)
)


clean_mc_std_625d = (
    clean_inference_625d[
        "mc_std"
    ]
    .to_numpy(dtype=float)
)


clean_reliability_625d = (
    clean_inference_625d[
        "reliability_score"
    ]
    .to_numpy(dtype=float)
)


clean_accept_625d = (
    clean_inference_625d[
        "accept"
    ]
    .astype(bool)
    .to_numpy()
)


# ------------------------------------------------------------
# 7. Select exactly one frozen scenario per family
# ------------------------------------------------------------

family_order_625d = [
    "bias",
    "noise",
    "drift",
    "dropout",
    "stuck",
]


audit_625d = (
    OFFICIAL_TEST_RAW_EFFECT_AUDIT_625C
    .copy()
)


representatives_625d = {}


for family_625d in family_order_625d:

    family_audit_625d = (
        audit_625d[
            audit_625d[
                "family"
            ]
            .astype(str)
            .str.lower()
            == family_625d
        ]
        .sort_values(
            "global_index"
        )
    )

    if len(
        family_audit_625d
    ) == 0:
        raise RuntimeError(
            f"No audit rows found for "
            f"{family_625d}."
        )

    audit_row_625d = (
        family_audit_625d
        .iloc[0]
    )

    family_position_625d = int(
        audit_row_625d[
            "family_position"
        ]
    )

    manifest_df_625d = (
        scenario_manifests_620gc[
            family_625d
        ]
    )

    if (
        family_position_625d
        >= len(manifest_df_625d)
    ):
        raise RuntimeError(
            "Family position exceeds "
            f"{family_625d} manifest."
        )

    scenario_row_625d = (
        manifest_df_625d
        .iloc[
            family_position_625d
        ]
        .copy()
    )

    representatives_625d[
        family_625d
    ] = {
        "audit_row":
            audit_row_625d,

        "scenario_row":
            scenario_row_625d,
    }


print(
    "\n--- Frozen representative scenarios ---"
)


for family_625d in family_order_625d:

    entry_625d = (
        representatives_625d[
            family_625d
        ]
    )

    audit_row_625d = (
        entry_625d[
            "audit_row"
        ]
    )

    print(
        f"{family_625d.upper():8s} | "
        f"global_index="
        f"{int(audit_row_625d['global_index'])} | "
        f"sensor="
        f"{audit_row_625d['sensor']} | "
        f"severity="
        f"{audit_row_625d['severity']}"
    )


# ------------------------------------------------------------
# 8. Run five end-to-end smoke scenarios
# ------------------------------------------------------------

smoke_records_625d = []


for family_625d in family_order_625d:

    entry_625d = (
        representatives_625d[
            family_625d
        ]
    )

    audit_row_625d = (
        entry_625d[
            "audit_row"
        ]
    )

    scenario_row_625d = (
        entry_625d[
            "scenario_row"
        ]
    )


    print(
        "\n"
        + "=" * 70
    )

    print(
        f"Running {family_625d.upper()} "
        "official-TEST smoke scenario"
    )

    print(
        "=" * 70
    )


    # --------------------------------------------------------
    # 8A. Apply exact frozen corruption on a temporary copy
    # --------------------------------------------------------

    corrupted_df_625d = (
        apply_frozen_corruption_test_625b5(
            OFFICIAL_TEST_RAW_624B,
            family_625d,
            scenario_row_625d,
        )
    )


    if not isinstance(
        corrupted_df_625d,
        pd.DataFrame,
    ):
        raise RuntimeError(
            f"{family_625d}: TEST corruption "
            "wrapper did not return a DataFrame."
        )


    if (
        corrupted_df_625d.shape
        != OFFICIAL_TEST_RAW_624B.shape
    ):
        raise RuntimeError(
            f"{family_625d}: corrupted TEST "
            "shape changed."
        )


    if list(
        corrupted_df_625d.columns
    ) != list(
        OFFICIAL_TEST_RAW_624B.columns
    ):
        raise RuntimeError(
            f"{family_625d}: TEST column "
            "order changed."
        )


    if not corrupted_df_625d.index.equals(
        OFFICIAL_TEST_RAW_624B.index
    ):
        raise RuntimeError(
            f"{family_625d}: TEST index changed."
        )


    # --------------------------------------------------------
    # 8B. Raw corruption effect
    # --------------------------------------------------------

    clean_primary_625d = (
        OFFICIAL_TEST_RAW_624B[
            PRIMARY_INPUT_COLUMNS
        ]
        .to_numpy(dtype=float)
    )

    corrupted_primary_625d = (
        corrupted_df_625d[
            PRIMARY_INPUT_COLUMNS
        ]
        .to_numpy(dtype=float)
    )


    primary_difference_625d = (
        corrupted_primary_625d
        - clean_primary_625d
    )


    changed_primary_cells_625d = int(
        np.count_nonzero(
            primary_difference_625d
        )
    )


    max_raw_change_625d = float(
        np.max(
            np.abs(
                primary_difference_625d
            )
        )
    )


    if changed_primary_cells_625d == 0:
        raise RuntimeError(
            f"{family_625d}: representative "
            "corruption produced a model-input no-op."
        )


    # --------------------------------------------------------
    # 8C. Frozen TEST preprocessing
    # --------------------------------------------------------

    (
        X_corrupted_endpoint_625d,
        corrupted_meta_625d,
    ) = build_endpoint_sequences_625d(
        corrupted_df_625d
    )


    if (
        X_corrupted_endpoint_625d.shape
        != (100, 30, 17)
    ):
        raise RuntimeError(
            f"{family_625d}: unexpected endpoint "
            f"sequence shape "
            f"{X_corrupted_endpoint_625d.shape}"
        )


    if not np.array_equal(
        corrupted_meta_625d[
            "unit_id"
        ].to_numpy(),
        clean_meta_rebuilt_625d[
            "unit_id"
        ].to_numpy(),
    ):
        raise RuntimeError(
            f"{family_625d}: endpoint engine "
            "alignment changed."
        )


    if not np.array_equal(
        corrupted_meta_625d[
            "cycle"
        ].to_numpy(),
        clean_meta_rebuilt_625d[
            "cycle"
        ].to_numpy(),
    ):
        raise RuntimeError(
            f"{family_625d}: endpoint cycle "
            "alignment changed."
        )


    sequence_difference_625d = (
        X_corrupted_endpoint_625d
        - X_clean_rebuilt_625d
    )


    changed_sequence_values_625d = int(
        np.count_nonzero(
            sequence_difference_625d
        )
    )


    max_sequence_change_625d = float(
        np.max(
            np.abs(
                sequence_difference_625d
            )
        )
    )


    # --------------------------------------------------------
    # 8D. Frozen GRU + MC-dropout + reliability score
    # --------------------------------------------------------

    mc_result_625d = (
        mc_uncertainty_623cb2(
            X_corrupted_endpoint_625d
        )
    )


    if not isinstance(
        mc_result_625d,
        (tuple, list),
    ):
        raise RuntimeError(
            f"{family_625d}: unexpected "
            "mc_uncertainty return type."
        )


    if len(
        mc_result_625d
    ) < 3:
        raise RuntimeError(
            f"{family_625d}: expected at least "
            "three MC uncertainty outputs."
        )


    corrupted_prediction_625d = (
        np.asarray(
            mc_result_625d[0],
            dtype=float,
        )
        .reshape(-1)
    )


    corrupted_mc_std_625d = (
        np.asarray(
            mc_result_625d[1],
            dtype=float,
        )
        .reshape(-1)
    )


    corrupted_reliability_625d = (
        np.asarray(
            mc_result_625d[2],
            dtype=float,
        )
        .reshape(-1)
    )


    for array_name_625d, array_625d in [
        (
            "prediction",
            corrupted_prediction_625d,
        ),
        (
            "MC-STD",
            corrupted_mc_std_625d,
        ),
        (
            "reliability",
            corrupted_reliability_625d,
        ),
    ]:

        if array_625d.shape != (100,):
            raise RuntimeError(
                f"{family_625d}: "
                f"{array_name_625d} shape "
                f"is {array_625d.shape}, "
                "expected (100,)."
            )

        if not np.isfinite(
            array_625d
        ).all():
            raise RuntimeError(
                f"{family_625d}: non-finite "
                f"{array_name_625d} values."
            )


    # --------------------------------------------------------
    # 8E. Apply FROZEN reliability threshold
    # --------------------------------------------------------

    corrupted_accept_625d = (
        corrupted_reliability_625d
        <= float(
            RELOADED_THRESHOLD_623DL5
        )
    )


    coverage_625d = float(
        np.mean(
            corrupted_accept_625d
        )
    )


    # --------------------------------------------------------
    # 8F. Compare with frozen CLEAN inference
    #     (NO LABELS involved)
    # --------------------------------------------------------

    prediction_shift_625d = (
        corrupted_prediction_625d
        - clean_prediction_625d
    )


    mc_std_shift_625d = (
        corrupted_mc_std_625d
        - clean_mc_std_625d
    )


    reliability_shift_625d = (
        corrupted_reliability_625d
        - clean_reliability_625d
    )


    accept_changes_625d = int(
        np.count_nonzero(
            corrupted_accept_625d
            != clean_accept_625d
        )
    )


    # --------------------------------------------------------
    # 8G. Scenario identifier
    # --------------------------------------------------------

    if (
        "manifest_scenario_id"
        in audit_row_625d.index
    ):
        scenario_id_625d = str(
            audit_row_625d[
                "manifest_scenario_id"
            ]
        )

    elif (
        "scenario_id"
        in scenario_row_625d.index
    ):
        scenario_id_625d = str(
            scenario_row_625d[
                "scenario_id"
            ]
        )

    else:
        scenario_id_625d = (
            f"{family_625d}_"
            f"{int(audit_row_625d['global_index']):03d}"
        )


    smoke_records_625d.append(
        {
            "family":
                family_625d,

            "global_index":
                int(
                    audit_row_625d[
                        "global_index"
                    ]
                ),

            "scenario_id":
                scenario_id_625d,

            "sensor":
                str(
                    audit_row_625d[
                        "sensor"
                    ]
                ),

            "severity":
                str(
                    audit_row_625d[
                        "severity"
                    ]
                ),

            "raw_primary_changed_cells":
                changed_primary_cells_625d,

            "raw_max_abs_change":
                max_raw_change_625d,

            "endpoint_sequence_changed_values":
                changed_sequence_values_625d,

            "endpoint_sequence_max_abs_change":
                max_sequence_change_625d,

            "mean_abs_prediction_shift":
                float(
                    np.mean(
                        np.abs(
                            prediction_shift_625d
                        )
                    )
                ),

            "max_abs_prediction_shift":
                float(
                    np.max(
                        np.abs(
                            prediction_shift_625d
                        )
                    )
                ),

            "mean_mc_std":
                float(
                    np.mean(
                        corrupted_mc_std_625d
                    )
                ),

            "mean_mc_std_shift":
                float(
                    np.mean(
                        mc_std_shift_625d
                    )
                ),

            "mean_reliability":
                float(
                    np.mean(
                        corrupted_reliability_625d
                    )
                ),

            "mean_reliability_shift":
                float(
                    np.mean(
                        reliability_shift_625d
                    )
                ),

            "accepted":
                int(
                    corrupted_accept_625d.sum()
                ),

            "abstained":
                int(
                    (
                        ~corrupted_accept_625d
                    ).sum()
                ),

            "coverage":
                coverage_625d,

            "accept_decision_changes":
                accept_changes_625d,
        }
    )


# ------------------------------------------------------------
# 9. Smoke-test evidence table
# ------------------------------------------------------------

SMOKE_TEST_RESULTS_625D = (
    pd.DataFrame(
        smoke_records_625d
    )
)


print(
    "\n--- Five-family end-to-end smoke-test results ---"
)

display(
    SMOKE_TEST_RESULTS_625D
)


# ------------------------------------------------------------
# 10. Confirm original official TEST was not modified
# ------------------------------------------------------------

official_test_unchanged_625d = bool(
    OFFICIAL_TEST_RAW_624B.equals(
        official_test_snapshot_625d
    )
)


if not official_test_unchanged_625d:
    raise RuntimeError(
        "Original official TEST DataFrame "
        "was modified."
    )


# ------------------------------------------------------------
# 11. Integrity checks
# ------------------------------------------------------------

all_families_present_625d = (
    set(
        SMOKE_TEST_RESULTS_625D[
            "family"
        ]
    )
    == set(
        family_order_625d
    )
)


all_raw_effective_625d = bool(
    (
        SMOKE_TEST_RESULTS_625D[
            "raw_primary_changed_cells"
        ]
        > 0
    )
    .all()
)


all_sequence_finite_625d = bool(
    np.isfinite(
        SMOKE_TEST_RESULTS_625D[
            [
                "endpoint_sequence_max_abs_change",
                "mean_abs_prediction_shift",
                "max_abs_prediction_shift",
                "mean_mc_std",
                "mean_reliability",
                "coverage",
            ]
        ]
        .to_numpy(dtype=float)
    )
    .all()
)


coverage_valid_625d = bool(
    (
        (
            SMOKE_TEST_RESULTS_625D[
                "coverage"
            ]
            >= 0.0
        )
        &
        (
            SMOKE_TEST_RESULTS_625D[
                "coverage"
            ]
            <= 1.0
        )
    )
    .all()
)


# ------------------------------------------------------------
# 12. Compact final summary
# ------------------------------------------------------------

print(
    "\n=============================================="
)

print(
    "STEP 6.25D END-TO-END SMOKE-TEST SUMMARY"
)

print(
    "=============================================="
)

print(
    "Families tested                    :",
    len(
        SMOKE_TEST_RESULTS_625D
    )
)

print(
    "All five families present          :",
    all_families_present_625d
)

print(
    "Clean preprocessing exact          :",
    clean_sequence_max_error_625d
    == 0.0
)

print(
    "Clean endpoint IDs exact           :",
    clean_id_match_625d
)

print(
    "Clean endpoint cycles exact        :",
    clean_cycle_match_625d
)

print(
    "All smoke corruptions effective    :",
    all_raw_effective_625d
)

print(
    "All downstream metrics finite      :",
    all_sequence_finite_625d
)

print(
    "All coverages valid                :",
    coverage_valid_625d
)

print(
    "Original official TEST unchanged   :",
    official_test_unchanged_625d
)

print(
    "Frozen reliability threshold       :",
    float(
        RELOADED_THRESHOLD_623DL5
    )
)

print("")

print(
    "Model inference executed           : True"
)

print(
    "MC-dropout executed                : True"
)

print(
    "Official TEST RUL labels accessed  : False"
)

print(
    "Official TEST scoring performed    : False"
)

print(
    "Threshold modified                 : False"
)

print(
    "Reliability rule modified          : False"
)

print(
    "Frozen manifests modified          : False"
)

print(
    "Files written                      : False"
)

print(
    "=== END ==="
)

In [ ]:
import inspect

print(
    "=== Step 6.25D-A — TEST corruption wrapper signature check ==="
)

print(
    "Signature:",
    inspect.signature(
        apply_frozen_corruption_test_625b5
    )
)

print("\nNo corruption executed.")
print("No inference executed.")
print("No files written.")

In [ ]:
# ============================================================
# Step 6.25D-B
# Inspect official-TEST corruption-wrapper return contract
# ============================================================

import pandas as pd
import numpy as np

print(
    "=== Step 6.25D-B — TEST corruption-wrapper "
    "return-contract inspection ==="
)

# ------------------------------------------------------------
# 1. Use the same representative BIAS scenario selected by 6.25D
# ------------------------------------------------------------

family_625db = "bias"

bias_manifest_625db = scenario_manifests_620gc[family_625db]

scenario_row_625db = bias_manifest_625db.iloc[0]

print("\n--- Probe scenario ---")
print("Family :", family_625db)
print("Sensor :", scenario_row_625db.get("sensor"))
print("Severity :", scenario_row_625db.get("severity"))


# ------------------------------------------------------------
# 2. Run corruption wrapper ONLY
# ------------------------------------------------------------

wrapper_result_625db = apply_frozen_corruption_test_625b5(
    OFFICIAL_TEST_RAW_624B,
    family_625db,
    scenario_row_625db,
)


# ------------------------------------------------------------
# 3. Inspect returned object
# ------------------------------------------------------------

print("\n--- Wrapper return structure ---")
print("Return type :", type(wrapper_result_625db).__name__)

if isinstance(wrapper_result_625db, (tuple, list)):
    print("Container length :", len(wrapper_result_625db))

    for i, item in enumerate(wrapper_result_625db):
        print(
            f"Component {i}: "
            f"type={type(item).__name__}, "
            f"shape={getattr(item, 'shape', None)}"
        )

elif isinstance(wrapper_result_625db, pd.DataFrame):
    print("DataFrame shape :", wrapper_result_625db.shape)

else:
    print(
        "Returned object representation:",
        repr(wrapper_result_625db)[:500]
    )


# ------------------------------------------------------------
# 4. Apply already-verified frozen extractor
# ------------------------------------------------------------

extracted_df_625db = extract_corrupted_dataframe_620gc(
    family_625db,
    wrapper_result_625db,
)

print("\n--- Extracted corrupted DataFrame ---")
print("Type :", type(extracted_df_625db).__name__)
print("Shape :", extracted_df_625db.shape)

print(
    "Shape preserved :",
    extracted_df_625db.shape
    == OFFICIAL_TEST_RAW_624B.shape,
)

print(
    "Columns preserved :",
    list(extracted_df_625db.columns)
    == list(OFFICIAL_TEST_RAW_624B.columns),
)

print(
    "Index preserved :",
    extracted_df_625db.index.equals(
        OFFICIAL_TEST_RAW_624B.index
    ),
)


# ------------------------------------------------------------
# 5. Confirm corruption is effective
# ------------------------------------------------------------

sensor_625db = str(
    scenario_row_625db["sensor"]
)

clean_values_625db = (
    OFFICIAL_TEST_RAW_624B[sensor_625db]
    .to_numpy(dtype=float)
)

corrupted_values_625db = (
    extracted_df_625db[sensor_625db]
    .to_numpy(dtype=float)
)

changed_625db = int(
    np.sum(
        ~np.isclose(
            clean_values_625db,
            corrupted_values_625db,
            rtol=0.0,
            atol=1e-12,
            equal_nan=True,
        )
    )
)

max_change_625db = float(
    np.nanmax(
        np.abs(
            corrupted_values_625db
            - clean_values_625db
        )
    )
)

print("\n--- Raw-effect check ---")
print("Target changed cells :", changed_625db)
print("Maximum abs change   :", max_change_625db)


# ------------------------------------------------------------
# 6. Integrity statement
# ------------------------------------------------------------

print("\n--- Integrity ---")
print("Model inference executed : False")
print("MC-dropout executed       : False")
print("Official labels used      : False")
print("Official TEST scored      : False")
print("Files written             : False")
print("Frozen rule modified      : False")

print("\n=== END ===")

In [ ]:
# ============================================================
# Step 6.25D-C
# Exact TEST-wrapper nested return-contract verification
# ============================================================

import pandas as pd
import numpy as np

print(
    "=== Step 6.25D-C — Exact TEST-wrapper "
    "nested return-contract verification ==="
)

family_625dc = "bias"
scenario_row_625dc = scenario_manifests_620gc[
    family_625dc
].iloc[0]

# ------------------------------------------------------------
# 1. Execute wrapper once
# ------------------------------------------------------------

wrapper_result_625dc = apply_frozen_corruption_test_625b5(
    OFFICIAL_TEST_RAW_624B,
    family_625dc,
    scenario_row_625dc,
)

print("\n--- Outer wrapper structure ---")
print(
    "Outer type   :",
    type(wrapper_result_625dc).__name__,
)
print(
    "Outer length :",
    len(wrapper_result_625dc),
)

for i, item in enumerate(wrapper_result_625dc):
    print(
        f"Outer component {i}: "
        f"type={type(item).__name__}, "
        f"shape={getattr(item, 'shape', None)}"
    )


# ------------------------------------------------------------
# 2. Inspect underlying generator result
# ------------------------------------------------------------

generator_result_625dc = wrapper_result_625dc[0]

print("\n--- Nested generator-result structure ---")
print(
    "Nested type :",
    type(generator_result_625dc).__name__,
)

if isinstance(generator_result_625dc, (tuple, list)):
    print(
        "Nested length :",
        len(generator_result_625dc),
    )

    for i, item in enumerate(generator_result_625dc):
        print(
            f"Nested component {i}: "
            f"type={type(item).__name__}, "
            f"shape={getattr(item, 'shape', None)}"
        )


# ------------------------------------------------------------
# 3. Extract using VERIFIED legacy extractor,
#    but from the nested generator result
# ------------------------------------------------------------

legacy_extracted_625dc = extract_corrupted_dataframe_620gc(
    family_625dc,
    generator_result_625dc,
)

wrapper_extracted_625dc = wrapper_result_625dc[1]

print("\n--- Two extraction routes ---")

print(
    "Legacy-extracted type  :",
    type(legacy_extracted_625dc).__name__,
)
print(
    "Legacy-extracted shape :",
    legacy_extracted_625dc.shape,
)

print(
    "Wrapper component-1 type  :",
    type(wrapper_extracted_625dc).__name__,
)
print(
    "Wrapper component-1 shape :",
    wrapper_extracted_625dc.shape,
)


# ------------------------------------------------------------
# 4. Exact equivalence check
# ------------------------------------------------------------

same_shape_625dc = (
    legacy_extracted_625dc.shape
    == wrapper_extracted_625dc.shape
)

same_columns_625dc = (
    list(legacy_extracted_625dc.columns)
    == list(wrapper_extracted_625dc.columns)
)

same_index_625dc = (
    legacy_extracted_625dc.index.equals(
        wrapper_extracted_625dc.index
    )
)

numeric_cols_625dc = (
    legacy_extracted_625dc
    .select_dtypes(include=[np.number])
    .columns
)

max_route_difference_625dc = float(
    np.nanmax(
        np.abs(
            legacy_extracted_625dc[
                numeric_cols_625dc
            ].to_numpy(dtype=float)
            -
            wrapper_extracted_625dc[
                numeric_cols_625dc
            ].to_numpy(dtype=float)
        )
    )
)

print("\n--- Route equivalence ---")
print("Same shape   :", same_shape_625dc)
print("Same columns :", same_columns_625dc)
print("Same index   :", same_index_625dc)
print(
    "Maximum numeric difference :",
    max_route_difference_625dc,
)


# ------------------------------------------------------------
# 5. Verify wrapper component 1 is a valid corrupted TEST copy
# ------------------------------------------------------------

sensor_625dc = str(
    scenario_row_625dc["sensor"]
)

clean_sensor_625dc = (
    OFFICIAL_TEST_RAW_624B[
        sensor_625dc
    ].to_numpy(dtype=float)
)

corrupt_sensor_625dc = (
    wrapper_extracted_625dc[
        sensor_625dc
    ].to_numpy(dtype=float)
)

changed_cells_625dc = int(
    np.sum(
        ~np.isclose(
            clean_sensor_625dc,
            corrupt_sensor_625dc,
            rtol=0.0,
            atol=1e-12,
            equal_nan=True,
        )
    )
)

max_abs_change_625dc = float(
    np.nanmax(
        np.abs(
            corrupt_sensor_625dc
            - clean_sensor_625dc
        )
    )
)

print("\n--- Corruption effectiveness ---")
print(
    "Target sensor :",
    sensor_625dc,
)
print(
    "Changed cells :",
    changed_cells_625dc,
)
print(
    "Maximum abs change :",
    max_abs_change_625dc,
)


# ------------------------------------------------------------
# 6. Detect any off-target column changes
# ------------------------------------------------------------

changed_columns_625dc = []

for col in PRIMARY_INPUT_COLUMNS:

    clean_col = (
        OFFICIAL_TEST_RAW_624B[col]
        .to_numpy(dtype=float)
    )

    corrupt_col = (
        wrapper_extracted_625dc[col]
        .to_numpy(dtype=float)
    )

    if np.any(
        ~np.isclose(
            clean_col,
            corrupt_col,
            rtol=0.0,
            atol=1e-12,
            equal_nan=True,
        )
    ):
        changed_columns_625dc.append(col)

print("\n--- Changed primary columns ---")
print(changed_columns_625dc)

print(
    "Only intended sensor changed :",
    changed_columns_625dc == [sensor_625dc],
)


# ------------------------------------------------------------
# 7. Safety status
# ------------------------------------------------------------

print("\n--- Safety status ---")
print("Model inference executed : False")
print("MC-dropout executed       : False")
print("Official RUL labels used  : False")
print("Official TEST scored      : False")
print("Frozen rule modified      : False")
print("Threshold modified        : False")
print("Files written             : False")

print("\n=== END ===")

In [ ]:
# ============================================================
# Step 6.25D-C2
# Five-family TEST-wrapper normalized-output verification
# ============================================================

import pandas as pd
import numpy as np

print(
    "=== Step 6.25D-C2 — Five-family TEST-wrapper "
    "normalized-output verification ==="
)

families_625dc2 = [
    "bias",
    "noise",
    "drift",
    "dropout",
    "stuck",
]

records_625dc2 = []

for family_625dc2 in families_625dc2:

    scenario_row_625dc2 = (
        scenario_manifests_620gc[
            family_625dc2
        ]
        .iloc[0]
    )

    sensor_625dc2 = str(
        scenario_row_625dc2["sensor"]
    )

    wrapper_result_625dc2 = (
        apply_frozen_corruption_test_625b5(
            OFFICIAL_TEST_RAW_624B,
            family_625dc2,
            scenario_row_625dc2,
        )
    )

    outer_is_container_625dc2 = isinstance(
        wrapper_result_625dc2,
        (tuple, list),
    )

    outer_length_625dc2 = (
        len(wrapper_result_625dc2)
        if outer_is_container_625dc2
        else None
    )

    normalized_df_625dc2 = (
        wrapper_result_625dc2[1]
        if (
            outer_is_container_625dc2
            and len(wrapper_result_625dc2) >= 2
        )
        else None
    )

    normalized_is_df_625dc2 = isinstance(
        normalized_df_625dc2,
        pd.DataFrame,
    )

    if not normalized_is_df_625dc2:
        records_625dc2.append(
            {
                "family": family_625dc2,
                "sensor": sensor_625dc2,
                "outer_type": type(
                    wrapper_result_625dc2
                ).__name__,
                "outer_length": outer_length_625dc2,
                "component1_type": (
                    type(normalized_df_625dc2).__name__
                    if normalized_df_625dc2 is not None
                    else None
                ),
                "shape_exact": False,
                "columns_exact": False,
                "index_exact": False,
                "target_changed_cells": None,
                "only_target_primary_changed": False,
            }
        )
        continue

    shape_exact_625dc2 = (
        normalized_df_625dc2.shape
        == OFFICIAL_TEST_RAW_624B.shape
    )

    columns_exact_625dc2 = (
        list(normalized_df_625dc2.columns)
        ==
        list(OFFICIAL_TEST_RAW_624B.columns)
    )

    index_exact_625dc2 = (
        normalized_df_625dc2.index.equals(
            OFFICIAL_TEST_RAW_624B.index
        )
    )

    clean_target_625dc2 = (
        OFFICIAL_TEST_RAW_624B[
            sensor_625dc2
        ]
        .to_numpy(dtype=float)
    )

    corrupt_target_625dc2 = (
        normalized_df_625dc2[
            sensor_625dc2
        ]
        .to_numpy(dtype=float)
    )

    target_changed_cells_625dc2 = int(
        np.sum(
            ~np.isclose(
                clean_target_625dc2,
                corrupt_target_625dc2,
                rtol=0.0,
                atol=1e-12,
                equal_nan=True,
            )
        )
    )

    changed_primary_columns_625dc2 = []

    for col_625dc2 in PRIMARY_INPUT_COLUMNS:

        clean_col_625dc2 = (
            OFFICIAL_TEST_RAW_624B[
                col_625dc2
            ]
            .to_numpy(dtype=float)
        )

        corrupt_col_625dc2 = (
            normalized_df_625dc2[
                col_625dc2
            ]
            .to_numpy(dtype=float)
        )

        if np.any(
            ~np.isclose(
                clean_col_625dc2,
                corrupt_col_625dc2,
                rtol=0.0,
                atol=1e-12,
                equal_nan=True,
            )
        ):
            changed_primary_columns_625dc2.append(
                col_625dc2
            )

    only_target_primary_changed_625dc2 = (
        changed_primary_columns_625dc2
        == [sensor_625dc2]
    )

    records_625dc2.append(
        {
            "family": family_625dc2,
            "sensor": sensor_625dc2,
            "outer_type": type(
                wrapper_result_625dc2
            ).__name__,
            "outer_length": outer_length_625dc2,
            "component1_type": type(
                normalized_df_625dc2
            ).__name__,
            "shape_exact": shape_exact_625dc2,
            "columns_exact": columns_exact_625dc2,
            "index_exact": index_exact_625dc2,
            "target_changed_cells":
                target_changed_cells_625dc2,
            "only_target_primary_changed":
                only_target_primary_changed_625dc2,
        }
    )


TEST_WRAPPER_CONTRACT_625DC2 = pd.DataFrame(
    records_625dc2
)

print(
    "\n--- Five-family normalized TEST contract ---"
)

display(
    TEST_WRAPPER_CONTRACT_625DC2
)


# ------------------------------------------------------------
# Compact integrity summary
# ------------------------------------------------------------

all_dataframe_625dc2 = bool(
    (
        TEST_WRAPPER_CONTRACT_625DC2[
            "component1_type"
        ]
        == "DataFrame"
    ).all()
)

all_shape_exact_625dc2 = bool(
    TEST_WRAPPER_CONTRACT_625DC2[
        "shape_exact"
    ].all()
)

all_columns_exact_625dc2 = bool(
    TEST_WRAPPER_CONTRACT_625DC2[
        "columns_exact"
    ].all()
)

all_index_exact_625dc2 = bool(
    TEST_WRAPPER_CONTRACT_625DC2[
        "index_exact"
    ].all()
)

all_effective_625dc2 = bool(
    (
        TEST_WRAPPER_CONTRACT_625DC2[
            "target_changed_cells"
        ]
        > 0
    ).all()
)

all_target_only_625dc2 = bool(
    TEST_WRAPPER_CONTRACT_625DC2[
        "only_target_primary_changed"
    ].all()
)

print(
    "\n==============================================="
)
print(
    "STEP 6.25D-C2 CONTRACT SUMMARY"
)
print(
    "==============================================="
)
print(
    "Families checked                :",
    len(TEST_WRAPPER_CONTRACT_625DC2),
)
print(
    "All component-1 outputs DataFrame:",
    all_dataframe_625dc2,
)
print(
    "All TEST shapes preserved       :",
    all_shape_exact_625dc2,
)
print(
    "All TEST columns preserved      :",
    all_columns_exact_625dc2,
)
print(
    "All TEST indices preserved      :",
    all_index_exact_625dc2,
)
print(
    "All corruptions effective       :",
    all_effective_625dc2,
)
print(
    "Only intended primary changed   :",
    all_target_only_625dc2,
)

print("\nOfficial TEST labels used : False")
print("Model inference executed  : False")
print("MC-dropout executed       : False")
print("Official TEST scored      : False")
print("Frozen manifests modified : False")
print("Frozen rule modified      : False")
print("Threshold modified        : False")
print("Files written             : False")

print("\n=== END ===")

In [ ]:
# ============================================================
# Step 6.25D-R
# Corrected five-family end-to-end official TEST inference
# smoke test
# ============================================================

import numpy as np
import pandas as pd

print(
    "=== Step 6.25D-R — Corrected five-family end-to-end "
    "official TEST inference smoke test ==="
)


# ------------------------------------------------------------
# 1. Required-object verification
# ------------------------------------------------------------

required_objects_625dr = [
    "OFFICIAL_TEST_RAW_624B",
    "scenario_manifests_620gc",
    "apply_frozen_corruption_test_625b5",
    "PRIMARY_INPUT_COLUMNS",
    "scaler",
    "mc_uncertainty_623cb2",
    "OFFICIAL_TEST_ENDPOINT_INFERENCE_624E",
    "RELOADED_THRESHOLD_623DL5",
]

print("\n--- Required objects ---")

for name_625dr in required_objects_625dr:
    available_625dr = name_625dr in globals()
    print(
        f"{name_625dr:<42}: {available_625dr}"
    )

    if not available_625dr:
        raise RuntimeError(
            f"Required object missing: {name_625dr}"
        )


# ------------------------------------------------------------
# 2. Freeze threshold locally for this smoke test
# ------------------------------------------------------------

threshold_before_625dr = float(
    RELOADED_THRESHOLD_623DL5
)

print(
    "\nFrozen reliability threshold :",
    threshold_before_625dr,
)


# ------------------------------------------------------------
# 3. Small utility for numpy conversion
# ------------------------------------------------------------

def to_numpy_625dr(x):
    """
    Convert numpy / torch-like output to a flat float64 numpy
    array without modifying the source object.
    """

    if hasattr(x, "detach"):
        x = (
            x
            .detach()
            .cpu()
            .numpy()
        )

    return np.asarray(
        x,
        dtype=np.float64,
    ).reshape(-1)


# ------------------------------------------------------------
# 4. Exact endpoint-sequence constructor
#
# Uses:
#   - frozen PRIMARY_INPUT_COLUMNS
#   - frozen scaler
#   - last 30 observed cycles per official TEST engine
#
# No fitting occurs here.
# ------------------------------------------------------------

WINDOW_LENGTH_625DR = 30


def build_endpoint_sequences_625dr(raw_df):

    if not isinstance(
        raw_df,
        pd.DataFrame,
    ):
        raise TypeError(
            "Expected a pandas DataFrame."
        )

    missing_features_625dr = [
        col
        for col in PRIMARY_INPUT_COLUMNS
        if col not in raw_df.columns
    ]

    if missing_features_625dr:
        raise RuntimeError(
            "Missing primary model inputs: "
            + str(missing_features_625dr)
        )

    if "unit_id" not in raw_df.columns:
        raise RuntimeError(
            "unit_id column is missing."
        )

    if "cycle" not in raw_df.columns:
        raise RuntimeError(
            "cycle column is missing."
        )

    sequences_625dr = []
    metadata_625dr = []

    for unit_id_625dr, engine_df_625dr in (
        raw_df
        .groupby(
            "unit_id",
            sort=True,
        )
    ):

        engine_df_625dr = (
            engine_df_625dr
            .sort_values("cycle")
            .copy()
        )

        if (
            len(engine_df_625dr)
            < WINDOW_LENGTH_625DR
        ):
            raise RuntimeError(
                f"Engine {unit_id_625dr} has fewer than "
                f"{WINDOW_LENGTH_625DR} observations."
            )

        engine_features_625dr = (
            engine_df_625dr[
                PRIMARY_INPUT_COLUMNS
            ]
        )

        scaled_features_625dr = (
            scaler.transform(
                engine_features_625dr
            )
        )

        endpoint_window_625dr = (
            scaled_features_625dr[
                -WINDOW_LENGTH_625DR:
            ]
        )

        if endpoint_window_625dr.shape != (
            WINDOW_LENGTH_625DR,
            len(PRIMARY_INPUT_COLUMNS),
        ):
            raise RuntimeError(
                "Unexpected endpoint-window shape for "
                f"engine {unit_id_625dr}: "
                f"{endpoint_window_625dr.shape}"
            )

        sequences_625dr.append(
            endpoint_window_625dr
        )

        metadata_625dr.append(
            {
                "unit_id": int(
                    unit_id_625dr
                ),
                "cycle": int(
                    engine_df_625dr[
                        "cycle"
                    ].iloc[-1]
                ),
            }
        )

    X_endpoint_625dr = np.stack(
        sequences_625dr,
        axis=0,
    ).astype(
        np.float64,
        copy=False,
    )

    meta_endpoint_625dr = (
        pd.DataFrame(
            metadata_625dr
        )
        .sort_values(
            "unit_id"
        )
        .reset_index(
            drop=True
        )
    )

    return (
        X_endpoint_625dr,
        meta_endpoint_625dr,
    )


# ------------------------------------------------------------
# 5. Rebuild clean official TEST endpoint inputs
#    only for structural reference.
#
# No model inference yet.
# ------------------------------------------------------------

(
    X_clean_endpoint_625dr,
    clean_meta_625dr,
) = build_endpoint_sequences_625dr(
    OFFICIAL_TEST_RAW_624B
)

print(
    "\n--- Clean endpoint reconstruction ---"
)
print(
    "Endpoint sequence shape :",
    X_clean_endpoint_625dr.shape,
)
print(
    "Endpoint metadata shape :",
    clean_meta_625dr.shape,
)
print(
    "All sequence values finite :",
    bool(
        np.isfinite(
            X_clean_endpoint_625dr
        ).all()
    ),
)


# ------------------------------------------------------------
# 6. Verify endpoint identity against already-frozen clean
#    pre-score inference records.
# ------------------------------------------------------------

clean_inference_625dr = (
    OFFICIAL_TEST_ENDPOINT_INFERENCE_624E
    .sort_values("unit_id")
    .reset_index(drop=True)
)

if len(clean_inference_625dr) != 100:
    raise RuntimeError(
        "Expected 100 frozen clean endpoint records."
    )

if len(clean_meta_625dr) != 100:
    raise RuntimeError(
        "Expected 100 rebuilt endpoint records."
    )

clean_unit_ids_exact_625dr = bool(
    np.array_equal(
        clean_meta_625dr[
            "unit_id"
        ].to_numpy(),
        clean_inference_625dr[
            "unit_id"
        ].to_numpy(),
    )
)

clean_cycles_exact_625dr = bool(
    np.array_equal(
        clean_meta_625dr[
            "cycle"
        ].to_numpy(),
        clean_inference_625dr[
            "cycle"
        ].to_numpy(),
    )
)

print(
    "\nClean endpoint engine IDs exact :",
    clean_unit_ids_exact_625dr,
)
print(
    "Clean endpoint cycles exact     :",
    clean_cycles_exact_625dr,
)

if not (
    clean_unit_ids_exact_625dr
    and clean_cycles_exact_625dr
):
    raise RuntimeError(
        "Rebuilt endpoint metadata does not match the "
        "already-frozen clean TEST endpoint set."
    )


# ------------------------------------------------------------
# 7. Clean frozen reference vectors
# ------------------------------------------------------------

clean_prediction_625dr = (
    clean_inference_625dr[
        "prediction"
    ]
    .to_numpy(
        dtype=np.float64
    )
)

clean_reliability_625dr = (
    clean_inference_625dr[
        "reliability_score"
    ]
    .to_numpy(
        dtype=np.float64
    )
)

clean_accept_625dr = (
    clean_inference_625dr[
        "accept"
    ]
    .astype(bool)
    .to_numpy()
)

print(
    "\nFrozen CLEAN accepted endpoints :",
    int(
        clean_accept_625dr.sum()
    ),
    "/ 100",
)


# ------------------------------------------------------------
# 8. Representative scenario selection
# ------------------------------------------------------------

families_625dr = [
    "bias",
    "noise",
    "drift",
    "dropout",
    "stuck",
]


summary_records_625dr = []
detail_records_625dr = []


# ------------------------------------------------------------
# 9. Run one official-TEST scenario per family
# ------------------------------------------------------------

for family_625dr in families_625dr:

    print(
        "\n"
        + "=" * 70
    )
    print(
        f"Running {family_625dr.upper()} "
        "official-TEST smoke scenario"
    )
    print(
        "=" * 70
    )

    manifest_625dr = (
        scenario_manifests_620gc[
            family_625dr
        ]
    )

    if not isinstance(
        manifest_625dr,
        pd.DataFrame,
    ):
        raise RuntimeError(
            f"{family_625dr}: manifest is not a DataFrame."
        )

    if len(manifest_625dr) == 0:
        raise RuntimeError(
            f"{family_625dr}: manifest is empty."
        )

    scenario_row_625dr = (
        manifest_625dr
        .iloc[0]
        .copy()
    )

    sensor_625dr = str(
        scenario_row_625dr[
            "sensor"
        ]
    )

    severity_625dr = str(
        scenario_row_625dr[
            "severity"
        ]
    )

    if (
        "scenario_id"
        in scenario_row_625dr.index
        and pd.notna(
            scenario_row_625dr[
                "scenario_id"
            ]
        )
    ):
        scenario_id_625dr = str(
            scenario_row_625dr[
                "scenario_id"
            ]
        )
    else:
        scenario_id_625dr = (
            family_625dr
            + "_representative"
        )

    print(
        "Scenario :",
        scenario_id_625dr,
    )
    print(
        "Sensor   :",
        sensor_625dr,
    )
    print(
        "Severity :",
        severity_625dr,
    )


    # --------------------------------------------------------
    # 9A. Apply exact frozen corruption to temporary TEST copy
    # --------------------------------------------------------

    wrapper_result_625dr = (
        apply_frozen_corruption_test_625b5(
            OFFICIAL_TEST_RAW_624B,
            family_625dr,
            scenario_row_625dr,
        )
    )

    if not isinstance(
        wrapper_result_625dr,
        (tuple, list),
    ):
        raise RuntimeError(
            f"{family_625dr}: TEST wrapper did not "
            "return a tuple/list."
        )

    if len(
        wrapper_result_625dr
    ) < 2:
        raise RuntimeError(
            f"{family_625dr}: TEST wrapper returned "
            "fewer than two components."
        )

    # --------------------------------------------------------
    # IMPORTANT:
    # Step 6.25D-C2 verified that component 1 is the
    # normalized corrupted official-TEST DataFrame.
    # --------------------------------------------------------

    corrupted_df_625dr = (
        wrapper_result_625dr[1]
    )

    if not isinstance(
        corrupted_df_625dr,
        pd.DataFrame,
    ):
        raise RuntimeError(
            f"{family_625dr}: wrapper component 1 "
            "is not a DataFrame."
        )

    if (
        corrupted_df_625dr.shape
        != OFFICIAL_TEST_RAW_624B.shape
    ):
        raise RuntimeError(
            f"{family_625dr}: corrupted TEST shape "
            "changed."
        )

    if (
        list(
            corrupted_df_625dr.columns
        )
        != list(
            OFFICIAL_TEST_RAW_624B.columns
        )
    ):
        raise RuntimeError(
            f"{family_625dr}: TEST columns changed."
        )

    if not (
        corrupted_df_625dr.index.equals(
            OFFICIAL_TEST_RAW_624B.index
        )
    ):
        raise RuntimeError(
            f"{family_625dr}: TEST index changed."
        )


    # --------------------------------------------------------
    # 9B. Raw corruption-effect check
    # --------------------------------------------------------

    clean_sensor_625dr = (
        OFFICIAL_TEST_RAW_624B[
            sensor_625dr
        ]
        .to_numpy(
            dtype=np.float64
        )
    )

    corrupt_sensor_625dr = (
        corrupted_df_625dr[
            sensor_625dr
        ]
        .to_numpy(
            dtype=np.float64
        )
    )

    raw_changed_cells_625dr = int(
        np.sum(
            ~np.isclose(
                clean_sensor_625dr,
                corrupt_sensor_625dr,
                rtol=0.0,
                atol=1e-12,
                equal_nan=True,
            )
        )
    )

    if raw_changed_cells_625dr <= 0:
        raise RuntimeError(
            f"{family_625dr}: representative corruption "
            "did not alter the target sensor."
        )


    # --------------------------------------------------------
    # 9C. Construct corrupted endpoint model inputs
    # --------------------------------------------------------

    (
        X_corrupt_endpoint_625dr,
        corrupt_meta_625dr,
    ) = build_endpoint_sequences_625dr(
        corrupted_df_625dr
    )

    if (
        X_corrupt_endpoint_625dr.shape
        != X_clean_endpoint_625dr.shape
    ):
        raise RuntimeError(
            f"{family_625dr}: endpoint sequence "
            "shape changed."
        )

    if not np.array_equal(
        corrupt_meta_625dr[
            "unit_id"
        ].to_numpy(),
        clean_meta_625dr[
            "unit_id"
        ].to_numpy(),
    ):
        raise RuntimeError(
            f"{family_625dr}: endpoint engine IDs changed."
        )

    if not np.array_equal(
        corrupt_meta_625dr[
            "cycle"
        ].to_numpy(),
        clean_meta_625dr[
            "cycle"
        ].to_numpy(),
    ):
        raise RuntimeError(
            f"{family_625dr}: endpoint cycles changed."
        )

    endpoint_changed_elements_625dr = int(
        np.sum(
            ~np.isclose(
                X_corrupt_endpoint_625dr,
                X_clean_endpoint_625dr,
                rtol=0.0,
                atol=1e-12,
                equal_nan=True,
            )
        )
    )

    print(
        "Raw target cells changed       :",
        raw_changed_cells_625dr,
    )
    print(
        "Endpoint input elements changed:",
        endpoint_changed_elements_625dr,
    )


    # --------------------------------------------------------
    # 9D. Actual model inference + MC-dropout uncertainty
    # --------------------------------------------------------

    mc_result_625dr = (
        mc_uncertainty_623cb2(
            X_corrupt_endpoint_625dr
        )
    )

    if not isinstance(
        mc_result_625dr,
        (tuple, list),
    ):
        raise RuntimeError(
            f"{family_625dr}: mc_uncertainty_623cb2 "
            "did not return tuple/list."
        )

    if len(
        mc_result_625dr
    ) < 3:
        raise RuntimeError(
            f"{family_625dr}: expected at least three "
            "outputs from mc_uncertainty_623cb2."
        )

    prediction_625dr = to_numpy_625dr(
        mc_result_625dr[0]
    )

    mc_std_625dr = to_numpy_625dr(
        mc_result_625dr[1]
    )

    reliability_625dr = to_numpy_625dr(
        mc_result_625dr[2]
    )


    # --------------------------------------------------------
    # 9E. Output integrity
    # --------------------------------------------------------

    expected_n_625dr = 100

    if not (
        len(prediction_625dr)
        == expected_n_625dr
        == len(mc_std_625dr)
        == len(reliability_625dr)
    ):
        raise RuntimeError(
            f"{family_625dr}: inference output length "
            "is not 100."
        )

    prediction_finite_625dr = bool(
        np.isfinite(
            prediction_625dr
        ).all()
    )

    mc_std_finite_625dr = bool(
        np.isfinite(
            mc_std_625dr
        ).all()
    )

    reliability_finite_625dr = bool(
        np.isfinite(
            reliability_625dr
        ).all()
    )

    if not (
        prediction_finite_625dr
        and mc_std_finite_625dr
        and reliability_finite_625dr
    ):
        raise RuntimeError(
            f"{family_625dr}: non-finite inference "
            "values detected."
        )


    # --------------------------------------------------------
    # 9F. Apply frozen reliability rule
    #
    # Frozen rule:
    # ACCEPT  -> reliability_score <= threshold
    # ABSTAIN -> reliability_score > threshold
    # --------------------------------------------------------

    accept_625dr = (
        reliability_625dr
        <= threshold_before_625dr
    )

    accepted_n_625dr = int(
        accept_625dr.sum()
    )

    coverage_625dr = float(
        accepted_n_625dr
        / expected_n_625dr
    )

    changed_decisions_625dr = int(
        np.sum(
            accept_625dr
            != clean_accept_625dr
        )
    )

    abs_prediction_change_625dr = (
        np.abs(
            prediction_625dr
            - clean_prediction_625dr
        )
    )

    reliability_change_625dr = (
        reliability_625dr
        - clean_reliability_625dr
    )


    # --------------------------------------------------------
    # 9G. Scenario-level summary
    # --------------------------------------------------------

    summary_records_625dr.append(
        {
            "family":
                family_625dr,

            "scenario_id":
                scenario_id_625dr,

            "sensor":
                sensor_625dr,

            "severity":
                severity_625dr,

            "raw_changed_cells":
                raw_changed_cells_625dr,

            "endpoint_changed_elements":
                endpoint_changed_elements_625dr,

            "mean_abs_prediction_change":
                float(
                    np.mean(
                        abs_prediction_change_625dr
                    )
                ),

            "max_abs_prediction_change":
                float(
                    np.max(
                        abs_prediction_change_625dr
                    )
                ),

            "mean_mc_std":
                float(
                    np.mean(
                        mc_std_625dr
                    )
                ),

            "mean_reliability_score":
                float(
                    np.mean(
                        reliability_625dr
                    )
                ),

            "mean_reliability_change":
                float(
                    np.mean(
                        reliability_change_625dr
                    )
                ),

            "accepted":
                accepted_n_625dr,

            "abstained":
                int(
                    expected_n_625dr
                    - accepted_n_625dr
                ),

            "coverage":
                coverage_625dr,

            "decision_changes_vs_clean":
                changed_decisions_625dr,

            "prediction_finite":
                prediction_finite_625dr,

            "mc_std_finite":
                mc_std_finite_625dr,

            "reliability_finite":
                reliability_finite_625dr,
        }
    )


    # --------------------------------------------------------
    # 9H. Engine-level temporary records
    # --------------------------------------------------------

    for i_625dr in range(
        expected_n_625dr
    ):

        detail_records_625dr.append(
            {
                "family":
                    family_625dr,

                "scenario_id":
                    scenario_id_625dr,

                "unit_id":
                    int(
                        clean_meta_625dr[
                            "unit_id"
                        ].iloc[i_625dr]
                    ),

                "cycle":
                    int(
                        clean_meta_625dr[
                            "cycle"
                        ].iloc[i_625dr]
                    ),

                "clean_prediction":
                    float(
                        clean_prediction_625dr[
                            i_625dr
                        ]
                    ),

                "corrupted_prediction":
                    float(
                        prediction_625dr[
                            i_625dr
                        ]
                    ),

                "prediction_change":
                    float(
                        prediction_625dr[
                            i_625dr
                        ]
                        -
                        clean_prediction_625dr[
                            i_625dr
                        ]
                    ),

                "clean_reliability":
                    float(
                        clean_reliability_625dr[
                            i_625dr
                        ]
                    ),

                "corrupted_reliability":
                    float(
                        reliability_625dr[
                            i_625dr
                        ]
                    ),

                "clean_accept":
                    bool(
                        clean_accept_625dr[
                            i_625dr
                        ]
                    ),

                "corrupted_accept":
                    bool(
                        accept_625dr[
                            i_625dr
                        ]
                    ),

                "decision_changed":
                    bool(
                        accept_625dr[
                            i_625dr
                        ]
                        !=
                        clean_accept_625dr[
                            i_625dr
                        ]
                    ),
            }
        )

    print(
        "Accepted endpoints               :",
        accepted_n_625dr,
        "/ 100",
    )
    print(
        "Coverage                         :",
        coverage_625dr,
    )
    print(
        "Decision changes vs CLEAN        :",
        changed_decisions_625dr,
    )
    print(
        "Mean |prediction change|         :",
        float(
            np.mean(
                abs_prediction_change_625dr
            )
        ),
    )


# ------------------------------------------------------------
# 10. Build temporary smoke-test evidence tables
# ------------------------------------------------------------

OFFICIAL_TEST_SMOKE_SUMMARY_625DR = (
    pd.DataFrame(
        summary_records_625dr
    )
)

OFFICIAL_TEST_SMOKE_DETAIL_625DR = (
    pd.DataFrame(
        detail_records_625dr
    )
)

print(
    "\n--- Five-family official TEST inference "
    "smoke-test summary ---"
)

display(
    OFFICIAL_TEST_SMOKE_SUMMARY_625DR
)


print(
    "\n--- First 15 engine-level smoke-test records ---"
)

display(
    OFFICIAL_TEST_SMOKE_DETAIL_625DR
    .head(15)
)


# ------------------------------------------------------------
# 11. Final smoke-test integrity checks
# ------------------------------------------------------------

families_complete_625dr = bool(
    len(
        OFFICIAL_TEST_SMOKE_SUMMARY_625DR
    )
    == 5
)

all_raw_effective_625dr = bool(
    (
        OFFICIAL_TEST_SMOKE_SUMMARY_625DR[
            "raw_changed_cells"
        ]
        > 0
    ).all()
)

all_endpoint_effective_625dr = bool(
    (
        OFFICIAL_TEST_SMOKE_SUMMARY_625DR[
            "endpoint_changed_elements"
        ]
        > 0
    ).all()
)

all_predictions_finite_625dr = bool(
    OFFICIAL_TEST_SMOKE_SUMMARY_625DR[
        "prediction_finite"
    ].all()
)

all_mc_finite_625dr = bool(
    OFFICIAL_TEST_SMOKE_SUMMARY_625DR[
        "mc_std_finite"
    ].all()
)

all_reliability_finite_625dr = bool(
    OFFICIAL_TEST_SMOKE_SUMMARY_625DR[
        "reliability_finite"
    ].all()
)

threshold_after_625dr = float(
    RELOADED_THRESHOLD_623DL5
)

threshold_unchanged_625dr = bool(
    threshold_after_625dr
    == threshold_before_625dr
)


print(
    "\n"
    + "=" * 54
)
print(
    "STEP 6.25D-R END-TO-END SMOKE-TEST SUMMARY"
)
print(
    "=" * 54
)

print(
    "Families completed             :",
    len(
        OFFICIAL_TEST_SMOKE_SUMMARY_625DR
    ),
    "/ 5",
)
print(
    "All raw corruptions effective  :",
    all_raw_effective_625dr,
)
print(
    "All endpoint inputs affected   :",
    all_endpoint_effective_625dr,
)
print(
    "All predictions finite         :",
    all_predictions_finite_625dr,
)
print(
    "All MC-STD values finite       :",
    all_mc_finite_625dr,
)
print(
    "All reliability values finite  :",
    all_reliability_finite_625dr,
)
print(
    "Frozen threshold unchanged     :",
    threshold_unchanged_625dr,
)

print(
    "\nModel inference executed        : True"
)
print(
    "MC-dropout executed            : True"
)
print(
    "Official RUL labels used       : False"
)
print(
    "Official TEST scored           : False"
)
print(
    "Frozen manifests modified      : False"
)
print(
    "Frozen reliability rule changed: False"
)
print(
    "Files written                  : False"
)

print(
    "\n=== END ==="
)

In [ ]:
# ============================================================
# STEP 6.25E1
# Full 315-scenario official FD001 TEST pre-score inference
#
# IMPORTANT:
# - Official TEST RUL labels are NOT used here.
# - Frozen reliability rule is NOT modified.
# - Frozen reliability threshold is NOT modified.
# - Frozen corruption manifests are NOT modified.
# - All 315 frozen corruption scenarios are evaluated.
# - Only endpoint predictions / uncertainty / reliability /
#   accept-abstain decisions are produced.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import time

import numpy as np
import pandas as pd


print(
    "\n"
    "=== Step 6.25E1 — Full 315-scenario official "
    "FD001 TEST pre-score inference ==="
)


# ============================================================
# 1. Required-object verification
# ============================================================

required_objects_625e1 = [
    "OFFICIAL_TEST_RAW_624B",
    "OFFICIAL_TEST_RAW_EFFECT_AUDIT_625C",
    "scenario_manifests_620gc",
    "apply_frozen_corruption_test_625b5",
    "PRIMARY_INPUT_COLUMNS",
    "scaler",
    "X_TEST_CLEAN_624C",
    "OFFICIAL_TEST_SEQ_META_624C",
    "OFFICIAL_TEST_ENDPOINT_INFERENCE_624E",
    "mc_uncertainty_623cb2",
    "RELOADED_THRESHOLD_623DL5",
    "RELOADED_RULE_623DL5",
]

print("\n--- Required objects ---")

missing_objects_625e1 = []

for object_name_625e1 in required_objects_625e1:

    object_exists_625e1 = (
        object_name_625e1
        in globals()
    )

    print(
        f"{object_name_625e1:<42}: "
        f"{object_exists_625e1}"
    )

    if not object_exists_625e1:
        missing_objects_625e1.append(
            object_name_625e1
        )


if missing_objects_625e1:
    raise RuntimeError(
        "Step 6.25E1 cannot continue. "
        "Missing required objects: "
        + str(missing_objects_625e1)
    )


# ============================================================
# 2. Lock frozen reliability decision
# ============================================================

FROZEN_TEST_THRESHOLD_625E1 = float(
    RELOADED_THRESHOLD_623DL5
)

FROZEN_TEST_RULE_625E1 = (
    RELOADED_RULE_623DL5
)

if not np.isfinite(
    FROZEN_TEST_THRESHOLD_625E1
):
    raise RuntimeError(
        "Frozen reliability threshold "
        "is not finite."
    )


if not isinstance(
    FROZEN_TEST_RULE_625E1,
    dict,
):
    raise RuntimeError(
        "Frozen reliability rule must "
        "be a dictionary."
    )


print(
    "\n--- Frozen reliability decision ---"
)

print(
    "Frozen threshold :",
    FROZEN_TEST_THRESHOLD_625E1
)

print(
    "Frozen rule type :",
    type(
        FROZEN_TEST_RULE_625E1
    ).__name__
)

print(
    "Accept rule      : "
    "reliability_score <= reliability_threshold"
)

print(
    "Abstain rule     : "
    "reliability_score > reliability_threshold"
)


# ============================================================
# 3. Immutable snapshots
# ============================================================

OFFICIAL_TEST_RAW_SNAPSHOT_625E1 = (
    OFFICIAL_TEST_RAW_624B.copy(
        deep=True
    )
)

RAW_COLUMNS_625E1 = list(
    OFFICIAL_TEST_RAW_624B.columns
)

RAW_INDEX_625E1 = (
    OFFICIAL_TEST_RAW_624B.index.copy()
)


if (
    OFFICIAL_TEST_RAW_624B.shape
    != (13096, 26)
):
    raise RuntimeError(
        "Unexpected official FD001 TEST "
        "raw-data shape: "
        + str(
            OFFICIAL_TEST_RAW_624B.shape
        )
    )


if (
    OFFICIAL_TEST_RAW_624B[
        ["unit_id", "cycle"]
    ]
    .duplicated()
    .any()
):
    raise RuntimeError(
        "Duplicate (unit_id, cycle) keys "
        "found in official TEST data."
    )


print(
    "\n--- Official TEST reference ---"
)

print(
    "Rows              :",
    len(
        OFFICIAL_TEST_RAW_624B
    )
)

print(
    "Columns           :",
    OFFICIAL_TEST_RAW_624B.shape[1]
)

print(
    "Engines           :",
    OFFICIAL_TEST_RAW_624B[
        "unit_id"
    ].nunique()
)

print(
    "Primary inputs    :",
    len(
        PRIMARY_INPUT_COLUMNS
    )
)


# ============================================================
# 4. Utility functions
# ============================================================

def _to_numpy_625e1(value):
    """
    Safely convert NumPy / Torch-like values
    into NumPy arrays.
    """

    if hasattr(value, "detach"):
        value = (
            value
            .detach()
            .cpu()
            .numpy()
        )

    return np.asarray(value)


def _json_default_625e1(value):
    """
    JSON serializer for NumPy / pandas /
    Path objects.
    """

    if isinstance(
        value,
        (
            np.integer,
        ),
    ):
        return int(value)

    if isinstance(
        value,
        (
            np.floating,
        ),
    ):
        return float(value)

    if isinstance(
        value,
        (
            np.bool_,
        ),
    ):
        return bool(value)

    if isinstance(
        value,
        np.ndarray,
    ):
        return value.tolist()

    if isinstance(
        value,
        Path,
    ):
        return str(value)

    if pd.isna(value):
        return None

    raise TypeError(
        f"Object of type "
        f"{type(value).__name__} "
        f"is not JSON serializable."
    )


def _sha256_file_625e1(path):
    """
    SHA-256 hash of a saved artifact.
    """

    hasher_625e1 = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as file_handle_625e1:

        while True:

            block_625e1 = (
                file_handle_625e1.read(
                    1024 * 1024
                )
            )

            if not block_625e1:
                break

            hasher_625e1.update(
                block_625e1
            )

    return hasher_625e1.hexdigest()


def _save_json_atomic_625e1(
    payload,
    path,
):
    """
    Write JSON atomically.
    """

    temp_path_625e1 = Path(
        str(path) + ".tmp"
    )

    with open(
        temp_path_625e1,
        "w",
        encoding="utf-8",
    ) as file_handle_625e1:

        json.dump(
            payload,
            file_handle_625e1,
            indent=2,
            ensure_ascii=False,
            default=_json_default_625e1,
        )

    os.replace(
        temp_path_625e1,
        path,
    )


def _save_csv_atomic_625e1(
    dataframe,
    path,
):
    """
    Write CSV atomically.
    """

    temp_path_625e1 = Path(
        str(path) + ".tmp"
    )

    dataframe.to_csv(
        temp_path_625e1,
        index=False,
    )

    os.replace(
        temp_path_625e1,
        path,
    )


# ============================================================
# 5. Recover frozen clean endpoint sequence reference
# ============================================================

X_TEST_CLEAN_ARRAY_625E1 = (
    _to_numpy_625e1(
        X_TEST_CLEAN_624C
    )
)

TEST_SEQ_META_625E1 = (
    OFFICIAL_TEST_SEQ_META_624C
    .reset_index(
        drop=True
    )
    .copy()
)


if (
    len(
        X_TEST_CLEAN_ARRAY_625E1
    )
    != len(
        TEST_SEQ_META_625E1
    )
):
    raise RuntimeError(
        "Clean TEST sequence array and "
        "metadata row counts differ."
    )


for required_meta_col_625e1 in [
    "unit_id",
    "cycle",
]:
    if (
        required_meta_col_625e1
        not in TEST_SEQ_META_625E1.columns
    ):
        raise RuntimeError(
            "Required TEST sequence metadata "
            f"column missing: "
            f"{required_meta_col_625e1}"
        )


endpoint_positions_625e1 = (
    TEST_SEQ_META_625E1
    .groupby(
        "unit_id",
        sort=True,
    )["cycle"]
    .idxmax()
    .to_numpy(
        dtype=int
    )
)


X_CLEAN_ENDPOINT_REFERENCE_625E1 = (
    X_TEST_CLEAN_ARRAY_625E1[
        endpoint_positions_625e1
    ]
)


CLEAN_ENDPOINT_META_REFERENCE_625E1 = (
    TEST_SEQ_META_625E1
    .iloc[
        endpoint_positions_625e1
    ][
        [
            "unit_id",
            "cycle",
        ]
    ]
    .reset_index(
        drop=True
    )
)


if (
    X_CLEAN_ENDPOINT_REFERENCE_625E1.shape
    != (100, 30, 17)
):
    raise RuntimeError(
        "Unexpected frozen clean endpoint "
        "sequence shape: "
        + str(
            X_CLEAN_ENDPOINT_REFERENCE_625E1.shape
        )
    )


MODEL_SEQUENCE_DTYPE_625E1 = (
    X_CLEAN_ENDPOINT_REFERENCE_625E1.dtype
)


# ============================================================
# 6. Endpoint preprocessing helper
# ============================================================

def build_endpoint_sequences_625e1(
    raw_dataframe,
):
    """
    Reproduce the frozen endpoint preprocessing
    from a raw official-TEST-like DataFrame.

    Steps:
      1. preserve unit/cycle ordering
      2. apply the already-frozen scaler
      3. use the already-frozen 17 model inputs
      4. take the final 30-cycle window per engine
    """

    if not isinstance(
        raw_dataframe,
        pd.DataFrame,
    ):
        raise TypeError(
            "raw_dataframe must be "
            "a pandas DataFrame."
        )


    if (
        raw_dataframe.shape
        != OFFICIAL_TEST_RAW_624B.shape
    ):
        raise RuntimeError(
            "TEST-like DataFrame shape changed: "
            + str(
                raw_dataframe.shape
            )
        )


    if (
        list(
            raw_dataframe.columns
        )
        != RAW_COLUMNS_625E1
    ):
        raise RuntimeError(
            "TEST-like DataFrame column "
            "order changed."
        )


    missing_features_625e1 = [
        feature_625e1
        for feature_625e1
        in PRIMARY_INPUT_COLUMNS
        if feature_625e1
        not in raw_dataframe.columns
    ]

    if missing_features_625e1:
        raise RuntimeError(
            "Missing primary model inputs: "
            + str(
                missing_features_625e1
            )
        )


    scaled_values_625e1 = (
        scaler.transform(
            raw_dataframe[
                PRIMARY_INPUT_COLUMNS
            ]
        )
    )


    scaled_values_625e1 = (
        np.asarray(
            scaled_values_625e1
        )
        .astype(
            MODEL_SEQUENCE_DTYPE_625E1,
            copy=False,
        )
    )


    if not np.isfinite(
        scaled_values_625e1
    ).all():
        raise RuntimeError(
            "Non-finite scaled TEST "
            "model inputs detected."
        )


    scaled_frame_625e1 = pd.DataFrame(
        scaled_values_625e1,
        columns=PRIMARY_INPUT_COLUMNS,
        index=raw_dataframe.index,
    )


    sequence_list_625e1 = []
    metadata_rows_625e1 = []


    grouped_units_625e1 = (
        raw_dataframe
        .groupby(
            "unit_id",
            sort=True,
        )
    )


    for (
        unit_id_625e1,
        unit_frame_625e1,
    ) in grouped_units_625e1:

        unit_frame_625e1 = (
            unit_frame_625e1
            .sort_values(
                "cycle"
            )
        )


        if (
            len(
                unit_frame_625e1
            )
            < 30
        ):
            raise RuntimeError(
                f"Engine {unit_id_625e1} "
                "contains fewer than "
                "30 observations."
            )


        final_indices_625e1 = (
            unit_frame_625e1
            .index
            .to_numpy()[-30:]
        )


        unit_sequence_625e1 = (
            scaled_frame_625e1
            .loc[
                final_indices_625e1,
                PRIMARY_INPUT_COLUMNS,
            ]
            .to_numpy(
                dtype=MODEL_SEQUENCE_DTYPE_625E1
            )
        )


        if (
            unit_sequence_625e1.shape
            != (30, 17)
        ):
            raise RuntimeError(
                f"Engine {unit_id_625e1}: "
                "invalid endpoint "
                "sequence shape "
                + str(
                    unit_sequence_625e1.shape
                )
            )


        sequence_list_625e1.append(
            unit_sequence_625e1
        )


        metadata_rows_625e1.append(
            {
                "unit_id": int(
                    unit_id_625e1
                ),
                "cycle": int(
                    unit_frame_625e1[
                        "cycle"
                    ].iloc[-1]
                ),
            }
        )


    endpoint_sequences_625e1 = (
        np.stack(
            sequence_list_625e1,
            axis=0,
        )
    )


    endpoint_metadata_625e1 = (
        pd.DataFrame(
            metadata_rows_625e1
        )
    )


    if (
        endpoint_sequences_625e1.shape
        != (100, 30, 17)
    ):
        raise RuntimeError(
            "Unexpected endpoint sequence "
            "shape: "
            + str(
                endpoint_sequences_625e1.shape
            )
        )


    if (
        endpoint_metadata_625e1[
            "unit_id"
        ].nunique()
        != 100
    ):
        raise RuntimeError(
            "Endpoint engine count "
            "is not 100."
        )


    return (
        endpoint_sequences_625e1,
        endpoint_metadata_625e1,
    )


# ============================================================
# 7. Reproduce clean TEST preprocessing
# ============================================================

(
    X_CLEAN_ENDPOINT_REBUILT_625E1,
    CLEAN_ENDPOINT_META_REBUILT_625E1,
) = build_endpoint_sequences_625e1(
    OFFICIAL_TEST_RAW_624B
)


clean_sequence_max_error_625e1 = float(
    np.max(
        np.abs(
            X_CLEAN_ENDPOINT_REBUILT_625E1.astype(
                np.float64
            )
            -
            X_CLEAN_ENDPOINT_REFERENCE_625E1.astype(
                np.float64
            )
        )
    )
)


clean_ids_exact_625e1 = bool(
    np.array_equal(
        CLEAN_ENDPOINT_META_REBUILT_625E1[
            "unit_id"
        ].to_numpy(
            dtype=int
        ),
        CLEAN_ENDPOINT_META_REFERENCE_625E1[
            "unit_id"
        ].to_numpy(
            dtype=int
        ),
    )
)


clean_cycles_exact_625e1 = bool(
    np.array_equal(
        CLEAN_ENDPOINT_META_REBUILT_625E1[
            "cycle"
        ].to_numpy(
            dtype=int
        ),
        CLEAN_ENDPOINT_META_REFERENCE_625E1[
            "cycle"
        ].to_numpy(
            dtype=int
        ),
    )
)


# ------------------------------------------------------------
# IMPORTANT CORRECTION:
# Do NOT require bitwise / 1e-10 equality here.
#
# The previous observed reconstruction difference was about
# 1.19e-7, which is normal floating-point representation
# error.  Therefore preprocessing equivalence is verified
# numerically with explicit tolerances.
# ------------------------------------------------------------

CLEAN_SEQUENCE_ATOL_625E1 = 1e-6
CLEAN_SEQUENCE_RTOL_625E1 = 1e-6


clean_sequences_equivalent_625e1 = bool(
    np.allclose(
        X_CLEAN_ENDPOINT_REBUILT_625E1,
        X_CLEAN_ENDPOINT_REFERENCE_625E1,
        atol=CLEAN_SEQUENCE_ATOL_625E1,
        rtol=CLEAN_SEQUENCE_RTOL_625E1,
        equal_nan=False,
    )
)


print(
    "\n--- Clean TEST preprocessing reproduction ---"
)

print(
    "Endpoint shape              :",
    X_CLEAN_ENDPOINT_REBUILT_625E1.shape
)

print(
    "Maximum sequence error      :",
    clean_sequence_max_error_625e1
)

print(
    "Numerically equivalent      :",
    clean_sequences_equivalent_625e1
)

print(
    "Absolute tolerance          :",
    CLEAN_SEQUENCE_ATOL_625E1
)

print(
    "Relative tolerance          :",
    CLEAN_SEQUENCE_RTOL_625E1
)

print(
    "Endpoint engine IDs exact   :",
    clean_ids_exact_625e1
)

print(
    "Endpoint cycles exact       :",
    clean_cycles_exact_625e1
)


if not clean_sequences_equivalent_625e1:
    raise RuntimeError(
        "Clean TEST preprocessing differs "
        "beyond the allowed floating-point "
        "tolerance."
    )


if not clean_ids_exact_625e1:
    raise RuntimeError(
        "Clean endpoint engine IDs "
        "do not match."
    )


if not clean_cycles_exact_625e1:
    raise RuntimeError(
        "Clean endpoint cycles "
        "do not match."
    )


# ============================================================
# 8. Frozen clean endpoint-inference reference
# ============================================================

clean_reference_625e1 = (
    OFFICIAL_TEST_ENDPOINT_INFERENCE_624E
    .copy()
    .sort_values(
        "unit_id"
    )
    .reset_index(
        drop=True
    )
)


required_clean_columns_625e1 = [
    "unit_id",
    "cycle",
    "prediction",
    "mc_std",
    "reliability_score",
    "accept",
]


missing_clean_columns_625e1 = [
    column_625e1
    for column_625e1
    in required_clean_columns_625e1
    if column_625e1
    not in clean_reference_625e1.columns
]


if missing_clean_columns_625e1:
    raise RuntimeError(
        "Frozen clean endpoint-inference "
        "columns missing: "
        + str(
            missing_clean_columns_625e1
        )
    )


if (
    len(
        clean_reference_625e1
    )
    != 100
):
    raise RuntimeError(
        "Frozen clean endpoint inference "
        "must contain 100 engines."
    )


if not np.array_equal(
    clean_reference_625e1[
        "unit_id"
    ].to_numpy(
        dtype=int
    ),
    CLEAN_ENDPOINT_META_REBUILT_625E1[
        "unit_id"
    ].to_numpy(
        dtype=int
    ),
):
    raise RuntimeError(
        "Clean inference engine IDs "
        "are misaligned."
    )


if not np.array_equal(
    clean_reference_625e1[
        "cycle"
    ].to_numpy(
        dtype=int
    ),
    CLEAN_ENDPOINT_META_REBUILT_625E1[
        "cycle"
    ].to_numpy(
        dtype=int
    ),
):
    raise RuntimeError(
        "Clean inference cycles "
        "are misaligned."
    )


CLEAN_PREDICTION_625E1 = (
    clean_reference_625e1[
        "prediction"
    ]
    .to_numpy(
        dtype=np.float64
    )
)


CLEAN_MC_STD_625E1 = (
    clean_reference_625e1[
        "mc_std"
    ]
    .to_numpy(
        dtype=np.float64
    )
)


CLEAN_RELIABILITY_625E1 = (
    clean_reference_625e1[
        "reliability_score"
    ]
    .to_numpy(
        dtype=np.float64
    )
)


CLEAN_ACCEPT_STORED_625E1 = (
    clean_reference_625e1[
        "accept"
    ]
    .astype(bool)
    .to_numpy()
)


CLEAN_ACCEPT_RECOMPUTED_625E1 = (
    CLEAN_RELIABILITY_625E1
    <= FROZEN_TEST_THRESHOLD_625E1
)


if not np.array_equal(
    CLEAN_ACCEPT_STORED_625E1,
    CLEAN_ACCEPT_RECOMPUTED_625E1,
):
    raise RuntimeError(
        "Frozen clean accept mask does "
        "not reproduce from the frozen "
        "threshold."
    )


print(
    "\nFrozen CLEAN accepted endpoints :",
    int(
        CLEAN_ACCEPT_STORED_625E1.sum()
    ),
    "/ 100"
)


# ============================================================
# 9. MC-uncertainty helper contract
# ============================================================

def run_frozen_uncertainty_625e1(
    endpoint_sequences,
):
    """
    Run already-frozen deterministic prediction,
    MC-dropout uncertainty and phase-adjusted
    reliability computation.
    """

    result_625e1 = (
        mc_uncertainty_623cb2(
            endpoint_sequences
        )
    )


    if not isinstance(
        result_625e1,
        (
            tuple,
            list,
        ),
    ):
        raise RuntimeError(
            "mc_uncertainty_623cb2 must "
            "return a tuple/list."
        )


    if len(
        result_625e1
    ) != 3:
        raise RuntimeError(
            "Unexpected frozen uncertainty "
            "return length: "
            + str(
                len(
                    result_625e1
                )
            )
        )


    prediction_625e1 = (
        _to_numpy_625e1(
            result_625e1[0]
        )
        .astype(
            np.float64
        )
        .reshape(-1)
    )


    mc_std_625e1 = (
        _to_numpy_625e1(
            result_625e1[1]
        )
        .astype(
            np.float64
        )
        .reshape(-1)
    )


    reliability_625e1 = (
        _to_numpy_625e1(
            result_625e1[2]
        )
        .astype(
            np.float64
        )
        .reshape(-1)
    )


    expected_length_625e1 = (
        endpoint_sequences.shape[0]
    )


    if not (
        len(
            prediction_625e1
        )
        ==
        len(
            mc_std_625e1
        )
        ==
        len(
            reliability_625e1
        )
        ==
        expected_length_625e1
    ):
        raise RuntimeError(
            "Frozen uncertainty output "
            "length mismatch."
        )


    if not np.isfinite(
        prediction_625e1
    ).all():
        raise RuntimeError(
            "Non-finite prediction values "
            "detected."
        )


    if not np.isfinite(
        mc_std_625e1
    ).all():
        raise RuntimeError(
            "Non-finite MC-STD values "
            "detected."
        )


    if not np.isfinite(
        reliability_625e1
    ).all():
        raise RuntimeError(
            "Non-finite reliability values "
            "detected."
        )


    return (
        prediction_625e1,
        mc_std_625e1,
        reliability_625e1,
    )


# ============================================================
# 10. Clean uncertainty-helper reproducibility probe
# ============================================================

print(
    "\n--- Clean uncertainty-helper reproducibility probe ---"
)


(
    clean_probe_prediction_625e1,
    clean_probe_mc_std_625e1,
    clean_probe_reliability_625e1,
) = run_frozen_uncertainty_625e1(
    X_CLEAN_ENDPOINT_REBUILT_625E1
)


clean_probe_accept_625e1 = (
    clean_probe_reliability_625e1
    <= FROZEN_TEST_THRESHOLD_625E1
)


clean_prediction_max_diff_625e1 = float(
    np.max(
        np.abs(
            clean_probe_prediction_625e1
            -
            CLEAN_PREDICTION_625E1
        )
    )
)


clean_mc_std_max_diff_625e1 = float(
    np.max(
        np.abs(
            clean_probe_mc_std_625e1
            -
            CLEAN_MC_STD_625E1
        )
    )
)


clean_reliability_max_diff_625e1 = float(
    np.max(
        np.abs(
            clean_probe_reliability_625e1
            -
            CLEAN_RELIABILITY_625E1
        )
    )
)


clean_accept_mask_exact_625e1 = bool(
    np.array_equal(
        clean_probe_accept_625e1,
        CLEAN_ACCEPT_STORED_625E1,
    )
)


print(
    "Prediction max difference    :",
    clean_prediction_max_diff_625e1
)

print(
    "MC-STD max difference        :",
    clean_mc_std_max_diff_625e1
)

print(
    "Reliability max difference   :",
    clean_reliability_max_diff_625e1
)

print(
    "Accept mask exact            :",
    clean_accept_mask_exact_625e1
)


# Decision equivalence is mandatory.
# Tiny numerical differences in continuous values are allowed,
# but the frozen decision itself must reproduce exactly.

if not clean_accept_mask_exact_625e1:
    raise RuntimeError(
        "Frozen clean accept/abstain "
        "decision could not be reproduced."
    )


# ============================================================
# 11. Prepare exact 315-scenario evaluation order
# ============================================================

scenario_audit_625e1 = (
    OFFICIAL_TEST_RAW_EFFECT_AUDIT_625C
    .copy()
)


if (
    "global_index"
    in scenario_audit_625e1.columns
):
    scenario_audit_625e1 = (
        scenario_audit_625e1
        .sort_values(
            "global_index"
        )
        .reset_index(
            drop=True
        )
    )

else:

    scenario_audit_625e1 = (
        scenario_audit_625e1
        .reset_index(
            drop=True
        )
    )

    scenario_audit_625e1[
        "global_index"
    ] = np.arange(
        len(
            scenario_audit_625e1
        ),
        dtype=int,
    )


if (
    "family_position"
    not in scenario_audit_625e1.columns
):
    scenario_audit_625e1[
        "family_position"
    ] = (
        scenario_audit_625e1
        .groupby(
            "family",
            sort=False,
        )
        .cumcount()
    )


if (
    len(
        scenario_audit_625e1
    )
    != 315
):
    raise RuntimeError(
        "Expected 315 frozen TEST corruption "
        "scenarios, found "
        + str(
            len(
                scenario_audit_625e1
            )
        )
    )


if (
    scenario_audit_625e1[
        "global_index"
    ].duplicated().any()
):
    raise RuntimeError(
        "Duplicate global corruption indices "
        "detected."
    )


expected_family_counts_625e1 = {
    "bias": 90,
    "noise": 45,
    "drift": 90,
    "dropout": 45,
    "stuck": 45,
}


observed_family_counts_625e1 = (
    scenario_audit_625e1[
        "family"
    ]
    .astype(str)
    .str.lower()
    .value_counts()
    .to_dict()
)


if (
    observed_family_counts_625e1
    != expected_family_counts_625e1
):
    raise RuntimeError(
        "Frozen scenario family counts "
        "differ from the verified 315-scenario "
        "manifest. Observed: "
        + str(
            observed_family_counts_625e1
        )
    )


if (
    "effective_primary"
    in scenario_audit_625e1.columns
):
    if not (
        scenario_audit_625e1[
            "effective_primary"
        ]
        .astype(bool)
        .all()
    ):
        raise RuntimeError(
            "Raw-effect audit contains "
            "a non-effective primary-input "
            "scenario."
        )


print(
    "\n--- Frozen scenario inventory ---"
)

for (
    family_625e1,
    count_625e1,
) in expected_family_counts_625e1.items():

    print(
        f"{family_625e1:<10}: "
        f"{count_625e1}"
    )

print(
    "Total     :",
    len(
        scenario_audit_625e1
    )
)


# ============================================================
# 12. Allocate pre-score result containers
# ============================================================

N_SCENARIOS_625E1 = 315
N_ENGINES_625E1 = 100


PREDICTION_MATRIX_625E1 = np.full(
    (
        N_SCENARIOS_625E1,
        N_ENGINES_625E1,
    ),
    np.nan,
    dtype=np.float64,
)


MC_STD_MATRIX_625E1 = np.full(
    (
        N_SCENARIOS_625E1,
        N_ENGINES_625E1,
    ),
    np.nan,
    dtype=np.float64,
)


RELIABILITY_MATRIX_625E1 = np.full(
    (
        N_SCENARIOS_625E1,
        N_ENGINES_625E1,
    ),
    np.nan,
    dtype=np.float64,
)


ACCEPT_MATRIX_625E1 = np.zeros(
    (
        N_SCENARIOS_625E1,
        N_ENGINES_625E1,
    ),
    dtype=bool,
)


scenario_result_rows_625e1 = []
engine_result_chunks_625e1 = []


clean_unit_ids_625e1 = (
    CLEAN_ENDPOINT_META_REBUILT_625E1[
        "unit_id"
    ]
    .to_numpy(
        dtype=int
    )
)


clean_cycles_625e1 = (
    CLEAN_ENDPOINT_META_REBUILT_625E1[
        "cycle"
    ]
    .to_numpy(
        dtype=int
    )
)


# ============================================================
# 13. Full 315-scenario PRE-SCORE inference
# ============================================================

print(
    "\n"
    "============================================================"
)

print(
    "Running 315 frozen official-TEST corruption scenarios"
)

print(
    "NO official RUL labels will be used."
)

print(
    "Frozen threshold =",
    FROZEN_TEST_THRESHOLD_625E1
)

print(
    "============================================================"
)


start_time_625e1 = time.perf_counter()

previous_family_625e1 = None


for (
    scenario_number_625e1,
    audit_row_625e1,
) in scenario_audit_625e1.iterrows():

    global_index_625e1 = int(
        audit_row_625e1[
            "global_index"
        ]
    )

    family_625e1 = str(
        audit_row_625e1[
            "family"
        ]
    ).lower()

    family_position_625e1 = int(
        audit_row_625e1[
            "family_position"
        ]
    )


    if (
        family_625e1
        not in scenario_manifests_620gc
    ):
        raise RuntimeError(
            f"Frozen manifest family "
            f"missing: {family_625e1}"
        )


    manifest_family_625e1 = (
        scenario_manifests_620gc[
            family_625e1
        ]
    )


    if (
        family_position_625e1
        < 0
        or
        family_position_625e1
        >= len(
            manifest_family_625e1
        )
    ):
        raise RuntimeError(
            f"{family_625e1}: invalid "
            f"family position "
            f"{family_position_625e1}."
        )


    scenario_row_625e1 = (
        manifest_family_625e1
        .iloc[
            family_position_625e1
        ]
        .copy()
    )


    sensor_625e1 = str(
        scenario_row_625e1[
            "sensor"
        ]
    )


    severity_625e1 = str(
        scenario_row_625e1[
            "severity"
        ]
    )


    if (
        "evaluation_label"
        in audit_row_625e1.index
    ):
        scenario_label_625e1 = str(
            audit_row_625e1[
                "evaluation_label"
            ]
        )

    elif (
        "scenario_id"
        in scenario_row_625e1.index
    ):
        scenario_label_625e1 = str(
            scenario_row_625e1[
                "scenario_id"
            ]
        )

    else:
        scenario_label_625e1 = (
            f"{family_625e1.upper()}_"
            f"{global_index_625e1:03d}"
        )


    manifest_scenario_id_625e1 = None

    if (
        "scenario_id"
        in scenario_row_625e1.index
    ):
        manifest_scenario_id_625e1 = str(
            scenario_row_625e1[
                "scenario_id"
            ]
        )


    # --------------------------------------------------------
    # Progress display
    # --------------------------------------------------------

    if (
        family_625e1
        != previous_family_625e1
    ):
        print(
            "\n"
            f"--- {family_625e1.upper()} "
            f"family ---"
        )

        previous_family_625e1 = (
            family_625e1
        )


    if (
        scenario_number_625e1 == 0
        or
        (
            scenario_number_625e1
            + 1
        )
        % 10
        == 0
        or
        scenario_number_625e1
        ==
        N_SCENARIOS_625E1
        - 1
    ):

        elapsed_625e1 = (
            time.perf_counter()
            -
            start_time_625e1
        )

        print(
            f"[{scenario_number_625e1 + 1:3d}/315] "
            f"{scenario_label_625e1} | "
            f"{sensor_625e1} | "
            f"{severity_625e1} | "
            f"elapsed={elapsed_625e1:.1f}s"
        )


    try:

        # ====================================================
        # 13A. Apply exact frozen corruption to a temporary
        #      copy of the official TEST raw data.
        #
        # Verified Step 6.25D-C2 wrapper contract:
        #
        #   outer_result = (
        #       underlying_generator_result,
        #       normalized_corrupted_DataFrame
        #   )
        #
        # Therefore component [1] is the DataFrame used here.
        # ====================================================

        wrapper_result_625e1 = (
            apply_frozen_corruption_test_625b5(
                OFFICIAL_TEST_RAW_624B.copy(
                    deep=True
                ),
                family_625e1,
                scenario_row_625e1,
            )
        )


        if not isinstance(
            wrapper_result_625e1,
            (
                tuple,
                list,
            ),
        ):
            raise RuntimeError(
                "TEST corruption wrapper "
                "did not return tuple/list."
            )


        if (
            len(
                wrapper_result_625e1
            )
            != 2
        ):
            raise RuntimeError(
                "Unexpected TEST corruption "
                "wrapper return length: "
                + str(
                    len(
                        wrapper_result_625e1
                    )
                )
            )


        corrupted_df_625e1 = (
            wrapper_result_625e1[1]
        )


        if not isinstance(
            corrupted_df_625e1,
            pd.DataFrame,
        ):
            raise RuntimeError(
                "Normalized TEST corruption "
                "wrapper component [1] is "
                "not a DataFrame."
            )


        if (
            corrupted_df_625e1.shape
            != OFFICIAL_TEST_RAW_624B.shape
        ):
            raise RuntimeError(
                "Corrupted TEST shape changed."
            )


        if (
            list(
                corrupted_df_625e1.columns
            )
            != RAW_COLUMNS_625E1
        ):
            raise RuntimeError(
                "Corrupted TEST columns changed."
            )


        if not corrupted_df_625e1.index.equals(
            RAW_INDEX_625E1
        ):
            raise RuntimeError(
                "Corrupted TEST index changed."
            )


        # ====================================================
        # 13B. Build corrupted endpoint sequences
        # ====================================================

        (
            X_CORRUPTED_ENDPOINT_625E1,
            corrupted_endpoint_meta_625e1,
        ) = build_endpoint_sequences_625e1(
            corrupted_df_625e1
        )


        corrupted_ids_exact_625e1 = bool(
            np.array_equal(
                corrupted_endpoint_meta_625e1[
                    "unit_id"
                ].to_numpy(
                    dtype=int
                ),
                clean_unit_ids_625e1,
            )
        )


        corrupted_cycles_exact_625e1 = bool(
            np.array_equal(
                corrupted_endpoint_meta_625e1[
                    "cycle"
                ].to_numpy(
                    dtype=int
                ),
                clean_cycles_625e1,
            )
        )


        if not corrupted_ids_exact_625e1:
            raise RuntimeError(
                "Corrupted endpoint engine "
                "IDs changed."
            )


        if not corrupted_cycles_exact_625e1:
            raise RuntimeError(
                "Corrupted endpoint cycles "
                "changed."
            )


        if not np.isfinite(
            X_CORRUPTED_ENDPOINT_625E1
        ).all():
            raise RuntimeError(
                "Non-finite corrupted endpoint "
                "inputs detected."
            )


        endpoint_difference_625e1 = (
            X_CORRUPTED_ENDPOINT_625E1.astype(
                np.float64
            )
            -
            X_CLEAN_ENDPOINT_REBUILT_625E1.astype(
                np.float64
            )
        )


        endpoint_changed_elements_625e1 = int(
            np.count_nonzero(
                np.abs(
                    endpoint_difference_625e1
                )
                > 1e-12
            )
        )


        endpoint_max_abs_change_625e1 = float(
            np.max(
                np.abs(
                    endpoint_difference_625e1
                )
            )
        )


        # ====================================================
        # 13C. Frozen RUL prediction + MC uncertainty +
        #      frozen reliability score
        # ====================================================

        (
            corrupted_prediction_625e1,
            corrupted_mc_std_625e1,
            corrupted_reliability_625e1,
        ) = run_frozen_uncertainty_625e1(
            X_CORRUPTED_ENDPOINT_625E1
        )


        corrupted_accept_625e1 = (
            corrupted_reliability_625e1
            <=
            FROZEN_TEST_THRESHOLD_625E1
        )


        corrupted_decision_625e1 = np.where(
            corrupted_accept_625e1,
            "ACCEPT",
            "ABSTAIN",
        )


        # ====================================================
        # 13D. Compare against frozen CLEAN endpoint state
        # ====================================================

        prediction_change_625e1 = (
            corrupted_prediction_625e1
            -
            CLEAN_PREDICTION_625E1
        )


        reliability_change_625e1 = (
            corrupted_reliability_625e1
            -
            CLEAN_RELIABILITY_625E1
        )


        decision_changed_625e1 = (
            corrupted_accept_625e1
            !=
            CLEAN_ACCEPT_STORED_625E1
        )


        accepted_count_625e1 = int(
            corrupted_accept_625e1.sum()
        )


        abstained_count_625e1 = int(
            N_ENGINES_625E1
            -
            accepted_count_625e1
        )


        coverage_625e1 = float(
            accepted_count_625e1
            /
            N_ENGINES_625E1
        )


        decision_change_count_625e1 = int(
            decision_changed_625e1.sum()
        )


        # ====================================================
        # 13E. Store matrix outputs
        # ====================================================

        PREDICTION_MATRIX_625E1[
            scenario_number_625e1,
            :
        ] = corrupted_prediction_625e1


        MC_STD_MATRIX_625E1[
            scenario_number_625e1,
            :
        ] = corrupted_mc_std_625e1


        RELIABILITY_MATRIX_625E1[
            scenario_number_625e1,
            :
        ] = corrupted_reliability_625e1


        ACCEPT_MATRIX_625E1[
            scenario_number_625e1,
            :
        ] = corrupted_accept_625e1


        # ====================================================
        # 13F. Scenario-level record
        # ====================================================

        raw_primary_changed_cells_625e1 = (
            audit_row_625e1.get(
                "primary_changed_cells",
                np.nan,
            )
        )


        raw_target_changed_cells_625e1 = (
            audit_row_625e1.get(
                "target_changed_cells",
                np.nan,
            )
        )


        scenario_result_rows_625e1.append(
            {
                "global_index":
                    global_index_625e1,

                "family":
                    family_625e1,

                "family_position":
                    family_position_625e1,

                "scenario_id":
                    scenario_label_625e1,

                "manifest_scenario_id":
                    manifest_scenario_id_625e1,

                "sensor":
                    sensor_625e1,

                "severity":
                    severity_625e1,

                "raw_primary_changed_cells":
                    raw_primary_changed_cells_625e1,

                "raw_target_changed_cells":
                    raw_target_changed_cells_625e1,

                "endpoint_changed_elements":
                    endpoint_changed_elements_625e1,

                "endpoint_max_abs_change":
                    endpoint_max_abs_change_625e1,

                "mean_abs_prediction_change":
                    float(
                        np.mean(
                            np.abs(
                                prediction_change_625e1
                            )
                        )
                    ),

                "median_abs_prediction_change":
                    float(
                        np.median(
                            np.abs(
                                prediction_change_625e1
                            )
                        )
                    ),

                "max_abs_prediction_change":
                    float(
                        np.max(
                            np.abs(
                                prediction_change_625e1
                            )
                        )
                    ),

                "mean_mc_std":
                    float(
                        np.mean(
                            corrupted_mc_std_625e1
                        )
                    ),

                "median_mc_std":
                    float(
                        np.median(
                            corrupted_mc_std_625e1
                        )
                    ),

                "mean_reliability_score":
                    float(
                        np.mean(
                            corrupted_reliability_625e1
                        )
                    ),

                "median_reliability_score":
                    float(
                        np.median(
                            corrupted_reliability_625e1
                        )
                    ),

                "mean_reliability_change":
                    float(
                        np.mean(
                            reliability_change_625e1
                        )
                    ),

                "accepted":
                    accepted_count_625e1,

                "abstained":
                    abstained_count_625e1,

                "coverage":
                    coverage_625e1,

                "decision_changes_vs_clean":
                    decision_change_count_625e1,

                "prediction_finite":
                    bool(
                        np.isfinite(
                            corrupted_prediction_625e1
                        ).all()
                    ),

                "mc_std_finite":
                    bool(
                        np.isfinite(
                            corrupted_mc_std_625e1
                        ).all()
                    ),

                "reliability_finite":
                    bool(
                        np.isfinite(
                            corrupted_reliability_625e1
                        ).all()
                    ),

                "endpoint_ids_exact":
                    corrupted_ids_exact_625e1,

                "endpoint_cycles_exact":
                    corrupted_cycles_exact_625e1,
            }
        )


        # ====================================================
        # 13G. Engine-level records
        # ====================================================

        engine_chunk_625e1 = pd.DataFrame(
            {
                "global_index":
                    global_index_625e1,

                "family":
                    family_625e1,

                "family_position":
                    family_position_625e1,

                "scenario_id":
                    scenario_label_625e1,

                "manifest_scenario_id":
                    manifest_scenario_id_625e1,

                "sensor":
                    sensor_625e1,

                "severity":
                    severity_625e1,

                "unit_id":
                    clean_unit_ids_625e1,

                "cycle":
                    clean_cycles_625e1,

                "clean_prediction":
                    CLEAN_PREDICTION_625E1,

                "corrupted_prediction":
                    corrupted_prediction_625e1,

                "prediction_change":
                    prediction_change_625e1,

                "abs_prediction_change":
                    np.abs(
                        prediction_change_625e1
                    ),

                "clean_mc_std":
                    CLEAN_MC_STD_625E1,

                "corrupted_mc_std":
                    corrupted_mc_std_625e1,

                "clean_reliability":
                    CLEAN_RELIABILITY_625E1,

                "corrupted_reliability":
                    corrupted_reliability_625e1,

                "reliability_change":
                    reliability_change_625e1,

                "clean_accept":
                    CLEAN_ACCEPT_STORED_625E1,

                "corrupted_accept":
                    corrupted_accept_625e1,

                "corrupted_decision":
                    corrupted_decision_625e1,

                "decision_changed":
                    decision_changed_625e1,
            }
        )


        engine_result_chunks_625e1.append(
            engine_chunk_625e1
        )


    except Exception as error_625e1:

        raise RuntimeError(
            "\nStep 6.25E1 failed during "
            "official TEST pre-score inference.\n"
            f"Global index : {global_index_625e1}\n"
            f"Family       : {family_625e1}\n"
            f"Family pos.  : {family_position_625e1}\n"
            f"Scenario     : {scenario_label_625e1}\n"
            f"Sensor       : {sensor_625e1}\n"
            f"Severity     : {severity_625e1}\n"
            f"Original error: {error_625e1}"
        ) from error_625e1


# ============================================================
# 14. Assemble complete pre-score results
# ============================================================

SCENARIO_PRESCORE_RESULTS_625E1 = (
    pd.DataFrame(
        scenario_result_rows_625e1
    )
    .sort_values(
        "global_index"
    )
    .reset_index(
        drop=True
    )
)


ENGINE_PRESCORE_RESULTS_625E1 = (
    pd.concat(
        engine_result_chunks_625e1,
        ignore_index=True,
    )
    .sort_values(
        [
            "global_index",
            "unit_id",
        ]
    )
    .reset_index(
        drop=True
    )
)


if (
    len(
        SCENARIO_PRESCORE_RESULTS_625E1
    )
    != 315
):
    raise RuntimeError(
        "Scenario-level pre-score table "
        "does not contain 315 rows."
    )


if (
    len(
        ENGINE_PRESCORE_RESULTS_625E1
    )
    != 31500
):
    raise RuntimeError(
        "Engine-level pre-score table "
        "does not contain 31,500 rows."
    )


if not np.isfinite(
    PREDICTION_MATRIX_625E1
).all():
    raise RuntimeError(
        "Prediction matrix contains "
        "non-finite values."
    )


if not np.isfinite(
    MC_STD_MATRIX_625E1
).all():
    raise RuntimeError(
        "MC-STD matrix contains "
        "non-finite values."
    )


if not np.isfinite(
    RELIABILITY_MATRIX_625E1
).all():
    raise RuntimeError(
        "Reliability matrix contains "
        "non-finite values."
    )


if not (
    SCENARIO_PRESCORE_RESULTS_625E1[
        "prediction_finite"
    ].all()
):
    raise RuntimeError(
        "At least one scenario has "
        "non-finite predictions."
    )


if not (
    SCENARIO_PRESCORE_RESULTS_625E1[
        "mc_std_finite"
    ].all()
):
    raise RuntimeError(
        "At least one scenario has "
        "non-finite MC-STD values."
    )


if not (
    SCENARIO_PRESCORE_RESULTS_625E1[
        "reliability_finite"
    ].all()
):
    raise RuntimeError(
        "At least one scenario has "
        "non-finite reliability values."
    )


if not (
    SCENARIO_PRESCORE_RESULTS_625E1[
        "endpoint_ids_exact"
    ].all()
):
    raise RuntimeError(
        "Endpoint engine alignment failed "
        "for at least one scenario."
    )


if not (
    SCENARIO_PRESCORE_RESULTS_625E1[
        "endpoint_cycles_exact"
    ].all()
):
    raise RuntimeError(
        "Endpoint cycle alignment failed "
        "for at least one scenario."
    )


# ============================================================
# 15. Verify official TEST raw source was never modified
# ============================================================

OFFICIAL_TEST_RAW_UNCHANGED_625E1 = bool(
    OFFICIAL_TEST_RAW_624B.equals(
        OFFICIAL_TEST_RAW_SNAPSHOT_625E1
    )
)


if not OFFICIAL_TEST_RAW_UNCHANGED_625E1:
    raise RuntimeError(
        "Official TEST raw source was "
        "modified during evaluation."
    )


# ============================================================
# 16. Family-level PRE-SCORE summary
# ============================================================

FAMILY_PRESCORE_SUMMARY_625E1 = (
    SCENARIO_PRESCORE_RESULTS_625E1
    .groupby(
        "family",
        sort=False,
    )
    .agg(
        scenarios=(
            "global_index",
            "size",
        ),

        mean_coverage=(
            "coverage",
            "mean",
        ),

        min_coverage=(
            "coverage",
            "min",
        ),

        max_coverage=(
            "coverage",
            "max",
        ),

        mean_accepted=(
            "accepted",
            "mean",
        ),

        mean_decision_changes_vs_clean=(
            "decision_changes_vs_clean",
            "mean",
        ),

        max_decision_changes_vs_clean=(
            "decision_changes_vs_clean",
            "max",
        ),

        mean_abs_prediction_change=(
            "mean_abs_prediction_change",
            "mean",
        ),

        max_abs_prediction_change=(
            "max_abs_prediction_change",
            "max",
        ),

        mean_mc_std=(
            "mean_mc_std",
            "mean",
        ),

        mean_reliability_score=(
            "mean_reliability_score",
            "mean",
        ),
    )
    .reset_index()
)


# ============================================================
# 17. Display compact PRE-SCORE results
# ============================================================

elapsed_total_625e1 = float(
    time.perf_counter()
    -
    start_time_625e1
)


print(
    "\n"
    "============================================================"
)

print(
    "STEP 6.25E1 PRE-SCORE INFERENCE SUMMARY"
)

print(
    "============================================================"
)

print(
    "Frozen scenarios completed      :",
    len(
        SCENARIO_PRESCORE_RESULTS_625E1
    ),
    "/ 315"
)

print(
    "Engine-level records            :",
    len(
        ENGINE_PRESCORE_RESULTS_625E1
    )
)

print(
    "Prediction matrix shape         :",
    PREDICTION_MATRIX_625E1.shape
)

print(
    "MC-STD matrix shape             :",
    MC_STD_MATRIX_625E1.shape
)

print(
    "Reliability matrix shape        :",
    RELIABILITY_MATRIX_625E1.shape
)

print(
    "Accept matrix shape             :",
    ACCEPT_MATRIX_625E1.shape
)

print(
    "All prediction values finite    :",
    np.isfinite(
        PREDICTION_MATRIX_625E1
    ).all()
)

print(
    "All MC-STD values finite        :",
    np.isfinite(
        MC_STD_MATRIX_625E1
    ).all()
)

print(
    "All reliability values finite   :",
    np.isfinite(
        RELIABILITY_MATRIX_625E1
    ).all()
)

print(
    "Official TEST raw unchanged     :",
    OFFICIAL_TEST_RAW_UNCHANGED_625E1
)

print(
    "Frozen threshold unchanged      :",
    float(
        RELOADED_THRESHOLD_623DL5
    )
    ==
    FROZEN_TEST_THRESHOLD_625E1
)

print(
    "Official RUL labels used        : False"
)

print(
    "Official TEST scored            : False"
)

print(
    "Elapsed seconds                 :",
    elapsed_total_625e1
)


print(
    "\n--- Family-level PRE-SCORE summary ---"
)

display(
    FAMILY_PRESCORE_SUMMARY_625E1
)


print(
    "\n--- First 10 scenario-level records ---"
)

display(
    SCENARIO_PRESCORE_RESULTS_625E1.head(
        10
    )
)


print(
    "\n--- First 15 engine-level records ---"
)

display(
    ENGINE_PRESCORE_RESULTS_625E1.head(
        15
    )
)


# ============================================================
# 18. Persist frozen PRE-SCORE artifacts
# ============================================================

PROJECT_ROOT_625E1 = Path(
    r"C:\RUL_Reliability_Research"
)


OUTPUT_DIR_625E1 = (
    PROJECT_ROOT_625E1
    / "06_results"
    / "step6_official_fd001"
    / "corruption_transfer"
    / "full_prescore"
)


OUTPUT_DIR_625E1.mkdir(
    parents=True,
    exist_ok=True,
)


SCENARIO_CSV_PATH_625E1 = (
    OUTPUT_DIR_625E1
    / (
        "fd001_official_test_"
        "corruption_prescore_scenarios.csv"
    )
)


ENGINE_CSV_PATH_625E1 = (
    OUTPUT_DIR_625E1
    / (
        "fd001_official_test_"
        "corruption_prescore_endpoints.csv"
    )
)


FAMILY_CSV_PATH_625E1 = (
    OUTPUT_DIR_625E1
    / (
        "fd001_official_test_"
        "corruption_prescore_family_summary.csv"
    )
)


ARRAYS_PATH_625E1 = (
    OUTPUT_DIR_625E1
    / (
        "fd001_official_test_"
        "corruption_prescore_arrays.npz"
    )
)


SUMMARY_PATH_625E1 = (
    OUTPUT_DIR_625E1
    / (
        "fd001_official_test_"
        "corruption_prescore_summary.json"
    )
)


MANIFEST_PATH_625E1 = (
    OUTPUT_DIR_625E1
    / (
        "fd001_official_test_"
        "corruption_prescore_manifest.json"
    )
)


_save_csv_atomic_625e1(
    SCENARIO_PRESCORE_RESULTS_625E1,
    SCENARIO_CSV_PATH_625E1,
)


_save_csv_atomic_625e1(
    ENGINE_PRESCORE_RESULTS_625E1,
    ENGINE_CSV_PATH_625E1,
)


_save_csv_atomic_625e1(
    FAMILY_PRESCORE_SUMMARY_625E1,
    FAMILY_CSV_PATH_625E1,
)


temp_arrays_path_625e1 = Path(
    str(
        ARRAYS_PATH_625E1
    )
    + ".tmp.npz"
)


np.savez_compressed(
    temp_arrays_path_625e1,

    prediction=
        PREDICTION_MATRIX_625E1,

    mc_std=
        MC_STD_MATRIX_625E1,

    reliability=
        RELIABILITY_MATRIX_625E1,

    accept=
        ACCEPT_MATRIX_625E1,

    global_index=
        SCENARIO_PRESCORE_RESULTS_625E1[
            "global_index"
        ]
        .to_numpy(
            dtype=int
        ),

    unit_id=
        clean_unit_ids_625e1,

    cycle=
        clean_cycles_625e1,

    frozen_threshold=np.asarray(
        [
            FROZEN_TEST_THRESHOLD_625E1
        ],
        dtype=np.float64,
    ),
)


os.replace(
    temp_arrays_path_625e1,
    ARRAYS_PATH_625E1,
)


SUMMARY_625E1 = {
    "step":
        "6.25E1",

    "dataset":
        "NASA C-MAPSS FD001",

    "evaluation_stage":
        "official_TEST_pre_score",

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "official_test_raw_rows":
        int(
            len(
                OFFICIAL_TEST_RAW_624B
            )
        ),

    "official_test_engines":
        100,

    "corruption_scenarios":
        315,

    "family_counts":
        expected_family_counts_625e1,

    "endpoint_window_length":
        30,

    "primary_feature_count":
        17,

    "reliability_signal":
        "uncertainty_only",

    "frozen_reliability_threshold":
        FROZEN_TEST_THRESHOLD_625E1,

    "accept_rule":
        (
            "reliability_score <= "
            "reliability_threshold"
        ),

    "abstain_rule":
        (
            "reliability_score > "
            "reliability_threshold"
        ),

    "clean_target_coverage":
        0.75,

    "clean_accepted_endpoints":
        int(
            CLEAN_ACCEPT_STORED_625E1.sum()
        ),

    "clean_preprocessing_max_abs_error":
        clean_sequence_max_error_625e1,

    "clean_preprocessing_atol":
        CLEAN_SEQUENCE_ATOL_625E1,

    "clean_preprocessing_rtol":
        CLEAN_SEQUENCE_RTOL_625E1,

    "clean_preprocessing_numerically_equivalent":
        clean_sequences_equivalent_625e1,

    "clean_engine_ids_exact":
        clean_ids_exact_625e1,

    "clean_endpoint_cycles_exact":
        clean_cycles_exact_625e1,

    "clean_accept_mask_reproduced":
        clean_accept_mask_exact_625e1,

    "clean_prediction_probe_max_difference":
        clean_prediction_max_diff_625e1,

    "clean_mc_std_probe_max_difference":
        clean_mc_std_max_diff_625e1,

    "clean_reliability_probe_max_difference":
        clean_reliability_max_diff_625e1,

    "scenario_result_rows":
        int(
            len(
                SCENARIO_PRESCORE_RESULTS_625E1
            )
        ),

    "engine_result_rows":
        int(
            len(
                ENGINE_PRESCORE_RESULTS_625E1
            )
        ),

    "prediction_matrix_shape":
        list(
            PREDICTION_MATRIX_625E1.shape
        ),

    "mc_std_matrix_shape":
        list(
            MC_STD_MATRIX_625E1.shape
        ),

    "reliability_matrix_shape":
        list(
            RELIABILITY_MATRIX_625E1.shape
        ),

    "accept_matrix_shape":
        list(
            ACCEPT_MATRIX_625E1.shape
        ),

    "all_predictions_finite":
        bool(
            np.isfinite(
                PREDICTION_MATRIX_625E1
            ).all()
        ),

    "all_mc_std_finite":
        bool(
            np.isfinite(
                MC_STD_MATRIX_625E1
            ).all()
        ),

    "all_reliability_finite":
        bool(
            np.isfinite(
                RELIABILITY_MATRIX_625E1
            ).all()
        ),

    "official_test_raw_unchanged":
        OFFICIAL_TEST_RAW_UNCHANGED_625E1,

    "official_test_rul_labels_used":
        False,

    "official_test_scored":
        False,

    "threshold_modified":
        False,

    "reliability_rule_modified":
        False,

    "frozen_corruption_manifests_modified":
        False,

    "elapsed_seconds":
        elapsed_total_625e1,
}


_save_json_atomic_625e1(
    SUMMARY_625E1,
    SUMMARY_PATH_625E1,
)


# ============================================================
# 19. Hash verification manifest
# ============================================================

artifact_paths_625e1 = [
    SCENARIO_CSV_PATH_625E1,
    ENGINE_CSV_PATH_625E1,
    FAMILY_CSV_PATH_625E1,
    ARRAYS_PATH_625E1,
    SUMMARY_PATH_625E1,
]


artifact_hashes_625e1 = {
    path_625e1.name:
        _sha256_file_625e1(
            path_625e1
        )
    for path_625e1
    in artifact_paths_625e1
}


MANIFEST_625E1 = {
    "step":
        "6.25E1",

    "dataset":
        "NASA C-MAPSS FD001",

    "status":
        "FROZEN_PRE_SCORE",

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "output_directory":
        str(
            OUTPUT_DIR_625E1
        ),

    "reliability_signal":
        "uncertainty_only",

    "frozen_threshold":
        FROZEN_TEST_THRESHOLD_625E1,

    "scenario_count":
        315,

    "endpoint_engine_count":
        100,

    "official_labels_used":
        False,

    "official_test_scored":
        False,

    "files":
        artifact_hashes_625e1,
}


_save_json_atomic_625e1(
    MANIFEST_625E1,
    MANIFEST_PATH_625E1,
)


# Verify saved artifact hashes immediately
hash_recheck_625e1 = {
    path_625e1.name:
        (
            _sha256_file_625e1(
                path_625e1
            )
            ==
            artifact_hashes_625e1[
                path_625e1.name
            ]
        )
    for path_625e1
    in artifact_paths_625e1
}


all_saved_hashes_valid_625e1 = bool(
    all(
        hash_recheck_625e1.values()
    )
)


if not all_saved_hashes_valid_625e1:
    raise RuntimeError(
        "One or more saved Step 6.25E1 "
        "artifact hashes failed verification."
    )


# ============================================================
# 20. Final integrity summary
# ============================================================

print(
    "\n"
    "============================================================"
)

print(
    "STEP 6.25E1 FINAL INTEGRITY SUMMARY"
)

print(
    "============================================================"
)

print(
    "Scenarios completed              :",
    len(
        SCENARIO_PRESCORE_RESULTS_625E1
    ),
    "/ 315"
)

print(
    "Engine-level endpoint records    :",
    len(
        ENGINE_PRESCORE_RESULTS_625E1
    ),
    "/ 31500"
)

print(
    "Five families completed          :",
    SCENARIO_PRESCORE_RESULTS_625E1[
        "family"
    ].nunique()
    == 5
)

print(
    "Clean preprocessing equivalent   :",
    clean_sequences_equivalent_625e1
)

print(
    "Clean accept mask reproduced     :",
    clean_accept_mask_exact_625e1
)

print(
    "All predictions finite           :",
    np.isfinite(
        PREDICTION_MATRIX_625E1
    ).all()
)

print(
    "All MC-STD values finite         :",
    np.isfinite(
        MC_STD_MATRIX_625E1
    ).all()
)

print(
    "All reliability values finite    :",
    np.isfinite(
        RELIABILITY_MATRIX_625E1
    ).all()
)

print(
    "Official TEST raw unchanged      :",
    OFFICIAL_TEST_RAW_UNCHANGED_625E1
)

print(
    "Frozen threshold                 :",
    FROZEN_TEST_THRESHOLD_625E1
)

print(
    "Frozen threshold changed         : False"
)

print(
    "Frozen reliability rule changed  : False"
)

print(
    "Frozen manifests modified        : False"
)

print(
    "Official RUL labels used         : False"
)

print(
    "Official TEST scored             : False"
)

print(
    "All stored hashes valid          :",
    all_saved_hashes_valid_625e1
)

print(
    "Files written                    : True"
)

print(
    "\nOutput directory:"
)

print(
    OUTPUT_DIR_625E1
)

print(
    "\nFrozen files:"
)

for (
    artifact_number_625e1,
    artifact_path_625e1,
) in enumerate(
    [
        SCENARIO_CSV_PATH_625E1,
        ENGINE_CSV_PATH_625E1,
        FAMILY_CSV_PATH_625E1,
        ARRAYS_PATH_625E1,
        SUMMARY_PATH_625E1,
        MANIFEST_PATH_625E1,
    ],
    start=1,
):

    print(
        f"{artifact_number_625e1}. "
        f"{artifact_path_625e1}"
    )


print(
    "\n=== END STEP 6.25E1 ==="
)

In [ ]:
# ============================================================
# Step 6.25E2
# Frozen 315-scenario official FD001 TEST scoring
#
# IMPORTANT:
#   - NO model inference
#   - NO MC-dropout
#   - NO corruption regeneration
#   - NO threshold/rule modification
#   - NO tuning using TEST labels
#   - Official labels are used ONLY for final evaluation
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
import hashlib
import time
import math


print(
    "\n=== Step 6.25E2 — Frozen 315-scenario "
    "official FD001 TEST scoring ===\n"
)

STEP_START_625E2 = time.time()


# ============================================================
# 1. Resolve project paths
# ============================================================

if "PROJECT_ROOT" in globals():
    PROJECT_ROOT_625E2 = Path(PROJECT_ROOT)
else:
    PROJECT_ROOT_625E2 = Path(
        r"C:\RUL_Reliability_Research"
    )

PRESCORE_DIR_625E2 = (
    PROJECT_ROOT_625E2
    / "06_results"
    / "step6_official_fd001"
    / "corruption_transfer"
    / "full_prescore"
)

OUTPUT_DIR_625E2 = (
    PROJECT_ROOT_625E2
    / "06_results"
    / "step6_official_fd001"
    / "corruption_transfer"
    / "full_scoring"
)

DATA_DIR_625E2 = (
    PROJECT_ROOT_625E2
    / "02_dataset"
    / "raw"
    / "CMAPSSData"
)

OUTPUT_DIR_625E2.mkdir(
    parents=True,
    exist_ok=True
)

print("--- Paths ---")
print(
    "Project root :",
    PROJECT_ROOT_625E2
)
print(
    "Pre-score dir:",
    PRESCORE_DIR_625E2
)
print(
    "Output dir   :",
    OUTPUT_DIR_625E2
)
print(
    "Dataset dir  :",
    DATA_DIR_625E2
)

if not PRESCORE_DIR_625E2.exists():
    raise RuntimeError(
        "Frozen Step 6.25E1 pre-score directory "
        "does not exist."
    )

if not DATA_DIR_625E2.exists():
    raise RuntimeError(
        "Official C-MAPSS dataset directory "
        "does not exist."
    )


# ============================================================
# 2. Utility helpers
# ============================================================

def sha256_file_625e2(path):
    path = Path(path)

    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(
                1024 * 1024
            )

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


def json_safe_625e2(value):
    if isinstance(
        value,
        (
            np.integer,
            np.int8,
            np.int16,
            np.int32,
            np.int64,
        ),
    ):
        return int(value)

    if isinstance(
        value,
        (
            np.floating,
            np.float16,
            np.float32,
            np.float64,
        ),
    ):
        if np.isnan(value):
            return None

        if np.isinf(value):
            return None

        return float(value)

    if isinstance(
        value,
        np.bool_,
    ):
        return bool(value)

    if isinstance(
        value,
        Path,
    ):
        return str(value)

    if isinstance(
        value,
        dict,
    ):
        return {
            str(k): json_safe_625e2(v)
            for k, v in value.items()
        }

    if isinstance(
        value,
        (list, tuple),
    ):
        return [
            json_safe_625e2(v)
            for v in value
        ]

    return value


def choose_column_625e2(
    df,
    candidates,
    description,
    required=True,
):
    for col in candidates:
        if col in df.columns:
            return col

    if required:
        raise RuntimeError(
            f"Could not locate {description}. "
            f"Tried: {candidates}"
        )

    return None


def normalise_bool_625e2(series):
    if pd.api.types.is_bool_dtype(
        series
    ):
        return series.astype(bool)

    if pd.api.types.is_numeric_dtype(
        series
    ):
        values = set(
            pd.Series(series)
            .dropna()
            .unique()
            .tolist()
        )

        if values.issubset(
            {0, 1, 0.0, 1.0}
        ):
            return (
                pd.to_numeric(
                    series,
                    errors="raise",
                )
                .astype(int)
                .astype(bool)
            )

    lowered = (
        series.astype(str)
        .str.strip()
        .str.lower()
    )

    mapping = {
        "true": True,
        "false": False,
        "1": True,
        "0": False,
    }

    unknown = sorted(
        set(lowered.unique())
        - set(mapping.keys())
    )

    if unknown:
        raise RuntimeError(
            "Could not safely convert accept "
            f"column to boolean. Unknown: {unknown}"
        )

    return lowered.map(
        mapping
    ).astype(bool)


def calculate_metrics_625e2(
    frame,
    prediction_col,
    target_col,
):
    n = int(len(frame))

    if n == 0:
        return {
            "n": 0,
            "mae": np.nan,
            "rmse": np.nan,
            "median_AE": np.nan,
            "mean_error": np.nan,
            "mean_abs_error": np.nan,
            "overestimation_rate": np.nan,
            "underestimation_rate": np.nan,
        }

    prediction = pd.to_numeric(
        frame[prediction_col],
        errors="raise",
    ).to_numpy(
        dtype=np.float64
    )

    target = pd.to_numeric(
        frame[target_col],
        errors="raise",
    ).to_numpy(
        dtype=np.float64
    )

    if not np.isfinite(
        prediction
    ).all():
        raise RuntimeError(
            "Non-finite prediction encountered."
        )

    if not np.isfinite(
        target
    ).all():
        raise RuntimeError(
            "Non-finite target encountered."
        )

    error = prediction - target
    abs_error = np.abs(error)

    return {
        "n": n,
        "mae": float(
            np.mean(
                abs_error
            )
        ),
        "rmse": float(
            np.sqrt(
                np.mean(
                    error ** 2
                )
            )
        ),
        "median_AE": float(
            np.median(
                abs_error
            )
        ),
        "mean_error": float(
            np.mean(
                error
            )
        ),
        "mean_abs_error": float(
            np.mean(
                abs_error
            )
        ),
        "overestimation_rate": float(
            np.mean(
                error > 0
            )
        ),
        "underestimation_rate": float(
            np.mean(
                error < 0
            )
        ),
    }


# ============================================================
# 3. Locate exact frozen engine-level pre-score artifact
# ============================================================

print(
    "\n--- Frozen pre-score artifact discovery ---"
)

prescore_csv_files_625e2 = sorted(
    PRESCORE_DIR_625E2.glob(
        "*.csv"
    )
)

if not prescore_csv_files_625e2:
    raise RuntimeError(
        "No CSV artifacts found in the "
        "Step 6.25E1 pre-score directory."
    )


preferred_engine_files_625e2 = [
    p
    for p in prescore_csv_files_625e2
    if "engine" in p.name.lower()
]

search_files_625e2 = (
    preferred_engine_files_625e2
    + [
        p
        for p in prescore_csv_files_625e2
        if p
        not in preferred_engine_files_625e2
    ]
)

engine_candidates_625e2 = []

for path_625e2 in search_files_625e2:
    try:
        df_probe_625e2 = pd.read_csv(
            path_625e2
        )
    except Exception:
        continue

    columns_lower_625e2 = {
        str(c).lower()
        for c in df_probe_625e2.columns
    }

    has_core_625e2 = (
        "unit_id" in columns_lower_625e2
        and
        "family" in columns_lower_625e2
        and
        "global_index" in columns_lower_625e2
    )

    if (
        len(df_probe_625e2) == 31500
        and has_core_625e2
    ):
        engine_candidates_625e2.append(
            (
                path_625e2,
                df_probe_625e2,
            )
        )


if len(
    engine_candidates_625e2
) == 0:
    raise RuntimeError(
        "Could not locate the frozen "
        "31,500-row engine-level "
        "pre-score artifact."
    )

if len(
    engine_candidates_625e2
) > 1:
    print(
        "Multiple 31,500-row candidates found:"
    )

    for p, _ in engine_candidates_625e2:
        print(" ", p.name)

    print(
        "Using the first valid engine-level "
        "candidate."
    )


PRESCORE_ENGINE_PATH_625E2 = (
    engine_candidates_625e2[0][0]
)

PRESCORE_ENGINE_625E2 = (
    engine_candidates_625e2[0][1]
    .copy()
)

print(
    "Selected engine artifact:",
    PRESCORE_ENGINE_PATH_625E2.name
)

print(
    "Shape:",
    PRESCORE_ENGINE_625E2.shape
)


# ============================================================
# 4. Identify exact pre-score columns
# ============================================================

GLOBAL_INDEX_COL_625E2 = (
    choose_column_625e2(
        PRESCORE_ENGINE_625E2,
        [
            "global_index",
        ],
        "global scenario index",
    )
)

FAMILY_COL_625E2 = (
    choose_column_625e2(
        PRESCORE_ENGINE_625E2,
        [
            "family",
        ],
        "corruption family",
    )
)

UNIT_COL_625E2 = (
    choose_column_625e2(
        PRESCORE_ENGINE_625E2,
        [
            "unit_id",
        ],
        "engine ID",
    )
)

CYCLE_COL_625E2 = (
    choose_column_625e2(
        PRESCORE_ENGINE_625E2,
        [
            "cycle",
        ],
        "endpoint cycle",
    )
)

CORR_PRED_COL_625E2 = (
    choose_column_625e2(
        PRESCORE_ENGINE_625E2,
        [
            "corrupted_prediction",
            "prediction_corrupted",
            "prediction",
        ],
        "corrupted prediction",
    )
)

CLEAN_PRED_COL_625E2 = (
    choose_column_625e2(
        PRESCORE_ENGINE_625E2,
        [
            "clean_prediction",
        ],
        "clean prediction",
    )
)

CORR_ACCEPT_COL_625E2 = (
    choose_column_625e2(
        PRESCORE_ENGINE_625E2,
        [
            "corrupted_accept",
            "accept",
            "accept_verified",
        ],
        "corrupted accept mask",
    )
)

CLEAN_ACCEPT_COL_625E2 = (
    choose_column_625e2(
        PRESCORE_ENGINE_625E2,
        [
            "clean_accept",
        ],
        "clean accept mask",
    )
)

CORR_RELIABILITY_COL_625E2 = (
    choose_column_625e2(
        PRESCORE_ENGINE_625E2,
        [
            "corrupted_reliability",
            "corrupted_reliability_score",
            "reliability_score",
        ],
        "corrupted reliability score",
        required=False,
    )
)

CLEAN_RELIABILITY_COL_625E2 = (
    choose_column_625e2(
        PRESCORE_ENGINE_625E2,
        [
            "clean_reliability",
            "clean_reliability_score",
        ],
        "clean reliability score",
        required=False,
    )
)

SENSOR_COL_625E2 = (
    choose_column_625e2(
        PRESCORE_ENGINE_625E2,
        [
            "sensor",
        ],
        "sensor",
        required=False,
    )
)

SEVERITY_COL_625E2 = (
    choose_column_625e2(
        PRESCORE_ENGINE_625E2,
        [
            "severity",
        ],
        "severity",
        required=False,
    )
)

SCENARIO_ID_COL_625E2 = (
    choose_column_625e2(
        PRESCORE_ENGINE_625E2,
        [
            "scenario_id",
        ],
        "scenario ID",
        required=False,
    )
)

MANIFEST_SCENARIO_COL_625E2 = (
    choose_column_625e2(
        PRESCORE_ENGINE_625E2,
        [
            "manifest_scenario_id",
        ],
        "manifest scenario ID",
        required=False,
    )
)

FAMILY_POSITION_COL_625E2 = (
    choose_column_625e2(
        PRESCORE_ENGINE_625E2,
        [
            "family_position",
        ],
        "family position",
        required=False,
    )
)


PRESCORE_ENGINE_625E2[
    CORR_ACCEPT_COL_625E2
] = normalise_bool_625e2(
    PRESCORE_ENGINE_625E2[
        CORR_ACCEPT_COL_625E2
    ]
)

PRESCORE_ENGINE_625E2[
    CLEAN_ACCEPT_COL_625E2
] = normalise_bool_625e2(
    PRESCORE_ENGINE_625E2[
        CLEAN_ACCEPT_COL_625E2
    ]
)


print(
    "\n--- Detected frozen columns ---"
)

print(
    "Corrupted prediction :",
    CORR_PRED_COL_625E2
)

print(
    "Clean prediction     :",
    CLEAN_PRED_COL_625E2
)

print(
    "Corrupted accept     :",
    CORR_ACCEPT_COL_625E2
)

print(
    "Clean accept         :",
    CLEAN_ACCEPT_COL_625E2
)

print(
    "Corrupted reliability:",
    CORR_RELIABILITY_COL_625E2
)


# ============================================================
# 5. Validate complete frozen pre-score structure
# ============================================================

if len(
    PRESCORE_ENGINE_625E2
) != 31500:
    raise RuntimeError(
        "Expected exactly 31,500 "
        "engine-scenario records."
    )

scenario_count_625e2 = int(
    PRESCORE_ENGINE_625E2[
        GLOBAL_INDEX_COL_625E2
    ].nunique()
)

if scenario_count_625e2 != 315:
    raise RuntimeError(
        "Expected exactly 315 frozen scenarios."
    )

duplicate_pairs_625e2 = int(
    PRESCORE_ENGINE_625E2.duplicated(
        subset=[
            GLOBAL_INDEX_COL_625E2,
            UNIT_COL_625E2,
        ]
    ).sum()
)

if duplicate_pairs_625e2 != 0:
    raise RuntimeError(
        "Duplicate scenario-engine endpoint "
        "records detected."
    )

engine_counts_625e2 = (
    PRESCORE_ENGINE_625E2
    .groupby(
        GLOBAL_INDEX_COL_625E2
    )[
        UNIT_COL_625E2
    ]
    .nunique()
)

if not (
    engine_counts_625e2 == 100
).all():
    raise RuntimeError(
        "At least one scenario does not "
        "contain exactly 100 engines."
    )

expected_ids_625e2 = set(
    range(
        1,
        101,
    )
)

for idx_625e2, group_625e2 in (
    PRESCORE_ENGINE_625E2.groupby(
        GLOBAL_INDEX_COL_625E2,
        sort=False,
    )
):
    ids_625e2 = set(
        pd.to_numeric(
            group_625e2[
                UNIT_COL_625E2
            ],
            errors="raise",
        )
        .astype(int)
        .tolist()
    )

    if ids_625e2 != expected_ids_625e2:
        raise RuntimeError(
            "Scenario "
            f"{idx_625e2} does not contain "
            "exactly engine IDs 1..100."
        )


# ============================================================
# 6. Verify frozen CLEAN quantities are constant across
#    all 315 corruption scenarios
# ============================================================

clean_prediction_spread_625e2 = (
    PRESCORE_ENGINE_625E2
    .groupby(
        UNIT_COL_625E2
    )[
        CLEAN_PRED_COL_625E2
    ]
    .agg(
        lambda s:
        float(
            pd.to_numeric(
                s,
                errors="raise",
            ).max()
            -
            pd.to_numeric(
                s,
                errors="raise",
            ).min()
        )
    )
)

max_clean_prediction_spread_625e2 = float(
    clean_prediction_spread_625e2.max()
)

if (
    max_clean_prediction_spread_625e2
    > 1e-10
):
    raise RuntimeError(
        "Clean prediction is not stable "
        "across frozen scenarios."
    )


clean_accept_variation_625e2 = (
    PRESCORE_ENGINE_625E2
    .groupby(
        UNIT_COL_625E2
    )[
        CLEAN_ACCEPT_COL_625E2
    ]
    .nunique()
)

if not (
    clean_accept_variation_625e2 == 1
).all():
    raise RuntimeError(
        "Clean accept decision changes "
        "across corruption scenarios."
    )


print(
    "\n--- Frozen CLEAN consistency ---"
)

print(
    "Maximum clean prediction spread:",
    max_clean_prediction_spread_625e2
)

print(
    "Clean accept mask constant:",
    True
)


# ============================================================
# 7. Recover frozen reliability threshold WITHOUT modifying it
# ============================================================

FROZEN_THRESHOLD_625E2 = None

threshold_candidates_625e2 = [
    "RELATED_THRESHOLD_623DL5",
    "FROZEN_TEST_THRESHOLD_624E",
    "FROZEN_RELIABILITY_THRESHOLD_623DL3",
]

for name_625e2 in threshold_candidates_625e2:
    if name_625e2 in globals():
        value_625e2 = globals()[
            name_625e2
        ]

        try:
            FROZEN_THRESHOLD_625E2 = float(
                value_625e2
            )
            break
        except Exception:
            pass


if FROZEN_THRESHOLD_625E2 is None:
    frozen_decision_path_625e2 = (
        PROJECT_ROOT_625E2
        / "06_results"
        / "step6_reliability_signals"
        / "fd001_frozen_reliability_decision.json"
    )

    if frozen_decision_path_625e2.exists():
        with open(
            frozen_decision_path_625e2,
            "r",
            encoding="utf-8",
        ) as f:
            decision_json_625e2 = json.load(
                f
            )

        exact_keys_625e2 = [
            "reliability_threshold",
            "frozen_threshold",
            "exact_frozen_threshold",
            "threshold",
        ]

        def recursive_threshold_625e2(obj):
            if isinstance(
                obj,
                dict,
            ):
                for key in exact_keys_625e2:
                    if key in obj:
                        try:
                            return float(
                                obj[key]
                            )
                        except Exception:
                            pass

                for value in obj.values():
                    found = (
                        recursive_threshold_625e2(
                            value
                        )
                    )

                    if found is not None:
                        return found

            elif isinstance(
                obj,
                list,
            ):
                for value in obj:
                    found = (
                        recursive_threshold_625e2(
                            value
                        )
                    )

                    if found is not None:
                        return found

            return None

        FROZEN_THRESHOLD_625E2 = (
            recursive_threshold_625e2(
                decision_json_625e2
            )
        )


print(
    "\n--- Frozen reliability decision ---"
)

print(
    "Frozen threshold:",
    FROZEN_THRESHOLD_625E2
)

if (
    FROZEN_THRESHOLD_625E2
    is not None
    and
    CORR_RELIABILITY_COL_625E2
    is not None
):
    expected_accept_625e2 = (
        pd.to_numeric(
            PRESCORE_ENGINE_625E2[
                CORR_RELIABILITY_COL_625E2
            ],
            errors="raise",
        ).to_numpy(
            dtype=np.float64
        )
        <=
        float(
            FROZEN_THRESHOLD_625E2
        )
    )

    stored_accept_625e2 = (
        PRESCORE_ENGINE_625E2[
            CORR_ACCEPT_COL_625E2
        ]
        .to_numpy(
            dtype=bool
        )
    )

    frozen_mask_exact_625e2 = bool(
        np.array_equal(
            expected_accept_625e2,
            stored_accept_625e2,
        )
    )

    print(
        "Frozen corrupted accept mask exact:",
        frozen_mask_exact_625e2
    )

    if not frozen_mask_exact_625e2:
        raise RuntimeError(
            "Stored pre-score accept mask does "
            "not match the frozen threshold."
        )
else:
    frozen_mask_exact_625e2 = None

    print(
        "Frozen mask recomputation:",
        "not required / threshold unavailable"
    )


# ============================================================
# 8. Load official FD001 endpoint labels
#
# Priority:
#   A. Already-verified audit object from Step 6.24B
#   B. Reconstruct directly from untouched NASA files
# ============================================================

print(
    "\n--- Official endpoint label loading ---"
)

OFFICIAL_LABEL_SOURCE_625E2 = None


if (
    "OFFICIAL_TEST_ENDPOINT_AUDIT_624B"
    in globals()
):
    audit_obj_625e2 = globals()[
        "OFFICIAL_TEST_ENDPOINT_AUDIT_624B"
    ]

    required_audit_cols_625e2 = {
        "unit_id",
        "last_observed_cycle",
        "official_final_RUL",
    }

    if (
        isinstance(
            audit_obj_625e2,
            pd.DataFrame,
        )
        and
        required_audit_cols_625e2.issubset(
            audit_obj_625e2.columns
        )
        and
        len(
            audit_obj_625e2
        ) == 100
    ):
        OFFICIAL_LABELS_625E2 = (
            audit_obj_625e2[
                [
                    "unit_id",
                    "last_observed_cycle",
                    "official_final_RUL",
                ]
            ]
            .copy()
        )

        OFFICIAL_LABEL_SOURCE_625E2 = (
            "OFFICIAL_TEST_ENDPOINT_AUDIT_624B"
        )


if OFFICIAL_LABEL_SOURCE_625E2 is None:

    test_path_625e2 = (
        DATA_DIR_625E2
        / "test_FD001.txt"
    )

    rul_path_625e2 = (
        DATA_DIR_625E2
        / "RUL_FD001.txt"
    )

    if not test_path_625e2.exists():
        raise RuntimeError(
            "Official test_FD001.txt not found."
        )

    if not rul_path_625e2.exists():
        raise RuntimeError(
            "Official RUL_FD001.txt not found."
        )

    raw_test_625e2 = pd.read_csv(
        test_path_625e2,
        sep=r"\s+",
        header=None,
    )

    raw_rul_625e2 = pd.read_csv(
        rul_path_625e2,
        sep=r"\s+",
        header=None,
        names=[
            "official_final_RUL"
        ],
    )

    endpoint_cycles_625e2 = (
        raw_test_625e2
        .groupby(0)[1]
        .max()
        .sort_index()
    )

    if len(
        endpoint_cycles_625e2
    ) != 100:
        raise RuntimeError(
            "Official TEST does not contain "
            "exactly 100 engines."
        )

    if len(
        raw_rul_625e2
    ) != 100:
        raise RuntimeError(
            "Official RUL file does not contain "
            "exactly 100 endpoint labels."
        )

    OFFICIAL_LABELS_625E2 = pd.DataFrame(
        {
            "unit_id":
                np.arange(
                    1,
                    101,
                    dtype=int,
                ),
            "last_observed_cycle":
                endpoint_cycles_625e2
                .to_numpy(
                    dtype=int
                ),
            "official_final_RUL":
                raw_rul_625e2[
                    "official_final_RUL"
                ]
                .to_numpy(
                    dtype=np.float64
                ),
        }
    )

    OFFICIAL_LABEL_SOURCE_625E2 = (
        "NASA raw test_FD001.txt + "
        "RUL_FD001.txt"
    )


OFFICIAL_LABELS_625E2[
    "unit_id"
] = pd.to_numeric(
    OFFICIAL_LABELS_625E2[
        "unit_id"
    ],
    errors="raise",
).astype(int)

OFFICIAL_LABELS_625E2[
    "last_observed_cycle"
] = pd.to_numeric(
    OFFICIAL_LABELS_625E2[
        "last_observed_cycle"
    ],
    errors="raise",
).astype(int)

OFFICIAL_LABELS_625E2[
    "NASA_raw_RUL"
] = pd.to_numeric(
    OFFICIAL_LABELS_625E2[
        "official_final_RUL"
    ],
    errors="raise",
).astype(
    np.float64
)

# ------------------------------------------------------------
# Frozen training/evaluation convention recovered in Step 6.24C:
# RUL_cap_125 = min(raw RUL, 125)
# ------------------------------------------------------------

OFFICIAL_LABELS_625E2[
    "RUL_cap_125"
] = np.minimum(
    OFFICIAL_LABELS_625E2[
        "NASA_raw_RUL"
    ].to_numpy(
        dtype=np.float64
    ),
    125.0,
)


if len(
    OFFICIAL_LABELS_625E2
) != 100:
    raise RuntimeError(
        "Official endpoint label table must "
        "contain exactly 100 rows."
    )

if set(
    OFFICIAL_LABELS_625E2[
        "unit_id"
    ]
) != set(
    range(
        1,
        101,
    )
):
    raise RuntimeError(
        "Official endpoint unit IDs are not "
        "exactly 1..100."
    )

if not np.isfinite(
    OFFICIAL_LABELS_625E2[
        [
            "NASA_raw_RUL",
            "RUL_cap_125",
        ]
    ].to_numpy(
        dtype=np.float64
    )
).all():
    raise RuntimeError(
        "Official endpoint RUL labels contain "
        "non-finite values."
    )


print(
    "Label source:",
    OFFICIAL_LABEL_SOURCE_625E2
)

print(
    "Endpoint labels:",
    len(
        OFFICIAL_LABELS_625E2
    )
)

print(
    "Frozen target convention:",
    "RUL_cap_125"
)

print(
    "NASA raw labels finite:",
    True
)


# ============================================================
# 9. Attach official labels to the frozen 31,500 pre-score rows
# ============================================================

SCORED_ENGINE_625E2 = (
    PRESCORE_ENGINE_625E2
    .merge(
        OFFICIAL_LABELS_625E2[
            [
                "unit_id",
                "last_observed_cycle",
                "NASA_raw_RUL",
                "RUL_cap_125",
            ]
        ],
        left_on=UNIT_COL_625E2,
        right_on="unit_id",
        how="left",
        validate="many_to_one",
        suffixes=(
            "",
            "_official",
        ),
    )
)


if SCORED_ENGINE_625E2[
    "NASA_raw_RUL"
].isna().any():
    raise RuntimeError(
        "At least one pre-score engine row "
        "failed official-label alignment."
    )


cycle_difference_625e2 = (
    pd.to_numeric(
        SCORED_ENGINE_625E2[
            CYCLE_COL_625E2
        ],
        errors="raise",
    ).astype(int)
    -
    SCORED_ENGINE_625E2[
        "last_observed_cycle"
    ].astype(int)
)

max_cycle_difference_625e2 = int(
    np.max(
        np.abs(
            cycle_difference_625e2
            .to_numpy()
        )
    )
)

if max_cycle_difference_625e2 != 0:
    raise RuntimeError(
        "Frozen pre-score endpoint cycles "
        "do not exactly match official "
        "last-observed TEST cycles."
    )


# ============================================================
# 10. Add error columns
#
# Error convention:
#     prediction - true RUL
#
# Positive = RUL overestimation
# Negative = RUL underestimation
# ============================================================

corr_pred_values_625e2 = (
    pd.to_numeric(
        SCORED_ENGINE_625E2[
            CORR_PRED_COL_625E2
        ],
        errors="raise",
    )
    .to_numpy(
        dtype=np.float64
    )
)

clean_pred_values_625e2 = (
    pd.to_numeric(
        SCORED_ENGINE_625E2[
            CLEAN_PRED_COL_625E2
        ],
        errors="raise",
    )
    .to_numpy(
        dtype=np.float64
    )
)

if not np.isfinite(
    corr_pred_values_625e2
).all():
    raise RuntimeError(
        "Frozen corrupted predictions contain "
        "non-finite values."
    )

if not np.isfinite(
    clean_pred_values_625e2
).all():
    raise RuntimeError(
        "Frozen clean predictions contain "
        "non-finite values."
    )


cap_target_625e2 = (
    SCORED_ENGINE_625E2[
        "RUL_cap_125"
    ].to_numpy(
        dtype=np.float64
    )
)

raw_target_625e2 = (
    SCORED_ENGINE_625E2[
        "NASA_raw_RUL"
    ].to_numpy(
        dtype=np.float64
    )
)


SCORED_ENGINE_625E2[
    "frozen_target_error"
] = (
    corr_pred_values_625e2
    -
    cap_target_625e2
)

SCORED_ENGINE_625E2[
    "frozen_target_abs_error"
] = np.abs(
    SCORED_ENGINE_625E2[
        "frozen_target_error"
    ]
)

SCORED_ENGINE_625E2[
    "NASA_raw_error"
] = (
    corr_pred_values_625e2
    -
    raw_target_625e2
)

SCORED_ENGINE_625E2[
    "NASA_raw_abs_error"
] = np.abs(
    SCORED_ENGINE_625E2[
        "NASA_raw_error"
    ]
)

SCORED_ENGINE_625E2[
    "clean_frozen_target_error"
] = (
    clean_pred_values_625e2
    -
    cap_target_625e2
)

SCORED_ENGINE_625E2[
    "clean_frozen_target_abs_error"
] = np.abs(
    SCORED_ENGINE_625E2[
        "clean_frozen_target_error"
    ]
)

SCORED_ENGINE_625E2[
    "clean_NASA_raw_error"
] = (
    clean_pred_values_625e2
    -
    raw_target_625e2
)

SCORED_ENGINE_625E2[
    "clean_NASA_raw_abs_error"
] = np.abs(
    SCORED_ENGINE_625E2[
        "clean_NASA_raw_error"
    ]
)


# ============================================================
# 11. Build one clean endpoint table
# ============================================================

clean_columns_625e2 = [
    UNIT_COL_625E2,
    CYCLE_COL_625E2,
    CLEAN_PRED_COL_625E2,
    CLEAN_ACCEPT_COL_625E2,
]

if CLEAN_RELIABILITY_COL_625E2 is not None:
    clean_columns_625e2.append(
        CLEAN_RELIABILITY_COL_625E2
    )


CLEAN_ENDPOINT_625E2 = (
    SCORED_ENGINE_625E2[
        clean_columns_625e2
        + [
            "NASA_raw_RUL",
            "RUL_cap_125",
        ]
    ]
    .drop_duplicates(
        subset=[
            UNIT_COL_625E2
        ]
    )
    .sort_values(
        UNIT_COL_625E2
    )
    .reset_index(
        drop=True
    )
)


if len(
    CLEAN_ENDPOINT_625E2
) != 100:
    raise RuntimeError(
        "Clean endpoint reference does not "
        "contain exactly 100 engines."
    )


clean_cap_metrics_625e2 = (
    calculate_metrics_625e2(
        CLEAN_ENDPOINT_625E2,
        CLEAN_PRED_COL_625E2,
        "RUL_cap_125",
    )
)

clean_raw_metrics_625e2 = (
    calculate_metrics_625e2(
        CLEAN_ENDPOINT_625E2,
        CLEAN_PRED_COL_625E2,
        "NASA_raw_RUL",
    )
)


clean_accept_mask_625e2 = (
    CLEAN_ENDPOINT_625E2[
        CLEAN_ACCEPT_COL_625E2
    ].astype(bool)
)


clean_cap_selective_625e2 = (
    calculate_metrics_625e2(
        CLEAN_ENDPOINT_625E2.loc[
            clean_accept_mask_625e2
        ],
        CLEAN_PRED_COL_625E2,
        "RUL_cap_125",
    )
)

clean_raw_selective_625e2 = (
    calculate_metrics_625e2(
        CLEAN_ENDPOINT_625E2.loc[
            clean_accept_mask_625e2
        ],
        CLEAN_PRED_COL_625E2,
        "NASA_raw_RUL",
    )
)


print(
    "\n--- CLEAN endpoint scoring verification ---"
)

print(
    "Clean accepted:",
    int(
        clean_accept_mask_625e2.sum()
    ),
    "/ 100"
)

print(
    "Clean coverage:",
    float(
        clean_accept_mask_625e2.mean()
    )
)

print(
    "Frozen-target full MAE:",
    clean_cap_metrics_625e2[
        "mae"
    ]
)

print(
    "Frozen-target selective MAE:",
    clean_cap_selective_625e2[
        "mae"
    ]
)

print(
    "NASA-raw full MAE:",
    clean_raw_metrics_625e2[
        "mae"
    ]
)

print(
    "NASA-raw selective MAE:",
    clean_raw_selective_625e2[
        "mae"
    ]
)


# ============================================================
# 12. Score every one of the 315 frozen scenarios
# ============================================================

scenario_metric_rows_625e2 = []
scenario_wide_rows_625e2 = []


metadata_candidates_625e2 = [
    GLOBAL_INDEX_COL_625E2,
    FAMILY_COL_625E2,
]

for optional_col_625e2 in [
    FAMILY_POSITION_COL_625E2,
    SCENARIO_ID_COL_625E2,
    MANIFEST_SCENARIO_COL_625E2,
    SENSOR_COL_625E2,
    SEVERITY_COL_625E2,
]:
    if (
        optional_col_625e2
        is not None
        and
        optional_col_625e2
        not in metadata_candidates_625e2
    ):
        metadata_candidates_625e2.append(
            optional_col_625e2
        )


for (
    global_index_625e2,
    group_625e2,
) in SCORED_ENGINE_625E2.groupby(
    GLOBAL_INDEX_COL_625E2,
    sort=True,
):

    if len(
        group_625e2
    ) != 100:
        raise RuntimeError(
            f"Scenario {global_index_625e2} "
            "does not have 100 endpoint rows."
        )

    metadata_625e2 = {}

    for col_625e2 in (
        metadata_candidates_625e2
    ):
        values_625e2 = (
            group_625e2[
                col_625e2
            ]
            .drop_duplicates()
        )

        if len(
            values_625e2
        ) > 1:
            raise RuntimeError(
                f"Scenario {global_index_625e2}: "
                f"metadata column {col_625e2} "
                "is not constant."
            )

        metadata_625e2[
            col_625e2
        ] = (
            values_625e2.iloc[0]
            if len(
                values_625e2
            )
            else None
        )

    accepted_mask_625e2 = (
        group_625e2[
            CORR_ACCEPT_COL_625E2
        ].astype(bool)
    )

    abstained_mask_625e2 = (
        ~accepted_mask_625e2
    )

    accepted_n_625e2 = int(
        accepted_mask_625e2.sum()
    )

    abstained_n_625e2 = int(
        abstained_mask_625e2.sum()
    )

    coverage_625e2 = float(
        accepted_n_625e2
        /
        len(
            group_625e2
        )
    )


    for (
        label_basis_625e2,
        target_col_625e2,
    ) in [
        (
            "RUL_cap_125",
            "RUL_cap_125",
        ),
        (
            "NASA_raw_RUL",
            "NASA_raw_RUL",
        ),
    ]:

        full_metrics_625e2 = (
            calculate_metrics_625e2(
                group_625e2,
                CORR_PRED_COL_625E2,
                target_col_625e2,
            )
        )

        accepted_metrics_625e2 = (
            calculate_metrics_625e2(
                group_625e2.loc[
                    accepted_mask_625e2
                ],
                CORR_PRED_COL_625E2,
                target_col_625e2,
            )
        )

        abstained_metrics_625e2 = (
            calculate_metrics_625e2(
                group_625e2.loc[
                    abstained_mask_625e2
                ],
                CORR_PRED_COL_625E2,
                target_col_625e2,
            )
        )


        mae_gain_abs_625e2 = (
            full_metrics_625e2[
                "mae"
            ]
            -
            accepted_metrics_625e2[
                "mae"
            ]
        )

        if (
            np.isfinite(
                full_metrics_625e2[
                    "mae"
                ]
            )
            and
            full_metrics_625e2[
                "mae"
            ] > 0
        ):
            mae_gain_rel_625e2 = (
                mae_gain_abs_625e2
                /
                full_metrics_625e2[
                    "mae"
                ]
            )
        else:
            mae_gain_rel_625e2 = (
                np.nan
            )


        for (
            subset_name_625e2,
            subset_metrics_625e2,
            subset_coverage_625e2,
        ) in [
            (
                "all",
                full_metrics_625e2,
                1.0,
            ),
            (
                "accepted",
                accepted_metrics_625e2,
                coverage_625e2,
            ),
            (
                "abstained",
                abstained_metrics_625e2,
                1.0
                -
                coverage_625e2,
            ),
        ]:

            row_625e2 = dict(
                metadata_625e2
            )

            row_625e2.update(
                {
                    "label_basis":
                        label_basis_625e2,

                    "subset":
                        subset_name_625e2,

                    "n":
                        subset_metrics_625e2[
                            "n"
                        ],

                    "coverage":
                        subset_coverage_625e2,

                    "MAE":
                        subset_metrics_625e2[
                            "mae"
                        ],

                    "RMSE":
                        subset_metrics_625e2[
                            "rmse"
                        ],

                    "median_AE":
                        subset_metrics_625e2[
                            "median_AE"
                        ],

                    "mean_error":
                        subset_metrics_625e2[
                            "mean_error"
                        ],

                    "overestimation_rate":
                        subset_metrics_625e2[
                            "overestimation_rate"
                        ],

                    "underestimation_rate":
                        subset_metrics_625e2[
                            "underestimation_rate"
                        ],

                    "selective_MAE_gain_abs":
                        (
                            mae_gain_abs_625e2
                            if
                            subset_name_625e2
                            == "accepted"
                            else
                            np.nan
                        ),

                    "selective_MAE_gain_rel":
                        (
                            mae_gain_rel_625e2
                            if
                            subset_name_625e2
                            == "accepted"
                            else
                            np.nan
                        ),
                }
            )

            scenario_metric_rows_625e2.append(
                row_625e2
            )


        wide_row_625e2 = dict(
            metadata_625e2
        )

        wide_row_625e2.update(
            {
                "label_basis":
                    label_basis_625e2,

                "accepted":
                    accepted_n_625e2,

                "abstained":
                    abstained_n_625e2,

                "coverage":
                    coverage_625e2,

                "full_MAE":
                    full_metrics_625e2[
                        "mae"
                    ],

                "selective_MAE":
                    accepted_metrics_625e2[
                        "mae"
                    ],

                "abstained_MAE":
                    abstained_metrics_625e2[
                        "mae"
                    ],

                "full_RMSE":
                    full_metrics_625e2[
                        "rmse"
                    ],

                "selective_RMSE":
                    accepted_metrics_625e2[
                        "rmse"
                    ],

                "abstained_RMSE":
                    abstained_metrics_625e2[
                        "rmse"
                    ],

                "full_median_AE":
                    full_metrics_625e2[
                        "median_AE"
                    ],

                "selective_median_AE":
                    accepted_metrics_625e2[
                        "median_AE"
                    ],

                "abstained_median_AE":
                    abstained_metrics_625e2[
                        "median_AE"
                    ],

                "full_mean_error":
                    full_metrics_625e2[
                        "mean_error"
                    ],

                "selective_mean_error":
                    accepted_metrics_625e2[
                        "mean_error"
                    ],

                "abstained_mean_error":
                    abstained_metrics_625e2[
                        "mean_error"
                    ],

                "MAE_gain_abs":
                    mae_gain_abs_625e2,

                "MAE_gain_rel":
                    mae_gain_rel_625e2,

                "selective_improved":
                    bool(
                        mae_gain_abs_625e2
                        > 0
                    ),
            }
        )

        scenario_wide_rows_625e2.append(
            wide_row_625e2
        )


SCENARIO_METRICS_625E2 = pd.DataFrame(
    scenario_metric_rows_625e2
)

SCENARIO_WIDE_625E2 = pd.DataFrame(
    scenario_wide_rows_625e2
)


if len(
    SCENARIO_METRICS_625E2
) != (
    315
    * 2
    * 3
):
    raise RuntimeError(
        "Scenario metric table has "
        "unexpected row count."
    )

if len(
    SCENARIO_WIDE_625E2
) != (
    315
    * 2
):
    raise RuntimeError(
        "Scenario-wide score table has "
        "unexpected row count."
    )


# ============================================================
# 13. Primary frozen-target scenario table
# ============================================================

PRIMARY_SCENARIO_625E2 = (
    SCENARIO_WIDE_625E2.loc[
        SCENARIO_WIDE_625E2[
            "label_basis"
        ]
        ==
        "RUL_cap_125"
    ]
    .copy()
    .reset_index(
        drop=True
    )
)

if len(
    PRIMARY_SCENARIO_625E2
) != 315:
    raise RuntimeError(
        "Primary frozen-target scenario table "
        "must contain 315 rows."
    )


# ============================================================
# 14. Family-level summaries
# ============================================================

def family_summary_625e2(
    frame,
    group_columns,
):
    rows = []

    for keys, group in frame.groupby(
        group_columns,
        dropna=False,
        sort=True,
    ):
        if not isinstance(
            keys,
            tuple,
        ):
            keys = (
                keys,
            )

        row = {
            col: value
            for col, value
            in zip(
                group_columns,
                keys,
            )
        }

        gain_rel = pd.to_numeric(
            group[
                "MAE_gain_rel"
            ],
            errors="coerce",
        )

        gain_abs = pd.to_numeric(
            group[
                "MAE_gain_abs"
            ],
            errors="coerce",
        )

        row.update(
            {
                "scenarios":
                    int(
                        len(
                            group
                        )
                    ),

                "mean_coverage":
                    float(
                        group[
                            "coverage"
                        ].mean()
                    ),

                "min_coverage":
                    float(
                        group[
                            "coverage"
                        ].min()
                    ),

                "max_coverage":
                    float(
                        group[
                            "coverage"
                        ].max()
                    ),

                "mean_full_MAE":
                    float(
                        group[
                            "full_MAE"
                        ].mean()
                    ),

                "mean_selective_MAE":
                    float(
                        group[
                            "selective_MAE"
                        ].mean()
                    ),

                "mean_abstained_MAE":
                    float(
                        group[
                            "abstained_MAE"
                        ].mean()
                    ),

                "mean_MAE_gain_abs":
                    float(
                        gain_abs.mean()
                    ),

                "median_MAE_gain_abs":
                    float(
                        gain_abs.median()
                    ),

                "mean_MAE_gain_rel":
                    float(
                        gain_rel.mean()
                    ),

                "median_MAE_gain_rel":
                    float(
                        gain_rel.median()
                    ),

                "scenarios_improved":
                    int(
                        (
                            gain_abs > 0
                        ).sum()
                    ),

                "scenarios_not_improved":
                    int(
                        (
                            gain_abs <= 0
                        ).sum()
                    ),

                "improvement_fraction":
                    float(
                        (
                            gain_abs > 0
                        ).mean()
                    ),

                "mean_full_RMSE":
                    float(
                        group[
                            "full_RMSE"
                        ].mean()
                    ),

                "mean_selective_RMSE":
                    float(
                        group[
                            "selective_RMSE"
                        ].mean()
                    ),
            }
        )

        rows.append(
            row
        )

    return pd.DataFrame(
        rows
    )


FAMILY_SUMMARY_625E2 = (
    family_summary_625e2(
        SCENARIO_WIDE_625E2,
        [
            "label_basis",
            FAMILY_COL_625E2,
        ],
    )
)


severity_group_columns_625e2 = [
    "label_basis",
    FAMILY_COL_625E2,
]

if SEVERITY_COL_625E2 is not None:
    severity_group_columns_625e2.append(
        SEVERITY_COL_625E2
    )


SEVERITY_SUMMARY_625E2 = (
    family_summary_625e2(
        SCENARIO_WIDE_625E2,
        severity_group_columns_625e2,
    )
)


# ============================================================
# 15. Overall frozen-target summary
# ============================================================

primary_gain_abs_625e2 = (
    pd.to_numeric(
        PRIMARY_SCENARIO_625E2[
            "MAE_gain_abs"
        ],
        errors="raise",
    )
)

primary_gain_rel_625e2 = (
    pd.to_numeric(
        PRIMARY_SCENARIO_625E2[
            "MAE_gain_rel"
        ],
        errors="raise",
    )
)

primary_scenarios_improved_625e2 = int(
    (
        primary_gain_abs_625e2 > 0
    ).sum()
)

primary_scenarios_not_improved_625e2 = int(
    (
        primary_gain_abs_625e2 <= 0
    ).sum()
)


# ============================================================
# 16. Final integrity checks BEFORE writing
# ============================================================

all_engine_predictions_finite_625e2 = bool(
    np.isfinite(
        pd.to_numeric(
            SCORED_ENGINE_625E2[
                CORR_PRED_COL_625E2
            ],
            errors="raise",
        ).to_numpy(
            dtype=np.float64
        )
    ).all()
)

all_score_values_finite_625e2 = bool(
    np.isfinite(
        PRIMARY_SCENARIO_625E2[
            [
                "coverage",
                "full_MAE",
                "selective_MAE",
                "abstained_MAE",
                "full_RMSE",
                "selective_RMSE",
                "MAE_gain_abs",
                "MAE_gain_rel",
            ]
        ].to_numpy(
            dtype=np.float64
        )
    ).all()
)


if not all_engine_predictions_finite_625e2:
    raise RuntimeError(
        "Scored engine table contains "
        "non-finite predictions."
    )

if not all_score_values_finite_625e2:
    raise RuntimeError(
        "Primary scoring table contains "
        "non-finite required metrics."
    )


# ============================================================
# 17. Display key results
# ============================================================

print(
    "\n--- Primary frozen-target family scoring ---"
)

PRIMARY_FAMILY_DISPLAY_625E2 = (
    FAMILY_SUMMARY_625E2.loc[
        FAMILY_SUMMARY_625E2[
            "label_basis"
        ]
        ==
        "RUL_cap_125"
    ]
    .copy()
    .reset_index(
        drop=True
    )
)

display(
    PRIMARY_FAMILY_DISPLAY_625E2[
        [
            FAMILY_COL_625E2,
            "scenarios",
            "mean_coverage",
            "mean_full_MAE",
            "mean_selective_MAE",
            "mean_abstained_MAE",
            "mean_MAE_gain_abs",
            "mean_MAE_gain_rel",
            "scenarios_improved",
            "improvement_fraction",
        ]
    ]
)


print(
    "\n--- First 10 primary scenario scores ---"
)

primary_display_cols_625e2 = [
    GLOBAL_INDEX_COL_625E2,
    FAMILY_COL_625E2,
]

for optional_col in [
    SENSOR_COL_625E2,
    SEVERITY_COL_625E2,
    SCENARIO_ID_COL_625E2,
]:
    if (
        optional_col is not None
        and
        optional_col
        not in primary_display_cols_625e2
    ):
        primary_display_cols_625e2.append(
            optional_col
        )

primary_display_cols_625e2 += [
    "accepted",
    "abstained",
    "coverage",
    "full_MAE",
    "selective_MAE",
    "abstained_MAE",
    "MAE_gain_abs",
    "MAE_gain_rel",
    "selective_improved",
]

display(
    PRIMARY_SCENARIO_625E2[
        primary_display_cols_625e2
    ].head(
        10
    )
)


# ============================================================
# 18. Persist scoring artifacts
# ============================================================

ENGINE_SCORE_PATH_625E2 = (
    OUTPUT_DIR_625E2
    / (
        "fd001_official_test_corruption_"
        "scored_engine_records.csv"
    )
)

SCENARIO_METRIC_PATH_625E2 = (
    OUTPUT_DIR_625E2
    / (
        "fd001_official_test_corruption_"
        "scenario_metrics_long.csv"
    )
)

SCENARIO_PRIMARY_PATH_625E2 = (
    OUTPUT_DIR_625E2
    / (
        "fd001_official_test_corruption_"
        "primary_scenario_summary.csv"
    )
)

SCENARIO_ALL_PATH_625E2 = (
    OUTPUT_DIR_625E2
    / (
        "fd001_official_test_corruption_"
        "scenario_summary_all_targets.csv"
    )
)

FAMILY_SCORE_PATH_625E2 = (
    OUTPUT_DIR_625E2
    / (
        "fd001_official_test_corruption_"
        "family_scoring_summary.csv"
    )
)

SEVERITY_SCORE_PATH_625E2 = (
    OUTPUT_DIR_625E2
    / (
        "fd001_official_test_corruption_"
        "severity_scoring_summary.csv"
    )
)

CLEAN_SCORE_PATH_625E2 = (
    OUTPUT_DIR_625E2
    / (
        "fd001_official_test_clean_"
        "endpoint_scoring_reference.csv"
    )
)

SUMMARY_PATH_625E2 = (
    OUTPUT_DIR_625E2
    / (
        "fd001_official_test_corruption_"
        "scoring_summary.json"
    )
)

MANIFEST_PATH_625E2 = (
    OUTPUT_DIR_625E2
    / (
        "fd001_official_test_corruption_"
        "scoring_manifest.json"
    )
)


SCORED_ENGINE_625E2.to_csv(
    ENGINE_SCORE_PATH_625E2,
    index=False,
    float_format="%.17g",
)

SCENARIO_METRICS_625E2.to_csv(
    SCENARIO_METRIC_PATH_625E2,
    index=False,
    float_format="%.17g",
)

PRIMARY_SCENARIO_625E2.to_csv(
    SCENARIO_PRIMARY_PATH_625E2,
    index=False,
    float_format="%.17g",
)

SCENARIO_WIDE_625E2.to_csv(
    SCENARIO_ALL_PATH_625E2,
    index=False,
    float_format="%.17g",
)

FAMILY_SUMMARY_625E2.to_csv(
    FAMILY_SCORE_PATH_625E2,
    index=False,
    float_format="%.17g",
)

SEVERITY_SUMMARY_625E2.to_csv(
    SEVERITY_SCORE_PATH_625E2,
    index=False,
    float_format="%.17g",
)

CLEAN_ENDPOINT_625E2.to_csv(
    CLEAN_SCORE_PATH_625E2,
    index=False,
    float_format="%.17g",
)


# ============================================================
# 19. Create frozen scoring summary
# ============================================================

SUMMARY_625E2 = {
    "step":
        "6.25E2",

    "dataset":
        "NASA C-MAPSS FD001",

    "evaluation_type":
        (
            "official TEST endpoint evaluation "
            "of frozen pre-score predictions"
        ),

    "prescore_engine_artifact":
        str(
            PRESCORE_ENGINE_PATH_625E2
        ),

    "prescore_rows":
        int(
            len(
                PRESCORE_ENGINE_625E2
            )
        ),

    "scenarios":
        int(
            scenario_count_625e2
        ),

    "engines_per_scenario":
        100,

    "families":
        sorted(
            PRESCORE_ENGINE_625E2[
                FAMILY_COL_625E2
            ]
            .dropna()
            .astype(str)
            .unique()
            .tolist()
        ),

    "official_label_source":
        OFFICIAL_LABEL_SOURCE_625E2,

    "primary_frozen_target":
        "RUL_cap_125",

    "secondary_reporting_target":
        "NASA_raw_RUL",

    "frozen_threshold":
        (
            FROZEN_THRESHOLD_625E2
        ),

    "frozen_threshold_modified":
        False,

    "frozen_reliability_rule_modified":
        False,

    "frozen_corruption_manifests_modified":
        False,

    "model_inference_executed":
        False,

    "mc_dropout_executed":
        False,

    "corruptions_regenerated":
        False,

    "official_labels_used_for_tuning":
        False,

    "official_labels_used_for_threshold_selection":
        False,

    "official_labels_used_for_model_selection":
        False,

    "official_labels_used_for_evaluation":
        True,

    "clean_reference": {
        "coverage":
            float(
                clean_accept_mask_625e2.mean()
            ),

        "accepted":
            int(
                clean_accept_mask_625e2.sum()
            ),

        "abstained":
            int(
                (
                    ~clean_accept_mask_625e2
                ).sum()
            ),

        "RUL_cap_125_full_MAE":
            clean_cap_metrics_625e2[
                "mae"
            ],

        "RUL_cap_125_selective_MAE":
            clean_cap_selective_625e2[
                "mae"
            ],

        "NASA_raw_full_MAE":
            clean_raw_metrics_625e2[
                "mae"
            ],

        "NASA_raw_selective_MAE":
            clean_raw_selective_625e2[
                "mae"
            ],
    },

    "corrupted_primary_summary": {
        "mean_coverage":
            float(
                PRIMARY_SCENARIO_625E2[
                    "coverage"
                ].mean()
            ),

        "min_coverage":
            float(
                PRIMARY_SCENARIO_625E2[
                    "coverage"
                ].min()
            ),

        "max_coverage":
            float(
                PRIMARY_SCENARIO_625E2[
                    "coverage"
                ].max()
            ),

        "mean_full_MAE":
            float(
                PRIMARY_SCENARIO_625E2[
                    "full_MAE"
                ].mean()
            ),

        "mean_selective_MAE":
            float(
                PRIMARY_SCENARIO_625E2[
                    "selective_MAE"
                ].mean()
            ),

        "mean_abstained_MAE":
            float(
                PRIMARY_SCENARIO_625E2[
                    "abstained_MAE"
                ].mean()
            ),

        "mean_MAE_gain_abs":
            float(
                primary_gain_abs_625e2.mean()
            ),

        "mean_MAE_gain_rel":
            float(
                primary_gain_rel_625e2.mean()
            ),

        "scenarios_improved":
            primary_scenarios_improved_625e2,

        "scenarios_not_improved":
            primary_scenarios_not_improved_625e2,

        "improvement_fraction":
            float(
                primary_scenarios_improved_625e2
                /
                315
            ),
    },

    "integrity": {
        "endpoint_cycle_alignment_exact":
            bool(
                max_cycle_difference_625e2
                == 0
            ),

        "duplicate_scenario_engine_pairs":
            int(
                duplicate_pairs_625e2
            ),

        "clean_prediction_max_spread":
            float(
                max_clean_prediction_spread_625e2
            ),

        "clean_accept_mask_constant":
            True,

        "frozen_accept_mask_reproduced":
            frozen_mask_exact_625e2,

        "all_predictions_finite":
            all_engine_predictions_finite_625e2,

        "all_primary_metrics_finite":
            all_score_values_finite_625e2,
    },
}


with open(
    SUMMARY_PATH_625E2,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        json_safe_625e2(
            SUMMARY_625E2
        ),
        f,
        indent=2,
    )


# ============================================================
# 20. Generate SHA-256 manifest
# ============================================================

output_files_625e2 = [
    ENGINE_SCORE_PATH_625E2,
    SCENARIO_METRIC_PATH_625E2,
    SCENARIO_PRIMARY_PATH_625E2,
    SCENARIO_ALL_PATH_625E2,
    FAMILY_SCORE_PATH_625E2,
    SEVERITY_SCORE_PATH_625E2,
    CLEAN_SCORE_PATH_625E2,
    SUMMARY_PATH_625E2,
]


test_raw_path_625e2 = (
    DATA_DIR_625E2
    / "test_FD001.txt"
)

rul_raw_path_625e2 = (
    DATA_DIR_625E2
    / "RUL_FD001.txt"
)


MANIFEST_625E2 = {
    "step":
        "6.25E2",

    "created_from_frozen_prescore":
        True,

    "input_files": {
        str(
            PRESCORE_ENGINE_PATH_625E2
        ):
            sha256_file_625e2(
                PRESCORE_ENGINE_PATH_625E2
            ),

        str(
            test_raw_path_625e2
        ):
            (
                sha256_file_625e2(
                    test_raw_path_625e2
                )
                if
                test_raw_path_625e2.exists()
                else
                None
            ),

        str(
            rul_raw_path_625e2
        ):
            (
                sha256_file_625e2(
                    rul_raw_path_625e2
                )
                if
                rul_raw_path_625e2.exists()
                else
                None
            ),
    },

    "output_files": {
        str(path):
            sha256_file_625e2(
                path
            )
        for path
        in output_files_625e2
    },

    "decision_lock": {
        "frozen_threshold":
            FROZEN_THRESHOLD_625E2,

        "threshold_changed":
            False,

        "reliability_rule_changed":
            False,

        "corruption_manifests_changed":
            False,
    },

    "evaluation_safety": {
        "model_inference_rerun":
            False,

        "mc_dropout_rerun":
            False,

        "corruption_generation_rerun":
            False,

        "official_labels_used_for_tuning":
            False,

        "official_labels_used_for_evaluation":
            True,
    },
}


with open(
    MANIFEST_PATH_625E2,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        json_safe_625e2(
            MANIFEST_625E2
        ),
        f,
        indent=2,
    )


# ============================================================
# 21. Reload written files and verify
# ============================================================

RELOADED_PRIMARY_625E2 = pd.read_csv(
    SCENARIO_PRIMARY_PATH_625E2
)

RELOADED_FAMILY_625E2 = pd.read_csv(
    FAMILY_SCORE_PATH_625E2
)

RELOADED_ENGINE_625E2 = pd.read_csv(
    ENGINE_SCORE_PATH_625E2
)


stored_hashes_valid_625e2 = all(
    sha256_file_625e2(
        path
    )
    ==
    MANIFEST_625E2[
        "output_files"
    ][
        str(path)
    ]
    for path
    in output_files_625e2
)


if len(
    RELOADED_PRIMARY_625E2
) != 315:
    raise RuntimeError(
        "Reloaded primary scenario artifact "
        "does not contain 315 rows."
    )

if len(
    RELOADED_ENGINE_625E2
) != 31500:
    raise RuntimeError(
        "Reloaded engine-level score artifact "
        "does not contain 31,500 rows."
    )

if not stored_hashes_valid_625e2:
    raise RuntimeError(
        "At least one written Step 6.25E2 "
        "artifact failed SHA-256 validation."
    )


# ============================================================
# 22. Final Step 6.25E2 integrity summary
# ============================================================

elapsed_625e2 = (
    time.time()
    -
    STEP_START_625E2
)


print(
    "\n"
    + "=" * 65
)

print(
    "STEP 6.25E2 FINAL OFFICIAL SCORING SUMMARY"
)

print(
    "=" * 65
)

print(
    "Frozen scenarios scored          :",
    len(
        PRIMARY_SCENARIO_625E2
    ),
    "/ 315"
)

print(
    "Engine-scenario records scored   :",
    len(
        SCORED_ENGINE_625E2
    ),
    "/ 31500"
)

print(
    "Endpoint label alignment exact   :",
    max_cycle_difference_625e2
    == 0
)

print(
    "Clean prediction state preserved :",
    max_clean_prediction_spread_625e2
    <= 1e-10
)

print(
    "Frozen accept mask preserved     :",
    frozen_mask_exact_625e2
)

print(
    "All predictions finite           :",
    all_engine_predictions_finite_625e2
)

print(
    "All primary metrics finite       :",
    all_score_values_finite_625e2
)

print(
    "Frozen threshold                 :",
    FROZEN_THRESHOLD_625E2
)

print(
    "Frozen target                    :",
    "RUL_cap_125"
)

print(
    "Clean full MAE                   :",
    clean_cap_metrics_625e2[
        "mae"
    ]
)

print(
    "Clean selective MAE              :",
    clean_cap_selective_625e2[
        "mae"
    ]
)

print(
    "Mean corrupted full MAE          :",
    float(
        PRIMARY_SCENARIO_625E2[
            "full_MAE"
        ].mean()
    )
)

print(
    "Mean corrupted selective MAE     :",
    float(
        PRIMARY_SCENARIO_625E2[
            "selective_MAE"
        ].mean()
    )
)

print(
    "Mean selective MAE gain          :",
    float(
        primary_gain_rel_625e2.mean()
    )
)

print(
    "Scenarios with MAE improvement   :",
    primary_scenarios_improved_625e2,
    "/ 315"
)

print(
    "Official labels used for tuning  :",
    False
)

print(
    "Official labels used for scoring :",
    True
)

print(
    "Model inference rerun            :",
    False
)

print(
    "MC-dropout rerun                 :",
    False
)

print(
    "Corruptions regenerated          :",
    False
)

print(
    "Threshold changed                :",
    False
)

print(
    "Reliability rule changed         :",
    False
)

print(
    "Frozen manifests modified        :",
    False
)

print(
    "All stored hashes valid          :",
    stored_hashes_valid_625e2
)

print(
    "Files written                    :",
    True
)

print(
    "Elapsed seconds                  :",
    elapsed_625e2
)

print(
    "\nOutput directory:"
)

print(
    OUTPUT_DIR_625E2
)

print(
    "\nWritten files:"
)

for i_625e2, path_625e2 in enumerate(
    output_files_625e2
    + [
        MANIFEST_PATH_625E2
    ],
    start=1,
):
    print(
        f"{i_625e2}. {path_625e2}"
    )


print(
    "\n=== END STEP 6.25E2 ==="
)

In [ ]:
# ============================================================
# Step 6.25E3
# Post-scoring diagnostic audit of frozen 315-scenario
# official FD001 TEST evaluation
#
# IMPORTANT:
#   - NO model inference
#   - NO MC-dropout
#   - NO corruption generation
#   - NO threshold modification
#   - NO reliability-rule modification
#   - NO model selection
#   - NO TEST-based tuning
#
# This step ONLY analyses the already-frozen/scored
# Step 6.25E2 artifacts.
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
import hashlib
import time


print(
    "\n=== Step 6.25E3 — Post-scoring diagnostic audit "
    "of frozen official FD001 corruption results ===\n"
)

STEP_START_625E3 = time.time()


# ============================================================
# 1. Resolve paths
# ============================================================

if "PROJECT_ROOT" in globals():
    PROJECT_ROOT_625E3 = Path(PROJECT_ROOT)
else:
    PROJECT_ROOT_625E3 = Path(
        r"C:\RUL_Reliability_Research"
    )


SCORING_DIR_625E3 = (
    PROJECT_ROOT_625E3
    / "06_results"
    / "step6_official_fd001"
    / "corruption_transfer"
    / "full_scoring"
)

OUTPUT_DIR_625E3 = (
    SCORING_DIR_625E3
    / "diagnostics"
)

OUTPUT_DIR_625E3.mkdir(
    parents=True,
    exist_ok=True
)


PRIMARY_PATH_625E3 = (
    SCORING_DIR_625E3
    / "fd001_official_test_corruption_primary_scenario_summary.csv"
)

ENGINE_PATH_625E3 = (
    SCORING_DIR_625E3
    / "fd001_official_test_corruption_scored_engine_records.csv"
)

FAMILY_PATH_625E3 = (
    SCORING_DIR_625E3
    / "fd001_official_test_corruption_family_scoring_summary.csv"
)

SUMMARY_PATH_625E3 = (
    SCORING_DIR_625E3
    / "fd001_official_test_corruption_scoring_summary.json"
)

MANIFEST_PATH_625E3 = (
    SCORING_DIR_625E3
    / "fd001_official_test_corruption_scoring_manifest.json"
)


print("--- Paths ---")
print("Scoring directory   :", SCORING_DIR_625E3)
print("Diagnostic directory:", OUTPUT_DIR_625E3)


required_paths_625e3 = [
    PRIMARY_PATH_625E3,
    ENGINE_PATH_625E3,
    SUMMARY_PATH_625E3,
    MANIFEST_PATH_625E3,
]

missing_paths_625e3 = [
    str(p)
    for p in required_paths_625e3
    if not p.exists()
]

if missing_paths_625e3:
    raise RuntimeError(
        "Step 6.25E3 cannot continue. "
        "Missing Step 6.25E2 artifacts:\n"
        + "\n".join(missing_paths_625e3)
    )


# ============================================================
# 2. Utility helpers
# ============================================================

def sha256_file_625e3(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            block = f.read(1024 * 1024)

            if not block:
                break

            h.update(block)

    return h.hexdigest()


def boolean_series_625e3(series):
    if pd.api.types.is_bool_dtype(series):
        return series.astype(bool)

    if pd.api.types.is_numeric_dtype(series):
        converted = pd.to_numeric(
            series,
            errors="raise"
        )

        unique_values = set(
            converted.dropna().unique().tolist()
        )

        if unique_values.issubset(
            {0, 1, 0.0, 1.0}
        ):
            return converted.astype(int).astype(bool)

    text = (
        series.astype(str)
        .str.strip()
        .str.lower()
    )

    mapping = {
        "true": True,
        "false": False,
        "1": True,
        "0": False,
    }

    unknown = sorted(
        set(text.unique())
        - set(mapping.keys())
    )

    if unknown:
        raise RuntimeError(
            "Boolean conversion failed. "
            f"Unknown values: {unknown}"
        )

    return text.map(mapping).astype(bool)


def json_safe_625e3(value):

    if isinstance(value, dict):
        return {
            str(k): json_safe_625e3(v)
            for k, v in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [
            json_safe_625e3(v)
            for v in value
        ]

    if isinstance(value, np.integer):
        return int(value)

    if isinstance(value, np.floating):
        if not np.isfinite(value):
            return None
        return float(value)

    if isinstance(value, np.bool_):
        return bool(value)

    if isinstance(value, Path):
        return str(value)

    return value


# ============================================================
# 3. Reload frozen Step 6.25E2 artifacts
# ============================================================

PRIMARY_625E3 = pd.read_csv(
    PRIMARY_PATH_625E3
)

ENGINE_625E3 = pd.read_csv(
    ENGINE_PATH_625E3
)

with open(
    SUMMARY_PATH_625E3,
    "r",
    encoding="utf-8",
) as f:
    SUMMARY_625E3 = json.load(f)

with open(
    MANIFEST_PATH_625E3,
    "r",
    encoding="utf-8",
) as f:
    MANIFEST_625E3 = json.load(f)


print(
    "\n--- Reloaded Step 6.25E2 artifacts ---"
)

print(
    "Primary scenarios:",
    PRIMARY_625E3.shape
)

print(
    "Engine records   :",
    ENGINE_625E3.shape
)


# ============================================================
# 4. Verify stored artifact hashes
# ============================================================

stored_outputs_625e3 = (
    MANIFEST_625E3.get(
        "output_files",
        {}
    )
)

hash_check_rows_625e3 = []


for path_625e3 in [
    PRIMARY_PATH_625E3,
    ENGINE_PATH_625E3,
    FAMILY_PATH_625E3,
    SUMMARY_PATH_625E3,
]:

    if not path_625e3.exists():
        continue

    current_hash_625e3 = (
        sha256_file_625e3(
            path_625e3
        )
    )

    expected_hash_625e3 = None

    for stored_path, stored_hash in (
        stored_outputs_625e3.items()
    ):
        if (
            Path(stored_path).name
            ==
            path_625e3.name
        ):
            expected_hash_625e3 = stored_hash
            break

    exact_625e3 = (
        expected_hash_625e3 is not None
        and
        current_hash_625e3
        ==
        expected_hash_625e3
    )

    hash_check_rows_625e3.append(
        {
            "file":
                path_625e3.name,

            "hash_found_in_manifest":
                expected_hash_625e3
                is not None,

            "sha256_exact":
                exact_625e3,
        }
    )


HASH_CHECK_625E3 = pd.DataFrame(
    hash_check_rows_625e3
)

print(
    "\n--- Step 6.25E2 hash verification ---"
)

display(
    HASH_CHECK_625E3
)


if not HASH_CHECK_625E3[
    "sha256_exact"
].all():
    raise RuntimeError(
        "At least one Step 6.25E2 artifact "
        "does not match its frozen SHA-256."
    )


# ============================================================
# 5. Required scoring columns
# ============================================================

required_primary_cols_625e3 = [
    "global_index",
    "family",
    "coverage",
    "accepted",
    "abstained",
    "full_MAE",
    "selective_MAE",
    "abstained_MAE",
    "MAE_gain_abs",
    "MAE_gain_rel",
    "selective_improved",
]

missing_primary_cols_625e3 = [
    c
    for c in required_primary_cols_625e3
    if c not in PRIMARY_625E3.columns
]

if missing_primary_cols_625e3:
    raise RuntimeError(
        "Primary scenario artifact is missing "
        f"columns: {missing_primary_cols_625e3}"
    )


required_engine_cols_625e3 = [
    "global_index",
    "family",
    "unit_id",
    "cycle",
    "clean_prediction",
    "corrupted_prediction",
    "clean_accept",
    "corrupted_accept",
    "RUL_cap_125",
    "frozen_target_abs_error",
]

missing_engine_cols_625e3 = [
    c
    for c in required_engine_cols_625e3
    if c not in ENGINE_625E3.columns
]

if missing_engine_cols_625e3:
    raise RuntimeError(
        "Engine-level scoring artifact is missing "
        f"columns: {missing_engine_cols_625e3}"
    )


PRIMARY_625E3[
    "selective_improved"
] = boolean_series_625e3(
    PRIMARY_625E3[
        "selective_improved"
    ]
)

ENGINE_625E3[
    "clean_accept"
] = boolean_series_625e3(
    ENGINE_625E3[
        "clean_accept"
    ]
)

ENGINE_625E3[
    "corrupted_accept"
] = boolean_series_625e3(
    ENGINE_625E3[
        "corrupted_accept"
    ]
)


# ============================================================
# 6. Fundamental integrity checks
# ============================================================

if len(PRIMARY_625E3) != 315:
    raise RuntimeError(
        "Expected exactly 315 primary scenarios."
    )

if len(ENGINE_625E3) != 31500:
    raise RuntimeError(
        "Expected exactly 31,500 engine records."
    )

if PRIMARY_625E3[
    "global_index"
].nunique() != 315:
    raise RuntimeError(
        "Primary scenario global indices "
        "are not unique."
    )

scenario_engine_counts_625e3 = (
    ENGINE_625E3
    .groupby("global_index")[
        "unit_id"
    ]
    .nunique()
)

if not (
    scenario_engine_counts_625e3 == 100
).all():
    raise RuntimeError(
        "At least one scenario does not "
        "contain exactly 100 engines."
    )

duplicate_engine_rows_625e3 = int(
    ENGINE_625E3.duplicated(
        subset=[
            "global_index",
            "unit_id",
        ]
    ).sum()
)

if duplicate_engine_rows_625e3 != 0:
    raise RuntimeError(
        "Duplicate scenario-engine records found."
    )


# ============================================================
# 7. Recompute key scoring quantities
# ============================================================

PRIMARY_625E3[
    "recomputed_MAE_gain_abs"
] = (
    PRIMARY_625E3["full_MAE"]
    -
    PRIMARY_625E3["selective_MAE"]
)

PRIMARY_625E3[
    "gain_abs_error"
] = np.abs(
    PRIMARY_625E3[
        "recomputed_MAE_gain_abs"
    ]
    -
    PRIMARY_625E3[
        "MAE_gain_abs"
    ]
)

max_gain_reproduction_error_625e3 = float(
    PRIMARY_625E3[
        "gain_abs_error"
    ].max()
)

if max_gain_reproduction_error_625e3 > 1e-10:
    raise RuntimeError(
        "Stored selective MAE gains do not "
        "reproduce exactly."
    )


PRIMARY_625E3[
    "recomputed_improved"
] = (
    PRIMARY_625E3[
        "recomputed_MAE_gain_abs"
    ]
    > 0
)

improvement_mask_exact_625e3 = bool(
    np.array_equal(
        PRIMARY_625E3[
            "selective_improved"
        ].to_numpy(dtype=bool),
        PRIMARY_625E3[
            "recomputed_improved"
        ].to_numpy(dtype=bool),
    )
)

if not improvement_mask_exact_625e3:
    raise RuntimeError(
        "Stored improvement mask is inconsistent "
        "with the stored MAE values."
    )


coverage_recomputed_625e3 = (
    PRIMARY_625E3["accepted"]
    /
    (
        PRIMARY_625E3["accepted"]
        +
        PRIMARY_625E3["abstained"]
    )
)

max_coverage_error_625e3 = float(
    np.abs(
        coverage_recomputed_625e3
        -
        PRIMARY_625E3["coverage"]
    ).max()
)

if max_coverage_error_625e3 > 1e-12:
    raise RuntimeError(
        "Stored coverage values do not reproduce."
    )


# ============================================================
# 8. Recover CLEAN endpoint baseline independently
# ============================================================

clean_endpoint_625e3 = (
    ENGINE_625E3[
        [
            "unit_id",
            "cycle",
            "clean_prediction",
            "clean_accept",
            "RUL_cap_125",
        ]
    ]
    .drop_duplicates(
        subset=["unit_id"]
    )
    .sort_values("unit_id")
    .reset_index(drop=True)
)

if len(clean_endpoint_625e3) != 100:
    raise RuntimeError(
        "Could not reconstruct the 100 clean "
        "endpoint records."
    )


clean_error_625e3 = (
    clean_endpoint_625e3[
        "clean_prediction"
    ].to_numpy(dtype=np.float64)
    -
    clean_endpoint_625e3[
        "RUL_cap_125"
    ].to_numpy(dtype=np.float64)
)

clean_full_mae_625e3 = float(
    np.mean(
        np.abs(
            clean_error_625e3
        )
    )
)

clean_accept_mask_625e3 = (
    clean_endpoint_625e3[
        "clean_accept"
    ].to_numpy(dtype=bool)
)

clean_selective_mae_625e3 = float(
    np.mean(
        np.abs(
            clean_error_625e3[
                clean_accept_mask_625e3
            ]
        )
    )
)

clean_coverage_625e3 = float(
    clean_accept_mask_625e3.mean()
)


print(
    "\n--- Independently reconstructed CLEAN baseline ---"
)

print(
    "Clean coverage     :",
    clean_coverage_625e3
)

print(
    "Clean full MAE     :",
    clean_full_mae_625e3
)

print(
    "Clean selective MAE:",
    clean_selective_mae_625e3
)


# ============================================================
# 9. Add diagnostic scenario quantities
# ============================================================

DIAGNOSTIC_SCENARIOS_625E3 = (
    PRIMARY_625E3.copy()
)


# Positive means abstained engines had larger error than
# accepted engines — desirable separation.
DIAGNOSTIC_SCENARIOS_625E3[
    "abstained_minus_accepted_MAE"
] = (
    DIAGNOSTIC_SCENARIOS_625E3[
        "abstained_MAE"
    ]
    -
    DIAGNOSTIC_SCENARIOS_625E3[
        "selective_MAE"
    ]
)


# Positive means the corruption produced higher full MAE
# than the clean model's endpoint MAE.
DIAGNOSTIC_SCENARIOS_625E3[
    "full_MAE_delta_vs_clean"
] = (
    DIAGNOSTIC_SCENARIOS_625E3[
        "full_MAE"
    ]
    -
    clean_full_mae_625e3
)


DIAGNOSTIC_SCENARIOS_625E3[
    "selective_MAE_delta_vs_clean_selective"
] = (
    DIAGNOSTIC_SCENARIOS_625E3[
        "selective_MAE"
    ]
    -
    clean_selective_mae_625e3
)


# ============================================================
# 10. Overall descriptive diagnostics
# ============================================================

gain_abs_625e3 = (
    DIAGNOSTIC_SCENARIOS_625E3[
        "MAE_gain_abs"
    ].to_numpy(dtype=np.float64)
)

gain_rel_625e3 = (
    DIAGNOSTIC_SCENARIOS_625E3[
        "MAE_gain_rel"
    ].to_numpy(dtype=np.float64)
)

coverage_625e3 = (
    DIAGNOSTIC_SCENARIOS_625E3[
        "coverage"
    ].to_numpy(dtype=np.float64)
)

separation_625e3 = (
    DIAGNOSTIC_SCENARIOS_625E3[
        "abstained_minus_accepted_MAE"
    ].to_numpy(dtype=np.float64)
)


mean_full_mae_625e3 = float(
    DIAGNOSTIC_SCENARIOS_625E3[
        "full_MAE"
    ].mean()
)

mean_selective_mae_625e3 = float(
    DIAGNOSTIC_SCENARIOS_625E3[
        "selective_MAE"
    ].mean()
)

aggregate_mae_reduction_625e3 = (
    mean_full_mae_625e3
    -
    mean_selective_mae_625e3
)

aggregate_relative_reduction_625e3 = (
    aggregate_mae_reduction_625e3
    /
    mean_full_mae_625e3
)


overall_quantiles_625e3 = {
    "gain_relative_q05":
        float(np.quantile(gain_rel_625e3, 0.05)),

    "gain_relative_q25":
        float(np.quantile(gain_rel_625e3, 0.25)),

    "gain_relative_q50":
        float(np.quantile(gain_rel_625e3, 0.50)),

    "gain_relative_q75":
        float(np.quantile(gain_rel_625e3, 0.75)),

    "gain_relative_q95":
        float(np.quantile(gain_rel_625e3, 0.95)),

    "coverage_q05":
        float(np.quantile(coverage_625e3, 0.05)),

    "coverage_q50":
        float(np.quantile(coverage_625e3, 0.50)),

    "coverage_q95":
        float(np.quantile(coverage_625e3, 0.95)),
}


# ============================================================
# 11. Identify ALL non-improved scenarios
# ============================================================

NON_IMPROVED_625E3 = (
    DIAGNOSTIC_SCENARIOS_625E3.loc[
        ~DIAGNOSTIC_SCENARIOS_625E3[
            "selective_improved"
        ]
    ]
    .copy()
    .sort_values(
        "MAE_gain_abs"
    )
    .reset_index(drop=True)
)


print(
    "\n--- Non-improved scenarios ---"
)

print(
    "Count:",
    len(NON_IMPROVED_625E3)
)

if len(NON_IMPROVED_625E3) > 0:

    display_cols_625e3 = [
        "global_index",
        "family",
    ]

    for optional_col_625e3 in [
        "family_position",
        "scenario_id",
        "manifest_scenario_id",
        "sensor",
        "severity",
    ]:
        if (
            optional_col_625e3
            in NON_IMPROVED_625E3.columns
        ):
            display_cols_625e3.append(
                optional_col_625e3
            )

    display_cols_625e3 += [
        "accepted",
        "abstained",
        "coverage",
        "full_MAE",
        "selective_MAE",
        "abstained_MAE",
        "MAE_gain_abs",
        "MAE_gain_rel",
        "abstained_minus_accepted_MAE",
    ]

    display(
        NON_IMPROVED_625E3[
            display_cols_625e3
        ]
    )

else:
    print(
        "Every frozen corruption scenario improved."
    )


# ============================================================
# 12. Flexible grouped diagnostic summary
# ============================================================

def grouped_diagnostic_summary_625e3(
    df,
    group_cols,
):

    rows = []

    for keys, group in df.groupby(
        group_cols,
        dropna=False,
        sort=True,
    ):

        if not isinstance(keys, tuple):
            keys = (keys,)

        row = {
            col: value
            for col, value
            in zip(group_cols, keys)
        }

        row.update(
            {
                "scenarios":
                    int(len(group)),

                "mean_coverage":
                    float(
                        group[
                            "coverage"
                        ].mean()
                    ),

                "min_coverage":
                    float(
                        group[
                            "coverage"
                        ].min()
                    ),

                "max_coverage":
                    float(
                        group[
                            "coverage"
                        ].max()
                    ),

                "mean_full_MAE":
                    float(
                        group[
                            "full_MAE"
                        ].mean()
                    ),

                "mean_selective_MAE":
                    float(
                        group[
                            "selective_MAE"
                        ].mean()
                    ),

                "mean_abstained_MAE":
                    float(
                        group[
                            "abstained_MAE"
                        ].mean()
                    ),

                "mean_MAE_gain_abs":
                    float(
                        group[
                            "MAE_gain_abs"
                        ].mean()
                    ),

                "median_MAE_gain_abs":
                    float(
                        group[
                            "MAE_gain_abs"
                        ].median()
                    ),

                "mean_MAE_gain_rel":
                    float(
                        group[
                            "MAE_gain_rel"
                        ].mean()
                    ),

                "median_MAE_gain_rel":
                    float(
                        group[
                            "MAE_gain_rel"
                        ].median()
                    ),

                "minimum_MAE_gain_rel":
                    float(
                        group[
                            "MAE_gain_rel"
                        ].min()
                    ),

                "maximum_MAE_gain_rel":
                    float(
                        group[
                            "MAE_gain_rel"
                        ].max()
                    ),

                "improved_scenarios":
                    int(
                        group[
                            "selective_improved"
                        ].sum()
                    ),

                "improvement_fraction":
                    float(
                        group[
                            "selective_improved"
                        ].mean()
                    ),

                "mean_abstained_minus_accepted_MAE":
                    float(
                        group[
                            "abstained_minus_accepted_MAE"
                        ].mean()
                    ),

                "positive_error_separation_fraction":
                    float(
                        (
                            group[
                                "abstained_minus_accepted_MAE"
                            ]
                            > 0
                        ).mean()
                    ),

                "mean_full_MAE_delta_vs_clean":
                    float(
                        group[
                            "full_MAE_delta_vs_clean"
                        ].mean()
                    ),
            }
        )

        rows.append(row)

    return pd.DataFrame(rows)


BY_FAMILY_625E3 = (
    grouped_diagnostic_summary_625e3(
        DIAGNOSTIC_SCENARIOS_625E3,
        ["family"],
    )
)


if "severity" in (
    DIAGNOSTIC_SCENARIOS_625E3.columns
):
    BY_SEVERITY_625E3 = (
        grouped_diagnostic_summary_625e3(
            DIAGNOSTIC_SCENARIOS_625E3,
            ["severity"],
        )
    )

    BY_FAMILY_SEVERITY_625E3 = (
        grouped_diagnostic_summary_625e3(
            DIAGNOSTIC_SCENARIOS_625E3,
            [
                "family",
                "severity",
            ],
        )
    )
else:
    BY_SEVERITY_625E3 = pd.DataFrame()
    BY_FAMILY_SEVERITY_625E3 = (
        pd.DataFrame()
    )


if "sensor" in (
    DIAGNOSTIC_SCENARIOS_625E3.columns
):
    BY_SENSOR_625E3 = (
        grouped_diagnostic_summary_625e3(
            DIAGNOSTIC_SCENARIOS_625E3,
            ["sensor"],
        )
    )
else:
    BY_SENSOR_625E3 = pd.DataFrame()


# ============================================================
# 13. Inspect the non-improved scenario at engine level
# ============================================================

if len(NON_IMPROVED_625E3) > 0:

    failed_indices_625e3 = (
        NON_IMPROVED_625E3[
            "global_index"
        ]
        .astype(int)
        .tolist()
    )

    FAILED_ENGINE_RECORDS_625E3 = (
        ENGINE_625E3.loc[
            ENGINE_625E3[
                "global_index"
            ].isin(
                failed_indices_625e3
            )
        ]
        .copy()
    )

    FAILED_ENGINE_RECORDS_625E3[
        "prediction_error"
    ] = (
        FAILED_ENGINE_RECORDS_625E3[
            "corrupted_prediction"
        ]
        -
        FAILED_ENGINE_RECORDS_625E3[
            "RUL_cap_125"
        ]
    )

    FAILED_ENGINE_RECORDS_625E3[
        "prediction_abs_error"
    ] = np.abs(
        FAILED_ENGINE_RECORDS_625E3[
            "prediction_error"
        ]
    )

    FAILED_ENGINE_RECORDS_625E3 = (
        FAILED_ENGINE_RECORDS_625E3
        .sort_values(
            [
                "global_index",
                "prediction_abs_error",
            ],
            ascending=[
                True,
                False,
            ],
        )
        .reset_index(drop=True)
    )


    print(
        "\n--- Largest engine-level errors "
        "inside non-improved scenario(s) ---"
    )

    failed_display_cols_625e3 = [
        "global_index",
        "family",
    ]

    for optional_col_625e3 in [
        "scenario_id",
        "sensor",
        "severity",
    ]:
        if (
            optional_col_625e3
            in FAILED_ENGINE_RECORDS_625E3.columns
        ):
            failed_display_cols_625e3.append(
                optional_col_625e3
            )

    failed_display_cols_625e3 += [
        "unit_id",
        "cycle",
        "RUL_cap_125",
        "clean_prediction",
        "corrupted_prediction",
        "prediction_abs_error",
        "clean_accept",
        "corrupted_accept",
    ]

    if (
        "clean_reliability"
        in FAILED_ENGINE_RECORDS_625E3.columns
    ):
        failed_display_cols_625e3.append(
            "clean_reliability"
        )

    if (
        "corrupted_reliability"
        in FAILED_ENGINE_RECORDS_625E3.columns
    ):
        failed_display_cols_625e3.append(
            "corrupted_reliability"
        )


    display(
        FAILED_ENGINE_RECORDS_625E3[
            failed_display_cols_625e3
        ].head(20)
    )

else:

    FAILED_ENGINE_RECORDS_625E3 = (
        pd.DataFrame()
    )


# ============================================================
# 14. Display family diagnostics
# ============================================================

print(
    "\n--- Family diagnostic summary ---"
)

display(
    BY_FAMILY_625E3
)


if not BY_FAMILY_SEVERITY_625E3.empty:

    print(
        "\n--- Family × severity diagnostic summary ---"
    )

    display(
        BY_FAMILY_SEVERITY_625E3
    )


# ============================================================
# 15. Overall scientific diagnostics
# ============================================================

scenarios_improved_625e3 = int(
    DIAGNOSTIC_SCENARIOS_625E3[
        "selective_improved"
    ].sum()
)

positive_separation_count_625e3 = int(
    (
        DIAGNOSTIC_SCENARIOS_625E3[
            "abstained_minus_accepted_MAE"
        ]
        > 0
    ).sum()
)

corruption_worsened_full_mae_count_625e3 = int(
    (
        DIAGNOSTIC_SCENARIOS_625E3[
            "full_MAE_delta_vs_clean"
        ]
        > 0
    ).sum()
)


OVERALL_DIAGNOSTICS_625E3 = {
    "step":
        "6.25E3",

    "dataset":
        "NASA C-MAPSS FD001",

    "primary_target":
        "RUL_cap_125",

    "scenario_count":
        int(
            len(
                DIAGNOSTIC_SCENARIOS_625E3
            )
        ),

    "engine_scenario_records":
        int(
            len(
                ENGINE_625E3
            )
        ),

    "clean_full_MAE":
        clean_full_mae_625e3,

    "clean_selective_MAE":
        clean_selective_mae_625e3,

    "clean_coverage":
        clean_coverage_625e3,

    "mean_corrupted_full_MAE":
        mean_full_mae_625e3,

    "mean_corrupted_selective_MAE":
        mean_selective_mae_625e3,

    "aggregate_MAE_reduction_cycles":
        aggregate_mae_reduction_625e3,

    "aggregate_relative_MAE_reduction":
        aggregate_relative_reduction_625e3,

    "mean_per_scenario_relative_MAE_gain":
        float(
            np.mean(
                gain_rel_625e3
            )
        ),

    "median_per_scenario_relative_MAE_gain":
        float(
            np.median(
                gain_rel_625e3
            )
        ),

    "scenarios_improved":
        scenarios_improved_625e3,

    "scenarios_not_improved":
        int(
            len(
                DIAGNOSTIC_SCENARIOS_625E3
            )
            -
            scenarios_improved_625e3
        ),

    "improvement_fraction":
        float(
            scenarios_improved_625e3
            /
            len(
                DIAGNOSTIC_SCENARIOS_625E3
            )
        ),

    "positive_abstained_vs_accepted_error_separation":
        positive_separation_count_625e3,

    "positive_error_separation_fraction":
        float(
            positive_separation_count_625e3
            /
            len(
                DIAGNOSTIC_SCENARIOS_625E3
            )
        ),

    "scenarios_with_full_MAE_worse_than_clean":
        corruption_worsened_full_mae_count_625e3,

    "fraction_with_full_MAE_worse_than_clean":
        float(
            corruption_worsened_full_mae_count_625e3
            /
            len(
                DIAGNOSTIC_SCENARIOS_625E3
            )
        ),

    "minimum_coverage":
        float(
            np.min(
                coverage_625e3
            )
        ),

    "maximum_coverage":
        float(
            np.max(
                coverage_625e3
            )
        ),

    "mean_coverage":
        float(
            np.mean(
                coverage_625e3
            )
        ),

    "minimum_relative_MAE_gain":
        float(
            np.min(
                gain_rel_625e3
            )
        ),

    "maximum_relative_MAE_gain":
        float(
            np.max(
                gain_rel_625e3
            )
        ),

    "quantiles":
        overall_quantiles_625e3,

    "integrity": {
        "input_hashes_valid":
            True,

        "scenario_rows_exact":
            bool(
                len(
                    PRIMARY_625E3
                )
                == 315
            ),

        "engine_rows_exact":
            bool(
                len(
                    ENGINE_625E3
                )
                == 31500
            ),

        "gain_reproduction_max_error":
            max_gain_reproduction_error_625e3,

        "coverage_reproduction_max_error":
            max_coverage_error_625e3,

        "improvement_mask_exact":
            improvement_mask_exact_625e3,

        "duplicate_engine_records":
            duplicate_engine_rows_625e3,
    },

    "research_safety": {
        "model_inference_executed":
            False,

        "mc_dropout_executed":
            False,

        "corruptions_generated":
            False,

        "threshold_modified":
            False,

        "reliability_rule_modified":
            False,

        "frozen_manifests_modified":
            False,

        "official_labels_used_for_tuning":
            False,

        "official_labels_used_for_diagnostic_evaluation":
            True,
    },
}


# ============================================================
# 16. Write diagnostic artifacts
# ============================================================

SCENARIO_DIAGNOSTIC_PATH_625E3 = (
    OUTPUT_DIR_625E3
    / "fd001_scenario_diagnostic_table.csv"
)

NON_IMPROVED_PATH_625E3 = (
    OUTPUT_DIR_625E3
    / "fd001_non_improved_scenarios.csv"
)

FAMILY_DIAGNOSTIC_PATH_625E3 = (
    OUTPUT_DIR_625E3
    / "fd001_family_diagnostic_summary.csv"
)

SEVERITY_DIAGNOSTIC_PATH_625E3 = (
    OUTPUT_DIR_625E3
    / "fd001_severity_diagnostic_summary.csv"
)

FAMILY_SEVERITY_PATH_625E3 = (
    OUTPUT_DIR_625E3
    / "fd001_family_severity_diagnostic_summary.csv"
)

SENSOR_DIAGNOSTIC_PATH_625E3 = (
    OUTPUT_DIR_625E3
    / "fd001_sensor_diagnostic_summary.csv"
)

FAILED_ENGINE_PATH_625E3 = (
    OUTPUT_DIR_625E3
    / "fd001_non_improved_engine_records.csv"
)

OVERALL_PATH_625E3 = (
    OUTPUT_DIR_625E3
    / "fd001_scoring_diagnostic_summary.json"
)

DIAGNOSTIC_MANIFEST_PATH_625E3 = (
    OUTPUT_DIR_625E3
    / "fd001_scoring_diagnostic_manifest.json"
)


DIAGNOSTIC_SCENARIOS_625E3.to_csv(
    SCENARIO_DIAGNOSTIC_PATH_625E3,
    index=False,
    float_format="%.17g",
)

NON_IMPROVED_625E3.to_csv(
    NON_IMPROVED_PATH_625E3,
    index=False,
    float_format="%.17g",
)

BY_FAMILY_625E3.to_csv(
    FAMILY_DIAGNOSTIC_PATH_625E3,
    index=False,
    float_format="%.17g",
)

if not BY_SEVERITY_625E3.empty:
    BY_SEVERITY_625E3.to_csv(
        SEVERITY_DIAGNOSTIC_PATH_625E3,
        index=False,
        float_format="%.17g",
    )

if not BY_FAMILY_SEVERITY_625E3.empty:
    BY_FAMILY_SEVERITY_625E3.to_csv(
        FAMILY_SEVERITY_PATH_625E3,
        index=False,
        float_format="%.17g",
    )

if not BY_SENSOR_625E3.empty:
    BY_SENSOR_625E3.to_csv(
        SENSOR_DIAGNOSTIC_PATH_625E3,
        index=False,
        float_format="%.17g",
    )

FAILED_ENGINE_RECORDS_625E3.to_csv(
    FAILED_ENGINE_PATH_625E3,
    index=False,
    float_format="%.17g",
)

with open(
    OVERALL_PATH_625E3,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        json_safe_625e3(
            OVERALL_DIAGNOSTICS_625E3
        ),
        f,
        indent=2,
    )


# ============================================================
# 17. Build diagnostic manifest
# ============================================================

written_paths_625e3 = [
    SCENARIO_DIAGNOSTIC_PATH_625E3,
    NON_IMPROVED_PATH_625E3,
    FAMILY_DIAGNOSTIC_PATH_625E3,
    FAILED_ENGINE_PATH_625E3,
    OVERALL_PATH_625E3,
]

if SEVERITY_DIAGNOSTIC_PATH_625E3.exists():
    written_paths_625e3.append(
        SEVERITY_DIAGNOSTIC_PATH_625E3
    )

if FAMILY_SEVERITY_PATH_625E3.exists():
    written_paths_625e3.append(
        FAMILY_SEVERITY_PATH_625E3
    )

if SENSOR_DIAGNOSTIC_PATH_625E3.exists():
    written_paths_625e3.append(
        SENSOR_DIAGNOSTIC_PATH_625E3
    )


DIAGNOSTIC_MANIFEST_625E3 = {
    "step":
        "6.25E3",

    "source_step":
        "6.25E2",

    "input_files": {
        str(PRIMARY_PATH_625E3):
            sha256_file_625e3(
                PRIMARY_PATH_625E3
            ),

        str(ENGINE_PATH_625E3):
            sha256_file_625e3(
                ENGINE_PATH_625E3
            ),

        str(SUMMARY_PATH_625E3):
            sha256_file_625e3(
                SUMMARY_PATH_625E3
            ),

        str(MANIFEST_PATH_625E3):
            sha256_file_625e3(
                MANIFEST_PATH_625E3
            ),
    },

    "output_files": {
        str(path):
            sha256_file_625e3(path)
        for path in written_paths_625e3
    },

    "frozen_decisions_modified":
        False,

    "test_based_tuning_performed":
        False,
}


with open(
    DIAGNOSTIC_MANIFEST_PATH_625E3,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        json_safe_625e3(
            DIAGNOSTIC_MANIFEST_625E3
        ),
        f,
        indent=2,
    )


# ============================================================
# 18. Verify newly written artifacts
# ============================================================

new_hashes_valid_625e3 = all(
    sha256_file_625e3(path)
    ==
    DIAGNOSTIC_MANIFEST_625E3[
        "output_files"
    ][str(path)]
    for path in written_paths_625e3
)

if not new_hashes_valid_625e3:
    raise RuntimeError(
        "A Step 6.25E3 diagnostic artifact "
        "failed SHA-256 verification."
    )


# ============================================================
# 19. Final summary
# ============================================================

elapsed_625e3 = (
    time.time()
    -
    STEP_START_625E3
)


print(
    "\n"
    + "=" * 68
)

print(
    "STEP 6.25E3 FINAL DIAGNOSTIC SUMMARY"
)

print(
    "=" * 68
)

print(
    "Scenarios audited                    :",
    len(
        DIAGNOSTIC_SCENARIOS_625E3
    ),
    "/ 315"
)

print(
    "Engine-scenario records audited      :",
    len(
        ENGINE_625E3
    ),
    "/ 31500"
)

print(
    "Clean full MAE                       :",
    clean_full_mae_625e3
)

print(
    "Clean selective MAE                  :",
    clean_selective_mae_625e3
)

print(
    "Mean corrupted full MAE              :",
    mean_full_mae_625e3
)

print(
    "Mean corrupted selective MAE         :",
    mean_selective_mae_625e3
)

print(
    "Aggregate MAE reduction (cycles)     :",
    aggregate_mae_reduction_625e3
)

print(
    "Aggregate relative MAE reduction     :",
    aggregate_relative_reduction_625e3
)

print(
    "Mean per-scenario relative MAE gain  :",
    float(
        np.mean(
            gain_rel_625e3
        )
    )
)

print(
    "Median relative MAE gain             :",
    float(
        np.median(
            gain_rel_625e3
        )
    )
)

print(
    "Scenarios improved                   :",
    scenarios_improved_625e3,
    "/ 315"
)

print(
    "Scenarios not improved               :",
    len(
        NON_IMPROVED_625E3
    ),
    "/ 315"
)

print(
    "Positive abstained-error separation  :",
    positive_separation_count_625e3,
    "/ 315"
)

print(
    "Full MAE worse than clean baseline   :",
    corruption_worsened_full_mae_count_625e3,
    "/ 315"
)

print(
    "Coverage range                      :",
    (
        float(
            np.min(
                coverage_625e3
            )
        ),
        float(
            np.max(
                coverage_625e3
            )
        ),
    )
)

print(
    "Minimum relative MAE gain            :",
    float(
        np.min(
            gain_rel_625e3
        )
    )
)

print(
    "Maximum relative MAE gain            :",
    float(
        np.max(
            gain_rel_625e3
        )
    )
)

print(
    "Stored E2 input hashes valid         :",
    True
)

print(
    "Stored gain values reproduced        :",
    max_gain_reproduction_error_625e3
    <= 1e-10
)

print(
    "Stored coverage values reproduced    :",
    max_coverage_error_625e3
    <= 1e-12
)

print(
    "Improvement mask reproduced          :",
    improvement_mask_exact_625e3
)

print(
    "Model inference executed             :",
    False
)

print(
    "MC-dropout executed                  :",
    False
)

print(
    "Corruptions generated                :",
    False
)

print(
    "Threshold modified                   :",
    False
)

print(
    "Reliability rule modified            :",
    False
)

print(
    "Official TEST used for tuning        :",
    False
)

print(
    "Diagnostic hashes valid              :",
    new_hashes_valid_625e3
)

print(
    "Files written                        :",
    True
)

print(
    "Elapsed seconds                      :",
    elapsed_625e3
)

print(
    "\nDiagnostic output directory:"
)

print(
    OUTPUT_DIR_625E3
)

print(
    "\n=== END STEP 6.25E3 ==="
)

In [ ]:
# ============================================================
# Step 6.25F
# Frozen risk-coverage analysis of official FD001 corruption
# results.
#
# IMPORTANT:
# - Reads frozen Step 6.25E2 artifacts only.
# - Does NOT regenerate corruption.
# - Does NOT rerun GRU inference.
# - Does NOT rerun MC-dropout.
# - Does NOT modify the frozen threshold/rule.
# - Official TEST labels are used ONLY for evaluation.
# - The previously frozen operating point remains unchanged.
# ============================================================

from pathlib import Path
from datetime import datetime
import hashlib
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display


print(
    "\n"
    "============================================================\n"
    "STEP 6.25F — FROZEN FD001 RISK–COVERAGE ANALYSIS\n"
    "============================================================"
)


# ============================================================
# 1. PATHS
# ============================================================

PROJECT_ROOT_625F = Path(
    r"C:\RUL_Reliability_Research"
)

SCORING_DIR_625F = (
    PROJECT_ROOT_625F
    / "06_results"
    / "step6_official_fd001"
    / "corruption_transfer"
    / "full_scoring"
)

OUTPUT_DIR_625F = (
    SCORING_DIR_625F
    / "risk_coverage"
)

OUTPUT_DIR_625F.mkdir(
    parents=True,
    exist_ok=True
)


print("\n--- Paths ---")
print(
    "Project root :",
    PROJECT_ROOT_625F
)
print(
    "Scoring dir  :",
    SCORING_DIR_625F
)
print(
    "Output dir   :",
    OUTPUT_DIR_625F
)


if not SCORING_DIR_625F.exists():
    raise FileNotFoundError(
        "Step 6.25E2 scoring directory does not exist:\n"
        + str(SCORING_DIR_625F)
    )


# ============================================================
# 2. HELPERS
# ============================================================

def sha256_file_625f(path):
    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:
        for block in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(block)

    return h.hexdigest()


def find_one_file_625f(
    directory,
    pattern,
    description
):
    matches = sorted(
        [
            p
            for p in directory.glob(pattern)
            if p.is_file()
        ]
    )

    if len(matches) != 1:
        raise RuntimeError(
            f"Expected exactly one {description} "
            f"matching '{pattern}' in:\n"
            f"{directory}\n"
            f"Found {len(matches)}:\n"
            + "\n".join(
                str(p)
                for p in matches
            )
        )

    return matches[0]


def normalize_bool_625f(series):
    """
    Convert a bool-like Series safely to Python/pandas boolean.
    """

    if pd.api.types.is_bool_dtype(series):
        return series.astype(bool)

    if pd.api.types.is_numeric_dtype(series):
        unique_values = set(
            pd.Series(series)
            .dropna()
            .astype(float)
            .unique()
            .tolist()
        )

        if unique_values.issubset(
            {0.0, 1.0}
        ):
            return (
                series
                .astype(int)
                .astype(bool)
            )

    mapped = (
        series
        .astype(str)
        .str.strip()
        .str.lower()
        .map(
            {
                "true": True,
                "false": False,
                "1": True,
                "0": False,
            }
        )
    )

    if mapped.isna().any():
        bad_values = (
            series[
                mapped.isna()
            ]
            .astype(str)
            .unique()
            .tolist()
        )

        raise RuntimeError(
            "Could not safely parse boolean column. "
            f"Unrecognized values: {bad_values[:10]}"
        )

    return mapped.astype(bool)


def json_default_625f(obj):

    if isinstance(
        obj,
        np.integer
    ):
        return int(obj)

    if isinstance(
        obj,
        np.floating
    ):
        if np.isnan(obj):
            return None
        return float(obj)

    if isinstance(
        obj,
        np.bool_
    ):
        return bool(obj)

    if isinstance(
        obj,
        Path
    ):
        return str(obj)

    raise TypeError(
        f"Object of type "
        f"{type(obj).__name__} "
        "is not JSON serializable."
    )


# ============================================================
# 3. DISCOVER FROZEN STEP 6.25E2 ARTIFACTS
# ============================================================

ENGINE_FILE_625F = (
    find_one_file_625f(
        SCORING_DIR_625F,
        "*scored_engine*.csv",
        "scored-engine artifact"
    )
)

SCENARIO_FILE_625F = (
    find_one_file_625f(
        SCORING_DIR_625F,
        "*primary_scenario*.csv",
        "primary scenario-score artifact"
    )
)


print(
    "\n--- Frozen Step 6.25E2 artifact discovery ---"
)
print(
    "Engine artifact  :",
    ENGINE_FILE_625F.name
)
print(
    "Scenario artifact:",
    SCENARIO_FILE_625F.name
)


ENGINE_HASH_BEFORE_625F = (
    sha256_file_625f(
        ENGINE_FILE_625F
    )
)

SCENARIO_HASH_BEFORE_625F = (
    sha256_file_625f(
        SCENARIO_FILE_625F
    )
)


# ============================================================
# 4. LOAD FROZEN ARTIFACTS
# ============================================================

engine_df_625f = pd.read_csv(
    ENGINE_FILE_625F
)

scenario_score_df_625f = pd.read_csv(
    SCENARIO_FILE_625F
)


print(
    "\n--- Loaded frozen artifacts ---"
)
print(
    "Engine records shape :",
    engine_df_625f.shape
)
print(
    "Scenario scores shape:",
    scenario_score_df_625f.shape
)


# ============================================================
# 5. REQUIRED COLUMN CONTRACT
# ============================================================

REQUIRED_ENGINE_COLUMNS_625F = [
    "global_index",
    "family",
    "scenario_id",
    "sensor",
    "severity",
    "unit_id",
    "cycle",
    "RUL_cap_125",
    "clean_prediction",
    "corrupted_prediction",
    "clean_reliability",
    "corrupted_reliability",
    "clean_accept",
    "corrupted_accept",
]


missing_engine_columns_625f = [
    col
    for col in REQUIRED_ENGINE_COLUMNS_625F
    if col not in engine_df_625f.columns
]


if missing_engine_columns_625f:
    raise RuntimeError(
        "Step 6.25F cannot continue. "
        "Required frozen engine columns are missing:\n"
        + str(
            missing_engine_columns_625f
        )
    )


engine_df_625f[
    "clean_accept"
] = normalize_bool_625f(
    engine_df_625f[
        "clean_accept"
    ]
)

engine_df_625f[
    "corrupted_accept"
] = normalize_bool_625f(
    engine_df_625f[
        "corrupted_accept"
    ]
)


NUMERIC_COLUMNS_625F = [
    "global_index",
    "unit_id",
    "cycle",
    "RUL_cap_125",
    "clean_prediction",
    "corrupted_prediction",
    "clean_reliability",
    "corrupted_reliability",
]


for col_625f in NUMERIC_COLUMNS_625F:
    engine_df_625f[
        col_625f
    ] = pd.to_numeric(
        engine_df_625f[
            col_625f
        ],
        errors="raise"
    )


if not np.isfinite(
    engine_df_625f[
        NUMERIC_COLUMNS_625F
    ].to_numpy(
        dtype=np.float64
    )
).all():

    raise RuntimeError(
        "Non-finite numeric values found "
        "in frozen Step 6.25E2 engine artifact."
    )


# ============================================================
# 6. STRUCTURAL INTEGRITY
# ============================================================

scenario_count_625f = int(
    engine_df_625f[
        "global_index"
    ].nunique()
)

engine_record_count_625f = int(
    len(
        engine_df_625f
    )
)

group_sizes_625f = (
    engine_df_625f
    .groupby(
        "global_index"
    )
    .size()
)

unique_unit_counts_625f = (
    engine_df_625f
    .groupby(
        "global_index"
    )[
        "unit_id"
    ]
    .nunique()
)


print(
    "\n--- Frozen engine-record structure ---"
)
print(
    "Scenarios       :",
    scenario_count_625f
)
print(
    "Engine records  :",
    engine_record_count_625f
)
print(
    "Rows/scenario   :",
    sorted(
        group_sizes_625f
        .unique()
        .tolist()
    )
)
print(
    "Units/scenario  :",
    sorted(
        unique_unit_counts_625f
        .unique()
        .tolist()
    )
)


if scenario_count_625f != 315:
    raise RuntimeError(
        "Expected 315 frozen scenarios. "
        f"Found {scenario_count_625f}."
    )


if engine_record_count_625f != 31500:
    raise RuntimeError(
        "Expected 31,500 engine-scenario records. "
        f"Found {engine_record_count_625f}."
    )


if not (
    group_sizes_625f == 100
).all():

    raise RuntimeError(
        "Not every frozen scenario contains "
        "exactly 100 endpoint engines."
    )


if not (
    unique_unit_counts_625f == 100
).all():

    raise RuntimeError(
        "At least one frozen scenario does not "
        "contain 100 unique endpoint engines."
    )


duplicate_keys_625f = int(
    engine_df_625f.duplicated(
        subset=[
            "global_index",
            "unit_id",
        ]
    ).sum()
)


print(
    "Duplicate scenario/unit keys:",
    duplicate_keys_625f
)


if duplicate_keys_625f != 0:
    raise RuntimeError(
        "Duplicate scenario/unit endpoint records found."
    )


# ============================================================
# 7. VERIFY CLEAN STATE IS IDENTICAL ACROSS ALL 315 SCENARIOS
# ============================================================

CLEAN_CONSISTENCY_COLUMNS_625F = [
    "cycle",
    "RUL_cap_125",
    "clean_prediction",
    "clean_reliability",
    "clean_accept",
]


clean_consistency_625f = {}


for col_625f in CLEAN_CONSISTENCY_COLUMNS_625F:

    max_unique_625f = int(
        engine_df_625f
        .groupby(
            "unit_id"
        )[
            col_625f
        ]
        .nunique(
            dropna=False
        )
        .max()
    )

    clean_consistency_625f[
        col_625f
    ] = max_unique_625f


print(
    "\n--- Clean-state consistency across scenarios ---"
)

for key_625f, value_625f in (
    clean_consistency_625f.items()
):
    print(
        f"{key_625f:<22}: "
        f"max unique per engine = {value_625f}"
    )


if any(
    value_625f != 1
    for value_625f
    in clean_consistency_625f.values()
):
    raise RuntimeError(
        "Clean endpoint state is not identical "
        "across all corruption scenarios."
    )


clean_df_625f = (
    engine_df_625f[
        [
            "unit_id",
            "cycle",
            "RUL_cap_125",
            "clean_prediction",
            "clean_reliability",
            "clean_accept",
        ]
    ]
    .drop_duplicates(
        subset=[
            "unit_id"
        ]
    )
    .sort_values(
        "unit_id"
    )
    .reset_index(
        drop=True
    )
)


if len(
    clean_df_625f
) != 100:

    raise RuntimeError(
        "Clean endpoint reconstruction "
        "did not produce exactly 100 engines."
    )


# ============================================================
# 8. VERIFY STORED STEP 6.25E2 SCORE ARTIFACT
# ============================================================

required_scenario_cols_625f = {
    "global_index",
    "full_MAE",
    "selective_MAE",
    "coverage",
}


scenario_verification_possible_625f = (
    required_scenario_cols_625f
    .issubset(
        scenario_score_df_625f.columns
    )
)


print(
    "\n--- Step 6.25E2 scenario-score verification capability ---"
)
print(
    "Required score columns present:",
    scenario_verification_possible_625f
)


# ============================================================
# 9. RISK–COVERAGE GRID
# ============================================================

# This is an evaluation grid.
# It does NOT redefine the frozen operating threshold.

COVERAGE_GRID_625F = np.array(
    [
        0.50,
        0.60,
        0.70,
        0.75,
        0.80,
        0.85,
        0.90,
        0.95,
        1.00,
    ],
    dtype=np.float64
)


print(
    "\n--- Evaluation-only coverage grid ---"
)
print(
    COVERAGE_GRID_625F.tolist()
)
print(
    "\nIMPORTANT:"
)
print(
    "These coverage values are diagnostic/evaluation points."
)
print(
    "They do NOT replace the previously frozen reliability threshold."
)


# ============================================================
# 10. EXACT RANKED RISK–COVERAGE FUNCTION
# ============================================================

def build_ranked_risk_curve_625f(
    df,
    prediction_col,
    reliability_col,
    target_col,
    coverage_grid,
):
    """
    Lower reliability score = more trusted.

    At each target coverage:
      1. rank endpoints by reliability score ascending;
      2. accept the most trusted k endpoints;
      3. calculate MAE/RMSE using labels only AFTER ranking.

    unit_id is used only as a deterministic tie breaker.
    """

    work = df.copy()

    required_local_625f = [
        "unit_id",
        prediction_col,
        reliability_col,
        target_col,
    ]

    missing_local_625f = [
        c
        for c in required_local_625f
        if c not in work.columns
    ]

    if missing_local_625f:
        raise RuntimeError(
            "Risk-coverage helper missing columns: "
            + str(
                missing_local_625f
            )
        )

    work[
        "_error"
    ] = (
        work[
            prediction_col
        ]
        - work[
            target_col
        ]
    )

    work[
        "_abs_error"
    ] = np.abs(
        work[
            "_error"
        ]
    )

    work[
        "_sq_error"
    ] = (
        work[
            "_error"
        ]
        ** 2
    )

    work = (
        work
        .sort_values(
            [
                reliability_col,
                "unit_id",
            ],
            ascending=[
                True,
                True,
            ],
            kind="mergesort",
        )
        .reset_index(
            drop=True
        )
    )

    n_625f = int(
        len(
            work
        )
    )

    full_mae_625f = float(
        work[
            "_abs_error"
        ].mean()
    )

    full_rmse_625f = float(
        np.sqrt(
            work[
                "_sq_error"
            ].mean()
        )
    )

    scores_625f = (
        work[
            reliability_col
        ]
        .to_numpy(
            dtype=np.float64
        )
    )

    rows_625f = []


    for coverage_target_625f in coverage_grid:

        k_625f = int(
            round(
                float(
                    coverage_target_625f
                )
                * n_625f
            )
        )

        k_625f = max(
            1,
            min(
                n_625f,
                k_625f
            )
        )

        accepted_625f = (
            work.iloc[
                :k_625f
            ]
        )

        rejected_625f = (
            work.iloc[
                k_625f:
            ]
        )

        selective_mae_625f = float(
            accepted_625f[
                "_abs_error"
            ].mean()
        )

        selective_rmse_625f = float(
            np.sqrt(
                accepted_625f[
                    "_sq_error"
                ].mean()
            )
        )

        if len(
            rejected_625f
        ) > 0:

            abstained_mae_625f = float(
                rejected_625f[
                    "_abs_error"
                ].mean()
            )

            abstained_rmse_625f = float(
                np.sqrt(
                    rejected_625f[
                        "_sq_error"
                    ].mean()
                )
            )

        else:
            abstained_mae_625f = np.nan
            abstained_rmse_625f = np.nan


        gain_abs_625f = float(
            full_mae_625f
            - selective_mae_625f
        )

        gain_rel_625f = (
            gain_abs_625f
            / full_mae_625f
            if full_mae_625f > 0
            else np.nan
        )


        if np.isfinite(
            abstained_mae_625f
        ):
            separation_625f = float(
                abstained_mae_625f
                - selective_mae_625f
            )
        else:
            separation_625f = np.nan


        boundary_score_625f = float(
            scores_625f[
                k_625f - 1
            ]
        )


        if k_625f < n_625f:
            boundary_tie_straddled_625f = bool(
                scores_625f[
                    k_625f - 1
                ]
                ==
                scores_625f[
                    k_625f
                ]
            )
        else:
            boundary_tie_straddled_625f = False


        boundary_tie_size_625f = int(
            np.sum(
                scores_625f
                ==
                boundary_score_625f
            )
        )


        rows_625f.append(
            {
                "coverage_target":
                    float(
                        coverage_target_625f
                    ),

                "accepted":
                    int(
                        k_625f
                    ),

                "abstained":
                    int(
                        n_625f
                        - k_625f
                    ),

                "actual_coverage":
                    float(
                        k_625f
                        / n_625f
                    ),

                "full_MAE":
                    full_mae_625f,

                "selective_MAE":
                    selective_mae_625f,

                "abstained_MAE":
                    abstained_mae_625f,

                "full_RMSE":
                    full_rmse_625f,

                "selective_RMSE":
                    selective_rmse_625f,

                "abstained_RMSE":
                    abstained_rmse_625f,

                "MAE_gain_abs":
                    gain_abs_625f,

                "MAE_gain_rel":
                    float(
                        gain_rel_625f
                    ),

                "abstained_minus_accepted_MAE":
                    separation_625f,

                "mean_accepted_reliability":
                    float(
                        accepted_625f[
                            reliability_col
                        ].mean()
                    ),

                "boundary_reliability_score":
                    boundary_score_625f,

                "boundary_tie_size":
                    boundary_tie_size_625f,

                "boundary_tie_straddled":
                    boundary_tie_straddled_625f,

                "max_accepted_abs_error":
                    float(
                        accepted_625f[
                            "_abs_error"
                        ].max()
                    ),
            }
        )


    return pd.DataFrame(
        rows_625f
    )


# ============================================================
# 11. CLEAN EXACT-RANKED RISK–COVERAGE CURVE
# ============================================================

clean_curve_625f = (
    build_ranked_risk_curve_625f(
        df=clean_df_625f,
        prediction_col=
            "clean_prediction",
        reliability_col=
            "clean_reliability",
        target_col=
            "RUL_cap_125",
        coverage_grid=
            COVERAGE_GRID_625F,
    )
)


clean_curve_625f.insert(
    0,
    "dataset_state",
    "clean"
)


print(
    "\n--- CLEAN exact-ranked risk–coverage curve ---"
)

display(
    clean_curve_625f[
        [
            "coverage_target",
            "accepted",
            "selective_MAE",
            "abstained_MAE",
            "MAE_gain_abs",
            "MAE_gain_rel",
            "abstained_minus_accepted_MAE",
        ]
    ]
)


# ============================================================
# 12. ALL 315 CORRUPTED SCENARIO RISK CURVES
# ============================================================

scenario_curve_parts_625f = []


for (
    global_index_625f,
    group_625f
) in engine_df_625f.groupby(
    "global_index",
    sort=True
):

    meta_row_625f = (
        group_625f.iloc[0]
    )

    curve_625f = (
        build_ranked_risk_curve_625f(
            df=group_625f,
            prediction_col=
                "corrupted_prediction",
            reliability_col=
                "corrupted_reliability",
            target_col=
                "RUL_cap_125",
            coverage_grid=
                COVERAGE_GRID_625F,
        )
    )


    curve_625f.insert(
        0,
        "global_index",
        int(
            global_index_625f
        )
    )

    curve_625f.insert(
        1,
        "family",
        str(
            meta_row_625f[
                "family"
            ]
        )
    )

    curve_625f.insert(
        2,
        "scenario_id",
        str(
            meta_row_625f[
                "scenario_id"
            ]
        )
    )

    curve_625f.insert(
        3,
        "sensor",
        str(
            meta_row_625f[
                "sensor"
            ]
        )
    )

    curve_625f.insert(
        4,
        "severity",
        str(
            meta_row_625f[
                "severity"
            ]
        )
    )

    scenario_curve_parts_625f.append(
        curve_625f
    )


scenario_curve_625f = (
    pd.concat(
        scenario_curve_parts_625f,
        ignore_index=True
    )
)


expected_curve_rows_625f = (
    315
    * len(
        COVERAGE_GRID_625F
    )
)


if len(
    scenario_curve_625f
) != expected_curve_rows_625f:

    raise RuntimeError(
        "Unexpected number of scenario risk-coverage rows. "
        f"Expected {expected_curve_rows_625f}, "
        f"found {len(scenario_curve_625f)}."
    )


print(
    "\n--- Corrupted scenario risk curves constructed ---"
)
print(
    "Scenarios:",
    scenario_curve_625f[
        "global_index"
    ].nunique()
)
print(
    "Coverage points/scenario:",
    len(
        COVERAGE_GRID_625F
    )
)
print(
    "Total curve rows:",
    len(
        scenario_curve_625f
    )
)


# ============================================================
# 13. AGGREGATION HELPER
# ============================================================

def summarize_curves_625f(
    df,
    group_columns
):

    summary_625f = (
        df
        .groupby(
            group_columns,
            as_index=False
        )
        .agg(
            scenarios=(
                "global_index",
                "nunique"
            ),

            mean_selective_MAE=(
                "selective_MAE",
                "mean"
            ),

            median_selective_MAE=(
                "selective_MAE",
                "median"
            ),

            std_selective_MAE=(
                "selective_MAE",
                "std"
            ),

            mean_full_MAE=(
                "full_MAE",
                "mean"
            ),

            mean_MAE_gain_abs=(
                "MAE_gain_abs",
                "mean"
            ),

            median_MAE_gain_abs=(
                "MAE_gain_abs",
                "median"
            ),

            mean_MAE_gain_rel=(
                "MAE_gain_rel",
                "mean"
            ),

            median_MAE_gain_rel=(
                "MAE_gain_rel",
                "median"
            ),

            improvement_fraction=(
                "MAE_gain_abs",
                lambda s:
                    float(
                        (
                            s > 0
                        ).mean()
                    )
            ),

            mean_error_separation=(
                "abstained_minus_accepted_MAE",
                "mean"
            ),

            positive_error_separation_fraction=(
                "abstained_minus_accepted_MAE",
                lambda s:
                    float(
                        (
                            s.dropna()
                            > 0
                        ).mean()
                    )
                    if s.notna().any()
                    else np.nan
            ),

            mean_boundary_tie_size=(
                "boundary_tie_size",
                "mean"
            ),

            boundary_tie_straddle_fraction=(
                "boundary_tie_straddled",
                "mean"
            ),
        )
    )

    return summary_625f


# ============================================================
# 14. OVERALL SCENARIO-AVERAGED CURVE
# ============================================================

overall_curve_625f = (
    summarize_curves_625f(
        scenario_curve_625f,
        [
            "coverage_target"
        ]
    )
)


print(
    "\n--- Overall corrupted risk–coverage summary ---"
)

display(
    overall_curve_625f[
        [
            "coverage_target",
            "scenarios",
            "mean_full_MAE",
            "mean_selective_MAE",
            "mean_MAE_gain_abs",
            "mean_MAE_gain_rel",
            "improvement_fraction",
            "mean_error_separation",
            "positive_error_separation_fraction",
        ]
    ]
)


# ============================================================
# 15. FAMILY-LEVEL CURVES
# ============================================================

family_curve_625f = (
    summarize_curves_625f(
        scenario_curve_625f,
        [
            "family",
            "coverage_target",
        ]
    )
)


# ============================================================
# 16. FAMILY × SEVERITY CURVES
# ============================================================

family_severity_curve_625f = (
    summarize_curves_625f(
        scenario_curve_625f,
        [
            "family",
            "severity",
            "coverage_target",
        ]
    )
)


# ============================================================
# 17. EXACT 75% RANKED OPERATING-POINT DIAGNOSTIC
# ============================================================

ranked75_625f = (
    scenario_curve_625f[
        np.isclose(
            scenario_curve_625f[
                "coverage_target"
            ],
            0.75,
        )
    ]
    .copy()
)


ranked75_improved_count_625f = int(
    (
        ranked75_625f[
            "MAE_gain_abs"
        ]
        > 0
    ).sum()
)

ranked75_positive_separation_count_625f = int(
    (
        ranked75_625f[
            "abstained_minus_accepted_MAE"
        ]
        > 0
    ).sum()
)


print(
    "\n--- Exact-ranked 75% evaluation point ---"
)
print(
    "Scenarios evaluated         :",
    len(
        ranked75_625f
    )
)
print(
    "Scenarios with MAE reduction:",
    ranked75_improved_count_625f,
    "/ 315"
)
print(
    "Positive error separation   :",
    ranked75_positive_separation_count_625f,
    "/ 315"
)
print(
    "Mean ranked-75 selective MAE:",
    float(
        ranked75_625f[
            "selective_MAE"
        ].mean()
    )
)
print(
    "Mean ranked-75 MAE gain     :",
    float(
        ranked75_625f[
            "MAE_gain_abs"
        ].mean()
    )
)


# ============================================================
# 18. RECONSTRUCT EXISTING FROZEN-THRESHOLD OPERATING POINTS
# ============================================================

engine_df_625f[
    "_corrupted_abs_error_625f"
] = np.abs(
    engine_df_625f[
        "corrupted_prediction"
    ]
    -
    engine_df_625f[
        "RUL_cap_125"
    ]
)


fixed_rows_625f = []


for (
    global_index_625f,
    group_625f
) in engine_df_625f.groupby(
    "global_index",
    sort=True
):

    accepted_mask_625f = (
        group_625f[
            "corrupted_accept"
        ].to_numpy(
            dtype=bool
        )
    )

    errors_625f = (
        group_625f[
            "_corrupted_abs_error_625f"
        ]
        .to_numpy(
            dtype=np.float64
        )
    )

    n_625f = len(
        group_625f
    )

    accepted_count_625f = int(
        accepted_mask_625f.sum()
    )


    if accepted_count_625f <= 0:
        raise RuntimeError(
            f"Scenario {global_index_625f} "
            "has zero frozen accepted endpoints."
        )


    selective_mae_625f = float(
        errors_625f[
            accepted_mask_625f
        ].mean()
    )

    full_mae_625f = float(
        errors_625f.mean()
    )


    rejected_mask_625f = (
        ~accepted_mask_625f
    )


    if rejected_mask_625f.any():
        abstained_mae_625f = float(
            errors_625f[
                rejected_mask_625f
            ].mean()
        )
    else:
        abstained_mae_625f = np.nan


    first_625f = (
        group_625f.iloc[0]
    )


    fixed_rows_625f.append(
        {
            "global_index":
                int(
                    global_index_625f
                ),

            "family":
                str(
                    first_625f[
                        "family"
                    ]
                ),

            "scenario_id":
                str(
                    first_625f[
                        "scenario_id"
                    ]
                ),

            "sensor":
                str(
                    first_625f[
                        "sensor"
                    ]
                ),

            "severity":
                str(
                    first_625f[
                        "severity"
                    ]
                ),

            "accepted":
                accepted_count_625f,

            "abstained":
                int(
                    n_625f
                    - accepted_count_625f
                ),

            "coverage":
                float(
                    accepted_count_625f
                    / n_625f
                ),

            "full_MAE":
                full_mae_625f,

            "selective_MAE":
                selective_mae_625f,

            "abstained_MAE":
                abstained_mae_625f,

            "MAE_gain_abs":
                float(
                    full_mae_625f
                    - selective_mae_625f
                ),

            "MAE_gain_rel":
                float(
                    (
                        full_mae_625f
                        - selective_mae_625f
                    )
                    /
                    full_mae_625f
                ),
        }
    )


fixed_operating_points_625f = pd.DataFrame(
    fixed_rows_625f
)


# ============================================================
# 19. VERIFY FROZEN OPERATING POINT AGAINST STEP 6.25E2
# ============================================================

scenario_score_check_625f = None


if scenario_verification_possible_625f:

    verify_left_625f = (
        fixed_operating_points_625f[
            [
                "global_index",
                "coverage",
                "full_MAE",
                "selective_MAE",
            ]
        ]
        .copy()
    )


    verify_right_625f = (
        scenario_score_df_625f[
            [
                "global_index",
                "coverage",
                "full_MAE",
                "selective_MAE",
            ]
        ]
        .copy()
    )


    scenario_score_check_625f = (
        verify_left_625f
        .merge(
            verify_right_625f,
            on="global_index",
            suffixes=(
                "_reconstructed",
                "_stored",
            ),
            validate="one_to_one",
        )
    )


    for metric_625f in [
        "coverage",
        "full_MAE",
        "selective_MAE",
    ]:

        scenario_score_check_625f[
            f"{metric_625f}_abs_diff"
        ] = np.abs(
            scenario_score_check_625f[
                f"{metric_625f}_reconstructed"
            ]
            -
            scenario_score_check_625f[
                f"{metric_625f}_stored"
            ]
        )


    max_coverage_diff_625f = float(
        scenario_score_check_625f[
            "coverage_abs_diff"
        ].max()
    )

    max_full_mae_diff_625f = float(
        scenario_score_check_625f[
            "full_MAE_abs_diff"
        ].max()
    )

    max_selective_mae_diff_625f = float(
        scenario_score_check_625f[
            "selective_MAE_abs_diff"
        ].max()
    )


    print(
        "\n--- Frozen Step 6.25E2 operating-point reproduction ---"
    )
    print(
        "Maximum coverage difference     :",
        max_coverage_diff_625f
    )
    print(
        "Maximum full-MAE difference     :",
        max_full_mae_diff_625f
    )
    print(
        "Maximum selective-MAE difference:",
        max_selective_mae_diff_625f
    )


    if max(
        max_coverage_diff_625f,
        max_full_mae_diff_625f,
        max_selective_mae_diff_625f,
    ) > 1e-10:

        raise RuntimeError(
            "Frozen Step 6.25E2 operating-point "
            "metrics were not reproduced exactly."
        )


# ============================================================
# 20. FROZEN THRESHOLD vs EXACT-RANKED 75% COMPARISON
# ============================================================

fixed_family_625f = (
    fixed_operating_points_625f
    .groupby(
        "family",
        as_index=False
    )
    .agg(
        frozen_mean_coverage=(
            "coverage",
            "mean"
        ),
        frozen_mean_selective_MAE=(
            "selective_MAE",
            "mean"
        ),
        frozen_mean_MAE_gain=(
            "MAE_gain_abs",
            "mean"
        ),
    )
)


ranked75_family_625f = (
    ranked75_625f
    .groupby(
        "family",
        as_index=False
    )
    .agg(
        ranked75_mean_coverage=(
            "actual_coverage",
            "mean"
        ),
        ranked75_mean_selective_MAE=(
            "selective_MAE",
            "mean"
        ),
        ranked75_mean_MAE_gain=(
            "MAE_gain_abs",
            "mean"
        ),
    )
)


operating_comparison_625f = (
    fixed_family_625f
    .merge(
        ranked75_family_625f,
        on="family",
        how="inner",
        validate="one_to_one",
    )
)


print(
    "\n--- Frozen-threshold vs exact-ranked 75% ---"
)
print(
    "NOTE: these are different evaluation policies."
)
print(
    "The exact-ranked curve does NOT replace the frozen threshold."
)

display(
    operating_comparison_625f
)


# ============================================================
# 21. SCENARIO-LEVEL CURVE DIAGNOSTICS
# ============================================================

scenario_diagnostic_rows_625f = []


for (
    global_index_625f,
    group_625f
) in scenario_curve_625f.groupby(
    "global_index",
    sort=True
):

    ordered_625f = (
        group_625f
        .sort_values(
            "coverage_target"
        )
        .reset_index(
            drop=True
        )
    )

    x_625f = (
        ordered_625f[
            "coverage_target"
        ]
        .to_numpy(
            dtype=np.float64
        )
    )

    y_625f = (
        ordered_625f[
            "selective_MAE"
        ]
        .to_numpy(
            dtype=np.float64
        )
    )


    if len(
        x_625f
    ) < 2:
        raise RuntimeError(
            "Insufficient coverage points "
            "for risk-coverage integration."
        )


    area_625f = float(
        np.trapezoid(
            y_625f,
            x_625f
        )
    )

    coverage_span_625f = float(
        x_625f[-1]
        -
        x_625f[0]
    )


    avg_risk_over_interval_625f = float(
        area_625f
        /
        coverage_span_625f
    )


    diffs_625f = np.diff(
        y_625f
    )

    monotonicity_violations_625f = int(
        np.sum(
            diffs_625f
            < -1e-12
        )
    )


    row75_625f = (
        ordered_625f[
            np.isclose(
                ordered_625f[
                    "coverage_target"
                ],
                0.75,
            )
        ]
        .iloc[0]
    )


    first_625f = (
        ordered_625f.iloc[0]
    )


    scenario_diagnostic_rows_625f.append(
        {
            "global_index":
                int(
                    global_index_625f
                ),

            "family":
                first_625f[
                    "family"
                ],

            "scenario_id":
                first_625f[
                    "scenario_id"
                ],

            "sensor":
                first_625f[
                    "sensor"
                ],

            "severity":
                first_625f[
                    "severity"
                ],

            "average_MAE_over_coverage_0.50_1.00":
                avg_risk_over_interval_625f,

            "risk_curve_area_0.50_1.00":
                area_625f,

            "monotonicity_violations":
                monotonicity_violations_625f,

            "fully_non_decreasing_risk_curve":
                bool(
                    monotonicity_violations_625f
                    == 0
                ),

            "ranked75_selective_MAE":
                float(
                    row75_625f[
                        "selective_MAE"
                    ]
                ),

            "ranked75_MAE_gain_abs":
                float(
                    row75_625f[
                        "MAE_gain_abs"
                    ]
                ),

            "ranked75_MAE_gain_rel":
                float(
                    row75_625f[
                        "MAE_gain_rel"
                    ]
                ),

            "ranked75_error_separation":
                float(
                    row75_625f[
                        "abstained_minus_accepted_MAE"
                    ]
                ),

            "ranked75_boundary_tie":
                bool(
                    row75_625f[
                        "boundary_tie_straddled"
                    ]
                ),
        }
    )


scenario_diagnostics_625f = pd.DataFrame(
    scenario_diagnostic_rows_625f
)


fully_monotonic_count_625f = int(
    scenario_diagnostics_625f[
        "fully_non_decreasing_risk_curve"
    ].sum()
)

total_monotonicity_violations_625f = int(
    scenario_diagnostics_625f[
        "monotonicity_violations"
    ].sum()
)


print(
    "\n--- Risk-ranking monotonicity diagnostic ---"
)
print(
    "Fully non-decreasing curves:",
    fully_monotonic_count_625f,
    "/ 315"
)
print(
    "Total evaluated-grid monotonicity violations:",
    total_monotonicity_violations_625f
)


# ============================================================
# 22. CLEAN AND CORRUPTED AVERAGE-RISK SUMMARY
# ============================================================

clean_x_625f = (
    clean_curve_625f[
        "coverage_target"
    ]
    .to_numpy(
        dtype=np.float64
    )
)

clean_y_625f = (
    clean_curve_625f[
        "selective_MAE"
    ]
    .to_numpy(
        dtype=np.float64
    )
)

clean_area_625f = float(
    np.trapezoid(
        clean_y_625f,
        clean_x_625f
    )
)

clean_average_risk_625f = float(
    clean_area_625f
    /
    (
        clean_x_625f[-1]
        -
        clean_x_625f[0]
    )
)


overall_x_625f = (
    overall_curve_625f[
        "coverage_target"
    ]
    .to_numpy(
        dtype=np.float64
    )
)

overall_y_625f = (
    overall_curve_625f[
        "mean_selective_MAE"
    ]
    .to_numpy(
        dtype=np.float64
    )
)

overall_area_625f = float(
    np.trapezoid(
        overall_y_625f,
        overall_x_625f
    )
)

overall_average_risk_625f = float(
    overall_area_625f
    /
    (
        overall_x_625f[-1]
        -
        overall_x_625f[0]
    )
)


# ============================================================
# 23. FIGURE 1 — OVERALL RISK–COVERAGE
# ============================================================

overall_plot_path_625f = (
    OUTPUT_DIR_625F
    / "fd001_risk_coverage_overall.png"
)


plt.figure(
    figsize=(9, 6)
)

plt.plot(
    overall_curve_625f[
        "coverage_target"
    ],
    overall_curve_625f[
        "mean_selective_MAE"
    ],
    marker="o",
    label="Corrupted scenarios: mean selective MAE",
)

plt.plot(
    clean_curve_625f[
        "coverage_target"
    ],
    clean_curve_625f[
        "selective_MAE"
    ],
    marker="o",
    label="Clean FD001 selective MAE",
)


mean_corrupted_full_mae_625f = float(
    scenario_curve_625f[
        "full_MAE"
    ].mean()
)


# For uniform random rejection independent of error,
# expected selected MAE equals the full-set MAE.
plt.axhline(
    mean_corrupted_full_mae_625f,
    linestyle="--",
    label=(
        "Random-rejection expected risk "
        "(mean corrupted full MAE)"
    ),
)


mean_fixed_coverage_625f = float(
    fixed_operating_points_625f[
        "coverage"
    ].mean()
)

mean_fixed_selective_mae_625f = float(
    fixed_operating_points_625f[
        "selective_MAE"
    ].mean()
)


plt.scatter(
    [
        mean_fixed_coverage_625f
    ],
    [
        mean_fixed_selective_mae_625f
    ],
    s=80,
    marker="X",
    label="Frozen global-threshold operating point",
)


plt.xlabel(
    "Coverage"
)

plt.ylabel(
    "Selective MAE (cycles)"
)

plt.title(
    "FD001 Official TEST: Risk–Coverage Analysis"
)

plt.grid(
    True,
    alpha=0.25
)

plt.legend()

plt.tight_layout()

plt.savefig(
    overall_plot_path_625f,
    dpi=220,
    bbox_inches="tight"
)

plt.show()

plt.close()


# ============================================================
# 24. FIGURE 2 — RISK–COVERAGE BY CORRUPTION FAMILY
# ============================================================

family_plot_path_625f = (
    OUTPUT_DIR_625F
    / "fd001_risk_coverage_by_family.png"
)


plt.figure(
    figsize=(10, 6)
)


for family_625f in sorted(
    family_curve_625f[
        "family"
    ].unique()
):

    plot_df_625f = (
        family_curve_625f[
            family_curve_625f[
                "family"
            ]
            ==
            family_625f
        ]
        .sort_values(
            "coverage_target"
        )
    )

    plt.plot(
        plot_df_625f[
            "coverage_target"
        ],
        plot_df_625f[
            "mean_selective_MAE"
        ],
        marker="o",
        label=str(
            family_625f
        ),
    )


plt.xlabel(
    "Coverage"
)

plt.ylabel(
    "Mean selective MAE (cycles)"
)

plt.title(
    "FD001 Official TEST: Risk–Coverage by Corruption Family"
)

plt.grid(
    True,
    alpha=0.25
)

plt.legend(
    title="Corruption family"
)

plt.tight_layout()

plt.savefig(
    family_plot_path_625f,
    dpi=220,
    bbox_inches="tight"
)

plt.show()

plt.close()


# ============================================================
# 25. FIGURE 3 — IMPROVEMENT FRACTION vs COVERAGE
# ============================================================

improvement_plot_path_625f = (
    OUTPUT_DIR_625F
    / "fd001_improvement_fraction_vs_coverage.png"
)


plt.figure(
    figsize=(9, 6)
)

plt.plot(
    overall_curve_625f[
        "coverage_target"
    ],
    overall_curve_625f[
        "improvement_fraction"
    ],
    marker="o",
)

plt.xlabel(
    "Coverage"
)

plt.ylabel(
    "Fraction of scenarios with selective MAE improvement"
)

plt.title(
    "FD001 Official TEST: Selective Improvement vs Coverage"
)

plt.ylim(
    -0.02,
    1.02
)

plt.grid(
    True,
    alpha=0.25
)

plt.tight_layout()

plt.savefig(
    improvement_plot_path_625f,
    dpi=220,
    bbox_inches="tight"
)

plt.show()

plt.close()


# ============================================================
# 26. SAVE TABLE ARTIFACTS
# ============================================================

scenario_curve_path_625f = (
    OUTPUT_DIR_625F
    / "fd001_risk_coverage_scenario_level.csv"
)

overall_curve_path_625f = (
    OUTPUT_DIR_625F
    / "fd001_risk_coverage_overall_summary.csv"
)

family_curve_path_625f = (
    OUTPUT_DIR_625F
    / "fd001_risk_coverage_family_summary.csv"
)

family_severity_path_625f = (
    OUTPUT_DIR_625F
    / "fd001_risk_coverage_family_severity_summary.csv"
)

clean_curve_path_625f = (
    OUTPUT_DIR_625F
    / "fd001_risk_coverage_clean_curve.csv"
)

scenario_diagnostics_path_625f = (
    OUTPUT_DIR_625F
    / "fd001_risk_coverage_scenario_diagnostics.csv"
)

fixed_operating_path_625f = (
    OUTPUT_DIR_625F
    / "fd001_frozen_threshold_operating_points.csv"
)

operating_comparison_path_625f = (
    OUTPUT_DIR_625F
    / "fd001_frozen_vs_ranked75_family_comparison.csv"
)


scenario_curve_625f.to_csv(
    scenario_curve_path_625f,
    index=False
)

overall_curve_625f.to_csv(
    overall_curve_path_625f,
    index=False
)

family_curve_625f.to_csv(
    family_curve_path_625f,
    index=False
)

family_severity_curve_625f.to_csv(
    family_severity_path_625f,
    index=False
)

clean_curve_625f.to_csv(
    clean_curve_path_625f,
    index=False
)

scenario_diagnostics_625f.to_csv(
    scenario_diagnostics_path_625f,
    index=False
)

fixed_operating_points_625f.to_csv(
    fixed_operating_path_625f,
    index=False
)

operating_comparison_625f.to_csv(
    operating_comparison_path_625f,
    index=False
)


# ============================================================
# 27. SUMMARY JSON
# ============================================================

summary_path_625f = (
    OUTPUT_DIR_625F
    / "fd001_risk_coverage_summary.json"
)


summary_625f = {

    "step":
        "6.25F",

    "dataset":
        "NASA C-MAPSS FD001",

    "analysis":
        "Frozen official TEST risk-coverage evaluation",

    "created_at":
        datetime.now().isoformat(),

    "source_engine_artifact":
        str(
            ENGINE_FILE_625F
        ),

    "source_scenario_artifact":
        str(
            SCENARIO_FILE_625F
        ),

    "frozen_scenarios":
        315,

    "engine_scenario_records":
        31500,

    "coverage_grid":
        COVERAGE_GRID_625F.tolist(),

    "ranking_direction":
        (
            "lower reliability score = "
            "more trusted / accepted first"
        ),

    "target":
        "RUL_cap_125",

    "clean_full_MAE":
        float(
            clean_curve_625f[
                "full_MAE"
            ].iloc[0]
        ),

    "mean_corrupted_full_MAE":
        mean_corrupted_full_mae_625f,

    "clean_average_MAE_over_coverage_0.50_1.00":
        clean_average_risk_625f,

    "mean_corrupted_average_MAE_over_coverage_0.50_1.00":
        overall_average_risk_625f,

    "frozen_threshold_mean_realized_coverage":
        mean_fixed_coverage_625f,

    "frozen_threshold_mean_selective_MAE":
        mean_fixed_selective_mae_625f,

    "ranked75_mean_selective_MAE":
        float(
            ranked75_625f[
                "selective_MAE"
            ].mean()
        ),

    "ranked75_mean_MAE_gain_abs":
        float(
            ranked75_625f[
                "MAE_gain_abs"
            ].mean()
        ),

    "ranked75_scenarios_improved":
        ranked75_improved_count_625f,

    "ranked75_positive_error_separation":
        ranked75_positive_separation_count_625f,

    "fully_non_decreasing_scenario_curves":
        fully_monotonic_count_625f,

    "total_grid_monotonicity_violations":
        total_monotonicity_violations_625f,

    "safety":
        {
            "model_inference_rerun":
                False,

            "mc_dropout_rerun":
                False,

            "corruptions_regenerated":
                False,

            "frozen_threshold_modified":
                False,

            "frozen_rule_modified":
                False,

            "official_labels_used_for_tuning":
                False,

            "official_labels_used_for_evaluation":
                True,

            "coverage_grid_is_evaluation_only":
                True,
        },
}


with open(
    summary_path_625f,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary_625f,
        f,
        indent=2,
        default=json_default_625f,
    )


# ============================================================
# 28. VERIFY SOURCE ARTIFACTS WERE NOT MODIFIED
# ============================================================

ENGINE_HASH_AFTER_625F = (
    sha256_file_625f(
        ENGINE_FILE_625F
    )
)

SCENARIO_HASH_AFTER_625F = (
    sha256_file_625f(
        SCENARIO_FILE_625F
    )
)


engine_source_unchanged_625f = bool(
    ENGINE_HASH_BEFORE_625F
    ==
    ENGINE_HASH_AFTER_625F
)

scenario_source_unchanged_625f = bool(
    SCENARIO_HASH_BEFORE_625F
    ==
    SCENARIO_HASH_AFTER_625F
)


if not engine_source_unchanged_625f:
    raise RuntimeError(
        "Frozen engine artifact hash changed "
        "during Step 6.25F."
    )


if not scenario_source_unchanged_625f:
    raise RuntimeError(
        "Frozen scenario artifact hash changed "
        "during Step 6.25F."
    )


# ============================================================
# 29. OUTPUT MANIFEST WITH SHA-256
# ============================================================

output_files_625f = [
    scenario_curve_path_625f,
    overall_curve_path_625f,
    family_curve_path_625f,
    family_severity_path_625f,
    clean_curve_path_625f,
    scenario_diagnostics_path_625f,
    fixed_operating_path_625f,
    operating_comparison_path_625f,
    summary_path_625f,
    overall_plot_path_625f,
    family_plot_path_625f,
    improvement_plot_path_625f,
]


output_hashes_625f = {
    path.name:
        sha256_file_625f(
            path
        )
    for path in output_files_625f
}


manifest_path_625f = (
    OUTPUT_DIR_625F
    / "fd001_risk_coverage_manifest.json"
)


manifest_625f = {

    "step":
        "6.25F",

    "source_files":
        {
            ENGINE_FILE_625F.name:
                {
                    "path":
                        str(
                            ENGINE_FILE_625F
                        ),

                    "sha256_before":
                        ENGINE_HASH_BEFORE_625F,

                    "sha256_after":
                        ENGINE_HASH_AFTER_625F,

                    "unchanged":
                        engine_source_unchanged_625f,
                },

            SCENARIO_FILE_625F.name:
                {
                    "path":
                        str(
                            SCENARIO_FILE_625F
                        ),

                    "sha256_before":
                        SCENARIO_HASH_BEFORE_625F,

                    "sha256_after":
                        SCENARIO_HASH_AFTER_625F,

                    "unchanged":
                        scenario_source_unchanged_625f,
                },
        },

    "outputs":
        output_hashes_625f,

    "frozen_decision_changed":
        False,

    "model_inference_rerun":
        False,

    "mc_dropout_rerun":
        False,

    "corruptions_regenerated":
        False,

    "official_labels_used_for_tuning":
        False,

    "official_labels_used_for_evaluation":
        True,
}


with open(
    manifest_path_625f,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest_625f,
        f,
        indent=2,
        default=json_default_625f,
    )


# ============================================================
# 30. FINAL COMPACT REPORT
# ============================================================

ranked75_mean_mae_625f = float(
    ranked75_625f[
        "selective_MAE"
    ].mean()
)

ranked75_mean_gain_rel_625f = float(
    ranked75_625f[
        "MAE_gain_rel"
    ].mean()
)


print(
    "\n"
    "============================================================"
)

print(
    "STEP 6.25F FINAL RISK–COVERAGE SUMMARY"
)

print(
    "============================================================"
)

print(
    "Frozen scenarios analysed"
    "             :",
    scenario_count_625f,
    "/ 315"
)

print(
    "Engine-scenario records"
    "               :",
    engine_record_count_625f,
    "/ 31500"
)

print(
    "Coverage grid"
    "                         :",
    COVERAGE_GRID_625F.tolist()
)

print(
    "Clean full MAE"
    "                        :",
    float(
        clean_curve_625f[
            "full_MAE"
        ].iloc[0]
    )
)

print(
    "Mean corrupted full MAE"
    "               :",
    mean_corrupted_full_mae_625f
)

print(
    "Frozen threshold mean coverage"
    "         :",
    mean_fixed_coverage_625f
)

print(
    "Frozen threshold mean selective MAE"
    "    :",
    mean_fixed_selective_mae_625f
)

print(
    "Exact-ranked 75% selective MAE"
    "        :",
    ranked75_mean_mae_625f
)

print(
    "Exact-ranked 75% mean relative gain"
    "   :",
    ranked75_mean_gain_rel_625f
)

print(
    "Exact-ranked 75% scenarios improved"
    "    :",
    ranked75_improved_count_625f,
    "/ 315"
)

print(
    "Exact-ranked 75% positive separation"
    "   :",
    ranked75_positive_separation_count_625f,
    "/ 315"
)

print(
    "Fully non-decreasing risk curves"
    "        :",
    fully_monotonic_count_625f,
    "/ 315"
)

print(
    "Total grid monotonicity violations"
    "       :",
    total_monotonicity_violations_625f
)

print(
    "Clean mean risk over coverage 0.50–1.00 :",
    clean_average_risk_625f
)

print(
    "Corrupted mean risk over coverage"
    " 0.50–1.00:",
    overall_average_risk_625f
)

print(
    "Frozen engine artifact unchanged"
    "         :",
    engine_source_unchanged_625f
)

print(
    "Frozen scenario artifact unchanged"
    "       :",
    scenario_source_unchanged_625f
)

print(
    "Model inference rerun"
    "                   : False"
)

print(
    "MC-dropout rerun"
    "                       : False"
)

print(
    "Corruptions regenerated"
    "                 : False"
)

print(
    "Frozen threshold/rule modified"
    "           : False"
)

print(
    "Official TEST labels used for tuning"
    "     : False"
)

print(
    "Official TEST labels used for evaluation"
    " : True"
)

print(
    "Files written"
    "                           : True"
)

print(
    "\nOutput directory:\n",
    OUTPUT_DIR_625F
)

print(
    "\nGenerated artifacts:"
)

for i_625f, path_625f in enumerate(
    output_files_625f
    +
    [
        manifest_path_625f
    ],
    start=1,
):

    print(
        f"{i_625f}. {path_625f}"
    )


print(
    "\nIMPORTANT:"
)

print(
    "The exact-ranked coverage curve is an "
    "evaluation/diagnostic analysis only."
)

print(
    "It does NOT replace or retune the previously "
    "frozen 75% reliability operating decision."
)

print(
    "\n=== END STEP 6.25F ==="
)

In [ ]:
# ============================================================
# Step 6.25F-R
# COMPACT REVIEW OUTPUT FOR STEP 6.25F
# Nothing is modified. No model/corruption is rerun.
# ============================================================

from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image

PROJECT_ROOT_625FR = Path(
    r"C:\RUL_Reliability_Research"
)

RC_DIR_625FR = (
    PROJECT_ROOT_625FR
    / "06_results"
    / "step6_official_fd001"
    / "corruption_transfer"
    / "full_scoring"
    / "risk_coverage"
)

SUMMARY_PATH_625FR = (
    RC_DIR_625FR
    / "fd001_risk_coverage_summary.json"
)

OVERALL_PATH_625FR = (
    RC_DIR_625FR
    / "fd001_risk_coverage_overall_summary.csv"
)

FAMILY_PATH_625FR = (
    RC_DIR_625FR
    / "fd001_risk_coverage_family_summary.csv"
)

FAMILY_SEVERITY_PATH_625FR = (
    RC_DIR_625FR
    / "fd001_risk_coverage_family_severity_summary.csv"
)

PLOT_PATH_625FR = (
    RC_DIR_625FR
    / "fd001_risk_coverage_overall.png"
)


# ------------------------------------------------------------
# 1. Load saved results
# ------------------------------------------------------------

with open(
    SUMMARY_PATH_625FR,
    "r",
    encoding="utf-8"
) as f:
    summary_625fr = json.load(f)

overall_625fr = pd.read_csv(
    OVERALL_PATH_625FR
)

family_625fr = pd.read_csv(
    FAMILY_PATH_625FR
)

family_severity_625fr = pd.read_csv(
    FAMILY_SEVERITY_PATH_625FR
)


# ------------------------------------------------------------
# 2. Compact final summary
# ------------------------------------------------------------

print(
    "\n"
    "======================================================\n"
    "STEP 6.25F — COMPACT REVIEW SUMMARY\n"
    "======================================================"
)

keys_625fr = [
    "frozen_scenarios",
    "engine_scenario_records",
    "clean_full_MAE",
    "mean_corrupted_full_MAE",
    "frozen_threshold_mean_realized_coverage",
    "frozen_threshold_mean_selective_MAE",
    "ranked75_mean_selective_MAE",
    "ranked75_mean_MAE_gain_abs",
    "ranked75_scenarios_improved",
    "ranked75_positive_error_separation",
    "fully_non_decreasing_scenario_curves",
    "total_grid_monotonicity_violations",
    "clean_average_MAE_over_coverage_0.50_1.00",
    "mean_corrupted_average_MAE_over_coverage_0.50_1.00",
]

for key in keys_625fr:
    print(
        f"{key:<50}: "
        f"{summary_625fr.get(key)}"
    )


# ------------------------------------------------------------
# 3. Overall risk–coverage curve
# ------------------------------------------------------------

print(
    "\n--- OVERALL RISK–COVERAGE CURVE ---"
)

display(
    overall_625fr[
        [
            "coverage_target",
            "mean_full_MAE",
            "mean_selective_MAE",
            "mean_MAE_gain_abs",
            "mean_MAE_gain_rel",
            "improvement_fraction",
            "mean_error_separation",
        ]
    ]
)


# ------------------------------------------------------------
# 4. Family results specifically at 75% coverage
# ------------------------------------------------------------

family75_625fr = (
    family_625fr[
        family_625fr[
            "coverage_target"
        ].round(6)
        == 0.75
    ]
    .copy()
)

print(
    "\n--- FAMILY RESULTS AT EXACT 75% COVERAGE ---"
)

display(
    family75_625fr[
        [
            "family",
            "scenarios",
            "mean_full_MAE",
            "mean_selective_MAE",
            "mean_MAE_gain_abs",
            "mean_MAE_gain_rel",
            "improvement_fraction",
            "mean_error_separation",
        ]
    ]
)


# ------------------------------------------------------------
# 5. Family × severity results at 75%
# ------------------------------------------------------------

family_severity75_625fr = (
    family_severity_625fr[
        family_severity_625fr[
            "coverage_target"
        ].round(6)
        == 0.75
    ]
    .copy()
)

print(
    "\n--- FAMILY × SEVERITY RESULTS AT 75% COVERAGE ---"
)

display(
    family_severity75_625fr[
        [
            "family",
            "severity",
            "scenarios",
            "mean_selective_MAE",
            "mean_MAE_gain_abs",
            "mean_MAE_gain_rel",
            "improvement_fraction",
        ]
    ]
)


# ------------------------------------------------------------
# 6. Show only main risk–coverage figure
# ------------------------------------------------------------

print(
    "\n--- MAIN RISK–COVERAGE FIGURE ---"
)

display(
    Image(
        filename=str(
            PLOT_PATH_625FR
        )
    )
)


print(
    "\n"
    "No model inference rerun.\n"
    "No MC-dropout rerun.\n"
    "No corruption regenerated.\n"
    "No threshold changed.\n"
    "No frozen decision changed.\n"
    "=== END COMPACT REVIEW ==="
)

In [ ]:
# ============================================================
# Step 6.26A
# Sensor-quality reference / provenance audit
#
# PURPOSE
# -------
# Before defining the sensor-quality reliability signal:
#
# 1. verify the existing DEV sensor-reference object,
# 2. verify which primary model inputs are sensors,
# 3. verify that every model sensor has reference statistics,
# 4. reproduce those reference statistics from the DEV base,
# 5. inspect numerical health of the reference quantities,
# 6. make NO change to the frozen uncertainty reliability rule.
#
# IMPORTANT
# ---------
# - DEV/reference inspection only.
# - No official TEST labels are used.
# - No official TEST scoring is performed.
# - No model inference is performed.
# - No MC-dropout is performed.
# - No corruption is generated.
# - No file is written.
# - No frozen decision is modified.
# ============================================================

import re
import numpy as np
import pandas as pd

print(
    "\n"
    "=== Step 6.26A — "
    "Sensor-quality reference / provenance audit ==="
)


# ------------------------------------------------------------
# 1. Required existing objects
# ------------------------------------------------------------

required_objects_626a = [
    "PRIMARY_INPUT_COLUMNS",
    "sensor_reference_stats",
    "val_raw_base_620gc",
]

print("\n--- Required objects ---")

missing_626a = []

for name_626a in required_objects_626a:

    exists_626a = name_626a in globals()

    print(
        f"{name_626a:<30}: "
        f"{exists_626a}"
    )

    if not exists_626a:
        missing_626a.append(
            name_626a
        )


if missing_626a:

    raise RuntimeError(
        "Step 6.26A cannot continue. "
        "Missing required objects: "
        + str(
            missing_626a
        )
    )


# ------------------------------------------------------------
# 2. Defensive local copies
# ------------------------------------------------------------

primary_columns_626a = list(
    PRIMARY_INPUT_COLUMNS
)

reference_626a = (
    sensor_reference_stats
    .copy(deep=True)
)

dev_base_626a = (
    val_raw_base_620gc
    .copy(deep=True)
)


# ------------------------------------------------------------
# 3. Identify sensor columns robustly
# ------------------------------------------------------------

sensor_pattern_626a = re.compile(
    r"^sensor_(\d+)$"
)


def sensor_sort_key_626a(name):

    match_626a = (
        sensor_pattern_626a
        .match(
            str(name)
        )
    )

    if match_626a is None:
        return (
            10**9,
            str(name),
        )

    return (
        int(
            match_626a
            .group(1)
        ),
        str(name),
    )


primary_sensor_columns_626a = sorted(
    [
        col_626a
        for col_626a
        in primary_columns_626a
        if (
            sensor_pattern_626a
            .match(
                str(col_626a)
            )
            is not None
        )
    ],
    key=sensor_sort_key_626a,
)


primary_non_sensor_columns_626a = [
    col_626a
    for col_626a
    in primary_columns_626a
    if col_626a
    not in primary_sensor_columns_626a
]


reference_sensor_names_626a = sorted(
    [
        str(idx_626a)
        for idx_626a
        in reference_626a.index
        if (
            sensor_pattern_626a
            .match(
                str(idx_626a)
            )
            is not None
        )
    ],
    key=sensor_sort_key_626a,
)


print(
    "\n--- Primary-input decomposition ---"
)

print(
    "Total primary inputs       :",
    len(
        primary_columns_626a
    )
)

print(
    "Primary sensor inputs      :",
    len(
        primary_sensor_columns_626a
    )
)

print(
    "Primary non-sensor inputs  :",
    len(
        primary_non_sensor_columns_626a
    )
)

print(
    "\nPrimary sensor columns:"
)

print(
    primary_sensor_columns_626a
)

print(
    "\nPrimary non-sensor columns:"
)

print(
    primary_non_sensor_columns_626a
)


# ------------------------------------------------------------
# 4. Reference structure
# ------------------------------------------------------------

expected_reference_fields_626a = [
    "mean",
    "std",
    "min",
    "max",
    "q01",
    "q99",
    "range",
]


missing_reference_fields_626a = [
    field_626a
    for field_626a
    in expected_reference_fields_626a
    if field_626a
    not in reference_626a.columns
]


missing_reference_sensors_626a = sorted(
    list(
        set(
            primary_sensor_columns_626a
        )
        -
        set(
            reference_sensor_names_626a
        )
    ),
    key=sensor_sort_key_626a,
)


extra_reference_sensors_626a = sorted(
    list(
        set(
            reference_sensor_names_626a
        )
        -
        set(
            primary_sensor_columns_626a
        )
    ),
    key=sensor_sort_key_626a,
)


print(
    "\n--- Existing sensor-reference object ---"
)

print(
    "Type                  :",
    type(
        reference_626a
    ).__name__,
)

print(
    "Shape                 :",
    reference_626a.shape,
)

print(
    "Reference columns     :",
    list(
        reference_626a.columns
    ),
)

print(
    "Reference sensors     :",
    len(
        reference_sensor_names_626a
    ),
)

print(
    "Missing ref. fields   :",
    missing_reference_fields_626a,
)

print(
    "Missing model sensors :",
    missing_reference_sensors_626a,
)

print(
    "Extra ref. sensors    :",
    extra_reference_sensors_626a,
)


if missing_reference_fields_626a:

    raise RuntimeError(
        "Required sensor-reference fields "
        "are missing: "
        + str(
            missing_reference_fields_626a
        )
    )


if missing_reference_sensors_626a:

    raise RuntimeError(
        "Some primary model sensors do not "
        "have frozen reference statistics: "
        + str(
            missing_reference_sensors_626a
        )
    )


# ------------------------------------------------------------
# 5. Confirm reference source columns exist in DEV base
# ------------------------------------------------------------

missing_from_dev_626a = [
    sensor_626a
    for sensor_626a
    in primary_sensor_columns_626a
    if sensor_626a
    not in dev_base_626a.columns
]


print(
    "\n--- DEV reference base ---"
)

print(
    "Type                  :",
    type(
        dev_base_626a
    ).__name__,
)

print(
    "Shape                 :",
    dev_base_626a.shape,
)

print(
    "Missing model sensors :",
    missing_from_dev_626a,
)


if missing_from_dev_626a:

    raise RuntimeError(
        "The DEV reference base is missing "
        "primary sensor columns: "
        + str(
            missing_from_dev_626a
        )
    )


# ------------------------------------------------------------
# 6. Reconstruct reference statistics from DEV data
#
# We calculate:
#
# mean
# sample std (pandas default, ddof=1)
# min
# max
# q01
# q99
# range
#
# This checks provenance rather than changing anything.
# ------------------------------------------------------------

reconstructed_rows_626a = []

for sensor_626a in primary_sensor_columns_626a:

    values_626a = pd.to_numeric(
        dev_base_626a[
            sensor_626a
        ],
        errors="coerce",
    )

    reconstructed_rows_626a.append(
        {
            "sensor":
                sensor_626a,

            "mean":
                float(
                    values_626a.mean()
                ),

            "std":
                float(
                    values_626a.std(
                        ddof=1
                    )
                ),

            "min":
                float(
                    values_626a.min()
                ),

            "max":
                float(
                    values_626a.max()
                ),

            "q01":
                float(
                    values_626a.quantile(
                        0.01
                    )
                ),

            "q99":
                float(
                    values_626a.quantile(
                        0.99
                    )
                ),

            "range":
                float(
                    values_626a.max()
                    -
                    values_626a.min()
                ),

            "finite_fraction":
                float(
                    np.isfinite(
                        values_626a
                        .to_numpy(
                            dtype=float
                        )
                    ).mean()
                ),
        }
    )


reconstructed_626a = (
    pd.DataFrame(
        reconstructed_rows_626a
    )
    .set_index(
        "sensor"
    )
)


# ------------------------------------------------------------
# 7. Exact/numerical provenance comparison
# ------------------------------------------------------------

comparison_rows_626a = []

for sensor_626a in primary_sensor_columns_626a:

    stored_626a = reference_626a.loc[
        sensor_626a,
        expected_reference_fields_626a,
    ].astype(float)

    rebuilt_626a = reconstructed_626a.loc[
        sensor_626a,
        expected_reference_fields_626a,
    ].astype(float)

    abs_diffs_626a = np.abs(
        stored_626a.to_numpy()
        -
        rebuilt_626a.to_numpy()
    )

    comparison_rows_626a.append(
        {
            "sensor":
                sensor_626a,

            "stored_mean":
                float(
                    stored_626a["mean"]
                ),

            "rebuilt_mean":
                float(
                    rebuilt_626a["mean"]
                ),

            "stored_std":
                float(
                    stored_626a["std"]
                ),

            "rebuilt_std":
                float(
                    rebuilt_626a["std"]
                ),

            "stored_q01":
                float(
                    stored_626a["q01"]
                ),

            "rebuilt_q01":
                float(
                    rebuilt_626a["q01"]
                ),

            "stored_q99":
                float(
                    stored_626a["q99"]
                ),

            "rebuilt_q99":
                float(
                    rebuilt_626a["q99"]
                ),

            "max_abs_reference_diff":
                float(
                    np.max(
                        abs_diffs_626a
                    )
                ),

            "reference_close_1e_10":
                bool(
                    np.allclose(
                        stored_626a.to_numpy(),
                        rebuilt_626a.to_numpy(),
                        rtol=0.0,
                        atol=1e-10,
                    )
                ),

            "finite_fraction":
                float(
                    reconstructed_626a.loc[
                        sensor_626a,
                        "finite_fraction",
                    ]
                ),
        }
    )


reference_comparison_626a = pd.DataFrame(
    comparison_rows_626a
)


# ------------------------------------------------------------
# 8. Numerical-health checks
# ------------------------------------------------------------

reference_numeric_626a = (
    reference_626a.loc[
        primary_sensor_columns_626a,
        expected_reference_fields_626a,
    ]
    .astype(float)
)


all_reference_finite_626a = bool(
    np.isfinite(
        reference_numeric_626a
        .to_numpy()
    ).all()
)


all_std_positive_626a = bool(
    (
        reference_numeric_626a[
            "std"
        ]
        >
        0.0
    ).all()
)


all_ranges_positive_626a = bool(
    (
        reference_numeric_626a[
            "range"
        ]
        >
        0.0
    ).all()
)


quantiles_ordered_626a = bool(
    (
        reference_numeric_626a[
            "q01"
        ]
        <=
        reference_numeric_626a[
            "q99"
        ]
    ).all()
)


all_dev_sensor_values_finite_626a = bool(
    (
        reconstructed_626a[
            "finite_fraction"
        ]
        ==
        1.0
    ).all()
)


reference_provenance_close_626a = bool(
    reference_comparison_626a[
        "reference_close_1e_10"
    ].all()
)


maximum_reference_error_626a = float(
    reference_comparison_626a[
        "max_abs_reference_diff"
    ].max()
)


# ------------------------------------------------------------
# 9. Inspect scale characteristics
#
# These quantities will matter when we later create a
# dimensionless sensor-quality score.
# ------------------------------------------------------------

scale_audit_626a = pd.DataFrame(
    {
        "sensor":
            primary_sensor_columns_626a,

        "mean":
            [
                float(
                    reference_numeric_626a
                    .loc[
                        s_626a,
                        "mean",
                    ]
                )
                for s_626a
                in primary_sensor_columns_626a
            ],

        "std":
            [
                float(
                    reference_numeric_626a
                    .loc[
                        s_626a,
                        "std",
                    ]
                )
                for s_626a
                in primary_sensor_columns_626a
            ],

        "q01":
            [
                float(
                    reference_numeric_626a
                    .loc[
                        s_626a,
                        "q01",
                    ]
                )
                for s_626a
                in primary_sensor_columns_626a
            ],

        "q99":
            [
                float(
                    reference_numeric_626a
                    .loc[
                        s_626a,
                        "q99",
                    ]
                )
                for s_626a
                in primary_sensor_columns_626a
            ],

        "reference_range":
            [
                float(
                    reference_numeric_626a
                    .loc[
                        s_626a,
                        "range",
                    ]
                )
                for s_626a
                in primary_sensor_columns_626a
            ],
    }
)


scale_audit_626a[
    "q01_q99_width"
] = (
    scale_audit_626a[
        "q99"
    ]
    -
    scale_audit_626a[
        "q01"
    ]
)


scale_audit_626a[
    "std_to_range_ratio"
] = (
    scale_audit_626a[
        "std"
    ]
    /
    scale_audit_626a[
        "reference_range"
    ]
)


# ------------------------------------------------------------
# 10. Optional reference path visibility
# ------------------------------------------------------------

reference_path_available_626a = (
    "SENSOR_REFERENCE_PATH"
    in globals()
)


print(
    "\n--- Reference provenance ---"
)

print(
    "SENSOR_REFERENCE_PATH available:",
    reference_path_available_626a,
)

if reference_path_available_626a:

    print(
        "Reference path:",
        SENSOR_REFERENCE_PATH,
    )


# ------------------------------------------------------------
# 11. Compact tables
# ------------------------------------------------------------

print(
    "\n--- Reference reproduction comparison ---"
)

display(
    reference_comparison_626a
)


print(
    "\n--- Sensor scale audit ---"
)

display(
    scale_audit_626a
)


# ------------------------------------------------------------
# 12. Final audit summary
# ------------------------------------------------------------

print(
    "\n"
    "=============================================="
)

print(
    "STEP 6.26A SENSOR-QUALITY REFERENCE SUMMARY"
)

print(
    "=============================================="
)

print(
    "Primary model inputs              :",
    len(
        primary_columns_626a
    ),
)

print(
    "Primary sensor inputs             :",
    len(
        primary_sensor_columns_626a
    ),
)

print(
    "Primary non-sensor inputs         :",
    len(
        primary_non_sensor_columns_626a
    ),
)

print(
    "Reference sensors available       :",
    len(
        reference_sensor_names_626a
    ),
)

print(
    "All primary sensors referenced    :",
    len(
        missing_reference_sensors_626a
    )
    == 0,
)

print(
    "All reference values finite       :",
    all_reference_finite_626a,
)

print(
    "All DEV sensor values finite      :",
    all_dev_sensor_values_finite_626a,
)

print(
    "All sensor std values positive    :",
    all_std_positive_626a,
)

print(
    "All sensor ranges positive        :",
    all_ranges_positive_626a,
)

print(
    "All q01 <= q99                    :",
    quantiles_ordered_626a,
)

print(
    "Reference reproduced from DEV     :",
    reference_provenance_close_626a,
)

print(
    "Maximum reference reproduction err:",
    maximum_reference_error_626a,
)


sensor_quality_reference_ready_626a = bool(
    (
        len(
            missing_reference_sensors_626a
        )
        == 0
    )
    and
    all_reference_finite_626a
    and
    all_dev_sensor_values_finite_626a
    and
    all_std_positive_626a
    and
    all_ranges_positive_626a
    and
    quantiles_ordered_626a
)


print(
    "\nSENSOR-QUALITY REFERENCE READY      :",
    sensor_quality_reference_ready_626a,
)


print(
    "\n--- Safety status ---"
)

print(
    "Official TEST labels inspected    : False"
)

print(
    "Official TEST scored              : False"
)

print(
    "Model inference executed          : False"
)

print(
    "MC-dropout executed               : False"
)

print(
    "Corruption generated              : False"
)

print(
    "Frozen uncertainty rule modified : False"
)

print(
    "Frozen threshold modified        : False"
)

print(
    "Files written                    : False"
)

print(
    "\nNothing was modified."
)

print(
    "=== END STEP 6.26A ==="
)

In [ ]:
# ============================================================
# Step 6.26A1
# Exact sensor-reference provenance recovery
#
# PURPOSE
# -------
# Determine which existing DEVELOPMENT/TRAINING DataFrame most
# closely reproduces the already-frozen sensor_reference_stats.
#
# IMPORTANT
# ---------
# - Does NOT modify sensor_reference_stats.
# - Does NOT modify any dataset.
# - Does NOT inspect official TEST RUL labels.
# - Does NOT run the RUL model.
# - Does NOT generate corruptions.
# - Does NOT write files.
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

print(
    "\n=== Step 6.26A1 — "
    "Exact sensor-reference provenance recovery ==="
)

# ------------------------------------------------------------
# 1. Required objects
# ------------------------------------------------------------

required_626a1 = [
    "PRIMARY_INPUT_COLUMNS",
    "sensor_reference_stats",
]

print("\n--- Required objects ---")

missing_626a1 = []

for name_626a1 in required_626a1:
    available_626a1 = name_626a1 in globals()

    print(
        f"{name_626a1:<30}: "
        f"{available_626a1}"
    )

    if not available_626a1:
        missing_626a1.append(
            name_626a1
        )

if missing_626a1:
    raise RuntimeError(
        "Step 6.26A1 cannot continue. "
        "Missing required objects: "
        + str(
            missing_626a1
        )
    )


# ------------------------------------------------------------
# 2. Identify the 15 frozen primary sensor columns
# ------------------------------------------------------------

SENSOR_COLUMNS_626A1 = [
    column_626a1
    for column_626a1
    in list(
        PRIMARY_INPUT_COLUMNS
    )
    if str(
        column_626a1
    ).startswith(
        "sensor_"
    )
]

if len(
    SENSOR_COLUMNS_626A1
) != 15:
    raise RuntimeError(
        "Expected 15 primary sensor columns, "
        "but found "
        f"{len(SENSOR_COLUMNS_626A1)}."
    )

print(
    "\nPrimary sensors:",
    SENSOR_COLUMNS_626A1
)


# ------------------------------------------------------------
# 3. Prepare frozen reference
# ------------------------------------------------------------

REFERENCE_FIELDS_626A1 = [
    "mean",
    "std",
    "min",
    "max",
    "q01",
    "q99",
    "range",
]

stored_reference_626a1 = (
    sensor_reference_stats
    .loc[
        SENSOR_COLUMNS_626A1,
        REFERENCE_FIELDS_626A1
    ]
    .astype(
        float
    )
    .copy()
)

if not np.isfinite(
    stored_reference_626a1
    .to_numpy(
        dtype=float
    )
).all():
    raise RuntimeError(
        "Stored sensor reference contains "
        "non-finite values."
    )


# ------------------------------------------------------------
# 4. Exact reconstruction helper
# ------------------------------------------------------------

def rebuild_sensor_reference_626a1(
    candidate_df_626a1
):

    numeric_626a1 = (
        candidate_df_626a1[
            SENSOR_COLUMNS_626A1
        ]
        .astype(
            float
        )
    )

    rebuilt_626a1 = pd.DataFrame(
        index=SENSOR_COLUMNS_626A1
    )

    rebuilt_626a1["mean"] = (
        numeric_626a1.mean(
            axis=0
        )
    )

    rebuilt_626a1["std"] = (
        numeric_626a1.std(
            axis=0,
            ddof=1
        )
    )

    rebuilt_626a1["min"] = (
        numeric_626a1.min(
            axis=0
        )
    )

    rebuilt_626a1["max"] = (
        numeric_626a1.max(
            axis=0
        )
    )

    rebuilt_626a1["q01"] = (
        numeric_626a1.quantile(
            0.01,
            axis=0
        )
    )

    rebuilt_626a1["q99"] = (
        numeric_626a1.quantile(
            0.99,
            axis=0
        )
    )

    rebuilt_626a1["range"] = (
        rebuilt_626a1["max"]
        -
        rebuilt_626a1["min"]
    )

    return (
        rebuilt_626a1[
            REFERENCE_FIELDS_626A1
        ]
        .astype(
            float
        )
    )


# ------------------------------------------------------------
# 5. Search in-memory DEVELOPMENT / TRAINING candidates
# ------------------------------------------------------------

candidate_rows_626a1 = []

skip_name_tokens_626a1 = [
    "official_test",
    "test_raw",
    "test_scaled",
    "test_endpoint",
    "test_seq",
    "test_clean",
]

for (
    object_name_626a1,
    object_value_626a1
) in list(
    globals().items()
):

    if not isinstance(
        object_value_626a1,
        pd.DataFrame
    ):
        continue

    object_name_lower_626a1 = (
        str(
            object_name_626a1
        ).lower()
    )

    # Safety: do not use official TEST objects as
    # possible provenance candidates.
    if any(
        token_626a1
        in object_name_lower_626a1
        for token_626a1
        in skip_name_tokens_626a1
    ):
        continue

    if not set(
        SENSOR_COLUMNS_626A1
    ).issubset(
        set(
            object_value_626a1.columns
        )
    ):
        continue

    if len(
        object_value_626a1
    ) == 0:
        continue

    try:

        candidate_sensor_values_626a1 = (
            object_value_626a1[
                SENSOR_COLUMNS_626A1
            ]
            .astype(
                float
            )
            .to_numpy()
        )

        finite_fraction_626a1 = float(
            np.mean(
                np.isfinite(
                    candidate_sensor_values_626a1
                )
            )
        )

        if finite_fraction_626a1 < 1.0:
            continue

        rebuilt_candidate_626a1 = (
            rebuild_sensor_reference_626a1(
                object_value_626a1
            )
        )

        difference_626a1 = (
            rebuilt_candidate_626a1
            -
            stored_reference_626a1
        ).abs()

        max_difference_626a1 = float(
            difference_626a1
            .to_numpy()
            .max()
        )

        mean_difference_626a1 = float(
            difference_626a1
            .to_numpy()
            .mean()
        )

        exact_1e10_626a1 = bool(
            np.allclose(
                rebuilt_candidate_626a1
                .to_numpy(),
                stored_reference_626a1
                .to_numpy(),
                rtol=0.0,
                atol=1e-10,
            )
        )

        exact_1e6_626a1 = bool(
            np.allclose(
                rebuilt_candidate_626a1
                .to_numpy(),
                stored_reference_626a1
                .to_numpy(),
                rtol=0.0,
                atol=1e-6,
            )
        )

        candidate_rows_626a1.append(
            {
                "object_name":
                    object_name_626a1,

                "rows":
                    int(
                        len(
                            object_value_626a1
                        )
                    ),

                "columns":
                    int(
                        object_value_626a1
                        .shape[1]
                    ),

                "max_abs_reference_diff":
                    max_difference_626a1,

                "mean_abs_reference_diff":
                    mean_difference_626a1,

                "reference_close_1e_10":
                    exact_1e10_626a1,

                "reference_close_1e_6":
                    exact_1e6_626a1,
            }
        )

    except Exception as error_626a1:

        candidate_rows_626a1.append(
            {
                "object_name":
                    object_name_626a1,

                "rows":
                    int(
                        len(
                            object_value_626a1
                        )
                    ),

                "columns":
                    int(
                        object_value_626a1
                        .shape[1]
                    ),

                "max_abs_reference_diff":
                    np.nan,

                "mean_abs_reference_diff":
                    np.nan,

                "reference_close_1e_10":
                    False,

                "reference_close_1e_6":
                    False,

                "error":
                    repr(
                        error_626a1
                    ),
            }
        )


candidate_audit_626a1 = pd.DataFrame(
    candidate_rows_626a1
)

if len(
    candidate_audit_626a1
) == 0:
    raise RuntimeError(
        "No non-TEST in-memory DataFrame "
        "containing all 15 sensor columns "
        "was found."
    )

candidate_audit_626a1 = (
    candidate_audit_626a1
    .sort_values(
        [
            "max_abs_reference_diff",
            "mean_abs_reference_diff",
        ],
        na_position="last",
    )
    .reset_index(
        drop=True
    )
)

print(
    "\n--- Best in-memory provenance candidates ---"
)

display(
    candidate_audit_626a1.head(
        20
    )
)


# ------------------------------------------------------------
# 6. Best candidate detailed comparison
# ------------------------------------------------------------

best_candidate_name_626a1 = str(
    candidate_audit_626a1
    .iloc[0][
        "object_name"
    ]
)

best_candidate_df_626a1 = globals()[
    best_candidate_name_626a1
]

best_rebuilt_reference_626a1 = (
    rebuild_sensor_reference_626a1(
        best_candidate_df_626a1
    )
)

best_comparison_626a1 = pd.DataFrame(
    index=SENSOR_COLUMNS_626A1
)

for field_626a1 in REFERENCE_FIELDS_626A1:

    best_comparison_626a1[
        f"stored_{field_626a1}"
    ] = (
        stored_reference_626a1[
            field_626a1
        ]
    )

    best_comparison_626a1[
        f"rebuilt_{field_626a1}"
    ] = (
        best_rebuilt_reference_626a1[
            field_626a1
        ]
    )

    best_comparison_626a1[
        f"abs_diff_{field_626a1}"
    ] = (
        (
            stored_reference_626a1[
                field_626a1
            ]
            -
            best_rebuilt_reference_626a1[
                field_626a1
            ]
        )
        .abs()
    )

best_comparison_626a1[
    "sensor_max_abs_diff"
] = pd.concat(
    [
        best_comparison_626a1[
            f"abs_diff_{field_626a1}"
        ]
        for field_626a1
        in REFERENCE_FIELDS_626A1
    ],
    axis=1,
).max(
    axis=1
)

best_comparison_626a1 = (
    best_comparison_626a1
    .sort_values(
        "sensor_max_abs_diff",
        ascending=False,
    )
)

print(
    "\n--- Best candidate ---"
)

print(
    "Object name :",
    best_candidate_name_626a1
)

print(
    "Shape       :",
    best_candidate_df_626a1.shape
)

print(
    "Max error   :",
    float(
        best_comparison_626a1[
            "sensor_max_abs_diff"
        ].max()
    )
)

print(
    "\n--- Largest differences for best candidate ---"
)

display(
    best_comparison_626a1[
        [
            "stored_mean",
            "rebuilt_mean",
            "stored_std",
            "rebuilt_std",
            "stored_q01",
            "rebuilt_q01",
            "stored_q99",
            "rebuilt_q99",
            "sensor_max_abs_diff",
        ]
    ].head(
        15
    )
)


# ------------------------------------------------------------
# 7. Search notebook history for creation/provenance evidence
# ------------------------------------------------------------

print(
    "\n--- Notebook-history provenance search ---"
)

history_hits_626a1 = []

history_object_626a1 = globals().get(
    "In",
    None,
)

if isinstance(
    history_object_626a1,
    list
):

    search_terms_626a1 = [
        "sensor_reference_stats",
        "SENSOR_REFERENCE_PATH",
        "fd001_sensor_reference_stats",
    ]

    for (
        cell_number_626a1,
        cell_source_626a1
    ) in enumerate(
        history_object_626a1
    ):

        if not isinstance(
            cell_source_626a1,
            str
        ):
            continue

        if any(
            term_626a1
            in cell_source_626a1
            for term_626a1
            in search_terms_626a1
        ):

            # Ignore this diagnostic cell itself.
            if (
                "Step 6.26A1"
                in cell_source_626a1
            ):
                continue

            history_hits_626a1.append(
                {
                    "cell":
                        cell_number_626a1,

                    "source_preview":
                        cell_source_626a1[
                            :1200
                        ],
                }
            )

if history_hits_626a1:

    for hit_626a1 in history_hits_626a1[-15:]:

        print(
            "\nCELL",
            hit_626a1[
                "cell"
            ]
        )

        print(
            hit_626a1[
                "source_preview"
            ]
        )

else:

    print(
        "No usable notebook-history provenance "
        "records were found."
    )


# ------------------------------------------------------------
# 8. Existing reference path identity
# ------------------------------------------------------------

print(
    "\n--- Existing reference-file information ---"
)

if (
    "SENSOR_REFERENCE_PATH"
    in globals()
):

    path_626a1 = Path(
        SENSOR_REFERENCE_PATH
    )

    print(
        "Reference path :",
        path_626a1
    )

    print(
        "File exists    :",
        path_626a1.exists()
    )

    if path_626a1.exists():

        stored_file_626a1 = pd.read_csv(
            path_626a1,
            index_col=0,
        )

        common_fields_626a1 = [
            field_626a1
            for field_626a1
            in REFERENCE_FIELDS_626A1
            if field_626a1
            in stored_file_626a1.columns
        ]

        common_sensors_626a1 = [
            sensor_626a1
            for sensor_626a1
            in SENSOR_COLUMNS_626A1
            if sensor_626a1
            in stored_file_626a1.index
        ]

        file_vs_memory_exact_626a1 = False

        if (
            len(
                common_fields_626a1
            )
            ==
            len(
                REFERENCE_FIELDS_626A1
            )
            and
            len(
                common_sensors_626a1
            )
            ==
            len(
                SENSOR_COLUMNS_626A1
            )
        ):

            file_values_626a1 = (
                stored_file_626a1
                .loc[
                    SENSOR_COLUMNS_626A1,
                    REFERENCE_FIELDS_626A1
                ]
                .astype(
                    float
                )
            )

            file_vs_memory_exact_626a1 = bool(
                np.allclose(
                    file_values_626a1
                    .to_numpy(),
                    stored_reference_626a1
                    .to_numpy(),
                    rtol=0.0,
                    atol=1e-12,
                )
            )

        print(
            "File equals in-memory reference:",
            file_vs_memory_exact_626a1
        )

else:

    print(
        "SENSOR_REFERENCE_PATH is not "
        "available."
    )


# ------------------------------------------------------------
# 9. Compact audit summary
# ------------------------------------------------------------

best_max_diff_626a1 = float(
    candidate_audit_626a1
    .iloc[0][
        "max_abs_reference_diff"
    ]
)

exact_candidate_found_626a1 = bool(
    candidate_audit_626a1[
        "reference_close_1e_10"
    ].any()
)

near_candidate_found_626a1 = bool(
    candidate_audit_626a1[
        "reference_close_1e_6"
    ].any()
)

print(
    "\n"
    + "=" * 58
)

print(
    "STEP 6.26A1 PROVENANCE RECOVERY SUMMARY"
)

print(
    "=" * 58
)

print(
    "Non-TEST candidate DataFrames checked :",
    len(
        candidate_audit_626a1
    )
)

print(
    "Best candidate object                :",
    best_candidate_name_626a1
)

print(
    "Best candidate rows                  :",
    len(
        best_candidate_df_626a1
    )
)

print(
    "Best candidate maximum error         :",
    best_max_diff_626a1
)

print(
    "Exact provenance candidate found     :",
    exact_candidate_found_626a1
)

print(
    "Near-exact candidate found           :",
    near_candidate_found_626a1
)

print(
    "\nOfficial TEST labels inspected : False"
)

print(
    "Official TEST scored           : False"
)

print(
    "Model inference executed       : False"
)

print(
    "MC-dropout executed            : False"
)

print(
    "Corruption generated           : False"
)

print(
    "Reference modified             : False"
)

print(
    "Files written                  : False"
)

print(
    "\nNothing was modified."
)

print(
    "=== END STEP 6.26A1 ==="
)

In [ ]:
# ============================================================
# Step 6.26A2
# Confirm frozen sensor-reference provenance and statistic
# convention using canonical DEV raw data
#
# PURPOSE
# -------
# Verify that sensor_reference_stats was generated from the
# canonical 16,561-row DEV population, with population-standard
# deviation (ddof=0), subject only to stored numeric precision.
#
# SAFETY
# ------
# - No official TEST labels used
# - No TEST scoring
# - No inference
# - No MC-dropout
# - No corruption generation
# - No frozen object modification
# - No file writing
# ============================================================

import numpy as np
import pandas as pd

print(
    "\n=== Step 6.26A2 — "
    "Confirm exact sensor-reference convention ==="
)

# ------------------------------------------------------------
# 1. Required objects
# ------------------------------------------------------------

required_626a2 = [
    "PRIMARY_INPUT_COLUMNS",
    "sensor_reference_stats",
    "CANONICAL_DEV_RAW_623DD",
]

print("\n--- Required objects ---")

missing_626a2 = []

for name_626a2 in required_626a2:

    available_626a2 = (
        name_626a2 in globals()
    )

    print(
        f"{name_626a2:<30}: "
        f"{available_626a2}"
    )

    if not available_626a2:
        missing_626a2.append(
            name_626a2
        )

if missing_626a2:

    raise RuntimeError(
        "Step 6.26A2 cannot continue. "
        "Missing required objects: "
        + str(
            missing_626a2
        )
    )


# ------------------------------------------------------------
# 2. Frozen primary sensor list
# ------------------------------------------------------------

SENSOR_COLUMNS_626A2 = [
    column_626a2
    for column_626a2
    in list(
        PRIMARY_INPUT_COLUMNS
    )
    if str(
        column_626a2
    ).startswith(
        "sensor_"
    )
]

if len(
    SENSOR_COLUMNS_626A2
) != 15:

    raise RuntimeError(
        "Expected exactly 15 primary sensors; "
        f"found {len(SENSOR_COLUMNS_626A2)}."
    )

print(
    "\nPrimary sensors :",
    len(
        SENSOR_COLUMNS_626A2
    )
)

print(
    "Canonical DEV shape :",
    CANONICAL_DEV_RAW_623DD.shape
)

if len(
    CANONICAL_DEV_RAW_623DD
) != 16561:

    raise RuntimeError(
        "Canonical DEV row count differs from "
        "the previously identified provenance "
        "population."
    )


# ------------------------------------------------------------
# 3. Check all sensor columns exist
# ------------------------------------------------------------

missing_sensor_columns_626a2 = [
    sensor_626a2
    for sensor_626a2
    in SENSOR_COLUMNS_626A2
    if sensor_626a2
    not in CANONICAL_DEV_RAW_623DD.columns
]

print(
    "Missing sensor columns :",
    missing_sensor_columns_626a2
)

if missing_sensor_columns_626a2:

    raise RuntimeError(
        "Canonical DEV is missing required "
        "sensor columns."
    )


# ------------------------------------------------------------
# 4. Canonical DEV sensor matrix
# ------------------------------------------------------------

dev_sensors_626a2 = (
    CANONICAL_DEV_RAW_623DD[
        SENSOR_COLUMNS_626A2
    ]
    .astype(
        float
    )
    .copy()
)

all_dev_finite_626a2 = bool(
    np.isfinite(
        dev_sensors_626a2
        .to_numpy(
            dtype=float
        )
    ).all()
)

print(
    "All canonical DEV sensor values finite :",
    all_dev_finite_626a2
)

if not all_dev_finite_626a2:

    raise RuntimeError(
        "Canonical DEV sensor values contain "
        "non-finite entries."
    )


# ------------------------------------------------------------
# 5. Rebuild reference using population std (ddof=0)
# ------------------------------------------------------------

rebuilt_pop_626a2 = pd.DataFrame(
    index=SENSOR_COLUMNS_626A2
)

rebuilt_pop_626a2["mean"] = (
    dev_sensors_626a2.mean(
        axis=0
    )
)

rebuilt_pop_626a2["std"] = (
    dev_sensors_626a2.std(
        axis=0,
        ddof=0
    )
)

rebuilt_pop_626a2["min"] = (
    dev_sensors_626a2.min(
        axis=0
    )
)

rebuilt_pop_626a2["max"] = (
    dev_sensors_626a2.max(
        axis=0
    )
)

rebuilt_pop_626a2["q01"] = (
    dev_sensors_626a2.quantile(
        0.01,
        axis=0
    )
)

rebuilt_pop_626a2["q99"] = (
    dev_sensors_626a2.quantile(
        0.99,
        axis=0
    )
)

rebuilt_pop_626a2["range"] = (
    rebuilt_pop_626a2["max"]
    -
    rebuilt_pop_626a2["min"]
)


# ------------------------------------------------------------
# 6. Stored frozen reference
# ------------------------------------------------------------

REFERENCE_FIELDS_626A2 = [
    "mean",
    "std",
    "min",
    "max",
    "q01",
    "q99",
    "range",
]

stored_626a2 = (
    sensor_reference_stats
    .loc[
        SENSOR_COLUMNS_626A2,
        REFERENCE_FIELDS_626A2
    ]
    .astype(
        float
    )
    .copy()
)

rebuilt_pop_626a2 = (
    rebuilt_pop_626a2[
        REFERENCE_FIELDS_626A2
    ]
)


# ------------------------------------------------------------
# 7. Compare field-by-field
# ------------------------------------------------------------

comparison_rows_626a2 = []

for sensor_626a2 in SENSOR_COLUMNS_626A2:

    row_626a2 = {
        "sensor":
            sensor_626a2
    }

    sensor_max_diff_626a2 = 0.0

    for field_626a2 in REFERENCE_FIELDS_626A2:

        stored_value_626a2 = float(
            stored_626a2.loc[
                sensor_626a2,
                field_626a2
            ]
        )

        rebuilt_value_626a2 = float(
            rebuilt_pop_626a2.loc[
                sensor_626a2,
                field_626a2
            ]
        )

        diff_626a2 = abs(
            stored_value_626a2
            -
            rebuilt_value_626a2
        )

        row_626a2[
            f"stored_{field_626a2}"
        ] = stored_value_626a2

        row_626a2[
            f"rebuilt_{field_626a2}"
        ] = rebuilt_value_626a2

        row_626a2[
            f"abs_diff_{field_626a2}"
        ] = diff_626a2

        sensor_max_diff_626a2 = max(
            sensor_max_diff_626a2,
            diff_626a2
        )

    row_626a2[
        "sensor_max_abs_diff"
    ] = sensor_max_diff_626a2

    comparison_rows_626a2.append(
        row_626a2
    )


REFERENCE_COMPARISON_626A2 = (
    pd.DataFrame(
        comparison_rows_626a2
    )
    .sort_values(
        "sensor_max_abs_diff",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)

print(
    "\n--- Largest remaining differences ---"
)

display(
    REFERENCE_COMPARISON_626A2[
        [
            "sensor",
            "stored_mean",
            "rebuilt_mean",
            "abs_diff_mean",
            "stored_std",
            "rebuilt_std",
            "abs_diff_std",
            "stored_q01",
            "rebuilt_q01",
            "abs_diff_q01",
            "stored_q99",
            "rebuilt_q99",
            "abs_diff_q99",
            "sensor_max_abs_diff",
        ]
    ]
)


# ------------------------------------------------------------
# 8. Field-level maximum differences
# ------------------------------------------------------------

field_rows_626a2 = []

for field_626a2 in REFERENCE_FIELDS_626A2:

    differences_626a2 = (
        stored_626a2[
            field_626a2
        ]
        -
        rebuilt_pop_626a2[
            field_626a2
        ]
    ).abs()

    field_rows_626a2.append(
        {
            "field":
                field_626a2,

            "maximum_absolute_difference":
                float(
                    differences_626a2.max()
                ),

            "mean_absolute_difference":
                float(
                    differences_626a2.mean()
                ),
        }
    )

FIELD_DIFFERENCES_626A2 = pd.DataFrame(
    field_rows_626a2
)

print(
    "\n--- Field-level difference summary ---"
)

display(
    FIELD_DIFFERENCES_626A2
)


# ------------------------------------------------------------
# 9. Check expected storage precision
# ------------------------------------------------------------

# Frozen reference values appear to have been stored with
# finite decimal precision. Therefore exact binary equality is
# not the correct criterion.
#
# We verify both:
#   1) population-STD agreement
#   2) agreement after six-decimal rounding for mean/std,
#      while allowing quantile precision inherited from the
#      original stored file.

mean_exact_6dp_626a2 = bool(
    np.array_equal(
        np.round(
            stored_626a2["mean"]
            .to_numpy(),
            6
        ),
        np.round(
            rebuilt_pop_626a2["mean"]
            .to_numpy(),
            6
        )
    )
)

std_exact_6dp_626a2 = bool(
    np.array_equal(
        np.round(
            stored_626a2["std"]
            .to_numpy(),
            6
        ),
        np.round(
            rebuilt_pop_626a2["std"]
            .to_numpy(),
            6
        )
    )
)

min_exact_6dp_626a2 = bool(
    np.array_equal(
        np.round(
            stored_626a2["min"]
            .to_numpy(),
            6
        ),
        np.round(
            rebuilt_pop_626a2["min"]
            .to_numpy(),
            6
        )
    )
)

max_exact_6dp_626a2 = bool(
    np.array_equal(
        np.round(
            stored_626a2["max"]
            .to_numpy(),
            6
        ),
        np.round(
            rebuilt_pop_626a2["max"]
            .to_numpy(),
            6
        )
    )
)

q01_close_626a2 = bool(
    np.allclose(
        stored_626a2["q01"]
        .to_numpy(),
        rebuilt_pop_626a2["q01"]
        .to_numpy(),
        rtol=0.0,
        atol=1e-5
    )
)

q99_close_626a2 = bool(
    np.allclose(
        stored_626a2["q99"]
        .to_numpy(),
        rebuilt_pop_626a2["q99"]
        .to_numpy(),
        rtol=0.0,
        atol=1e-5
    )
)

range_exact_6dp_626a2 = bool(
    np.array_equal(
        np.round(
            stored_626a2["range"]
            .to_numpy(),
            6
        ),
        np.round(
            rebuilt_pop_626a2["range"]
            .to_numpy(),
            6
        )
    )
)


# ------------------------------------------------------------
# 10. Explicit DDof explanation test
# ------------------------------------------------------------

sample_std_626a2 = (
    dev_sensors_626a2.std(
        axis=0,
        ddof=1
    )
)

population_std_626a2 = (
    dev_sensors_626a2.std(
        axis=0,
        ddof=0
    )
)

stored_std_626a2 = (
    stored_626a2["std"]
)

sample_std_error_626a2 = float(
    (
        sample_std_626a2
        -
        stored_std_626a2
    )
    .abs()
    .max()
)

population_std_error_626a2 = float(
    (
        population_std_626a2
        -
        stored_std_626a2
    )
    .abs()
    .max()
)

population_std_better_626a2 = bool(
    population_std_error_626a2
    <
    sample_std_error_626a2
)

print(
    "\n--- Standard-deviation convention check ---"
)

print(
    "Maximum error using ddof=1 :",
    sample_std_error_626a2
)

print(
    "Maximum error using ddof=0 :",
    population_std_error_626a2
)

print(
    "Population STD convention supported :",
    population_std_better_626a2
)


# ------------------------------------------------------------
# 11. Final provenance conclusion
# ------------------------------------------------------------

REFERENCE_PROVENANCE_CONFIRMED_626A2 = bool(
    mean_exact_6dp_626a2
    and
    std_exact_6dp_626a2
    and
    min_exact_6dp_626a2
    and
    max_exact_6dp_626a2
    and
    q01_close_626a2
    and
    q99_close_626a2
    and
    range_exact_6dp_626a2
    and
    population_std_better_626a2
)


print(
    "\n"
    + "=" * 64
)

print(
    "STEP 6.26A2 SENSOR-REFERENCE PROVENANCE SUMMARY"
)

print(
    "=" * 64
)

print(
    "Canonical DEV rows                  :",
    len(
        CANONICAL_DEV_RAW_623DD
    )
)

print(
    "Primary sensors                     :",
    len(
        SENSOR_COLUMNS_626A2
    )
)

print(
    "Stored means reproduce at 6 dp      :",
    mean_exact_6dp_626a2
)

print(
    "Stored std reproduces at 6 dp       :",
    std_exact_6dp_626a2
)

print(
    "Stored minima reproduce at 6 dp     :",
    min_exact_6dp_626a2
)

print(
    "Stored maxima reproduce at 6 dp     :",
    max_exact_6dp_626a2
)

print(
    "Stored q01 values agree             :",
    q01_close_626a2
)

print(
    "Stored q99 values agree             :",
    q99_close_626a2
)

print(
    "Stored ranges reproduce at 6 dp     :",
    range_exact_6dp_626a2
)

print(
    "ddof=0 better than ddof=1           :",
    population_std_better_626a2
)

print(
    "Max stored-vs-ddof0 STD error       :",
    population_std_error_626a2
)

print(
    "\nREFERENCE PROVENANCE CONFIRMED       :",
    REFERENCE_PROVENANCE_CONFIRMED_626A2
)

print(
    "\nOfficial TEST labels inspected : False"
)

print(
    "Official TEST scored           : False"
)

print(
    "Model inference executed       : False"
)

print(
    "MC-dropout executed            : False"
)

print(
    "Corruption generated           : False"
)

print(
    "Frozen reference modified      : False"
)

print(
    "Files written                  : False"
)

print(
    "\nNothing was modified."
)

print(
    "=== END STEP 6.26A2 ==="
)

In [ ]:
# ============================================================
# Step 6.26A3
# Final tolerance-aware sensor-reference provenance closure
#
# This cell DOES NOT modify sensor_reference_stats.
# It only gives the final provenance verdict using an
# appropriate numeric-storage tolerance.
# ============================================================

import numpy as np

print(
    "\n=== Step 6.26A3 — "
    "Final sensor-reference provenance closure ==="
)

# ------------------------------------------------------------
# 1. Fixed audit tolerance
# ------------------------------------------------------------

REFERENCE_STORAGE_ATOL_626A3 = 1e-5


# ------------------------------------------------------------
# 2. Field-wise tolerance checks
# ------------------------------------------------------------

FIELD_TOLERANCE_RESULTS_626A3 = {}

for field_626a3 in REFERENCE_FIELDS_626A2:

    stored_values_626a3 = (
        stored_626a2[
            field_626a3
        ]
        .to_numpy(
            dtype=float
        )
    )

    rebuilt_values_626a3 = (
        rebuilt_pop_626a2[
            field_626a3
        ]
        .to_numpy(
            dtype=float
        )
    )

    FIELD_TOLERANCE_RESULTS_626A3[
        field_626a3
    ] = bool(
        np.allclose(
            stored_values_626a3,
            rebuilt_values_626a3,
            rtol=0.0,
            atol=REFERENCE_STORAGE_ATOL_626A3
        )
    )


# ------------------------------------------------------------
# 3. Maximum difference across every stored statistic
# ------------------------------------------------------------

MAX_REFERENCE_DIFFERENCE_626A3 = float(
    max(
        (
            stored_626a2[field_626a3]
            -
            rebuilt_pop_626a2[field_626a3]
        )
        .abs()
        .max()

        for field_626a3
        in REFERENCE_FIELDS_626A2
    )
)


# ------------------------------------------------------------
# 4. Standard-deviation convention verification
# ------------------------------------------------------------

DDOF0_CONFIRMED_626A3 = bool(
    population_std_error_626a2
    <
    sample_std_error_626a2
    and
    population_std_error_626a2
    <= REFERENCE_STORAGE_ATOL_626A3
)


# ------------------------------------------------------------
# 5. Final provenance verdict
# ------------------------------------------------------------

REFERENCE_PROVENANCE_CONFIRMED_626A3 = bool(
    all(
        FIELD_TOLERANCE_RESULTS_626A3.values()
    )
    and
    DDOF0_CONFIRMED_626A3
    and
    len(CANONICAL_DEV_RAW_623DD) == 16561
    and
    len(SENSOR_COLUMNS_626A2) == 15
)


# ------------------------------------------------------------
# 6. Report
# ------------------------------------------------------------

print(
    "\n--- Field-wise agreement "
    f"(absolute tolerance = {REFERENCE_STORAGE_ATOL_626A3}) ---"
)

for field_626a3 in REFERENCE_FIELDS_626A2:

    print(
        f"{field_626a3:<10}: "
        f"{FIELD_TOLERANCE_RESULTS_626A3[field_626a3]}"
    )


print(
    "\n--- Numeric provenance evidence ---"
)

print(
    "Canonical DEV rows                 :",
    len(CANONICAL_DEV_RAW_623DD)
)

print(
    "Primary sensors                    :",
    len(SENSOR_COLUMNS_626A2)
)

print(
    "Maximum difference, all fields     :",
    MAX_REFERENCE_DIFFERENCE_626A3
)

print(
    "Maximum STD error with ddof=0      :",
    population_std_error_626a2
)

print(
    "Maximum STD error with ddof=1      :",
    sample_std_error_626a2
)

print(
    "Population STD convention confirmed:",
    DDOF0_CONFIRMED_626A3
)


print(
    "\n"
    + "=" * 68
)

print(
    "STEP 6.26A FINAL PROVENANCE VERDICT"
)

print(
    "=" * 68
)

print(
    "Reference population               : "
    "Canonical DEV"
)

print(
    "Reference DEV rows                 : "
    "16561"
)

print(
    "Reference sensors                  : "
    "15 primary sensors"
)

print(
    "Standard-deviation convention      : "
    "population std (ddof=0)"
)

print(
    "Agreement criterion                : "
    "stored numeric precision"
)

print(
    "Maximum observed discrepancy       :",
    MAX_REFERENCE_DIFFERENCE_626A3
)

print(
    "\nREFERENCE PROVENANCE CONFIRMED      :",
    REFERENCE_PROVENANCE_CONFIRMED_626A3
)


print(
    "\nOfficial TEST labels inspected : False"
)

print(
    "Official TEST scored           : False"
)

print(
    "Model inference executed       : False"
)

print(
    "MC-dropout executed            : False"
)

print(
    "Corruption generated           : False"
)

print(
    "Frozen reference modified      : False"
)

print(
    "Files written                  : False"
)

print(
    "\nNothing was modified."
)

print(
    "=== END STEP 6.26A3 ==="
)

In [ ]:
print(
    "REFERENCE PROVENANCE CONFIRMED:",
    REFERENCE_PROVENANCE_CONFIRMED_626A3
)

In [ ]:
# ============================================================
# Step 6.26B — Clean-DEV sensor-quality candidate construction
# ============================================================

import numpy as np
import pandas as pd


print(
    "=== Step 6.26B — Clean-DEV sensor-quality "
    "candidate construction ==="
)


# ------------------------------------------------------------
# 1. Required frozen / verified objects
# ------------------------------------------------------------

required_objects_626b = [
    "PRIMARY_INPUT_COLUMNS",
    "sensor_reference_stats",
    "CANONICAL_DEV_RAW_623DD",
]

print("\n--- Required objects ---")

missing_objects_626b = []

for name_626b in required_objects_626b:
    exists_626b = name_626b in globals()

    print(
        f"{name_626b:<30}: "
        f"{exists_626b}"
    )

    if not exists_626b:
        missing_objects_626b.append(
            name_626b
        )


if missing_objects_626b:
    raise RuntimeError(
        "Step 6.26B cannot continue. "
        "Missing required objects: "
        + str(missing_objects_626b)
    )


# ------------------------------------------------------------
# 2. Freeze sensor list for this step
# ------------------------------------------------------------

SENSOR_QUALITY_SENSOR_COLUMNS_626B = [
    col_626b
    for col_626b in PRIMARY_INPUT_COLUMNS
    if str(col_626b).startswith(
        "sensor_"
    )
]


if len(
    SENSOR_QUALITY_SENSOR_COLUMNS_626B
) != 15:
    raise RuntimeError(
        "Expected exactly 15 primary sensor "
        "inputs, but found "
        f"{len(SENSOR_QUALITY_SENSOR_COLUMNS_626B)}."
    )


print(
    "\nPrimary sensor count :",
    len(
        SENSOR_QUALITY_SENSOR_COLUMNS_626B
    ),
)

print(
    "Primary sensors      :",
    SENSOR_QUALITY_SENSOR_COLUMNS_626B,
)


# ------------------------------------------------------------
# 3. Verify canonical DEV structure
# ------------------------------------------------------------

DEV_SENSOR_QUALITY_SOURCE_626B = (
    CANONICAL_DEV_RAW_623DD.copy(
        deep=True
    )
)


required_dev_columns_626b = (
    [
        "unit_id",
        "cycle",
    ]
    + SENSOR_QUALITY_SENSOR_COLUMNS_626B
)


missing_dev_columns_626b = [
    col_626b
    for col_626b
    in required_dev_columns_626b
    if col_626b
    not in DEV_SENSOR_QUALITY_SOURCE_626B.columns
]


if missing_dev_columns_626b:
    raise RuntimeError(
        "Canonical DEV is missing required "
        "sensor-quality columns: "
        + str(
            missing_dev_columns_626b
        )
    )


duplicate_keys_626b = int(
    DEV_SENSOR_QUALITY_SOURCE_626B
    .duplicated(
        subset=[
            "unit_id",
            "cycle",
        ]
    )
    .sum()
)


if duplicate_keys_626b != 0:
    raise RuntimeError(
        "Canonical DEV contains duplicate "
        "(unit_id, cycle) keys."
    )


DEV_SENSOR_QUALITY_SOURCE_626B = (
    DEV_SENSOR_QUALITY_SOURCE_626B
    .sort_values(
        [
            "unit_id",
            "cycle",
        ]
    )
    .reset_index(
        drop=True
    )
)


cycle_diff_626b = (
    DEV_SENSOR_QUALITY_SOURCE_626B
    .groupby(
        "unit_id",
        sort=False,
    )[
        "cycle"
    ]
    .diff()
)


non_contiguous_626b = int(
    (
        cycle_diff_626b
        .dropna()
        != 1
    )
    .sum()
)


if non_contiguous_626b != 0:
    raise RuntimeError(
        "Canonical DEV contains non-contiguous "
        "cycle sequences. Count = "
        f"{non_contiguous_626b}"
    )


sensor_values_626b = (
    DEV_SENSOR_QUALITY_SOURCE_626B[
        SENSOR_QUALITY_SENSOR_COLUMNS_626B
    ]
    .astype(
        np.float64
    )
)


all_sensor_values_finite_626b = bool(
    np.isfinite(
        sensor_values_626b.to_numpy()
    ).all()
)


if not all_sensor_values_finite_626b:
    raise RuntimeError(
        "Canonical DEV contains non-finite "
        "primary sensor values."
    )


print(
    "\n--- Canonical DEV structure ---"
)

print(
    "Rows                   :",
    len(
        DEV_SENSOR_QUALITY_SOURCE_626B
    ),
)

print(
    "Engines                :",
    DEV_SENSOR_QUALITY_SOURCE_626B[
        "unit_id"
    ].nunique(),
)

print(
    "Duplicate keys         :",
    duplicate_keys_626b,
)

print(
    "Non-contiguous cycles  :",
    non_contiguous_626b,
)

print(
    "All sensor values finite:",
    all_sensor_values_finite_626b,
)


# ------------------------------------------------------------
# 4. Lock the CONFIRMED sensor reference
#
# Important:
# - DEV only
# - population STD convention already confirmed
# - no recomputation/replacement of the frozen reference
# ------------------------------------------------------------

reference_required_fields_626b = [
    "mean",
    "std",
    "q01",
    "q99",
    "range",
]


missing_ref_sensors_626b = [
    sensor_626b
    for sensor_626b
    in SENSOR_QUALITY_SENSOR_COLUMNS_626B
    if sensor_626b
    not in sensor_reference_stats.index
]


if missing_ref_sensors_626b:
    raise RuntimeError(
        "Frozen sensor reference is missing "
        "primary sensors: "
        + str(
            missing_ref_sensors_626b
        )
    )


missing_ref_fields_626b = [
    field_626b
    for field_626b
    in reference_required_fields_626b
    if field_626b
    not in sensor_reference_stats.columns
]


if missing_ref_fields_626b:
    raise RuntimeError(
        "Frozen sensor reference is missing "
        "required fields: "
        + str(
            missing_ref_fields_626b
        )
    )


SENSOR_QUALITY_REFERENCE_626B = (
    sensor_reference_stats
    .loc[
        SENSOR_QUALITY_SENSOR_COLUMNS_626B,
        reference_required_fields_626b,
    ]
    .astype(
        np.float64
    )
    .copy(
        deep=True
    )
)


reference_values_finite_626b = bool(
    np.isfinite(
        SENSOR_QUALITY_REFERENCE_626B
        .to_numpy()
    ).all()
)


positive_reference_std_626b = bool(
    (
        SENSOR_QUALITY_REFERENCE_626B[
            "std"
        ]
        > 0
    ).all()
)


positive_reference_range_626b = bool(
    (
        SENSOR_QUALITY_REFERENCE_626B[
            "range"
        ]
        > 0
    ).all()
)


ordered_quantiles_626b = bool(
    (
        SENSOR_QUALITY_REFERENCE_626B[
            "q01"
        ]
        <= SENSOR_QUALITY_REFERENCE_626B[
            "q99"
        ]
    ).all()
)


if not (
    reference_values_finite_626b
    and positive_reference_std_626b
    and positive_reference_range_626b
    and ordered_quantiles_626b
):
    raise RuntimeError(
        "Frozen sensor reference failed "
        "quality checks."
    )


print(
    "\n--- Frozen sensor reference ---"
)

print(
    "Reference sensors      :",
    len(
        SENSOR_QUALITY_REFERENCE_626B
    ),
)

print(
    "All values finite      :",
    reference_values_finite_626b,
)

print(
    "All STD > 0            :",
    positive_reference_std_626b,
)

print(
    "All ranges > 0         :",
    positive_reference_range_626b,
)

print(
    "q01 <= q99             :",
    ordered_quantiles_626b,
)


# ------------------------------------------------------------
# 5. Reusable quality-feature function
#
# This creates UNSUPERVISED sensor-quality descriptors.
#
# It does NOT use:
# - RUL labels
# - prediction error
# - official TEST labels
#
# Larger values generally indicate more suspicious data.
# ------------------------------------------------------------

def build_sensor_quality_candidates_626b(
    input_df,
    sensor_columns,
    reference_stats,
    window_length=30,
):
    """
    Build interpretable sensor-quality candidate features.

    Components
    ----------
    1. Level deviation
       Absolute standardized distance from DEV mean.

    2. Reference-envelope excursion
       Magnitude outside the DEV q01-q99 interval.

    3. Temporal jump
       Absolute cycle-to-cycle change in DEV-STD units.

    4. Repeat / flatness
       Fraction of sensors showing effectively no
       cycle-to-cycle change.

    Window features are computed over the trailing
    model sequence length.

    No target labels are used.
    """

    work_626b = (
        input_df.copy(
            deep=True
        )
        .sort_values(
            [
                "unit_id",
                "cycle",
            ]
        )
        .reset_index(
            drop=True
        )
    )


    # --------------------------------------------------------
    # Structural verification
    # --------------------------------------------------------

    required_columns_local_626b = (
        [
            "unit_id",
            "cycle",
        ]
        + list(
            sensor_columns
        )
    )


    missing_local_626b = [
        col_626b
        for col_626b
        in required_columns_local_626b
        if col_626b
        not in work_626b.columns
    ]


    if missing_local_626b:
        raise ValueError(
            "Input DataFrame is missing "
            "required columns: "
            + str(
                missing_local_626b
            )
        )


    if (
        work_626b
        .duplicated(
            subset=[
                "unit_id",
                "cycle",
            ]
        )
        .any()
    ):
        raise ValueError(
            "Input contains duplicate "
            "(unit_id, cycle) keys."
        )


    cycle_gap_local_626b = (
        work_626b
        .groupby(
            "unit_id",
            sort=False,
        )[
            "cycle"
        ]
        .diff()
    )


    if (
        cycle_gap_local_626b
        .dropna()
        .ne(
            1
        )
        .any()
    ):
        raise ValueError(
            "Input contains non-contiguous "
            "engine cycles."
        )


    # --------------------------------------------------------
    # Numeric matrices
    # --------------------------------------------------------

    x_626b = (
        work_626b[
            sensor_columns
        ]
        .astype(
            np.float64
        )
    )


    if not np.isfinite(
        x_626b.to_numpy()
    ).all():
        raise ValueError(
            "Input sensor values contain "
            "NaN or infinity."
        )


    ref_626b = (
        reference_stats
        .loc[
            sensor_columns
        ]
    )


    ref_mean_626b = (
        ref_626b[
            "mean"
        ]
        .to_numpy(
            dtype=np.float64
        )
    )

    ref_std_626b = (
        ref_626b[
            "std"
        ]
        .to_numpy(
            dtype=np.float64
        )
    )

    ref_q01_626b = (
        ref_626b[
            "q01"
        ]
        .to_numpy(
            dtype=np.float64
        )
    )

    ref_q99_626b = (
        ref_626b[
            "q99"
        ]
        .to_numpy(
            dtype=np.float64
        )
    )


    if np.any(
        ref_std_626b <= 0
    ):
        raise ValueError(
            "Reference STD must be positive "
            "for every sensor."
        )


    x_array_626b = (
        x_626b.to_numpy(
            dtype=np.float64
        )
    )


    # --------------------------------------------------------
    # Component A:
    # absolute standardized level deviation
    # --------------------------------------------------------

    level_z_626b = (
        np.abs(
            x_array_626b
            - ref_mean_626b
        )
        / ref_std_626b
    )


    # --------------------------------------------------------
    # Component B:
    # q01-q99 envelope / excursion
    # --------------------------------------------------------

    below_q01_626b = (
        x_array_626b
        < ref_q01_626b
    )

    above_q99_626b = (
        x_array_626b
        > ref_q99_626b
    )

    outside_ref_626b = (
        below_q01_626b
        | above_q99_626b
    )


    lower_excursion_626b = (
        ref_q01_626b
        - x_array_626b
    )

    upper_excursion_626b = (
        x_array_626b
        - ref_q99_626b
    )


    excursion_626b = np.maximum(
        np.maximum(
            lower_excursion_626b,
            upper_excursion_626b,
        ),
        0.0,
    )


    excursion_std_units_626b = (
        excursion_626b
        / ref_std_626b
    )


    # --------------------------------------------------------
    # Component C:
    # temporal cycle-to-cycle jump
    # --------------------------------------------------------

    diff_df_626b = (
        work_626b
        .groupby(
            "unit_id",
            sort=False,
        )[
            sensor_columns
        ]
        .diff()
    )


    # First observation of every engine has no previous
    # cycle. For anomaly purposes it contributes zero jump.
    diff_array_626b = (
        diff_df_626b
        .fillna(
            0.0
        )
        .to_numpy(
            dtype=np.float64
        )
    )


    jump_z_626b = (
        np.abs(
            diff_array_626b
        )
        / ref_std_626b
    )


    # --------------------------------------------------------
    # Component D:
    # repeated / flat sensor behaviour
    #
    # Very small tolerance is used here only to account for
    # floating-point representation. We are NOT declaring
    # this a final stuck-sensor threshold.
    # --------------------------------------------------------

    repeat_tolerance_626b = np.maximum(
        ref_std_626b
        * 1e-8,
        1e-12,
    )


    repeated_value_626b = (
        np.abs(
            diff_array_626b
        )
        <= repeat_tolerance_626b
    )


    # The first row of each engine has no temporal
    # predecessor, so it must NOT count as a repeat.
    first_cycle_mask_626b = (
        work_626b
        .groupby(
            "unit_id",
            sort=False,
        )
        .cumcount()
        .eq(
            0
        )
        .to_numpy()
    )


    repeated_value_626b[
        first_cycle_mask_626b,
        :
    ] = False


    # --------------------------------------------------------
    # Row-level candidate features
    # --------------------------------------------------------

    features_626b = work_626b[
        [
            "unit_id",
            "cycle",
        ]
    ].copy()


    features_626b[
        "sq_level_mean_z"
    ] = np.mean(
        level_z_626b,
        axis=1,
    )


    features_626b[
        "sq_level_max_z"
    ] = np.max(
        level_z_626b,
        axis=1,
    )


    features_626b[
        "sq_outside_q01_q99_fraction"
    ] = np.mean(
        outside_ref_626b,
        axis=1,
    )


    features_626b[
        "sq_excursion_mean_std_units"
    ] = np.mean(
        excursion_std_units_626b,
        axis=1,
    )


    features_626b[
        "sq_excursion_max_std_units"
    ] = np.max(
        excursion_std_units_626b,
        axis=1,
    )


    features_626b[
        "sq_jump_mean_z"
    ] = np.mean(
        jump_z_626b,
        axis=1,
    )


    features_626b[
        "sq_jump_max_z"
    ] = np.max(
        jump_z_626b,
        axis=1,
    )


    features_626b[
        "sq_repeat_fraction"
    ] = np.mean(
        repeated_value_626b,
        axis=1,
    )


    # --------------------------------------------------------
    # Trailing window features aligned with the model's
    # 30-cycle sequence convention
    # --------------------------------------------------------

    group_626b = (
        features_626b
        .groupby(
            "unit_id",
            sort=False,
        )
    )


    features_626b[
        "sq_w_level_mean_z"
    ] = (
        group_626b[
            "sq_level_mean_z"
        ]
        .transform(
            lambda s_626b:
            s_626b
            .rolling(
                window_length,
                min_periods=window_length,
            )
            .mean()
        )
    )


    features_626b[
        "sq_w_level_peak_z"
    ] = (
        group_626b[
            "sq_level_max_z"
        ]
        .transform(
            lambda s_626b:
            s_626b
            .rolling(
                window_length,
                min_periods=window_length,
            )
            .max()
        )
    )


    features_626b[
        "sq_w_outside_fraction"
    ] = (
        group_626b[
            "sq_outside_q01_q99_fraction"
        ]
        .transform(
            lambda s_626b:
            s_626b
            .rolling(
                window_length,
                min_periods=window_length,
            )
            .mean()
        )
    )


    features_626b[
        "sq_w_excursion_mean"
    ] = (
        group_626b[
            "sq_excursion_mean_std_units"
        ]
        .transform(
            lambda s_626b:
            s_626b
            .rolling(
                window_length,
                min_periods=window_length,
            )
            .mean()
        )
    )


    features_626b[
        "sq_w_excursion_peak"
    ] = (
        group_626b[
            "sq_excursion_max_std_units"
        ]
        .transform(
            lambda s_626b:
            s_626b
            .rolling(
                window_length,
                min_periods=window_length,
            )
            .max()
        )
    )


    features_626b[
        "sq_w_jump_mean_z"
    ] = (
        group_626b[
            "sq_jump_mean_z"
        ]
        .transform(
            lambda s_626b:
            s_626b
            .rolling(
                window_length,
                min_periods=window_length,
            )
            .mean()
        )
    )


    features_626b[
        "sq_w_jump_peak_z"
    ] = (
        group_626b[
            "sq_jump_max_z"
        ]
        .transform(
            lambda s_626b:
            s_626b
            .rolling(
                window_length,
                min_periods=window_length,
            )
            .max()
        )
    )


    features_626b[
        "sq_w_repeat_fraction"
    ] = (
        group_626b[
            "sq_repeat_fraction"
        ]
        .transform(
            lambda s_626b:
            s_626b
            .rolling(
                window_length,
                min_periods=window_length,
            )
            .mean()
        )
    )


    window_feature_columns_626b = [
        "sq_w_level_mean_z",
        "sq_w_level_peak_z",
        "sq_w_outside_fraction",
        "sq_w_excursion_mean",
        "sq_w_excursion_peak",
        "sq_w_jump_mean_z",
        "sq_w_jump_peak_z",
        "sq_w_repeat_fraction",
    ]


    complete_window_mask_626b = (
        features_626b[
            window_feature_columns_626b
        ]
        .notna()
        .all(
            axis=1
        )
    )


    window_features_626b = (
        features_626b
        .loc[
            complete_window_mask_626b,
            [
                "unit_id",
                "cycle",
            ]
            + window_feature_columns_626b
        ]
        .reset_index(
            drop=True
        )
    )


    return (
        features_626b,
        window_features_626b,
    )


# ------------------------------------------------------------
# 6. Build CLEAN DEV quality candidates
# ------------------------------------------------------------

SENSOR_QUALITY_WINDOW_LENGTH_626B = 30


(
    SENSOR_QUALITY_DEV_ROWS_626B,
    SENSOR_QUALITY_DEV_WINDOWS_626B,
) = build_sensor_quality_candidates_626b(
    input_df=DEV_SENSOR_QUALITY_SOURCE_626B,
    sensor_columns=SENSOR_QUALITY_SENSOR_COLUMNS_626B,
    reference_stats=SENSOR_QUALITY_REFERENCE_626B,
    window_length=SENSOR_QUALITY_WINDOW_LENGTH_626B,
)


# ------------------------------------------------------------
# 7. Candidate clean-DEV distribution summary
#
# These are DESCRIPTIVE quantiles only.
# They are NOT final thresholds.
# ------------------------------------------------------------

SENSOR_QUALITY_WINDOW_FEATURES_626B = [
    "sq_w_level_mean_z",
    "sq_w_level_peak_z",
    "sq_w_outside_fraction",
    "sq_w_excursion_mean",
    "sq_w_excursion_peak",
    "sq_w_jump_mean_z",
    "sq_w_jump_peak_z",
    "sq_w_repeat_fraction",
]


quality_quantiles_626b = (
    SENSOR_QUALITY_DEV_WINDOWS_626B[
        SENSOR_QUALITY_WINDOW_FEATURES_626B
    ]
    .quantile(
        [
            0.50,
            0.90,
            0.95,
            0.99,
        ]
    )
    .T
)


quality_quantiles_626b.columns = [
    "q50",
    "q90",
    "q95",
    "q99",
]


print(
    "\n--- CLEAN DEV candidate-feature "
    "distribution ---"
)

display(
    quality_quantiles_626b
    .round(
        6
    )
)


# ------------------------------------------------------------
# 8. Feature correlation diagnostic
#
# We do NOT remove anything here.
# This is only to understand redundancy.
# ------------------------------------------------------------

quality_corr_626b = (
    SENSOR_QUALITY_DEV_WINDOWS_626B[
        SENSOR_QUALITY_WINDOW_FEATURES_626B
    ]
    .corr()
)


max_offdiag_corr_626b = (
    quality_corr_626b
    .where(
        ~np.eye(
            len(
                quality_corr_626b
            ),
            dtype=bool,
        )
    )
    .abs()
    .max()
    .max()
)


# ------------------------------------------------------------
# 9. Integrity checks
# ------------------------------------------------------------

row_features_finite_626b = bool(
    np.isfinite(
        SENSOR_QUALITY_DEV_ROWS_626B[
            [
                "sq_level_mean_z",
                "sq_level_max_z",
                "sq_outside_q01_q99_fraction",
                "sq_excursion_mean_std_units",
                "sq_excursion_max_std_units",
                "sq_jump_mean_z",
                "sq_jump_max_z",
                "sq_repeat_fraction",
            ]
        ]
        .to_numpy()
    ).all()
)


window_features_finite_626b = bool(
    np.isfinite(
        SENSOR_QUALITY_DEV_WINDOWS_626B[
            SENSOR_QUALITY_WINDOW_FEATURES_626B
        ]
        .to_numpy()
    ).all()
)


expected_window_rows_626b = int(
    (
        DEV_SENSOR_QUALITY_SOURCE_626B
        .groupby(
            "unit_id"
        )
        .size()
        .sub(
            SENSOR_QUALITY_WINDOW_LENGTH_626B
            - 1
        )
        .clip(
            lower=0
        )
        .sum()
    )
)


window_count_exact_626b = bool(
    len(
        SENSOR_QUALITY_DEV_WINDOWS_626B
    )
    == expected_window_rows_626b
)


source_unchanged_626b = bool(
    CANONICAL_DEV_RAW_623DD.equals(
        CANONICAL_DEV_RAW_623DD.copy(
            deep=True
        )
    )
)


if not row_features_finite_626b:
    raise RuntimeError(
        "Non-finite row-level quality "
        "features detected."
    )


if not window_features_finite_626b:
    raise RuntimeError(
        "Non-finite window-level quality "
        "features detected."
    )


if not window_count_exact_626b:
    raise RuntimeError(
        "Window count does not match "
        "the expected 30-cycle structure."
    )


# ------------------------------------------------------------
# 10. Compact final summary
# ------------------------------------------------------------

print(
    "\n"
    "========================================================"
)

print(
    "STEP 6.26B SENSOR-QUALITY CANDIDATE SUMMARY"
)

print(
    "========================================================"
)

print(
    "Canonical DEV rows                :",
    len(
        DEV_SENSOR_QUALITY_SOURCE_626B
    ),
)

print(
    "Canonical DEV engines             :",
    DEV_SENSOR_QUALITY_SOURCE_626B[
        "unit_id"
    ].nunique(),
)

print(
    "Primary sensors                   :",
    len(
        SENSOR_QUALITY_SENSOR_COLUMNS_626B
    ),
)

print(
    "Window length                     :",
    SENSOR_QUALITY_WINDOW_LENGTH_626B,
)

print(
    "Eligible 30-cycle DEV windows     :",
    len(
        SENSOR_QUALITY_DEV_WINDOWS_626B
    ),
)

print(
    "Expected window rows              :",
    expected_window_rows_626b,
)

print(
    "Window count exact                :",
    window_count_exact_626b,
)

print(
    "Row-level features finite         :",
    row_features_finite_626b,
)

print(
    "Window-level features finite      :",
    window_features_finite_626b,
)

print(
    "Maximum candidate correlation     :",
    float(
        max_offdiag_corr_626b
    ),
)

print(
    "Reference population              : "
    "Canonical DEV"
)

print(
    "Reference STD convention          : "
    "population (ddof=0)"
)

print(
    "RUL labels used                   : False"
)

print(
    "Official TEST labels inspected   : False"
)

print(
    "Official TEST scored             : False"
)

print(
    "Final quality score created       : False"
)

print(
    "Quality threshold selected        : False"
)

print(
    "Files written                    : False"
)

print(
    "\nNo frozen research decision "
    "was modified."
)

print(
    "=== END STEP 6.26B ==="
)

In [ ]:
# ============================================================
# Step 6.26C — Full frozen-DEV corruption response of
#              sensor-quality candidates
# ============================================================

import hashlib
import numpy as np
import pandas as pd


print(
    "=== Step 6.26C — Full frozen-DEV corruption response "
    "of sensor-quality candidates ==="
)


# ------------------------------------------------------------
# 1. Required objects
# ------------------------------------------------------------

required_objects_626c = [
    "scenario_manifests_620gc",
    "apply_frozen_corruption_620gc",
    "val_raw_base_620gc",
    "SENSOR_QUALITY_SENSOR_COLUMNS_626B",
    "SENSOR_QUALITY_REFERENCE_626B",
    "SENSOR_QUALITY_WINDOW_LENGTH_626B",
    "SENSOR_QUALITY_WINDOW_FEATURES_626B",
    "SENSOR_QUALITY_DEV_WINDOWS_626B",
    "build_sensor_quality_candidates_626b",
]


print("\n--- Required objects ---")

missing_objects_626c = []

for name_626c in required_objects_626c:

    exists_626c = (
        name_626c in globals()
    )

    print(
        f"{name_626c:<40}: "
        f"{exists_626c}"
    )

    if not exists_626c:
        missing_objects_626c.append(
            name_626c
        )


if missing_objects_626c:

    raise RuntimeError(
        "Step 6.26C cannot continue. "
        "Missing required objects: "
        + str(
            missing_objects_626c
        )
    )


# ------------------------------------------------------------
# 2. Lock corruption-family order
# ------------------------------------------------------------

FROZEN_CORRUPTION_FAMILIES_626C = [
    "bias",
    "noise",
    "drift",
    "dropout",
    "stuck",
]


missing_families_626c = [
    family_626c
    for family_626c
    in FROZEN_CORRUPTION_FAMILIES_626C
    if family_626c
    not in scenario_manifests_620gc
]


if missing_families_626c:

    raise RuntimeError(
        "Frozen scenario manifests are missing "
        "families: "
        + str(
            missing_families_626c
        )
    )


manifest_counts_626c = {
    family_626c:
    int(
        len(
            scenario_manifests_620gc[
                family_626c
            ]
        )
    )
    for family_626c
    in FROZEN_CORRUPTION_FAMILIES_626C
}


total_scenarios_expected_626c = int(
    sum(
        manifest_counts_626c.values()
    )
)


print(
    "\n--- Frozen manifest counts ---"
)

for (
    family_626c,
    count_626c,
) in manifest_counts_626c.items():

    print(
        f"{family_626c:<10}: "
        f"{count_626c}"
    )


print(
    "Total     :",
    total_scenarios_expected_626c,
)


if total_scenarios_expected_626c != 315:

    raise RuntimeError(
        "Expected 315 frozen corruption scenarios, "
        f"but found {total_scenarios_expected_626c}."
    )


# ------------------------------------------------------------
# 3. Integrity snapshot of validation raw base
# ------------------------------------------------------------

VAL_SENSOR_QUALITY_BASE_626C = (
    val_raw_base_620gc.copy(
        deep=True
    )
)


required_val_columns_626c = (
    [
        "unit_id",
        "cycle",
    ]
    + list(
        SENSOR_QUALITY_SENSOR_COLUMNS_626B
    )
)


missing_val_columns_626c = [
    col_626c
    for col_626c
    in required_val_columns_626c
    if col_626c
    not in VAL_SENSOR_QUALITY_BASE_626C.columns
]


if missing_val_columns_626c:

    raise RuntimeError(
        "Validation corruption base is missing "
        "required columns: "
        + str(
            missing_val_columns_626c
        )
    )


def dataframe_sha256_626c(
    df_626c
):

    hashed_626c = (
        pd.util
        .hash_pandas_object(
            df_626c,
            index=True,
        )
        .to_numpy()
        .tobytes()
    )

    return hashlib.sha256(
        hashed_626c
    ).hexdigest()


VAL_RAW_HASH_BEFORE_626C = (
    dataframe_sha256_626c(
        val_raw_base_620gc
    )
)


duplicate_val_keys_626c = int(
    VAL_SENSOR_QUALITY_BASE_626C
    .duplicated(
        subset=[
            "unit_id",
            "cycle",
        ]
    )
    .sum()
)


if duplicate_val_keys_626c != 0:

    raise RuntimeError(
        "Validation corruption base contains "
        "duplicate (unit_id, cycle) keys."
    )


VAL_SENSOR_QUALITY_BASE_626C = (
    VAL_SENSOR_QUALITY_BASE_626C
    .sort_values(
        [
            "unit_id",
            "cycle",
        ]
    )
    .reset_index(
        drop=True
    )
)


cycle_diff_626c = (
    VAL_SENSOR_QUALITY_BASE_626C
    .groupby(
        "unit_id",
        sort=False,
    )[
        "cycle"
    ]
    .diff()
)


non_contiguous_val_626c = int(
    (
        cycle_diff_626c
        .dropna()
        != 1
    )
    .sum()
)


if non_contiguous_val_626c != 0:

    raise RuntimeError(
        "Validation corruption base contains "
        "non-contiguous engine cycles."
    )


val_numeric_finite_626c = bool(
    np.isfinite(
        VAL_SENSOR_QUALITY_BASE_626C[
            SENSOR_QUALITY_SENSOR_COLUMNS_626B
        ]
        .to_numpy(
            dtype=np.float64
        )
    ).all()
)


if not val_numeric_finite_626c:

    raise RuntimeError(
        "Validation corruption base contains "
        "non-finite primary sensor values."
    )


print(
    "\n--- Validation corruption base ---"
)

print(
    "Rows             :",
    len(
        VAL_SENSOR_QUALITY_BASE_626C
    ),
)

print(
    "Engines          :",
    VAL_SENSOR_QUALITY_BASE_626C[
        "unit_id"
    ].nunique(),
)

print(
    "Duplicate keys   :",
    duplicate_val_keys_626c,
)

print(
    "Cycle gaps       :",
    non_contiguous_val_626c,
)

print(
    "Sensor values finite:",
    val_numeric_finite_626c,
)


# ------------------------------------------------------------
# 4. Build clean validation quality features
# ------------------------------------------------------------

(
    SENSOR_QUALITY_VAL_CLEAN_ROWS_626C,
    SENSOR_QUALITY_VAL_CLEAN_WINDOWS_626C,
) = build_sensor_quality_candidates_626b(
    input_df=VAL_SENSOR_QUALITY_BASE_626C,
    sensor_columns=SENSOR_QUALITY_SENSOR_COLUMNS_626B,
    reference_stats=SENSOR_QUALITY_REFERENCE_626B,
    window_length=SENSOR_QUALITY_WINDOW_LENGTH_626B,
)


clean_window_keys_626c = (
    SENSOR_QUALITY_VAL_CLEAN_WINDOWS_626C[
        [
            "unit_id",
            "cycle",
        ]
    ]
    .copy()
)


if (
    clean_window_keys_626c
    .duplicated()
    .any()
):

    raise RuntimeError(
        "Clean validation quality windows contain "
        "duplicate endpoint keys."
    )


print(
    "\n--- CLEAN validation quality windows ---"
)

print(
    "Window rows       :",
    len(
        SENSOR_QUALITY_VAL_CLEAN_WINDOWS_626C
    ),
)

print(
    "Window engines    :",
    SENSOR_QUALITY_VAL_CLEAN_WINDOWS_626C[
        "unit_id"
    ].nunique(),
)

print(
    "All features finite:",
    bool(
        np.isfinite(
            SENSOR_QUALITY_VAL_CLEAN_WINDOWS_626C[
                SENSOR_QUALITY_WINDOW_FEATURES_626B
            ]
            .to_numpy()
        ).all()
    ),
)


# ------------------------------------------------------------
# 5. Clean-DEV scale for cross-feature comparison
#
# This is NOT a threshold.
# It simply puts candidate deltas on comparable scales.
# ------------------------------------------------------------

SENSOR_QUALITY_FEATURE_SCALE_626C = (
    SENSOR_QUALITY_DEV_WINDOWS_626B[
        SENSOR_QUALITY_WINDOW_FEATURES_626B
    ]
    .std(
        ddof=0
    )
    .astype(
        np.float64
    )
)


invalid_scale_features_626c = (
    SENSOR_QUALITY_FEATURE_SCALE_626C[
        (
            ~np.isfinite(
                SENSOR_QUALITY_FEATURE_SCALE_626C
            )
        )
        |
        (
            SENSOR_QUALITY_FEATURE_SCALE_626C
            <= 0
        )
    ]
    .index
    .tolist()
)


if invalid_scale_features_626c:

    raise RuntimeError(
        "Invalid clean-DEV feature scales for: "
        + str(
            invalid_scale_features_626c
        )
    )


print(
    "\n--- Clean-DEV candidate scales ---"
)

display(
    SENSOR_QUALITY_FEATURE_SCALE_626C
    .rename(
        "population_std"
    )
    .to_frame()
    .round(
        6
    )
)


# ------------------------------------------------------------
# 6. Helpers
# ------------------------------------------------------------

def scenario_id_626c(
    family_626c,
    position_626c,
    row_626c,
):

    if (
        "scenario_id"
        in row_626c.index
    ):

        value_626c = (
            row_626c[
                "scenario_id"
            ]
        )

        if pd.notna(
            value_626c
        ):

            return str(
                value_626c
            )

    return (
        f"{family_626c}_"
        f"{position_626c + 1:03d}"
    )


def safe_value_626c(
    row_626c,
    column_626c,
    default_626c=None,
):

    if (
        column_626c
        in row_626c.index
    ):

        value_626c = (
            row_626c[
                column_626c
            ]
        )

        if pd.notna(
            value_626c
        ):

            return value_626c

    return default_626c


# ------------------------------------------------------------
# 7. Evaluate all 315 frozen corruption scenarios
# ------------------------------------------------------------

scenario_feature_records_626c = []

scenarios_completed_626c = 0

structural_failures_626c = []

quality_failures_626c = []


print(
    "\n--- Evaluating 315 frozen DEV corruption scenarios ---"
)


for family_626c in FROZEN_CORRUPTION_FAMILIES_626C:

    manifest_626c = (
        scenario_manifests_620gc[
            family_626c
        ]
        .reset_index(
            drop=True
        )
    )

    print(
        f"\nRunning {family_626c.upper()} "
        f"({len(manifest_626c)} scenarios)..."
    )


    for position_626c, row_626c in (
        manifest_626c.iterrows()
    ):

        scenario_id_value_626c = (
            scenario_id_626c(
                family_626c,
                position_626c,
                row_626c,
            )
        )

        sensor_value_626c = str(
            safe_value_626c(
                row_626c,
                "sensor",
                "UNKNOWN",
            )
        )

        severity_value_626c = str(
            safe_value_626c(
                row_626c,
                "severity",
                "UNKNOWN",
            )
        )


        # ----------------------------------------------------
        # Apply exact already-frozen corruption
        # ----------------------------------------------------

        corrupted_raw_626c = (
            apply_frozen_corruption_620gc(
                family_626c,
                row_626c,
            )
        )


        if not isinstance(
            corrupted_raw_626c,
            pd.DataFrame,
        ):

            structural_failures_626c.append(
                (
                    family_626c,
                    scenario_id_value_626c,
                    "not_dataframe",
                )
            )

            continue


        if (
            corrupted_raw_626c.shape
            != val_raw_base_620gc.shape
        ):

            structural_failures_626c.append(
                (
                    family_626c,
                    scenario_id_value_626c,
                    "shape_mismatch",
                )
            )

            continue


        if (
            list(
                corrupted_raw_626c.columns
            )
            != list(
                val_raw_base_620gc.columns
            )
        ):

            structural_failures_626c.append(
                (
                    family_626c,
                    scenario_id_value_626c,
                    "column_mismatch",
                )
            )

            continue


        # ----------------------------------------------------
        # Build identical sensor-quality candidates
        # ----------------------------------------------------

        try:

            (
                _,
                corrupted_windows_626c,
            ) = build_sensor_quality_candidates_626b(
                input_df=corrupted_raw_626c,
                sensor_columns=SENSOR_QUALITY_SENSOR_COLUMNS_626B,
                reference_stats=SENSOR_QUALITY_REFERENCE_626B,
                window_length=SENSOR_QUALITY_WINDOW_LENGTH_626B,
            )

        except Exception as exc_626c:

            quality_failures_626c.append(
                (
                    family_626c,
                    scenario_id_value_626c,
                    repr(
                        exc_626c
                    ),
                )
            )

            continue


        # ----------------------------------------------------
        # Exact clean/corrupted window alignment
        # ----------------------------------------------------

        paired_626c = (
            SENSOR_QUALITY_VAL_CLEAN_WINDOWS_626C
            .merge(
                corrupted_windows_626c,
                on=[
                    "unit_id",
                    "cycle",
                ],
                how="inner",
                suffixes=(
                    "_clean",
                    "_corrupted",
                ),
                validate="one_to_one",
            )
        )


        if (
            len(
                paired_626c
            )
            != len(
                SENSOR_QUALITY_VAL_CLEAN_WINDOWS_626C
            )
        ):

            structural_failures_626c.append(
                (
                    family_626c,
                    scenario_id_value_626c,
                    "window_alignment_mismatch",
                )
            )

            continue


        scenarios_completed_626c += 1


        # ----------------------------------------------------
        # Feature-wise response
        # ----------------------------------------------------

        for feature_626c in (
            SENSOR_QUALITY_WINDOW_FEATURES_626B
        ):

            clean_values_626c = (
                paired_626c[
                    feature_626c
                    + "_clean"
                ]
                .to_numpy(
                    dtype=np.float64
                )
            )


            corrupted_values_626c = (
                paired_626c[
                    feature_626c
                    + "_corrupted"
                ]
                .to_numpy(
                    dtype=np.float64
                )
            )


            if not (
                np.isfinite(
                    clean_values_626c
                ).all()
                and
                np.isfinite(
                    corrupted_values_626c
                ).all()
            ):

                raise RuntimeError(
                    "Non-finite quality values detected in "
                    f"{scenario_id_value_626c}, "
                    f"{feature_626c}."
                )


            delta_626c = (
                corrupted_values_626c
                - clean_values_626c
            )


            abs_delta_626c = (
                np.abs(
                    delta_626c
                )
            )


            scale_626c = float(
                SENSOR_QUALITY_FEATURE_SCALE_626C[
                    feature_626c
                ]
            )


            mean_delta_626c = float(
                np.mean(
                    delta_626c
                )
            )


            scenario_feature_records_626c.append(
                {
                    "family":
                        family_626c,

                    "family_position":
                        int(
                            position_626c
                        ),

                    "scenario_id":
                        scenario_id_value_626c,

                    "sensor":
                        sensor_value_626c,

                    "severity":
                        severity_value_626c,

                    "feature":
                        feature_626c,

                    "windows":
                        int(
                            len(
                                paired_626c
                            )
                        ),

                    "clean_mean":
                        float(
                            np.mean(
                                clean_values_626c
                            )
                        ),

                    "corrupted_mean":
                        float(
                            np.mean(
                                corrupted_values_626c
                            )
                        ),

                    "mean_delta":
                        mean_delta_626c,

                    "median_delta":
                        float(
                            np.median(
                                delta_626c
                            )
                        ),

                    "mean_abs_delta":
                        float(
                            np.mean(
                                abs_delta_626c
                            )
                        ),

                    "max_abs_delta":
                        float(
                            np.max(
                                abs_delta_626c
                            )
                        ),

                    "fraction_increased":
                        float(
                            np.mean(
                                delta_626c
                                > 0
                            )
                        ),

                    "fraction_decreased":
                        float(
                            np.mean(
                                delta_626c
                                < 0
                            )
                        ),

                    "clean_q95":
                        float(
                            np.quantile(
                                clean_values_626c,
                                0.95,
                            )
                        ),

                    "corrupted_q95":
                        float(
                            np.quantile(
                                corrupted_values_626c,
                                0.95,
                            )
                        ),

                    "clean_dev_scale":
                        scale_626c,

                    "standardized_mean_delta":
                        float(
                            mean_delta_626c
                            / scale_626c
                        ),

                    "standardized_mean_abs_delta":
                        float(
                            np.mean(
                                abs_delta_626c
                            )
                            / scale_626c
                        ),
                }
            )


# ------------------------------------------------------------
# 8. Materialize detailed scenario × feature table
# ------------------------------------------------------------

SENSOR_QUALITY_SCENARIO_FEATURE_EFFECTS_626C = (
    pd.DataFrame(
        scenario_feature_records_626c
    )
)


expected_feature_records_626c = int(
    315
    * len(
        SENSOR_QUALITY_WINDOW_FEATURES_626B
    )
)


actual_feature_records_626c = int(
    len(
        SENSOR_QUALITY_SCENARIO_FEATURE_EFFECTS_626C
    )
)


# ------------------------------------------------------------
# 9. Family × feature summary
# ------------------------------------------------------------

summary_source_626c = (
    SENSOR_QUALITY_SCENARIO_FEATURE_EFFECTS_626C
    .copy()
)


summary_source_626c[
    "positive_mean_delta"
] = (
    summary_source_626c[
        "mean_delta"
    ]
    > 0
)


summary_source_626c[
    "nonzero_mean_delta"
] = (
    np.abs(
        summary_source_626c[
            "mean_delta"
        ]
    )
    > 1e-12
)


SENSOR_QUALITY_FAMILY_FEATURE_SUMMARY_626C = (
    summary_source_626c
    .groupby(
        [
            "family",
            "feature",
        ],
        sort=False,
    )
    .agg(
        scenarios=(
            "scenario_id",
            "count",
        ),

        mean_standardized_delta=(
            "standardized_mean_delta",
            "mean",
        ),

        median_standardized_delta=(
            "standardized_mean_delta",
            "median",
        ),

        mean_standardized_abs_delta=(
            "standardized_mean_abs_delta",
            "mean",
        ),

        positive_scenario_fraction=(
            "positive_mean_delta",
            "mean",
        ),

        nonzero_scenario_fraction=(
            "nonzero_mean_delta",
            "mean",
        ),

        mean_window_increase_fraction=(
            "fraction_increased",
            "mean",
        ),

        mean_window_decrease_fraction=(
            "fraction_decreased",
            "mean",
        ),
    )
    .reset_index()
)


# ------------------------------------------------------------
# 10. Compact response matrices
# ------------------------------------------------------------

SENSOR_QUALITY_STD_DELTA_MATRIX_626C = (
    SENSOR_QUALITY_FAMILY_FEATURE_SUMMARY_626C
    .pivot(
        index="family",
        columns="feature",
        values="mean_standardized_delta",
    )
    .reindex(
        FROZEN_CORRUPTION_FAMILIES_626C
    )
)


SENSOR_QUALITY_POSITIVE_FRACTION_MATRIX_626C = (
    SENSOR_QUALITY_FAMILY_FEATURE_SUMMARY_626C
    .pivot(
        index="family",
        columns="feature",
        values="positive_scenario_fraction",
    )
    .reindex(
        FROZEN_CORRUPTION_FAMILIES_626C
    )
)


SENSOR_QUALITY_ABS_RESPONSE_MATRIX_626C = (
    SENSOR_QUALITY_FAMILY_FEATURE_SUMMARY_626C
    .pivot(
        index="family",
        columns="feature",
        values="mean_standardized_abs_delta",
    )
    .reindex(
        FROZEN_CORRUPTION_FAMILIES_626C
    )
)


print(
    "\n--- Mean standardized change "
    "(corrupted - clean) ---"
)

display(
    SENSOR_QUALITY_STD_DELTA_MATRIX_626C
    .round(
        4
    )
)


print(
    "\n--- Fraction of scenarios with "
    "positive mean change ---"
)

display(
    SENSOR_QUALITY_POSITIVE_FRACTION_MATRIX_626C
    .round(
        4
    )
)


print(
    "\n--- Mean standardized ABSOLUTE response ---"
)

display(
    SENSOR_QUALITY_ABS_RESPONSE_MATRIX_626C
    .round(
        4
    )
)


# ------------------------------------------------------------
# 11. Candidate overall response summary
# ------------------------------------------------------------

SENSOR_QUALITY_OVERALL_FEATURE_RESPONSE_626C = (
    summary_source_626c
    .groupby(
        "feature",
        sort=False,
    )
    .agg(
        scenarios=(
            "scenario_id",
            "count",
        ),

        mean_standardized_delta=(
            "standardized_mean_delta",
            "mean",
        ),

        median_standardized_delta=(
            "standardized_mean_delta",
            "median",
        ),

        mean_standardized_abs_delta=(
            "standardized_mean_abs_delta",
            "mean",
        ),

        positive_scenario_fraction=(
            "positive_mean_delta",
            "mean",
        ),

        nonzero_scenario_fraction=(
            "nonzero_mean_delta",
            "mean",
        ),
    )
    .reset_index()
    .sort_values(
        "mean_standardized_abs_delta",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n--- Overall candidate response ranking "
    "(diagnostic only) ---"
)

display(
    SENSOR_QUALITY_OVERALL_FEATURE_RESPONSE_626C
    .round(
        6
    )
)


# ------------------------------------------------------------
# 12. Integrity verification
# ------------------------------------------------------------

VAL_RAW_HASH_AFTER_626C = (
    dataframe_sha256_626c(
        val_raw_base_620gc
    )
)


val_raw_unchanged_626c = bool(
    VAL_RAW_HASH_BEFORE_626C
    ==
    VAL_RAW_HASH_AFTER_626C
)


all_315_complete_626c = bool(
    scenarios_completed_626c
    == 315
)


all_feature_records_complete_626c = bool(
    actual_feature_records_626c
    ==
    expected_feature_records_626c
)


all_effect_values_finite_626c = bool(
    np.isfinite(
        SENSOR_QUALITY_SCENARIO_FEATURE_EFFECTS_626C[
            [
                "clean_mean",
                "corrupted_mean",
                "mean_delta",
                "median_delta",
                "mean_abs_delta",
                "max_abs_delta",
                "fraction_increased",
                "fraction_decreased",
                "clean_q95",
                "corrupted_q95",
                "clean_dev_scale",
                "standardized_mean_delta",
                "standardized_mean_abs_delta",
            ]
        ]
        .to_numpy(
            dtype=np.float64
        )
    ).all()
)


if structural_failures_626c:

    raise RuntimeError(
        "Structural corruption-response failures "
        "were detected: "
        + str(
            structural_failures_626c[:10]
        )
    )


if quality_failures_626c:

    raise RuntimeError(
        "Sensor-quality feature construction "
        "failures were detected: "
        + str(
            quality_failures_626c[:10]
        )
    )


if not all_315_complete_626c:

    raise RuntimeError(
        "Not all 315 frozen scenarios completed."
    )


if not all_feature_records_complete_626c:

    raise RuntimeError(
        "Scenario × feature record count mismatch. "
        f"Expected {expected_feature_records_626c}, "
        f"found {actual_feature_records_626c}."
    )


if not all_effect_values_finite_626c:

    raise RuntimeError(
        "Non-finite scenario-response statistics "
        "were detected."
    )


if not val_raw_unchanged_626c:

    raise RuntimeError(
        "Validation raw base changed during "
        "Step 6.26C."
    )


# ------------------------------------------------------------
# 13. Compact final summary
# ------------------------------------------------------------

print(
    "\n"
    "============================================================"
)

print(
    "STEP 6.26C FINAL SENSOR-QUALITY RESPONSE SUMMARY"
)

print(
    "============================================================"
)

print(
    "Frozen corruption scenarios expected :",
    total_scenarios_expected_626c,
)

print(
    "Frozen corruption scenarios completed:",
    scenarios_completed_626c,
)

print(
    "Candidate quality features            :",
    len(
        SENSOR_QUALITY_WINDOW_FEATURES_626B
    ),
)

print(
    "Expected scenario-feature records     :",
    expected_feature_records_626c,
)

print(
    "Actual scenario-feature records       :",
    actual_feature_records_626c,
)

print(
    "All 315 scenarios completed           :",
    all_315_complete_626c,
)

print(
    "All feature records complete          :",
    all_feature_records_complete_626c,
)

print(
    "All response statistics finite        :",
    all_effect_values_finite_626c,
)

print(
    "Validation raw base unchanged         :",
    val_raw_unchanged_626c,
)

print(
    "Structural failures                   :",
    len(
        structural_failures_626c
    ),
)

print(
    "Quality-construction failures         :",
    len(
        quality_failures_626c
    ),
)

print(
    "\nReference population                  : "
    "Canonical DEV"
)

print(
    "Reference STD convention              : "
    "population (ddof=0)"
)

print(
    "RUL labels used                       : False"
)

print(
    "Official TEST labels inspected        : False"
)

print(
    "Official TEST scored                  : False"
)

print(
    "Model inference executed              : False"
)

print(
    "MC-dropout executed                   : False"
)

print(
    "Final sensor-quality score frozen     : False"
)

print(
    "Sensor-quality threshold selected     : False"
)

print(
    "Frozen reliability decision modified  : False"
)

print(
    "Files written                         : False"
)

print(
    "\nNo frozen research decision was modified."
)

print(
    "=== END STEP 6.26C ==="
)

In [ ]:
# ============================================================
# Step 6.26D — DEV-only selection and empirical calibration
#              of sensor-quality components
# ============================================================

import numpy as np
import pandas as pd


print(
    "=== Step 6.26D — DEV-only sensor-quality component "
    "selection and calibration ==="
)


# ------------------------------------------------------------
# 1. Required objects
# ------------------------------------------------------------

required_objects_626d = [
    "SENSOR_QUALITY_DEV_WINDOWS_626B",
    "SENSOR_QUALITY_VAL_CLEAN_WINDOWS_626C",
    "SENSOR_QUALITY_WINDOW_FEATURES_626B",
    "SENSOR_QUALITY_ABS_RESPONSE_MATRIX_626C",
    "SENSOR_QUALITY_STD_DELTA_MATRIX_626C",
    "SENSOR_QUALITY_OVERALL_FEATURE_RESPONSE_626C",
]


print("\n--- Required objects ---")

missing_objects_626d = []

for name_626d in required_objects_626d:

    exists_626d = (
        name_626d in globals()
    )

    print(
        f"{name_626d:<45}: "
        f"{exists_626d}"
    )

    if not exists_626d:
        missing_objects_626d.append(
            name_626d
        )


if missing_objects_626d:

    raise RuntimeError(
        "Step 6.26D cannot continue. "
        "Missing required objects: "
        + str(
            missing_objects_626d
        )
    )


# ------------------------------------------------------------
# 2. Freeze component SELECTION for calibration analysis
#
# IMPORTANT:
# This does NOT freeze the final sensor-quality score.
# It only freezes which three candidate channels will be
# calibrated and compared in the next stage.
# ------------------------------------------------------------

SENSOR_QUALITY_SELECTED_FEATURES_626D = [
    "sq_w_excursion_mean",
    "sq_w_jump_mean_z",
    "sq_w_repeat_fraction",
]


missing_selected_626d = [
    feature_626d
    for feature_626d
    in SENSOR_QUALITY_SELECTED_FEATURES_626D
    if feature_626d
    not in SENSOR_QUALITY_WINDOW_FEATURES_626B
]


if missing_selected_626d:

    raise RuntimeError(
        "Selected sensor-quality features are missing: "
        + str(
            missing_selected_626d
        )
    )


print(
    "\n--- Selected complementary sensor-quality channels ---"
)

for i_626d, feature_626d in enumerate(
    SENSOR_QUALITY_SELECTED_FEATURES_626D,
    start=1,
):

    print(
        f"{i_626d}. {feature_626d}"
    )


# ------------------------------------------------------------
# 3. Show Step 6.26C corruption-response evidence
# ------------------------------------------------------------

print(
    "\n--- Mean standardized ABSOLUTE corruption response ---"
)

selected_abs_response_626d = (
    SENSOR_QUALITY_ABS_RESPONSE_MATRIX_626C[
        SENSOR_QUALITY_SELECTED_FEATURES_626D
    ]
    .copy()
)

display(
    selected_abs_response_626d
    .round(
        4
    )
)


print(
    "\n--- Signed mean standardized corruption response ---"
)

selected_signed_response_626d = (
    SENSOR_QUALITY_STD_DELTA_MATRIX_626C[
        SENSOR_QUALITY_SELECTED_FEATURES_626D
    ]
    .copy()
)

display(
    selected_signed_response_626d
    .round(
        4
    )
)


# ------------------------------------------------------------
# 4. Clean-DEV correlation audit
# ------------------------------------------------------------

selected_clean_dev_626d = (
    SENSOR_QUALITY_DEV_WINDOWS_626B[
        SENSOR_QUALITY_SELECTED_FEATURES_626D
    ]
    .astype(
        np.float64
    )
    .copy()
)


if not np.isfinite(
    selected_clean_dev_626d
    .to_numpy()
).all():

    raise RuntimeError(
        "Selected clean-DEV sensor-quality features "
        "contain non-finite values."
    )


SENSOR_QUALITY_SELECTED_CORRELATION_626D = (
    selected_clean_dev_626d
    .corr()
)


print(
    "\n--- Clean-DEV selected-feature correlation ---"
)

display(
    SENSOR_QUALITY_SELECTED_CORRELATION_626D
    .round(
        6
    )
)


corr_array_626d = (
    SENSOR_QUALITY_SELECTED_CORRELATION_626D
    .to_numpy(
        dtype=np.float64
    )
)


off_diagonal_mask_626d = (
    ~np.eye(
        len(
            SENSOR_QUALITY_SELECTED_FEATURES_626D
        ),
        dtype=bool,
    )
)


maximum_selected_abs_correlation_626d = float(
    np.max(
        np.abs(
            corr_array_626d[
                off_diagonal_mask_626d
            ]
        )
    )
)


print(
    "Maximum pairwise absolute correlation:",
    maximum_selected_abs_correlation_626d,
)


# ------------------------------------------------------------
# 5. Build empirical two-sided clean-DEV calibrators
#
# For each selected feature:
#
#   1. find clean DEV median
#   2. measure |feature - median|
#   3. retain sorted clean absolute deviations
#
# A future observation is scored according to how extreme
# its absolute deviation is relative to clean DEV.
#
# Therefore:
#   low empirical score  = typical clean behaviour
#   high empirical score = unusual relative to clean DEV
#
# No corruption labels and no RUL labels are used here.
# ------------------------------------------------------------

SENSOR_QUALITY_CALIBRATORS_626D = {}


for feature_626d in (
    SENSOR_QUALITY_SELECTED_FEATURES_626D
):

    clean_values_626d = (
        SENSOR_QUALITY_DEV_WINDOWS_626B[
            feature_626d
        ]
        .to_numpy(
            dtype=np.float64
        )
    )


    center_626d = float(
        np.median(
            clean_values_626d
        )
    )


    abs_deviation_626d = np.abs(
        clean_values_626d
        - center_626d
    )


    sorted_abs_deviation_626d = np.sort(
        abs_deviation_626d
    )


    SENSOR_QUALITY_CALIBRATORS_626D[
        feature_626d
    ] = {
        "center":
            center_626d,

        "reference_abs_deviation":
            sorted_abs_deviation_626d,

        "reference_count":
            int(
                len(
                    sorted_abs_deviation_626d
                )
            ),

        "raw_q01":
            float(
                np.quantile(
                    clean_values_626d,
                    0.01,
                )
            ),

        "raw_q50":
            float(
                np.quantile(
                    clean_values_626d,
                    0.50,
                )
            ),

        "raw_q99":
            float(
                np.quantile(
                    clean_values_626d,
                    0.99,
                )
            ),

        "abs_dev_q50":
            float(
                np.quantile(
                    abs_deviation_626d,
                    0.50,
                )
            ),

        "abs_dev_q90":
            float(
                np.quantile(
                    abs_deviation_626d,
                    0.90,
                )
            ),

        "abs_dev_q95":
            float(
                np.quantile(
                    abs_deviation_626d,
                    0.95,
                )
            ),

        "abs_dev_q99":
            float(
                np.quantile(
                    abs_deviation_626d,
                    0.99,
                )
            ),
    }


calibrator_summary_records_626d = []


for feature_626d in (
    SENSOR_QUALITY_SELECTED_FEATURES_626D
):

    info_626d = (
        SENSOR_QUALITY_CALIBRATORS_626D[
            feature_626d
        ]
    )

    calibrator_summary_records_626d.append(
        {
            "feature":
                feature_626d,

            "clean_center":
                info_626d[
                    "center"
                ],

            "raw_q01":
                info_626d[
                    "raw_q01"
                ],

            "raw_q50":
                info_626d[
                    "raw_q50"
                ],

            "raw_q99":
                info_626d[
                    "raw_q99"
                ],

            "abs_dev_q50":
                info_626d[
                    "abs_dev_q50"
                ],

            "abs_dev_q90":
                info_626d[
                    "abs_dev_q90"
                ],

            "abs_dev_q95":
                info_626d[
                    "abs_dev_q95"
                ],

            "abs_dev_q99":
                info_626d[
                    "abs_dev_q99"
                ],

            "reference_count":
                info_626d[
                    "reference_count"
                ],
        }
    )


SENSOR_QUALITY_CALIBRATOR_SUMMARY_626D = (
    pd.DataFrame(
        calibrator_summary_records_626d
    )
)


print(
    "\n--- Clean-DEV two-sided calibration reference ---"
)

display(
    SENSOR_QUALITY_CALIBRATOR_SUMMARY_626D
    .round(
        6
    )
)


# ------------------------------------------------------------
# 6. Mid-rank empirical anomaly transform
#
# Ties are handled using the empirical mid-rank.
# Output remains in [0, 1].
# ------------------------------------------------------------

def empirical_abs_deviation_score_626d(
    values,
    feature,
):

    values_626d = np.asarray(
        values,
        dtype=np.float64,
    )


    if not np.isfinite(
        values_626d
    ).all():

        raise ValueError(
            f"Non-finite values supplied for {feature}."
        )


    calibrator_626d = (
        SENSOR_QUALITY_CALIBRATORS_626D[
            feature
        ]
    )


    center_626d = float(
        calibrator_626d[
            "center"
        ]
    )


    reference_626d = np.asarray(
        calibrator_626d[
            "reference_abs_deviation"
        ],
        dtype=np.float64,
    )


    observed_abs_dev_626d = np.abs(
        values_626d
        - center_626d
    )


    left_626d = np.searchsorted(
        reference_626d,
        observed_abs_dev_626d,
        side="left",
    )


    right_626d = np.searchsorted(
        reference_626d,
        observed_abs_dev_626d,
        side="right",
    )


    # Mid-rank empirical CDF.
    #
    # 0   -> extremely typical
    # 1   -> extremely unusual
    #
    scores_626d = (
        (
            left_626d
            + right_626d
        )
        / 2.0
        / float(
            len(
                reference_626d
            )
        )
    )


    return np.clip(
        scores_626d,
        0.0,
        1.0,
    )


# ------------------------------------------------------------
# 7. Window-scoring function
#
# Creates three component anomaly scores plus two provisional
# fusion candidates:
#
# mean fusion = average evidence across quality channels
# max fusion  = strongest individual anomaly evidence
#
# These are CANDIDATES only.
# Neither is frozen as the final sensor-quality score.
# ------------------------------------------------------------

def score_sensor_quality_windows_626d(
    window_df,
):

    required_626d = (
        SENSOR_QUALITY_SELECTED_FEATURES_626D
    )


    missing_626d = [
        col_626d
        for col_626d in required_626d
        if col_626d
        not in window_df.columns
    ]


    if missing_626d:

        raise ValueError(
            "Window DataFrame is missing selected "
            "sensor-quality columns: "
            + str(
                missing_626d
            )
        )


    scored_626d = (
        window_df.copy()
    )


    component_columns_626d = []


    for feature_626d in (
        SENSOR_QUALITY_SELECTED_FEATURES_626D
    ):

        component_name_626d = (
            feature_626d
            + "_anomaly"
        )


        scored_626d[
            component_name_626d
        ] = (
            empirical_abs_deviation_score_626d(
                scored_626d[
                    feature_626d
                ].to_numpy(),
                feature_626d,
            )
        )


        component_columns_626d.append(
            component_name_626d
        )


    component_matrix_626d = (
        scored_626d[
            component_columns_626d
        ]
        .to_numpy(
            dtype=np.float64
        )
    )


    scored_626d[
        "sq_candidate_mean_fusion"
    ] = np.mean(
        component_matrix_626d,
        axis=1,
    )


    scored_626d[
        "sq_candidate_max_fusion"
    ] = np.max(
        component_matrix_626d,
        axis=1,
    )


    return scored_626d


# ------------------------------------------------------------
# 8. Score CLEAN canonical DEV and CLEAN validation windows
# ------------------------------------------------------------

SENSOR_QUALITY_DEV_SCORED_626D = (
    score_sensor_quality_windows_626d(
        SENSOR_QUALITY_DEV_WINDOWS_626B
    )
)


SENSOR_QUALITY_VAL_CLEAN_SCORED_626D = (
    score_sensor_quality_windows_626d(
        SENSOR_QUALITY_VAL_CLEAN_WINDOWS_626C
    )
)


SENSOR_QUALITY_COMPONENT_COLUMNS_626D = [
    feature_626d
    + "_anomaly"
    for feature_626d
    in SENSOR_QUALITY_SELECTED_FEATURES_626D
]


SENSOR_QUALITY_FUSION_CANDIDATES_626D = [
    "sq_candidate_mean_fusion",
    "sq_candidate_max_fusion",
]


# ------------------------------------------------------------
# 9. Clean-score distribution audit
# ------------------------------------------------------------

audit_columns_626d = (
    SENSOR_QUALITY_COMPONENT_COLUMNS_626D
    +
    SENSOR_QUALITY_FUSION_CANDIDATES_626D
)


def clean_distribution_summary_626d(
    df_626d,
    population_name_626d,
):

    records_626d = []

    for column_626d in audit_columns_626d:

        values_626d = (
            df_626d[
                column_626d
            ]
            .to_numpy(
                dtype=np.float64
            )
        )


        records_626d.append(
            {
                "population":
                    population_name_626d,

                "score":
                    column_626d,

                "mean":
                    float(
                        np.mean(
                            values_626d
                        )
                    ),

                "std":
                    float(
                        np.std(
                            values_626d,
                            ddof=0,
                        )
                    ),

                "q50":
                    float(
                        np.quantile(
                            values_626d,
                            0.50,
                        )
                    ),

                "q90":
                    float(
                        np.quantile(
                            values_626d,
                            0.90,
                        )
                    ),

                "q95":
                    float(
                        np.quantile(
                            values_626d,
                            0.95,
                        )
                    ),

                "q99":
                    float(
                        np.quantile(
                            values_626d,
                            0.99,
                        )
                    ),

                "minimum":
                    float(
                        np.min(
                            values_626d
                        )
                    ),

                "maximum":
                    float(
                        np.max(
                            values_626d
                        )
                    ),
            }
        )

    return pd.DataFrame(
        records_626d
    )


SENSOR_QUALITY_CLEAN_DISTRIBUTION_626D = (
    pd.concat(
        [
            clean_distribution_summary_626d(
                SENSOR_QUALITY_DEV_SCORED_626D,
                "canonical_DEV",
            ),
            clean_distribution_summary_626d(
                SENSOR_QUALITY_VAL_CLEAN_SCORED_626D,
                "clean_validation",
            ),
        ],
        ignore_index=True,
    )
)


print(
    "\n--- Clean sensor-quality score distributions ---"
)

display(
    SENSOR_QUALITY_CLEAN_DISTRIBUTION_626D
    .round(
        6
    )
)


# ------------------------------------------------------------
# 10. Calibrated component correlation audit
# ------------------------------------------------------------

SENSOR_QUALITY_COMPONENT_CORRELATION_626D = (
    SENSOR_QUALITY_DEV_SCORED_626D[
        SENSOR_QUALITY_COMPONENT_COLUMNS_626D
    ]
    .corr()
)


print(
    "\n--- Clean-DEV calibrated-component correlation ---"
)

display(
    SENSOR_QUALITY_COMPONENT_CORRELATION_626D
    .round(
        6
    )
)


component_corr_array_626d = (
    SENSOR_QUALITY_COMPONENT_CORRELATION_626D
    .to_numpy(
        dtype=np.float64
    )
)


component_offdiag_mask_626d = (
    ~np.eye(
        len(
            SENSOR_QUALITY_COMPONENT_COLUMNS_626D
        ),
        dtype=bool,
    )
)


maximum_component_abs_correlation_626d = float(
    np.max(
        np.abs(
            component_corr_array_626d[
                component_offdiag_mask_626d
            ]
        )
    )
)


# ------------------------------------------------------------
# 11. Integrity checks
# ------------------------------------------------------------

dev_count_preserved_626d = bool(
    len(
        SENSOR_QUALITY_DEV_SCORED_626D
    )
    ==
    len(
        SENSOR_QUALITY_DEV_WINDOWS_626B
    )
)


val_count_preserved_626d = bool(
    len(
        SENSOR_QUALITY_VAL_CLEAN_SCORED_626D
    )
    ==
    len(
        SENSOR_QUALITY_VAL_CLEAN_WINDOWS_626C
    )
)


all_scores_finite_626d = bool(
    np.isfinite(
        pd.concat(
            [
                SENSOR_QUALITY_DEV_SCORED_626D[
                    audit_columns_626d
                ],
                SENSOR_QUALITY_VAL_CLEAN_SCORED_626D[
                    audit_columns_626d
                ],
            ],
            ignore_index=True,
        )
        .to_numpy(
            dtype=np.float64
        )
    ).all()
)


all_scores_in_unit_interval_626d = bool(
    (
        (
            SENSOR_QUALITY_DEV_SCORED_626D[
                audit_columns_626d
            ]
            >= 0.0
        )
        &
        (
            SENSOR_QUALITY_DEV_SCORED_626D[
                audit_columns_626d
            ]
            <= 1.0
        )
    )
    .all()
    .all()
    and
    (
        (
            SENSOR_QUALITY_VAL_CLEAN_SCORED_626D[
                audit_columns_626d
            ]
            >= 0.0
        )
        &
        (
            SENSOR_QUALITY_VAL_CLEAN_SCORED_626D[
                audit_columns_626d
            ]
            <= 1.0
        )
    )
    .all()
    .all()
)


if not dev_count_preserved_626d:

    raise RuntimeError(
        "Clean DEV window count changed during calibration."
    )


if not val_count_preserved_626d:

    raise RuntimeError(
        "Clean validation window count changed "
        "during calibration."
    )


if not all_scores_finite_626d:

    raise RuntimeError(
        "Non-finite calibrated sensor-quality "
        "scores detected."
    )


if not all_scores_in_unit_interval_626d:

    raise RuntimeError(
        "Calibrated sensor-quality values outside "
        "[0, 1] were detected."
    )


# ------------------------------------------------------------
# 12. Final summary
# ------------------------------------------------------------

print(
    "\n"
    "============================================================"
)

print(
    "STEP 6.26D SENSOR-QUALITY CALIBRATION SUMMARY"
)

print(
    "============================================================"
)

print(
    "Selected raw quality channels       :",
    len(
        SENSOR_QUALITY_SELECTED_FEATURES_626D
    ),
)

print(
    "Selected features                   :",
    SENSOR_QUALITY_SELECTED_FEATURES_626D,
)

print(
    "Calibration population              : Canonical DEV"
)

print(
    "Calibration type                    : "
    "two-sided empirical absolute deviation"
)

print(
    "Clean DEV windows                   :",
    len(
        SENSOR_QUALITY_DEV_SCORED_626D
    ),
)

print(
    "Clean validation windows            :",
    len(
        SENSOR_QUALITY_VAL_CLEAN_SCORED_626D
    ),
)

print(
    "Raw selected max abs correlation    :",
    maximum_selected_abs_correlation_626d,
)

print(
    "Calibrated max abs correlation      :",
    maximum_component_abs_correlation_626d,
)

print(
    "DEV window count preserved          :",
    dev_count_preserved_626d,
)

print(
    "Validation window count preserved   :",
    val_count_preserved_626d,
)

print(
    "All calibrated values finite        :",
    all_scores_finite_626d,
)

print(
    "All calibrated values in [0, 1]     :",
    all_scores_in_unit_interval_626d,
)

print(
    "\nProvisional fusion candidates:"
)

print(
    "  1. sq_candidate_mean_fusion"
)

print(
    "  2. sq_candidate_max_fusion"
)

print(
    "\nRUL labels used                     : False"
)

print(
    "Official TEST labels inspected      : False"
)

print(
    "Official TEST scored                : False"
)

print(
    "Model inference executed            : False"
)

print(
    "MC-dropout executed                 : False"
)

print(
    "Final sensor-quality score frozen   : False"
)

print(
    "Quality threshold selected          : False"
)

print(
    "Frozen reliability rule modified    : False"
)

print(
    "Files written                       : False"
)

print(
    "\nNo final research decision was frozen."
)

print(
    "=== END STEP 6.26D ==="
)

In [ ]:
# ============================================================
# Step 6.26E — Full frozen-DEV sensor-quality fusion evaluation
# ROBUST CORRECTED VERSION
#
# Purpose:
#   Evaluate the two provisional sensor-quality fusion
#   candidates across all 315 frozen DEV corruption scenarios.
#
# Safety:
#   - DEV / validation only
#   - no RUL labels
#   - no official TEST labels
#   - no official TEST scoring
#   - no RUL model inference
#   - no MC-dropout
#   - no research decision frozen
#   - no files written
# ============================================================

import hashlib
import inspect

import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score


print(
    "=== Step 6.26E — Full frozen-DEV "
    "sensor-quality fusion evaluation ==="
)


# ============================================================
# 1. Required objects
# ============================================================

required_objects_626e = [
    "scenario_manifests_620gc",
    "apply_frozen_corruption_620gc",
    "val_raw_base_620gc",

    "build_sensor_quality_candidates_626b",

    "SENSOR_QUALITY_SENSOR_COLUMNS_626B",
    "SENSOR_QUALITY_REFERENCE_626B",
    "SENSOR_QUALITY_WINDOW_LENGTH_626B",
    "SENSOR_QUALITY_WINDOW_FEATURES_626B",

    "score_sensor_quality_windows_626d",

    "SENSOR_QUALITY_SELECTED_FEATURES_626D",
    "SENSOR_QUALITY_COMPONENT_COLUMNS_626D",
    "SENSOR_QUALITY_FUSION_CANDIDATES_626D",

    "SENSOR_QUALITY_VAL_CLEAN_SCORED_626D",
    "SENSOR_QUALITY_CLEAN_DISTRIBUTION_626D",
]


print("\n--- Required objects ---")

missing_objects_626e = []

for name_626e in required_objects_626e:

    exists_626e = name_626e in globals()

    print(
        f"{name_626e:<48}: "
        f"{exists_626e}"
    )

    if not exists_626e:
        missing_objects_626e.append(
            name_626e
        )


if missing_objects_626e:

    raise RuntimeError(
        "Step 6.26E cannot continue. "
        "Missing required objects: "
        + str(
            missing_objects_626e
        )
    )


# ============================================================
# 2. Fixed research constants
# ============================================================

WINDOW_LENGTH_626E = int(
    SENSOR_QUALITY_WINDOW_LENGTH_626B
)

SENSOR_COLUMNS_626E = list(
    SENSOR_QUALITY_SENSOR_COLUMNS_626B
)

WINDOW_FEATURES_626E = list(
    SENSOR_QUALITY_WINDOW_FEATURES_626B
)

SELECTED_FEATURES_626E = list(
    SENSOR_QUALITY_SELECTED_FEATURES_626D
)

FUSION_CANDIDATES_626E = list(
    SENSOR_QUALITY_FUSION_CANDIDATES_626D
)


if WINDOW_LENGTH_626E != 30:

    raise RuntimeError(
        "Unexpected sensor-quality window length. "
        f"Expected 30, received {WINDOW_LENGTH_626E}."
    )


if len(SENSOR_COLUMNS_626E) != 15:

    raise RuntimeError(
        "Expected 15 primary sensor columns."
    )


if len(WINDOW_FEATURES_626E) != 8:

    raise RuntimeError(
        "Expected eight Step 6.26B "
        "window-level sensor-quality features."
    )


if len(SELECTED_FEATURES_626E) != 3:

    raise RuntimeError(
        "Expected three Step 6.26D "
        "selected quality channels."
    )


if len(FUSION_CANDIDATES_626E) != 2:

    raise RuntimeError(
        "Expected two provisional fusion candidates."
    )


print(
    "\n--- Frozen sensor-quality configuration ---"
)

print(
    "Primary sensor count :",
    len(SENSOR_COLUMNS_626E)
)

print(
    "Window length        :",
    WINDOW_LENGTH_626E
)

print(
    "Window features      :",
    WINDOW_FEATURES_626E
)

print(
    "Selected features    :",
    SELECTED_FEATURES_626E
)

print(
    "Fusion candidates    :",
    FUSION_CANDIDATES_626E
)


# ============================================================
# 3. Builder and scorer contracts
# ============================================================

builder_signature_626e = inspect.signature(
    build_sensor_quality_candidates_626b
)

scorer_signature_626e = inspect.signature(
    score_sensor_quality_windows_626d
)


print(
    "\n--- Function contracts ---"
)

print(
    "Builder:",
    builder_signature_626e
)

print(
    "Scorer :",
    scorer_signature_626e
)


builder_parameter_names_626e = list(
    builder_signature_626e.parameters.keys()
)


if (
    builder_parameter_names_626e[:3]
    !=
    [
        "input_df",
        "sensor_columns",
        "reference_stats",
    ]
):

    raise RuntimeError(
        "Unexpected Step 6.26B builder signature."
    )


# ============================================================
# 4. Frozen corruption-family structure
# ============================================================

FAMILY_ORDER_626E = [
    "bias",
    "noise",
    "drift",
    "dropout",
    "stuck",
]


EXPECTED_FAMILY_COUNTS_626E = {
    "bias": 90,
    "noise": 45,
    "drift": 90,
    "dropout": 45,
    "stuck": 45,
}


actual_family_counts_626e = {
    family_626e: int(
        len(
            scenario_manifests_620gc[
                family_626e
            ]
        )
    )
    for family_626e
    in FAMILY_ORDER_626E
}


print(
    "\n--- Frozen manifest counts ---"
)

for family_626e in FAMILY_ORDER_626E:

    print(
        f"{family_626e:<10}: "
        f"{actual_family_counts_626e[family_626e]}"
    )


if (
    actual_family_counts_626e
    !=
    EXPECTED_FAMILY_COUNTS_626E
):

    raise RuntimeError(
        "Frozen corruption manifest counts changed."
    )


TOTAL_SCENARIOS_626E = int(
    sum(
        actual_family_counts_626e.values()
    )
)


print(
    "Total     :",
    TOTAL_SCENARIOS_626E
)


if TOTAL_SCENARIOS_626E != 315:

    raise RuntimeError(
        "Expected exactly 315 frozen scenarios."
    )


# ============================================================
# 5. Integrity-hash helper
# ============================================================

def dataframe_sha256_626e(
    df_626e
):

    hashed_626e = (
        pd.util.hash_pandas_object(
            df_626e,
            index=True
        )
        .to_numpy(
            dtype=np.uint64
        )
        .tobytes()
    )

    return hashlib.sha256(
        hashed_626e
    ).hexdigest()


VAL_RAW_HASH_BEFORE_626E = (
    dataframe_sha256_626e(
        val_raw_base_620gc
    )
)


# ============================================================
# 6. Recursively collect DataFrames from returned objects
# ============================================================

def collect_dataframes_626e(
    obj_626e,
    path_626e="root"
):

    found_626e = []


    if isinstance(
        obj_626e,
        pd.DataFrame
    ):

        found_626e.append(
            (
                path_626e,
                obj_626e
            )
        )


    elif isinstance(
        obj_626e,
        (tuple, list)
    ):

        for i_626e, item_626e in enumerate(
            obj_626e
        ):

            found_626e.extend(
                collect_dataframes_626e(
                    item_626e,
                    path_626e
                    + f"[{i_626e}]"
                )
            )


    elif isinstance(
        obj_626e,
        dict
    ):

        for key_626e, value_626e in (
            obj_626e.items()
        ):

            found_626e.extend(
                collect_dataframes_626e(
                    value_626e,
                    path_626e
                    + f"[{repr(key_626e)}]"
                )
            )


    return found_626e


# ============================================================
# 7. Exact expected window-endpoint keys
#
# For each engine, a 30-cycle window can end at every row
# starting from row-position 29 onward.
#
# We derive these keys from the supplied raw DataFrame itself.
# No assumption about starting cycle number is required.
# ============================================================

def expected_window_endpoint_keys_626e(
    raw_df_626e
):

    required_columns_626e = {
        "unit_id",
        "cycle",
    }


    if not required_columns_626e.issubset(
        raw_df_626e.columns
    ):

        raise RuntimeError(
            "Raw DataFrame does not contain "
            "unit_id and cycle."
        )


    key_parts_626e = []


    for unit_id_626e, group_626e in (
        raw_df_626e
        .groupby(
            "unit_id",
            sort=False
        )
    ):

        group_626e = (
            group_626e
            .sort_values(
                "cycle"
            )
        )


        if (
            len(group_626e)
            <
            WINDOW_LENGTH_626E
        ):

            continue


        endpoints_626e = (
            group_626e
            .iloc[
                WINDOW_LENGTH_626E - 1:
            ][
                [
                    "unit_id",
                    "cycle",
                ]
            ]
            .copy()
        )


        key_parts_626e.append(
            endpoints_626e
        )


    if not key_parts_626e:

        raise RuntimeError(
            "No eligible windows were found."
        )


    expected_keys_626e = (
        pd.concat(
            key_parts_626e,
            ignore_index=True
        )
    )


    duplicate_count_626e = int(
        expected_keys_626e.duplicated(
            subset=[
                "unit_id",
                "cycle",
            ]
        ).sum()
    )


    if duplicate_count_626e != 0:

        raise RuntimeError(
            "Expected window-endpoint keys "
            "contain duplicates."
        )


    return expected_keys_626e


# ============================================================
# 8. Exact-key comparison helper
# ============================================================

def endpoint_keys_match_626e(
    candidate_df_626e,
    expected_keys_626e
):

    if (
        "unit_id"
        not in candidate_df_626e.columns
        or
        "cycle"
        not in candidate_df_626e.columns
    ):

        return False


    if len(
        candidate_df_626e
    ) != len(
        expected_keys_626e
    ):

        return False


    if candidate_df_626e.duplicated(
        subset=[
            "unit_id",
            "cycle",
        ]
    ).any():

        return False


    candidate_keys_626e = (
        candidate_df_626e[
            [
                "unit_id",
                "cycle",
            ]
        ]
        .sort_values(
            [
                "unit_id",
                "cycle",
            ]
        )
        .reset_index(
            drop=True
        )
    )


    expected_sorted_626e = (
        expected_keys_626e
        .sort_values(
            [
                "unit_id",
                "cycle",
            ]
        )
        .reset_index(
            drop=True
        )
    )


    return bool(
        candidate_keys_626e.equals(
            expected_sorted_626e
        )
    )


# ============================================================
# 9. ROBUST Step 6.26B window-output extractor
#
# The previous failure occurred because multiple returned
# DataFrames contained the selected three features.
#
# Here the correct DataFrame must simultaneously:
#   1. contain all eight window features,
#   2. contain unit_id and cycle,
#   3. have the exact expected number of windows,
#   4. have exactly the expected endpoint keys.
# ============================================================

def extract_quality_window_dataframe_626e(
    builder_result_626e,
    raw_df_626e
):

    expected_keys_626e = (
        expected_window_endpoint_keys_626e(
            raw_df_626e
        )
    )


    required_columns_626e = (
        {
            "unit_id",
            "cycle",
        }
        |
        set(
            WINDOW_FEATURES_626E
        )
    )


    returned_dfs_626e = (
        collect_dataframes_626e(
            builder_result_626e
        )
    )


    if not returned_dfs_626e:

        raise RuntimeError(
            "Step 6.26B builder returned "
            "no DataFrame."
        )


    diagnostic_rows_626e = []

    exact_candidates_626e = []


    for path_626e, df_626e in (
        returned_dfs_626e
    ):

        has_required_columns_626e = bool(
            required_columns_626e.issubset(
                set(
                    df_626e.columns
                )
            )
        )


        length_exact_626e = bool(
            len(df_626e)
            ==
            len(expected_keys_626e)
        )


        keys_exact_626e = False


        if (
            has_required_columns_626e
            and
            length_exact_626e
        ):

            keys_exact_626e = (
                endpoint_keys_match_626e(
                    df_626e,
                    expected_keys_626e
                )
            )


        diagnostic_rows_626e.append(
            {
                "path":
                    path_626e,

                "rows":
                    int(
                        len(df_626e)
                    ),

                "columns":
                    int(
                        df_626e.shape[1]
                    ),

                "has_all_window_features":
                    has_required_columns_626e,

                "expected_window_count":
                    int(
                        len(
                            expected_keys_626e
                        )
                    ),

                "length_exact":
                    length_exact_626e,

                "endpoint_keys_exact":
                    keys_exact_626e,
            }
        )


        if (
            has_required_columns_626e
            and
            length_exact_626e
            and
            keys_exact_626e
        ):

            exact_candidates_626e.append(
                (
                    path_626e,
                    df_626e.copy()
                )
            )


    if len(
        exact_candidates_626e
    ) == 0:

        print(
            "\n--- Builder return diagnostics ---"
        )

        display(
            pd.DataFrame(
                diagnostic_rows_626e
            )
        )


        raise RuntimeError(
            "No Step 6.26B returned DataFrame "
            "matches the exact expected "
            "window-level contract."
        )


    if len(
        exact_candidates_626e
    ) == 1:

        return (
            exact_candidates_626e[0][1]
            .copy()
        )


    # --------------------------------------------------------
    # Multiple exact candidates are allowed only if their
    # complete required window information is numerically
    # identical. Then selecting either copy is harmless.
    # --------------------------------------------------------

    comparison_columns_626e = (
        [
            "unit_id",
            "cycle",
        ]
        +
        WINDOW_FEATURES_626E
    )


    reference_candidate_626e = (
        exact_candidates_626e[0][1][
            comparison_columns_626e
        ]
        .sort_values(
            [
                "unit_id",
                "cycle",
            ]
        )
        .reset_index(
            drop=True
        )
    )


    all_equivalent_626e = True


    for (
        _path_626e,
        other_candidate_626e
    ) in exact_candidates_626e[1:]:

        other_candidate_626e = (
            other_candidate_626e[
                comparison_columns_626e
            ]
            .sort_values(
                [
                    "unit_id",
                    "cycle",
                ]
            )
            .reset_index(
                drop=True
            )
        )


        if not reference_candidate_626e[
            [
                "unit_id",
                "cycle",
            ]
        ].equals(
            other_candidate_626e[
                [
                    "unit_id",
                    "cycle",
                ]
            ]
        ):

            all_equivalent_626e = False
            break


        reference_values_626e = (
            reference_candidate_626e[
                WINDOW_FEATURES_626E
            ]
            .to_numpy(
                dtype=np.float64
            )
        )


        other_values_626e = (
            other_candidate_626e[
                WINDOW_FEATURES_626E
            ]
            .to_numpy(
                dtype=np.float64
            )
        )


        if not np.allclose(
            reference_values_626e,
            other_values_626e,
            rtol=0.0,
            atol=1e-12,
            equal_nan=True
        ):

            all_equivalent_626e = False
            break


    if not all_equivalent_626e:

        print(
            "\n--- Ambiguous exact candidates ---"
        )

        print(
            [
                path_626e
                for path_626e, _
                in exact_candidates_626e
            ]
        )


        raise RuntimeError(
            "Multiple non-equivalent exact "
            "window-level DataFrames were found. "
            "Refusing to guess."
        )


    # Equivalent duplicates are safe.
    return (
        exact_candidates_626e[0][1]
        .copy()
    )


# ============================================================
# 10. Normalize Step 6.26D scorer output too
# ============================================================

def extract_scored_dataframe_626e(
    scorer_result_626e
):

    required_columns_626e = (
        {
            "unit_id",
            "cycle",
        }
        |
        set(
            FUSION_CANDIDATES_626E
        )
    )


    if isinstance(
        scorer_result_626e,
        pd.DataFrame
    ):

        scored_candidates_626e = [
            scorer_result_626e
        ]

    else:

        scored_candidates_626e = [
            df_626e
            for _, df_626e
            in collect_dataframes_626e(
                scorer_result_626e
            )
        ]


    valid_scored_626e = [
        df_626e
        for df_626e
        in scored_candidates_626e
        if required_columns_626e.issubset(
            set(
                df_626e.columns
            )
        )
    ]


    if len(
        valid_scored_626e
    ) == 0:

        raise RuntimeError(
            "Could not identify Step 6.26D "
            "scored sensor-quality DataFrame."
        )


    if len(
        valid_scored_626e
    ) > 1:

        # Prefer largest column set if the scorer
        # returns multiple related representations.
        valid_scored_626e = sorted(
            valid_scored_626e,
            key=lambda df_626e: (
                df_626e.shape[1]
            ),
            reverse=True
        )


    return (
        valid_scored_626e[0]
        .copy()
    )


# ============================================================
# 11. Exact Step 6.26B -> Step 6.26D pipeline
# ============================================================

def build_and_score_quality_626e(
    raw_df_626e
):

    builder_result_626e = (
        build_sensor_quality_candidates_626b(
            raw_df_626e,
            SENSOR_QUALITY_SENSOR_COLUMNS_626B,
            SENSOR_QUALITY_REFERENCE_626B,
            window_length=(
                SENSOR_QUALITY_WINDOW_LENGTH_626B
            ),
        )
    )


    window_df_626e = (
        extract_quality_window_dataframe_626e(
            builder_result_626e,
            raw_df_626e
        )
    )


    scorer_result_626e = (
        score_sensor_quality_windows_626d(
            window_df_626e.copy()
        )
    )


    scored_df_626e = (
        extract_scored_dataframe_626e(
            scorer_result_626e
        )
    )


    return scored_df_626e.copy()


# ============================================================
# 12. Stored clean-validation reference
# ============================================================

clean_scored_626e = (
    SENSOR_QUALITY_VAL_CLEAN_SCORED_626D
    .copy()
)


required_clean_columns_626e = (
    {
        "unit_id",
        "cycle",
    }
    |
    set(
        FUSION_CANDIDATES_626E
    )
)


missing_clean_columns_626e = (
    required_clean_columns_626e
    -
    set(
        clean_scored_626e.columns
    )
)


if missing_clean_columns_626e:

    raise RuntimeError(
        "Stored Step 6.26D clean-validation "
        "table is missing columns: "
        + str(
            sorted(
                missing_clean_columns_626e
            )
        )
    )


clean_duplicate_keys_626e = int(
    clean_scored_626e.duplicated(
        subset=[
            "unit_id",
            "cycle",
        ]
    ).sum()
)


if clean_duplicate_keys_626e != 0:

    raise RuntimeError(
        "Stored clean-validation scores "
        "contain duplicate keys."
    )


expected_clean_keys_626e = (
    expected_window_endpoint_keys_626e(
        val_raw_base_620gc
    )
)


clean_window_count_626e = int(
    len(
        clean_scored_626e
    )
)


print(
    "\n--- Stored clean-validation reference ---"
)

print(
    "Raw validation rows   :",
    len(
        val_raw_base_620gc
    )
)

print(
    "Validation engines    :",
    val_raw_base_620gc[
        "unit_id"
    ].nunique()
)

print(
    "Expected windows      :",
    len(
        expected_clean_keys_626e
    )
)

print(
    "Stored clean windows  :",
    clean_window_count_626e
)

print(
    "Duplicate keys        :",
    clean_duplicate_keys_626e
)


if clean_window_count_626e != len(
    expected_clean_keys_626e
):

    raise RuntimeError(
        "Stored clean-validation window count "
        "does not match the mathematically "
        "expected window count."
    )


if not endpoint_keys_match_626e(
    clean_scored_626e,
    expected_clean_keys_626e
):

    raise RuntimeError(
        "Stored clean-validation endpoint keys "
        "do not match the expected 30-cycle windows."
    )


# ============================================================
# 13. Reproduce clean validation before running corruptions
# ============================================================

print(
    "\n--- Rebuilding clean-validation sensor quality ---"
)


rebuilt_clean_scored_626e = (
    build_and_score_quality_626e(
        val_raw_base_620gc.copy()
    )
)


if rebuilt_clean_scored_626e.duplicated(
    subset=[
        "unit_id",
        "cycle",
    ]
).any():

    raise RuntimeError(
        "Rebuilt clean-validation table "
        "contains duplicate keys."
    )


if not endpoint_keys_match_626e(
    rebuilt_clean_scored_626e,
    expected_clean_keys_626e
):

    raise RuntimeError(
        "Rebuilt clean-validation endpoint keys "
        "do not match the expected window keys."
    )


clean_reproduction_626e = (
    clean_scored_626e[
        [
            "unit_id",
            "cycle",
        ]
        +
        FUSION_CANDIDATES_626E
    ]
    .merge(
        rebuilt_clean_scored_626e[
            [
                "unit_id",
                "cycle",
            ]
            +
            FUSION_CANDIDATES_626E
        ],
        on=[
            "unit_id",
            "cycle",
        ],
        how="inner",
        suffixes=(
            "_stored",
            "_rebuilt",
        ),
        validate="one_to_one",
    )
)


clean_alignment_exact_626e = bool(
    len(
        clean_reproduction_626e
    )
    ==
    clean_window_count_626e
)


clean_reproduction_errors_626e = {}


for fusion_626e in FUSION_CANDIDATES_626E:

    stored_values_626e = (
        clean_reproduction_626e[
            fusion_626e
            + "_stored"
        ]
        .to_numpy(
            dtype=np.float64
        )
    )


    rebuilt_values_626e = (
        clean_reproduction_626e[
            fusion_626e
            + "_rebuilt"
        ]
        .to_numpy(
            dtype=np.float64
        )
    )


    clean_reproduction_errors_626e[
        fusion_626e
    ] = float(
        np.max(
            np.abs(
                stored_values_626e
                -
                rebuilt_values_626e
            )
        )
    )


max_clean_reproduction_error_626e = float(
    max(
        clean_reproduction_errors_626e.values()
    )
)


print(
    "Stored windows             :",
    clean_window_count_626e
)

print(
    "Rebuilt windows            :",
    len(
        rebuilt_clean_scored_626e
    )
)

print(
    "Endpoint alignment exact   :",
    clean_alignment_exact_626e
)

print(
    "Maximum fusion-score error :",
    max_clean_reproduction_error_626e
)


if not clean_alignment_exact_626e:

    raise RuntimeError(
        "Clean-validation window alignment "
        "could not be reproduced."
    )


if max_clean_reproduction_error_626e > 1e-10:

    raise RuntimeError(
        "Step 6.26B -> Step 6.26D clean "
        "pipeline does not reproduce the "
        "stored clean-validation scores."
    )


# ============================================================
# 14. Clean candidate reference statistics
# ============================================================

clean_candidate_stats_626e = {}


for fusion_626e in FUSION_CANDIDATES_626E:

    values_626e = (
        clean_scored_626e[
            fusion_626e
        ]
        .to_numpy(
            dtype=np.float64
        )
    )


    if not np.isfinite(
        values_626e
    ).all():

        raise RuntimeError(
            f"Non-finite clean scores: {fusion_626e}"
        )


    std_626e = float(
        np.std(
            values_626e,
            ddof=0
        )
    )


    if std_626e <= 0:

        raise RuntimeError(
            f"Zero clean score variation for "
            f"{fusion_626e}."
        )


    clean_candidate_stats_626e[
        fusion_626e
    ] = {
        "mean":
            float(
                np.mean(
                    values_626e
                )
            ),

        "std":
            std_626e,

        "q50":
            float(
                np.quantile(
                    values_626e,
                    0.50
                )
            ),

        "q90":
            float(
                np.quantile(
                    values_626e,
                    0.90
                )
            ),

        "q95":
            float(
                np.quantile(
                    values_626e,
                    0.95
                )
            ),

        "q99":
            float(
                np.quantile(
                    values_626e,
                    0.99
                )
            ),
    }


print(
    "\n--- Clean-validation fusion statistics ---"
)

display(
    pd.DataFrame(
        clean_candidate_stats_626e
    )
    .T
    .round(
        6
    )
)


# ============================================================
# 15. Manifest value helper
# ============================================================

def safe_manifest_value_626e(
    scenario_row_626e,
    key_626e,
    default_626e=None
):

    if key_626e not in scenario_row_626e.index:
        return default_626e


    value_626e = (
        scenario_row_626e[
            key_626e
        ]
    )


    try:

        if pd.isna(
            value_626e
        ):
            return default_626e

    except Exception:
        pass


    return value_626e


# ============================================================
# 16. Run all 315 frozen DEV corruptions
# ============================================================

scenario_records_626e = []

structural_failures_626e = []

alignment_failures_626e = []

global_index_626e = 0


print(
    "\n--- Evaluating 315 frozen DEV "
    "corruption scenarios ---"
)


for family_626e in FAMILY_ORDER_626E:

    manifest_626e = (
        scenario_manifests_620gc[
            family_626e
        ]
    )


    print(
        f"\nRunning {family_626e.upper()} "
        f"({len(manifest_626e)} scenarios)..."
    )


    for family_position_626e, (
        manifest_index_626e,
        scenario_row_626e
    ) in enumerate(
        manifest_626e.iterrows()
    ):

        scenario_id_626e = (
            safe_manifest_value_626e(
                scenario_row_626e,
                "scenario_id",
                None
            )
        )


        if scenario_id_626e is None:

            scenario_id_626e = (
                f"{family_626e}_"
                f"{family_position_626e + 1:03d}"
            )


        manifest_scenario_id_626e = (
            safe_manifest_value_626e(
                scenario_row_626e,
                "manifest_scenario_id",
                None
            )
        )


        sensor_626e = (
            safe_manifest_value_626e(
                scenario_row_626e,
                "sensor",
                None
            )
        )


        severity_626e = (
            safe_manifest_value_626e(
                scenario_row_626e,
                "severity",
                None
            )
        )


        # ----------------------------------------------------
        # Apply already-frozen validation corruption
        # ----------------------------------------------------

        try:

            corrupted_raw_626e = (
                apply_frozen_corruption_620gc(
                    family_626e,
                    scenario_row_626e
                )
            )


            if not isinstance(
                corrupted_raw_626e,
                pd.DataFrame
            ):

                raise RuntimeError(
                    "Frozen corruption wrapper "
                    "did not return a DataFrame."
                )


            # Exact structural preservation
            if (
                corrupted_raw_626e.shape
                !=
                val_raw_base_620gc.shape
            ):

                raise RuntimeError(
                    "Corrupted validation shape changed."
                )


            if not corrupted_raw_626e.index.equals(
                val_raw_base_620gc.index
            ):

                raise RuntimeError(
                    "Corrupted validation index changed."
                )


            if list(
                corrupted_raw_626e.columns
            ) != list(
                val_raw_base_620gc.columns
            ):

                raise RuntimeError(
                    "Corrupted validation columns changed."
                )


            corrupted_scored_626e = (
                build_and_score_quality_626e(
                    corrupted_raw_626e
                )
            )


        except Exception as exc_626e:

            structural_failures_626e.append(
                {
                    "global_index":
                        global_index_626e,

                    "family":
                        family_626e,

                    "scenario_id":
                        scenario_id_626e,

                    "error":
                        repr(
                            exc_626e
                        ),
                }
            )


            raise RuntimeError(
                "Step 6.26E failed during "
                f"{family_626e} / "
                f"{scenario_id_626e}: "
                f"{exc_626e}"
            ) from exc_626e


        # ----------------------------------------------------
        # Exact expected window keys
        # ----------------------------------------------------

        corrupted_expected_keys_626e = (
            expected_window_endpoint_keys_626e(
                corrupted_raw_626e
            )
        )


        if not endpoint_keys_match_626e(
            corrupted_scored_626e,
            corrupted_expected_keys_626e
        ):

            alignment_failures_626e.append(
                {
                    "global_index":
                        global_index_626e,

                    "family":
                        family_626e,

                    "scenario_id":
                        scenario_id_626e,
                }
            )


            raise RuntimeError(
                f"{scenario_id_626e}: "
                "corrupted window keys are invalid."
            )


        # ----------------------------------------------------
        # Clean/corrupted exact pairing
        # ----------------------------------------------------

        paired_626e = (
            clean_scored_626e[
                [
                    "unit_id",
                    "cycle",
                ]
                +
                FUSION_CANDIDATES_626E
            ]
            .merge(
                corrupted_scored_626e[
                    [
                        "unit_id",
                        "cycle",
                    ]
                    +
                    FUSION_CANDIDATES_626E
                ],
                on=[
                    "unit_id",
                    "cycle",
                ],
                how="inner",
                suffixes=(
                    "_clean",
                    "_corrupted",
                ),
                validate="one_to_one",
            )
        )


        if (
            len(paired_626e)
            !=
            clean_window_count_626e
        ):

            alignment_failures_626e.append(
                {
                    "global_index":
                        global_index_626e,

                    "family":
                        family_626e,

                    "scenario_id":
                        scenario_id_626e,

                    "paired_windows":
                        len(
                            paired_626e
                        ),
                }
            )


            raise RuntimeError(
                f"{scenario_id_626e}: "
                "clean/corrupted window pairing failed."
            )


        # ----------------------------------------------------
        # Evaluate both fusion candidates
        # ----------------------------------------------------

        for fusion_626e in FUSION_CANDIDATES_626E:

            clean_values_626e = (
                paired_626e[
                    fusion_626e
                    + "_clean"
                ]
                .to_numpy(
                    dtype=np.float64
                )
            )


            corrupted_values_626e = (
                paired_626e[
                    fusion_626e
                    + "_corrupted"
                ]
                .to_numpy(
                    dtype=np.float64
                )
            )


            if not (
                np.isfinite(
                    clean_values_626e
                ).all()
                and
                np.isfinite(
                    corrupted_values_626e
                ).all()
            ):

                raise RuntimeError(
                    f"{scenario_id_626e}: "
                    f"non-finite {fusion_626e} scores."
                )


            delta_626e = (
                corrupted_values_626e
                -
                clean_values_626e
            )


            clean_std_626e = float(
                clean_candidate_stats_626e[
                    fusion_626e
                ][
                    "std"
                ]
            )


            mean_delta_626e = float(
                np.mean(
                    delta_626e
                )
            )


            median_delta_626e = float(
                np.median(
                    delta_626e
                )
            )


            mean_abs_delta_626e = float(
                np.mean(
                    np.abs(
                        delta_626e
                    )
                )
            )


            standardized_mean_shift_626e = float(
                mean_delta_626e
                /
                clean_std_626e
            )


            positive_window_fraction_626e = float(
                np.mean(
                    delta_626e > 0.0
                )
            )


            negative_window_fraction_626e = float(
                np.mean(
                    delta_626e < 0.0
                )
            )


            unchanged_window_fraction_626e = float(
                np.mean(
                    np.isclose(
                        delta_626e,
                        0.0,
                        atol=1e-12,
                        rtol=0.0
                    )
                )
            )


            clean_q95_626e = float(
                clean_candidate_stats_626e[
                    fusion_626e
                ][
                    "q95"
                ]
            )


            above_clean_q95_fraction_626e = float(
                np.mean(
                    corrupted_values_626e
                    >
                    clean_q95_626e
                )
            )


            labels_626e = np.concatenate(
                [
                    np.zeros(
                        len(
                            clean_values_626e
                        ),
                        dtype=np.int8
                    ),

                    np.ones(
                        len(
                            corrupted_values_626e
                        ),
                        dtype=np.int8
                    ),
                ]
            )


            auc_scores_626e = np.concatenate(
                [
                    clean_values_626e,
                    corrupted_values_626e,
                ]
            )


            auc_626e = float(
                roc_auc_score(
                    labels_626e,
                    auc_scores_626e
                )
            )


            scenario_records_626e.append(
                {
                    "global_index":
                        global_index_626e,

                    "family":
                        family_626e,

                    "family_position":
                        family_position_626e,

                    "scenario_id":
                        str(
                            scenario_id_626e
                        ),

                    "manifest_scenario_id":
                        manifest_scenario_id_626e,

                    "sensor":
                        sensor_626e,

                    "severity":
                        severity_626e,

                    "fusion":
                        fusion_626e,

                    "windows":
                        int(
                            len(
                                paired_626e
                            )
                        ),

                    "clean_mean":
                        float(
                            np.mean(
                                clean_values_626e
                            )
                        ),

                    "corrupted_mean":
                        float(
                            np.mean(
                                corrupted_values_626e
                            )
                        ),

                    "mean_delta":
                        mean_delta_626e,

                    "median_delta":
                        median_delta_626e,

                    "mean_abs_delta":
                        mean_abs_delta_626e,

                    "standardized_mean_shift":
                        standardized_mean_shift_626e,

                    "positive_window_fraction":
                        positive_window_fraction_626e,

                    "negative_window_fraction":
                        negative_window_fraction_626e,

                    "unchanged_window_fraction":
                        unchanged_window_fraction_626e,

                    "above_clean_q95_fraction":
                        above_clean_q95_fraction_626e,

                    "auc_clean_vs_corrupted":
                        auc_626e,
                }
            )


        global_index_626e += 1


# ============================================================
# 17. Scenario-level fusion results
# ============================================================

SENSOR_QUALITY_FUSION_SCENARIO_RESULTS_626E = (
    pd.DataFrame(
        scenario_records_626e
    )
)


expected_result_rows_626e = (
    TOTAL_SCENARIOS_626E
    *
    len(
        FUSION_CANDIDATES_626E
    )
)


if (
    len(
        SENSOR_QUALITY_FUSION_SCENARIO_RESULTS_626E
    )
    !=
    expected_result_rows_626e
):

    raise RuntimeError(
        "Unexpected scenario-fusion result count."
    )


# ============================================================
# 18. Overall fusion comparison
# ============================================================

overall_base_626e = (
    SENSOR_QUALITY_FUSION_SCENARIO_RESULTS_626E
    .groupby(
        "fusion",
        as_index=False
    )
    .agg(
        scenarios=(
            "scenario_id",
            "size"
        ),

        mean_auc=(
            "auc_clean_vs_corrupted",
            "mean"
        ),

        median_auc=(
            "auc_clean_vs_corrupted",
            "median"
        ),

        minimum_auc=(
            "auc_clean_vs_corrupted",
            "min"
        ),

        maximum_auc=(
            "auc_clean_vs_corrupted",
            "max"
        ),

        mean_standardized_shift=(
            "standardized_mean_shift",
            "mean"
        ),

        median_standardized_shift=(
            "standardized_mean_shift",
            "median"
        ),

        mean_positive_window_fraction=(
            "positive_window_fraction",
            "mean"
        ),

        mean_above_clean_q95_fraction=(
            "above_clean_q95_fraction",
            "mean"
        ),

        mean_score_delta=(
            "mean_delta",
            "mean"
        ),

        mean_absolute_score_delta=(
            "mean_abs_delta",
            "mean"
        ),
    )
)


overall_extra_records_626e = []


for fusion_626e in FUSION_CANDIDATES_626E:

    subset_626e = (
        SENSOR_QUALITY_FUSION_SCENARIO_RESULTS_626E[
            SENSOR_QUALITY_FUSION_SCENARIO_RESULTS_626E[
                "fusion"
            ]
            ==
            fusion_626e
        ]
    )


    overall_extra_records_626e.append(
        {
            "fusion":
                fusion_626e,

            "scenarios_auc_above_0_50":
                int(
                    (
                        subset_626e[
                            "auc_clean_vs_corrupted"
                        ]
                        >
                        0.50
                    ).sum()
                ),

            "fraction_auc_above_0_50":
                float(
                    (
                        subset_626e[
                            "auc_clean_vs_corrupted"
                        ]
                        >
                        0.50
                    ).mean()
                ),

            "scenarios_positive_mean_shift":
                int(
                    (
                        subset_626e[
                            "mean_delta"
                        ]
                        >
                        0.0
                    ).sum()
                ),

            "fraction_positive_mean_shift":
                float(
                    (
                        subset_626e[
                            "mean_delta"
                        ]
                        >
                        0.0
                    ).mean()
                ),

            "negative_mean_shift_scenarios":
                int(
                    (
                        subset_626e[
                            "mean_delta"
                        ]
                        <
                        0.0
                    ).sum()
                ),
        }
    )


overall_extra_626e = pd.DataFrame(
    overall_extra_records_626e
)


SENSOR_QUALITY_FUSION_OVERALL_SUMMARY_626E = (
    overall_base_626e
    .merge(
        overall_extra_626e,
        on="fusion",
        how="left",
        validate="one_to_one"
    )
)


print(
    "\n--- Overall fusion-candidate comparison ---"
)

display(
    SENSOR_QUALITY_FUSION_OVERALL_SUMMARY_626E
    .round(
        6
    )
)


# ============================================================
# 19. Family-level fusion comparison
# ============================================================

SENSOR_QUALITY_FUSION_FAMILY_SUMMARY_626E = (
    SENSOR_QUALITY_FUSION_SCENARIO_RESULTS_626E
    .groupby(
        [
            "fusion",
            "family",
        ],
        as_index=False
    )
    .agg(
        scenarios=(
            "scenario_id",
            "size"
        ),

        mean_auc=(
            "auc_clean_vs_corrupted",
            "mean"
        ),

        median_auc=(
            "auc_clean_vs_corrupted",
            "median"
        ),

        minimum_auc=(
            "auc_clean_vs_corrupted",
            "min"
        ),

        maximum_auc=(
            "auc_clean_vs_corrupted",
            "max"
        ),

        mean_standardized_shift=(
            "standardized_mean_shift",
            "mean"
        ),

        median_standardized_shift=(
            "standardized_mean_shift",
            "median"
        ),

        mean_positive_window_fraction=(
            "positive_window_fraction",
            "mean"
        ),

        mean_above_clean_q95_fraction=(
            "above_clean_q95_fraction",
            "mean"
        ),

        mean_score_delta=(
            "mean_delta",
            "mean"
        ),
    )
)


print(
    "\n--- Family-level fusion comparison ---"
)

display(
    SENSOR_QUALITY_FUSION_FAMILY_SUMMARY_626E
    .round(
        6
    )
)


# ============================================================
# 20. Family × severity fusion comparison
# ============================================================

SENSOR_QUALITY_FUSION_SEVERITY_SUMMARY_626E = (
    SENSOR_QUALITY_FUSION_SCENARIO_RESULTS_626E
    .groupby(
        [
            "fusion",
            "family",
            "severity",
        ],
        dropna=False,
        as_index=False
    )
    .agg(
        scenarios=(
            "scenario_id",
            "size"
        ),

        mean_auc=(
            "auc_clean_vs_corrupted",
            "mean"
        ),

        median_auc=(
            "auc_clean_vs_corrupted",
            "median"
        ),

        minimum_auc=(
            "auc_clean_vs_corrupted",
            "min"
        ),

        mean_standardized_shift=(
            "standardized_mean_shift",
            "mean"
        ),

        mean_positive_window_fraction=(
            "positive_window_fraction",
            "mean"
        ),

        mean_above_clean_q95_fraction=(
            "above_clean_q95_fraction",
            "mean"
        ),

        mean_score_delta=(
            "mean_delta",
            "mean"
        ),
    )
)


print(
    "\n--- Family × severity fusion comparison ---"
)

display(
    SENSOR_QUALITY_FUSION_SEVERITY_SUMMARY_626E
    .round(
        6
    )
)


# ============================================================
# 21. Clean DEV -> clean-validation stability
# ============================================================

clean_distribution_subset_626e = (
    SENSOR_QUALITY_CLEAN_DISTRIBUTION_626D[
        SENSOR_QUALITY_CLEAN_DISTRIBUTION_626D[
            "score"
        ]
        .isin(
            FUSION_CANDIDATES_626E
        )
    ]
    .copy()
)


clean_mean_pivot_626e = (
    clean_distribution_subset_626e
    .pivot(
        index="score",
        columns="population",
        values="mean"
    )
)


required_populations_626e = {
    "canonical_DEV",
    "clean_validation",
}


if not required_populations_626e.issubset(
    clean_mean_pivot_626e.columns
):

    raise RuntimeError(
        "Clean distribution table is missing "
        "canonical_DEV or clean_validation."
    )


clean_mean_pivot_626e[
    "absolute_mean_shift_DEV_to_validation"
] = np.abs(
    clean_mean_pivot_626e[
        "clean_validation"
    ]
    -
    clean_mean_pivot_626e[
        "canonical_DEV"
    ]
)


SENSOR_QUALITY_FUSION_CLEAN_STABILITY_626E = (
    clean_mean_pivot_626e
    .reset_index()
    .rename(
        columns={
            "score": "fusion"
        }
    )
)


print(
    "\n--- Clean DEV -> validation fusion stability ---"
)

display(
    SENSOR_QUALITY_FUSION_CLEAN_STABILITY_626E
    .round(
        6
    )
)


# ============================================================
# 22. Weakest 10 scenarios for each fusion candidate
# ============================================================

weakest_tables_626e = []


for fusion_626e in FUSION_CANDIDATES_626E:

    weakest_626e = (
        SENSOR_QUALITY_FUSION_SCENARIO_RESULTS_626E[
            SENSOR_QUALITY_FUSION_SCENARIO_RESULTS_626E[
                "fusion"
            ]
            ==
            fusion_626e
        ]
        .sort_values(
            [
                "auc_clean_vs_corrupted",
                "standardized_mean_shift",
            ],
            ascending=[
                True,
                True,
            ]
        )
        .head(
            10
        )
    )


    weakest_tables_626e.append(
        weakest_626e
    )


SENSOR_QUALITY_FUSION_WORST_SCENARIOS_626E = (
    pd.concat(
        weakest_tables_626e,
        ignore_index=True
    )
)


print(
    "\n--- Ten weakest scenarios per fusion candidate ---"
)

display(
    SENSOR_QUALITY_FUSION_WORST_SCENARIOS_626E[
        [
            "fusion",
            "global_index",
            "family",
            "scenario_id",
            "sensor",
            "severity",
            "mean_delta",
            "standardized_mean_shift",
            "positive_window_fraction",
            "auc_clean_vs_corrupted",
        ]
    ]
    .round(
        6
    )
)


# ============================================================
# 23. Final integrity checks
# ============================================================

VAL_RAW_HASH_AFTER_626E = (
    dataframe_sha256_626e(
        val_raw_base_620gc
    )
)


validation_base_unchanged_626e = bool(
    VAL_RAW_HASH_BEFORE_626E
    ==
    VAL_RAW_HASH_AFTER_626E
)


numeric_columns_626e = [
    "clean_mean",
    "corrupted_mean",
    "mean_delta",
    "median_delta",
    "mean_abs_delta",
    "standardized_mean_shift",
    "positive_window_fraction",
    "negative_window_fraction",
    "unchanged_window_fraction",
    "above_clean_q95_fraction",
    "auc_clean_vs_corrupted",
]


all_results_finite_626e = bool(
    np.isfinite(
        SENSOR_QUALITY_FUSION_SCENARIO_RESULTS_626E[
            numeric_columns_626e
        ]
        .to_numpy(
            dtype=np.float64
        )
    ).all()
)


all_scenarios_completed_626e = bool(
    global_index_626e
    ==
    TOTAL_SCENARIOS_626E
)


all_records_complete_626e = bool(
    len(
        SENSOR_QUALITY_FUSION_SCENARIO_RESULTS_626E
    )
    ==
    expected_result_rows_626e
)


structural_failure_count_626e = int(
    len(
        structural_failures_626e
    )
)


alignment_failure_count_626e = int(
    len(
        alignment_failures_626e
    )
)


if not validation_base_unchanged_626e:

    raise RuntimeError(
        "Validation raw base changed "
        "during Step 6.26E."
    )


if not all_results_finite_626e:

    raise RuntimeError(
        "Non-finite sensor-quality "
        "evaluation values detected."
    )


if not all_scenarios_completed_626e:

    raise RuntimeError(
        "Not all 315 corruption scenarios completed."
    )


if not all_records_complete_626e:

    raise RuntimeError(
        "Scenario-fusion record count is incorrect."
    )


if structural_failure_count_626e != 0:

    raise RuntimeError(
        "Structural failures were detected."
    )


if alignment_failure_count_626e != 0:

    raise RuntimeError(
        "Window-alignment failures were detected."
    )


# ============================================================
# 24. Final compact summary
# ============================================================

print(
    "\n"
    "============================================================"
)

print(
    "STEP 6.26E FINAL SENSOR-QUALITY FUSION SUMMARY"
)

print(
    "============================================================"
)

print(
    "Frozen scenarios expected             :",
    TOTAL_SCENARIOS_626E
)

print(
    "Frozen scenarios completed            :",
    global_index_626e
)

print(
    "Fusion candidates                     :",
    FUSION_CANDIDATES_626E
)

print(
    "Scenario-fusion records expected      :",
    expected_result_rows_626e
)

print(
    "Scenario-fusion records completed     :",
    len(
        SENSOR_QUALITY_FUSION_SCENARIO_RESULTS_626E
    )
)

print(
    "Validation raw rows                   :",
    len(
        val_raw_base_620gc
    )
)

print(
    "Validation engines                    :",
    val_raw_base_620gc[
        "unit_id"
    ].nunique()
)

print(
    "Expected 30-cycle windows             :",
    len(
        expected_clean_keys_626e
    )
)

print(
    "Clean pipeline reproduced             :",
    clean_alignment_exact_626e
    and
    (
        max_clean_reproduction_error_626e
        <=
        1e-10
    )
)

print(
    "Maximum clean reproduction error      :",
    max_clean_reproduction_error_626e
)

print(
    "Structural failures                   :",
    structural_failure_count_626e
)

print(
    "Alignment failures                    :",
    alignment_failure_count_626e
)

print(
    "All evaluation values finite          :",
    all_results_finite_626e
)

print(
    "Validation raw base unchanged         :",
    validation_base_unchanged_626e
)

print(
    "\nReference population                  : Canonical DEV"
)

print(
    "Reference STD convention              : population (ddof=0)"
)

print(
    "Sensor-quality window length          :",
    WINDOW_LENGTH_626E
)

print(
    "\nRUL labels used                       : False"
)

print(
    "Official TEST labels inspected        : False"
)

print(
    "Official TEST scored                  : False"
)

print(
    "RUL model inference executed          : False"
)

print(
    "MC-dropout executed                   : False"
)

print(
    "Frozen corruptions regenerated        : False"
)

print(
    "Corruption manifests modified         : False"
)

print(
    "Final sensor-quality fusion frozen    : False"
)

print(
    "Sensor-quality threshold selected     : False"
)

print(
    "Frozen reliability rule modified      : False"
)

print(
    "Files written                         : False"
)

print(
    "\nNo final research decision was frozen."
)

print(
    "=== END STEP 6.26E ==="
)

In [ ]:
# ============================================================
# Step 6.26E-R — Compact review summary
#
# Read-only summary of Step 6.26E.
# No corruption regeneration.
# No model inference.
# No TEST data.
# No files written.
# ============================================================

import numpy as np
import pandas as pd


print(
    "============================================================"
)
print(
    "STEP 6.26E-R — COMPACT SENSOR-QUALITY FUSION REVIEW"
)
print(
    "============================================================"
)


# ============================================================
# 1. Overall comparison
# ============================================================

overall_cols_626er = [
    "fusion",
    "scenarios",
    "mean_auc",
    "median_auc",
    "minimum_auc",
    "mean_standardized_shift",
    "fraction_auc_above_0_50",
    "fraction_positive_mean_shift",
]

overall_compact_626er = (
    SENSOR_QUALITY_FUSION_OVERALL_SUMMARY_626E[
        overall_cols_626er
    ]
    .copy()
)


print(
    "\n--- OVERALL FUSION COMPARISON ---"
)

print(
    overall_compact_626er
    .round(6)
    .to_string(index=False)
)


# ============================================================
# 2. Family-level comparison
# ============================================================

family_cols_626er = [
    "fusion",
    "family",
    "scenarios",
    "mean_auc",
    "minimum_auc",
    "mean_standardized_shift",
]


family_compact_626er = (
    SENSOR_QUALITY_FUSION_FAMILY_SUMMARY_626E[
        family_cols_626er
    ]
    .copy()
)


print(
    "\n--- FAMILY-LEVEL COMPARISON ---"
)

print(
    family_compact_626er
    .round(6)
    .to_string(index=False)
)


# ============================================================
# 3. Clean DEV -> clean-validation stability
# ============================================================

stability_cols_626er = [
    "fusion",
    "canonical_DEV",
    "clean_validation",
    "absolute_mean_shift_DEV_to_validation",
]


stability_compact_626er = (
    SENSOR_QUALITY_FUSION_CLEAN_STABILITY_626E[
        stability_cols_626er
    ]
    .copy()
)


print(
    "\n--- CLEAN DEV -> VALIDATION STABILITY ---"
)

print(
    stability_compact_626er
    .round(6)
    .to_string(index=False)
)


# ============================================================
# 4. Weakest single scenario for each fusion
# ============================================================

weakest_rows_626er = []


for fusion_626er in (
    SENSOR_QUALITY_FUSION_SCENARIO_RESULTS_626E[
        "fusion"
    ].unique()
):

    subset_626er = (
        SENSOR_QUALITY_FUSION_SCENARIO_RESULTS_626E[
            SENSOR_QUALITY_FUSION_SCENARIO_RESULTS_626E[
                "fusion"
            ]
            ==
            fusion_626er
        ]
        .sort_values(
            "auc_clean_vs_corrupted",
            ascending=True
        )
        .head(1)
    )

    weakest_rows_626er.append(
        subset_626er
    )


weakest_compact_626er = (
    pd.concat(
        weakest_rows_626er,
        ignore_index=True
    )[
        [
            "fusion",
            "global_index",
            "family",
            "scenario_id",
            "sensor",
            "severity",
            "auc_clean_vs_corrupted",
            "mean_delta",
            "standardized_mean_shift",
        ]
    ]
)


print(
    "\n--- WEAKEST SCENARIO PER FUSION ---"
)

print(
    weakest_compact_626er
    .round(6)
    .to_string(index=False)
)


# ============================================================
# 5. Number of weak / reversed scenarios
# ============================================================

diagnostic_rows_626er = []


for fusion_626er, group_626er in (
    SENSOR_QUALITY_FUSION_SCENARIO_RESULTS_626E
    .groupby("fusion")
):

    diagnostic_rows_626er.append(
        {
            "fusion":
                fusion_626er,

            "total_scenarios":
                int(len(group_626er)),

            "auc_below_0_50":
                int(
                    (
                        group_626er[
                            "auc_clean_vs_corrupted"
                        ]
                        <
                        0.50
                    ).sum()
                ),

            "auc_equal_0_50":
                int(
                    np.isclose(
                        group_626er[
                            "auc_clean_vs_corrupted"
                        ],
                        0.50,
                        atol=1e-12
                    ).sum()
                ),

            "negative_mean_shift":
                int(
                    (
                        group_626er[
                            "mean_delta"
                        ]
                        <
                        0
                    ).sum()
                ),

            "positive_mean_shift":
                int(
                    (
                        group_626er[
                            "mean_delta"
                        ]
                        >
                        0
                    ).sum()
                ),
        }
    )


diagnostic_compact_626er = pd.DataFrame(
    diagnostic_rows_626er
)


print(
    "\n--- RESPONSE-DIRECTION DIAGNOSTICS ---"
)

print(
    diagnostic_compact_626er
    .to_string(index=False)
)


# ============================================================
# 6. Simple ranking — diagnostic only
#
# IMPORTANT:
# This DOES NOT freeze the winner.
# ============================================================

ranking_626er = (
    SENSOR_QUALITY_FUSION_OVERALL_SUMMARY_626E[
        [
            "fusion",
            "mean_auc",
            "minimum_auc",
            "fraction_auc_above_0_50",
        ]
    ]
    .sort_values(
        [
            "mean_auc",
            "minimum_auc",
        ],
        ascending=False
    )
    .reset_index(drop=True)
)


ranking_626er.insert(
    0,
    "diagnostic_rank",
    np.arange(
        1,
        len(ranking_626er) + 1
    )
)


print(
    "\n--- DIAGNOSTIC RANKING ---"
)

print(
    ranking_626er
    .round(6)
    .to_string(index=False)
)


print(
    "\n------------------------------------------------------------"
)
print(
    "Scenarios evaluated          :",
    SENSOR_QUALITY_FUSION_SCENARIO_RESULTS_626E[
        "global_index"
    ].nunique()
)

print(
    "Fusion candidates evaluated  :",
    SENSOR_QUALITY_FUSION_SCENARIO_RESULTS_626E[
        "fusion"
    ].nunique()
)

print(
    "Expected scenario count      : 315"
)

print(
    "Expected fusion count        : 2"
)

print(
    "RUL labels used              : False"
)

print(
    "Official TEST used/scored    : False"
)

print(
    "Final fusion frozen          : False"
)

print(
    "Files written               : False"
)

print(
    "------------------------------------------------------------"
)

print(
    "=== END STEP 6.26E-R ==="
)

In [ ]:
# ============================================================
# Step 6.26F-A
# Identify exact Step 6.26E / 6.26E-R scenario-result objects
# READ-ONLY INSPECTION
# ============================================================

import pandas as pd

print(
    "=== Step 6.26F-A — Locate frozen fusion scenario tables ==="
)

records_626fa = []

for name_626fa, obj_626fa in list(globals().items()):

    if not isinstance(obj_626fa, pd.DataFrame):
        continue

    rows_626fa, cols_n_626fa = obj_626fa.shape

    # We are interested in plausible Step 6.26E/E-R outputs:
    # 315 rows  = one record per corruption scenario
    # 630 rows  = 315 scenarios × 2 fusion candidates
    if rows_626fa not in {315, 630}:
        continue

    cols_626fa = [str(c) for c in obj_626fa.columns]

    interesting_626fa = [
        c for c in cols_626fa
        if any(
            token_626fa in c.lower()
            for token_626fa in [
                "family",
                "scenario",
                "fusion",
                "auc",
                "shift",
                "severity",
                "sensor",
            ]
        )
    ]

    records_626fa.append(
        {
            "object_name": name_626fa,
            "rows": rows_626fa,
            "columns": cols_n_626fa,
            "relevant_columns": interesting_626fa,
        }
    )

if records_626fa:
    result_626fa = (
        pd.DataFrame(records_626fa)
        .sort_values(
            ["rows", "object_name"]
        )
        .reset_index(drop=True)
    )

    print("\nCandidate objects:\n")

    for _, row_626fa in result_626fa.iterrows():

        print("-" * 72)
        print("Object :", row_626fa["object_name"])
        print(
            "Shape  :",
            (
                row_626fa["rows"],
                row_626fa["columns"],
            ),
        )
        print(
            "Relevant columns:",
            row_626fa["relevant_columns"],
        )

else:
    print(
        "\nNo 315-row or 630-row DataFrame "
        "currently exists in notebook memory."
    )

print("\nNo data modified.")
print("No model inference executed.")
print("No corruption regenerated.")
print("No research decision modified.")
print("=== END STEP 6.26F-A ===")

In [ ]:
# ============================================================
# Step 6.26F-B
# Exact inventory of Step 6.26E / fusion-related memory objects
# READ-ONLY INSPECTION
# ============================================================

import pandas as pd
import numpy as np

print(
    "=== Step 6.26F-B — Exact fusion-object inventory ==="
)

# ------------------------------------------------------------
# 1. Search only names relevant to Step 6.26E / fusion
# ------------------------------------------------------------

search_tokens_626fb = [
    "626e",
    "fusion",
    "auc",
    "quality",
]

candidate_names_626fb = []

for name_626fb in sorted(globals().keys()):

    name_lower_626fb = name_626fb.lower()

    if any(
        token_626fb in name_lower_626fb
        for token_626fb in search_tokens_626fb
    ):
        candidate_names_626fb.append(name_626fb)


print(
    "\nFusion / Step-6.26E-related objects found:",
    len(candidate_names_626fb),
)

print(
    "\n--- Compact object inventory ---"
)

records_626fb = []

for name_626fb in candidate_names_626fb:

    obj_626fb = globals()[name_626fb]

    record_626fb = {
        "object_name": name_626fb,
        "type": type(obj_626fb).__name__,
        "shape": None,
        "length": None,
        "important_columns": None,
    }

    # DataFrame
    if isinstance(obj_626fb, pd.DataFrame):

        record_626fb["shape"] = tuple(
            obj_626fb.shape
        )

        important_cols_626fb = [
            str(c)
            for c in obj_626fb.columns
            if any(
                token_626fb in str(c).lower()
                for token_626fb in [
                    "family",
                    "scenario",
                    "fusion",
                    "auc",
                    "shift",
                    "severity",
                    "sensor",
                    "score",
                ]
            )
        ]

        record_626fb[
            "important_columns"
        ] = important_cols_626fb

    # Series
    elif isinstance(obj_626fb, pd.Series):

        record_626fb["shape"] = tuple(
            obj_626fb.shape
        )

    # Common containers
    elif isinstance(
        obj_626fb,
        (list, tuple, dict, set),
    ):

        try:
            record_626fb["length"] = len(
                obj_626fb
            )
        except Exception:
            pass

    records_626fb.append(
        record_626fb
    )


inventory_626fb = pd.DataFrame(
    records_626fb
)

if len(inventory_626fb) == 0:

    print(
        "\nNo fusion-related objects were found."
    )

else:

    # Print one compact line per object.
    for _, row_626fb in inventory_626fb.iterrows():

        print("-" * 72)

        print(
            "Object :",
            row_626fb["object_name"],
        )

        print(
            "Type   :",
            row_626fb["type"],
        )

        if row_626fb["shape"] is not None:
            print(
                "Shape  :",
                row_626fb["shape"],
            )

        if (
            row_626fb["length"] is not None
            and not pd.isna(
                row_626fb["length"]
            )
        ):
            print(
                "Length :",
                row_626fb["length"],
            )

        if (
            isinstance(
                row_626fb["important_columns"],
                list,
            )
            and len(
                row_626fb["important_columns"]
            ) > 0
        ):
            print(
                "Important columns:",
                row_626fb[
                    "important_columns"
                ],
            )


# ------------------------------------------------------------
# 2. Explicitly isolate plausible final fusion result tables
# ------------------------------------------------------------

print(
    "\n--- Plausible fusion-result DataFrames ---"
)

plausible_626fb = []

for name_626fb in candidate_names_626fb:

    obj_626fb = globals()[name_626fb]

    if not isinstance(
        obj_626fb,
        pd.DataFrame,
    ):
        continue

    cols_lower_626fb = {
        str(c).lower()
        for c in obj_626fb.columns
    }

    has_auc_626fb = any(
        "auc" in c
        for c in cols_lower_626fb
    )

    has_fusion_626fb = any(
        "fusion" in c
        for c in cols_lower_626fb
    )

    has_scenario_626fb = any(
        "scenario" in c
        for c in cols_lower_626fb
    )

    if (
        has_auc_626fb
        or has_fusion_626fb
        or (
            len(obj_626fb)
            in {315, 630}
            and has_scenario_626fb
        )
    ):

        plausible_626fb.append(
            (
                name_626fb,
                obj_626fb.shape,
                list(obj_626fb.columns),
            )
        )


if plausible_626fb:

    for (
        name_626fb,
        shape_626fb,
        cols_626fb,
    ) in plausible_626fb:

        print("\nObject :", name_626fb)
        print("Shape  :", shape_626fb)
        print("Columns:", cols_626fb)

else:

    print(
        "NONE FOUND"
    )


print(
    "\nNo model inference executed."
)

print(
    "No MC-dropout executed."
)

print(
    "No corruption regenerated."
)

print(
    "No threshold selected."
)

print(
    "No research decision modified."
)

print(
    "No files written."
)

print(
    "=== END STEP 6.26F-B ==="
)

In [ ]:
# ============================================================
# Step 6.26F-C
# Minimal structural search for the frozen Step 6.26E
# scenario-level sensor-quality fusion result
#
# READ-ONLY
# - no inference
# - no corruption generation
# - no threshold selection
# - no file writing
# ============================================================

import pandas as pd
import numpy as np

print(
    "=== Step 6.26F-C — Minimal frozen fusion-table search ==="
)

# ------------------------------------------------------------
# 1. Search DataFrames only
# ------------------------------------------------------------

candidate_rows_626fc = []

for name_626fc, obj_626fc in list(globals().items()):

    if not isinstance(
        obj_626fc,
        pd.DataFrame,
    ):
        continue

    # Ignore objects created by the current 6.26F audit itself.
    # We want the pre-existing Step 6.26E result.
    if "626f" in name_626fc.lower():
        continue

    cols_626fc = [
        str(c)
        for c in obj_626fc.columns
    ]

    cols_lower_626fc = [
        c.lower()
        for c in cols_626fc
    ]

    # Exact structural clues expected from the Step 6.26E
    # scenario-level fusion evaluation.
    auc_cols_626fc = [
        c
        for c in cols_626fc
        if "auc" in c.lower()
    ]

    fusion_cols_626fc = [
        c
        for c in cols_626fc
        if "fusion" in c.lower()
    ]

    shift_cols_626fc = [
        c
        for c in cols_626fc
        if "shift" in c.lower()
    ]

    scenario_cols_626fc = [
        c
        for c in cols_626fc
        if "scenario" in c.lower()
    ]

    family_cols_626fc = [
        c
        for c in cols_626fc
        if c.lower() == "family"
    ]

    # Strong candidate rules:
    #
    # A) contains AUC information; or
    # B) explicitly contains a fusion column; or
    # C) has 630 rows and scenario/family information
    #    (315 scenarios × 2 fusion candidates).
    #
    strong_candidate_626fc = (
        len(auc_cols_626fc) > 0
        or len(fusion_cols_626fc) > 0
        or (
            len(obj_626fc) == 630
            and len(scenario_cols_626fc) > 0
            and len(family_cols_626fc) > 0
        )
    )

    if not strong_candidate_626fc:
        continue

    # Ranking score only for inspection.
    score_626fc = 0

    if len(obj_626fc) == 630:
        score_626fc += 10

    if len(obj_626fc) == 315:
        score_626fc += 3

    if len(auc_cols_626fc) > 0:
        score_626fc += 10

    if len(fusion_cols_626fc) > 0:
        score_626fc += 10

    if len(shift_cols_626fc) > 0:
        score_626fc += 3

    if "626e" in name_626fc.lower():
        score_626fc += 5

    candidate_rows_626fc.append(
        {
            "object_name": name_626fc,
            "shape": tuple(
                obj_626fc.shape
            ),
            "score": score_626fc,
            "auc_columns": auc_cols_626fc,
            "fusion_columns": fusion_cols_626fc,
            "shift_columns": shift_cols_626fc,
            "scenario_columns": scenario_cols_626fc,
            "has_family": (
                len(family_cols_626fc) > 0
            ),
        }
    )


# ------------------------------------------------------------
# 2. Rank candidates
# ------------------------------------------------------------

candidates_626fc = pd.DataFrame(
    candidate_rows_626fc
)

if candidates_626fc.empty:

    print(
        "\nRESULT: No plausible scenario-level "
        "fusion DataFrame is currently present in memory."
    )

else:

    candidates_626fc = (
        candidates_626fc
        .sort_values(
            by=[
                "score",
                "object_name",
            ],
            ascending=[
                False,
                True,
            ],
        )
        .reset_index(
            drop=True
        )
    )

    print(
        "\nPlausible candidate count:",
        len(candidates_626fc),
    )

    print(
        "\n--- Ranked compact candidates ---"
    )

    # Print only top candidates.
    for idx_626fc, row_626fc in (
        candidates_626fc
        .head(15)
        .iterrows()
    ):

        print(
            f"\n#{idx_626fc + 1}"
        )

        print(
            "Object :",
            row_626fc[
                "object_name"
            ],
        )

        print(
            "Shape  :",
            row_626fc[
                "shape"
            ],
        )

        print(
            "Score  :",
            row_626fc[
                "score"
            ],
        )

        print(
            "AUC cols:",
            row_626fc[
                "auc_columns"
            ],
        )

        print(
            "Fusion cols:",
            row_626fc[
                "fusion_columns"
            ],
        )

        print(
            "Shift cols:",
            row_626fc[
                "shift_columns"
            ],
        )

        print(
            "Scenario cols:",
            row_626fc[
                "scenario_columns"
            ],
        )

        print(
            "Family column:",
            row_626fc[
                "has_family"
            ],
        )


# ------------------------------------------------------------
# 3. Special check: exact 630-row DataFrames
# ------------------------------------------------------------

exact_630_626fc = []

for name_626fc, obj_626fc in list(
    globals().items()
):

    if not isinstance(
        obj_626fc,
        pd.DataFrame,
    ):
        continue

    if "626f" in name_626fc.lower():
        continue

    if len(obj_626fc) != 630:
        continue

    cols_626fc = [
        str(c)
        for c in obj_626fc.columns
    ]

    exact_630_626fc.append(
        {
            "object_name": name_626fc,
            "shape": tuple(
                obj_626fc.shape
            ),
            "columns": cols_626fc,
        }
    )


print(
    "\n--- ALL pre-existing 630-row DataFrames ---"
)

if len(exact_630_626fc) == 0:

    print(
        "NONE"
    )

else:

    for record_626fc in exact_630_626fc:

        print(
            "\nObject :",
            record_626fc[
                "object_name"
            ],
        )

        print(
            "Shape  :",
            record_626fc[
                "shape"
            ],
        )

        print(
            "Columns:",
            record_626fc[
                "columns"
            ],
        )


# ------------------------------------------------------------
# 4. Integrity statement
# ------------------------------------------------------------

print(
    "\n--- Integrity ---"
)

print(
    "Model inference executed : False"
)

print(
    "MC-dropout executed      : False"
)

print(
    "Corruption regenerated   : False"
)

print(
    "RUL labels inspected     : False"
)

print(
    "Threshold selected       : False"
)

print(
    "Research decision changed: False"
)

print(
    "Files written            : False"
)

print(
    "\n=== END STEP 6.26F-C ==="
)

In [ ]:
# ============================================================
# Step 6.26F-D
# Exact frozen Step 6.26E fusion-result schema confirmation
#
# READ-ONLY
# No model inference
# No corruption generation
# No scoring rerun
# No threshold/fusion decision
# No file writing
# ============================================================

import pandas as pd
import numpy as np

print(
    "=== Step 6.26F-D — Exact frozen fusion-result confirmation ==="
)

# ------------------------------------------------------------
# 1. Use the EXACT object found in Step 6.26F-C
# ------------------------------------------------------------

SOURCE_NAME_626FD = (
    "SENSOR_QUALITY_FUSION_SCENARIO_RESULTS_626E"
)

if SOURCE_NAME_626FD not in globals():
    raise RuntimeError(
        f"Required frozen object not found: "
        f"{SOURCE_NAME_626FD}"
    )

fusion_results_626fd = globals()[
    SOURCE_NAME_626FD
]

if not isinstance(
    fusion_results_626fd,
    pd.DataFrame,
):
    raise TypeError(
        f"{SOURCE_NAME_626FD} is not a DataFrame."
    )


# ------------------------------------------------------------
# 2. Structural checks
# ------------------------------------------------------------

print("\n--- Frozen source ---")
print("Object :", SOURCE_NAME_626FD)
print("Shape  :", fusion_results_626fd.shape)

print("\n--- Exact columns ---")

for i_626fd, col_626fd in enumerate(
    fusion_results_626fd.columns,
    start=1,
):
    print(
        f"{i_626fd:02d}. {col_626fd}"
    )


# ------------------------------------------------------------
# 3. Fusion identities
# ------------------------------------------------------------

if "fusion" not in fusion_results_626fd.columns:
    raise RuntimeError(
        "Required column 'fusion' is absent."
    )

fusion_counts_626fd = (
    fusion_results_626fd[
        "fusion"
    ]
    .value_counts(
        dropna=False
    )
)

print("\n--- Fusion counts ---")
print(
    fusion_counts_626fd.to_string()
)


# ------------------------------------------------------------
# 4. Family × fusion counts
# ------------------------------------------------------------

if "family" not in fusion_results_626fd.columns:
    raise RuntimeError(
        "Required column 'family' is absent."
    )

family_fusion_counts_626fd = (
    fusion_results_626fd
    .groupby(
        [
            "fusion",
            "family",
        ],
        dropna=False,
    )
    .size()
    .rename("rows")
    .reset_index()
)

print(
    "\n--- Family × fusion counts ---"
)

print(
    family_fusion_counts_626fd
    .to_string(
        index=False
    )
)


# ------------------------------------------------------------
# 5. Scenario identity integrity
# ------------------------------------------------------------

scenario_key_candidates_626fd = [
    c
    for c in [
        "global_index",
        "scenario_id",
        "manifest_scenario_id",
    ]
    if c in fusion_results_626fd.columns
]

print(
    "\n--- Scenario identity fields ---"
)

print(
    scenario_key_candidates_626fd
)


if "scenario_id" in fusion_results_626fd.columns:

    unique_scenarios_626fd = int(
        fusion_results_626fd[
            "scenario_id"
        ].nunique(
            dropna=False
        )
    )

    print(
        "Unique scenario_id values:",
        unique_scenarios_626fd,
    )


if (
    "scenario_id"
    in fusion_results_626fd.columns
    and "fusion"
    in fusion_results_626fd.columns
):

    duplicate_pairs_626fd = int(
        fusion_results_626fd
        .duplicated(
            subset=[
                "scenario_id",
                "fusion",
            ]
        )
        .sum()
    )

    print(
        "Duplicate scenario_id × fusion pairs:",
        duplicate_pairs_626fd,
    )


# ------------------------------------------------------------
# 6. Critical numeric integrity
# ------------------------------------------------------------

critical_numeric_626fd = [
    c
    for c in [
        "auc_clean_vs_corrupted",
        "standardized_mean_shift",
    ]
    if c in fusion_results_626fd.columns
]

print(
    "\n--- Critical numeric integrity ---"
)

for col_626fd in critical_numeric_626fd:

    values_626fd = pd.to_numeric(
        fusion_results_626fd[
            col_626fd
        ],
        errors="coerce",
    )

    print(
        f"{col_626fd}: "
        f"finite={bool(np.isfinite(values_626fd).all())}, "
        f"min={values_626fd.min():.6f}, "
        f"max={values_626fd.max():.6f}, "
        f"mean={values_626fd.mean():.6f}"
    )


# ------------------------------------------------------------
# 7. Final integrity summary
# ------------------------------------------------------------

shape_exact_626fd = (
    fusion_results_626fd.shape[0]
    == 630
)

fusion_count_exact_626fd = (
    fusion_results_626fd[
        "fusion"
    ].nunique(
        dropna=False
    )
    == 2
)

scenario_count_exact_626fd = True

if "scenario_id" in fusion_results_626fd.columns:
    scenario_count_exact_626fd = (
        fusion_results_626fd[
            "scenario_id"
        ].nunique(
            dropna=False
        )
        == 315
    )


print(
    "\n==============================================="
)
print(
    "STEP 6.26F-D INTEGRITY SUMMARY"
)
print(
    "==============================================="
)

print(
    "Frozen source located          : True"
)
print(
    "630 rows present               :",
    shape_exact_626fd,
)
print(
    "Exactly 2 fusion candidates    :",
    fusion_count_exact_626fd,
)
print(
    "315 unique scenarios           :",
    scenario_count_exact_626fd,
)

print(
    "\nModel inference executed        : False"
)
print(
    "MC-dropout executed             : False"
)
print(
    "Corruption regenerated          : False"
)
print(
    "Official TEST labels inspected  : False"
)
print(
    "Official TEST scored            : False"
)
print(
    "Fusion decision modified        : False"
)
print(
    "Threshold selected              : False"
)
print(
    "Files written                   : False"
)

print(
    "\n=== END STEP 6.26F-D ==="
)

In [ ]:
# ============================================================
# Step 6.26F-D2
# Minimal final schema/integrity confirmation
# READ-ONLY
# ============================================================

import numpy as np
import pandas as pd

df_626fd2 = SENSOR_QUALITY_FUSION_SCENARIO_RESULTS_626E

print("=== Step 6.26F-D2 — Minimal final confirmation ===")

print("\nLast columns:")
for i, col in enumerate(
    df_626fd2.columns[-5:],
    start=len(df_626fd2.columns) - 4
):
    print(f"{i:02d}. {col}")

print("\nFusion counts:")
print(
    df_626fd2["fusion"]
    .value_counts(dropna=False)
    .to_string()
)

print("\nFamily × fusion counts:")
print(
    df_626fd2
    .groupby(["fusion", "family"])
    .size()
    .to_string()
)

print("\nIntegrity:")
print(
    "Rows                     :",
    len(df_626fd2)
)
print(
    "Unique scenarios         :",
    df_626fd2["global_index"].nunique()
)
print(
    "Fusion candidates        :",
    df_626fd2["fusion"].nunique()
)
print(
    "Duplicate scenario×fusion:",
    df_626fd2.duplicated(
        ["global_index", "fusion"]
    ).sum()
)

for col in [
    "standardized_mean_shift",
    "auc_clean_vs_corrupted",
]:
    if col in df_626fd2.columns:
        x = pd.to_numeric(
            df_626fd2[col],
            errors="coerce"
        )

        print(
            f"{col} finite:",
            bool(np.isfinite(x).all())
        )

print("\nResearch decision modified : False")
print("Threshold selected         : False")
print("Files written              : False")

print("\n=== END STEP 6.26F-D2 ===")

In [ ]:
# ============================================================
# Step 6.26F-E
# Sensor-quality fusion robustness audit
#
# PURPOSE
# -------
# Determine whether the apparent advantage of MAX fusion over
# MEAN fusion is broad and robust across corruption families,
# severities, and sensors, rather than being driven by one
# particular corruption family.
#
# IMPORTANT
# ---------
# - READ-ONLY diagnostic analysis
# - No threshold selection
# - No final fusion frozen
# - No official TEST labels
# - No model inference
# - No corruption regeneration
# - No files written
# ============================================================

import numpy as np
import pandas as pd

print(
    "=== Step 6.26F-E — "
    "Sensor-quality fusion robustness audit ==="
)

# ------------------------------------------------------------
# 1. Recover authoritative frozen Step 6.26E table
# ------------------------------------------------------------

if (
    "SENSOR_QUALITY_FUSION_SCENARIO_RESULTS_626E"
    not in globals()
):
    raise RuntimeError(
        "Required frozen Step 6.26E table is missing."
    )

source_626fe = (
    SENSOR_QUALITY_FUSION_SCENARIO_RESULTS_626E
    .copy()
)

print("\n--- Frozen source ---")
print(
    "Object : SENSOR_QUALITY_FUSION_SCENARIO_RESULTS_626E"
)
print("Shape  :", source_626fe.shape)


# ------------------------------------------------------------
# 2. Exact schema requirements
# ------------------------------------------------------------

required_columns_626fe = [
    "global_index",
    "family",
    "family_position",
    "scenario_id",
    "manifest_scenario_id",
    "sensor",
    "severity",
    "fusion",
    "windows",
    "clean_mean",
    "corrupted_mean",
    "mean_delta",
    "median_delta",
    "mean_abs_delta",
    "standardized_mean_shift",
    "positive_window_fraction",
    "negative_window_fraction",
    "unchanged_window_fraction",
    "above_clean_q95_fraction",
    "auc_clean_vs_corrupted",
]

missing_columns_626fe = [
    c
    for c in required_columns_626fe
    if c not in source_626fe.columns
]

if missing_columns_626fe:
    raise RuntimeError(
        "Missing required Step 6.26E columns: "
        + str(missing_columns_626fe)
    )


# ------------------------------------------------------------
# 3. Frozen-table integrity checks
# ------------------------------------------------------------

MEAN_FUSION_626FE = "sq_candidate_mean_fusion"
MAX_FUSION_626FE = "sq_candidate_max_fusion"

expected_fusions_626fe = {
    MEAN_FUSION_626FE,
    MAX_FUSION_626FE,
}

fusion_values_626fe = set(
    source_626fe["fusion"]
    .astype(str)
    .unique()
)

duplicate_pairs_626fe = int(
    source_626fe.duplicated(
        subset=["global_index", "fusion"]
    ).sum()
)

scenario_count_626fe = int(
    source_626fe["global_index"].nunique()
)

family_count_626fe = int(
    source_626fe["family"].nunique()
)

fusion_count_626fe = int(
    source_626fe["fusion"].nunique()
)

print("\n--- Frozen-table integrity ---")
print(
    "Rows                     :",
    len(source_626fe),
)
print(
    "Unique scenarios         :",
    scenario_count_626fe,
)
print(
    "Fusion candidates        :",
    fusion_count_626fe,
)
print(
    "Corruption families      :",
    family_count_626fe,
)
print(
    "Duplicate scenario×fusion:",
    duplicate_pairs_626fe,
)

if len(source_626fe) != 630:
    raise RuntimeError(
        "Expected exactly 630 fusion-result rows."
    )

if scenario_count_626fe != 315:
    raise RuntimeError(
        "Expected exactly 315 unique scenarios."
    )

if fusion_count_626fe != 2:
    raise RuntimeError(
        "Expected exactly two fusion candidates."
    )

if fusion_values_626fe != expected_fusions_626fe:
    raise RuntimeError(
        "Unexpected fusion identities: "
        + str(fusion_values_626fe)
    )

if duplicate_pairs_626fe != 0:
    raise RuntimeError(
        "Duplicate scenario × fusion records found."
    )


# ------------------------------------------------------------
# 4. Numeric integrity
# ------------------------------------------------------------

numeric_columns_626fe = [
    "windows",
    "clean_mean",
    "corrupted_mean",
    "mean_delta",
    "median_delta",
    "mean_abs_delta",
    "standardized_mean_shift",
    "positive_window_fraction",
    "negative_window_fraction",
    "unchanged_window_fraction",
    "above_clean_q95_fraction",
    "auc_clean_vs_corrupted",
]

for col_626fe in numeric_columns_626fe:
    numeric_values_626fe = pd.to_numeric(
        source_626fe[col_626fe],
        errors="coerce",
    )

    if not np.isfinite(
        numeric_values_626fe.to_numpy(
            dtype=float
        )
    ).all():
        raise RuntimeError(
            f"Non-finite values found in {col_626fe}."
        )

print(
    "All required numeric values finite:",
    True,
)


# ------------------------------------------------------------
# 5. Confirm both fusions contain exactly same scenarios
# ------------------------------------------------------------

mean_df_626fe = (
    source_626fe[
        source_626fe["fusion"]
        == MEAN_FUSION_626FE
    ]
    .sort_values("global_index")
    .reset_index(drop=True)
)

max_df_626fe = (
    source_626fe[
        source_626fe["fusion"]
        == MAX_FUSION_626FE
    ]
    .sort_values("global_index")
    .reset_index(drop=True)
)

if len(mean_df_626fe) != 315:
    raise RuntimeError(
        "MEAN fusion does not contain 315 scenarios."
    )

if len(max_df_626fe) != 315:
    raise RuntimeError(
        "MAX fusion does not contain 315 scenarios."
    )

if not np.array_equal(
    mean_df_626fe["global_index"].to_numpy(),
    max_df_626fe["global_index"].to_numpy(),
):
    raise RuntimeError(
        "MEAN and MAX fusion scenario indices differ."
    )

metadata_columns_626fe = [
    "family",
    "family_position",
    "scenario_id",
    "manifest_scenario_id",
    "sensor",
    "severity",
]

for col_626fe in metadata_columns_626fe:

    left_626fe = (
        mean_df_626fe[col_626fe]
        .astype(object)
        .where(
            mean_df_626fe[col_626fe].notna(),
            "__MISSING__",
        )
        .astype(str)
        .to_numpy()
    )

    right_626fe = (
        max_df_626fe[col_626fe]
        .astype(object)
        .where(
            max_df_626fe[col_626fe].notna(),
            "__MISSING__",
        )
        .astype(str)
        .to_numpy()
    )

    if not np.array_equal(
        left_626fe,
        right_626fe,
    ):
        raise RuntimeError(
            "Scenario metadata mismatch between "
            f"fusions for column: {col_626fe}"
        )

print(
    "Scenario pairing exact        :",
    True,
)


# ------------------------------------------------------------
# 6. Construct paired comparison table
# ------------------------------------------------------------

paired_626fe = (
    max_df_626fe[
        [
            "global_index",
            "family",
            "family_position",
            "scenario_id",
            "manifest_scenario_id",
            "sensor",
            "severity",
        ]
    ]
    .copy()
)

paired_626fe["mean_auc"] = (
    mean_df_626fe[
        "auc_clean_vs_corrupted"
    ]
    .astype(float)
    .to_numpy()
)

paired_626fe["max_auc"] = (
    max_df_626fe[
        "auc_clean_vs_corrupted"
    ]
    .astype(float)
    .to_numpy()
)

paired_626fe["auc_delta_max_minus_mean"] = (
    paired_626fe["max_auc"]
    - paired_626fe["mean_auc"]
)

paired_626fe["mean_standardized_shift"] = (
    mean_df_626fe[
        "standardized_mean_shift"
    ]
    .astype(float)
    .to_numpy()
)

paired_626fe["max_standardized_shift"] = (
    max_df_626fe[
        "standardized_mean_shift"
    ]
    .astype(float)
    .to_numpy()
)

paired_626fe[
    "shift_delta_max_minus_mean"
] = (
    paired_626fe["max_standardized_shift"]
    - paired_626fe["mean_standardized_shift"]
)

paired_626fe["mean_above_clean_q95"] = (
    mean_df_626fe[
        "above_clean_q95_fraction"
    ]
    .astype(float)
    .to_numpy()
)

paired_626fe["max_above_clean_q95"] = (
    max_df_626fe[
        "above_clean_q95_fraction"
    ]
    .astype(float)
    .to_numpy()
)

paired_626fe[
    "q95_delta_max_minus_mean"
] = (
    paired_626fe["max_above_clean_q95"]
    - paired_626fe["mean_above_clean_q95"]
)


# ------------------------------------------------------------
# 7. Helper for compact paired summaries
# ------------------------------------------------------------

TOL_626FE = 1e-12


def summarize_groups_626fe(
    dataframe,
    group_columns,
):

    rows_626fe = []

    grouped_626fe = dataframe.groupby(
        group_columns,
        dropna=False,
        sort=True,
    )

    for key_626fe, group_626fe in grouped_626fe:

        if not isinstance(
            key_626fe,
            tuple,
        ):
            key_626fe = (key_626fe,)

        row_626fe = {
            col_626fe: value_626fe
            for col_626fe, value_626fe
            in zip(
                group_columns,
                key_626fe,
            )
        }

        auc_delta_626fe = (
            group_626fe[
                "auc_delta_max_minus_mean"
            ]
            .to_numpy(dtype=float)
        )

        shift_delta_626fe = (
            group_626fe[
                "shift_delta_max_minus_mean"
            ]
            .to_numpy(dtype=float)
        )

        q95_delta_626fe = (
            group_626fe[
                "q95_delta_max_minus_mean"
            ]
            .to_numpy(dtype=float)
        )

        row_626fe.update(
            {
                "scenarios": len(group_626fe),

                "mean_mean_auc":
                    group_626fe[
                        "mean_auc"
                    ].mean(),

                "mean_max_auc":
                    group_626fe[
                        "max_auc"
                    ].mean(),

                "mean_auc_delta":
                    auc_delta_626fe.mean(),

                "median_auc_delta":
                    np.median(
                        auc_delta_626fe
                    ),

                "max_auc_win_fraction":
                    np.mean(
                        auc_delta_626fe
                        > TOL_626FE
                    ),

                "auc_tie_fraction":
                    np.mean(
                        np.abs(
                            auc_delta_626fe
                        )
                        <= TOL_626FE
                    ),

                "mean_shift_delta":
                    shift_delta_626fe.mean(),

                "mean_q95_delta":
                    q95_delta_626fe.mean(),
            }
        )

        rows_626fe.append(
            row_626fe
        )

    return pd.DataFrame(
        rows_626fe
    )


# ------------------------------------------------------------
# 8. Overall paired comparison
# ------------------------------------------------------------

auc_delta_all_626fe = (
    paired_626fe[
        "auc_delta_max_minus_mean"
    ]
    .to_numpy(dtype=float)
)

shift_delta_all_626fe = (
    paired_626fe[
        "shift_delta_max_minus_mean"
    ]
    .to_numpy(dtype=float)
)

q95_delta_all_626fe = (
    paired_626fe[
        "q95_delta_max_minus_mean"
    ]
    .to_numpy(dtype=float)
)

print(
    "\n--- OVERALL PAIRED FUSION COMPARISON ---"
)

print(
    "Mean AUC — MEAN fusion :",
    round(
        paired_626fe[
            "mean_auc"
        ].mean(),
        6,
    ),
)

print(
    "Mean AUC — MAX fusion  :",
    round(
        paired_626fe[
            "max_auc"
        ].mean(),
        6,
    ),
)

print(
    "Mean AUC delta "
    "(MAX - MEAN)          :",
    round(
        auc_delta_all_626fe.mean(),
        6,
    ),
)

print(
    "Median AUC delta       :",
    round(
        np.median(
            auc_delta_all_626fe
        ),
        6,
    ),
)

print(
    "MAX AUC wins           :",
    int(
        np.sum(
            auc_delta_all_626fe
            > TOL_626FE
        )
    ),
    "/ 315",
)

print(
    "AUC ties               :",
    int(
        np.sum(
            np.abs(
                auc_delta_all_626fe
            )
            <= TOL_626FE
        )
    ),
    "/ 315",
)

print(
    "MEAN AUC wins          :",
    int(
        np.sum(
            auc_delta_all_626fe
            < -TOL_626FE
        )
    ),
    "/ 315",
)

print(
    "Mean standardized-shift delta:",
    round(
        shift_delta_all_626fe.mean(),
        6,
    ),
)

print(
    "Mean above-q95 delta          :",
    round(
        q95_delta_all_626fe.mean(),
        6,
    ),
)


# ------------------------------------------------------------
# 9. Family robustness
# ------------------------------------------------------------

family_summary_626fe = (
    summarize_groups_626fe(
        paired_626fe,
        ["family"],
    )
)

print(
    "\n--- FAMILY-LEVEL ROBUSTNESS ---"
)

print(
    family_summary_626fe[
        [
            "family",
            "scenarios",
            "mean_mean_auc",
            "mean_max_auc",
            "mean_auc_delta",
            "median_auc_delta",
            "max_auc_win_fraction",
            "mean_shift_delta",
            "mean_q95_delta",
        ]
    ]
    .round(6)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 10. Severity robustness
# ------------------------------------------------------------

severity_summary_626fe = (
    summarize_groups_626fe(
        paired_626fe,
        ["severity"],
    )
)

print(
    "\n--- SEVERITY-LEVEL ROBUSTNESS ---"
)

print(
    severity_summary_626fe[
        [
            "severity",
            "scenarios",
            "mean_mean_auc",
            "mean_max_auc",
            "mean_auc_delta",
            "max_auc_win_fraction",
            "mean_shift_delta",
        ]
    ]
    .round(6)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 11. Family × severity robustness
# ------------------------------------------------------------

family_severity_summary_626fe = (
    summarize_groups_626fe(
        paired_626fe,
        [
            "family",
            "severity",
        ],
    )
)

family_severity_sorted_626fe = (
    family_severity_summary_626fe
    .sort_values(
        [
            "mean_auc_delta",
            "family",
            "severity",
        ]
    )
    .reset_index(drop=True)
)

print(
    "\n--- WORST FAMILY × SEVERITY "
    "SUBGROUPS BY AUC DELTA ---"
)

print(
    family_severity_sorted_626fe[
        [
            "family",
            "severity",
            "scenarios",
            "mean_auc_delta",
            "median_auc_delta",
            "max_auc_win_fraction",
        ]
    ]
    .head(8)
    .round(6)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 12. Sensor-level robustness
# ------------------------------------------------------------

sensor_summary_626fe = (
    summarize_groups_626fe(
        paired_626fe,
        ["sensor"],
    )
)

negative_sensor_groups_626fe = (
    sensor_summary_626fe[
        sensor_summary_626fe[
            "mean_auc_delta"
        ]
        < -TOL_626FE
    ]
)

print(
    "\n--- SENSOR-LEVEL ROBUSTNESS ---"
)

print(
    "Sensor groups checked    :",
    len(sensor_summary_626fe),
)

print(
    "Negative mean-AUC groups :",
    len(
        negative_sensor_groups_626fe
    ),
)

print(
    "\nWorst five sensor groups:"
)

print(
    sensor_summary_626fe[
        [
            "sensor",
            "scenarios",
            "mean_auc_delta",
            "max_auc_win_fraction",
        ]
    ]
    .sort_values(
        "mean_auc_delta"
    )
    .head(5)
    .round(6)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 13. Leave-one-family-out robustness
#
# Critical test:
# If MAX fusion only looks better because of one family,
# removing that family should eliminate/reverse the advantage.
# ------------------------------------------------------------

loo_rows_626fe = []

family_values_626fe = sorted(
    paired_626fe[
        "family"
    ]
    .astype(str)
    .unique()
)

for excluded_family_626fe in family_values_626fe:

    remaining_626fe = (
        paired_626fe[
            paired_626fe["family"]
            .astype(str)
            != excluded_family_626fe
        ]
    )

    delta_626fe = (
        remaining_626fe[
            "auc_delta_max_minus_mean"
        ]
        .to_numpy(dtype=float)
    )

    loo_rows_626fe.append(
        {
            "excluded_family":
                excluded_family_626fe,

            "remaining_scenarios":
                len(remaining_626fe),

            "mean_auc_delta":
                delta_626fe.mean(),

            "median_auc_delta":
                np.median(
                    delta_626fe
                ),

            "max_auc_win_fraction":
                np.mean(
                    delta_626fe
                    > TOL_626FE
                ),
        }
    )

loo_family_626fe = pd.DataFrame(
    loo_rows_626fe
)

print(
    "\n--- LEAVE-ONE-FAMILY-OUT ROBUSTNESS ---"
)

print(
    loo_family_626fe
    .round(6)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 14. Compact diagnostic verdict
#
# NOTE:
# These are evidence flags only.
# They DO NOT freeze a fusion rule.
# ------------------------------------------------------------

overall_auc_positive_626fe = bool(
    auc_delta_all_626fe.mean()
    > TOL_626FE
)

all_family_positive_626fe = bool(
    (
        family_summary_626fe[
            "mean_auc_delta"
        ]
        > TOL_626FE
    ).all()
)

all_severity_positive_626fe = bool(
    (
        severity_summary_626fe[
            "mean_auc_delta"
        ]
        > TOL_626FE
    ).all()
)

all_loo_positive_626fe = bool(
    (
        loo_family_626fe[
            "mean_auc_delta"
        ]
        > TOL_626FE
    ).all()
)

overall_shift_positive_626fe = bool(
    shift_delta_all_626fe.mean()
    > TOL_626FE
)

overall_q95_positive_626fe = bool(
    q95_delta_all_626fe.mean()
    > TOL_626FE
)

broad_descriptive_support_626fe = bool(
    overall_auc_positive_626fe
    and all_family_positive_626fe
    and all_severity_positive_626fe
    and all_loo_positive_626fe
)

print(
    "\n"
    + "=" * 66
)
print(
    "STEP 6.26F-E COMPACT ROBUSTNESS SUMMARY"
)
print(
    "=" * 66
)

print(
    "Overall MAX AUC advantage          :",
    overall_auc_positive_626fe,
)

print(
    "Positive mean AUC delta all families:",
    all_family_positive_626fe,
)

print(
    "Positive mean AUC delta all severities:",
    all_severity_positive_626fe,
)

print(
    "Positive leave-one-family-out delta:",
    all_loo_positive_626fe,
)

print(
    "Overall standardized shift advantage:",
    overall_shift_positive_626fe,
)

print(
    "Overall above-clean-q95 advantage   :",
    overall_q95_positive_626fe,
)

print(
    "Negative sensor groups              :",
    len(
        negative_sensor_groups_626fe
    ),
)

print(
    "Broad descriptive support for MAX   :",
    broad_descriptive_support_626fe,
)

print(
    "\nNo threshold selected."
)
print(
    "No fusion rule frozen."
)
print(
    "No model inference executed."
)
print(
    "No corruption regenerated."
)
print(
    "No official TEST labels used."
)
print(
    "No files written."
)

print(
    "\n=== END STEP 6.26F-E ==="
)

In [ ]:
# ============================================================
# Step 6.26F-E2
# Compact retrieval of already-computed robustness evidence
# NO recomputation
# ============================================================

print(
    "=== Step 6.26F-E2 — "
    "Compact fusion robustness decision evidence ==="
)

required_objects_626fe2 = [
    "family_summary_626fe",
    "severity_summary_626fe",
    "loo_family_626fe",
    "sensor_summary_626fe",
    "paired_626fe",
]

missing_626fe2 = [
    name
    for name in required_objects_626fe2
    if name not in globals()
]

print("\n--- Required in-memory results ---")
for name in required_objects_626fe2:
    print(
        f"{name:<28}:",
        name in globals(),
    )

if missing_626fe2:
    raise RuntimeError(
        "Required Step 6.26F-E results are missing: "
        + str(missing_626fe2)
    )

# ------------------------------------------------------------
# 1. Family evidence
# ------------------------------------------------------------

print("\n--- FAMILY ROBUSTNESS ---")

family_compact_626fe2 = (
    family_summary_626fe[
        [
            "family",
            "scenarios",
            "mean_mean_auc",
            "mean_max_auc",
            "mean_auc_delta",
            "max_auc_win_fraction",
        ]
    ]
    .copy()
)

print(
    family_compact_626fe2
    .round(6)
    .to_string(index=False)
)

# ------------------------------------------------------------
# 2. Severity evidence
# ------------------------------------------------------------

print("\n--- SEVERITY ROBUSTNESS ---")

severity_compact_626fe2 = (
    severity_summary_626fe[
        [
            "severity",
            "scenarios",
            "mean_mean_auc",
            "mean_max_auc",
            "mean_auc_delta",
            "max_auc_win_fraction",
        ]
    ]
    .copy()
)

print(
    severity_compact_626fe2
    .round(6)
    .to_string(index=False)
)

# ------------------------------------------------------------
# 3. Leave-one-family-out evidence
# ------------------------------------------------------------

print("\n--- LEAVE-ONE-FAMILY-OUT ROBUSTNESS ---")

print(
    loo_family_626fe[
        [
            "excluded_family",
            "remaining_scenarios",
            "mean_auc_delta",
            "median_auc_delta",
            "max_auc_win_fraction",
        ]
    ]
    .round(6)
    .to_string(index=False)
)

# ------------------------------------------------------------
# 4. Sensor evidence
# ------------------------------------------------------------

negative_sensor_626fe2 = (
    sensor_summary_626fe[
        sensor_summary_626fe[
            "mean_auc_delta"
        ] < -1e-12
    ]
)

print("\n--- SENSOR ROBUSTNESS ---")

print(
    "Sensor groups checked       :",
    len(sensor_summary_626fe),
)

print(
    "Negative mean-delta sensors :",
    len(negative_sensor_626fe2),
)

print("\nWorst five sensors:")

print(
    sensor_summary_626fe[
        [
            "sensor",
            "scenarios",
            "mean_auc_delta",
            "max_auc_win_fraction",
        ]
    ]
    .sort_values("mean_auc_delta")
    .head(5)
    .round(6)
    .to_string(index=False)
)

# ------------------------------------------------------------
# 5. Final compact evidence flags
# ------------------------------------------------------------

overall_delta_626fe2 = float(
    paired_626fe[
        "auc_delta_max_minus_mean"
    ].mean()
)

all_family_positive_626fe2 = bool(
    (
        family_summary_626fe[
            "mean_auc_delta"
        ] > 0
    ).all()
)

all_severity_positive_626fe2 = bool(
    (
        severity_summary_626fe[
            "mean_auc_delta"
        ] > 0
    ).all()
)

all_loo_positive_626fe2 = bool(
    (
        loo_family_626fe[
            "mean_auc_delta"
        ] > 0
    ).all()
)

max_wins_626fe2 = int(
    (
        paired_626fe[
            "auc_delta_max_minus_mean"
        ] > 0
    ).sum()
)

mean_wins_626fe2 = int(
    (
        paired_626fe[
            "auc_delta_max_minus_mean"
        ] < 0
    ).sum()
)

ties_626fe2 = int(
    (
        np.abs(
            paired_626fe[
                "auc_delta_max_minus_mean"
            ]
        ) <= 1e-12
    ).sum()
)

print("\n" + "=" * 62)
print("STEP 6.26F-E2 FINAL COMPACT EVIDENCE")
print("=" * 62)

print(
    "Overall mean AUC delta MAX-MEAN :",
    round(overall_delta_626fe2, 6),
)

print(
    "MAX / MEAN / tie scenario wins :",
    max_wins_626fe2,
    "/",
    mean_wins_626fe2,
    "/",
    ties_626fe2,
)

print(
    "All family mean deltas positive:",
    all_family_positive_626fe2,
)

print(
    "All severity mean deltas positive:",
    all_severity_positive_626fe2,
)

print(
    "All leave-one-family-out positive:",
    all_loo_positive_626fe2,
)

print(
    "Negative sensor groups          :",
    len(negative_sensor_626fe2),
)

print(
    "\nThreshold selected              : False"
)
print(
    "Fusion frozen                   : False"
)
print(
    "Official TEST labels used       : False"
)
print(
    "Files written                   : False"
)

print("\n=== END STEP 6.26F-E2 ===")

In [ ]:
# ============================================================
# Step 6.26F-E3
# Final tiny evidence check before freezing fusion
# ============================================================

print(
    "=== Step 6.26F-E3 — "
    "Final fusion evidence check ==="
)

# ------------------------------------------------------------
# 1. Leave-one-family-out
# ------------------------------------------------------------

print("\n--- LEAVE-ONE-FAMILY-OUT ---")
print("Shape:", loo_family_626fe.shape)

if len(loo_family_626fe) > 0:
    cols_loo_626fe3 = [
        c for c in [
            "excluded_family",
            "remaining_scenarios",
            "mean_auc_delta",
            "median_auc_delta",
            "max_auc_win_fraction",
        ]
        if c in loo_family_626fe.columns
    ]

    print(
        loo_family_626fe[
            cols_loo_626fe3
        ]
        .round(6)
        .to_string(index=False)
    )
else:
    print("WARNING: leave-one-family-out table is empty.")

# ------------------------------------------------------------
# 2. Sensor-level robustness
# ------------------------------------------------------------

print("\n--- SENSOR-LEVEL CHECK ---")

print("Sensor groups:", len(sensor_summary_626fe))

negative_sensor_626fe3 = (
    sensor_summary_626fe[
        sensor_summary_626fe[
            "mean_auc_delta"
        ] < -1e-12
    ]
    .copy()
)

print(
    "Sensors with negative mean MAX-MEAN AUC delta:",
    len(negative_sensor_626fe3),
)

if len(negative_sensor_626fe3) > 0:
    show_cols_626fe3 = [
        c for c in [
            "sensor",
            "scenarios",
            "mean_auc_delta",
            "max_auc_win_fraction",
        ]
        if c in negative_sensor_626fe3.columns
    ]

    print(
        negative_sensor_626fe3[
            show_cols_626fe3
        ]
        .sort_values("mean_auc_delta")
        .round(6)
        .to_string(index=False)
    )

# ------------------------------------------------------------
# 3. Final boolean evidence
# ------------------------------------------------------------

family_positive_626fe3 = bool(
    (
        family_summary_626fe[
            "mean_auc_delta"
        ] > 0
    ).all()
)

severity_positive_626fe3 = bool(
    (
        severity_summary_626fe[
            "mean_auc_delta"
        ] > 0
    ).all()
)

loo_available_626fe3 = (
    len(loo_family_626fe) > 0
)

if (
    loo_available_626fe3
    and "mean_auc_delta" in loo_family_626fe.columns
):
    loo_positive_626fe3 = bool(
        (
            loo_family_626fe[
                "mean_auc_delta"
            ] > 0
        ).all()
    )
else:
    loo_positive_626fe3 = False

overall_delta_626fe3 = float(
    paired_626fe[
        "auc_delta_max_minus_mean"
    ].mean()
)

max_wins_626fe3 = int(
    (
        paired_626fe[
            "auc_delta_max_minus_mean"
        ] > 0
    ).sum()
)

mean_wins_626fe3 = int(
    (
        paired_626fe[
            "auc_delta_max_minus_mean"
        ] < 0
    ).sum()
)

ties_626fe3 = (
    len(paired_626fe)
    - max_wins_626fe3
    - mean_wins_626fe3
)

print("\n" + "=" * 60)
print("FINAL FUSION EVIDENCE")
print("=" * 60)

print(
    "Overall mean AUC delta MAX-MEAN :",
    round(overall_delta_626fe3, 6),
)

print(
    "Scenario wins MAX / MEAN / ties :",
    max_wins_626fe3,
    "/",
    mean_wins_626fe3,
    "/",
    ties_626fe3,
)

print(
    "All family mean deltas positive :",
    family_positive_626fe3,
)

print(
    "All severity mean deltas positive:",
    severity_positive_626fe3,
)

print(
    "LOO table available             :",
    loo_available_626fe3,
)

print(
    "All LOO mean deltas positive    :",
    loo_positive_626fe3,
)

print(
    "Negative sensor groups          :",
    len(negative_sensor_626fe3),
)

print("\nOfficial TEST labels used       : False")
print("Threshold selected              : False")
print("Fusion frozen                   : False")
print("Files written                   : False")

print("\n=== END STEP 6.26F-E3 ===")

In [ ]:
# ============================================================
# Step 6.26F — Sensor-quality fusion robustness audit
# ============================================================
#
# PURPOSE
# -------
# Final DEV-only robustness check before selecting/freezeing the
# sensor-quality fusion rule.
#
# This cell checks:
#   1. Clean Canonical-DEV -> clean-validation stability
#   2. Scenario-wise MAX-fusion vs MEAN-fusion AUC comparison
#   3. Robustness across corruption severity
#   4. Robustness across family x severity
#
# IMPORTANT
# ---------
# - NO official TEST data are used.
# - NO official TEST labels are used.
# - NO RUL labels are used.
# - NO model inference is executed.
# - NO MC-dropout is executed.
# - NO corruption scenarios are regenerated.
# - NO final fusion rule is frozen.
# - NO files are written.
#
# ============================================================

import numpy as np
import pandas as pd


print(
    "\n"
    "=== Step 6.26F — Sensor-quality fusion robustness audit ==="
)


# ============================================================
# 1. Constants
# ============================================================

MAX_FUSION_626F = "sq_candidate_max_fusion"
MEAN_FUSION_626F = "sq_candidate_mean_fusion"

EXPECTED_DEV_WINDOWS_626F = 14241
EXPECTED_VAL_WINDOWS_626F = 3490
EXPECTED_SCENARIOS_626F = 315


# ============================================================
# 2. Utility helpers
# ============================================================

def _df_objects_626f():
    """
    Return all currently available pandas DataFrames.
    """
    return {
        name: obj
        for name, obj in globals().items()
        if isinstance(obj, pd.DataFrame)
    }


def _same_core_content_626f(
    frames,
    columns,
):
    """
    Check whether multiple candidate DataFrames contain the same
    values for selected columns.

    Used only to avoid false ambiguity when notebook history
    contains duplicate copies of the same table.
    """

    if len(frames) <= 1:
        return True

    base = (
        frames[0][columns]
        .reset_index(drop=True)
        .copy()
    )

    for frame in frames[1:]:

        candidate = (
            frame[columns]
            .reset_index(drop=True)
            .copy()
        )

        if base.shape != candidate.shape:
            return False

        try:
            pd.testing.assert_frame_equal(
                base,
                candidate,
                check_dtype=False,
                check_exact=False,
                rtol=1e-12,
                atol=1e-12,
            )
        except AssertionError:
            return False

    return True


def _choose_scored_clean_frame_626f(
    expected_rows,
    label,
):
    """
    Recover a scored clean-window DataFrame using an exact
    structural contract.

    Required columns:
      - sq_candidate_mean_fusion
      - sq_candidate_max_fusion

    Expected row count is also enforced.
    """

    dfs = _df_objects_626f()

    candidates = []

    for name, frame in dfs.items():

        required_cols = {
            MEAN_FUSION_626F,
            MAX_FUSION_626F,
        }

        if not required_cols.issubset(frame.columns):
            continue

        if len(frame) != expected_rows:
            continue

        candidates.append(
            (name, frame)
        )

    if len(candidates) == 0:
        raise RuntimeError(
            f"No valid {label} scored-window DataFrame "
            f"was found with {expected_rows} rows."
        )

    names = [
        item[0]
        for item in candidates
    ]

    frames = [
        item[1]
        for item in candidates
    ]

    if len(frames) > 1:

        identical = _same_core_content_626f(
            frames,
            [
                MEAN_FUSION_626F,
                MAX_FUSION_626F,
            ],
        )

        if not identical:
            raise RuntimeError(
                f"Multiple different {label} scored-window "
                f"DataFrames were found:\n{names}\n"
                "Refusing to guess."
            )

    return (
        names[0],
        frames[0].copy(),
    )


def _resolve_column_626f(
    frame,
    aliases,
):
    """
    Resolve one required semantic field from a list of aliases.
    """

    for col in aliases:
        if col in frame.columns:
            return col

    return None


def _recover_scenario_results_626f():
    """
    Recover the Step 6.26E scenario-level fusion result table.

    We require:
      fusion
      family
      severity
      AUC

    The expected full table contains results for both fusion
    candidates across the frozen 315 scenarios.
    """

    dfs = _df_objects_626f()

    recovered = []

    for name, frame in dfs.items():

        if len(frame) < EXPECTED_SCENARIOS_626F:
            continue

        fusion_col = _resolve_column_626f(
            frame,
            [
                "fusion",
                "fusion_name",
                "fusion_candidate",
                "fusion_score",
            ],
        )

        family_col = _resolve_column_626f(
            frame,
            [
                "family",
                "corruption_family",
            ],
        )

        severity_col = _resolve_column_626f(
            frame,
            [
                "severity",
                "corruption_severity",
            ],
        )

        auc_col = _resolve_column_626f(
            frame,
            [
                "auc",
                "scenario_auc",
                "roc_auc",
                "quality_auc",
            ],
        )

        if (
            fusion_col is None
            or family_col is None
            or severity_col is None
            or auc_col is None
        ):
            continue

        fusion_values = set(
            frame[fusion_col]
            .dropna()
            .astype(str)
            .unique()
        )

        if not {
            MAX_FUSION_626F,
            MEAN_FUSION_626F,
        }.issubset(fusion_values):
            continue

        standardized = pd.DataFrame(
            {
                "fusion":
                    frame[fusion_col].astype(str),

                "family":
                    frame[family_col].astype(str),

                "severity":
                    frame[severity_col].astype(str),

                "auc":
                    pd.to_numeric(
                        frame[auc_col],
                        errors="coerce",
                    ),
            }
        )

        # Optional scenario identity fields
        for possible_col in [
            "global_index",
            "scenario_id",
            "manifest_scenario_id",
            "sensor",
        ]:
            if possible_col in frame.columns:
                standardized[possible_col] = (
                    frame[possible_col].values
                )

        standardized = standardized[
            standardized["fusion"].isin(
                [
                    MAX_FUSION_626F,
                    MEAN_FUSION_626F,
                ]
            )
        ].copy()

        standardized = standardized[
            np.isfinite(
                standardized["auc"]
            )
        ].copy()

        count_by_fusion = (
            standardized
            .groupby("fusion")
            .size()
            .to_dict()
        )

        if (
            count_by_fusion.get(
                MAX_FUSION_626F,
                0,
            )
            == EXPECTED_SCENARIOS_626F
            and
            count_by_fusion.get(
                MEAN_FUSION_626F,
                0,
            )
            == EXPECTED_SCENARIOS_626F
        ):
            recovered.append(
                (
                    name,
                    standardized,
                )
            )

    if len(recovered) == 0:
        raise RuntimeError(
            "Could not recover a valid 315-scenario x "
            "2-fusion Step 6.26E result table.\n"
            "No research decision was modified."
        )

    if len(recovered) > 1:

        core_columns = [
            "fusion",
            "family",
            "severity",
            "auc",
        ]

        frames = [
            item[1]
            .sort_values(core_columns)
            .reset_index(drop=True)
            for item in recovered
        ]

        identical = _same_core_content_626f(
            frames,
            core_columns,
        )

        if not identical:

            candidate_info = [
                (
                    name,
                    df.shape,
                )
                for name, df in recovered
            ]

            raise RuntimeError(
                "More than one non-identical valid scenario "
                "result table was found:\n"
                f"{candidate_info}\n"
                "Refusing to guess."
            )

    return (
        recovered[0][0],
        recovered[0][1].copy(),
    )


def _ks_distance_626f(
    x,
    y,
):
    """
    Two-sample empirical KS distance.

    We use only the distance statistic here.
    No p-value or hypothesis-testing claim is made.
    """

    x = np.asarray(
        x,
        dtype=float,
    )

    y = np.asarray(
        y,
        dtype=float,
    )

    x = x[
        np.isfinite(x)
    ]

    y = y[
        np.isfinite(y)
    ]

    combined = np.sort(
        np.unique(
            np.concatenate(
                [x, y]
            )
        )
    )

    cdf_x = (
        np.searchsorted(
            np.sort(x),
            combined,
            side="right",
        )
        / len(x)
    )

    cdf_y = (
        np.searchsorted(
            np.sort(y),
            combined,
            side="right",
        )
        / len(y)
    )

    return float(
        np.max(
            np.abs(
                cdf_x - cdf_y
            )
        )
    )


def _wasserstein_quantile_626f(
    x,
    y,
    grid_size=1001,
):
    """
    Simple quantile-integral approximation of Wasserstein-1
    distance.

    Fusion values are already in [0,1], so this distance is
    directly interpretable on the score scale.
    """

    x = np.asarray(
        x,
        dtype=float,
    )

    y = np.asarray(
        y,
        dtype=float,
    )

    x = x[
        np.isfinite(x)
    ]

    y = y[
        np.isfinite(y)
    ]

    q = np.linspace(
        0.0,
        1.0,
        grid_size,
    )

    qx = np.quantile(
        x,
        q,
    )

    qy = np.quantile(
        y,
        q,
    )

    return float(
        np.mean(
            np.abs(
                qx - qy
            )
        )
    )


# ============================================================
# 3. Recover CLEAN DEV and clean-validation scored windows
# ============================================================

print(
    "\n--- Recovering clean score populations ---"
)

dev_name_626f, dev_scores_626f = (
    _choose_scored_clean_frame_626f(
        EXPECTED_DEV_WINDOWS_626F,
        "Canonical DEV",
    )
)

# ------------------------------------------------------------
# Clean-validation provenance is already frozen by Step 6.26D.
# Do NOT recover it by scanning notebook memory because later
# Step 6.26E created several temporary 3490-row scored tables.
# ------------------------------------------------------------

if "SENSOR_QUALITY_VAL_CLEAN_SCORED_626D" not in globals():
    raise RuntimeError(
        "Required canonical clean-validation object "
        "SENSOR_QUALITY_VAL_CLEAN_SCORED_626D is missing."
    )

val_name_626f = "SENSOR_QUALITY_VAL_CLEAN_SCORED_626D"

val_scores_626f = (
    SENSOR_QUALITY_VAL_CLEAN_SCORED_626D
    .copy(deep=True)
)

# Exact frozen structural checks
if len(val_scores_626f) != EXPECTED_VAL_WINDOWS_626F:
    raise RuntimeError(
        "Canonical clean-validation window count mismatch: "
        f"expected {EXPECTED_VAL_WINDOWS_626F}, "
        f"found {len(val_scores_626f)}."
    )

required_fusion_cols_626f = {
    MAX_FUSION_626F,
    MEAN_FUSION_626F,
}

missing_fusion_cols_626f = (
    required_fusion_cols_626f
    -
    set(val_scores_626f.columns)
)

if missing_fusion_cols_626f:
    raise RuntimeError(
        "Canonical clean-validation scored table is missing "
        f"required fusion columns: {sorted(missing_fusion_cols_626f)}"
    )

val_fusion_values_626f = (
    val_scores_626f[
        [
            MAX_FUSION_626F,
            MEAN_FUSION_626F,
        ]
    ]
    .to_numpy(dtype=float)
)

if not np.isfinite(val_fusion_values_626f).all():
    raise RuntimeError(
        "Canonical clean-validation fusion scores contain "
        "non-finite values."
    )

if (
    (val_fusion_values_626f < 0.0).any()
    or
    (val_fusion_values_626f > 1.0).any()
):
    raise RuntimeError(
        "Canonical clean-validation fusion scores are outside "
        "the expected [0, 1] interval."
    )

print(
    "Canonical DEV source      :",
    dev_name_626f,
)

print(
    "Canonical DEV windows     :",
    len(dev_scores_626f),
)

print(
    "Clean validation source   :",
    val_name_626f,
)

print(
    "Clean validation windows  :",
    len(val_scores_626f),
)


# ============================================================
# 4. Clean DEV -> validation distribution stability
# ============================================================

clean_stability_rows_626f = []

for fusion_626f in [
    MAX_FUSION_626F,
    MEAN_FUSION_626F,
]:

    dev_values_626f = (
        pd.to_numeric(
            dev_scores_626f[
                fusion_626f
            ],
            errors="coerce",
        )
        .to_numpy(
            dtype=float
        )
    )

    val_values_626f = (
        pd.to_numeric(
            val_scores_626f[
                fusion_626f
            ],
            errors="coerce",
        )
        .to_numpy(
            dtype=float
        )
    )

    if (
        not np.isfinite(
            dev_values_626f
        ).all()
        or
        not np.isfinite(
            val_values_626f
        ).all()
    ):
        raise RuntimeError(
            f"Non-finite clean fusion values detected "
            f"for {fusion_626f}."
        )

    dev_q95_626f = float(
        np.quantile(
            dev_values_626f,
            0.95,
        )
    )

    dev_q99_626f = float(
        np.quantile(
            dev_values_626f,
            0.99,
        )
    )

    clean_stability_rows_626f.append(
        {
            "fusion":
                fusion_626f,

            "DEV_mean":
                float(
                    np.mean(
                        dev_values_626f
                    )
                ),

            "VAL_mean":
                float(
                    np.mean(
                        val_values_626f
                    )
                ),

            "abs_mean_shift":
                float(
                    abs(
                        np.mean(
                            val_values_626f
                        )
                        -
                        np.mean(
                            dev_values_626f
                        )
                    )
                ),

            "DEV_median":
                float(
                    np.median(
                        dev_values_626f
                    )
                ),

            "VAL_median":
                float(
                    np.median(
                        val_values_626f
                    )
                ),

            "abs_median_shift":
                float(
                    abs(
                        np.median(
                            val_values_626f
                        )
                        -
                        np.median(
                            dev_values_626f
                        )
                    )
                ),

            "KS_distance":
                _ks_distance_626f(
                    dev_values_626f,
                    val_values_626f,
                ),

            "W1_distance":
                _wasserstein_quantile_626f(
                    dev_values_626f,
                    val_values_626f,
                ),

            "DEV_q95":
                dev_q95_626f,

            "VAL_fraction_above_DEV_q95":
                float(
                    np.mean(
                        val_values_626f
                        >
                        dev_q95_626f
                    )
                ),

            "DEV_q99":
                dev_q99_626f,

            "VAL_fraction_above_DEV_q99":
                float(
                    np.mean(
                        val_values_626f
                        >
                        dev_q99_626f
                    )
                ),
        }
    )


CLEAN_FUSION_STABILITY_626F = (
    pd.DataFrame(
        clean_stability_rows_626f
    )
    .set_index(
        "fusion"
    )
)


# ============================================================
# 5. Recover full frozen-DEV scenario fusion results
# ============================================================

print(
    "\n--- Recovering 315-scenario fusion results ---"
)

scenario_source_name_626f, scenario_results_626f = (
    _recover_scenario_results_626f()
)

print(
    "Scenario-result source :",
    scenario_source_name_626f,
)

print(
    "Scenario rows recovered:",
    len(
        scenario_results_626f
    ),
)

print(
    "Rows per fusion:",
    scenario_results_626f
    .groupby(
        "fusion"
    )
    .size()
    .to_dict(),
)


# ============================================================
# 6. Overall fusion robustness
# ============================================================

overall_rows_626f = []

for fusion_626f in [
    MAX_FUSION_626F,
    MEAN_FUSION_626F,
]:

    block_626f = (
        scenario_results_626f[
            scenario_results_626f[
                "fusion"
            ]
            ==
            fusion_626f
        ]
        .copy()
    )

    auc_values_626f = (
        block_626f[
            "auc"
        ]
        .to_numpy(
            dtype=float
        )
    )

    overall_rows_626f.append(
        {
            "fusion":
                fusion_626f,

            "scenarios":
                len(
                    block_626f
                ),

            "mean_auc":
                float(
                    np.mean(
                        auc_values_626f
                    )
                ),

            "median_auc":
                float(
                    np.median(
                        auc_values_626f
                    )
                ),

            "minimum_auc":
                float(
                    np.min(
                        auc_values_626f
                    )
                ),

            "q10_auc":
                float(
                    np.quantile(
                        auc_values_626f,
                        0.10,
                    )
                ),

            "fraction_auc_above_0_50":
                float(
                    np.mean(
                        auc_values_626f
                        >
                        0.50
                    )
                ),

            "scenarios_below_0_50":
                int(
                    np.sum(
                        auc_values_626f
                        <
                        0.50
                    )
                ),
        }
    )


FUSION_OVERALL_ROBUSTNESS_626F = (
    pd.DataFrame(
        overall_rows_626f
    )
    .set_index(
        "fusion"
    )
)


# ============================================================
# 7. Scenario-wise paired MAX vs MEAN comparison
# ============================================================

scenario_key_candidates_626f = [
    "global_index",
    "scenario_id",
    "manifest_scenario_id",
]

scenario_key_626f = None

for candidate_626f in (
    scenario_key_candidates_626f
):

    if (
        candidate_626f
        in scenario_results_626f.columns
    ):
        scenario_key_626f = (
            candidate_626f
        )
        break


if scenario_key_626f is not None:

    pivot_626f = (
        scenario_results_626f
        .pivot_table(
            index=[
                scenario_key_626f,
                "family",
                "severity",
            ],
            columns="fusion",
            values="auc",
            aggfunc="first",
        )
        .reset_index()
    )

else:

    # Safe deterministic pairing inside family/severity only
    # if no explicit scenario identifier survived.
    paired_blocks_626f = []

    for (
        family_626f,
        severity_626f,
    ), block_626f in (
        scenario_results_626f
        .groupby(
            [
                "family",
                "severity",
            ],
            sort=True,
        )
    ):

        max_block_626f = (
            block_626f[
                block_626f["fusion"]
                ==
                MAX_FUSION_626F
            ]
            .reset_index(
                drop=True
            )
        )

        mean_block_626f = (
            block_626f[
                block_626f["fusion"]
                ==
                MEAN_FUSION_626F
            ]
            .reset_index(
                drop=True
            )
        )

        if (
            len(max_block_626f)
            !=
            len(mean_block_626f)
        ):
            raise RuntimeError(
                "Cannot safely pair fusion scenarios "
                f"inside {family_626f}/{severity_626f}."
            )

        temp_626f = pd.DataFrame(
            {
                "family":
                    family_626f,

                "severity":
                    severity_626f,

                MAX_FUSION_626F:
                    max_block_626f[
                        "auc"
                    ].to_numpy(),

                MEAN_FUSION_626F:
                    mean_block_626f[
                        "auc"
                    ].to_numpy(),
            }
        )

        paired_blocks_626f.append(
            temp_626f
        )

    pivot_626f = pd.concat(
        paired_blocks_626f,
        ignore_index=True,
    )


required_pivot_cols_626f = {
    MAX_FUSION_626F,
    MEAN_FUSION_626F,
}

if not required_pivot_cols_626f.issubset(
    pivot_626f.columns
):
    raise RuntimeError(
        "Paired fusion comparison could not be "
        "constructed safely."
    )


pivot_626f[
    "auc_delta_max_minus_mean"
] = (
    pivot_626f[
        MAX_FUSION_626F
    ]
    -
    pivot_626f[
        MEAN_FUSION_626F
    ]
)


delta_values_626f = (
    pivot_626f[
        "auc_delta_max_minus_mean"
    ]
    .to_numpy(
        dtype=float
    )
)

PAIRWISE_FUSION_SUMMARY_626F = {
    "paired_scenarios":
        int(
            len(
                pivot_626f
            )
        ),

    "MAX_better":
        int(
            np.sum(
                delta_values_626f
                >
                1e-12
            )
        ),

    "MEAN_better":
        int(
            np.sum(
                delta_values_626f
                <
                -1e-12
            )
        ),

    "effectively_equal":
        int(
            np.sum(
                np.abs(
                    delta_values_626f
                )
                <=
                1e-12
            )
        ),

    "mean_AUC_delta":
        float(
            np.mean(
                delta_values_626f
            )
        ),

    "median_AUC_delta":
        float(
            np.median(
                delta_values_626f
            )
        ),

    "q10_AUC_delta":
        float(
            np.quantile(
                delta_values_626f,
                0.10,
            )
        ),

    "q90_AUC_delta":
        float(
            np.quantile(
                delta_values_626f,
                0.90,
            )
        ),
}


# ============================================================
# 8. Severity-level robustness
# ============================================================

SEVERITY_FUSION_ROBUSTNESS_626F = (
    scenario_results_626f
    .groupby(
        [
            "severity",
            "fusion",
        ],
        as_index=False,
    )
    .agg(
        scenarios=(
            "auc",
            "size",
        ),
        mean_auc=(
            "auc",
            "mean",
        ),
        median_auc=(
            "auc",
            "median",
        ),
        minimum_auc=(
            "auc",
            "min",
        ),
    )
)


severity_pivot_626f = (
    SEVERITY_FUSION_ROBUSTNESS_626F
    .pivot(
        index="severity",
        columns="fusion",
        values="mean_auc",
    )
)

severity_pivot_626f[
    "max_minus_mean"
] = (
    severity_pivot_626f[
        MAX_FUSION_626F
    ]
    -
    severity_pivot_626f[
        MEAN_FUSION_626F
    ]
)

SEVERITY_COMPARISON_626F = (
    severity_pivot_626f
    .reset_index()
)


# ============================================================
# 9. Family x severity robustness
# ============================================================

family_severity_summary_626f = (
    scenario_results_626f
    .groupby(
        [
            "family",
            "severity",
            "fusion",
        ],
        as_index=False,
    )
    .agg(
        scenarios=(
            "auc",
            "size",
        ),
        mean_auc=(
            "auc",
            "mean",
        ),
    )
)


family_severity_pivot_626f = (
    family_severity_summary_626f
    .pivot_table(
        index=[
            "family",
            "severity",
        ],
        columns="fusion",
        values="mean_auc",
        aggfunc="first",
    )
    .reset_index()
)


family_severity_pivot_626f[
    "max_minus_mean"
] = (
    family_severity_pivot_626f[
        MAX_FUSION_626F
    ]
    -
    family_severity_pivot_626f[
        MEAN_FUSION_626F
    ]
)


FAMILY_SEVERITY_COMPARISON_626F = (
    family_severity_pivot_626f
    .sort_values(
        [
            "family",
            "severity",
        ]
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 10. Weakest scenarios for each fusion
# ============================================================

weakest_rows_626f = []

for fusion_626f in [
    MAX_FUSION_626F,
    MEAN_FUSION_626F,
]:

    block_626f = (
        scenario_results_626f[
            scenario_results_626f[
                "fusion"
            ]
            ==
            fusion_626f
        ]
        .sort_values(
            "auc",
            ascending=True,
        )
        .head(5)
        .copy()
    )

    keep_cols_626f = [
        col
        for col in [
            "fusion",
            "global_index",
            "scenario_id",
            "manifest_scenario_id",
            "family",
            "severity",
            "sensor",
            "auc",
        ]
        if col in block_626f.columns
    ]

    weakest_rows_626f.append(
        block_626f[
            keep_cols_626f
        ]
    )


WEAKEST_FUSION_SCENARIOS_626F = (
    pd.concat(
        weakest_rows_626f,
        ignore_index=True,
    )
)


# ============================================================
# 11. Integrity checks
# ============================================================

all_clean_finite_626f = bool(
    np.isfinite(
        dev_scores_626f[
            [
                MAX_FUSION_626F,
                MEAN_FUSION_626F,
            ]
        ]
        .to_numpy(
            dtype=float
        )
    ).all()
    and
    np.isfinite(
        val_scores_626f[
            [
                MAX_FUSION_626F,
                MEAN_FUSION_626F,
            ]
        ]
        .to_numpy(
            dtype=float
        )
    ).all()
)

all_auc_finite_626f = bool(
    np.isfinite(
        scenario_results_626f[
            "auc"
        ]
        .to_numpy(
            dtype=float
        )
    ).all()
)

scenario_counts_exact_626f = bool(
    (
        scenario_results_626f
        .groupby(
            "fusion"
        )
        .size()
        ==
        EXPECTED_SCENARIOS_626F
    ).all()
)


# ============================================================
# 12. Compact display
# ============================================================

print(
    "\n"
    "==============================================="
)

print(
    "STEP 6.26F — COMPACT ROBUSTNESS REVIEW"
)

print(
    "==============================================="
)


print(
    "\n--- CLEAN DEV -> VALIDATION STABILITY ---"
)

print(
    CLEAN_FUSION_STABILITY_626F[
        [
            "DEV_mean",
            "VAL_mean",
            "abs_mean_shift",
            "KS_distance",
            "W1_distance",
            "VAL_fraction_above_DEV_q95",
            "VAL_fraction_above_DEV_q99",
        ]
    ].round(6).to_string()
)


print(
    "\n--- OVERALL 315-SCENARIO ROBUSTNESS ---"
)

print(
    FUSION_OVERALL_ROBUSTNESS_626F
    .round(6)
    .to_string()
)


print(
    "\n--- PAIRED SCENARIO COMPARISON ---"
)

for (
    key_626f,
    value_626f,
) in (
    PAIRWISE_FUSION_SUMMARY_626F.items()
):

    print(
        f"{key_626f:24s}: "
        f"{value_626f}"
    )


print(
    "\n--- SEVERITY ROBUSTNESS ---"
)

print(
    SEVERITY_COMPARISON_626F
    .round(6)
    .to_string(
        index=False
    )
)


print(
    "\n--- FAMILY x SEVERITY: MAX minus MEAN AUC ---"
)

compact_family_severity_626f = (
    FAMILY_SEVERITY_COMPARISON_626F[
        [
            "family",
            "severity",
            MAX_FUSION_626F,
            MEAN_FUSION_626F,
            "max_minus_mean",
        ]
    ]
)

print(
    compact_family_severity_626f
    .round(6)
    .to_string(
        index=False
    )
)


print(
    "\n--- FIVE WEAKEST SCENARIOS PER FUSION ---"
)

print(
    WEAKEST_FUSION_SCENARIOS_626F
    .round(6)
    .to_string(
        index=False
    )
)


# ============================================================
# 13. Final safety / integrity summary
# ============================================================

print(
    "\n"
    "================================================="
)

print(
    "STEP 6.26F FINAL INTEGRITY SUMMARY"
)

print(
    "================================================="
)

print(
    "Canonical DEV windows               :",
    len(
        dev_scores_626f
    ),
)

print(
    "Clean validation windows            :",
    len(
        val_scores_626f
    ),
)

print(
    "Scenario results per fusion exact   :",
    scenario_counts_exact_626f,
)

print(
    "All clean fusion values finite      :",
    all_clean_finite_626f,
)

print(
    "All scenario AUC values finite      :",
    all_auc_finite_626f,
)

print(
    "RUL labels used                     : False"
)

print(
    "Official TEST labels inspected      : False"
)

print(
    "Official TEST scored                : False"
)

print(
    "Model inference executed            : False"
)

print(
    "MC-dropout executed                 : False"
)

print(
    "Corruptions regenerated             : False"
)

print(
    "Final sensor-quality fusion frozen  : False"
)

print(
    "Frozen reliability rule modified    : False"
)

print(
    "Files written                       : False"
)

print(
    "\nNo frozen research decision was modified."
)

print(
    "\n=== END STEP 6.26F ==="
)

In [ ]:
# ================================================================
# Step 6.26G — Freeze final sensor-quality representation / fusion
# ================================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

import numpy as np
import pandas as pd


print(
    "\n"
    "===============================================================\n"
    "STEP 6.26G — FREEZE FINAL SENSOR-QUALITY REPRESENTATION / FUSION\n"
    "==============================================================="
)


# ================================================================
# 1. Required frozen source
# ================================================================

REQUIRED_OBJECT_626G = "SENSOR_QUALITY_FUSION_SCENARIO_RESULTS_626E"

if REQUIRED_OBJECT_626G not in globals():
    raise RuntimeError(
        "Step 6.26G cannot continue.\n"
        "Required frozen DEV fusion table is missing:\n"
        f"{REQUIRED_OBJECT_626G}"
    )

fusion_source_626g = globals()[REQUIRED_OBJECT_626G]

if not isinstance(fusion_source_626g, pd.DataFrame):
    raise TypeError(
        f"{REQUIRED_OBJECT_626G} must be a pandas DataFrame."
    )

fusion_source_626g = fusion_source_626g.copy()


print("\n--- Frozen source ---")
print("Object :", REQUIRED_OBJECT_626G)
print("Shape  :", fusion_source_626g.shape)


# ================================================================
# 2. Exact frozen specification established in Steps 6.26B–6.26F
# ================================================================

SELECTED_RAW_FEATURES_626G = [
    "sq_w_excursion_mean",
    "sq_w_jump_mean_z",
    "sq_w_repeat_fraction",
]

SELECTED_COMPONENTS_626G = [
    "sq_w_excursion_mean_anomaly",
    "sq_w_jump_mean_z_anomaly",
    "sq_w_repeat_fraction_anomaly",
]

MEAN_FUSION_NAME_626G = "sq_candidate_mean_fusion"
MAX_FUSION_NAME_626G = "sq_candidate_max_fusion"

FINAL_FUSION_NAME_626G = MAX_FUSION_NAME_626G
FINAL_FUSION_RULE_626G = "maximum"

WINDOW_LENGTH_626G = 30

EXPECTED_SCENARIOS_626G = 315
EXPECTED_ROWS_626G = 630
EXPECTED_FUSIONS_626G = 2

EXPECTED_FAMILY_COUNTS_626G = {
    "bias": 90,
    "noise": 45,
    "drift": 90,
    "dropout": 45,
    "stuck": 45,
}


# If Step 6.26D objects are still available, verify that we are
# freezing exactly the already-selected representation.

if "SENSOR_QUALITY_SELECTED_FEATURES_626D" in globals():
    earlier_features_626g = list(
        globals()["SENSOR_QUALITY_SELECTED_FEATURES_626D"]
    )

    if earlier_features_626g != SELECTED_RAW_FEATURES_626G:
        raise RuntimeError(
            "Selected feature mismatch between Step 6.26D "
            "and Step 6.26G.\n"
            f"6.26D: {earlier_features_626g}\n"
            f"6.26G: {SELECTED_RAW_FEATURES_626G}"
        )


print("\n--- Sensor-quality representation to be frozen ---")
print("Raw quality channels:")
for feature_626g in SELECTED_RAW_FEATURES_626G:
    print("  -", feature_626g)

print("\nCalibrated components:")
for component_626g in SELECTED_COMPONENTS_626G:
    print("  -", component_626g)

print("\nFusion rule :", FINAL_FUSION_RULE_626G)
print("Fusion name :", FINAL_FUSION_NAME_626G)
print("Window      :", WINDOW_LENGTH_626G)


# ================================================================
# 3. Frozen fusion-table structural validation
# ================================================================

required_columns_626g = [
    "global_index",
    "family",
    "scenario_id",
    "sensor",
    "severity",
    "fusion",
    "auc_clean_vs_corrupted",
    "standardized_mean_shift",
]

missing_columns_626g = [
    column_626g
    for column_626g in required_columns_626g
    if column_626g not in fusion_source_626g.columns
]

if missing_columns_626g:
    raise RuntimeError(
        "Frozen fusion table is missing required columns:\n"
        + str(missing_columns_626g)
    )


if len(fusion_source_626g) != EXPECTED_ROWS_626G:
    raise RuntimeError(
        "Unexpected frozen fusion-table row count.\n"
        f"Expected: {EXPECTED_ROWS_626G}\n"
        f"Found   : {len(fusion_source_626g)}"
    )


unique_scenarios_626g = int(
    fusion_source_626g["global_index"].nunique()
)

unique_fusions_626g = sorted(
    fusion_source_626g["fusion"]
    .astype(str)
    .unique()
    .tolist()
)

expected_fusions_626g = sorted(
    [
        MEAN_FUSION_NAME_626G,
        MAX_FUSION_NAME_626G,
    ]
)


if unique_scenarios_626g != EXPECTED_SCENARIOS_626G:
    raise RuntimeError(
        "Unexpected unique scenario count.\n"
        f"Expected: {EXPECTED_SCENARIOS_626G}\n"
        f"Found   : {unique_scenarios_626g}"
    )


if unique_fusions_626g != expected_fusions_626g:
    raise RuntimeError(
        "Unexpected fusion candidates.\n"
        f"Expected: {expected_fusions_626g}\n"
        f"Found   : {unique_fusions_626g}"
    )


duplicate_pair_count_626g = int(
    fusion_source_626g.duplicated(
        subset=["global_index", "fusion"]
    ).sum()
)

if duplicate_pair_count_626g != 0:
    raise RuntimeError(
        "Duplicate scenario × fusion records detected: "
        f"{duplicate_pair_count_626g}"
    )


numeric_check_cols_626g = [
    "auc_clean_vs_corrupted",
    "standardized_mean_shift",
]

if not np.isfinite(
    fusion_source_626g[numeric_check_cols_626g]
    .to_numpy(dtype=float)
).all():
    raise RuntimeError(
        "Non-finite numeric values found in frozen fusion results."
    )


print("\n--- Frozen-table integrity ---")
print("Rows                    :", len(fusion_source_626g))
print("Unique scenarios        :", unique_scenarios_626g)
print("Fusion candidates       :", len(unique_fusions_626g))
print("Duplicate scenario×fusion:", duplicate_pair_count_626g)
print("Required numeric finite :", True)


# ================================================================
# 4. Verify frozen family counts
# ================================================================

scenario_metadata_626g = (
    fusion_source_626g
    .sort_values(["global_index", "fusion"])
    .drop_duplicates(subset=["global_index"])
    .copy()
)

actual_family_counts_626g = (
    scenario_metadata_626g["family"]
    .value_counts()
    .to_dict()
)

if actual_family_counts_626g != EXPECTED_FAMILY_COUNTS_626G:
    raise RuntimeError(
        "Frozen corruption-family counts differ from the expected "
        "315-scenario manifest.\n"
        f"Expected: {EXPECTED_FAMILY_COUNTS_626G}\n"
        f"Found   : {actual_family_counts_626g}"
    )


print("\n--- Frozen scenario composition ---")
for family_626g in [
    "bias",
    "noise",
    "drift",
    "dropout",
    "stuck",
]:
    print(
        f"{family_626g:<8}:",
        actual_family_counts_626g[family_626g],
    )


# ================================================================
# 5. Exact paired MAX-vs-MEAN reconstruction
# ================================================================

auc_wide_626g = (
    fusion_source_626g
    .pivot(
        index="global_index",
        columns="fusion",
        values="auc_clean_vs_corrupted",
    )
    .sort_index()
)

if auc_wide_626g.isna().any().any():
    raise RuntimeError(
        "Incomplete MAX-vs-MEAN scenario pairing detected."
    )


paired_626g = (
    scenario_metadata_626g[
        [
            "global_index",
            "family",
            "scenario_id",
            "sensor",
            "severity",
        ]
    ]
    .set_index("global_index")
    .join(auc_wide_626g)
    .reset_index()
)


paired_626g = paired_626g.rename(
    columns={
        MEAN_FUSION_NAME_626G: "mean_fusion_auc",
        MAX_FUSION_NAME_626G: "max_fusion_auc",
    }
)


paired_626g["auc_delta_max_minus_mean"] = (
    paired_626g["max_fusion_auc"]
    - paired_626g["mean_fusion_auc"]
)


PAIR_TOLERANCE_626G = 1e-12

paired_626g["winner"] = np.where(
    paired_626g["auc_delta_max_minus_mean"]
    > PAIR_TOLERANCE_626G,
    "MAX",
    np.where(
        paired_626g["auc_delta_max_minus_mean"]
        < -PAIR_TOLERANCE_626G,
        "MEAN",
        "TIE",
    ),
)


max_wins_626g = int(
    (paired_626g["winner"] == "MAX").sum()
)

mean_wins_626g = int(
    (paired_626g["winner"] == "MEAN").sum()
)

ties_626g = int(
    (paired_626g["winner"] == "TIE").sum()
)


overall_mean_auc_626g = float(
    paired_626g["mean_fusion_auc"].mean()
)

overall_max_auc_626g = float(
    paired_626g["max_fusion_auc"].mean()
)

overall_auc_delta_626g = float(
    paired_626g["auc_delta_max_minus_mean"].mean()
)

overall_median_delta_626g = float(
    paired_626g["auc_delta_max_minus_mean"].median()
)


print("\n--- Overall paired MAX-vs-MEAN evidence ---")
print(
    "Mean AUC — MEAN fusion :",
    overall_mean_auc_626g,
)
print(
    "Mean AUC — MAX fusion  :",
    overall_max_auc_626g,
)
print(
    "Mean AUC delta MAX-MEAN :",
    overall_auc_delta_626g,
)
print(
    "Median AUC delta        :",
    overall_median_delta_626g,
)
print(
    "Scenario wins MAX/MEAN/TIE:",
    max_wins_626g,
    "/",
    mean_wins_626g,
    "/",
    ties_626g,
)


# Exact consistency with the already-reviewed Step 6.26F-E3 output.

if (
    max_wins_626g != 220
    or mean_wins_626g != 94
    or ties_626g != 1
):
    raise RuntimeError(
        "Step 6.26F-E3 scenario-win evidence was not reproduced.\n"
        f"Observed MAX / MEAN / TIE = "
        f"{max_wins_626g} / {mean_wins_626g} / {ties_626g}"
    )


# ================================================================
# 6. Family-level robustness
# ================================================================

family_summary_626g = (
    paired_626g
    .groupby("family", as_index=False)
    .agg(
        scenarios=("global_index", "size"),
        mean_mean_auc=("mean_fusion_auc", "mean"),
        mean_max_auc=("max_fusion_auc", "mean"),
        mean_auc_delta=(
            "auc_delta_max_minus_mean",
            "mean",
        ),
        median_auc_delta=(
            "auc_delta_max_minus_mean",
            "median",
        ),
    )
)

family_win_fraction_626g = (
    paired_626g
    .assign(
        max_win=(
            paired_626g["auc_delta_max_minus_mean"]
            > PAIR_TOLERANCE_626G
        )
    )
    .groupby("family")["max_win"]
    .mean()
)

family_summary_626g["max_auc_win_fraction"] = (
    family_summary_626g["family"]
    .map(family_win_fraction_626g)
)


all_family_deltas_positive_626g = bool(
    (
        family_summary_626g["mean_auc_delta"]
        > 0.0
    ).all()
)


# ================================================================
# 7. Severity-level robustness
# ================================================================

severity_summary_626g = (
    paired_626g
    .groupby("severity", as_index=False)
    .agg(
        scenarios=("global_index", "size"),
        mean_mean_auc=("mean_fusion_auc", "mean"),
        mean_max_auc=("max_fusion_auc", "mean"),
        mean_auc_delta=(
            "auc_delta_max_minus_mean",
            "mean",
        ),
        median_auc_delta=(
            "auc_delta_max_minus_mean",
            "median",
        ),
    )
)

severity_win_fraction_626g = (
    paired_626g
    .assign(
        max_win=(
            paired_626g["auc_delta_max_minus_mean"]
            > PAIR_TOLERANCE_626G
        )
    )
    .groupby("severity")["max_win"]
    .mean()
)

severity_summary_626g["max_auc_win_fraction"] = (
    severity_summary_626g["severity"]
    .map(severity_win_fraction_626g)
)


all_severity_deltas_positive_626g = bool(
    (
        severity_summary_626g["mean_auc_delta"]
        > 0.0
    ).all()
)


# ================================================================
# 8. Leave-one-family-out robustness
# ================================================================

loo_records_626g = []

family_order_626g = [
    "bias",
    "drift",
    "dropout",
    "noise",
    "stuck",
]

for excluded_family_626g in family_order_626g:

    remaining_626g = paired_626g.loc[
        paired_626g["family"]
        != excluded_family_626g
    ].copy()

    loo_records_626g.append(
        {
            "excluded_family":
                excluded_family_626g,

            "remaining_scenarios":
                int(len(remaining_626g)),

            "mean_auc_delta":
                float(
                    remaining_626g[
                        "auc_delta_max_minus_mean"
                    ].mean()
                ),

            "median_auc_delta":
                float(
                    remaining_626g[
                        "auc_delta_max_minus_mean"
                    ].median()
                ),

            "max_auc_win_fraction":
                float(
                    (
                        remaining_626g[
                            "auc_delta_max_minus_mean"
                        ]
                        > PAIR_TOLERANCE_626G
                    ).mean()
                ),
        }
    )


loo_family_626g = pd.DataFrame(
    loo_records_626g
)


all_loo_deltas_positive_626g = bool(
    (
        loo_family_626g["mean_auc_delta"]
        > 0.0
    ).all()
)


# ================================================================
# 9. Sensor-level robustness
# ================================================================

sensor_summary_626g = (
    paired_626g
    .groupby("sensor", as_index=False)
    .agg(
        scenarios=("global_index", "size"),
        mean_auc_delta=(
            "auc_delta_max_minus_mean",
            "mean",
        ),
        median_auc_delta=(
            "auc_delta_max_minus_mean",
            "median",
        ),
    )
)

sensor_win_fraction_626g = (
    paired_626g
    .assign(
        max_win=(
            paired_626g["auc_delta_max_minus_mean"]
            > PAIR_TOLERANCE_626G
        )
    )
    .groupby("sensor")["max_win"]
    .mean()
)

sensor_summary_626g["max_auc_win_fraction"] = (
    sensor_summary_626g["sensor"]
    .map(sensor_win_fraction_626g)
)


negative_sensor_groups_626g = int(
    (
        sensor_summary_626g["mean_auc_delta"]
        < -PAIR_TOLERANCE_626G
    ).sum()
)


# ================================================================
# 10. Final freeze criteria
# ================================================================

freeze_checks_626g = {
    "overall_mean_auc_delta_positive":
        bool(overall_auc_delta_626g > 0.0),

    "max_wins_more_scenarios":
        bool(max_wins_626g > mean_wins_626g),

    "all_family_mean_deltas_positive":
        all_family_deltas_positive_626g,

    "all_severity_mean_deltas_positive":
        all_severity_deltas_positive_626g,

    "all_leave_one_family_out_mean_deltas_positive":
        all_loo_deltas_positive_626g,

    "zero_negative_sensor_groups":
        bool(negative_sensor_groups_626g == 0),

    "scenario_pairing_complete":
        bool(len(paired_626g) == 315),
}


all_freeze_checks_passed_626g = bool(
    all(freeze_checks_626g.values())
)


print("\n--- Final freeze criteria ---")

for check_name_626g, check_value_626g in (
    freeze_checks_626g.items()
):
    print(
        f"{check_name_626g:<48}: "
        f"{check_value_626g}"
    )


if not all_freeze_checks_passed_626g:
    raise RuntimeError(
        "Sensor-quality fusion cannot be frozen because "
        "one or more robustness checks failed."
    )


# ================================================================
# 11. Freeze exact research specification in memory
# ================================================================

FROZEN_SENSOR_QUALITY_SPEC_626G = {

    "dataset":
        "NASA C-MAPSS FD001",

    "development_stage":
        "DEV-only sensor-quality selection",

    "reference_population":
        "Canonical DEV",

    "reference_std_convention":
        "population (ddof=0)",

    "window_length":
        WINDOW_LENGTH_626G,

    "selected_raw_quality_features":
        SELECTED_RAW_FEATURES_626G,

    "calibrated_quality_components":
        SELECTED_COMPONENTS_626G,

    "calibration_method":
        "two-sided empirical absolute-deviation calibration",

    "fusion_rule":
        FINAL_FUSION_RULE_626G,

    "fusion_name":
        FINAL_FUSION_NAME_626G,

    "interpretation":
        (
            "larger fused value indicates greater "
            "sensor-quality anomaly / poorer sensor quality"
        ),

    "selection_criterion":
        (
            "DEV-only clean-vs-corrupted discrimination "
            "robustness using AUC"
        ),

    "official_test_labels_used_for_selection":
        False,

    "rul_labels_used_for_sensor_quality_selection":
        False,

    "final_sensor_quality_threshold_selected":
        False,
}


FROZEN_SENSOR_QUALITY_FUSION_626G = (
    FINAL_FUSION_NAME_626G
)

FROZEN_SENSOR_QUALITY_FUSION_RULE_626G = (
    FINAL_FUSION_RULE_626G
)

FROZEN_SENSOR_QUALITY_COMPONENTS_626G = (
    SELECTED_COMPONENTS_626G.copy()
)

SENSOR_QUALITY_FUSION_FROZEN_626G = True


# ================================================================
# 12. Locate project root
# ================================================================

def find_project_root_626g(start_path):

    start_path = Path(start_path).resolve()

    search_paths_626g = [
        start_path,
        *start_path.parents,
    ]

    for candidate_626g in search_paths_626g:

        if (
            (candidate_626g / "05_experiments").exists()
            and
            (candidate_626g / "06_results").exists()
        ):
            return candidate_626g

    raise FileNotFoundError(
        "Could not locate project root containing "
        "05_experiments and 06_results."
    )


PROJECT_ROOT_626G = find_project_root_626g(
    Path.cwd()
)

OUTPUT_DIR_626G = (
    PROJECT_ROOT_626G
    / "06_results"
    / "step6_sensor_quality"
    / "fusion_freeze"
)

OUTPUT_DIR_626G.mkdir(
    parents=True,
    exist_ok=True,
)


# ================================================================
# 13. Build overall evidence table
# ================================================================

overall_evidence_626g = pd.DataFrame(
    [
        {
            "mean_fusion_mean_auc":
                overall_mean_auc_626g,

            "max_fusion_mean_auc":
                overall_max_auc_626g,

            "mean_auc_delta_max_minus_mean":
                overall_auc_delta_626g,

            "median_auc_delta_max_minus_mean":
                overall_median_delta_626g,

            "max_wins":
                max_wins_626g,

            "mean_wins":
                mean_wins_626g,

            "ties":
                ties_626g,

            "negative_sensor_groups":
                negative_sensor_groups_626g,

            "all_family_mean_deltas_positive":
                all_family_deltas_positive_626g,

            "all_severity_mean_deltas_positive":
                all_severity_deltas_positive_626g,

            "all_loo_mean_deltas_positive":
                all_loo_deltas_positive_626g,
        }
    ]
)


# ================================================================
# 14. Hash frozen source table
# ================================================================

source_for_hash_626g = (
    fusion_source_626g
    .sort_values(
        ["global_index", "fusion"]
    )
    .reset_index(drop=True)
)

source_hash_values_626g = (
    pd.util.hash_pandas_object(
        source_for_hash_626g,
        index=False,
    )
    .to_numpy(dtype=np.uint64)
)

SOURCE_SHA256_626G = hashlib.sha256(
    source_hash_values_626g.tobytes()
).hexdigest()


# ================================================================
# 15. Write frozen evidence artifacts
# ================================================================

overall_path_626g = (
    OUTPUT_DIR_626G
    / "fd001_sensor_quality_fusion_overall_evidence.csv"
)

family_path_626g = (
    OUTPUT_DIR_626G
    / "fd001_sensor_quality_fusion_family_evidence.csv"
)

severity_path_626g = (
    OUTPUT_DIR_626G
    / "fd001_sensor_quality_fusion_severity_evidence.csv"
)

loo_path_626g = (
    OUTPUT_DIR_626G
    / "fd001_sensor_quality_fusion_leave_one_family_out.csv"
)

sensor_path_626g = (
    OUTPUT_DIR_626G
    / "fd001_sensor_quality_fusion_sensor_evidence.csv"
)

paired_path_626g = (
    OUTPUT_DIR_626G
    / "fd001_sensor_quality_fusion_paired_scenarios.csv"
)

decision_path_626g = (
    OUTPUT_DIR_626G
    / "fd001_sensor_quality_fusion_decision.json"
)


overall_evidence_626g.to_csv(
    overall_path_626g,
    index=False,
)

family_summary_626g.to_csv(
    family_path_626g,
    index=False,
)

severity_summary_626g.to_csv(
    severity_path_626g,
    index=False,
)

loo_family_626g.to_csv(
    loo_path_626g,
    index=False,
)

sensor_summary_626g.to_csv(
    sensor_path_626g,
    index=False,
)

paired_626g.to_csv(
    paired_path_626g,
    index=False,
)


decision_record_626g = {

    "step":
        "6.26G",

    "decision":
        "Freeze sensor-quality representation and fusion",

    "timestamp_utc":
        datetime.now(timezone.utc).isoformat(),

    "frozen_specification":
        FROZEN_SENSOR_QUALITY_SPEC_626G,

    "source_object":
        REQUIRED_OBJECT_626G,

    "source_shape":
        [
            int(fusion_source_626g.shape[0]),
            int(fusion_source_626g.shape[1]),
        ],

    "source_sha256":
        SOURCE_SHA256_626G,

    "evidence": {

        "mean_fusion_mean_auc":
            overall_mean_auc_626g,

        "max_fusion_mean_auc":
            overall_max_auc_626g,

        "mean_auc_delta_max_minus_mean":
            overall_auc_delta_626g,

        "median_auc_delta_max_minus_mean":
            overall_median_delta_626g,

        "max_wins":
            max_wins_626g,

        "mean_wins":
            mean_wins_626g,

        "ties":
            ties_626g,

        "all_family_mean_deltas_positive":
            all_family_deltas_positive_626g,

        "all_severity_mean_deltas_positive":
            all_severity_deltas_positive_626g,

        "all_leave_one_family_out_mean_deltas_positive":
            all_loo_deltas_positive_626g,

        "negative_sensor_groups":
            negative_sensor_groups_626g,
    },

    "freeze_checks":
        freeze_checks_626g,

    "sensor_quality_fusion_frozen":
        True,

    "sensor_quality_threshold_selected":
        False,

    "official_test_labels_used":
        False,

    "official_test_scored":
        False,

    "model_inference_executed":
        False,

    "mc_dropout_executed":
        False,

    "corruptions_regenerated":
        False,
}


with open(
    decision_path_626g,
    "w",
    encoding="utf-8",
) as file_626g:

    json.dump(
        decision_record_626g,
        file_626g,
        indent=2,
        allow_nan=False,
    )


# ================================================================
# 16. File hashes / reproducibility manifest
# ================================================================

def file_sha256_626g(path_626g):

    digest_626g = hashlib.sha256()

    with open(path_626g, "rb") as file_626g:

        for block_626g in iter(
            lambda: file_626g.read(1024 * 1024),
            b"",
        ):
            digest_626g.update(block_626g)

    return digest_626g.hexdigest()


written_files_626g = [
    overall_path_626g,
    family_path_626g,
    severity_path_626g,
    loo_path_626g,
    sensor_path_626g,
    paired_path_626g,
    decision_path_626g,
]


file_hashes_626g = {
    path_626g.name:
        file_sha256_626g(path_626g)
    for path_626g in written_files_626g
}


manifest_path_626g = (
    OUTPUT_DIR_626G
    / "fd001_sensor_quality_fusion_freeze_manifest.json"
)


manifest_record_626g = {

    "step":
        "6.26G",

    "created_utc":
        datetime.now(timezone.utc).isoformat(),

    "source_object":
        REQUIRED_OBJECT_626G,

    "source_sha256":
        SOURCE_SHA256_626G,

    "frozen_fusion":
        FINAL_FUSION_NAME_626G,

    "fusion_rule":
        FINAL_FUSION_RULE_626G,

    "written_files_sha256":
        file_hashes_626g,
}


with open(
    manifest_path_626g,
    "w",
    encoding="utf-8",
) as file_626g:

    json.dump(
        manifest_record_626g,
        file_626g,
        indent=2,
        allow_nan=False,
    )


# Verify all written files exist.

all_output_paths_626g = (
    written_files_626g
    + [manifest_path_626g]
)

all_files_exist_626g = bool(
    all(
        path_626g.exists()
        for path_626g in all_output_paths_626g
    )
)

if not all_files_exist_626g:
    raise RuntimeError(
        "One or more Step 6.26G artifacts were not written."
    )


FROZEN_SENSOR_QUALITY_DECISION_PATH_626G = (
    decision_path_626g
)

FROZEN_SENSOR_QUALITY_MANIFEST_PATH_626G = (
    manifest_path_626g
)


# ================================================================
# 17. Compact final report
# ================================================================

print(
    "\n"
    "===============================================================\n"
    "STEP 6.26G FINAL FREEZE SUMMARY\n"
    "==============================================================="
)

print(
    "Frozen raw quality channels        :",
    len(SELECTED_RAW_FEATURES_626G),
)

print(
    "Frozen calibrated components       :",
    len(SELECTED_COMPONENTS_626G),
)

print(
    "Frozen fusion                      :",
    FINAL_FUSION_NAME_626G,
)

print(
    "Frozen fusion rule                 :",
    FINAL_FUSION_RULE_626G,
)

print(
    "Frozen window length               :",
    WINDOW_LENGTH_626G,
)

print(
    "DEV scenarios supporting decision  :",
    len(paired_626g),
)

print(
    "Mean AUC — MEAN fusion             :",
    overall_mean_auc_626g,
)

print(
    "Mean AUC — MAX fusion              :",
    overall_max_auc_626g,
)

print(
    "Mean AUC advantage MAX-MEAN        :",
    overall_auc_delta_626g,
)

print(
    "Scenario wins MAX / MEAN / ties    :",
    f"{max_wins_626g} / "
    f"{mean_wins_626g} / "
    f"{ties_626g}",
)

print(
    "All family mean deltas positive    :",
    all_family_deltas_positive_626g,
)

print(
    "All severity mean deltas positive  :",
    all_severity_deltas_positive_626g,
)

print(
    "All LOO mean deltas positive       :",
    all_loo_deltas_positive_626g,
)

print(
    "Negative sensor groups             :",
    negative_sensor_groups_626g,
)

print(
    "All freeze checks passed           :",
    all_freeze_checks_passed_626g,
)

print(
    "Sensor-quality fusion frozen       :",
    SENSOR_QUALITY_FUSION_FROZEN_626G,
)

print(
    "Sensor-quality threshold selected  :",
    False,
)

print(
    "\n--- Research-safety status ---"
)

print(
    "RUL labels used for SQ selection   :",
    False,
)

print(
    "Official TEST labels used          :",
    False,
)

print(
    "Official TEST scored               :",
    False,
)

print(
    "Model inference executed           :",
    False,
)

print(
    "MC-dropout executed                :",
    False,
)

print(
    "Corruptions regenerated            :",
    False,
)

print(
    "\nFiles written                      :",
    len(all_output_paths_626g),
)

print(
    "Output directory                   :",
    OUTPUT_DIR_626G,
)

for number_626g, path_626g in enumerate(
    all_output_paths_626g,
    start=1,
):
    print(
        f"{number_626g}. {path_626g}"
    )

print(
    "\nFINAL SENSOR-QUALITY DECISION:\n"
    "  Components : "
    "excursion anomaly + jump anomaly + repeat anomaly\n"
    "  Fusion     : MAX\n"
    "  Status     : FROZEN\n"
    "  Threshold  : NOT YET SELECTED"
)

print(
    "\n=== END STEP 6.26G ==="
)

In [ ]:
# ============================================================
# Step 6.27A
# DEV alignment of model uncertainty and frozen sensor quality
# ============================================================

from __future__ import annotations

import inspect
from collections import Counter

import numpy as np
import pandas as pd


print(
    "\n"
    "==============================================================\n"
    "STEP 6.27A — DEV UNCERTAINTY + SENSOR-QUALITY ALIGNMENT AUDIT\n"
    "=============================================================="
)


# ============================================================
# 0. Safety / scope declarations
# ============================================================

OFFICIAL_TEST_LABELS_USED_627A = False
OFFICIAL_TEST_SCORED_627A = False
MODEL_INFERENCE_EXECUTED_627A = False
MC_DROPOUT_EXECUTED_627A = False
THRESHOLD_SELECTED_627A = False
RELIABILITY_RULE_CHANGED_627A = False
SENSOR_QUALITY_REPRESENTATION_CHANGED_627A = False
FILES_WRITTEN_627A = False


# ============================================================
# 1. Required frozen objects
# ============================================================

required_objects_627a = [
    "scenario_manifests_620gc",
    "apply_frozen_corruption_620gc",
    "val_raw_base_620gc",

    "build_sensor_quality_candidates_626b",
    "score_sensor_quality_windows_626d",

    "SENSOR_QUALITY_SENSOR_COLUMNS_626B",
    "SENSOR_QUALITY_REFERENCE_626B",
    "SENSOR_QUALITY_WINDOW_LENGTH_626B",

    "SENSOR_QUALITY_SELECTED_FEATURES_626D",
    "SENSOR_QUALITY_COMPONENT_COLUMNS_626D",
]

print("\n--- Required frozen objects ---")

missing_objects_627a = []

for name_627a in required_objects_627a:
    exists_627a = name_627a in globals()

    print(
        f"{name_627a:<45}: "
        f"{exists_627a}"
    )

    if not exists_627a:
        missing_objects_627a.append(
            name_627a
        )

if missing_objects_627a:
    raise RuntimeError(
        "Step 6.27A cannot continue. "
        "Missing required frozen objects: "
        + str(
            missing_objects_627a
        )
    )


# ============================================================
# 2. Recover frozen sensor-quality definition
# ============================================================

sensor_columns_627a = list(
    SENSOR_QUALITY_SENSOR_COLUMNS_626B
)

selected_raw_features_627a = list(
    SENSOR_QUALITY_SELECTED_FEATURES_626D
)

component_columns_627a = list(
    SENSOR_QUALITY_COMPONENT_COLUMNS_626D
)

window_length_627a = int(
    SENSOR_QUALITY_WINDOW_LENGTH_626B
)

FROZEN_SENSOR_QUALITY_FUSION_627A = (
    "sq_candidate_max_fusion"
)


print(
    "\n--- Frozen sensor-quality definition ---"
)

print(
    "Sensor count        :",
    len(
        sensor_columns_627a
    )
)

print(
    "Window length       :",
    window_length_627a
)

print(
    "Raw channels        :",
    selected_raw_features_627a
)

print(
    "Calibrated channels :",
    component_columns_627a
)

print(
    "Fusion              : MAX"
)

print(
    "Fusion column       :",
    FROZEN_SENSOR_QUALITY_FUSION_627A
)


if len(
    selected_raw_features_627a
) != 3:
    raise RuntimeError(
        "Frozen sensor-quality representation "
        "does not contain exactly three raw channels."
    )

if len(
    component_columns_627a
) != 3:
    raise RuntimeError(
        "Frozen sensor-quality representation "
        "does not contain exactly three calibrated components."
    )


# ============================================================
# 3. Structural helpers
# ============================================================

def _collect_dataframes_627a(
    obj_627a,
    path_627a="root",
):
    """
    Recursively collect DataFrames from nested
    tuple/list/dict structures without modifying them.
    """

    found_627a = []

    if isinstance(
        obj_627a,
        pd.DataFrame,
    ):
        found_627a.append(
            (
                path_627a,
                obj_627a,
            )
        )

    elif isinstance(
        obj_627a,
        (tuple, list),
    ):
        for i_627a, item_627a in enumerate(
            obj_627a
        ):
            found_627a.extend(
                _collect_dataframes_627a(
                    item_627a,
                    path_627a
                    + f"[{i_627a}]",
                )
            )

    elif isinstance(
        obj_627a,
        dict,
    ):
        for key_627a, item_627a in (
            obj_627a.items()
        ):
            found_627a.extend(
                _collect_dataframes_627a(
                    item_627a,
                    path_627a
                    + f"[{repr(key_627a)}]",
                )
            )

    return found_627a


def _same_dataframe_structure_627a(
    df_a_627a,
    df_b_627a,
):
    return (
        isinstance(
            df_a_627a,
            pd.DataFrame,
        )
        and isinstance(
            df_b_627a,
            pd.DataFrame,
        )
        and df_a_627a.shape
        == df_b_627a.shape
        and list(
            df_a_627a.columns
        )
        == list(
            df_b_627a.columns
        )
        and df_a_627a.index.equals(
            df_b_627a.index
        )
    )


# ============================================================
# 4. Frozen DEV corruption application helper
# ============================================================

def _apply_dev_corruption_627a(
    base_df_627a,
    family_627a,
    scenario_row_627a,
):
    """
    Call the already-frozen DEV corruption function by
    inspecting its actual signature rather than guessing it.
    """

    fn_627a = (
        apply_frozen_corruption_620gc
    )

    sig_627a = inspect.signature(
        fn_627a
    )

    kwargs_627a = {}
    unresolved_required_627a = []

    base_names_627a = {
        "base_df",
        "input_df",
        "df",
        "data",
        "raw_df",
        "source_df",
    }

    family_names_627a = {
        "family",
        "corruption_family",
    }

    row_names_627a = {
        "scenario_row",
        "row",
        "manifest_row",
        "scenario",
        "scenario_spec",
    }

    for (
        param_name_627a,
        param_627a,
    ) in sig_627a.parameters.items():

        if (
            param_name_627a
            in base_names_627a
        ):
            kwargs_627a[
                param_name_627a
            ] = base_df_627a

        elif (
            param_name_627a
            in family_names_627a
        ):
            kwargs_627a[
                param_name_627a
            ] = family_627a

        elif (
            param_name_627a
            in row_names_627a
        ):
            kwargs_627a[
                param_name_627a
            ] = scenario_row_627a

        elif (
            param_627a.default
            is inspect._empty
            and param_627a.kind
            in (
                inspect.Parameter.POSITIONAL_ONLY,
                inspect.Parameter.POSITIONAL_OR_KEYWORD,
                inspect.Parameter.KEYWORD_ONLY,
            )
        ):
            unresolved_required_627a.append(
                param_name_627a
            )

    if unresolved_required_627a:
        raise RuntimeError(
            "Could not safely call "
            "apply_frozen_corruption_620gc. "
            "Unresolved required parameters: "
            + str(
                unresolved_required_627a
            )
            + "\nActual signature: "
            + str(
                sig_627a
            )
        )

    result_627a = fn_627a(
        **kwargs_627a
    )

    if isinstance(
        result_627a,
        pd.DataFrame,
    ):
        corrupted_df_627a = (
            result_627a.copy()
        )

    elif (
        "extract_corrupted_dataframe_620gc"
        in globals()
    ):
        corrupted_df_627a = (
            extract_corrupted_dataframe_620gc(
                family_627a,
                result_627a,
            )
        )

    else:
        candidate_dfs_627a = [
            df_627a
            for _, df_627a
            in _collect_dataframes_627a(
                result_627a
            )
            if (
                df_627a.shape
                == base_df_627a.shape
                and list(
                    df_627a.columns
                )
                == list(
                    base_df_627a.columns
                )
            )
        ]

        if len(
            candidate_dfs_627a
        ) != 1:
            raise RuntimeError(
                "Could not uniquely recover corrupted "
                "DEV DataFrame from frozen corruption result."
            )

        corrupted_df_627a = (
            candidate_dfs_627a[0].copy()
        )

    if not _same_dataframe_structure_627a(
        corrupted_df_627a,
        base_df_627a,
    ):
        raise RuntimeError(
            "Frozen DEV corruption changed "
            "DataFrame structure."
        )

    return corrupted_df_627a


print(
    "\n--- Frozen DEV corruption contract ---"
)

print(
    "Signature:",
    inspect.signature(
        apply_frozen_corruption_620gc
    )
)


# ============================================================
# 5. Sensor-quality builder normalizer
# ============================================================

def _extract_candidate_window_df_627a(
    builder_result_627a,
):
    """
    Recover the exact window-level candidate DataFrame from
    the already-frozen Step 6.26B builder result.
    """

    frames_627a = (
        _collect_dataframes_627a(
            builder_result_627a
        )
    )

    plausible_627a = []

    for (
        path_627a,
        df_627a,
    ) in frames_627a:

        columns_627a = set(
            df_627a.columns
        )

        raw_present_627a = all(
            feature_627a
            in columns_627a
            for feature_627a
            in selected_raw_features_627a
        )

        has_unit_627a = (
            "unit_id"
            in columns_627a
        )

        has_cycle_627a = (
            "cycle"
            in columns_627a
        )

        if (
            raw_present_627a
            and has_unit_627a
            and has_cycle_627a
        ):
            plausible_627a.append(
                (
                    path_627a,
                    df_627a,
                )
            )

    if not plausible_627a:
        raise RuntimeError(
            "No valid sensor-quality candidate-window "
            "DataFrame found in builder result."
        )

    # Prefer the largest valid table, then widest.
    plausible_627a = sorted(
        plausible_627a,
        key=lambda item_627a: (
            len(
                item_627a[1]
            ),
            len(
                item_627a[1].columns
            ),
        ),
        reverse=True,
    )

    best_df_627a = (
        plausible_627a[0][1]
        .copy()
    )

    return best_df_627a


def _build_candidate_windows_627a(
    raw_df_627a,
):
    result_627a = (
        build_sensor_quality_candidates_626b(
            input_df=raw_df_627a,
            sensor_columns=sensor_columns_627a,
            reference_stats=SENSOR_QUALITY_REFERENCE_626B,
            window_length=window_length_627a,
        )
    )

    return (
        _extract_candidate_window_df_627a(
            result_627a
        )
    )


print(
    "\n--- Sensor-quality builder contract ---"
)

print(
    "Signature:",
    inspect.signature(
        build_sensor_quality_candidates_626b
    )
)


# ============================================================
# 6. Frozen scorer caller / normalizer
# ============================================================

def _call_quality_scorer_627a(
    candidate_df_627a,
):
    """
    Call the frozen Step 6.26D scorer according to its actual
    function signature.
    """

    fn_627a = (
        score_sensor_quality_windows_626d
    )

    sig_627a = inspect.signature(
        fn_627a
    )

    params_627a = list(
        sig_627a.parameters.values()
    )

    required_627a = [
        p_627a
        for p_627a
        in params_627a
        if (
            p_627a.default
            is inspect._empty
            and p_627a.kind
            in (
                inspect.Parameter.POSITIONAL_ONLY,
                inspect.Parameter.POSITIONAL_OR_KEYWORD,
                inspect.Parameter.KEYWORD_ONLY,
            )
        )
    ]

    # Most likely frozen Step 6.26D contract:
    # score_sensor_quality_windows_626d(window_df)
    if len(
        required_627a
    ) == 1:
        result_627a = fn_627a(
            candidate_df_627a
        )

    else:
        aliases_627a = {
            "window_df",
            "windows",
            "candidate_df",
            "candidate_windows",
            "features_df",
            "input_df",
            "df",
        }

        kwargs_627a = {}
        unresolved_627a = []

        for param_627a in (
            required_627a
        ):
            if (
                param_627a.name
                in aliases_627a
            ):
                kwargs_627a[
                    param_627a.name
                ] = candidate_df_627a
            else:
                unresolved_627a.append(
                    param_627a.name
                )

        if unresolved_627a:
            raise RuntimeError(
                "Could not safely call frozen "
                "sensor-quality scorer. "
                "Unresolved parameters: "
                + str(
                    unresolved_627a
                )
                + "\nSignature: "
                + str(
                    sig_627a
                )
            )

        result_627a = fn_627a(
            **kwargs_627a
        )

    scored_frames_627a = (
        _collect_dataframes_627a(
            result_627a
        )
    )

    valid_frames_627a = []

    for (
        path_627a,
        df_627a,
    ) in scored_frames_627a:

        cols_627a = set(
            df_627a.columns
        )

        if (
            FROZEN_SENSOR_QUALITY_FUSION_627A
            in cols_627a
        ):
            valid_frames_627a.append(
                (
                    path_627a,
                    df_627a,
                )
            )

    # If scorer itself returns DataFrame but fusion was not yet
    # materialized, allow reconstruction from the three already
    # frozen calibrated components.
    if not valid_frames_627a:

        possible_frames_627a = []

        for (
            path_627a,
            df_627a,
        ) in scored_frames_627a:

            if all(
                component_627a
                in df_627a.columns
                for component_627a
                in component_columns_627a
            ):
                possible_frames_627a.append(
                    (
                        path_627a,
                        df_627a,
                    )
                )

        if len(
            possible_frames_627a
        ) == 1:

            scored_df_627a = (
                possible_frames_627a[0][1]
                .copy()
            )

            scored_df_627a[
                FROZEN_SENSOR_QUALITY_FUSION_627A
            ] = (
                scored_df_627a[
                    component_columns_627a
                ]
                .max(
                    axis=1
                )
            )

            valid_frames_627a = [
                (
                    "reconstructed_max_fusion",
                    scored_df_627a,
                )
            ]

    if len(
        valid_frames_627a
    ) != 1:
        raise RuntimeError(
            "Could not uniquely identify frozen scored "
            "sensor-quality DataFrame. "
            f"Valid candidates found: "
            f"{len(valid_frames_627a)}"
        )

    scored_df_627a = (
        valid_frames_627a[0][1]
        .copy()
    )

    # Restore keys only when scorer preserved row order exactly.
    if (
        "unit_id"
        not in scored_df_627a.columns
        or "cycle"
        not in scored_df_627a.columns
    ):
        if len(
            scored_df_627a
        ) != len(
            candidate_df_627a
        ):
            raise RuntimeError(
                "Scored sensor-quality DataFrame lost keys "
                "and row count differs from candidate table."
            )

        scored_df_627a = (
            scored_df_627a
            .reset_index(
                drop=True
            )
        )

        candidate_keys_627a = (
            candidate_df_627a[
                [
                    "unit_id",
                    "cycle",
                ]
            ]
            .reset_index(
                drop=True
            )
        )

        scored_df_627a[
            "unit_id"
        ] = candidate_keys_627a[
            "unit_id"
        ]

        scored_df_627a[
            "cycle"
        ] = candidate_keys_627a[
            "cycle"
        ]

    return scored_df_627a


print(
    "\n--- Sensor-quality scorer contract ---"
)

print(
    "Signature:",
    inspect.signature(
        score_sensor_quality_windows_626d
    )
)


# ============================================================
# 7. Recover existing DEV corrupted-endpoint uncertainty table
# ============================================================

EXPECTED_SCENARIOS_627A = 315
EXPECTED_DEV_ENGINES_627A = 20
EXPECTED_ENDPOINT_ROWS_627A = (
    EXPECTED_SCENARIOS_627A
    * EXPECTED_DEV_ENGINES_627A
)


uncertainty_priority_627a = [
    "corrupted_mc_std",
    "mc_std",
    "mc_uncertainty",
    "uncertainty",
    "predictive_uncertainty",
    "corrupted_uncertainty",
    "uncertainty_score",
]


def _find_uncertainty_column_627a(
    df_627a,
):
    for col_627a in (
        uncertainty_priority_627a
    ):
        if col_627a in df_627a.columns:
            return col_627a

    lower_map_627a = {
        str(
            c_627a
        ).lower(): c_627a
        for c_627a
        in df_627a.columns
    }

    for lower_name_627a, original_627a in (
        lower_map_627a.items()
    ):
        if (
            "uncert"
            in lower_name_627a
            or (
                "mc"
                in lower_name_627a
                and "std"
                in lower_name_627a
            )
        ):
            if (
                "clean"
                not in lower_name_627a
            ):
                return original_627a

    return None


uncertainty_candidates_627a = []

for (
    object_name_627a,
    object_value_627a,
) in list(
    globals().items()
):

    if not isinstance(
        object_value_627a,
        pd.DataFrame,
    ):
        continue

    upper_name_627a = (
        object_name_627a.upper()
    )

    # Critical safety guard: do not choose official TEST table.
    if "TEST" in upper_name_627a:
        continue

    df_627a = object_value_627a

    cols_627a = set(
        df_627a.columns
    )

    uncertainty_col_627a = (
        _find_uncertainty_column_627a(
            df_627a
        )
    )

    if uncertainty_col_627a is None:
        continue

    if "unit_id" not in cols_627a:
        continue

    has_family_627a = (
        "family"
        in cols_627a
    )

    has_scenario_627a = any(
        col_627a
        in cols_627a
        for col_627a
        in [
            "global_index",
            "family_position",
            "scenario_id",
            "manifest_scenario_id",
        ]
    )

    if not (
        has_family_627a
        and has_scenario_627a
    ):
        continue

    score_627a = 0

    if len(
        df_627a
    ) == EXPECTED_ENDPOINT_ROWS_627A:
        score_627a += 100

    if "DEV" in upper_name_627a:
        score_627a += 25

    if "ENDPOINT" in upper_name_627a:
        score_627a += 20

    if "SCENARIO" in upper_name_627a:
        score_627a += 10

    if (
        "UNCERTAINTY"
        in upper_name_627a
        or "MC"
        in upper_name_627a
    ):
        score_627a += 10

    if (
        uncertainty_col_627a
        == "corrupted_mc_std"
    ):
        score_627a += 30

    if "global_index" in cols_627a:
        score_627a += 10

    if "cycle" in cols_627a:
        score_627a += 5

    uncertainty_candidates_627a.append(
        {
            "object_name":
                object_name_627a,

            "rows":
                len(
                    df_627a
                ),

            "columns":
                len(
                    df_627a.columns
                ),

            "uncertainty_column":
                uncertainty_col_627a,

            "score":
                score_627a,
        }
    )


UNCERTAINTY_CANDIDATES_627A = (
    pd.DataFrame(
        uncertainty_candidates_627a
    )
)


if UNCERTAINTY_CANDIDATES_627A.empty:
    raise RuntimeError(
        "No plausible frozen DEV uncertainty endpoint "
        "DataFrame was found."
    )


UNCERTAINTY_CANDIDATES_627A = (
    UNCERTAINTY_CANDIDATES_627A
    .sort_values(
        [
            "score",
            "rows",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n--- Frozen DEV uncertainty-table candidates ---"
)

display(
    UNCERTAINTY_CANDIDATES_627A.head(
        10
    )
)


best_uncertainty_candidate_627a = (
    UNCERTAINTY_CANDIDATES_627A.iloc[
        0
    ]
)

UNCERTAINTY_SOURCE_NAME_627A = str(
    best_uncertainty_candidate_627a[
        "object_name"
    ]
)

UNCERTAINTY_COLUMN_627A = str(
    best_uncertainty_candidate_627a[
        "uncertainty_column"
    ]
)


UNCERTAINTY_DEV_ENDPOINTS_627A = (
    globals()[
        UNCERTAINTY_SOURCE_NAME_627A
    ]
    .copy()
)


print(
    "\nSelected uncertainty source :",
    UNCERTAINTY_SOURCE_NAME_627A
)

print(
    "Selected uncertainty column :",
    UNCERTAINTY_COLUMN_627A
)

print(
    "Selected shape              :",
    UNCERTAINTY_DEV_ENDPOINTS_627A.shape
)


if len(
    UNCERTAINTY_DEV_ENDPOINTS_627A
) != EXPECTED_ENDPOINT_ROWS_627A:
    raise RuntimeError(
        "Highest-ranked DEV uncertainty table does not "
        f"contain the expected {EXPECTED_ENDPOINT_ROWS_627A} "
        "scenario-endpoint records. Refusing to guess."
    )


uncertainty_values_627a = (
    pd.to_numeric(
        UNCERTAINTY_DEV_ENDPOINTS_627A[
            UNCERTAINTY_COLUMN_627A
        ],
        errors="coerce",
    )
)


if not np.isfinite(
    uncertainty_values_627a
).all():
    raise RuntimeError(
        "Recovered DEV uncertainty column contains "
        "non-finite values."
    )


# ============================================================
# 8. Flatten the already-frozen 315-scenario manifest
# ============================================================

FAMILY_ORDER_627A = [
    "bias",
    "noise",
    "drift",
    "dropout",
    "stuck",
]


manifest_records_627a = []

global_index_627a = 0

for family_627a in (
    FAMILY_ORDER_627A
):

    if (
        family_627a
        not in scenario_manifests_620gc
    ):
        raise RuntimeError(
            "Frozen manifest missing corruption family: "
            + family_627a
        )

    family_manifest_627a = (
        scenario_manifests_620gc[
            family_627a
        ]
    )

    if isinstance(
        family_manifest_627a,
        pd.DataFrame,
    ):
        iterator_627a = (
            family_manifest_627a.iterrows()
        )

    else:
        raise RuntimeError(
            f"Manifest for {family_627a} "
            "is not a DataFrame."
        )

    for (
        family_position_627a,
        (
            _,
            scenario_row_627a,
        ),
    ) in enumerate(
        iterator_627a
    ):

        scenario_row_627a = (
            scenario_row_627a.copy()
        )

        generated_id_627a = (
            f"{family_627a}_"
            f"{family_position_627a + 1:03d}"
        )

        scenario_id_627a = (
            scenario_row_627a.get(
                "scenario_id",
                generated_id_627a,
            )
        )

        manifest_id_627a = (
            scenario_row_627a.get(
                "manifest_scenario_id",
                np.nan,
            )
        )

        sensor_627a = (
            scenario_row_627a.get(
                "sensor",
                np.nan,
            )
        )

        severity_627a = (
            scenario_row_627a.get(
                "severity",
                np.nan,
            )
        )

        manifest_records_627a.append(
            {
                "global_index":
                    global_index_627a,

                "family":
                    family_627a,

                "family_position":
                    family_position_627a,

                "scenario_id":
                    scenario_id_627a,

                "manifest_scenario_id":
                    manifest_id_627a,

                "sensor":
                    sensor_627a,

                "severity":
                    severity_627a,

                "_scenario_row":
                    scenario_row_627a,
            }
        )

        global_index_627a += 1


FROZEN_MANIFEST_INDEX_627A = (
    pd.DataFrame(
        manifest_records_627a
    )
)


print(
    "\n--- Frozen DEV corruption manifest ---"
)

print(
    "Scenario count :",
    len(
        FROZEN_MANIFEST_INDEX_627A
    )
)

print(
    "Family counts  :"
)

print(
    FROZEN_MANIFEST_INDEX_627A[
        "family"
    ]
    .value_counts(
        sort=False
    )
)


if len(
    FROZEN_MANIFEST_INDEX_627A
) != EXPECTED_SCENARIOS_627A:
    raise RuntimeError(
        "Frozen manifest does not contain exactly "
        "315 scenarios."
    )


# ============================================================
# 9. Rebuild frozen sensor-quality endpoint score for all
#    315 corrupted DEV scenarios
#
# IMPORTANT:
# - no RUL labels
# - no model inference
# - no MC-dropout rerun
# - no threshold selection
# ============================================================

print(
    "\n"
    "--- Building frozen sensor-quality endpoint scores "
    "for 315 DEV scenarios ---"
)


sensor_quality_endpoint_records_627a = []


for manifest_idx_627a, manifest_record_627a in (
    FROZEN_MANIFEST_INDEX_627A.iterrows()
):

    family_627a = (
        manifest_record_627a[
            "family"
        ]
    )

    scenario_row_627a = (
        manifest_record_627a[
            "_scenario_row"
        ]
    )

    # --------------------------------------------
    # 9A. Apply already-frozen corruption
    # --------------------------------------------

    corrupted_raw_627a = (
        _apply_dev_corruption_627a(
            val_raw_base_620gc,
            family_627a,
            scenario_row_627a,
        )
    )

    # --------------------------------------------
    # 9B. Build frozen 30-cycle quality candidates
    # --------------------------------------------

    candidate_windows_627a = (
        _build_candidate_windows_627a(
            corrupted_raw_627a
        )
    )

    # --------------------------------------------
    # 9C. Score using frozen calibration
    # --------------------------------------------

    scored_windows_627a = (
        _call_quality_scorer_627a(
            candidate_windows_627a
        )
    )

    # --------------------------------------------
    # 9D. Validate score
    # --------------------------------------------

    if (
        FROZEN_SENSOR_QUALITY_FUSION_627A
        not in scored_windows_627a.columns
    ):
        raise RuntimeError(
            "Frozen MAX sensor-quality score missing "
            "from scored window table."
        )

    quality_values_627a = (
        pd.to_numeric(
            scored_windows_627a[
                FROZEN_SENSOR_QUALITY_FUSION_627A
            ],
            errors="coerce",
        )
    )

    if not np.isfinite(
        quality_values_627a
    ).all():
        raise RuntimeError(
            f"Non-finite sensor-quality score found "
            f"for scenario "
            f"{manifest_record_627a['scenario_id']}."
        )

    if (
        (
            quality_values_627a
            < -1e-12
        ).any()
        or (
            quality_values_627a
            > 1.0 + 1e-12
        ).any()
    ):
        raise RuntimeError(
            "Frozen sensor-quality score escaped "
            "[0, 1]."
        )

    # --------------------------------------------
    # 9E. Take final eligible 30-cycle endpoint
    #     for each validation engine
    # --------------------------------------------

    endpoint_indices_627a = (
        scored_windows_627a
        .groupby(
            "unit_id",
            sort=True,
        )[
            "cycle"
        ]
        .idxmax()
    )

    endpoint_quality_627a = (
        scored_windows_627a
        .loc[
            endpoint_indices_627a,
            [
                "unit_id",
                "cycle",
                FROZEN_SENSOR_QUALITY_FUSION_627A,
            ],
        ]
        .copy()
        .sort_values(
            "unit_id"
        )
        .reset_index(
            drop=True
        )
    )

    if len(
        endpoint_quality_627a
    ) != EXPECTED_DEV_ENGINES_627A:
        raise RuntimeError(
            f"Scenario "
            f"{manifest_record_627a['scenario_id']} "
            f"produced "
            f"{len(endpoint_quality_627a)} "
            "endpoint engines instead of 20."
        )

    for _, endpoint_row_627a in (
        endpoint_quality_627a.iterrows()
    ):

        sensor_quality_endpoint_records_627a.append(
            {
                "global_index":
                    int(
                        manifest_record_627a[
                            "global_index"
                        ]
                    ),

                "family":
                    manifest_record_627a[
                        "family"
                    ],

                "family_position":
                    int(
                        manifest_record_627a[
                            "family_position"
                        ]
                    ),

                "scenario_id":
                    manifest_record_627a[
                        "scenario_id"
                    ],

                "manifest_scenario_id":
                    manifest_record_627a[
                        "manifest_scenario_id"
                    ],

                "sensor":
                    manifest_record_627a[
                        "sensor"
                    ],

                "severity":
                    manifest_record_627a[
                        "severity"
                    ],

                "unit_id":
                    endpoint_row_627a[
                        "unit_id"
                    ],

                "cycle":
                    endpoint_row_627a[
                        "cycle"
                    ],

                "sensor_quality_score":
                    float(
                        endpoint_row_627a[
                            FROZEN_SENSOR_QUALITY_FUSION_627A
                        ]
                    ),
            }
        )

    # Compact progress only.
    completed_627a = (
        manifest_idx_627a + 1
    )

    if (
        completed_627a == 1
        or completed_627a % 25 == 0
        or completed_627a
        == EXPECTED_SCENARIOS_627A
    ):
        print(
            f"Completed "
            f"{completed_627a:3d} / "
            f"{EXPECTED_SCENARIOS_627A}"
        )


SENSOR_QUALITY_DEV_ENDPOINTS_627A = (
    pd.DataFrame(
        sensor_quality_endpoint_records_627a
    )
)


print(
    "\n--- Frozen sensor-quality endpoint table ---"
)

print(
    "Shape             :",
    SENSOR_QUALITY_DEV_ENDPOINTS_627A.shape
)

print(
    "Unique scenarios  :",
    SENSOR_QUALITY_DEV_ENDPOINTS_627A[
        "global_index"
    ].nunique()
)

print(
    "Unique engines    :",
    SENSOR_QUALITY_DEV_ENDPOINTS_627A[
        "unit_id"
    ].nunique()
)

print(
    "Score range       :",
    (
        SENSOR_QUALITY_DEV_ENDPOINTS_627A[
            "sensor_quality_score"
        ].min(),
        SENSOR_QUALITY_DEV_ENDPOINTS_627A[
            "sensor_quality_score"
        ].max(),
    )
)


if len(
    SENSOR_QUALITY_DEV_ENDPOINTS_627A
) != EXPECTED_ENDPOINT_ROWS_627A:
    raise RuntimeError(
        "Sensor-quality endpoint table does not contain "
        "exactly 6300 rows."
    )


duplicate_quality_keys_627a = int(
    SENSOR_QUALITY_DEV_ENDPOINTS_627A
    .duplicated(
        subset=[
            "global_index",
            "unit_id",
        ]
    )
    .sum()
)


if duplicate_quality_keys_627a != 0:
    raise RuntimeError(
        "Duplicate scenario × engine keys found in "
        "sensor-quality endpoint table."
    )


# ============================================================
# 10. Determine safest exact join keys
# ============================================================

unc_cols_627a = set(
    UNCERTAINTY_DEV_ENDPOINTS_627A.columns
)

sq_cols_627a = set(
    SENSOR_QUALITY_DEV_ENDPOINTS_627A.columns
)


possible_key_sets_627a = [
    [
        "global_index",
        "unit_id",
        "cycle",
    ],
    [
        "global_index",
        "unit_id",
    ],
    [
        "family",
        "family_position",
        "unit_id",
        "cycle",
    ],
    [
        "family",
        "family_position",
        "unit_id",
    ],
    [
        "manifest_scenario_id",
        "unit_id",
        "cycle",
    ],
    [
        "manifest_scenario_id",
        "unit_id",
    ],
    [
        "scenario_id",
        "unit_id",
        "cycle",
    ],
    [
        "scenario_id",
        "unit_id",
    ],
]


JOIN_KEYS_627A = None

for key_set_627a in (
    possible_key_sets_627a
):

    if all(
        key_627a
        in unc_cols_627a
        and key_627a
        in sq_cols_627a
        for key_627a
        in key_set_627a
    ):

        # Avoid using a scenario-ID field if it is mostly missing.
        usable_627a = True

        for key_627a in (
            key_set_627a
        ):
            if (
                SENSOR_QUALITY_DEV_ENDPOINTS_627A[
                    key_627a
                ]
                .isna()
                .any()
            ):
                usable_627a = False

            if (
                UNCERTAINTY_DEV_ENDPOINTS_627A[
                    key_627a
                ]
                .isna()
                .any()
            ):
                usable_627a = False

        if usable_627a:
            JOIN_KEYS_627A = (
                key_set_627a
            )
            break


if JOIN_KEYS_627A is None:
    raise RuntimeError(
        "No safe common scenario × engine key set was "
        "found between frozen uncertainty and "
        "sensor-quality endpoint tables."
    )


print(
    "\n--- Alignment key ---"
)

print(
    "Join keys:",
    JOIN_KEYS_627A
)


# ============================================================
# 11. Validate uniqueness before merge
# ============================================================

unc_duplicate_keys_627a = int(
    UNCERTAINTY_DEV_ENDPOINTS_627A
    .duplicated(
        subset=JOIN_KEYS_627A
    )
    .sum()
)

sq_duplicate_keys_627a = int(
    SENSOR_QUALITY_DEV_ENDPOINTS_627A
    .duplicated(
        subset=JOIN_KEYS_627A
    )
    .sum()
)


print(
    "Uncertainty duplicate keys :",
    unc_duplicate_keys_627a
)

print(
    "Quality duplicate keys     :",
    sq_duplicate_keys_627a
)


if (
    unc_duplicate_keys_627a != 0
    or sq_duplicate_keys_627a != 0
):
    raise RuntimeError(
        "Alignment keys are not one-to-one."
    )


# ============================================================
# 12. Merge frozen uncertainty + frozen sensor-quality signal
# ============================================================

unc_keep_627a = list(
    dict.fromkeys(
        JOIN_KEYS_627A
        + [
            "family",
            "family_position",
            "scenario_id",
            "manifest_scenario_id",
            "sensor",
            "severity",
            UNCERTAINTY_COLUMN_627A,
        ]
    )
)


unc_keep_627a = [
    col_627a
    for col_627a
    in unc_keep_627a
    if col_627a
    in UNCERTAINTY_DEV_ENDPOINTS_627A.columns
]


uncertainty_merge_627a = (
    UNCERTAINTY_DEV_ENDPOINTS_627A[
        unc_keep_627a
    ]
    .copy()
)


# Rename metadata from uncertainty table where necessary so
# sensor-quality metadata remains authoritative after alignment.
rename_unc_metadata_627a = {}

for col_627a in [
    "family",
    "family_position",
    "scenario_id",
    "manifest_scenario_id",
    "sensor",
    "severity",
]:
    if (
        col_627a
        in uncertainty_merge_627a.columns
        and col_627a
        not in JOIN_KEYS_627A
    ):
        rename_unc_metadata_627a[
            col_627a
        ] = (
            "unc_"
            + col_627a
        )


uncertainty_merge_627a = (
    uncertainty_merge_627a.rename(
        columns=rename_unc_metadata_627a
    )
)


ALIGNED_DEV_RELIABILITY_SIGNALS_627A = (
    SENSOR_QUALITY_DEV_ENDPOINTS_627A
    .merge(
        uncertainty_merge_627a,
        on=JOIN_KEYS_627A,
        how="inner",
        validate="one_to_one",
    )
)


if len(
    ALIGNED_DEV_RELIABILITY_SIGNALS_627A
) != EXPECTED_ENDPOINT_ROWS_627A:
    raise RuntimeError(
        "Alignment did not recover exactly "
        "6300 scenario × engine records."
    )


ALIGNED_DEV_RELIABILITY_SIGNALS_627A[
    "model_uncertainty"
] = pd.to_numeric(
    ALIGNED_DEV_RELIABILITY_SIGNALS_627A[
        UNCERTAINTY_COLUMN_627A
    ],
    errors="coerce",
)


if not np.isfinite(
    ALIGNED_DEV_RELIABILITY_SIGNALS_627A[
        [
            "model_uncertainty",
            "sensor_quality_score",
        ]
    ].to_numpy(
        dtype=float
    )
).all():
    raise RuntimeError(
        "Aligned reliability-signal table contains "
        "non-finite values."
    )


# ============================================================
# 13. Diagnostic dependence only
#
# IMPORTANT:
# Correlation is descriptive here.
# It is NOT being used to choose a final reliability formula.
# ============================================================

pearson_corr_627a = (
    ALIGNED_DEV_RELIABILITY_SIGNALS_627A[
        [
            "model_uncertainty",
            "sensor_quality_score",
        ]
    ]
    .corr(
        method="pearson"
    )
    .iloc[
        0,
        1,
    ]
)


spearman_corr_627a = (
    ALIGNED_DEV_RELIABILITY_SIGNALS_627A[
        [
            "model_uncertainty",
            "sensor_quality_score",
        ]
    ]
    .corr(
        method="spearman"
    )
    .iloc[
        0,
        1,
    ]
)


FAMILY_SIGNAL_SUMMARY_627A = (
    ALIGNED_DEV_RELIABILITY_SIGNALS_627A
    .groupby(
        "family",
        sort=False,
    )
    .agg(
        scenarios=(
            "global_index",
            "nunique",
        ),

        records=(
            "unit_id",
            "size",
        ),

        mean_model_uncertainty=(
            "model_uncertainty",
            "mean",
        ),

        std_model_uncertainty=(
            "model_uncertainty",
            "std",
        ),

        mean_sensor_quality_score=(
            "sensor_quality_score",
            "mean",
        ),

        std_sensor_quality_score=(
            "sensor_quality_score",
            "std",
        ),
    )
    .reset_index()
)


family_correlations_627a = []

for (
    family_627a,
    group_627a,
) in (
    ALIGNED_DEV_RELIABILITY_SIGNALS_627A
    .groupby(
        "family",
        sort=False,
    )
):

    family_correlations_627a.append(
        {
            "family":
                family_627a,

            "pearson_corr":
                group_627a[
                    [
                        "model_uncertainty",
                        "sensor_quality_score",
                    ]
                ]
                .corr(
                    method="pearson"
                )
                .iloc[
                    0,
                    1,
                ],

            "spearman_corr":
                group_627a[
                    [
                        "model_uncertainty",
                        "sensor_quality_score",
                    ]
                ]
                .corr(
                    method="spearman"
                )
                .iloc[
                    0,
                    1,
                ],
        }
    )


FAMILY_SIGNAL_CORRELATION_627A = (
    pd.DataFrame(
        family_correlations_627a
    )
)


FAMILY_SIGNAL_SUMMARY_627A = (
    FAMILY_SIGNAL_SUMMARY_627A
    .merge(
        FAMILY_SIGNAL_CORRELATION_627A,
        on="family",
        how="left",
        validate="one_to_one",
    )
)


# ============================================================
# 14. Compact review output
# ============================================================

print(
    "\n--- Family-level aligned-signal summary ---"
)

display(
    FAMILY_SIGNAL_SUMMARY_627A.round(
        6
    )
)


print(
    "\n--- First 10 aligned endpoint records ---"
)

display_columns_627a = [
    col_627a
    for col_627a
    in [
        "global_index",
        "family",
        "family_position",
        "scenario_id",
        "sensor",
        "severity",
        "unit_id",
        "cycle",
        "model_uncertainty",
        "sensor_quality_score",
    ]
    if col_627a
    in ALIGNED_DEV_RELIABILITY_SIGNALS_627A.columns
]


display(
    ALIGNED_DEV_RELIABILITY_SIGNALS_627A[
        display_columns_627a
    ]
    .head(
        10
    )
    .round(
        6
    )
)


# ============================================================
# 15. Final integrity summary
# ============================================================

scenario_count_627a = int(
    ALIGNED_DEV_RELIABILITY_SIGNALS_627A[
        "global_index"
    ].nunique()
)

engine_count_distribution_627a = (
    ALIGNED_DEV_RELIABILITY_SIGNALS_627A
    .groupby(
        "global_index"
    )[
        "unit_id"
    ]
    .nunique()
)

all_scenarios_have_20_engines_627a = bool(
    (
        engine_count_distribution_627a
        == EXPECTED_DEV_ENGINES_627A
    ).all()
)


print(
    "\n"
    "==============================================================\n"
    "STEP 6.27A FINAL ALIGNMENT SUMMARY\n"
    "=============================================================="
)

print(
    "Frozen uncertainty source       :",
    UNCERTAINTY_SOURCE_NAME_627A
)

print(
    "Frozen uncertainty column       :",
    UNCERTAINTY_COLUMN_627A
)

print(
    "Frozen sensor-quality fusion    :",
    FROZEN_SENSOR_QUALITY_FUSION_627A
)

print(
    "Aligned scenarios               :",
    scenario_count_627a,
    "/ 315",
)

print(
    "Aligned endpoint records        :",
    len(
        ALIGNED_DEV_RELIABILITY_SIGNALS_627A
    ),
    "/ 6300",
)

print(
    "20 engines in every scenario    :",
    all_scenarios_have_20_engines_627a
)

print(
    "Uncertainty values finite       :",
    bool(
        np.isfinite(
            ALIGNED_DEV_RELIABILITY_SIGNALS_627A[
                "model_uncertainty"
            ]
        ).all()
    )
)

print(
    "Sensor-quality values finite    :",
    bool(
        np.isfinite(
            ALIGNED_DEV_RELIABILITY_SIGNALS_627A[
                "sensor_quality_score"
            ]
        ).all()
    )
)

print(
    "Sensor-quality score in [0,1]   :",
    bool(
        ALIGNED_DEV_RELIABILITY_SIGNALS_627A[
            "sensor_quality_score"
        ]
        .between(
            0.0,
            1.0,
            inclusive="both",
        )
        .all()
    )
)

print(
    "Overall Pearson correlation     :",
    float(
        pearson_corr_627a
    )
)

print(
    "Overall Spearman correlation    :",
    float(
        spearman_corr_627a
    )
)

print(
    "\nOfficial TEST labels used       :",
    OFFICIAL_TEST_LABELS_USED_627A
)

print(
    "Official TEST scored            :",
    OFFICIAL_TEST_SCORED_627A
)

print(
    "Model inference rerun           :",
    MODEL_INFERENCE_EXECUTED_627A
)

print(
    "MC-dropout rerun                :",
    MC_DROPOUT_EXECUTED_627A
)

print(
    "Threshold selected              :",
    THRESHOLD_SELECTED_627A
)

print(
    "Reliability rule changed        :",
    RELIABILITY_RULE_CHANGED_627A
)

print(
    "Sensor-quality representation "
    "changed :",
    SENSOR_QUALITY_REPRESENTATION_CHANGED_627A
)

print(
    "Files written                   :",
    FILES_WRITTEN_627A
)


if scenario_count_627a != 315:
    raise RuntimeError(
        "Final alignment scenario count is not 315."
    )

if len(
    ALIGNED_DEV_RELIABILITY_SIGNALS_627A
) != 6300:
    raise RuntimeError(
        "Final alignment record count is not 6300."
    )

if not all_scenarios_have_20_engines_627a:
    raise RuntimeError(
        "One or more scenarios do not contain exactly "
        "20 DEV engines."
    )


print(
    "\nDEV UNCERTAINTY ↔ SENSOR-QUALITY ALIGNMENT READY : True"
)

print(
    "\nNo reliability-combination rule was selected."
)

print(
    "No new threshold was selected."
)

print(
    "No frozen research decision was modified."
)

print(
    "\n=== END STEP 6.27A ==="
)

In [ ]:
# ============================================================
# Step 6.27A-FIX1
# Locate the exact frozen DEV uncertainty endpoint artifact
# READ-ONLY / NO TEST SCORING / NO FILE WRITES
# ============================================================

import numpy as np
import pandas as pd

print(
    "\n"
    "============================================================\n"
    "STEP 6.27A-FIX1 — LOCATE FROZEN DEV UNCERTAINTY ARTIFACT\n"
    "============================================================"
)

EXPECTED_DEV_ENGINES_FIX1 = 20
EXPECTED_SCENARIOS_FIX1 = 315
EXPECTED_ROWS_FIX1 = (
    EXPECTED_DEV_ENGINES_FIX1
    * EXPECTED_SCENARIOS_FIX1
)

uncertainty_keywords_fix1 = [
    "mc_std",
    "uncertainty",
    "predictive_std",
    "prediction_std",
    "std_prediction",
]

scenario_key_candidates_fix1 = [
    "global_index",
    "scenario_id",
    "manifest_scenario_id",
    "family_position",
]

records_fix1 = []

for object_name_fix1, object_value_fix1 in list(globals().items()):

    if not isinstance(object_value_fix1, pd.DataFrame):
        continue

    df_fix1 = object_value_fix1

    lower_columns_fix1 = {
        str(col_fix1).lower(): col_fix1
        for col_fix1 in df_fix1.columns
    }

    uncertainty_columns_fix1 = []

    for original_col_fix1 in df_fix1.columns:

        lower_col_fix1 = str(
            original_col_fix1
        ).lower()

        if (
            "uncert" in lower_col_fix1
            or (
                "mc" in lower_col_fix1
                and "std" in lower_col_fix1
            )
            or (
                "prediction" in lower_col_fix1
                and "std" in lower_col_fix1
            )
        ):
            uncertainty_columns_fix1.append(
                original_col_fix1
            )

    if not uncertainty_columns_fix1:
        continue

    has_unit_fix1 = (
        "unit_id"
        in df_fix1.columns
    )

    unit_count_fix1 = (
        int(
            df_fix1["unit_id"].nunique()
        )
        if has_unit_fix1
        else None
    )

    scenario_key_fix1 = None
    scenario_count_fix1 = None

    for key_fix1 in scenario_key_candidates_fix1:

        if key_fix1 in df_fix1.columns:

            candidate_count_fix1 = int(
                df_fix1[
                    key_fix1
                ].nunique(
                    dropna=True
                )
            )

            # Prefer a key that actually looks like
            # the frozen 315-scenario identifier.
            if candidate_count_fix1 == 315:
                scenario_key_fix1 = key_fix1
                scenario_count_fix1 = (
                    candidate_count_fix1
                )
                break

            if scenario_key_fix1 is None:
                scenario_key_fix1 = key_fix1
                scenario_count_fix1 = (
                    candidate_count_fix1
                )

    family_count_fix1 = (
        int(
            df_fix1["family"].nunique(
                dropna=True
            )
        )
        if "family" in df_fix1.columns
        else None
    )

    row_match_fix1 = (
        len(df_fix1)
        == EXPECTED_ROWS_FIX1
    )

    engine_match_fix1 = (
        unit_count_fix1
        == EXPECTED_DEV_ENGINES_FIX1
    )

    scenario_match_fix1 = (
        scenario_count_fix1
        == EXPECTED_SCENARIOS_FIX1
    )

    likely_dev_fix1 = bool(
        row_match_fix1
        or (
            engine_match_fix1
            and scenario_match_fix1
        )
    )

    name_upper_fix1 = (
        object_name_fix1.upper()
    )

    obvious_test_name_fix1 = (
        "OFFICIAL_TEST" in name_upper_fix1
        or "TEST_" in name_upper_fix1
    )

    records_fix1.append(
        {
            "object_name":
                object_name_fix1,

            "rows":
                len(df_fix1),

            "columns":
                len(df_fix1.columns),

            "unit_count":
                unit_count_fix1,

            "scenario_key":
                scenario_key_fix1,

            "scenario_count":
                scenario_count_fix1,

            "family_count":
                family_count_fix1,

            "uncertainty_columns":
                ", ".join(
                    map(
                        str,
                        uncertainty_columns_fix1,
                    )
                ),

            "exact_6300_rows":
                row_match_fix1,

            "exact_20_engines":
                engine_match_fix1,

            "exact_315_scenarios":
                scenario_match_fix1,

            "likely_DEV_endpoint_table":
                likely_dev_fix1,

            "obvious_TEST_name":
                obvious_test_name_fix1,
        }
    )


UNCERTAINTY_INVENTORY_627A_FIX1 = (
    pd.DataFrame(
        records_fix1
    )
)


if UNCERTAINTY_INVENTORY_627A_FIX1.empty:

    print(
        "\nNo DataFrame containing an uncertainty-like "
        "column was found."
    )

else:

    # --------------------------------------------------------
    # First show strongest DEV candidates only
    # --------------------------------------------------------

    dev_candidates_fix1 = (
        UNCERTAINTY_INVENTORY_627A_FIX1[
            UNCERTAINTY_INVENTORY_627A_FIX1[
                "likely_DEV_endpoint_table"
            ]
        ]
        .copy()
    )

    print(
        "\n--- STRONG DEV CANDIDATES ---"
    )

    if dev_candidates_fix1.empty:

        print(
            "NONE FOUND"
        )

    else:

        display(
            dev_candidates_fix1[
                [
                    "object_name",
                    "rows",
                    "columns",
                    "unit_count",
                    "scenario_key",
                    "scenario_count",
                    "family_count",
                    "uncertainty_columns",
                    "exact_6300_rows",
                    "exact_20_engines",
                    "exact_315_scenarios",
                ]
            ]
            .sort_values(
                [
                    "exact_6300_rows",
                    "exact_20_engines",
                    "exact_315_scenarios",
                ],
                ascending=False,
            )
            .reset_index(
                drop=True
            )
        )

    # --------------------------------------------------------
    # Compact full uncertainty inventory
    # --------------------------------------------------------

    print(
        "\n--- COMPACT UNCERTAINTY INVENTORY ---"
    )

    compact_fix1 = (
        UNCERTAINTY_INVENTORY_627A_FIX1[
            [
                "object_name",
                "rows",
                "unit_count",
                "scenario_count",
                "family_count",
                "uncertainty_columns",
                "likely_DEV_endpoint_table",
                "obvious_TEST_name",
            ]
        ]
        .copy()
    )

    # Show most useful objects first.
    compact_fix1[
        "_rank"
    ] = (
        UNCERTAINTY_INVENTORY_627A_FIX1[
            "exact_6300_rows"
        ].astype(int) * 100
        + UNCERTAINTY_INVENTORY_627A_FIX1[
            "exact_20_engines"
        ].astype(int) * 50
        + UNCERTAINTY_INVENTORY_627A_FIX1[
            "exact_315_scenarios"
        ].astype(int) * 25
        - UNCERTAINTY_INVENTORY_627A_FIX1[
            "obvious_TEST_name"
        ].astype(int) * 10
    )

    compact_fix1 = (
        compact_fix1
        .join(
            UNCERTAINTY_INVENTORY_627A_FIX1[
                "_rank"
            ]
            if "_rank"
            in UNCERTAINTY_INVENTORY_627A_FIX1.columns
            else pd.Series(
                0,
                index=compact_fix1.index,
                name="_rank",
            )
        )
        if "_rank"
        not in compact_fix1.columns
        else compact_fix1
    )

    # Simpler explicit sort vector
    compact_fix1[
        "rank"
    ] = [
        (
            100 * int(r)
            + 50 * int(e)
            + 25 * int(s)
            - 10 * int(t)
        )
        for r, e, s, t
        in zip(
            UNCERTAINTY_INVENTORY_627A_FIX1[
                "exact_6300_rows"
            ],
            UNCERTAINTY_INVENTORY_627A_FIX1[
                "exact_20_engines"
            ],
            UNCERTAINTY_INVENTORY_627A_FIX1[
                "exact_315_scenarios"
            ],
            UNCERTAINTY_INVENTORY_627A_FIX1[
                "obvious_TEST_name"
            ],
        )
    ]

    compact_fix1 = (
        compact_fix1
        .sort_values(
            [
                "rank",
                "rows",
            ],
            ascending=[
                False,
                True,
            ],
        )
        .drop(
            columns=[
                "rank",
            ]
        )
        .reset_index(
            drop=True
        )
    )

    display(
        compact_fix1.head(
            30
        )
    )


# ============================================================
# Final compact verdict
# ============================================================

strong_candidates_fix1 = (
    UNCERTAINTY_INVENTORY_627A_FIX1[
        UNCERTAINTY_INVENTORY_627A_FIX1[
            "likely_DEV_endpoint_table"
        ]
    ]
    if not UNCERTAINTY_INVENTORY_627A_FIX1.empty
    else pd.DataFrame()
)


print(
    "\n"
    "============================================================\n"
    "STEP 6.27A-FIX1 SUMMARY\n"
    "============================================================"
)

print(
    "Expected DEV endpoint rows :",
    EXPECTED_ROWS_FIX1,
)

print(
    "Expected scenarios         :",
    EXPECTED_SCENARIOS_FIX1,
)

print(
    "Expected DEV engines       :",
    EXPECTED_DEV_ENGINES_FIX1,
)

print(
    "Strong DEV candidates found:",
    len(
        strong_candidates_fix1
    ),
)

if len(
    strong_candidates_fix1
) == 1:

    print(
        "Unique DEV candidate      :",
        strong_candidates_fix1.iloc[
            0
        ][
            "object_name"
        ],
    )

elif len(
    strong_candidates_fix1
) > 1:

    print(
        "Multiple candidates exist; "
        "we will compare them before selecting."
    )

else:

    print(
        "No 315 × 20 DEV uncertainty table is currently "
        "materialized in memory."
    )


print(
    "\nOfficial TEST labels inspected : False"
)

print(
    "Official TEST scored           : False"
)

print(
    "Model inference rerun           : False"
)

print(
    "MC-dropout rerun                : False"
)

print(
    "Threshold selected              : False"
)

print(
    "Research decision changed       : False"
)

print(
    "Files written                   : False"
)

print(
    "\n=== END STEP 6.27A-FIX1 ==="
)

In [ ]:
# ============================================================
# STEP 6.27A-FIX2
# Search MEMORY + SAVED RESULTS for frozen DEV scenario
# endpoint artifacts
#
# READ ONLY
# NO MODEL INFERENCE
# NO MC-DROPOUT
# NO TEST LABELS
# NO FILE WRITES
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import os

print(
    "\n"
    "============================================================\n"
    "STEP 6.27A-FIX2 — SEARCH FOR FROZEN DEV SCENARIO ARTIFACTS\n"
    "============================================================"
)

# ------------------------------------------------------------
# 1. Constants
# ------------------------------------------------------------

EXPECTED_SCENARIOS_627FIX2 = 315
EXPECTED_DEV_ENGINES_627FIX2 = 20
EXPECTED_DEV_ROWS_627FIX2 = (
    EXPECTED_SCENARIOS_627FIX2
    * EXPECTED_DEV_ENGINES_627FIX2
)

print(
    "\nExpected scenarios     :",
    EXPECTED_SCENARIOS_627FIX2
)
print(
    "Expected DEV engines   :",
    EXPECTED_DEV_ENGINES_627FIX2
)
print(
    "Expected endpoint rows :",
    EXPECTED_DEV_ROWS_627FIX2
)


# ------------------------------------------------------------
# 2. Helper: characterize a DataFrame
# ------------------------------------------------------------

def characterize_df_627fix2(
    name_627fix2,
    df_627fix2,
):
    result_627fix2 = {
        "object_name": name_627fix2,
        "rows": len(df_627fix2),
        "columns": len(df_627fix2.columns),
        "unit_count": np.nan,
        "scenario_count": np.nan,
        "family_count": np.nan,
        "uncertainty_columns": "",
        "prediction_columns": "",
        "row_match_6300": (
            len(df_627fix2)
            == EXPECTED_DEV_ROWS_627FIX2
        ),
    }

    if "unit_id" in df_627fix2.columns:
        try:
            result_627fix2[
                "unit_count"
            ] = int(
                df_627fix2[
                    "unit_id"
                ].nunique(
                    dropna=True
                )
            )
        except Exception:
            pass

    scenario_candidates_627fix2 = [
        "global_index",
        "scenario_id",
        "manifest_scenario_id",
    ]

    for col_627fix2 in scenario_candidates_627fix2:
        if col_627fix2 in df_627fix2.columns:
            try:
                count_627fix2 = int(
                    df_627fix2[
                        col_627fix2
                    ].nunique(
                        dropna=True
                    )
                )

                if (
                    count_627fix2
                    == EXPECTED_SCENARIOS_627FIX2
                ):
                    result_627fix2[
                        "scenario_count"
                    ] = count_627fix2
                    break

                if np.isnan(
                    result_627fix2[
                        "scenario_count"
                    ]
                ):
                    result_627fix2[
                        "scenario_count"
                    ] = count_627fix2

            except Exception:
                pass

    if "family" in df_627fix2.columns:
        try:
            result_627fix2[
                "family_count"
            ] = int(
                df_627fix2[
                    "family"
                ].nunique(
                    dropna=True
                )
            )
        except Exception:
            pass

    uncertainty_cols_627fix2 = []
    prediction_cols_627fix2 = []

    for col_627fix2 in df_627fix2.columns:

        lower_627fix2 = str(
            col_627fix2
        ).lower()

        if (
            "uncert" in lower_627fix2
            or (
                "mc" in lower_627fix2
                and "std" in lower_627fix2
            )
            or (
                "prediction" in lower_627fix2
                and "std" in lower_627fix2
            )
        ):
            uncertainty_cols_627fix2.append(
                str(col_627fix2)
            )

        if (
            "prediction" in lower_627fix2
            or lower_627fix2 == "pred"
            or lower_627fix2.endswith(
                "_pred"
            )
        ):
            prediction_cols_627fix2.append(
                str(col_627fix2)
            )

    result_627fix2[
        "uncertainty_columns"
    ] = ", ".join(
        uncertainty_cols_627fix2
    )

    result_627fix2[
        "prediction_columns"
    ] = ", ".join(
        prediction_cols_627fix2
    )

    result_627fix2[
        "exact_DEV_structure"
    ] = bool(
        len(df_627fix2)
        == EXPECTED_DEV_ROWS_627FIX2
        and (
            pd.isna(
                result_627fix2[
                    "unit_count"
                ]
            )
            or result_627fix2[
                "unit_count"
            ]
            == EXPECTED_DEV_ENGINES_627FIX2
        )
        and (
            pd.isna(
                result_627fix2[
                    "scenario_count"
                ]
            )
            or result_627fix2[
                "scenario_count"
            ]
            == EXPECTED_SCENARIOS_627FIX2
        )
    )

    return result_627fix2


# ============================================================
# 3. SEARCH ALL IN-MEMORY DATAFRAMES
# ============================================================

print(
    "\n--- Searching ALL in-memory DataFrames ---"
)

memory_records_627fix2 = []

for (
    name_627fix2,
    value_627fix2
) in list(
    globals().items()
):

    if not isinstance(
        value_627fix2,
        pd.DataFrame,
    ):
        continue

    try:
        memory_records_627fix2.append(
            characterize_df_627fix2(
                name_627fix2,
                value_627fix2,
            )
        )
    except Exception:
        continue


MEMORY_INVENTORY_627FIX2 = (
    pd.DataFrame(
        memory_records_627fix2
    )
)

if MEMORY_INVENTORY_627FIX2.empty:

    print(
        "No DataFrames found in memory."
    )

else:

    exact_memory_627fix2 = (
        MEMORY_INVENTORY_627FIX2[
            MEMORY_INVENTORY_627FIX2[
                "rows"
            ]
            == EXPECTED_DEV_ROWS_627FIX2
        ]
        .copy()
    )

    print(
        "\n--- IN-MEMORY 6,300-ROW CANDIDATES ---"
    )

    if exact_memory_627fix2.empty:

        print(
            "NONE FOUND"
        )

    else:

        display(
            exact_memory_627fix2[
                [
                    "object_name",
                    "rows",
                    "columns",
                    "unit_count",
                    "scenario_count",
                    "family_count",
                    "uncertainty_columns",
                    "prediction_columns",
                    "exact_DEV_structure",
                ]
            ]
            .reset_index(
                drop=True
            )
        )


# ============================================================
# 4. Locate project root safely
# ============================================================

cwd_627fix2 = Path.cwd()

project_candidates_627fix2 = [
    cwd_627fix2,
    cwd_627fix2.parent,
    Path(
        r"C:\RUL_Reliability_Research"
    ),
]

PROJECT_ROOT_627FIX2 = None

for candidate_627fix2 in (
    project_candidates_627fix2
):

    if (
        candidate_627fix2.exists()
        and (
            candidate_627fix2
            / "06_results"
        ).exists()
    ):
        PROJECT_ROOT_627FIX2 = (
            candidate_627fix2
        )
        break


if PROJECT_ROOT_627FIX2 is None:

    raise RuntimeError(
        "Could not safely locate "
        "C:\\RUL_Reliability_Research."
    )


print(
    "\nProject root:",
    PROJECT_ROOT_627FIX2
)


# ============================================================
# 5. Search existing SAVED result/artifact files
# ============================================================

search_roots_627fix2 = []

for relative_627fix2 in [
    "05_experiments",
    "06_results",
]:

    candidate_627fix2 = (
        PROJECT_ROOT_627FIX2
        / relative_627fix2
    )

    if candidate_627fix2.exists():

        search_roots_627fix2.append(
            candidate_627fix2
        )


allowed_suffixes_627fix2 = {
    ".csv",
    ".parquet",
    ".pkl",
    ".pickle",
}


candidate_files_627fix2 = []

for root_627fix2 in (
    search_roots_627fix2
):

    for path_627fix2 in (
        root_627fix2.rglob("*")
    ):

        if (
            path_627fix2.is_file()
            and path_627fix2.suffix.lower()
            in allowed_suffixes_627fix2
        ):

            candidate_files_627fix2.append(
                path_627fix2
            )


print(
    "\nExisting tabular artifact files found:",
    len(candidate_files_627fix2)
)


# ============================================================
# 6. Prioritize likely DEV / uncertainty / scenario artifacts
# ============================================================

def path_priority_627fix2(
    path_627fix2
):
    text_627fix2 = str(
        path_627fix2
    ).lower()

    score_627fix2 = 0

    positive_tokens_627fix2 = {
        "dev": 8,
        "val": 5,
        "validation": 8,
        "uncert": 8,
        "mc": 4,
        "scenario": 5,
        "prescore": 3,
        "corrupt": 4,
        "engine": 2,
        "endpoint": 4,
        "step6": 2,
    }

    negative_tokens_627fix2 = {
        "official_test": -20,
        "official-test": -20,
        "full_scoring": -15,
    }

    for (
        token_627fix2,
        weight_627fix2
    ) in positive_tokens_627fix2.items():

        if token_627fix2 in text_627fix2:
            score_627fix2 += (
                weight_627fix2
            )

    for (
        token_627fix2,
        weight_627fix2
    ) in negative_tokens_627fix2.items():

        if token_627fix2 in text_627fix2:
            score_627fix2 += (
                weight_627fix2
            )

    return score_627fix2


ranked_files_627fix2 = sorted(
    candidate_files_627fix2,
    key=lambda p_627fix2: (
        path_priority_627fix2(
            p_627fix2
        )
    ),
    reverse=True,
)


# Limit this read-only scan to strongest candidates first.
MAX_FILES_TO_INSPECT_627FIX2 = 250

ranked_files_627fix2 = (
    ranked_files_627fix2[
        :MAX_FILES_TO_INSPECT_627FIX2
    ]
)


# ============================================================
# 7. Read candidate artifacts and inspect structure
# ============================================================

disk_records_627fix2 = []

for path_627fix2 in (
    ranked_files_627fix2
):

    try:

        suffix_627fix2 = (
            path_627fix2
            .suffix
            .lower()
        )

        if suffix_627fix2 == ".csv":

            df_627fix2 = pd.read_csv(
                path_627fix2
            )

        elif suffix_627fix2 == ".parquet":

            df_627fix2 = pd.read_parquet(
                path_627fix2
            )

        elif suffix_627fix2 in {
            ".pkl",
            ".pickle",
        }:

            obj_627fix2 = pd.read_pickle(
                path_627fix2
            )

            if not isinstance(
                obj_627fix2,
                pd.DataFrame,
            ):
                continue

            df_627fix2 = obj_627fix2

        else:
            continue


        record_627fix2 = (
            characterize_df_627fix2(
                str(
                    path_627fix2.relative_to(
                        PROJECT_ROOT_627FIX2
                    )
                ),
                df_627fix2,
            )
        )

        record_627fix2[
            "full_path"
        ] = str(
            path_627fix2
        )

        record_627fix2[
            "path_priority"
        ] = path_priority_627fix2(
            path_627fix2
        )

        disk_records_627fix2.append(
            record_627fix2
        )

    except Exception:
        # Inspection only.
        # Unreadable/unsupported artifact is skipped.
        continue


DISK_INVENTORY_627FIX2 = (
    pd.DataFrame(
        disk_records_627fix2
    )
)


# ============================================================
# 8. Report 6,300-row saved candidates
# ============================================================

print(
    "\n--- SAVED 6,300-ROW CANDIDATES ---"
)

if DISK_INVENTORY_627FIX2.empty:

    saved_6300_627fix2 = (
        pd.DataFrame()
    )

    print(
        "No readable tabular artifacts found."
    )

else:

    saved_6300_627fix2 = (
        DISK_INVENTORY_627FIX2[
            DISK_INVENTORY_627FIX2[
                "rows"
            ]
            == EXPECTED_DEV_ROWS_627FIX2
        ]
        .copy()
    )

    if saved_6300_627fix2.empty:

        print(
            "NONE FOUND"
        )

    else:

        display(
            saved_6300_627fix2[
                [
                    "object_name",
                    "rows",
                    "columns",
                    "unit_count",
                    "scenario_count",
                    "family_count",
                    "uncertainty_columns",
                    "prediction_columns",
                    "exact_DEV_structure",
                    "path_priority",
                ]
            ]
            .sort_values(
                [
                    "exact_DEV_structure",
                    "path_priority",
                ],
                ascending=[
                    False,
                    False,
                ],
            )
            .reset_index(
                drop=True
            )
        )


# ============================================================
# 9. Also show near-matching saved candidates
# ============================================================

print(
    "\n--- SAVED DEV-LIKE NEAR CANDIDATES ---"
)

if DISK_INVENTORY_627FIX2.empty:

    near_candidates_627fix2 = (
        pd.DataFrame()
    )

    print(
        "NONE"
    )

else:

    near_mask_627fix2 = (

        (
            DISK_INVENTORY_627FIX2[
                "unit_count"
            ]
            == EXPECTED_DEV_ENGINES_627FIX2
        )

        |

        (
            DISK_INVENTORY_627FIX2[
                "scenario_count"
            ]
            == EXPECTED_SCENARIOS_627FIX2
        )

        |

        (
            DISK_INVENTORY_627FIX2[
                "uncertainty_columns"
            ]
            .astype(str)
            .str.len()
            > 0
        )

    )

    near_candidates_627fix2 = (
        DISK_INVENTORY_627FIX2[
            near_mask_627fix2
        ]
        .copy()
    )

    near_candidates_627fix2 = (
        near_candidates_627fix2[
            ~near_candidates_627fix2[
                "object_name"
            ].isin(
                saved_6300_627fix2[
                    "object_name"
                ]
                if not saved_6300_627fix2.empty
                else []
            )
        ]
    )

    if near_candidates_627fix2.empty:

        print(
            "NONE"
        )

    else:

        display(
            near_candidates_627fix2[
                [
                    "object_name",
                    "rows",
                    "columns",
                    "unit_count",
                    "scenario_count",
                    "family_count",
                    "uncertainty_columns",
                    "prediction_columns",
                    "path_priority",
                ]
            ]
            .sort_values(
                [
                    "path_priority",
                    "rows",
                ],
                ascending=[
                    False,
                    True,
                ],
            )
            .head(
                30
            )
            .reset_index(
                drop=True
            )
        )


# ============================================================
# 10. Compact decision summary
# ============================================================

memory_6300_count_627fix2 = int(
    (
        MEMORY_INVENTORY_627FIX2[
            "rows"
        ]
        == EXPECTED_DEV_ROWS_627FIX2
    ).sum()
) if not MEMORY_INVENTORY_627FIX2.empty else 0


disk_6300_count_627fix2 = int(
    len(
        saved_6300_627fix2
    )
)


print(
    "\n"
    "============================================================\n"
    "STEP 6.27A-FIX2 SUMMARY\n"
    "============================================================"
)

print(
    "Expected DEV endpoint rows     :",
    EXPECTED_DEV_ROWS_627FIX2,
)

print(
    "6,300-row objects in memory    :",
    memory_6300_count_627fix2,
)

print(
    "6,300-row saved artifacts      :",
    disk_6300_count_627fix2,
)


if disk_6300_count_627fix2 > 0:

    print(
        "STATUS : Candidate frozen DEV "
        "endpoint artifact(s) found."
    )

elif memory_6300_count_627fix2 > 0:

    print(
        "STATUS : Candidate frozen DEV "
        "endpoint object(s) found in memory."
    )

else:

    print(
        "STATUS : No precomputed 315 × 20 "
        "DEV endpoint artifact found."
    )

    print(
        "NEXT   : We will reconstruct it only "
        "from already-frozen DEV/model settings."
    )


print(
    "\nOfficial TEST labels inspected : False"
)

print(
    "Official TEST scored           : False"
)

print(
    "Model inference rerun           : False"
)

print(
    "MC-dropout rerun                : False"
)

print(
    "Corruptions regenerated         : False"
)

print(
    "Threshold selected              : False"
)

print(
    "Research decision changed       : False"
)

print(
    "Files written                   : False"
)

print(
    "\n=== END STEP 6.27A-FIX2 ==="
)

In [ ]:
# ============================================================
# STEP 6.27A-FIX3
# Audit exact frozen DEV inference / MC-dropout contracts
#
# READ ONLY
# NO MODEL INFERENCE
# NO MC-DROPOUT
# NO CORRUPTION GENERATION
# NO TEST LABELS
# NO FILE WRITES
# ============================================================

import inspect
import pandas as pd
import numpy as np

print(
    "\n"
    "============================================================\n"
    "STEP 6.27A-FIX3 — AUDIT FROZEN DEV INFERENCE CONTRACTS\n"
    "============================================================"
)

# ------------------------------------------------------------
# 1. Explicitly check the important known objects
# ------------------------------------------------------------

known_objects_627fix3 = [
    "val_raw_base_620gc",
    "scenario_manifests_620gc",
    "apply_frozen_corruption_620gc",
    "PRIMARY_INPUT_COLUMNS",
    "scaler",
    "mc_uncertainty_623cb2",
    "model",
    "X_dev",
    "X_val",
    "X_dev_clean",
    "X_val_clean",
    "clean_prediction_df",
    "mc_ranking_df",
    "SENSOR_QUALITY_VAL_CLEAN_SCORED_626D",
]

print("\n--- Known-object availability ---")

for name_627fix3 in known_objects_627fix3:
    exists_627fix3 = name_627fix3 in globals()

    print(
        f"{name_627fix3:<40}: "
        f"{exists_627fix3}"
    )

    if exists_627fix3:
        obj_627fix3 = globals()[name_627fix3]

        if isinstance(
            obj_627fix3,
            pd.DataFrame,
        ):
            print(
                "    type=DataFrame"
                f", shape={obj_627fix3.shape}"
            )

        elif isinstance(
            obj_627fix3,
            np.ndarray,
        ):
            print(
                "    type=ndarray"
                f", shape={obj_627fix3.shape}"
            )

        else:
            shape_627fix3 = getattr(
                obj_627fix3,
                "shape",
                None,
            )

            print(
                f"    type={type(obj_627fix3).__name__}"
                f", shape={shape_627fix3}"
            )


# ------------------------------------------------------------
# 2. Print exact function signatures for known functions
# ------------------------------------------------------------

print("\n--- Exact known function signatures ---")

for name_627fix3 in [
    "apply_frozen_corruption_620gc",
    "mc_uncertainty_623cb2",
]:

    if name_627fix3 not in globals():
        print(
            f"{name_627fix3}: NOT FOUND"
        )
        continue

    obj_627fix3 = globals()[name_627fix3]

    if callable(obj_627fix3):
        try:
            print(
                f"{name_627fix3}: "
                f"{inspect.signature(obj_627fix3)}"
            )
        except Exception as exc_627fix3:
            print(
                f"{name_627fix3}: "
                f"signature unavailable "
                f"({exc_627fix3})"
            )


# ------------------------------------------------------------
# 3. Search all callable objects for likely preprocessing /
#    sequence / endpoint / inference helpers
# ------------------------------------------------------------

keywords_627fix3 = [
    "sequence",
    "window",
    "endpoint",
    "preprocess",
    "scale",
    "predict",
    "inference",
    "mc",
    "uncert",
    "dev",
    "validation",
]

callable_records_627fix3 = []

for (
    name_627fix3,
    obj_627fix3,
) in list(globals().items()):

    if not callable(obj_627fix3):
        continue

    lower_name_627fix3 = (
        str(name_627fix3).lower()
    )

    if not any(
        keyword_627fix3 in lower_name_627fix3
        for keyword_627fix3
        in keywords_627fix3
    ):
        continue

    try:
        signature_627fix3 = str(
            inspect.signature(
                obj_627fix3
            )
        )
    except Exception:
        signature_627fix3 = (
            "<signature unavailable>"
        )

    callable_records_627fix3.append(
        {
            "object_name": name_627fix3,
            "signature": signature_627fix3,
        }
    )


CALLABLE_INVENTORY_627FIX3 = (
    pd.DataFrame(
        callable_records_627fix3
    )
)


print(
    "\n--- Candidate reusable functions ---"
)

if CALLABLE_INVENTORY_627FIX3.empty:
    print("NONE FOUND")
else:
    display(
        CALLABLE_INVENTORY_627FIX3
        .sort_values(
            "object_name"
        )
        .reset_index(
            drop=True
        )
    )


# ------------------------------------------------------------
# 4. Search arrays/DataFrames that look like DEV/validation
#    sequence or endpoint objects
# ------------------------------------------------------------

data_records_627fix3 = []

for (
    name_627fix3,
    obj_627fix3,
) in list(globals().items()):

    lower_name_627fix3 = (
        str(name_627fix3).lower()
    )

    if not any(
        token_627fix3 in lower_name_627fix3
        for token_627fix3 in [
            "dev",
            "val",
            "validation",
            "sequence",
            "window",
            "endpoint",
        ]
    ):
        continue

    if isinstance(
        obj_627fix3,
        pd.DataFrame,
    ):

        data_records_627fix3.append(
            {
                "object_name": name_627fix3,
                "type": "DataFrame",
                "shape": str(
                    obj_627fix3.shape
                ),
                "columns": ", ".join(
                    map(
                        str,
                        obj_627fix3.columns[
                            :12
                        ],
                    )
                ),
            }
        )

    elif isinstance(
        obj_627fix3,
        np.ndarray,
    ):

        data_records_627fix3.append(
            {
                "object_name": name_627fix3,
                "type": "ndarray",
                "shape": str(
                    obj_627fix3.shape
                ),
                "columns": "",
            }
        )


DEV_DATA_INVENTORY_627FIX3 = (
    pd.DataFrame(
        data_records_627fix3
    )
)


print(
    "\n--- DEV / validation data-object inventory ---"
)

if DEV_DATA_INVENTORY_627FIX3.empty:
    print("NONE FOUND")
else:
    display(
        DEV_DATA_INVENTORY_627FIX3
        .sort_values(
            [
                "type",
                "object_name",
            ]
        )
        .reset_index(
            drop=True
        )
    )


# ------------------------------------------------------------
# 5. Look specifically for 20-engine and 3490-window objects
# ------------------------------------------------------------

print(
    "\n--- Structurally important DEV objects ---"
)

important_rows_627fix3 = []

for (
    name_627fix3,
    obj_627fix3,
) in list(globals().items()):

    if isinstance(
        obj_627fix3,
        pd.DataFrame,
    ):

        rows_627fix3 = len(
            obj_627fix3
        )

        unit_count_627fix3 = np.nan

        if "unit_id" in obj_627fix3.columns:
            try:
                unit_count_627fix3 = int(
                    obj_627fix3[
                        "unit_id"
                    ].nunique()
                )
            except Exception:
                pass

        if (
            rows_627fix3 in {
                3490,
                4070,
            }
            or unit_count_627fix3 == 20
        ):
            important_rows_627fix3.append(
                {
                    "object_name":
                        name_627fix3,
                    "rows":
                        rows_627fix3,
                    "columns":
                        len(
                            obj_627fix3.columns
                        ),
                    "unit_count":
                        unit_count_627fix3,
                    "first_columns":
                        ", ".join(
                            map(
                                str,
                                obj_627fix3.columns[
                                    :15
                                ],
                            )
                        ),
                }
            )

    elif isinstance(
        obj_627fix3,
        np.ndarray,
    ):

        shape_627fix3 = (
            obj_627fix3.shape
        )

        if (
            len(shape_627fix3) > 0
            and (
                shape_627fix3[0]
                in {
                    20,
                    3490,
                }
            )
        ):
            important_rows_627fix3.append(
                {
                    "object_name":
                        name_627fix3,
                    "rows":
                        shape_627fix3[0],
                    "columns":
                        (
                            shape_627fix3[1]
                            if len(
                                shape_627fix3
                            ) > 1
                            else np.nan
                        ),
                    "unit_count":
                        np.nan,
                    "first_columns":
                        "",
                }
            )


IMPORTANT_DEV_OBJECTS_627FIX3 = (
    pd.DataFrame(
        important_rows_627fix3
    )
)


if IMPORTANT_DEV_OBJECTS_627FIX3.empty:
    print("NONE FOUND")
else:
    display(
        IMPORTANT_DEV_OBJECTS_627FIX3
        .sort_values(
            "object_name"
        )
        .reset_index(
            drop=True
        )
    )


# ------------------------------------------------------------
# 6. Safety summary
# ------------------------------------------------------------

print(
    "\n"
    "============================================================\n"
    "STEP 6.27A-FIX3 SUMMARY\n"
    "============================================================"
)

print(
    "Callable candidates found :",
    len(
        CALLABLE_INVENTORY_627FIX3
    ),
)

print(
    "DEV-like data objects      :",
    len(
        DEV_DATA_INVENTORY_627FIX3
    ),
)

print(
    "Important DEV structures  :",
    len(
        IMPORTANT_DEV_OBJECTS_627FIX3
    ),
)

print(
    "\nOfficial TEST labels inspected : False"
)
print(
    "Official TEST scored           : False"
)
print(
    "Model inference executed       : False"
)
print(
    "MC-dropout executed            : False"
)
print(
    "Corruption generated           : False"
)
print(
    "Threshold selected             : False"
)
print(
    "Research decision changed      : False"
)
print(
    "Files written                  : False"
)

print(
    "\n=== END STEP 6.27A-FIX3 ==="
)

In [ ]:
# ============================================================
# Step 6.27B
# Single-scenario frozen validation uncertainty reconstruction
# CONTRACT SMOKE TEST ONLY
#
# PURPOSE
# -------
# Verify the exact existing frozen chain:
#
# validation raw data
#   -> frozen corruption
#   -> endpoint sequence reconstruction
#   -> frozen MC-dropout uncertainty
#
# IMPORTANT
# ---------
# - One corruption scenario only
# - No official TEST labels
# - No scoring
# - No threshold selection
# - No research decision change
# - No files written
# ============================================================

import numpy as np
import pandas as pd
import inspect

print(
    "\n"
    "============================================================\n"
    "STEP 6.27B — SINGLE-SCENARIO FROZEN UNCERTAINTY SMOKE TEST\n"
    "============================================================"
)


# ============================================================
# 1. Required frozen objects
# ============================================================

required_627b = [
    "val_raw_base_620gc",
    "scenario_manifests_620gc",
    "_apply_dev_corruption_627a",
    "build_endpoint_sequences_625e1",
    "run_frozen_uncertainty_625e1",
]

print("\n--- Required frozen objects ---")

missing_627b = []

for name_627b in required_627b:
    exists_627b = name_627b in globals()

    print(
        f"{name_627b:<38}: "
        f"{exists_627b}"
    )

    if not exists_627b:
        missing_627b.append(
            name_627b
        )

if missing_627b:
    raise RuntimeError(
        "Step 6.27B cannot continue. "
        "Missing required frozen objects: "
        + str(
            missing_627b
        )
    )


# ============================================================
# 2. Display exact contracts
# ============================================================

print(
    "\n--- Frozen function contracts ---"
)

print(
    "_apply_dev_corruption_627a:",
    inspect.signature(
        _apply_dev_corruption_627a
    ),
)

print(
    "build_endpoint_sequences_625e1:",
    inspect.signature(
        build_endpoint_sequences_625e1
    ),
)

print(
    "run_frozen_uncertainty_625e1:",
    inspect.signature(
        run_frozen_uncertainty_625e1
    ),
)


# ============================================================
# 3. Frozen validation-base integrity
# ============================================================

BASE_627B = (
    val_raw_base_620gc
    .copy(
        deep=True
    )
)

print(
    "\n--- Frozen validation base ---"
)

print(
    "Shape        :",
    BASE_627B.shape,
)

if not isinstance(
    BASE_627B,
    pd.DataFrame,
):
    raise RuntimeError(
        "val_raw_base_620gc is not a DataFrame."
    )

required_key_columns_627b = [
    "unit_id",
    "cycle",
]

missing_keys_627b = [
    col_627b
    for col_627b
    in required_key_columns_627b
    if col_627b
    not in BASE_627B.columns
]

if missing_keys_627b:
    raise RuntimeError(
        "Validation base is missing key columns: "
        + str(
            missing_keys_627b
        )
    )

base_engines_627b = (
    pd.to_numeric(
        BASE_627B[
            "unit_id"
        ],
        errors="raise",
    )
    .astype(int)
    .nunique()
)

duplicate_keys_627b = int(
    BASE_627B.duplicated(
        subset=[
            "unit_id",
            "cycle",
        ]
    ).sum()
)

print(
    "Engines      :",
    base_engines_627b,
)

print(
    "Duplicate keys:",
    duplicate_keys_627b,
)

if base_engines_627b != 20:
    raise RuntimeError(
        "Expected exactly 20 frozen validation engines, "
        f"found {base_engines_627b}."
    )

if duplicate_keys_627b != 0:
    raise RuntimeError(
        "Frozen validation base contains duplicate "
        "(unit_id, cycle) keys."
    )


# ============================================================
# 4. Select ONE deterministic frozen corruption scenario
#
# Prefer BIAS simply because it is straightforward for
# a first contract test.
# ============================================================

print(
    "\n--- Selecting one frozen scenario ---"
)

if not isinstance(
    scenario_manifests_620gc,
    dict,
):
    raise RuntimeError(
        "scenario_manifests_620gc is not a dict."
    )

family_627b = "bias"

if family_627b not in scenario_manifests_620gc:
    raise RuntimeError(
        "Frozen bias manifest was not found."
    )

manifest_627b = (
    scenario_manifests_620gc[
        family_627b
    ]
)

if isinstance(
    manifest_627b,
    pd.DataFrame,
):
    if len(
        manifest_627b
    ) == 0:
        raise RuntimeError(
            "Bias manifest is empty."
        )

    scenario_row_627b = (
        manifest_627b
        .iloc[0]
        .copy()
    )

elif isinstance(
    manifest_627b,
    (list, tuple),
):
    if len(
        manifest_627b
    ) == 0:
        raise RuntimeError(
            "Bias manifest is empty."
        )

    scenario_row_627b = (
        manifest_627b[0]
    )

else:
    raise RuntimeError(
        "Unsupported bias-manifest type: "
        + type(
            manifest_627b
        ).__name__
    )

print(
    "Family       :",
    family_627b,
)

print(
    "Manifest type:",
    type(
        manifest_627b
    ).__name__,
)

print(
    "Scenario type:",
    type(
        scenario_row_627b
    ).__name__,
)

if isinstance(
    scenario_row_627b,
    pd.Series,
):
    for field_627b in [
        "scenario_id",
        "sensor",
        "severity",
    ]:
        if field_627b in scenario_row_627b.index:
            print(
                f"{field_627b:<13}:",
                scenario_row_627b[
                    field_627b
                ],
            )


# ============================================================
# 5. Apply exact established frozen corruption
# ============================================================

print(
    "\n--- Applying frozen corruption ---"
)

corrupted_result_627b = (
    _apply_dev_corruption_627a(
        BASE_627B.copy(
            deep=True
        ),
        family_627b,
        scenario_row_627b,
    )
)

print(
    "Returned type:",
    type(
        corrupted_result_627b
    ).__name__,
)


# ============================================================
# 6. Recover corrupted DataFrame conservatively
# ============================================================

def _recover_dataframe_627b(
    obj_627b,
):
    """
    Recover a DataFrame without guessing between
    multiple plausible DataFrames.
    """

    if isinstance(
        obj_627b,
        pd.DataFrame,
    ):
        return obj_627b.copy(
            deep=True
        )

    candidates_627b = []

    if isinstance(
        obj_627b,
        (tuple, list),
    ):
        for item_627b in obj_627b:
            if isinstance(
                item_627b,
                pd.DataFrame,
            ):
                candidates_627b.append(
                    item_627b
                )

    elif isinstance(
        obj_627b,
        dict,
    ):
        for item_627b in obj_627b.values():
            if isinstance(
                item_627b,
                pd.DataFrame,
            ):
                candidates_627b.append(
                    item_627b
                )

    if len(
        candidates_627b
    ) == 1:
        return (
            candidates_627b[0]
            .copy(
                deep=True
            )
        )

    if len(
        candidates_627b
    ) == 0:
        raise RuntimeError(
            "Frozen corruption call did not expose "
            "a DataFrame."
        )

    raise RuntimeError(
        "Frozen corruption call exposed more than one "
        "possible DataFrame. Refusing to guess."
    )


CORRUPTED_RAW_627B = (
    _recover_dataframe_627b(
        corrupted_result_627b
    )
)

print(
    "Recovered shape:",
    CORRUPTED_RAW_627B.shape,
)

if (
    CORRUPTED_RAW_627B.shape
    != BASE_627B.shape
):
    raise RuntimeError(
        "Frozen corruption changed validation "
        "DataFrame shape."
    )

if list(
    CORRUPTED_RAW_627B.columns
) != list(
    BASE_627B.columns
):
    raise RuntimeError(
        "Frozen corruption changed validation columns."
    )

if not (
    CORRUPTED_RAW_627B[
        [
            "unit_id",
            "cycle",
        ]
    ]
    .reset_index(
        drop=True
    )
    .equals(
        BASE_627B[
            [
                "unit_id",
                "cycle",
            ]
        ]
        .reset_index(
            drop=True
        )
    )
):
    raise RuntimeError(
        "Frozen corruption changed endpoint keys."
    )


# ============================================================
# 7. Confirm corruption is actually effective
# ============================================================

numeric_cols_627b = [
    col_627b
    for col_627b
    in BASE_627B.columns
    if (
        pd.api.types.is_numeric_dtype(
            BASE_627B[
                col_627b
            ]
        )
        and col_627b
        not in [
            "unit_id",
            "cycle",
        ]
    )
]

base_numeric_627b = (
    BASE_627B[
        numeric_cols_627b
    ]
    .to_numpy(
        dtype=float
    )
)

corrupted_numeric_627b = (
    CORRUPTED_RAW_627B[
        numeric_cols_627b
    ]
    .to_numpy(
        dtype=float
    )
)

changed_mask_627b = (
    ~np.isclose(
        base_numeric_627b,
        corrupted_numeric_627b,
        rtol=0.0,
        atol=0.0,
        equal_nan=True,
    )
)

changed_numeric_cells_627b = int(
    changed_mask_627b.sum()
)

print(
    "Changed numeric cells:",
    changed_numeric_cells_627b,
)

if changed_numeric_cells_627b == 0:
    raise RuntimeError(
        "Selected frozen corruption produced no "
        "numeric change."
    )


# ============================================================
# 8. Build endpoint sequences using EXISTING frozen builder
# ============================================================

print(
    "\n--- Building corrupted endpoint sequences ---"
)

endpoint_result_627b = (
    build_endpoint_sequences_625e1(
        CORRUPTED_RAW_627B
    )
)

print(
    "Builder return type:",
    type(
        endpoint_result_627b
    ).__name__,
)

if isinstance(
    endpoint_result_627b,
    (tuple, list),
):
    print(
        "Builder container length:",
        len(
            endpoint_result_627b
        ),
    )

    for i_627b, item_627b in enumerate(
        endpoint_result_627b
    ):
        print(
            f"  component {i_627b}: "
            f"type={type(item_627b).__name__}, "
            f"shape={getattr(item_627b, 'shape', None)}"
        )


# ============================================================
# 9. Recover the endpoint sequence tensor
# ============================================================

def _recover_endpoint_tensor_627b(
    obj_627b,
):
    """
    Require exactly one plausible 3-D numeric sequence
    tensor with shape (*, 30, 17).
    """

    candidates_627b = []

    def inspect_item_627b(
        item_627b
    ):
        if isinstance(
            item_627b,
            np.ndarray,
        ):
            if (
                item_627b.ndim == 3
                and item_627b.shape[1] == 30
                and item_627b.shape[2] == 17
            ):
                candidates_627b.append(
                    item_627b
                )

    inspect_item_627b(
        obj_627b
    )

    if isinstance(
        obj_627b,
        (tuple, list),
    ):
        for item_627b in obj_627b:
            inspect_item_627b(
                item_627b
            )

    elif isinstance(
        obj_627b,
        dict,
    ):
        for item_627b in obj_627b.values():
            inspect_item_627b(
                item_627b
            )

    if len(
        candidates_627b
    ) == 0:
        raise RuntimeError(
            "No plausible (n,30,17) endpoint-sequence "
            "array found."
        )

    if len(
        candidates_627b
    ) > 1:
        raise RuntimeError(
            "More than one plausible endpoint sequence "
            "array found. Refusing to guess."
        )

    return np.asarray(
        candidates_627b[0]
    )


X_ENDPOINT_627B = (
    _recover_endpoint_tensor_627b(
        endpoint_result_627b
    )
)

print(
    "Recovered endpoint tensor:",
    X_ENDPOINT_627B.shape,
)

if X_ENDPOINT_627B.shape != (
    20,
    30,
    17,
):
    raise RuntimeError(
        "Expected exactly one endpoint sequence "
        "for each of 20 validation engines: "
        "(20,30,17). Found "
        + str(
            X_ENDPOINT_627B.shape
        )
    )

if not np.isfinite(
    X_ENDPOINT_627B
).all():
    raise RuntimeError(
        "Endpoint sequence tensor contains "
        "non-finite values."
    )


# ============================================================
# 10. Run exact frozen uncertainty wrapper
# ============================================================

print(
    "\n--- Running frozen endpoint uncertainty ---"
)

uncertainty_result_627b = (
    run_frozen_uncertainty_625e1(
        X_ENDPOINT_627B
    )
)

print(
    "Uncertainty return type:",
    type(
        uncertainty_result_627b
    ).__name__,
)

if isinstance(
    uncertainty_result_627b,
    (tuple, list),
):
    print(
        "Uncertainty container length:",
        len(
            uncertainty_result_627b
        ),
    )

    for i_627b, item_627b in enumerate(
        uncertainty_result_627b
    ):
        print(
            f"  component {i_627b}: "
            f"type={type(item_627b).__name__}, "
            f"shape={getattr(item_627b, 'shape', None)}"
        )


# ============================================================
# 11. Recover exactly one 20-element uncertainty vector
# ============================================================

def _recover_uncertainty_vector_627b(
    obj_627b,
):
    """
    Find exactly one plausible 20-value numeric vector.

    If a DataFrame is returned, prefer columns whose
    names explicitly identify MC uncertainty.
    """

    named_candidates_627b = []
    generic_candidates_627b = []

    def add_array_627b(
        arr_627b,
        label_627b,
        named_627b=False,
    ):
        arr_627b = np.asarray(
            arr_627b
        )

        arr_627b = np.squeeze(
            arr_627b
        )

        if (
            arr_627b.ndim == 1
            and len(
                arr_627b
            ) == 20
            and np.issubdtype(
                arr_627b.dtype,
                np.number,
            )
        ):
            record_627b = (
                label_627b,
                arr_627b.astype(
                    float
                ),
            )

            if named_627b:
                named_candidates_627b.append(
                    record_627b
                )
            else:
                generic_candidates_627b.append(
                    record_627b
                )

    if isinstance(
        obj_627b,
        pd.DataFrame,
    ):
        for col_627b in obj_627b.columns:
            col_lower_627b = str(
                col_627b
            ).lower()

            if (
                "uncert" in col_lower_627b
                or "mc_std" in col_lower_627b
                or "dropout_std" in col_lower_627b
            ):
                add_array_627b(
                    pd.to_numeric(
                        obj_627b[
                            col_627b
                        ],
                        errors="coerce",
                    ).to_numpy(),
                    str(
                        col_627b
                    ),
                    named_627b=True,
                )

    elif isinstance(
        obj_627b,
        dict,
    ):
        for key_627b, value_627b in obj_627b.items():

            key_lower_627b = str(
                key_627b
            ).lower()

            add_array_627b(
                value_627b,
                str(
                    key_627b
                ),
                named_627b=(
                    "uncert"
                    in key_lower_627b
                    or "mc_std"
                    in key_lower_627b
                    or "dropout_std"
                    in key_lower_627b
                ),
            )

    elif isinstance(
        obj_627b,
        (tuple, list),
    ):
        for i_627b, value_627b in enumerate(
            obj_627b
        ):
            add_array_627b(
                value_627b,
                f"component_{i_627b}",
                named_627b=False,
            )

    else:
        add_array_627b(
            obj_627b,
            "direct_return",
            named_627b=False,
        )

    candidates_627b = (
        named_candidates_627b
        if len(
            named_candidates_627b
        ) > 0
        else generic_candidates_627b
    )

    if len(
        candidates_627b
    ) == 0:
        raise RuntimeError(
            "Could not recover any 20-value "
            "uncertainty vector."
        )

    if len(
        candidates_627b
    ) > 1:
        labels_627b = [
            item_627b[0]
            for item_627b
            in candidates_627b
        ]

        raise RuntimeError(
            "More than one plausible 20-value "
            "uncertainty vector found: "
            + str(
                labels_627b
            )
            + ". Refusing to guess."
        )

    return candidates_627b[0]


(
    uncertainty_source_627b,
    uncertainty_values_627b,
) = _recover_uncertainty_vector_627b(
    uncertainty_result_627b
)

print(
    "Recovered uncertainty source:",
    uncertainty_source_627b,
)

print(
    "Recovered uncertainty shape :",
    uncertainty_values_627b.shape,
)

print(
    "Uncertainty min             :",
    float(
        np.min(
            uncertainty_values_627b
        )
    ),
)

print(
    "Uncertainty mean            :",
    float(
        np.mean(
            uncertainty_values_627b
        )
    ),
)

print(
    "Uncertainty max             :",
    float(
        np.max(
            uncertainty_values_627b
        )
    ),
)

if not np.isfinite(
    uncertainty_values_627b
).all():
    raise RuntimeError(
        "Recovered uncertainty contains "
        "non-finite values."
    )


# ============================================================
# 12. Preserve smoke-test objects only in memory
# ============================================================

SMOKE_CORRUPTED_RAW_627B = (
    CORRUPTED_RAW_627B.copy(
        deep=True
    )
)

SMOKE_ENDPOINT_SEQUENCES_627B = (
    X_ENDPOINT_627B.copy()
)

SMOKE_UNCERTAINTY_627B = (
    uncertainty_values_627b.copy()
)


# ============================================================
# 13. Final compact verdict
# ============================================================

print(
    "\n"
    "============================================================\n"
    "STEP 6.27B SMOKE-TEST SUMMARY\n"
    "============================================================"
)

print(
    "Frozen validation engines       :",
    base_engines_627b,
)

print(
    "Scenario family                 :",
    family_627b,
)

print(
    "Corruption effective             :",
    changed_numeric_cells_627b > 0,
)

print(
    "Endpoint sequences reconstructed :",
    X_ENDPOINT_627B.shape,
)

print(
    "Exactly 20 endpoint sequences    :",
    X_ENDPOINT_627B.shape[0] == 20,
)

print(
    "Frozen uncertainty values        :",
    len(
        uncertainty_values_627b
    ),
)

print(
    "All uncertainty values finite    :",
    bool(
        np.isfinite(
            uncertainty_values_627b
        ).all()
    ),
)

print(
    "\nOfficial TEST labels inspected : False"
)

print(
    "Official TEST scored           : False"
)

print(
    "Threshold selected             : False"
)

print(
    "Research decision changed      : False"
)

print(
    "Files written                  : False"
)

print(
    "\nNo frozen research decision was modified."
)

print(
    "\n=== END STEP 6.27B ==="
)

In [ ]:
# ============================================================
# Step 6.27B-R
# Exact validation preprocessing reconstruction audit
#
# PURPOSE
# -------
# Prove that the frozen validation sequence preprocessing can
# be reconstructed exactly from:
#
#   val_raw_base_620gc
#   PRIMARY_INPUT_COLUMNS
#   scaler
#
# by comparing against the already-frozen X_val_clean.
#
# NO:
# - official TEST labels
# - TEST scoring
# - model inference
# - MC-dropout
# - threshold selection
# - files written
# ============================================================

import numpy as np
import pandas as pd

print(
    "\n"
    "============================================================\n"
    "STEP 6.27B-R — EXACT VALIDATION PREPROCESSING AUDIT\n"
    "============================================================"
)


# ============================================================
# 1. Required frozen objects
# ============================================================

required_627br = [
    "val_raw_base_620gc",
    "PRIMARY_INPUT_COLUMNS",
    "scaler",
    "X_val_clean",
]

print("\n--- Required frozen objects ---")

missing_627br = []

for name_627br in required_627br:

    exists_627br = (
        name_627br in globals()
    )

    print(
        f"{name_627br:<30}: "
        f"{exists_627br}"
    )

    if not exists_627br:
        missing_627br.append(
            name_627br
        )


if missing_627br:

    raise RuntimeError(
        "Step 6.27B-R cannot continue. "
        "Missing required objects: "
        + str(
            missing_627br
        )
    )


# ============================================================
# 2. Freeze local copies
# ============================================================

RAW_627BR = (
    val_raw_base_620gc
    .copy(
        deep=True
    )
)

PRIMARY_COLUMNS_627BR = list(
    PRIMARY_INPUT_COLUMNS
)

X_REFERENCE_627BR = np.asarray(
    X_val_clean
)


print(
    "\n--- Existing frozen validation state ---"
)

print(
    "Raw validation shape :",
    RAW_627BR.shape
)

print(
    "Primary inputs       :",
    len(
        PRIMARY_COLUMNS_627BR
    )
)

print(
    "Frozen X_val_clean   :",
    X_REFERENCE_627BR.shape
)


# ============================================================
# 3. Basic contract checks
# ============================================================

if not isinstance(
    RAW_627BR,
    pd.DataFrame
):
    raise RuntimeError(
        "val_raw_base_620gc is not a DataFrame."
    )


if "unit_id" not in RAW_627BR.columns:

    raise RuntimeError(
        "unit_id is missing from validation raw data."
    )


if "cycle" not in RAW_627BR.columns:

    raise RuntimeError(
        "cycle is missing from validation raw data."
    )


missing_primary_627br = [
    col_627br
    for col_627br
    in PRIMARY_COLUMNS_627BR
    if col_627br
    not in RAW_627BR.columns
]


if missing_primary_627br:

    raise RuntimeError(
        "Frozen validation data is missing "
        "primary model inputs: "
        + str(
            missing_primary_627br
        )
    )


if len(
    PRIMARY_COLUMNS_627BR
) != 17:

    raise RuntimeError(
        "Expected 17 frozen primary inputs, found "
        + str(
            len(
                PRIMARY_COLUMNS_627BR
            )
        )
    )


# ============================================================
# 4. Establish deterministic validation ordering
# ============================================================

ORDERED_RAW_627BR = (
    RAW_627BR
    .sort_values(
        [
            "unit_id",
            "cycle",
        ],
        kind="stable"
    )
    .reset_index(
        drop=True
    )
)


engine_count_627br = int(
    ORDERED_RAW_627BR[
        "unit_id"
    ].nunique()
)


duplicate_keys_627br = int(
    ORDERED_RAW_627BR
    .duplicated(
        subset=[
            "unit_id",
            "cycle",
        ]
    )
    .sum()
)


print(
    "\n--- Validation structure ---"
)

print(
    "Engines        :",
    engine_count_627br
)

print(
    "Duplicate keys :",
    duplicate_keys_627br
)


if engine_count_627br != 20:

    raise RuntimeError(
        "Expected 20 frozen validation engines, "
        f"found {engine_count_627br}."
    )


if duplicate_keys_627br != 0:

    raise RuntimeError(
        "Duplicate (unit_id, cycle) keys found."
    )


# ============================================================
# 5. Confirm cycles are contiguous within every engine
# ============================================================

non_contiguous_627br = []

for unit_627br, group_627br in (
    ORDERED_RAW_627BR.groupby(
        "unit_id",
        sort=True
    )
):

    cycles_627br = (
        pd.to_numeric(
            group_627br[
                "cycle"
            ],
            errors="raise"
        )
        .to_numpy(
            dtype=int
        )
    )

    if len(
        cycles_627br
    ) > 1:

        if not np.all(
            np.diff(
                cycles_627br
            ) == 1
        ):

            non_contiguous_627br.append(
                int(
                    unit_627br
                )
            )


print(
    "Non-contiguous engines:",
    len(
        non_contiguous_627br
    )
)


if non_contiguous_627br:

    raise RuntimeError(
        "Validation cycles are not contiguous for engines: "
        + str(
            non_contiguous_627br
        )
    )


# ============================================================
# 6. Apply the already-frozen scaler
#
# IMPORTANT:
# use exact PRIMARY_INPUT_COLUMNS ordering.
# ============================================================

PRIMARY_FRAME_627BR = (
    ORDERED_RAW_627BR[
        PRIMARY_COLUMNS_627BR
    ]
    .copy()
)


SCALED_ROWS_627BR = np.asarray(
    scaler.transform(
        PRIMARY_FRAME_627BR
    ),
    dtype=float
)


print(
    "\n--- Frozen scaling reconstruction ---"
)

print(
    "Scaled row matrix:",
    SCALED_ROWS_627BR.shape
)


if SCALED_ROWS_627BR.shape != (
    len(
        ORDERED_RAW_627BR
    ),
    17,
):

    raise RuntimeError(
        "Unexpected scaled validation shape: "
        + str(
            SCALED_ROWS_627BR.shape
        )
    )


if not np.isfinite(
    SCALED_ROWS_627BR
).all():

    raise RuntimeError(
        "Scaled validation values contain "
        "non-finite values."
    )


# ============================================================
# 7. Reconstruct every 30-cycle validation window
# ============================================================

WINDOW_LENGTH_627BR = 30

all_windows_627br = []

window_meta_627br = []

endpoint_indices_627br = []

running_window_index_627br = 0


for unit_627br, group_627br in (
    ORDERED_RAW_627BR
    .groupby(
        "unit_id",
        sort=True
    )
):

    row_indices_627br = (
        group_627br.index
        .to_numpy(
            dtype=int
        )
    )

    unit_scaled_627br = (
        SCALED_ROWS_627BR[
            row_indices_627br
        ]
    )

    unit_cycles_627br = (
        group_627br[
            "cycle"
        ]
        .to_numpy(
            dtype=int
        )
    )

    n_rows_627br = len(
        group_627br
    )

    n_windows_627br = (
        n_rows_627br
        - WINDOW_LENGTH_627BR
        + 1
    )


    if n_windows_627br <= 0:

        raise RuntimeError(
            f"Engine {unit_627br} contains fewer than "
            f"{WINDOW_LENGTH_627BR} cycles."
        )


    for start_627br in range(
        n_windows_627br
    ):

        stop_627br = (
            start_627br
            + WINDOW_LENGTH_627BR
        )

        sequence_627br = (
            unit_scaled_627br[
                start_627br:
                stop_627br
            ]
        )

        all_windows_627br.append(
            sequence_627br
        )

        window_meta_627br.append(
            {
                "unit_id":
                    int(
                        unit_627br
                    ),

                "cycle":
                    int(
                        unit_cycles_627br[
                            stop_627br - 1
                        ]
                    ),
            }
        )

        running_window_index_627br += 1


    endpoint_indices_627br.append(
        running_window_index_627br - 1
    )


X_REBUILT_627BR = np.stack(
    all_windows_627br,
    axis=0
)


META_REBUILT_627BR = pd.DataFrame(
    window_meta_627br
)


ENDPOINT_INDICES_627BR = np.asarray(
    endpoint_indices_627br,
    dtype=int
)


print(
    "\n--- Reconstructed rolling windows ---"
)

print(
    "Rebuilt X shape      :",
    X_REBUILT_627BR.shape
)

print(
    "Metadata shape       :",
    META_REBUILT_627BR.shape
)

print(
    "Endpoint count       :",
    len(
        ENDPOINT_INDICES_627BR
    )
)


# ============================================================
# 8. Expected window count from raw trajectories
# ============================================================

expected_windows_627br = int(

    ORDERED_RAW_627BR
    .groupby(
        "unit_id",
        sort=True
    )
    .size()
    .sub(
        WINDOW_LENGTH_627BR - 1
    )
    .sum()
)


print(
    "Expected windows     :",
    expected_windows_627br
)


if expected_windows_627br != 3490:

    raise RuntimeError(
        "Unexpected expected validation-window count: "
        + str(
            expected_windows_627br
        )
    )


if X_REBUILT_627BR.shape != (
    3490,
    30,
    17,
):

    raise RuntimeError(
        "Reconstructed validation tensor has "
        "unexpected shape: "
        + str(
            X_REBUILT_627BR.shape
        )
    )


# ============================================================
# 9. Compare directly with frozen X_val_clean
# ============================================================

if X_REFERENCE_627BR.shape != (
    3490,
    30,
    17,
):

    raise RuntimeError(
        "Frozen X_val_clean has unexpected shape: "
        + str(
            X_REFERENCE_627BR.shape
        )
    )


abs_diff_627br = np.abs(
    X_REBUILT_627BR
    - X_REFERENCE_627BR
)


max_abs_error_627br = float(
    np.max(
        abs_diff_627br
    )
)


mean_abs_error_627br = float(
    np.mean(
        abs_diff_627br
    )
)


exact_equal_627br = bool(
    np.array_equal(
        X_REBUILT_627BR,
        X_REFERENCE_627BR
    )
)


close_1e12_627br = bool(
    np.allclose(
        X_REBUILT_627BR,
        X_REFERENCE_627BR,
        rtol=0.0,
        atol=1e-12
    )
)


close_1e10_627br = bool(
    np.allclose(
        X_REBUILT_627BR,
        X_REFERENCE_627BR,
        rtol=0.0,
        atol=1e-10
    )
)


print(
    "\n--- Exact frozen-preprocessing comparison ---"
)

print(
    "Array exactly equal :",
    exact_equal_627br
)

print(
    "Close at 1e-12      :",
    close_1e12_627br
)

print(
    "Close at 1e-10      :",
    close_1e10_627br
)

print(
    "Maximum abs error   :",
    max_abs_error_627br
)

print(
    "Mean abs error      :",
    mean_abs_error_627br
)


# ============================================================
# 10. Endpoint extraction
# ============================================================

X_ENDPOINTS_REBUILT_627BR = (
    X_REBUILT_627BR[
        ENDPOINT_INDICES_627BR
    ]
)


endpoint_meta_627br = (
    META_REBUILT_627BR
    .iloc[
        ENDPOINT_INDICES_627BR
    ]
    .reset_index(
        drop=True
    )
)


print(
    "\n--- Reconstructed validation endpoints ---"
)

print(
    "Endpoint tensor shape:",
    X_ENDPOINTS_REBUILT_627BR.shape
)

print(
    "Endpoint metadata    :",
    endpoint_meta_627br.shape
)

print(
    "Unique endpoint units:",
    endpoint_meta_627br[
        "unit_id"
    ].nunique()
)


if X_ENDPOINTS_REBUILT_627BR.shape != (
    20,
    30,
    17,
):

    raise RuntimeError(
        "Expected 20 endpoint sequences; found "
        + str(
            X_ENDPOINTS_REBUILT_627BR.shape
        )
    )


if endpoint_meta_627br[
    "unit_id"
].nunique() != 20:

    raise RuntimeError(
        "Endpoint metadata does not contain "
        "exactly 20 engines."
    )


# ============================================================
# 11. Do NOT continue if frozen preprocessing is not reproduced
# ============================================================

PREPROCESSING_REPRODUCED_627BR = bool(
    close_1e10_627br
)


if not PREPROCESSING_REPRODUCED_627BR:

    raise RuntimeError(
        "Frozen validation preprocessing was NOT "
        "reproduced within 1e-10. "
        "Do not continue to corrupted uncertainty."
    )


# ============================================================
# 12. Preserve proven reconstruction in memory
# ============================================================

VALIDATION_X_REBUILT_627BR = (
    X_REBUILT_627BR.copy()
)

VALIDATION_META_REBUILT_627BR = (
    META_REBUILT_627BR.copy(
        deep=True
    )
)

VALIDATION_ENDPOINT_X_627BR = (
    X_ENDPOINTS_REBUILT_627BR.copy()
)

VALIDATION_ENDPOINT_META_627BR = (
    endpoint_meta_627br.copy(
        deep=True
    )
)


# ============================================================
# 13. Final compact summary
# ============================================================

print(
    "\n"
    "============================================================\n"
    "STEP 6.27B-R FINAL SUMMARY\n"
    "============================================================"
)

print(
    "Validation raw rows              :",
    len(
        ORDERED_RAW_627BR
    )
)

print(
    "Validation engines               :",
    engine_count_627br
)

print(
    "Frozen primary features          :",
    len(
        PRIMARY_COLUMNS_627BR
    )
)

print(
    "Window length                    :",
    WINDOW_LENGTH_627BR
)

print(
    "Expected validation windows      :",
    expected_windows_627br
)

print(
    "Reconstructed validation windows :",
    X_REBUILT_627BR.shape[0]
)

print(
    "Frozen X_val_clean shape         :",
    X_REFERENCE_627BR.shape
)

print(
    "Maximum reconstruction error     :",
    max_abs_error_627br
)

print(
    "Mean reconstruction error        :",
    mean_abs_error_627br
)

print(
    "Exact equality                   :",
    exact_equal_627br
)

print(
    "Equivalent within 1e-10          :",
    close_1e10_627br
)

print(
    "Endpoint tensor                  :",
    X_ENDPOINTS_REBUILT_627BR.shape
)

print(
    "Frozen preprocessing reproduced  :",
    PREPROCESSING_REPRODUCED_627BR
)

print(
    "\nOfficial TEST labels inspected : False"
)

print(
    "Official TEST scored           : False"
)

print(
    "Model inference executed       : False"
)

print(
    "MC-dropout executed            : False"
)

print(
    "Threshold selected             : False"
)

print(
    "Research decision changed      : False"
)

print(
    "Files written                  : False"
)

print(
    "\n=== END STEP 6.27B-R ==="
)

In [ ]:
# ============================================================
# Step 6.27B-R1
# Diagnose validation reconstruction numerical discrepancy
# ============================================================

import numpy as np

print(
    "\n"
    "============================================================\n"
    "STEP 6.27B-R1 — NUMERICAL RECONSTRUCTION DIAGNOSTIC\n"
    "============================================================"
)

required_627br1 = [
    "X_REBUILT_627BR",
    "X_REFERENCE_627BR",
]

missing_627br1 = [
    name
    for name in required_627br1
    if name not in globals()
]

if missing_627br1:
    raise RuntimeError(
        "Required reconstruction objects are missing: "
        + str(missing_627br1)
        + ". Re-run Step 6.27B-R only up to the comparison section."
    )


rebuilt_627br1 = np.asarray(
    X_REBUILT_627BR
)

reference_627br1 = np.asarray(
    X_REFERENCE_627BR
)


print("\n--- Array contract ---")

print(
    "Rebuilt shape     :",
    rebuilt_627br1.shape
)

print(
    "Reference shape   :",
    reference_627br1.shape
)

print(
    "Rebuilt dtype     :",
    rebuilt_627br1.dtype
)

print(
    "Reference dtype   :",
    reference_627br1.dtype
)


if rebuilt_627br1.shape != reference_627br1.shape:
    raise RuntimeError(
        "Shapes differ. This is not a floating-point issue."
    )


# ------------------------------------------------------------
# Raw difference
# ------------------------------------------------------------

diff_627br1 = np.abs(
    rebuilt_627br1.astype(np.float64)
    -
    reference_627br1.astype(np.float64)
)

max_diff_627br1 = float(
    np.max(diff_627br1)
)

mean_diff_627br1 = float(
    np.mean(diff_627br1)
)

median_diff_627br1 = float(
    np.median(diff_627br1)
)

nonzero_count_627br1 = int(
    np.count_nonzero(diff_627br1)
)

total_values_627br1 = int(
    diff_627br1.size
)


print("\n--- Raw numerical difference ---")

print(
    "Maximum absolute difference :",
    max_diff_627br1
)

print(
    "Mean absolute difference    :",
    mean_diff_627br1
)

print(
    "Median absolute difference  :",
    median_diff_627br1
)

print(
    "Different elements          :",
    f"{nonzero_count_627br1} / {total_values_627br1}"
)


# ------------------------------------------------------------
# Tolerance ladder
# ------------------------------------------------------------

print("\n--- Absolute-tolerance ladder ---")

for atol_627br1 in [
    1e-12,
    1e-10,
    1e-9,
    1e-8,
    1e-7,
    5e-7,
    1e-6,
    1e-5,
]:

    passed_627br1 = bool(
        np.allclose(
            rebuilt_627br1,
            reference_627br1,
            rtol=0.0,
            atol=atol_627br1
        )
    )

    print(
        f"atol={atol_627br1:<8g} : "
        f"{passed_627br1}"
    )


# ------------------------------------------------------------
# Compare after matching float32 representation
# ------------------------------------------------------------

rebuilt_f32_627br1 = (
    rebuilt_627br1
    .astype(np.float32)
)

reference_f32_627br1 = (
    reference_627br1
    .astype(np.float32)
)

f32_exact_627br1 = bool(
    np.array_equal(
        rebuilt_f32_627br1,
        reference_f32_627br1
    )
)

f32_diff_627br1 = np.abs(
    rebuilt_f32_627br1.astype(np.float64)
    -
    reference_f32_627br1.astype(np.float64)
)

f32_max_diff_627br1 = float(
    np.max(f32_diff_627br1)
)


print(
    "\n--- Float32-normalized comparison ---"
)

print(
    "Exact after float32 cast     :",
    f32_exact_627br1
)

print(
    "Max difference after cast    :",
    f32_max_diff_627br1
)


# ------------------------------------------------------------
# Location of largest mismatch
# ------------------------------------------------------------

max_index_627br1 = np.unravel_index(
    np.argmax(diff_627br1),
    diff_627br1.shape
)

rebuilt_value_627br1 = float(
    rebuilt_627br1[
        max_index_627br1
    ]
)

reference_value_627br1 = float(
    reference_627br1[
        max_index_627br1
    ]
)


print(
    "\n--- Largest mismatch ---"
)

print(
    "Index           :",
    max_index_627br1
)

print(
    "Rebuilt value   :",
    rebuilt_value_627br1
)

print(
    "Reference value :",
    reference_value_627br1
)

print(
    "Absolute error  :",
    max_diff_627br1
)


# ------------------------------------------------------------
# Structural slice checks
# ------------------------------------------------------------

first_window_diff_627br1 = float(
    np.max(
        np.abs(
            rebuilt_627br1[0].astype(np.float64)
            -
            reference_627br1[0].astype(np.float64)
        )
    )
)

last_window_diff_627br1 = float(
    np.max(
        np.abs(
            rebuilt_627br1[-1].astype(np.float64)
            -
            reference_627br1[-1].astype(np.float64)
        )
    )
)


print(
    "\n--- Boundary-window comparison ---"
)

print(
    "First window max difference :",
    first_window_diff_627br1
)

print(
    "Last window max difference  :",
    last_window_diff_627br1
)


# ------------------------------------------------------------
# Interpretation guard
# ------------------------------------------------------------

precision_only_627br1 = bool(
    max_diff_627br1 <= 1e-6
)

strong_precision_match_627br1 = bool(
    max_diff_627br1 <= 5e-7
)


print(
    "\n"
    "============================================================\n"
    "STEP 6.27B-R1 DIAGNOSTIC SUMMARY\n"
    "============================================================"
)

print(
    "Same shape                     :",
    rebuilt_627br1.shape
    ==
    reference_627br1.shape
)

print(
    "Rebuilt dtype                  :",
    rebuilt_627br1.dtype
)

print(
    "Frozen-reference dtype         :",
    reference_627br1.dtype
)

print(
    "Maximum absolute difference    :",
    max_diff_627br1
)

print(
    "Equivalent within 5e-7         :",
    strong_precision_match_627br1
)

print(
    "Equivalent within 1e-6         :",
    precision_only_627br1
)

print(
    "Exact after float32 conversion :",
    f32_exact_627br1
)

print(
    "\nNo research decision changed."
)

print(
    "No model inference executed."
)

print(
    "No MC-dropout executed."
)

print(
    "No official TEST labels used."
)

print(
    "No files written."
)

print(
    "\n=== END STEP 6.27B-R1 ==="
)

In [ ]:
# ============================================================
# Step 6.27B-R2
# Final closure of frozen validation preprocessing equivalence
# ============================================================

import numpy as np

print(
    "\n"
    "============================================================\n"
    "STEP 6.27B-R2 — FINAL VALIDATION PREPROCESSING CLOSURE\n"
    "============================================================"
)

# ------------------------------------------------------------
# 1. Required diagnostic objects
# ------------------------------------------------------------

required_627br2 = [
    "X_REBUILT_627BR",
    "X_REFERENCE_627BR",
]

missing_627br2 = [
    name
    for name in required_627br2
    if name not in globals()
]

if missing_627br2:
    raise RuntimeError(
        "Required Step 6.27B-R reconstruction objects are missing: "
        + str(missing_627br2)
    )


rebuilt_627br2 = np.asarray(
    X_REBUILT_627BR
)

reference_627br2 = np.asarray(
    X_REFERENCE_627BR
)


# ------------------------------------------------------------
# 2. Structural equivalence
# ------------------------------------------------------------

shape_exact_627br2 = bool(
    rebuilt_627br2.shape
    ==
    reference_627br2.shape
)

if not shape_exact_627br2:
    raise RuntimeError(
        "Validation reconstruction shape differs from "
        "the frozen validation reference."
    )


print("\n--- Structural contract ---")

print(
    "Rebuilt shape             :",
    rebuilt_627br2.shape
)

print(
    "Frozen reference shape    :",
    reference_627br2.shape
)

print(
    "Shape exact               :",
    shape_exact_627br2
)

print(
    "Rebuilt dtype             :",
    rebuilt_627br2.dtype
)

print(
    "Frozen reference dtype    :",
    reference_627br2.dtype
)


# ------------------------------------------------------------
# 3. Finite-value contract
# ------------------------------------------------------------

rebuilt_finite_627br2 = bool(
    np.isfinite(
        rebuilt_627br2
    ).all()
)

reference_finite_627br2 = bool(
    np.isfinite(
        reference_627br2
    ).all()
)

if not rebuilt_finite_627br2:
    raise RuntimeError(
        "Reconstructed validation tensor contains "
        "non-finite values."
    )

if not reference_finite_627br2:
    raise RuntimeError(
        "Frozen validation reference contains "
        "non-finite values."
    )


# ------------------------------------------------------------
# 4. High-precision numerical difference
# ------------------------------------------------------------

difference_627br2 = np.abs(
    rebuilt_627br2.astype(np.float64)
    -
    reference_627br2.astype(np.float64)
)

max_abs_diff_627br2 = float(
    np.max(
        difference_627br2
    )
)

mean_abs_diff_627br2 = float(
    np.mean(
        difference_627br2
    )
)

median_abs_diff_627br2 = float(
    np.median(
        difference_627br2
    )
)


# ------------------------------------------------------------
# 5. Stored-dtype comparison
#
# The frozen reference is the authoritative stored tensor.
# Therefore the reconstruction is also compared after casting
# to the frozen reference dtype.
# ------------------------------------------------------------

rebuilt_reference_dtype_627br2 = (
    rebuilt_627br2.astype(
        reference_627br2.dtype
    )
)

exact_at_stored_dtype_627br2 = bool(
    np.array_equal(
        rebuilt_reference_dtype_627br2,
        reference_627br2
    )
)

stored_dtype_diff_627br2 = np.abs(
    rebuilt_reference_dtype_627br2.astype(np.float64)
    -
    reference_627br2.astype(np.float64)
)

stored_dtype_max_diff_627br2 = float(
    np.max(
        stored_dtype_diff_627br2
    )
)


# ------------------------------------------------------------
# 6. Dtype-aware numerical tolerance
#
# Empirical diagnostic:
# observed maximum reconstruction difference ≈ 2.21e-7
#
# Frozen reference:
# float32
#
# 5e-7 therefore safely distinguishes this known numerical
# representation effect from any material preprocessing change.
# ------------------------------------------------------------

VALIDATION_PREPROCESSING_ATOL_627BR2 = 5e-7

numerically_equivalent_627br2 = bool(
    np.allclose(
        rebuilt_627br2,
        reference_627br2,
        rtol=0.0,
        atol=VALIDATION_PREPROCESSING_ATOL_627BR2
    )
)

if not numerically_equivalent_627br2:
    raise RuntimeError(
        "Validation reconstruction exceeds the "
        "verified dtype-aware tolerance. "
        f"Maximum difference = {max_abs_diff_627br2}; "
        f"allowed tolerance = "
        f"{VALIDATION_PREPROCESSING_ATOL_627BR2}."
    )


# ------------------------------------------------------------
# 7. Boundary-window verification
# ------------------------------------------------------------

first_window_max_diff_627br2 = float(
    np.max(
        np.abs(
            rebuilt_627br2[0].astype(np.float64)
            -
            reference_627br2[0].astype(np.float64)
        )
    )
)

last_window_max_diff_627br2 = float(
    np.max(
        np.abs(
            rebuilt_627br2[-1].astype(np.float64)
            -
            reference_627br2[-1].astype(np.float64)
        )
    )
)

boundary_windows_valid_627br2 = bool(
    first_window_max_diff_627br2
    <= VALIDATION_PREPROCESSING_ATOL_627BR2
    and
    last_window_max_diff_627br2
    <= VALIDATION_PREPROCESSING_ATOL_627BR2
)

if not boundary_windows_valid_627br2:
    raise RuntimeError(
        "Boundary validation windows do not reproduce "
        "the frozen preprocessing state."
    )


# ------------------------------------------------------------
# 8. Freeze proven reconstruction object IN MEMORY
#
# No disk files are written here.
# ------------------------------------------------------------

VALIDATION_X_REBUILT_627BR2 = (
    rebuilt_627br2
    .astype(
        reference_627br2.dtype,
        copy=True
    )
)

VALIDATION_PREPROCESSING_TOLERANCE_627BR2 = float(
    VALIDATION_PREPROCESSING_ATOL_627BR2
)

VALIDATION_PREPROCESSING_REPRODUCED_627BR2 = True

VALIDATION_PREPROCESSING_EVIDENCE_627BR2 = {
    "shape": tuple(
        reference_627br2.shape
    ),
    "reconstructed_dtype": str(
        rebuilt_627br2.dtype
    ),
    "reference_dtype": str(
        reference_627br2.dtype
    ),
    "max_abs_difference": float(
        max_abs_diff_627br2
    ),
    "mean_abs_difference": float(
        mean_abs_diff_627br2
    ),
    "median_abs_difference": float(
        median_abs_diff_627br2
    ),
    "verification_atol": float(
        VALIDATION_PREPROCESSING_ATOL_627BR2
    ),
    "exact_after_reference_dtype_cast": bool(
        exact_at_stored_dtype_627br2
    ),
    "stored_dtype_max_difference": float(
        stored_dtype_max_diff_627br2
    ),
    "first_window_max_difference": float(
        first_window_max_diff_627br2
    ),
    "last_window_max_difference": float(
        last_window_max_diff_627br2
    ),
    "structurally_identical": True,
    "numerically_equivalent": True,
}


# ------------------------------------------------------------
# 9. Final concise evidence
# ------------------------------------------------------------

print(
    "\n--- Numerical equivalence evidence ---"
)

print(
    "Rebuilt values finite          :",
    rebuilt_finite_627br2
)

print(
    "Reference values finite        :",
    reference_finite_627br2
)

print(
    "Maximum absolute difference    :",
    max_abs_diff_627br2
)

print(
    "Mean absolute difference       :",
    mean_abs_diff_627br2
)

print(
    "Median absolute difference     :",
    median_abs_diff_627br2
)

print(
    "Verified absolute tolerance    :",
    VALIDATION_PREPROCESSING_ATOL_627BR2
)

print(
    "Equivalent at tolerance        :",
    numerically_equivalent_627br2
)

print(
    "Exact after reference-dtype cast:",
    exact_at_stored_dtype_627br2
)

print(
    "Stored-dtype max difference    :",
    stored_dtype_max_diff_627br2
)

print(
    "First-window max difference    :",
    first_window_max_diff_627br2
)

print(
    "Last-window max difference     :",
    last_window_max_diff_627br2
)


print(
    "\n"
    "============================================================\n"
    "STEP 6.27B-R2 FINAL VERDICT\n"
    "============================================================"
)

print(
    "Validation structure reproduced      : True"
)

print(
    "Validation preprocessing reproduced  :",
    VALIDATION_PREPROCESSING_REPRODUCED_627BR2
)

print(
    "Frozen reference precision           :",
    reference_627br2.dtype
)

print(
    "Verified reconstruction tolerance    :",
    VALIDATION_PREPROCESSING_ATOL_627BR2
)

print(
    "Proven validation tensor preserved   : True"
)

print(
    "\nOfficial TEST labels inspected        : False"
)

print(
    "Official TEST scored                  : False"
)

print(
    "Model inference executed              : False"
)

print(
    "MC-dropout executed                   : False"
)

print(
    "Corruptions regenerated               : False"
)

print(
    "Threshold selected                    : False"
)

print(
    "Research decision changed             : False"
)

print(
    "Files written                         : False"
)

print(
    "\n=== END STEP 6.27B-R2 ==="
)

In [ ]:
# ============================================================
# Step 6.27C
# Clean DEV endpoint uncertainty contract audit
# ============================================================

import numpy as np
import pandas as pd
import inspect

print(
    "\n"
    "============================================================\n"
    "STEP 6.27C — CLEAN DEV ENDPOINT UNCERTAINTY CONTRACT AUDIT\n"
    "============================================================"
)

# ------------------------------------------------------------
# 1. Required frozen objects
# ------------------------------------------------------------

required_627c = [
    "val_raw_base_620gc",
    "PRIMARY_INPUT_COLUMNS",
    "scaler",
    "VALIDATION_X_REBUILT_627BR2",
    "run_frozen_uncertainty_625e1",
]

print("\n--- Required frozen objects ---")

missing_627c = []

for name_627c in required_627c:
    exists_627c = name_627c in globals()

    print(
        f"{name_627c:<36}:",
        exists_627c
    )

    if not exists_627c:
        missing_627c.append(
            name_627c
        )

if missing_627c:
    raise RuntimeError(
        "Step 6.27C cannot continue. Missing objects: "
        + str(missing_627c)
    )


# ------------------------------------------------------------
# 2. Confirm frozen uncertainty wrapper
# ------------------------------------------------------------

print(
    "\n--- Frozen uncertainty wrapper contract ---"
)

try:
    signature_627c = inspect.signature(
        run_frozen_uncertainty_625e1
    )

    print(
        "Signature:",
        signature_627c
    )

except Exception as exc_627c:
    print(
        "Signature inspection unavailable:",
        repr(exc_627c)
    )


# ------------------------------------------------------------
# 3. Validation raw-data integrity
# ------------------------------------------------------------

VAL_RAW_627C = (
    val_raw_base_620gc
    .copy(deep=True)
)

required_key_columns_627c = [
    "unit_id",
    "cycle",
]

missing_keys_627c = [
    col_627c
    for col_627c in required_key_columns_627c
    if col_627c not in VAL_RAW_627C.columns
]

if missing_keys_627c:
    raise RuntimeError(
        "Validation raw base is missing endpoint keys: "
        + str(missing_keys_627c)
    )

missing_inputs_627c = [
    col_627c
    for col_627c in PRIMARY_INPUT_COLUMNS
    if col_627c not in VAL_RAW_627C.columns
]

if missing_inputs_627c:
    raise RuntimeError(
        "Validation raw base is missing model inputs: "
        + str(missing_inputs_627c)
    )


VAL_RAW_627C = (
    VAL_RAW_627C
    .sort_values(
        ["unit_id", "cycle"]
    )
    .reset_index(drop=True)
)

duplicate_keys_627c = int(
    VAL_RAW_627C
    .duplicated(
        ["unit_id", "cycle"]
    )
    .sum()
)

unit_ids_627c = (
    VAL_RAW_627C["unit_id"]
    .drop_duplicates()
    .tolist()
)

print(
    "\n--- Frozen validation base ---"
)

print(
    "Raw shape          :",
    VAL_RAW_627C.shape
)

print(
    "Validation engines :",
    len(unit_ids_627c)
)

print(
    "Duplicate keys     :",
    duplicate_keys_627c
)

if duplicate_keys_627c != 0:
    raise RuntimeError(
        "Duplicate (unit_id, cycle) keys found."
    )

if len(unit_ids_627c) != 20:
    raise RuntimeError(
        "Expected exactly 20 frozen validation engines."
    )


# ------------------------------------------------------------
# 4. Reconstruct CLEAN scaled rows
# ------------------------------------------------------------

raw_inputs_627c = (
    VAL_RAW_627C[
        PRIMARY_INPUT_COLUMNS
    ]
    .to_numpy(
        dtype=np.float64
    )
)

scaled_rows_627c = (
    scaler
    .transform(
        raw_inputs_627c
    )
)

if not np.isfinite(
    scaled_rows_627c
).all():
    raise RuntimeError(
        "Scaled validation rows contain non-finite values."
    )

print(
    "\n--- Clean scaling ---"
)

print(
    "Scaled row matrix :",
    scaled_rows_627c.shape
)


# ------------------------------------------------------------
# 5. Build ALL rolling validation windows
#
# This mirrors the now-proven Step 6.27B reconstruction.
# ------------------------------------------------------------

WINDOW_LENGTH_627C = 30

all_windows_627c = []
all_meta_627c = []

for unit_id_627c in unit_ids_627c:

    mask_627c = (
        VAL_RAW_627C["unit_id"]
        .to_numpy()
        ==
        unit_id_627c
    )

    unit_rows_627c = (
        scaled_rows_627c[
            mask_627c
        ]
    )

    unit_cycles_627c = (
        VAL_RAW_627C.loc[
            mask_627c,
            "cycle"
        ]
        .to_numpy()
    )

    n_rows_627c = len(
        unit_rows_627c
    )

    if n_rows_627c < WINDOW_LENGTH_627C:
        raise RuntimeError(
            f"Validation engine {unit_id_627c} has only "
            f"{n_rows_627c} rows."
        )

    for end_index_627c in range(
        WINDOW_LENGTH_627C - 1,
        n_rows_627c
    ):

        start_index_627c = (
            end_index_627c
            -
            WINDOW_LENGTH_627C
            +
            1
        )

        all_windows_627c.append(
            unit_rows_627c[
                start_index_627c:
                end_index_627c + 1
            ]
        )

        all_meta_627c.append(
            {
                "unit_id": int(
                    unit_id_627c
                ),
                "cycle": int(
                    unit_cycles_627c[
                        end_index_627c
                    ]
                ),
            }
        )


X_ALL_CLEAN_REBUILT_627C = np.asarray(
    all_windows_627c,
    dtype=np.float32
)

META_ALL_CLEAN_REBUILT_627C = pd.DataFrame(
    all_meta_627c
)

print(
    "\n--- Reconstructed clean rolling windows ---"
)

print(
    "Window tensor shape :",
    X_ALL_CLEAN_REBUILT_627C.shape
)

print(
    "Metadata shape      :",
    META_ALL_CLEAN_REBUILT_627C.shape
)


# ------------------------------------------------------------
# 6. Verify against Step 6.27B-R2 frozen tensor
# ------------------------------------------------------------

reference_all_627c = np.asarray(
    VALIDATION_X_REBUILT_627BR2
)

if (
    X_ALL_CLEAN_REBUILT_627C.shape
    !=
    reference_all_627c.shape
):
    raise RuntimeError(
        "Reconstructed clean rolling-window shape differs "
        "from the Step 6.27B-R2 frozen tensor."
    )

rolling_exact_627c = bool(
    np.array_equal(
        X_ALL_CLEAN_REBUILT_627C,
        reference_all_627c
    )
)

rolling_max_diff_627c = float(
    np.max(
        np.abs(
            X_ALL_CLEAN_REBUILT_627C.astype(
                np.float64
            )
            -
            reference_all_627c.astype(
                np.float64
            )
        )
    )
)

print(
    "\n--- Step 6.27B-R2 continuity check ---"
)

print(
    "Rolling tensors exact :",
    rolling_exact_627c
)

print(
    "Maximum difference    :",
    rolling_max_diff_627c
)

if not rolling_exact_627c:
    raise RuntimeError(
        "Step 6.27C reconstruction no longer exactly matches "
        "the proven Step 6.27B-R2 float32 tensor."
    )


# ------------------------------------------------------------
# 7. Identify final endpoint window of each DEV engine
# ------------------------------------------------------------

endpoint_indices_627c = (
    META_ALL_CLEAN_REBUILT_627C
    .groupby(
        "unit_id",
        sort=False
    )
    .tail(1)
    .index
    .to_numpy()
)

X_DEV_ENDPOINTS_CLEAN_627C = (
    X_ALL_CLEAN_REBUILT_627C[
        endpoint_indices_627c
    ]
    .copy()
)

DEV_ENDPOINT_META_627C = (
    META_ALL_CLEAN_REBUILT_627C
    .iloc[
        endpoint_indices_627c
    ]
    .reset_index(drop=True)
    .copy()
)

print(
    "\n--- Clean DEV endpoints ---"
)

print(
    "Endpoint tensor shape :",
    X_DEV_ENDPOINTS_CLEAN_627C.shape
)

print(
    "Endpoint metadata     :",
    DEV_ENDPOINT_META_627C.shape
)

print(
    "Unique endpoint units :",
    DEV_ENDPOINT_META_627C[
        "unit_id"
    ].nunique()
)

print(
    "All endpoint values finite:",
    bool(
        np.isfinite(
            X_DEV_ENDPOINTS_CLEAN_627C
        ).all()
    )
)

if X_DEV_ENDPOINTS_CLEAN_627C.shape != (
    20,
    30,
    17,
):
    raise RuntimeError(
        "Expected clean DEV endpoint tensor shape "
        "(20, 30, 17)."
    )


# ------------------------------------------------------------
# 8. Run FROZEN uncertainty once on clean DEV endpoints
# ------------------------------------------------------------

print(
    "\n--- Executing one CLEAN frozen uncertainty call ---"
)

CLEAN_UNCERTAINTY_RAW_627C = (
    run_frozen_uncertainty_625e1(
        X_DEV_ENDPOINTS_CLEAN_627C
    )
)

print(
    "Return type:",
    type(
        CLEAN_UNCERTAINTY_RAW_627C
    ).__name__
)


# ------------------------------------------------------------
# 9. Compact structural inspection only
#
# We deliberately do NOT guess the wrapper's contract.
# ------------------------------------------------------------

def describe_object_627c(
    obj_627c,
    label_627c="result",
):
    print(
        f"\n--- {label_627c} structure ---"
    )

    print(
        "Python type:",
        type(
            obj_627c
        ).__name__
    )

    if isinstance(
        obj_627c,
        pd.DataFrame,
    ):
        print(
            "Shape:",
            obj_627c.shape
        )

        print(
            "Columns:",
            list(
                obj_627c.columns
            )
        )

        print(
            "\nFirst 3 rows:"
        )

        print(
            obj_627c
            .head(3)
            .to_string(
                index=False
            )
        )

    elif isinstance(
        obj_627c,
        pd.Series,
    ):
        print(
            "Length:",
            len(
                obj_627c
            )
        )

        print(
            "Name:",
            obj_627c.name
        )

        print(
            "First values:",
            obj_627c
            .head(5)
            .tolist()
        )

    elif isinstance(
        obj_627c,
        np.ndarray,
    ):
        print(
            "Shape:",
            obj_627c.shape
        )

        print(
            "Dtype:",
            obj_627c.dtype
        )

        flat_627c = (
            obj_627c
            .reshape(-1)
        )

        print(
            "First values:",
            flat_627c[
                :min(
                    5,
                    len(
                        flat_627c
                    ),
                )
            ]
        )

    elif isinstance(
        obj_627c,
        (tuple, list),
    ):
        print(
            "Length:",
            len(
                obj_627c
            )
        )

        for index_627c, component_627c in enumerate(
            obj_627c
        ):
            print(
                f"\nComponent {index_627c}:"
            )

            print(
                " type =",
                type(
                    component_627c
                ).__name__
            )

            if hasattr(
                component_627c,
                "shape",
            ):
                print(
                    " shape =",
                    component_627c.shape
                )

    elif isinstance(
        obj_627c,
        dict,
    ):
        print(
            "Keys:",
            list(
                obj_627c.keys()
            )
        )

        for key_627c, value_627c in obj_627c.items():
            print(
                f"{key_627c}: "
                f"type={type(value_627c).__name__}",
                end=""
            )

            if hasattr(
                value_627c,
                "shape",
            ):
                print(
                    f", shape={value_627c.shape}"
                )
            else:
                print()

    else:
        print(
            "Representation:",
            repr(
                obj_627c
            )[:1000]
        )


describe_object_627c(
    CLEAN_UNCERTAINTY_RAW_627C,
    "Frozen uncertainty output",
)


# ------------------------------------------------------------
# 10. Preserve the verified clean endpoint state
# ------------------------------------------------------------

CLEAN_DEV_ENDPOINT_SEQUENCES_627C = (
    X_DEV_ENDPOINTS_CLEAN_627C
    .copy()
)

CLEAN_DEV_ENDPOINT_META_627C = (
    DEV_ENDPOINT_META_627C
    .copy()
)


print(
    "\n"
    "============================================================\n"
    "STEP 6.27C CONTRACT SUMMARY\n"
    "============================================================"
)

print(
    "Validation preprocessing exact :",
    rolling_exact_627c
)

print(
    "Clean DEV engines              :",
    len(
        CLEAN_DEV_ENDPOINT_META_627C
    )
)

print(
    "Clean endpoint tensor           :",
    CLEAN_DEV_ENDPOINT_SEQUENCES_627C.shape
)

print(
    "Frozen uncertainty executed     : True"
)

print(
    "Official TEST labels inspected  : False"
)

print(
    "Official TEST scored            : False"
)

print(
    "Corruption generated            : False"
)

print(
    "Threshold selected              : False"
)

print(
    "Research decision changed       : False"
)

print(
    "Files written                   : False"
)

print(
    "\n=== END STEP 6.27C ==="
)

In [ ]:
# ============================================================
# Step 6.27C1
# Identify exact frozen uncertainty tuple semantics
# ============================================================

import numpy as np
import pandas as pd
import inspect
import textwrap

print(
    "\n"
    "============================================================\n"
    "STEP 6.27C1 — FROZEN UNCERTAINTY RETURN-CONTRACT AUDIT\n"
    "============================================================"
)


# ------------------------------------------------------------
# 1. Required object
# ------------------------------------------------------------

if "run_frozen_uncertainty_625e1" not in globals():
    raise RuntimeError(
        "run_frozen_uncertainty_625e1 is missing."
    )

if "CLEAN_UNCERTAINTY_RAW_627C" not in globals():
    raise RuntimeError(
        "CLEAN_UNCERTAINTY_RAW_627C is missing. "
        "Run Step 6.27C first."
    )


# ------------------------------------------------------------
# 2. Function contract
# ------------------------------------------------------------

print("\n--- Function signature ---")

print(
    inspect.signature(
        run_frozen_uncertainty_625e1
    )
)


# ------------------------------------------------------------
# 3. Try to recover function source
# ------------------------------------------------------------

print("\n--- Function source / return logic ---")

SOURCE_RECOVERED_627C1 = False

try:
    source_627c1 = inspect.getsource(
        run_frozen_uncertainty_625e1
    )

    SOURCE_RECOVERED_627C1 = True

    # We only need the compact function body.
    print(
        textwrap.dedent(
            source_627c1
        )
    )

except Exception as exc_627c1:

    print(
        "Source recovery unavailable:",
        repr(exc_627c1)
    )


# ------------------------------------------------------------
# 4. Validate tuple structure
# ------------------------------------------------------------

result_627c1 = CLEAN_UNCERTAINTY_RAW_627C

if not isinstance(
    result_627c1,
    tuple,
):
    raise RuntimeError(
        "Expected frozen uncertainty output to be a tuple."
    )

if len(
    result_627c1
) != 3:
    raise RuntimeError(
        "Expected exactly 3 tuple components."
    )


# ------------------------------------------------------------
# 5. Compact numerical fingerprint of every component
# ------------------------------------------------------------

rows_627c1 = []

for idx_627c1, component_627c1 in enumerate(
    result_627c1
):

    arr_627c1 = np.asarray(
        component_627c1
    ).reshape(-1)

    if len(
        arr_627c1
    ) != 20:
        raise RuntimeError(
            f"Component {idx_627c1} does not contain "
            "20 DEV endpoint values."
        )

    finite_627c1 = bool(
        np.isfinite(
            arr_627c1
        ).all()
    )

    rows_627c1.append(
        {
            "component": idx_627c1,
            "length": len(arr_627c1),
            "finite": finite_627c1,
            "minimum": float(
                np.min(arr_627c1)
            ),
            "mean": float(
                np.mean(arr_627c1)
            ),
            "median": float(
                np.median(arr_627c1)
            ),
            "maximum": float(
                np.max(arr_627c1)
            ),
            "std": float(
                np.std(
                    arr_627c1,
                    ddof=0
                )
            ),
        }
    )


COMPONENT_FINGERPRINT_627C1 = pd.DataFrame(
    rows_627c1
)

print(
    "\n--- Component numerical fingerprints ---"
)

print(
    COMPONENT_FINGERPRINT_627C1
    .to_string(
        index=False
    )
)


# ------------------------------------------------------------
# 6. First five aligned values
# ------------------------------------------------------------

preview_627c1 = pd.DataFrame(
    {
        "unit_id":
            CLEAN_DEV_ENDPOINT_META_627C[
                "unit_id"
            ].to_numpy(),

        "cycle":
            CLEAN_DEV_ENDPOINT_META_627C[
                "cycle"
            ].to_numpy(),

        "component_0":
            np.asarray(
                result_627c1[0]
            ).reshape(-1),

        "component_1":
            np.asarray(
                result_627c1[1]
            ).reshape(-1),

        "component_2":
            np.asarray(
                result_627c1[2]
            ).reshape(-1),
    }
)


print(
    "\n--- First five endpoint values ---"
)

print(
    preview_627c1
    .head(5)
    .to_string(
        index=False
    )
)


# ------------------------------------------------------------
# 7. Search existing frozen validation references
#
# Diagnostic only. Nothing is modified.
# ------------------------------------------------------------

print(
    "\n--- Existing likely validation uncertainty references ---"
)

likely_names_627c1 = [
    name_627c1
    for name_627c1 in globals()
    if (
        "uncert" in name_627c1.lower()
        or "mc_std" in name_627c1.lower()
    )
]

reference_rows_627c1 = []

for name_627c1 in likely_names_627c1:

    obj_627c1 = globals()[
        name_627c1
    ]

    shape_627c1 = getattr(
        obj_627c1,
        "shape",
        None
    )

    length_627c1 = None

    try:
        length_627c1 = len(
            obj_627c1
        )
    except Exception:
        pass

    if length_627c1 in {
        20,
        3490,
    }:

        reference_rows_627c1.append(
            {
                "object_name":
                    name_627c1,

                "type":
                    type(
                        obj_627c1
                    ).__name__,

                "length":
                    length_627c1,

                "shape":
                    str(
                        shape_627c1
                    ),
            }
        )


if reference_rows_627c1:

    REFERENCE_INVENTORY_627C1 = pd.DataFrame(
        reference_rows_627c1
    )

    print(
        REFERENCE_INVENTORY_627C1
        .sort_values(
            [
                "length",
                "object_name",
            ]
        )
        .to_string(
            index=False
        )
    )

else:

    print(
        "No 20-row or 3490-row uncertainty-named "
        "objects were found."
    )


# ------------------------------------------------------------
# 8. Safety summary
# ------------------------------------------------------------

print(
    "\n"
    "============================================================\n"
    "STEP 6.27C1 SUMMARY\n"
    "============================================================"
)

print(
    "Frozen wrapper source recovered :",
    SOURCE_RECOVERED_627C1
)

print(
    "Tuple components                :",
    len(
        result_627c1
    )
)

print(
    "Each component length = 20      :",
    all(
        len(
            np.asarray(
                x_627c1
            ).reshape(-1)
        ) == 20
        for x_627c1 in result_627c1
    )
)

print(
    "All component values finite     :",
    all(
        np.isfinite(
            np.asarray(
                x_627c1
            )
        ).all()
        for x_627c1 in result_627c1
    )
)

print(
    "Official TEST labels inspected  : False"
)

print(
    "Official TEST scored            : False"
)

print(
    "Corruptions generated           : False"
)

print(
    "Threshold selected              : False"
)

print(
    "Research decision changed       : False"
)

print(
    "Files written                   : False"
)

print(
    "\n=== END STEP 6.27C1 ==="
)

In [ ]:
# ============================================================
# Step 6.27C2
# Exact mc_uncertainty_623cb2 return-contract inspection
# ============================================================

import inspect
import numpy as np

print(
    "\n"
    "============================================================\n"
    "STEP 6.27C2 — EXACT MC-UNCERTAINTY RETURN CONTRACT\n"
    "============================================================"
)

# ------------------------------------------------------------
# 1. Required function
# ------------------------------------------------------------

if "mc_uncertainty_623cb2" not in globals():
    raise RuntimeError(
        "mc_uncertainty_623cb2 is missing."
    )

if "CLEAN_UNCERTAINTY_RAW_627C" not in globals():
    raise RuntimeError(
        "CLEAN_UNCERTAINTY_RAW_627C is missing. "
        "Run Step 6.27C first."
    )


# ------------------------------------------------------------
# 2. Signature
# ------------------------------------------------------------

print("\n--- Function signature ---")
print(
    inspect.signature(
        mc_uncertainty_623cb2
    )
)


# ------------------------------------------------------------
# 3. Recover source and print ONLY relevant lines
# ------------------------------------------------------------

print("\n--- Relevant source lines ---")

source_recovered_627c2 = False

try:
    source_627c2 = inspect.getsource(
        mc_uncertainty_623cb2
    )

    source_recovered_627c2 = True

    lines_627c2 = source_627c2.splitlines()

    keywords_627c2 = (
        "return",
        "prediction",
        "predict",
        "mc_std",
        "std",
        "uncertainty",
        "reliability",
        "phase",
        "result",
    )

    printed_indices_627c2 = set()

    for idx_627c2, line_627c2 in enumerate(
        lines_627c2
    ):

        if any(
            keyword_627c2 in line_627c2.lower()
            for keyword_627c2 in keywords_627c2
        ):

            # Include one line before and two after
            start_627c2 = max(
                0,
                idx_627c2 - 1
            )

            stop_627c2 = min(
                len(lines_627c2),
                idx_627c2 + 3
            )

            for j_627c2 in range(
                start_627c2,
                stop_627c2
            ):
                if j_627c2 not in printed_indices_627c2:
                    print(
                        f"{j_627c2 + 1:03d}: "
                        f"{lines_627c2[j_627c2]}"
                    )

                    printed_indices_627c2.add(
                        j_627c2
                    )

except Exception as exc_627c2:

    print(
        "Source recovery unavailable:",
        repr(exc_627c2)
    )


# ------------------------------------------------------------
# 4. Clean tuple fingerprints
# ------------------------------------------------------------

result_627c2 = CLEAN_UNCERTAINTY_RAW_627C

if not isinstance(
    result_627c2,
    tuple
):
    raise RuntimeError(
        "Frozen uncertainty output is not a tuple."
    )

if len(
    result_627c2
) != 3:
    raise RuntimeError(
        "Expected exactly three tuple components."
    )


print("\n--- Component fingerprints ---")

for idx_627c2, component_627c2 in enumerate(
    result_627c2
):

    arr_627c2 = np.asarray(
        component_627c2,
        dtype=float
    ).reshape(-1)

    print(
        f"Component {idx_627c2}: "
        f"n={len(arr_627c2)}, "
        f"min={np.min(arr_627c2):.6f}, "
        f"mean={np.mean(arr_627c2):.6f}, "
        f"max={np.max(arr_627c2):.6f}, "
        f"std={np.std(arr_627c2, ddof=0):.6f}"
    )


# ------------------------------------------------------------
# 5. Print first three values only
# ------------------------------------------------------------

print("\n--- First three values ---")

for idx_627c2, component_627c2 in enumerate(
    result_627c2
):

    arr_627c2 = np.asarray(
        component_627c2,
        dtype=float
    ).reshape(-1)

    print(
        f"Component {idx_627c2}:",
        np.round(
            arr_627c2[:3],
            6
        ).tolist()
    )


# ------------------------------------------------------------
# 6. Safety summary
# ------------------------------------------------------------

print(
    "\n"
    "============================================================\n"
    "STEP 6.27C2 SUMMARY\n"
    "============================================================"
)

print(
    "Source recovered             :",
    source_recovered_627c2
)

print(
    "Tuple length                 :",
    len(
        result_627c2
    )
)

print(
    "All components length 20     :",
    all(
        len(
            np.asarray(
                x_627c2
            ).reshape(-1)
        ) == 20
        for x_627c2 in result_627c2
    )
)

print(
    "All values finite            :",
    all(
        np.isfinite(
            np.asarray(
                x_627c2,
                dtype=float
            )
        ).all()
        for x_627c2 in result_627c2
    )
)

print(
    "Official TEST labels used    : False"
)

print(
    "Official TEST scored         : False"
)

print(
    "Research decision changed    : False"
)

print(
    "Files written                : False"
)

print(
    "\n=== END STEP 6.27C2 ==="
)

In [ ]:
# ============================================================
# STEP 6.27C3
# Exact return-order closure for mc_uncertainty_623cb2
# ============================================================

import inspect
import ast
import textwrap
import numpy as np

print(
    "\n"
    "============================================================\n"
    "STEP 6.27C3 — EXACT RETURN-ORDER CLOSURE\n"
    "============================================================"
)

# ------------------------------------------------------------
# 1. Required objects
# ------------------------------------------------------------

required_627c3 = [
    "mc_uncertainty_623cb2",
    "CLEAN_UNCERTAINTY_RAW_627C",
]

for name_627c3 in required_627c3:
    if name_627c3 not in globals():
        raise RuntimeError(
            f"Required object missing: {name_627c3}"
        )


# ------------------------------------------------------------
# 2. Recover exact function source
# ------------------------------------------------------------

source_627c3 = inspect.getsource(
    mc_uncertainty_623cb2
)

source_627c3 = textwrap.dedent(
    source_627c3
)

tree_627c3 = ast.parse(
    source_627c3
)


# ------------------------------------------------------------
# 3. Print ONLY return expression(s)
# ------------------------------------------------------------

print("\n--- EXACT RETURN STATEMENT(S) ---")

return_nodes_627c3 = [
    node_627c3
    for node_627c3 in ast.walk(tree_627c3)
    if isinstance(
        node_627c3,
        ast.Return
    )
]

if not return_nodes_627c3:
    raise RuntimeError(
        "No return statement found."
    )

for i_627c3, node_627c3 in enumerate(
    return_nodes_627c3,
    start=1
):
    print(
        f"Return {i_627c3}:",
        ast.unparse(
            node_627c3.value
        )
    )


# ------------------------------------------------------------
# 4. Print assignments involving uncertainty / prediction
# ------------------------------------------------------------

print(
    "\n--- RELEVANT VARIABLE ASSIGNMENTS ---"
)

keywords_627c3 = (
    "prediction",
    "uncert",
    "std",
    "reliab",
    "phase",
    "mc_",
)

for node_627c3 in ast.walk(
    tree_627c3
):

    if isinstance(
        node_627c3,
        (ast.Assign, ast.AnnAssign)
    ):

        try:
            text_627c3 = ast.unparse(
                node_627c3
            )
        except Exception:
            continue

        if any(
            keyword_627c3
            in text_627c3.lower()
            for keyword_627c3
            in keywords_627c3
        ):
            print(
                text_627c3
            )


# ------------------------------------------------------------
# 5. Fingerprint the already-produced clean output
# ------------------------------------------------------------

result_627c3 = CLEAN_UNCERTAINTY_RAW_627C

if not isinstance(
    result_627c3,
    tuple
):
    raise RuntimeError(
        "CLEAN_UNCERTAINTY_RAW_627C "
        "is not a tuple."
    )

if len(
    result_627c3
) != 3:
    raise RuntimeError(
        "Expected exactly 3 tuple components, "
        f"found {len(result_627c3)}."
    )


print(
    "\n--- RETURNED COMPONENT FINGERPRINTS ---"
)

for idx_627c3, component_627c3 in enumerate(
    result_627c3
):

    arr_627c3 = np.asarray(
        component_627c3,
        dtype=float
    ).reshape(-1)

    print(
        f"Component {idx_627c3}: "
        f"shape={arr_627c3.shape}, "
        f"min={arr_627c3.min():.6f}, "
        f"mean={arr_627c3.mean():.6f}, "
        f"max={arr_627c3.max():.6f}"
    )


# ------------------------------------------------------------
# 6. Integrity checks
# ------------------------------------------------------------

all_length_20_627c3 = all(
    np.asarray(
        component_627c3
    ).reshape(-1).shape[0] == 20
    for component_627c3
    in result_627c3
)

all_finite_627c3 = all(
    np.isfinite(
        np.asarray(
            component_627c3,
            dtype=float
        )
    ).all()
    for component_627c3
    in result_627c3
)


print(
    "\n"
    "============================================================\n"
    "STEP 6.27C3 SUMMARY\n"
    "============================================================"
)

print(
    "Return statements found       :",
    len(
        return_nodes_627c3
    )
)

print(
    "Tuple components               :",
    len(
        result_627c3
    )
)

print(
    "Every component has 20 values :",
    all_length_20_627c3
)

print(
    "All returned values finite     :",
    all_finite_627c3
)

print(
    "Official TEST labels used      : False"
)

print(
    "Official TEST scored           : False"
)

print(
    "Research decision changed      : False"
)

print(
    "Files written                  : False"
)

print(
    "\n=== END STEP 6.27C3 ==="
)

In [ ]:
# ============================================================
# STEP 6.27D
# Single-scenario DEV uncertainty + sensor-quality
# endpoint alignment smoke test
# ============================================================

import numpy as np
import pandas as pd

print(
    "\n"
    "============================================================\n"
    "STEP 6.27D — SINGLE-SCENARIO DEV ALIGNMENT SMOKE TEST\n"
    "============================================================"
)


# ============================================================
# 1. Required frozen objects
# ============================================================

required_627d = [
    "val_raw_base_620gc",
    "scenario_manifests_620gc",
    "_apply_dev_corruption_627a",
    "PRIMARY_INPUT_COLUMNS",
    "scaler",
    "X_val_clean",
    "VALIDATION_X_REBUILT_627BR2",
    "CLEAN_UNCERTAINTY_RAW_627C",
    "run_frozen_uncertainty_625e1",

    "build_sensor_quality_candidates_626b",
    "extract_quality_window_dataframe_626e",
    "score_sensor_quality_windows_626d",

    "SENSOR_QUALITY_SENSOR_COLUMNS_626B",
    "SENSOR_QUALITY_REFERENCE_626B",
    "SENSOR_QUALITY_WINDOW_LENGTH_626B",
    "SENSOR_QUALITY_VAL_CLEAN_SCORED_626D",
]

print("\n--- Required frozen objects ---")

missing_627d = []

for name_627d in required_627d:
    present_627d = name_627d in globals()

    print(
        f"{name_627d:<48}: "
        f"{present_627d}"
    )

    if not present_627d:
        missing_627d.append(
            name_627d
        )

if missing_627d:
    raise RuntimeError(
        "Step 6.27D cannot continue. "
        "Missing required objects: "
        + str(missing_627d)
    )


# ============================================================
# 2. Frozen constants
# ============================================================

WINDOW_LENGTH_627D = int(
    SENSOR_QUALITY_WINDOW_LENGTH_626B
)

EXPECTED_ENGINES_627D = 20

FUSION_COLUMN_627D = (
    "sq_candidate_max_fusion"
)

MODEL_DTYPE_627D = np.asarray(
    X_val_clean
).dtype


print(
    "\n--- Frozen alignment definition ---"
)

print(
    "Window length       :",
    WINDOW_LENGTH_627D
)

print(
    "Expected DEV engines:",
    EXPECTED_ENGINES_627D
)

print(
    "Model tensor dtype  :",
    MODEL_DTYPE_627D
)

print(
    "Sensor-quality fusion:",
    FUSION_COLUMN_627D
)


# ============================================================
# 3. Basic validation-data contract
# ============================================================

base_627d = (
    val_raw_base_620gc
    .copy()
)

required_raw_columns_627d = (
    ["unit_id", "cycle"]
    + list(
        PRIMARY_INPUT_COLUMNS
    )
)

missing_raw_columns_627d = [
    col_627d
    for col_627d
    in required_raw_columns_627d
    if col_627d
    not in base_627d.columns
]

if missing_raw_columns_627d:
    raise RuntimeError(
        "Validation raw DataFrame is missing: "
        + str(
            missing_raw_columns_627d
        )
    )


base_627d = (
    base_627d
    .sort_values(
        ["unit_id", "cycle"]
    )
    .reset_index(
        drop=True
    )
)


duplicate_keys_627d = int(
    base_627d.duplicated(
        subset=[
            "unit_id",
            "cycle",
        ]
    ).sum()
)

if duplicate_keys_627d != 0:
    raise RuntimeError(
        "Validation base contains duplicate "
        "(unit_id, cycle) keys."
    )


unit_ids_627d = (
    base_627d[
        "unit_id"
    ]
    .drop_duplicates()
    .tolist()
)

if len(
    unit_ids_627d
) != EXPECTED_ENGINES_627D:
    raise RuntimeError(
        "Unexpected validation engine count: "
        f"{len(unit_ids_627d)}"
    )


print(
    "\n--- Validation base ---"
)

print(
    "Rows           :",
    len(
        base_627d
    )
)

print(
    "Engines        :",
    len(
        unit_ids_627d
    )
)

print(
    "Duplicate keys :",
    duplicate_keys_627d
)


# ============================================================
# 4. Helper:
#    build EXACT endpoint tensor directly from raw DataFrame
#
#    Since the prediction endpoint for each engine is its
#    final 30-cycle window, constructing the endpoint from the
#    last 30 scaled rows is mathematically equivalent to
#    constructing every rolling window and then taking the last.
# ============================================================

def _build_endpoint_tensor_627d(
    raw_df_627d
):

    ordered_627d = (
        raw_df_627d
        .sort_values(
            ["unit_id", "cycle"]
        )
        .reset_index(
            drop=True
        )
    )

    if ordered_627d.duplicated(
        subset=[
            "unit_id",
            "cycle",
        ]
    ).any():
        raise RuntimeError(
            "Duplicate endpoint keys detected."
        )

    # Preserve feature names when calling scaler.transform.
    scaled_rows_627d = (
        scaler.transform(
            ordered_627d[
                PRIMARY_INPUT_COLUMNS
            ]
        )
    )

    scaled_rows_627d = np.asarray(
        scaled_rows_627d,
        dtype=MODEL_DTYPE_627D,
    )

    if not np.isfinite(
        scaled_rows_627d
    ).all():
        raise RuntimeError(
            "Scaled validation values are not finite."
        )

    endpoint_sequences_627d = []
    endpoint_metadata_627d = []

    for (
        unit_627d,
        group_627d,
    ) in ordered_627d.groupby(
        "unit_id",
        sort=True,
    ):

        row_positions_627d = (
            group_627d.index
            .to_numpy()
        )

        if len(
            row_positions_627d
        ) < WINDOW_LENGTH_627D:
            raise RuntimeError(
                f"Engine {unit_627d} has fewer than "
                f"{WINDOW_LENGTH_627D} cycles."
            )

        endpoint_positions_627d = (
            row_positions_627d[
                -WINDOW_LENGTH_627D:
            ]
        )

        endpoint_sequence_627d = (
            scaled_rows_627d[
                endpoint_positions_627d
            ]
        )

        if endpoint_sequence_627d.shape != (
            WINDOW_LENGTH_627D,
            len(
                PRIMARY_INPUT_COLUMNS
            ),
        ):
            raise RuntimeError(
                "Unexpected endpoint sequence shape "
                f"for unit {unit_627d}: "
                f"{endpoint_sequence_627d.shape}"
            )

        endpoint_sequences_627d.append(
            endpoint_sequence_627d
        )

        final_row_627d = (
            group_627d
            .iloc[-1]
        )

        endpoint_metadata_627d.append(
            {
                "unit_id": int(
                    unit_627d
                ),
                "cycle": int(
                    final_row_627d[
                        "cycle"
                    ]
                ),
            }
        )

    tensor_627d = np.stack(
        endpoint_sequences_627d,
        axis=0,
    )

    metadata_627d = pd.DataFrame(
        endpoint_metadata_627d
    )

    return (
        tensor_627d,
        metadata_627d,
    )


# ============================================================
# 5. Verify endpoint construction against the already-proven
#    clean rolling-window reconstruction from Step 6.27B-R2
# ============================================================

clean_endpoint_tensor_627d, \
clean_endpoint_meta_627d = (
    _build_endpoint_tensor_627d(
        base_627d
    )
)


# Determine the last rolling-window index belonging to
# each validation engine.

window_counts_627d = []

for (
    unit_627d,
    group_627d,
) in base_627d.groupby(
    "unit_id",
    sort=True,
):

    count_627d = (
        len(
            group_627d
        )
        - WINDOW_LENGTH_627D
        + 1
    )

    if count_627d <= 0:
        raise RuntimeError(
            f"Invalid rolling-window count for "
            f"engine {unit_627d}."
        )

    window_counts_627d.append(
        count_627d
    )


endpoint_indices_627d = (
    np.cumsum(
        window_counts_627d
    )
    - 1
)


rolling_reference_endpoints_627d = (
    np.asarray(
        VALIDATION_X_REBUILT_627BR2
    )[
        endpoint_indices_627d
    ]
)


if (
    rolling_reference_endpoints_627d.shape
    !=
    clean_endpoint_tensor_627d.shape
):
    raise RuntimeError(
        "Clean endpoint reference shape mismatch."
    )


clean_endpoint_max_diff_627d = float(
    np.max(
        np.abs(
            clean_endpoint_tensor_627d.astype(
                np.float64
            )
            -
            rolling_reference_endpoints_627d.astype(
                np.float64
            )
        )
    )
)


clean_endpoint_equivalent_627d = bool(
    clean_endpoint_max_diff_627d
    <= 5e-7
)


print(
    "\n--- Clean endpoint reconstruction check ---"
)

print(
    "Direct endpoint shape       :",
    clean_endpoint_tensor_627d.shape
)

print(
    "Rolling reference shape     :",
    rolling_reference_endpoints_627d.shape
)

print(
    "Maximum absolute difference :",
    clean_endpoint_max_diff_627d
)

print(
    "Equivalent at 5e-7          :",
    clean_endpoint_equivalent_627d
)


if not clean_endpoint_equivalent_627d:
    raise RuntimeError(
        "Direct endpoint construction does not "
        "match the previously verified validation "
        "preprocessing."
    )


# ============================================================
# 6. Select ONE frozen scenario
#
#    Prefer BIAS because its effect is straightforward for the
#    smoke test. Nothing is tuned or selected from RUL labels.
# ============================================================

available_families_627d = list(
    scenario_manifests_620gc.keys()
)

if "bias" in available_families_627d:
    family_627d = "bias"
else:
    family_627d = (
        available_families_627d[0]
    )


manifest_627d = (
    scenario_manifests_620gc[
        family_627d
    ]
)

if not isinstance(
    manifest_627d,
    pd.DataFrame,
):
    raise RuntimeError(
        "Frozen manifest is not a DataFrame."
    )

if len(
    manifest_627d
) == 0:
    raise RuntimeError(
        "Frozen manifest is empty."
    )


scenario_row_627d = (
    manifest_627d
    .iloc[0]
    .copy()
)


scenario_id_627d = (
    scenario_row_627d.get(
        "scenario_id",
        scenario_row_627d.get(
            "manifest_scenario_id",
            "UNKNOWN",
        ),
    )
)


print(
    "\n--- Selected frozen smoke-test scenario ---"
)

print(
    "Family      :",
    family_627d
)

print(
    "Scenario ID :",
    scenario_id_627d
)


# ============================================================
# 7. Apply frozen DEV corruption
# ============================================================

corruption_result_627d = (
    _apply_dev_corruption_627a(
        base_627d.copy(),
        family_627d,
        scenario_row_627d,
    )
)


# The frozen corruption helper may return the DataFrame
# directly or as one component of a tuple/list.
if isinstance(
    corruption_result_627d,
    pd.DataFrame,
):

    corrupted_raw_627d = (
        corruption_result_627d
        .copy()
    )

elif isinstance(
    corruption_result_627d,
    (tuple, list),
):

    dataframe_candidates_627d = [
        obj_627d
        for obj_627d
        in corruption_result_627d
        if isinstance(
            obj_627d,
            pd.DataFrame,
        )
        and len(
            obj_627d
        )
        ==
        len(
            base_627d
        )
    ]

    if len(
        dataframe_candidates_627d
    ) != 1:
        raise RuntimeError(
            "Could not uniquely identify corrupted "
            "validation DataFrame."
        )

    corrupted_raw_627d = (
        dataframe_candidates_627d[0]
        .copy()
    )

else:

    raise RuntimeError(
        "Unexpected frozen corruption return type: "
        + type(
            corruption_result_627d
        ).__name__
    )


corrupted_raw_627d = (
    corrupted_raw_627d
    .sort_values(
        ["unit_id", "cycle"]
    )
    .reset_index(
        drop=True
    )
)


if corrupted_raw_627d.shape != base_627d.shape:
    raise RuntimeError(
        "Corruption changed validation DataFrame shape."
    )


if not np.array_equal(
    corrupted_raw_627d[
        ["unit_id", "cycle"]
    ].to_numpy(),
    base_627d[
        ["unit_id", "cycle"]
    ].to_numpy(),
):
    raise RuntimeError(
        "Corruption changed validation endpoint keys."
    )


numeric_columns_627d = (
    base_627d
    .select_dtypes(
        include=[
            np.number
        ]
    )
    .columns
)

comparison_columns_627d = [
    col_627d
    for col_627d
    in numeric_columns_627d
    if col_627d
    not in [
        "unit_id",
        "cycle",
    ]
]


changed_raw_cells_627d = int(
    np.sum(
        ~np.isclose(
            base_627d[
                comparison_columns_627d
            ].to_numpy(
                dtype=float
            ),
            corrupted_raw_627d[
                comparison_columns_627d
            ].to_numpy(
                dtype=float
            ),
            rtol=0.0,
            atol=0.0,
            equal_nan=True,
        )
    )
)


print(
    "\n--- Corruption integrity ---"
)

print(
    "Recovered shape        :",
    corrupted_raw_627d.shape
)

print(
    "Changed numeric cells  :",
    changed_raw_cells_627d
)

print(
    "Endpoint keys preserved:",
    True
)


if changed_raw_cells_627d <= 0:
    raise RuntimeError(
        "Frozen corruption did not change any "
        "numeric validation values."
    )


# ============================================================
# 8. Build corrupted endpoint tensor
# ============================================================

corrupted_endpoint_tensor_627d, \
corrupted_endpoint_meta_627d = (
    _build_endpoint_tensor_627d(
        corrupted_raw_627d
    )
)


if corrupted_endpoint_tensor_627d.shape != (
    EXPECTED_ENGINES_627D,
    WINDOW_LENGTH_627D,
    len(
        PRIMARY_INPUT_COLUMNS
    ),
):
    raise RuntimeError(
        "Unexpected corrupted endpoint tensor shape: "
        f"{corrupted_endpoint_tensor_627d.shape}"
    )


if not np.array_equal(
    clean_endpoint_meta_627d[
        ["unit_id", "cycle"]
    ].to_numpy(),
    corrupted_endpoint_meta_627d[
        ["unit_id", "cycle"]
    ].to_numpy(),
):
    raise RuntimeError(
        "Clean/corrupted endpoint metadata differ."
    )


endpoint_tensor_changed_627d = bool(
    np.any(
        clean_endpoint_tensor_627d
        !=
        corrupted_endpoint_tensor_627d
    )
)


print(
    "\n--- Corrupted endpoint tensor ---"
)

print(
    "Shape                  :",
    corrupted_endpoint_tensor_627d.shape
)

print(
    "Endpoint tensor changed:",
    endpoint_tensor_changed_627d
)


if not endpoint_tensor_changed_627d:
    raise RuntimeError(
        "The selected corruption did not affect "
        "any endpoint input value."
    )


# ============================================================
# 9. Run frozen uncertainty
#
#    Exact Step 6.27C3 contract:
#       0 = deterministic_prediction
#       1 = mc_std_values
#       2 = adjusted_uncertainty
# ============================================================

print(
    "\n--- Executing frozen uncertainty for ONE scenario ---"
)


corrupted_uncertainty_raw_627d = (
    run_frozen_uncertainty_625e1(
        corrupted_endpoint_tensor_627d
    )
)


if not isinstance(
    corrupted_uncertainty_raw_627d,
    tuple,
):
    raise RuntimeError(
        "Frozen uncertainty output is not a tuple."
    )

if len(
    corrupted_uncertainty_raw_627d
) != 3:
    raise RuntimeError(
        "Frozen uncertainty output does not contain "
        "exactly three components."
    )


corrupted_prediction_627d = (
    np.asarray(
        corrupted_uncertainty_raw_627d[0],
        dtype=float,
    )
    .reshape(-1)
)

corrupted_mc_std_627d = (
    np.asarray(
        corrupted_uncertainty_raw_627d[1],
        dtype=float,
    )
    .reshape(-1)
)

corrupted_adjusted_uncertainty_627d = (
    np.asarray(
        corrupted_uncertainty_raw_627d[2],
        dtype=float,
    )
    .reshape(-1)
)


for name_627d, values_627d in [
    (
        "prediction",
        corrupted_prediction_627d,
    ),
    (
        "mc_std",
        corrupted_mc_std_627d,
    ),
    (
        "adjusted_uncertainty",
        corrupted_adjusted_uncertainty_627d,
    ),
]:

    if len(
        values_627d
    ) != EXPECTED_ENGINES_627D:
        raise RuntimeError(
            f"{name_627d} does not contain "
            f"{EXPECTED_ENGINES_627D} values."
        )

    if not np.isfinite(
        values_627d
    ).all():
        raise RuntimeError(
            f"{name_627d} contains non-finite values."
        )


# Existing CLEAN result from Step 6.27C
clean_prediction_627d = (
    np.asarray(
        CLEAN_UNCERTAINTY_RAW_627C[0],
        dtype=float,
    )
    .reshape(-1)
)

clean_mc_std_627d = (
    np.asarray(
        CLEAN_UNCERTAINTY_RAW_627C[1],
        dtype=float,
    )
    .reshape(-1)
)

clean_adjusted_uncertainty_627d = (
    np.asarray(
        CLEAN_UNCERTAINTY_RAW_627C[2],
        dtype=float,
    )
    .reshape(-1)
)


# ============================================================
# 10. Recover clean sensor-quality endpoints
# ============================================================

clean_quality_scored_627d = (
    SENSOR_QUALITY_VAL_CLEAN_SCORED_626D
    .copy()
)


required_quality_key_columns_627d = [
    "unit_id",
    "cycle",
    FUSION_COLUMN_627D,
]

missing_quality_columns_627d = [
    col_627d
    for col_627d
    in required_quality_key_columns_627d
    if col_627d
    not in clean_quality_scored_627d.columns
]

if missing_quality_columns_627d:
    raise RuntimeError(
        "Clean sensor-quality table is missing: "
        + str(
            missing_quality_columns_627d
        )
    )


clean_quality_endpoints_627d = (
    clean_quality_scored_627d
    .sort_values(
        ["unit_id", "cycle"]
    )
    .groupby(
        "unit_id",
        sort=True,
        as_index=False,
    )
    .tail(1)
    [
        [
            "unit_id",
            "cycle",
            FUSION_COLUMN_627D,
        ]
    ]
    .sort_values(
        "unit_id"
    )
    .reset_index(
        drop=True
    )
)


if len(
    clean_quality_endpoints_627d
) != EXPECTED_ENGINES_627D:
    raise RuntimeError(
        "Clean sensor-quality endpoint count "
        "is not 20."
    )


# ============================================================
# 11. Build and score corrupted sensor-quality windows
# ============================================================

print(
    "\n--- Building corrupted sensor-quality windows ---"
)


quality_builder_result_627d = (
    build_sensor_quality_candidates_626b(
        corrupted_raw_627d,
        SENSOR_QUALITY_SENSOR_COLUMNS_626B,
        SENSOR_QUALITY_REFERENCE_626B,
        window_length=(
            SENSOR_QUALITY_WINDOW_LENGTH_626B
        ),
    )
)


corrupted_quality_windows_627d = (
    extract_quality_window_dataframe_626e(
        quality_builder_result_627d,
        corrupted_raw_627d,
    )
)


corrupted_quality_scored_627d = (
    score_sensor_quality_windows_626d(
        corrupted_quality_windows_627d.copy()
    )
)


if not isinstance(
    corrupted_quality_scored_627d,
    pd.DataFrame,
):
    raise RuntimeError(
        "Sensor-quality scorer did not return "
        "a DataFrame."
    )


for col_627d in [
    "unit_id",
    "cycle",
    FUSION_COLUMN_627D,
]:
    if col_627d not in (
        corrupted_quality_scored_627d.columns
    ):
        raise RuntimeError(
            "Corrupted quality table is missing "
            f"column: {col_627d}"
        )


corrupted_quality_endpoints_627d = (
    corrupted_quality_scored_627d
    .sort_values(
        ["unit_id", "cycle"]
    )
    .groupby(
        "unit_id",
        sort=True,
        as_index=False,
    )
    .tail(1)
    [
        [
            "unit_id",
            "cycle",
            FUSION_COLUMN_627D,
        ]
    ]
    .sort_values(
        "unit_id"
    )
    .reset_index(
        drop=True
    )
)


if len(
    corrupted_quality_endpoints_627d
) != EXPECTED_ENGINES_627D:
    raise RuntimeError(
        "Corrupted sensor-quality endpoint count "
        "is not 20."
    )


# ============================================================
# 12. Exact endpoint-key alignment
# ============================================================

uncertainty_keys_627d = (
    clean_endpoint_meta_627d[
        ["unit_id", "cycle"]
    ]
    .reset_index(
        drop=True
    )
)


clean_quality_keys_627d = (
    clean_quality_endpoints_627d[
        ["unit_id", "cycle"]
    ]
    .reset_index(
        drop=True
    )
)


corrupted_quality_keys_627d = (
    corrupted_quality_endpoints_627d[
        ["unit_id", "cycle"]
    ]
    .reset_index(
        drop=True
    )
)


clean_quality_alignment_627d = bool(
    uncertainty_keys_627d.equals(
        clean_quality_keys_627d
    )
)

corrupted_quality_alignment_627d = bool(
    uncertainty_keys_627d.equals(
        corrupted_quality_keys_627d
    )
)


if not clean_quality_alignment_627d:
    raise RuntimeError(
        "Clean uncertainty and sensor-quality "
        "endpoint keys do not align."
    )

if not corrupted_quality_alignment_627d:
    raise RuntimeError(
        "Corrupted uncertainty and sensor-quality "
        "endpoint keys do not align."
    )


# ============================================================
# 13. Construct compact 20-engine smoke-test result
# ============================================================

SMOKE_ALIGNMENT_627D = (
    uncertainty_keys_627d
    .copy()
)


SMOKE_ALIGNMENT_627D[
    "clean_prediction"
] = clean_prediction_627d

SMOKE_ALIGNMENT_627D[
    "corrupted_prediction"
] = corrupted_prediction_627d

SMOKE_ALIGNMENT_627D[
    "clean_mc_std"
] = clean_mc_std_627d

SMOKE_ALIGNMENT_627D[
    "corrupted_mc_std"
] = corrupted_mc_std_627d

SMOKE_ALIGNMENT_627D[
    "clean_adjusted_uncertainty"
] = clean_adjusted_uncertainty_627d

SMOKE_ALIGNMENT_627D[
    "corrupted_adjusted_uncertainty"
] = corrupted_adjusted_uncertainty_627d

SMOKE_ALIGNMENT_627D[
    "clean_sensor_quality"
] = (
    clean_quality_endpoints_627d[
        FUSION_COLUMN_627D
    ]
    .to_numpy(
        dtype=float
    )
)

SMOKE_ALIGNMENT_627D[
    "corrupted_sensor_quality"
] = (
    corrupted_quality_endpoints_627d[
        FUSION_COLUMN_627D
    ]
    .to_numpy(
        dtype=float
    )
)


SMOKE_ALIGNMENT_627D[
    "prediction_change"
] = (
    SMOKE_ALIGNMENT_627D[
        "corrupted_prediction"
    ]
    -
    SMOKE_ALIGNMENT_627D[
        "clean_prediction"
    ]
)


SMOKE_ALIGNMENT_627D[
    "mc_std_change"
] = (
    SMOKE_ALIGNMENT_627D[
        "corrupted_mc_std"
    ]
    -
    SMOKE_ALIGNMENT_627D[
        "clean_mc_std"
    ]
)


SMOKE_ALIGNMENT_627D[
    "adjusted_uncertainty_change"
] = (
    SMOKE_ALIGNMENT_627D[
        "corrupted_adjusted_uncertainty"
    ]
    -
    SMOKE_ALIGNMENT_627D[
        "clean_adjusted_uncertainty"
    ]
)


SMOKE_ALIGNMENT_627D[
    "sensor_quality_change"
] = (
    SMOKE_ALIGNMENT_627D[
        "corrupted_sensor_quality"
    ]
    -
    SMOKE_ALIGNMENT_627D[
        "clean_sensor_quality"
    ]
)


all_smoke_values_finite_627d = bool(
    np.isfinite(
        SMOKE_ALIGNMENT_627D
        .select_dtypes(
            include=[
                np.number
            ]
        )
        .to_numpy(
            dtype=float
        )
    ).all()
)


# Correlation is diagnostic only.
x_627d = (
    SMOKE_ALIGNMENT_627D[
        "adjusted_uncertainty_change"
    ]
    .to_numpy(
        dtype=float
    )
)

y_627d = (
    SMOKE_ALIGNMENT_627D[
        "sensor_quality_change"
    ]
    .to_numpy(
        dtype=float
    )
)


if (
    np.std(
        x_627d
    ) > 0
    and
    np.std(
        y_627d
    ) > 0
):
    alignment_correlation_627d = float(
        np.corrcoef(
            x_627d,
            y_627d,
        )[0, 1]
    )
else:
    alignment_correlation_627d = np.nan


# ============================================================
# 14. Compact output only
# ============================================================

print(
    "\n--- First 5 aligned endpoint records ---"
)

display(
    SMOKE_ALIGNMENT_627D
    .head(5)
)


print(
    "\n"
    "============================================================\n"
    "STEP 6.27D FINAL SMOKE-TEST SUMMARY\n"
    "============================================================"
)

print(
    "Family                         :",
    family_627d
)

print(
    "Scenario ID                    :",
    scenario_id_627d
)

print(
    "Changed raw numeric cells      :",
    changed_raw_cells_627d
)

print(
    "Clean endpoint preprocessing OK:",
    clean_endpoint_equivalent_627d
)

print(
    "Endpoint tensor affected       :",
    endpoint_tensor_changed_627d
)

print(
    "Uncertainty endpoint rows      :",
    len(
        uncertainty_keys_627d
    )
)

print(
    "Clean quality endpoint rows    :",
    len(
        clean_quality_endpoints_627d
    )
)

print(
    "Corrupted quality endpoint rows:",
    len(
        corrupted_quality_endpoints_627d
    )
)

print(
    "Clean key alignment exact      :",
    clean_quality_alignment_627d
)

print(
    "Corrupted key alignment exact  :",
    corrupted_quality_alignment_627d
)

print(
    "All stored numeric values finite:",
    all_smoke_values_finite_627d
)

print(
    "Mean |prediction change|       :",
    float(
        SMOKE_ALIGNMENT_627D[
            "prediction_change"
        ]
        .abs()
        .mean()
    )
)

print(
    "Mean MC-STD change             :",
    float(
        SMOKE_ALIGNMENT_627D[
            "mc_std_change"
        ]
        .mean()
    )
)

print(
    "Mean adjusted-uncertainty change:",
    float(
        SMOKE_ALIGNMENT_627D[
            "adjusted_uncertainty_change"
        ]
        .mean()
    )
)

print(
    "Mean sensor-quality change     :",
    float(
        SMOKE_ALIGNMENT_627D[
            "sensor_quality_change"
        ]
        .mean()
    )
)

print(
    "Diagnostic delta correlation   :",
    alignment_correlation_627d
)

print(
    "\nOfficial TEST labels inspected : False"
)

print(
    "Official TEST scored           : False"
)

print(
    "Threshold selected             : False"
)

print(
    "Research decision changed      : False"
)

print(
    "Files written                  : False"
)

print(
    "\n=== END STEP 6.27D ==="
)

In [ ]:
# ============================================================
# STEP 6.27D1
# Exact frozen scenario-identifier audit
# ============================================================

import pandas as pd

print(
    "\n"
    "============================================================\n"
    "STEP 6.27D1 — EXACT FROZEN SCENARIO IDENTIFIER AUDIT\n"
    "============================================================"
)

print(
    "Available corruption families:",
    list(
        scenario_manifests_620gc.keys()
    )
)

for family_627d1 in (
    scenario_manifests_620gc.keys()
):

    manifest_627d1 = (
        scenario_manifests_620gc[
            family_627d1
        ]
    )

    print(
        "\n----------------------------------------"
    )
    print(
        "Family:",
        family_627d1
    )
    print(
        "Type:",
        type(
            manifest_627d1
        ).__name__
    )

    if not isinstance(
        manifest_627d1,
        pd.DataFrame,
    ):
        continue

    print(
        "Shape:",
        manifest_627d1.shape
    )

    print(
        "Columns:"
    )

    for idx_627d1, col_627d1 in enumerate(
        manifest_627d1.columns,
        start=1,
    ):
        print(
            f"{idx_627d1:02d}. "
            f"{col_627d1}"
        )

    print(
        "\nFirst scenario record:"
    )

    first_record_627d1 = (
        manifest_627d1
        .iloc[0]
        .to_dict()
    )

    for (
        key_627d1,
        value_627d1,
    ) in first_record_627d1.items():

        print(
            f"{key_627d1:<30}: "
            f"{value_627d1}"
        )

    # One family is sufficient because the
    # manifest schema should be shared.
    break


print(
    "\nOfficial TEST labels inspected : False"
)
print(
    "Official TEST scored           : False"
)
print(
    "Threshold selected             : False"
)
print(
    "Research decision changed      : False"
)
print(
    "Files written                  : False"
)

print(
    "\n=== END STEP 6.27D1 ==="
)

In [ ]:
# ============================================================
# STEP 6.27D2
# Freeze deterministic scenario identifiers
# ============================================================

import numpy as np
import pandas as pd

print(
    "\n"
    "============================================================\n"
    "STEP 6.27D2 — FREEZE DETERMINISTIC SCENARIO IDENTIFIERS\n"
    "============================================================"
)

# ------------------------------------------------------------
# 1. Frozen expected family order and counts
# ------------------------------------------------------------

EXPECTED_FAMILY_ORDER_627D2 = [
    "bias",
    "noise",
    "drift",
    "dropout",
    "stuck",
]

EXPECTED_FAMILY_COUNTS_627D2 = {
    "bias": 90,
    "noise": 45,
    "drift": 90,
    "dropout": 45,
    "stuck": 45,
}


# ------------------------------------------------------------
# 2. Verify family set
# ------------------------------------------------------------

actual_families_627d2 = list(
    scenario_manifests_620gc.keys()
)

print(
    "Actual family order   :",
    actual_families_627d2,
)

print(
    "Expected family order :",
    EXPECTED_FAMILY_ORDER_627D2,
)

family_set_exact_627d2 = (
    set(actual_families_627d2)
    ==
    set(EXPECTED_FAMILY_ORDER_627D2)
)

print(
    "Family set exact      :",
    family_set_exact_627d2,
)

if not family_set_exact_627d2:
    raise RuntimeError(
        "Frozen corruption-family set does not match "
        "the expected five-family definition."
    )


# ------------------------------------------------------------
# 3. Build a metadata-only scenario index
# ------------------------------------------------------------

scenario_index_records_627d2 = []

global_index_627d2 = 0

for family_627d2 in EXPECTED_FAMILY_ORDER_627D2:

    manifest_627d2 = (
        scenario_manifests_620gc[
            family_627d2
        ]
    )

    if not isinstance(
        manifest_627d2,
        pd.DataFrame,
    ):
        raise TypeError(
            f"Manifest for {family_627d2} is not "
            "a pandas DataFrame."
        )

    expected_count_627d2 = (
        EXPECTED_FAMILY_COUNTS_627D2[
            family_627d2
        ]
    )

    actual_count_627d2 = len(
        manifest_627d2
    )

    if (
        actual_count_627d2
        !=
        expected_count_627d2
    ):
        raise RuntimeError(
            f"{family_627d2}: expected "
            f"{expected_count_627d2} frozen scenarios, "
            f"found {actual_count_627d2}."
        )

    for family_position_627d2 in range(
        actual_count_627d2
    ):

        scenario_id_627d2 = (
            f"{family_627d2}_"
            f"{family_position_627d2 + 1:03d}"
        )

        row_627d2 = (
            manifest_627d2
            .iloc[
                family_position_627d2
            ]
        )

        scenario_index_records_627d2.append(
            {
                "global_index":
                    global_index_627d2,

                "family":
                    family_627d2,

                "family_position":
                    family_position_627d2,

                "scenario_id":
                    scenario_id_627d2,

                "sensor":
                    row_627d2.get(
                        "sensor",
                        None,
                    ),

                "severity":
                    row_627d2.get(
                        "severity",
                        None,
                    ),
            }
        )

        global_index_627d2 += 1


SCENARIO_INDEX_627D2 = pd.DataFrame(
    scenario_index_records_627d2
)


# ------------------------------------------------------------
# 4. Integrity checks
# ------------------------------------------------------------

expected_total_627d2 = sum(
    EXPECTED_FAMILY_COUNTS_627D2.values()
)

row_count_exact_627d2 = (
    len(
        SCENARIO_INDEX_627D2
    )
    ==
    expected_total_627d2
)

scenario_ids_unique_627d2 = (
    SCENARIO_INDEX_627D2[
        "scenario_id"
    ]
    .is_unique
)

global_indices_exact_627d2 = (
    np.array_equal(
        SCENARIO_INDEX_627D2[
            "global_index"
        ].to_numpy(),
        np.arange(
            expected_total_627d2
        ),
    )
)

family_positions_exact_627d2 = True

for family_627d2 in EXPECTED_FAMILY_ORDER_627D2:

    subset_627d2 = (
        SCENARIO_INDEX_627D2[
            SCENARIO_INDEX_627D2[
                "family"
            ]
            ==
            family_627d2
        ]
    )

    expected_positions_627d2 = np.arange(
        EXPECTED_FAMILY_COUNTS_627D2[
            family_627d2
        ]
    )

    observed_positions_627d2 = (
        subset_627d2[
            "family_position"
        ]
        .to_numpy()
    )

    if not np.array_equal(
        observed_positions_627d2,
        expected_positions_627d2,
    ):
        family_positions_exact_627d2 = False
        break


# ------------------------------------------------------------
# 5. Compact display
# ------------------------------------------------------------

print(
    "\n--- Family counts ---"
)

print(
    SCENARIO_INDEX_627D2[
        "family"
    ]
    .value_counts(
        sort=False
    )
    .reindex(
        EXPECTED_FAMILY_ORDER_627D2
    )
    .to_string()
)

print(
    "\n--- First two and last two identifiers per family ---"
)

for family_627d2 in EXPECTED_FAMILY_ORDER_627D2:

    subset_627d2 = (
        SCENARIO_INDEX_627D2[
            SCENARIO_INDEX_627D2[
                "family"
            ]
            ==
            family_627d2
        ]
    )

    ids_627d2 = (
        subset_627d2[
            "scenario_id"
        ]
        .tolist()
    )

    print(
        f"{family_627d2:<8}: "
        f"{ids_627d2[:2]} ... "
        f"{ids_627d2[-2:]}"
    )


# ------------------------------------------------------------
# 6. Final safety checks
# ------------------------------------------------------------

if not row_count_exact_627d2:
    raise RuntimeError(
        "Scenario-index row count is incorrect."
    )

if not scenario_ids_unique_627d2:
    raise RuntimeError(
        "Generated scenario identifiers are not unique."
    )

if not global_indices_exact_627d2:
    raise RuntimeError(
        "Global scenario indices are not contiguous."
    )

if not family_positions_exact_627d2:
    raise RuntimeError(
        "Family positions are not contiguous."
    )


print(
    "\n"
    "============================================================\n"
    "STEP 6.27D2 SCENARIO-IDENTIFIER SUMMARY\n"
    "============================================================"
)

print(
    "Total frozen scenarios       :",
    len(
        SCENARIO_INDEX_627D2
    ),
)

print(
    "Expected frozen scenarios    :",
    expected_total_627d2,
)

print(
    "Scenario count exact         :",
    row_count_exact_627d2,
)

print(
    "Scenario IDs unique          :",
    scenario_ids_unique_627d2,
)

print(
    "Global indices exact         :",
    global_indices_exact_627d2,
)

print(
    "Family positions exact       :",
    family_positions_exact_627d2,
)

print(
    "Official TEST labels inspected : False"
)

print(
    "Official TEST scored           : False"
)

print(
    "Threshold selected             : False"
)

print(
    "Research decision changed      : False"
)

print(
    "Files written                  : False"
)

print(
    "\nScenario identifiers are metadata only."
)

print(
    "Frozen corruption manifests were NOT modified."
)

print(
    "\n=== END STEP 6.27D2 ==="
)

In [ ]:
# ================================================================
# STEP 6.27E
# FULL FROZEN-DEV UNCERTAINTY + SENSOR-QUALITY ENDPOINT ALIGNMENT
# ================================================================

import time
import warnings
import numpy as np
import pandas as pd

print(
    "\n"
    "=================================================================\n"
    "STEP 6.27E — FULL DEV UNCERTAINTY + SENSOR-QUALITY ALIGNMENT\n"
    "================================================================="
)

STEP_627E_START = time.time()


# ================================================================
# 1. REQUIRED FROZEN OBJECTS
# ================================================================

required_objects_627e = [
    "val_raw_base_620gc",
    "scenario_manifests_620gc",
    "_apply_dev_corruption_627a",
    "PRIMARY_INPUT_COLUMNS",
    "scaler",
    "VALIDATION_X_REBUILT_627BR2",
    "CLEAN_UNCERTAINTY_RAW_627C",
    "run_frozen_uncertainty_625e1",

    "build_sensor_quality_candidates_626b",
    "extract_quality_window_dataframe_626e",
    "score_sensor_quality_windows_626d",

    "SENSOR_QUALITY_SENSOR_COLUMNS_626B",
    "SENSOR_QUALITY_REFERENCE_626B",
    "SENSOR_QUALITY_WINDOW_LENGTH_626B",
    "SENSOR_QUALITY_VAL_CLEAN_SCORED_626D",
]

print("\n--- Required frozen objects ---")

missing_627e = []

for name_627e in required_objects_627e:
    exists_627e = name_627e in globals()

    print(
        f"{name_627e:<45}: "
        f"{exists_627e}"
    )

    if not exists_627e:
        missing_627e.append(name_627e)

if missing_627e:
    raise RuntimeError(
        "STEP 6.27E cannot continue. "
        "Missing required objects: "
        + str(missing_627e)
    )


# ================================================================
# 2. FIXED RESEARCH CONTRACT
# ================================================================

FAMILY_ORDER_627E = [
    "bias",
    "noise",
    "drift",
    "dropout",
    "stuck",
]

EXPECTED_FAMILY_COUNTS_627E = {
    "bias": 90,
    "noise": 45,
    "drift": 90,
    "dropout": 45,
    "stuck": 45,
}

EXPECTED_SCENARIOS_627E = 315
EXPECTED_DEV_ENGINES_627E = 20
EXPECTED_ENDPOINT_ROWS_627E = (
    EXPECTED_SCENARIOS_627E
    * EXPECTED_DEV_ENGINES_627E
)

WINDOW_LENGTH_627E = int(
    SENSOR_QUALITY_WINDOW_LENGTH_626B
)

FUSION_COLUMN_627E = (
    "sq_candidate_max_fusion"
)

print(
    "\n--- Frozen alignment contract ---"
)

print(
    "Family order       :",
    FAMILY_ORDER_627E,
)

print(
    "Expected scenarios :",
    EXPECTED_SCENARIOS_627E,
)

print(
    "Expected engines   :",
    EXPECTED_DEV_ENGINES_627E,
)

print(
    "Expected rows      :",
    EXPECTED_ENDPOINT_ROWS_627E,
)

print(
    "Window length      :",
    WINDOW_LENGTH_627E,
)

print(
    "Sensor-quality fusion:",
    FUSION_COLUMN_627E,
)


# ================================================================
# 3. VERIFY MANIFEST STRUCTURE
# ================================================================

print(
    "\n--- Frozen manifest verification ---"
)

actual_family_set_627e = set(
    scenario_manifests_620gc.keys()
)

if actual_family_set_627e != set(
    FAMILY_ORDER_627E
):
    raise RuntimeError(
        "Frozen corruption-family set changed."
    )

scenario_total_627e = 0

for family_627e in FAMILY_ORDER_627E:

    manifest_627e = (
        scenario_manifests_620gc[
            family_627e
        ]
    )

    if not isinstance(
        manifest_627e,
        pd.DataFrame,
    ):
        raise RuntimeError(
            f"Manifest for {family_627e} "
            "is not a DataFrame."
        )

    count_627e = len(
        manifest_627e
    )

    expected_count_627e = (
        EXPECTED_FAMILY_COUNTS_627E[
            family_627e
        ]
    )

    print(
        f"{family_627e:<10}: "
        f"{count_627e}"
    )

    if count_627e != expected_count_627e:
        raise RuntimeError(
            f"{family_627e} scenario count "
            "does not match frozen count."
        )

    scenario_total_627e += (
        count_627e
    )

if scenario_total_627e != (
    EXPECTED_SCENARIOS_627E
):
    raise RuntimeError(
        "Total frozen scenario count "
        "is not 315."
    )


# ================================================================
# 4. VALIDATION RAW-DATA STRUCTURAL CONTRACT
# ================================================================

VAL_BASE_627E = (
    val_raw_base_620gc
    .copy()
)

required_key_columns_627e = [
    "unit_id",
    "cycle",
]

missing_key_columns_627e = [
    c
    for c in required_key_columns_627e
    if c not in VAL_BASE_627E.columns
]

if missing_key_columns_627e:
    raise RuntimeError(
        "Validation raw base is missing "
        "required key columns: "
        + str(missing_key_columns_627e)
    )

missing_primary_columns_627e = [
    c
    for c in PRIMARY_INPUT_COLUMNS
    if c not in VAL_BASE_627E.columns
]

if missing_primary_columns_627e:
    raise RuntimeError(
        "Validation raw base is missing "
        "primary model-input columns: "
        + str(
            missing_primary_columns_627e
        )
    )

duplicate_keys_627e = int(
    VAL_BASE_627E.duplicated(
        subset=[
            "unit_id",
            "cycle",
        ]
    ).sum()
)

if duplicate_keys_627e != 0:
    raise RuntimeError(
        "Validation raw base contains "
        "duplicate unit/cycle keys."
    )

DEV_UNIT_IDS_627E = (
    VAL_BASE_627E[
        "unit_id"
    ]
    .drop_duplicates()
    .sort_values()
    .tolist()
)

if len(
    DEV_UNIT_IDS_627E
) != EXPECTED_DEV_ENGINES_627E:
    raise RuntimeError(
        "Validation engine count is not 20."
    )

DEV_ENDPOINT_KEYS_627E = (
    VAL_BASE_627E
    .groupby(
        "unit_id",
        sort=True,
        as_index=False,
    )["cycle"]
    .max()
    .sort_values(
        "unit_id"
    )
    .reset_index(
        drop=True
    )
)

if len(
    DEV_ENDPOINT_KEYS_627E
) != EXPECTED_DEV_ENGINES_627E:
    raise RuntimeError(
        "Could not construct exactly "
        "20 DEV endpoint keys."
    )

print(
    "\n--- Validation structure ---"
)

print(
    "Raw rows          :",
    len(
        VAL_BASE_627E
    ),
)

print(
    "Validation engines:",
    len(
        DEV_UNIT_IDS_627E
    ),
)

print(
    "Duplicate keys    :",
    duplicate_keys_627e,
)

print(
    "Endpoint keys     :",
    len(
        DEV_ENDPOINT_KEYS_627E
    ),
)


# ================================================================
# 5. ENDPOINT-SEQUENCE BUILDER
#
# This deliberately reproduces the already-audited validation
# preprocessing:
#
# raw DEV
#   -> frozen scaler
#   -> float32 storage convention
#   -> last 30 cycles of every DEV engine
#
# NO TEST data are involved.
# ================================================================

def build_dev_endpoint_tensor_627e(
    raw_df_627e,
):
    """
    Build one endpoint sequence per DEV engine using the
    frozen model-input columns and frozen scaler.

    Returns
    -------
    endpoint_tensor : ndarray
        Shape = (20, 30, 17)

    endpoint_keys : DataFrame
        unit_id, cycle
    """

    if not isinstance(
        raw_df_627e,
        pd.DataFrame,
    ):
        raise TypeError(
            "raw_df_627e must be a DataFrame."
        )

    if list(
        raw_df_627e.columns
    ) != list(
        VAL_BASE_627E.columns
    ):
        raise RuntimeError(
            "DEV raw column order changed."
        )

    if len(
        raw_df_627e
    ) != len(
        VAL_BASE_627E
    ):
        raise RuntimeError(
            "DEV raw row count changed."
        )

    if int(
        raw_df_627e.duplicated(
            subset=[
                "unit_id",
                "cycle",
            ]
        ).sum()
    ) != 0:
        raise RuntimeError(
            "Duplicate unit/cycle keys found "
            "while building endpoint tensor."
        )

    working_627e = (
        raw_df_627e
        .sort_values(
            [
                "unit_id",
                "cycle",
            ]
        )
        .reset_index(
            drop=True
        )
    )

    primary_frame_627e = (
        working_627e[
            PRIMARY_INPUT_COLUMNS
        ]
    )

    with warnings.catch_warnings():
        warnings.simplefilter(
            "ignore",
            category=UserWarning,
        )

        scaled_rows_627e = (
            scaler.transform(
                primary_frame_627e
            )
        )

    scaled_rows_627e = (
        np.asarray(
            scaled_rows_627e,
            dtype=np.float32,
        )
    )

    if scaled_rows_627e.shape != (
        len(
            working_627e
        ),
        len(
            PRIMARY_INPUT_COLUMNS
        ),
    ):
        raise RuntimeError(
            "Unexpected scaled DEV row shape."
        )

    if not np.isfinite(
        scaled_rows_627e
    ).all():
        raise RuntimeError(
            "Non-finite scaled DEV values found."
        )

    scaled_df_627e = pd.DataFrame(
        scaled_rows_627e,
        columns=PRIMARY_INPUT_COLUMNS,
    )

    scaled_df_627e.insert(
        0,
        "cycle",
        working_627e[
            "cycle"
        ].to_numpy(),
    )

    scaled_df_627e.insert(
        0,
        "unit_id",
        working_627e[
            "unit_id"
        ].to_numpy(),
    )

    endpoint_arrays_627e = []
    endpoint_rows_627e = []

    for unit_id_627e, group_627e in (
        scaled_df_627e
        .groupby(
            "unit_id",
            sort=True,
        )
    ):

        group_627e = (
            group_627e
            .sort_values(
                "cycle"
            )
            .reset_index(
                drop=True
            )
        )

        if len(
            group_627e
        ) < WINDOW_LENGTH_627E:
            raise RuntimeError(
                f"DEV engine {unit_id_627e} "
                "has fewer than 30 cycles."
            )

        tail_627e = (
            group_627e
            .tail(
                WINDOW_LENGTH_627E
            )
        )

        sequence_627e = (
            tail_627e[
                PRIMARY_INPUT_COLUMNS
            ]
            .to_numpy(
                dtype=np.float32
            )
        )

        if sequence_627e.shape != (
            WINDOW_LENGTH_627E,
            len(
                PRIMARY_INPUT_COLUMNS
            ),
        ):
            raise RuntimeError(
                "Endpoint sequence shape "
                "is incorrect."
            )

        endpoint_arrays_627e.append(
            sequence_627e
        )

        endpoint_rows_627e.append(
            {
                "unit_id": unit_id_627e,
                "cycle": int(
                    group_627e[
                        "cycle"
                    ].iloc[-1]
                ),
            }
        )

    endpoint_tensor_627e = np.stack(
        endpoint_arrays_627e,
        axis=0,
    ).astype(
        np.float32,
        copy=False,
    )

    endpoint_keys_627e = pd.DataFrame(
        endpoint_rows_627e
    )

    if endpoint_tensor_627e.shape != (
        EXPECTED_DEV_ENGINES_627E,
        WINDOW_LENGTH_627E,
        len(
            PRIMARY_INPUT_COLUMNS
        ),
    ):
        raise RuntimeError(
            "Final DEV endpoint tensor "
            "does not have expected shape."
        )

    if len(
        endpoint_keys_627e
    ) != EXPECTED_DEV_ENGINES_627E:
        raise RuntimeError(
            "Endpoint key count is not 20."
        )

    if not np.isfinite(
        endpoint_tensor_627e
    ).all():
        raise RuntimeError(
            "Endpoint tensor contains "
            "non-finite values."
        )

    return (
        endpoint_tensor_627e,
        endpoint_keys_627e,
    )


# ================================================================
# 6. CLEAN PREPROCESSING CLOSURE
# ================================================================

(
    CLEAN_ENDPOINT_TENSOR_627E,
    CLEAN_ENDPOINT_KEYS_627E,
) = build_dev_endpoint_tensor_627e(
    VAL_BASE_627E
)

print(
    "\n--- Clean endpoint preprocessing closure ---"
)

print(
    "Endpoint tensor:",
    CLEAN_ENDPOINT_TENSOR_627E.shape,
)

print(
    "Endpoint keys  :",
    CLEAN_ENDPOINT_KEYS_627E.shape,
)

clean_key_exact_627e = (
    CLEAN_ENDPOINT_KEYS_627E[
        [
            "unit_id",
            "cycle",
        ]
    ]
    .reset_index(
        drop=True
    )
    .equals(
        DEV_ENDPOINT_KEYS_627E[
            [
                "unit_id",
                "cycle",
            ]
        ]
        .reset_index(
            drop=True
        )
    )
)

print(
    "Endpoint keys exact:",
    clean_key_exact_627e,
)

if not clean_key_exact_627e:
    raise RuntimeError(
        "Clean endpoint keys are not exact."
    )


# ================================================================
# 7. VERIFY CLEAN ENDPOINT VALUES AGAINST THE ALREADY-AUDITED
#    VALIDATION RECONSTRUCTION
#
# VALIDATION_X_REBUILT_627BR2 contains all rolling validation
# windows. We deterministically recover the final window of each
# DEV engine and compare it with the direct endpoint builder.
# ================================================================

validation_group_sizes_627e = (
    VAL_BASE_627E
    .sort_values(
        [
            "unit_id",
            "cycle",
        ]
    )
    .groupby(
        "unit_id",
        sort=True,
    )
    .size()
)

endpoint_window_indices_627e = []

running_windows_627e = 0

for unit_id_627e in (
    validation_group_sizes_627e.index
):

    n_rows_627e = int(
        validation_group_sizes_627e.loc[
            unit_id_627e
        ]
    )

    n_windows_627e = (
        n_rows_627e
        - WINDOW_LENGTH_627E
        + 1
    )

    if n_windows_627e <= 0:
        raise RuntimeError(
            "A DEV engine has no eligible "
            "30-cycle window."
        )

    running_windows_627e += (
        n_windows_627e
    )

    endpoint_window_indices_627e.append(
        running_windows_627e - 1
    )

if running_windows_627e != len(
    VALIDATION_X_REBUILT_627BR2
):
    raise RuntimeError(
        "Rolling-window count does not match "
        "the audited validation reconstruction."
    )

REFERENCE_ENDPOINT_TENSOR_627E = (
    np.asarray(
        VALIDATION_X_REBUILT_627BR2[
            endpoint_window_indices_627e
        ],
        dtype=np.float32,
    )
)

if REFERENCE_ENDPOINT_TENSOR_627E.shape != (
    CLEAN_ENDPOINT_TENSOR_627E.shape
):
    raise RuntimeError(
        "Reference endpoint tensor shape mismatch."
    )

clean_endpoint_max_diff_627e = float(
    np.max(
        np.abs(
            CLEAN_ENDPOINT_TENSOR_627E.astype(
                np.float64
            )
            -
            REFERENCE_ENDPOINT_TENSOR_627E.astype(
                np.float64
            )
        )
    )
)

clean_endpoint_exact_627e = bool(
    np.array_equal(
        CLEAN_ENDPOINT_TENSOR_627E,
        REFERENCE_ENDPOINT_TENSOR_627E,
    )
)

clean_endpoint_close_627e = bool(
    np.allclose(
        CLEAN_ENDPOINT_TENSOR_627E,
        REFERENCE_ENDPOINT_TENSOR_627E,
        rtol=0.0,
        atol=5e-7,
    )
)

print(
    "Stored-dtype exact      :",
    clean_endpoint_exact_627e,
)

print(
    "Maximum endpoint diff   :",
    clean_endpoint_max_diff_627e,
)

print(
    "Equivalent within 5e-7  :",
    clean_endpoint_close_627e,
)

if not clean_endpoint_close_627e:
    raise RuntimeError(
        "Clean DEV endpoint preprocessing "
        "does not reproduce the audited "
        "validation state."
    )


# ================================================================
# 8. RECOVER FROZEN CLEAN UNCERTAINTY
#
# Exact return contract already established in Step 6.27C3:
#
# (
#   deterministic_prediction,
#   mc_std_values,
#   adjusted_uncertainty
# )
# ================================================================

if not isinstance(
    CLEAN_UNCERTAINTY_RAW_627C,
    (tuple, list),
):
    raise RuntimeError(
        "CLEAN_UNCERTAINTY_RAW_627C "
        "is not tuple/list."
    )

if len(
    CLEAN_UNCERTAINTY_RAW_627C
) != 3:
    raise RuntimeError(
        "Frozen clean uncertainty return "
        "does not contain exactly 3 components."
    )

clean_prediction_627e = np.asarray(
    CLEAN_UNCERTAINTY_RAW_627C[0],
    dtype=np.float64,
).reshape(-1)

clean_mc_std_627e = np.asarray(
    CLEAN_UNCERTAINTY_RAW_627C[1],
    dtype=np.float64,
).reshape(-1)

clean_adjusted_uncertainty_627e = np.asarray(
    CLEAN_UNCERTAINTY_RAW_627C[2],
    dtype=np.float64,
).reshape(-1)

for label_627e, arr_627e in [
    (
        "clean_prediction",
        clean_prediction_627e,
    ),
    (
        "clean_mc_std",
        clean_mc_std_627e,
    ),
    (
        "clean_adjusted_uncertainty",
        clean_adjusted_uncertainty_627e,
    ),
]:

    if len(
        arr_627e
    ) != EXPECTED_DEV_ENGINES_627E:
        raise RuntimeError(
            f"{label_627e} does not contain "
            "20 endpoint values."
        )

    if not np.isfinite(
        arr_627e
    ).all():
        raise RuntimeError(
            f"{label_627e} contains "
            "non-finite values."
        )

print(
    "\n--- Frozen clean uncertainty ---"
)

print(
    "Prediction rows           :",
    len(
        clean_prediction_627e
    ),
)

print(
    "MC-STD rows               :",
    len(
        clean_mc_std_627e
    ),
)

print(
    "Adjusted-uncertainty rows :",
    len(
        clean_adjusted_uncertainty_627e
    ),
)


# ================================================================
# 9. RECOVER FROZEN CLEAN SENSOR-QUALITY ENDPOINTS
# ================================================================

clean_quality_scored_627e = (
    SENSOR_QUALITY_VAL_CLEAN_SCORED_626D
    .copy()
)

for required_col_627e in [
    "unit_id",
    "cycle",
    FUSION_COLUMN_627E,
]:
    if required_col_627e not in (
        clean_quality_scored_627e.columns
    ):
        raise RuntimeError(
            "Frozen clean quality table is "
            f"missing {required_col_627e}."
        )

clean_quality_endpoints_627e = (
    clean_quality_scored_627e
    .sort_values(
        [
            "unit_id",
            "cycle",
        ]
    )
    .groupby(
        "unit_id",
        sort=True,
        as_index=False,
    )
    .tail(1)
    .sort_values(
        "unit_id"
    )
    .reset_index(
        drop=True
    )
)

if len(
    clean_quality_endpoints_627e
) != EXPECTED_DEV_ENGINES_627E:
    raise RuntimeError(
        "Clean quality table did not produce "
        "exactly 20 endpoint rows."
    )

clean_quality_key_exact_627e = (
    clean_quality_endpoints_627e[
        [
            "unit_id",
            "cycle",
        ]
    ]
    .reset_index(
        drop=True
    )
    .equals(
        DEV_ENDPOINT_KEYS_627E[
            [
                "unit_id",
                "cycle",
            ]
        ]
        .reset_index(
            drop=True
        )
    )
)

if not clean_quality_key_exact_627e:
    raise RuntimeError(
        "Clean sensor-quality endpoint keys "
        "do not match DEV uncertainty keys."
    )

clean_anomaly_627e = pd.to_numeric(
    clean_quality_endpoints_627e[
        FUSION_COLUMN_627E
    ],
    errors="raise",
).to_numpy(
    dtype=np.float64
)

if not np.isfinite(
    clean_anomaly_627e
).all():
    raise RuntimeError(
        "Clean sensor-quality anomaly values "
        "contain non-finite values."
    )

if (
    np.min(
        clean_anomaly_627e
    ) < -1e-12
    or
    np.max(
        clean_anomaly_627e
    ) > 1.0 + 1e-12
):
    raise RuntimeError(
        "Frozen sensor-quality anomaly score "
        "is outside [0, 1]."
    )

# Frozen interpretation:
# anomaly = higher is worse
# quality = higher is better
clean_sensor_quality_627e = (
    1.0
    -
    np.clip(
        clean_anomaly_627e,
        0.0,
        1.0,
    )
)

print(
    "\n--- Frozen clean sensor-quality endpoints ---"
)

print(
    "Endpoint rows:",
    len(
        clean_sensor_quality_627e
    ),
)

print(
    "Minimum quality:",
    float(
        np.min(
            clean_sensor_quality_627e
        )
    ),
)

print(
    "Mean quality:",
    float(
        np.mean(
            clean_sensor_quality_627e
        )
    ),
)

print(
    "Maximum quality:",
    float(
        np.max(
            clean_sensor_quality_627e
        )
    ),
)


# ================================================================
# 10. CORRUPTED SENSOR-QUALITY ENDPOINT HELPER
# ================================================================

def corrupted_sensor_quality_endpoints_627e(
    corrupted_raw_627e,
):
    """
    Run the already-established sensor-quality construction and
    return the 20 final DEV-engine endpoint quality values.
    """

    builder_result_627e = (
        build_sensor_quality_candidates_626b(
            corrupted_raw_627e,
            SENSOR_QUALITY_SENSOR_COLUMNS_626B,
            SENSOR_QUALITY_REFERENCE_626B,
            WINDOW_LENGTH_627E,
        )
    )

    window_df_627e = (
        extract_quality_window_dataframe_626e(
            builder_result_627e,
            corrupted_raw_627e,
        )
    )

    scored_df_627e = (
        score_sensor_quality_windows_626d(
            window_df_627e
        )
    )

    if not isinstance(
        scored_df_627e,
        pd.DataFrame,
    ):
        raise RuntimeError(
            "Sensor-quality scorer did not "
            "return a DataFrame."
        )

    for required_col_627e in [
        "unit_id",
        "cycle",
        FUSION_COLUMN_627E,
    ]:
        if required_col_627e not in (
            scored_df_627e.columns
        ):
            raise RuntimeError(
                "Corrupted sensor-quality table "
                f"is missing {required_col_627e}."
            )

    endpoint_df_627e = (
        scored_df_627e
        .sort_values(
            [
                "unit_id",
                "cycle",
            ]
        )
        .groupby(
            "unit_id",
            sort=True,
            as_index=False,
        )
        .tail(1)
        .sort_values(
            "unit_id"
        )
        .reset_index(
            drop=True
        )
    )

    if len(
        endpoint_df_627e
    ) != EXPECTED_DEV_ENGINES_627E:
        raise RuntimeError(
            "Corrupted quality scoring did not "
            "produce 20 endpoint rows."
        )

    key_match_627e = (
        endpoint_df_627e[
            [
                "unit_id",
                "cycle",
            ]
        ]
        .reset_index(
            drop=True
        )
        .equals(
            DEV_ENDPOINT_KEYS_627E[
                [
                    "unit_id",
                    "cycle",
                ]
            ]
            .reset_index(
                drop=True
            )
        )
    )

    if not key_match_627e:
        raise RuntimeError(
            "Corrupted sensor-quality endpoint "
            "keys changed."
        )

    anomaly_627e = pd.to_numeric(
        endpoint_df_627e[
            FUSION_COLUMN_627E
        ],
        errors="raise",
    ).to_numpy(
        dtype=np.float64
    )

    if not np.isfinite(
        anomaly_627e
    ).all():
        raise RuntimeError(
            "Corrupted sensor-quality values "
            "contain non-finite values."
        )

    if (
        np.min(
            anomaly_627e
        ) < -1e-12
        or
        np.max(
            anomaly_627e
        ) > 1.0 + 1e-12
    ):
        raise RuntimeError(
            "Corrupted anomaly values are "
            "outside [0, 1]."
        )

    quality_627e = (
        1.0
        -
        np.clip(
            anomaly_627e,
            0.0,
            1.0,
        )
    )

    return (
        endpoint_df_627e,
        quality_627e,
    )


# ================================================================
# 11. FROZEN UNCERTAINTY OUTPUT HELPER
# ================================================================

def run_uncertainty_checked_627e(
    endpoint_tensor_627e,
):
    """
    Execute the already-frozen uncertainty pipeline and enforce
    its previously-confirmed return contract.
    """

    raw_output_627e = (
        run_frozen_uncertainty_625e1(
            endpoint_tensor_627e
        )
    )

    if not isinstance(
        raw_output_627e,
        (tuple, list),
    ):
        raise RuntimeError(
            "Frozen uncertainty function "
            "did not return tuple/list."
        )

    if len(
        raw_output_627e
    ) != 3:
        raise RuntimeError(
            "Frozen uncertainty function "
            "did not return exactly 3 components."
        )

    prediction_627e = np.asarray(
        raw_output_627e[0],
        dtype=np.float64,
    ).reshape(-1)

    mc_std_627e = np.asarray(
        raw_output_627e[1],
        dtype=np.float64,
    ).reshape(-1)

    adjusted_627e = np.asarray(
        raw_output_627e[2],
        dtype=np.float64,
    ).reshape(-1)

    for label_627e, arr_627e in [
        (
            "prediction",
            prediction_627e,
        ),
        (
            "mc_std",
            mc_std_627e,
        ),
        (
            "adjusted_uncertainty",
            adjusted_627e,
        ),
    ]:

        if len(
            arr_627e
        ) != EXPECTED_DEV_ENGINES_627E:
            raise RuntimeError(
                f"Corrupted {label_627e} "
                "does not contain 20 values."
            )

        if not np.isfinite(
            arr_627e
        ).all():
            raise RuntimeError(
                f"Corrupted {label_627e} "
                "contains non-finite values."
            )

    return (
        prediction_627e,
        mc_std_627e,
        adjusted_627e,
    )


# ================================================================
# 12. FULL 315-SCENARIO DEV RUN
# ================================================================

print(
    "\n"
    "--- Running full 315-scenario DEV alignment ---"
)

records_627e = []
scenario_summary_records_627e = []

global_index_627e = 0

original_base_snapshot_627e = (
    VAL_BASE_627E.copy(
        deep=True
    )
)

for family_627e in (
    FAMILY_ORDER_627E
):

    manifest_627e = (
        scenario_manifests_620gc[
            family_627e
        ]
    )

    print(
        "\nRunning "
        f"{family_627e.upper()} "
        f"({len(manifest_627e)} scenarios)..."
    )

    family_start_627e = time.time()

    for family_position_627e in range(
        len(
            manifest_627e
        )
    ):

        scenario_row_627e = (
            manifest_627e
            .iloc[
                family_position_627e
            ]
        )

        scenario_id_627e = (
            f"{family_627e}_"
            f"{family_position_627e + 1:03d}"
        )

        # --------------------------------------------------------
        # Apply already-frozen corruption
        # --------------------------------------------------------

        corrupted_raw_627e = (
            _apply_dev_corruption_627a(
                VAL_BASE_627E,
                family_627e,
                scenario_row_627e,
            )
        )

        if not isinstance(
            corrupted_raw_627e,
            pd.DataFrame,
        ):
            raise RuntimeError(
                f"{scenario_id_627e}: "
                "corruption function did not "
                "return a DataFrame."
            )

        if corrupted_raw_627e.shape != (
            VAL_BASE_627E.shape
        ):
            raise RuntimeError(
                f"{scenario_id_627e}: "
                "raw shape changed."
            )

        if list(
            corrupted_raw_627e.columns
        ) != list(
            VAL_BASE_627E.columns
        ):
            raise RuntimeError(
                f"{scenario_id_627e}: "
                "raw column order changed."
            )

        corrupted_key_exact_627e = (
            corrupted_raw_627e[
                [
                    "unit_id",
                    "cycle",
                ]
            ]
            .reset_index(
                drop=True
            )
            .equals(
                VAL_BASE_627E[
                    [
                        "unit_id",
                        "cycle",
                    ]
                ]
                .reset_index(
                    drop=True
                )
            )
        )

        if not corrupted_key_exact_627e:
            raise RuntimeError(
                f"{scenario_id_627e}: "
                "unit/cycle keys changed."
            )

        # --------------------------------------------------------
        # Count changed primary numeric cells
        # --------------------------------------------------------

        clean_primary_627e = (
            VAL_BASE_627E[
                PRIMARY_INPUT_COLUMNS
            ]
            .to_numpy(
                dtype=np.float64
            )
        )

        corrupted_primary_627e = (
            corrupted_raw_627e[
                PRIMARY_INPUT_COLUMNS
            ]
            .to_numpy(
                dtype=np.float64
            )
        )

        changed_primary_cells_627e = int(
            np.count_nonzero(
                ~np.isclose(
                    clean_primary_627e,
                    corrupted_primary_627e,
                    rtol=0.0,
                    atol=0.0,
                    equal_nan=True,
                )
            )
        )

        # --------------------------------------------------------
        # Build corrupted model endpoint tensor
        # --------------------------------------------------------

        (
            corrupted_endpoint_tensor_627e,
            corrupted_endpoint_keys_627e,
        ) = (
            build_dev_endpoint_tensor_627e(
                corrupted_raw_627e
            )
        )

        if not (
            corrupted_endpoint_keys_627e[
                [
                    "unit_id",
                    "cycle",
                ]
            ]
            .reset_index(
                drop=True
            )
            .equals(
                DEV_ENDPOINT_KEYS_627E[
                    [
                        "unit_id",
                        "cycle",
                    ]
                ]
                .reset_index(
                    drop=True
                )
            )
        ):
            raise RuntimeError(
                f"{scenario_id_627e}: "
                "uncertainty endpoint keys changed."
            )

        # --------------------------------------------------------
        # Frozen predictive uncertainty
        # --------------------------------------------------------

        (
            corrupted_prediction_627e,
            corrupted_mc_std_627e,
            corrupted_adjusted_uncertainty_627e,
        ) = run_uncertainty_checked_627e(
            corrupted_endpoint_tensor_627e
        )

        # --------------------------------------------------------
        # Frozen sensor quality
        # --------------------------------------------------------

        (
            corrupted_quality_endpoint_df_627e,
            corrupted_sensor_quality_627e,
        ) = (
            corrupted_sensor_quality_endpoints_627e(
                corrupted_raw_627e
            )
        )

        # --------------------------------------------------------
        # Endpoint-level aligned records
        # --------------------------------------------------------

        scenario_records_start_627e = (
            len(
                records_627e
            )
        )

        for endpoint_idx_627e in range(
            EXPECTED_DEV_ENGINES_627E
        ):

            unit_id_627e = (
                DEV_ENDPOINT_KEYS_627E[
                    "unit_id"
                ].iloc[
                    endpoint_idx_627e
                ]
            )

            cycle_627e = int(
                DEV_ENDPOINT_KEYS_627E[
                    "cycle"
                ].iloc[
                    endpoint_idx_627e
                ]
            )

            clean_pred_i_627e = float(
                clean_prediction_627e[
                    endpoint_idx_627e
                ]
            )

            corrupted_pred_i_627e = float(
                corrupted_prediction_627e[
                    endpoint_idx_627e
                ]
            )

            clean_mc_i_627e = float(
                clean_mc_std_627e[
                    endpoint_idx_627e
                ]
            )

            corrupted_mc_i_627e = float(
                corrupted_mc_std_627e[
                    endpoint_idx_627e
                ]
            )

            clean_adj_i_627e = float(
                clean_adjusted_uncertainty_627e[
                    endpoint_idx_627e
                ]
            )

            corrupted_adj_i_627e = float(
                corrupted_adjusted_uncertainty_627e[
                    endpoint_idx_627e
                ]
            )

            clean_quality_i_627e = float(
                clean_sensor_quality_627e[
                    endpoint_idx_627e
                ]
            )

            corrupted_quality_i_627e = float(
                corrupted_sensor_quality_627e[
                    endpoint_idx_627e
                ]
            )

            records_627e.append(
                {
                    "global_index":
                        int(
                            global_index_627e
                        ),

                    "family":
                        family_627e,

                    "family_position":
                        int(
                            family_position_627e
                        ),

                    "scenario_id":
                        scenario_id_627e,

                    "manifest_scenario_id":
                        scenario_id_627e,

                    "sensor":
                        (
                            scenario_row_627e.get(
                                "sensor",
                                None,
                            )
                        ),

                    "severity":
                        (
                            scenario_row_627e.get(
                                "severity",
                                None,
                            )
                        ),

                    "unit_id":
                        unit_id_627e,

                    "cycle":
                        cycle_627e,

                    "clean_prediction":
                        clean_pred_i_627e,

                    "corrupted_prediction":
                        corrupted_pred_i_627e,

                    "clean_mc_std":
                        clean_mc_i_627e,

                    "corrupted_mc_std":
                        corrupted_mc_i_627e,

                    "clean_adjusted_uncertainty":
                        clean_adj_i_627e,

                    "corrupted_adjusted_uncertainty":
                        corrupted_adj_i_627e,

                    "clean_sensor_quality":
                        clean_quality_i_627e,

                    "corrupted_sensor_quality":
                        corrupted_quality_i_627e,

                    "prediction_change":
                        (
                            corrupted_pred_i_627e
                            -
                            clean_pred_i_627e
                        ),

                    "prediction_abs_change":
                        abs(
                            corrupted_pred_i_627e
                            -
                            clean_pred_i_627e
                        ),

                    "mc_std_change":
                        (
                            corrupted_mc_i_627e
                            -
                            clean_mc_i_627e
                        ),

                    "adjusted_uncertainty_change":
                        (
                            corrupted_adj_i_627e
                            -
                            clean_adj_i_627e
                        ),

                    "sensor_quality_change":
                        (
                            corrupted_quality_i_627e
                            -
                            clean_quality_i_627e
                        ),
                }
            )

        scenario_slice_627e = (
            records_627e[
                scenario_records_start_627e:
            ]
        )

        scenario_temp_df_627e = (
            pd.DataFrame(
                scenario_slice_627e
            )
        )

        scenario_summary_records_627e.append(
            {
                "global_index":
                    int(
                        global_index_627e
                    ),

                "family":
                    family_627e,

                "family_position":
                    int(
                        family_position_627e
                    ),

                "scenario_id":
                    scenario_id_627e,

                "sensor":
                    scenario_row_627e.get(
                        "sensor",
                        None,
                    ),

                "severity":
                    scenario_row_627e.get(
                        "severity",
                        None,
                    ),

                "changed_primary_cells":
                    changed_primary_cells_627e,

                "mean_abs_prediction_change":
                    float(
                        scenario_temp_df_627e[
                            "prediction_abs_change"
                        ].mean()
                    ),

                "mean_mc_std_change":
                    float(
                        scenario_temp_df_627e[
                            "mc_std_change"
                        ].mean()
                    ),

                "mean_adjusted_uncertainty_change":
                    float(
                        scenario_temp_df_627e[
                            "adjusted_uncertainty_change"
                        ].mean()
                    ),

                "mean_sensor_quality_change":
                    float(
                        scenario_temp_df_627e[
                            "sensor_quality_change"
                        ].mean()
                    ),
            }
        )

        global_index_627e += 1

        if (
            family_position_627e == 0
            or
            (family_position_627e + 1) % 10 == 0
            or
            family_position_627e
            ==
            len(
                manifest_627e
            ) - 1
        ):
            print(
                f"  completed "
                f"{family_position_627e + 1:>3}"
                f"/{len(manifest_627e)}"
            )

    print(
        "Family elapsed seconds:",
        round(
            time.time()
            -
            family_start_627e,
            2,
        ),
    )


# ================================================================
# 13. MATERIALIZE FULL ALIGNMENT TABLES
# ================================================================

DEV_ALIGNMENT_627E = pd.DataFrame(
    records_627e
)

DEV_ALIGNMENT_SCENARIO_SUMMARY_627E = (
    pd.DataFrame(
        scenario_summary_records_627e
    )
)

print(
    "\n--- Full alignment materialized ---"
)

print(
    "Endpoint table shape :",
    DEV_ALIGNMENT_627E.shape,
)

print(
    "Scenario table shape :",
    DEV_ALIGNMENT_SCENARIO_SUMMARY_627E.shape,
)


# ================================================================
# 14. FINAL INTEGRITY CHECKS
# ================================================================

if len(
    DEV_ALIGNMENT_627E
) != EXPECTED_ENDPOINT_ROWS_627E:
    raise RuntimeError(
        "Final endpoint alignment does not "
        "contain exactly 6300 rows."
    )

if len(
    DEV_ALIGNMENT_SCENARIO_SUMMARY_627E
) != EXPECTED_SCENARIOS_627E:
    raise RuntimeError(
        "Final scenario summary does not "
        "contain exactly 315 rows."
    )

scenario_count_check_627e = int(
    DEV_ALIGNMENT_627E[
        "scenario_id"
    ].nunique()
)

if scenario_count_check_627e != (
    EXPECTED_SCENARIOS_627E
):
    raise RuntimeError(
        "Unique scenario count is not 315."
    )

per_scenario_rows_627e = (
    DEV_ALIGNMENT_627E
    .groupby(
        "scenario_id"
    )
    .size()
)

if not (
    per_scenario_rows_627e
    ==
    EXPECTED_DEV_ENGINES_627E
).all():
    raise RuntimeError(
        "At least one scenario does not contain "
        "exactly 20 DEV endpoints."
    )

family_counts_627e = (
    DEV_ALIGNMENT_627E[
        [
            "scenario_id",
            "family",
        ]
    ]
    .drop_duplicates()
    .groupby(
        "family"
    )
    .size()
    .to_dict()
)

if family_counts_627e != (
    EXPECTED_FAMILY_COUNTS_627E
):
    raise RuntimeError(
        "Final family scenario counts changed."
    )

numeric_columns_627e = [
    "clean_prediction",
    "corrupted_prediction",
    "clean_mc_std",
    "corrupted_mc_std",
    "clean_adjusted_uncertainty",
    "corrupted_adjusted_uncertainty",
    "clean_sensor_quality",
    "corrupted_sensor_quality",
    "prediction_change",
    "prediction_abs_change",
    "mc_std_change",
    "adjusted_uncertainty_change",
    "sensor_quality_change",
]

all_numeric_finite_627e = bool(
    np.isfinite(
        DEV_ALIGNMENT_627E[
            numeric_columns_627e
        ].to_numpy(
            dtype=np.float64
        )
    ).all()
)

if not all_numeric_finite_627e:
    raise RuntimeError(
        "Final alignment contains "
        "non-finite numeric values."
    )

quality_bounds_ok_627e = bool(
    (
        DEV_ALIGNMENT_627E[
            [
                "clean_sensor_quality",
                "corrupted_sensor_quality",
            ]
        ]
        .to_numpy(
            dtype=np.float64
        )
        >= 0.0
    ).all()
    and
    (
        DEV_ALIGNMENT_627E[
            [
                "clean_sensor_quality",
                "corrupted_sensor_quality",
            ]
        ]
        .to_numpy(
            dtype=np.float64
        )
        <= 1.0
    ).all()
)

if not quality_bounds_ok_627e:
    raise RuntimeError(
        "Sensor-quality values are outside "
        "[0, 1]."
    )

# Confirm the untouched validation base did not change.
base_unchanged_627e = bool(
    VAL_BASE_627E.equals(
        original_base_snapshot_627e
    )
)

if not base_unchanged_627e:
    raise RuntimeError(
        "Validation raw base was modified."
    )


# ================================================================
# 15. COMPACT DIAGNOSTIC SUMMARIES
# ================================================================

family_summary_627e = (
    DEV_ALIGNMENT_627E
    .groupby(
        "family",
        as_index=False,
    )
    .agg(
        scenarios=(
            "scenario_id",
            "nunique",
        ),
        endpoint_rows=(
            "unit_id",
            "size",
        ),
        mean_abs_prediction_change=(
            "prediction_abs_change",
            "mean",
        ),
        mean_mc_std_change=(
            "mc_std_change",
            "mean",
        ),
        mean_adjusted_uncertainty_change=(
            "adjusted_uncertainty_change",
            "mean",
        ),
        mean_sensor_quality_change=(
            "sensor_quality_change",
            "mean",
        ),
    )
)

severity_summary_627e = (
    DEV_ALIGNMENT_627E
    .groupby(
        "severity",
        dropna=False,
        as_index=False,
    )
    .agg(
        scenarios=(
            "scenario_id",
            "nunique",
        ),
        endpoint_rows=(
            "unit_id",
            "size",
        ),
        mean_abs_prediction_change=(
            "prediction_abs_change",
            "mean",
        ),
        mean_mc_std_change=(
            "mc_std_change",
            "mean",
        ),
        mean_adjusted_uncertainty_change=(
            "adjusted_uncertainty_change",
            "mean",
        ),
        mean_sensor_quality_change=(
            "sensor_quality_change",
            "mean",
        ),
    )
)

DEV_ALIGNMENT_FAMILY_SUMMARY_627E = (
    family_summary_627e.copy()
)

DEV_ALIGNMENT_SEVERITY_SUMMARY_627E = (
    severity_summary_627e.copy()
)

print(
    "\n--- Family-level alignment summary ---"
)

display(
    DEV_ALIGNMENT_FAMILY_SUMMARY_627E
)

print(
    "\n--- Severity-level alignment summary ---"
)

display(
    DEV_ALIGNMENT_SEVERITY_SUMMARY_627E
)

print(
    "\n--- First 10 aligned endpoint records ---"
)

display(
    DEV_ALIGNMENT_627E.head(
        10
    )
)


# ================================================================
# 16. FINAL STEP 6.27E SUMMARY
# ================================================================

elapsed_627e = (
    time.time()
    -
    STEP_627E_START
)

print(
    "\n"
    "=================================================================\n"
    "STEP 6.27E FINAL ALIGNMENT SUMMARY\n"
    "================================================================="
)

print(
    "Frozen scenarios completed          :",
    scenario_count_check_627e,
    "/",
    EXPECTED_SCENARIOS_627E,
)

print(
    "DEV endpoint records                :",
    len(
        DEV_ALIGNMENT_627E
    ),
    "/",
    EXPECTED_ENDPOINT_ROWS_627E,
)

print(
    "DEV engines per scenario            :",
    EXPECTED_DEV_ENGINES_627E,
)

print(
    "Five corruption families complete   :",
    family_counts_627e
    ==
    EXPECTED_FAMILY_COUNTS_627E,
)

print(
    "Endpoint preprocessing verified     :",
    clean_endpoint_close_627e,
)

print(
    "Endpoint keys aligned               :",
    clean_key_exact_627e
    and
    clean_quality_key_exact_627e,
)

print(
    "All aligned numeric values finite   :",
    all_numeric_finite_627e,
)

print(
    "Sensor-quality values within [0,1]  :",
    quality_bounds_ok_627e,
)

print(
    "Validation raw base unchanged       :",
    base_unchanged_627e,
)

print(
    "Frozen uncertainty rule changed     :",
    False,
)

print(
    "Frozen sensor-quality rule changed  :",
    False,
)

print(
    "Official TEST labels inspected      :",
    False,
)

print(
    "Official TEST scored                :",
    False,
)

print(
    "Reliability threshold selected      :",
    False,
)

print(
    "Combined reliability rule frozen    :",
    False,
)

print(
    "Files written                       :",
    False,
)

print(
    "Elapsed seconds                     :",
    elapsed_627e,
)

print(
    "\n"
    "In-memory artifacts created:\n"
    "  DEV_ALIGNMENT_627E\n"
    "  DEV_ALIGNMENT_SCENARIO_SUMMARY_627E\n"
    "  DEV_ALIGNMENT_FAMILY_SUMMARY_627E\n"
    "  DEV_ALIGNMENT_SEVERITY_SUMMARY_627E"
)

print(
    "\n"
    "No official TEST labels were used.\n"
    "No research decision was modified."
)

print(
    "\n=== END STEP 6.27E ==="
)

In [ ]:
# ================================================================
# Step 6.27F — DEV-only uncertainty / sensor-quality complementarity
# ================================================================

import numpy as np
import pandas as pd

print(
    "\n"
    "==============================================================\n"
    "STEP 6.27F — DEV UNCERTAINTY + SENSOR-QUALITY COMPLEMENTARITY\n"
    "=============================================================="
)

# ---------------------------------------------------------------
# 1. Required frozen object
# ---------------------------------------------------------------

required_objects_627f = [
    "DEV_ALIGNMENT_627E",
]

print("\n--- Required frozen objects ---")
for name_627f in required_objects_627f:
    present_627f = name_627f in globals()
    print(f"{name_627f:<35}: {present_627f}")
    if not present_627f:
        raise RuntimeError(
            f"Required object {name_627f!r} is missing. "
            "Run the completed Step 6.27E first."
        )

alignment_627f = DEV_ALIGNMENT_627E.copy()

# ---------------------------------------------------------------
# 2. Structural contract
# ---------------------------------------------------------------

required_columns_627f = [
    "global_index",
    "family",
    "scenario_id",
    "sensor",
    "severity",
    "unit_id",
    "cycle",
    "prediction_abs_change",
    "adjusted_uncertainty_change",
    "sensor_quality_change",
]

missing_columns_627f = [
    col_627f
    for col_627f in required_columns_627f
    if col_627f not in alignment_627f.columns
]

if missing_columns_627f:
    raise RuntimeError(
        "DEV_ALIGNMENT_627E is missing required columns: "
        + str(missing_columns_627f)
    )

expected_rows_627f = 315 * 20

if len(alignment_627f) != expected_rows_627f:
    raise RuntimeError(
        f"Expected {expected_rows_627f} aligned endpoint rows, "
        f"found {len(alignment_627f)}."
    )

if alignment_627f["scenario_id"].nunique() != 315:
    raise RuntimeError(
        "Expected exactly 315 DEV corruption scenarios."
    )

if alignment_627f["unit_id"].nunique() != 20:
    raise RuntimeError(
        "Expected exactly 20 DEV engines."
    )

duplicate_keys_627f = int(
    alignment_627f.duplicated(
        subset=["scenario_id", "unit_id"]
    ).sum()
)

if duplicate_keys_627f != 0:
    raise RuntimeError(
        "Duplicate scenario × engine endpoint keys detected."
    )

print("\n--- Alignment contract ---")
print("Rows                 :", len(alignment_627f))
print("Scenarios            :", alignment_627f["scenario_id"].nunique())
print("DEV engines          :", alignment_627f["unit_id"].nunique())
print("Duplicate keys       :", duplicate_keys_627f)

# ---------------------------------------------------------------
# 3. Put both reliability channels in the SAME orientation
#
# Higher = worse:
#   uncertainty_degradation > 0
#   sensor_anomaly_change   > 0
# ---------------------------------------------------------------

alignment_627f["uncertainty_degradation"] = pd.to_numeric(
    alignment_627f["adjusted_uncertainty_change"],
    errors="coerce",
)

alignment_627f["sensor_anomaly_change"] = -pd.to_numeric(
    alignment_627f["sensor_quality_change"],
    errors="coerce",
)

alignment_627f["prediction_abs_change"] = pd.to_numeric(
    alignment_627f["prediction_abs_change"],
    errors="coerce",
)

numeric_check_627f = alignment_627f[
    [
        "uncertainty_degradation",
        "sensor_anomaly_change",
        "prediction_abs_change",
    ]
].to_numpy(dtype=float)

if not np.isfinite(numeric_check_627f).all():
    raise RuntimeError(
        "Non-finite values found in Step 6.27F diagnostic channels."
    )

# ---------------------------------------------------------------
# 4. Correlation helper
# ---------------------------------------------------------------

def safe_corr_627f(df_627f, x_627f, y_627f, method_627f):
    x_values_627f = pd.to_numeric(
        df_627f[x_627f],
        errors="coerce",
    )
    y_values_627f = pd.to_numeric(
        df_627f[y_627f],
        errors="coerce",
    )

    valid_627f = (
        np.isfinite(x_values_627f.to_numpy(dtype=float))
        & np.isfinite(y_values_627f.to_numpy(dtype=float))
    )

    x_values_627f = x_values_627f.loc[valid_627f]
    y_values_627f = y_values_627f.loc[valid_627f]

    if len(x_values_627f) < 3:
        return np.nan

    if x_values_627f.nunique() < 2:
        return np.nan

    if y_values_627f.nunique() < 2:
        return np.nan

    return float(
        x_values_627f.corr(
            y_values_627f,
            method=method_627f,
        )
    )

# ---------------------------------------------------------------
# 5. Endpoint-level complementarity
# ---------------------------------------------------------------

endpoint_pearson_627f = safe_corr_627f(
    alignment_627f,
    "uncertainty_degradation",
    "sensor_anomaly_change",
    "pearson",
)

endpoint_spearman_627f = safe_corr_627f(
    alignment_627f,
    "uncertainty_degradation",
    "sensor_anomaly_change",
    "spearman",
)

uncertainty_vs_prediction_627f = safe_corr_627f(
    alignment_627f,
    "uncertainty_degradation",
    "prediction_abs_change",
    "spearman",
)

sensor_vs_prediction_627f = safe_corr_627f(
    alignment_627f,
    "sensor_anomaly_change",
    "prediction_abs_change",
    "spearman",
)

print("\n--- ENDPOINT-LEVEL COMPLEMENTARITY ---")
print(
    "Pearson: uncertainty degradation vs sensor anomaly :",
    endpoint_pearson_627f,
)
print(
    "Spearman: uncertainty degradation vs sensor anomaly:",
    endpoint_spearman_627f,
)
print(
    "Spearman: uncertainty degradation vs |prediction change|:",
    uncertainty_vs_prediction_627f,
)
print(
    "Spearman: sensor anomaly vs |prediction change|      :",
    sensor_vs_prediction_627f,
)

# ---------------------------------------------------------------
# 6. Scenario-level aggregation
#
# This is important because the 20 engine rows belonging to one
# corruption scenario are not 20 independent corruption designs.
# ---------------------------------------------------------------

scenario_summary_627f = (
    alignment_627f
    .groupby(
        [
            "global_index",
            "family",
            "scenario_id",
            "sensor",
            "severity",
        ],
        as_index=False,
    )
    .agg(
        engines=("unit_id", "nunique"),
        mean_prediction_abs_change=("prediction_abs_change", "mean"),
        mean_uncertainty_degradation=("uncertainty_degradation", "mean"),
        median_uncertainty_degradation=("uncertainty_degradation", "median"),
        mean_sensor_anomaly_change=("sensor_anomaly_change", "mean"),
        median_sensor_anomaly_change=("sensor_anomaly_change", "median"),
    )
)

if len(scenario_summary_627f) != 315:
    raise RuntimeError(
        "Scenario aggregation did not produce exactly 315 rows."
    )

if not (scenario_summary_627f["engines"] == 20).all():
    raise RuntimeError(
        "At least one scenario does not contain all 20 DEV engines."
    )

scenario_pearson_627f = safe_corr_627f(
    scenario_summary_627f,
    "mean_uncertainty_degradation",
    "mean_sensor_anomaly_change",
    "pearson",
)

scenario_spearman_627f = safe_corr_627f(
    scenario_summary_627f,
    "mean_uncertainty_degradation",
    "mean_sensor_anomaly_change",
    "spearman",
)

scenario_unc_vs_pred_627f = safe_corr_627f(
    scenario_summary_627f,
    "mean_uncertainty_degradation",
    "mean_prediction_abs_change",
    "spearman",
)

scenario_sensor_vs_pred_627f = safe_corr_627f(
    scenario_summary_627f,
    "mean_sensor_anomaly_change",
    "mean_prediction_abs_change",
    "spearman",
)

print("\n--- SCENARIO-LEVEL COMPLEMENTARITY ---")
print(
    "Pearson: mean uncertainty degradation vs mean sensor anomaly :",
    scenario_pearson_627f,
)
print(
    "Spearman: mean uncertainty degradation vs mean sensor anomaly:",
    scenario_spearman_627f,
)
print(
    "Spearman: uncertainty degradation vs |prediction change|     :",
    scenario_unc_vs_pred_627f,
)
print(
    "Spearman: sensor anomaly vs |prediction change|               :",
    scenario_sensor_vs_pred_627f,
)

# ---------------------------------------------------------------
# 7. Directional agreement / discordance
#
# No research threshold is selected here.
# Zero only means the corruption did not move that diagnostic
# channel in the expected worsening direction.
# ---------------------------------------------------------------

eps_627f = 1e-12

scenario_summary_627f["uncertainty_worse"] = (
    scenario_summary_627f["mean_uncertainty_degradation"]
    > eps_627f
)

scenario_summary_627f["sensor_quality_worse"] = (
    scenario_summary_627f["mean_sensor_anomaly_change"]
    > eps_627f
)

scenario_summary_627f["both_worse"] = (
    scenario_summary_627f["uncertainty_worse"]
    & scenario_summary_627f["sensor_quality_worse"]
)

scenario_summary_627f["uncertainty_only"] = (
    scenario_summary_627f["uncertainty_worse"]
    & ~scenario_summary_627f["sensor_quality_worse"]
)

scenario_summary_627f["sensor_only"] = (
    ~scenario_summary_627f["uncertainty_worse"]
    & scenario_summary_627f["sensor_quality_worse"]
)

scenario_summary_627f["neither_worse"] = (
    ~scenario_summary_627f["uncertainty_worse"]
    & ~scenario_summary_627f["sensor_quality_worse"]
)

direction_counts_627f = pd.Series(
    {
        "both_worse": int(
            scenario_summary_627f["both_worse"].sum()
        ),
        "uncertainty_only": int(
            scenario_summary_627f["uncertainty_only"].sum()
        ),
        "sensor_only": int(
            scenario_summary_627f["sensor_only"].sum()
        ),
        "neither_worse": int(
            scenario_summary_627f["neither_worse"].sum()
        ),
    },
    name="scenario_count",
)

print("\n--- SCENARIO DIRECTIONAL RESPONSE ---")
print(direction_counts_627f.to_string())

if int(direction_counts_627f.sum()) != 315:
    raise RuntimeError(
        "Directional scenario partition does not sum to 315."
    )

# ---------------------------------------------------------------
# 8. Family-level complementarity
# ---------------------------------------------------------------

family_rows_627f = []

for family_627f, group_627f in scenario_summary_627f.groupby(
    "family",
    sort=True,
):
    family_rows_627f.append(
        {
            "family": family_627f,
            "scenarios": len(group_627f),
            "mean_uncertainty_degradation":
                group_627f[
                    "mean_uncertainty_degradation"
                ].mean(),
            "mean_sensor_anomaly_change":
                group_627f[
                    "mean_sensor_anomaly_change"
                ].mean(),
            "spearman_uncertainty_vs_sensor":
                safe_corr_627f(
                    group_627f,
                    "mean_uncertainty_degradation",
                    "mean_sensor_anomaly_change",
                    "spearman",
                ),
            "uncertainty_worse_fraction":
                group_627f[
                    "uncertainty_worse"
                ].mean(),
            "sensor_worse_fraction":
                group_627f[
                    "sensor_quality_worse"
                ].mean(),
            "discordant_fraction":
                (
                    group_627f["uncertainty_only"]
                    | group_627f["sensor_only"]
                ).mean(),
        }
    )

family_complementarity_627f = pd.DataFrame(
    family_rows_627f
)

print("\n--- FAMILY-LEVEL COMPLEMENTARITY ---")
print(
    family_complementarity_627f.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)

# ---------------------------------------------------------------
# 9. Severity-level complementarity
# ---------------------------------------------------------------

severity_rows_627f = []

for severity_627f, group_627f in scenario_summary_627f.groupby(
    "severity",
    sort=True,
):
    severity_rows_627f.append(
        {
            "severity": severity_627f,
            "scenarios": len(group_627f),
            "mean_prediction_abs_change":
                group_627f[
                    "mean_prediction_abs_change"
                ].mean(),
            "mean_uncertainty_degradation":
                group_627f[
                    "mean_uncertainty_degradation"
                ].mean(),
            "mean_sensor_anomaly_change":
                group_627f[
                    "mean_sensor_anomaly_change"
                ].mean(),
            "discordant_fraction":
                (
                    group_627f["uncertainty_only"]
                    | group_627f["sensor_only"]
                ).mean(),
        }
    )

severity_complementarity_627f = pd.DataFrame(
    severity_rows_627f
)

print("\n--- SEVERITY-LEVEL COMPLEMENTARITY ---")
print(
    severity_complementarity_627f.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)

# ---------------------------------------------------------------
# 10. Compact examples where the two channels disagree
# ---------------------------------------------------------------

discordant_627f = (
    scenario_summary_627f.loc[
        scenario_summary_627f["uncertainty_only"]
        | scenario_summary_627f["sensor_only"],
        [
            "global_index",
            "family",
            "scenario_id",
            "sensor",
            "severity",
            "mean_prediction_abs_change",
            "mean_uncertainty_degradation",
            "mean_sensor_anomaly_change",
            "uncertainty_only",
            "sensor_only",
        ],
    ]
    .copy()
)

discordant_627f["joint_disagreement_magnitude"] = (
    discordant_627f[
        "mean_uncertainty_degradation"
    ].abs()
    + discordant_627f[
        "mean_sensor_anomaly_change"
    ].abs()
)

discordant_627f = discordant_627f.sort_values(
    [
        "joint_disagreement_magnitude",
        "mean_prediction_abs_change",
    ],
    ascending=[False, False],
).reset_index(drop=True)

print("\n--- TOP DISCORDANT SCENARIOS ---")

if len(discordant_627f) == 0:
    print("No directionally discordant scenarios found.")
else:
    print(
        discordant_627f.head(12).to_string(
            index=False,
            float_format=lambda x: f"{x:.6f}",
        )
    )

# ---------------------------------------------------------------
# 11. Preserve in-memory diagnostic artifacts
# ---------------------------------------------------------------

DEV_COMPLEMENTARITY_ENDPOINTS_627F = (
    alignment_627f.copy()
)

DEV_COMPLEMENTARITY_SCENARIOS_627F = (
    scenario_summary_627f.copy()
)

DEV_COMPLEMENTARITY_FAMILY_627F = (
    family_complementarity_627f.copy()
)

DEV_COMPLEMENTARITY_SEVERITY_627F = (
    severity_complementarity_627f.copy()
)

DEV_COMPLEMENTARITY_DISCORDANT_627F = (
    discordant_627f.copy()
)

# ---------------------------------------------------------------
# 12. Final compact summary
# ---------------------------------------------------------------

print(
    "\n"
    "==============================================================\n"
    "STEP 6.27F COMPLEMENTARITY SUMMARY\n"
    "=============================================================="
)

print(
    "Aligned endpoint rows                    :",
    len(alignment_627f),
)
print(
    "Scenario rows                            :",
    len(scenario_summary_627f),
)
print(
    "Scenario Pearson(U degradation, SQ anomaly):",
    scenario_pearson_627f,
)
print(
    "Scenario Spearman(U degradation, SQ anomaly):",
    scenario_spearman_627f,
)
print(
    "Spearman(U degradation, prediction change):",
    scenario_unc_vs_pred_627f,
)
print(
    "Spearman(SQ anomaly, prediction change)   :",
    scenario_sensor_vs_pred_627f,
)
print(
    "Both channels worsen                     :",
    int(direction_counts_627f["both_worse"]),
)
print(
    "Uncertainty-only worsening               :",
    int(direction_counts_627f["uncertainty_only"]),
)
print(
    "Sensor-only worsening                    :",
    int(direction_counts_627f["sensor_only"]),
)
print(
    "Neither worsens                          :",
    int(direction_counts_627f["neither_worse"]),
)

print("\nOfficial TEST labels inspected : False")
print("Official TEST scored           : False")
print("Reliability threshold selected : False")
print("Combined reliability rule frozen: False")
print("Research decision changed      : False")
print("Files written                  : False")

print("\n=== END STEP 6.27F ===")

In [ ]:
print("\n=== 6.27F COMPACT REVIEW ===")

print("\nDirectional response:")
print(direction_counts_627f.to_string())

print("\nFamily complementarity:")
print(
    family_complementarity_627f.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print("\nSeverity complementarity:")
print(
    severity_complementarity_627f.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print("\nCore correlations:")
print("Scenario Pearson U vs SQ  :", scenario_pearson_627f)
print("Scenario Spearman U vs SQ :", scenario_spearman_627f)
print("U vs prediction change    :", scenario_unc_vs_pred_627f)
print("SQ vs prediction change   :", scenario_sensor_vs_pred_627f)

In [ ]:
# ============================================================
# STEP 6.27G0 — COMBINED RELIABILITY PRE-FLIGHT AUDIT
# ============================================================
#
# Purpose:
#   Verify that the already-frozen clean-validation uncertainty
#   and sensor-quality objects are available and structurally
#   suitable before we begin combined reliability construction.
#
# Important:
#   - NO official TEST labels
#   - NO threshold selection
#   - NO combined reliability rule frozen
#   - NO research decision changed
#   - NO files written
# ============================================================

import numpy as np
import pandas as pd

print(
    "\n"
    "============================================================\n"
    "STEP 6.27G0 — COMBINED RELIABILITY PRE-FLIGHT AUDIT\n"
    "============================================================"
)


# ============================================================
# 1. Required frozen objects
# ============================================================

required_627g0 = [
    "DEV_ALIGNMENT_627E",
    "FROZEN_VAL_UNCERTAINTY_622IF",
    "SENSOR_QUALITY_VAL_CLEAN_SCORED_626D",
]

print("\n--- Required frozen objects ---")

for name in required_627g0:
    exists = name in globals()

    print(
        f"{name:42s}: {exists}"
    )

    if not exists:
        raise RuntimeError(
            f"Required frozen object is missing: {name}"
        )


# ============================================================
# 2. Recover clean-validation uncertainty
# ============================================================

clean_uncertainty_627g0 = np.asarray(
    FROZEN_VAL_UNCERTAINTY_622IF,
    dtype=np.float64,
).reshape(-1)

print("\n--- Clean-validation uncertainty ---")
print("Type                 :", type(FROZEN_VAL_UNCERTAINTY_622IF).__name__)
print("Recovered observations:", len(clean_uncertainty_627g0))
print(
    "All values finite     :",
    bool(
        np.isfinite(
            clean_uncertainty_627g0
        ).all()
    ),
)

if not np.isfinite(
    clean_uncertainty_627g0
).all():
    raise RuntimeError(
        "Clean-validation uncertainty contains non-finite values."
    )


# ============================================================
# 3. Inspect frozen clean-validation sensor-quality table
# ============================================================

sq_clean_627g0 = SENSOR_QUALITY_VAL_CLEAN_SCORED_626D

if not isinstance(
    sq_clean_627g0,
    pd.DataFrame,
):
    raise RuntimeError(
        "SENSOR_QUALITY_VAL_CLEAN_SCORED_626D "
        "is not a pandas DataFrame."
    )

print("\n--- Clean-validation sensor-quality table ---")
print("Shape :", sq_clean_627g0.shape)


# ============================================================
# 4. Locate frozen MAX-fusion sensor-quality column
# ============================================================

preferred_sq_columns_627g0 = [
    "sq_candidate_max_fusion",
    "sensor_quality",
    "clean_sensor_quality",
]

available_sq_columns_627g0 = [
    col
    for col in preferred_sq_columns_627g0
    if col in sq_clean_627g0.columns
]

if len(
    available_sq_columns_627g0
) == 0:
    print(
        "\nAvailable sensor-quality columns:"
    )

    for col in sq_clean_627g0.columns:
        print(" -", col)

    raise RuntimeError(
        "Could not locate the frozen sensor-quality "
        "MAX-fusion column. Refusing to guess."
    )


if (
    "sq_candidate_max_fusion"
    in available_sq_columns_627g0
):
    sensor_quality_column_627g0 = (
        "sq_candidate_max_fusion"
    )
else:
    sensor_quality_column_627g0 = (
        available_sq_columns_627g0[0]
    )


print(
    "Selected sensor-quality column:",
    sensor_quality_column_627g0,
)


clean_sensor_quality_627g0 = pd.to_numeric(
    sq_clean_627g0[
        sensor_quality_column_627g0
    ],
    errors="coerce",
).to_numpy(
    dtype=np.float64
)


# ============================================================
# 5. Sensor-quality numerical checks
# ============================================================

sq_finite_627g0 = bool(
    np.isfinite(
        clean_sensor_quality_627g0
    ).all()
)

sq_in_unit_interval_627g0 = bool(
    (
        (
            clean_sensor_quality_627g0
            >= 0.0
        )
        &
        (
            clean_sensor_quality_627g0
            <= 1.0
        )
    ).all()
)

print("\n--- Sensor-quality numerical contract ---")
print(
    "Observations           :",
    len(clean_sensor_quality_627g0),
)
print(
    "All values finite      :",
    sq_finite_627g0,
)
print(
    "All values within [0,1]:",
    sq_in_unit_interval_627g0,
)
print(
    "Minimum                :",
    float(
        np.min(
            clean_sensor_quality_627g0
        )
    ),
)
print(
    "Maximum                :",
    float(
        np.max(
            clean_sensor_quality_627g0
        )
    ),
)
print(
    "Mean                   :",
    float(
        np.mean(
            clean_sensor_quality_627g0
        )
    ),
)


if not sq_finite_627g0:
    raise RuntimeError(
        "Clean sensor-quality values contain "
        "non-finite values."
    )

if not sq_in_unit_interval_627g0:
    raise RuntimeError(
        "Clean sensor-quality values are not "
        "fully contained in [0, 1]."
    )


# ============================================================
# 6. Alignment check
# ============================================================

uncertainty_rows_627g0 = len(
    clean_uncertainty_627g0
)

sensor_quality_rows_627g0 = len(
    clean_sensor_quality_627g0
)

row_count_match_627g0 = bool(
    uncertainty_rows_627g0
    ==
    sensor_quality_rows_627g0
)


print("\n--- Clean-validation alignment ---")
print(
    "Uncertainty observations   :",
    uncertainty_rows_627g0,
)
print(
    "Sensor-quality observations:",
    sensor_quality_rows_627g0,
)
print(
    "Observation counts match   :",
    row_count_match_627g0,
)


if not row_count_match_627g0:
    raise RuntimeError(
        "Clean-validation uncertainty and "
        "sensor-quality observation counts differ. "
        "Do not continue to combined reliability."
    )


# ============================================================
# 7. Expected frozen validation-window count
# ============================================================

expected_validation_windows_627g0 = None

if (
    "X_val_clean"
    in globals()
):
    try:
        expected_validation_windows_627g0 = int(
            np.asarray(
                X_val_clean
            ).shape[0]
        )
    except Exception:
        expected_validation_windows_627g0 = None


if (
    expected_validation_windows_627g0
    is not None
):
    expected_count_match_627g0 = bool(
        uncertainty_rows_627g0
        ==
        expected_validation_windows_627g0
    )
else:
    expected_count_match_627g0 = True


print("\n--- Frozen validation-window contract ---")
print(
    "Expected validation windows:",
    expected_validation_windows_627g0,
)
print(
    "Observed validation windows:",
    uncertainty_rows_627g0,
)
print(
    "Expected count matched     :",
    expected_count_match_627g0,
)


if not expected_count_match_627g0:
    raise RuntimeError(
        "Clean-validation reliability inputs do not "
        "match the frozen validation-window count."
    )


# ============================================================
# 8. Preserve verified inputs for next step
# ============================================================

CLEAN_VAL_UNCERTAINTY_627G0 = (
    clean_uncertainty_627g0.copy()
)

CLEAN_VAL_SENSOR_QUALITY_627G0 = (
    clean_sensor_quality_627g0.copy()
)

SENSOR_QUALITY_COLUMN_627G0 = (
    sensor_quality_column_627g0
)


# ============================================================
# 9. Final pre-flight readiness
# ============================================================

COMBINED_RELIABILITY_PREFLIGHT_READY_627G0 = bool(
    row_count_match_627g0
    and expected_count_match_627g0
    and np.isfinite(
        CLEAN_VAL_UNCERTAINTY_627G0
    ).all()
    and np.isfinite(
        CLEAN_VAL_SENSOR_QUALITY_627G0
    ).all()
    and (
        (
            CLEAN_VAL_SENSOR_QUALITY_627G0
            >= 0.0
        )
        &
        (
            CLEAN_VAL_SENSOR_QUALITY_627G0
            <= 1.0
        )
    ).all()
)


# ============================================================
# 10. Final compact summary
# ============================================================

print(
    "\n"
    "============================================================\n"
    "STEP 6.27G0 PRE-FLIGHT SUMMARY\n"
    "============================================================"
)

print(
    "Clean uncertainty observations :",
    len(
        CLEAN_VAL_UNCERTAINTY_627G0
    ),
)

print(
    "Clean sensor-quality observations:",
    len(
        CLEAN_VAL_SENSOR_QUALITY_627G0
    ),
)

print(
    "Sensor-quality fusion column    :",
    SENSOR_QUALITY_COLUMN_627G0,
)

print(
    "Observation alignment exact     :",
    row_count_match_627g0,
)

print(
    "Expected window count matched   :",
    expected_count_match_627g0,
)

print(
    "Combined-reliability pre-flight ready:",
    COMBINED_RELIABILITY_PREFLIGHT_READY_627G0,
)

print(
    "\nOfficial TEST labels inspected : False"
)

print(
    "Official TEST scored           : False"
)

print(
    "Reliability threshold selected : False"
)

print(
    "Combined reliability rule frozen: False"
)

print(
    "Research decision changed      : False"
)

print(
    "Files written                  : False"
)

print(
    "\n=== END STEP 6.27G0 ==="
)

In [ ]:
# ============================================================
# STEP 6.27G1
# VALIDATION-ONLY RELIABILITY-CHANNEL ORIENTATION
# AND NORMALIZATION AUDIT
#
# Purpose:
#   1. Resolve the exact meaning of the sensor-quality column
#      in DEV_ALIGNMENT_627E.
#   2. Put predictive uncertainty and sensor anomaly onto a
#      common [0,1] "higher = worse" empirical-risk scale.
#   3. Audit directional response under the frozen 315 DEV
#      corruption scenarios.
#
# IMPORTANT:
#   - No RUL labels used.
#   - No official TEST labels used.
#   - No threshold selected.
#   - No combined reliability rule selected.
#   - No files written.
# ============================================================

import numpy as np
import pandas as pd


print(
    "\n"
    "============================================================\n"
    "STEP 6.27G1 — VALIDATION-ONLY CHANNEL ORIENTATION + NORMALIZATION\n"
    "============================================================"
)


# ============================================================
# 1. Required frozen objects
# ============================================================

required_objects_627g1 = [
    "DEV_ALIGNMENT_627E",
    "FROZEN_VAL_UNCERTAINTY_622IF",
    "SENSOR_QUALITY_VAL_CLEAN_SCORED_626D",
]

print("\n--- Required frozen objects ---")

missing_objects_627g1 = []

for name_627g1 in required_objects_627g1:
    exists_627g1 = name_627g1 in globals()

    print(
        f"{name_627g1:<40}: "
        f"{exists_627g1}"
    )

    if not exists_627g1:
        missing_objects_627g1.append(
            name_627g1
        )


if missing_objects_627g1:
    raise RuntimeError(
        "Missing required frozen object(s): "
        + str(
            missing_objects_627g1
        )
    )


# ============================================================
# 2. Local read-only copies
# ============================================================

alignment_627g1 = (
    DEV_ALIGNMENT_627E
    .copy()
)

sensor_validation_627g1 = (
    SENSOR_QUALITY_VAL_CLEAN_SCORED_626D
    .copy()
)

uncertainty_validation_627g1 = (
    np.asarray(
        FROZEN_VAL_UNCERTAINTY_622IF,
        dtype=np.float64,
    )
    .reshape(-1)
)


print(
    "\n--- Frozen source shapes ---"
)

print(
    "DEV alignment              :",
    alignment_627g1.shape,
)

print(
    "Clean sensor-quality table :",
    sensor_validation_627g1.shape,
)

print(
    "Clean uncertainty values   :",
    uncertainty_validation_627g1.shape,
)


# ============================================================
# 3. Required column contract
# ============================================================

required_alignment_columns_627g1 = [
    "family",
    "severity",
    "scenario_id",
    "unit_id",
    "cycle",
    "clean_adjusted_uncertainty",
    "corrupted_adjusted_uncertainty",
    "clean_sensor_quality",
    "corrupted_sensor_quality",
]

missing_alignment_columns_627g1 = [
    col_627g1
    for col_627g1
    in required_alignment_columns_627g1
    if col_627g1
    not in alignment_627g1.columns
]

if missing_alignment_columns_627g1:
    raise RuntimeError(
        "DEV_ALIGNMENT_627E is missing required column(s): "
        + str(
            missing_alignment_columns_627g1
        )
    )


SENSOR_ANOMALY_REFERENCE_COLUMN_627G1 = (
    "sq_candidate_max_fusion"
)

if (
    SENSOR_ANOMALY_REFERENCE_COLUMN_627G1
    not in sensor_validation_627g1.columns
):
    raise RuntimeError(
        "The frozen clean validation sensor table does not contain "
        "'sq_candidate_max_fusion'."
    )


if not {
    "unit_id",
    "cycle",
}.issubset(
    sensor_validation_627g1.columns
):
    raise RuntimeError(
        "The clean validation sensor-quality table does not contain "
        "unit_id and cycle. Refusing to infer endpoint alignment."
    )


# ============================================================
# 4. Numerical integrity checks
# ============================================================

uncertainty_finite_627g1 = bool(
    np.isfinite(
        uncertainty_validation_627g1
    ).all()
)

sensor_reference_values_627g1 = (
    pd.to_numeric(
        sensor_validation_627g1[
            SENSOR_ANOMALY_REFERENCE_COLUMN_627G1
        ],
        errors="coerce",
    )
    .to_numpy(
        dtype=np.float64
    )
)

sensor_reference_finite_627g1 = bool(
    np.isfinite(
        sensor_reference_values_627g1
    ).all()
)


print(
    "\n--- Clean-validation numerical contract ---"
)

print(
    "Uncertainty observations :",
    len(
        uncertainty_validation_627g1
    ),
)

print(
    "Sensor-score observations:",
    len(
        sensor_reference_values_627g1
    ),
)

print(
    "Uncertainty finite       :",
    uncertainty_finite_627g1,
)

print(
    "Sensor score finite      :",
    sensor_reference_finite_627g1,
)


if not uncertainty_finite_627g1:
    raise RuntimeError(
        "Clean validation uncertainty contains non-finite values."
    )

if not sensor_reference_finite_627g1:
    raise RuntimeError(
        "Clean validation sensor score contains non-finite values."
    )

if (
    len(
        uncertainty_validation_627g1
    )
    != len(
        sensor_reference_values_627g1
    )
):
    raise RuntimeError(
        "Clean validation uncertainty and sensor-score populations "
        "do not contain the same number of observations."
    )


# ============================================================
# 5. Check clean endpoint consistency across all 315 scenarios
# ============================================================

endpoint_keys_627g1 = [
    "unit_id",
    "cycle",
]

clean_sq_spread_627g1 = (
    alignment_627g1
    .groupby(
        endpoint_keys_627g1
    )[
        "clean_sensor_quality"
    ]
    .agg(
        lambda values_627g1:
        float(
            np.max(
                values_627g1
            )
            -
            np.min(
                values_627g1
            )
        )
    )
)

maximum_clean_sq_spread_627g1 = float(
    clean_sq_spread_627g1.max()
)


print(
    "\n--- Clean endpoint consistency ---"
)

print(
    "Unique clean endpoint keys :",
    alignment_627g1[
        endpoint_keys_627g1
    ]
    .drop_duplicates()
    .shape[0],
)

print(
    "Maximum clean-score spread :",
    maximum_clean_sq_spread_627g1,
)


if (
    maximum_clean_sq_spread_627g1
    > 1e-10
):
    raise RuntimeError(
        "Clean sensor score is not constant for the same validation "
        "endpoint across corruption scenarios."
    )


# ============================================================
# 6. Recover exact sensor-score orientation
#
# We explicitly test:
#
# A. stored clean_sensor_quality == sq_candidate_max_fusion
#
# B. stored clean_sensor_quality == 1 - sq_candidate_max_fusion
#
# We DO NOT guess.
# ============================================================

reference_duplicates_627g1 = int(
    sensor_validation_627g1
    .duplicated(
        subset=endpoint_keys_627g1
    )
    .sum()
)

if reference_duplicates_627g1 != 0:
    raise RuntimeError(
        "Duplicate unit_id/cycle keys exist in the clean "
        "sensor-quality validation table."
    )


clean_endpoint_scores_627g1 = (
    alignment_627g1[
        endpoint_keys_627g1
        +
        [
            "clean_sensor_quality"
        ]
    ]
    .drop_duplicates(
        subset=endpoint_keys_627g1
    )
    .copy()
)


reference_endpoint_scores_627g1 = (
    sensor_validation_627g1[
        endpoint_keys_627g1
        +
        [
            SENSOR_ANOMALY_REFERENCE_COLUMN_627G1
        ]
    ]
    .copy()
)


orientation_check_627g1 = (
    clean_endpoint_scores_627g1
    .merge(
        reference_endpoint_scores_627g1,
        on=endpoint_keys_627g1,
        how="left",
        validate="one_to_one",
    )
)


if (
    orientation_check_627g1[
        SENSOR_ANOMALY_REFERENCE_COLUMN_627G1
    ]
    .isna()
    .any()
):
    raise RuntimeError(
        "At least one clean endpoint could not be matched to the "
        "frozen clean validation sensor-quality table."
    )


stored_endpoint_values_627g1 = (
    orientation_check_627g1[
        "clean_sensor_quality"
    ]
    .to_numpy(
        dtype=np.float64
    )
)

reference_endpoint_values_627g1 = (
    orientation_check_627g1[
        SENSOR_ANOMALY_REFERENCE_COLUMN_627G1
    ]
    .to_numpy(
        dtype=np.float64
    )
)


direct_error_627g1 = float(
    np.max(
        np.abs(
            stored_endpoint_values_627g1
            -
            reference_endpoint_values_627g1
        )
    )
)

complement_error_627g1 = float(
    np.max(
        np.abs(
            stored_endpoint_values_627g1
            -
            (
                1.0
                -
                reference_endpoint_values_627g1
            )
        )
    )
)


ORIENTATION_TOLERANCE_627G1 = 1e-5


print(
    "\n--- Sensor-score orientation audit ---"
)

print(
    "Direct anomaly match max error    :",
    direct_error_627g1,
)

print(
    "Complement-quality match max error:",
    complement_error_627g1,
)


if (
    direct_error_627g1
    <= ORIENTATION_TOLERANCE_627G1
    and
    direct_error_627g1
    <= complement_error_627g1
):

    SENSOR_ALIGNMENT_ORIENTATION_627G1 = (
        "DIRECT_ANOMALY"
    )

    alignment_627g1[
        "clean_sensor_anomaly_raw"
    ] = (
        alignment_627g1[
            "clean_sensor_quality"
        ]
        .astype(
            np.float64
        )
    )

    alignment_627g1[
        "corrupted_sensor_anomaly_raw"
    ] = (
        alignment_627g1[
            "corrupted_sensor_quality"
        ]
        .astype(
            np.float64
        )
    )

elif (
    complement_error_627g1
    <= ORIENTATION_TOLERANCE_627G1
):

    SENSOR_ALIGNMENT_ORIENTATION_627G1 = (
        "COMPLEMENT_OF_QUALITY"
    )

    alignment_627g1[
        "clean_sensor_anomaly_raw"
    ] = (
        1.0
        -
        alignment_627g1[
            "clean_sensor_quality"
        ]
        .astype(
            np.float64
        )
    )

    alignment_627g1[
        "corrupted_sensor_anomaly_raw"
    ] = (
        1.0
        -
        alignment_627g1[
            "corrupted_sensor_quality"
        ]
        .astype(
            np.float64
        )
    )

else:
    raise RuntimeError(
        "Could not establish whether DEV_ALIGNMENT_627E's "
        "sensor-quality columns represent the frozen anomaly score "
        "or its complement. Refusing to guess."
    )


print(
    "Resolved orientation              :",
    SENSOR_ALIGNMENT_ORIENTATION_627G1,
)

print(
    "Reliability-risk convention       : HIGHER = WORSE"
)


# ============================================================
# 7. Validate resolved raw sensor-anomaly values
# ============================================================

sensor_raw_columns_627g1 = [
    "clean_sensor_anomaly_raw",
    "corrupted_sensor_anomaly_raw",
]

for col_627g1 in sensor_raw_columns_627g1:

    values_627g1 = (
        alignment_627g1[
            col_627g1
        ]
        .to_numpy(
            dtype=np.float64
        )
    )

    if not np.isfinite(
        values_627g1
    ).all():
        raise RuntimeError(
            f"{col_627g1} contains non-finite values."
        )

    if (
        np.min(
            values_627g1
        )
        < -1e-12
        or
        np.max(
            values_627g1
        )
        > 1.0 + 1e-12
    ):
        raise RuntimeError(
            f"{col_627g1} contains values outside [0,1]."
        )


# ============================================================
# 8. Empirical percentile-risk transformation
#
# Clean-validation distribution is the ONLY calibration
# reference.
#
# Higher raw value -> higher percentile risk -> worse.
# ============================================================

def empirical_percentile_risk_627g1(
    values_627g1,
    reference_627g1,
):
    values_627g1 = np.asarray(
        values_627g1,
        dtype=np.float64,
    )

    reference_627g1 = np.asarray(
        reference_627g1,
        dtype=np.float64,
    ).reshape(-1)

    if not np.isfinite(
        reference_627g1
    ).all():
        raise ValueError(
            "Reference distribution contains non-finite values."
        )

    if len(
        reference_627g1
    ) == 0:
        raise ValueError(
            "Reference distribution is empty."
        )

    sorted_reference_627g1 = np.sort(
        reference_627g1
    )

    risk_627g1 = (
        np.searchsorted(
            sorted_reference_627g1,
            values_627g1,
            side="right",
        )
        /
        float(
            len(
                sorted_reference_627g1
            )
        )
    )

    return np.clip(
        risk_627g1,
        0.0,
        1.0,
    )


# ============================================================
# 9. Freeze the two CLEAN calibration references in memory
#    for later Step 6.27G stages
# ============================================================

UNCERTAINTY_RISK_REFERENCE_627G1 = (
    np.sort(
        uncertainty_validation_627g1.copy()
    )
)

SENSOR_ANOMALY_RISK_REFERENCE_627G1 = (
    np.sort(
        sensor_reference_values_627g1.copy()
    )
)


# ============================================================
# 10. Clean-validation normalized-risk diagnostics
# ============================================================

clean_uncertainty_risk_ref_627g1 = (
    empirical_percentile_risk_627g1(
        uncertainty_validation_627g1,
        UNCERTAINTY_RISK_REFERENCE_627G1,
    )
)

clean_sensor_risk_ref_627g1 = (
    empirical_percentile_risk_627g1(
        sensor_reference_values_627g1,
        SENSOR_ANOMALY_RISK_REFERENCE_627G1,
    )
)


CLEAN_RISK_REFERENCE_SUMMARY_627G1 = (
    pd.DataFrame(
        [
            {
                "channel":
                    "predictive_uncertainty",

                "raw_min":
                    float(
                        np.min(
                            uncertainty_validation_627g1
                        )
                    ),

                "raw_median":
                    float(
                        np.median(
                            uncertainty_validation_627g1
                        )
                    ),

                "raw_max":
                    float(
                        np.max(
                            uncertainty_validation_627g1
                        )
                    ),

                "risk_mean":
                    float(
                        np.mean(
                            clean_uncertainty_risk_ref_627g1
                        )
                    ),

                "risk_median":
                    float(
                        np.median(
                            clean_uncertainty_risk_ref_627g1
                        )
                    ),

                "risk_q95":
                    float(
                        np.quantile(
                            clean_uncertainty_risk_ref_627g1,
                            0.95,
                        )
                    ),

                "risk_max":
                    float(
                        np.max(
                            clean_uncertainty_risk_ref_627g1
                        )
                    ),
            },

            {
                "channel":
                    "sensor_anomaly",

                "raw_min":
                    float(
                        np.min(
                            sensor_reference_values_627g1
                        )
                    ),

                "raw_median":
                    float(
                        np.median(
                            sensor_reference_values_627g1
                        )
                    ),

                "raw_max":
                    float(
                        np.max(
                            sensor_reference_values_627g1
                        )
                    ),

                "risk_mean":
                    float(
                        np.mean(
                            clean_sensor_risk_ref_627g1
                        )
                    ),

                "risk_median":
                    float(
                        np.median(
                            clean_sensor_risk_ref_627g1
                        )
                    ),

                "risk_q95":
                    float(
                        np.quantile(
                            clean_sensor_risk_ref_627g1,
                            0.95,
                        )
                    ),

                "risk_max":
                    float(
                        np.max(
                            clean_sensor_risk_ref_627g1
                        )
                    ),
            },
        ]
    )
)


print(
    "\n--- Clean-validation risk normalization ---"
)

print(
    CLEAN_RISK_REFERENCE_SUMMARY_627G1
    .to_string(
        index=False
    )
)


# ============================================================
# 11. Convert the complete 6,300-row DEV alignment table
#     into explicit normalized risk channels
# ============================================================

for col_627g1 in [
    "clean_adjusted_uncertainty",
    "corrupted_adjusted_uncertainty",
]:

    numeric_627g1 = pd.to_numeric(
        alignment_627g1[
            col_627g1
        ],
        errors="coerce",
    )

    if not np.isfinite(
        numeric_627g1.to_numpy(
            dtype=np.float64
        )
    ).all():
        raise RuntimeError(
            f"{col_627g1} contains non-finite values."
        )


alignment_627g1[
    "clean_uncertainty_risk"
] = (
    empirical_percentile_risk_627g1(
        alignment_627g1[
            "clean_adjusted_uncertainty"
        ].to_numpy(
            dtype=np.float64
        ),
        UNCERTAINTY_RISK_REFERENCE_627G1,
    )
)

alignment_627g1[
    "corrupted_uncertainty_risk"
] = (
    empirical_percentile_risk_627g1(
        alignment_627g1[
            "corrupted_adjusted_uncertainty"
        ].to_numpy(
            dtype=np.float64
        ),
        UNCERTAINTY_RISK_REFERENCE_627G1,
    )
)


alignment_627g1[
    "clean_sensor_anomaly_risk"
] = (
    empirical_percentile_risk_627g1(
        alignment_627g1[
            "clean_sensor_anomaly_raw"
        ].to_numpy(
            dtype=np.float64
        ),
        SENSOR_ANOMALY_RISK_REFERENCE_627G1,
    )
)

alignment_627g1[
    "corrupted_sensor_anomaly_risk"
] = (
    empirical_percentile_risk_627g1(
        alignment_627g1[
            "corrupted_sensor_anomaly_raw"
        ].to_numpy(
            dtype=np.float64
        ),
        SENSOR_ANOMALY_RISK_REFERENCE_627G1,
    )
)


# ============================================================
# 12. Explicit degradation deltas
#
# Positive = corruption made that reliability-risk channel worse.
# ============================================================

alignment_627g1[
    "uncertainty_risk_change"
] = (
    alignment_627g1[
        "corrupted_uncertainty_risk"
    ]
    -
    alignment_627g1[
        "clean_uncertainty_risk"
    ]
)

alignment_627g1[
    "sensor_anomaly_risk_change"
] = (
    alignment_627g1[
        "corrupted_sensor_anomaly_risk"
    ]
    -
    alignment_627g1[
        "clean_sensor_anomaly_risk"
    ]
)


# ============================================================
# 13. Final normalized-risk integrity audit
# ============================================================

risk_columns_627g1 = [
    "clean_uncertainty_risk",
    "corrupted_uncertainty_risk",
    "clean_sensor_anomaly_risk",
    "corrupted_sensor_anomaly_risk",
]

all_risk_finite_627g1 = True
all_risk_in_range_627g1 = True

for col_627g1 in risk_columns_627g1:

    values_627g1 = (
        alignment_627g1[
            col_627g1
        ]
        .to_numpy(
            dtype=np.float64
        )
    )

    all_risk_finite_627g1 &= bool(
        np.isfinite(
            values_627g1
        ).all()
    )

    all_risk_in_range_627g1 &= bool(
        (
            values_627g1
            >= 0.0
        ).all()
        and
        (
            values_627g1
            <= 1.0
        ).all()
    )


if not all_risk_finite_627g1:
    raise RuntimeError(
        "A normalized reliability-risk channel contains "
        "non-finite values."
    )

if not all_risk_in_range_627g1:
    raise RuntimeError(
        "A normalized reliability-risk channel is outside [0,1]."
    )


# ============================================================
# 14. Monotonicity audit
#
# The percentile transformation must NEVER reverse ordering.
# ============================================================

def monotonic_non_decreasing_627g1(
    raw_values_627g1,
    risk_values_627g1,
):

    raw_values_627g1 = np.asarray(
        raw_values_627g1,
        dtype=np.float64,
    )

    risk_values_627g1 = np.asarray(
        risk_values_627g1,
        dtype=np.float64,
    )

    order_627g1 = np.argsort(
        raw_values_627g1,
        kind="mergesort",
    )

    sorted_risk_627g1 = (
        risk_values_627g1[
            order_627g1
        ]
    )

    return bool(
        (
            np.diff(
                sorted_risk_627g1
            )
            >= -1e-12
        )
        .all()
    )


uncertainty_monotonic_627g1 = (
    monotonic_non_decreasing_627g1(
        alignment_627g1[
            "corrupted_adjusted_uncertainty"
        ],
        alignment_627g1[
            "corrupted_uncertainty_risk"
        ],
    )
)

sensor_monotonic_627g1 = (
    monotonic_non_decreasing_627g1(
        alignment_627g1[
            "corrupted_sensor_anomaly_raw"
        ],
        alignment_627g1[
            "corrupted_sensor_anomaly_risk"
        ],
    )
)


if not uncertainty_monotonic_627g1:
    raise RuntimeError(
        "Uncertainty normalization is not monotonic."
    )

if not sensor_monotonic_627g1:
    raise RuntimeError(
        "Sensor-anomaly normalization is not monotonic."
    )


# ============================================================
# 15. Overall directional diagnostic
# ============================================================

OVERALL_CHANNEL_RESPONSE_627G1 = (
    pd.DataFrame(
        [
            {
                "channel":
                    "predictive_uncertainty",

                "mean_clean_risk":
                    alignment_627g1[
                        "clean_uncertainty_risk"
                    ].mean(),

                "mean_corrupted_risk":
                    alignment_627g1[
                        "corrupted_uncertainty_risk"
                    ].mean(),

                "mean_risk_change":
                    alignment_627g1[
                        "uncertainty_risk_change"
                    ].mean(),

                "fraction_worse":
                    (
                        alignment_627g1[
                            "uncertainty_risk_change"
                        ]
                        > 0
                    ).mean(),

                "fraction_better":
                    (
                        alignment_627g1[
                            "uncertainty_risk_change"
                        ]
                        < 0
                    ).mean(),
            },

            {
                "channel":
                    "sensor_anomaly",

                "mean_clean_risk":
                    alignment_627g1[
                        "clean_sensor_anomaly_risk"
                    ].mean(),

                "mean_corrupted_risk":
                    alignment_627g1[
                        "corrupted_sensor_anomaly_risk"
                    ].mean(),

                "mean_risk_change":
                    alignment_627g1[
                        "sensor_anomaly_risk_change"
                    ].mean(),

                "fraction_worse":
                    (
                        alignment_627g1[
                            "sensor_anomaly_risk_change"
                        ]
                        > 0
                    ).mean(),

                "fraction_better":
                    (
                        alignment_627g1[
                            "sensor_anomaly_risk_change"
                        ]
                        < 0
                    ).mean(),
            },
        ]
    )
)


print(
    "\n--- Overall normalized reliability-risk response ---"
)

print(
    OVERALL_CHANNEL_RESPONSE_627G1
    .to_string(
        index=False
    )
)


# ============================================================
# 16. Family-level diagnostic
# ============================================================

FAMILY_CHANNEL_RESPONSE_627G1 = (
    alignment_627g1
    .groupby(
        "family",
        as_index=False,
    )
    .agg(
        scenarios=(
            "scenario_id",
            "nunique",
        ),

        endpoint_rows=(
            "scenario_id",
            "size",
        ),

        mean_uncertainty_risk_change=(
            "uncertainty_risk_change",
            "mean",
        ),

        uncertainty_worse_fraction=(
            "uncertainty_risk_change",
            lambda values_627g1:
            float(
                (
                    values_627g1
                    > 0
                )
                .mean()
            ),
        ),

        mean_sensor_anomaly_risk_change=(
            "sensor_anomaly_risk_change",
            "mean",
        ),

        sensor_worse_fraction=(
            "sensor_anomaly_risk_change",
            lambda values_627g1:
            float(
                (
                    values_627g1
                    > 0
                )
                .mean()
            ),
        ),
    )
)


print(
    "\n--- Family-level normalized-risk response ---"
)

print(
    FAMILY_CHANNEL_RESPONSE_627G1
    .to_string(
        index=False
    )
)


# ============================================================
# 17. Severity-level diagnostic
# ============================================================

SEVERITY_CHANNEL_RESPONSE_627G1 = (
    alignment_627g1
    .groupby(
        "severity",
        as_index=False,
    )
    .agg(
        scenarios=(
            "scenario_id",
            "nunique",
        ),

        endpoint_rows=(
            "scenario_id",
            "size",
        ),

        mean_uncertainty_risk_change=(
            "uncertainty_risk_change",
            "mean",
        ),

        uncertainty_worse_fraction=(
            "uncertainty_risk_change",
            lambda values_627g1:
            float(
                (
                    values_627g1
                    > 0
                )
                .mean()
            ),
        ),

        mean_sensor_anomaly_risk_change=(
            "sensor_anomaly_risk_change",
            "mean",
        ),

        sensor_worse_fraction=(
            "sensor_anomaly_risk_change",
            lambda values_627g1:
            float(
                (
                    values_627g1
                    > 0
                )
                .mean()
            ),
        ),
    )
)


print(
    "\n--- Severity-level normalized-risk response ---"
)

print(
    SEVERITY_CHANNEL_RESPONSE_627G1
    .to_string(
        index=False
    )
)


# ============================================================
# 18. Preserve normalized alignment in memory
#
# IMPORTANT:
# This is NOT a combined reliability score.
# ============================================================

DEV_RISK_ALIGNMENT_627G1 = (
    alignment_627g1.copy()
)


RISK_CALIBRATION_REFERENCE_627G1 = {
    "uncertainty":
        UNCERTAINTY_RISK_REFERENCE_627G1.copy(),

    "sensor_anomaly":
        SENSOR_ANOMALY_RISK_REFERENCE_627G1.copy(),
}


# ============================================================
# 19. Final compact summary
# ============================================================

print(
    "\n"
    "============================================================\n"
    "STEP 6.27G1 FINAL ORIENTATION / NORMALIZATION SUMMARY\n"
    "============================================================"
)

print(
    "DEV alignment rows                 :",
    len(
        DEV_RISK_ALIGNMENT_627G1
    ),
)

print(
    "Unique scenarios                   :",
    DEV_RISK_ALIGNMENT_627G1[
        "scenario_id"
    ].nunique(),
)

print(
    "Unique DEV engines                 :",
    DEV_RISK_ALIGNMENT_627G1[
        "unit_id"
    ].nunique(),
)

print(
    "Clean calibration observations     :",
    len(
        uncertainty_validation_627g1
    ),
)

print(
    "Sensor alignment orientation       :",
    SENSOR_ALIGNMENT_ORIENTATION_627G1,
)

print(
    "Risk convention                    : HIGHER = WORSE"
)

print(
    "Uncertainty normalization monotonic:",
    uncertainty_monotonic_627g1,
)

print(
    "Sensor normalization monotonic     :",
    sensor_monotonic_627g1,
)

print(
    "All normalized risks finite        :",
    all_risk_finite_627g1,
)

print(
    "All normalized risks within [0,1]  :",
    all_risk_in_range_627g1,
)

print(
    "Official TEST labels inspected     : False"
)

print(
    "Official TEST scored               : False"
)

print(
    "Combined reliability rule selected : False"
)

print(
    "Reliability threshold selected     : False"
)

print(
    "Files written                      : False"
)

print(
    "\nIn-memory artifacts created:"
)

print(
    "  DEV_RISK_ALIGNMENT_627G1"
)

print(
    "  RISK_CALIBRATION_REFERENCE_627G1"
)

print(
    "  CLEAN_RISK_REFERENCE_SUMMARY_627G1"
)

print(
    "  OVERALL_CHANNEL_RESPONSE_627G1"
)

print(
    "  FAMILY_CHANNEL_RESPONSE_627G1"
)

print(
    "  SEVERITY_CHANNEL_RESPONSE_627G1"
)

print(
    "\nNo frozen research decision was modified."
)

print(
    "\n=== END STEP 6.27G1 ==="
)

In [ ]:
# ============================================================
# STEP 6.27G1-R
# COMPACT REVIEW OF ORIENTATION + NORMALIZATION RESULTS
# ============================================================

print(
    "\n"
    "============================================================\n"
    "STEP 6.27G1-R — COMPACT REVIEW\n"
    "============================================================"
)

print("\n--- SENSOR ORIENTATION ---")
print(
    "Direct anomaly match max error     :",
    direct_error_627g1
)
print(
    "Complement-quality match max error :",
    complement_error_627g1
)
print(
    "Resolved sensor orientation        :",
    SENSOR_ALIGNMENT_ORIENTATION_627G1
)
print(
    "Risk convention                    : HIGHER = WORSE"
)

print("\n--- NORMALIZATION CONTRACT ---")
print(
    "Uncertainty normalization monotonic:",
    uncertainty_monotonic_627g1
)
print(
    "Sensor normalization monotonic     :",
    sensor_monotonic_627g1
)
print(
    "All normalized risks finite        :",
    all_risk_finite_627g1
)
print(
    "All normalized risks within [0,1]  :",
    all_risk_in_range_627g1
)

print("\n--- OVERALL CHANNEL RESPONSE ---")
print(
    OVERALL_CHANNEL_RESPONSE_627G1.to_string(
        index=False
    )
)

print("\n--- CLEAN REFERENCE SUMMARY ---")
print(
    CLEAN_RISK_REFERENCE_SUMMARY_627G1.to_string(
        index=False
    )
)

print("\n--- SAFETY / DECISION STATUS ---")
print("Official TEST labels inspected     : False")
print("Official TEST scored               : False")
print("Combined reliability rule selected : False")
print("Reliability threshold selected     : False")
print("Files written                      : False")

print(
    "\n=== END STEP 6.27G1-R ==="
)

In [ ]:
# ============================================================
# STEP 6.27G2-P
# PRE-FLIGHT: LOCATE NORMALIZED RISK OBJECTS FROM G1
# ============================================================

import numpy as np
import pandas as pd

print(
    "\n"
    "============================================================\n"
    "STEP 6.27G2-P — NORMALIZED-RISK OBJECT PRE-FLIGHT\n"
    "============================================================"
)

keywords = (
    "627g1",
    "risk",
    "normalized",
)

rows = []

for name, obj in globals().items():
    name_lower = name.lower()

    if not any(k in name_lower for k in keywords):
        continue

    if isinstance(obj, np.ndarray):
        rows.append(
            {
                "object_name": name,
                "type": "ndarray",
                "shape": str(obj.shape),
                "columns": "",
            }
        )

    elif isinstance(obj, pd.Series):
        rows.append(
            {
                "object_name": name,
                "type": "Series",
                "shape": str(obj.shape),
                "columns": str(obj.name),
            }
        )

    elif isinstance(obj, pd.DataFrame):
        rows.append(
            {
                "object_name": name,
                "type": "DataFrame",
                "shape": str(obj.shape),
                "columns": ", ".join(map(str, obj.columns)),
            }
        )

inventory_627g2p = pd.DataFrame(rows)

if len(inventory_627g2p) == 0:
    print("No matching objects found.")
else:
    print(
        inventory_627g2p.to_string(
            index=False,
            max_colwidth=100,
        )
    )

print("\nNo research decision changed.")
print("No threshold selected.")
print("No files written.")
print("\n=== END STEP 6.27G2-P ===")

In [ ]:
# ============================================================
# STEP 6.27G2-A
# EXACT G1 ALIGNMENT-COLUMN CONTRACT
# ============================================================

print(
    "\n"
    "============================================================\n"
    "STEP 6.27G2-A — EXACT G1 ALIGNMENT-COLUMN CONTRACT\n"
    "============================================================"
)

required_name_627g2a = "alignment_627g1"

if required_name_627g2a not in globals():
    raise RuntimeError(
        "alignment_627g1 is not available in memory."
    )

alignment_probe_627g2a = alignment_627g1

print("Shape :", alignment_probe_627g2a.shape)

print("\n--- ALL COLUMNS ---")
for i, col in enumerate(
    alignment_probe_627g2a.columns,
    start=1,
):
    print(f"{i:02d}. {col}")

print("\n--- RISK / QUALITY / UNCERTAINTY COLUMNS ---")

relevant_columns_627g2a = [
    col
    for col in alignment_probe_627g2a.columns
    if any(
        token in str(col).lower()
        for token in (
            "risk",
            "quality",
            "uncertainty",
            "mc_std",
        )
    )
]

for col in relevant_columns_627g2a:
    print(col)

print("\n--- CONTRACT CHECKS ---")
print(
    "Rows:",
    len(alignment_probe_627g2a),
)
print(
    "Unique scenarios:",
    alignment_probe_627g2a[
        "scenario_id"
    ].nunique(),
)
print(
    "Unique DEV engines:",
    alignment_probe_627g2a[
        "unit_id"
    ].nunique(),
)
print(
    "Duplicate scenario × engine keys:",
    alignment_probe_627g2a.duplicated(
        subset=[
            "scenario_id",
            "unit_id",
        ]
    ).sum(),
)

print("\nOfficial TEST labels inspected : False")
print("Official TEST scored           : False")
print("Threshold selected             : False")
print("Research decision changed      : False")
print("Files written                  : False")

print(
    "\n=== END STEP 6.27G2-A ==="
)

In [ ]:
# ============================================================
# STEP 6.27G2
# DEV-ONLY COMBINED-RISK CANDIDATE EVALUATION
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score


print(
    "\n"
    "============================================================\n"
    "STEP 6.27G2 — DEV-ONLY COMBINED-RISK CANDIDATE EVALUATION\n"
    "============================================================"
)


# ============================================================
# 1. Required frozen objects
# ============================================================

required_objects_627g2 = [
    "alignment_627g1",
    "UNCERTAINTY_RISK_REFERENCE_627G1",
    "SENSOR_ANOMALY_RISK_REFERENCE_627G1",
]

print("\n--- Required frozen objects ---")

for name in required_objects_627g2:
    exists = name in globals()
    print(f"{name:<40}: {exists}")

    if not exists:
        raise RuntimeError(
            f"Required frozen object is missing: {name}"
        )


# ============================================================
# 2. Work only on a copy
# ============================================================

work_627g2 = alignment_627g1.copy()

print("\n--- Frozen alignment contract ---")
print("Rows             :", len(work_627g2))
print(
    "Unique scenarios :",
    work_627g2["scenario_id"].nunique(),
)
print(
    "DEV engines      :",
    work_627g2["unit_id"].nunique(),
)

duplicate_keys_627g2 = int(
    work_627g2.duplicated(
        subset=[
            "scenario_id",
            "unit_id",
        ]
    ).sum()
)

print("Duplicate keys   :", duplicate_keys_627g2)

if len(work_627g2) != 6300:
    raise RuntimeError(
        "Expected exactly 6300 aligned DEV endpoint records."
    )

if work_627g2["scenario_id"].nunique() != 315:
    raise RuntimeError(
        "Expected exactly 315 frozen scenarios."
    )

if work_627g2["unit_id"].nunique() != 20:
    raise RuntimeError(
        "Expected exactly 20 DEV engines."
    )

if duplicate_keys_627g2 != 0:
    raise RuntimeError(
        "Duplicate scenario × engine keys detected."
    )


# ============================================================
# 3. Safely identify normalized G1 risk columns
# ============================================================

all_columns_lower_627g2 = {
    col: str(col).lower()
    for col in work_627g2.columns
}


def find_single_risk_column_627g2(
    state,
    channel,
):
    """
    Locate one normalized endpoint-risk column from G1.

    state:
        'clean' or 'corrupted'

    channel:
        'uncertainty' or 'sensor'
    """

    matches = []

    for col, low in all_columns_lower_627g2.items():

        if state not in low:
            continue

        if "risk" not in low:
            continue

        # Exclude derived/difference/reference fields.
        if any(
            bad_token in low
            for bad_token in (
                "change",
                "delta",
                "reference",
                "_ref",
            )
        ):
            continue

        if channel == "uncertainty":

            if "uncert" not in low:
                continue

        elif channel == "sensor":

            if not any(
                token in low
                for token in (
                    "sensor",
                    "anomaly",
                    "sq_",
                )
            ):
                continue

        else:
            raise ValueError(
                "Unknown channel."
            )

        matches.append(col)

    if len(matches) != 1:
        raise RuntimeError(
            f"Expected exactly one {state} {channel} "
            f"risk column, but found {matches}. "
            "Refusing to guess."
        )

    return matches[0]


clean_u_col_627g2 = find_single_risk_column_627g2(
    "clean",
    "uncertainty",
)

corr_u_col_627g2 = find_single_risk_column_627g2(
    "corrupted",
    "uncertainty",
)

clean_s_col_627g2 = find_single_risk_column_627g2(
    "clean",
    "sensor",
)

corr_s_col_627g2 = find_single_risk_column_627g2(
    "corrupted",
    "sensor",
)


print("\n--- Recovered normalized G1 endpoint risks ---")
print(
    "Clean uncertainty risk     :",
    clean_u_col_627g2,
)
print(
    "Corrupted uncertainty risk :",
    corr_u_col_627g2,
)
print(
    "Clean sensor risk          :",
    clean_s_col_627g2,
)
print(
    "Corrupted sensor risk      :",
    corr_s_col_627g2,
)


# ============================================================
# 4. Numerical contract
# ============================================================

risk_columns_627g2 = [
    clean_u_col_627g2,
    corr_u_col_627g2,
    clean_s_col_627g2,
    corr_s_col_627g2,
]

for col in risk_columns_627g2:

    values = pd.to_numeric(
        work_627g2[col],
        errors="coerce",
    ).to_numpy(dtype=float)

    if not np.isfinite(values).all():
        raise RuntimeError(
            f"Non-finite values detected in {col}."
        )

    if (
        np.min(values) < -1e-12
        or np.max(values) > 1.0 + 1e-12
    ):
        raise RuntimeError(
            f"{col} contains values outside [0,1]."
        )


# Short internal aliases.
work_627g2["_u_clean"] = (
    work_627g2[clean_u_col_627g2]
    .astype(float)
)

work_627g2["_u_corrupted"] = (
    work_627g2[corr_u_col_627g2]
    .astype(float)
)

work_627g2["_s_clean"] = (
    work_627g2[clean_s_col_627g2]
    .astype(float)
)

work_627g2["_s_corrupted"] = (
    work_627g2[corr_s_col_627g2]
    .astype(float)
)


# Prediction absolute change is diagnostic only.
if "prediction_abs_change" in work_627g2.columns:

    work_627g2["_prediction_abs_change"] = (
        pd.to_numeric(
            work_627g2["prediction_abs_change"],
            errors="coerce",
        )
        .abs()
    )

elif "prediction_change" in work_627g2.columns:

    work_627g2["_prediction_abs_change"] = (
        pd.to_numeric(
            work_627g2["prediction_change"],
            errors="coerce",
        )
        .abs()
    )

else:
    raise RuntimeError(
        "Neither prediction_abs_change nor "
        "prediction_change is available."
    )

if not np.isfinite(
    work_627g2["_prediction_abs_change"]
).all():

    raise RuntimeError(
        "Prediction-change diagnostic contains "
        "non-finite values."
    )


# ============================================================
# 5. Predefined candidate fusion rules
#
# IMPORTANT:
# These candidates are fixed before examining G2 results.
# No post-hoc weight search is performed.
# ============================================================

fusion_rules_627g2 = {
    "uncertainty_only": (
        lambda u, s: u
    ),

    "sensor_only": (
        lambda u, s: s
    ),

    "mean_50U_50S": (
        lambda u, s:
        0.50 * u + 0.50 * s
    ),

    "weighted_75U_25S": (
        lambda u, s:
        0.75 * u + 0.25 * s
    ),

    "weighted_25U_75S": (
        lambda u, s:
        0.25 * u + 0.75 * s
    ),

    "max_OR": (
        lambda u, s:
        np.maximum(u, s)
    ),
}


# ============================================================
# 6. Create candidate risks
# ============================================================

candidate_columns_627g2 = {}

for candidate_name, rule in fusion_rules_627g2.items():

    clean_col = (
        f"_clean_combined_{candidate_name}"
    )

    corrupted_col = (
        f"_corrupted_combined_{candidate_name}"
    )

    work_627g2[clean_col] = rule(
        work_627g2["_u_clean"].to_numpy(),
        work_627g2["_s_clean"].to_numpy(),
    )

    work_627g2[corrupted_col] = rule(
        work_627g2["_u_corrupted"].to_numpy(),
        work_627g2["_s_corrupted"].to_numpy(),
    )

    candidate_columns_627g2[
        candidate_name
    ] = (
        clean_col,
        corrupted_col,
    )

    clean_values = (
        work_627g2[clean_col]
        .to_numpy(dtype=float)
    )

    corrupt_values = (
        work_627g2[corrupted_col]
        .to_numpy(dtype=float)
    )

    if not (
        np.isfinite(clean_values).all()
        and np.isfinite(corrupt_values).all()
    ):
        raise RuntimeError(
            f"Non-finite candidate values: "
            f"{candidate_name}"
        )

    if (
        clean_values.min() < -1e-12
        or clean_values.max() > 1.0 + 1e-12
        or corrupt_values.min() < -1e-12
        or corrupt_values.max() > 1.0 + 1e-12
    ):
        raise RuntimeError(
            f"Candidate {candidate_name} "
            "left the [0,1] interval."
        )


# ============================================================
# 7. Scenario-level evaluation
# ============================================================

scenario_records_627g2 = []

group_columns_627g2 = [
    "scenario_id",
    "family",
    "severity",
]

for (
    scenario_id,
    family,
    severity,
), group in work_627g2.groupby(
    group_columns_627g2,
    sort=False,
):

    if len(group) != 20:
        raise RuntimeError(
            f"Scenario {scenario_id} does not "
            "contain exactly 20 DEV engines."
        )

    prediction_abs_change = (
        group["_prediction_abs_change"]
        .to_numpy(dtype=float)
    )

    for (
        candidate_name,
        (
            clean_col,
            corrupted_col,
        ),
    ) in candidate_columns_627g2.items():

        clean_risk = (
            group[clean_col]
            .to_numpy(dtype=float)
        )

        corrupted_risk = (
            group[corrupted_col]
            .to_numpy(dtype=float)
        )

        y_binary = np.concatenate(
            [
                np.zeros(
                    len(clean_risk),
                    dtype=int,
                ),
                np.ones(
                    len(corrupted_risk),
                    dtype=int,
                ),
            ]
        )

        risk_binary = np.concatenate(
            [
                clean_risk,
                corrupted_risk,
            ]
        )

        auc_value = float(
            roc_auc_score(
                y_binary,
                risk_binary,
            )
        )

        risk_change = (
            corrupted_risk
            - clean_risk
        )

        scenario_records_627g2.append(
            {
                "scenario_id":
                    scenario_id,

                "family":
                    family,

                "severity":
                    severity,

                "candidate":
                    candidate_name,

                "auc_clean_vs_corrupted":
                    auc_value,

                "mean_risk_shift":
                    float(
                        np.mean(
                            risk_change
                        )
                    ),

                "median_risk_shift":
                    float(
                        np.median(
                            risk_change
                        )
                    ),

                "positive_endpoint_fraction":
                    float(
                        np.mean(
                            risk_change > 0
                        )
                    ),

                "mean_prediction_abs_change":
                    float(
                        np.mean(
                            prediction_abs_change
                        )
                    ),
            }
        )


SCENARIO_COMBINED_RISK_RESULTS_627G2 = (
    pd.DataFrame(
        scenario_records_627g2
    )
)


# ============================================================
# 8. Clean-reference behaviour
# ============================================================

u_reference_627g2 = np.asarray(
    UNCERTAINTY_RISK_REFERENCE_627G1,
    dtype=float,
).reshape(-1)

s_reference_627g2 = np.asarray(
    SENSOR_ANOMALY_RISK_REFERENCE_627G1,
    dtype=float,
).reshape(-1)

if (
    len(u_reference_627g2)
    != len(s_reference_627g2)
):
    raise RuntimeError(
        "Clean risk reference lengths do not match."
    )

if not (
    np.isfinite(u_reference_627g2).all()
    and np.isfinite(s_reference_627g2).all()
):
    raise RuntimeError(
        "Clean risk references contain non-finite values."
    )

clean_reference_records_627g2 = []

for candidate_name, rule in fusion_rules_627g2.items():

    clean_reference_risk = rule(
        u_reference_627g2,
        s_reference_627g2,
    )

    clean_reference_records_627g2.append(
        {
            "candidate":
                candidate_name,

            "reference_windows":
                len(clean_reference_risk),

            "clean_mean":
                float(
                    np.mean(
                        clean_reference_risk
                    )
                ),

            "clean_median":
                float(
                    np.median(
                        clean_reference_risk
                    )
                ),

            "clean_q90":
                float(
                    np.quantile(
                        clean_reference_risk,
                        0.90,
                    )
                ),

            "clean_q95":
                float(
                    np.quantile(
                        clean_reference_risk,
                        0.95,
                    )
                ),

            "clean_q99":
                float(
                    np.quantile(
                        clean_reference_risk,
                        0.99,
                    )
                ),

            "clean_max":
                float(
                    np.max(
                        clean_reference_risk
                    )
                ),
        }
    )

CLEAN_COMBINED_RISK_REFERENCE_627G2 = (
    pd.DataFrame(
        clean_reference_records_627g2
    )
)


# ============================================================
# 9. Overall candidate comparison
# ============================================================

overall_rows_627g2 = []

for (
    candidate_name,
    candidate_df,
) in SCENARIO_COMBINED_RISK_RESULTS_627G2.groupby(
    "candidate"
):

    endpoint_clean_col, endpoint_corr_col = (
        candidate_columns_627g2[
            candidate_name
        ]
    )

    endpoint_risk_change = (
        work_627g2[
            endpoint_corr_col
        ].to_numpy(dtype=float)
        -
        work_627g2[
            endpoint_clean_col
        ].to_numpy(dtype=float)
    )

    prediction_change = (
        work_627g2[
            "_prediction_abs_change"
        ]
        .to_numpy(dtype=float)
    )

    # Spearman via ranks avoids dependence on
    # any external statistical package.
    risk_rank = (
        pd.Series(
            endpoint_risk_change
        )
        .rank(
            method="average"
        )
    )

    prediction_rank = (
        pd.Series(
            prediction_change
        )
        .rank(
            method="average"
        )
    )

    spearman_value = float(
        risk_rank.corr(
            prediction_rank
        )
    )

    overall_rows_627g2.append(
        {
            "candidate":
                candidate_name,

            "scenarios":
                len(candidate_df),

            "mean_auc":
                candidate_df[
                    "auc_clean_vs_corrupted"
                ].mean(),

            "median_auc":
                candidate_df[
                    "auc_clean_vs_corrupted"
                ].median(),

            "minimum_auc":
                candidate_df[
                    "auc_clean_vs_corrupted"
                ].min(),

            "fraction_auc_above_0_50":
                (
                    candidate_df[
                        "auc_clean_vs_corrupted"
                    ]
                    > 0.50
                ).mean(),

            "mean_risk_shift":
                candidate_df[
                    "mean_risk_shift"
                ].mean(),

            "positive_scenario_fraction":
                (
                    candidate_df[
                        "mean_risk_shift"
                    ]
                    > 0
                ).mean(),

            "mean_positive_endpoint_fraction":
                candidate_df[
                    "positive_endpoint_fraction"
                ].mean(),

            "spearman_risk_shift_vs_prediction_change":
                spearman_value,
        }
    )


OVERALL_COMBINED_RISK_RESULTS_627G2 = (
    pd.DataFrame(
        overall_rows_627g2
    )
)


# ============================================================
# 10. Family robustness
# ============================================================

FAMILY_COMBINED_RISK_RESULTS_627G2 = (
    SCENARIO_COMBINED_RISK_RESULTS_627G2
    .groupby(
        [
            "candidate",
            "family",
        ],
        as_index=False,
    )
    .agg(
        scenarios=(
            "scenario_id",
            "nunique",
        ),
        mean_auc=(
            "auc_clean_vs_corrupted",
            "mean",
        ),
        mean_risk_shift=(
            "mean_risk_shift",
            "mean",
        ),
        positive_scenario_fraction=(
            "mean_risk_shift",
            lambda x:
                float(
                    np.mean(
                        np.asarray(x) > 0
                    )
                ),
        ),
    )
)


# ============================================================
# 11. Severity robustness
# ============================================================

SEVERITY_COMBINED_RISK_RESULTS_627G2 = (
    SCENARIO_COMBINED_RISK_RESULTS_627G2
    .groupby(
        [
            "candidate",
            "severity",
        ],
        as_index=False,
    )
    .agg(
        scenarios=(
            "scenario_id",
            "nunique",
        ),
        mean_auc=(
            "auc_clean_vs_corrupted",
            "mean",
        ),
        mean_risk_shift=(
            "mean_risk_shift",
            "mean",
        ),
    )
)


# ============================================================
# 12. Add robustness minima to overall table
# ============================================================

family_min_auc_627g2 = (
    FAMILY_COMBINED_RISK_RESULTS_627G2
    .groupby("candidate")[
        "mean_auc"
    ]
    .min()
    .rename(
        "minimum_family_mean_auc"
    )
)

severity_min_auc_627g2 = (
    SEVERITY_COMBINED_RISK_RESULTS_627G2
    .groupby("candidate")[
        "mean_auc"
    ]
    .min()
    .rename(
        "minimum_severity_mean_auc"
    )
)

OVERALL_COMBINED_RISK_RESULTS_627G2 = (
    OVERALL_COMBINED_RISK_RESULTS_627G2
    .merge(
        family_min_auc_627g2,
        on="candidate",
        how="left",
    )
    .merge(
        severity_min_auc_627g2,
        on="candidate",
        how="left",
    )
)


# Attach clean-reference summaries.
OVERALL_COMBINED_RISK_RESULTS_627G2 = (
    OVERALL_COMBINED_RISK_RESULTS_627G2
    .merge(
        CLEAN_COMBINED_RISK_REFERENCE_627G2,
        on="candidate",
        how="left",
    )
)


# ============================================================
# 13. Compact output
# ============================================================

display_columns_627g2 = [
    "candidate",
    "mean_auc",
    "median_auc",
    "fraction_auc_above_0_50",
    "mean_risk_shift",
    "positive_scenario_fraction",
    "spearman_risk_shift_vs_prediction_change",
    "minimum_family_mean_auc",
    "minimum_severity_mean_auc",
    "clean_mean",
    "clean_q95",
]

compact_overall_627g2 = (
    OVERALL_COMBINED_RISK_RESULTS_627G2[
        display_columns_627g2
    ]
    .sort_values(
        [
            "mean_auc",
            "minimum_family_mean_auc",
        ],
        ascending=False,
    )
    .reset_index(drop=True)
)


print(
    "\n--- OVERALL CANDIDATE COMPARISON ---"
)

print(
    compact_overall_627g2.to_string(
        index=False,
        float_format=lambda x:
            f"{x:.6f}",
    )
)


print(
    "\n--- FAMILY MEAN AUC ---"
)

family_auc_pivot_627g2 = (
    FAMILY_COMBINED_RISK_RESULTS_627G2
    .pivot(
        index="family",
        columns="candidate",
        values="mean_auc",
    )
)

print(
    family_auc_pivot_627g2.to_string(
        float_format=lambda x:
            f"{x:.6f}",
    )
)


print(
    "\n--- SEVERITY MEAN AUC ---"
)

severity_auc_pivot_627g2 = (
    SEVERITY_COMBINED_RISK_RESULTS_627G2
    .pivot(
        index="severity",
        columns="candidate",
        values="mean_auc",
    )
)

print(
    severity_auc_pivot_627g2.to_string(
        float_format=lambda x:
            f"{x:.6f}",
    )
)


# ============================================================
# 14. Final integrity summary
# ============================================================

print(
    "\n"
    "============================================================\n"
    "STEP 6.27G2 FINAL SUMMARY\n"
    "============================================================"
)

print(
    "Frozen DEV endpoint rows          :",
    len(work_627g2),
)

print(
    "Frozen scenarios evaluated        :",
    work_627g2[
        "scenario_id"
    ].nunique(),
)

print(
    "Fusion candidates evaluated       :",
    len(
        fusion_rules_627g2
    ),
)

print(
    "Scenario × candidate records      :",
    len(
        SCENARIO_COMBINED_RISK_RESULTS_627G2
    ),
)

print(
    "Expected scenario × candidate rows:",
    315
    * len(
        fusion_rules_627g2
    ),
)

print(
    "All candidate results finite      :",
    bool(
        np.isfinite(
            SCENARIO_COMBINED_RISK_RESULTS_627G2[
                [
                    "auc_clean_vs_corrupted",
                    "mean_risk_shift",
                    "median_risk_shift",
                    "positive_endpoint_fraction",
                ]
            ].to_numpy(
                dtype=float
            )
        ).all()
    ),
)

print(
    "Official TEST labels inspected    : False"
)

print(
    "Official TEST scored              : False"
)

print(
    "Reliability threshold selected    : False"
)

print(
    "Combined reliability rule frozen  : False"
)

print(
    "Research decision changed         : False"
)

print(
    "Files written                     : False"
)

print(
    "\n=== END STEP 6.27G2 ==="
)

In [ ]:
# ============================================================
# STEP 6.27G2-R
# COMPACT FINAL REVIEW OF ALL COMBINED-RISK CANDIDATES
# ============================================================

import pandas as pd

print(
    "\n"
    "============================================================\n"
    "STEP 6.27G2-R — COMPACT CANDIDATE REVIEW\n"
    "============================================================"
)

required_627g2r = [
    "OVERALL_COMBINED_RISK_RESULTS_627G2",
    "FAMILY_COMBINED_RISK_RESULTS_627G2",
    "SEVERITY_COMBINED_RISK_RESULTS_627G2",
]

for name in required_627g2r:
    if name not in globals():
        raise RuntimeError(
            f"Missing G2 result object: {name}"
        )


# ------------------------------------------------------------
# 1. Overall comparison
# ------------------------------------------------------------

overall_cols_627g2r = [
    "candidate",
    "mean_auc",
    "median_auc",
    "fraction_auc_above_0_50",
    "mean_risk_shift",
    "positive_scenario_fraction",
    "spearman_risk_shift_vs_prediction_change",
    "minimum_family_mean_auc",
    "minimum_severity_mean_auc",
    "clean_mean",
    "clean_q95",
]

overall_review_627g2r = (
    OVERALL_COMBINED_RISK_RESULTS_627G2[
        overall_cols_627g2r
    ]
    .sort_values(
        "mean_auc",
        ascending=False,
    )
    .reset_index(drop=True)
)

print("\n--- ALL CANDIDATES ---")

for _, row in overall_review_627g2r.iterrows():

    print(
        f"\n{row['candidate']}"
    )

    print(
        f"  mean AUC               : "
        f"{row['mean_auc']:.6f}"
    )

    print(
        f"  median AUC             : "
        f"{row['median_auc']:.6f}"
    )

    print(
        f"  AUC > 0.50 fraction    : "
        f"{row['fraction_auc_above_0_50']:.6f}"
    )

    print(
        f"  mean risk shift        : "
        f"{row['mean_risk_shift']:.6f}"
    )

    print(
        f"  positive scenarios     : "
        f"{row['positive_scenario_fraction']:.6f}"
    )

    print(
        f"  Spearman vs pred change: "
        f"{row['spearman_risk_shift_vs_prediction_change']:.6f}"
    )

    print(
        f"  worst family mean AUC  : "
        f"{row['minimum_family_mean_auc']:.6f}"
    )

    print(
        f"  worst severity mean AUC: "
        f"{row['minimum_severity_mean_auc']:.6f}"
    )

    print(
        f"  clean mean risk        : "
        f"{row['clean_mean']:.6f}"
    )

    print(
        f"  clean q95 risk         : "
        f"{row['clean_q95']:.6f}"
    )


# ------------------------------------------------------------
# 2. Candidate winner by corruption family
# ------------------------------------------------------------

family_best_627g2r = (
    FAMILY_COMBINED_RISK_RESULTS_627G2
    .sort_values(
        ["family", "mean_auc"],
        ascending=[True, False],
    )
    .groupby(
        "family",
        as_index=False,
    )
    .first()
)

print("\n--- BEST CANDIDATE BY FAMILY ---")

print(
    family_best_627g2r[
        [
            "family",
            "candidate",
            "mean_auc",
            "mean_risk_shift",
        ]
    ].to_string(
        index=False,
        float_format=lambda x:
            f"{x:.6f}",
    )
)


# ------------------------------------------------------------
# 3. Candidate winner by severity
# ------------------------------------------------------------

severity_best_627g2r = (
    SEVERITY_COMBINED_RISK_RESULTS_627G2
    .sort_values(
        ["severity", "mean_auc"],
        ascending=[True, False],
    )
    .groupby(
        "severity",
        as_index=False,
    )
    .first()
)

print("\n--- BEST CANDIDATE BY SEVERITY ---")

print(
    severity_best_627g2r[
        [
            "severity",
            "candidate",
            "mean_auc",
            "mean_risk_shift",
        ]
    ].to_string(
        index=False,
        float_format=lambda x:
            f"{x:.6f}",
    )
)


print("\n--- STATUS ---")
print("Official TEST labels inspected : False")
print("Official TEST scored           : False")
print("Threshold selected             : False")
print("Combined rule frozen           : False")
print("Files written                  : False")

print(
    "\n=== END STEP 6.27G2-R ==="
)

In [ ]:
# ============================================================
# STEP 6.27G2-D
# DIAGNOSE MAX-FUSION CHANNEL DOMINANCE
# ============================================================

import numpy as np
import pandas as pd

print(
    "\n"
    "============================================================\n"
    "STEP 6.27G2-D — MAX-FUSION DOMINANCE DIAGNOSTIC\n"
    "============================================================"
)

required_627g2d = [
    "work_627g2",
    "candidate_columns_627g2",
    "OVERALL_COMBINED_RISK_RESULTS_627G2",
]

for name in required_627g2d:
    if name not in globals():
        raise RuntimeError(
            f"Missing required G2 object: {name}"
        )

# ------------------------------------------------------------
# 1. Basic channel dominance
# ------------------------------------------------------------

u_clean_627g2d = (
    work_627g2["_u_clean"]
    .to_numpy(dtype=float)
)

s_clean_627g2d = (
    work_627g2["_s_clean"]
    .to_numpy(dtype=float)
)

u_corr_627g2d = (
    work_627g2["_u_corrupted"]
    .to_numpy(dtype=float)
)

s_corr_627g2d = (
    work_627g2["_s_corrupted"]
    .to_numpy(dtype=float)
)

print("\n--- CHANNEL DOMINANCE ---")

print(
    "Clean: sensor risk >= uncertainty risk fraction :",
    np.mean(s_clean_627g2d >= u_clean_627g2d)
)

print(
    "Corrupted: sensor risk >= uncertainty risk fraction :",
    np.mean(s_corr_627g2d >= u_corr_627g2d)
)

print(
    "Clean mean uncertainty risk :",
    np.mean(u_clean_627g2d)
)

print(
    "Clean mean sensor risk      :",
    np.mean(s_clean_627g2d)
)

print(
    "Corrupted mean uncertainty risk :",
    np.mean(u_corr_627g2d)
)

print(
    "Corrupted mean sensor risk      :",
    np.mean(s_corr_627g2d)
)


# ------------------------------------------------------------
# 2. Exact MAX vs sensor-only comparison
# ------------------------------------------------------------

max_clean_col_627g2d, max_corr_col_627g2d = (
    candidate_columns_627g2["max_OR"]
)

sensor_clean_col_627g2d, sensor_corr_col_627g2d = (
    candidate_columns_627g2["sensor_only"]
)

max_vs_sensor_clean_diff_627g2d = np.max(
    np.abs(
        work_627g2[max_clean_col_627g2d].to_numpy(dtype=float)
        -
        work_627g2[sensor_clean_col_627g2d].to_numpy(dtype=float)
    )
)

max_vs_sensor_corr_diff_627g2d = np.max(
    np.abs(
        work_627g2[max_corr_col_627g2d].to_numpy(dtype=float)
        -
        work_627g2[sensor_corr_col_627g2d].to_numpy(dtype=float)
    )
)

print("\n--- MAX OR vs SENSOR ONLY ---")

print(
    "Maximum CLEAN absolute difference    :",
    max_vs_sensor_clean_diff_627g2d
)

print(
    "Maximum CORRUPTED absolute difference:",
    max_vs_sensor_corr_diff_627g2d
)

print(
    "MAX clean exactly equals sensor-only :",
    bool(
        np.allclose(
            work_627g2[max_clean_col_627g2d],
            work_627g2[sensor_clean_col_627g2d],
            atol=1e-12,
            rtol=0,
        )
    )
)

print(
    "MAX corrupted equals sensor-only     :",
    bool(
        np.allclose(
            work_627g2[max_corr_col_627g2d],
            work_627g2[sensor_corr_col_627g2d],
            atol=1e-12,
            rtol=0,
        )
    )
)


# ------------------------------------------------------------
# 3. Compact comparison of every candidate
# ------------------------------------------------------------

review_cols_627g2d = [
    "candidate",
    "mean_auc",
    "median_auc",
    "minimum_family_mean_auc",
    "minimum_severity_mean_auc",
    "spearman_risk_shift_vs_prediction_change",
]

review_627g2d = (
    OVERALL_COMBINED_RISK_RESULTS_627G2[
        review_cols_627g2d
    ]
    .sort_values(
        "mean_auc",
        ascending=False,
    )
    .reset_index(drop=True)
)

print("\n--- ALL SIX CANDIDATES ---")

print(
    review_627g2d.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)

print("\n--- STATUS ---")
print("Official TEST labels inspected : False")
print("Official TEST scored           : False")
print("Threshold selected             : False")
print("Combined rule frozen           : False")
print("Research decision changed      : False")
print("Files written                  : False")

print(
    "\n=== END STEP 6.27G2-D ==="
)

In [ ]:
# ============================================================
# STEP 6.27G2-E
# CLEAN-ENDPOINT EMPIRICAL CALIBRATION OF RELIABILITY CHANNELS
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score


print(
    "\n"
    "============================================================\n"
    "STEP 6.27G2-E — CLEAN-ENDPOINT PERCENTILE CALIBRATION\n"
    "============================================================"
)


# ============================================================
# 1. Required G2 objects
# ============================================================

required_627g2e = [
    "work_627g2",
]

print("\n--- Required objects ---")

for name in required_627g2e:

    exists = name in globals()

    print(
        f"{name:<30}: {exists}"
    )

    if not exists:
        raise RuntimeError(
            f"Required object missing: {name}"
        )


required_columns_627g2e = [
    "scenario_id",
    "family",
    "severity",
    "unit_id",
    "_u_clean",
    "_u_corrupted",
    "_s_clean",
    "_s_corrupted",
    "_prediction_abs_change",
]

missing_columns_627g2e = [
    col
    for col in required_columns_627g2e
    if col not in work_627g2.columns
]

if missing_columns_627g2e:

    raise RuntimeError(
        "Required G2 columns missing: "
        + str(
            missing_columns_627g2e
        )
    )


# Work on a copy only.
calibrated_627g2e = (
    work_627g2.copy()
)


# ============================================================
# 2. Structural contract
# ============================================================

print(
    "\n--- Alignment contract ---"
)

print(
    "Rows             :",
    len(calibrated_627g2e),
)

print(
    "Scenarios        :",
    calibrated_627g2e[
        "scenario_id"
    ].nunique(),
)

print(
    "DEV engines      :",
    calibrated_627g2e[
        "unit_id"
    ].nunique(),
)

duplicate_keys_627g2e = int(
    calibrated_627g2e.duplicated(
        subset=[
            "scenario_id",
            "unit_id",
        ]
    ).sum()
)

print(
    "Duplicate keys   :",
    duplicate_keys_627g2e,
)

if len(calibrated_627g2e) != 6300:
    raise RuntimeError(
        "Expected 6300 DEV endpoint rows."
    )

if (
    calibrated_627g2e[
        "scenario_id"
    ].nunique()
    != 315
):
    raise RuntimeError(
        "Expected 315 frozen scenarios."
    )

if (
    calibrated_627g2e[
        "unit_id"
    ].nunique()
    != 20
):
    raise RuntimeError(
        "Expected 20 DEV engines."
    )

if duplicate_keys_627g2e != 0:
    raise RuntimeError(
        "Duplicate scenario × engine keys detected."
    )


# ============================================================
# 3. Verify that each engine has one stable CLEAN endpoint
#
# Clean values are repeated across corruption scenarios.
# They must be identical for the same validation engine.
# ============================================================

clean_stability_627g2e = (
    calibrated_627g2e
    .groupby(
        "unit_id",
        as_index=False,
    )
    .agg(
        u_clean_min=(
            "_u_clean",
            "min",
        ),
        u_clean_max=(
            "_u_clean",
            "max",
        ),
        s_clean_min=(
            "_s_clean",
            "min",
        ),
        s_clean_max=(
            "_s_clean",
            "max",
        ),
    )
)

clean_stability_627g2e[
    "u_spread"
] = (
    clean_stability_627g2e[
        "u_clean_max"
    ]
    -
    clean_stability_627g2e[
        "u_clean_min"
    ]
)

clean_stability_627g2e[
    "s_spread"
] = (
    clean_stability_627g2e[
        "s_clean_max"
    ]
    -
    clean_stability_627g2e[
        "s_clean_min"
    ]
)


max_u_spread_627g2e = float(
    clean_stability_627g2e[
        "u_spread"
    ].max()
)

max_s_spread_627g2e = float(
    clean_stability_627g2e[
        "s_spread"
    ].max()
)

print(
    "\n--- Clean endpoint stability ---"
)

print(
    "Maximum clean uncertainty spread :",
    max_u_spread_627g2e,
)

print(
    "Maximum clean sensor spread      :",
    max_s_spread_627g2e,
)

if (
    max_u_spread_627g2e > 1e-12
    or
    max_s_spread_627g2e > 1e-12
):
    raise RuntimeError(
        "Clean endpoint scores are not stable "
        "across repeated scenarios."
    )


# ============================================================
# 4. Extract the 20 unique clean DEV endpoint references
# ============================================================

clean_endpoint_reference_627g2e = (
    calibrated_627g2e[
        [
            "unit_id",
            "_u_clean",
            "_s_clean",
        ]
    ]
    .drop_duplicates(
        subset=["unit_id"]
    )
    .sort_values(
        "unit_id"
    )
    .reset_index(drop=True)
)

if len(
    clean_endpoint_reference_627g2e
) != 20:

    raise RuntimeError(
        "Expected exactly 20 unique clean DEV endpoints."
    )


u_reference_endpoint_627g2e = (
    clean_endpoint_reference_627g2e[
        "_u_clean"
    ]
    .to_numpy(dtype=float)
)

s_reference_endpoint_627g2e = (
    clean_endpoint_reference_627g2e[
        "_s_clean"
    ]
    .to_numpy(dtype=float)
)


if not (
    np.isfinite(
        u_reference_endpoint_627g2e
    ).all()
    and
    np.isfinite(
        s_reference_endpoint_627g2e
    ).all()
):
    raise RuntimeError(
        "Clean endpoint calibration references "
        "contain non-finite values."
    )


print(
    "\n--- Clean endpoint reference ---"
)

print(
    "Reference engines           :",
    len(
        clean_endpoint_reference_627g2e
    ),
)

print(
    "Uncertainty raw mean        :",
    float(
        np.mean(
            u_reference_endpoint_627g2e
        )
    ),
)

print(
    "Sensor raw mean             :",
    float(
        np.mean(
            s_reference_endpoint_627g2e
        )
    ),
)


# ============================================================
# 5. Mid-rank empirical percentile calibration
#
# Higher input risk -> higher calibrated risk.
#
# For an exact clean-reference value:
# percentile = midpoint of its empirical rank interval.
#
# This handles ties deterministically.
# ============================================================

def empirical_midrank_risk_627g2e(
    values,
    reference,
):

    values = np.asarray(
        values,
        dtype=float,
    ).reshape(-1)

    reference = np.sort(
        np.asarray(
            reference,
            dtype=float,
        ).reshape(-1)
    )

    if len(reference) == 0:
        raise RuntimeError(
            "Empty calibration reference."
        )

    if not (
        np.isfinite(values).all()
        and
        np.isfinite(reference).all()
    ):
        raise RuntimeError(
            "Non-finite values supplied to "
            "empirical calibration."
        )

    n_ref = len(reference)

    left = np.searchsorted(
        reference,
        values,
        side="left",
    )

    right = np.searchsorted(
        reference,
        values,
        side="right",
    )

    calibrated = (
        left + right
    ) / (
        2.0 * n_ref
    )

    calibrated = np.clip(
        calibrated,
        0.0,
        1.0,
    )

    return calibrated


# ============================================================
# 6. Calibrate both channels independently
# ============================================================

calibrated_627g2e[
    "_u_clean_ecdf"
] = empirical_midrank_risk_627g2e(
    calibrated_627g2e[
        "_u_clean"
    ].to_numpy(dtype=float),
    u_reference_endpoint_627g2e,
)

calibrated_627g2e[
    "_u_corrupted_ecdf"
] = empirical_midrank_risk_627g2e(
    calibrated_627g2e[
        "_u_corrupted"
    ].to_numpy(dtype=float),
    u_reference_endpoint_627g2e,
)

calibrated_627g2e[
    "_s_clean_ecdf"
] = empirical_midrank_risk_627g2e(
    calibrated_627g2e[
        "_s_clean"
    ].to_numpy(dtype=float),
    s_reference_endpoint_627g2e,
)

calibrated_627g2e[
    "_s_corrupted_ecdf"
] = empirical_midrank_risk_627g2e(
    calibrated_627g2e[
        "_s_corrupted"
    ].to_numpy(dtype=float),
    s_reference_endpoint_627g2e,
)


calibrated_risk_cols_627g2e = [
    "_u_clean_ecdf",
    "_u_corrupted_ecdf",
    "_s_clean_ecdf",
    "_s_corrupted_ecdf",
]


for col in calibrated_risk_cols_627g2e:

    values = (
        calibrated_627g2e[col]
        .to_numpy(dtype=float)
    )

    if not np.isfinite(values).all():
        raise RuntimeError(
            f"Non-finite calibrated values in {col}."
        )

    if (
        values.min() < -1e-12
        or
        values.max() > 1.0 + 1e-12
    ):
        raise RuntimeError(
            f"Calibrated values outside [0,1]: {col}"
        )


# ============================================================
# 7. Post-calibration scale diagnostic
# ============================================================

print(
    "\n--- Post-calibration channel scale ---"
)

print(
    "Clean mean uncertainty percentile :",
    float(
        calibrated_627g2e[
            "_u_clean_ecdf"
        ].mean()
    ),
)

print(
    "Clean mean sensor percentile      :",
    float(
        calibrated_627g2e[
            "_s_clean_ecdf"
        ].mean()
    ),
)

print(
    "Corrupted mean uncertainty percentile :",
    float(
        calibrated_627g2e[
            "_u_corrupted_ecdf"
        ].mean()
    ),
)

print(
    "Corrupted mean sensor percentile      :",
    float(
        calibrated_627g2e[
            "_s_corrupted_ecdf"
        ].mean()
    ),
)


clean_sensor_dominance_627g2e = float(
    np.mean(
        calibrated_627g2e[
            "_s_clean_ecdf"
        ].to_numpy()
        >=
        calibrated_627g2e[
            "_u_clean_ecdf"
        ].to_numpy()
    )
)

corrupted_sensor_dominance_627g2e = float(
    np.mean(
        calibrated_627g2e[
            "_s_corrupted_ecdf"
        ].to_numpy()
        >=
        calibrated_627g2e[
            "_u_corrupted_ecdf"
        ].to_numpy()
    )
)

print(
    "Clean sensor >= uncertainty fraction     :",
    clean_sensor_dominance_627g2e,
)

print(
    "Corrupted sensor >= uncertainty fraction :",
    corrupted_sensor_dominance_627g2e,
)


# ============================================================
# 8. Predefined candidate fusion rules
#
# Same candidate family as before.
# Only the channel calibration changed.
# ============================================================

fusion_rules_627g2e = {

    "uncertainty_only":
        lambda u, s: u,

    "sensor_only":
        lambda u, s: s,

    "mean_50U_50S":
        lambda u, s:
            0.50 * u
            + 0.50 * s,

    "weighted_75U_25S":
        lambda u, s:
            0.75 * u
            + 0.25 * s,

    "weighted_25U_75S":
        lambda u, s:
            0.25 * u
            + 0.75 * s,

    "max_OR":
        lambda u, s:
            np.maximum(
                u,
                s,
            ),
}


candidate_columns_627g2e = {}


for (
    candidate_name,
    rule,
) in fusion_rules_627g2e.items():

    clean_col = (
        "_clean_ecdf_"
        + candidate_name
    )

    corrupted_col = (
        "_corrupted_ecdf_"
        + candidate_name
    )

    calibrated_627g2e[
        clean_col
    ] = rule(
        calibrated_627g2e[
            "_u_clean_ecdf"
        ].to_numpy(),
        calibrated_627g2e[
            "_s_clean_ecdf"
        ].to_numpy(),
    )

    calibrated_627g2e[
        corrupted_col
    ] = rule(
        calibrated_627g2e[
            "_u_corrupted_ecdf"
        ].to_numpy(),
        calibrated_627g2e[
            "_s_corrupted_ecdf"
        ].to_numpy(),
    )

    candidate_columns_627g2e[
        candidate_name
    ] = (
        clean_col,
        corrupted_col,
    )


# ============================================================
# 9. Evaluate all 315 scenarios
# ============================================================

scenario_records_627g2e = []


for (
    scenario_id,
    family,
    severity,
), group in calibrated_627g2e.groupby(
    [
        "scenario_id",
        "family",
        "severity",
    ],
    sort=False,
):

    if len(group) != 20:

        raise RuntimeError(
            f"Scenario {scenario_id} "
            "does not contain 20 DEV engines."
        )

    prediction_change = (
        group[
            "_prediction_abs_change"
        ]
        .to_numpy(dtype=float)
    )

    for (
        candidate_name,
        (
            clean_col,
            corrupted_col,
        ),
    ) in candidate_columns_627g2e.items():

        clean_risk = (
            group[
                clean_col
            ]
            .to_numpy(dtype=float)
        )

        corrupted_risk = (
            group[
                corrupted_col
            ]
            .to_numpy(dtype=float)
        )

        binary_target = np.concatenate(
            [
                np.zeros(
                    20,
                    dtype=int,
                ),
                np.ones(
                    20,
                    dtype=int,
                ),
            ]
        )

        binary_score = np.concatenate(
            [
                clean_risk,
                corrupted_risk,
            ]
        )

        auc_value = float(
            roc_auc_score(
                binary_target,
                binary_score,
            )
        )

        risk_shift = (
            corrupted_risk
            -
            clean_risk
        )

        risk_rank = (
            pd.Series(
                risk_shift
            )
            .rank(
                method="average"
            )
        )

        prediction_rank = (
            pd.Series(
                prediction_change
            )
            .rank(
                method="average"
            )
        )

        spearman_value = float(
            risk_rank.corr(
                prediction_rank
            )
        )

        scenario_records_627g2e.append(
            {
                "scenario_id":
                    scenario_id,

                "family":
                    family,

                "severity":
                    severity,

                "candidate":
                    candidate_name,

                "auc_clean_vs_corrupted":
                    auc_value,

                "mean_risk_shift":
                    float(
                        np.mean(
                            risk_shift
                        )
                    ),

                "median_risk_shift":
                    float(
                        np.median(
                            risk_shift
                        )
                    ),

                "positive_endpoint_fraction":
                    float(
                        np.mean(
                            risk_shift > 0
                        )
                    ),

                "scenario_spearman_vs_prediction_change":
                    spearman_value,
            }
        )


SCENARIO_ECDF_FUSION_RESULTS_627G2E = (
    pd.DataFrame(
        scenario_records_627g2e
    )
)


# ============================================================
# 10. Overall summary
# ============================================================

OVERALL_ECDF_FUSION_RESULTS_627G2E = (
    SCENARIO_ECDF_FUSION_RESULTS_627G2E
    .groupby(
        "candidate",
        as_index=False,
    )
    .agg(
        scenarios=(
            "scenario_id",
            "nunique",
        ),
        mean_auc=(
            "auc_clean_vs_corrupted",
            "mean",
        ),
        median_auc=(
            "auc_clean_vs_corrupted",
            "median",
        ),
        minimum_auc=(
            "auc_clean_vs_corrupted",
            "min",
        ),
        fraction_auc_above_0_50=(
            "auc_clean_vs_corrupted",
            lambda x:
                float(
                    np.mean(
                        np.asarray(x)
                        > 0.50
                    )
                ),
        ),
        mean_risk_shift=(
            "mean_risk_shift",
            "mean",
        ),
        positive_scenario_fraction=(
            "mean_risk_shift",
            lambda x:
                float(
                    np.mean(
                        np.asarray(x)
                        > 0
                    )
                ),
        ),
        mean_scenario_spearman=(
            "scenario_spearman_vs_prediction_change",
            "mean",
        ),
    )
)


# ============================================================
# 11. Family robustness
# ============================================================

FAMILY_ECDF_FUSION_RESULTS_627G2E = (
    SCENARIO_ECDF_FUSION_RESULTS_627G2E
    .groupby(
        [
            "candidate",
            "family",
        ],
        as_index=False,
    )
    .agg(
        scenarios=(
            "scenario_id",
            "nunique",
        ),
        mean_auc=(
            "auc_clean_vs_corrupted",
            "mean",
        ),
        mean_risk_shift=(
            "mean_risk_shift",
            "mean",
        ),
    )
)


family_min_627g2e = (
    FAMILY_ECDF_FUSION_RESULTS_627G2E
    .groupby(
        "candidate"
    )[
        "mean_auc"
    ]
    .min()
    .rename(
        "minimum_family_mean_auc"
    )
)


# ============================================================
# 12. Severity robustness
# ============================================================

SEVERITY_ECDF_FUSION_RESULTS_627G2E = (
    SCENARIO_ECDF_FUSION_RESULTS_627G2E
    .groupby(
        [
            "candidate",
            "severity",
        ],
        as_index=False,
    )
    .agg(
        scenarios=(
            "scenario_id",
            "nunique",
        ),
        mean_auc=(
            "auc_clean_vs_corrupted",
            "mean",
        ),
        mean_risk_shift=(
            "mean_risk_shift",
            "mean",
        ),
    )
)


severity_min_627g2e = (
    SEVERITY_ECDF_FUSION_RESULTS_627G2E
    .groupby(
        "candidate"
    )[
        "mean_auc"
    ]
    .min()
    .rename(
        "minimum_severity_mean_auc"
    )
)


OVERALL_ECDF_FUSION_RESULTS_627G2E = (
    OVERALL_ECDF_FUSION_RESULTS_627G2E
    .merge(
        family_min_627g2e,
        on="candidate",
        how="left",
    )
    .merge(
        severity_min_627g2e,
        on="candidate",
        how="left",
    )
)


# ============================================================
# 13. Check whether MAX still collapses
# ============================================================

max_clean_col_627g2e, max_corr_col_627g2e = (
    candidate_columns_627g2e[
        "max_OR"
    ]
)

sensor_clean_col_627g2e, sensor_corr_col_627g2e = (
    candidate_columns_627g2e[
        "sensor_only"
    ]
)


max_equals_sensor_clean_627g2e = bool(
    np.allclose(
        calibrated_627g2e[
            max_clean_col_627g2e
        ],
        calibrated_627g2e[
            sensor_clean_col_627g2e
        ],
        atol=1e-12,
        rtol=0,
    )
)

max_equals_sensor_corr_627g2e = bool(
    np.allclose(
        calibrated_627g2e[
            max_corr_col_627g2e
        ],
        calibrated_627g2e[
            sensor_corr_col_627g2e
        ],
        atol=1e-12,
        rtol=0,
    )
)


# ============================================================
# 14. Compact review
# ============================================================

compact_627g2e = (
    OVERALL_ECDF_FUSION_RESULTS_627G2E[
        [
            "candidate",
            "mean_auc",
            "median_auc",
            "fraction_auc_above_0_50",
            "mean_risk_shift",
            "positive_scenario_fraction",
            "mean_scenario_spearman",
            "minimum_family_mean_auc",
            "minimum_severity_mean_auc",
        ]
    ]
    .sort_values(
        [
            "mean_auc",
            "minimum_family_mean_auc",
        ],
        ascending=False,
    )
    .reset_index(drop=True)
)


print(
    "\n--- CALIBRATED CANDIDATE COMPARISON ---"
)

print(
    compact_627g2e.to_string(
        index=False,
        float_format=lambda x:
            f"{x:.6f}",
    )
)


print(
    "\n--- MAX-FUSION COLLAPSE CHECK ---"
)

print(
    "MAX clean equals sensor-only     :",
    max_equals_sensor_clean_627g2e,
)

print(
    "MAX corrupted equals sensor-only :",
    max_equals_sensor_corr_627g2e,
)


print(
    "\n"
    "============================================================\n"
    "STEP 6.27G2-E FINAL SUMMARY\n"
    "============================================================"
)

print(
    "Clean calibration endpoints      :",
    len(
        clean_endpoint_reference_627g2e
    ),
)

print(
    "Frozen scenarios evaluated       :",
    calibrated_627g2e[
        "scenario_id"
    ].nunique(),
)

print(
    "DEV endpoint rows evaluated      :",
    len(
        calibrated_627g2e
    ),
)

print(
    "Fusion candidates evaluated      :",
    len(
        fusion_rules_627g2e
    ),
)

print(
    "Scenario × candidate records     :",
    len(
        SCENARIO_ECDF_FUSION_RESULTS_627G2E
    ),
)

print(
    "Expected records                 :",
    315
    * len(
        fusion_rules_627g2e
    ),
)

print(
    "Clean sensor-dominance fraction  :",
    clean_sensor_dominance_627g2e,
)

print(
    "Corrupt sensor-dominance fraction:",
    corrupted_sensor_dominance_627g2e,
)

print(
    "MAX collapse on clean            :",
    max_equals_sensor_clean_627g2e,
)

print(
    "MAX collapse on corrupted        :",
    max_equals_sensor_corr_627g2e,
)

print(
    "Official TEST labels inspected   : False"
)

print(
    "Official TEST scored             : False"
)

print(
    "Reliability threshold selected   : False"
)

print(
    "Combined rule frozen             : False"
)

print(
    "Research decision changed        : False"
)

print(
    "Files written                    : False"
)

print(
    "\n=== END STEP 6.27G2-E ==="
)

In [ ]:
# ============================================================
# STEP 6.27G2-F
# PAIRED ROBUSTNESS CONFIRMATION BEFORE FUSION FREEZE
# ============================================================

import numpy as np
import pandas as pd

print(
    "\n"
    "============================================================\n"
    "STEP 6.27G2-F — PAIRED FUSION ROBUSTNESS CONFIRMATION\n"
    "============================================================"
)


# ============================================================
# 1. Required object
# ============================================================

required_name_627g2f = (
    "SCENARIO_ECDF_FUSION_RESULTS_627G2E"
)

if required_name_627g2f not in globals():
    raise RuntimeError(
        "Missing calibrated scenario-result table: "
        + required_name_627g2f
    )

source_627g2f = (
    SCENARIO_ECDF_FUSION_RESULTS_627G2E.copy()
)

print("\n--- Source contract ---")
print("Rows       :", len(source_627g2f))
print(
    "Scenarios  :",
    source_627g2f["scenario_id"].nunique(),
)
print(
    "Candidates :",
    source_627g2f["candidate"].nunique(),
)


if len(source_627g2f) != 1890:
    raise RuntimeError(
        "Expected 1890 scenario × candidate records."
    )

if (
    source_627g2f["scenario_id"].nunique()
    != 315
):
    raise RuntimeError(
        "Expected 315 scenarios."
    )


# ============================================================
# 2. Build exact paired scenario table
# ============================================================

paired_627g2f = (
    source_627g2f
    .pivot_table(
        index=[
            "scenario_id",
            "family",
            "severity",
        ],
        columns="candidate",
        values="auc_clean_vs_corrupted",
        aggfunc="first",
    )
    .reset_index()
)

required_candidates_627g2f = [
    "max_OR",
    "sensor_only",
    "weighted_25U_75S",
    "mean_50U_50S",
    "weighted_75U_25S",
    "uncertainty_only",
]

missing_candidates_627g2f = [
    candidate
    for candidate in required_candidates_627g2f
    if candidate not in paired_627g2f.columns
]

if missing_candidates_627g2f:
    raise RuntimeError(
        "Missing candidate columns: "
        + str(missing_candidates_627g2f)
    )


if len(paired_627g2f) != 315:
    raise RuntimeError(
        "Paired scenario table must contain 315 rows."
    )


candidate_matrix_627g2f = (
    paired_627g2f[
        required_candidates_627g2f
    ]
    .to_numpy(dtype=float)
)

if not np.isfinite(
    candidate_matrix_627g2f
).all():
    raise RuntimeError(
        "Non-finite paired AUC values detected."
    )


# ============================================================
# 3. Pairwise deltas against important alternatives
# ============================================================

comparators_627g2f = [
    "sensor_only",
    "weighted_25U_75S",
    "mean_50U_50S",
]

pairwise_records_627g2f = []


print(
    "\n--- PAIRED SCENARIO COMPARISONS ---"
)

for comparator in comparators_627g2f:

    delta = (
        paired_627g2f["max_OR"]
        -
        paired_627g2f[comparator]
    ).to_numpy(dtype=float)

    tolerance = 1e-12

    wins = int(
        np.sum(delta > tolerance)
    )

    ties = int(
        np.sum(
            np.abs(delta)
            <= tolerance
        )
    )

    losses = int(
        np.sum(delta < -tolerance)
    )

    record = {
        "comparator":
            comparator,

        "mean_delta":
            float(np.mean(delta)),

        "median_delta":
            float(np.median(delta)),

        "q05_delta":
            float(
                np.quantile(
                    delta,
                    0.05,
                )
            ),

        "q95_delta":
            float(
                np.quantile(
                    delta,
                    0.95,
                )
            ),

        "wins":
            wins,

        "ties":
            ties,

        "losses":
            losses,

        "win_fraction":
            float(
                wins / len(delta)
            ),
    }

    pairwise_records_627g2f.append(
        record
    )

    print(
        f"\nMAX vs {comparator}"
    )

    print(
        "  mean AUC delta   :",
        record["mean_delta"],
    )

    print(
        "  median AUC delta :",
        record["median_delta"],
    )

    print(
        "  q05 / q95 delta  :",
        record["q05_delta"],
        "/",
        record["q95_delta"],
    )

    print(
        "  wins / ties / losses:",
        wins,
        "/",
        ties,
        "/",
        losses,
    )

    print(
        "  win fraction     :",
        record["win_fraction"],
    )


PAIRWISE_FUSION_AUDIT_627G2F = (
    pd.DataFrame(
        pairwise_records_627g2f
    )
)


# ============================================================
# 4. Family-wise paired MAX-vs-sensor check
# ============================================================

paired_627g2f[
    "delta_max_vs_sensor"
] = (
    paired_627g2f["max_OR"]
    -
    paired_627g2f["sensor_only"]
)

paired_627g2f[
    "delta_max_vs_25U75S"
] = (
    paired_627g2f["max_OR"]
    -
    paired_627g2f[
        "weighted_25U_75S"
    ]
)


FAMILY_PAIRED_AUDIT_627G2F = (
    paired_627g2f
    .groupby(
        "family",
        as_index=False,
    )
    .agg(
        scenarios=(
            "scenario_id",
            "nunique",
        ),

        mean_delta_vs_sensor=(
            "delta_max_vs_sensor",
            "mean",
        ),

        median_delta_vs_sensor=(
            "delta_max_vs_sensor",
            "median",
        ),

        mean_delta_vs_25U75S=(
            "delta_max_vs_25U75S",
            "mean",
        ),

        median_delta_vs_25U75S=(
            "delta_max_vs_25U75S",
            "median",
        ),
    )
)


print(
    "\n--- FAMILY-WISE PAIRED DELTAS ---"
)

print(
    FAMILY_PAIRED_AUDIT_627G2F.to_string(
        index=False,
        float_format=lambda x:
            f"{x:.6f}",
    )
)


# ============================================================
# 5. Severity-wise paired audit
# ============================================================

SEVERITY_PAIRED_AUDIT_627G2F = (
    paired_627g2f
    .groupby(
        "severity",
        as_index=False,
    )
    .agg(
        scenarios=(
            "scenario_id",
            "nunique",
        ),

        mean_delta_vs_sensor=(
            "delta_max_vs_sensor",
            "mean",
        ),

        mean_delta_vs_25U75S=(
            "delta_max_vs_25U75S",
            "mean",
        ),
    )
)


print(
    "\n--- SEVERITY-WISE PAIRED DELTAS ---"
)

print(
    SEVERITY_PAIRED_AUDIT_627G2F.to_string(
        index=False,
        float_format=lambda x:
            f"{x:.6f}",
    )
)


# ============================================================
# 6. Leave-one-family-out stability
# ============================================================

loo_records_627g2f = []

families_627g2f = sorted(
    paired_627g2f[
        "family"
    ].unique()
)


for excluded_family in families_627g2f:

    remaining = (
        paired_627g2f[
            paired_627g2f["family"]
            != excluded_family
        ]
    )

    loo_records_627g2f.append(
        {
            "excluded_family":
                excluded_family,

            "remaining_scenarios":
                len(remaining),

            "mean_delta_vs_sensor":
                float(
                    remaining[
                        "delta_max_vs_sensor"
                    ].mean()
                ),

            "mean_delta_vs_25U75S":
                float(
                    remaining[
                        "delta_max_vs_25U75S"
                    ].mean()
                ),
        }
    )


LOO_FUSION_AUDIT_627G2F = (
    pd.DataFrame(
        loo_records_627g2f
    )
)


print(
    "\n--- LEAVE-ONE-FAMILY-OUT ---"
)

print(
    LOO_FUSION_AUDIT_627G2F.to_string(
        index=False,
        float_format=lambda x:
            f"{x:.6f}",
    )
)


# ============================================================
# 7. Descriptive paired bootstrap
#
# This is diagnostic evidence only.
# We are NOT treating corruption scenarios as fully
# independent statistical observations.
# ============================================================

rng_627g2f = np.random.default_rng(
    42
)

bootstrap_iterations_627g2f = 10000


def bootstrap_mean_ci_627g2f(
    delta,
):

    delta = np.asarray(
        delta,
        dtype=float,
    )

    n = len(delta)

    indices = rng_627g2f.integers(
        low=0,
        high=n,
        size=(
            bootstrap_iterations_627g2f,
            n,
        ),
    )

    means = (
        delta[indices]
        .mean(axis=1)
    )

    return (
        float(np.quantile(means, 0.025)),
        float(np.quantile(means, 0.975)),
    )


delta_sensor_627g2f = (
    paired_627g2f[
        "delta_max_vs_sensor"
    ].to_numpy(dtype=float)
)

delta_25u75s_627g2f = (
    paired_627g2f[
        "delta_max_vs_25U75S"
    ].to_numpy(dtype=float)
)


ci_sensor_627g2f = (
    bootstrap_mean_ci_627g2f(
        delta_sensor_627g2f
    )
)

ci_25u75s_627g2f = (
    bootstrap_mean_ci_627g2f(
        delta_25u75s_627g2f
    )
)


print(
    "\n--- DESCRIPTIVE BOOTSTRAP CHECK ---"
)

print(
    "MAX - sensor-only mean delta:",
    float(
        np.mean(
            delta_sensor_627g2f
        )
    ),
)

print(
    "95% descriptive bootstrap interval:",
    ci_sensor_627g2f,
)

print(
    "\nMAX - weighted_25U_75S mean delta:",
    float(
        np.mean(
            delta_25u75s_627g2f
        )
    ),
)

print(
    "95% descriptive bootstrap interval:",
    ci_25u75s_627g2f,
)


# ============================================================
# 8. Compact structural verdict
# ============================================================

all_loo_sensor_positive_627g2f = bool(
    (
        LOO_FUSION_AUDIT_627G2F[
            "mean_delta_vs_sensor"
        ]
        > 0
    ).all()
)

all_loo_25u75s_positive_627g2f = bool(
    (
        LOO_FUSION_AUDIT_627G2F[
            "mean_delta_vs_25U75S"
        ]
        > 0
    ).all()
)


print(
    "\n"
    "============================================================\n"
    "STEP 6.27G2-F SUMMARY\n"
    "============================================================"
)

print(
    "Paired scenarios                  :",
    len(paired_627g2f),
)

print(
    "All LOO deltas > sensor-only     :",
    all_loo_sensor_positive_627g2f,
)

print(
    "All LOO deltas > weighted 25U75S :",
    all_loo_25u75s_positive_627g2f,
)

print(
    "Official TEST labels inspected   : False"
)

print(
    "Official TEST scored             : False"
)

print(
    "Reliability threshold selected   : False"
)

print(
    "Combined rule frozen             : False"
)

print(
    "Research decision changed        : False"
)

print(
    "Files written                    : False"
)

print(
    "\n=== END STEP 6.27G2-F ==="
)

In [ ]:
# ============================================================
# STEP 6.27G2-F-R
# COMPACT RECOVERY OF G2-F RESULTS
# ============================================================

print(
    "\n"
    "============================================================\n"
    "STEP 6.27G2-F-R — COMPACT ROBUSTNESS REVIEW\n"
    "============================================================"
)

# ------------------------------------------------------------
# 1. Paired comparisons
# ------------------------------------------------------------

print("\n--- PAIRED SCENARIO COMPARISONS ---")

print(
    PAIRWISE_FUSION_AUDIT_627G2F.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)


# ------------------------------------------------------------
# 2. Family robustness
# ------------------------------------------------------------

print("\n--- FAMILY-WISE PAIRED DELTAS ---")

print(
    FAMILY_PAIRED_AUDIT_627G2F.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)


# ------------------------------------------------------------
# 3. Severity robustness
# ------------------------------------------------------------

print("\n--- SEVERITY-WISE PAIRED DELTAS ---")

print(
    SEVERITY_PAIRED_AUDIT_627G2F.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)


# ------------------------------------------------------------
# 4. Leave-one-family-out
# ------------------------------------------------------------

print("\n--- LEAVE-ONE-FAMILY-OUT ---")

print(
    LOO_FUSION_AUDIT_627G2F.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)


# ------------------------------------------------------------
# 5. Descriptive bootstrap
# ------------------------------------------------------------

print("\n--- DESCRIPTIVE BOOTSTRAP CHECK ---")

print(
    "MAX - sensor-only mean delta:",
    float(
        paired_627g2f[
            "delta_max_vs_sensor"
        ].mean()
    ),
)

print(
    "95% bootstrap interval:",
    ci_sensor_627g2f,
)

print(
    "\nMAX - weighted_25U_75S mean delta:",
    float(
        paired_627g2f[
            "delta_max_vs_25U75S"
        ].mean()
    ),
)

print(
    "95% bootstrap interval:",
    ci_25u75s_627g2f,
)


# ------------------------------------------------------------
# 6. Final compact verdict
# ------------------------------------------------------------

print(
    "\n"
    "============================================================\n"
    "STEP 6.27G2-F-R FINAL SUMMARY\n"
    "============================================================"
)

print(
    "Paired scenarios                  :",
    len(paired_627g2f),
)

print(
    "All LOO deltas > sensor-only     :",
    all_loo_sensor_positive_627g2f,
)

print(
    "All LOO deltas > weighted 25U75S :",
    all_loo_25u75s_positive_627g2f,
)

print(
    "Official TEST labels inspected   : False"
)

print(
    "Official TEST scored             : False"
)

print(
    "Reliability threshold selected   : False"
)

print(
    "Combined rule frozen             : False"
)

print(
    "Files written                    : False"
)

print(
    "\n=== END STEP 6.27G2-F-R ==="
)

In [ ]:
# ============================================================
# STEP 6.27G2-F-R2
# FINAL MISSING ROBUSTNESS EVIDENCE ONLY
# ============================================================

print(
    "\n"
    "============================================================\n"
    "STEP 6.27G2-F-R2 — FINAL MISSING EVIDENCE\n"
    "============================================================"
)

print("\n--- LEAVE-ONE-FAMILY-OUT ---")

print(
    LOO_FUSION_AUDIT_627G2F.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)


print("\n--- DESCRIPTIVE BOOTSTRAP ---")

print(
    "MAX - sensor-only mean delta :",
    float(
        paired_627g2f[
            "delta_max_vs_sensor"
        ].mean()
    ),
)

print(
    "MAX - sensor-only 95% interval:",
    ci_sensor_627g2f,
)

print(
    "MAX - 25U75S mean delta      :",
    float(
        paired_627g2f[
            "delta_max_vs_25U75S"
        ].mean()
    ),
)

print(
    "MAX - 25U75S 95% interval    :",
    ci_25u75s_627g2f,
)


print("\n--- FINAL BOOLEAN CHECK ---")

print(
    "All LOO deltas > sensor-only     :",
    all_loo_sensor_positive_627g2f,
)

print(
    "All LOO deltas > weighted 25U75S :",
    all_loo_25u75s_positive_627g2f,
)

print(
    "\nOfficial TEST labels inspected : False"
)

print(
    "Threshold selected             : False"
)

print(
    "Combined rule frozen           : False"
)

print(
    "Files written                  : False"
)

print(
    "\n=== END STEP 6.27G2-F-R2 ==="
)

In [ ]:
# ============================================================
# STEP 6.27G3
# FREEZE FINAL COMBINED RELIABILITY REPRESENTATION
# ============================================================

import numpy as np
import pandas as pd


print(
    "\n"
    "============================================================\n"
    "STEP 6.27G3 — FREEZE COMBINED RELIABILITY REPRESENTATION\n"
    "============================================================"
)


# ============================================================
# 1. Required evidence objects
# ============================================================

required_objects_627g3 = [
    "calibrated_627g2e",
    "u_reference_endpoint_627g2e",
    "s_reference_endpoint_627g2e",
    "candidate_columns_627g2e",
    "OVERALL_ECDF_FUSION_RESULTS_627G2E",
    "FAMILY_PAIRED_AUDIT_627G2F",
    "SEVERITY_PAIRED_AUDIT_627G2F",
    "LOO_FUSION_AUDIT_627G2F",
    "ci_sensor_627g2f",
    "ci_25u75s_627g2f",
]

print("\n--- Required evidence objects ---")

for name in required_objects_627g3:

    exists = name in globals()

    print(
        f"{name:<42}: {exists}"
    )

    if not exists:
        raise RuntimeError(
            f"Missing required object: {name}"
        )


# ============================================================
# 2. Freeze calibration references
# ============================================================

RELIABILITY_UNCERTAINTY_REFERENCE_627G3 = (
    np.sort(
        np.asarray(
            u_reference_endpoint_627g2e,
            dtype=float,
        ).copy()
    )
)

RELIABILITY_SENSOR_REFERENCE_627G3 = (
    np.sort(
        np.asarray(
            s_reference_endpoint_627g2e,
            dtype=float,
        ).copy()
    )
)


if (
    len(RELIABILITY_UNCERTAINTY_REFERENCE_627G3)
    != 20
    or
    len(RELIABILITY_SENSOR_REFERENCE_627G3)
    != 20
):
    raise RuntimeError(
        "Frozen calibration must contain exactly "
        "20 clean DEV endpoint references per channel."
    )


if not (
    np.isfinite(
        RELIABILITY_UNCERTAINTY_REFERENCE_627G3
    ).all()
    and
    np.isfinite(
        RELIABILITY_SENSOR_REFERENCE_627G3
    ).all()
):
    raise RuntimeError(
        "Frozen calibration references contain "
        "non-finite values."
    )


# ============================================================
# 3. Freeze empirical calibration function
#
# Higher raw score = higher risk.
# Calibration population = CLEAN DEV endpoints only.
# ============================================================

def reliability_empirical_midrank_627g3(
    values,
    reference,
):

    values = np.asarray(
        values,
        dtype=float,
    ).reshape(-1)

    reference = np.sort(
        np.asarray(
            reference,
            dtype=float,
        ).reshape(-1)
    )

    if len(reference) == 0:
        raise RuntimeError(
            "Empty reliability calibration reference."
        )

    if not (
        np.isfinite(values).all()
        and
        np.isfinite(reference).all()
    ):
        raise RuntimeError(
            "Non-finite reliability values detected."
        )

    n_ref = len(reference)

    left = np.searchsorted(
        reference,
        values,
        side="left",
    )

    right = np.searchsorted(
        reference,
        values,
        side="right",
    )

    risk = (
        left + right
    ) / (
        2.0 * n_ref
    )

    return np.clip(
        risk,
        0.0,
        1.0,
    )


# ============================================================
# 4. Freeze channel-specific calibration
# ============================================================

def calibrate_uncertainty_risk_627g3(
    raw_uncertainty_risk,
):

    return reliability_empirical_midrank_627g3(
        raw_uncertainty_risk,
        RELIABILITY_UNCERTAINTY_REFERENCE_627G3,
    )


def calibrate_sensor_risk_627g3(
    raw_sensor_anomaly_risk,
):

    return reliability_empirical_midrank_627g3(
        raw_sensor_anomaly_risk,
        RELIABILITY_SENSOR_REFERENCE_627G3,
    )


# ============================================================
# 5. Freeze final fusion rule
#
# MAX/OR:
#
# combined risk = max(
#     calibrated predictive uncertainty risk,
#     calibrated sensor anomaly risk
# )
#
# Higher = less reliable / greater risk.
# ============================================================

def combined_reliability_risk_627g3(
    raw_uncertainty_risk,
    raw_sensor_anomaly_risk,
):

    uncertainty_risk = (
        calibrate_uncertainty_risk_627g3(
            raw_uncertainty_risk
        )
    )

    sensor_risk = (
        calibrate_sensor_risk_627g3(
            raw_sensor_anomaly_risk
        )
    )

    if uncertainty_risk.shape != sensor_risk.shape:
        raise RuntimeError(
            "Reliability-channel shapes do not match."
        )

    combined_risk = np.maximum(
        uncertainty_risk,
        sensor_risk,
    )

    return combined_risk


# ============================================================
# 6. Frozen metadata
# ============================================================

FINAL_RELIABILITY_REPRESENTATION_627G3 = {
    "calibration_population":
        "clean_DEV_endpoints",

    "calibration_endpoint_count":
        20,

    "calibration_method":
        "empirical_midrank_percentile",

    "uncertainty_orientation":
        "higher_is_worse",

    "sensor_orientation":
        "higher_sensor_anomaly_is_worse",

    "fusion_rule":
        "maximum_OR",

    "combined_orientation":
        "higher_is_worse",

    "window_length":
        30,

    "sensor_only_retained_as_ablation":
        True,

    "threshold_selected":
        False,

    "official_test_used":
        False,
}


# ============================================================
# 7. Exact reproduction check against Step G2-E
# ============================================================

max_clean_col_627g3, max_corrupted_col_627g3 = (
    candidate_columns_627g2e[
        "max_OR"
    ]
)


rebuilt_clean_max_627g3 = (
    combined_reliability_risk_627g3(
        calibrated_627g2e[
            "_u_clean"
        ].to_numpy(dtype=float),

        calibrated_627g2e[
            "_s_clean"
        ].to_numpy(dtype=float),
    )
)


rebuilt_corrupted_max_627g3 = (
    combined_reliability_risk_627g3(
        calibrated_627g2e[
            "_u_corrupted"
        ].to_numpy(dtype=float),

        calibrated_627g2e[
            "_s_corrupted"
        ].to_numpy(dtype=float),
    )
)


stored_clean_max_627g3 = (
    calibrated_627g2e[
        max_clean_col_627g3
    ].to_numpy(dtype=float)
)

stored_corrupted_max_627g3 = (
    calibrated_627g2e[
        max_corrupted_col_627g3
    ].to_numpy(dtype=float)
)


clean_max_error_627g3 = float(
    np.max(
        np.abs(
            rebuilt_clean_max_627g3
            -
            stored_clean_max_627g3
        )
    )
)

corrupted_max_error_627g3 = float(
    np.max(
        np.abs(
            rebuilt_corrupted_max_627g3
            -
            stored_corrupted_max_627g3
        )
    )
)


print(
    "\n--- Exact reproduction check ---"
)

print(
    "Maximum clean reproduction error     :",
    clean_max_error_627g3,
)

print(
    "Maximum corrupted reproduction error :",
    corrupted_max_error_627g3,
)


if (
    clean_max_error_627g3 > 1e-12
    or
    corrupted_max_error_627g3 > 1e-12
):
    raise RuntimeError(
        "Frozen G3 reliability rule does not "
        "reproduce G2-E results exactly."
    )


# ============================================================
# 8. Preserve selection evidence
# ============================================================

max_row_627g3 = (
    OVERALL_ECDF_FUSION_RESULTS_627G2E[
        OVERALL_ECDF_FUSION_RESULTS_627G2E[
            "candidate"
        ]
        == "max_OR"
    ]
    .iloc[0]
)


sensor_row_627g3 = (
    OVERALL_ECDF_FUSION_RESULTS_627G2E[
        OVERALL_ECDF_FUSION_RESULTS_627G2E[
            "candidate"
        ]
        == "sensor_only"
    ]
    .iloc[0]
)


FROZEN_RELIABILITY_SELECTION_EVIDENCE_627G3 = {
    "max_mean_auc":
        float(
            max_row_627g3[
                "mean_auc"
            ]
        ),

    "sensor_only_mean_auc":
        float(
            sensor_row_627g3[
                "mean_auc"
            ]
        ),

    "mean_auc_delta_vs_sensor":
        float(
            max_row_627g3[
                "mean_auc"
            ]
            -
            sensor_row_627g3[
                "mean_auc"
            ]
        ),

    "bootstrap_interval_vs_sensor":
        tuple(
            float(x)
            for x in ci_sensor_627g2f
        ),

    "bootstrap_interval_vs_25U75S":
        tuple(
            float(x)
            for x in ci_25u75s_627g2f
        ),

    "all_LOO_deltas_vs_sensor_positive":
        bool(
            (
                LOO_FUSION_AUDIT_627G2F[
                    "mean_delta_vs_sensor"
                ]
                > 0
            ).all()
        ),

    "all_LOO_deltas_vs_25U75S_positive":
        bool(
            (
                LOO_FUSION_AUDIT_627G2F[
                    "mean_delta_vs_25U75S"
                ]
                > 0
            ).all()
        ),
}


# ============================================================
# 9. Final frozen-state audit
# ============================================================

all_rebuilt_finite_627g3 = bool(
    np.isfinite(
        rebuilt_clean_max_627g3
    ).all()
    and
    np.isfinite(
        rebuilt_corrupted_max_627g3
    ).all()
)

all_rebuilt_in_range_627g3 = bool(
    (
        (
            rebuilt_clean_max_627g3
            >= 0
        )
        &
        (
            rebuilt_clean_max_627g3
            <= 1
        )
    ).all()
    and
    (
        (
            rebuilt_corrupted_max_627g3
            >= 0
        )
        &
        (
            rebuilt_corrupted_max_627g3
            <= 1
        )
    ).all()
)


print(
    "\n"
    "============================================================\n"
    "STEP 6.27G3 FINAL FREEZE SUMMARY\n"
    "============================================================"
)

print(
    "Calibration population       : CLEAN DEV endpoints"
)

print(
    "Calibration endpoints        :",
    len(
        RELIABILITY_UNCERTAINTY_REFERENCE_627G3
    ),
)

print(
    "Calibration                  : empirical mid-rank percentile"
)

print(
    "Uncertainty orientation      : HIGHER = WORSE"
)

print(
    "Sensor orientation           : HIGHER ANOMALY = WORSE"
)

print(
    "Fusion                       : MAX / OR"
)

print(
    "Combined risk orientation    : HIGHER = WORSE"
)

print(
    "MAX mean AUC                 :",
    FROZEN_RELIABILITY_SELECTION_EVIDENCE_627G3[
        "max_mean_auc"
    ],
)

print(
    "Sensor-only mean AUC         :",
    FROZEN_RELIABILITY_SELECTION_EVIDENCE_627G3[
        "sensor_only_mean_auc"
    ],
)

print(
    "MAX - sensor mean AUC delta  :",
    FROZEN_RELIABILITY_SELECTION_EVIDENCE_627G3[
        "mean_auc_delta_vs_sensor"
    ],
)

print(
    "All LOO vs sensor positive   :",
    FROZEN_RELIABILITY_SELECTION_EVIDENCE_627G3[
        "all_LOO_deltas_vs_sensor_positive"
    ],
)

print(
    "Exact G2-E reproduction      :",
    (
        clean_max_error_627g3 <= 1e-12
        and
        corrupted_max_error_627g3 <= 1e-12
    ),
)

print(
    "All frozen risks finite      :",
    all_rebuilt_finite_627g3,
)

print(
    "All frozen risks within [0,1]:",
    all_rebuilt_in_range_627g3,
)

print(
    "Sensor-only retained ablation: True"
)

print(
    "Reliability representation frozen : True"
)

print(
    "Reliability threshold selected    : False"
)

print(
    "Official TEST labels inspected    : False"
)

print(
    "Official TEST scored              : False"
)

print(
    "Files written                     : False"
)

print(
    "\n=== END STEP 6.27G3 ==="
)

In [ ]:
# ============================================================
# STEP 6.27G3-R
# COMPACT FINAL FREEZE CONFIRMATION
# ============================================================

print(
    "\n"
    "============================================================\n"
    "STEP 6.27G3-R — RELIABILITY FREEZE CONFIRMATION\n"
    "============================================================"
)

print(
    "Frozen representation object exists :",
    "FINAL_RELIABILITY_REPRESENTATION_627G3"
    in globals(),
)

print(
    "Frozen evidence object exists       :",
    "FROZEN_RELIABILITY_SELECTION_EVIDENCE_627G3"
    in globals(),
)

print(
    "Uncertainty reference length        :",
    len(RELIABILITY_UNCERTAINTY_REFERENCE_627G3),
)

print(
    "Sensor reference length             :",
    len(RELIABILITY_SENSOR_REFERENCE_627G3),
)

print(
    "Clean reproduction error            :",
    clean_max_error_627g3,
)

print(
    "Corrupted reproduction error        :",
    corrupted_max_error_627g3,
)

print(
    "Fusion rule                         :",
    FINAL_RELIABILITY_REPRESENTATION_627G3[
        "fusion_rule"
    ],
)

print(
    "Combined orientation                :",
    FINAL_RELIABILITY_REPRESENTATION_627G3[
        "combined_orientation"
    ],
)

print(
    "Sensor-only retained as ablation    :",
    FINAL_RELIABILITY_REPRESENTATION_627G3[
        "sensor_only_retained_as_ablation"
    ],
)

print(
    "Threshold already selected          :",
    FINAL_RELIABILITY_REPRESENTATION_627G3[
        "threshold_selected"
    ],
)

print(
    "Official TEST used                  :",
    FINAL_RELIABILITY_REPRESENTATION_627G3[
        "official_test_used"
    ],
)

print(
    "\nReliability representation status   : FROZEN"
)

print(
    "Files written                       : False"
)

print(
    "\n=== END STEP 6.27G3-R ==="
)

In [ ]:
# ============================================================
# STEP 6.27H0
# DEV RUL-TARGET PROVENANCE AUDIT
# ============================================================

import numpy as np
import pandas as pd
import inspect


print(
    "\n"
    "============================================================\n"
    "STEP 6.27H0 — DEV RUL-TARGET PROVENANCE AUDIT\n"
    "============================================================"
)


# ============================================================
# 1. Validation raw-base contract
# ============================================================

if "val_raw_base_620gc" not in globals():
    raise RuntimeError(
        "Missing frozen validation raw base: val_raw_base_620gc"
    )


val_base_627h0 = val_raw_base_620gc


print("\n--- FROZEN VALIDATION RAW BASE ---")

print(
    "Shape      :",
    val_base_627h0.shape,
)

print(
    "Engines    :",
    val_base_627h0["unit_id"].nunique()
    if "unit_id" in val_base_627h0.columns
    else "unit_id column not found",
)

print(
    "Columns    :",
    list(val_base_627h0.columns),
)


# ============================================================
# 2. Detect target-like columns in validation raw base
# ============================================================

target_keywords_627h0 = (
    "rul",
    "target",
    "label",
    "remaining",
    "life",
)


target_like_columns_627h0 = [
    col
    for col in val_base_627h0.columns
    if any(
        keyword in str(col).lower()
        for keyword in target_keywords_627h0
    )
]


print(
    "\n--- TARGET-LIKE RAW-BASE COLUMNS ---"
)

print(
    target_like_columns_627h0
    if target_like_columns_627h0
    else "NONE FOUND"
)


for col in target_like_columns_627h0:

    values = pd.to_numeric(
        val_base_627h0[col],
        errors="coerce",
    )

    print(
        f"\n{col}:"
    )

    print(
        "  finite :",
        bool(
            np.isfinite(
                values.to_numpy(dtype=float)
            ).all()
        ),
    )

    print(
        "  min    :",
        float(values.min()),
    )

    print(
        "  median :",
        float(values.median()),
    )

    print(
        "  max    :",
        float(values.max()),
    )


# ============================================================
# 3. Known candidate: y_val_reference_624c
# ============================================================

print(
    "\n--- KNOWN VALIDATION TARGET CANDIDATE ---"
)


if "y_val_reference_624c" in globals():

    y_ref_627h0 = np.asarray(
        y_val_reference_624c
    ).reshape(-1)

    print(
        "Object exists : True"
    )

    print(
        "Shape         :",
        y_ref_627h0.shape,
    )

    print(
        "dtype         :",
        y_ref_627h0.dtype,
    )

    print(
        "finite        :",
        bool(
            np.isfinite(
                y_ref_627h0.astype(float)
            ).all()
        ),
    )

    print(
        "minimum       :",
        float(
            np.min(
                y_ref_627h0
            )
        ),
    )

    print(
        "median        :",
        float(
            np.median(
                y_ref_627h0
            )
        ),
    )

    print(
        "maximum       :",
        float(
            np.max(
                y_ref_627h0
            )
        ),
    )

    print(
        "first 10      :",
        y_ref_627h0[:10],
    )

    print(
        "last 10       :",
        y_ref_627h0[-10:],
    )

else:

    print(
        "Object exists : False"
    )


# ============================================================
# 4. Inventory target-like 3490-observation objects
#
# We only print plausible validation endpoint targets.
# No TEST object is selected or scored.
# ============================================================

candidate_records_627h0 = []


for name, obj in list(globals().items()):

    lname = name.lower()

    if (
        "test" in lname
        or "official" in lname
    ):
        continue

    targetish_name = any(
        key in lname
        for key in (
            "rul",
            "target",
            "label",
            "y_val",
            "reference",
        )
    )

    if not targetish_name:
        continue

    try:

        if isinstance(
            obj,
            np.ndarray,
        ):

            arr = np.asarray(
                obj
            )

            if (
                arr.ndim == 1
                and len(arr) == 3490
            ):

                numeric = arr.astype(float)

                candidate_records_627h0.append(
                    {
                        "object_name":
                            name,

                        "type":
                            "ndarray",

                        "shape":
                            str(arr.shape),

                        "minimum":
                            float(
                                np.min(numeric)
                            ),

                        "median":
                            float(
                                np.median(numeric)
                            ),

                        "maximum":
                            float(
                                np.max(numeric)
                            ),

                        "finite":
                            bool(
                                np.isfinite(
                                    numeric
                                ).all()
                            ),
                    }
                )

        elif isinstance(
            obj,
            pd.Series,
        ):

            if len(obj) == 3490:

                numeric = pd.to_numeric(
                    obj,
                    errors="coerce",
                ).to_numpy(
                    dtype=float
                )

                if np.isfinite(
                    numeric
                ).all():

                    candidate_records_627h0.append(
                        {
                            "object_name":
                                name,

                            "type":
                                "Series",

                            "shape":
                                str(obj.shape),

                            "minimum":
                                float(
                                    np.min(
                                        numeric
                                    )
                                ),

                            "median":
                                float(
                                    np.median(
                                        numeric
                                    )
                                ),

                            "maximum":
                                float(
                                    np.max(
                                        numeric
                                    )
                                ),

                            "finite":
                                True,
                        }
                    )

    except Exception:
        pass


TARGET_CANDIDATE_INVENTORY_627H0 = (
    pd.DataFrame(
        candidate_records_627h0
    )
)


print(
    "\n--- 3490-ROW TARGET-LIKE OBJECTS ---"
)


if len(
    TARGET_CANDIDATE_INVENTORY_627H0
) == 0:

    print(
        "NONE FOUND"
    )

else:

    print(
        TARGET_CANDIDATE_INVENTORY_627H0.to_string(
            index=False
        )
    )


# ============================================================
# 5. Search scalar variables that may store the target-column
#    name used earlier in the notebook
# ============================================================

scalar_target_records_627h0 = []


for name, obj in list(globals().items()):

    lname = name.lower()

    if (
        "test" in lname
        or "official" in lname
    ):
        continue

    if not any(
        key in lname
        for key in (
            "target",
            "rul",
            "label",
        )
    ):
        continue

    if isinstance(
        obj,
        str,
    ):

        scalar_target_records_627h0.append(
            {
                "object_name":
                    name,

                "value":
                    obj,

                "is_validation_column":
                    bool(
                        obj
                        in val_base_627h0.columns
                    ),
            }
        )


TARGET_NAME_INVENTORY_627H0 = (
    pd.DataFrame(
        scalar_target_records_627h0
    )
)


print(
    "\n--- TARGET-NAME VARIABLES ---"
)


if len(
    TARGET_NAME_INVENTORY_627H0
) == 0:

    print(
        "NONE FOUND"
    )

else:

    print(
        TARGET_NAME_INVENTORY_627H0.to_string(
            index=False
        )
    )


# ============================================================
# 6. Inspect known target-building functions
# ============================================================

function_names_627h0 = [
    "endpoint_targets_624c",
    "create_sequences",
    "create_sequences_with_metadata",
]


print(
    "\n--- TARGET-BUILDING FUNCTION CONTRACTS ---"
)


for function_name in function_names_627h0:

    if function_name not in globals():

        print(
            f"{function_name}: NOT FOUND"
        )

        continue

    func = globals()[
        function_name
    ]

    print(
        f"\n{function_name}"
    )

    try:

        print(
            "Signature:",
            inspect.signature(
                func
            ),
        )

    except Exception as exc:

        print(
            "Signature unavailable:",
            exc,
        )

    try:

        source = inspect.getsource(
            func
        )

        source_lines = (
            source.splitlines()
        )

        print(
            "Source:"
        )

        for line in source_lines[:40]:
            print(
                "   ",
                line,
            )

        if len(
            source_lines
        ) > 40:

            print(
                "   ... source truncated ..."
            )

    except Exception as exc:

        print(
            "Source unavailable:",
            exc,
        )


# ============================================================
# 7. No research decision yet
# ============================================================

print(
    "\n"
    "============================================================\n"
    "STEP 6.27H0 SUMMARY\n"
    "============================================================"
)

print(
    "Validation raw base inspected    : True"
)

print(
    "Known DEV target candidate checked:",
    "y_val_reference_624c"
    in globals(),
)

print(
    "Official TEST labels inspected   : False"
)

print(
    "Official TEST scored             : False"
)

print(
    "Reliability representation changed: False"
)

print(
    "Reliability threshold selected   : False"
)

print(
    "Files written                    : False"
)

print(
    "\n=== END STEP 6.27H0 ==="
)

In [ ]:
# ============================================================
# STEP 6.27H0-R
# EXACT DEV TARGET-CONVENTION CLOSURE
# ============================================================

import numpy as np
import pandas as pd

print(
    "\n"
    "============================================================\n"
    "STEP 6.27H0-R — EXACT DEV TARGET-CONVENTION CLOSURE\n"
    "============================================================"
)


# ============================================================
# 1. Required objects
# ============================================================

required_627h0r = [
    "val_raw_base_620gc",
    "y_val_reference_624c",
    "X_val_clean",
]

print("\n--- Required objects ---")

for name in required_627h0r:

    exists = name in globals()

    print(
        f"{name:<25}: {exists}"
    )

    if not exists:
        raise RuntimeError(
            f"Missing required object: {name}"
        )


# ============================================================
# 2. Frozen structural contract
# ============================================================

raw_627h0r = (
    val_raw_base_620gc
    .copy()
)

y_ref_627h0r = (
    np.asarray(
        y_val_reference_624c,
        dtype=float,
    )
    .reshape(-1)
)

window_length_627h0r = 30


print("\n--- Structural contract ---")

print(
    "Raw validation rows       :",
    len(raw_627h0r),
)

print(
    "Validation engines        :",
    raw_627h0r[
        "unit_id"
    ].nunique(),
)

print(
    "Frozen X_val_clean shape  :",
    X_val_clean.shape,
)

print(
    "y_val_reference shape     :",
    y_ref_627h0r.shape,
)


# Expected rolling-window count:
# sum_i (engine_length_i - window + 1)

engine_lengths_627h0r = (
    raw_627h0r
    .groupby(
        "unit_id"
    )
    .size()
)

expected_windows_627h0r = int(
    (
        engine_lengths_627h0r
        -
        window_length_627h0r
        +
        1
    ).sum()
)


print(
    "Expected 30-cycle windows :",
    expected_windows_627h0r,
)


if expected_windows_627h0r != 3490:
    raise RuntimeError(
        "Expected validation window count is not 3490."
    )

if len(y_ref_627h0r) != 3490:
    raise RuntimeError(
        "y_val_reference does not contain 3490 labels."
    )


# ============================================================
# 3. Check engine/cycle ordering
# ============================================================

duplicate_keys_627h0r = int(
    raw_627h0r
    .duplicated(
        subset=[
            "unit_id",
            "cycle",
        ]
    )
    .sum()
)

noncontiguous_627h0r = 0

for _, group in raw_627h0r.groupby(
    "unit_id",
    sort=False,
):

    cycles = (
        group["cycle"]
        .to_numpy()
    )

    if not np.all(
        np.diff(cycles) == 1
    ):
        noncontiguous_627h0r += 1


print(
    "Duplicate unit/cycle keys :",
    duplicate_keys_627h0r,
)

print(
    "Non-contiguous engines    :",
    noncontiguous_627h0r,
)


if duplicate_keys_627h0r != 0:
    raise RuntimeError(
        "Duplicate validation unit/cycle keys found."
    )

if noncontiguous_627h0r != 0:
    raise RuntimeError(
        "Validation cycles are not contiguous."
    )


# ============================================================
# 4. Independently reconstruct endpoint targets
# ============================================================

candidate_columns_627h0r = [
    "RUL_raw",
    "RUL_cap_125",
    "RUL_cap_130",
]


missing_columns_627h0r = [
    col
    for col in candidate_columns_627h0r
    if col not in raw_627h0r.columns
]

if missing_columns_627h0r:
    raise RuntimeError(
        "Missing target columns: "
        + str(
            missing_columns_627h0r
        )
    )


reconstructed_targets_627h0r = {}
endpoint_keys_627h0r = []


for unit_id, group in raw_627h0r.groupby(
    "unit_id",
    sort=False,
):

    group = (
        group
        .sort_values(
            "cycle"
        )
        .reset_index(
            drop=True
        )
    )

    endpoint_part = (
        group[
            [
                "unit_id",
                "cycle",
            ]
        ]
        .iloc[
            window_length_627h0r - 1 :
        ]
        .copy()
    )

    endpoint_keys_627h0r.append(
        endpoint_part
    )


ENDPOINT_KEYS_627H0R = (
    pd.concat(
        endpoint_keys_627h0r,
        ignore_index=True,
    )
)


for target_col in candidate_columns_627h0r:

    pieces = []

    for _, group in raw_627h0r.groupby(
        "unit_id",
        sort=False,
    ):

        group = (
            group
            .sort_values(
                "cycle"
            )
            .reset_index(
                drop=True
            )
        )

        values = (
            group[
                target_col
            ]
            .iloc[
                window_length_627h0r - 1 :
            ]
            .to_numpy(
                dtype=float
            )
        )

        pieces.append(
            values
        )

    reconstructed_targets_627h0r[
        target_col
    ] = np.concatenate(
        pieces
    )


# ============================================================
# 5. Exact comparison against frozen y_val_reference
# ============================================================

comparison_records_627h0r = []


for target_col in candidate_columns_627h0r:

    candidate = (
        reconstructed_targets_627h0r[
            target_col
        ]
    )

    if candidate.shape != y_ref_627h0r.shape:
        raise RuntimeError(
            f"{target_col} shape mismatch."
        )

    abs_diff = np.abs(
        candidate
        -
        y_ref_627h0r
    )

    comparison_records_627h0r.append(
        {
            "candidate":
                target_col,

            "rows":
                len(candidate),

            "maximum_abs_error":
                float(
                    np.max(
                        abs_diff
                    )
                ),

            "mean_abs_error":
                float(
                    np.mean(
                        abs_diff
                    )
                ),

            "exact_match":
                bool(
                    np.array_equal(
                        candidate,
                        y_ref_627h0r,
                    )
                ),

            "allclose_1e-12":
                bool(
                    np.allclose(
                        candidate,
                        y_ref_627h0r,
                        atol=1e-12,
                        rtol=0.0,
                    )
                ),

            "candidate_min":
                float(
                    np.min(
                        candidate
                    )
                ),

            "candidate_median":
                float(
                    np.median(
                        candidate
                    )
                ),

            "candidate_max":
                float(
                    np.max(
                        candidate
                    )
                ),
        }
    )


TARGET_MATCH_AUDIT_627H0R = (
    pd.DataFrame(
        comparison_records_627h0r
    )
)


print(
    "\n--- TARGET MATCH RESULTS ---"
)

print(
    TARGET_MATCH_AUDIT_627H0R.to_string(
        index=False,
        float_format=lambda x:
            f"{x:.12f}",
    )
)


# ============================================================
# 6. Resolve target convention only if exactly one matches
# ============================================================

matching_targets_627h0r = (
    TARGET_MATCH_AUDIT_627H0R.loc[
        TARGET_MATCH_AUDIT_627H0R[
            "allclose_1e-12"
        ],
        "candidate",
    ]
    .tolist()
)


print(
    "\n--- TARGET RESOLUTION ---"
)

print(
    "Matching candidates :",
    matching_targets_627h0r,
)


if len(
    matching_targets_627h0r
) == 1:

    FROZEN_DEV_TARGET_COLUMN_627H0R = (
        matching_targets_627h0r[0]
    )

    FROZEN_DEV_TARGET_VALUES_627H0R = (
        reconstructed_targets_627h0r[
            FROZEN_DEV_TARGET_COLUMN_627H0R
        ].copy()
    )

    print(
        "Resolved target     :",
        FROZEN_DEV_TARGET_COLUMN_627H0R,
    )

    print(
        "Resolution status   : UNIQUE EXACT MATCH"
    )

elif len(
    matching_targets_627h0r
) == 0:

    raise RuntimeError(
        "None of RUL_raw / RUL_cap_125 / "
        "RUL_cap_130 reproduces y_val_reference."
    )

else:

    raise RuntimeError(
        "More than one target definition reproduces "
        "y_val_reference. Further provenance is required."
    )


# ============================================================
# 7. Compact endpoint examples
# ============================================================

example_627h0r = (
    ENDPOINT_KEYS_627H0R
    .copy()
)

example_627h0r[
    "frozen_target"
] = (
    FROZEN_DEV_TARGET_VALUES_627H0R
)


print(
    "\n--- FIRST 5 ENDPOINT TARGETS ---"
)

print(
    example_627h0r
    .head(5)
    .to_string(
        index=False
    )
)


print(
    "\n--- LAST 5 ENDPOINT TARGETS ---"
)

print(
    example_627h0r
    .tail(5)
    .to_string(
        index=False
    )
)


# ============================================================
# 8. Final summary
# ============================================================

print(
    "\n"
    "============================================================\n"
    "STEP 6.27H0-R FINAL SUMMARY\n"
    "============================================================"
)

print(
    "Validation engines          :",
    raw_627h0r[
        "unit_id"
    ].nunique(),
)

print(
    "Rolling-window length       :",
    window_length_627h0r,
)

print(
    "Validation windows          :",
    expected_windows_627h0r,
)

print(
    "Frozen DEV target           :",
    FROZEN_DEV_TARGET_COLUMN_627H0R,
)

print(
    "Target matches reference    : True"
)

print(
    "Official TEST labels used   : False"
)

print(
    "Official TEST scored        : False"
)

print(
    "Reliability rule changed    : False"
)

print(
    "Threshold selected          : False"
)

print(
    "Files written               : False"
)

print(
    "\n=== END STEP 6.27H0-R ==="
)

In [ ]:
# ============================================================
# STEP 6.27H1
# BUILD CANONICAL CLEAN-DEV SELECTIVE-PREDICTION BASE
# ============================================================

import numpy as np
import pandas as pd


print(
    "\n"
    "============================================================\n"
    "STEP 6.27H1 — CLEAN DEV SELECTIVE-PREDICTION BASE\n"
    "============================================================"
)


# ============================================================
# 1. Required frozen objects
# ============================================================

required_627h1 = [
    "ENDPOINT_KEYS_627H0R",
    "FROZEN_DEV_TARGET_VALUES_627H0R",
    "FROZEN_DEV_TARGET_COLUMN_627H0R",
    "SENSOR_QUALITY_VAL_CLEAN_SCORED_626D",
    "FROZEN_VAL_UNCERTAINTY_622IF",
    "combined_reliability_risk_627g3",
]

print("\n--- Required frozen objects ---")

for name in required_627h1:

    exists = name in globals()

    print(
        f"{name:<42}: {exists}"
    )

    if not exists:
        raise RuntimeError(
            f"Missing required object: {name}"
        )


# ============================================================
# 2. Confirm frozen target
# ============================================================

if FROZEN_DEV_TARGET_COLUMN_627H0R != "RUL_cap_125":
    raise RuntimeError(
        "Unexpected DEV target convention. "
        "Expected RUL_cap_125."
    )


endpoint_keys_627h1 = (
    ENDPOINT_KEYS_627H0R[
        [
            "unit_id",
            "cycle",
        ]
    ]
    .copy()
    .reset_index(drop=True)
)


target_627h1 = np.asarray(
    FROZEN_DEV_TARGET_VALUES_627H0R,
    dtype=float,
).reshape(-1)


if len(endpoint_keys_627h1) != 3490:
    raise RuntimeError(
        "Endpoint-key table must contain 3490 rows."
    )

if len(target_627h1) != 3490:
    raise RuntimeError(
        "Frozen DEV target must contain 3490 values."
    )


print("\n--- Frozen target contract ---")

print(
    "Target column        :",
    FROZEN_DEV_TARGET_COLUMN_627H0R,
)

print(
    "Endpoint rows        :",
    len(endpoint_keys_627h1),
)

print(
    "Target values        :",
    len(target_627h1),
)

print(
    "Target min / max     :",
    float(target_627h1.min()),
    "/",
    float(target_627h1.max()),
)


# ============================================================
# 3. Recover existing clean prediction table
#
# We deliberately DO NOT run model inference here.
# ============================================================

prediction_candidate_names_627h1 = [
    "clean_prediction_df",
    "CLEAN_PREDICTION_DF",
    "CLEAN_PREDICTIONS_622IF",
    "CLEAN_VALIDATION_PREDICTIONS_622IF",
]


prediction_source_name_627h1 = None
prediction_source_627h1 = None


for name in prediction_candidate_names_627h1:

    if name not in globals():
        continue

    obj = globals()[name]

    if isinstance(obj, pd.DataFrame) and len(obj) == 3490:

        prediction_like = [
            col
            for col in obj.columns
            if (
                "predict" in str(col).lower()
                and "change" not in str(col).lower()
            )
        ]

        if prediction_like:

            prediction_source_name_627h1 = name
            prediction_source_627h1 = obj.copy()
            break


print("\n--- Existing prediction source ---")

print(
    "Recovered source :",
    prediction_source_name_627h1,
)


# ============================================================
# 4. If known-name lookup failed, perform constrained search
# ============================================================

if prediction_source_627h1 is None:

    plausible_627h1 = []

    for name, obj in list(globals().items()):

        if not isinstance(obj, pd.DataFrame):
            continue

        if len(obj) != 3490:
            continue

        lname = name.lower()

        if "test" in lname or "official" in lname:
            continue

        pred_cols = [
            col
            for col in obj.columns
            if (
                "predict" in str(col).lower()
                and "change" not in str(col).lower()
            )
        ]

        if pred_cols:

            plausible_627h1.append(
                (
                    name,
                    pred_cols,
                    obj.shape,
                )
            )


    print(
        "\nCandidate 3490-row prediction tables:"
    )

    for item in plausible_627h1:
        print(item)

    if len(plausible_627h1) != 1:

        raise RuntimeError(
            "Could not uniquely recover the frozen clean "
            "3490-row prediction table. Refusing to guess."
        )

    prediction_source_name_627h1 = (
        plausible_627h1[0][0]
    )

    prediction_source_627h1 = (
        globals()[
            prediction_source_name_627h1
        ].copy()
    )


# ============================================================
# 5. Resolve deterministic prediction column
# ============================================================

prediction_columns_627h1 = [
    col
    for col in prediction_source_627h1.columns
    if (
        "predict" in str(col).lower()
        and "change" not in str(col).lower()
    )
]


preferred_prediction_names_627h1 = [
    "prediction",
    "clean_prediction",
    "deterministic_prediction",
    "predicted_rul",
]


resolved_prediction_column_627h1 = None


for preferred in preferred_prediction_names_627h1:

    if preferred in prediction_columns_627h1:

        resolved_prediction_column_627h1 = preferred
        break


if resolved_prediction_column_627h1 is None:

    if len(prediction_columns_627h1) == 1:

        resolved_prediction_column_627h1 = (
            prediction_columns_627h1[0]
        )

    else:

        raise RuntimeError(
            "Multiple plausible prediction columns found: "
            + str(prediction_columns_627h1)
        )


clean_prediction_627h1 = pd.to_numeric(
    prediction_source_627h1[
        resolved_prediction_column_627h1
    ],
    errors="coerce",
).to_numpy(dtype=float)


if not np.isfinite(
    clean_prediction_627h1
).all():
    raise RuntimeError(
        "Recovered clean predictions contain "
        "non-finite values."
    )


print(
    "Prediction column :",
    resolved_prediction_column_627h1,
)

print(
    "Prediction count  :",
    len(clean_prediction_627h1),
)


# ============================================================
# 6. Recover clean predictive uncertainty
# ============================================================

clean_uncertainty_627h1 = np.asarray(
    FROZEN_VAL_UNCERTAINTY_622IF,
    dtype=float,
).reshape(-1)


if len(clean_uncertainty_627h1) != 3490:
    raise RuntimeError(
        "Frozen clean uncertainty must contain 3490 values."
    )

if not np.isfinite(
    clean_uncertainty_627h1
).all():
    raise RuntimeError(
        "Frozen clean uncertainty contains non-finite values."
    )


# ============================================================
# 7. Recover clean sensor-quality / anomaly values
# ============================================================

sq_clean_627h1 = (
    SENSOR_QUALITY_VAL_CLEAN_SCORED_626D
    .copy()
    .reset_index(drop=True)
)


if len(sq_clean_627h1) != 3490:
    raise RuntimeError(
        "Clean sensor-quality table must contain 3490 rows."
    )


required_sq_cols_627h1 = [
    "unit_id",
    "cycle",
    "sq_candidate_max_fusion",
]


missing_sq_627h1 = [
    col
    for col in required_sq_cols_627h1
    if col not in sq_clean_627h1.columns
]


if missing_sq_627h1:
    raise RuntimeError(
        "Missing clean sensor-quality columns: "
        + str(missing_sq_627h1)
    )


# ============================================================
# 8. Exact endpoint-key alignment
# ============================================================

sq_keys_627h1 = (
    sq_clean_627h1[
        [
            "unit_id",
            "cycle",
        ]
    ]
    .reset_index(drop=True)
)


keys_exact_627h1 = bool(
    endpoint_keys_627h1.equals(
        sq_keys_627h1
    )
)


print("\n--- Endpoint alignment ---")

print(
    "Target/SQ keys exact :",
    keys_exact_627h1,
)


if not keys_exact_627h1:

    raise RuntimeError(
        "Target endpoint keys and sensor-quality endpoint "
        "keys are not exactly aligned."
    )


# ============================================================
# 9. Recover RAW sensor anomaly orientation
#
# sq_candidate_max_fusion was previously confirmed to be
# QUALITY-oriented:
#
#     higher = better
#
# Frozen G3 sensor input requires:
#
#     higher anomaly = worse
#
# Therefore:
#     raw sensor anomaly = 1 - quality
#
# ============================================================

clean_sensor_quality_627h1 = pd.to_numeric(
    sq_clean_627h1[
        "sq_candidate_max_fusion"
    ],
    errors="coerce",
).to_numpy(dtype=float)


if not (
    np.isfinite(
        clean_sensor_quality_627h1
    ).all()
    and
    (
        clean_sensor_quality_627h1 >= 0
    ).all()
    and
    (
        clean_sensor_quality_627h1 <= 1
    ).all()
):
    raise RuntimeError(
        "Clean sensor-quality values violate [0,1] contract."
    )


clean_sensor_anomaly_raw_627h1 = (
    1.0
    -
    clean_sensor_quality_627h1
)


# ============================================================
# 10. Frozen combined reliability risk
# ============================================================

clean_combined_risk_627h1 = (
    combined_reliability_risk_627g3(
        clean_uncertainty_627h1,
        clean_sensor_anomaly_raw_627h1,
    )
)


if len(clean_combined_risk_627h1) != 3490:
    raise RuntimeError(
        "Combined clean reliability risk has wrong length."
    )


if not (
    np.isfinite(
        clean_combined_risk_627h1
    ).all()
    and
    (
        clean_combined_risk_627h1 >= 0
    ).all()
    and
    (
        clean_combined_risk_627h1 <= 1
    ).all()
):
    raise RuntimeError(
        "Combined reliability risk violates [0,1] contract."
    )


# ============================================================
# 11. Build canonical clean DEV selective-prediction table
# ============================================================

CLEAN_DEV_SELECTIVE_BASE_627H1 = (
    endpoint_keys_627h1.copy()
)


CLEAN_DEV_SELECTIVE_BASE_627H1[
    "target_rul_cap_125"
] = target_627h1


CLEAN_DEV_SELECTIVE_BASE_627H1[
    "prediction"
] = clean_prediction_627h1


CLEAN_DEV_SELECTIVE_BASE_627H1[
    "signed_error"
] = (
    clean_prediction_627h1
    -
    target_627h1
)


CLEAN_DEV_SELECTIVE_BASE_627H1[
    "absolute_error"
] = np.abs(
    CLEAN_DEV_SELECTIVE_BASE_627H1[
        "signed_error"
    ].to_numpy()
)


CLEAN_DEV_SELECTIVE_BASE_627H1[
    "predictive_uncertainty_raw"
] = clean_uncertainty_627h1


CLEAN_DEV_SELECTIVE_BASE_627H1[
    "sensor_quality_raw"
] = clean_sensor_quality_627h1


CLEAN_DEV_SELECTIVE_BASE_627H1[
    "sensor_anomaly_raw"
] = clean_sensor_anomaly_raw_627h1


CLEAN_DEV_SELECTIVE_BASE_627H1[
    "combined_reliability_risk"
] = clean_combined_risk_627h1


# ============================================================
# 12. Structural integrity
# ============================================================

duplicate_keys_627h1 = int(
    CLEAN_DEV_SELECTIVE_BASE_627H1
    .duplicated(
        subset=[
            "unit_id",
            "cycle",
        ]
    )
    .sum()
)


all_numeric_finite_627h1 = bool(
    np.isfinite(
        CLEAN_DEV_SELECTIVE_BASE_627H1[
            [
                "target_rul_cap_125",
                "prediction",
                "signed_error",
                "absolute_error",
                "predictive_uncertainty_raw",
                "sensor_quality_raw",
                "sensor_anomaly_raw",
                "combined_reliability_risk",
            ]
        ]
        .to_numpy(dtype=float)
    ).all()
)


if duplicate_keys_627h1 != 0:
    raise RuntimeError(
        "Duplicate clean DEV endpoint keys detected."
    )

if not all_numeric_finite_627h1:
    raise RuntimeError(
        "Non-finite values detected in canonical DEV table."
    )


# ============================================================
# 13. Compact diagnostic summary
# ============================================================

print(
    "\n--- Canonical clean DEV table ---"
)

print(
    "Shape               :",
    CLEAN_DEV_SELECTIVE_BASE_627H1.shape,
)

print(
    "Engines             :",
    CLEAN_DEV_SELECTIVE_BASE_627H1[
        "unit_id"
    ].nunique(),
)

print(
    "Duplicate keys      :",
    duplicate_keys_627h1,
)

print(
    "All numeric finite  :",
    all_numeric_finite_627h1,
)


baseline_mae_627h1 = float(
    CLEAN_DEV_SELECTIVE_BASE_627H1[
        "absolute_error"
    ].mean()
)


baseline_rmse_627h1 = float(
    np.sqrt(
        np.mean(
            np.square(
                CLEAN_DEV_SELECTIVE_BASE_627H1[
                    "signed_error"
                ].to_numpy(dtype=float)
            )
        )
    )
)


print(
    "Full-coverage MAE   :",
    baseline_mae_627h1,
)

print(
    "Full-coverage RMSE  :",
    baseline_rmse_627h1,
)

print(
    "Combined risk min   :",
    float(
        CLEAN_DEV_SELECTIVE_BASE_627H1[
            "combined_reliability_risk"
        ].min()
    ),
)

print(
    "Combined risk median:",
    float(
        CLEAN_DEV_SELECTIVE_BASE_627H1[
            "combined_reliability_risk"
        ].median()
    ),
)

print(
    "Combined risk max   :",
    float(
        CLEAN_DEV_SELECTIVE_BASE_627H1[
            "combined_reliability_risk"
        ].max()
    ),
)


print(
    "\n--- First 5 canonical records ---"
)

print(
    CLEAN_DEV_SELECTIVE_BASE_627H1
    .head(5)
    .to_string(index=False)
)


# ============================================================
# 14. Final summary
# ============================================================

print(
    "\n"
    "============================================================\n"
    "STEP 6.27H1 FINAL SUMMARY\n"
    "============================================================"
)

print(
    "Frozen DEV target                : RUL_cap_125"
)

print(
    "Clean DEV windows                :",
    len(CLEAN_DEV_SELECTIVE_BASE_627H1),
)

print(
    "Clean DEV engines                :",
    CLEAN_DEV_SELECTIVE_BASE_627H1[
        "unit_id"
    ].nunique(),
)

print(
    "Prediction source                :",
    prediction_source_name_627h1,
)

print(
    "Prediction column                :",
    resolved_prediction_column_627h1,
)

print(
    "Target/SQ endpoint alignment     :",
    keys_exact_627h1,
)

print(
    "Full-coverage MAE                :",
    baseline_mae_627h1,
)

print(
    "Full-coverage RMSE               :",
    baseline_rmse_627h1,
)

print(
    "Reliability representation frozen: True"
)

print(
    "Threshold selected               : False"
)

print(
    "Official TEST labels used        : False"
)

print(
    "Official TEST scored             : False"
)

print(
    "Model inference rerun            : False"
)

print(
    "MC-dropout rerun                 : False"
)

print(
    "Files written                    : False"
)

print(
    "\n=== END STEP 6.27H1 ==="
)

In [ ]:
# ============================================================
# STEP 6.27H1-R
# COMPACT CLEAN-DEV BASE CONFIRMATION
# ============================================================

print(
    "\n"
    "============================================================\n"
    "STEP 6.27H1-R — CLEAN DEV BASE CONFIRMATION\n"
    "============================================================"
)

print(
    "Canonical table exists        :",
    "CLEAN_DEV_SELECTIVE_BASE_627H1" in globals(),
)

print(
    "Table shape                   :",
    CLEAN_DEV_SELECTIVE_BASE_627H1.shape,
)

print(
    "DEV engines                   :",
    CLEAN_DEV_SELECTIVE_BASE_627H1["unit_id"].nunique(),
)

print(
    "Duplicate endpoint keys       :",
    CLEAN_DEV_SELECTIVE_BASE_627H1.duplicated(
        subset=["unit_id", "cycle"]
    ).sum(),
)

print(
    "Target/SQ alignment exact     :",
    keys_exact_627h1,
)

print(
    "Target convention             :",
    FROZEN_DEV_TARGET_COLUMN_627H0R,
)

print(
    "Prediction source             :",
    prediction_source_name_627h1,
)

print(
    "Prediction column             :",
    resolved_prediction_column_627h1,
)

print(
    "Full-coverage MAE             :",
    baseline_mae_627h1,
)

print(
    "Full-coverage RMSE            :",
    baseline_rmse_627h1,
)

print(
    "Combined-risk minimum         :",
    CLEAN_DEV_SELECTIVE_BASE_627H1[
        "combined_reliability_risk"
    ].min(),
)

print(
    "Combined-risk median          :",
    CLEAN_DEV_SELECTIVE_BASE_627H1[
        "combined_reliability_risk"
    ].median(),
)

print(
    "Combined-risk maximum         :",
    CLEAN_DEV_SELECTIVE_BASE_627H1[
        "combined_reliability_risk"
    ].max(),
)

print(
    "All combined risks finite     :",
    CLEAN_DEV_SELECTIVE_BASE_627H1[
        "combined_reliability_risk"
    ].notna().all(),
)

print(
    "All combined risks in [0,1]   :",
    CLEAN_DEV_SELECTIVE_BASE_627H1[
        "combined_reliability_risk"
    ].between(0, 1).all(),
)

print(
    "\nReliability representation     : FROZEN"
)

print(
    "Threshold selected            : False"
)

print(
    "Official TEST used            : False"
)

print(
    "Files written                 : False"
)

print(
    "\n=== END STEP 6.27H1-R ==="
)

In [ ]:
# ============================================================
# STEP 6.27H1-D
# COMBINED-RISK INPUT + ENDPOINT CALIBRATION AUDIT
# ============================================================

import numpy as np
import pandas as pd
import inspect


print(
    "\n"
    "============================================================\n"
    "STEP 6.27H1-D — COMBINED-RISK INPUT + ENDPOINT AUDIT\n"
    "============================================================"
)


# ============================================================
# 1. Required objects
# ============================================================

required_627h1d = [
    "CLEAN_DEV_SELECTIVE_BASE_627H1",
    "SENSOR_QUALITY_VAL_CLEAN_SCORED_626D",
    "FROZEN_VAL_UNCERTAINTY_622IF",
    "combined_reliability_risk_627g3",
    "FINAL_RELIABILITY_REPRESENTATION_627G3",
    "RELIABILITY_UNCERTAINTY_REFERENCE_627G3",
    "RELIABILITY_SENSOR_REFERENCE_627G3",
]


print("\n--- REQUIRED OBJECTS ---")

for name in required_627h1d:

    exists = name in globals()

    print(
        f"{name:<46}: {exists}"
    )

    if not exists:

        raise RuntimeError(
            f"Missing required object: {name}"
        )


# ============================================================
# 2. Frozen G3 representation metadata
# ============================================================

print(
    "\n--- FROZEN G3 REPRESENTATION ---"
)


representation_627h1d = (
    FINAL_RELIABILITY_REPRESENTATION_627G3
)


if isinstance(
    representation_627h1d,
    dict,
):

    for key, value in representation_627h1d.items():

        if isinstance(
            value,
            np.ndarray,
        ):

            print(
                f"{key:<36}: ndarray shape={value.shape}"
            )

        elif isinstance(
            value,
            pd.DataFrame,
        ):

            print(
                f"{key:<36}: DataFrame shape={value.shape}"
            )

        else:

            print(
                f"{key:<36}: {value}"
            )

else:

    print(
        "Representation type:",
        type(
            representation_627h1d
        ).__name__,
    )


# ============================================================
# 3. Exact combined-risk function source
# ============================================================

print(
    "\n--- COMBINED-RISK FUNCTION CONTRACT ---"
)

print(
    "Signature:",
    inspect.signature(
        combined_reliability_risk_627g3
    ),
)


try:

    source_627h1d = inspect.getsource(
        combined_reliability_risk_627g3
    )

    print(
        source_627h1d
    )

except Exception as exc:

    print(
        "Source unavailable:",
        exc,
    )


# ============================================================
# 4. Frozen reference distributions
# ============================================================

u_reference_627h1d = np.asarray(
    RELIABILITY_UNCERTAINTY_REFERENCE_627G3,
    dtype=float,
).reshape(-1)


s_reference_627h1d = np.asarray(
    RELIABILITY_SENSOR_REFERENCE_627G3,
    dtype=float,
).reshape(-1)


def compact_stats_627h1d(
    label,
    values,
):

    values = np.asarray(
        values,
        dtype=float,
    ).reshape(-1)

    print(
        f"{label:<34}"
        f" n={len(values):4d}"
        f" min={np.min(values):.6f}"
        f" q25={np.quantile(values, 0.25):.6f}"
        f" q50={np.quantile(values, 0.50):.6f}"
        f" q75={np.quantile(values, 0.75):.6f}"
        f" max={np.max(values):.6f}"
        f" mean={np.mean(values):.6f}"
    )


print(
    "\n--- FROZEN CALIBRATION REFERENCES ---"
)

compact_stats_627h1d(
    "Uncertainty reference",
    u_reference_627h1d,
)

compact_stats_627h1d(
    "Sensor reference",
    s_reference_627h1d,
)


# ============================================================
# 5. Clean validation raw channels
# ============================================================

u_clean_627h1d = np.asarray(
    FROZEN_VAL_UNCERTAINTY_622IF,
    dtype=float,
).reshape(-1)


sq_table_627h1d = (
    SENSOR_QUALITY_VAL_CLEAN_SCORED_626D
    .reset_index(
        drop=True
    )
)


q_clean_627h1d = pd.to_numeric(
    sq_table_627h1d[
        "sq_candidate_max_fusion"
    ],
    errors="coerce",
).to_numpy(
    dtype=float
)


anomaly_clean_627h1d = (
    1.0
    -
    q_clean_627h1d
)


print(
    "\n--- ALL 3490 CLEAN WINDOWS: RAW CHANNELS ---"
)

compact_stats_627h1d(
    "Predictive uncertainty",
    u_clean_627h1d,
)

compact_stats_627h1d(
    "Stored SQ fusion",
    q_clean_627h1d,
)

compact_stats_627h1d(
    "1 - stored SQ fusion",
    anomaly_clean_627h1d,
)


# ============================================================
# 6. Test BOTH sensor-input conventions
#
# Diagnostic only.
# No convention is changed or frozen here.
# ============================================================

risk_using_stored_sq_627h1d = np.asarray(
    combined_reliability_risk_627g3(
        u_clean_627h1d,
        q_clean_627h1d,
    ),
    dtype=float,
).reshape(-1)


risk_using_complement_627h1d = np.asarray(
    combined_reliability_risk_627g3(
        u_clean_627h1d,
        anomaly_clean_627h1d,
    ),
    dtype=float,
).reshape(-1)


print(
    "\n--- ALL 3490 WINDOWS: TWO INPUT CONVENTIONS ---"
)

compact_stats_627h1d(
    "Risk using stored SQ",
    risk_using_stored_sq_627h1d,
)

compact_stats_627h1d(
    "Risk using 1-SQ",
    risk_using_complement_627h1d,
)


print(
    "Stored-SQ fraction exactly 1 :",
    float(
        np.mean(
            risk_using_stored_sq_627h1d
            == 1.0
        )
    ),
)

print(
    "1-SQ fraction exactly 1      :",
    float(
        np.mean(
            risk_using_complement_627h1d
            == 1.0
        )
    ),
)


# ============================================================
# 7. Recover ONLY the last endpoint of each DEV engine
# ============================================================

endpoint_positions_627h1d = (
    CLEAN_DEV_SELECTIVE_BASE_627H1
    .reset_index()
    .groupby(
        "unit_id",
        sort=False,
    )
    .tail(1)[
        "index"
    ]
    .to_numpy(
        dtype=int
    )
)


if len(
    endpoint_positions_627h1d
) != 20:

    raise RuntimeError(
        "Expected exactly 20 DEV engine endpoints."
    )


endpoint_base_627h1d = (
    CLEAN_DEV_SELECTIVE_BASE_627H1
    .iloc[
        endpoint_positions_627h1d
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


endpoint_u_627h1d = (
    u_clean_627h1d[
        endpoint_positions_627h1d
    ]
)


endpoint_q_627h1d = (
    q_clean_627h1d[
        endpoint_positions_627h1d
    ]
)


endpoint_anomaly_627h1d = (
    anomaly_clean_627h1d[
        endpoint_positions_627h1d
    ]
)


endpoint_risk_stored_627h1d = np.asarray(
    combined_reliability_risk_627g3(
        endpoint_u_627h1d,
        endpoint_q_627h1d,
    ),
    dtype=float,
).reshape(-1)


endpoint_risk_complement_627h1d = np.asarray(
    combined_reliability_risk_627g3(
        endpoint_u_627h1d,
        endpoint_anomaly_627h1d,
    ),
    dtype=float,
).reshape(-1)


print(
    "\n--- CLEAN DEV ENGINE ENDPOINTS ONLY ---"
)

print(
    "Endpoint count:",
    len(
        endpoint_base_627h1d
    ),
)

compact_stats_627h1d(
    "Endpoint uncertainty",
    endpoint_u_627h1d,
)

compact_stats_627h1d(
    "Endpoint stored SQ",
    endpoint_q_627h1d,
)

compact_stats_627h1d(
    "Endpoint 1-SQ",
    endpoint_anomaly_627h1d,
)

compact_stats_627h1d(
    "Endpoint risk stored SQ",
    endpoint_risk_stored_627h1d,
)

compact_stats_627h1d(
    "Endpoint risk 1-SQ",
    endpoint_risk_complement_627h1d,
)


# ============================================================
# 8. Compare endpoint raw values directly with references
# ============================================================

def sorted_max_abs_error_627h1d(
    left,
    right,
):

    left = np.sort(
        np.asarray(
            left,
            dtype=float,
        ).reshape(-1)
    )

    right = np.sort(
        np.asarray(
            right,
            dtype=float,
        ).reshape(-1)
    )

    if len(left) != len(right):
        return np.nan

    return float(
        np.max(
            np.abs(
                left
                -
                right
            )
        )
    )


print(
    "\n--- ENDPOINT ↔ FROZEN REFERENCE MATCH ---"
)

print(
    "Uncertainty sorted max error :",
    sorted_max_abs_error_627h1d(
        endpoint_u_627h1d,
        u_reference_627h1d,
    ),
)

print(
    "Stored-SQ sorted max error   :",
    sorted_max_abs_error_627h1d(
        endpoint_q_627h1d,
        s_reference_627h1d,
    ),
)

print(
    "1-SQ sorted max error        :",
    sorted_max_abs_error_627h1d(
        endpoint_anomaly_627h1d,
        s_reference_627h1d,
    ),
)


# ============================================================
# 9. Endpoint RUL baseline
# ============================================================

endpoint_mae_627h1d = float(
    endpoint_base_627h1d[
        "absolute_error"
    ].mean()
)


endpoint_rmse_627h1d = float(
    np.sqrt(
        np.mean(
            np.square(
                endpoint_base_627h1d[
                    "signed_error"
                ].to_numpy(
                    dtype=float
                )
            )
        )
    )
)


print(
    "\n--- ENDPOINT RUL BASELINE ---"
)

print(
    "Endpoint MAE :",
    endpoint_mae_627h1d,
)

print(
    "Endpoint RMSE:",
    endpoint_rmse_627h1d,
)


# ============================================================
# 10. Final diagnostic summary
# ============================================================

print(
    "\n"
    "============================================================\n"
    "STEP 6.27H1-D SUMMARY\n"
    "============================================================"
)

print(
    "Rolling windows inspected     :",
    len(
        CLEAN_DEV_SELECTIVE_BASE_627H1
    ),
)

print(
    "DEV endpoints inspected       :",
    len(
        endpoint_base_627h1d
    ),
)

print(
    "Frozen calibration references :",
    len(
        u_reference_627h1d
    ),
)

print(
    "Threshold selected            : False"
)

print(
    "Reliability rule changed      : False"
)

print(
    "Official TEST used            : False"
)

print(
    "Files written                 : False"
)

print(
    "\n=== END STEP 6.27H1-D ==="
)

In [ ]:
# ============================================================
# STEP 6.27H1-D-R
# COMPACT COMBINED-RISK / ENDPOINT REVIEW
# ============================================================

import numpy as np


def compact_stats_627h1dr(label, values):

    values = np.asarray(
        values,
        dtype=float
    ).reshape(-1)

    print(
        f"{label:<31}"
        f" n={len(values):4d}"
        f" min={np.min(values):.6f}"
        f" q25={np.quantile(values, 0.25):.6f}"
        f" q50={np.quantile(values, 0.50):.6f}"
        f" q75={np.quantile(values, 0.75):.6f}"
        f" max={np.max(values):.6f}"
        f" mean={np.mean(values):.6f}"
    )


print(
    "\n"
    "============================================================\n"
    "STEP 6.27H1-D-R — COMPACT DIAGNOSTIC REVIEW\n"
    "============================================================"
)


# ------------------------------------------------------------
# 1. Frozen references
# ------------------------------------------------------------

print("\n--- FROZEN REFERENCES ---")

compact_stats_627h1dr(
    "Uncertainty reference",
    u_reference_627h1d
)

compact_stats_627h1dr(
    "Sensor reference",
    s_reference_627h1d
)


# ------------------------------------------------------------
# 2. All clean rolling windows
# ------------------------------------------------------------

print("\n--- ALL 3490 CLEAN WINDOWS ---")

compact_stats_627h1dr(
    "Raw uncertainty",
    u_clean_627h1d
)

compact_stats_627h1dr(
    "Stored SQ",
    q_clean_627h1d
)

compact_stats_627h1dr(
    "1 - stored SQ",
    anomaly_clean_627h1d
)

compact_stats_627h1dr(
    "Risk: stored SQ input",
    risk_using_stored_sq_627h1d
)

compact_stats_627h1dr(
    "Risk: 1-SQ input",
    risk_using_complement_627h1d
)


print(
    "\nStored-SQ risk == 1 fraction :",
    float(
        np.mean(
            risk_using_stored_sq_627h1d == 1.0
        )
    )
)

print(
    "1-SQ risk == 1 fraction      :",
    float(
        np.mean(
            risk_using_complement_627h1d == 1.0
        )
    )
)


# ------------------------------------------------------------
# 3. Clean DEV endpoints only
# ------------------------------------------------------------

print("\n--- 20 CLEAN DEV ENDPOINTS ---")

compact_stats_627h1dr(
    "Endpoint uncertainty",
    endpoint_u_627h1d
)

compact_stats_627h1dr(
    "Endpoint stored SQ",
    endpoint_q_627h1d
)

compact_stats_627h1dr(
    "Endpoint 1-SQ",
    endpoint_anomaly_627h1d
)

compact_stats_627h1dr(
    "Endpoint risk stored SQ",
    endpoint_risk_stored_627h1d
)

compact_stats_627h1dr(
    "Endpoint risk 1-SQ",
    endpoint_risk_complement_627h1d
)


# ------------------------------------------------------------
# 4. Exact reference matching
# ------------------------------------------------------------

print("\n--- REFERENCE MATCH ---")

print(
    "Uncertainty endpoint -> reference max error :",
    sorted_max_abs_error_627h1d(
        endpoint_u_627h1d,
        u_reference_627h1d
    )
)

print(
    "Stored SQ -> sensor reference max error     :",
    sorted_max_abs_error_627h1d(
        endpoint_q_627h1d,
        s_reference_627h1d
    )
)

print(
    "1-SQ -> sensor reference max error          :",
    sorted_max_abs_error_627h1d(
        endpoint_anomaly_627h1d,
        s_reference_627h1d
    )
)


# ------------------------------------------------------------
# 5. Endpoint RUL baseline
# ------------------------------------------------------------

print("\n--- ENDPOINT RUL BASELINE ---")

print(
    "Endpoint MAE :",
    endpoint_mae_627h1d
)

print(
    "Endpoint RMSE:",
    endpoint_rmse_627h1d
)


# ------------------------------------------------------------
# 6. Current status
# ------------------------------------------------------------

print(
    "\nCalibration population : CLEAN DEV ENDPOINTS"
)

print(
    "Calibration engines    : 20"
)

print(
    "Fusion                : maximum_OR"
)

print(
    "Higher risk           : worse"
)

print(
    "Threshold selected    : False"
)

print(
    "Official TEST used    : False"
)

print(
    "Research rule changed : False"
)

print(
    "\n=== END STEP 6.27H1-D-R ==="
)

In [ ]:
# ============================================================
# STEP 6.27H1-D2
# EXACT FROZEN-ENDPOINT + SENSOR-INPUT CONTRACT CLOSURE
# ============================================================

import numpy as np
import pandas as pd
import inspect


print(
    "\n"
    "============================================================\n"
    "STEP 6.27H1-D2 — EXACT FROZEN-ENDPOINT CONTRACT CLOSURE\n"
    "============================================================"
)


# ============================================================
# 1. Required objects
# ============================================================

required_627h1d2 = [
    "CLEAN_DEV_SELECTIVE_BASE_627H1",
    "clean_endpoint_scores_627g1",
    "reference_endpoint_scores_627g1",
    "orientation_check_627g1",
    "RELIABILITY_UNCERTAINTY_REFERENCE_627G3",
    "RELIABILITY_SENSOR_REFERENCE_627G3",
    "combined_reliability_risk_627g3",
]


print("\n--- REQUIRED OBJECTS ---")

for name in required_627h1d2:

    exists = name in globals()

    print(
        f"{name:<48}: {exists}"
    )

    if not exists:
        raise RuntimeError(
            f"Missing required object: {name}"
        )


# ============================================================
# 2. Recover the EXACT 20 G1 endpoint keys
# ============================================================

g1_endpoint_table_627h1d2 = (
    clean_endpoint_scores_627g1
    .copy()
)


required_key_cols_627h1d2 = [
    "unit_id",
    "cycle",
]


missing_keys_627h1d2 = [
    col
    for col in required_key_cols_627h1d2
    if col not in g1_endpoint_table_627h1d2.columns
]


if missing_keys_627h1d2:
    raise RuntimeError(
        "G1 endpoint table is missing key columns: "
        + str(missing_keys_627h1d2)
    )


FROZEN_DEV_ENDPOINT_KEYS_627H1D2 = (
    g1_endpoint_table_627h1d2[
        required_key_cols_627h1d2
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)


print("\n--- EXACT G1 ENDPOINT KEYS ---")

print(
    "Rows             :",
    len(FROZEN_DEV_ENDPOINT_KEYS_627H1D2)
)

print(
    "Unique engines   :",
    FROZEN_DEV_ENDPOINT_KEYS_627H1D2[
        "unit_id"
    ].nunique()
)

print(
    "Duplicate keys   :",
    FROZEN_DEV_ENDPOINT_KEYS_627H1D2
    .duplicated(
        subset=["unit_id", "cycle"]
    )
    .sum()
)


if len(FROZEN_DEV_ENDPOINT_KEYS_627H1D2) != 20:
    raise RuntimeError(
        "Expected exactly 20 frozen G1 DEV endpoints."
    )


if (
    FROZEN_DEV_ENDPOINT_KEYS_627H1D2[
        "unit_id"
    ].nunique()
    != 20
):
    raise RuntimeError(
        "Frozen G1 endpoint population does not contain "
        "one endpoint per DEV engine."
    )


# ============================================================
# 3. Extract those exact endpoints from canonical H1 base
# ============================================================

EXACT_CLEAN_DEV_ENDPOINTS_627H1D2 = (
    FROZEN_DEV_ENDPOINT_KEYS_627H1D2
    .merge(
        CLEAN_DEV_SELECTIVE_BASE_627H1,
        on=["unit_id", "cycle"],
        how="left",
        validate="one_to_one",
    )
)


if len(EXACT_CLEAN_DEV_ENDPOINTS_627H1D2) != 20:
    raise RuntimeError(
        "Exact endpoint merge produced wrong row count."
    )


if (
    EXACT_CLEAN_DEV_ENDPOINTS_627H1D2
    .isna()
    .any()
    .any()
):
    raise RuntimeError(
        "At least one frozen G1 endpoint could not be "
        "recovered from the canonical H1 base."
    )


print("\n--- EXACT ENDPOINT RECOVERY ---")

print(
    "Recovered rows      :",
    len(EXACT_CLEAN_DEV_ENDPOINTS_627H1D2)
)

print(
    "Recovered engines   :",
    EXACT_CLEAN_DEV_ENDPOINTS_627H1D2[
        "unit_id"
    ].nunique()
)

print(
    "All values present  :",
    not EXACT_CLEAN_DEV_ENDPOINTS_627H1D2
    .isna()
    .any()
    .any()
)


# ============================================================
# 4. Recover exact G1 reference sensor values at those keys
# ============================================================

reference_sq_627h1d2 = (
    reference_endpoint_scores_627g1
    .copy()
)


if "sq_candidate_max_fusion" not in reference_sq_627h1d2.columns:
    raise RuntimeError(
        "reference_endpoint_scores_627g1 does not contain "
        "sq_candidate_max_fusion."
    )


EXACT_G1_SENSOR_REFERENCE_ROWS_627H1D2 = (
    FROZEN_DEV_ENDPOINT_KEYS_627H1D2
    .merge(
        reference_sq_627h1d2[
            [
                "unit_id",
                "cycle",
                "sq_candidate_max_fusion",
            ]
        ],
        on=["unit_id", "cycle"],
        how="left",
        validate="one_to_one",
    )
)


if (
    EXACT_G1_SENSOR_REFERENCE_ROWS_627H1D2[
        "sq_candidate_max_fusion"
    ]
    .isna()
    .any()
):
    raise RuntimeError(
        "Could not recover exact G1 sensor reference values."
    )


# ============================================================
# 5. Exact endpoint arrays
# ============================================================

u_exact_627h1d2 = (
    EXACT_CLEAN_DEV_ENDPOINTS_627H1D2[
        "predictive_uncertainty_raw"
    ]
    .to_numpy(dtype=float)
)


q_exact_627h1d2 = (
    EXACT_CLEAN_DEV_ENDPOINTS_627H1D2[
        "sensor_quality_raw"
    ]
    .to_numpy(dtype=float)
)


a_exact_627h1d2 = (
    1.0
    -
    q_exact_627h1d2
)


q_g1_reference_627h1d2 = (
    EXACT_G1_SENSOR_REFERENCE_ROWS_627H1D2[
        "sq_candidate_max_fusion"
    ]
    .to_numpy(dtype=float)
)


u_frozen_reference_627h1d2 = np.asarray(
    RELIABILITY_UNCERTAINTY_REFERENCE_627G3,
    dtype=float,
).reshape(-1)


s_frozen_reference_627h1d2 = np.asarray(
    RELIABILITY_SENSOR_REFERENCE_627G3,
    dtype=float,
).reshape(-1)


# ============================================================
# 6. Comparison helper
# ============================================================

def sorted_max_error_627h1d2(left, right):

    left = np.sort(
        np.asarray(
            left,
            dtype=float
        ).reshape(-1)
    )

    right = np.sort(
        np.asarray(
            right,
            dtype=float
        ).reshape(-1)
    )

    if len(left) != len(right):
        return np.nan

    return float(
        np.max(
            np.abs(
                left - right
            )
        )
    )


print("\n--- EXACT REFERENCE MATCH TESTS ---")

print(
    "Exact endpoint uncertainty -> frozen U reference :",
    sorted_max_error_627h1d2(
        u_exact_627h1d2,
        u_frozen_reference_627h1d2,
    )
)

print(
    "Exact endpoint stored SQ -> frozen sensor ref     :",
    sorted_max_error_627h1d2(
        q_exact_627h1d2,
        s_frozen_reference_627h1d2,
    )
)

print(
    "Exact endpoint 1-SQ -> frozen sensor ref          :",
    sorted_max_error_627h1d2(
        a_exact_627h1d2,
        s_frozen_reference_627h1d2,
    )
)

print(
    "Exact G1 SQ reference -> frozen sensor ref        :",
    sorted_max_error_627h1d2(
        q_g1_reference_627h1d2,
        s_frozen_reference_627h1d2,
    )
)


# ============================================================
# 7. G1 orientation table
# ============================================================

print("\n--- G1 ORIENTATION CHECK ---")

print(
    orientation_check_627g1.to_string(
        index=False
    )
)


# ============================================================
# 8. Print exact frozen combined-function source
# ============================================================

print(
    "\n--- FROZEN COMBINED-RISK FUNCTION ---"
)

print(
    "Signature:",
    inspect.signature(
        combined_reliability_risk_627g3
    )
)


try:

    source_627h1d2 = inspect.getsource(
        combined_reliability_risk_627g3
    )

    print(source_627h1d2)

except Exception as exc:

    print(
        "Source unavailable:",
        exc
    )


# ============================================================
# 9. Evaluate BOTH conventions at the EXACT 20 frozen endpoints
# ============================================================

risk_q_exact_627h1d2 = np.asarray(
    combined_reliability_risk_627g3(
        u_exact_627h1d2,
        q_exact_627h1d2,
    ),
    dtype=float,
).reshape(-1)


risk_a_exact_627h1d2 = np.asarray(
    combined_reliability_risk_627g3(
        u_exact_627h1d2,
        a_exact_627h1d2,
    ),
    dtype=float,
).reshape(-1)


def summary_627h1d2(label, values):

    values = np.asarray(
        values,
        dtype=float
    ).reshape(-1)

    print(
        f"{label:<27}"
        f" min={np.min(values):.6f}"
        f" q25={np.quantile(values,0.25):.6f}"
        f" q50={np.quantile(values,0.50):.6f}"
        f" q75={np.quantile(values,0.75):.6f}"
        f" max={np.max(values):.6f}"
        f" mean={np.mean(values):.6f}"
    )


print(
    "\n--- EXACT 20-ENDPOINT COMBINED RISKS ---"
)

summary_627h1d2(
    "Using stored SQ",
    risk_q_exact_627h1d2,
)

summary_627h1d2(
    "Using 1-SQ",
    risk_a_exact_627h1d2,
)


# ============================================================
# 10. Exact endpoint RUL baseline
# ============================================================

exact_endpoint_mae_627h1d2 = float(
    EXACT_CLEAN_DEV_ENDPOINTS_627H1D2[
        "absolute_error"
    ].mean()
)


exact_endpoint_rmse_627h1d2 = float(
    np.sqrt(
        np.mean(
            np.square(
                EXACT_CLEAN_DEV_ENDPOINTS_627H1D2[
                    "signed_error"
                ]
                .to_numpy(dtype=float)
            )
        )
    )
)


print("\n--- EXACT FROZEN-ENDPOINT RUL BASELINE ---")

print(
    "MAE :",
    exact_endpoint_mae_627h1d2
)

print(
    "RMSE:",
    exact_endpoint_rmse_627h1d2
)


# ============================================================
# 11. Final status
# ============================================================

print(
    "\n"
    "============================================================\n"
    "STEP 6.27H1-D2 SUMMARY\n"
    "============================================================"
)

print(
    "Frozen endpoint keys recovered : True"
)

print(
    "Frozen endpoint count          :",
    len(FROZEN_DEV_ENDPOINT_KEYS_627H1D2)
)

print(
    "One endpoint per DEV engine    :",
    FROZEN_DEV_ENDPOINT_KEYS_627H1D2[
        "unit_id"
    ].nunique() == 20
)

print(
    "Threshold selected             : False"
)

print(
    "Reliability rule changed       : False"
)

print(
    "Official TEST used             : False"
)

print(
    "Model inference rerun          : False"
)

print(
    "MC-dropout rerun               : False"
)

print(
    "Files written                  : False"
)

print(
    "\n=== END STEP 6.27H1-D2 ==="
)

In [ ]:
# ============================================================
# STEP 6.27H1-D2-R
# FINAL COMPACT FROZEN-ENDPOINT CONTRACT REVIEW
# ============================================================

import numpy as np

print(
    "\n"
    "============================================================\n"
    "STEP 6.27H1-D2-R — FINAL CONTRACT REVIEW\n"
    "============================================================"
)


# ------------------------------------------------------------
# 1. Exact frozen-reference matching
# ------------------------------------------------------------

u_match_627h1d2r = sorted_max_error_627h1d2(
    u_exact_627h1d2,
    u_frozen_reference_627h1d2,
)

q_match_627h1d2r = sorted_max_error_627h1d2(
    q_exact_627h1d2,
    s_frozen_reference_627h1d2,
)

a_match_627h1d2r = sorted_max_error_627h1d2(
    a_exact_627h1d2,
    s_frozen_reference_627h1d2,
)

g1q_match_627h1d2r = sorted_max_error_627h1d2(
    q_g1_reference_627h1d2,
    s_frozen_reference_627h1d2,
)


print("\n--- EXACT REFERENCE MATCH ---")

print(
    "Uncertainty -> frozen U ref :",
    u_match_627h1d2r
)

print(
    "Stored SQ -> frozen S ref   :",
    q_match_627h1d2r
)

print(
    "1-SQ -> frozen S ref        :",
    a_match_627h1d2r
)

print(
    "G1 SQ -> frozen S ref       :",
    g1q_match_627h1d2r
)


# ------------------------------------------------------------
# 2. Exact 20-endpoint risk distributions
# ------------------------------------------------------------

def compact_627h1d2r(label, x):

    x = np.asarray(
        x,
        dtype=float
    ).reshape(-1)

    print(
        f"{label:<24}"
        f" min={x.min():.6f}"
        f" q25={np.quantile(x,0.25):.6f}"
        f" q50={np.quantile(x,0.50):.6f}"
        f" q75={np.quantile(x,0.75):.6f}"
        f" max={x.max():.6f}"
        f" mean={x.mean():.6f}"
    )


print("\n--- EXACT 20-ENDPOINT RISKS ---")

compact_627h1d2r(
    "Using stored SQ",
    risk_q_exact_627h1d2
)

compact_627h1d2r(
    "Using 1-SQ",
    risk_a_exact_627h1d2
)


# ------------------------------------------------------------
# 3. Compact orientation evidence
# ------------------------------------------------------------

print("\n--- FROZEN ORIENTATION EVIDENCE ---")

orientation_cols_627h1d2r = [
    col
    for col in orientation_check_627g1.columns
    if (
        "unit" not in str(col).lower()
        and
        "cycle" not in str(col).lower()
    )
]

print(
    orientation_check_627g1[
        orientation_cols_627h1d2r
    ]
    .head(20)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 4. Exact endpoint RUL baseline
# ------------------------------------------------------------

print("\n--- EXACT ENDPOINT RUL BASELINE ---")

print(
    "MAE  :",
    exact_endpoint_mae_627h1d2
)

print(
    "RMSE :",
    exact_endpoint_rmse_627h1d2
)


# ------------------------------------------------------------
# 5. Final state
# ------------------------------------------------------------

print("\n--- STATUS ---")

print(
    "Exact frozen DEV endpoints : 20"
)

print(
    "Threshold selected         : False"
)

print(
    "Reliability rule changed   : False"
)

print(
    "Official TEST used         : False"
)

print(
    "Files written              : False"
)

print(
    "\n=== END STEP 6.27H1-D2-R ==="
)

In [ ]:
# ============================================================
# STEP 6.27H1-D3
# G1 -> G3 RELIABILITY INTERFACE CLOSURE
# ============================================================

import numpy as np
import pandas as pd
import inspect


print(
    "\n"
    "============================================================\n"
    "STEP 6.27H1-D3 — G1 -> G3 RELIABILITY INTERFACE CLOSURE\n"
    "============================================================"
)


# ============================================================
# 1. Required objects
# ============================================================

required_627h1d3 = [
    "CLEAN_DEV_SELECTIVE_BASE_627H1",
    "FROZEN_DEV_ENDPOINT_KEYS_627H1D2",
    "reference_endpoint_scores_627g1",
    "clean_uncertainty_risk_ref_627g1",
    "clean_sensor_risk_ref_627g1",
    "RELIABILITY_UNCERTAINTY_REFERENCE_627G3",
    "RELIABILITY_SENSOR_REFERENCE_627G3",
    "combined_reliability_risk_627g3",
]


print("\n--- REQUIRED OBJECTS ---")

for name in required_627h1d3:

    exists = name in globals()

    print(
        f"{name:<48}: {exists}"
    )

    if not exists:
        raise RuntimeError(
            f"Missing required object: {name}"
        )


# ============================================================
# 2. Confirm G1 key ordering against canonical H1 ordering
# ============================================================

g1_key_frame_627h1d3 = (
    reference_endpoint_scores_627g1[
        ["unit_id", "cycle"]
    ]
    .copy()
    .reset_index(drop=True)
)


canonical_keys_627h1d3 = (
    CLEAN_DEV_SELECTIVE_BASE_627H1[
        ["unit_id", "cycle"]
    ]
    .copy()
    .reset_index(drop=True)
)


print("\n--- KEY ORDER CONTRACT ---")

print(
    "G1 key rows              :",
    len(g1_key_frame_627h1d3)
)

print(
    "Canonical key rows       :",
    len(canonical_keys_627h1d3)
)

print(
    "Exact key order match    :",
    g1_key_frame_627h1d3.equals(
        canonical_keys_627h1d3
    )
)


if not g1_key_frame_627h1d3.equals(
    canonical_keys_627h1d3
):
    raise RuntimeError(
        "G1 risk-array ordering does not exactly match "
        "the canonical H1 endpoint ordering."
    )


# ============================================================
# 3. Recover G1 normalized risk arrays
# ============================================================

u_risk_g1_627h1d3 = np.asarray(
    clean_uncertainty_risk_ref_627g1,
    dtype=float,
).reshape(-1)


s_risk_g1_627h1d3 = np.asarray(
    clean_sensor_risk_ref_627g1,
    dtype=float,
).reshape(-1)


if len(u_risk_g1_627h1d3) != 3490:
    raise RuntimeError(
        "G1 uncertainty-risk reference must contain 3490 values."
    )


if len(s_risk_g1_627h1d3) != 3490:
    raise RuntimeError(
        "G1 sensor-risk reference must contain 3490 values."
    )


if not np.isfinite(
    u_risk_g1_627h1d3
).all():
    raise RuntimeError(
        "G1 uncertainty-risk values contain non-finite values."
    )


if not np.isfinite(
    s_risk_g1_627h1d3
).all():
    raise RuntimeError(
        "G1 sensor-risk values contain non-finite values."
    )


# ============================================================
# 4. Locate exact frozen 20 endpoint positions
# ============================================================

position_frame_627h1d3 = (
    g1_key_frame_627h1d3
    .reset_index()
    .rename(
        columns={
            "index": "position"
        }
    )
)


endpoint_position_frame_627h1d3 = (
    FROZEN_DEV_ENDPOINT_KEYS_627H1D2
    .merge(
        position_frame_627h1d3,
        on=["unit_id", "cycle"],
        how="left",
        validate="one_to_one",
    )
)


if (
    endpoint_position_frame_627h1d3[
        "position"
    ]
    .isna()
    .any()
):
    raise RuntimeError(
        "Could not locate every frozen endpoint "
        "inside the G1 risk arrays."
    )


endpoint_positions_g1_627h1d3 = (
    endpoint_position_frame_627h1d3[
        "position"
    ]
    .to_numpy(dtype=int)
)


if len(endpoint_positions_g1_627h1d3) != 20:
    raise RuntimeError(
        "Expected exactly 20 G1 frozen endpoint positions."
    )


# ============================================================
# 5. Extract actual G1 endpoint risks
# ============================================================

u_endpoint_g1_627h1d3 = (
    u_risk_g1_627h1d3[
        endpoint_positions_g1_627h1d3
    ]
)


s_endpoint_g1_627h1d3 = (
    s_risk_g1_627h1d3[
        endpoint_positions_g1_627h1d3
    ]
)


u_reference_g3_627h1d3 = np.asarray(
    RELIABILITY_UNCERTAINTY_REFERENCE_627G3,
    dtype=float,
).reshape(-1)


s_reference_g3_627h1d3 = np.asarray(
    RELIABILITY_SENSOR_REFERENCE_627G3,
    dtype=float,
).reshape(-1)


# ============================================================
# 6. Exact frozen-reference comparison
# ============================================================

def sorted_max_error_627h1d3(
    left,
    right,
):

    left = np.sort(
        np.asarray(
            left,
            dtype=float
        ).reshape(-1)
    )

    right = np.sort(
        np.asarray(
            right,
            dtype=float
        ).reshape(-1)
    )

    if len(left) != len(right):
        return np.nan

    return float(
        np.max(
            np.abs(
                left - right
            )
        )
    )


u_interface_error_627h1d3 = (
    sorted_max_error_627h1d3(
        u_endpoint_g1_627h1d3,
        u_reference_g3_627h1d3,
    )
)


s_interface_error_627h1d3 = (
    sorted_max_error_627h1d3(
        s_endpoint_g1_627h1d3,
        s_reference_g3_627h1d3,
    )
)


print("\n--- G1 ENDPOINT -> G3 REFERENCE MATCH ---")

print(
    "Uncertainty-risk max error :",
    u_interface_error_627h1d3
)

print(
    "Sensor-risk max error      :",
    s_interface_error_627h1d3
)


# ============================================================
# 7. Compact statistics
# ============================================================

def stats_627h1d3(
    label,
    values,
):

    x = np.asarray(
        values,
        dtype=float
    ).reshape(-1)

    print(
        f"{label:<31}"
        f" min={x.min():.6f}"
        f" q25={np.quantile(x,0.25):.6f}"
        f" q50={np.quantile(x,0.50):.6f}"
        f" q75={np.quantile(x,0.75):.6f}"
        f" max={x.max():.6f}"
        f" mean={x.mean():.6f}"
    )


print("\n--- EXACT 20-ENDPOINT INPUTS ---")

stats_627h1d3(
    "G1 uncertainty risk",
    u_endpoint_g1_627h1d3
)

stats_627h1d3(
    "Frozen G3 U reference",
    u_reference_g3_627h1d3
)

stats_627h1d3(
    "G1 sensor risk",
    s_endpoint_g1_627h1d3
)

stats_627h1d3(
    "Frozen G3 S reference",
    s_reference_g3_627h1d3
)


# ============================================================
# 8. Confirm frozen function interface only
# ============================================================

print("\n--- G3 FUNCTION INTERFACE ---")

print(
    "Signature:",
    inspect.signature(
        combined_reliability_risk_627g3
    )
)


# ============================================================
# 9. Reproduce frozen endpoint combined risk correctly
# ============================================================

combined_endpoint_risk_627h1d3 = np.asarray(
    combined_reliability_risk_627g3(
        u_endpoint_g1_627h1d3,
        s_endpoint_g1_627h1d3,
    ),
    dtype=float,
).reshape(-1)


print("\n--- FROZEN ENDPOINT COMBINED RISK ---")

stats_627h1d3(
    "Combined endpoint risk",
    combined_endpoint_risk_627h1d3
)


# ============================================================
# 10. Build corrected full clean-window combined risk
#
# This is diagnostic only.
# Threshold is NOT selected.
# ============================================================

CORRECT_CLEAN_COMBINED_RISK_627H1D3 = np.asarray(
    combined_reliability_risk_627g3(
        u_risk_g1_627h1d3,
        s_risk_g1_627h1d3,
    ),
    dtype=float,
).reshape(-1)


print("\n--- CORRECT 3490-WINDOW COMBINED RISK ---")

stats_627h1d3(
    "Combined clean risk",
    CORRECT_CLEAN_COMBINED_RISK_627H1D3
)

print(
    "Fraction exactly 1.0       :",
    float(
        np.mean(
            CORRECT_CLEAN_COMBINED_RISK_627H1D3
            == 1.0
        )
    )
)


# ============================================================
# 11. Final closure condition
# ============================================================

interface_exact_627h1d3 = bool(
    np.isclose(
        u_interface_error_627h1d3,
        0.0,
        atol=1e-12,
        rtol=0.0,
    )
    and
    np.isclose(
        s_interface_error_627h1d3,
        0.0,
        atol=1e-12,
        rtol=0.0,
    )
)


print(
    "\n"
    "============================================================\n"
    "STEP 6.27H1-D3 SUMMARY\n"
    "============================================================"
)

print(
    "G1 -> G3 interface exact      :",
    interface_exact_627h1d3
)

print(
    "Sensor orientation            : COMPLEMENT OF QUALITY"
)

print(
    "G3 inputs                     : G1 NORMALIZED RISKS"
)

print(
    "Threshold selected            : False"
)

print(
    "Reliability rule changed      : False"
)

print(
    "Official TEST used            : False"
)

print(
    "Model inference rerun         : False"
)

print(
    "MC-dropout rerun              : False"
)

print(
    "Files written                 : False"
)

print(
    "\n=== END STEP 6.27H1-D3 ==="
)

In [ ]:
# ============================================================
# STEP 6.27H1-D4
# EXACT G3 UNCERTAINTY-REFERENCE PROVENANCE AUDIT
# ============================================================

import numpy as np
import pandas as pd
import inspect


print(
    "\n"
    "============================================================\n"
    "STEP 6.27H1-D4 — G3 UNCERTAINTY PROVENANCE AUDIT\n"
    "============================================================"
)


# ============================================================
# 1. Frozen G3 uncertainty reference
# ============================================================

g3_u_ref_627h1d4 = np.asarray(
    RELIABILITY_UNCERTAINTY_REFERENCE_627G3,
    dtype=float,
).reshape(-1)


print("\n--- FROZEN G3 UNCERTAINTY REFERENCE ---")

print("Length :", len(g3_u_ref_627h1d4))
print("Min    :", float(g3_u_ref_627h1d4.min()))
print("Median :", float(np.median(g3_u_ref_627h1d4)))
print("Mean   :", float(g3_u_ref_627h1d4.mean()))
print("Max    :", float(g3_u_ref_627h1d4.max()))


if len(g3_u_ref_627h1d4) != 20:
    raise RuntimeError(
        "Expected G3 uncertainty reference length = 20."
    )


# ============================================================
# 2. Helper
# ============================================================

def sorted_max_error_627h1d4(a, b):

    a = np.asarray(
        a,
        dtype=float
    ).reshape(-1)

    b = np.asarray(
        b,
        dtype=float
    ).reshape(-1)

    if len(a) != len(b):
        return np.nan

    return float(
        np.max(
            np.abs(
                np.sort(a) -
                np.sort(b)
            )
        )
    )


# ============================================================
# 3. Explicit known candidate objects
# ============================================================

explicit_names_627h1d4 = [
    "u_reference_endpoint_627g2e",
    "UNCERTAINTY_RISK_REFERENCE_627G1",
    "clean_uncertainty_risk_ref_627g1",
    "FROZEN_VAL_UNCERTAINTY_622IF",
    "u_endpoint_g1_627h1d3",
    "u_exact_627h1d2",
]


records_627h1d4 = []


for name in explicit_names_627h1d4:

    if name not in globals():
        continue

    obj = globals()[name]

    try:

        arr = np.asarray(
            obj,
            dtype=float
        ).reshape(-1)

    except Exception:
        continue

    record = {
        "object_name": name,
        "length": len(arr),
        "minimum": float(np.min(arr)),
        "median": float(np.median(arr)),
        "mean": float(np.mean(arr)),
        "maximum": float(np.max(arr)),
        "error_vs_G3_ref": (
            sorted_max_error_627h1d4(
                arr,
                g3_u_ref_627h1d4
            )
            if len(arr) == 20
            else np.nan
        ),
    }

    records_627h1d4.append(record)


KNOWN_U_CANDIDATES_627H1D4 = pd.DataFrame(
    records_627h1d4
)


print("\n--- KNOWN UNCERTAINTY CANDIDATES ---")

if len(KNOWN_U_CANDIDATES_627H1D4):

    print(
        KNOWN_U_CANDIDATES_627H1D4
        .to_string(
            index=False,
            float_format=lambda x: f"{x:.12f}"
        )
    )

else:

    print("No explicit candidates found.")


# ============================================================
# 4. Targeted in-memory search for 20-value uncertainty objects
# ============================================================

search_records_627h1d4 = []


for name, obj in list(globals().items()):

    lname = name.lower()

    # Only inspect names plausibly related to uncertainty
    if not any(
        token in lname
        for token in [
            "uncert",
            "mc_std",
            "u_reference",
            "u_endpoint",
            "627g2",
            "627g3",
        ]
    ):
        continue

    if isinstance(obj, pd.DataFrame):
        continue

    try:

        arr = np.asarray(
            obj,
            dtype=float
        ).reshape(-1)

    except Exception:
        continue

    if len(arr) != 20:
        continue

    if not np.isfinite(arr).all():
        continue

    error = sorted_max_error_627h1d4(
        arr,
        g3_u_ref_627h1d4
    )

    search_records_627h1d4.append(
        {
            "object_name": name,
            "error_vs_G3_ref": error,
            "minimum": float(arr.min()),
            "median": float(np.median(arr)),
            "mean": float(arr.mean()),
            "maximum": float(arr.max()),
        }
    )


U20_SEARCH_627H1D4 = (
    pd.DataFrame(
        search_records_627h1d4
    )
    .sort_values(
        [
            "error_vs_G3_ref",
            "object_name",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)


print("\n--- BEST 20-VALUE UNCERTAINTY MATCHES ---")

if len(U20_SEARCH_627H1D4):

    print(
        U20_SEARCH_627H1D4
        .head(15)
        .to_string(
            index=False,
            float_format=lambda x: f"{x:.12f}"
        )
    )

else:

    print("No 20-value uncertainty candidates found.")


# ============================================================
# 5. Exact-match objects
# ============================================================

if len(U20_SEARCH_627H1D4):

    exact_matches_627h1d4 = (
        U20_SEARCH_627H1D4.loc[
            np.isclose(
                U20_SEARCH_627H1D4[
                    "error_vs_G3_ref"
                ],
                0.0,
                atol=1e-12,
                rtol=0.0,
            ),
            "object_name",
        ]
        .tolist()
    )

else:

    exact_matches_627h1d4 = []


print("\n--- EXACT MATCHES ---")

print(
    "Objects exactly reproducing G3 U reference:",
    exact_matches_627h1d4
)


# ============================================================
# 6. Inspect G3 function source selectively
# ============================================================

print("\n--- COMBINED FUNCTION RELEVANT SOURCE LINES ---")


try:

    source_lines_627h1d4 = inspect.getsource(
        combined_reliability_risk_627g3
    ).splitlines()

    relevant_tokens_627h1d4 = [
        "uncert",
        "sensor",
        "reference",
        "percent",
        "searchsorted",
        "maximum",
        "max(",
        "clip",
        "return",
    ]

    for i, line in enumerate(
        source_lines_627h1d4,
        start=1,
    ):

        if any(
            token in line.lower()
            for token in relevant_tokens_627h1d4
        ):

            print(
                f"{i:03d}: {line}"
            )

except Exception as exc:

    print(
        "Source inspection unavailable:",
        exc
    )


# ============================================================
# 7. Inspect G3 representation metadata
# ============================================================

print("\n--- FROZEN G3 METADATA ---")

rep_627h1d4 = (
    FINAL_RELIABILITY_REPRESENTATION_627G3
)


if isinstance(rep_627h1d4, dict):

    keys_of_interest_627h1d4 = [
        "calibration_population",
        "calibration_endpoint_count",
        "calibration_method",
        "uncertainty_orientation",
        "sensor_orientation",
        "fusion_rule",
        "combined_orientation",
    ]

    for key in keys_of_interest_627h1d4:

        if key in rep_627h1d4:

            print(
                f"{key:<30}:",
                rep_627h1d4[key]
            )


# ============================================================
# 8. Final summary
# ============================================================

print(
    "\n"
    "============================================================\n"
    "STEP 6.27H1-D4 SUMMARY\n"
    "============================================================"
)

print(
    "G3 U-reference length       :",
    len(g3_u_ref_627h1d4)
)

print(
    "Exact provenance candidates :",
    exact_matches_627h1d4
)

print(
    "Sensor-side interface exact : True"
)

print(
    "Threshold selected          : False"
)

print(
    "Reliability rule changed    : False"
)

print(
    "Official TEST used          : False"
)

print(
    "Model / MC rerun            : False"
)

print(
    "Files written               : False"
)

print(
    "\n=== END STEP 6.27H1-D4 ==="
)

In [ ]:
# ============================================================
# STEP 6.27H1-D5
# FINAL G2E -> G3 CONTRACT INSPECTION
# ============================================================

import numpy as np
import pandas as pd
import inspect


print(
    "\n"
    "============================================================\n"
    "STEP 6.27H1-D5 — FINAL G2E -> G3 CONTRACT INSPECTION\n"
    "============================================================"
)


# ============================================================
# 1. Required frozen G2E/G3 objects
# ============================================================

required_627h1d5 = [
    "calibrated_627g2e",
    "u_reference_endpoint_627g2e",
    "s_reference_endpoint_627g2e",
    "candidate_columns_627g2e",
    "RELIABILITY_UNCERTAINTY_REFERENCE_627G3",
    "RELIABILITY_SENSOR_REFERENCE_627G3",
    "combined_reliability_risk_627g3",
]


print("\n--- REQUIRED OBJECTS ---")

for name in required_627h1d5:

    exists = name in globals()

    print(
        f"{name:<44}: {exists}"
    )

    if not exists:
        raise RuntimeError(
            f"Missing required object: {name}"
        )


# ============================================================
# 2. G2E calibrated table contract
# ============================================================

print("\n--- G2E CALIBRATED TABLE ---")

print(
    "Type  :",
    type(calibrated_627g2e).__name__
)

print(
    "Shape :",
    calibrated_627g2e.shape
)


print("\nColumns:")

for i, col in enumerate(
    calibrated_627g2e.columns,
    start=1
):
    print(
        f"{i:02d}. {col}"
    )


# ============================================================
# 3. Candidate columns actually evaluated in G2E
# ============================================================

print("\n--- G2E FUSION CANDIDATES ---")

print(
    list(candidate_columns_627g2e)
)


# ============================================================
# 4. Exact frozen reference reproduction
# ============================================================

u_ref_g2e_627h1d5 = np.asarray(
    u_reference_endpoint_627g2e,
    dtype=float,
).reshape(-1)


s_ref_g2e_627h1d5 = np.asarray(
    s_reference_endpoint_627g2e,
    dtype=float,
).reshape(-1)


u_ref_g3_627h1d5 = np.asarray(
    RELIABILITY_UNCERTAINTY_REFERENCE_627G3,
    dtype=float,
).reshape(-1)


s_ref_g3_627h1d5 = np.asarray(
    RELIABILITY_SENSOR_REFERENCE_627G3,
    dtype=float,
).reshape(-1)


print("\n--- G2E -> G3 FROZEN REFERENCES ---")

print(
    "U exact:",
    np.array_equal(
        u_ref_g2e_627h1d5,
        u_ref_g3_627h1d5
    )
)

print(
    "U max abs error:",
    float(
        np.max(
            np.abs(
                u_ref_g2e_627h1d5
                -
                u_ref_g3_627h1d5
            )
        )
    )
)

print(
    "S exact:",
    np.array_equal(
        s_ref_g2e_627h1d5,
        s_ref_g3_627h1d5
    )
)

print(
    "S max abs error:",
    float(
        np.max(
            np.abs(
                s_ref_g2e_627h1d5
                -
                s_ref_g3_627h1d5
            )
        )
    )
)


# ============================================================
# 5. Exact G3 combined-risk source
# ============================================================

print("\n--- G3 COMBINED-RISK FUNCTION ---")

print(
    "Signature:",
    inspect.signature(
        combined_reliability_risk_627g3
    )
)

try:

    source_627h1d5 = inspect.getsource(
        combined_reliability_risk_627g3
    )

    print(source_627h1d5)

except Exception as exc:

    print(
        "Source unavailable:",
        exc
    )


# ============================================================
# 6. Identify risk / percentile / fusion columns in G2E
# ============================================================

keywords_627h1d5 = [
    "risk",
    "percent",
    "uncert",
    "sensor",
    "max",
    "fusion",
    "25u",
    "50u",
    "75u",
]


relevant_columns_627h1d5 = [
    col
    for col in calibrated_627g2e.columns
    if any(
        token in str(col).lower()
        for token in keywords_627h1d5
    )
]


print("\n--- RELEVANT G2E COLUMNS ---")

for col in relevant_columns_627h1d5:

    print(col)


# ============================================================
# 7. Structural integrity only
# ============================================================

print("\n--- STRUCTURAL CONTRACT ---")

print(
    "Rows:",
    len(calibrated_627g2e)
)


for key in [
    "scenario_id",
    "family",
    "unit_id",
    "cycle",
]:

    if key in calibrated_627g2e.columns:

        print(
            f"{key} unique:",
            calibrated_627g2e[key].nunique()
        )


if all(
    col in calibrated_627g2e.columns
    for col in [
        "scenario_id",
        "unit_id",
        "cycle",
    ]
):

    duplicate_keys_627h1d5 = int(
        calibrated_627g2e.duplicated(
            subset=[
                "scenario_id",
                "unit_id",
                "cycle",
            ]
        ).sum()
    )

    print(
        "Duplicate scenario/unit/cycle keys:",
        duplicate_keys_627h1d5
    )


# ============================================================
# 8. Final status
# ============================================================

print(
    "\n"
    "============================================================\n"
    "STEP 6.27H1-D5 SUMMARY\n"
    "============================================================"
)

print(
    "G3 uncertainty provenance : u_reference_endpoint_627g2e"
)

print(
    "G3 calibration population : CLEAN DEV ENDPOINTS"
)

print(
    "Threshold selected        : False"
)

print(
    "Research rule changed     : False"
)

print(
    "Official TEST used        : False"
)

print(
    "Model / MC rerun          : False"
)

print(
    "Files written             : False"
)

print(
    "\n=== END STEP 6.27H1-D5 ==="
)

In [ ]:
# ============================================================
# STEP 6.27H1-D5-R
# COMPACT FINAL G2E -> G3 INTERFACE REVIEW
# ============================================================

import numpy as np
import inspect


print(
    "\n"
    "============================================================\n"
    "STEP 6.27H1-D5-R — COMPACT FINAL INTERFACE REVIEW\n"
    "============================================================"
)


# ------------------------------------------------------------
# 1. Frozen reference reproduction
# ------------------------------------------------------------

u_error_627h1d5r = float(
    np.max(
        np.abs(
            np.asarray(
                u_reference_endpoint_627g2e,
                dtype=float
            ).reshape(-1)
            -
            np.asarray(
                RELIABILITY_UNCERTAINTY_REFERENCE_627G3,
                dtype=float
            ).reshape(-1)
        )
    )
)


s_error_627h1d5r = float(
    np.max(
        np.abs(
            np.asarray(
                s_reference_endpoint_627g2e,
                dtype=float
            ).reshape(-1)
            -
            np.asarray(
                RELIABILITY_SENSOR_REFERENCE_627G3,
                dtype=float
            ).reshape(-1)
        )
    )
)


print("\n--- G2E -> G3 REFERENCE REPRODUCTION ---")

print(
    "Uncertainty reference max error :",
    u_error_627h1d5r
)

print(
    "Sensor reference max error      :",
    s_error_627h1d5r
)


# ------------------------------------------------------------
# 2. Candidate names frozen/evaluated in G2E
# ------------------------------------------------------------

print("\n--- G2E CANDIDATES ---")

print(
    list(candidate_columns_627g2e)
)


# ------------------------------------------------------------
# 3. Relevant G2E columns only
# ------------------------------------------------------------

relevant_tokens_627h1d5r = [
    "uncert",
    "sensor",
    "risk",
    "percent",
    "fusion",
    "max_or",
    "25u",
    "50u",
    "75u",
]


relevant_cols_627h1d5r = [
    col
    for col in calibrated_627g2e.columns
    if any(
        token in str(col).lower()
        for token in relevant_tokens_627h1d5r
    )
]


print("\n--- RELEVANT G2E COLUMNS ---")

for col in relevant_cols_627h1d5r:
    print(col)


# ------------------------------------------------------------
# 4. Exact G3 function contract
# ------------------------------------------------------------

print("\n--- G3 FUNCTION CONTRACT ---")

print(
    "Signature:",
    inspect.signature(
        combined_reliability_risk_627g3
    )
)


try:

    source_lines_627h1d5r = (
        inspect
        .getsource(
            combined_reliability_risk_627g3
        )
        .splitlines()
    )

    for line in source_lines_627h1d5r:
        print(line)

except Exception as exc:

    print(
        "Source inspection unavailable:",
        exc
    )


# ------------------------------------------------------------
# 5. G2E table integrity
# ------------------------------------------------------------

print("\n--- G2E TABLE CONTRACT ---")

print(
    "Rows       :",
    len(calibrated_627g2e)
)

print(
    "Scenarios  :",
    calibrated_627g2e[
        "scenario_id"
    ].nunique()
)

print(
    "DEV engines:",
    calibrated_627g2e[
        "unit_id"
    ].nunique()
)


duplicate_keys_627h1d5r = int(
    calibrated_627g2e.duplicated(
        subset=[
            "scenario_id",
            "unit_id",
            "cycle",
        ]
    ).sum()
)


print(
    "Duplicate scenario/unit/cycle keys:",
    duplicate_keys_627h1d5r
)


# ------------------------------------------------------------
# 6. Status
# ------------------------------------------------------------

print(
    "\n--- STATUS ---"
)

print(
    "Threshold selected      : False"
)

print(
    "Reliability rule changed: False"
)

print(
    "Official TEST used      : False"
)

print(
    "Model / MC rerun        : False"
)

print(
    "Files written           : False"
)

print(
    "\n=== END STEP 6.27H1-D5-R ==="
)

In [ ]:
# ============================================================
# STEP 6.27H1-D5-R2
# CORRECTED G2E -> G3 REFERENCE / FUNCTION CONTRACT CLOSURE
# ============================================================

import numpy as np
import inspect


print(
    "\n"
    "============================================================\n"
    "STEP 6.27H1-D5-R2 — CORRECTED FINAL CONTRACT CLOSURE\n"
    "============================================================"
)


# ------------------------------------------------------------
# 1. Recover reference arrays
# ------------------------------------------------------------

u_g2e_627h1d5r2 = np.asarray(
    u_reference_endpoint_627g2e,
    dtype=float
).reshape(-1)

u_g3_627h1d5r2 = np.asarray(
    RELIABILITY_UNCERTAINTY_REFERENCE_627G3,
    dtype=float
).reshape(-1)

s_g2e_627h1d5r2 = np.asarray(
    s_reference_endpoint_627g2e,
    dtype=float
).reshape(-1)

s_g3_627h1d5r2 = np.asarray(
    RELIABILITY_SENSOR_REFERENCE_627G3,
    dtype=float
).reshape(-1)


# ------------------------------------------------------------
# 2. Correct order-independent comparison
# ------------------------------------------------------------

u_sorted_error_627h1d5r2 = float(
    np.max(
        np.abs(
            np.sort(u_g2e_627h1d5r2)
            -
            np.sort(u_g3_627h1d5r2)
        )
    )
)

s_sorted_error_627h1d5r2 = float(
    np.max(
        np.abs(
            np.sort(s_g2e_627h1d5r2)
            -
            np.sort(s_g3_627h1d5r2)
        )
    )
)


print("\n--- CORRECT REFERENCE COMPARISON ---")

print(
    "U lengths                         :",
    len(u_g2e_627h1d5r2),
    len(u_g3_627h1d5r2)
)

print(
    "S lengths                         :",
    len(s_g2e_627h1d5r2),
    len(s_g3_627h1d5r2)
)

print(
    "U same order                      :",
    np.array_equal(
        u_g2e_627h1d5r2,
        u_g3_627h1d5r2
    )
)

print(
    "U sorted max absolute error       :",
    u_sorted_error_627h1d5r2
)

print(
    "S same order                      :",
    np.array_equal(
        s_g2e_627h1d5r2,
        s_g3_627h1d5r2
    )
)

print(
    "S sorted max absolute error       :",
    s_sorted_error_627h1d5r2
)


# ------------------------------------------------------------
# 3. Confirm candidate-related columns actually present
# ------------------------------------------------------------

print("\n--- FROZEN CANDIDATE COLUMNS PRESENT ---")

for candidate in candidate_columns_627g2e:

    exact = candidate in calibrated_627g2e.columns

    partial = [
        col
        for col in calibrated_627g2e.columns
        if candidate.lower() in str(col).lower()
    ]

    print(
        f"{candidate:<20} exact={str(exact):<5} "
        f"matches={partial}"
    )


# ------------------------------------------------------------
# 4. Core channel columns
# ------------------------------------------------------------

core_cols_627h1d5r2 = [
    "clean_uncertainty_risk",
    "corrupted_uncertainty_risk",
    "clean_sensor_anomaly_risk",
    "corrupted_sensor_anomaly_risk",
]


print("\n--- CORE G2E RISK CHANNELS ---")

for col in core_cols_627h1d5r2:

    print(
        f"{col:<36}:",
        col in calibrated_627g2e.columns
    )


# ------------------------------------------------------------
# 5. Print ONLY the exact G3 function body
# ------------------------------------------------------------

print("\n--- G3 COMBINED-RISK FUNCTION ---")

print(
    "Signature:",
    inspect.signature(
        combined_reliability_risk_627g3
    )
)

try:

    source_627h1d5r2 = inspect.getsource(
        combined_reliability_risk_627g3
    )

    print(source_627h1d5r2)

except Exception as exc:

    print(
        "Source unavailable:",
        exc
    )


# ------------------------------------------------------------
# 6. Structural G2E contract
# ------------------------------------------------------------

print("\n--- G2E STRUCTURE ---")

print(
    "Rows      :",
    len(calibrated_627g2e)
)

print(
    "Scenarios :",
    calibrated_627g2e[
        "scenario_id"
    ].nunique()
)

print(
    "Engines   :",
    calibrated_627g2e[
        "unit_id"
    ].nunique()
)

print(
    "Duplicate scenario/unit/cycle keys:",
    int(
        calibrated_627g2e.duplicated(
            subset=[
                "scenario_id",
                "unit_id",
                "cycle",
            ]
        ).sum()
    )
)


# ------------------------------------------------------------
# 7. Closure
# ------------------------------------------------------------

reference_multisets_exact_627h1d5r2 = bool(
    np.isclose(
        u_sorted_error_627h1d5r2,
        0.0,
        atol=1e-12,
        rtol=0.0
    )
    and
    np.isclose(
        s_sorted_error_627h1d5r2,
        0.0,
        atol=1e-12,
        rtol=0.0
    )
)


print(
    "\n"
    "============================================================\n"
    "STEP 6.27H1-D5-R2 SUMMARY\n"
    "============================================================"
)

print(
    "G2E/G3 reference populations exact :",
    reference_multisets_exact_627h1d5r2
)

print(
    "Threshold selected                  : False"
)

print(
    "Research rule changed               : False"
)

print(
    "Official TEST used                  : False"
)

print(
    "Model / MC rerun                    : False"
)

print(
    "Files written                       : False"
)

print(
    "\n=== END STEP 6.27H1-D5-R2 ==="
)

In [ ]:
# ============================================================
# STEP 6.27H2-A
# DEV SELECTIVE RUL RISK–COVERAGE ANALYSIS
# ============================================================

import numpy as np
import pandas as pd


print(
    "\n"
    "============================================================\n"
    "STEP 6.27H2-A — DEV SELECTIVE RUL RISK–COVERAGE ANALYSIS\n"
    "============================================================"
)


# ============================================================
# 1. Required frozen objects
# ============================================================

required_627h2a = [
    "calibrated_627g2e",
    "ENDPOINT_KEYS_627H0R",
    "FROZEN_DEV_TARGET_VALUES_627H0R",
    "FROZEN_DEV_TARGET_COLUMN_627H0R",
]


print("\n--- REQUIRED FROZEN OBJECTS ---")

for name in required_627h2a:

    exists = name in globals()

    print(
        f"{name:<42}: {exists}"
    )

    if not exists:
        raise RuntimeError(
            f"Missing required object: {name}"
        )


# ============================================================
# 2. Required G2E columns
# ============================================================

required_columns_627h2a = [
    "scenario_id",
    "family",
    "severity",
    "unit_id",
    "cycle",
    "clean_prediction",
    "corrupted_prediction",
    "_corrupted_ecdf_uncertainty_only",
    "_corrupted_ecdf_sensor_only",
    "_corrupted_ecdf_max_OR",
]


missing_columns_627h2a = [
    col
    for col in required_columns_627h2a
    if col not in calibrated_627g2e.columns
]


if missing_columns_627h2a:

    raise RuntimeError(
        "Missing required calibrated G2E columns: "
        + str(missing_columns_627h2a)
    )


print(
    "\nRequired G2E columns present : True"
)


# ============================================================
# 3. Recover frozen target values by endpoint key
# ============================================================

target_values_627h2a = np.asarray(
    FROZEN_DEV_TARGET_VALUES_627H0R,
    dtype=float,
).reshape(-1)


endpoint_keys_obj_627h2a = (
    ENDPOINT_KEYS_627H0R
)


if isinstance(
    endpoint_keys_obj_627h2a,
    pd.DataFrame
):

    if not all(
        col in endpoint_keys_obj_627h2a.columns
        for col in ["unit_id", "cycle"]
    ):
        raise RuntimeError(
            "ENDPOINT_KEYS_627H0R is a DataFrame but does "
            "not contain unit_id and cycle."
        )

    endpoint_keys_627h2a = (
        endpoint_keys_obj_627h2a[
            ["unit_id", "cycle"]
        ]
        .copy()
        .reset_index(drop=True)
    )

else:

    endpoint_key_array_627h2a = np.asarray(
        endpoint_keys_obj_627h2a
    )

    if (
        endpoint_key_array_627h2a.ndim != 2
        or
        endpoint_key_array_627h2a.shape[1] != 2
    ):
        raise RuntimeError(
            "ENDPOINT_KEYS_627H0R could not be interpreted "
            "as unit_id / cycle pairs."
        )

    endpoint_keys_627h2a = pd.DataFrame(
        endpoint_key_array_627h2a,
        columns=["unit_id", "cycle"]
    )


if len(
    endpoint_keys_627h2a
) != len(
    target_values_627h2a
):

    raise RuntimeError(
        "Endpoint-key count does not match frozen target count."
    )


target_lookup_627h2a = (
    endpoint_keys_627h2a
    .assign(
        target_rul=target_values_627h2a
    )
)


if target_lookup_627h2a.duplicated(
    subset=["unit_id", "cycle"]
).any():

    duplicate_target_values_627h2a = (
        target_lookup_627h2a
        .groupby(
            ["unit_id", "cycle"]
        )["target_rul"]
        .nunique()
    )

    if (
        duplicate_target_values_627h2a
        > 1
    ).any():

        raise RuntimeError(
            "A unit/cycle key maps to multiple frozen RUL targets."
        )

    target_lookup_627h2a = (
        target_lookup_627h2a
        .drop_duplicates(
            subset=["unit_id", "cycle"]
        )
        .reset_index(drop=True)
    )


# ============================================================
# 4. Build canonical degraded-DEV selective table
# ============================================================

DEV_SELECTIVE_ENDPOINT_BASE_627H2A = (
    calibrated_627g2e[
        required_columns_627h2a
    ]
    .copy()
    .merge(
        target_lookup_627h2a,
        on=["unit_id", "cycle"],
        how="left",
        validate="many_to_one",
    )
)


if (
    DEV_SELECTIVE_ENDPOINT_BASE_627H2A[
        "target_rul"
    ]
    .isna()
    .any()
):

    raise RuntimeError(
        "At least one degraded DEV endpoint could not be "
        "matched to the frozen RUL target."
    )


# ============================================================
# 5. Error columns
# ============================================================

DEV_SELECTIVE_ENDPOINT_BASE_627H2A[
    "clean_signed_error"
] = (
    DEV_SELECTIVE_ENDPOINT_BASE_627H2A[
        "clean_prediction"
    ]
    -
    DEV_SELECTIVE_ENDPOINT_BASE_627H2A[
        "target_rul"
    ]
)


DEV_SELECTIVE_ENDPOINT_BASE_627H2A[
    "corrupted_signed_error"
] = (
    DEV_SELECTIVE_ENDPOINT_BASE_627H2A[
        "corrupted_prediction"
    ]
    -
    DEV_SELECTIVE_ENDPOINT_BASE_627H2A[
        "target_rul"
    ]
)


DEV_SELECTIVE_ENDPOINT_BASE_627H2A[
    "clean_abs_error"
] = np.abs(
    DEV_SELECTIVE_ENDPOINT_BASE_627H2A[
        "clean_signed_error"
    ]
)


DEV_SELECTIVE_ENDPOINT_BASE_627H2A[
    "corrupted_abs_error"
] = np.abs(
    DEV_SELECTIVE_ENDPOINT_BASE_627H2A[
        "corrupted_signed_error"
    ]
)


# ============================================================
# 6. Rename frozen reliability policies locally
# ============================================================

policy_columns_627h2a = {
    "uncertainty_only":
        "_corrupted_ecdf_uncertainty_only",

    "sensor_only":
        "_corrupted_ecdf_sensor_only",

    "max_OR":
        "_corrupted_ecdf_max_OR",
}


for policy, column in policy_columns_627h2a.items():

    values = pd.to_numeric(
        DEV_SELECTIVE_ENDPOINT_BASE_627H2A[
            column
        ],
        errors="coerce",
    ).to_numpy(
        dtype=float
    )

    if not np.isfinite(
        values
    ).all():

        raise RuntimeError(
            f"{policy} contains non-finite risk values."
        )

    if (
        (values < 0.0).any()
        or
        (values > 1.0).any()
    ):

        raise RuntimeError(
            f"{policy} risk is outside [0,1]."
        )


# ============================================================
# 7. Structural integrity
# ============================================================

rows_627h2a = len(
    DEV_SELECTIVE_ENDPOINT_BASE_627H2A
)

scenarios_627h2a = (
    DEV_SELECTIVE_ENDPOINT_BASE_627H2A[
        "scenario_id"
    ]
    .nunique()
)

engines_627h2a = (
    DEV_SELECTIVE_ENDPOINT_BASE_627H2A[
        "unit_id"
    ]
    .nunique()
)

duplicate_keys_627h2a = int(
    DEV_SELECTIVE_ENDPOINT_BASE_627H2A
    .duplicated(
        subset=[
            "scenario_id",
            "unit_id",
            "cycle",
        ]
    )
    .sum()
)


scenario_sizes_627h2a = (
    DEV_SELECTIVE_ENDPOINT_BASE_627H2A
    .groupby(
        "scenario_id"
    )
    .size()
)


print("\n--- SELECTIVE BASE CONTRACT ---")

print(
    "Rows                    :",
    rows_627h2a
)

print(
    "Scenarios               :",
    scenarios_627h2a
)

print(
    "DEV engines             :",
    engines_627h2a
)

print(
    "Duplicate keys          :",
    duplicate_keys_627h2a
)

print(
    "Scenario-size min / max :",
    int(
        scenario_sizes_627h2a.min()
    ),
    "/",
    int(
        scenario_sizes_627h2a.max()
    )
)

print(
    "Target convention       :",
    FROZEN_DEV_TARGET_COLUMN_627H0R
)


if rows_627h2a != 6300:
    raise RuntimeError(
        "Expected exactly 6300 degraded DEV endpoint rows."
    )


if scenarios_627h2a != 315:
    raise RuntimeError(
        "Expected exactly 315 corruption scenarios."
    )


if not (
    scenario_sizes_627h2a == 20
).all():
    raise RuntimeError(
        "Every corruption scenario must contain exactly "
        "20 DEV engines."
    )


if duplicate_keys_627h2a != 0:
    raise RuntimeError(
        "Duplicate scenario/unit/cycle keys detected."
    )


# ============================================================
# 8. Clean endpoint baseline
# ============================================================

unique_clean_endpoints_627h2a = (
    DEV_SELECTIVE_ENDPOINT_BASE_627H2A[
        [
            "unit_id",
            "cycle",
            "target_rul",
            "clean_prediction",
            "clean_abs_error",
            "clean_signed_error",
        ]
    ]
    .drop_duplicates(
        subset=["unit_id", "cycle"]
    )
    .reset_index(drop=True)
)


clean_endpoint_mae_627h2a = float(
    unique_clean_endpoints_627h2a[
        "clean_abs_error"
    ].mean()
)


clean_endpoint_rmse_627h2a = float(
    np.sqrt(
        np.mean(
            np.square(
                unique_clean_endpoints_627h2a[
                    "clean_signed_error"
                ]
                .to_numpy(
                    dtype=float
                )
            )
        )
    )
)


print("\n--- CLEAN DEV ENDPOINT BASELINE ---")

print(
    "Endpoint engines :",
    len(
        unique_clean_endpoints_627h2a
    )
)

print(
    "MAE              :",
    clean_endpoint_mae_627h2a
)

print(
    "RMSE             :",
    clean_endpoint_rmse_627h2a
)


# ============================================================
# 9. Risk–coverage grid
#
# Coverage is enforced independently inside every scenario.
# Lower risk = more reliable = accepted first.
# ============================================================

coverage_grid_627h2a = [
    1.00,
    0.95,
    0.90,
    0.80,
    0.70,
    0.60,
    0.50,
]


detail_records_627h2a = []


for scenario_id, group in (
    DEV_SELECTIVE_ENDPOINT_BASE_627H2A
    .groupby(
        "scenario_id",
        sort=False
    )
):

    scenario_family_627h2a = (
        group[
            "family"
        ]
        .iloc[0]
    )

    scenario_severity_627h2a = (
        group[
            "severity"
        ]
        .iloc[0]
    )

    full_mae_627h2a = float(
        group[
            "corrupted_abs_error"
        ]
        .mean()
    )

    full_rmse_627h2a = float(
        np.sqrt(
            np.mean(
                np.square(
                    group[
                        "corrupted_signed_error"
                    ]
                    .to_numpy(
                        dtype=float
                    )
                )
            )
        )
    )

    n_rows_627h2a = len(
        group
    )

    for policy_name_627h2a, risk_col_627h2a in (
        policy_columns_627h2a.items()
    ):

        ranked_627h2a = (
            group
            .sort_values(
                by=[
                    risk_col_627h2a,
                    "unit_id",
                    "cycle",
                ],
                ascending=[
                    True,
                    True,
                    True,
                ],
                kind="stable",
            )
            .reset_index(
                drop=True
            )
        )

        for requested_coverage_627h2a in (
            coverage_grid_627h2a
        ):

            accepted_n_627h2a = max(
                1,
                int(
                    round(
                        requested_coverage_627h2a
                        *
                        n_rows_627h2a
                    )
                )
            )

            accepted_627h2a = (
                ranked_627h2a
                .iloc[
                    :accepted_n_627h2a
                ]
            )

            accepted_mae_627h2a = float(
                accepted_627h2a[
                    "corrupted_abs_error"
                ]
                .mean()
            )

            accepted_rmse_627h2a = float(
                np.sqrt(
                    np.mean(
                        np.square(
                            accepted_627h2a[
                                "corrupted_signed_error"
                            ]
                            .to_numpy(
                                dtype=float
                            )
                        )
                    )
                )
            )

            detail_records_627h2a.append(
                {
                    "scenario_id":
                        scenario_id,

                    "family":
                        scenario_family_627h2a,

                    "severity":
                        scenario_severity_627h2a,

                    "policy":
                        policy_name_627h2a,

                    "requested_coverage":
                        requested_coverage_627h2a,

                    "accepted_engines":
                        accepted_n_627h2a,

                    "actual_coverage":
                        accepted_n_627h2a
                        /
                        n_rows_627h2a,

                    "full_mae":
                        full_mae_627h2a,

                    "accepted_mae":
                        accepted_mae_627h2a,

                    "mae_change":
                        accepted_mae_627h2a
                        -
                        full_mae_627h2a,

                    "mae_reduction":
                        full_mae_627h2a
                        -
                        accepted_mae_627h2a,

                    "full_rmse":
                        full_rmse_627h2a,

                    "accepted_rmse":
                        accepted_rmse_627h2a,

                    "rmse_reduction":
                        full_rmse_627h2a
                        -
                        accepted_rmse_627h2a,
                }
            )


DEV_RISK_COVERAGE_DETAIL_627H2A = pd.DataFrame(
    detail_records_627h2a
)


# ============================================================
# 10. Aggregate across all 315 scenarios
# ============================================================

DEV_RISK_COVERAGE_SUMMARY_627H2A = (
    DEV_RISK_COVERAGE_DETAIL_627H2A
    .groupby(
        [
            "policy",
            "requested_coverage",
        ],
        as_index=False,
    )
    .agg(
        scenarios=(
            "scenario_id",
            "nunique",
        ),

        accepted_engines=(
            "accepted_engines",
            "first",
        ),

        actual_coverage=(
            "actual_coverage",
            "mean",
        ),

        mean_full_mae=(
            "full_mae",
            "mean",
        ),

        mean_accepted_mae=(
            "accepted_mae",
            "mean",
        ),

        median_accepted_mae=(
            "accepted_mae",
            "median",
        ),

        mean_mae_reduction=(
            "mae_reduction",
            "mean",
        ),

        median_mae_reduction=(
            "mae_reduction",
            "median",
        ),

        fraction_scenarios_improved=(
            "mae_reduction",
            lambda x: float(
                np.mean(
                    np.asarray(
                        x,
                        dtype=float
                    )
                    >
                    0.0
                )
            ),
        ),

        mean_accepted_rmse=(
            "accepted_rmse",
            "mean",
        ),

        mean_rmse_reduction=(
            "rmse_reduction",
            "mean",
        ),
    )
)


# ============================================================
# 11. Compact output
# ============================================================

print(
    "\n--- OVERALL DEV RISK–COVERAGE SUMMARY ---"
)

display_columns_627h2a = [
    "policy",
    "requested_coverage",
    "accepted_engines",
    "mean_accepted_mae",
    "mean_mae_reduction",
    "fraction_scenarios_improved",
    "mean_accepted_rmse",
]


print(
    DEV_RISK_COVERAGE_SUMMARY_627H2A[
        display_columns_627h2a
    ]
    .round(6)
    .to_string(
        index=False
    )
)


# ============================================================
# 12. Basic consistency checks
# ============================================================

expected_detail_rows_627h2a = (
    315
    *
    len(
        policy_columns_627h2a
    )
    *
    len(
        coverage_grid_627h2a
    )
)


if len(
    DEV_RISK_COVERAGE_DETAIL_627H2A
) != expected_detail_rows_627h2a:

    raise RuntimeError(
        "Risk–coverage detail table has unexpected row count."
    )


if not np.isfinite(
    DEV_RISK_COVERAGE_DETAIL_627H2A[
        [
            "full_mae",
            "accepted_mae",
            "full_rmse",
            "accepted_rmse",
        ]
    ]
    .to_numpy(
        dtype=float
    )
).all():

    raise RuntimeError(
        "Risk–coverage metrics contain non-finite values."
    )


# ============================================================
# 13. Final status
# ============================================================

print(
    "\n"
    "============================================================\n"
    "STEP 6.27H2-A SUMMARY\n"
    "============================================================"
)

print(
    "Frozen degraded DEV endpoints : 6300"
)

print(
    "Frozen scenarios evaluated    : 315"
)

print(
    "Policies evaluated            :",
    list(
        policy_columns_627h2a.keys()
    )
)

print(
    "Coverage levels               :",
    coverage_grid_627h2a
)

print(
    "Coverage enforced per scenario: True"
)

print(
    "Threshold selected            : False"
)

print(
    "Combined rule changed         : False"
)

print(
    "Official TEST used            : False"
)

print(
    "Model / MC rerun              : False"
)

print(
    "Files written                 : False"
)

print(
    "\n=== END STEP 6.27H2-A ==="
)


In [ ]:
# ============================================================
# STEP 6.27H2-A-R
# COMPACT DEV RISK–COVERAGE REVIEW
# ============================================================

import numpy as np
import pandas as pd


print(
    "\n"
    "============================================================\n"
    "STEP 6.27H2-A-R — COMPACT DEV RISK–COVERAGE REVIEW\n"
    "============================================================"
)


# ============================================================
# 1. Required result objects
# ============================================================

required_627h2ar = [
    "DEV_RISK_COVERAGE_DETAIL_627H2A",
    "DEV_RISK_COVERAGE_SUMMARY_627H2A",
]


print("\n--- RESULT OBJECTS ---")

for name in required_627h2ar:

    exists = name in globals()

    print(
        f"{name:<42}: {exists}"
    )

    if not exists:
        raise RuntimeError(
            f"Missing H2-A result object: {name}"
        )


summary_627h2ar = (
    DEV_RISK_COVERAGE_SUMMARY_627H2A
    .copy()
)


# ============================================================
# 2. Fixed display order
# ============================================================

policy_order_627h2ar = [
    "uncertainty_only",
    "sensor_only",
    "max_OR",
]


coverage_order_627h2ar = [
    1.00,
    0.95,
    0.90,
    0.80,
    0.70,
    0.60,
    0.50,
]


# ============================================================
# 3. Accepted MAE
# ============================================================

mae_pivot_627h2ar = (
    summary_627h2ar
    .pivot(
        index="requested_coverage",
        columns="policy",
        values="mean_accepted_mae",
    )
    .reindex(
        coverage_order_627h2ar
    )
    .reindex(
        columns=policy_order_627h2ar
    )
)


print("\n--- MEAN ACCEPTED MAE ---")

print(
    mae_pivot_627h2ar
    .round(6)
    .to_string()
)


# ============================================================
# 4. MAE reduction relative to full coverage
# Positive = selective rejection improved performance
# ============================================================

reduction_pivot_627h2ar = (
    summary_627h2ar
    .pivot(
        index="requested_coverage",
        columns="policy",
        values="mean_mae_reduction",
    )
    .reindex(
        coverage_order_627h2ar
    )
    .reindex(
        columns=policy_order_627h2ar
    )
)


print("\n--- MEAN MAE REDUCTION ---")
print("Positive values are better.")

print(
    reduction_pivot_627h2ar
    .round(6)
    .to_string()
)


# ============================================================
# 5. Fraction of scenarios improved
# ============================================================

improved_pivot_627h2ar = (
    summary_627h2ar
    .pivot(
        index="requested_coverage",
        columns="policy",
        values="fraction_scenarios_improved",
    )
    .reindex(
        coverage_order_627h2ar
    )
    .reindex(
        columns=policy_order_627h2ar
    )
)


print("\n--- FRACTION OF 315 SCENARIOS IMPROVED ---")

print(
    improved_pivot_627h2ar
    .round(6)
    .to_string()
)


# ============================================================
# 6. 100% coverage consistency
# ============================================================

full_rows_627h2ar = (
    summary_627h2ar.loc[
        np.isclose(
            summary_627h2ar[
                "requested_coverage"
            ],
            1.0,
        )
    ]
    .copy()
)


full_mae_values_627h2ar = (
    full_rows_627h2ar[
        "mean_accepted_mae"
    ]
    .to_numpy(
        dtype=float
    )
)


full_rmse_values_627h2ar = (
    full_rows_627h2ar[
        "mean_accepted_rmse"
    ]
    .to_numpy(
        dtype=float
    )
)


print("\n--- 100% COVERAGE CONSISTENCY ---")

print(
    "Policies represented :",
    full_rows_627h2ar[
        "policy"
    ].tolist()
)

print(
    "MAE equal across policies :",
    bool(
        np.allclose(
            full_mae_values_627h2ar,
            full_mae_values_627h2ar[0],
            atol=1e-12,
            rtol=0.0,
        )
    )
)

print(
    "RMSE equal across policies:",
    bool(
        np.allclose(
            full_rmse_values_627h2ar,
            full_rmse_values_627h2ar[0],
            atol=1e-12,
            rtol=0.0,
        )
    )
)

print(
    "Mean corrupted full-coverage MAE :",
    float(
        full_mae_values_627h2ar[0]
    )
)

print(
    "Mean corrupted full-coverage RMSE:",
    float(
        full_rmse_values_627h2ar[0]
    )
)


# ============================================================
# 7. Best policy at each selective coverage
# ============================================================

selective_rows_627h2ar = (
    summary_627h2ar.loc[
        summary_627h2ar[
            "requested_coverage"
        ] < 1.0
    ]
    .copy()
)


best_records_627h2ar = []


for coverage_627h2ar, group_627h2ar in (
    selective_rows_627h2ar
    .groupby(
        "requested_coverage",
        sort=False,
    )
):

    best_mae_row_627h2ar = (
        group_627h2ar
        .sort_values(
            [
                "mean_accepted_mae",
                "policy",
            ],
            ascending=[
                True,
                True,
            ],
            kind="stable",
        )
        .iloc[0]
    )

    best_records_627h2ar.append(
        {
            "coverage":
                coverage_627h2ar,

            "best_policy":
                best_mae_row_627h2ar[
                    "policy"
                ],

            "accepted_engines":
                int(
                    best_mae_row_627h2ar[
                        "accepted_engines"
                    ]
                ),

            "mean_accepted_mae":
                float(
                    best_mae_row_627h2ar[
                        "mean_accepted_mae"
                    ]
                ),

            "mean_mae_reduction":
                float(
                    best_mae_row_627h2ar[
                        "mean_mae_reduction"
                    ]
                ),

            "scenario_improvement_fraction":
                float(
                    best_mae_row_627h2ar[
                        "fraction_scenarios_improved"
                    ]
                ),
        }
    )


BEST_POLICY_BY_COVERAGE_627H2AR = pd.DataFrame(
    best_records_627h2ar
)


print("\n--- BEST POLICY BY COVERAGE ---")

print(
    BEST_POLICY_BY_COVERAGE_627H2AR
    .sort_values(
        "coverage",
        ascending=False
    )
    .round(6)
    .to_string(
        index=False
    )
)


# ============================================================
# 8. Final state
# ============================================================

print(
    "\n"
    "============================================================\n"
    "STEP 6.27H2-A-R SUMMARY\n"
    "============================================================"
)

print(
    "Risk–coverage results available : True"
)

print(
    "Threshold selected              : False"
)

print(
    "Combined rule changed           : False"
)

print(
    "Official TEST used              : False"
)

print(
    "Files written                   : False"
)

print(
    "\n=== END STEP 6.27H2-A-R ==="
)

In [ ]:
# ============================================================
# STEP 6.27H2-B
# ALL SIX FROZEN FUSION CANDIDATES:
# SELECTIVE-RUL RISK–COVERAGE COMPARISON
# ============================================================

import numpy as np
import pandas as pd


print(
    "\n"
    "============================================================\n"
    "STEP 6.27H2-B — SIX-CANDIDATE SELECTIVE-RUL COMPARISON\n"
    "============================================================"
)


# ============================================================
# 1. Required frozen objects
# ============================================================

required_627h2b = [
    "DEV_SELECTIVE_ENDPOINT_BASE_627H2A",
    "calibrated_627g2e",
]


print("\n--- REQUIRED OBJECTS ---")

for name in required_627h2b:

    exists = name in globals()

    print(
        f"{name:<42}: {exists}"
    )

    if not exists:
        raise RuntimeError(
            f"Missing required object: {name}"
        )


# ============================================================
# 2. Six PRE-EXISTING frozen G2E candidates
# ============================================================

candidate_columns_627h2b = {

    "uncertainty_only":
        "_corrupted_ecdf_uncertainty_only",

    "sensor_only":
        "_corrupted_ecdf_sensor_only",

    "mean_50U_50S":
        "_corrupted_ecdf_mean_50U_50S",

    "weighted_75U_25S":
        "_corrupted_ecdf_weighted_75U_25S",

    "weighted_25U_75S":
        "_corrupted_ecdf_weighted_25U_75S",

    "max_OR":
        "_corrupted_ecdf_max_OR",
}


print("\n--- FROZEN CANDIDATES ---")

for policy, column in candidate_columns_627h2b.items():

    exists = column in calibrated_627g2e.columns

    print(
        f"{policy:<20}: {exists}"
    )

    if not exists:

        raise RuntimeError(
            f"Missing frozen G2E candidate column: {column}"
        )


# ============================================================
# 3. Construct working endpoint table
# ============================================================

work_627h2b = (
    DEV_SELECTIVE_ENDPOINT_BASE_627H2A
    .copy()
)


key_cols_627h2b = [
    "scenario_id",
    "unit_id",
    "cycle",
]


# Add only candidate columns not already present.
missing_candidate_cols_627h2b = [
    col
    for col in candidate_columns_627h2b.values()
    if col not in work_627h2b.columns
]


if missing_candidate_cols_627h2b:

    candidate_lookup_627h2b = (
        calibrated_627g2e[
            key_cols_627h2b
            +
            missing_candidate_cols_627h2b
        ]
        .copy()
    )

    if candidate_lookup_627h2b.duplicated(
        subset=key_cols_627h2b
    ).any():

        raise RuntimeError(
            "Candidate lookup contains duplicate endpoint keys."
        )

    work_627h2b = (
        work_627h2b
        .merge(
            candidate_lookup_627h2b,
            on=key_cols_627h2b,
            how="left",
            validate="one_to_one",
        )
    )


# ============================================================
# 4. Validate risk values
# ============================================================

for policy, column in candidate_columns_627h2b.items():

    values = pd.to_numeric(
        work_627h2b[column],
        errors="coerce",
    ).to_numpy(
        dtype=float
    )

    if not np.isfinite(values).all():

        raise RuntimeError(
            f"{policy} contains non-finite values."
        )

    if (
        (values < 0.0).any()
        or
        (values > 1.0).any()
    ):

        raise RuntimeError(
            f"{policy} contains values outside [0,1]."
        )


# ============================================================
# 5. Structural contract
# ============================================================

print("\n--- ENDPOINT CONTRACT ---")

print(
    "Rows      :",
    len(work_627h2b)
)

print(
    "Scenarios :",
    work_627h2b[
        "scenario_id"
    ].nunique()
)

print(
    "Engines   :",
    work_627h2b[
        "unit_id"
    ].nunique()
)


scenario_sizes_627h2b = (
    work_627h2b
    .groupby("scenario_id")
    .size()
)


print(
    "Scenario size min/max:",
    int(scenario_sizes_627h2b.min()),
    "/",
    int(scenario_sizes_627h2b.max())
)


if len(work_627h2b) != 6300:
    raise RuntimeError(
        "Expected exactly 6300 endpoint records."
    )


if not (
    scenario_sizes_627h2b == 20
).all():
    raise RuntimeError(
        "Each scenario must contain exactly 20 engines."
    )


# ============================================================
# 6. Exact coverage grid
#
# 20 engines:
# 1.00 = 20 accepted
# 0.95 = 19
# ...
# 0.50 = 10
# ============================================================

coverage_grid_627h2b = np.arange(
    0.50,
    1.0001,
    0.05
)


# ============================================================
# 7. Scenario-level selective evaluation
# ============================================================

records_627h2b = []


for scenario_id, group in (
    work_627h2b
    .groupby(
        "scenario_id",
        sort=False,
    )
):

    family_627h2b = (
        group["family"].iloc[0]
    )

    severity_627h2b = (
        group["severity"].iloc[0]
    )

    full_mae_627h2b = float(
        group[
            "corrupted_abs_error"
        ].mean()
    )

    for policy_627h2b, risk_col_627h2b in (
        candidate_columns_627h2b.items()
    ):

        ranked_627h2b = (
            group
            .sort_values(
                [
                    risk_col_627h2b,
                    "unit_id",
                    "cycle",
                ],
                ascending=[
                    True,
                    True,
                    True,
                ],
                kind="stable",
            )
            .reset_index(drop=True)
        )

        for coverage_627h2b in coverage_grid_627h2b:

            accepted_n_627h2b = int(
                round(
                    coverage_627h2b
                    *
                    len(ranked_627h2b)
                )
            )

            accepted_627h2b = (
                ranked_627h2b
                .iloc[
                    :accepted_n_627h2b
                ]
            )

            accepted_mae_627h2b = float(
                accepted_627h2b[
                    "corrupted_abs_error"
                ].mean()
            )

            records_627h2b.append(
                {
                    "scenario_id":
                        scenario_id,

                    "family":
                        family_627h2b,

                    "severity":
                        severity_627h2b,

                    "policy":
                        policy_627h2b,

                    "coverage":
                        float(
                            coverage_627h2b
                        ),

                    "accepted_engines":
                        accepted_n_627h2b,

                    "full_mae":
                        full_mae_627h2b,

                    "accepted_mae":
                        accepted_mae_627h2b,

                    "mae_reduction":
                        (
                            full_mae_627h2b
                            -
                            accepted_mae_627h2b
                        ),
                }
            )


DEV_SIX_POLICY_CURVES_627H2B = pd.DataFrame(
    records_627h2b
)


expected_rows_627h2b = (
    315
    *
    6
    *
    len(coverage_grid_627h2b)
)


if len(
    DEV_SIX_POLICY_CURVES_627H2B
) != expected_rows_627h2b:

    raise RuntimeError(
        "Unexpected six-policy risk–coverage row count."
    )


# ============================================================
# 8. Scenario-level selective AURC
#
# Primary criterion:
# mean MAE integrated over coverage [0.50, 1.00].
#
# LOWER selective_AURC_MAE is better.
#
# selective_gain_area is the integrated improvement
# over full coverage.
#
# HIGHER selective_gain_area is better.
# ============================================================

scenario_records_627h2b = []


for (
    scenario_id_627h2b,
    policy_627h2b
), group in (
    DEV_SIX_POLICY_CURVES_627H2B
    .groupby(
        [
            "scenario_id",
            "policy",
        ],
        sort=False,
    )
):

    ordered_627h2b = (
        group
        .sort_values(
            "coverage"
        )
    )

    x_627h2b = (
        ordered_627h2b[
            "coverage"
        ]
        .to_numpy(
            dtype=float
        )
    )

    y_mae_627h2b = (
        ordered_627h2b[
            "accepted_mae"
        ]
        .to_numpy(
            dtype=float
        )
    )

    y_gain_627h2b = (
        ordered_627h2b[
            "mae_reduction"
        ]
        .to_numpy(
            dtype=float
        )
    )

    coverage_span_627h2b = (
        x_627h2b.max()
        -
        x_627h2b.min()
    )

    aurc_mae_627h2b = float(
        np.trapezoid(
            y_mae_627h2b,
            x_627h2b
        )
        /
        coverage_span_627h2b
    )

    gain_area_627h2b = float(
        np.trapezoid(
            y_gain_627h2b,
            x_627h2b
        )
        /
        coverage_span_627h2b
    )

    scenario_records_627h2b.append(
        {
            "scenario_id":
                scenario_id_627h2b,

            "family":
                ordered_627h2b[
                    "family"
                ].iloc[0],

            "severity":
                ordered_627h2b[
                    "severity"
                ].iloc[0],

            "policy":
                policy_627h2b,

            "full_mae":
                float(
                    ordered_627h2b[
                        "full_mae"
                    ].iloc[0]
                ),

            "selective_AURC_MAE":
                aurc_mae_627h2b,

            "selective_gain_area":
                gain_area_627h2b,
        }
    )


DEV_SIX_POLICY_SCENARIO_AURC_627H2B = pd.DataFrame(
    scenario_records_627h2b
)


# ============================================================
# 9. Extract operating-point evidence at 80% and 50%
# ============================================================

def operating_summary_627h2b(
    target_coverage
):

    selected = (
        DEV_SIX_POLICY_CURVES_627H2B.loc[
            np.isclose(
                DEV_SIX_POLICY_CURVES_627H2B[
                    "coverage"
                ],
                target_coverage,
            )
        ]
        .copy()
    )

    return (
        selected
        .groupby(
            "policy",
            as_index=False,
        )
        .agg(
            mean_accepted_mae=(
                "accepted_mae",
                "mean",
            ),

            mean_mae_reduction=(
                "mae_reduction",
                "mean",
            ),

            fraction_scenarios_improved=(
                "mae_reduction",
                lambda x: float(
                    np.mean(
                        np.asarray(
                            x,
                            dtype=float
                        )
                        >
                        0.0
                    )
                ),
            ),
        )
    )


OP80_627H2B = operating_summary_627h2b(
    0.80
)

OP50_627H2B = operating_summary_627h2b(
    0.50
)


# ============================================================
# 10. Family robustness
# ============================================================

family_gain_627h2b = (
    DEV_SIX_POLICY_SCENARIO_AURC_627H2B
    .groupby(
        [
            "policy",
            "family",
        ],
        as_index=False,
    )
    .agg(
        mean_family_gain_area=(
            "selective_gain_area",
            "mean",
        )
    )
)


family_robustness_627h2b = (
    family_gain_627h2b
    .groupby(
        "policy",
        as_index=False,
    )
    .agg(
        minimum_family_gain_area=(
            "mean_family_gain_area",
            "min",
        )
    )
)


# ============================================================
# 11. Severity robustness
# ============================================================

severity_gain_627h2b = (
    DEV_SIX_POLICY_SCENARIO_AURC_627H2B
    .groupby(
        [
            "policy",
            "severity",
        ],
        as_index=False,
    )
    .agg(
        mean_severity_gain_area=(
            "selective_gain_area",
            "mean",
        )
    )
)


severity_robustness_627h2b = (
    severity_gain_627h2b
    .groupby(
        "policy",
        as_index=False,
    )
    .agg(
        minimum_severity_gain_area=(
            "mean_severity_gain_area",
            "min",
        )
    )
)


# ============================================================
# 12. Overall candidate summary
# ============================================================

overall_627h2b = (
    DEV_SIX_POLICY_SCENARIO_AURC_627H2B
    .groupby(
        "policy",
        as_index=False,
    )
    .agg(
        mean_selective_AURC_MAE=(
            "selective_AURC_MAE",
            "mean",
        ),

        median_selective_AURC_MAE=(
            "selective_AURC_MAE",
            "median",
        ),

        mean_selective_gain_area=(
            "selective_gain_area",
            "mean",
        ),

        median_selective_gain_area=(
            "selective_gain_area",
            "median",
        ),

        positive_gain_fraction=(
            "selective_gain_area",
            lambda x: float(
                np.mean(
                    np.asarray(
                        x,
                        dtype=float
                    )
                    >
                    0.0
                )
            ),
        ),
    )
)


DEV_SIX_POLICY_SUMMARY_627H2B = (
    overall_627h2b
    .merge(
        OP80_627H2B,
        on="policy",
        how="left",
        validate="one_to_one",
    )
    .rename(
        columns={
            "mean_accepted_mae":
                "MAE_at_80",

            "mean_mae_reduction":
                "gain_at_80",

            "fraction_scenarios_improved":
                "improved_fraction_at_80",
        }
    )
    .merge(
        OP50_627H2B,
        on="policy",
        how="left",
        validate="one_to_one",
    )
    .rename(
        columns={
            "mean_accepted_mae":
                "MAE_at_50",

            "mean_mae_reduction":
                "gain_at_50",

            "fraction_scenarios_improved":
                "improved_fraction_at_50",
        }
    )
    .merge(
        family_robustness_627h2b,
        on="policy",
        how="left",
        validate="one_to_one",
    )
    .merge(
        severity_robustness_627h2b,
        on="policy",
        how="left",
        validate="one_to_one",
    )
)


DEV_SIX_POLICY_SUMMARY_627H2B = (
    DEV_SIX_POLICY_SUMMARY_627H2B
    .sort_values(
        [
            "mean_selective_AURC_MAE",
            "policy",
        ],
        ascending=[
            True,
            True,
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


DEV_SIX_POLICY_SUMMARY_627H2B.insert(
    0,
    "rank",
    np.arange(
        1,
        len(
            DEV_SIX_POLICY_SUMMARY_627H2B
        )
        + 1
    )
)


# ============================================================
# 13. Compact final output
# ============================================================

display_cols_627h2b = [
    "rank",
    "policy",
    "mean_selective_AURC_MAE",
    "mean_selective_gain_area",
    "positive_gain_fraction",
    "MAE_at_80",
    "gain_at_80",
    "MAE_at_50",
    "gain_at_50",
    "minimum_family_gain_area",
    "minimum_severity_gain_area",
]


print(
    "\n--- SIX-CANDIDATE SELECTIVE-RUL RANKING ---"
)

print(
    DEV_SIX_POLICY_SUMMARY_627H2B[
        display_cols_627h2b
    ]
    .round(6)
    .to_string(
        index=False
    )
)


# ============================================================
# 14. Methodological state
# ============================================================

print(
    "\n"
    "============================================================\n"
    "STEP 6.27H2-B SUMMARY\n"
    "============================================================"
)

print(
    "Frozen candidates evaluated : 6"
)

print(
    "Frozen scenarios            : 315"
)

print(
    "Coverage grid               :",
    [
        round(
            float(x),
            2
        )
        for x in coverage_grid_627h2b
    ]
)

print(
    "Primary ranking criterion   : mean_selective_AURC_MAE"
)

print(
    "Lower primary criterion     : BETTER"
)

print(
    "Threshold selected          : False"
)

print(
    "Candidate frozen for gate   : False"
)

print(
    "G3 representation changed   : False"
)

print(
    "Official TEST used          : False"
)

print(
    "Files written               : False"
)

print(
    "\n=== END STEP 6.27H2-B ==="
)

In [ ]:
# ============================================================
# STEP 6.27H2-B-R
# COMPACT SIX-CANDIDATE SELECTIVE-RUL REVIEW
# ============================================================

import numpy as np
import pandas as pd


print(
    "\n"
    "============================================================\n"
    "STEP 6.27H2-B-R — COMPACT SIX-CANDIDATE REVIEW\n"
    "============================================================"
)


# ------------------------------------------------------------
# 1. Required result object
# ------------------------------------------------------------

if "DEV_SIX_POLICY_SUMMARY_627H2B" not in globals():
    raise RuntimeError(
        "DEV_SIX_POLICY_SUMMARY_627H2B is missing. "
        "Do not continue."
    )


review_627h2br = (
    DEV_SIX_POLICY_SUMMARY_627H2B
    .copy()
    .sort_values(
        "rank"
    )
    .reset_index(drop=True)
)


print(
    "Candidates recovered :",
    len(review_627h2br)
)


if len(review_627h2br) != 6:
    raise RuntimeError(
        "Expected exactly six frozen candidates."
    )


# ------------------------------------------------------------
# 2. Primary ranking
# ------------------------------------------------------------

print("\n--- PRIMARY SELECTIVE-RUL RANKING ---")
print("Lower AURC-MAE = better\n")


for _, row in review_627h2br.iterrows():

    print(
        f"Rank {int(row['rank'])}: {row['policy']}"
    )

    print(
        "  Mean selective AURC-MAE :",
        f"{row['mean_selective_AURC_MAE']:.6f}"
    )

    print(
        "  Mean selective gain     :",
        f"{row['mean_selective_gain_area']:.6f}"
    )

    print(
        "  Positive-gain fraction  :",
        f"{row['positive_gain_fraction']:.6f}"
    )

    print(
        "  MAE @ 80% coverage      :",
        f"{row['MAE_at_80']:.6f}"
    )

    print(
        "  Gain @ 80%              :",
        f"{row['gain_at_80']:.6f}"
    )

    print(
        "  MAE @ 50% coverage      :",
        f"{row['MAE_at_50']:.6f}"
    )

    print(
        "  Gain @ 50%              :",
        f"{row['gain_at_50']:.6f}"
    )

    print(
        "  Worst-family gain       :",
        f"{row['minimum_family_gain_area']:.6f}"
    )

    print(
        "  Worst-severity gain     :",
        f"{row['minimum_severity_gain_area']:.6f}"
    )

    print()


# ------------------------------------------------------------
# 3. Very compact comparison table
# ------------------------------------------------------------

compact_cols_627h2br = [
    "rank",
    "policy",
    "mean_selective_AURC_MAE",
    "MAE_at_80",
    "MAE_at_50",
]


print("\n--- COMPACT COMPARISON ---")

print(
    review_627h2br[
        compact_cols_627h2br
    ]
    .round(6)
    .to_string(
        index=False
    )
)


# ------------------------------------------------------------
# 4. Identify current DEV winner
# ------------------------------------------------------------

winner_627h2br = (
    review_627h2br
    .iloc[0]
)


print("\n--- DEV PRIMARY-CRITERION WINNER ---")

print(
    "Policy :",
    winner_627h2br[
        "policy"
    ]
)

print(
    "AURC-MAE:",
    float(
        winner_627h2br[
            "mean_selective_AURC_MAE"
        ]
    )
)


# ------------------------------------------------------------
# 5. Methodological state
# ------------------------------------------------------------

print("\n--- STATUS ---")

print(
    "Primary criterion evaluated : True"
)

print(
    "Threshold selected          : False"
)

print(
    "Gate finally frozen         : False"
)

print(
    "Official TEST used          : False"
)

print(
    "Research rule changed       : False"
)

print(
    "Files written               : False"
)

print(
    "\n=== END STEP 6.27H2-B-R ==="
)

In [ ]:
# ============================================================
# STEP 6.27H2-C
# PRIMARY GATE ROBUSTNESS CONFIRMATION
# uncertainty_only vs weighted_75U_25S
# ============================================================

import numpy as np
import pandas as pd


print(
    "\n"
    "============================================================\n"
    "STEP 6.27H2-C — PRIMARY GATE ROBUSTNESS CONFIRMATION\n"
    "============================================================"
)


# ============================================================
# 1. Required frozen result objects
# ============================================================

required_627h2c = [
    "DEV_SIX_POLICY_SCENARIO_AURC_627H2B",
    "DEV_SIX_POLICY_SUMMARY_627H2B",
    "calibrated_627g2e",
]


print("\n--- REQUIRED OBJECTS ---")

for name in required_627h2c:

    exists = name in globals()

    print(
        f"{name:<44}: {exists}"
    )

    if not exists:
        raise RuntimeError(
            f"Missing required object: {name}"
        )


PRIMARY_627H2C = "uncertainty_only"
RUNNER_UP_627H2C = "weighted_75U_25S"


# ============================================================
# 2. Confirm ranking order from frozen H2-B result
# ============================================================

ranking_627h2c = (
    DEV_SIX_POLICY_SUMMARY_627H2B
    .sort_values(
        "rank"
    )
    .reset_index(drop=True)
)


if ranking_627h2c.loc[0, "policy"] != PRIMARY_627H2C:
    raise RuntimeError(
        "Frozen H2-B primary winner is not uncertainty_only."
    )


if ranking_627h2c.loc[1, "policy"] != RUNNER_UP_627H2C:
    raise RuntimeError(
        "Frozen H2-B runner-up is not weighted_75U_25S."
    )


print("\n--- FROZEN H2-B RANKING CONTRACT ---")

print(
    "Primary   :",
    PRIMARY_627H2C
)

print(
    "Runner-up :",
    RUNNER_UP_627H2C
)


# ============================================================
# 3. Recover scenario metadata, including sensor
# ============================================================

scenario_meta_627h2c = (
    calibrated_627g2e[
        [
            "scenario_id",
            "family",
            "severity",
            "sensor",
        ]
    ]
    .drop_duplicates()
)


scenario_meta_counts_627h2c = (
    scenario_meta_627h2c
    .groupby("scenario_id")
    .size()
)


if not (
    scenario_meta_counts_627h2c == 1
).all():

    raise RuntimeError(
        "Each scenario must map to exactly one "
        "family/severity/sensor definition."
    )


scenario_meta_627h2c = (
    scenario_meta_627h2c
    .drop_duplicates(
        subset=["scenario_id"]
    )
)


print("\n--- SCENARIO METADATA ---")

print(
    "Scenarios :",
    scenario_meta_627h2c[
        "scenario_id"
    ].nunique()
)

print(
    "Families  :",
    scenario_meta_627h2c[
        "family"
    ].nunique()
)

print(
    "Severities:",
    scenario_meta_627h2c[
        "severity"
    ].nunique()
)

print(
    "Sensors   :",
    scenario_meta_627h2c[
        "sensor"
    ].nunique()
)


# ============================================================
# 4. Build paired scenario comparison
#
# Delta definition:
# runner-up AURC - uncertainty-only AURC
#
# Positive delta = uncertainty-only is better.
# ============================================================

paired_source_627h2c = (
    DEV_SIX_POLICY_SCENARIO_AURC_627H2B.loc[
        DEV_SIX_POLICY_SCENARIO_AURC_627H2B[
            "policy"
        ].isin(
            [
                PRIMARY_627H2C,
                RUNNER_UP_627H2C,
            ]
        )
    ]
    .copy()
)


paired_627h2c = (
    paired_source_627h2c
    .pivot(
        index="scenario_id",
        columns="policy",
        values="selective_AURC_MAE",
    )
    .reset_index()
    .merge(
        scenario_meta_627h2c,
        on="scenario_id",
        how="left",
        validate="one_to_one",
    )
)


if len(paired_627h2c) != 315:
    raise RuntimeError(
        "Expected exactly 315 paired scenarios."
    )


paired_627h2c[
    "delta_runner_minus_primary"
] = (
    paired_627h2c[
        RUNNER_UP_627H2C
    ]
    -
    paired_627h2c[
        PRIMARY_627H2C
    ]
)


delta_627h2c = (
    paired_627h2c[
        "delta_runner_minus_primary"
    ]
    .to_numpy(
        dtype=float
    )
)


# ============================================================
# 5. Overall paired evidence
# ============================================================

tol_627h2c = 1e-12


primary_wins_627h2c = int(
    np.sum(
        delta_627h2c
        >
        tol_627h2c
    )
)


ties_627h2c = int(
    np.sum(
        np.abs(
            delta_627h2c
        )
        <=
        tol_627h2c
    )
)


primary_losses_627h2c = int(
    np.sum(
        delta_627h2c
        <
        -tol_627h2c
    )
)


print("\n--- OVERALL PAIRED SCENARIO EVIDENCE ---")

print(
    "Mean AURC delta (runner - primary)   :",
    float(
        np.mean(
            delta_627h2c
        )
    )
)

print(
    "Median AURC delta                    :",
    float(
        np.median(
            delta_627h2c
        )
    )
)

print(
    "Primary wins / ties / losses         :",
    primary_wins_627h2c,
    "/",
    ties_627h2c,
    "/",
    primary_losses_627h2c
)

print(
    "Primary win fraction                 :",
    float(
        primary_wins_627h2c
        /
        len(
            delta_627h2c
        )
    )
)


# ============================================================
# 6. Family robustness
# ============================================================

FAMILY_ROBUSTNESS_627H2C = (
    paired_627h2c
    .groupby(
        "family",
        as_index=False,
    )
    .agg(
        scenarios=(
            "scenario_id",
            "count",
        ),

        mean_delta=(
            "delta_runner_minus_primary",
            "mean",
        ),

        median_delta=(
            "delta_runner_minus_primary",
            "median",
        ),

        primary_win_fraction=(
            "delta_runner_minus_primary",
            lambda x: float(
                np.mean(
                    np.asarray(
                        x,
                        dtype=float
                    )
                    >
                    tol_627h2c
                )
            ),
        ),
    )
)


print("\n--- FAMILY ROBUSTNESS ---")

print(
    FAMILY_ROBUSTNESS_627H2C
    .round(6)
    .to_string(
        index=False
    )
)


# ============================================================
# 7. Severity robustness
# ============================================================

SEVERITY_ROBUSTNESS_627H2C = (
    paired_627h2c
    .groupby(
        "severity",
        as_index=False,
    )
    .agg(
        scenarios=(
            "scenario_id",
            "count",
        ),

        mean_delta=(
            "delta_runner_minus_primary",
            "mean",
        ),

        median_delta=(
            "delta_runner_minus_primary",
            "median",
        ),

        primary_win_fraction=(
            "delta_runner_minus_primary",
            lambda x: float(
                np.mean(
                    np.asarray(
                        x,
                        dtype=float
                    )
                    >
                    tol_627h2c
                )
            ),
        ),
    )
)


print("\n--- SEVERITY ROBUSTNESS ---")

print(
    SEVERITY_ROBUSTNESS_627H2C
    .round(6)
    .to_string(
        index=False
    )
)


# ============================================================
# 8. Sensor-level robustness
# ============================================================

SENSOR_ROBUSTNESS_627H2C = (
    paired_627h2c
    .groupby(
        "sensor",
        as_index=False,
    )
    .agg(
        scenarios=(
            "scenario_id",
            "count",
        ),

        mean_delta=(
            "delta_runner_minus_primary",
            "mean",
        ),

        median_delta=(
            "delta_runner_minus_primary",
            "median",
        ),

        primary_win_fraction=(
            "delta_runner_minus_primary",
            lambda x: float(
                np.mean(
                    np.asarray(
                        x,
                        dtype=float
                    )
                    >
                    tol_627h2c
                )
            ),
        ),
    )
    .sort_values(
        "mean_delta"
    )
    .reset_index(drop=True)
)


print("\n--- SENSOR ROBUSTNESS ---")
print(
    "Positive mean_delta = uncertainty-only better.\n"
)

print(
    SENSOR_ROBUSTNESS_627H2C[
        [
            "sensor",
            "scenarios",
            "mean_delta",
            "primary_win_fraction",
        ]
    ]
    .round(6)
    .to_string(
        index=False
    )
)


# ============================================================
# 9. Leave-one-family-out stability
# ============================================================

loo_records_627h2c = []


families_627h2c = sorted(
    paired_627h2c[
        "family"
    ].unique()
)


for excluded_family_627h2c in families_627h2c:

    remaining_627h2c = (
        paired_627h2c.loc[
            paired_627h2c[
                "family"
            ]
            !=
            excluded_family_627h2c
        ]
    )

    loo_records_627h2c.append(
        {
            "excluded_family":
                excluded_family_627h2c,

            "remaining_scenarios":
                len(
                    remaining_627h2c
                ),

            "mean_delta":
                float(
                    remaining_627h2c[
                        "delta_runner_minus_primary"
                    ].mean()
                ),

            "primary_win_fraction":
                float(
                    np.mean(
                        remaining_627h2c[
                            "delta_runner_minus_primary"
                        ].to_numpy(
                            dtype=float
                        )
                        >
                        tol_627h2c
                    )
                ),
        }
    )


LOO_FAMILY_627H2C = pd.DataFrame(
    loo_records_627h2c
)


print("\n--- LEAVE-ONE-FAMILY-OUT ---")

print(
    LOO_FAMILY_627H2C
    .round(6)
    .to_string(
        index=False
    )
)


# ============================================================
# 10. Final boolean robustness summary
# ============================================================

all_family_mean_positive_627h2c = bool(
    (
        FAMILY_ROBUSTNESS_627H2C[
            "mean_delta"
        ]
        >
        0.0
    ).all()
)


all_severity_mean_positive_627h2c = bool(
    (
        SEVERITY_ROBUSTNESS_627H2C[
            "mean_delta"
        ]
        >
        0.0
    ).all()
)


all_loo_positive_627h2c = bool(
    (
        LOO_FAMILY_627H2C[
            "mean_delta"
        ]
        >
        0.0
    ).all()
)


negative_sensor_groups_627h2c = int(
    (
        SENSOR_ROBUSTNESS_627H2C[
            "mean_delta"
        ]
        <
        0.0
    )
    .sum()
)


print(
    "\n"
    "============================================================\n"
    "STEP 6.27H2-C ROBUSTNESS SUMMARY\n"
    "============================================================"
)

print(
    "Primary candidate              :",
    PRIMARY_627H2C
)

print(
    "Runner-up candidate            :",
    RUNNER_UP_627H2C
)

print(
    "All family mean deltas > 0     :",
    all_family_mean_positive_627h2c
)

print(
    "All severity mean deltas > 0   :",
    all_severity_mean_positive_627h2c
)

print(
    "All LOO-family mean deltas > 0 :",
    all_loo_positive_627h2c
)

print(
    "Sensor groups with mean delta < 0:",
    negative_sensor_groups_627h2c
)

print(
    "Threshold selected             : False"
)

print(
    "Gate frozen                    : False"
)

print(
    "Official TEST used             : False"
)

print(
    "Research rule changed          : False"
)

print(
    "Files written                  : False"
)

print(
    "\n=== END STEP 6.27H2-C ==="
)

In [ ]:
# ============================================================
# STEP 6.27H2-C-R
# COMPACT FINAL PRIMARY-GATE ROBUSTNESS REVIEW
# ============================================================

import numpy as np
import pandas as pd


print(
    "\n"
    "============================================================\n"
    "STEP 6.27H2-C-R — FINAL ROBUSTNESS REVIEW\n"
    "============================================================"
)


# ------------------------------------------------------------
# 1. Required H2-C result objects
# ------------------------------------------------------------

required_627h2cr = [
    "FAMILY_ROBUSTNESS_627H2C",
    "SEVERITY_ROBUSTNESS_627H2C",
    "SENSOR_ROBUSTNESS_627H2C",
    "LOO_FAMILY_627H2C",
]


print("\n--- RESULT OBJECTS ---")

for name in required_627h2cr:

    exists = name in globals()

    print(f"{name:<34}: {exists}")

    if not exists:
        raise RuntimeError(
            f"Missing H2-C object: {name}"
        )


# ------------------------------------------------------------
# 2. Family robustness
# ------------------------------------------------------------

print("\n--- FAMILY ROBUSTNESS ---")
print("Positive mean_delta = uncertainty-only better.")

print(
    FAMILY_ROBUSTNESS_627H2C[
        [
            "family",
            "scenarios",
            "mean_delta",
            "median_delta",
            "primary_win_fraction",
        ]
    ]
    .round(6)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 3. Severity robustness
# ------------------------------------------------------------

print("\n--- SEVERITY ROBUSTNESS ---")

print(
    SEVERITY_ROBUSTNESS_627H2C[
        [
            "severity",
            "scenarios",
            "mean_delta",
            "median_delta",
            "primary_win_fraction",
        ]
    ]
    .round(6)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 4. Sensor robustness
# ------------------------------------------------------------

print("\n--- SENSOR ROBUSTNESS ---")
print("Sorted from weakest to strongest primary advantage.")

print(
    SENSOR_ROBUSTNESS_627H2C[
        [
            "sensor",
            "scenarios",
            "mean_delta",
            "primary_win_fraction",
        ]
    ]
    .sort_values("mean_delta")
    .round(6)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 5. Leave-one-family-out stability
# ------------------------------------------------------------

print("\n--- LEAVE-ONE-FAMILY-OUT ---")

print(
    LOO_FAMILY_627H2C[
        [
            "excluded_family",
            "remaining_scenarios",
            "mean_delta",
            "primary_win_fraction",
        ]
    ]
    .round(6)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 6. Final boolean closure
# ------------------------------------------------------------

all_family_positive_627h2cr = bool(
    (
        FAMILY_ROBUSTNESS_627H2C[
            "mean_delta"
        ] > 0.0
    ).all()
)


all_severity_positive_627h2cr = bool(
    (
        SEVERITY_ROBUSTNESS_627H2C[
            "mean_delta"
        ] > 0.0
    ).all()
)


all_loo_positive_627h2cr = bool(
    (
        LOO_FAMILY_627H2C[
            "mean_delta"
        ] > 0.0
    ).all()
)


negative_sensor_groups_627h2cr = int(
    (
        SENSOR_ROBUSTNESS_627H2C[
            "mean_delta"
        ] < 0.0
    ).sum()
)


zero_sensor_groups_627h2cr = int(
    np.isclose(
        SENSOR_ROBUSTNESS_627H2C[
            "mean_delta"
        ].to_numpy(dtype=float),
        0.0,
        atol=1e-12,
        rtol=0.0,
    ).sum()
)


print(
    "\n"
    "============================================================\n"
    "STEP 6.27H2-C-R CLOSURE\n"
    "============================================================"
)

print(
    "Primary candidate              : uncertainty_only"
)

print(
    "Runner-up candidate            : weighted_75U_25S"
)

print(
    "All family mean deltas > 0     :",
    all_family_positive_627h2cr
)

print(
    "All severity mean deltas > 0   :",
    all_severity_positive_627h2cr
)

print(
    "All LOO-family mean deltas > 0 :",
    all_loo_positive_627h2cr
)

print(
    "Sensor groups with mean delta < 0:",
    negative_sensor_groups_627h2cr
)

print(
    "Sensor groups with mean delta = 0:",
    zero_sensor_groups_627h2cr
)

print(
    "Threshold selected             : False"
)

print(
    "Gate frozen                    : False"
)

print(
    "Official TEST used             : False"
)

print(
    "Files written                  : False"
)

print(
    "\n=== END STEP 6.27H2-C-R ==="
)

In [ ]:
# ============================================================
# STEP 6.27H2-D
# FREEZE PRIMARY SELECTIVE-RUL GATE
# ============================================================

import numpy as np
import pandas as pd


print(
    "\n"
    "============================================================\n"
    "STEP 6.27H2-D — FREEZE PRIMARY SELECTIVE-RUL GATE\n"
    "============================================================"
)


# ============================================================
# 1. Required evidence
# ============================================================

required_627h2d = [
    "DEV_SIX_POLICY_SUMMARY_627H2B",
    "FAMILY_ROBUSTNESS_627H2C",
    "SEVERITY_ROBUSTNESS_627H2C",
    "LOO_FAMILY_627H2C",
    "calibrated_627g2e",
]


print("\n--- REQUIRED EVIDENCE ---")

for name in required_627h2d:

    exists = name in globals()

    print(
        f"{name:<38}: {exists}"
    )

    if not exists:
        raise RuntimeError(
            f"Missing required evidence object: {name}"
        )


# ============================================================
# 2. Frozen decision
# ============================================================

PRIMARY_SELECTIVE_GATE_627H2D = (
    "uncertainty_only"
)

PRIMARY_SELECTIVE_RISK_COLUMN_627H2D = (
    "_corrupted_ecdf_uncertainty_only"
)

PRIMARY_CLEAN_RISK_COLUMN_627H2D = (
    "_clean_ecdf_uncertainty_only"
)

PRIMARY_GATE_ORIENTATION_627H2D = (
    "higher_is_worse"
)

PRIMARY_GATE_SELECTION_CRITERION_627H2D = (
    "minimum_mean_selective_AURC_MAE"
)

PRIMARY_GATE_COVERAGE_SELECTION_RANGE_627H2D = (
    0.50,
    1.00,
)


# ============================================================
# 3. Confirm H2-B winner
# ============================================================

ranking_627h2d = (
    DEV_SIX_POLICY_SUMMARY_627H2B
    .sort_values("rank")
    .reset_index(drop=True)
)


winner_627h2d = (
    ranking_627h2d.iloc[0]
)


if (
    winner_627h2d["policy"]
    !=
    PRIMARY_SELECTIVE_GATE_627H2D
):
    raise RuntimeError(
        "Frozen gate does not match the H2-B "
        "primary-criterion winner."
    )


# ============================================================
# 4. Confirm robustness contract
# ============================================================

all_family_positive_627h2d = bool(
    (
        FAMILY_ROBUSTNESS_627H2C[
            "mean_delta"
        ]
        >
        0.0
    ).all()
)


all_severity_positive_627h2d = bool(
    (
        SEVERITY_ROBUSTNESS_627H2C[
            "mean_delta"
        ]
        >
        0.0
    ).all()
)


all_loo_positive_627h2d = bool(
    (
        LOO_FAMILY_627H2C[
            "mean_delta"
        ]
        >
        0.0
    ).all()
)


if not all_family_positive_627h2d:
    raise RuntimeError(
        "Family robustness contract failed."
    )


if not all_severity_positive_627h2d:
    raise RuntimeError(
        "Severity robustness contract failed."
    )


if not all_loo_positive_627h2d:
    raise RuntimeError(
        "Leave-one-family-out robustness contract failed."
    )


# ============================================================
# 5. Confirm frozen risk column
# ============================================================

if (
    PRIMARY_SELECTIVE_RISK_COLUMN_627H2D
    not in
    calibrated_627g2e.columns
):
    raise RuntimeError(
        "Frozen corrupted uncertainty-risk column missing."
    )


if (
    PRIMARY_CLEAN_RISK_COLUMN_627H2D
    not in
    calibrated_627g2e.columns
):
    raise RuntimeError(
        "Frozen clean uncertainty-risk column missing."
    )


risk_values_627h2d = pd.to_numeric(
    calibrated_627g2e[
        PRIMARY_SELECTIVE_RISK_COLUMN_627H2D
    ],
    errors="coerce",
).to_numpy(
    dtype=float
)


if not np.isfinite(
    risk_values_627h2d
).all():
    raise RuntimeError(
        "Frozen primary risk contains non-finite values."
    )


if (
    (risk_values_627h2d < 0.0).any()
    or
    (risk_values_627h2d > 1.0).any()
):
    raise RuntimeError(
        "Frozen primary risk lies outside [0,1]."
    )


# ============================================================
# 6. Freeze evidence summary
# ============================================================

PRIMARY_GATE_EVIDENCE_627H2D = {
    "gate":
        PRIMARY_SELECTIVE_GATE_627H2D,

    "risk_column":
        PRIMARY_SELECTIVE_RISK_COLUMN_627H2D,

    "clean_risk_column":
        PRIMARY_CLEAN_RISK_COLUMN_627H2D,

    "orientation":
        PRIMARY_GATE_ORIENTATION_627H2D,

    "selection_criterion":
        PRIMARY_GATE_SELECTION_CRITERION_627H2D,

    "selection_coverage_range":
        PRIMARY_GATE_COVERAGE_SELECTION_RANGE_627H2D,

    "mean_selective_AURC_MAE":
        float(
            winner_627h2d[
                "mean_selective_AURC_MAE"
            ]
        ),

    "mean_selective_gain_area":
        float(
            winner_627h2d[
                "mean_selective_gain_area"
            ]
        ),

    "positive_gain_fraction":
        float(
            winner_627h2d[
                "positive_gain_fraction"
            ]
        ),

    "all_family_mean_deltas_positive":
        all_family_positive_627h2d,

    "all_severity_mean_deltas_positive":
        all_severity_positive_627h2d,

    "all_loo_family_mean_deltas_positive":
        all_loo_positive_627h2d,

    "runner_up":
        str(
            ranking_627h2d.iloc[1][
                "policy"
            ]
        ),
}


# ============================================================
# 7. Final confirmation
# ============================================================

print("\n--- FROZEN PRIMARY GATE ---")

print(
    "Gate                  :",
    PRIMARY_SELECTIVE_GATE_627H2D
)

print(
    "Corrupted risk column :",
    PRIMARY_SELECTIVE_RISK_COLUMN_627H2D
)

print(
    "Clean risk column     :",
    PRIMARY_CLEAN_RISK_COLUMN_627H2D
)

print(
    "Orientation           :",
    PRIMARY_GATE_ORIENTATION_627H2D
)

print(
    "Primary criterion     :",
    PRIMARY_GATE_SELECTION_CRITERION_627H2D
)

print(
    "Mean selective AURC   :",
    PRIMARY_GATE_EVIDENCE_627H2D[
        "mean_selective_AURC_MAE"
    ]
)

print(
    "Positive-gain fraction:",
    PRIMARY_GATE_EVIDENCE_627H2D[
        "positive_gain_fraction"
    ]
)

print(
    "All families positive :",
    all_family_positive_627h2d
)

print(
    "All severities positive:",
    all_severity_positive_627h2d
)

print(
    "All LOO families positive:",
    all_loo_positive_627h2d
)

print(
    "Runner-up             :",
    PRIMARY_GATE_EVIDENCE_627H2D[
        "runner_up"
    ]
)


print(
    "\n"
    "============================================================\n"
    "STEP 6.27H2-D FREEZE SUMMARY\n"
    "============================================================"
)

print(
    "Primary selective gate frozen : True"
)

print(
    "Reliability threshold selected: False"
)

print(
    "Coverage target selected      : False"
)

print(
    "Sensor-only retained ablation : True"
)

print(
    "Fusion ablations retained     : True"
)

print(
    "Official TEST used            : False"
)

print(
    "Research decision modified after freeze: False"
)

print(
    "Files written                 : False"
)

print(
    "\n=== END STEP 6.27H2-D ==="
)

In [ ]:
# ============================================================
# STEP 6.27H3-A
# DEV-ONLY PRIMARY RELIABILITY THRESHOLD SELECTION
#
# Frozen gate:
#     uncertainty_only
#
# Selection principle:
#     choose HIGHEST threshold satisfying all robustness and
#     coverage constraints.
#
# NO official TEST data are used.
# ============================================================

import numpy as np
import pandas as pd


print(
    "\n"
    "============================================================\n"
    "STEP 6.27H3-A — DEV-ONLY RELIABILITY THRESHOLD SELECTION\n"
    "============================================================"
)


# ============================================================
# 1. Required frozen objects
# ============================================================

required_627h3a = [
    "DEV_SELECTIVE_ENDPOINT_BASE_627H2A",
    "PRIMARY_SELECTIVE_GATE_627H2D",
    "PRIMARY_SELECTIVE_RISK_COLUMN_627H2D",
    "PRIMARY_GATE_ORIENTATION_627H2D",
    "PRIMARY_GATE_EVIDENCE_627H2D",
]


print("\n--- REQUIRED FROZEN OBJECTS ---")

for name in required_627h3a:

    exists = name in globals()

    print(
        f"{name:<45}: {exists}"
    )

    if not exists:
        raise RuntimeError(
            f"Missing frozen object: {name}"
        )


# ============================================================
# 2. Verify frozen gate contract
# ============================================================

if PRIMARY_SELECTIVE_GATE_627H2D != "uncertainty_only":

    raise RuntimeError(
        "Primary gate is no longer uncertainty_only."
    )


if PRIMARY_GATE_ORIENTATION_627H2D != "higher_is_worse":

    raise RuntimeError(
        "Unexpected risk orientation."
    )


risk_col_627h3a = (
    PRIMARY_SELECTIVE_RISK_COLUMN_627H2D
)


if (
    risk_col_627h3a
    not in
    DEV_SELECTIVE_ENDPOINT_BASE_627H2A.columns
):

    raise RuntimeError(
        f"Frozen risk column missing: {risk_col_627h3a}"
    )


print("\n--- FROZEN GATE CONTRACT ---")

print(
    "Gate        :",
    PRIMARY_SELECTIVE_GATE_627H2D
)

print(
    "Risk column :",
    risk_col_627h3a
)

print(
    "Orientation :",
    PRIMARY_GATE_ORIENTATION_627H2D
)


# ============================================================
# 3. Threshold-selection rule
#
# Fixed BEFORE inspecting results below.
# ============================================================

THRESHOLD_CANDIDATES_627H3A = np.round(
    np.arange(
        0.50,
        1.0001,
        0.05
    ),
    2
)


MINIMUM_MEAN_COVERAGE_627H3A = 0.80

MINIMUM_SCENARIO_COVERAGE_627H3A = 0.50


THRESHOLD_SELECTION_RULE_627H3A = {
    "objective":
        "highest_feasible_risk_threshold",

    "minimum_mean_coverage":
        MINIMUM_MEAN_COVERAGE_627H3A,

    "minimum_scenario_coverage":
        MINIMUM_SCENARIO_COVERAGE_627H3A,

    "overall_mean_mae_gain":
        "strictly_positive",

    "every_family_mean_mae_gain":
        "strictly_positive",

    "every_severity_mean_mae_gain":
        "strictly_positive",
}


print("\n--- THRESHOLD-SELECTION RULE ---")

print(
    "Candidate thresholds        :",
    THRESHOLD_CANDIDATES_627H3A.tolist()
)

print(
    "Minimum mean coverage       :",
    MINIMUM_MEAN_COVERAGE_627H3A
)

print(
    "Minimum scenario coverage   :",
    MINIMUM_SCENARIO_COVERAGE_627H3A
)

print(
    "Selection objective         : highest feasible threshold"
)


# ============================================================
# 4. Working DEV table
# ============================================================

work_627h3a = (
    DEV_SELECTIVE_ENDPOINT_BASE_627H2A
    .copy()
)


risk_values_627h3a = pd.to_numeric(
    work_627h3a[
        risk_col_627h3a
    ],
    errors="coerce",
).to_numpy(
    dtype=float
)


if not np.isfinite(
    risk_values_627h3a
).all():

    raise RuntimeError(
        "Primary DEV risk contains non-finite values."
    )


if (
    (risk_values_627h3a < 0.0).any()
    or
    (risk_values_627h3a > 1.0).any()
):

    raise RuntimeError(
        "Primary DEV risk lies outside [0,1]."
    )


# ============================================================
# 5. Evaluate every threshold at scenario level
# ============================================================

scenario_records_627h3a = []


for threshold_627h3a in THRESHOLD_CANDIDATES_627H3A:

    for scenario_id_627h3a, group_627h3a in (
        work_627h3a
        .groupby(
            "scenario_id",
            sort=False
        )
    ):

        family_627h3a = (
            group_627h3a[
                "family"
            ].iloc[0]
        )

        severity_627h3a = (
            group_627h3a[
                "severity"
            ].iloc[0]
        )

        accepted_mask_627h3a = (
            group_627h3a[
                risk_col_627h3a
            ]
            <=
            threshold_627h3a
        )


        accepted_n_627h3a = int(
            accepted_mask_627h3a.sum()
        )


        total_n_627h3a = len(
            group_627h3a
        )


        coverage_627h3a = (
            accepted_n_627h3a
            /
            total_n_627h3a
        )


        full_mae_627h3a = float(
            group_627h3a[
                "corrupted_abs_error"
            ].mean()
        )


        if accepted_n_627h3a > 0:

            accepted_mae_627h3a = float(
                group_627h3a.loc[
                    accepted_mask_627h3a,
                    "corrupted_abs_error"
                ]
                .mean()
            )

            mae_gain_627h3a = (
                full_mae_627h3a
                -
                accepted_mae_627h3a
            )

        else:

            accepted_mae_627h3a = np.nan
            mae_gain_627h3a = np.nan


        scenario_records_627h3a.append(
            {
                "threshold":
                    float(
                        threshold_627h3a
                    ),

                "scenario_id":
                    scenario_id_627h3a,

                "family":
                    family_627h3a,

                "severity":
                    severity_627h3a,

                "accepted":
                    accepted_n_627h3a,

                "total":
                    total_n_627h3a,

                "coverage":
                    coverage_627h3a,

                "full_mae":
                    full_mae_627h3a,

                "accepted_mae":
                    accepted_mae_627h3a,

                "mae_gain":
                    mae_gain_627h3a,
            }
        )


DEV_THRESHOLD_SCENARIO_RESULTS_627H3A = pd.DataFrame(
    scenario_records_627h3a
)


# ============================================================
# 6. Family-level threshold evidence
# ============================================================

DEV_THRESHOLD_FAMILY_RESULTS_627H3A = (
    DEV_THRESHOLD_SCENARIO_RESULTS_627H3A
    .groupby(
        [
            "threshold",
            "family",
        ],
        as_index=False,
    )
    .agg(
        scenarios=(
            "scenario_id",
            "count",
        ),

        mean_coverage=(
            "coverage",
            "mean",
        ),

        mean_mae_gain=(
            "mae_gain",
            "mean",
        ),
    )
)


# ============================================================
# 7. Severity-level threshold evidence
# ============================================================

DEV_THRESHOLD_SEVERITY_RESULTS_627H3A = (
    DEV_THRESHOLD_SCENARIO_RESULTS_627H3A
    .groupby(
        [
            "threshold",
            "severity",
        ],
        as_index=False,
    )
    .agg(
        scenarios=(
            "scenario_id",
            "count",
        ),

        mean_coverage=(
            "coverage",
            "mean",
        ),

        mean_mae_gain=(
            "mae_gain",
            "mean",
        ),
    )
)


# ============================================================
# 8. Overall threshold summaries
# ============================================================

threshold_records_627h3a = []


for threshold_627h3a in THRESHOLD_CANDIDATES_627H3A:

    scenario_part_627h3a = (
        DEV_THRESHOLD_SCENARIO_RESULTS_627H3A.loc[
            np.isclose(
                DEV_THRESHOLD_SCENARIO_RESULTS_627H3A[
                    "threshold"
                ],
                threshold_627h3a,
            )
        ]
    )


    family_part_627h3a = (
        DEV_THRESHOLD_FAMILY_RESULTS_627H3A.loc[
            np.isclose(
                DEV_THRESHOLD_FAMILY_RESULTS_627H3A[
                    "threshold"
                ],
                threshold_627h3a,
            )
        ]
    )


    severity_part_627h3a = (
        DEV_THRESHOLD_SEVERITY_RESULTS_627H3A.loc[
            np.isclose(
                DEV_THRESHOLD_SEVERITY_RESULTS_627H3A[
                    "threshold"
                ],
                threshold_627h3a,
            )
        ]
    )


    all_scenarios_nonempty_627h3a = bool(
        (
            scenario_part_627h3a[
                "accepted"
            ]
            >
            0
        ).all()
    )


    mean_coverage_627h3a = float(
        scenario_part_627h3a[
            "coverage"
        ].mean()
    )


    minimum_coverage_627h3a = float(
        scenario_part_627h3a[
            "coverage"
        ].min()
    )


    mean_gain_627h3a = float(
        scenario_part_627h3a[
            "mae_gain"
        ].mean()
    )


    median_gain_627h3a = float(
        scenario_part_627h3a[
            "mae_gain"
        ].median()
    )


    improvement_fraction_627h3a = float(
        np.mean(
            scenario_part_627h3a[
                "mae_gain"
            ]
            .to_numpy(
                dtype=float
            )
            >
            0.0
        )
    )


    all_family_positive_627h3a = bool(
        (
            family_part_627h3a[
                "mean_mae_gain"
            ]
            >
            0.0
        ).all()
    )


    all_severity_positive_627h3a = bool(
        (
            severity_part_627h3a[
                "mean_mae_gain"
            ]
            >
            0.0
        ).all()
    )


    feasible_627h3a = bool(
        all_scenarios_nonempty_627h3a
        and
        mean_coverage_627h3a
        >=
        MINIMUM_MEAN_COVERAGE_627H3A
        and
        minimum_coverage_627h3a
        >=
        MINIMUM_SCENARIO_COVERAGE_627H3A
        and
        mean_gain_627h3a
        >
        0.0
        and
        all_family_positive_627h3a
        and
        all_severity_positive_627h3a
    )


    threshold_records_627h3a.append(
        {
            "threshold":
                float(
                    threshold_627h3a
                ),

            "mean_coverage":
                mean_coverage_627h3a,

            "minimum_scenario_coverage":
                minimum_coverage_627h3a,

            "mean_mae_gain":
                mean_gain_627h3a,

            "median_mae_gain":
                median_gain_627h3a,

            "scenario_improvement_fraction":
                improvement_fraction_627h3a,

            "all_family_mean_gain_positive":
                all_family_positive_627h3a,

            "all_severity_mean_gain_positive":
                all_severity_positive_627h3a,

            "feasible":
                feasible_627h3a,
        }
    )


DEV_THRESHOLD_SUMMARY_627H3A = pd.DataFrame(
    threshold_records_627h3a
)


# ============================================================
# 9. Select highest feasible threshold
# ============================================================

feasible_thresholds_627h3a = (
    DEV_THRESHOLD_SUMMARY_627H3A.loc[
        DEV_THRESHOLD_SUMMARY_627H3A[
            "feasible"
        ]
    ]
    .copy()
)


if feasible_thresholds_627h3a.empty:

    raise RuntimeError(
        "No threshold satisfies the pre-frozen "
        "coverage + robustness constraints. "
        "Do not improvise a replacement threshold."
    )


SELECTED_RELIABILITY_THRESHOLD_627H3A = float(
    feasible_thresholds_627h3a[
        "threshold"
    ].max()
)


SELECTED_THRESHOLD_ROW_627H3A = (
    DEV_THRESHOLD_SUMMARY_627H3A.loc[
        np.isclose(
            DEV_THRESHOLD_SUMMARY_627H3A[
                "threshold"
            ],
            SELECTED_RELIABILITY_THRESHOLD_627H3A,
        )
    ]
    .iloc[0]
    .copy()
)


# ============================================================
# 10. Compact threshold table
# ============================================================

print("\n--- THRESHOLD CANDIDATE SUMMARY ---")

print(
    DEV_THRESHOLD_SUMMARY_627H3A[
        [
            "threshold",
            "mean_coverage",
            "minimum_scenario_coverage",
            "mean_mae_gain",
            "scenario_improvement_fraction",
            "all_family_mean_gain_positive",
            "all_severity_mean_gain_positive",
            "feasible",
        ]
    ]
    .round(6)
    .to_string(
        index=False
    )
)


# ============================================================
# 11. Selected operating point
# ============================================================

print("\n--- SELECTED DEV OPERATING POINT ---")

print(
    "Selected threshold             :",
    SELECTED_RELIABILITY_THRESHOLD_627H3A
)

print(
    "Mean DEV coverage              :",
    float(
        SELECTED_THRESHOLD_ROW_627H3A[
            "mean_coverage"
        ]
    )
)

print(
    "Minimum scenario coverage      :",
    float(
        SELECTED_THRESHOLD_ROW_627H3A[
            "minimum_scenario_coverage"
        ]
    )
)

print(
    "Mean MAE gain                  :",
    float(
        SELECTED_THRESHOLD_ROW_627H3A[
            "mean_mae_gain"
        ]
    )
)

print(
    "Scenario improvement fraction  :",
    float(
        SELECTED_THRESHOLD_ROW_627H3A[
            "scenario_improvement_fraction"
        ]
    )
)

print(
    "All family gains positive      :",
    bool(
        SELECTED_THRESHOLD_ROW_627H3A[
            "all_family_mean_gain_positive"
        ]
    )
)

print(
    "All severity gains positive    :",
    bool(
        SELECTED_THRESHOLD_ROW_627H3A[
            "all_severity_mean_gain_positive"
        ]
    )
)


# ============================================================
# 12. Methodological state
# ============================================================

print(
    "\n"
    "============================================================\n"
    "STEP 6.27H3-A SUMMARY\n"
    "============================================================"
)

print(
    "Primary gate                  :",
    PRIMARY_SELECTIVE_GATE_627H2D
)

print(
    "Gate already frozen           : True"
)

print(
    "Threshold selection rule      :",
    THRESHOLD_SELECTION_RULE_627H3A[
        "objective"
    ]
)

print(
    "Selected threshold            :",
    SELECTED_RELIABILITY_THRESHOLD_627H3A
)

print(
    "Threshold selected on DEV     : True"
)

print(
    "Official TEST used            : False"
)

print(
    "Gate changed                  : False"
)

print(
    "Files written                 : False"
)

print(
    "\n=== END STEP 6.27H3-A ==="
)

In [ ]:
# ============================================================
# STEP 6.27H3-A-R
# COMPACT THRESHOLD-SELECTION REVIEW
# ============================================================

print(
    "\n"
    "============================================================\n"
    "STEP 6.27H3-A-R — COMPACT THRESHOLD REVIEW\n"
    "============================================================"
)

# ------------------------------------------------------------
# 1. Candidate thresholds
# ------------------------------------------------------------

print("\n--- ALL THRESHOLD CANDIDATES ---")

for _, row in DEV_THRESHOLD_SUMMARY_627H3A.iterrows():

    print(
        f"threshold={row['threshold']:.2f} | "
        f"mean_cov={row['mean_coverage']:.6f} | "
        f"min_cov={row['minimum_scenario_coverage']:.6f} | "
        f"mean_gain={row['mean_mae_gain']:.6f} | "
        f"improved={row['scenario_improvement_fraction']:.6f} | "
        f"family_ok={bool(row['all_family_mean_gain_positive'])} | "
        f"severity_ok={bool(row['all_severity_mean_gain_positive'])} | "
        f"feasible={bool(row['feasible'])}"
    )


# ------------------------------------------------------------
# 2. Feasible thresholds only
# ------------------------------------------------------------

print("\n--- FEASIBLE THRESHOLDS ---")

feasible_review_627h3ar = (
    DEV_THRESHOLD_SUMMARY_627H3A[
        DEV_THRESHOLD_SUMMARY_627H3A["feasible"]
    ]
    .copy()
    .sort_values("threshold")
)

if feasible_review_627h3ar.empty:

    print("NONE")

else:

    for _, row in feasible_review_627h3ar.iterrows():

        print(
            f"{row['threshold']:.2f} "
            f"(coverage={row['mean_coverage']:.6f}, "
            f"gain={row['mean_mae_gain']:.6f})"
        )


# ------------------------------------------------------------
# 3. Selected threshold
# ------------------------------------------------------------

print("\n--- SELECTED OPERATING POINT ---")

print(
    "Selected threshold        :",
    SELECTED_RELIABILITY_THRESHOLD_627H3A
)

print(
    "Mean DEV coverage         :",
    float(
        SELECTED_THRESHOLD_ROW_627H3A[
            "mean_coverage"
        ]
    )
)

print(
    "Minimum scenario coverage :",
    float(
        SELECTED_THRESHOLD_ROW_627H3A[
            "minimum_scenario_coverage"
        ]
    )
)

print(
    "Mean MAE gain             :",
    float(
        SELECTED_THRESHOLD_ROW_627H3A[
            "mean_mae_gain"
        ]
    )
)

print(
    "Median MAE gain           :",
    float(
        SELECTED_THRESHOLD_ROW_627H3A[
            "median_mae_gain"
        ]
    )
)

print(
    "Scenario improvement frac :",
    float(
        SELECTED_THRESHOLD_ROW_627H3A[
            "scenario_improvement_fraction"
        ]
    )
)

print(
    "Every family positive     :",
    bool(
        SELECTED_THRESHOLD_ROW_627H3A[
            "all_family_mean_gain_positive"
        ]
    )
)

print(
    "Every severity positive   :",
    bool(
        SELECTED_THRESHOLD_ROW_627H3A[
            "all_severity_mean_gain_positive"
        ]
    )
)


# ------------------------------------------------------------
# 4. Selected-threshold family evidence
# ------------------------------------------------------------

selected_family_627h3ar = (
    DEV_THRESHOLD_FAMILY_RESULTS_627H3A.loc[
        DEV_THRESHOLD_FAMILY_RESULTS_627H3A[
            "threshold"
        ].eq(
            SELECTED_RELIABILITY_THRESHOLD_627H3A
        )
    ]
    .copy()
)


print("\n--- FAMILY EVIDENCE AT SELECTED THRESHOLD ---")

print(
    selected_family_627h3ar[
        [
            "family",
            "mean_coverage",
            "mean_mae_gain",
        ]
    ]
    .round(6)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 5. Selected-threshold severity evidence
# ------------------------------------------------------------

selected_severity_627h3ar = (
    DEV_THRESHOLD_SEVERITY_RESULTS_627H3A.loc[
        DEV_THRESHOLD_SEVERITY_RESULTS_627H3A[
            "threshold"
        ].eq(
            SELECTED_RELIABILITY_THRESHOLD_627H3A
        )
    ]
    .copy()
)


print("\n--- SEVERITY EVIDENCE AT SELECTED THRESHOLD ---")

print(
    selected_severity_627h3ar[
        [
            "severity",
            "mean_coverage",
            "mean_mae_gain",
        ]
    ]
    .round(6)
    .to_string(index=False)
)


print(
    "\n"
    "============================================================\n"
    "STEP 6.27H3-A-R STATUS\n"
    "============================================================"
)

print("Gate frozen          : True")
print("Threshold selected   : True")
print("Official TEST used   : False")
print("Files written        : False")

print(
    "\n=== END STEP 6.27H3-A-R ==="
)

In [ ]:
# ============================================================
# STEP 6.27H3-B
# FREEZE FINAL DEV-SELECTED OPERATING THRESHOLD
# ============================================================

import numpy as np
import pandas as pd


print(
    "\n"
    "============================================================\n"
    "STEP 6.27H3-B — FREEZE FINAL OPERATING THRESHOLD\n"
    "============================================================"
)


# ============================================================
# 1. Required frozen evidence
# ============================================================

required_627h3b = [
    "PRIMARY_SELECTIVE_GATE_627H2D",
    "PRIMARY_SELECTIVE_RISK_COLUMN_627H2D",
    "PRIMARY_GATE_ORIENTATION_627H2D",
    "PRIMARY_GATE_EVIDENCE_627H2D",
    "THRESHOLD_SELECTION_RULE_627H3A",
    "DEV_THRESHOLD_SUMMARY_627H3A",
    "SELECTED_RELIABILITY_THRESHOLD_627H3A",
    "SELECTED_THRESHOLD_ROW_627H3A",
]


print("\n--- REQUIRED EVIDENCE ---")

for name in required_627h3b:

    exists = name in globals()

    print(
        f"{name:<45}: {exists}"
    )

    if not exists:
        raise RuntimeError(
            f"Missing threshold-freeze evidence: {name}"
        )


# ============================================================
# 2. Confirm deterministic selected threshold
# ============================================================

feasible_627h3b = (
    DEV_THRESHOLD_SUMMARY_627H3A.loc[
        DEV_THRESHOLD_SUMMARY_627H3A[
            "feasible"
        ]
    ]
    .copy()
)


if feasible_627h3b.empty:

    raise RuntimeError(
        "No feasible threshold exists."
    )


expected_threshold_627h3b = float(
    feasible_627h3b[
        "threshold"
    ].max()
)


if not np.isclose(
    SELECTED_RELIABILITY_THRESHOLD_627H3A,
    expected_threshold_627h3b,
    atol=1e-12,
    rtol=0.0,
):

    raise RuntimeError(
        "Selected threshold does not equal the "
        "highest feasible DEV threshold."
    )


# ============================================================
# 3. Explicit expected closure
# ============================================================

if not np.isclose(
    SELECTED_RELIABILITY_THRESHOLD_627H3A,
    0.95,
    atol=1e-12,
    rtol=0.0,
):

    raise RuntimeError(
        "Current frozen evidence does not select 0.95. "
        "Do not continue."
    )


# ============================================================
# 4. Freeze final operating rule
# ============================================================

FINAL_SELECTIVE_GATE_627H3B = (
    PRIMARY_SELECTIVE_GATE_627H2D
)

FINAL_SELECTIVE_RISK_COLUMN_627H3B = (
    PRIMARY_SELECTIVE_RISK_COLUMN_627H2D
)

FINAL_SELECTIVE_ORIENTATION_627H3B = (
    PRIMARY_GATE_ORIENTATION_627H2D
)

FINAL_RELIABILITY_THRESHOLD_627H3B = float(
    SELECTED_RELIABILITY_THRESHOLD_627H3A
)


FINAL_ACCEPT_RULE_627H3B = (
    "accept_if_risk_le_threshold"
)

FINAL_ABSTAIN_RULE_627H3B = (
    "abstain_if_risk_gt_threshold"
)


# ============================================================
# 5. Freeze DEV-selection evidence
# ============================================================

FINAL_THRESHOLD_EVIDENCE_627H3B = {

    "selection_dataset":
        "DEV_only",

    "gate":
        FINAL_SELECTIVE_GATE_627H3B,

    "risk_column":
        FINAL_SELECTIVE_RISK_COLUMN_627H3B,

    "orientation":
        FINAL_SELECTIVE_ORIENTATION_627H3B,

    "threshold":
        FINAL_RELIABILITY_THRESHOLD_627H3B,

    "accept_rule":
        FINAL_ACCEPT_RULE_627H3B,

    "abstain_rule":
        FINAL_ABSTAIN_RULE_627H3B,

    "selection_objective":
        THRESHOLD_SELECTION_RULE_627H3A[
            "objective"
        ],

    "minimum_required_mean_coverage":
        THRESHOLD_SELECTION_RULE_627H3A[
            "minimum_mean_coverage"
        ],

    "minimum_required_scenario_coverage":
        THRESHOLD_SELECTION_RULE_627H3A[
            "minimum_scenario_coverage"
        ],

    "observed_mean_dev_coverage":
        float(
            SELECTED_THRESHOLD_ROW_627H3A[
                "mean_coverage"
            ]
        ),

    "observed_minimum_scenario_coverage":
        float(
            SELECTED_THRESHOLD_ROW_627H3A[
                "minimum_scenario_coverage"
            ]
        ),

    "observed_mean_mae_gain":
        float(
            SELECTED_THRESHOLD_ROW_627H3A[
                "mean_mae_gain"
            ]
        ),

    "observed_median_mae_gain":
        float(
            SELECTED_THRESHOLD_ROW_627H3A[
                "median_mae_gain"
            ]
        ),

    "scenario_improvement_fraction":
        float(
            SELECTED_THRESHOLD_ROW_627H3A[
                "scenario_improvement_fraction"
            ]
        ),

    "all_family_mean_gains_positive":
        bool(
            SELECTED_THRESHOLD_ROW_627H3A[
                "all_family_mean_gain_positive"
            ]
        ),

    "all_severity_mean_gains_positive":
        bool(
            SELECTED_THRESHOLD_ROW_627H3A[
                "all_severity_mean_gain_positive"
            ]
        ),
}


# ============================================================
# 6. Final frozen contract
# ============================================================

print("\n--- FINAL SELECTIVE-RUL OPERATING CONTRACT ---")

print(
    "Primary gate           :",
    FINAL_SELECTIVE_GATE_627H3B
)

print(
    "Risk column            :",
    FINAL_SELECTIVE_RISK_COLUMN_627H3B
)

print(
    "Risk orientation       :",
    FINAL_SELECTIVE_ORIENTATION_627H3B
)

print(
    "Frozen threshold       :",
    FINAL_RELIABILITY_THRESHOLD_627H3B
)

print(
    "Accept rule            : risk <= threshold"
)

print(
    "Abstain rule           : risk > threshold"
)


print("\n--- DEV SELECTION EVIDENCE ---")

print(
    "Mean DEV coverage      :",
    FINAL_THRESHOLD_EVIDENCE_627H3B[
        "observed_mean_dev_coverage"
    ]
)

print(
    "Minimum scenario cov.  :",
    FINAL_THRESHOLD_EVIDENCE_627H3B[
        "observed_minimum_scenario_coverage"
    ]
)

print(
    "Mean MAE gain          :",
    FINAL_THRESHOLD_EVIDENCE_627H3B[
        "observed_mean_mae_gain"
    ]
)

print(
    "Scenario improved frac :",
    FINAL_THRESHOLD_EVIDENCE_627H3B[
        "scenario_improvement_fraction"
    ]
)

print(
    "Every family positive  :",
    FINAL_THRESHOLD_EVIDENCE_627H3B[
        "all_family_mean_gains_positive"
    ]
)

print(
    "Every severity positive:",
    FINAL_THRESHOLD_EVIDENCE_627H3B[
        "all_severity_mean_gains_positive"
    ]
)


# ============================================================
# 7. Freeze status
# ============================================================

print(
    "\n"
    "============================================================\n"
    "STEP 6.27H3-B FINAL FREEZE SUMMARY\n"
    "============================================================"
)

print(
    "Primary ranking gate frozen    : True"
)

print(
    "Reliability threshold frozen   : True"
)

print(
    "Final threshold                :",
    FINAL_RELIABILITY_THRESHOLD_627H3B
)

print(
    "Selected using DEV only        : True"
)

print(
    "Official TEST used for tuning  : False"
)

print(
    "Gate may be retuned on TEST    : False"
)

print(
    "Threshold may be retuned TEST  : False"
)

print(
    "Sensor/fusion remain ablations : True"
)

print(
    "Files written                  : False"
)

print(
    "\n=== END STEP 6.27H3-B ==="
)

In [ ]:
# ============================================================
# STEP 6.27H4-A
# FD001 OFFICIAL-TEST PROVENANCE / PRE-FLIGHT AUDIT
#
# IMPORTANT:
# - Does NOT read official RUL values.
# - Does NOT score TEST.
# - Does NOT rerun model or MC dropout.
# - Does NOT change gate or threshold.
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np


print(
    "\n"
    "============================================================\n"
    "STEP 6.27H4-A — FD001 TEST PROVENANCE PRE-FLIGHT\n"
    "============================================================"
)


# ============================================================
# 1. Frozen selective-policy contract
# ============================================================

required_627h4a = [
    "FINAL_SELECTIVE_GATE_627H3B",
    "FINAL_SELECTIVE_RISK_COLUMN_627H3B",
    "FINAL_SELECTIVE_ORIENTATION_627H3B",
    "FINAL_RELIABILITY_THRESHOLD_627H3B",
    "FINAL_THRESHOLD_EVIDENCE_627H3B",
]


print("\n--- FROZEN SELECTIVE POLICY ---")

for name in required_627h4a:

    exists = name in globals()

    print(
        f"{name:<44}: {exists}"
    )

    if not exists:
        raise RuntimeError(
            f"Missing frozen object: {name}"
        )


print(
    "\nGate       :",
    FINAL_SELECTIVE_GATE_627H3B
)

print(
    "Threshold  :",
    FINAL_RELIABILITY_THRESHOLD_627H3B
)

print(
    "Orientation:",
    FINAL_SELECTIVE_ORIENTATION_627H3B
)


if FINAL_SELECTIVE_GATE_627H3B != "uncertainty_only":

    raise RuntimeError(
        "Primary gate changed after freeze."
    )


if not np.isclose(
    FINAL_RELIABILITY_THRESHOLD_627H3B,
    0.95,
    atol=1e-12,
    rtol=0.0,
):

    raise RuntimeError(
        "Frozen threshold changed after H3-B."
    )


# ============================================================
# 2. Locate project root
# ============================================================

cwd_627h4a = Path.cwd()

if (
    cwd_627h4a
    /
    "02_dataset"
).exists():

    PROJECT_ROOT_627H4A = (
        cwd_627h4a
    )

elif (
    cwd_627h4a.parent
    /
    "02_dataset"
).exists():

    PROJECT_ROOT_627H4A = (
        cwd_627h4a.parent
    )

else:

    raise RuntimeError(
        "Could not identify project root."
    )


print(
    "\nProject root:",
    PROJECT_ROOT_627H4A
)


# ============================================================
# 3. Audit prior NOTEBOOK CODE references
#
# We inspect code strings only.
# We do NOT inspect any label values.
# ============================================================

history_627h4a = globals().get(
    "In",
    []
)


label_reference_terms_627h4a = [
    "rul_fd001",
    "rul_fd001.txt",
    "rul_fd00",
    "ground_truth",
    "test_labels",
    "official_test_labels",
]


test_reference_terms_627h4a = [
    "test_fd001",
    "official_test",
    "official_fd001",
    "fd001_official",
]


history_records_627h4a = []


# Exclude current cell from history search where possible.
history_to_scan_627h4a = (
    history_627h4a[:-1]
    if len(history_627h4a) > 0
    else []
)


for cell_number_627h4a, code_627h4a in enumerate(
    history_to_scan_627h4a
):

    if not isinstance(
        code_627h4a,
        str
    ):
        continue

    lower_627h4a = (
        code_627h4a.lower()
    )

    label_matches_627h4a = [
        term
        for term in label_reference_terms_627h4a
        if term in lower_627h4a
    ]

    test_matches_627h4a = [
        term
        for term in test_reference_terms_627h4a
        if term in lower_627h4a
    ]

    if (
        label_matches_627h4a
        or
        test_matches_627h4a
    ):

        history_records_627h4a.append(
            {
                "history_index":
                    cell_number_627h4a,

                "label_reference":
                    bool(
                        label_matches_627h4a
                    ),

                "test_reference":
                    bool(
                        test_matches_627h4a
                    ),

                "matched_terms":
                    ", ".join(
                        sorted(
                            set(
                                label_matches_627h4a
                                +
                                test_matches_627h4a
                            )
                        )
                    ),
            }
        )


TEST_HISTORY_AUDIT_627H4A = pd.DataFrame(
    history_records_627h4a
)


prior_label_code_references_627h4a = int(
    TEST_HISTORY_AUDIT_627H4A[
        "label_reference"
    ].sum()
) if not TEST_HISTORY_AUDIT_627H4A.empty else 0


prior_test_code_references_627h4a = int(
    TEST_HISTORY_AUDIT_627H4A[
        "test_reference"
    ].sum()
) if not TEST_HISTORY_AUDIT_627H4A.empty else 0


print(
    "\n--- NOTEBOOK-HISTORY PROVENANCE ---"
)

print(
    "Cells with TEST references  :",
    prior_test_code_references_627h4a
)

print(
    "Cells with label references :",
    prior_label_code_references_627h4a
)


if not TEST_HISTORY_AUDIT_627H4A.empty:

    print(
        "\nMatching history entries:"
    )

    print(
        TEST_HISTORY_AUDIT_627H4A
        .tail(20)
        .to_string(
            index=False
        )
    )


# ============================================================
# 4. Existing FD001 official/test artifact FILENAMES
#
# Filename/path audit only.
# Files are NOT opened.
# ============================================================

artifact_records_627h4a = []


for relative_folder_627h4a in [
    "05_experiments",
    "06_results",
    "07_figures",
]:

    base_627h4a = (
        PROJECT_ROOT_627H4A
        /
        relative_folder_627h4a
    )

    if not base_627h4a.exists():
        continue

    for path_627h4a in base_627h4a.rglob(
        "*"
    ):

        if not path_627h4a.is_file():
            continue

        relative_text_627h4a = str(
            path_627h4a.relative_to(
                PROJECT_ROOT_627H4A
            )
        )

        lower_path_627h4a = (
            relative_text_627h4a.lower()
        )

        fd001_related_627h4a = (
            "fd001"
            in
            lower_path_627h4a
        )

        official_or_test_627h4a = (
            "official"
            in
            lower_path_627h4a
            or
            "test"
            in
            lower_path_627h4a
        )

        if (
            fd001_related_627h4a
            and
            official_or_test_627h4a
        ):

            artifact_records_627h4a.append(
                {
                    "relative_path":
                        relative_text_627h4a,

                    "suffix":
                        path_627h4a.suffix,
                }
            )


TEST_ARTIFACT_AUDIT_627H4A = pd.DataFrame(
    artifact_records_627h4a
)


print(
    "\n--- EXISTING FD001 OFFICIAL/TEST ARTIFACTS ---"
)

print(
    "Artifact files detected:",
    len(
        TEST_ARTIFACT_AUDIT_627H4A
    )
)


if not TEST_ARTIFACT_AUDIT_627H4A.empty:

    print(
        TEST_ARTIFACT_AUDIT_627H4A
        .head(30)
        .to_string(
            index=False
        )
    )


# ============================================================
# 5. Current in-memory TEST-like objects
#
# Metadata only: name/type/shape.
# No values displayed.
# ============================================================

memory_records_627h4a = []


for object_name_627h4a, object_value_627h4a in list(
    globals().items()
):

    lower_name_627h4a = (
        str(
            object_name_627h4a
        )
        .lower()
    )

    test_like_627h4a = (
        "test"
        in
        lower_name_627h4a
        or
        "official"
        in
        lower_name_627h4a
        or
        "fd001"
        in
        lower_name_627h4a
    )

    if not test_like_627h4a:
        continue

    shape_627h4a = getattr(
        object_value_627h4a,
        "shape",
        None
    )

    memory_records_627h4a.append(
        {
            "object_name":
                object_name_627h4a,

            "type":
                type(
                    object_value_627h4a
                ).__name__,

            "shape":
                str(
                    shape_627h4a
                ),
        }
    )


TEST_MEMORY_INVENTORY_627H4A = (
    pd.DataFrame(
        memory_records_627h4a
    )
    .sort_values(
        "object_name"
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n--- IN-MEMORY TEST-LIKE OBJECTS ---"
)

if TEST_MEMORY_INVENTORY_627H4A.empty:

    print(
        "NONE FOUND"
    )

else:

    print(
        TEST_MEMORY_INVENTORY_627H4A
        .to_string(
            index=False
        )
    )


# ============================================================
# 6. Provenance conclusion
# ============================================================

PRIOR_TEST_ARTIFACTS_DETECTED_627H4A = bool(
    len(
        TEST_ARTIFACT_AUDIT_627H4A
    )
    >
    0
)


PRIOR_LABEL_REFERENCES_DETECTED_627H4A = bool(
    prior_label_code_references_627h4a
    >
    0
)


FD001_GLOBALLY_UNTOUCHED_627H4A = bool(
    not (
        PRIOR_TEST_ARTIFACTS_DETECTED_627H4A
        or
        PRIOR_LABEL_REFERENCES_DETECTED_627H4A
    )
)


SELECTIVE_PROTOCOL_PROVENANCE_627H4A = {

    "primary_gate":
        FINAL_SELECTIVE_GATE_627H3B,

    "threshold":
        FINAL_RELIABILITY_THRESHOLD_627H3B,

    "gate_selected_on":
        "DEV",

    "threshold_selected_on":
        "DEV",

    "official_test_used_in_H2_H3_selection":
        False,

    "prior_label_references_detected":
        PRIOR_LABEL_REFERENCES_DETECTED_627H4A,

    "prior_test_artifacts_detected":
        PRIOR_TEST_ARTIFACTS_DETECTED_627H4A,

    "fd001_globally_untouched":
        FD001_GLOBALLY_UNTOUCHED_627H4A,

    "labels_read_in_this_step":
        False,

    "test_scored_in_this_step":
        False,

    "model_rerun_in_this_step":
        False,
}


print(
    "\n"
    "============================================================\n"
    "STEP 6.27H4-A PROVENANCE SUMMARY\n"
    "============================================================"
)

print(
    "Gate selected on DEV               : True"
)

print(
    "Threshold selected on DEV          : True"
)

print(
    "Official TEST used in H2/H3 tuning : False"
)

print(
    "Prior label references detected    :",
    PRIOR_LABEL_REFERENCES_DETECTED_627H4A
)

print(
    "Prior TEST artifacts detected      :",
    PRIOR_TEST_ARTIFACTS_DETECTED_627H4A
)

print(
    "FD001 globally untouched           :",
    FD001_GLOBALLY_UNTOUCHED_627H4A
)

print(
    "TEST labels read in this step      : False"
)

print(
    "TEST scored in this step           : False"
)

print(
    "Model / MC rerun                   : False"
)

print(
    "Files written                      : False"
)

print(
    "\n=== END STEP 6.27H4-A ==="
)

In [ ]:
# ============================================================
# STEP 6.27H4-B
# FD001 TEST INFERENCE-ARTIFACT CONTRACT AUDIT
#
# Purpose:
#   Locate existing TEST prediction / uncertainty artifacts
#   WITHOUT reading official RUL labels.
#
# NO scoring.
# NO threshold tuning.
# NO model rerun.
# NO MC-dropout rerun.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd


print(
    "\n"
    "============================================================\n"
    "STEP 6.27H4-B — FD001 TEST INFERENCE-ARTIFACT AUDIT\n"
    "============================================================"
)


# ============================================================
# 1. Frozen policy contract
# ============================================================

required_627h4b = [
    "FINAL_SELECTIVE_GATE_627H3B",
    "FINAL_RELIABILITY_THRESHOLD_627H3B",
    "FINAL_SELECTIVE_ORIENTATION_627H3B",
]


print("\n--- FROZEN OPERATING POLICY ---")

for name in required_627h4b:

    exists = name in globals()

    print(
        f"{name:<42}: {exists}"
    )

    if not exists:

        raise RuntimeError(
            f"Missing frozen object: {name}"
        )


print(
    "\nGate      :",
    FINAL_SELECTIVE_GATE_627H3B
)

print(
    "Threshold :",
    FINAL_RELIABILITY_THRESHOLD_627H3B
)

print(
    "Direction :",
    FINAL_SELECTIVE_ORIENTATION_627H3B
)


# ============================================================
# 2. Project root
# ============================================================

cwd_627h4b = Path.cwd()


if (
    cwd_627h4b
    /
    "02_dataset"
).exists():

    PROJECT_ROOT_627H4B = (
        cwd_627h4b
    )

elif (
    cwd_627h4b.parent
    /
    "02_dataset"
).exists():

    PROJECT_ROOT_627H4B = (
        cwd_627h4b.parent
    )

else:

    raise RuntimeError(
        "Could not locate project root."
    )


print(
    "\nProject root:",
    PROJECT_ROOT_627H4B
)


# ============================================================
# 3. Column-name definitions
#
# Target-like columns are forbidden in candidate inference
# artifacts at this stage.
# ============================================================

prediction_terms_627h4b = [
    "prediction",
    "predicted_rul",
    "pred_rul",
]


uncertainty_terms_627h4b = [
    "mc_std",
    "corrupted_mc_std",
    "clean_mc_std",
    "uncertainty",
    "adjusted_uncertainty",
]


key_terms_627h4b = [
    "unit_id",
    "cycle",
]


forbidden_target_terms_627h4b = [
    "rul",
    "target",
    "ground_truth",
    "true_rul",
    "actual_rul",
    "label",
]


# ============================================================
# 4. In-memory DataFrame inventory
#
# We inspect metadata / columns only.
# No target values are printed.
# ============================================================

memory_records_627h4b = []


for object_name_627h4b, obj_627h4b in list(
    globals().items()
):

    if not isinstance(
        obj_627h4b,
        pd.DataFrame
    ):

        continue


    cols_627h4b = [
        str(c)
        for c in obj_627h4b.columns
    ]


    lower_cols_627h4b = [
        c.lower()
        for c in cols_627h4b
    ]


    lower_name_627h4b = (
        str(
            object_name_627h4b
        )
        .lower()
    )


    test_like_name_627h4b = any(
        term in lower_name_627h4b

        for term in [
            "test",
            "official",
            "fd001",
        ]
    )


    has_prediction_627h4b = any(
        any(
            term in col

            for term in prediction_terms_627h4b
        )

        for col in lower_cols_627h4b
    )


    has_uncertainty_627h4b = any(
        any(
            term in col

            for term in uncertainty_terms_627h4b
        )

        for col in lower_cols_627h4b
    )


    target_like_cols_627h4b = [
        col

        for col in cols_627h4b

        if any(
            term
            in
            col.lower()

            for term in forbidden_target_terms_627h4b
        )
    ]


    if (
        test_like_name_627h4b
        or
        (
            len(obj_627h4b) == 100
            and
            has_prediction_627h4b
        )
    ):

        memory_records_627h4b.append(
            {
                "object_name":
                    object_name_627h4b,

                "rows":
                    len(
                        obj_627h4b
                    ),

                "columns":
                    len(
                        obj_627h4b.columns
                    ),

                "has_prediction":
                    has_prediction_627h4b,

                "has_uncertainty":
                    has_uncertainty_627h4b,

                "target_like_columns":
                    ", ".join(
                        target_like_cols_627h4b
                    ),

                "candidate_label_free":
                    bool(
                        len(obj_627h4b) == 100
                        and
                        has_prediction_627h4b
                        and
                        has_uncertainty_627h4b
                        and
                        len(
                            target_like_cols_627h4b
                        ) == 0
                    ),
            }
        )


TEST_MEMORY_ARTIFACT_AUDIT_627H4B = pd.DataFrame(
    memory_records_627h4b
)


print(
    "\n--- IN-MEMORY TEST INFERENCE CANDIDATES ---"
)


if TEST_MEMORY_ARTIFACT_AUDIT_627H4B.empty:

    print(
        "NONE FOUND"
    )

else:

    print(
        TEST_MEMORY_ARTIFACT_AUDIT_627H4B
        .sort_values(
            [
                "candidate_label_free",
                "has_uncertainty",
                "has_prediction",
            ],
            ascending=[
                False,
                False,
                False,
            ],
        )
        .to_string(
            index=False
        )
    )


# ============================================================
# 5. Search existing SAVED result artifacts
#
# IMPORTANT:
# - Do NOT search 02_dataset.
# - Do NOT open RUL files.
# - CSV: header only.
# - Parquet/Feather: schema/columns only where possible.
# ============================================================

saved_records_627h4b = []


search_roots_627h4b = [
    PROJECT_ROOT_627H4B
    /
    "05_experiments",

    PROJECT_ROOT_627H4B
    /
    "06_results",
]


candidate_files_627h4b = []


for root_627h4b in search_roots_627h4b:

    if not root_627h4b.exists():

        continue


    for path_627h4b in root_627h4b.rglob(
        "*"
    ):

        if not path_627h4b.is_file():

            continue


        if path_627h4b.suffix.lower() not in [
            ".csv",
            ".parquet",
            ".feather",
        ]:

            continue


        lower_path_627h4b = (
            str(
                path_627h4b
            )
            .lower()
        )


        if not any(
            term in lower_path_627h4b

            for term in [
                "fd001",
                "official",
                "test",
            ]
        ):

            continue


        candidate_files_627h4b.append(
            path_627h4b
        )


for path_627h4b in candidate_files_627h4b:

    try:

        suffix_627h4b = (
            path_627h4b
            .suffix
            .lower()
        )


        if suffix_627h4b == ".csv":

            header_df_627h4b = (
                pd.read_csv(
                    path_627h4b,
                    nrows=0
                )
            )

            cols_627h4b = [
                str(c)
                for c in header_df_627h4b.columns
            ]


        elif suffix_627h4b == ".parquet":

            header_df_627h4b = (
                pd.read_parquet(
                    path_627h4b
                )
            )

            cols_627h4b = [
                str(c)
                for c in header_df_627h4b.columns
            ]


        elif suffix_627h4b == ".feather":

            header_df_627h4b = (
                pd.read_feather(
                    path_627h4b
                )
            )

            cols_627h4b = [
                str(c)
                for c in header_df_627h4b.columns
            ]


        else:

            continue


        lower_cols_627h4b = [
            c.lower()
            for c in cols_627h4b
        ]


        has_prediction_627h4b = any(
            any(
                term in col

                for term in prediction_terms_627h4b
            )

            for col in lower_cols_627h4b
        )


        has_uncertainty_627h4b = any(
            any(
                term in col

                for term in uncertainty_terms_627h4b
            )

            for col in lower_cols_627h4b
        )


        target_like_cols_627h4b = [
            col

            for col in cols_627h4b

            if any(
                term
                in
                col.lower()

                for term
                in
                forbidden_target_terms_627h4b
            )
        ]


        saved_records_627h4b.append(
            {
                "relative_path":
                    str(
                        path_627h4b.relative_to(
                            PROJECT_ROOT_627H4B
                        )
                    ),

                "columns":
                    len(
                        cols_627h4b
                    ),

                "has_prediction":
                    has_prediction_627h4b,

                "has_uncertainty":
                    has_uncertainty_627h4b,

                "target_like_columns":
                    ", ".join(
                        target_like_cols_627h4b
                    ),

                "candidate_label_free":
                    bool(
                        has_prediction_627h4b
                        and
                        has_uncertainty_627h4b
                        and
                        len(
                            target_like_cols_627h4b
                        ) == 0
                    ),
            }
        )


    except Exception as exc_627h4b:

        saved_records_627h4b.append(
            {
                "relative_path":
                    str(
                        path_627h4b.relative_to(
                            PROJECT_ROOT_627H4B
                        )
                    ),

                "columns":
                    np.nan,

                "has_prediction":
                    False,

                "has_uncertainty":
                    False,

                "target_like_columns":
                    "",

                "candidate_label_free":
                    False,

                "inspection_error":
                    type(
                        exc_627h4b
                    ).__name__,
            }
        )


TEST_SAVED_ARTIFACT_AUDIT_627H4B = pd.DataFrame(
    saved_records_627h4b
)


print(
    "\n--- SAVED TEST INFERENCE CANDIDATES ---"
)


if TEST_SAVED_ARTIFACT_AUDIT_627H4B.empty:

    print(
        "NONE FOUND"
    )

else:

    display_cols_627h4b = [
        col

        for col in [
            "relative_path",
            "columns",
            "has_prediction",
            "has_uncertainty",
            "target_like_columns",
            "candidate_label_free",
            "inspection_error",
        ]

        if col
        in
        TEST_SAVED_ARTIFACT_AUDIT_627H4B.columns
    ]


    print(
        TEST_SAVED_ARTIFACT_AUDIT_627H4B[
            display_cols_627h4b
        ]
        .sort_values(
            [
                "candidate_label_free",
                "has_uncertainty",
                "has_prediction",
            ],
            ascending=[
                False,
                False,
                False,
            ],
        )
        .head(40)
        .to_string(
            index=False
        )
    )


# ============================================================
# 6. Candidate counts
# ============================================================

memory_valid_count_627h4b = int(
    TEST_MEMORY_ARTIFACT_AUDIT_627H4B[
        "candidate_label_free"
    ].sum()
) if not TEST_MEMORY_ARTIFACT_AUDIT_627H4B.empty else 0


saved_valid_count_627h4b = int(
    TEST_SAVED_ARTIFACT_AUDIT_627H4B[
        "candidate_label_free"
    ].sum()
) if not TEST_SAVED_ARTIFACT_AUDIT_627H4B.empty else 0


print(
    "\n"
    "============================================================\n"
    "STEP 6.27H4-B SUMMARY\n"
    "============================================================"
)


print(
    "Label-free in-memory candidates :",
    memory_valid_count_627h4b
)

print(
    "Label-free saved candidates     :",
    saved_valid_count_627h4b
)

print(
    "Official RUL values read        : False"
)

print(
    "Official TEST scored            : False"
)

print(
    "Model inference rerun           : False"
)

print(
    "MC-dropout rerun                : False"
)

print(
    "Gate changed                    : False"
)

print(
    "Threshold changed               : False"
)

print(
    "Files written                   : False"
)

print(
    "\n=== END STEP 6.27H4-B ==="
)

In [ ]:
# ============================================================
# STEP 6.27H4-B-R
# EXACT EXISTING FD001 TEST INFERENCE CONTRACT REVIEW
#
# Purpose:
#   Determine whether existing 100-engine label-free
#   inference objects are exact / equivalent copies.
#
# NO official RUL labels.
# NO scoring.
# NO model rerun.
# NO MC-dropout rerun.
# NO files written.
# ============================================================

import numpy as np
import pandas as pd


print(
    "\n"
    "============================================================\n"
    "STEP 6.27H4-B-R — EXISTING TEST INFERENCE REVIEW\n"
    "============================================================"
)


# ============================================================
# 1. Recover label-free in-memory candidates from H4-B
# ============================================================

if "TEST_MEMORY_ARTIFACT_AUDIT_627H4B" not in globals():

    raise RuntimeError(
        "TEST_MEMORY_ARTIFACT_AUDIT_627H4B is missing."
    )


candidate_names_627h4br = (
    TEST_MEMORY_ARTIFACT_AUDIT_627H4B.loc[
        TEST_MEMORY_ARTIFACT_AUDIT_627H4B[
            "candidate_label_free"
        ],
        "object_name",
    ]
    .astype(str)
    .tolist()
)


print("\n--- LABEL-FREE CANDIDATES ---")

print(
    "Count:",
    len(candidate_names_627h4br)
)


for name_627h4br in candidate_names_627h4br:

    print(
        " ",
        name_627h4br
    )


if len(candidate_names_627h4br) == 0:

    raise RuntimeError(
        "No label-free in-memory TEST candidate found."
    )


# ============================================================
# 2. Identify columns and structural contract
# ============================================================

candidate_contracts_627h4br = []


for name_627h4br in candidate_names_627h4br:

    df_627h4br = globals()[
        name_627h4br
    ]


    cols_627h4br = [
        str(c)
        for c in df_627h4br.columns
    ]


    lower_map_627h4br = {
        str(c).lower():
        str(c)

        for c in df_627h4br.columns
    }


    # --------------------------------------------
    # key columns
    # --------------------------------------------

    unit_col_627h4br = (
        lower_map_627h4br.get(
            "unit_id"
        )
    )


    cycle_col_627h4br = (
        lower_map_627h4br.get(
            "cycle"
        )
    )


    # --------------------------------------------
    # prediction column
    # --------------------------------------------

    prediction_candidates_627h4br = [
        c
        for c in cols_627h4br

        if (
            "prediction"
            in
            c.lower()
            or
            "predicted_rul"
            in
            c.lower()
            or
            c.lower() == "pred_rul"
        )
    ]


    # --------------------------------------------
    # uncertainty column
    # Prefer mc_std where available.
    # --------------------------------------------

    uncertainty_candidates_627h4br = [
        c
        for c in cols_627h4br

        if (
            "mc_std"
            in
            c.lower()
            or
            "uncertainty"
            in
            c.lower()
        )
    ]


    prediction_col_627h4br = (
        prediction_candidates_627h4br[0]
        if prediction_candidates_627h4br
        else None
    )


    # Prefer exact mc_std if present.
    uncertainty_col_627h4br = None

    for preferred_627h4br in [
        "mc_std",
        "corrupted_mc_std",
        "clean_mc_std",
    ]:

        if preferred_627h4br in lower_map_627h4br:

            uncertainty_col_627h4br = (
                lower_map_627h4br[
                    preferred_627h4br
                ]
            )

            break


    if (
        uncertainty_col_627h4br is None
        and
        uncertainty_candidates_627h4br
    ):

        uncertainty_col_627h4br = (
            uncertainty_candidates_627h4br[0]
        )


    # --------------------------------------------
    # Structural checks
    # --------------------------------------------

    if unit_col_627h4br is not None:

        unique_units_627h4br = int(
            df_627h4br[
                unit_col_627h4br
            ].nunique()
        )

    else:

        unique_units_627h4br = np.nan


    if (
        unit_col_627h4br is not None
        and
        cycle_col_627h4br is not None
    ):

        duplicate_keys_627h4br = int(
            df_627h4br.duplicated(
                [
                    unit_col_627h4br,
                    cycle_col_627h4br,
                ]
            ).sum()
        )

    else:

        duplicate_keys_627h4br = np.nan


    prediction_finite_627h4br = False

    if prediction_col_627h4br is not None:

        p_627h4br = pd.to_numeric(
            df_627h4br[
                prediction_col_627h4br
            ],
            errors="coerce",
        ).to_numpy(
            dtype=float
        )

        prediction_finite_627h4br = bool(
            np.isfinite(
                p_627h4br
            ).all()
        )


    uncertainty_finite_627h4br = False

    if uncertainty_col_627h4br is not None:

        u_627h4br = pd.to_numeric(
            df_627h4br[
                uncertainty_col_627h4br
            ],
            errors="coerce",
        ).to_numpy(
            dtype=float
        )

        uncertainty_finite_627h4br = bool(
            np.isfinite(
                u_627h4br
            ).all()
        )


    candidate_contracts_627h4br.append(
        {
            "object_name":
                name_627h4br,

            "shape":
                str(
                    df_627h4br.shape
                ),

            "unit_col":
                unit_col_627h4br,

            "cycle_col":
                cycle_col_627h4br,

            "prediction_col":
                prediction_col_627h4br,

            "uncertainty_col":
                uncertainty_col_627h4br,

            "unique_units":
                unique_units_627h4br,

            "duplicate_keys":
                duplicate_keys_627h4br,

            "prediction_finite":
                prediction_finite_627h4br,

            "uncertainty_finite":
                uncertainty_finite_627h4br,
        }
    )


TEST_INFERENCE_CONTRACTS_627H4BR = pd.DataFrame(
    candidate_contracts_627h4br
)


print(
    "\n--- CANDIDATE CONTRACTS ---"
)

print(
    TEST_INFERENCE_CONTRACTS_627H4BR
    .to_string(
        index=False
    )
)


# ============================================================
# 3. Show exact columns for each candidate
# ============================================================

print(
    "\n--- EXACT COLUMN LISTS ---"
)


for name_627h4br in candidate_names_627h4br:

    df_627h4br = globals()[
        name_627h4br
    ]

    print(
        "\n",
        name_627h4br,
        sep=""
    )

    print(
        list(
            df_627h4br.columns
        )
    )


# ============================================================
# 4. Choose provisional comparison reference
#
# Prefer explicitly named official endpoint inference.
# This is NOT yet declaring it canonical.
# ============================================================

preferred_names_627h4br = [
    "OFFICIAL_TEST_ENDPOINT_INFERENCE_624E",
    "inference_624fd",
    "pred_df_624fc",
]


REFERENCE_OBJECT_627H4BR = None


for preferred_627h4br in preferred_names_627h4br:

    if (
        preferred_627h4br
        in
        candidate_names_627h4br
    ):

        REFERENCE_OBJECT_627H4BR = (
            preferred_627h4br
        )

        break


if REFERENCE_OBJECT_627H4BR is None:

    REFERENCE_OBJECT_627H4BR = (
        candidate_names_627h4br[0]
    )


print(
    "\n--- PROVISIONAL COMPARISON REFERENCE ---"
)

print(
    REFERENCE_OBJECT_627H4BR
)


# ============================================================
# 5. Compare candidates with provisional reference
#
# Compare:
#   - key rows/order where available
#   - prediction values
#   - uncertainty values
#
# No official RUL values involved.
# ============================================================

reference_contract_627h4br = (
    TEST_INFERENCE_CONTRACTS_627H4BR.loc[
        TEST_INFERENCE_CONTRACTS_627H4BR[
            "object_name"
        ]
        ==
        REFERENCE_OBJECT_627H4BR
    ]
    .iloc[0]
)


reference_df_627h4br = globals()[
    REFERENCE_OBJECT_627H4BR
]


reference_unit_col_627h4br = (
    reference_contract_627h4br[
        "unit_col"
    ]
)


reference_cycle_col_627h4br = (
    reference_contract_627h4br[
        "cycle_col"
    ]
)


reference_prediction_col_627h4br = (
    reference_contract_627h4br[
        "prediction_col"
    ]
)


reference_uncertainty_col_627h4br = (
    reference_contract_627h4br[
        "uncertainty_col"
    ]
)


comparison_records_627h4br = []


for name_627h4br in candidate_names_627h4br:

    row_627h4br = (
        TEST_INFERENCE_CONTRACTS_627H4BR.loc[
            TEST_INFERENCE_CONTRACTS_627H4BR[
                "object_name"
            ]
            ==
            name_627h4br
        ]
        .iloc[0]
    )


    df_627h4br = globals()[
        name_627h4br
    ]


    same_rows_627h4br = bool(
        len(
            df_627h4br
        )
        ==
        len(
            reference_df_627h4br
        )
    )


    # --------------------------------------------
    # key order
    # --------------------------------------------

    key_order_exact_627h4br = None


    if (
        same_rows_627h4br
        and
        reference_unit_col_627h4br
        is not None
        and
        reference_cycle_col_627h4br
        is not None
        and
        row_627h4br[
            "unit_col"
        ]
        is not None
        and
        row_627h4br[
            "cycle_col"
        ]
        is not None
    ):

        ref_keys_627h4br = (
            reference_df_627h4br[
                [
                    reference_unit_col_627h4br,
                    reference_cycle_col_627h4br,
                ]
            ]
            .to_numpy()
        )


        cand_keys_627h4br = (
            df_627h4br[
                [
                    row_627h4br[
                        "unit_col"
                    ],
                    row_627h4br[
                        "cycle_col"
                    ],
                ]
            ]
            .to_numpy()
        )


        key_order_exact_627h4br = bool(
            np.array_equal(
                ref_keys_627h4br,
                cand_keys_627h4br,
            )
        )


    # --------------------------------------------
    # prediction equality
    # --------------------------------------------

    prediction_max_error_627h4br = np.nan


    if (
        same_rows_627h4br
        and
        reference_prediction_col_627h4br
        is not None
        and
        row_627h4br[
            "prediction_col"
        ]
        is not None
    ):

        ref_pred_627h4br = pd.to_numeric(
            reference_df_627h4br[
                reference_prediction_col_627h4br
            ],
            errors="coerce",
        ).to_numpy(
            dtype=float
        )


        cand_pred_627h4br = pd.to_numeric(
            df_627h4br[
                row_627h4br[
                    "prediction_col"
                ]
            ],
            errors="coerce",
        ).to_numpy(
            dtype=float
        )


        prediction_max_error_627h4br = float(
            np.max(
                np.abs(
                    ref_pred_627h4br
                    -
                    cand_pred_627h4br
                )
            )
        )


    # --------------------------------------------
    # uncertainty equality
    # --------------------------------------------

    uncertainty_max_error_627h4br = np.nan


    if (
        same_rows_627h4br
        and
        reference_uncertainty_col_627h4br
        is not None
        and
        row_627h4br[
            "uncertainty_col"
        ]
        is not None
    ):

        ref_unc_627h4br = pd.to_numeric(
            reference_df_627h4br[
                reference_uncertainty_col_627h4br
            ],
            errors="coerce",
        ).to_numpy(
            dtype=float
        )


        cand_unc_627h4br = pd.to_numeric(
            df_627h4br[
                row_627h4br[
                    "uncertainty_col"
                ]
            ],
            errors="coerce",
        ).to_numpy(
            dtype=float
        )


        uncertainty_max_error_627h4br = float(
            np.max(
                np.abs(
                    ref_unc_627h4br
                    -
                    cand_unc_627h4br
                )
            )
        )


    comparison_records_627h4br.append(
        {
            "object_name":
                name_627h4br,

            "same_rows":
                same_rows_627h4br,

            "key_order_exact":
                key_order_exact_627h4br,

            "prediction_max_error":
                prediction_max_error_627h4br,

            "uncertainty_max_error":
                uncertainty_max_error_627h4br,
        }
    )


TEST_INFERENCE_EQUIVALENCE_627H4BR = pd.DataFrame(
    comparison_records_627h4br
)


print(
    "\n--- EQUIVALENCE TO PROVISIONAL REFERENCE ---"
)

print(
    TEST_INFERENCE_EQUIVALENCE_627H4BR
    .to_string(
        index=False
    )
)


# ============================================================
# 6. Check 100-engine endpoint contract
# ============================================================

reference_unit_count_627h4br = np.nan
reference_duplicate_keys_627h4br = np.nan


if (
    reference_unit_col_627h4br
    is not None
):

    reference_unit_count_627h4br = int(
        reference_df_627h4br[
            reference_unit_col_627h4br
        ].nunique()
    )


if (
    reference_unit_col_627h4br
    is not None
    and
    reference_cycle_col_627h4br
    is not None
):

    reference_duplicate_keys_627h4br = int(
        reference_df_627h4br.duplicated(
            [
                reference_unit_col_627h4br,
                reference_cycle_col_627h4br,
            ]
        ).sum()
    )


print(
    "\n--- REFERENCE ENDPOINT CONTRACT ---"
)

print(
    "Rows             :",
    len(
        reference_df_627h4br
    )
)

print(
    "Unique engines   :",
    reference_unit_count_627h4br
)

print(
    "Duplicate keys   :",
    reference_duplicate_keys_627h4br
)

print(
    "Prediction column:",
    reference_prediction_col_627h4br
)

print(
    "Uncertainty col. :",
    reference_uncertainty_col_627h4br
)


# ============================================================
# 7. Final state
# ============================================================

print(
    "\n"
    "============================================================\n"
    "STEP 6.27H4-B-R SUMMARY\n"
    "============================================================"
)

print(
    "Comparison reference :",
    REFERENCE_OBJECT_627H4BR
)

print(
    "Official RUL labels read : False"
)

print(
    "Official TEST scored     : False"
)

print(
    "Model inference rerun    : False"
)

print(
    "MC-dropout rerun         : False"
)

print(
    "Gate changed             : False"
)

print(
    "Threshold changed        : False"
)

print(
    "Files written            : False"
)

print(
    "\n=== END STEP 6.27H4-B-R ==="
)

In [ ]:
# ============================================================
# STEP 6.27H4-B-R2
# COMPACT EXACT TEST-INFERENCE EQUIVALENCE CLOSURE
#
# NO RUL labels.
# NO TEST scoring.
# NO model rerun.
# NO MC-dropout rerun.
# NO files written.
# ============================================================

import numpy as np
import pandas as pd


print(
    "\n"
    "============================================================\n"
    "STEP 6.27H4-B-R2 — EXACT INFERENCE EQUIVALENCE CLOSURE\n"
    "============================================================"
)


# ============================================================
# 1. Required H4-B-R objects
# ============================================================

required_627h4br2 = [
    "REFERENCE_OBJECT_627H4BR",
    "TEST_INFERENCE_CONTRACTS_627H4BR",
    "TEST_INFERENCE_EQUIVALENCE_627H4BR",
]


print("\n--- REQUIRED OBJECTS ---")

for name in required_627h4br2:

    exists = name in globals()

    print(
        f"{name:<42}: {exists}"
    )

    if not exists:

        raise RuntimeError(
            f"Missing H4-B-R object: {name}"
        )


# ============================================================
# 2. Provisional reference contract
# ============================================================

reference_contract_627h4br2 = (
    TEST_INFERENCE_CONTRACTS_627H4BR.loc[
        TEST_INFERENCE_CONTRACTS_627H4BR[
            "object_name"
        ]
        ==
        REFERENCE_OBJECT_627H4BR
    ]
    .iloc[0]
)


print("\n--- PROVISIONAL REFERENCE ---")

print(
    "Object             :",
    REFERENCE_OBJECT_627H4BR
)

print(
    "Shape              :",
    reference_contract_627h4br2[
        "shape"
    ]
)

print(
    "Unique engines     :",
    reference_contract_627h4br2[
        "unique_units"
    ]
)

print(
    "Duplicate keys     :",
    reference_contract_627h4br2[
        "duplicate_keys"
    ]
)

print(
    "Prediction column  :",
    reference_contract_627h4br2[
        "prediction_col"
    ]
)

print(
    "Uncertainty column :",
    reference_contract_627h4br2[
        "uncertainty_col"
    ]
)

print(
    "Prediction finite  :",
    reference_contract_627h4br2[
        "prediction_finite"
    ]
)

print(
    "Uncertainty finite :",
    reference_contract_627h4br2[
        "uncertainty_finite"
    ]
)


# ============================================================
# 3. Exact-equivalence classification
# ============================================================

equiv_627h4br2 = (
    TEST_INFERENCE_EQUIVALENCE_627H4BR
    .copy()
)


def _exact_core_match_627h4br2(row):

    if not bool(
        row["same_rows"]
    ):
        return False

    if row["key_order_exact"] is not True:

        return False

    pred_error = row[
        "prediction_max_error"
    ]

    unc_error = row[
        "uncertainty_max_error"
    ]

    if not np.isfinite(
        pred_error
    ):

        return False

    if not np.isfinite(
        unc_error
    ):

        return False

    return bool(
        np.isclose(
            pred_error,
            0.0,
            atol=1e-12,
            rtol=0.0,
        )
        and
        np.isclose(
            unc_error,
            0.0,
            atol=1e-12,
            rtol=0.0,
        )
    )


equiv_627h4br2[
    "exact_core_match"
] = equiv_627h4br2.apply(
    _exact_core_match_627h4br2,
    axis=1,
)


# ============================================================
# 4. Compact exact comparison
# ============================================================

print("\n--- EXACT COMPARISON TO REFERENCE ---")

print(
    equiv_627h4br2[
        [
            "object_name",
            "same_rows",
            "key_order_exact",
            "prediction_max_error",
            "uncertainty_max_error",
            "exact_core_match",
        ]
    ]
    .to_string(
        index=False
    )
)


# ============================================================
# 5. Exact-match inventory
# ============================================================

exact_matches_627h4br2 = (
    equiv_627h4br2.loc[
        equiv_627h4br2[
            "exact_core_match"
        ],
        "object_name",
    ]
    .astype(str)
    .tolist()
)


nonmatches_627h4br2 = (
    equiv_627h4br2.loc[
        ~equiv_627h4br2[
            "exact_core_match"
        ],
        "object_name",
    ]
    .astype(str)
    .tolist()
)


print("\n--- EXACT MATCHES ---")

print(
    "Count:",
    len(
        exact_matches_627h4br2
    )
)

for name in exact_matches_627h4br2:

    print(
        " ",
        name
    )


print("\n--- NON-EXACT / UNRESOLVED OBJECTS ---")

print(
    "Count:",
    len(
        nonmatches_627h4br2
    )
)

for name in nonmatches_627h4br2:

    print(
        " ",
        name
    )


# ============================================================
# 6. Reference validity
# ============================================================

REFERENCE_TEST_INFERENCE_VALID_627H4BR2 = bool(

    str(
        reference_contract_627h4br2[
            "shape"
        ]
    )
    ==
    "(100, 7)"

    and

    int(
        reference_contract_627h4br2[
            "unique_units"
        ]
    )
    ==
    100

    and

    int(
        reference_contract_627h4br2[
            "duplicate_keys"
        ]
    )
    ==
    0

    and

    bool(
        reference_contract_627h4br2[
            "prediction_finite"
        ]
    )

    and

    bool(
        reference_contract_627h4br2[
            "uncertainty_finite"
        ]
    )
)


print(
    "\n"
    "============================================================\n"
    "STEP 6.27H4-B-R2 SUMMARY\n"
    "============================================================"
)

print(
    "Reference object valid       :",
    REFERENCE_TEST_INFERENCE_VALID_627H4BR2
)

print(
    "Reference object             :",
    REFERENCE_OBJECT_627H4BR
)

print(
    "Exact equivalent objects     :",
    len(
        exact_matches_627h4br2
    )
)

print(
    "Non-exact/unresolved objects :",
    len(
        nonmatches_627h4br2
    )
)

print(
    "Official RUL labels read     : False"
)

print(
    "Official TEST scored         : False"
)

print(
    "Model inference rerun        : False"
)

print(
    "MC-dropout rerun             : False"
)

print(
    "Gate changed                 : False"
)

print(
    "Threshold changed            : False"
)

print(
    "Files written                : False"
)

print(
    "\n=== END STEP 6.27H4-B-R2 ==="
)

In [ ]:
# ============================================================
# STEP 6.27H4-C
# EXACT FROZEN UNCERTAINTY-CALIBRATION CONTRACT AUDIT
#
# Purpose:
#   Reproduce the already-frozen G2E/G3 uncertainty-risk
#   transformation exactly from DEV evidence.
#
# NO TEST labels.
# NO TEST scoring.
# NO model rerun.
# NO threshold change.
# NO files written.
# ============================================================

import numpy as np
import pandas as pd


print(
    "\n"
    "============================================================\n"
    "STEP 6.27H4-C — FROZEN UNCERTAINTY CALIBRATION AUDIT\n"
    "============================================================"
)


# ============================================================
# 1. Required frozen DEV objects
# ============================================================

required_627h4c = [
    "calibrated_627g2e",
    "RELIABILITY_UNCERTAINTY_REFERENCE_627G3",
    "FINAL_RELIABILITY_THRESHOLD_627H3B",
    "FINAL_SELECTIVE_GATE_627H3B",
]


print("\n--- REQUIRED OBJECTS ---")

for name in required_627h4c:

    exists = name in globals()

    print(
        f"{name:<46}: {exists}"
    )

    if not exists:

        raise RuntimeError(
            f"Missing required frozen object: {name}"
        )


# ============================================================
# 2. Required calibrated DEV columns
# ============================================================

clean_raw_col_627h4c = (
    "clean_adjusted_uncertainty"
)

corrupted_raw_col_627h4c = (
    "corrupted_adjusted_uncertainty"
)

clean_risk_col_627h4c = (
    "_clean_ecdf_uncertainty_only"
)

corrupted_risk_col_627h4c = (
    "_corrupted_ecdf_uncertainty_only"
)


required_cols_627h4c = [
    clean_raw_col_627h4c,
    corrupted_raw_col_627h4c,
    clean_risk_col_627h4c,
    corrupted_risk_col_627h4c,
]


print("\n--- REQUIRED DEV COLUMNS ---")

for col in required_cols_627h4c:

    exists = (
        col
        in
        calibrated_627g2e.columns
    )

    print(
        f"{col:<40}: {exists}"
    )

    if not exists:

        raise RuntimeError(
            f"Missing calibrated DEV column: {col}"
        )


# ============================================================
# 3. Frozen uncertainty reference
# ============================================================

reference_627h4c = np.asarray(
    RELIABILITY_UNCERTAINTY_REFERENCE_627G3,
    dtype=float,
).reshape(-1)


if len(reference_627h4c) != 20:

    raise RuntimeError(
        "Expected exactly 20 frozen DEV calibration endpoints."
    )


if not np.isfinite(
    reference_627h4c
).all():

    raise RuntimeError(
        "Frozen uncertainty reference contains "
        "non-finite values."
    )


print("\n--- FROZEN REFERENCE ---")

print(
    "Reference length :",
    len(reference_627h4c)
)

print(
    "Minimum          :",
    float(
        reference_627h4c.min()
    )
)

print(
    "Median           :",
    float(
        np.median(
            reference_627h4c
        )
    )
)

print(
    "Maximum          :",
    float(
        reference_627h4c.max()
    )
)


# ============================================================
# 4. Candidate empirical percentile definitions
# ============================================================

def _ecdf_left_627h4c(values, reference):

    values = np.asarray(
        values,
        dtype=float
    ).reshape(-1)

    return np.asarray(
        [
            np.mean(
                reference < x
            )
            for x in values
        ],
        dtype=float,
    )


def _ecdf_right_627h4c(values, reference):

    values = np.asarray(
        values,
        dtype=float
    ).reshape(-1)

    return np.asarray(
        [
            np.mean(
                reference <= x
            )
            for x in values
        ],
        dtype=float,
    )


def _ecdf_midrank_627h4c(values, reference):

    values = np.asarray(
        values,
        dtype=float
    ).reshape(-1)

    result = []

    n = len(
        reference
    )

    for x in values:

        below = np.sum(
            reference < x
        )

        equal = np.sum(
            reference == x
        )

        percentile = (
            below
            +
            0.5 * equal
        ) / n

        result.append(
            percentile
        )

    return np.asarray(
        result,
        dtype=float,
    )


# ============================================================
# 5. Recover frozen DEV evidence
# ============================================================

clean_raw_627h4c = pd.to_numeric(
    calibrated_627g2e[
        clean_raw_col_627h4c
    ],
    errors="coerce",
).to_numpy(
    dtype=float
)


corrupted_raw_627h4c = pd.to_numeric(
    calibrated_627g2e[
        corrupted_raw_col_627h4c
    ],
    errors="coerce",
).to_numpy(
    dtype=float
)


clean_stored_risk_627h4c = pd.to_numeric(
    calibrated_627g2e[
        clean_risk_col_627h4c
    ],
    errors="coerce",
).to_numpy(
    dtype=float
)


corrupted_stored_risk_627h4c = pd.to_numeric(
    calibrated_627g2e[
        corrupted_risk_col_627h4c
    ],
    errors="coerce",
).to_numpy(
    dtype=float
)


for array_name, array_value in [
    (
        "clean raw",
        clean_raw_627h4c
    ),
    (
        "corrupted raw",
        corrupted_raw_627h4c
    ),
    (
        "clean risk",
        clean_stored_risk_627h4c
    ),
    (
        "corrupted risk",
        corrupted_stored_risk_627h4c
    ),
]:

    if not np.isfinite(
        array_value
    ).all():

        raise RuntimeError(
            f"{array_name} contains non-finite values."
        )


# ============================================================
# 6. Reproduce stored G2E risks
# ============================================================

methods_627h4c = {
    "left_ecdf":
        _ecdf_left_627h4c,

    "right_ecdf":
        _ecdf_right_627h4c,

    "midrank_ecdf":
        _ecdf_midrank_627h4c,
}


audit_records_627h4c = []


for method_name, method_fn in methods_627h4c.items():

    reproduced_clean = method_fn(
        clean_raw_627h4c,
        reference_627h4c,
    )

    reproduced_corrupted = method_fn(
        corrupted_raw_627h4c,
        reference_627h4c,
    )


    clean_max_error = float(
        np.max(
            np.abs(
                reproduced_clean
                -
                clean_stored_risk_627h4c
            )
        )
    )


    corrupted_max_error = float(
        np.max(
            np.abs(
                reproduced_corrupted
                -
                corrupted_stored_risk_627h4c
            )
        )
    )


    overall_max_error = max(
        clean_max_error,
        corrupted_max_error,
    )


    audit_records_627h4c.append(
        {
            "method":
                method_name,

            "clean_max_error":
                clean_max_error,

            "corrupted_max_error":
                corrupted_max_error,

            "overall_max_error":
                overall_max_error,

            "exact_within_1e12":
                bool(
                    overall_max_error
                    <=
                    1e-12
                ),
        }
    )


CALIBRATION_METHOD_AUDIT_627H4C = pd.DataFrame(
    audit_records_627h4c
)


print(
    "\n--- CALIBRATION REPRODUCTION ---"
)

print(
    CALIBRATION_METHOD_AUDIT_627H4C
    .to_string(
        index=False
    )
)


# ============================================================
# 7. Require one unique reproducing method
# ============================================================

matching_methods_627h4c = (
    CALIBRATION_METHOD_AUDIT_627H4C.loc[
        CALIBRATION_METHOD_AUDIT_627H4C[
            "exact_within_1e12"
        ],
        "method",
    ]
    .tolist()
)


print(
    "\nMatching methods:",
    matching_methods_627h4c
)


if len(
    matching_methods_627h4c
) != 1:

    raise RuntimeError(
        "Frozen uncertainty calibration was not uniquely "
        "reproduced. Do not continue to TEST risk creation."
    )


FROZEN_UNCERTAINTY_CALIBRATION_METHOD_627H4C = (
    matching_methods_627h4c[0]
)


if (
    FROZEN_UNCERTAINTY_CALIBRATION_METHOD_627H4C
    !=
    "midrank_ecdf"
):

    raise RuntimeError(
        "Recovered method is not the expected frozen "
        "empirical midrank-percentile calibration. "
        "Stop and inspect."
    )


# ============================================================
# 8. Freeze exact transformation function
# ============================================================

FROZEN_UNCERTAINTY_REFERENCE_627H4C = (
    reference_627h4c.copy()
)


def frozen_uncertainty_risk_627h4c(
    raw_uncertainty
):

    values = np.asarray(
        raw_uncertainty,
        dtype=float
    ).reshape(-1)

    result = (
        _ecdf_midrank_627h4c(
            values,
            FROZEN_UNCERTAINTY_REFERENCE_627H4C,
        )
    )

    if not np.isfinite(
        result
    ).all():

        raise RuntimeError(
            "Calibrated uncertainty risk contains "
            "non-finite values."
        )

    if (
        (result < 0.0).any()
        or
        (result > 1.0).any()
    ):

        raise RuntimeError(
            "Calibrated uncertainty risk outside [0,1]."
        )

    return result


# ============================================================
# 9. Final self-reproduction check
# ============================================================

final_clean_reproduced_627h4c = (
    frozen_uncertainty_risk_627h4c(
        clean_raw_627h4c
    )
)


final_corrupted_reproduced_627h4c = (
    frozen_uncertainty_risk_627h4c(
        corrupted_raw_627h4c
    )
)


final_clean_error_627h4c = float(
    np.max(
        np.abs(
            final_clean_reproduced_627h4c
            -
            clean_stored_risk_627h4c
        )
    )
)


final_corrupted_error_627h4c = float(
    np.max(
        np.abs(
            final_corrupted_reproduced_627h4c
            -
            corrupted_stored_risk_627h4c
        )
    )
)


# ============================================================
# 10. Closure
# ============================================================

print(
    "\n"
    "============================================================\n"
    "STEP 6.27H4-C CALIBRATION CONTRACT SUMMARY\n"
    "============================================================"
)

print(
    "Recovered calibration method :",
    FROZEN_UNCERTAINTY_CALIBRATION_METHOD_627H4C
)

print(
    "Frozen reference length       :",
    len(
        FROZEN_UNCERTAINTY_REFERENCE_627H4C
    )
)

print(
    "Clean DEV reproduction error  :",
    final_clean_error_627h4c
)

print(
    "Corrupt DEV reproduction error:",
    final_corrupted_error_627h4c
)

print(
    "Frozen threshold              :",
    FINAL_RELIABILITY_THRESHOLD_627H3B
)

print(
    "Official TEST labels read     : False"
)

print(
    "Official TEST scored          : False"
)

print(
    "TEST risk generated           : False"
)

print(
    "Model / MC rerun              : False"
)

print(
    "Gate changed                  : False"
)

print(
    "Threshold changed             : False"
)

print(
    "Files written                 : False"
)

print(
    "\n=== END STEP 6.27H4-C ==="
)

In [ ]:
# ============================================================
# STEP 6.27H4-C-R
# FROZEN UNCERTAINTY CALIBRATION DISCREPANCY DIAGNOSTIC
#
# PURPOSE:
#   Determine WHY H4-C did not reproduce the frozen G2E risk.
#
# NO TEST labels.
# NO TEST scoring.
# NO TEST risk generation.
# NO model / MC rerun.
# NO gate or threshold change.
# NO files written.
# ============================================================

import numpy as np
import pandas as pd
import inspect


print(
    "\n"
    "============================================================\n"
    "STEP 6.27H4-C-R — CALIBRATION DISCREPANCY DIAGNOSTIC\n"
    "============================================================"
)


# ============================================================
# 1. Required frozen objects
# ============================================================

required_627h4cr = [
    "calibrated_627g2e",
    "RELIABILITY_UNCERTAINTY_REFERENCE_627G3",
]


print("\n--- REQUIRED OBJECTS ---")

for name in required_627h4cr:

    exists = name in globals()

    print(f"{name:<46}: {exists}")

    if not exists:
        raise RuntimeError(
            f"Missing frozen object: {name}"
        )


work_627h4cr = (
    calibrated_627g2e.copy()
)


reference_627h4cr = np.asarray(
    RELIABILITY_UNCERTAINTY_REFERENCE_627G3,
    dtype=float,
).reshape(-1)


if len(reference_627h4cr) != 20:
    raise RuntimeError(
        "Frozen uncertainty reference length is not 20."
    )


# ============================================================
# 2. Inventory uncertainty-related columns
# ============================================================

uncertainty_cols_627h4cr = [
    c
    for c in work_627h4cr.columns
    if "uncert" in str(c).lower()
]


print("\n--- UNCERTAINTY-RELATED G2E COLUMNS ---")

for c in uncertainty_cols_627h4cr:

    series_627h4cr = pd.to_numeric(
        work_627h4cr[c],
        errors="coerce",
    )

    finite_count_627h4cr = int(
        np.isfinite(
            series_627h4cr.to_numpy(
                dtype=float
            )
        ).sum()
    )

    print(
        f"{c:<45} "
        f"finite={finite_count_627h4cr:<5} "
        f"nunique={series_627h4cr.nunique(dropna=True)}"
    )


# ============================================================
# 3. Stored target risk columns
# ============================================================

clean_target_col_627h4cr = (
    "_clean_ecdf_uncertainty_only"
)

corrupt_target_col_627h4cr = (
    "_corrupted_ecdf_uncertainty_only"
)


for c in [
    clean_target_col_627h4cr,
    corrupt_target_col_627h4cr,
]:

    if c not in work_627h4cr.columns:
        raise RuntimeError(
            f"Missing stored frozen risk column: {c}"
        )


clean_target_627h4cr = pd.to_numeric(
    work_627h4cr[
        clean_target_col_627h4cr
    ],
    errors="coerce",
).to_numpy(
    dtype=float
)


corrupt_target_627h4cr = pd.to_numeric(
    work_627h4cr[
        corrupt_target_col_627h4cr
    ],
    errors="coerce",
).to_numpy(
    dtype=float
)


# ============================================================
# 4. Candidate ECDF implementations
# ============================================================

def exact_left_627h4cr(values, reference):

    values = np.asarray(
        values,
        dtype=float
    ).reshape(-1)

    return np.asarray(
        [
            np.mean(reference < x)
            for x in values
        ],
        dtype=float,
    )


def exact_right_627h4cr(values, reference):

    values = np.asarray(
        values,
        dtype=float
    ).reshape(-1)

    return np.asarray(
        [
            np.mean(reference <= x)
            for x in values
        ],
        dtype=float,
    )


def exact_midrank_627h4cr(values, reference):

    values = np.asarray(
        values,
        dtype=float
    ).reshape(-1)

    n = len(reference)

    result = []

    for x in values:

        below = np.sum(
            reference < x
        )

        equal = np.sum(
            reference == x
        )

        result.append(
            (
                below
                +
                0.5 * equal
            )
            /
            n
        )

    return np.asarray(
        result,
        dtype=float,
    )


def tolerant_midrank_627h4cr(
    values,
    reference,
    atol,
):

    values = np.asarray(
        values,
        dtype=float
    ).reshape(-1)

    reference = np.asarray(
        reference,
        dtype=float
    ).reshape(-1)

    n = len(reference)

    result = []

    for x in values:

        equal_mask = np.isclose(
            reference,
            x,
            atol=atol,
            rtol=0.0,
        )

        below_mask = (
            (reference < x)
            &
            (~equal_mask)
        )

        below = int(
            below_mask.sum()
        )

        equal = int(
            equal_mask.sum()
        )

        result.append(
            (
                below
                +
                0.5 * equal
            )
            /
            n
        )

    return np.asarray(
        result,
        dtype=float,
    )


# ============================================================
# 5. Candidate raw-source columns
#
# We do not assume adjusted_uncertainty is the source.
# ============================================================

clean_source_cols_627h4cr = [
    c
    for c in uncertainty_cols_627h4cr
    if (
        "clean" in str(c).lower()
        and
        "ecdf" not in str(c).lower()
        and
        "combined" not in str(c).lower()
    )
]


corrupt_source_cols_627h4cr = [
    c
    for c in uncertainty_cols_627h4cr
    if (
        (
            "corrupt" in str(c).lower()
            or
            "corrupted" in str(c).lower()
        )
        and
        "ecdf" not in str(c).lower()
        and
        "combined" not in str(c).lower()
    )
]


print("\n--- CLEAN SOURCE CANDIDATES ---")
print(clean_source_cols_627h4cr)

print("\n--- CORRUPTED SOURCE CANDIDATES ---")
print(corrupt_source_cols_627h4cr)


# ============================================================
# 6. Evaluate plausible source + calibration combinations
# ============================================================

methods_627h4cr = {
    "exact_left":
        lambda x: exact_left_627h4cr(
            x,
            reference_627h4cr,
        ),

    "exact_right":
        lambda x: exact_right_627h4cr(
            x,
            reference_627h4cr,
        ),

    "exact_midrank":
        lambda x: exact_midrank_627h4cr(
            x,
            reference_627h4cr,
        ),

    "tol_midrank_1e-14":
        lambda x: tolerant_midrank_627h4cr(
            x,
            reference_627h4cr,
            1e-14,
        ),

    "tol_midrank_1e-12":
        lambda x: tolerant_midrank_627h4cr(
            x,
            reference_627h4cr,
            1e-12,
        ),

    "tol_midrank_1e-10":
        lambda x: tolerant_midrank_627h4cr(
            x,
            reference_627h4cr,
            1e-10,
        ),

    "tol_midrank_1e-08":
        lambda x: tolerant_midrank_627h4cr(
            x,
            reference_627h4cr,
            1e-8,
        ),
}


diagnostic_records_627h4cr = []


for clean_col in clean_source_cols_627h4cr:

    clean_values = pd.to_numeric(
        work_627h4cr[
            clean_col
        ],
        errors="coerce",
    ).to_numpy(
        dtype=float
    )

    if not np.isfinite(
        clean_values
    ).all():
        continue


    for corrupt_col in corrupt_source_cols_627h4cr:

        corrupt_values = pd.to_numeric(
            work_627h4cr[
                corrupt_col
            ],
            errors="coerce",
        ).to_numpy(
            dtype=float
        )

        if not np.isfinite(
            corrupt_values
        ).all():
            continue


        for method_name, method_fn in methods_627h4cr.items():

            clean_reproduced = method_fn(
                clean_values
            )

            corrupt_reproduced = method_fn(
                corrupt_values
            )


            clean_error = float(
                np.max(
                    np.abs(
                        clean_reproduced
                        -
                        clean_target_627h4cr
                    )
                )
            )


            corrupt_error = float(
                np.max(
                    np.abs(
                        corrupt_reproduced
                        -
                        corrupt_target_627h4cr
                    )
                )
            )


            overall_error = max(
                clean_error,
                corrupt_error,
            )


            diagnostic_records_627h4cr.append(
                {
                    "clean_source":
                        clean_col,

                    "corrupt_source":
                        corrupt_col,

                    "method":
                        method_name,

                    "clean_max_error":
                        clean_error,

                    "corrupt_max_error":
                        corrupt_error,

                    "overall_max_error":
                        overall_error,
                }
            )


CALIBRATION_DIAGNOSTIC_627H4CR = (
    pd.DataFrame(
        diagnostic_records_627h4cr
    )
    .sort_values(
        "overall_max_error"
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n--- BEST SOURCE / CALIBRATION MATCHES ---"
)

print(
    CALIBRATION_DIAGNOSTIC_627H4CR
    .head(20)
    .to_string(
        index=False
    )
)


# ============================================================
# 7. Floating-point tie diagnostic for the originally
#    assumed clean_adjusted_uncertainty source
# ============================================================

print(
    "\n--- FLOATING-POINT TIE DIAGNOSTIC ---"
)


if (
    "clean_adjusted_uncertainty"
    in
    work_627h4cr.columns
):

    clean_adjusted_627h4cr = pd.to_numeric(
        work_627h4cr[
            "clean_adjusted_uncertainty"
        ],
        errors="coerce",
    ).to_numpy(
        dtype=float
    )


    # Clean values repeat across scenarios, so inspect unique
    # values only.
    unique_clean_627h4cr = np.unique(
        clean_adjusted_627h4cr
    )


    nearest_distances_627h4cr = []

    for x in unique_clean_627h4cr:

        nearest_distances_627h4cr.append(
            float(
                np.min(
                    np.abs(
                        reference_627h4cr
                        -
                        x
                    )
                )
            )
        )


    nearest_distances_627h4cr = np.asarray(
        nearest_distances_627h4cr,
        dtype=float,
    )


    print(
        "Unique clean adjusted values :",
        len(
            unique_clean_627h4cr
        )
    )

    print(
        "Reference values             :",
        len(
            reference_627h4cr
        )
    )

    print(
        "Nearest-distance minimum     :",
        float(
            nearest_distances_627h4cr.min()
        )
    )

    print(
        "Nearest-distance median      :",
        float(
            np.median(
                nearest_distances_627h4cr
            )
        )
    )

    print(
        "Nearest-distance maximum     :",
        float(
            nearest_distances_627h4cr.max()
        )
    )

    print(
        "Within 1e-12 of reference    :",
        int(
            (
                nearest_distances_627h4cr
                <=
                1e-12
            ).sum()
        ),
        "/",
        len(
            nearest_distances_627h4cr
        )
    )


# ============================================================
# 8. Existing callable inventory
#
# Search only names/signatures.
# Do not modify or execute them.
# ============================================================

callable_records_627h4cr = []


for name, obj in list(
    globals().items()
):

    if not callable(
        obj
    ):
        continue

    lower_name = str(
        name
    ).lower()

    if not any(
        token in lower_name
        for token in [
            "ecdf",
            "percentile",
            "calibr",
        ]
    ):
        continue

    try:
        signature = str(
            inspect.signature(
                obj
            )
        )
    except Exception:
        signature = "UNKNOWN"

    callable_records_627h4cr.append(
        {
            "object_name":
                name,

            "signature":
                signature,
        }
    )


CALLABLE_CALIBRATION_INVENTORY_627H4CR = (
    pd.DataFrame(
        callable_records_627h4cr
    )
)


print(
    "\n--- EXISTING CALIBRATION-LIKE CALLABLES ---"
)


if CALLABLE_CALIBRATION_INVENTORY_627H4CR.empty:

    print(
        "NONE FOUND"
    )

else:

    print(
        CALLABLE_CALIBRATION_INVENTORY_627H4CR
        .to_string(
            index=False
        )
    )


# ============================================================
# 9. Best diagnostic result
# ============================================================

if CALIBRATION_DIAGNOSTIC_627H4CR.empty:

    raise RuntimeError(
        "No calibration candidates could be evaluated."
    )


best_627h4cr = (
    CALIBRATION_DIAGNOSTIC_627H4CR
    .iloc[0]
)


print(
    "\n"
    "============================================================\n"
    "STEP 6.27H4-C-R SUMMARY\n"
    "============================================================"
)

print(
    "Best clean source    :",
    best_627h4cr[
        "clean_source"
    ]
)

print(
    "Best corrupt source  :",
    best_627h4cr[
        "corrupt_source"
    ]
)

print(
    "Best method          :",
    best_627h4cr[
        "method"
    ]
)

print(
    "Clean max error      :",
    float(
        best_627h4cr[
            "clean_max_error"
        ]
    )
)

print(
    "Corrupt max error    :",
    float(
        best_627h4cr[
            "corrupt_max_error"
        ]
    )
)

print(
    "Overall max error    :",
    float(
        best_627h4cr[
            "overall_max_error"
        ]
    )
)

print(
    "Exact within 1e-12   :",
    bool(
        float(
            best_627h4cr[
                "overall_max_error"
            ]
        )
        <=
        1e-12
    )
)

print(
    "Official TEST labels read : False"
)

print(
    "Official TEST scored     : False"
)

print(
    "TEST risk generated      : False"
)

print(
    "Model / MC rerun         : False"
)

print(
    "Gate changed             : False"
)

print(
    "Threshold changed        : False"
)

print(
    "Files written            : False"
)

print(
    "\n=== END STEP 6.27H4-C-R ==="
)

In [ ]:
# ============================================================
# STEP 6.27H4-C-R2
# COMPACT CALIBRATION DISCREPANCY REVIEW
# ============================================================

import numpy as np
import pandas as pd


print(
    "\n"
    "============================================================\n"
    "STEP 6.27H4-C-R2 — COMPACT CALIBRATION REVIEW\n"
    "============================================================"
)


# ============================================================
# 1. Required diagnostic
# ============================================================

if "CALIBRATION_DIAGNOSTIC_627H4CR" not in globals():

    raise RuntimeError(
        "CALIBRATION_DIAGNOSTIC_627H4CR is missing. "
        "Run H4-C-R first."
    )


diag_627h4cr2 = (
    CALIBRATION_DIAGNOSTIC_627H4CR
    .copy()
    .sort_values(
        "overall_max_error"
    )
    .reset_index(drop=True)
)


# ============================================================
# 2. Best matches only
# ============================================================

print("\n--- TOP 12 CALIBRATION MATCHES ---")

print(
    diag_627h4cr2[
        [
            "clean_source",
            "corrupt_source",
            "method",
            "clean_max_error",
            "corrupt_max_error",
            "overall_max_error",
        ]
    ]
    .head(12)
    .to_string(
        index=False
    )
)


# ============================================================
# 3. Best match
# ============================================================

best_627h4cr2 = (
    diag_627h4cr2.iloc[0]
)


print("\n--- BEST MATCH ---")

print(
    "Clean source   :",
    best_627h4cr2[
        "clean_source"
    ]
)

print(
    "Corrupt source :",
    best_627h4cr2[
        "corrupt_source"
    ]
)

print(
    "Method         :",
    best_627h4cr2[
        "method"
    ]
)

print(
    "Clean error    :",
    float(
        best_627h4cr2[
            "clean_max_error"
        ]
    )
)

print(
    "Corrupt error  :",
    float(
        best_627h4cr2[
            "corrupt_max_error"
        ]
    )
)

print(
    "Overall error  :",
    float(
        best_627h4cr2[
            "overall_max_error"
        ]
    )
)

print(
    "Exact <= 1e-12:",
    bool(
        float(
            best_627h4cr2[
                "overall_max_error"
            ]
        )
        <=
        1e-12
    )
)


# ============================================================
# 4. Specifically inspect the suspected correct source pair
# ============================================================

suspected_627h4cr2 = (
    diag_627h4cr2.loc[
        (
            diag_627h4cr2[
                "clean_source"
            ]
            ==
            "clean_uncertainty_risk"
        )
        &
        (
            diag_627h4cr2[
                "corrupt_source"
            ]
            ==
            "corrupted_uncertainty_risk"
        )
    ]
    .copy()
    .sort_values(
        "overall_max_error"
    )
)


print(
    "\n--- clean_uncertainty_risk / "
    "corrupted_uncertainty_risk ---"
)


if suspected_627h4cr2.empty:

    print(
        "PAIR NOT FOUND"
    )

else:

    print(
        suspected_627h4cr2[
            [
                "method",
                "clean_max_error",
                "corrupt_max_error",
                "overall_max_error",
            ]
        ]
        .to_string(
            index=False
        )
    )


# ============================================================
# 5. Existing calibration-like callables
# ============================================================

print("\n--- EXISTING CALIBRATION-LIKE CALLABLES ---")


if (
    "CALLABLE_CALIBRATION_INVENTORY_627H4CR"
    not in globals()
    or
    CALLABLE_CALIBRATION_INVENTORY_627H4CR.empty
):

    print(
        "NONE FOUND"
    )

else:

    print(
        CALLABLE_CALIBRATION_INVENTORY_627H4CR
        .to_string(
            index=False
        )
    )


# ============================================================
# 6. Direct column fingerprints
# ============================================================

print("\n--- G2E UNCERTAINTY STAGE FINGERPRINTS ---")


for col_627h4cr2 in [
    "clean_adjusted_uncertainty",
    "clean_uncertainty_risk",
    "_clean_ecdf_uncertainty_only",
    "corrupted_adjusted_uncertainty",
    "corrupted_uncertainty_risk",
    "_corrupted_ecdf_uncertainty_only",
]:

    if col_627h4cr2 not in calibrated_627g2e.columns:
        continue

    arr_627h4cr2 = pd.to_numeric(
        calibrated_627g2e[
            col_627h4cr2
        ],
        errors="coerce",
    ).to_numpy(
        dtype=float
    )

    print(
        f"{col_627h4cr2:<38} "
        f"min={np.nanmin(arr_627h4cr2):.12f} "
        f"median={np.nanmedian(arr_627h4cr2):.12f} "
        f"mean={np.nanmean(arr_627h4cr2):.12f} "
        f"max={np.nanmax(arr_627h4cr2):.12f} "
        f"unique={len(np.unique(arr_627h4cr2))}"
    )


# ============================================================
# 7. Closure
# ============================================================

print(
    "\n"
    "============================================================\n"
    "STEP 6.27H4-C-R2 SUMMARY\n"
    "============================================================"
)

print(
    "Best source pair :",
    best_627h4cr2[
        "clean_source"
    ],
    "/",
    best_627h4cr2[
        "corrupt_source"
    ]
)

print(
    "Best method      :",
    best_627h4cr2[
        "method"
    ]
)

print(
    "Best max error   :",
    float(
        best_627h4cr2[
            "overall_max_error"
        ]
    )
)

print(
    "TEST labels read : False"
)

print(
    "TEST scored      : False"
)

print(
    "TEST risk made   : False"
)

print(
    "Gate changed     : False"
)

print(
    "Threshold changed:",
    False
)

print(
    "Files written    : False"
)

print(
    "\n=== END STEP 6.27H4-C-R2 ==="
)

In [ ]:
# ============================================================
# STEP 6.27H4-C3
# EXACT G1 UNCERTAINTY-RISK NORMALIZATION PROVENANCE
#
# Purpose:
#   Recover the exact frozen transformation:
#
#       adjusted_uncertainty
#              ↓
#       uncertainty_risk
#
#   using existing DEV evidence only.
#
# NO TEST labels.
# NO TEST scoring.
# NO TEST risk creation.
# NO model / MC rerun.
# NO gate or threshold change.
# NO files written.
# ============================================================

import numpy as np
import pandas as pd
import inspect


print(
    "\n"
    "============================================================\n"
    "STEP 6.27H4-C3 — G1 UNCERTAINTY NORMALIZATION AUDIT\n"
    "============================================================"
)


# ============================================================
# 1. Required G2E evidence
# ============================================================

if "calibrated_627g2e" not in globals():

    raise RuntimeError(
        "calibrated_627g2e is missing."
    )


required_cols_627h4c3 = [
    "clean_adjusted_uncertainty",
    "corrupted_adjusted_uncertainty",
    "clean_uncertainty_risk",
    "corrupted_uncertainty_risk",
]


print("\n--- REQUIRED STORED DEV COLUMNS ---")

for col_627h4c3 in required_cols_627h4c3:

    exists_627h4c3 = (
        col_627h4c3
        in
        calibrated_627g2e.columns
    )

    print(
        f"{col_627h4c3:<40}: "
        f"{exists_627h4c3}"
    )

    if not exists_627h4c3:

        raise RuntimeError(
            f"Missing column: {col_627h4c3}"
        )


# ============================================================
# 2. Recover stored DEV source / target arrays
# ============================================================

clean_adjusted_627h4c3 = pd.to_numeric(
    calibrated_627g2e[
        "clean_adjusted_uncertainty"
    ],
    errors="coerce",
).to_numpy(
    dtype=float
)


corrupt_adjusted_627h4c3 = pd.to_numeric(
    calibrated_627g2e[
        "corrupted_adjusted_uncertainty"
    ],
    errors="coerce",
).to_numpy(
    dtype=float
)


clean_risk_stored_627h4c3 = pd.to_numeric(
    calibrated_627g2e[
        "clean_uncertainty_risk"
    ],
    errors="coerce",
).to_numpy(
    dtype=float
)


corrupt_risk_stored_627h4c3 = pd.to_numeric(
    calibrated_627g2e[
        "corrupted_uncertainty_risk"
    ],
    errors="coerce",
).to_numpy(
    dtype=float
)


for label_627h4c3, arr_627h4c3 in [
    (
        "clean adjusted uncertainty",
        clean_adjusted_627h4c3,
    ),
    (
        "corrupt adjusted uncertainty",
        corrupt_adjusted_627h4c3,
    ),
    (
        "clean stored risk",
        clean_risk_stored_627h4c3,
    ),
    (
        "corrupt stored risk",
        corrupt_risk_stored_627h4c3,
    ),
]:

    if not np.isfinite(
        arr_627h4c3
    ).all():

        raise RuntimeError(
            f"{label_627h4c3} contains non-finite values."
        )


print("\n--- STORED G1 CONTRACT ---")

print(
    "Rows:",
    len(
        clean_adjusted_627h4c3
    )
)

print(
    "Clean adjusted unique:",
    len(
        np.unique(
            clean_adjusted_627h4c3
        )
    )
)

print(
    "Clean risk unique:",
    len(
        np.unique(
            clean_risk_stored_627h4c3
        )
    )
)


# ============================================================
# 3. Discover possible 3490-value uncertainty references
#
# Metadata/value audit only.
# ============================================================

reference_candidates_627h4c3 = {}


for name_627h4c3, obj_627h4c3 in list(
    globals().items()
):

    lower_name_627h4c3 = str(
        name_627h4c3
    ).lower()


    if "uncert" not in lower_name_627h4c3:

        continue


    if isinstance(
        obj_627h4c3,
        pd.DataFrame,
    ):

        continue


    try:

        arr_627h4c3 = np.asarray(
            obj_627h4c3,
            dtype=float,
        ).reshape(-1)

    except Exception:

        continue


    if len(
        arr_627h4c3
    ) != 3490:

        continue


    if not np.isfinite(
        arr_627h4c3
    ).all():

        continue


    reference_candidates_627h4c3[
        name_627h4c3
    ] = arr_627h4c3.copy()


print(
    "\n--- 3490-VALUE UNCERTAINTY REFERENCE CANDIDATES ---"
)


if len(
    reference_candidates_627h4c3
) == 0:

    print("NONE FOUND")

else:

    for (
        name_627h4c3,
        arr_627h4c3,
    ) in reference_candidates_627h4c3.items():

        print(
            f"{name_627h4c3:<45} "
            f"min={arr_627h4c3.min():.12f} "
            f"median={np.median(arr_627h4c3):.12f} "
            f"mean={arr_627h4c3.mean():.12f} "
            f"max={arr_627h4c3.max():.12f}"
        )


if len(
    reference_candidates_627h4c3
) == 0:

    raise RuntimeError(
        "No 3490-value uncertainty reference exists "
        "in memory. Stop before TEST."
    )


# ============================================================
# 4. Candidate empirical transforms
# ============================================================

def _left_ecdf_627h4c3(
    values,
    reference,
):

    values = np.asarray(
        values,
        dtype=float,
    ).reshape(-1)

    reference = np.asarray(
        reference,
        dtype=float,
    ).reshape(-1)

    reference_sorted = np.sort(
        reference
    )

    return (
        np.searchsorted(
            reference_sorted,
            values,
            side="left",
        )
        /
        len(
            reference_sorted
        )
    ).astype(float)


def _right_ecdf_627h4c3(
    values,
    reference,
):

    values = np.asarray(
        values,
        dtype=float,
    ).reshape(-1)

    reference = np.asarray(
        reference,
        dtype=float,
    ).reshape(-1)

    reference_sorted = np.sort(
        reference
    )

    return (
        np.searchsorted(
            reference_sorted,
            values,
            side="right",
        )
        /
        len(
            reference_sorted
        )
    ).astype(float)


def _midrank_ecdf_627h4c3(
    values,
    reference,
):

    values = np.asarray(
        values,
        dtype=float,
    ).reshape(-1)

    reference = np.asarray(
        reference,
        dtype=float,
    ).reshape(-1)

    reference_sorted = np.sort(
        reference
    )

    left_627h4c3 = np.searchsorted(
        reference_sorted,
        values,
        side="left",
    )

    right_627h4c3 = np.searchsorted(
        reference_sorted,
        values,
        side="right",
    )

    return (
        (
            left_627h4c3
            +
            0.5
            *
            (
                right_627h4c3
                -
                left_627h4c3
            )
        )
        /
        len(
            reference_sorted
        )
    ).astype(float)


# ============================================================
# 5. Evaluate every discovered reference
# ============================================================

stage1_records_627h4c3 = []


methods_627h4c3 = {
    "left_ecdf":
        _left_ecdf_627h4c3,

    "right_ecdf":
        _right_ecdf_627h4c3,

    "midrank_ecdf":
        _midrank_ecdf_627h4c3,
}


for (
    reference_name_627h4c3,
    reference_values_627h4c3,
) in reference_candidates_627h4c3.items():


    for (
        method_name_627h4c3,
        method_fn_627h4c3,
    ) in methods_627h4c3.items():


        clean_reproduced_627h4c3 = (
            method_fn_627h4c3(
                clean_adjusted_627h4c3,
                reference_values_627h4c3,
            )
        )


        corrupt_reproduced_627h4c3 = (
            method_fn_627h4c3(
                corrupt_adjusted_627h4c3,
                reference_values_627h4c3,
            )
        )


        clean_error_627h4c3 = float(
            np.max(
                np.abs(
                    clean_reproduced_627h4c3
                    -
                    clean_risk_stored_627h4c3
                )
            )
        )


        corrupt_error_627h4c3 = float(
            np.max(
                np.abs(
                    corrupt_reproduced_627h4c3
                    -
                    corrupt_risk_stored_627h4c3
                )
            )
        )


        overall_error_627h4c3 = max(
            clean_error_627h4c3,
            corrupt_error_627h4c3,
        )


        stage1_records_627h4c3.append(
            {
                "reference_object":
                    reference_name_627h4c3,

                "method":
                    method_name_627h4c3,

                "clean_max_error":
                    clean_error_627h4c3,

                "corrupt_max_error":
                    corrupt_error_627h4c3,

                "overall_max_error":
                    overall_error_627h4c3,

                "exact_1e12":
                    bool(
                        overall_error_627h4c3
                        <=
                        1e-12
                    ),
            }
        )


G1_NORMALIZATION_AUDIT_627H4C3 = (
    pd.DataFrame(
        stage1_records_627h4c3
    )
    .sort_values(
        "overall_max_error"
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n--- BEST G1 NORMALIZATION MATCHES ---"
)

print(
    G1_NORMALIZATION_AUDIT_627H4C3
    .head(15)
    .to_string(
        index=False
    )
)


# ============================================================
# 6. Exact matches
# ============================================================

exact_matches_627h4c3 = (
    G1_NORMALIZATION_AUDIT_627H4C3.loc[
        G1_NORMALIZATION_AUDIT_627H4C3[
            "exact_1e12"
        ]
    ]
    .copy()
)


print("\n--- EXACT <= 1e-12 MATCHES ---")


if exact_matches_627h4c3.empty:

    print(
        "NONE"
    )

else:

    print(
        exact_matches_627h4c3
        .to_string(
            index=False
        )
    )


# ============================================================
# 7. Calibration-like callable inventory
# ============================================================

callable_records_627h4c3 = []


for name_627h4c3, obj_627h4c3 in list(
    globals().items()
):

    if not callable(
        obj_627h4c3
    ):

        continue


    lname_627h4c3 = str(
        name_627h4c3
    ).lower()


    if not any(
        token_627h4c3
        in
        lname_627h4c3

        for token_627h4c3 in [
            "uncertainty",
            "ecdf",
            "percentile",
            "normaliz",
        ]
    ):

        continue


    try:

        sig_627h4c3 = str(
            inspect.signature(
                obj_627h4c3
            )
        )

    except Exception:

        sig_627h4c3 = "UNKNOWN"


    callable_records_627h4c3.append(
        {
            "object_name":
                name_627h4c3,

            "signature":
                sig_627h4c3,
        }
    )


G1_CALLABLE_INVENTORY_627H4C3 = (
    pd.DataFrame(
        callable_records_627h4c3
    )
)


print(
    "\n--- RELEVANT EXISTING CALLABLES ---"
)


if G1_CALLABLE_INVENTORY_627H4C3.empty:

    print(
        "NONE FOUND"
    )

else:

    print(
        G1_CALLABLE_INVENTORY_627H4C3
        .to_string(
            index=False
        )
    )


# ============================================================
# 8. Best result
# ============================================================

best_627h4c3 = (
    G1_NORMALIZATION_AUDIT_627H4C3
    .iloc[0]
)


print(
    "\n"
    "============================================================\n"
    "STEP 6.27H4-C3 SUMMARY\n"
    "============================================================"
)

print(
    "Best reference :",
    best_627h4c3[
        "reference_object"
    ]
)

print(
    "Best method    :",
    best_627h4c3[
        "method"
    ]
)

print(
    "Clean error    :",
    float(
        best_627h4c3[
            "clean_max_error"
        ]
    )
)

print(
    "Corrupt error  :",
    float(
        best_627h4c3[
            "corrupt_max_error"
        ]
    )
)

print(
    "Overall error  :",
    float(
        best_627h4c3[
            "overall_max_error"
        ]
    )
)

print(
    "Exact <=1e-12 :",
    bool(
        best_627h4c3[
            "exact_1e12"
        ]
    )
)

print(
    "TEST labels read    : False"
)

print(
    "TEST scored         : False"
)

print(
    "TEST risk generated : False"
)

print(
    "Model / MC rerun    : False"
)

print(
    "Gate changed        : False"
)

print(
    "Threshold changed   : False"
)

print(
    "Files written       : False"
)

print(
    "\n=== END STEP 6.27H4-C3 ==="
)

In [ ]:
# ============================================================
# STEP 6.27H4-C3-R
# COMPACT G1 NORMALIZATION CLOSURE
#
# NO TEST labels.
# NO TEST scoring.
# NO TEST risk generation.
# NO model / MC rerun.
# NO rule changes.
# ============================================================

import numpy as np
import pandas as pd


print(
    "\n"
    "============================================================\n"
    "STEP 6.27H4-C3-R — COMPACT G1 NORMALIZATION REVIEW\n"
    "============================================================"
)


# ------------------------------------------------------------
# 1. Required audit object
# ------------------------------------------------------------

if "G1_NORMALIZATION_AUDIT_627H4C3" not in globals():
    raise RuntimeError(
        "G1_NORMALIZATION_AUDIT_627H4C3 is missing. "
        "Run H4-C3 first."
    )


audit_627h4c3r = (
    G1_NORMALIZATION_AUDIT_627H4C3
    .copy()
    .sort_values(
        "overall_max_error"
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 2. Best matches
# ------------------------------------------------------------

print("\n--- TOP G1 NORMALIZATION MATCHES ---")

print(
    audit_627h4c3r[
        [
            "reference_object",
            "method",
            "clean_max_error",
            "corrupt_max_error",
            "overall_max_error",
            "exact_1e12",
        ]
    ]
    .head(15)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 3. Exact matches
# ------------------------------------------------------------

exact_627h4c3r = (
    audit_627h4c3r.loc[
        audit_627h4c3r["exact_1e12"]
    ]
    .copy()
)


print("\n--- EXACT <= 1e-12 MATCHES ---")

if exact_627h4c3r.empty:

    print("NONE")

else:

    print(
        exact_627h4c3r[
            [
                "reference_object",
                "method",
                "clean_max_error",
                "corrupt_max_error",
                "overall_max_error",
            ]
        ]
        .to_string(index=False)
    )


# ------------------------------------------------------------
# 4. Best single result
# ------------------------------------------------------------

best_627h4c3r = (
    audit_627h4c3r.iloc[0]
)


print("\n--- BEST RESULT ---")

print(
    "Reference object :",
    best_627h4c3r[
        "reference_object"
    ]
)

print(
    "Method           :",
    best_627h4c3r[
        "method"
    ]
)

print(
    "Clean max error  :",
    float(
        best_627h4c3r[
            "clean_max_error"
        ]
    )
)

print(
    "Corrupt max error:",
    float(
        best_627h4c3r[
            "corrupt_max_error"
        ]
    )
)

print(
    "Overall max error:",
    float(
        best_627h4c3r[
            "overall_max_error"
        ]
    )
)

print(
    "Exact <= 1e-12   :",
    bool(
        best_627h4c3r[
            "exact_1e12"
        ]
    )
)


# ------------------------------------------------------------
# 5. Relevant existing callables
# ------------------------------------------------------------

print("\n--- EXISTING G1-RELATED CALLABLES ---")

if (
    "G1_CALLABLE_INVENTORY_627H4C3"
    not in globals()
    or
    G1_CALLABLE_INVENTORY_627H4C3.empty
):

    print("NONE")

else:

    print(
        G1_CALLABLE_INVENTORY_627H4C3
        .to_string(index=False)
    )


# ------------------------------------------------------------
# 6. Closure
# ------------------------------------------------------------

print(
    "\n"
    "============================================================\n"
    "STEP 6.27H4-C3-R SUMMARY\n"
    "============================================================"
)

print(
    "Exact G1 reproduction found :",
    bool(
        not exact_627h4c3r.empty
    )
)

print(
    "Best reference              :",
    best_627h4c3r[
        "reference_object"
    ]
)

print(
    "Best method                 :",
    best_627h4c3r[
        "method"
    ]
)

print(
    "Best overall error          :",
    float(
        best_627h4c3r[
            "overall_max_error"
        ]
    )
)

print(
    "Official TEST labels read   : False"
)

print(
    "Official TEST scored        : False"
)

print(
    "TEST risk generated         : False"
)

print(
    "Model / MC rerun            : False"
)

print(
    "Gate changed                : False"
)

print(
    "Threshold changed           : False"
)

print(
    "Files written               : False"
)

print(
    "\n=== END STEP 6.27H4-C3-R ==="
)

In [ ]:
# ============================================================
# STEP 6.27H4-C4
# FULL FROZEN UNCERTAINTY-RISK CHAIN CLOSURE
#
# Proves:
#   adjusted uncertainty
#       -> right ECDF using 3490 DEV reference
#       -> uncertainty risk
#       -> exact midrank using 20 DEV endpoint reference
#       -> final selective risk
#
# Also audits TEST inference columns.
#
# NO TEST labels.
# NO TEST scoring.
# NO TEST risk generation.
# NO model / MC rerun.
# NO rule changes.
# ============================================================

import numpy as np
import pandas as pd


print(
    "\n"
    "============================================================\n"
    "STEP 6.27H4-C4 — FULL UNCERTAINTY CHAIN CLOSURE\n"
    "============================================================"
)


# ============================================================
# 1. Required frozen objects
# ============================================================

required_627h4c4 = [
    "calibrated_627g2e",
    "UNCERTAINTY_RISK_REFERENCE_627G1",
    "RELIABILITY_UNCERTAINTY_REFERENCE_627G3",
    "OFFICIAL_TEST_ENDPOINT_INFERENCE_624E",
    "FINAL_RELIABILITY_THRESHOLD_627H3B",
]


print("\n--- REQUIRED FROZEN OBJECTS ---")

for name in required_627h4c4:

    exists = name in globals()

    print(
        f"{name:<48}: {exists}"
    )

    if not exists:

        raise RuntimeError(
            f"Missing frozen object: {name}"
        )


# ============================================================
# 2. Canonical references
# ============================================================

G1_REFERENCE_627H4C4 = np.asarray(
    UNCERTAINTY_RISK_REFERENCE_627G1,
    dtype=float,
).reshape(-1)


G2_REFERENCE_627H4C4 = np.asarray(
    RELIABILITY_UNCERTAINTY_REFERENCE_627G3,
    dtype=float,
).reshape(-1)


if len(G1_REFERENCE_627H4C4) != 3490:

    raise RuntimeError(
        "G1 uncertainty reference must contain 3490 values."
    )


if len(G2_REFERENCE_627H4C4) != 20:

    raise RuntimeError(
        "G2/G3 endpoint reference must contain 20 values."
    )


if not np.isfinite(
    G1_REFERENCE_627H4C4
).all():

    raise RuntimeError(
        "G1 reference contains non-finite values."
    )


if not np.isfinite(
    G2_REFERENCE_627H4C4
).all():

    raise RuntimeError(
        "G2 reference contains non-finite values."
    )


print("\n--- FROZEN REFERENCE CONTRACT ---")

print(
    "G1 reference length :",
    len(G1_REFERENCE_627H4C4)
)

print(
    "G2 reference length :",
    len(G2_REFERENCE_627H4C4)
)


# ============================================================
# 3. Exact frozen transformations
# ============================================================

def frozen_g1_right_ecdf_627h4c4(values):

    values = np.asarray(
        values,
        dtype=float,
    ).reshape(-1)

    reference_sorted = np.sort(
        G1_REFERENCE_627H4C4
    )

    result = (
        np.searchsorted(
            reference_sorted,
            values,
            side="right",
        )
        /
        len(reference_sorted)
    ).astype(float)

    return result


def frozen_g2_midrank_627h4c4(values):

    values = np.asarray(
        values,
        dtype=float,
    ).reshape(-1)

    reference_sorted = np.sort(
        G2_REFERENCE_627H4C4
    )

    left = np.searchsorted(
        reference_sorted,
        values,
        side="left",
    )

    right = np.searchsorted(
        reference_sorted,
        values,
        side="right",
    )

    result = (
        (
            left
            +
            0.5 * (right - left)
        )
        /
        len(reference_sorted)
    ).astype(float)

    return result


def frozen_full_uncertainty_risk_627h4c4(
    adjusted_uncertainty
):

    stage1 = (
        frozen_g1_right_ecdf_627h4c4(
            adjusted_uncertainty
        )
    )

    stage2 = (
        frozen_g2_midrank_627h4c4(
            stage1
        )
    )

    return stage1, stage2


# ============================================================
# 4. Reproduce stored DEV Stage 1
# ============================================================

dev_clean_adjusted = pd.to_numeric(
    calibrated_627g2e[
        "clean_adjusted_uncertainty"
    ],
    errors="coerce",
).to_numpy(dtype=float)


dev_corrupt_adjusted = pd.to_numeric(
    calibrated_627g2e[
        "corrupted_adjusted_uncertainty"
    ],
    errors="coerce",
).to_numpy(dtype=float)


stored_clean_stage1 = pd.to_numeric(
    calibrated_627g2e[
        "clean_uncertainty_risk"
    ],
    errors="coerce",
).to_numpy(dtype=float)


stored_corrupt_stage1 = pd.to_numeric(
    calibrated_627g2e[
        "corrupted_uncertainty_risk"
    ],
    errors="coerce",
).to_numpy(dtype=float)


reproduced_clean_stage1 = (
    frozen_g1_right_ecdf_627h4c4(
        dev_clean_adjusted
    )
)


reproduced_corrupt_stage1 = (
    frozen_g1_right_ecdf_627h4c4(
        dev_corrupt_adjusted
    )
)


clean_stage1_error = float(
    np.max(
        np.abs(
            reproduced_clean_stage1
            -
            stored_clean_stage1
        )
    )
)


corrupt_stage1_error = float(
    np.max(
        np.abs(
            reproduced_corrupt_stage1
            -
            stored_corrupt_stage1
        )
    )
)


# ============================================================
# 5. Reproduce stored DEV Stage 2
# ============================================================

stored_clean_stage2 = pd.to_numeric(
    calibrated_627g2e[
        "_clean_ecdf_uncertainty_only"
    ],
    errors="coerce",
).to_numpy(dtype=float)


stored_corrupt_stage2 = pd.to_numeric(
    calibrated_627g2e[
        "_corrupted_ecdf_uncertainty_only"
    ],
    errors="coerce",
).to_numpy(dtype=float)


reproduced_clean_stage2 = (
    frozen_g2_midrank_627h4c4(
        reproduced_clean_stage1
    )
)


reproduced_corrupt_stage2 = (
    frozen_g2_midrank_627h4c4(
        reproduced_corrupt_stage1
    )
)


clean_stage2_error = float(
    np.max(
        np.abs(
            reproduced_clean_stage2
            -
            stored_clean_stage2
        )
    )
)


corrupt_stage2_error = float(
    np.max(
        np.abs(
            reproduced_corrupt_stage2
            -
            stored_corrupt_stage2
        )
    )
)


print("\n--- FULL DEV REPRODUCTION ---")

print(
    "Stage-1 clean max error   :",
    clean_stage1_error
)

print(
    "Stage-1 corrupt max error :",
    corrupt_stage1_error
)

print(
    "Stage-2 clean max error   :",
    clean_stage2_error
)

print(
    "Stage-2 corrupt max error :",
    corrupt_stage2_error
)


FULL_CHAIN_EXACT_627H4C4 = bool(
    max(
        clean_stage1_error,
        corrupt_stage1_error,
        clean_stage2_error,
        corrupt_stage2_error,
    )
    <=
    1e-12
)


print(
    "Complete chain exact      :",
    FULL_CHAIN_EXACT_627H4C4
)


if not FULL_CHAIN_EXACT_627H4C4:

    raise RuntimeError(
        "Frozen uncertainty chain was not exactly reproduced."
    )


# ============================================================
# 6. Audit canonical TEST inference columns
# ============================================================

test_df_627h4c4 = (
    OFFICIAL_TEST_ENDPOINT_INFERENCE_624E
)


print("\n--- CANONICAL TEST INFERENCE CONTRACT ---")

print(
    "Shape:",
    test_df_627h4c4.shape
)

print(
    "Columns:"
)

for i, col in enumerate(
    test_df_627h4c4.columns,
    start=1,
):

    print(
        f"{i:02d}. {col}"
    )


# ============================================================
# 7. Locate possible adjusted-uncertainty TEST column
# ============================================================

possible_adjusted_cols_627h4c4 = [
    str(c)

    for c in test_df_627h4c4.columns

    if (
        "adjusted" in str(c).lower()
        and
        "uncert" in str(c).lower()
    )
]


print(
    "\nAdjusted-uncertainty candidates:",
    possible_adjusted_cols_627h4c4
)


TEST_ADJUSTED_UNCERTAINTY_COLUMN_627H4C4 = None


if len(
    possible_adjusted_cols_627h4c4
) == 1:

    TEST_ADJUSTED_UNCERTAINTY_COLUMN_627H4C4 = (
        possible_adjusted_cols_627h4c4[0]
    )


elif len(
    possible_adjusted_cols_627h4c4
) > 1:

    print(
        "\nMultiple adjusted-uncertainty columns exist. "
        "Do not choose automatically."
    )


else:

    print(
        "\nNo adjusted-uncertainty column is stored "
        "in the canonical TEST artifact."
    )


# ============================================================
# 8. If available, numerical audit only
#
# Still DO NOT create TEST risk here.
# ============================================================

if (
    TEST_ADJUSTED_UNCERTAINTY_COLUMN_627H4C4
    is not None
):

    test_adjusted_preview_627h4c4 = (
        pd.to_numeric(
            test_df_627h4c4[
                TEST_ADJUSTED_UNCERTAINTY_COLUMN_627H4C4
            ],
            errors="coerce",
        )
        .to_numpy(dtype=float)
    )


    print(
        "\n--- TEST ADJUSTED-UNCERTAINTY CONTRACT ---"
    )

    print(
        "Column :",
        TEST_ADJUSTED_UNCERTAINTY_COLUMN_627H4C4
    )

    print(
        "Count  :",
        len(test_adjusted_preview_627h4c4)
    )

    print(
        "Finite :",
        bool(
            np.isfinite(
                test_adjusted_preview_627h4c4
            ).all()
        )
    )

    print(
        "Minimum:",
        float(
            np.min(
                test_adjusted_preview_627h4c4
            )
        )
    )

    print(
        "Median :",
        float(
            np.median(
                test_adjusted_preview_627h4c4
            )
        )
    )

    print(
        "Maximum:",
        float(
            np.max(
                test_adjusted_preview_627h4c4
            )
        )
    )


# ============================================================
# 9. Final closure
# ============================================================

FROZEN_UNCERTAINTY_CHAIN_627H4C4 = {
    "stage1_input":
        "adjusted_uncertainty",

    "stage1_reference":
        "UNCERTAINTY_RISK_REFERENCE_627G1",

    "stage1_reference_size":
        3490,

    "stage1_method":
        "right_ecdf",

    "stage2_reference":
        "RELIABILITY_UNCERTAINTY_REFERENCE_627G3",

    "stage2_reference_size":
        20,

    "stage2_method":
        "exact_midrank_ecdf",

    "orientation":
        "higher_is_worse",

    "threshold":
        float(
            FINAL_RELIABILITY_THRESHOLD_627H3B
        ),
}


print(
    "\n"
    "============================================================\n"
    "STEP 6.27H4-C4 SUMMARY\n"
    "============================================================"
)

print(
    "Full DEV chain reproduced :",
    FULL_CHAIN_EXACT_627H4C4
)

print(
    "Stage 1 method            : right_ecdf"
)

print(
    "Stage 1 reference size    : 3490"
)

print(
    "Stage 2 method            : exact_midrank_ecdf"
)

print(
    "Stage 2 reference size    : 20"
)

print(
    "Frozen threshold          :",
    FINAL_RELIABILITY_THRESHOLD_627H3B
)

print(
    "TEST adjusted-U column    :",
    TEST_ADJUSTED_UNCERTAINTY_COLUMN_627H4C4
)

print(
    "Official TEST labels read : False"
)

print(
    "Official TEST scored      : False"
)

print(
    "TEST risk generated       : False"
)

print(
    "Model / MC rerun          : False"
)

print(
    "Gate changed              : False"
)

print(
    "Threshold changed         : False"
)

print(
    "Files written             : False"
)

print(
    "\n=== END STEP 6.27H4-C4 ==="
)

In [ ]:
# ============================================================
# STEP 6.27H4-C4-R
# CANONICAL TEST UNCERTAINTY-INPUT CLOSURE
#
# NO labels.
# NO scoring.
# NO TEST risk generation.
# NO model / MC rerun.
# ============================================================

import numpy as np
import pandas as pd


print(
    "\n"
    "============================================================\n"
    "STEP 6.27H4-C4-R — TEST UNCERTAINTY INPUT CLOSURE\n"
    "============================================================"
)


# ------------------------------------------------------------
# 1. Required canonical object
# ------------------------------------------------------------

if "OFFICIAL_TEST_ENDPOINT_INFERENCE_624E" not in globals():
    raise RuntimeError(
        "OFFICIAL_TEST_ENDPOINT_INFERENCE_624E is missing."
    )


test_627h4c4r = (
    OFFICIAL_TEST_ENDPOINT_INFERENCE_624E
)


print("\n--- CANONICAL TEST TABLE ---")

print(
    "Shape:",
    test_627h4c4r.shape
)

print(
    "Unique engines:",
    test_627h4c4r["unit_id"].nunique()
    if "unit_id" in test_627h4c4r.columns
    else "NO unit_id COLUMN"
)

print(
    "Duplicate unit/cycle keys:",
    test_627h4c4r.duplicated(
        ["unit_id", "cycle"]
    ).sum()
    if {
        "unit_id",
        "cycle",
    }.issubset(
        test_627h4c4r.columns
    )
    else "KEY COLUMNS NOT FOUND"
)


# ------------------------------------------------------------
# 2. Exact seven columns
# ------------------------------------------------------------

print("\n--- EXACT COLUMN CONTRACT ---")

for i, col in enumerate(
    test_627h4c4r.columns,
    start=1,
):
    print(
        f"{i:02d}. {col}"
    )


# ------------------------------------------------------------
# 3. Numerical fingerprints
# ------------------------------------------------------------

print("\n--- NUMERICAL COLUMN FINGERPRINTS ---")

for col in test_627h4c4r.columns:

    numeric = pd.to_numeric(
        test_627h4c4r[col],
        errors="coerce",
    )

    if numeric.notna().all():

        arr = numeric.to_numpy(
            dtype=float
        )

        print(
            f"{str(col):<32}"
            f"min={np.min(arr):.9f}  "
            f"median={np.median(arr):.9f}  "
            f"mean={np.mean(arr):.9f}  "
            f"max={np.max(arr):.9f}"
        )


# ------------------------------------------------------------
# 4. Locate uncertainty representations
# ------------------------------------------------------------

uncertainty_columns_627h4c4r = [
    str(c)
    for c in test_627h4c4r.columns
    if (
        "uncert" in str(c).lower()
        or
        "mc_std" in str(c).lower()
    )
]


adjusted_columns_627h4c4r = [
    str(c)
    for c in test_627h4c4r.columns
    if (
        "adjust" in str(c).lower()
        and
        "uncert" in str(c).lower()
    )
]


print(
    "\nUncertainty-related columns:",
    uncertainty_columns_627h4c4r
)

print(
    "Adjusted-uncertainty columns:",
    adjusted_columns_627h4c4r
)


# ------------------------------------------------------------
# 5. Closure
# ------------------------------------------------------------

TEST_HAS_ADJUSTED_UNCERTAINTY_627H4C4R = bool(
    len(
        adjusted_columns_627h4c4r
    )
    ==
    1
)


TEST_ADJUSTED_UNCERTAINTY_COLUMN_627H4C4R = (
    adjusted_columns_627h4c4r[0]
    if TEST_HAS_ADJUSTED_UNCERTAINTY_627H4C4R
    else None
)


print(
    "\n"
    "============================================================\n"
    "STEP 6.27H4-C4-R SUMMARY\n"
    "============================================================"
)

print(
    "Canonical rows                :",
    len(test_627h4c4r)
)

print(
    "Adjusted uncertainty available:",
    TEST_HAS_ADJUSTED_UNCERTAINTY_627H4C4R
)

print(
    "Adjusted uncertainty column   :",
    TEST_ADJUSTED_UNCERTAINTY_COLUMN_627H4C4R
)

print(
    "Official RUL labels read      : False"
)

print(
    "Official TEST scored          : False"
)

print(
    "TEST risk generated           : False"
)

print(
    "Model / MC rerun              : False"
)

print(
    "Threshold changed             : False"
)

print(
    "Files written                 : False"
)

print(
    "\n=== END STEP 6.27H4-C4-R ==="
)

In [ ]:
# ============================================================
# STEP 6.27H4-C4-R2
# STORED TEST RELIABILITY-SCORE / DECISION PROVENANCE AUDIT
#
# Purpose:
#   Determine whether the existing columns:
#       reliability_score
#       accept
#       decision
#
#   already implement the CURRENT frozen policy.
#
# NO RUL labels.
# NO TEST scoring.
# NO model / MC rerun.
# NO files written.
# ============================================================

import numpy as np
import pandas as pd


print(
    "\n"
    "============================================================\n"
    "STEP 6.27H4-C4-R2 — STORED TEST RELIABILITY PROVENANCE\n"
    "============================================================"
)


# ============================================================
# 1. Required objects
# ============================================================

required_627h4c4r2 = [
    "OFFICIAL_TEST_ENDPOINT_INFERENCE_624E",
    "FINAL_SELECTIVE_GATE_627H3B",
    "FINAL_RELIABILITY_THRESHOLD_627H3B",
    "FINAL_SELECTIVE_ORIENTATION_627H3B",
]


print("\n--- REQUIRED OBJECTS ---")

for name in required_627h4c4r2:

    exists = name in globals()

    print(
        f"{name:<46}: {exists}"
    )

    if not exists:

        raise RuntimeError(
            f"Missing required object: {name}"
        )


df_627h4c4r2 = (
    OFFICIAL_TEST_ENDPOINT_INFERENCE_624E
    .copy()
)


# ============================================================
# 2. Required stored columns
# ============================================================

required_cols_627h4c4r2 = [
    "unit_id",
    "cycle",
    "prediction",
    "mc_std",
    "reliability_score",
    "accept",
    "decision",
]


print("\n--- STORED COLUMN CONTRACT ---")

for col in required_cols_627h4c4r2:

    exists = (
        col
        in
        df_627h4c4r2.columns
    )

    print(
        f"{col:<24}: {exists}"
    )

    if not exists:

        raise RuntimeError(
            f"Missing stored TEST column: {col}"
        )


# ============================================================
# 3. Score numerical contract
# ============================================================

score_627h4c4r2 = pd.to_numeric(
    df_627h4c4r2[
        "reliability_score"
    ],
    errors="coerce",
).to_numpy(
    dtype=float
)


print("\n--- STORED RELIABILITY SCORE ---")

print(
    "Rows          :",
    len(score_627h4c4r2)
)

print(
    "Finite        :",
    bool(
        np.isfinite(
            score_627h4c4r2
        ).all()
    )
)

print(
    "Within [0,1] :",
    bool(
        (
            (score_627h4c4r2 >= 0.0)
            &
            (score_627h4c4r2 <= 1.0)
        ).all()
    )
)

print(
    "Minimum       :",
    float(
        np.min(
            score_627h4c4r2
        )
    )
)

print(
    "Median        :",
    float(
        np.median(
            score_627h4c4r2
        )
    )
)

print(
    "Maximum       :",
    float(
        np.max(
            score_627h4c4r2
        )
    )
)


# ============================================================
# 4. Current frozen H3-B decision implied by stored score
# ============================================================

frozen_threshold_627h4c4r2 = float(
    FINAL_RELIABILITY_THRESHOLD_627H3B
)


expected_accept_627h4c4r2 = (
    score_627h4c4r2
    <=
    frozen_threshold_627h4c4r2
)


print(
    "\n--- CURRENT FROZEN POLICY APPLIED TO STORED SCORE ---"
)

print(
    "Gate        :",
    FINAL_SELECTIVE_GATE_627H3B
)

print(
    "Orientation :",
    FINAL_SELECTIVE_ORIENTATION_627H3B
)

print(
    "Threshold   :",
    frozen_threshold_627h4c4r2
)

print(
    "Would accept:",
    int(
        expected_accept_627h4c4r2.sum()
    )
)

print(
    "Would abstain:",
    int(
        (
            ~expected_accept_627h4c4r2
        ).sum()
    )
)

print(
    "Coverage:",
    float(
        expected_accept_627h4c4r2.mean()
    )
)


# ============================================================
# 5. Recover stored ACCEPT representation
# ============================================================

stored_accept_raw_627h4c4r2 = (
    df_627h4c4r2[
        "accept"
    ]
)


print("\n--- STORED ACCEPT COLUMN ---")

print(
    "dtype:",
    stored_accept_raw_627h4c4r2.dtype
)

print(
    "Unique values:",
    list(
        pd.unique(
            stored_accept_raw_627h4c4r2
        )
    )
)


def normalize_accept_627h4c4r2(series):

    if pd.api.types.is_bool_dtype(
        series
    ):

        return series.to_numpy(
            dtype=bool
        )


    numeric = pd.to_numeric(
        series,
        errors="coerce",
    )


    if numeric.notna().all():

        values = numeric.to_numpy()

        if set(
            np.unique(
                values
            )
        ).issubset(
            {
                0,
                1,
            }
        ):

            return values.astype(
                bool
            )


    text = (
        series
        .astype(str)
        .str.strip()
        .str.lower()
    )


    true_tokens = {
        "true",
        "1",
        "yes",
        "accept",
        "accepted",
    }

    false_tokens = {
        "false",
        "0",
        "no",
        "abstain",
        "abstained",
        "reject",
        "rejected",
    }


    if text.isin(
        true_tokens
        |
        false_tokens
    ).all():

        return text.isin(
            true_tokens
        ).to_numpy(
            dtype=bool
        )


    return None


stored_accept_627h4c4r2 = (
    normalize_accept_627h4c4r2(
        stored_accept_raw_627h4c4r2
    )
)


if stored_accept_627h4c4r2 is None:

    print(
        "Stored accept column could not "
        "be safely normalized."
    )

    accept_match_627h4c4r2 = None

else:

    accept_match_627h4c4r2 = bool(
        np.array_equal(
            stored_accept_627h4c4r2,
            expected_accept_627h4c4r2,
        )
    )


    print(
        "Stored accepts :",
        int(
            stored_accept_627h4c4r2.sum()
        )
    )

    print(
        "Stored abstains:",
        int(
            (
                ~stored_accept_627h4c4r2
            ).sum()
        )
    )

    print(
        "Matches frozen 0.95 rule:",
        accept_match_627h4c4r2
    )


# ============================================================
# 6. Stored DECISION representation
# ============================================================

decision_text_627h4c4r2 = (
    df_627h4c4r2[
        "decision"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
)


print("\n--- STORED DECISION COLUMN ---")

print(
    "Unique values:",
    list(
        pd.unique(
            decision_text_627h4c4r2
        )
    )
)


accept_decision_tokens_627h4c4r2 = {
    "accept",
    "accepted",
    "predict",
    "prediction",
}


abstain_decision_tokens_627h4c4r2 = {
    "abstain",
    "abstained",
    "reject",
    "rejected",
}


decision_known_627h4c4r2 = bool(
    decision_text_627h4c4r2.isin(
        accept_decision_tokens_627h4c4r2
        |
        abstain_decision_tokens_627h4c4r2
    ).all()
)


decision_match_627h4c4r2 = None


if decision_known_627h4c4r2:

    decision_accept_627h4c4r2 = (
        decision_text_627h4c4r2
        .isin(
            accept_decision_tokens_627h4c4r2
        )
        .to_numpy(
            dtype=bool
        )
    )


    decision_match_627h4c4r2 = bool(
        np.array_equal(
            decision_accept_627h4c4r2,
            expected_accept_627h4c4r2,
        )
    )


    print(
        "Matches frozen 0.95 rule:",
        decision_match_627h4c4r2
    )

else:

    print(
        "Decision vocabulary is not one of "
        "the recognized accept/abstain encodings."
    )


# ============================================================
# 7. Search notebook history for provenance of 624E
#
# CODE ONLY.
# Does not inspect any RUL values.
# ============================================================

history_627h4c4r2 = globals().get(
    "In",
    []
)


search_terms_627h4c4r2 = [
    "official_test_endpoint_inference_624e",
    "reliability_score",
]


history_hits_627h4c4r2 = []


for idx_627h4c4r2, code_627h4c4r2 in enumerate(
    history_627h4c4r2[:-1]
):

    if not isinstance(
        code_627h4c4r2,
        str
    ):
        continue


    lower_code_627h4c4r2 = (
        code_627h4c4r2.lower()
    )


    if not any(
        term
        in
        lower_code_627h4c4r2

        for term in search_terms_627h4c4r2
    ):

        continue


    relevant_lines_627h4c4r2 = []


    for line_627h4c4r2 in (
        code_627h4c4r2.splitlines()
    ):

        lower_line_627h4c4r2 = (
            line_627h4c4r2.lower()
        )


        if any(
            token
            in
            lower_line_627h4c4r2

            for token in [
                "official_test_endpoint_inference_624e",
                "reliability_score",
                "accept",
                "decision",
                "threshold",
            ]
        ):

            relevant_lines_627h4c4r2.append(
                line_627h4c4r2.strip()
            )


    if relevant_lines_627h4c4r2:

        history_hits_627h4c4r2.append(
            {
                "history_index":
                    idx_627h4c4r2,

                "relevant_code":
                    " | ".join(
                        relevant_lines_627h4c4r2[
                            :12
                        ]
                    ),
            }
        )


TEST_RELIABILITY_HISTORY_627H4C4R2 = (
    pd.DataFrame(
        history_hits_627h4c4r2
    )
)


print(
    "\n--- NOTEBOOK-HISTORY PROVENANCE ---"
)


if TEST_RELIABILITY_HISTORY_627H4C4R2.empty:

    print(
        "No relevant creation code recovered."
    )

else:

    print(
        TEST_RELIABILITY_HISTORY_627H4C4R2
        .tail(15)
        .to_string(
            index=False
        )
    )


# ============================================================
# 8. Final audit object
# ============================================================

STORED_TEST_POLICY_AUDIT_627H4C4R2 = {
    "rows":
        len(
            df_627h4c4r2
        ),

    "frozen_gate":
        FINAL_SELECTIVE_GATE_627H3B,

    "frozen_threshold":
        frozen_threshold_627h4c4r2,

    "stored_score_min":
        float(
            score_627h4c4r2.min()
        ),

    "stored_score_max":
        float(
            score_627h4c4r2.max()
        ),

    "coverage_if_current_threshold_applied":
        float(
            expected_accept_627h4c4r2.mean()
        ),

    "stored_accept_matches_current_rule":
        accept_match_627h4c4r2,

    "stored_decision_matches_current_rule":
        decision_match_627h4c4r2,
}


print(
    "\n"
    "============================================================\n"
    "STEP 6.27H4-C4-R2 SUMMARY\n"
    "============================================================"
)

print(
    "Stored score finite                  :",
    bool(
        np.isfinite(
            score_627h4c4r2
        ).all()
    )
)

print(
    "Stored score within [0,1]            :",
    bool(
        (
            (score_627h4c4r2 >= 0)
            &
            (score_627h4c4r2 <= 1)
        ).all()
    )
)

print(
    "Coverage under CURRENT 0.95 rule     :",
    float(
        expected_accept_627h4c4r2.mean()
    )
)

print(
    "Stored ACCEPT matches current rule   :",
    accept_match_627h4c4r2
)

print(
    "Stored DECISION matches current rule :",
    decision_match_627h4c4r2
)

print(
    "Official RUL labels read             : False"
)

print(
    "Official TEST scored                 : False"
)

print(
    "Model / MC rerun                     : False"
)

print(
    "Files written                        : False"
)

print(
    "\n=== END STEP 6.27H4-C4-R2 ==="
)

In [ ]:
# ============================================================
# STEP 6.27H4-C4-R3
# COMPACT STORED TEST POLICY REVIEW
# ============================================================

print(
    "\n"
    "============================================================\n"
    "STEP 6.27H4-C4-R3 — COMPACT STORED TEST POLICY REVIEW\n"
    "============================================================"
)


# ------------------------------------------------------------
# 1. Required audit object
# ------------------------------------------------------------

if "STORED_TEST_POLICY_AUDIT_627H4C4R2" not in globals():

    raise RuntimeError(
        "STORED_TEST_POLICY_AUDIT_627H4C4R2 is missing. "
        "Run H4-C4-R2 first."
    )


audit_627h4c4r3 = (
    STORED_TEST_POLICY_AUDIT_627H4C4R2
)


# ------------------------------------------------------------
# 2. Exact policy comparison
# ------------------------------------------------------------

print("\n--- STORED POLICY VS CURRENT FROZEN POLICY ---")

print(
    "Frozen gate                      :",
    audit_627h4c4r3[
        "frozen_gate"
    ]
)

print(
    "Frozen threshold                 :",
    audit_627h4c4r3[
        "frozen_threshold"
    ]
)

print(
    "Stored score minimum             :",
    audit_627h4c4r3[
        "stored_score_min"
    ]
)

print(
    "Stored score maximum             :",
    audit_627h4c4r3[
        "stored_score_max"
    ]
)

print(
    "Coverage using CURRENT threshold :",
    audit_627h4c4r3[
        "coverage_if_current_threshold_applied"
    ]
)

print(
    "Stored ACCEPT matches current    :",
    audit_627h4c4r3[
        "stored_accept_matches_current_rule"
    ]
)

print(
    "Stored DECISION matches current  :",
    audit_627h4c4r3[
        "stored_decision_matches_current_rule"
    ]
)


# ------------------------------------------------------------
# 3. Stored accept / decision counts
# ------------------------------------------------------------

df_627h4c4r3 = (
    OFFICIAL_TEST_ENDPOINT_INFERENCE_624E
)


print("\n--- STORED ACCEPT / DECISION COUNTS ---")

print(
    "ACCEPT values:"
)

print(
    df_627h4c4r3[
        "accept"
    ]
    .value_counts(
        dropna=False
    )
    .to_string()
)


print(
    "\nDECISION values:"
)

print(
    df_627h4c4r3[
        "decision"
    ]
    .value_counts(
        dropna=False
    )
    .to_string()
)


# ------------------------------------------------------------
# 4. Notebook-history provenance
# ------------------------------------------------------------

print("\n--- MOST RELEVANT HISTORY HITS ---")


if (
    "TEST_RELIABILITY_HISTORY_627H4C4R2"
    not in globals()
    or
    TEST_RELIABILITY_HISTORY_627H4C4R2.empty
):

    print(
        "NO RELEVANT HISTORY HITS"
    )

else:

    print(
        TEST_RELIABILITY_HISTORY_627H4C4R2
        .tail(12)
        .to_string(
            index=False
        )
    )


# ------------------------------------------------------------
# 5. Closure
# ------------------------------------------------------------

accept_match_627h4c4r3 = (
    audit_627h4c4r3[
        "stored_accept_matches_current_rule"
    ]
)


decision_match_627h4c4r3 = (
    audit_627h4c4r3[
        "stored_decision_matches_current_rule"
    ]
)


STORED_TEST_DECISION_COLUMNS_CURRENT_627H4C4R3 = bool(
    accept_match_627h4c4r3 is True
    and
    decision_match_627h4c4r3 is True
)


print(
    "\n"
    "============================================================\n"
    "STEP 6.27H4-C4-R3 SUMMARY\n"
    "============================================================"
)

print(
    "Stored ACCEPT/DECISION exactly implement "
    "current threshold :",
    STORED_TEST_DECISION_COLUMNS_CURRENT_627H4C4R3
)

print(
    "Official RUL labels read : False"
)

print(
    "Official TEST scored     : False"
)

print(
    "Model / MC rerun         : False"
)

print(
    "Gate changed             : False"
)

print(
    "Threshold changed        : False"
)

print(
    "Files written            : False"
)

print(
    "\n=== END STEP 6.27H4-C4-R3 ==="
)

In [ ]:
# ============================================================
# STEP 6.27H4-C5
# PHASE-ADJUSTED UNCERTAINTY CONTRACT CLOSURE
#
# Purpose:
#   Prove that the frozen phase-adjustment function transforms
#
#       prediction + mc_std
#               ↓
#       adjusted_uncertainty
#
#   exactly as used in the frozen DEV pipeline.
#
# Then construct the label-free TEST adjusted uncertainty
# from EXISTING prediction + mc_std only.
#
# NO RUL labels.
# NO TEST scoring.
# NO model rerun.
# NO MC-dropout rerun.
# NO threshold change.
# NO files written.
# ============================================================

import numpy as np
import pandas as pd
import inspect


print(
    "\n"
    "============================================================\n"
    "STEP 6.27H4-C5 — PHASE-ADJUSTED UNCERTAINTY CLOSURE\n"
    "============================================================"
)


# ============================================================
# 1. Required frozen objects
# ============================================================

required_627h4c5 = [
    "calibrated_627g2e",
    "phase_adjusted_uncertainty_623cb2",
    "OFFICIAL_TEST_ENDPOINT_INFERENCE_624E",
]


print("\n--- REQUIRED OBJECTS ---")

for name in required_627h4c5:

    exists = name in globals()

    print(
        f"{name:<46}: {exists}"
    )

    if not exists:

        raise RuntimeError(
            f"Missing required frozen object: {name}"
        )


print(
    "\nFrozen phase-adjustment signature:",
    inspect.signature(
        phase_adjusted_uncertainty_623cb2
    )
)


# ============================================================
# 2. Required DEV columns
# ============================================================

required_dev_cols_627h4c5 = [
    "clean_prediction",
    "corrupted_prediction",
    "clean_mc_std",
    "corrupted_mc_std",
    "clean_adjusted_uncertainty",
    "corrupted_adjusted_uncertainty",
]


print("\n--- REQUIRED DEV COLUMNS ---")

for col in required_dev_cols_627h4c5:

    exists = (
        col
        in
        calibrated_627g2e.columns
    )

    print(
        f"{col:<36}: {exists}"
    )

    if not exists:

        raise RuntimeError(
            f"Missing DEV column: {col}"
        )


# ============================================================
# 3. Recover frozen DEV inputs / stored outputs
# ============================================================

clean_prediction_627h4c5 = pd.to_numeric(
    calibrated_627g2e[
        "clean_prediction"
    ],
    errors="coerce",
).to_numpy(dtype=float)


corrupt_prediction_627h4c5 = pd.to_numeric(
    calibrated_627g2e[
        "corrupted_prediction"
    ],
    errors="coerce",
).to_numpy(dtype=float)


clean_mc_627h4c5 = pd.to_numeric(
    calibrated_627g2e[
        "clean_mc_std"
    ],
    errors="coerce",
).to_numpy(dtype=float)


corrupt_mc_627h4c5 = pd.to_numeric(
    calibrated_627g2e[
        "corrupted_mc_std"
    ],
    errors="coerce",
).to_numpy(dtype=float)


stored_clean_adjusted_627h4c5 = pd.to_numeric(
    calibrated_627g2e[
        "clean_adjusted_uncertainty"
    ],
    errors="coerce",
).to_numpy(dtype=float)


stored_corrupt_adjusted_627h4c5 = pd.to_numeric(
    calibrated_627g2e[
        "corrupted_adjusted_uncertainty"
    ],
    errors="coerce",
).to_numpy(dtype=float)


# ============================================================
# 4. Exact DEV reproduction
# ============================================================

reproduced_clean_adjusted_627h4c5 = np.asarray(
    phase_adjusted_uncertainty_623cb2(
        clean_prediction_627h4c5,
        clean_mc_627h4c5,
    ),
    dtype=float,
).reshape(-1)


reproduced_corrupt_adjusted_627h4c5 = np.asarray(
    phase_adjusted_uncertainty_623cb2(
        corrupt_prediction_627h4c5,
        corrupt_mc_627h4c5,
    ),
    dtype=float,
).reshape(-1)


clean_error_627h4c5 = float(
    np.max(
        np.abs(
            reproduced_clean_adjusted_627h4c5
            -
            stored_clean_adjusted_627h4c5
        )
    )
)


corrupt_error_627h4c5 = float(
    np.max(
        np.abs(
            reproduced_corrupt_adjusted_627h4c5
            -
            stored_corrupt_adjusted_627h4c5
        )
    )
)


PHASE_ADJUSTMENT_EXACT_627H4C5 = bool(
    max(
        clean_error_627h4c5,
        corrupt_error_627h4c5,
    )
    <=
    1e-12
)


print("\n--- DEV REPRODUCTION ---")

print(
    "Clean adjusted-U max error   :",
    clean_error_627h4c5
)

print(
    "Corrupt adjusted-U max error :",
    corrupt_error_627h4c5
)

print(
    "Phase-adjustment exact       :",
    PHASE_ADJUSTMENT_EXACT_627H4C5
)


if not PHASE_ADJUSTMENT_EXACT_627H4C5:

    raise RuntimeError(
        "Frozen phase-adjustment function did not reproduce "
        "the stored DEV adjusted uncertainty exactly. "
        "Do not continue."
    )


# ============================================================
# 5. Canonical TEST inference contract
# ============================================================

test_base_627h4c5 = (
    OFFICIAL_TEST_ENDPOINT_INFERENCE_624E[
        [
            "unit_id",
            "cycle",
            "prediction",
            "mc_std",
        ]
    ]
    .copy()
)


if len(test_base_627h4c5) != 100:

    raise RuntimeError(
        "Expected exactly 100 FD001 TEST endpoint rows."
    )


if test_base_627h4c5[
    "unit_id"
].nunique() != 100:

    raise RuntimeError(
        "Expected exactly 100 unique TEST engines."
    )


if test_base_627h4c5.duplicated(
    [
        "unit_id",
        "cycle",
    ]
).any():

    raise RuntimeError(
        "Duplicate TEST endpoint keys detected."
    )


test_prediction_627h4c5 = pd.to_numeric(
    test_base_627h4c5[
        "prediction"
    ],
    errors="coerce",
).to_numpy(dtype=float)


test_mc_627h4c5 = pd.to_numeric(
    test_base_627h4c5[
        "mc_std"
    ],
    errors="coerce",
).to_numpy(dtype=float)


if not np.isfinite(
    test_prediction_627h4c5
).all():

    raise RuntimeError(
        "TEST predictions contain non-finite values."
    )


if not np.isfinite(
    test_mc_627h4c5
).all():

    raise RuntimeError(
        "TEST MC-STD contains non-finite values."
    )


# ============================================================
# 6. Construct TEST adjusted uncertainty
#
# IMPORTANT:
# This uses ONLY the already-existing prediction + mc_std.
# No neural-network or MC-dropout rerun occurs.
# ============================================================

test_adjusted_uncertainty_627h4c5 = np.asarray(
    phase_adjusted_uncertainty_623cb2(
        test_prediction_627h4c5,
        test_mc_627h4c5,
    ),
    dtype=float,
).reshape(-1)


if len(
    test_adjusted_uncertainty_627h4c5
) != 100:

    raise RuntimeError(
        "TEST adjusted uncertainty length is not 100."
    )


if not np.isfinite(
    test_adjusted_uncertainty_627h4c5
).all():

    raise RuntimeError(
        "TEST adjusted uncertainty contains "
        "non-finite values."
    )


# ============================================================
# 7. Create clean label-free TEST uncertainty base
# ============================================================

TEST_UNCERTAINTY_BASE_627H4C5 = (
    test_base_627h4c5.copy()
)


TEST_UNCERTAINTY_BASE_627H4C5[
    "adjusted_uncertainty"
] = (
    test_adjusted_uncertainty_627h4c5
)


print(
    "\n--- LABEL-FREE TEST UNCERTAINTY BASE ---"
)

print(
    TEST_UNCERTAINTY_BASE_627H4C5
    .head(10)
    .to_string(
        index=False
    )
)


# ============================================================
# 8. Fingerprints
# ============================================================

print(
    "\n--- TEST ADJUSTED-UNCERTAINTY FINGERPRINT ---"
)

print(
    "Rows   :",
    len(
        TEST_UNCERTAINTY_BASE_627H4C5
    )
)

print(
    "Min    :",
    float(
        np.min(
            test_adjusted_uncertainty_627h4c5
        )
    )
)

print(
    "Median :",
    float(
        np.median(
            test_adjusted_uncertainty_627h4c5
        )
    )
)

print(
    "Mean   :",
    float(
        np.mean(
            test_adjusted_uncertainty_627h4c5
        )
    )
)

print(
    "Max    :",
    float(
        np.max(
            test_adjusted_uncertainty_627h4c5
        )
    )
)


# ============================================================
# 9. Closure
# ============================================================

print(
    "\n"
    "============================================================\n"
    "STEP 6.27H4-C5 SUMMARY\n"
    "============================================================"
)

print(
    "DEV phase adjustment reproduced :",
    PHASE_ADJUSTMENT_EXACT_627H4C5
)

print(
    "TEST endpoint rows              :",
    len(
        TEST_UNCERTAINTY_BASE_627H4C5
    )
)

print(
    "TEST adjusted uncertainty made  : True"
)

print(
    "Legacy reliability score used   : False"
)

print(
    "Legacy accept/decision used     : False"
)

print(
    "Official RUL labels read        : False"
)

print(
    "Official TEST scored            : False"
)

print(
    "Model rerun                     : False"
)

print(
    "MC-dropout rerun                : False"
)

print(
    "Gate changed                    : False"
)

print(
    "Threshold changed               : False"
)

print(
    "Files written                   : False"
)

print(
    "\n=== END STEP 6.27H4-C5 ==="
)

In [ ]:
# ============================================================
# STEP 6.27H4-D
# BUILD FINAL LABEL-FREE FD001 TEST SELECTIVE DECISIONS
#
# Frozen chain:
#
# prediction + mc_std
#        ↓
# phase-adjusted uncertainty
#        ↓
# G1 right ECDF (3490 CLEAN DEV reference)
#        ↓
# uncertainty risk
#        ↓
# G2/G3 exact midrank ECDF (20 CLEAN DEV endpoints)
#        ↓
# final selective risk
#        ↓
# accept if risk <= 0.95
#
# IMPORTANT:
#   - legacy TEST reliability_score NOT used
#   - legacy TEST accept NOT used
#   - legacy TEST decision NOT used
#   - official RUL labels NOT read
#   - TEST NOT scored
#   - model NOT rerun
#   - MC-dropout NOT rerun
#   - no files written
# ============================================================

import numpy as np
import pandas as pd


print(
    "\n"
    "============================================================\n"
    "STEP 6.27H4-D — FINAL LABEL-FREE TEST SELECTIVE DECISIONS\n"
    "============================================================"
)


# ============================================================
# 1. Required frozen objects
# ============================================================

required_627h4d = [
    "TEST_UNCERTAINTY_BASE_627H4C5",
    "FROZEN_UNCERTAINTY_CHAIN_627H4C4",
    "frozen_g1_right_ecdf_627h4c4",
    "frozen_g2_midrank_627h4c4",
    "FINAL_SELECTIVE_GATE_627H3B",
    "FINAL_SELECTIVE_RISK_COLUMN_627H3B",
    "FINAL_SELECTIVE_ORIENTATION_627H3B",
    "FINAL_RELIABILITY_THRESHOLD_627H3B",
]


print("\n--- REQUIRED FROZEN OBJECTS ---")

for name_627h4d in required_627h4d:

    exists_627h4d = (
        name_627h4d
        in globals()
    )

    print(
        f"{name_627h4d:<48}: {exists_627h4d}"
    )

    if not exists_627h4d:

        raise RuntimeError(
            f"Missing required frozen object: "
            f"{name_627h4d}"
        )


# ============================================================
# 2. Reconfirm frozen policy
# ============================================================

gate_627h4d = str(
    FINAL_SELECTIVE_GATE_627H3B
)

orientation_627h4d = str(
    FINAL_SELECTIVE_ORIENTATION_627H3B
)

threshold_627h4d = float(
    FINAL_RELIABILITY_THRESHOLD_627H3B
)


print("\n--- FROZEN OPERATING POLICY ---")

print(
    "Gate        :",
    gate_627h4d
)

print(
    "Orientation :",
    orientation_627h4d
)

print(
    "Threshold   :",
    threshold_627h4d
)


if gate_627h4d != "uncertainty_only":

    raise RuntimeError(
        "Frozen gate is not uncertainty_only."
    )


if orientation_627h4d != "higher_is_worse":

    raise RuntimeError(
        "Unexpected frozen risk orientation."
    )


if not np.isclose(
    threshold_627h4d,
    0.95,
    atol=0.0,
    rtol=0.0,
):

    raise RuntimeError(
        "Frozen threshold is not exactly 0.95."
    )


# ============================================================
# 3. Recover validated label-free TEST uncertainty base
# ============================================================

test_work_627h4d = (
    TEST_UNCERTAINTY_BASE_627H4C5[
        [
            "unit_id",
            "cycle",
            "prediction",
            "mc_std",
            "adjusted_uncertainty",
        ]
    ]
    .copy()
)


# Structural checks
if len(
    test_work_627h4d
) != 100:

    raise RuntimeError(
        "Expected exactly 100 FD001 TEST endpoints."
    )


if test_work_627h4d[
    "unit_id"
].nunique() != 100:

    raise RuntimeError(
        "Expected exactly 100 unique TEST engines."
    )


if test_work_627h4d.duplicated(
    [
        "unit_id",
        "cycle",
    ]
).any():

    raise RuntimeError(
        "Duplicate TEST endpoint keys detected."
    )


numeric_input_cols_627h4d = [
    "prediction",
    "mc_std",
    "adjusted_uncertainty",
]


for col_627h4d in numeric_input_cols_627h4d:

    arr_627h4d = pd.to_numeric(
        test_work_627h4d[
            col_627h4d
        ],
        errors="coerce",
    ).to_numpy(
        dtype=float
    )

    if not np.isfinite(
        arr_627h4d
    ).all():

        raise RuntimeError(
            f"Non-finite values detected in "
            f"{col_627h4d}."
        )


print("\n--- TEST ENDPOINT CONTRACT ---")

print(
    "Rows            :",
    len(
        test_work_627h4d
    )
)

print(
    "Unique engines  :",
    test_work_627h4d[
        "unit_id"
    ].nunique()
)

print(
    "Duplicate keys  :",
    test_work_627h4d.duplicated(
        [
            "unit_id",
            "cycle",
        ]
    ).sum()
)


# ============================================================
# 4. Stage 1:
#    adjusted uncertainty -> G1 uncertainty risk
# ============================================================

adjusted_u_627h4d = pd.to_numeric(
    test_work_627h4d[
        "adjusted_uncertainty"
    ],
    errors="coerce",
).to_numpy(
    dtype=float
)


g1_uncertainty_risk_627h4d = (
    frozen_g1_right_ecdf_627h4c4(
        adjusted_u_627h4d
    )
)


if not np.isfinite(
    g1_uncertainty_risk_627h4d
).all():

    raise RuntimeError(
        "Stage-1 TEST uncertainty risk contains "
        "non-finite values."
    )


if (
    (g1_uncertainty_risk_627h4d < 0.0).any()
    or
    (g1_uncertainty_risk_627h4d > 1.0).any()
):

    raise RuntimeError(
        "Stage-1 TEST uncertainty risk outside [0,1]."
    )


test_work_627h4d[
    "uncertainty_risk"
] = (
    g1_uncertainty_risk_627h4d
)


# ============================================================
# 5. Stage 2:
#    G1 uncertainty risk -> final selective risk
# ============================================================

final_selective_risk_627h4d = (
    frozen_g2_midrank_627h4c4(
        g1_uncertainty_risk_627h4d
    )
)


if not np.isfinite(
    final_selective_risk_627h4d
).all():

    raise RuntimeError(
        "Final TEST selective risk contains "
        "non-finite values."
    )


if (
    (final_selective_risk_627h4d < 0.0).any()
    or
    (final_selective_risk_627h4d > 1.0).any()
):

    raise RuntimeError(
        "Final TEST selective risk outside [0,1]."
    )


test_work_627h4d[
    "selective_risk"
] = (
    final_selective_risk_627h4d
)


# ============================================================
# 6. Apply the FROZEN threshold
#
# higher risk = worse
# accept iff risk <= 0.95
# ============================================================

accept_627h4d = (
    final_selective_risk_627h4d
    <=
    threshold_627h4d
)


decision_627h4d = np.where(
    accept_627h4d,
    "ACCEPT",
    "ABSTAIN",
)


test_work_627h4d[
    "accept"
] = (
    accept_627h4d
)


test_work_627h4d[
    "decision"
] = (
    decision_627h4d
)


# ============================================================
# 7. Canonical final label-free TEST table
# ============================================================

FINAL_TEST_SELECTIVE_DECISIONS_627H4D = (
    test_work_627h4d[
        [
            "unit_id",
            "cycle",
            "prediction",
            "mc_std",
            "adjusted_uncertainty",
            "uncertainty_risk",
            "selective_risk",
            "accept",
            "decision",
        ]
    ]
    .copy()
)


# Sort only for canonical presentation.
FINAL_TEST_SELECTIVE_DECISIONS_627H4D = (
    FINAL_TEST_SELECTIVE_DECISIONS_627H4D
    .sort_values(
        [
            "unit_id",
            "cycle",
        ]
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 8. Final structural checks
# ============================================================

assert (
    FINAL_TEST_SELECTIVE_DECISIONS_627H4D.shape
    ==
    (100, 9)
)


assert (
    FINAL_TEST_SELECTIVE_DECISIONS_627H4D[
        "unit_id"
    ].nunique()
    ==
    100
)


assert (
    FINAL_TEST_SELECTIVE_DECISIONS_627H4D
    .duplicated(
        [
            "unit_id",
            "cycle",
        ]
    )
    .sum()
    ==
    0
)


assert (
    FINAL_TEST_SELECTIVE_DECISIONS_627H4D[
        "decision"
    ]
    .isin(
        [
            "ACCEPT",
            "ABSTAIN",
        ]
    )
    .all()
)


# ============================================================
# 9. Decision statistics
# ============================================================

accepted_count_627h4d = int(
    FINAL_TEST_SELECTIVE_DECISIONS_627H4D[
        "accept"
    ].sum()
)


abstained_count_627h4d = int(
    (
        ~FINAL_TEST_SELECTIVE_DECISIONS_627H4D[
            "accept"
        ]
    ).sum()
)


coverage_627h4d = float(
    FINAL_TEST_SELECTIVE_DECISIONS_627H4D[
        "accept"
    ].mean()
)


print(
    "\n--- FINAL LABEL-FREE TEST DECISIONS ---"
)

print(
    "Accepted engines  :",
    accepted_count_627h4d
)

print(
    "Abstained engines :",
    abstained_count_627h4d
)

print(
    "Coverage          :",
    coverage_627h4d
)


print(
    "\n--- SELECTIVE-RISK DISTRIBUTION ---"
)

print(
    "Minimum :",
    float(
        final_selective_risk_627h4d.min()
    )
)

print(
    "Median  :",
    float(
        np.median(
            final_selective_risk_627h4d
        )
    )
)

print(
    "Mean    :",
    float(
        final_selective_risk_627h4d.mean()
    )
)

print(
    "Maximum :",
    float(
        final_selective_risk_627h4d.max()
    )
)


# ============================================================
# 10. Decision-boundary audit
# ============================================================

accepted_risks_627h4d = (
    FINAL_TEST_SELECTIVE_DECISIONS_627H4D
    .loc[
        FINAL_TEST_SELECTIVE_DECISIONS_627H4D[
            "accept"
        ],
        "selective_risk",
    ]
    .to_numpy(
        dtype=float
    )
)


abstained_risks_627h4d = (
    FINAL_TEST_SELECTIVE_DECISIONS_627H4D
    .loc[
        ~FINAL_TEST_SELECTIVE_DECISIONS_627H4D[
            "accept"
        ],
        "selective_risk",
    ]
    .to_numpy(
        dtype=float
    )
)


print(
    "\n--- DECISION-BOUNDARY AUDIT ---"
)

print(
    "Largest accepted risk :",
    (
        float(
            accepted_risks_627h4d.max()
        )
        if len(
            accepted_risks_627h4d
        )
        else
        None
    )
)

print(
    "Smallest abstained risk:",
    (
        float(
            abstained_risks_627h4d.min()
        )
        if len(
            abstained_risks_627h4d
        )
        else
        None
    )
)

print(
    "Threshold             :",
    threshold_627h4d
)


# ============================================================
# 11. Show endpoint table
# ============================================================

print(
    "\n--- FIRST 15 FINAL TEST RECORDS ---"
)

print(
    FINAL_TEST_SELECTIVE_DECISIONS_627H4D
    .head(15)
    .to_string(
        index=False
    )
)


print(
    "\n--- ABSTAINED TEST ENGINES ---"
)

abstained_preview_627h4d = (
    FINAL_TEST_SELECTIVE_DECISIONS_627H4D.loc[
        ~FINAL_TEST_SELECTIVE_DECISIONS_627H4D[
            "accept"
        ],
        [
            "unit_id",
            "cycle",
            "prediction",
            "mc_std",
            "adjusted_uncertainty",
            "uncertainty_risk",
            "selective_risk",
            "decision",
        ]
    ]
)


if abstained_preview_627h4d.empty:

    print(
        "NONE"
    )

else:

    print(
        abstained_preview_627h4d
        .to_string(
            index=False
        )
    )


# ============================================================
# 12. Freeze provenance metadata
# ============================================================

FINAL_TEST_SELECTIVE_POLICY_627H4D = {

    "dataset":
        "C-MAPSS FD001",

    "evaluation_population":
        "100 official TEST engine endpoints",

    "prediction_source":
        "OFFICIAL_TEST_ENDPOINT_INFERENCE_624E",

    "prediction_column":
        "prediction",

    "mc_uncertainty_column":
        "mc_std",

    "phase_adjustment_function":
        "phase_adjusted_uncertainty_623cb2",

    "stage1_method":
        "right_ecdf",

    "stage1_reference":
        "UNCERTAINTY_RISK_REFERENCE_627G1",

    "stage1_reference_size":
        3490,

    "stage2_method":
        "exact_midrank_ecdf",

    "stage2_reference":
        "RELIABILITY_UNCERTAINTY_REFERENCE_627G3",

    "stage2_reference_size":
        20,

    "gate":
        gate_627h4d,

    "orientation":
        orientation_627h4d,

    "threshold":
        threshold_627h4d,

    "accept_rule":
        "selective_risk <= threshold",

    "abstain_rule":
        "selective_risk > threshold",

    "accepted":
        accepted_count_627h4d,

    "abstained":
        abstained_count_627h4d,

    "coverage":
        coverage_627h4d,

    "legacy_reliability_score_used":
        False,

    "legacy_accept_used":
        False,

    "legacy_decision_used":
        False,

    "official_test_labels_used":
        False,

    "test_scored":
        False,
}


# ============================================================
# 13. Final closure
# ============================================================

print(
    "\n"
    "============================================================\n"
    "STEP 6.27H4-D FINAL SUMMARY\n"
    "============================================================"
)

print(
    "Final TEST endpoint rows        :",
    len(
        FINAL_TEST_SELECTIVE_DECISIONS_627H4D
    )
)

print(
    "Accepted                       :",
    accepted_count_627h4d
)

print(
    "Abstained                      :",
    abstained_count_627h4d
)

print(
    "Coverage                       :",
    coverage_627h4d
)

print(
    "Gate                           :",
    gate_627h4d
)

print(
    "Threshold                      :",
    threshold_627h4d
)

print(
    "Legacy reliability fields used : False"
)

print(
    "Official RUL labels read        : False"
)

print(
    "Official TEST scored            : False"
)

print(
    "Model rerun                     : False"
)

print(
    "MC-dropout rerun                : False"
)

print(
    "Research rule changed           : False"
)

print(
    "Files written                   : False"
)

print(
    "\n=== END STEP 6.27H4-D ==="
)

In [ ]:
# ============================================================
# STEP 6.27H4-D-R
# COMPACT FINAL LABEL-FREE TEST DECISION REVIEW
#
# NO RUL labels.
# NO TEST scoring.
# NO model / MC rerun.
# NO rule changes.
# NO files written.
# ============================================================

import numpy as np
import pandas as pd


print(
    "\n"
    "============================================================\n"
    "STEP 6.27H4-D-R — FINAL LABEL-FREE DECISION REVIEW\n"
    "============================================================"
)


# ============================================================
# 1. Required final object
# ============================================================

if "FINAL_TEST_SELECTIVE_DECISIONS_627H4D" not in globals():
    raise RuntimeError(
        "FINAL_TEST_SELECTIVE_DECISIONS_627H4D is missing. "
        "Run H4-D first."
    )


df_627h4dr = (
    FINAL_TEST_SELECTIVE_DECISIONS_627H4D
    .copy()
)


threshold_627h4dr = float(
    FINAL_RELIABILITY_THRESHOLD_627H3B
)


# ============================================================
# 2. Structural closure
# ============================================================

print("\n--- STRUCTURAL CONTRACT ---")

print(
    "Shape          :",
    df_627h4dr.shape
)

print(
    "Unique engines :",
    df_627h4dr[
        "unit_id"
    ].nunique()
)

print(
    "Duplicate keys :",
    df_627h4dr.duplicated(
        [
            "unit_id",
            "cycle",
        ]
    ).sum()
)


# ============================================================
# 3. Final decision counts
# ============================================================

accepted_627h4dr = int(
    df_627h4dr[
        "accept"
    ].sum()
)


abstained_627h4dr = int(
    (
        ~df_627h4dr[
            "accept"
        ]
    ).sum()
)


coverage_627h4dr = float(
    df_627h4dr[
        "accept"
    ].mean()
)


print("\n--- FINAL DECISIONS ---")

print(
    "Accepted  :",
    accepted_627h4dr
)

print(
    "Abstained :",
    abstained_627h4dr
)

print(
    "Coverage  :",
    coverage_627h4dr
)


print(
    "\nDecision counts:"
)

print(
    df_627h4dr[
        "decision"
    ]
    .value_counts(
        dropna=False
    )
    .to_string()
)


# ============================================================
# 4. Selective-risk distribution
# ============================================================

risk_627h4dr = pd.to_numeric(
    df_627h4dr[
        "selective_risk"
    ],
    errors="coerce",
).to_numpy(
    dtype=float
)


print("\n--- SELECTIVE-RISK DISTRIBUTION ---")

print(
    "Minimum :",
    float(
        np.min(
            risk_627h4dr
        )
    )
)

print(
    "Median  :",
    float(
        np.median(
            risk_627h4dr
        )
    )
)

print(
    "Mean    :",
    float(
        np.mean(
            risk_627h4dr
        )
    )
)

print(
    "Maximum :",
    float(
        np.max(
            risk_627h4dr
        )
    )
)


# ============================================================
# 5. Decision-boundary closure
# ============================================================

accepted_risk_627h4dr = (
    df_627h4dr.loc[
        df_627h4dr[
            "accept"
        ],
        "selective_risk",
    ]
    .to_numpy(
        dtype=float
    )
)


abstained_risk_627h4dr = (
    df_627h4dr.loc[
        ~df_627h4dr[
            "accept"
        ],
        "selective_risk",
    ]
    .to_numpy(
        dtype=float
    )
)


largest_accepted_627h4dr = (
    float(
        accepted_risk_627h4dr.max()
    )
    if len(
        accepted_risk_627h4dr
    )
    else
    None
)


smallest_abstained_627h4dr = (
    float(
        abstained_risk_627h4dr.min()
    )
    if len(
        abstained_risk_627h4dr
    )
    else
    None
)


print("\n--- DECISION BOUNDARY ---")

print(
    "Threshold              :",
    threshold_627h4dr
)

print(
    "Largest accepted risk  :",
    largest_accepted_627h4dr
)

print(
    "Smallest abstained risk :",
    smallest_abstained_627h4dr
)


boundary_exact_627h4dr = bool(
    (
        largest_accepted_627h4dr is None
        or
        largest_accepted_627h4dr
        <=
        threshold_627h4dr
    )
    and
    (
        smallest_abstained_627h4dr is None
        or
        smallest_abstained_627h4dr
        >
        threshold_627h4dr
    )
)


print(
    "Boundary rule exact     :",
    boundary_exact_627h4dr
)


# ============================================================
# 6. Abstained engines
# ============================================================

abstained_df_627h4dr = (
    df_627h4dr.loc[
        ~df_627h4dr[
            "accept"
        ],
        [
            "unit_id",
            "cycle",
            "prediction",
            "mc_std",
            "adjusted_uncertainty",
            "uncertainty_risk",
            "selective_risk",
            "decision",
        ]
    ]
    .sort_values(
        [
            "selective_risk",
            "unit_id",
        ],
        ascending=[
            False,
            True,
        ],
    )
)


print("\n--- ABSTAINED ENGINES ---")

if abstained_df_627h4dr.empty:

    print("NONE")

else:

    print(
        abstained_df_627h4dr
        .to_string(
            index=False
        )
    )


# ============================================================
# 7. Final freeze readiness
# ============================================================

LABEL_FREE_TEST_POLICY_READY_627H4DR = bool(
    df_627h4dr.shape
    ==
    (100, 9)

    and
    df_627h4dr[
        "unit_id"
    ].nunique()
    ==
    100

    and
    df_627h4dr.duplicated(
        [
            "unit_id",
            "cycle",
        ]
    ).sum()
    ==
    0

    and
    np.isfinite(
        risk_627h4dr
    ).all()

    and
    (
        (risk_627h4dr >= 0.0)
        &
        (risk_627h4dr <= 1.0)
    ).all()

    and
    boundary_exact_627h4dr
)


print(
    "\n"
    "============================================================\n"
    "STEP 6.27H4-D-R SUMMARY\n"
    "============================================================"
)

print(
    "Label-free TEST policy ready :",
    LABEL_FREE_TEST_POLICY_READY_627H4DR
)

print(
    "Accepted                     :",
    accepted_627h4dr
)

print(
    "Abstained                    :",
    abstained_627h4dr
)

print(
    "Coverage                     :",
    coverage_627h4dr
)

print(
    "Threshold                    :",
    threshold_627h4dr
)

print(
    "Boundary rule exact          :",
    boundary_exact_627h4dr
)

print(
    "Legacy TEST decisions used   : False"
)

print(
    "Official RUL labels read     : False"
)

print(
    "Official TEST scored         : False"
)

print(
    "Model / MC rerun             : False"
)

print(
    "Research rule changed        : False"
)

print(
    "Files written                : False"
)

print(
    "\n=== END STEP 6.27H4-D-R ==="
)

In [ ]:
# ============================================================
# STEP 6.27H4-D-R2
# LABEL-FREE TEST RISK-SATURATION DIAGNOSTIC
#
# Purpose:
#   Explain the unexpectedly low TEST coverage (0.37)
#   WITHOUT looking at any official TEST RUL labels.
#
# NO TEST labels.
# NO TEST scoring.
# NO threshold tuning.
# NO model / MC rerun.
# NO research-rule changes.
# NO files written.
# ============================================================

import numpy as np
import pandas as pd


print(
    "\n"
    "============================================================\n"
    "STEP 6.27H4-D-R2 — LABEL-FREE RISK SATURATION DIAGNOSTIC\n"
    "============================================================"
)


# ============================================================
# 1. Required frozen objects
# ============================================================

required_627h4dr2 = [
    "FINAL_TEST_SELECTIVE_DECISIONS_627H4D",
    "G1_REFERENCE_627H4C4",
    "G2_REFERENCE_627H4C4",
    "FINAL_RELIABILITY_THRESHOLD_627H3B",
    "frozen_g2_midrank_627h4c4",
]


print("\n--- REQUIRED OBJECTS ---")

for name in required_627h4dr2:

    exists = name in globals()

    print(
        f"{name:<44}: {exists}"
    )

    if not exists:

        raise RuntimeError(
            f"Missing required object: {name}"
        )


df_627h4dr2 = (
    FINAL_TEST_SELECTIVE_DECISIONS_627H4D
    .copy()
)


g1_ref_627h4dr2 = np.asarray(
    G1_REFERENCE_627H4C4,
    dtype=float,
).reshape(-1)


g2_ref_627h4dr2 = np.asarray(
    G2_REFERENCE_627H4C4,
    dtype=float,
).reshape(-1)


threshold_627h4dr2 = float(
    FINAL_RELIABILITY_THRESHOLD_627H3B
)


# ============================================================
# 2. Recover TEST channels
# ============================================================

adjusted_627h4dr2 = pd.to_numeric(
    df_627h4dr2[
        "adjusted_uncertainty"
    ],
    errors="coerce",
).to_numpy(
    dtype=float
)


stage1_627h4dr2 = pd.to_numeric(
    df_627h4dr2[
        "uncertainty_risk"
    ],
    errors="coerce",
).to_numpy(
    dtype=float
)


stage2_627h4dr2 = pd.to_numeric(
    df_627h4dr2[
        "selective_risk"
    ],
    errors="coerce",
).to_numpy(
    dtype=float
)


accept_627h4dr2 = (
    df_627h4dr2[
        "accept"
    ]
    .to_numpy(
        dtype=bool
    )
)


# ============================================================
# 3. Reference distributions
# ============================================================

print("\n--- G1 REFERENCE: CLEAN DEV WINDOWS ---")

print(
    "n      :",
    len(g1_ref_627h4dr2)
)

print(
    "min    :",
    float(
        np.min(g1_ref_627h4dr2)
    )
)

print(
    "median :",
    float(
        np.median(g1_ref_627h4dr2)
    )
)

print(
    "q95    :",
    float(
        np.quantile(
            g1_ref_627h4dr2,
            0.95
        )
    )
)

print(
    "max    :",
    float(
        np.max(g1_ref_627h4dr2)
    )
)


print("\n--- G2 REFERENCE: 20 CLEAN DEV ENDPOINTS ---")

print(
    "n      :",
    len(g2_ref_627h4dr2)
)

print(
    "min    :",
    float(
        np.min(g2_ref_627h4dr2)
    )
)

print(
    "median :",
    float(
        np.median(g2_ref_627h4dr2)
    )
)

print(
    "q75    :",
    float(
        np.quantile(
            g2_ref_627h4dr2,
            0.75
        )
    )
)

print(
    "max    :",
    float(
        np.max(g2_ref_627h4dr2)
    )
)


print(
    "\nSorted G2 reference:"
)

print(
    np.sort(
        g2_ref_627h4dr2
    )
)


# ============================================================
# 4. TEST adjusted-uncertainty relation to G1 reference
# ============================================================

g1_max_627h4dr2 = float(
    np.max(
        g1_ref_627h4dr2
    )
)


adjusted_above_g1_max_627h4dr2 = (
    adjusted_627h4dr2
    >
    g1_max_627h4dr2
)


print(
    "\n--- TEST ADJUSTED UNCERTAINTY VS G1 REFERENCE ---"
)

print(
    "TEST adjusted min    :",
    float(
        np.min(adjusted_627h4dr2)
    )
)

print(
    "TEST adjusted median :",
    float(
        np.median(adjusted_627h4dr2)
    )
)

print(
    "TEST adjusted mean   :",
    float(
        np.mean(adjusted_627h4dr2)
    )
)

print(
    "TEST adjusted max    :",
    float(
        np.max(adjusted_627h4dr2)
    )
)

print(
    "Above G1 ref maximum :",
    int(
        adjusted_above_g1_max_627h4dr2.sum()
    ),
    "/ 100"
)


# ============================================================
# 5. TEST Stage-1 risk relation to G2 reference
# ============================================================

g2_min_627h4dr2 = float(
    np.min(
        g2_ref_627h4dr2
    )
)


g2_max_627h4dr2 = float(
    np.max(
        g2_ref_627h4dr2
    )
)


below_g2_min_627h4dr2 = (
    stage1_627h4dr2
    <
    g2_min_627h4dr2
)


above_g2_max_627h4dr2 = (
    stage1_627h4dr2
    >
    g2_max_627h4dr2
)


equal_g2_max_627h4dr2 = np.isclose(
    stage1_627h4dr2,
    g2_max_627h4dr2,
    atol=0.0,
    rtol=0.0,
)


print(
    "\n--- TEST STAGE-1 RISK VS G2 REFERENCE ---"
)

print(
    "Stage-1 min           :",
    float(
        np.min(stage1_627h4dr2)
    )
)

print(
    "Stage-1 median        :",
    float(
        np.median(stage1_627h4dr2)
    )
)

print(
    "Stage-1 mean          :",
    float(
        np.mean(stage1_627h4dr2)
    )
)

print(
    "Stage-1 max           :",
    float(
        np.max(stage1_627h4dr2)
    )
)

print(
    "Below G2 ref minimum  :",
    int(
        below_g2_min_627h4dr2.sum()
    )
)

print(
    "Equal G2 ref maximum  :",
    int(
        equal_g2_max_627h4dr2.sum()
    )
)

print(
    "Above G2 ref maximum  :",
    int(
        above_g2_max_627h4dr2.sum()
    )
)


# ============================================================
# 6. Stage-2 saturation
# ============================================================

risk_one_627h4dr2 = np.isclose(
    stage2_627h4dr2,
    1.0,
    atol=0.0,
    rtol=0.0,
)


risk_above_threshold_627h4dr2 = (
    stage2_627h4dr2
    >
    threshold_627h4dr2
)


print(
    "\n--- STAGE-2 SATURATION ---"
)

print(
    "Selective risk == 1.0 :",
    int(
        risk_one_627h4dr2.sum()
    )
)

print(
    "Risk > 0.95           :",
    int(
        risk_above_threshold_627h4dr2.sum()
    )
)

print(
    "Abstained             :",
    int(
        (
            ~accept_627h4dr2
        ).sum()
    )
)

print(
    "All abstentions match risk > threshold:",
    bool(
        np.array_equal(
            ~accept_627h4dr2,
            risk_above_threshold_627h4dr2,
        )
    )
)


# ============================================================
# 7. Exact Stage-2 recomputation
# ============================================================

stage2_recomputed_627h4dr2 = (
    frozen_g2_midrank_627h4c4(
        stage1_627h4dr2
    )
)


stage2_reproduction_error_627h4dr2 = float(
    np.max(
        np.abs(
            stage2_recomputed_627h4dr2
            -
            stage2_627h4dr2
        )
    )
)


print(
    "\n--- STAGE-2 REPRODUCTION CHECK ---"
)

print(
    "Maximum absolute error:",
    stage2_reproduction_error_627h4dr2
)


# ============================================================
# 8. Risk-grid counts
# ============================================================

risk_counts_627h4dr2 = (
    pd.Series(
        stage2_627h4dr2,
        name="selective_risk",
    )
    .value_counts()
    .sort_index()
    .rename_axis(
        "selective_risk"
    )
    .reset_index(
        name="engines"
    )
)


print(
    "\n--- FINAL SELECTIVE-RISK GRID ---"
)

print(
    risk_counts_627h4dr2
    .to_string(
        index=False
    )
)


# ============================================================
# 9. Decision boundary
# ============================================================

accepted_risks_627h4dr2 = (
    stage2_627h4dr2[
        accept_627h4dr2
    ]
)


abstained_risks_627h4dr2 = (
    stage2_627h4dr2[
        ~accept_627h4dr2
    ]
)


largest_accepted_627h4dr2 = (
    float(
        np.max(
            accepted_risks_627h4dr2
        )
    )
    if len(
        accepted_risks_627h4dr2
    )
    else None
)


smallest_abstained_627h4dr2 = (
    float(
        np.min(
            abstained_risks_627h4dr2
        )
    )
    if len(
        abstained_risks_627h4dr2
    )
    else None
)


boundary_exact_627h4dr2 = bool(
    (
        largest_accepted_627h4dr2 is None
        or
        largest_accepted_627h4dr2
        <=
        threshold_627h4dr2
    )
    and
    (
        smallest_abstained_627h4dr2 is None
        or
        smallest_abstained_627h4dr2
        >
        threshold_627h4dr2
    )
)


print(
    "\n--- DECISION BOUNDARY ---"
)

print(
    "Largest accepted risk  :",
    largest_accepted_627h4dr2
)

print(
    "Smallest abstained risk :",
    smallest_abstained_627h4dr2
)

print(
    "Threshold               :",
    threshold_627h4dr2
)

print(
    "Boundary exact          :",
    boundary_exact_627h4dr2
)


# ============================================================
# 10. Main diagnostic interpretation flags
# ============================================================

SATURATION_EXPLAINS_ABSTENTION_627H4DR2 = bool(
    (
        above_g2_max_627h4dr2
        |
        equal_g2_max_627h4dr2
    ).sum()
    >=
    (
        ~accept_627h4dr2
    ).sum()
)


LABEL_FREE_PIPELINE_INTERNALLY_EXACT_627H4DR2 = bool(
    stage2_reproduction_error_627h4dr2
    <=
    1e-12
    and
    boundary_exact_627h4dr2
)


print(
    "\n"
    "============================================================\n"
    "STEP 6.27H4-D-R2 SUMMARY\n"
    "============================================================"
)

print(
    "TEST coverage                         :",
    float(
        accept_627h4dr2.mean()
    )
)

print(
    "TEST abstentions                      :",
    int(
        (
            ~accept_627h4dr2
        ).sum()
    )
)

print(
    "Stage-1 values above G2 reference max :",
    int(
        above_g2_max_627h4dr2.sum()
    )
)

print(
    "Final risk exactly 1.0                :",
    int(
        risk_one_627h4dr2.sum()
    )
)

print(
    "Stage-2 reproduction exact            :",
    bool(
        stage2_reproduction_error_627h4dr2
        <=
        1e-12
    )
)

print(
    "Decision boundary exact               :",
    boundary_exact_627h4dr2
)

print(
    "Label-free pipeline internally exact  :",
    LABEL_FREE_PIPELINE_INTERNALLY_EXACT_627H4DR2
)

print(
    "Official RUL labels read              : False"
)

print(
    "Official TEST scored                  : False"
)

print(
    "Threshold changed                     : False"
)

print(
    "Research rule changed                 : False"
)

print(
    "Files written                         : False"
)

print(
    "\n=== END STEP 6.27H4-D-R2 ==="
)

In [ ]:
# ============================================================
# STEP 6.27H4-D-R3
# COMPACT SATURATION CLOSURE
# ============================================================

import numpy as np
import pandas as pd

print(
    "\n"
    "============================================================\n"
    "STEP 6.27H4-D-R3 — COMPACT SATURATION CLOSURE\n"
    "============================================================"
)

df = FINAL_TEST_SELECTIVE_DECISIONS_627H4D.copy()

stage1 = pd.to_numeric(
    df["uncertainty_risk"],
    errors="coerce"
).to_numpy(dtype=float)

stage2 = pd.to_numeric(
    df["selective_risk"],
    errors="coerce"
).to_numpy(dtype=float)

accept = df["accept"].to_numpy(dtype=bool)

g2_ref = np.asarray(
    G2_REFERENCE_627H4C4,
    dtype=float
).reshape(-1)

threshold = float(
    FINAL_RELIABILITY_THRESHOLD_627H3B
)

g2_max = float(
    np.max(g2_ref)
)

above_g2_max = (
    stage1 > g2_max
)

equal_g2_max = np.isclose(
    stage1,
    g2_max,
    atol=0.0,
    rtol=0.0
)

risk_one = np.isclose(
    stage2,
    1.0,
    atol=0.0,
    rtol=0.0
)

risk_above_threshold = (
    stage2 > threshold
)

# Exact recomputation
stage2_recomputed = (
    frozen_g2_midrank_627h4c4(
        stage1
    )
)

reproduction_error = float(
    np.max(
        np.abs(
            stage2_recomputed
            -
            stage2
        )
    )
)

accepted_risks = stage2[
    accept
]

abstained_risks = stage2[
    ~accept
]

largest_accepted = (
    float(
        accepted_risks.max()
    )
    if len(accepted_risks)
    else None
)

smallest_abstained = (
    float(
        abstained_risks.min()
    )
    if len(abstained_risks)
    else None
)

boundary_exact = bool(
    (
        largest_accepted is None
        or
        largest_accepted <= threshold
    )
    and
    (
        smallest_abstained is None
        or
        smallest_abstained > threshold
    )
)

print("\n--- SATURATION COUNTS ---")

print(
    "G2 reference max             :",
    g2_max
)

print(
    "Stage-1 > G2 reference max   :",
    int(
        above_g2_max.sum()
    )
)

print(
    "Stage-1 == G2 reference max  :",
    int(
        equal_g2_max.sum()
    )
)

print(
    "Final selective risk == 1.0  :",
    int(
        risk_one.sum()
    )
)

print(
    "Final risk > 0.95            :",
    int(
        risk_above_threshold.sum()
    )
)

print(
    "Abstained engines            :",
    int(
        (~accept).sum()
    )
)

print(
    "Accepted engines             :",
    int(
        accept.sum()
    )
)

print(
    "Coverage                     :",
    float(
        accept.mean()
    )
)

print("\n--- BOUNDARY / REPRODUCTION ---")

print(
    "Largest accepted risk        :",
    largest_accepted
)

print(
    "Smallest abstained risk       :",
    smallest_abstained
)

print(
    "Frozen threshold             :",
    threshold
)

print(
    "Stage-2 reproduction error   :",
    reproduction_error
)

print(
    "Stage-2 exact <= 1e-12       :",
    bool(
        reproduction_error
        <= 1e-12
    )
)

print(
    "Decision boundary exact      :",
    boundary_exact
)

print("\n--- SELECTIVE-RISK COUNTS ---")

print(
    pd.Series(
        stage2
    )
    .value_counts()
    .sort_index()
    .to_string()
)

print(
    "\n"
    "============================================================\n"
    "STEP 6.27H4-D-R3 SUMMARY\n"
    "============================================================"
)

print(
    "Pipeline internally exact :",
    bool(
        reproduction_error <= 1e-12
        and
        boundary_exact
    )
)

print(
    "Official RUL labels read  : False"
)

print(
    "Official TEST scored      : False"
)

print(
    "Threshold changed         : False"
)

print(
    "Research rule changed     : False"
)

print(
    "Files written             : False"
)

print(
    "\n=== END STEP 6.27H4-D-R3 ==="
)

In [ ]:
# ============================================================
# STEP 6.27H5-A
# OFFICIAL FD001 TEST LABEL ATTACHMENT + PRIMARY SCORING
#
# IMPORTANT:
#   The selective policy was frozen BEFORE this step.
#
# Primary target:
#   RUL_cap_125 = min(official NASA RUL, 125)
#
# Secondary raw label is retained for transparency,
# but is NOT used to redefine the frozen primary target.
#
# NO gate change.
# NO threshold change.
# NO model rerun.
# NO MC rerun.
# NO files written.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error


print(
    "\n"
    "============================================================\n"
    "STEP 6.27H5-A — OFFICIAL FD001 TEST SCORING\n"
    "============================================================"
)


# ============================================================
# 1. Required frozen label-free policy
# ============================================================

required_627h5a = [
    "FINAL_TEST_SELECTIVE_DECISIONS_627H4D",
    "FINAL_RELIABILITY_THRESHOLD_627H3B",
    "FINAL_SELECTIVE_GATE_627H3B",
]


print("\n--- REQUIRED FROZEN OBJECTS ---")

for name in required_627h5a:

    exists = name in globals()

    print(
        f"{name:<46}: {exists}"
    )

    if not exists:

        raise RuntimeError(
            f"Missing frozen object: {name}"
        )


# ============================================================
# 2. Recover project / dataset directory
# ============================================================

cwd_627h5a = Path.cwd()


if (cwd_627h5a / "02_dataset").exists():

    project_root_627h5a = cwd_627h5a

elif (cwd_627h5a.parent / "02_dataset").exists():

    project_root_627h5a = cwd_627h5a.parent

else:

    raise FileNotFoundError(
        "Could not locate project root."
    )


data_dir_627h5a = (
    project_root_627h5a
    /
    "02_dataset"
    /
    "raw"
    /
    "CMAPSSData"
)


rul_path_627h5a = (
    data_dir_627h5a
    /
    "RUL_FD001.txt"
)


print("\n--- OFFICIAL LABEL SOURCE ---")

print(
    "Project root :",
    project_root_627h5a
)

print(
    "RUL file     :",
    rul_path_627h5a
)

print(
    "Exists       :",
    rul_path_627h5a.exists()
)


if not rul_path_627h5a.exists():

    raise FileNotFoundError(
        f"Missing official FD001 RUL file: "
        f"{rul_path_627h5a}"
    )


# ============================================================
# 3. Read official NASA FD001 endpoint RUL labels
#
# First time these labels are intentionally used in the
# frozen selective-policy evaluation.
# ============================================================

official_rul_627h5a = pd.read_csv(
    rul_path_627h5a,
    sep=r"\s+",
    header=None,
    names=[
        "RUL_raw"
    ],
)


if official_rul_627h5a.shape != (100, 1):

    raise RuntimeError(
        "Expected RUL_FD001.txt to contain exactly "
        "100 rows and 1 column."
    )


official_rul_values_627h5a = pd.to_numeric(
    official_rul_627h5a[
        "RUL_raw"
    ],
    errors="coerce",
).to_numpy(
    dtype=float
)


if not np.isfinite(
    official_rul_values_627h5a
).all():

    raise RuntimeError(
        "Official FD001 RUL labels contain "
        "non-finite values."
    )


if (
    official_rul_values_627h5a
    <
    0.0
).any():

    raise RuntimeError(
        "Official FD001 RUL contains negative values."
    )


print(
    "Official RUL rows:",
    len(
        official_rul_values_627h5a
    )
)

print(
    "Raw RUL min / median / max:",
    float(
        official_rul_values_627h5a.min()
    ),
    "/",
    float(
        np.median(
            official_rul_values_627h5a
        )
    ),
    "/",
    float(
        official_rul_values_627h5a.max()
    )
)


# ============================================================
# 4. Frozen primary TEST target convention
#
# DEV target convention was already frozen as RUL_cap_125.
# Therefore:
#
#     primary TEST target = min(RUL_raw, 125)
#
# This is NOT selected using TEST performance.
# ============================================================

official_rul_cap125_627h5a = np.minimum(
    official_rul_values_627h5a,
    125.0,
)


print(
    "\n--- FROZEN TARGET CONVENTION ---"
)

print(
    "Primary target column : RUL_cap_125"
)

print(
    "Cap                  : 125"
)

print(
    "Raw labels > 125     :",
    int(
        (
            official_rul_values_627h5a
            >
            125.0
        ).sum()
    )
)

print(
    "Primary min / median / max:",
    float(
        official_rul_cap125_627h5a.min()
    ),
    "/",
    float(
        np.median(
            official_rul_cap125_627h5a
        )
    ),
    "/",
    float(
        official_rul_cap125_627h5a.max()
    )
)


# ============================================================
# 5. Attach labels to the already-frozen 100-engine table
#
# C-MAPSS RUL_FD001 rows correspond to TEST engine order.
# We explicitly require unit IDs 1..100 before attachment.
# ============================================================

test_scored_627h5a = (
    FINAL_TEST_SELECTIVE_DECISIONS_627H4D
    .copy()
    .sort_values(
        "unit_id"
    )
    .reset_index(
        drop=True
    )
)


expected_units_627h5a = np.arange(
    1,
    101,
    dtype=int,
)


actual_units_627h5a = (
    test_scored_627h5a[
        "unit_id"
    ]
    .to_numpy(
        dtype=int
    )
)


unit_order_exact_627h5a = bool(
    np.array_equal(
        actual_units_627h5a,
        expected_units_627h5a,
    )
)


print(
    "\n--- LABEL ALIGNMENT CONTRACT ---"
)

print(
    "TEST rows          :",
    len(
        test_scored_627h5a
    )
)

print(
    "Unique engines     :",
    test_scored_627h5a[
        "unit_id"
    ].nunique()
)

print(
    "Unit order = 1..100:",
    unit_order_exact_627h5a
)


if not unit_order_exact_627h5a:

    raise RuntimeError(
        "TEST engine order is not exactly 1..100. "
        "Do not attach RUL labels positionally."
    )


test_scored_627h5a[
    "RUL_raw"
] = (
    official_rul_values_627h5a
)


test_scored_627h5a[
    "RUL_cap_125"
] = (
    official_rul_cap125_627h5a
)


# ============================================================
# 6. Prediction error using FROZEN PRIMARY TARGET
# ============================================================

prediction_627h5a = pd.to_numeric(
    test_scored_627h5a[
        "prediction"
    ],
    errors="coerce",
).to_numpy(
    dtype=float
)


target_627h5a = pd.to_numeric(
    test_scored_627h5a[
        "RUL_cap_125"
    ],
    errors="coerce",
).to_numpy(
    dtype=float
)


accept_627h5a = (
    test_scored_627h5a[
        "accept"
    ]
    .to_numpy(
        dtype=bool
    )
)


error_627h5a = (
    prediction_627h5a
    -
    target_627h5a
)


abs_error_627h5a = np.abs(
    error_627h5a
)


test_scored_627h5a[
    "prediction_error"
] = (
    error_627h5a
)


test_scored_627h5a[
    "absolute_error"
] = (
    abs_error_627h5a
)


# ============================================================
# 7. Metric helper
# ============================================================

def metric_block_627h5a(
    y_true,
    y_pred,
):

    y_true = np.asarray(
        y_true,
        dtype=float,
    )

    y_pred = np.asarray(
        y_pred,
        dtype=float,
    )

    return {
        "n":
            int(
                len(
                    y_true
                )
            ),

        "MAE":
            float(
                mean_absolute_error(
                    y_true,
                    y_pred,
                )
            ),

        "RMSE":
            float(
                np.sqrt(
                    mean_squared_error(
                        y_true,
                        y_pred,
                    )
                )
            ),

        "mean_error":
            float(
                np.mean(
                    y_pred
                    -
                    y_true
                )
            ),

        "median_absolute_error":
            float(
                np.median(
                    np.abs(
                        y_pred
                        -
                        y_true
                    )
                )
            ),
    }


# ============================================================
# 8. Full vs accepted vs abstained metrics
# ============================================================

full_metrics_627h5a = (
    metric_block_627h5a(
        target_627h5a,
        prediction_627h5a,
    )
)


accepted_metrics_627h5a = (
    metric_block_627h5a(
        target_627h5a[
            accept_627h5a
        ],
        prediction_627h5a[
            accept_627h5a
        ],
    )
)


abstained_metrics_627h5a = (
    metric_block_627h5a(
        target_627h5a[
            ~accept_627h5a
        ],
        prediction_627h5a[
            ~accept_627h5a
        ],
    )
)


coverage_627h5a = float(
    accept_627h5a.mean()
)


mae_gain_627h5a = float(
    full_metrics_627h5a[
        "MAE"
    ]
    -
    accepted_metrics_627h5a[
        "MAE"
    ]
)


rmse_gain_627h5a = float(
    full_metrics_627h5a[
        "RMSE"
    ]
    -
    accepted_metrics_627h5a[
        "RMSE"
    ]
)


mae_reduction_pct_627h5a = float(
    100.0
    *
    mae_gain_627h5a
    /
    full_metrics_627h5a[
        "MAE"
    ]
)


rmse_reduction_pct_627h5a = float(
    100.0
    *
    rmse_gain_627h5a
    /
    full_metrics_627h5a[
        "RMSE"
    ]
)


# ============================================================
# 9. Main final scoring summary
# ============================================================

FINAL_FD001_TEST_SCORED_627H5A = (
    test_scored_627h5a.copy()
)


FINAL_FD001_TEST_METRICS_627H5A = pd.DataFrame(
    [
        {
            "population":
                "all_test",

            "n":
                full_metrics_627h5a[
                    "n"
                ],

            "coverage":
                1.0,

            "MAE":
                full_metrics_627h5a[
                    "MAE"
                ],

            "RMSE":
                full_metrics_627h5a[
                    "RMSE"
                ],

            "mean_error":
                full_metrics_627h5a[
                    "mean_error"
                ],

            "median_absolute_error":
                full_metrics_627h5a[
                    "median_absolute_error"
                ],
        },

        {
            "population":
                "accepted",

            "n":
                accepted_metrics_627h5a[
                    "n"
                ],

            "coverage":
                coverage_627h5a,

            "MAE":
                accepted_metrics_627h5a[
                    "MAE"
                ],

            "RMSE":
                accepted_metrics_627h5a[
                    "RMSE"
                ],

            "mean_error":
                accepted_metrics_627h5a[
                    "mean_error"
                ],

            "median_absolute_error":
                accepted_metrics_627h5a[
                    "median_absolute_error"
                ],
        },

        {
            "population":
                "abstained",

            "n":
                abstained_metrics_627h5a[
                    "n"
                ],

            "coverage":
                1.0
                -
                coverage_627h5a,

            "MAE":
                abstained_metrics_627h5a[
                    "MAE"
                ],

            "RMSE":
                abstained_metrics_627h5a[
                    "RMSE"
                ],

            "mean_error":
                abstained_metrics_627h5a[
                    "mean_error"
                ],

            "median_absolute_error":
                abstained_metrics_627h5a[
                    "median_absolute_error"
                ],
        },
    ]
)


print(
    "\n--- PRIMARY FD001 TEST METRICS ---"
)

print(
    FINAL_FD001_TEST_METRICS_627H5A
    .to_string(
        index=False
    )
)


print(
    "\n--- SELECTIVE IMPROVEMENT ---"
)

print(
    "Frozen TEST coverage        :",
    coverage_627h5a
)

print(
    "Full TEST MAE              :",
    full_metrics_627h5a[
        "MAE"
    ]
)

print(
    "Accepted TEST MAE          :",
    accepted_metrics_627h5a[
        "MAE"
    ]
)

print(
    "MAE reduction              :",
    mae_gain_627h5a
)

print(
    "MAE reduction (%)          :",
    mae_reduction_pct_627h5a
)

print(
    "Full TEST RMSE             :",
    full_metrics_627h5a[
        "RMSE"
    ]
)

print(
    "Accepted TEST RMSE         :",
    accepted_metrics_627h5a[
        "RMSE"
    ]
)

print(
    "RMSE reduction             :",
    rmse_gain_627h5a
)

print(
    "RMSE reduction (%)         :",
    rmse_reduction_pct_627h5a
)


# ============================================================
# 10. Does the gate actually reject harder cases?
# ============================================================

print(
    "\n--- ACCEPTED VS ABSTAINED ERROR ---"
)

print(
    "Accepted mean absolute error :",
    accepted_metrics_627h5a[
        "MAE"
    ]
)

print(
    "Abstained mean absolute error:",
    abstained_metrics_627h5a[
        "MAE"
    ]
)

print(
    "Abstained minus accepted MAE :",
    float(
        abstained_metrics_627h5a[
            "MAE"
        ]
        -
        accepted_metrics_627h5a[
            "MAE"
        ]
    )
)


# ============================================================
# 11. Risk-error association on official TEST
# ============================================================

risk_627h5a = pd.to_numeric(
    test_scored_627h5a[
        "selective_risk"
    ],
    errors="coerce",
).to_numpy(
    dtype=float
)


risk_error_pearson_627h5a = float(
    pd.Series(
        risk_627h5a
    ).corr(
        pd.Series(
            abs_error_627h5a
        ),
        method="pearson",
    )
)


risk_error_spearman_627h5a = float(
    pd.Series(
        risk_627h5a
    ).corr(
        pd.Series(
            abs_error_627h5a
        ),
        method="spearman",
    )
)


print(
    "\n--- TEST RISK / ERROR ASSOCIATION ---"
)

print(
    "Pearson risk vs |error|  :",
    risk_error_pearson_627h5a
)

print(
    "Spearman risk vs |error| :",
    risk_error_spearman_627h5a
)


# ============================================================
# 12. Final provenance record
# ============================================================

FINAL_FD001_TEST_EVIDENCE_627H5A = {

    "dataset":
        "C-MAPSS FD001",

    "official_test_engines":
        100,

    "target_convention":
        "RUL_cap_125",

    "raw_rul_retained":
        True,

    "policy_frozen_before_labels":
        True,

    "gate":
        str(
            FINAL_SELECTIVE_GATE_627H3B
        ),

    "threshold":
        float(
            FINAL_RELIABILITY_THRESHOLD_627H3B
        ),

    "coverage":
        coverage_627h5a,

    "accepted_engines":
        int(
            accept_627h5a.sum()
        ),

    "abstained_engines":
        int(
            (
                ~accept_627h5a
            ).sum()
        ),

    "full_mae":
        full_metrics_627h5a[
            "MAE"
        ],

    "accepted_mae":
        accepted_metrics_627h5a[
            "MAE"
        ],

    "abstained_mae":
        abstained_metrics_627h5a[
            "MAE"
        ],

    "full_rmse":
        full_metrics_627h5a[
            "RMSE"
        ],

    "accepted_rmse":
        accepted_metrics_627h5a[
            "RMSE"
        ],

    "mae_reduction_pct":
        mae_reduction_pct_627h5a,

    "rmse_reduction_pct":
        rmse_reduction_pct_627h5a,

    "risk_abs_error_pearson":
        risk_error_pearson_627h5a,

    "risk_abs_error_spearman":
        risk_error_spearman_627h5a,

    "model_rerun_after_labels":
        False,

    "mc_rerun_after_labels":
        False,

    "threshold_changed_after_labels":
        False,

    "gate_changed_after_labels":
        False,
}


# ============================================================
# 13. Closure
# ============================================================

print(
    "\n"
    "============================================================\n"
    "STEP 6.27H5-A FINAL SUMMARY\n"
    "============================================================"
)

print(
    "Official TEST RUL labels used       : True"
)

print(
    "Primary target                     : RUL_cap_125"
)

print(
    "Policy frozen before labels        : True"
)

print(
    "TEST engines                       :",
    len(
        test_scored_627h5a
    )
)

print(
    "Accepted / abstained              :",
    int(
        accept_627h5a.sum()
    ),
    "/",
    int(
        (
            ~accept_627h5a
        ).sum()
    )
)

print(
    "Coverage                           :",
    coverage_627h5a
)

print(
    "Full MAE / accepted MAE            :",
    full_metrics_627h5a[
        "MAE"
    ],
    "/",
    accepted_metrics_627h5a[
        "MAE"
    ]
)

print(
    "Full RMSE / accepted RMSE          :",
    full_metrics_627h5a[
        "RMSE"
    ],
    "/",
    accepted_metrics_627h5a[
        "RMSE"
    ]
)

print(
    "Gate changed after labels          : False"
)

print(
    "Threshold changed after labels     : False"
)

print(
    "Model / MC rerun after labels      : False"
)

print(
    "Files written                      : False"
)

print(
    "\n=== END STEP 6.27H5-A ==="
)

In [ ]:
# ============================================================
# STEP 6.27H5-A-R
# COMPACT OFFICIAL FD001 TEST SCORE REVIEW
# ============================================================

print(
    "\n"
    "============================================================\n"
    "STEP 6.27H5-A-R — COMPACT FINAL TEST SCORE REVIEW\n"
    "============================================================"
)


# ------------------------------------------------------------
# Required result objects
# ------------------------------------------------------------

required = [
    "FINAL_FD001_TEST_METRICS_627H5A",
    "FINAL_FD001_TEST_EVIDENCE_627H5A",
]

for name in required:
    if name not in globals():
        raise RuntimeError(
            f"{name} is missing. Run H5-A first."
        )


# ------------------------------------------------------------
# 1. Main population metrics
# ------------------------------------------------------------

print("\n--- PRIMARY FD001 TEST METRICS ---")

print(
    FINAL_FD001_TEST_METRICS_627H5A
    .to_string(index=False)
)


# ------------------------------------------------------------
# 2. Selective improvement
# ------------------------------------------------------------

ev = FINAL_FD001_TEST_EVIDENCE_627H5A

print("\n--- SELECTIVE IMPROVEMENT ---")

print(
    "Coverage              :",
    ev["coverage"]
)

print(
    "Accepted engines      :",
    ev["accepted_engines"]
)

print(
    "Abstained engines     :",
    ev["abstained_engines"]
)

print(
    "Full TEST MAE         :",
    ev["full_mae"]
)

print(
    "Accepted TEST MAE     :",
    ev["accepted_mae"]
)

print(
    "Abstained TEST MAE    :",
    ev["abstained_mae"]
)

print(
    "MAE reduction (%)     :",
    ev["mae_reduction_pct"]
)

print(
    "Full TEST RMSE        :",
    ev["full_rmse"]
)

print(
    "Accepted TEST RMSE    :",
    ev["accepted_rmse"]
)

print(
    "RMSE reduction (%)    :",
    ev["rmse_reduction_pct"]
)


# ------------------------------------------------------------
# 3. Does abstention remove harder cases?
# ------------------------------------------------------------

print("\n--- GATE DISCRIMINATION ---")

print(
    "Abstained - accepted MAE :",
    ev["abstained_mae"]
    -
    ev["accepted_mae"]
)

print(
    "Pearson risk vs |error|   :",
    ev["risk_abs_error_pearson"]
)

print(
    "Spearman risk vs |error|  :",
    ev["risk_abs_error_spearman"]
)


# ------------------------------------------------------------
# 4. Frozen-policy provenance
# ------------------------------------------------------------

print("\n--- EXPERIMENT INTEGRITY ---")

print(
    "Target convention             :",
    ev["target_convention"]
)

print(
    "Policy frozen before labels   :",
    ev["policy_frozen_before_labels"]
)

print(
    "Gate                          :",
    ev["gate"]
)

print(
    "Threshold                     :",
    ev["threshold"]
)

print(
    "Model rerun after labels      :",
    ev["model_rerun_after_labels"]
)

print(
    "MC rerun after labels         :",
    ev["mc_rerun_after_labels"]
)

print(
    "Threshold changed after labels:",
    ev["threshold_changed_after_labels"]
)

print(
    "Gate changed after labels     :",
    ev["gate_changed_after_labels"]
)


print(
    "\n"
    "============================================================\n"
    "STEP 6.27H5-A-R SUMMARY\n"
    "============================================================"
)

print(
    "Official TEST scored : True"
)

print(
    "Primary target       : RUL_cap_125"
)

print(
    "Coverage             :",
    ev["coverage"]
)

print(
    "Full MAE             :",
    ev["full_mae"]
)

print(
    "Accepted MAE         :",
    ev["accepted_mae"]
)

print(
    "Abstained MAE        :",
    ev["abstained_mae"]
)

print(
    "MAE reduction (%)    :",
    ev["mae_reduction_pct"]
)

print(
    "Full RMSE            :",
    ev["full_rmse"]
)

print(
    "Accepted RMSE        :",
    ev["accepted_rmse"]
)

print(
    "RMSE reduction (%)   :",
    ev["rmse_reduction_pct"]
)

print(
    "\n=== END STEP 6.27H5-A-R ==="
)

In [ ]:
# ============================================================
# STEP 6.27H5-B
# FD001 RAW-RUL BENCHMARK COMPARABILITY
#
# IMPORTANT:
#   - frozen decisions are NOT changed
#   - threshold is NOT changed
#   - gate is NOT changed
#   - model / MC are NOT rerun
#
# Primary model-aligned target remains RUL_cap_125.
# This step adds raw official RUL evaluation only for
# benchmark transparency/comparability.
# ============================================================

import numpy as np
import pandas as pd


print(
    "\n"
    "============================================================\n"
    "STEP 6.27H5-B — RAW-RUL BENCHMARK COMPARABILITY\n"
    "============================================================"
)


# ------------------------------------------------------------
# 1. Required scored object
# ------------------------------------------------------------

if "FINAL_FD001_TEST_SCORED_627H5A" not in globals():
    raise RuntimeError(
        "FINAL_FD001_TEST_SCORED_627H5A is missing."
    )


df_627h5b = (
    FINAL_FD001_TEST_SCORED_627H5A
    .copy()
)


required_cols_627h5b = [
    "unit_id",
    "prediction",
    "RUL_raw",
    "RUL_cap_125",
    "selective_risk",
    "accept",
]


print("\n--- REQUIRED COLUMNS ---")

for col in required_cols_627h5b:

    exists = col in df_627h5b.columns

    print(
        f"{col:<24}: {exists}"
    )

    if not exists:
        raise RuntimeError(
            f"Missing required column: {col}"
        )


# ------------------------------------------------------------
# 2. Recover arrays
# ------------------------------------------------------------

prediction = pd.to_numeric(
    df_627h5b["prediction"],
    errors="coerce"
).to_numpy(dtype=float)


rul_raw = pd.to_numeric(
    df_627h5b["RUL_raw"],
    errors="coerce"
).to_numpy(dtype=float)


rul_cap = pd.to_numeric(
    df_627h5b["RUL_cap_125"],
    errors="coerce"
).to_numpy(dtype=float)


risk = pd.to_numeric(
    df_627h5b["selective_risk"],
    errors="coerce"
).to_numpy(dtype=float)


accept = (
    df_627h5b["accept"]
    .to_numpy(dtype=bool)
)


for name, values in [
    ("prediction", prediction),
    ("RUL_raw", rul_raw),
    ("RUL_cap_125", rul_cap),
    ("selective_risk", risk),
]:

    if not np.isfinite(values).all():
        raise RuntimeError(
            f"{name} contains non-finite values."
        )


# ------------------------------------------------------------
# 3. Metric helper
# ------------------------------------------------------------

def metrics_627h5b(
    y_true,
    y_pred
):

    y_true = np.asarray(
        y_true,
        dtype=float
    )

    y_pred = np.asarray(
        y_pred,
        dtype=float
    )

    error = (
        y_pred
        -
        y_true
    )

    return {
        "n":
            int(len(y_true)),

        "MAE":
            float(
                np.mean(
                    np.abs(error)
                )
            ),

        "RMSE":
            float(
                np.sqrt(
                    np.mean(
                        error ** 2
                    )
                )
            ),

        "mean_error":
            float(
                np.mean(error)
            ),

        "median_abs_error":
            float(
                np.median(
                    np.abs(error)
                )
            ),
    }


# ------------------------------------------------------------
# 4. Raw official RUL metrics
# ------------------------------------------------------------

raw_all = metrics_627h5b(
    rul_raw,
    prediction
)


raw_accepted = metrics_627h5b(
    rul_raw[accept],
    prediction[accept]
)


raw_abstained = metrics_627h5b(
    rul_raw[~accept],
    prediction[~accept]
)


raw_mae_reduction_pct = float(
    100.0
    *
    (
        raw_all["MAE"]
        -
        raw_accepted["MAE"]
    )
    /
    raw_all["MAE"]
)


raw_rmse_reduction_pct = float(
    100.0
    *
    (
        raw_all["RMSE"]
        -
        raw_accepted["RMSE"]
    )
    /
    raw_all["RMSE"]
)


# ------------------------------------------------------------
# 5. Risk / error association for BOTH target conventions
# ------------------------------------------------------------

cap_abs_error = np.abs(
    prediction
    -
    rul_cap
)


raw_abs_error = np.abs(
    prediction
    -
    rul_raw
)


cap_pearson = float(
    pd.Series(risk).corr(
        pd.Series(cap_abs_error),
        method="pearson"
    )
)


cap_spearman = float(
    pd.Series(risk).corr(
        pd.Series(cap_abs_error),
        method="spearman"
    )
)


raw_pearson = float(
    pd.Series(risk).corr(
        pd.Series(raw_abs_error),
        method="pearson"
    )
)


raw_spearman = float(
    pd.Series(risk).corr(
        pd.Series(raw_abs_error),
        method="spearman"
    )
)


# ------------------------------------------------------------
# 6. Dual-target table
# ------------------------------------------------------------

FINAL_FD001_DUAL_TARGET_METRICS_627H5B = pd.DataFrame(
    [
        {
            "target": "RUL_cap_125",
            "population": "all_test",
            "n": 100,
            "MAE": float(
                np.mean(cap_abs_error)
            ),
            "RMSE": float(
                np.sqrt(
                    np.mean(
                        (
                            prediction
                            -
                            rul_cap
                        ) ** 2
                    )
                )
            ),
        },

        {
            "target": "RUL_cap_125",
            "population": "accepted",
            "n": int(accept.sum()),
            "MAE": float(
                np.mean(
                    cap_abs_error[accept]
                )
            ),
            "RMSE": float(
                np.sqrt(
                    np.mean(
                        (
                            prediction[accept]
                            -
                            rul_cap[accept]
                        ) ** 2
                    )
                )
            ),
        },

        {
            "target": "RUL_cap_125",
            "population": "abstained",
            "n": int((~accept).sum()),
            "MAE": float(
                np.mean(
                    cap_abs_error[~accept]
                )
            ),
            "RMSE": float(
                np.sqrt(
                    np.mean(
                        (
                            prediction[~accept]
                            -
                            rul_cap[~accept]
                        ) ** 2
                    )
                )
            ),
        },

        {
            "target": "RUL_raw",
            "population": "all_test",
            "n": raw_all["n"],
            "MAE": raw_all["MAE"],
            "RMSE": raw_all["RMSE"],
        },

        {
            "target": "RUL_raw",
            "population": "accepted",
            "n": raw_accepted["n"],
            "MAE": raw_accepted["MAE"],
            "RMSE": raw_accepted["RMSE"],
        },

        {
            "target": "RUL_raw",
            "population": "abstained",
            "n": raw_abstained["n"],
            "MAE": raw_abstained["MAE"],
            "RMSE": raw_abstained["RMSE"],
        },
    ]
)


print(
    "\n--- DUAL-TARGET METRICS ---"
)

print(
    FINAL_FD001_DUAL_TARGET_METRICS_627H5B
    .to_string(index=False)
)


# ------------------------------------------------------------
# 7. Raw benchmark selective effect
# ------------------------------------------------------------

print(
    "\n--- RAW-RUL SELECTIVE EFFECT ---"
)

print(
    "Coverage                :",
    float(accept.mean())
)

print(
    "Raw full MAE            :",
    raw_all["MAE"]
)

print(
    "Raw accepted MAE        :",
    raw_accepted["MAE"]
)

print(
    "Raw abstained MAE       :",
    raw_abstained["MAE"]
)

print(
    "Raw MAE reduction (%)   :",
    raw_mae_reduction_pct
)

print(
    "Raw full RMSE           :",
    raw_all["RMSE"]
)

print(
    "Raw accepted RMSE       :",
    raw_accepted["RMSE"]
)

print(
    "Raw abstained RMSE      :",
    raw_abstained["RMSE"]
)

print(
    "Raw RMSE reduction (%)  :",
    raw_rmse_reduction_pct
)


# ------------------------------------------------------------
# 8. Risk-error association closure
# ------------------------------------------------------------

print(
    "\n--- RISK / ABSOLUTE-ERROR ASSOCIATION ---"
)

print(
    "Cap-125 Pearson  :",
    cap_pearson
)

print(
    "Cap-125 Spearman :",
    cap_spearman
)

print(
    "Raw-RUL Pearson  :",
    raw_pearson
)

print(
    "Raw-RUL Spearman :",
    raw_spearman
)


# ------------------------------------------------------------
# 9. Cap-effect audit
# ------------------------------------------------------------

above_cap = (
    rul_raw > 125.0
)


print(
    "\n--- TARGET-CONVENTION AUDIT ---"
)

print(
    "Raw RUL > 125 engines :",
    int(
        above_cap.sum()
    )
)

print(
    "Of those accepted      :",
    int(
        (
            above_cap
            &
            accept
        ).sum()
    )
)

print(
    "Of those abstained     :",
    int(
        (
            above_cap
            &
            (~accept)
        ).sum()
    )
)


# ------------------------------------------------------------
# 10. Freeze evidence
# ------------------------------------------------------------

FINAL_FD001_DUAL_TARGET_EVIDENCE_627H5B = {

    "coverage":
        float(
            accept.mean()
        ),

    "raw_full_mae":
        raw_all["MAE"],

    "raw_accepted_mae":
        raw_accepted["MAE"],

    "raw_abstained_mae":
        raw_abstained["MAE"],

    "raw_mae_reduction_pct":
        raw_mae_reduction_pct,

    "raw_full_rmse":
        raw_all["RMSE"],

    "raw_accepted_rmse":
        raw_accepted["RMSE"],

    "raw_abstained_rmse":
        raw_abstained["RMSE"],

    "raw_rmse_reduction_pct":
        raw_rmse_reduction_pct,

    "cap125_risk_error_pearson":
        cap_pearson,

    "cap125_risk_error_spearman":
        cap_spearman,

    "raw_risk_error_pearson":
        raw_pearson,

    "raw_risk_error_spearman":
        raw_spearman,

    "threshold_changed":
        False,

    "gate_changed":
        False,

    "model_rerun":
        False,

    "mc_rerun":
        False,
}


print(
    "\n"
    "============================================================\n"
    "STEP 6.27H5-B SUMMARY\n"
    "============================================================"
)

print(
    "Frozen policy unchanged : True"
)

print(
    "Primary target remains  : RUL_cap_125"
)

print(
    "Raw RUL also evaluated  : True"
)

print(
    "Coverage                :",
    float(accept.mean())
)

print(
    "Raw full / accepted MAE :",
    raw_all["MAE"],
    "/",
    raw_accepted["MAE"]
)

print(
    "Raw MAE reduction (%)   :",
    raw_mae_reduction_pct
)

print(
    "Raw full / accepted RMSE:",
    raw_all["RMSE"],
    "/",
    raw_accepted["RMSE"]
)

print(
    "Raw RMSE reduction (%)  :",
    raw_rmse_reduction_pct
)

print(
    "Gate changed            : False"
)

print(
    "Threshold changed       : False"
)

print(
    "Model / MC rerun        : False"
)

print(
    "Files written           : False"
)

print(
    "\n=== END STEP 6.27H5-B ==="
)

In [ ]:
# ============================================================
# STEP 6.27H5-C
# ENGINE-LEVEL BOOTSTRAP UNCERTAINTY FOR FINAL FD001 RESULTS
#
# Purpose:
#   Quantify uncertainty around the frozen selective-RUL
#   improvement on the official FD001 TEST population.
#
# Bootstrap unit:
#   ENGINE ENDPOINT (100 independent TEST engines)
#
# Evaluates BOTH:
#   1. RUL_cap_125  [PRIMARY]
#   2. RUL_raw      [SECONDARY / benchmark transparency]
#
# IMPORTANT:
#   - accept / abstain decisions remain fixed
#   - threshold remains 0.95
#   - gate remains uncertainty_only
#   - no model rerun
#   - no MC rerun
#   - no threshold tuning
#   - no files written
# ============================================================

import numpy as np
import pandas as pd


print(
    "\n"
    "============================================================\n"
    "STEP 6.27H5-C — ENGINE-LEVEL BOOTSTRAP UNCERTAINTY\n"
    "============================================================"
)


# ============================================================
# 1. Required frozen scored object
# ============================================================

if "FINAL_FD001_TEST_SCORED_627H5A" not in globals():
    raise RuntimeError(
        "FINAL_FD001_TEST_SCORED_627H5A is missing."
    )


df_627h5c = (
    FINAL_FD001_TEST_SCORED_627H5A
    .copy()
    .sort_values("unit_id")
    .reset_index(drop=True)
)


required_cols_627h5c = [
    "unit_id",
    "prediction",
    "RUL_cap_125",
    "RUL_raw",
    "accept",
]


print("\n--- REQUIRED COLUMNS ---")

for col in required_cols_627h5c:

    exists = col in df_627h5c.columns

    print(
        f"{col:<22}: {exists}"
    )

    if not exists:
        raise RuntimeError(
            f"Missing required column: {col}"
        )


# ============================================================
# 2. Structural contract
# ============================================================

n_engines_627h5c = len(
    df_627h5c
)


if n_engines_627h5c != 100:
    raise RuntimeError(
        "Expected exactly 100 FD001 TEST engines."
    )


if df_627h5c["unit_id"].nunique() != 100:
    raise RuntimeError(
        "Expected exactly 100 unique TEST engines."
    )


prediction_627h5c = pd.to_numeric(
    df_627h5c["prediction"],
    errors="coerce"
).to_numpy(dtype=float)


cap_target_627h5c = pd.to_numeric(
    df_627h5c["RUL_cap_125"],
    errors="coerce"
).to_numpy(dtype=float)


raw_target_627h5c = pd.to_numeric(
    df_627h5c["RUL_raw"],
    errors="coerce"
).to_numpy(dtype=float)


accept_627h5c = (
    df_627h5c["accept"]
    .to_numpy(dtype=bool)
)


print("\n--- FROZEN TEST CONTRACT ---")

print(
    "TEST engines       :",
    n_engines_627h5c
)

print(
    "Accepted engines   :",
    int(accept_627h5c.sum())
)

print(
    "Abstained engines  :",
    int((~accept_627h5c).sum())
)

print(
    "Observed coverage  :",
    float(accept_627h5c.mean())
)


# ============================================================
# 3. Bootstrap configuration
# ============================================================

BOOTSTRAP_REPS_627H5C = 10000
BOOTSTRAP_SEED_627H5C = 62750


rng_627h5c = np.random.default_rng(
    BOOTSTRAP_SEED_627H5C
)


print("\n--- BOOTSTRAP CONFIGURATION ---")

print(
    "Replicates :",
    BOOTSTRAP_REPS_627H5C
)

print(
    "Seed       :",
    BOOTSTRAP_SEED_627H5C
)

print(
    "Unit       : TEST engine endpoint"
)


# ============================================================
# 4. Metric calculation
# ============================================================

metric_names_627h5c = [
    "coverage",
    "full_mae",
    "accepted_mae",
    "abstained_mae",
    "mae_gain",
    "mae_reduction_pct",
    "full_rmse",
    "accepted_rmse",
    "abstained_rmse",
    "rmse_gain",
    "rmse_reduction_pct",
    "abstained_minus_accepted_mae",
]


def metric_vector_627h5c(
    prediction,
    target,
    accept
):

    prediction = np.asarray(
        prediction,
        dtype=float
    )

    target = np.asarray(
        target,
        dtype=float
    )

    accept = np.asarray(
        accept,
        dtype=bool
    )

    if (
        accept.sum() == 0
        or
        (~accept).sum() == 0
    ):
        return None


    error = (
        prediction
        -
        target
    )

    abs_error = np.abs(
        error
    )

    sq_error = (
        error ** 2
    )


    full_mae = float(
        abs_error.mean()
    )

    accepted_mae = float(
        abs_error[
            accept
        ].mean()
    )

    abstained_mae = float(
        abs_error[
            ~accept
        ].mean()
    )


    full_rmse = float(
        np.sqrt(
            sq_error.mean()
        )
    )

    accepted_rmse = float(
        np.sqrt(
            sq_error[
                accept
            ].mean()
        )
    )

    abstained_rmse = float(
        np.sqrt(
            sq_error[
                ~accept
            ].mean()
        )
    )


    mae_gain = float(
        full_mae
        -
        accepted_mae
    )

    rmse_gain = float(
        full_rmse
        -
        accepted_rmse
    )


    mae_reduction_pct = float(
        100.0
        *
        mae_gain
        /
        full_mae
    )


    rmse_reduction_pct = float(
        100.0
        *
        rmse_gain
        /
        full_rmse
    )


    return np.asarray(
        [
            float(
                accept.mean()
            ),

            full_mae,
            accepted_mae,
            abstained_mae,
            mae_gain,
            mae_reduction_pct,
            full_rmse,
            accepted_rmse,
            abstained_rmse,
            rmse_gain,
            rmse_reduction_pct,

            float(
                abstained_mae
                -
                accepted_mae
            ),
        ],
        dtype=float
    )


# ============================================================
# 5. Original point estimates
# ============================================================

point_cap_627h5c = (
    metric_vector_627h5c(
        prediction_627h5c,
        cap_target_627h5c,
        accept_627h5c,
    )
)


point_raw_627h5c = (
    metric_vector_627h5c(
        prediction_627h5c,
        raw_target_627h5c,
        accept_627h5c,
    )
)


# ============================================================
# 6. Bootstrap
#
# Engines are sampled WITH replacement.
#
# Crucially:
#   the already-frozen accept status travels with each
#   resampled engine.
# ============================================================

boot_cap_627h5c = np.full(
    (
        BOOTSTRAP_REPS_627H5C,
        len(metric_names_627h5c),
    ),
    np.nan,
    dtype=float
)


boot_raw_627h5c = np.full(
    (
        BOOTSTRAP_REPS_627H5C,
        len(metric_names_627h5c),
    ),
    np.nan,
    dtype=float
)


for b_627h5c in range(
    BOOTSTRAP_REPS_627H5C
):

    idx_627h5c = rng_627h5c.integers(
        low=0,
        high=n_engines_627h5c,
        size=n_engines_627h5c,
    )


    pred_b_627h5c = (
        prediction_627h5c[
            idx_627h5c
        ]
    )


    accept_b_627h5c = (
        accept_627h5c[
            idx_627h5c
        ]
    )


    cap_result_627h5c = (
        metric_vector_627h5c(
            pred_b_627h5c,
            cap_target_627h5c[
                idx_627h5c
            ],
            accept_b_627h5c,
        )
    )


    raw_result_627h5c = (
        metric_vector_627h5c(
            pred_b_627h5c,
            raw_target_627h5c[
                idx_627h5c
            ],
            accept_b_627h5c,
        )
    )


    if cap_result_627h5c is not None:

        boot_cap_627h5c[
            b_627h5c,
            :
        ] = (
            cap_result_627h5c
        )


    if raw_result_627h5c is not None:

        boot_raw_627h5c[
            b_627h5c,
            :
        ] = (
            raw_result_627h5c
        )


# ============================================================
# 7. Valid replicate check
# ============================================================

valid_cap_627h5c = np.isfinite(
    boot_cap_627h5c
).all(
    axis=1
)


valid_raw_627h5c = np.isfinite(
    boot_raw_627h5c
).all(
    axis=1
)


print("\n--- VALID BOOTSTRAP REPLICATES ---")

print(
    "Cap-125:",
    int(
        valid_cap_627h5c.sum()
    ),
    "/",
    BOOTSTRAP_REPS_627H5C
)

print(
    "Raw RUL:",
    int(
        valid_raw_627h5c.sum()
    ),
    "/",
    BOOTSTRAP_REPS_627H5C
)


if valid_cap_627h5c.sum() < 9500:
    raise RuntimeError(
        "Too many invalid Cap-125 bootstrap replicates."
    )


if valid_raw_627h5c.sum() < 9500:
    raise RuntimeError(
        "Too many invalid raw-RUL bootstrap replicates."
    )


# ============================================================
# 8. Confidence-interval helper
# ============================================================

def ci_table_627h5c(
    target_name,
    point_values,
    bootstrap_values,
    valid_mask
):

    rows = []


    use_metrics = [
        "coverage",
        "full_mae",
        "accepted_mae",
        "abstained_mae",
        "mae_gain",
        "mae_reduction_pct",
        "full_rmse",
        "accepted_rmse",
        "rmse_gain",
        "rmse_reduction_pct",
        "abstained_minus_accepted_mae",
    ]


    valid_boot = (
        bootstrap_values[
            valid_mask
        ]
    )


    for metric in use_metrics:

        j = metric_names_627h5c.index(
            metric
        )


        values = (
            valid_boot[
                :,
                j
            ]
        )


        q025, q50, q975 = np.quantile(
            values,
            [
                0.025,
                0.50,
                0.975,
            ]
        )


        rows.append(
            {
                "target":
                    target_name,

                "metric":
                    metric,

                "point":
                    float(
                        point_values[j]
                    ),

                "ci_2.5":
                    float(q025),

                "bootstrap_median":
                    float(q50),

                "ci_97.5":
                    float(q975),
            }
        )


    return pd.DataFrame(
        rows
    )


ci_cap_627h5c = ci_table_627h5c(
    "RUL_cap_125",
    point_cap_627h5c,
    boot_cap_627h5c,
    valid_cap_627h5c,
)


ci_raw_627h5c = ci_table_627h5c(
    "RUL_raw",
    point_raw_627h5c,
    boot_raw_627h5c,
    valid_raw_627h5c,
)


FINAL_FD001_BOOTSTRAP_CI_627H5C = pd.concat(
    [
        ci_cap_627h5c,
        ci_raw_627h5c,
    ],
    ignore_index=True
)


# ============================================================
# 9. Stability of selective benefit
# ============================================================

def stability_row_627h5c(
    target_name,
    bootstrap_values,
    valid_mask
):

    valid_boot = (
        bootstrap_values[
            valid_mask
        ]
    )


    mae_gain_col = metric_names_627h5c.index(
        "mae_gain"
    )

    rmse_gain_col = metric_names_627h5c.index(
        "rmse_gain"
    )

    discrimination_col = metric_names_627h5c.index(
        "abstained_minus_accepted_mae"
    )


    return {
        "target":
            target_name,

        "bootstrap_replicates":
            int(
                len(valid_boot)
            ),

        "fraction_MAE_gain_positive":
            float(
                np.mean(
                    valid_boot[
                        :,
                        mae_gain_col
                    ]
                    >
                    0.0
                )
            ),

        "fraction_RMSE_gain_positive":
            float(
                np.mean(
                    valid_boot[
                        :,
                        rmse_gain_col
                    ]
                    >
                    0.0
                )
            ),

        "fraction_abstained_MAE_higher":
            float(
                np.mean(
                    valid_boot[
                        :,
                        discrimination_col
                    ]
                    >
                    0.0
                )
            ),
    }


FINAL_FD001_BOOTSTRAP_STABILITY_627H5C = pd.DataFrame(
    [
        stability_row_627h5c(
            "RUL_cap_125",
            boot_cap_627h5c,
            valid_cap_627h5c,
        ),

        stability_row_627h5c(
            "RUL_raw",
            boot_raw_627h5c,
            valid_raw_627h5c,
        ),
    ]
)


# ============================================================
# 10. Compact publication-relevant output
# ============================================================

important_metrics_627h5c = [
    "coverage",
    "mae_gain",
    "mae_reduction_pct",
    "rmse_gain",
    "rmse_reduction_pct",
    "abstained_minus_accepted_mae",
]


print(
    "\n--- 95% ENGINE-LEVEL BOOTSTRAP INTERVALS ---"
)

print(
    FINAL_FD001_BOOTSTRAP_CI_627H5C.loc[
        FINAL_FD001_BOOTSTRAP_CI_627H5C[
            "metric"
        ].isin(
            important_metrics_627h5c
        )
    ]
    .to_string(
        index=False
    )
)


print(
    "\n--- BOOTSTRAP BENEFIT STABILITY ---"
)

print(
    FINAL_FD001_BOOTSTRAP_STABILITY_627H5C
    .to_string(
        index=False
    )
)


# ============================================================
# 11. Freeze statistical evidence
# ============================================================

FINAL_FD001_BOOTSTRAP_EVIDENCE_627H5C = {

    "bootstrap_unit":
        "official TEST engine endpoint",

    "bootstrap_replicates":
        BOOTSTRAP_REPS_627H5C,

    "seed":
        BOOTSTRAP_SEED_627H5C,

    "confidence_interval":
        "percentile 95%",

    "primary_target":
        "RUL_cap_125",

    "secondary_target":
        "RUL_raw",

    "gate_changed":
        False,

    "threshold_changed":
        False,

    "model_rerun":
        False,

    "mc_rerun":
        False,
}


print(
    "\n"
    "============================================================\n"
    "STEP 6.27H5-C SUMMARY\n"
    "============================================================"
)

print(
    "Bootstrap unit           : TEST engine"
)

print(
    "Replicates               :",
    BOOTSTRAP_REPS_627H5C
)

print(
    "CI                       : percentile 95%"
)

print(
    "Primary target           : RUL_cap_125"
)

print(
    "Secondary target         : RUL_raw"
)

print(
    "Frozen decisions changed : False"
)

print(
    "Gate changed             : False"
)

print(
    "Threshold changed        : False"
)

print(
    "Model / MC rerun         : False"
)

print(
    "Files written            : False"
)

print(
    "\n=== END STEP 6.27H5-C ==="
)

In [ ]:
# ============================================================
# STEP 6.27H5-C-R
# COMPACT BOOTSTRAP EVIDENCE REVIEW
#
# No recomputation.
# No TEST-policy change.
# No files written.
# ============================================================

import pandas as pd

print(
    "\n"
    "============================================================\n"
    "STEP 6.27H5-C-R — COMPACT BOOTSTRAP EVIDENCE REVIEW\n"
    "============================================================"
)


# ------------------------------------------------------------
# 1. Required result objects
# ------------------------------------------------------------

required_627h5cr = [
    "FINAL_FD001_BOOTSTRAP_CI_627H5C",
    "FINAL_FD001_BOOTSTRAP_STABILITY_627H5C",
    "FINAL_FD001_BOOTSTRAP_EVIDENCE_627H5C",
]


print("\n--- REQUIRED RESULT OBJECTS ---")

for name in required_627h5cr:

    exists = name in globals()

    print(
        f"{name:<46}: {exists}"
    )

    if not exists:
        raise RuntimeError(
            f"Missing result object: {name}"
        )


# ------------------------------------------------------------
# 2. Publication-relevant confidence intervals
# ------------------------------------------------------------

important_metrics_627h5cr = [
    "coverage",
    "mae_gain",
    "mae_reduction_pct",
    "rmse_gain",
    "rmse_reduction_pct",
    "abstained_minus_accepted_mae",
]


review_ci_627h5cr = (
    FINAL_FD001_BOOTSTRAP_CI_627H5C.loc[
        FINAL_FD001_BOOTSTRAP_CI_627H5C[
            "metric"
        ].isin(
            important_metrics_627h5cr
        )
    ]
    .copy()
)


print(
    "\n--- 95% ENGINE-LEVEL BOOTSTRAP INTERVALS ---"
)

print(
    review_ci_627h5cr
    .to_string(
        index=False
    )
)


# ------------------------------------------------------------
# 3. Benefit stability
# ------------------------------------------------------------

print(
    "\n--- BOOTSTRAP BENEFIT STABILITY ---"
)

print(
    FINAL_FD001_BOOTSTRAP_STABILITY_627H5C
    .to_string(
        index=False
    )
)


# ------------------------------------------------------------
# 4. Primary target evidence extraction
# ------------------------------------------------------------

primary_627h5cr = (
    review_ci_627h5cr[
        review_ci_627h5cr[
            "target"
        ]
        ==
        "RUL_cap_125"
    ]
    .set_index(
        "metric"
    )
)


mae_gain_low_627h5cr = float(
    primary_627h5cr.loc[
        "mae_gain",
        "ci_2.5"
    ]
)


mae_gain_high_627h5cr = float(
    primary_627h5cr.loc[
        "mae_gain",
        "ci_97.5"
    ]
)


rmse_gain_low_627h5cr = float(
    primary_627h5cr.loc[
        "rmse_gain",
        "ci_2.5"
    ]
)


rmse_gain_high_627h5cr = float(
    primary_627h5cr.loc[
        "rmse_gain",
        "ci_97.5"
    ]
)


disc_low_627h5cr = float(
    primary_627h5cr.loc[
        "abstained_minus_accepted_mae",
        "ci_2.5"
    ]
)


disc_high_627h5cr = float(
    primary_627h5cr.loc[
        "abstained_minus_accepted_mae",
        "ci_97.5"
    ]
)


# ------------------------------------------------------------
# 5. Evidence flags
#
# These are descriptive bootstrap checks, not hypothesis-test
# p-values.
# ------------------------------------------------------------

PRIMARY_MAE_GAIN_CI_POSITIVE_627H5CR = bool(
    mae_gain_low_627h5cr > 0.0
)


PRIMARY_RMSE_GAIN_CI_POSITIVE_627H5CR = bool(
    rmse_gain_low_627h5cr > 0.0
)


PRIMARY_DISCRIMINATION_CI_POSITIVE_627H5CR = bool(
    disc_low_627h5cr > 0.0
)


print(
    "\n--- PRIMARY RUL_cap_125 EVIDENCE ---"
)

print(
    "MAE-gain 95% CI                :",
    (
        mae_gain_low_627h5cr,
        mae_gain_high_627h5cr,
    )
)

print(
    "MAE-gain CI entirely > 0       :",
    PRIMARY_MAE_GAIN_CI_POSITIVE_627H5CR
)

print(
    "RMSE-gain 95% CI               :",
    (
        rmse_gain_low_627h5cr,
        rmse_gain_high_627h5cr,
    )
)

print(
    "RMSE-gain CI entirely > 0      :",
    PRIMARY_RMSE_GAIN_CI_POSITIVE_627H5CR
)

print(
    "Abstained-minus-accepted MAE CI:",
    (
        disc_low_627h5cr,
        disc_high_627h5cr,
    )
)

print(
    "Discrimination CI > 0          :",
    PRIMARY_DISCRIMINATION_CI_POSITIVE_627H5CR
)


# ------------------------------------------------------------
# 6. Final compact closure
# ------------------------------------------------------------

print(
    "\n"
    "============================================================\n"
    "STEP 6.27H5-C-R SUMMARY\n"
    "============================================================"
)

print(
    "Primary target                   : RUL_cap_125"
)

print(
    "MAE selective gain CI positive  :",
    PRIMARY_MAE_GAIN_CI_POSITIVE_627H5CR
)

print(
    "RMSE selective gain CI positive :",
    PRIMARY_RMSE_GAIN_CI_POSITIVE_627H5CR
)

print(
    "Rejected-set MAE CI higher      :",
    PRIMARY_DISCRIMINATION_CI_POSITIVE_627H5CR
)

print(
    "Bootstrap recomputed             : False"
)

print(
    "Gate changed                     : False"
)

print(
    "Threshold changed                : False"
)

print(
    "Model / MC rerun                 : False"
)

print(
    "Files written                    : False"
)

print(
    "\n=== END STEP 6.27H5-C-R ==="
)

In [ ]:
# ============================================================
# STEP 6.27H5-C-R2
# FINAL BOOTSTRAP STABILITY CLOSURE
# ============================================================

print(
    "\n"
    "============================================================\n"
    "STEP 6.27H5-C-R2 — FINAL BOOTSTRAP STABILITY CLOSURE\n"
    "============================================================"
)

print(
    FINAL_FD001_BOOTSTRAP_STABILITY_627H5C
    .to_string(index=False)
)

print(
    "\nInterpretation aid:"
)

for _, row in FINAL_FD001_BOOTSTRAP_STABILITY_627H5C.iterrows():

    print(
        f"\nTarget: {row['target']}"
    )

    print(
        "Fraction MAE gain > 0           :",
        row["fraction_MAE_gain_positive"]
    )

    print(
        "Fraction RMSE gain > 0          :",
        row["fraction_RMSE_gain_positive"]
    )

    print(
        "Fraction abstained MAE > accepted:",
        row["fraction_abstained_MAE_higher"]
    )

print(
    "\nGate changed       : False"
)

print(
    "Threshold changed  : False"
)

print(
    "Model / MC rerun   : False"
)

print(
    "Files written      : False"
)

print(
    "\n=== END STEP 6.27H5-C-R2 ==="
)

In [ ]:
# ============================================================
# STEP 6.27H5-D
# FREEZE / PERSIST FINAL FD001 EVIDENCE
#
# Purpose:
#   Save the completed FD001 experiment exactly as obtained.
#
# IMPORTANT:
#   - NO recomputation
#   - NO model rerun
#   - NO MC rerun
#   - NO threshold change
#   - NO gate change
#   - NO result editing
# ============================================================

from pathlib import Path
import json
import pandas as pd
import numpy as np


print(
    "\n"
    "============================================================\n"
    "STEP 6.27H5-D — FREEZE FINAL FD001 EVIDENCE\n"
    "============================================================"
)


# ============================================================
# 1. Required final objects
# ============================================================

required_627h5d = [
    "FINAL_FD001_TEST_SCORED_627H5A",
    "FINAL_FD001_TEST_METRICS_627H5A",
    "FINAL_FD001_DUAL_TARGET_METRICS_627H5B",
    "FINAL_FD001_BOOTSTRAP_CI_627H5C",
    "FINAL_FD001_BOOTSTRAP_STABILITY_627H5C",
    "FINAL_FD001_TEST_EVIDENCE_627H5A",
    "FINAL_FD001_DUAL_TARGET_EVIDENCE_627H5B",
    "FINAL_FD001_BOOTSTRAP_EVIDENCE_627H5C",
]


print("\n--- REQUIRED FINAL OBJECTS ---")

for name in required_627h5d:

    exists = name in globals()

    print(
        f"{name:<48}: {exists}"
    )

    if not exists:
        raise RuntimeError(
            f"Missing required final object: {name}"
        )


# ============================================================
# 2. Locate project root
# ============================================================

cwd_627h5d = Path.cwd()


if (cwd_627h5d / "06_results").exists():

    project_root_627h5d = cwd_627h5d

elif (cwd_627h5d.parent / "06_results").exists():

    project_root_627h5d = cwd_627h5d.parent

else:

    raise FileNotFoundError(
        "Could not locate project root containing 06_results."
    )


output_dir_627h5d = (
    project_root_627h5d
    /
    "06_results"
    /
    "fd001_final"
)


output_dir_627h5d.mkdir(
    parents=True,
    exist_ok=True
)


print(
    "\nOutput directory:",
    output_dir_627h5d
)


# ============================================================
# 3. Save canonical scored engine table
# ============================================================

path_scored_627h5d = (
    output_dir_627h5d
    /
    "fd001_final_engine_results.csv"
)


FINAL_FD001_TEST_SCORED_627H5A.to_csv(
    path_scored_627h5d,
    index=False
)


# ============================================================
# 4. Save main metric tables
# ============================================================

path_primary_627h5d = (
    output_dir_627h5d
    /
    "fd001_primary_metrics.csv"
)


FINAL_FD001_TEST_METRICS_627H5A.to_csv(
    path_primary_627h5d,
    index=False
)


path_dual_627h5d = (
    output_dir_627h5d
    /
    "fd001_dual_target_metrics.csv"
)


FINAL_FD001_DUAL_TARGET_METRICS_627H5B.to_csv(
    path_dual_627h5d,
    index=False
)


path_ci_627h5d = (
    output_dir_627h5d
    /
    "fd001_bootstrap_95ci.csv"
)


FINAL_FD001_BOOTSTRAP_CI_627H5C.to_csv(
    path_ci_627h5d,
    index=False
)


path_stability_627h5d = (
    output_dir_627h5d
    /
    "fd001_bootstrap_stability.csv"
)


FINAL_FD001_BOOTSTRAP_STABILITY_627H5C.to_csv(
    path_stability_627h5d,
    index=False
)


# ============================================================
# 5. JSON-safe conversion helper
# ============================================================

def json_safe_627h5d(value):

    if isinstance(
        value,
        (
            np.integer,
            np.int8,
            np.int16,
            np.int32,
            np.int64,
        )
    ):
        return int(value)

    if isinstance(
        value,
        (
            np.floating,
            np.float16,
            np.float32,
            np.float64,
        )
    ):
        return float(value)

    if isinstance(
        value,
        np.bool_
    ):
        return bool(value)

    if isinstance(
        value,
        dict
    ):
        return {
            str(k):
            json_safe_627h5d(v)

            for k, v in value.items()
        }

    if isinstance(
        value,
        (list, tuple)
    ):
        return [
            json_safe_627h5d(v)
            for v in value
        ]

    return value


# ============================================================
# 6. Freeze provenance / evidence metadata
# ============================================================

final_evidence_627h5d = {

    "experiment":
        "FD001 final selective-RUL evaluation",

    "primary_target":
        "RUL_cap_125",

    "secondary_target":
        "RUL_raw",

    "gate":
        "uncertainty_only",

    "threshold":
        0.95,

    "risk_orientation":
        "higher_is_worse",

    "policy_frozen_before_test_labels":
        True,

    "test_engines":
        100,

    "accepted_engines":
        37,

    "abstained_engines":
        63,

    "coverage":
        0.37,

    "primary_full_mae":
        10.322472620010377,

    "primary_accepted_mae":
        8.18384464367016,

    "primary_abstained_mae":
        11.57849225162567,

    "primary_mae_reduction_pct":
        20.718175335185023,

    "primary_full_rmse":
        14.02137228896183,

    "primary_accepted_rmse":
        11.633176004334077,

    "primary_rmse_reduction_pct":
        17.03254314492337,

    "bootstrap_replicates":
        10000,

    "bootstrap_seed":
        62750,

    "primary_fraction_mae_gain_positive":
        0.9643,

    "primary_fraction_rmse_gain_positive":
        0.9329,

    "primary_fraction_abstained_mae_higher":
        0.9643,

    "primary_mae_gain_95ci":
        [
            -0.210485,
            4.514620,
        ],

    "primary_rmse_gain_95ci":
        [
            -0.697134,
            5.953767,
        ],

    "interpretation":
        (
            "Positive selective-performance point estimates "
            "with strong bootstrap directional stability, "
            "but 95% percentile confidence intervals include zero."
        ),

    "threshold_changed_after_test":
        False,

    "gate_changed_after_test":
        False,

    "model_rerun_after_test":
        False,

    "mc_rerun_after_test":
        False,
}


final_evidence_627h5d[
    "source_H5A"
] = (
    FINAL_FD001_TEST_EVIDENCE_627H5A
)


final_evidence_627h5d[
    "source_H5B"
] = (
    FINAL_FD001_DUAL_TARGET_EVIDENCE_627H5B
)


final_evidence_627h5d[
    "source_H5C"
] = (
    FINAL_FD001_BOOTSTRAP_EVIDENCE_627H5C
)


final_evidence_627h5d = (
    json_safe_627h5d(
        final_evidence_627h5d
    )
)


path_json_627h5d = (
    output_dir_627h5d
    /
    "fd001_final_evidence.json"
)


with open(
    path_json_627h5d,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        final_evidence_627h5d,
        f,
        indent=2,
    )


# ============================================================
# 7. Verify files
# ============================================================

written_files_627h5d = [
    path_scored_627h5d,
    path_primary_627h5d,
    path_dual_627h5d,
    path_ci_627h5d,
    path_stability_627h5d,
    path_json_627h5d,
]


print("\n--- WRITTEN FILES ---")

for path in written_files_627h5d:

    print(
        path.name,
        " | exists =",
        path.exists(),
        " | bytes =",
        (
            path.stat().st_size
            if path.exists()
            else None
        )
    )


all_written_627h5d = all(
    path.exists()
    and
    path.stat().st_size > 0

    for path in written_files_627h5d
)


# ============================================================
# 8. Freeze completion flag
# ============================================================

FD001_FINAL_EVIDENCE_FROZEN_627H5D = bool(
    all_written_627h5d
)


print(
    "\n"
    "============================================================\n"
    "STEP 6.27H5-D FINAL SUMMARY\n"
    "============================================================"
)

print(
    "FD001 evidence frozen       :",
    FD001_FINAL_EVIDENCE_FROZEN_627H5D
)

print(
    "Files written               :",
    len(
        written_files_627h5d
    )
)

print(
    "Primary target              : RUL_cap_125"
)

print(
    "Gate                        : uncertainty_only"
)

print(
    "Threshold                   : 0.95"
)

print(
    "Coverage                    : 0.37"
)

print(
    "Primary MAE reduction (%)   : 20.718175335185023"
)

print(
    "Primary RMSE reduction (%)  : 17.03254314492337"
)

print(
    "95% MAE-gain CI excludes 0  : False"
)

print(
    "Research decision modified  : False"
)

print(
    "\n=== END STEP 6.27H5-D ==="
)